# Credit Card Default — Ultra-Extensive Exploratory Analysis (Taiwan, Apr–Sep 2005)

**Goal:** understand *why and where* clients default. This notebook does **not** build predictive models for deployment.
Where a model appears (Section 9), it is used only as a **pattern-discovery lens** (interaction mining, rule extraction), never as the deliverable.

**Downstream use:** every analytic view here is designed to map onto a **Tableau** sheet. Section 21 exports clean, labelled, Tableau-ready CSVs.

**Data:** UCI / Kaggle *Default of Credit Card Clients* — 30,000 clients, 23 features, binary target (`default.payment.next.month`).

| Block | Columns | Meaning |
|---|---|---|
| Static | `LIMIT_BAL`, `SEX`, `EDUCATION`, `MARRIAGE`, `AGE` | credit line (NT$, includes family/supplementary), demographics |
| Repayment status | `PAY_0, PAY_2..PAY_6` → renamed `PAY_1..PAY_6` | Sep → Apr. −2 no consumption, −1 paid in full, 0 revolving (min paid), 1..9 months delay |
| Bill statement | `BILL_AMT1..6` | Sep → Apr statement balance |
| Payment | `PAY_AMT1..6` | amount paid in Sep → Apr (PAY_AMT*i* settles BILL_AMT*i+1*) |
| Target | `DEFAULT` | 1 = default next month (Oct 2005) |

**Index-to-month map (important, it is reversed):** 1 = Sep, 2 = Aug, 3 = Jul, 4 = Jun, 5 = May, 6 = Apr.

### Contents
**Part I - Foundations**
0. Setup, data load, cleaning, core feature layer
1. Data quality, undocumented codes, anomalies
2. Univariate profiles
3. Bivariate: every feature vs default (rates, tests, WoE/IV)
4. Interactions: two- and three-way default-rate surfaces
5. Temporal behaviour: trajectories, transitions, roll rates
6. **Share of Wallet**
7. Engineered behavioural features, correlation, redundancy
8. Segmentation and personas
9. Hidden-pattern discovery (GPU): interaction mining, rules, embeddings
10. Statistical robustness checks
11. Snapshot or panel? Data-structure audit and deep time-series / panel analysis

**Part II - Deep dives**

12. Column forensics: what the codes and amounts really mean (accounting identities, status semantics, label noise)
13. The full association atlas: every column vs every column, networks, differential structure
14. Conditional and higher-order interactions: what matters once we know X
15. Demographics in depth: lifecycle, credit-line assignment, gap decomposition
16. Money-flow dynamics and trajectory shapes
17. Early warning, thresholds and tipping points
18. Exposure economics: risk vs revenue, policy scenarios
19. Sub-population case studies

**Part III - Story and delivery**

20. Story synthesis: narrative arc, hero charts, storyboard
21. Tableau exports + data dictionary + dashboard blueprint
A. Appendix: optional automated EDA reports

## 0. Setup
Runs on **Kaggle** (GPU accelerator recommended — Section 9 uses it) and falls back to CPU anywhere else.

In [ ]:
import os, sys, glob, json, math, warnings, itertools, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
np.random.seed(42)
RNG = np.random.default_rng(42)

# ---------- plotting style ----------
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 12,
                     "figure.figsize": (10, 5), "axes.spines.top": False, "axes.spines.right": False})
C_OK, C_DEF, C_NEU, C_ACC = "#2b8cbe", "#e34a33", "#969696", "#31a354"   # non-default, default, neutral, accent
PAL_TARGET = {0: C_OK, 1: C_DEF, "No default": C_OK, "Default": C_DEF}
CMAP_RATE = "rocket_r"     # sequential map for default-rate heatmaps
CMAP_DIV = "RdBu_r"        # diverging map (lift / correlation)

# ---------- environment ----------
IS_KAGGLE = os.path.exists("/kaggle")
def _gpu_available():
    try:
        import subprocess
        return subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
    except Exception:
        return False
GPU = _gpu_available()
print(f"Kaggle: {IS_KAGGLE} | GPU: {GPU} | pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
# ---------- locate data ----------
CANDIDATES = [
    "/kaggle/input/default-of-credit-card-clients-dataset/UCI_Credit_Card.csv",
    "../input/UCI_Credit_Card.csv",
    "build/data/UCI_Credit_Card.csv",
    "data/UCI_Credit_Card.csv",
    "UCI_Credit_Card.csv",
]
CANDIDATES += glob.glob("/kaggle/input/**/UCI_Credit_Card.csv", recursive=True)
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_PATH, "UCI_Credit_Card.csv not found - add the dataset 'uciml/default-of-credit-card-clients-dataset' to the notebook"
print("Data:", DATA_PATH)

OUT = Path("/kaggle/working" if IS_KAGGLE else "outputs")
FIG_DIR, TAB_DIR = OUT / "figures", OUT / "tableau"
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

raw = pd.read_csv(DATA_PATH)
print(raw.shape)
raw.head()

### 0.1 Standardise names and keep the raw codes
`PAY_0` is renamed `PAY_1` so all three monthly blocks share the index 1..6. The target becomes `DEFAULT`.

In [ ]:
df = raw.rename(columns={"PAY_0": "PAY_1", "default.payment.next.month": "DEFAULT"}).copy()
df["ID"] = df["ID"].astype(int)
for c in df.columns:
    if c not in ("LIMIT_BAL",) and df[c].dtype == float and (df[c] % 1 == 0).all():
        df[c] = df[c].astype(int)

IDX = [1, 2, 3, 4, 5, 6]
MONTH = {1: "Sep", 2: "Aug", 3: "Jul", 4: "Jun", 5: "May", 6: "Apr"}
MONTH_ORDER = ["Apr", "May", "Jun", "Jul", "Aug", "Sep"]          # chronological
PAY_COLS  = [f"PAY_{i}" for i in IDX]
BILL_COLS = [f"BILL_AMT{i}" for i in IDX]
PAMT_COLS = [f"PAY_AMT{i}" for i in IDX]
DEMO_COLS = ["SEX", "EDUCATION", "MARRIAGE", "AGE"]
TARGET = "DEFAULT"
BASE_RATE = df[TARGET].mean()
print(f"Rows {len(df):,} | defaulters {df[TARGET].sum():,} | base default rate {BASE_RATE:.2%}")

### 0.2 Labels and bands (used everywhere, and exported to Tableau)
* `EDUCATION` 0, 5, 6 are undocumented; 4 = "others". Kept raw in `EDUCATION`; `EDU_LABEL` folds 0/4/5/6 into **Other/Unknown** (Section 1 checks whether that fold is safe).
* `MARRIAGE` 0 is undocumented → **Other**.
* Repayment status codes get readable labels and a coarse group.

In [ ]:
SEX_MAP = {1: "Male", 2: "Female"}
EDU_MAP = {1: "Graduate school", 2: "University", 3: "High school", 4: "Other/Unknown", 5: "Other/Unknown", 6: "Other/Unknown", 0: "Other/Unknown"}
EDU_ORDER = ["Graduate school", "University", "High school", "Other/Unknown"]
MAR_MAP = {1: "Married", 2: "Single", 3: "Other", 0: "Other"}
MAR_ORDER = ["Married", "Single", "Other"]
STATUS_LABEL = {-2: "-2 No consumption", -1: "-1 Paid in full", 0: "0 Revolving (min paid)",
                1: "1 Late 1m", 2: "2 Late 2m", 3: "3 Late 3m", 4: "4 Late 4m", 5: "5 Late 5m",
                6: "6 Late 6m", 7: "7 Late 7m", 8: "8 Late 8m", 9: "9 Late 9m+"}
def status_group(v):
    if v <= -2: return "Inactive"
    if v == -1: return "Paid in full"
    if v == 0:  return "Revolving"
    if v == 1:  return "Late 1m"
    if v == 2:  return "Late 2m"
    return "Late 3m+"
STATUS_GROUP_ORDER = ["Inactive", "Paid in full", "Revolving", "Late 1m", "Late 2m", "Late 3m+"]
STATUS_GRP_ORDER = STATUS_GROUP_ORDER

df["SEX_LABEL"] = df["SEX"].map(SEX_MAP)
df["EDU_LABEL"] = df["EDUCATION"].map(EDU_MAP)
df["MAR_LABEL"] = df["MARRIAGE"].map(MAR_MAP)
df["DEFAULT_LABEL"] = df[TARGET].map({0: "No default", 1: "Default"})

AGE_BINS = [20, 25, 30, 35, 40, 45, 50, 60, 80]
AGE_LABELS = ["21-25", "26-30", "31-35", "36-40", "41-45", "46-50", "51-60", "61+"]
df["AGE_BAND"] = pd.cut(df["AGE"], AGE_BINS, labels=AGE_LABELS)

LIMIT_BINS = [0, 50_000, 100_000, 150_000, 200_000, 300_000, 500_000, 1_000_001]
LIMIT_LABELS = ["<=50k", "50-100k", "100-150k", "150-200k", "200-300k", "300-500k", ">500k"]
df["LIMIT_BAND"] = pd.cut(df["LIMIT_BAL"], LIMIT_BINS, labels=LIMIT_LABELS)

for i in IDX:
    df[f"STATUS_GRP_{i}"] = df[f"PAY_{i}"].map(status_group)
df["STATUS_GRP_SEP"] = df["STATUS_GRP_1"]
df[["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND", "STATUS_GRP_SEP"]].describe()

### 0.3 Core behavioural feature layer
Built once here so every section (and Tableau) uses identical definitions.

| Feature | Definition |
|---|---|
| `UTIL_i` | `BILL_AMTi / LIMIT_BAL` (utilisation in month *i*) |
| `PAY_RATIO_i` (i=1..5) | `PAY_AMTi / BILL_AMT(i+1)` — share of the *previous* statement repaid; NaN when previous bill ≤ 0; capped at 5 |
| `SPEND_i` (i=1..5) | `BILL_AMTi − BILL_AMT(i+1) + PAY_AMTi` — implied new purchases (+ interest/fees) in month *i* |
| `N_LATE` / `MAX_DELAY` | months with status ≥ 1 / worst status in 6 months |
| `LATE_STREAK_MAX` / `LATE_STREAK_RECENT` | longest run of late months / run ending in Sep |
| `N_INACTIVE`, `N_PAIDFULL`, `N_REVOLVE` | months with status −2 / −1 / 0 |
| `AVG_UTIL`, `MAX_UTIL`, `UTIL_TREND` | mean / max utilisation; OLS slope of utilisation Apr→Sep (per month) |
| `TOTAL_BILL`, `TOTAL_PAY` | 6-month sums of bills / payments |
| `PAY_TO_BILL_5M` | **share of bills repaid (5 statements, aligned)**: `sum(PAY_AMT1..5) / sum(max(BILL_AMT2..6, 0))`; NaN when the denominator is 0. Use this one. |
| `PAY_TO_BILL_6M` | legacy: total paid / total billed over the same six months, NOT aligned (a payment settles the previous month's bill); use `PAY_TO_BILL_5M` |
| `PAYRATIO_BAND` | bands of `PAY_TO_BILL_5M` (0, 0-5%, ..., >100%; "No bill" when NaN) |
| `BILL_TREND` | OLS slope of bill Apr→Sep (NT$/month) |
| `N_ZERO_PAY` | months with zero payment |
| `OVERLIMIT_ANY`, `NEG_BILL_ANY` | any bill > limit / any negative bill |
| `DELAY_TREND` | status Sep − status Apr (positive = deteriorating) |

In [ ]:
B = df[BILL_COLS].to_numpy(float)          # col 0 = Sep ... col 5 = Apr
P = df[PAMT_COLS].to_numpy(float)
S = df[PAY_COLS].to_numpy(int)
L = df["LIMIT_BAL"].to_numpy(float)[:, None]

# utilisation
U = B / L
for k, i in enumerate(IDX):
    df[f"UTIL_{i}"] = U[:, k]
df["AVG_UTIL"] = U.mean(1)
df["MAX_UTIL"] = U.max(1)

# payment ratio vs previous statement
for i in IDX[:-1]:
    prev = df[f"BILL_AMT{i+1}"].astype(float)
    df[f"PAY_RATIO_{i}"] = np.where(prev > 0, (df[f"PAY_AMT{i}"] / prev.where(prev > 0)).clip(upper=5), np.nan)
df["AVG_PAY_RATIO"] = df[[f"PAY_RATIO_{i}" for i in IDX[:-1]]].mean(1)

# implied spend (new purchases + charges)
for i in IDX[:-1]:
    df[f"SPEND_{i}"] = df[f"BILL_AMT{i}"] - df[f"BILL_AMT{i+1}"] + df[f"PAY_AMT{i}"]
SPEND_COLS = [f"SPEND_{i}" for i in IDX[:-1]]
df["AVG_SPEND"] = df[SPEND_COLS].mean(1)

# delinquency counts and streaks
late = S >= 1
df["N_LATE"] = late.sum(1)
df["MAX_DELAY"] = S.max(1)
df["N_INACTIVE"] = (S == -2).sum(1)
df["N_PAIDFULL"] = (S == -1).sum(1)
df["N_REVOLVE"] = (S == 0).sum(1)
chron = late[:, ::-1]                      # Apr -> Sep
def _max_run(row):
    best = cur = 0
    for v in row:
        cur = cur + 1 if v else 0
        best = max(best, cur)
    return best
df["LATE_STREAK_MAX"] = [_max_run(r) for r in chron]
rec = np.zeros(len(df), int)
for k in range(6):                         # Sep backwards
    still = (rec == k) & late[:, k]
    rec[still] += 1
df["LATE_STREAK_RECENT"] = rec
df["DELAY_TREND"] = S[:, 0] - S[:, 5]

# sums, ratios, trends
df["TOTAL_BILL"] = B.sum(1)
df["TOTAL_PAY"] = P.sum(1)
pos_bill = np.clip(B, 0, None).sum(1)
df["PAY_TO_BILL_6M"] = np.where(pos_bill > 0, P.sum(1) / np.where(pos_bill > 0, pos_bill, 1), np.nan)
# aligned version: PAY_AMT1..5 settle BILL_AMT2..6 (the September payment pays the August statement)
pos_prev = np.clip(B[:, 1:], 0, None).sum(1)
df["PAY_TO_BILL_5M"] = np.where(pos_prev > 0, P[:, :5].sum(1) / np.where(pos_prev > 0, pos_prev, 1), np.nan)
df["AVG_BILL"] = B.mean(1)
df["AVG_PAY"] = P.mean(1)
t = np.arange(6) - 2.5                     # chronological time index Apr..Sep
def _slope(M):                             # M columns Sep..Apr -> slope over chronological time
    Mc = M[:, ::-1]
    return ((Mc - Mc.mean(1, keepdims=True)) * t).sum(1) / (t ** 2).sum()
df["BILL_TREND"] = _slope(B)
df["UTIL_TREND"] = _slope(U)
df["PAY_TREND"] = _slope(P)
df["N_ZERO_PAY"] = (P == 0).sum(1)
df["OVERLIMIT_ANY"] = (B > L).any(1).astype(int)
df["N_OVERLIMIT"] = (B > L).sum(1)
df["NEG_BILL_ANY"] = (B < 0).any(1).astype(int)

# convenient bands for Tableau / crosstabs
df["UTIL_BAND"] = pd.cut(df["AVG_UTIL"], [-np.inf, 0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, np.inf],
                         labels=["<=0", "0-10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%", ">100%"])
df["N_LATE_BAND"] = pd.cut(df["N_LATE"], [-1, 0, 1, 2, 3, 6], labels=["0", "1", "2", "3", "4-6"])
df["PAYRATIO_BAND"] = pd.cut(df["PAY_TO_BILL_5M"], [-np.inf, 0, 0.05, 0.1, 0.25, 0.5, 1.0, np.inf],
                             labels=["0", "0-5%", "5-10%", "10-25%", "25-50%", "50-100%", ">100%"])
df["PAYRATIO_BAND"] = df["PAYRATIO_BAND"].cat.add_categories("No bill").fillna("No bill")

FEATURE_COLS = ["AVG_UTIL", "MAX_UTIL", "UTIL_TREND", "AVG_PAY_RATIO", "PAY_TO_BILL_6M", "AVG_SPEND",
                "N_LATE", "MAX_DELAY", "LATE_STREAK_MAX", "LATE_STREAK_RECENT", "N_INACTIVE", "N_PAIDFULL",
                "N_REVOLVE", "DELAY_TREND", "TOTAL_BILL", "TOTAL_PAY", "AVG_BILL", "AVG_PAY", "BILL_TREND",
                "PAY_TREND", "N_ZERO_PAY", "OVERLIMIT_ANY", "N_OVERLIMIT", "NEG_BILL_ANY"]
RAW_NUM = ["LIMIT_BAL", "AGE"] + BILL_COLS + PAMT_COLS
print(df.shape)
df[FEATURE_COLS].describe().T

### 0.4 Shared helper functions
Used by every section: default-rate tables with Wilson CIs, rate plots, Cramér's V, WoE/IV, figure saving, Tableau export.

In [ ]:
def wilson_ci(k, n, z=1.96):
    k, n = np.asarray(k, float), np.asarray(n, float)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = k / n
        den = 1 + z**2 / n
        c = (p + z**2 / (2 * n)) / den
        h = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / den
    return c - h, c + h

def rate_table(data, by, target=TARGET, min_n=0):
    # Default-rate table for one column or a list of columns. Returns n, defaults, rate, CI, share, lift.
    by = [by] if isinstance(by, str) else list(by)
    g = data.groupby(by, observed=True)[target].agg(n="size", defaults="sum").reset_index()
    g["rate"] = g["defaults"] / g["n"]
    g["ci_lo"], g["ci_hi"] = wilson_ci(g["defaults"], g["n"])
    g["share_clients"] = g["n"] / g["n"].sum()
    g["share_defaults"] = g["defaults"] / g["defaults"].sum()
    g["lift"] = g["rate"] / data[target].mean()
    return g[g["n"] >= min_n].reset_index(drop=True)

def plot_rate(data, col, ax=None, order=None, title=None, rotate=0, min_n=0):
    # Bars = client count (left axis); red line + CI = default rate (right axis); dashed = base rate.
    t = rate_table(data, col, min_n=min_n)
    if order is not None:
        t = t.set_index(col).reindex([o for o in order if o in set(t[col])]).reset_index()
    ax = ax or plt.subplots(figsize=(10, 4.5))[1]
    x = np.arange(len(t))
    ax.bar(x, t["n"], color=C_NEU, alpha=.45)
    ax.set_ylabel("clients")
    ax.set_xticks(x); ax.set_xticklabels(t[col].astype(str), rotation=rotate, ha="right" if rotate else "center")
    ax2 = ax.twinx()
    ax2.errorbar(x, t["rate"], yerr=[t["rate"] - t["ci_lo"], t["ci_hi"] - t["rate"]], color=C_DEF, marker="o", lw=2, capsize=3)
    ax2.axhline(BASE_RATE, ls="--", color="k", lw=1)
    ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax2.set_ylabel("default rate"); ax2.grid(False)
    ax2.set_ylim(0, max(0.05, float(t["ci_hi"].max()) * 1.1))
    ax.set_title(title or f"Default rate by {col}"); ax.set_xlabel("")
    return t

def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum(); r, k = ct.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return float(np.sqrt(phi2 / max(1e-12, min(kc - 1, rc - 1))))

def woe_iv(data, col, target=TARGET, bins=10, eps=0.5):
    # Weight of Evidence / Information Value. Numeric columns with many levels are quantile-binned.
    s = data[col]
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > bins:
        s = pd.qcut(s, bins, duplicates="drop")
    t = pd.crosstab(s, data[target])
    t.columns = ["good", "bad"]
    dg = (t["good"] + eps) / (t["good"].sum() + eps)
    db = (t["bad"] + eps) / (t["bad"].sum() + eps)
    t["woe"] = np.log(dg / db)
    t["iv_part"] = (dg - db) * t["woe"]
    t["rate"] = t["bad"] / (t["good"] + t["bad"])
    return t, float(t["iv_part"].sum())

def savefig(name, fig=None):
    (fig or plt.gcf()).savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=120)

def export(frame, name, index=False):
    path = TAB_DIR / f"{name}.csv"
    frame.to_csv(path, index=index)
    print(f"exported {path}  {frame.shape}")

def pct(x):
    return f"{x:.1%}"

# ---------- story facts: every section can register headline numbers for the story synthesis (Section 20) ----------
STORY_FACTS = []
def add_fact(section, theme, fact, value=None, chart=None, strength=None):
    # section: e.g. "12.3"; theme: short tag (e.g. "recency", "wallet", "demographics"); fact: one plain sentence
    # containing the numbers; value: the key number; chart: figure name saved with savefig; strength: 1 (minor) .. 3 (headline)
    STORY_FACTS.append({"section": str(section), "theme": theme, "fact": fact, "value": value, "chart": chart, "strength": strength})

print("helpers ready")

## 0B. Data dictionary: every column, what it means, how to read it

**Why this section exists.** Every later chart rests on about 80 columns: 25 raw ones and about 55 built in Section 0.
This section explains each one in plain words: what it is, the exact formula, its units and range, how to read a value,
and where the definition is weak. Then it takes **five real clients** and prints their raw months next to every
derived value, so you can redo each formula by hand. A final check recomputes every formula independently and stops
the notebook if anything does not match.

Columns added by later sections (Sections 5-8 such as `TRAJ_*`, `SOW_*` including the quarterly paydown `SOW_QPD_*`, `FE_*`, `RISK_TIER`, and the prefixed deep-dive columns `PN_`, `CF_`, `CI_`, `DM_`, `MF_`, `EW_`, `CS_`) are explained in **Section 22 (Column glossary)**.

**How to use it.** Read 0B.1 (the month order) first. Everything else depends on it.

### Words we use, and their one meaning
Each word below has **one** meaning across the whole notebook. If a chart or sentence uses the word, this is what it means.

| Word | What it means here | Rule in the data | Example |
|---|---|---|---|
| **Late** (in a month) | The bank recorded a payment delay that month. | status code >= 1 | Status 2 in July = late in July. |
| **Never late** | No delay recorded in any of the six months. | status <= 0 in all six months (`N_LATE = 0`) | About two in three clients; they still make up about 35% of defaulters. |
| **Clean in Sep** | Not late at the last statement we see. | `PAY_1 <= 0` | Late in May, fine in Sep = clean in Sep. |
| **Paid in full (code)** | The **bank's status code** says the bill was paid in full. | status = -1 | `N_PAIDFULL = 6`: 1,992 clients, 14.3% default. |
| **Cleared every bill (money)** | The **amounts** show every bill was paid off. Different from the code. | payment >= previous bill whenever a bill was due (May-Sep) | 5,339 clients; 725 of them still defaulted. |
| **Revolver** | Status code 0: carried a balance and paid at least the minimum (typically about 4% of the bill). | status = 0 | Paid 3,000 on a 67,060 bill. (The money-based 6-95% band is called **partial payer**, not revolver.) |
| **Inactive** | Status code -2: nothing was due that month. | status = -2 | Not the same as "card unused": about 4 in 10 such months still show a bill. |
| **Dormant** | No activity at all: zero bills and zero payments in all six months. | all `BILL_AMTi = 0` and all `PAY_AMTi = 0` | 795 clients, and 37.6% of them default. |
| **Utilisation** | How much of the credit line the bill uses. | bill / credit line (`UTIL_i`); say "Sep" or "average" | Bill 45,000 on a 100,000 line = 45%. |
| **Exposure / EAD** | What the bank has lent out in Sep / the part of it that defaulted. | `max(BILL_AMT1, 0)`; EAD = exposure x DEFAULT | Sep bill 41,087 and defaulted -> EAD 41,087. |
| **Headroom** | The unused part of the credit line. | `max(LIMIT_BAL - exposure, 0)` | Line 100,000, bill 45,000 -> 55,000. |
| **Over limit** | The bill went above the credit line in at least one month. | any `BILL_AMTi > LIMIT_BAL` | Bill 64,400 on a 50,000 line. |
| **Cured** | Was late early on, then clean for the last three months. | late in Apr-Jun, never late Jul-Sep (`CS_G09_CURED`) | Late in May, clean Jul-Sep. |
| **Base rate** | The share of all 30,000 clients who default in October. | mean of `DEFAULT` | 22.1%. Any "default rate" is out of all clients unless a group is named. |

### 0B.1 The shape of the data: six monthly snapshots per client

**One row = one client.** There are 30,000 rows and 30,000 different clients. Each row holds the same six calendar
months (April to September 2005) side by side in "wide" columns. So the file contains 30,000 x 6 = 180,000
client-months, but it is **not** a long panel: there is no row per month, no account-opening date, no tenure,
and no history before April. Every client is observed over exactly the same six months. The outcome (`DEFAULT`)
is measured once, for October 2005.

**The index runs backwards in time.** Index 1 is the **most recent** month (September). Index 6 is the **oldest** (April).

| Calendar month (2005) | Apr | May | Jun | Jul | Aug | Sep | Oct |
|---|---|---|---|---|---|---|---|
| Index *i* | 6 | 5 | 4 | 3 | 2 | **1** | - |
| Repayment status | `PAY_6` | `PAY_5` | `PAY_4` | `PAY_3` | `PAY_2` | `PAY_1` (raw `PAY_0`) | - |
| Bill statement | `BILL_AMT6` | `BILL_AMT5` | `BILL_AMT4` | `BILL_AMT3` | `BILL_AMT2` | `BILL_AMT1` | - |
| Payment made in the month | `PAY_AMT6` | `PAY_AMT5` | `PAY_AMT4` | `PAY_AMT3` | `PAY_AMT2` | `PAY_AMT1` | - |
| ...which settles the statement of | Mar (not in data) | Apr = `BILL_AMT6` | May = `BILL_AMT5` | Jun = `BILL_AMT4` | Jul = `BILL_AMT3` | Aug = `BILL_AMT2` | - |
| Target | | | | | | | `DEFAULT` |

**Read it like a statement.** The September statement (`BILL_AMT1`) is the August statement (`BILL_AMT2`), minus what
the client paid in September (`PAY_AMT1`), plus new charges and interest in September. So the payment with index *i*
belongs to the bill with index *i + 1*. Example: `PAY_AMT1` (paid in Sep) settles `BILL_AMT2` (the Aug statement).
Cell 0B.4 proves this pairing from the data. The September bill itself would be paid in October, which we never see.

**About the names.** The source file has `PAY_0, PAY_2, PAY_3, ..., PAY_6`. There is no `PAY_1`, but the bill and
payment columns start at 1. This is a naming slip in the source, not a missing month. Section 0 renames `PAY_0` to
`PAY_1` so all three blocks share the index 1..6. There is **no month 0**, no `BILL_AMT0`, and no "bal0" (opening
balance): the oldest balance we have is the April statement, `BILL_AMT6`.

In [ ]:
S00B_T0 = time.time()
S00B_TIMELINE = pd.DataFrame({
    "index_i": IDX[::-1],
    "month_2005": [MONTH[i] for i in IDX[::-1]],
    "status_col": [f"PAY_{i}" + (" (raw PAY_0)" if i == 1 else "") for i in IDX[::-1]],
    "bill_col": [f"BILL_AMT{i}" for i in IDX[::-1]],
    "payment_col": [f"PAY_AMT{i}" for i in IDX[::-1]],
    "payment_settles": ["BILL_AMT7 = Mar statement (not in data)"] + [f"BILL_AMT{i + 1} ({MONTH[i + 1]} statement)" for i in IDX[::-1][1:]],
    "recency": ["oldest"] + [""] * 4 + ["most recent"]})
display(S00B_TIMELINE)

# the same client seen as six snapshots (long form): one row per month, chronological
S00B_ID_DEMO = 31
_r = df.loc[df["ID"] == S00B_ID_DEMO].iloc[0]
display(pd.DataFrame([{"ID": S00B_ID_DEMO, "month": MONTH[i], "index_i": i, "status": int(_r[f"PAY_{i}"]), "bill": _r[f"BILL_AMT{i}"],
                       "paid_this_month": _r[f"PAY_AMT{i}"], "prev_statement": _r[f"BILL_AMT{i + 1}"] if i < 6 else np.nan}
                      for i in IDX[::-1]]))
print(f"rows {len(df):,} | unique IDs {df['ID'].nunique():,} | client-months {6 * len(df):,} | every client covers the same 6 months Apr-Sep 2005")

### 0B.2 Raw columns: the client (static) block

These are measured once per client.

| Column | What it is | Values and units | How to read it (examples) | Watch out |
|---|---|---|---|---|
| `ID` | Row key, 1 to 30,000 | integer | Only a label. ID 15 is not "more" than ID 14. Use it to join Tableau extracts. | No meaning. Never use it as a feature. |
| `LIMIT_BAL` | Credit line granted by the bank | NT$ 10,000 to 1,000,000 (New Taiwan dollars, 2005) | 50,000 = small line; 200,000 = about the median; 500,000+ = premium. A client with 100,000 and a bill of 45,000 uses 45% of the line. | Covers the client **and** family / supplementary cards. Set in round steps (10k, 20k, 50k...). It is the bank's decision, not a client trait. |
| `SEX` | Gender | 1 = male, 2 = female | `SEX = 2` means female (60% of clients). | Gender is a protected trait. Describe it, do not use it to make decisions. |
| `EDUCATION` | Highest education | 1 = graduate school, 2 = university, 3 = high school, 4 = others, **0, 5, 6 = undocumented** | 2 is the most common (47%). | 0/5/6 are not in the official codebook. `EDU_LABEL` folds 0/4/5/6 into "Other/Unknown". That group defaults at only about 7% (see Section 15.6). |
| `MARRIAGE` | Marital status | 1 = married, 2 = single, 3 = others, **0 = undocumented** | 2 (single) is the most common (53%). | 0 has 54 rows; folded into "Other". |
| `AGE` | Age in years | 21 to 79 | 35 = a 35-year-old client. | Measured once; it does not change over the window. |
| `DEFAULT` | Target: did the client default on the payment due in **October 2005**? | 1 = yes, 0 = no. Raw name `default.payment.next.month`. | 22.1% of clients have 1. | It is one month **after** the window. Any column built from it (for example a cell's default rate) is an outcome summary, not a predictor. |

In [ ]:
S00B_DEMO = []
for _c, _m in [("SEX", SEX_MAP), ("EDUCATION", EDU_MAP), ("MARRIAGE", MAR_MAP)]:
    _t = rate_table(df, _c)
    for _, _row in _t.iterrows():
        S00B_DEMO.append({"column": _c, "code": int(_row[_c]), "label_used": _m.get(int(_row[_c])), "n": int(_row["n"]),
                          "share": _row["share_clients"], "default_rate": _row["rate"]})
S00B_DEMO = pd.DataFrame(S00B_DEMO)
display(S00B_DEMO.round(4))
display(df[["LIMIT_BAL", "AGE"]].describe(percentiles=[.05, .25, .5, .75, .95]).T.round(0))
print(f"LIMIT_BAL distinct values: {df['LIMIT_BAL'].nunique()} | share that are multiples of 10,000: {(df['LIMIT_BAL'] % 10_000 == 0).mean():.1%}")
print(f"DEFAULT: {df[TARGET].sum():,} of {len(df):,} = {BASE_RATE:.2%}")

### 0B.3 Raw columns: the repayment status `PAY_1 .. PAY_6`

One code per month. It says how the client handled the bill that was due that month.

| Code | Official meaning | What the money usually looks like | Example |
|---|---|---|---|
| **-2** | "No consumption": card not used, nothing (or almost nothing) to pay | Bill is often 0. When there is a bill, it is small and paid off. | Bill 0, payment 0, status -2. |
| **-1** | "Paid duly": paid the full balance | Payment is about equal to the previous statement (median ratio = 1.0). | Aug bill 17,265; Sep payment 17,270; Sep status -1. |
| **0** | "Revolving": paid at least the minimum, carried the rest | Payment is a small share of the previous bill (median about 5%). Balance stays large. | Aug bill 67,060; Sep payment 3,000 (4.5%); Sep status 0. |
| **1** | Payment delay of 1 month | **Almost only in September** (see below). | Sep status 1. |
| **2** | Payment delay of 2 months | Still often pays something (median about 5% of the bill), but too little or too late. | Typical Aug status-2 row (medians): bill 27,689, payment 1,646. |
| **3 .. 8** | Delay of 3 .. 8 months | Median payment is **0**. The balance keeps rolling. | Status 3, payment 0. |
| 9 | "9 months and above" | Never appears in this file (max is 8). | - |

**How to read the scale.** Codes 1..8 are a count of months behind. Codes -2, -1, 0 are **not** on that scale: they are
three kinds of "not late". So `0 - (-2) = 2` is not "two months worse". Treat the code as an ordered category, not a number.

**The code-1 quirk.** Code 1 appears 3,688 times in September but only 28 times in August, 4 in July, 2 in June and
never in April or May. In April-August a first missed payment shows up directly as code **2**. So "Late 1m" is a
September-only category, and a 2 in Jul is roughly what a 1 in Sep is. Any rule like "status >= 2" means different
things in Sep and in earlier months. Many Sep code-1 clients have a small or even negative Sep bill (see the table below).

The cell below shows the counts by month, and the money pattern behind each code. "Payment / previous bill" pairs
`PAY_AMTi` with `BILL_AMT(i+1)`, as explained in 0B.1.

In [ ]:
S00B_CODE_BY_MONTH = pd.DataFrame({MONTH[i]: df[f"PAY_{i}"].value_counts() for i in IDX[::-1]}).fillna(0).astype(int).sort_index()
S00B_CODE_BY_MONTH.index.name = "status code"
print("Count of each status code by month (Apr -> Sep):")
display(S00B_CODE_BY_MONTH)

_rows = []
for _lab, _months in [("Apr-Aug pooled (i = 2..5)", [2, 3, 4, 5]), ("Sep (i = 1)", [1])]:
    _st = np.concatenate([df[f"PAY_{i}"].to_numpy() for i in _months])
    _bill = np.concatenate([df[f"BILL_AMT{i}"].to_numpy(float) for i in _months])
    _prev = np.concatenate([df[f"BILL_AMT{i + 1}"].to_numpy(float) for i in _months])
    _pay = np.concatenate([df[f"PAY_AMT{i}"].to_numpy(float) for i in _months])
    _y = np.concatenate([df[TARGET].to_numpy() for i in _months])
    _ratio = np.where(_prev > 0, _pay / np.where(_prev > 0, _prev, 1), np.nan)
    for _code in sorted(np.unique(_st)):
        _m = _st == _code
        _rr = _ratio[_m]
        _rows.append({"months": _lab, "status": int(_code), "client_months": int(_m.sum()),
                      "median_prev_bill": float(np.median(_prev[_m])), "median_payment": float(np.median(_pay[_m])),
                      "median_pay_over_prev_bill": float(np.nanmedian(_rr)) if np.isfinite(_rr).any() else np.nan,
                      "share_paid_>=95%_of_prev": float(np.nanmean(_rr >= .95)) if np.isfinite(_rr).any() else np.nan,
                      "share_paid_zero": float((_pay[_m] == 0).mean()), "share_this_bill_<=0": float((_bill[_m] <= 0).mean()),
                      "default_rate": float(_y[_m].mean())})
S00B_CODE_MONEY = pd.DataFrame(_rows)
display(S00B_CODE_MONEY.round(3))
print("Read: -1 rows pay ~100% of the previous bill; 0 rows pay a few %; 3+ rows mostly pay nothing. "
      "Code 1 is a Sep-only phenomenon and its money pattern differs from code 2.")

### 0B.4 Raw columns: bills `BILL_AMT1..6` and payments `PAY_AMT1..6`

| Column | What it is | Units, range | How to read it | Watch out |
|---|---|---|---|---|
| `BILL_AMTi` | Balance on the statement of month *i* (1 = Sep ... 6 = Apr) | NT$; can be **negative** (about -340k to 1.66M) | 45,000 = the client owed 45,000 on that statement. 0 = nothing owed. -2,000 = the bank owes the client 2,000 (over-payment or refund). | It is a **stock** (what is owed), not spending. A balance carried for 3 months appears in 3 bills. It can exceed `LIMIT_BAL` (over limit). |
| `PAY_AMTi` | Amount paid during month *i* | NT$ >= 0 (0 to about 1.7M) | 3,000 in `PAY_AMT1` = the client paid 3,000 in September, toward the **August** statement (`BILL_AMT2`). | Pairs with the **next** index's bill. `PAY_AMT6` (paid in Apr) settles a March bill we do not have. |

**Proof of the pairing.** If `PAY_AMTi` settles `BILL_AMT(i+1)`, full payers should often pay **exactly** the previous
bill. The cell counts exact matches both ways. It also checks the accounting identity: implied new charges
`BILL_AMTi - BILL_AMT(i+1) + PAY_AMTi` should rarely be negative under the right pairing.

In [ ]:
_rows = []
for i in IDX[:-1]:
    _pay, _prev, _same = df[f"PAY_AMT{i}"], df[f"BILL_AMT{i + 1}"], df[f"BILL_AMT{i}"]
    _rows.append({"payment": f"PAY_AMT{i} ({MONTH[i]})",
                  "exact match with BILL_AMT(i+1) (previous statement)": int(((_pay == _prev) & (_prev > 0)).sum()),
                  "exact match with BILL_AMTi (same-month statement)": int(((_pay == _same) & (_same > 0)).sum()),
                  "share spend < 0, pairing i with i+1 (used)": float(((_same - _prev + _pay) < 0).mean()),
                  "share spend < 0, alternative pairing": float(((_same - _prev + df[f"PAY_AMT{i + 1}"]) < 0).mean())})
S00B_PAIRING = pd.DataFrame(_rows)
display(S00B_PAIRING.round(4))
print("Exact matches are 6-7x more frequent with the PREVIOUS statement: PAY_AMTi settles BILL_AMT(i+1).")
display(df[BILL_COLS + PAMT_COLS].describe(percentiles=[.05, .5, .95]).T.round(0))
print(f"negative bills: {(df[BILL_COLS] < 0).to_numpy().mean():.2%} of client-months | zero bills: {(df[BILL_COLS] == 0).to_numpy().mean():.2%} | "
      f"zero payments: {(df[PAMT_COLS] == 0).to_numpy().mean():.2%}")

### 0B.5 Section 0 labels and bands (readable versions of raw columns)

| Column | Definition | Example | Watch out |
|---|---|---|---|
| `SEX_LABEL` | `SEX` mapped 1 -> Male, 2 -> Female | `SEX = 2` -> "Female" | - |
| `EDU_LABEL` | `EDUCATION` mapped 1 -> Graduate school, 2 -> University, 3 -> High school, **0/4/5/6 -> Other/Unknown** | `EDUCATION = 5` -> "Other/Unknown" | Mixes the documented "others" (4) with three undocumented codes. Raw codes are kept in `EDUCATION` (and in `DM_EDU_RAW_LABEL`, Section 15). |
| `MAR_LABEL` | `MARRIAGE` mapped 1 -> Married, 2 -> Single, **0/3 -> Other** | `MARRIAGE = 0` -> "Other" | - |
| `DEFAULT_LABEL` | `DEFAULT` mapped 0 -> "No default", 1 -> "Default" | - | - |
| `AGE_BAND` | Age cut into (20,25], (25,30], ..., (45,50], (50,60], (60,80] -> "21-25", "26-30", ..., "46-50", "51-60", "61+" | Age 25 -> "21-25"; age 26 -> "26-30"; age 55 -> "51-60" | Right edge is included. The last two bands are 10 and 20 years wide. |
| `LIMIT_BAND` | `LIMIT_BAL` cut at 50k, 100k, 150k, 200k, 300k, 500k (right edge included) | 50,000 -> "<=50k"; 50,001-100,000 -> "50-100k"; 1,000,000 -> ">500k" | Bands are unequal widths, chosen for readable counts. |
| `STATUS_GRP_i` (i = 1..6) | Status code of month *i* grouped: -2 -> Inactive, -1 -> Paid in full, 0 -> Revolving, 1 -> Late 1m, 2 -> Late 2m, 3..8 -> Late 3m+ | `PAY_3 = 4` -> `STATUS_GRP_3` = "Late 3m+" | "Late 1m" is almost empty outside Sep (code-1 quirk). |
| `STATUS_GRP_SEP` | Same as `STATUS_GRP_1` (September), kept under a readable name | `PAY_1 = 0` -> "Revolving" | - |

### 0B.6 Section 0 monthly features: utilisation, payment ratio, implied spend

**`UTIL_i` (i = 1..6): utilisation in month *i*.**
Formula: `UTIL_i = BILL_AMTi / LIMIT_BAL`. Unit: share of the credit line (0.45 = 45%). Range in data: about -1.5 to 10.7 (extreme rows are rare; 93% of client-months lie between 0 and 1).
* Limit 100,000, Sep bill 45,000 -> `UTIL_1 = 0.45`: the client uses 45% of the line.
* Limit 50,000, Sep bill 64,400 -> `UTIL_1 = 1.29`: the client is 29% **over** the limit.
* Bill -2,000 on a 100,000 line -> `UTIL_1 = -0.02`: the account is in credit.

*Sanity check:* values above 1 are real (over-limit balances; 13% of clients have at least one month above 1).
Negative values mean a credit balance, not "negative usage". The bill includes interest and fees, so it is
"balance owed / line", not "purchases / line".

**`PAY_RATIO_i` (i = 1..5): share of the previous statement repaid.**
Formula: `PAY_RATIO_i = PAY_AMTi / BILL_AMT(i+1)`, set to NaN when `BILL_AMT(i+1) <= 0`, capped at 5. Unit: share of the previous bill. There is no `PAY_RATIO_6` because the March bill is missing.
* Aug bill 20,000; Sep payment 20,000 -> `PAY_RATIO_1 = 1.00`: paid the bill in full.
* Aug bill 20,000; Sep payment 1,000 -> `PAY_RATIO_1 = 0.05`: about a minimum payment.
* Aug bill 0 (or negative); Sep payment 500 -> `PAY_RATIO_1 = NaN`: no bill was due, so "share repaid" has no meaning.
* Aug bill 1,000; Sep payment 150,000 -> 150, capped to `5`.

*Sanity check:* undefined in 11-16% of client-months (no positive previous bill), so averages of it cover different
months for different clients. The cap at 5 hides big pre-payments (about 30-45 rows per month hit the cap).

**`SPEND_i` (i = 1..5): implied new charges in month *i*.**
Formula: `SPEND_i = BILL_AMTi - BILL_AMT(i+1) + PAY_AMTi`. Unit: NT$. It comes from the statement identity
"new bill = old bill - payment + new charges".
* Aug bill 20,000; Sep payment 5,000; Sep bill 30,000 -> `SPEND_1 = 30,000 - 20,000 + 5,000 = 15,000` of new purchases, interest and fees.
* Aug bill 20,000; Sep payment 0; Sep bill 20,300 -> `SPEND_1 = 300`: no real use, only interest.
* Aug bill 20,000; Sep payment 0; Sep bill 18,000 -> `SPEND_1 = -2,000`: the balance fell without a payment (refund, reversal, write-off).

*Sanity check:* **can go negative** (about 0.4% of client-months). It mixes purchases with interest and fees, so
a revolver with zero purchases still shows positive "spend" (the interest). It is "net new charges", not "shopping".

### 0B.7 Section 0 summary features (one value per client)

Chronological order below is always Apr -> Sep. "Late" means status >= 1.

| Column | Definition and formula | Units, range | Examples | Sanity check / weakness |
|---|---|---|---|---|
| `AVG_UTIL` | mean(`UTIL_1..6`) | share of line; about -0.2 to 5.4 | Utilisations 0.40, 0.42, 0.45, 0.47, 0.50, 0.52 -> 0.46. Inactive card -> 0. | Over-limit and credit months pull the mean both ways. |
| `MAX_UTIL` | max(`UTIL_1..6`) | share of line | Same client -> 0.52. One month at 1.29 -> 1.29. | One spike decides it. |
| `UTIL_TREND` | OLS slope of `UTIL` on time (Apr = -2.5 ... Sep = +2.5) | share of line per month | 0.10, 0.20, ..., 0.60 -> +0.10 (10 points a month). Flat -> 0. | Six points; one jump moves it a lot. |
| `AVG_PAY_RATIO` | mean of the **available** `PAY_RATIO_1..5` (NaN skipped) | share | Ratios 1.0, 1.0, NaN, 1.0, 1.0 -> 1.0. All NaN -> NaN (5% of clients). | Averages different months for different clients; capped values (5) dominate a mean. |
| `AVG_SPEND` | mean(`SPEND_1..5`) | NT$ / month | Spends 15k, 15k, 10k, 20k, 15k -> 15,000. | Inherits negative spends and interest. |
| `N_LATE` | number of months with status >= 1 | 0-6 | Apr..Sep = 2, 2, 2, 0, 0, 2 -> 4. | Code 1 only exists in Sep, so a 1-month slip counts only if it happens in Sep. |
| `MAX_DELAY` | max(status over 6 months) | -2 to 8 | -1,-1,-1,-1,-1,-1 -> -1. 0,0,2,2,0,0 -> 2. | -2 / -1 / 0 are kinds of "not late", not degrees. |
| `N_INACTIVE`, `N_PAIDFULL`, `N_REVOLVE` | months with status -2 / -1 / 0 | 0-6 each | All -1 -> N_PAIDFULL = 6. | `N_LATE + N_INACTIVE + N_PAIDFULL + N_REVOLVE = 6` always. `N_PAIDFULL = 6` (code view: 1,992 clients, 284 defaulters, 14.3%) is not the same as "cleared every bill by amount" (money view: 725 defaulters). |
| `LATE_STREAK_MAX` | longest run of consecutive late months | 0-6 | 2, 2, 2, 0, 0, 2 -> 3 (Apr-Jun). | Ignores how deep the delay was. |
| `LATE_STREAK_RECENT` | run of late months ending in Sep (0 if Sep is not late) | 0-6 | 2, 2, 2, 0, 0, 2 -> 1. 0, 0, 2, 3, 4, 5 -> 4. | 0 for anyone current in Sep, even if late in Aug. |
| `DELAY_TREND` | `PAY_1 - PAY_6` (Sep code minus Apr code) | -7 to +7 | Apr -1, Sep 2 -> +3 (worse). Apr 3, Sep 0 -> -3 (better). | **Weak:** Apr -2, Sep 0 gives +2, yet "started revolving" is not a delay. Uses only two months. |
| `TOTAL_BILL` | sum of the six bills | NT$ | 50,000 carried in all six months -> 300,000. | Counts the same debt up to six times. It is "balance-months", not "amount borrowed". |
| `TOTAL_PAY` | sum of the six payments | NT$ | 2,500 a month -> 15,000. | `PAY_AMT6` settles a March bill outside the window. |
| `PAY_TO_BILL_5M` | **Share of bills repaid, aligned.** `sum(PAY_AMT1..5) / sum(max(BILL_AMT2..6, 0))`; NaN if those bills sum to 0 | ratio; 1.0 = repaid everything billed | Revolver paying 3,000 a month on ~60k bills: 15,000 / 302,704 = 0.05. Full payer: about 1.0. | Each payment is matched with the bill it settles (May-Sep payments vs Apr-Aug bills). Still sums balances, so carried debt counts once per month. Not capped. |
| `PAY_TO_BILL_6M` | **Legacy, not aligned.** `TOTAL_PAY / sum(max(BILL_AMTi, 0))` over the same six months; NaN if no positive bill | ratio; 0 to about 800 | Revolver: 18,000 / 373,591 = 0.05. | **Weak:** Apr-Sep payments against Apr-Sep bills, so one month out of step; kept only for older charts. Use `PAY_TO_BILL_5M`. |
| `AVG_BILL`, `AVG_PAY` | mean of the six bills / payments | NT$ | - | Same stock-vs-flow caveat as above. |
| `BILL_TREND`, `PAY_TREND` | OLS slope of bill / payment on time (Apr..Sep) | NT$ per month | Bills 10k, 20k, ..., 60k -> +10,000 / month. | Not scaled by limit; big lines dominate. |
| `N_ZERO_PAY` | months with `PAY_AMTi = 0` | 0-6 | Inactive card -> 6. | **Weak:** mixes "owed money and paid nothing" with "owed nothing" (about 36% of zero-payment months are inactive months). |
| `OVERLIMIT_ANY`, `N_OVERLIMIT` | any / number of months with bill > limit | 0/1; 0-6 | Bill 64,400 on a 50,000 line in 3 months -> 1; 3. | Strict ">": a bill exactly at the limit is not over. |
| `NEG_BILL_ANY` | 1 if any bill < 0 | 0/1 | Credit balance in one month -> 1. | - |
| `UTIL_BAND` | `AVG_UTIL` cut: <=0, 0-10%, 10-30%, 30-50%, 50-70%, 70-90%, 90-100%, >100% (right edge included) | category | 0.46 -> "30-50%". 0 -> "<=0". | "<=0" mixes unused cards with credit balances. |
| `N_LATE_BAND` | `N_LATE` as 0, 1, 2, 3, 4-6 | category | 4 -> "4-6". | - |
| `PAYRATIO_BAND` | Share of bills repaid (`PAY_TO_BILL_5M` once Section 0 builds it; `PAY_TO_BILL_6M` in older runs) cut: 0, 0-5%, 5-10%, 10-25%, 25-50%, 50-100%, >100%; NaN -> "No bill" | category | 0.05 -> "0-5%" (right edge included). | Cell 0B.10 prints which source this run uses. |

The cells below make all of this concrete with real clients.

### 0B.8 Five real clients, month by month

Five clients picked by **fixed, transparent rules** (the first ID that meets each rule), so you can find them again:

| Persona | Rule |
|---|---|
| Clean revolver | status 0 in all six months, no default, every bill > 20k and below 80% of the line, paid something each month |
| Full payer | status -1 in all six months, no default, every bill > 1k |
| Late payer who defaults | default, Sep status >= 2, late in >= 4 months, every bill > 0, at least two payments |
| Inactive card | status -2 in all six months, all bills and payments 0 |
| Over-limit | bill above the line in >= 3 months, every bill > 0 |

For each one: the raw months in time order (Apr -> Sep) with the monthly features underneath.

In [ ]:
_S = df[PAY_COLS].to_numpy(); _B = df[BILL_COLS].to_numpy(float); _P = df[PAMT_COLS].to_numpy(float); _L = df["LIMIT_BAL"].to_numpy(float)[:, None]
S00B_RULES = {
    "Clean revolver": (_S == 0).all(axis=1) & (df[TARGET].to_numpy() == 0) & (_B > 20_000).all(axis=1) & (_B < 0.8 * _L).all(axis=1) & (_P[:, :5] > 0).all(axis=1),
    "Full payer": (_S == -1).all(axis=1) & (df[TARGET].to_numpy() == 0) & (_B > 1_000).all(axis=1),
    "Late payer who defaults": (df[TARGET].to_numpy() == 1) & (_S[:, 0] >= 2) & ((_S >= 1).sum(axis=1) >= 4) & (_B > 0).all(axis=1) & ((_P > 0).sum(axis=1) >= 2),
    "Inactive card": (_S == -2).all(axis=1) & (_B == 0).all(axis=1) & (_P == 0).all(axis=1),
    "Over-limit": ((_B > _L).sum(axis=1) >= 3) & (_B > 0).all(axis=1)}
S00B_CASES = {k: int(df.loc[m, "ID"].min()) for k, m in S00B_RULES.items()}
print("clients meeting each rule:", {k: int(m.sum()) for k, m in S00B_RULES.items()})
print("chosen IDs:", S00B_CASES)

def s00b_month_table(cid):
    r = df.loc[df["ID"] == cid].iloc[0]
    cols = {}
    for i in IDX[::-1]:                                  # Apr .. Sep
        cols[f"{MONTH[i]} (i={i})"] = {
            "status PAY_i": int(r[f"PAY_{i}"]), "status group": r[f"STATUS_GRP_{i}"],
            "BILL_AMTi (statement)": f"{r[f'BILL_AMT{i}']:,.0f}", "PAY_AMTi (paid this month)": f"{r[f'PAY_AMT{i}']:,.0f}",
            "...settles statement of": MONTH[i + 1] if i < 6 else "Mar (n/a)",
            "UTIL_i": f"{r[f'UTIL_{i}']:.3f}",
            "PAY_RATIO_i": (f"{r[f'PAY_RATIO_{i}']:.3f}" if pd.notna(r[f"PAY_RATIO_{i}"]) else "NaN") if i < 6 else "n/a",
            "SPEND_i": f"{r[f'SPEND_{i}']:,.0f}" if i < 6 else "n/a"}
    return pd.DataFrame(cols)

for _name, _cid in S00B_CASES.items():
    _r = df.loc[df["ID"] == _cid].iloc[0]
    display(Markdown(f"**{_name}: ID {_cid}** | limit NT$ {_r['LIMIT_BAL']:,.0f} | {_r['SEX_LABEL']}, {_r['AGE']} y, {_r['EDU_LABEL']}, {_r['MAR_LABEL']} | "
                     f"**DEFAULT = {_r[TARGET]}** ({_r['DEFAULT_LABEL']})"))
    display(s00b_month_table(_cid))

**All Section 0 summary features for the five clients side by side.** Read each column against the month tables above.

In [ ]:
S00B_SUMMARY_COLS = ["AVG_UTIL", "MAX_UTIL", "UTIL_TREND", "AVG_PAY_RATIO", "AVG_SPEND", "N_LATE", "MAX_DELAY", "N_INACTIVE", "N_PAIDFULL",
                     "N_REVOLVE", "LATE_STREAK_MAX", "LATE_STREAK_RECENT", "DELAY_TREND", "TOTAL_BILL", "TOTAL_PAY", "PAY_TO_BILL_6M"] + (["PAY_TO_BILL_5M"] if "PAY_TO_BILL_5M" in df.columns else []) + ["AVG_BILL",
                     "AVG_PAY", "BILL_TREND", "PAY_TREND", "N_ZERO_PAY", "OVERLIMIT_ANY", "N_OVERLIMIT", "NEG_BILL_ANY",
                     "UTIL_BAND", "N_LATE_BAND", "PAYRATIO_BAND", "AGE_BAND", "LIMIT_BAND", "STATUS_GRP_SEP"]
S00B_CASE_TABLE = df.set_index("ID").loc[list(S00B_CASES.values()), S00B_SUMMARY_COLS].T
S00B_CASE_TABLE.columns = [f"{k} (ID {v})" for k, v in S00B_CASES.items()]
display(S00B_CASE_TABLE.astype(object).map(lambda v: f"{v:,.4f}" if isinstance(v, (float, np.floating)) else v))
export(df.loc[df["ID"].isin(S00B_CASES.values()), ["ID", "LIMIT_BAL", "AGE", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL", TARGET] + PAY_COLS + BILL_COLS + PAMT_COLS
              + [f"UTIL_{i}" for i in IDX] + [f"PAY_RATIO_{i}" for i in IDX[:-1]] + SPEND_COLS + S00B_SUMMARY_COLS], "s00b_example_clients")

### 0B.9 Redo every formula by hand

For each example client the cell writes out the arithmetic with the client's own numbers, computes the value
independently, and compares it with the value stored in `df`. `OK` means they agree.

In [ ]:
def s00b_fmt(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NaN"
    if isinstance(v, (int, np.integer)) or float(v).is_integer() and abs(v) >= 100:
        return f"{v:,.0f}"
    return f"{v:,.4f}"

def s00b_runs(flags):
    s = "".join("L" if f else "." for f in flags)
    return s, max((len(x) for x in s.split(".")), default=0), len(s) - len(s.rstrip("L")) if s.endswith("L") else 0

def s00b_explain(cid):
    r = df.loc[df["ID"] == cid].iloc[0]
    L = float(r["LIMIT_BAL"])
    ch = IDX[::-1]                                                     # 6..1 = Apr..Sep
    bills = [float(r[f"BILL_AMT{i}"]) for i in ch]; pays = [float(r[f"PAY_AMT{i}"]) for i in ch]; st = [int(r[f"PAY_{i}"]) for i in ch]
    utils = [b / L for b in bills]
    out = []
    def chk(name, text, val):
        stored = r[name]
        ok = (pd.isna(val) and pd.isna(stored)) or (not pd.isna(val) and not pd.isna(stored) and np.isclose(float(val), float(stored), rtol=1e-9, atol=1e-9))
        out.append(f"  {name:<18} = {text} = {s00b_fmt(val)}   [df: {s00b_fmt(stored)}]  {'OK' if ok else 'MISMATCH'}")
        return ok
    oks = []
    oks.append(chk("UTIL_1", f"BILL_AMT1 / LIMIT_BAL = {r['BILL_AMT1']:,.0f} / {L:,.0f}", r["BILL_AMT1"] / L))
    oks.append(chk("UTIL_6", f"BILL_AMT6 / LIMIT_BAL = {r['BILL_AMT6']:,.0f} / {L:,.0f}", r["BILL_AMT6"] / L))
    for i in (1, 5):
        prev = float(r[f"BILL_AMT{i + 1}"])
        v = min(r[f"PAY_AMT{i}"] / prev, 5) if prev > 0 else np.nan
        oks.append(chk(f"PAY_RATIO_{i}", f"PAY_AMT{i} / BILL_AMT{i + 1} = {r[f'PAY_AMT{i}']:,.0f} / {prev:,.0f}" + ("" if prev > 0 else " (prev bill <= 0 -> NaN)"), v))
        oks.append(chk(f"SPEND_{i}", f"BILL_AMT{i} - BILL_AMT{i + 1} + PAY_AMT{i} = {r[f'BILL_AMT{i}']:,.0f} - ({prev:,.0f}) + {r[f'PAY_AMT{i}']:,.0f}",
                       r[f"BILL_AMT{i}"] - prev + r[f"PAY_AMT{i}"]))
    oks.append(chk("AVG_UTIL", "mean of utilisations Apr..Sep [" + ", ".join(f"{u:.3f}" for u in utils) + "]", float(np.mean(utils))))
    oks.append(chk("MAX_UTIL", "max of the same six", float(np.max(utils))))
    t = np.arange(6) - 2.5
    oks.append(chk("UTIL_TREND", "OLS slope of utilisation on t = -2.5..+2.5", float(np.polyfit(t, utils, 1)[0])))
    oks.append(chk("BILL_TREND", "OLS slope of bills [" + ", ".join(f"{b:,.0f}" for b in bills) + "]", float(np.polyfit(t, bills, 1)[0])))
    oks.append(chk("PAY_TREND", "OLS slope of payments [" + ", ".join(f"{p:,.0f}" for p in pays) + "]", float(np.polyfit(t, pays, 1)[0])))
    prs = [r[f"PAY_RATIO_{i}"] for i in IDX[:-1]]
    oks.append(chk("AVG_PAY_RATIO", "mean of available PAY_RATIO_1..5 [" + ", ".join(s00b_fmt(p) for p in prs) + "]", float(np.nanmean(prs)) if np.isfinite(prs).any() else np.nan))
    sp = [r[f"SPEND_{i}"] for i in IDX[:-1]]
    oks.append(chk("AVG_SPEND", "mean of SPEND_1..5 [" + ", ".join(f"{s:,.0f}" for s in sp) + "]", float(np.mean(sp))))
    seq, longest, recent = s00b_runs([s >= 1 for s in st])
    oks.append(chk("N_LATE", f"months with status >= 1 in Apr..Sep {st}", sum(s >= 1 for s in st)))
    oks.append(chk("MAX_DELAY", f"max of {st}", max(st)))
    oks.append(chk("N_INACTIVE", f"months with status -2", sum(s == -2 for s in st)))
    oks.append(chk("N_PAIDFULL", f"months with status -1", sum(s == -1 for s in st)))
    oks.append(chk("N_REVOLVE", f"months with status 0", sum(s == 0 for s in st)))
    oks.append(chk("LATE_STREAK_MAX", f"longest run of L in '{seq}' (L = late, Apr..Sep)", longest))
    oks.append(chk("LATE_STREAK_RECENT", f"run of L ending in Sep in '{seq}'", recent))
    oks.append(chk("DELAY_TREND", f"PAY_1 - PAY_6 = {st[-1]} - ({st[0]})", st[-1] - st[0]))
    pos = sum(max(b, 0) for b in bills)
    oks.append(chk("TOTAL_BILL", "sum of the six bills", sum(bills)))
    oks.append(chk("TOTAL_PAY", "sum of the six payments", sum(pays)))
    oks.append(chk("PAY_TO_BILL_6M", f"(legacy) TOTAL_PAY / sum(max(bill, 0)) = {sum(pays):,.0f} / {pos:,.0f}", sum(pays) / pos if pos > 0 else np.nan))
    p5 = sum(float(r[f"PAY_AMT{i}"]) for i in IDX[:-1]); b5 = sum(max(float(r[f"BILL_AMT{i + 1}"]), 0) for i in IDX[:-1])
    if "PAY_TO_BILL_5M" in df.columns:
        oks.append(chk("PAY_TO_BILL_5M", f"PAY_AMT1..5 / sum(max(BILL_AMT2..6, 0)) = {p5:,.0f} / {b5:,.0f}", p5 / b5 if b5 > 0 else np.nan))
    else:
        out.append(f"  PAY_TO_BILL_5M (aligned, not yet in df) = {p5:,.0f} / {b5:,.0f} = {s00b_fmt(p5 / b5 if b5 > 0 else np.nan)}")
    oks.append(chk("N_ZERO_PAY", f"months with payment = 0 in [" + ", ".join(f"{p:,.0f}" for p in pays) + "]", sum(p == 0 for p in pays)))
    oks.append(chk("N_OVERLIMIT", f"months with bill > {L:,.0f}", sum(b > L for b in bills)))
    oks.append(chk("OVERLIMIT_ANY", "1 if N_OVERLIMIT > 0", int(any(b > L for b in bills))))
    oks.append(chk("NEG_BILL_ANY", "1 if any bill < 0", int(any(b < 0 for b in bills))))
    out.append(f"  bands: AVG_UTIL {r['AVG_UTIL']:.3f} -> UTIL_BAND '{r['UTIL_BAND']}' | N_LATE {r['N_LATE']} -> N_LATE_BAND '{r['N_LATE_BAND']}' | "
               f"PAY_TO_BILL_6M {s00b_fmt(r['PAY_TO_BILL_6M'])} -> PAYRATIO_BAND '{r['PAYRATIO_BAND']}' | AGE {r['AGE']} -> '{r['AGE_BAND']}' | LIMIT {L:,.0f} -> '{r['LIMIT_BAND']}'")
    return out, all(oks)

S00B_HAND_OK = {}
for _name, _cid in S00B_CASES.items():
    _lines, S00B_HAND_OK[_name] = s00b_explain(_cid)
    print(f"=== {_name} (ID {_cid}) ===")
    print("\n".join(_lines)); print()
assert all(S00B_HAND_OK.values()), f"hand-check mismatch: {S00B_HAND_OK}"
print("all hand checks OK:", S00B_HAND_OK)

### 0B.9b The key "how much of the debt is paid down" measure: quarterly paydown (built in Section 6)

Because `PAY_TO_BILL_6M` is one month out of step (0B.11), Section 6 builds a cleaner measure, `SOW_QPD_1`:

`SOW_QPD_1 = (PAY_AMT1 + PAY_AMT2 + PAY_AMT3) / (BILL_AMT2 + BILL_AMT3 + BILL_AMT4)`

In words: **what the client paid in Jul + Aug + Sep, divided by what the Jun + Jul + Aug statements asked for.** Each
payment is matched with the statement it settles. Negative bills count as 0. If the three bills add up to 0 or less there
is nothing to pay down, so the value is missing ("No balance"). It is capped at 5 (500%).

| Value | Meaning |
|---|---|
| 1.0 (100%) | Paid off everything that was billed: a full payer. |
| 0.05 (5%) | Paid a small slice: a minimum payer, the debt barely moves. |
| 0 | Paid nothing although money was owed. |
| above 1 | Paid more than billed: clearing older debt or prepaying. |
| missing | Nothing was owed in Jun-Aug. |

Related columns (all Section 6): `SOW_QPD_2` / `SOW_QPD_3` = the same window one / two months earlier (overlapping);
`SOW_QPD_PRIOR` = payments May + Jun / statements Apr + May (two months only, because the March statement is missing);
`SOW_QPD_CHG` = `SOW_QPD_1 - SOW_QPD_PRIOR` (positive = paying down faster lately); `SOW_QPD_BAND`, `SOW_QPD_CHG_BAND` = readable bands.
The cell computes it by hand for the five example clients.

In [ ]:
def s00b_qpd(r, pays, bills):
    p = sum(float(r[f"PAY_AMT{i}"]) for i in pays); b = sum(max(float(r[f"BILL_AMT{i}"]), 0) for i in bills)
    return p, b, (min(p / b, 5) if b > 0 else np.nan)
_rows = []
for _name, _cid in S00B_CASES.items():
    _r = df.loc[df["ID"] == _cid].iloc[0]
    _p, _b, _q = s00b_qpd(_r, [1, 2, 3], [2, 3, 4])
    _pp, _bp, _qp = s00b_qpd(_r, [4, 5], [5, 6])
    print(f"{_name} (ID {_cid}): SOW_QPD_1 = ({_r['PAY_AMT1']:,.0f} + {_r['PAY_AMT2']:,.0f} + {_r['PAY_AMT3']:,.0f}) / "
          f"({_r['BILL_AMT2']:,.0f} + {_r['BILL_AMT3']:,.0f} + {_r['BILL_AMT4']:,.0f}) = {_p:,.0f} / {_b:,.0f} = {s00b_fmt(_q)}"
          f"   | SOW_QPD_PRIOR = {_pp:,.0f} / {_bp:,.0f} = {s00b_fmt(_qp)}   | PAY_TO_BILL_6M (misaligned) = {s00b_fmt(_r['PAY_TO_BILL_6M'])}")
    _rows.append({"persona": _name, "ID": _cid, "SOW_QPD_1": _q, "SOW_QPD_PRIOR": _qp, "SOW_QPD_CHG": _q - _qp,
                  "PAY_TO_BILL_6M": _r["PAY_TO_BILL_6M"], "outcome": _r["DEFAULT_LABEL"]})
display(pd.DataFrame(_rows).round(4))

### 0B.10 Automated check: recompute every Section 0 formula for all 30,000 clients

Each feature is rebuilt with **different code** from Section 0 (plain pandas, `np.polyfit`, string-based run counting,
explicit if/else banding) and compared row by row. The cell stops the notebook if any column disagrees.

In [ ]:
S00B_CHECKS = []
def s00b_check(name, got, exp, kind="num"):
    got, exp = pd.Series(got).reset_index(drop=True), pd.Series(exp).reset_index(drop=True)
    if kind == "num":
        g, e = got.to_numpy(float), exp.to_numpy(float)
        bad = ~(np.isclose(g, e, rtol=1e-9, atol=1e-6) | (np.isnan(g) & np.isnan(e)))
    else:
        bad = (got.astype(str).to_numpy() != exp.astype(str).to_numpy())
    S00B_CHECKS.append({"column": name, "rows_checked": len(got), "mismatches": int(bad.sum())})

_lim = df["LIMIT_BAL"].astype(float)
for i in IDX:
    s00b_check(f"UTIL_{i}", df[f"UTIL_{i}"], df[f"BILL_AMT{i}"] / _lim)
for i in IDX[:-1]:
    _prev = df[f"BILL_AMT{i + 1}"].astype(float)
    _pr = (df[f"PAY_AMT{i}"] / _prev).where(_prev > 0).clip(upper=5)
    s00b_check(f"PAY_RATIO_{i}", df[f"PAY_RATIO_{i}"], _pr)
    s00b_check(f"SPEND_{i}", df[f"SPEND_{i}"], df[f"BILL_AMT{i}"] - _prev + df[f"PAY_AMT{i}"])
_U = pd.concat([df[f"BILL_AMT{i}"] / _lim for i in IDX], axis=1)
s00b_check("AVG_UTIL", df["AVG_UTIL"], _U.mean(axis=1))
s00b_check("MAX_UTIL", df["MAX_UTIL"], _U.max(axis=1))
s00b_check("AVG_PAY_RATIO", df["AVG_PAY_RATIO"], df[[f"PAY_RATIO_{i}" for i in IDX[:-1]]].mean(axis=1, skipna=True))
s00b_check("AVG_SPEND", df["AVG_SPEND"], df[SPEND_COLS].mean(axis=1))
_St = df[PAY_COLS]
s00b_check("N_LATE", df["N_LATE"], (_St >= 1).sum(axis=1))
s00b_check("MAX_DELAY", df["MAX_DELAY"], _St.max(axis=1))
s00b_check("N_INACTIVE", df["N_INACTIVE"], (_St == -2).sum(axis=1))
s00b_check("N_PAIDFULL", df["N_PAIDFULL"], (_St == -1).sum(axis=1))
s00b_check("N_REVOLVE", df["N_REVOLVE"], (_St == 0).sum(axis=1))
_seq = ["".join("L" if v >= 1 else "." for v in row) for row in df[PAY_COLS[::-1]].to_numpy()]   # Apr..Sep
s00b_check("LATE_STREAK_MAX", df["LATE_STREAK_MAX"], [max(len(x) for x in s.split(".")) for s in _seq])
s00b_check("LATE_STREAK_RECENT", df["LATE_STREAK_RECENT"], [len(s) - len(s.rstrip("L")) for s in _seq])
s00b_check("DELAY_TREND", df["DELAY_TREND"], df["PAY_1"] - df["PAY_6"])
s00b_check("TOTAL_BILL", df["TOTAL_BILL"], df[BILL_COLS].sum(axis=1))
s00b_check("TOTAL_PAY", df["TOTAL_PAY"], df[PAMT_COLS].sum(axis=1))
_posb = df[BILL_COLS].clip(lower=0).sum(axis=1)
s00b_check("PAY_TO_BILL_6M", df["PAY_TO_BILL_6M"], (df[PAMT_COLS].sum(axis=1) / _posb).where(_posb > 0))
_den5 = df[[f"BILL_AMT{i + 1}" for i in IDX[:-1]]].clip(lower=0).sum(axis=1)
S00B_PTB5 = (df[[f"PAY_AMT{i}" for i in IDX[:-1]]].sum(axis=1) / _den5).where(_den5 > 0)
if "PAY_TO_BILL_5M" in df.columns:
    s00b_check("PAY_TO_BILL_5M", df["PAY_TO_BILL_5M"], S00B_PTB5)
s00b_check("AVG_BILL", df["AVG_BILL"], df[BILL_COLS].mean(axis=1))
s00b_check("AVG_PAY", df["AVG_PAY"], df[PAMT_COLS].mean(axis=1))
_t = np.arange(6) - 2.5
s00b_check("BILL_TREND", df["BILL_TREND"], np.polyfit(_t, df[BILL_COLS[::-1]].to_numpy(float).T, 1)[0])
s00b_check("UTIL_TREND", df["UTIL_TREND"], np.polyfit(_t, _U.iloc[:, ::-1].to_numpy(float).T, 1)[0])
s00b_check("PAY_TREND", df["PAY_TREND"], np.polyfit(_t, df[PAMT_COLS[::-1]].to_numpy(float).T, 1)[0])
s00b_check("N_ZERO_PAY", df["N_ZERO_PAY"], (df[PAMT_COLS] == 0).sum(axis=1))
_over = df[BILL_COLS].gt(_lim, axis=0)
s00b_check("OVERLIMIT_ANY", df["OVERLIMIT_ANY"], _over.any(axis=1).astype(int))
s00b_check("N_OVERLIMIT", df["N_OVERLIMIT"], _over.sum(axis=1))
s00b_check("NEG_BILL_ANY", df["NEG_BILL_ANY"], (df[BILL_COLS] < 0).any(axis=1).astype(int))

# categorical labels and bands with explicit if/else rules
def _grp(v):
    return {-2: "Inactive", -1: "Paid in full", 0: "Revolving", 1: "Late 1m", 2: "Late 2m"}.get(v, "Late 3m+" if v >= 3 else "Inactive")
for i in IDX:
    s00b_check(f"STATUS_GRP_{i}", df[f"STATUS_GRP_{i}"], df[f"PAY_{i}"].map(_grp), "cat")
s00b_check("STATUS_GRP_SEP", df["STATUS_GRP_SEP"], df["PAY_1"].map(_grp), "cat")
s00b_check("SEX_LABEL", df["SEX_LABEL"], df["SEX"].map(lambda v: "Male" if v == 1 else "Female"), "cat")
s00b_check("EDU_LABEL", df["EDU_LABEL"], df["EDUCATION"].map(lambda v: {1: "Graduate school", 2: "University", 3: "High school"}.get(v, "Other/Unknown")), "cat")
s00b_check("MAR_LABEL", df["MAR_LABEL"], df["MARRIAGE"].map(lambda v: {1: "Married", 2: "Single"}.get(v, "Other")), "cat")
s00b_check("DEFAULT_LABEL", df["DEFAULT_LABEL"], df[TARGET].map(lambda v: "Default" if v == 1 else "No default"), "cat")
def _age(a):
    for hi, lab in [(25, "21-25"), (30, "26-30"), (35, "31-35"), (40, "36-40"), (45, "41-45"), (50, "46-50"), (60, "51-60")]:
        if a <= hi: return lab
    return "61+"
s00b_check("AGE_BAND", df["AGE_BAND"], df["AGE"].map(_age), "cat")
def _lim_band(x):
    for hi, lab in [(50_000, "<=50k"), (100_000, "50-100k"), (150_000, "100-150k"), (200_000, "150-200k"), (300_000, "200-300k"), (500_000, "300-500k")]:
        if x <= hi: return lab
    return ">500k"
s00b_check("LIMIT_BAND", df["LIMIT_BAND"], df["LIMIT_BAL"].map(_lim_band), "cat")
def _util_band(u):
    for hi, lab in [(0, "<=0"), (.1, "0-10%"), (.3, "10-30%"), (.5, "30-50%"), (.7, "50-70%"), (.9, "70-90%"), (1.0, "90-100%")]:
        if u <= hi: return lab
    return ">100%"
s00b_check("UTIL_BAND", df["UTIL_BAND"], df["AVG_UTIL"].map(_util_band), "cat")
s00b_check("N_LATE_BAND", df["N_LATE_BAND"], df["N_LATE"].map(lambda n: "4-6" if n >= 4 else str(n)), "cat")
def _pr_band(x):
    if pd.isna(x): return "No bill"
    for hi, lab in [(0, "0"), (.05, "0-5%"), (.1, "5-10%"), (.25, "10-25%"), (.5, "25-50%"), (1.0, "50-100%")]:
        if x <= hi: return lab
    return ">100%"
_m5 = (df["PAYRATIO_BAND"].astype(str) == S00B_PTB5.map(_pr_band).astype(str)).mean()
_m6 = (df["PAYRATIO_BAND"].astype(str) == df["PAY_TO_BILL_6M"].map(_pr_band).astype(str)).mean()
S00B_PRB_SRC = "PAY_TO_BILL_5M" if _m5 >= _m6 else "PAY_TO_BILL_6M"
print(f"PAYRATIO_BAND is built from {S00B_PRB_SRC} in this run (agreement {max(_m5, _m6):.2%})")
s00b_check(f"PAYRATIO_BAND (from {S00B_PRB_SRC})", df["PAYRATIO_BAND"], (S00B_PTB5 if S00B_PRB_SRC == "PAY_TO_BILL_5M" else df["PAY_TO_BILL_6M"]).map(_pr_band), "cat")
s00b_check("identity N_LATE+N_INACTIVE+N_PAIDFULL+N_REVOLVE = 6", df["N_LATE"] + df["N_INACTIVE"] + df["N_PAIDFULL"] + df["N_REVOLVE"], np.full(len(df), 6))

S00B_CHECKS = pd.DataFrame(S00B_CHECKS)
S00B_CHECKS["result"] = np.where(S00B_CHECKS["mismatches"] == 0, "OK", "MISMATCH")
print(f"{(S00B_CHECKS['result'] == 'OK').sum()} of {len(S00B_CHECKS)} checks pass")
display(S00B_CHECKS[S00B_CHECKS["result"] != "OK"] if (S00B_CHECKS["result"] != "OK").any() else S00B_CHECKS.head(8))
assert (S00B_CHECKS["mismatches"] == 0).all(), "a Section 0 formula does not match its definition"

### 0B.11 Edge cases and weak spots, measured

The formulas are computed correctly (0B.10). This cell asks a different question: **do the definitions make sense?**
It counts how often each known edge case happens, so you can judge how much it matters.

In [ ]:
_U6 = df[[f"UTIL_{i}" for i in IDX]].to_numpy(float)
_PR5 = df[[f"PAY_RATIO_{i}" for i in IDX[:-1]]].to_numpy(float)
_SP5 = df[SPEND_COLS].to_numpy(float)
_ptb = df["PAY_TO_BILL_6M"]
_den_al = df[[f"BILL_AMT{i + 1}" for i in IDX[:-1]]].clip(lower=0).sum(axis=1)
_ptb_aligned = (df[[f"PAY_AMT{i}" for i in IDX[:-1]]].sum(axis=1) / _den_al).where(_den_al > 0)
_zero_pay = df[PAMT_COLS].to_numpy() == 0
_inact = df[PAY_COLS].to_numpy() == -2
_dt = df["DELAY_TREND"]
S00B_EDGE = pd.DataFrame([
    ("UTIL_i", "client-months with utilisation > 1 (over limit)", float((_U6 > 1).mean())),
    ("UTIL_i", "clients with any month over limit", float((_U6 > 1).any(axis=1).mean())),
    ("UTIL_i", "client-months with utilisation < 0 (credit balance)", float((_U6 < 0).mean())),
    ("PAY_RATIO_i", "client-months undefined (previous bill <= 0)", float(np.isnan(_PR5).mean())),
    ("PAY_RATIO_i", "client-months hitting the cap of 5", float((_PR5 == 5).mean())),
    ("AVG_PAY_RATIO", "clients with no defined ratio at all (NaN)", float(df["AVG_PAY_RATIO"].isna().mean())),
    ("SPEND_i", "client-months with negative implied spend", float((_SP5 < 0).mean())),
    ("SPEND_i", "client-months with spend > 0 but < 1% of limit (interest-only look)", float(((_SP5 > 0) & (_SP5 < 0.01 * df[["LIMIT_BAL"]].to_numpy())).mean())),
    ("PAY_TO_BILL_6M", "clients with value > 1", float((_ptb > 1).mean())),
    ("PAY_TO_BILL_6M", "maximum value", float(_ptb.max())),
    ("PAY_TO_BILL_6M", "Spearman rho with the aligned PAY_TO_BILL_5M (PAY_AMT1..5 / BILL_AMT2..6)", float(stats.spearmanr(_ptb, _ptb_aligned, nan_policy="omit")[0])),
    ("PAY_TO_BILL_6M", "clients whose band changes under the aligned version", float((pd.cut(_ptb, [-np.inf, 0, .05, .1, .25, .5, 1, np.inf]).astype(str) != pd.cut(_ptb_aligned, [-np.inf, 0, .05, .1, .25, .5, 1, np.inf]).astype(str)).mean())),
    ("N_ZERO_PAY", "share of zero-payment months that are inactive (status -2) months", float((_zero_pay & _inact).sum() / _zero_pay.sum())),
    ("N_ZERO_PAY", "share of zero-payment months with previous bill <= 0 (nothing was owed)", float((_zero_pay[:, :5] & (df[[f"BILL_AMT{i + 1}" for i in IDX[:-1]]].to_numpy() <= 0)).sum() / _zero_pay[:, :5].sum())),
    ("DELAY_TREND", "clients with DELAY_TREND > 0 who are NOT late in Sep (rise is within -2/-1/0)", float(((_dt > 0) & (df["PAY_1"] <= 0)).sum() / max((_dt > 0).sum(), 1))),
    ("MAX_DELAY", "clients whose worst status is -2 (never used, never late)", float((df["MAX_DELAY"] == -2).mean())),
    ("STATUS code 1", "share of all code-1 client-months that fall in Sep", float((df["PAY_1"] == 1).sum() / (df[PAY_COLS] == 1).to_numpy().sum())),
    ("STATUS -2", "share of status -2 client-months with a positive bill", float(((df[PAY_COLS].to_numpy() == -2) & (df[BILL_COLS].to_numpy() > 0)).sum() / _inact.sum())),
    ("Inactive card", "default rate of clients with all-zero bills and payments", float(df.loc[(df[BILL_COLS] == 0).all(axis=1) & (df[PAMT_COLS] == 0).all(axis=1), TARGET].mean())),
], columns=["column", "edge case", "value"])
display(S00B_EDGE.round(4))
export(S00B_EDGE, "s00b_edge_cases")

**How to read the weak spots** (each is measured in the table above):

1. **`PAY_TO_BILL_6M` (legacy) is misaligned and unbounded.** Its numerator is the six payments made Apr-Sep. Those settle the
   Mar-Aug bills. Its denominator is the six Apr-Sep bills. So it is one month out of step (the table shows how many clients
   change band). The aligned replacement is `PAY_TO_BILL_5M = PAY_AMT1..5 / BILL_AMT2..6`, and `PAYRATIO_BAND` is being rebuilt
   from it. Both versions sum balances, so a debt carried for six months counts six times: read them as "payments per NT$ of
   balance-months". For "how much of the debt is paid down" over a recent quarter, use `SOW_QPD_1` (0B.9b).
   `CI_PAYR5` (Section 14) still uses the legacy version.
2. **`DELAY_TREND` treats status codes as numbers.** A move from -2 (unused) to 0 (revolving) counts as +2
   "deterioration". Many positive values belong to clients who are not late in September.
3. **`N_ZERO_PAY` mixes two stories.** "Owed money and paid nothing" (a risk sign) and "owed nothing" (an unused card) both count.
4. **`PAY_RATIO_i` and `AVG_PAY_RATIO` have holes.** NaN when no bill was due. The average uses whichever months exist,
   and the cap at 5 hides large pre-payments.
5. **`SPEND_i` is net new charges, not purchases.** It includes interest and fees, and it can be negative.
6. **Status code 1 is a September-only code.** `N_LATE`, `MAX_DELAY` and every "status >= 1" rule count a one-month
   slip only if it happens in September.
7. **`UTIL_i > 1` and `< 0` are real**, not errors. Bands handle them with ">100%" and "<=0". Note that "<=0" mixes unused cards and credit balances.
8. **Inactive is not safe.** Cards with zero bills and zero payments in all six months still default at a rate well
   above the base rate (last row of the table). "No activity" is not "no risk", and the -2 code alone does not mean "zero bill".

### 0B.12 Machine-readable dictionary (exported, and reused by Section 22)
One row per raw or Section 0 column: section, plain definition with formula, how to read it, caveat. Section 22
reuses `S00B_DICT` so that its "definition missing" flag only fires for genuinely undocumented columns.

In [ ]:
S00B_DICT = {
    "ID": ("raw", "Row key 1..30,000.", "A label only; use it to join tables.", "No meaning; never a feature."),
    "LIMIT_BAL": ("raw", "Credit line granted (NT$ 10k-1M), covers client and family/supplementary cards.", "200,000 is about the median; bill / limit = utilisation.", "Set by the bank in round steps; not a client trait."),
    "SEX": ("raw", "1 = male, 2 = female.", "Use SEX_LABEL in charts.", "Protected trait: describe, do not decide on it."),
    "EDUCATION": ("raw", "1 grad school, 2 university, 3 high school, 4 others, 0/5/6 undocumented.", "Use EDU_LABEL (0/4/5/6 -> Other/Unknown).", "Other/Unknown defaults at ~7%: likely a special segment."),
    "MARRIAGE": ("raw", "1 married, 2 single, 3 others, 0 undocumented.", "Use MAR_LABEL.", "0 has 54 rows."),
    "AGE": ("raw", "Age in years (21-79), measured once.", "35 = 35 years old.", "Does not change over the window."),
    TARGET: ("raw", "1 if the client defaulted on the payment due in Oct 2005 (raw name default.payment.next.month).", "Base rate 22.1%.", "One month after the window; anything built from it is an outcome summary."),
    "SEX_LABEL": ("0", "SEX as Male / Female.", "Example: SEX = 2 -> Female (60% of clients).", "Describe only; do not decide on gender."),
    "EDU_LABEL": ("0", "EDUCATION as Graduate school / University / High school / Other/Unknown (0,4,5,6).", "Example: EDUCATION = 5 -> Other/Unknown.", "Folds documented 'others' with undocumented codes."),
    "MAR_LABEL": ("0", "MARRIAGE as Married / Single / Other (0,3).", "Example: MARRIAGE = 0 -> Other.", "Other is small (~1%)."),
    "DEFAULT_LABEL": ("0", "DEFAULT as 'No default' / 'Default'.", "Example: DEFAULT = 1 -> Default.", "It is the outcome, not a feature."),
    "AGE_BAND": ("0", "AGE cut (20,25],(25,30],...,(45,50],(50,60],(60,80].", "Age 26 -> '26-30'.", "Last two bands are wider."),
    "LIMIT_BAND": ("0", "LIMIT_BAL cut at 50k,100k,150k,200k,300k,500k (right edge included).", "50,000 -> '<=50k'.", "Unequal widths."),
    "STATUS_GRP_SEP": ("0", "Sep status grouped: Inactive(-2), Paid in full(-1), Revolving(0), Late 1m(1), Late 2m(2), Late 3m+(3-8).", "Same as STATUS_GRP_1.", "Late 1m is Sep-only."),
    "AVG_UTIL": ("0", "mean(UTIL_1..6).", "0.46 = 46% of the line used on average.", "Over-limit and credit months both included."),
    "MAX_UTIL": ("0", "max(UTIL_1..6).", "1.29 = 29% over the limit in the worst month.", "One spike decides it."),
    "UTIL_TREND": ("0", "OLS slope of utilisation on time Apr..Sep (t = -2.5..2.5).", "+0.10 = +10 points of the line per month.", "Six points; sensitive to one jump."),
    "AVG_PAY_RATIO": ("0", "mean of available PAY_RATIO_1..5 (NaN skipped).", "1.0 = pays the previous bill in full; 0.05 = ~minimum.", "Different months for different clients; cap 5 inflates means."),
    "AVG_SPEND": ("0", "mean(SPEND_1..5), NT$ per month.", "15,000 = 15k of net new charges a month.", "Includes interest; can be negative."),
    "N_LATE": ("0", "months with status >= 1 (0-6).", "4 = late in four of six months.", "Code 1 exists only in Sep."),
    "MAX_DELAY": ("0", "worst status code over six months (-2..8).", "2 = at worst two months behind.", "-2/-1/0 are kinds of 'not late'."),
    "N_INACTIVE": ("0", "months with status -2.", "6 = never used in the window.", "-2 months can still carry a small bill."),
    "N_PAIDFULL": ("0", "months with status -1 (the bank's CODE says paid in full).", "6 = coded paid in full every month: 1,992 clients, 284 defaulted (14.3%).", "Code-based. The 725 defaulters often quoted are a different, MONEY-based group: 'cleared every bill by amount' (Section 1, CS_G03)."),
    "N_REVOLVE": ("0", "months with status 0.", "6 = revolver all window.", "N_LATE+N_INACTIVE+N_PAIDFULL+N_REVOLVE = 6."),
    "LATE_STREAK_MAX": ("0", "longest run of consecutive late months (status >= 1).", "3 = three late months in a row.", "Ignores depth of delay."),
    "LATE_STREAK_RECENT": ("0", "run of late months ending in Sep.", "1 = late in Sep only (Aug was not late).", "0 if current in Sep."),
    "DELAY_TREND": ("0", "PAY_1 - PAY_6 (Sep code minus Apr code).", "+3 = from paid-in-full (-1) to 2 months late.", "Treats codes as numbers: -2 -> 0 gives +2 without any delay."),
    "TOTAL_BILL": ("0", "sum of six bills (NT$).", "Balance-months, not amount borrowed.", "Carried debt counted up to six times."),
    "TOTAL_PAY": ("0", "sum of six payments (NT$).", "Example: 3,000 a month -> 18,000.", "PAY_AMT6 settles a March bill outside the window."),
    "PAY_TO_BILL_6M": ("0", "LEGACY, not aligned: TOTAL_PAY / sum(max(BILL_AMTi,0)) over Apr-Sep; NaN if no positive bill.", "0.05 = pays ~5% of balances; ~1 = full payer. Example: ID 15 -> 18,000 / 373,591 = 0.048.", "One month out of step, double-counts carried balances, uncapped (max ~800). Use PAY_TO_BILL_5M."),
    "PAY_TO_BILL_5M": ("0", "Share of bills repaid, aligned: sum(PAY_AMT1..5) / sum(max(BILL_AMT2..6, 0)); NaN if those bills sum to 0.", "1.0 = repaid everything billed; 0.05 = about the minimum. Example: ID 15 -> 15,000 / 302,704 = 0.050.", "Sums balances, so carried debt counts once per month; not capped."),
    "AVG_BILL": ("0", "mean of six bills (NT$).", "Example: ID 15 -> 62,265 (average debt carried).", "Stock, not flow."),
    "AVG_PAY": ("0", "mean of six payments (NT$).", "Example: ID 15 -> 3,000 a month.", "Says nothing about how much was owed."),
    "BILL_TREND": ("0", "OLS slope of bills Apr..Sep (NT$ per month).", "+10,000 = bill grows 10k a month.", "Not scaled by limit."),
    "PAY_TREND": ("0", "OLS slope of payments Apr..Sep (NT$ per month).", "-500 = payments shrink 500 a month.", "One lump payment tilts the slope."),
    "N_ZERO_PAY": ("0", "months with PAY_AMTi = 0.", "6 = never paid anything (or never owed anything).", "Mixes non-payment with nothing owed."),
    "OVERLIMIT_ANY": ("0", "1 if any bill > LIMIT_BAL.", "Example: ID 6 (bill 64,400 on 50,000) -> 1.", "Strict >."),
    "N_OVERLIMIT": ("0", "months with bill > LIMIT_BAL.", "3 = over the line in three months.", "Interest alone can push a full card over."),
    "NEG_BILL_ANY": ("0", "1 if any bill < 0 (credit balance).", "Example: bill -2,000 in one month -> 1.", "Credit balances are rare (~2% of months)."),
    "UTIL_BAND": ("0", "AVG_UTIL banded <=0, 0-10%, 10-30%, 30-50%, 50-70%, 70-90%, 90-100%, >100%.", "0.46 -> '30-50%'.", "'<=0' mixes unused and credit balances."),
    "N_LATE_BAND": ("0", "N_LATE as 0,1,2,3,4-6.", "Example: 4 late months -> '4-6'.", "Inherits the Sep-only code-1 issue."),
    "PAYRATIO_BAND": ("0", "PAY_TO_BILL_5M (aligned; PAY_TO_BILL_6M in older runs) banded 0, 0-5%, 5-10%, 10-25%, 25-50%, 50-100%, >100%, No bill.", "0.05 -> '0-5%'.", "Check which source the run used (0B.10 prints it)."),
}
for i in IDX:
    S00B_DICT[f"PAY_{i}"] = ("raw", f"Repayment status in {MONTH[i]} 2005 (index {i}{'; raw name PAY_0' if i == 1 else ''}): -2 no consumption, -1 paid in full, 0 revolving, 1-8 months of delay.",
                             "Ordered category; 0 = paid the minimum, 2 = two months behind.", "Code 1 almost only in Sep; -2/-1/0 are not a numeric scale.")
    S00B_DICT[f"BILL_AMT{i}"] = ("raw", f"Statement balance for {MONTH[i]} 2005 (NT$, can be negative).", "45,000 = owed 45k on that statement.", "A stock; can exceed the limit.")
    S00B_DICT[f"PAY_AMT{i}"] = ("raw", f"Amount paid during {MONTH[i]} 2005 (NT$), settling the " + (f"{MONTH[i + 1]} statement BILL_AMT{i + 1}." if i < 6 else "March statement (not in data)."),
                                "3,000 = paid 3k that month.", "Pairs with the NEXT index's bill.")
    S00B_DICT[f"STATUS_GRP_{i}"] = ("0", f"{MONTH[i]} status grouped (Inactive / Paid in full / Revolving / Late 1m / Late 2m / Late 3m+).", "Example: status 4 -> Late 3m+.", "Late 1m is Sep-only.")
    S00B_DICT[f"UTIL_{i}"] = ("0", f"BILL_AMT{i} / LIMIT_BAL ({MONTH[i]} utilisation).", "0.45 = 45% of line; >1 over limit; <0 credit.", "Bill includes interest/fees.")
for i in IDX[:-1]:
    S00B_DICT[f"PAY_RATIO_{i}"] = ("0", f"PAY_AMT{i} / BILL_AMT{i + 1}: share of the {MONTH[i + 1]} statement paid in {MONTH[i]}; NaN if that bill <= 0; capped at 5.", "1.0 = paid in full; 0.05 = ~minimum.", "Undefined when nothing was owed; cap hides pre-payments.")
    S00B_DICT[f"SPEND_{i}"] = ("0", f"BILL_AMT{i} - BILL_AMT{i + 1} + PAY_AMT{i}: net new charges in {MONTH[i]} (NT$).", "15,000 = 15k of purchases + interest + fees.", "Can be negative (refunds/reversals).")
S00B_DICT_FRAME = pd.DataFrame([(k,) + v for k, v in S00B_DICT.items()], columns=["column", "section", "definition", "how_to_read", "caveat"])
_missing = [c for c in df.columns if c not in S00B_DICT]
print(f"documented columns: {len(S00B_DICT)} | df columns now: {df.shape[1]} | undocumented at this point: {_missing if _missing else 'none'}")
export(S00B_DICT_FRAME, "s00b_data_dictionary")
print(f"Section 0B runtime {time.time() - S00B_T0:.1f}s")

## 1. Data quality, undocumented codes and anomalies
Before any pattern is trusted we audit the table itself: schema, duplicates, codes that the data dictionary does not define,
internal contradictions between the repayment-status codes and the money columns, outliers, and digit-level integrity checks.
Every check feeds a single **data-quality scorecard** (1.12), exported for a Tableau "data health" sheet.

All flags are held in a side frame `dq` (same index as `df`); **no `df` columns are added or changed in this section.**

**Section guide**

- **What it does:** Audits the raw table before we trust any pattern. It checks missing values, duplicates, undefined codes, contradictions between status codes and money columns, outliers and digit patterns. Nothing is dropped; every issue becomes a flag.
- **Question it answers:** Is the data clean enough to analyse, and which oddities need a caveat?
- **Inputs / new columns:** all 25 raw columns. No `df` columns are added; flags live in a side table `dq`.
- **How to read the charts:** bars = number of clients; red line or dots = default rate; dashed line = 22.1% base rate (share of all clients who default). Benford charts compare observed first-digit shares (bars) with the expected curve.
- **Key takeaways:**
  - 30,000 rows, 25 columns, zero missing cells.
  - Status codes -2 (card not used) and 0 (revolving: minimum paid) are undocumented but fill most cells; code 1 (late 1 month) appears almost only in September.
  - Education codes 0/4/5/6 fold safely into Other/Unknown (468 clients, 7.05% default).
  - Dormant accounts (all six bills zero) default at 36.6%; over-limit clients at 30.1%.
  - 725 clients paid every bill in full and still defaulted (13.6%).
- **Where it feeds:** the data-quality appendix and `s01_quality_scorecard.csv` (Tableau "data health" sheet); the dormant and over-limit flags feed the risk dashboard.

### 1.1 Schema: dtypes, missing values, cardinality, ranges
One row per raw column. `doc_range` is the range implied by the UCI data dictionary; `in_doc_range` says whether every observed value respects it.

In [ ]:
RAW_COLS = ["ID", "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"] + PAY_COLS + BILL_COLS + PAMT_COLS + [TARGET]
DOC_RANGE = {"ID": (1, 30000), "LIMIT_BAL": (0, None), "SEX": (1, 2), "EDUCATION": (1, 4), "MARRIAGE": (1, 3),
             "AGE": (18, 100), TARGET: (0, 1)}
for c in PAY_COLS:
    DOC_RANGE[c] = (-1, 9)        # data dictionary documents only -1 (pay duly) and 1..9 (months delay)
for c in PAMT_COLS:
    DOC_RANGE[c] = (0, None)      # payments cannot be negative
for c in BILL_COLS:
    DOC_RANGE[c] = (None, None)   # no documented bound (credit balances can be negative)

rows = []
for c in RAW_COLS:
    s = df[c]
    lo, hi = DOC_RANGE[c]
    bad = pd.Series(False, index=s.index)
    if lo is not None: bad |= s < lo
    if hi is not None: bad |= s > hi
    rows.append({"column": c, "dtype": str(s.dtype), "n_missing": int(s.isna().sum()), "n_unique": int(s.nunique()),
                 "min": s.min(), "max": s.max(), "doc_range": f"[{lo}, {hi}]", "in_doc_range": not bad.any(),
                 "n_outside_doc": int(bad.sum())})
schema = pd.DataFrame(rows)
print(f"Total missing cells: {int(df[RAW_COLS].isna().sum().sum())} | memory {df[RAW_COLS].memory_usage(deep=True).sum()/1e6:.1f} MB")
schema

**Observations:**
- The table has 30,000 rows, 25 columns and zero missing cells.
- EDUCATION has 345 values outside the documented 1-4 range and MARRIAGE has 54 outside 1-3.
- Every PAY_x column uses the undocumented codes -2 and 0: from 2,759 (Sep) to 4,895 (Apr) cells per month fall outside the documented range.
- Bills can be negative (minimum NT$ -339,603) and one payment reaches NT$ 1.68M, so money columns need log scales.
- Story use: background only (data-quality appendix).

### 1.2 Duplicates and constant columns
`ID` must be unique. We also look for identical client records once `ID` is removed (possible double entry), and whether duplicate
records ever carry *conflicting* targets (which would indicate label noise).

In [ ]:
dq = pd.DataFrame(index=df.index)
FEAT_RAW = [c for c in RAW_COLS if c not in ("ID", TARGET)]

n_dup_id = int(df["ID"].duplicated().sum())
dup_mask = df.duplicated(subset=FEAT_RAW, keep=False)
dq["dup_features"] = dup_mask
dup_groups = df[dup_mask].groupby(FEAT_RAW, dropna=False)
conflicting = int((dup_groups[TARGET].nunique() > 1).sum())
const_cols = [c for c in RAW_COLS if df[c].nunique() <= 1]
summary_dup = pd.DataFrame({
    "check": ["duplicate IDs", "rows in duplicate feature groups (excl. ID)", "extra copies (rows removable)",
              "duplicate groups", "groups with conflicting DEFAULT", "constant columns"],
    "value": [n_dup_id, int(dup_mask.sum()), int(df.duplicated(subset=FEAT_RAW).sum()),
              int(dup_groups.ngroups) if dup_mask.any() else 0, conflicting, len(const_cols)]})
display(summary_dup)
if dup_mask.any():
    dd = df[dup_mask]
    print(f"Default rate among duplicated rows: {dd[TARGET].mean():.2%} (base {BASE_RATE:.2%})")
    print("Typical profile of duplicates (mostly all-zero activity?):")
    print(f"  share with all 6 bills = 0: {(dd[BILL_COLS] == 0).all(1).mean():.1%} | all statuses = -2: {(dd[PAY_COLS] == -2).all(1).mean():.1%}")
    display(dd[["ID", "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"] + PAY_COLS[:3] + BILL_COLS[:2] + [TARGET]]
            .sort_values(FEAT_RAW[:5]).head(10))

**Observations:**
- No ID is duplicated and no column is constant.
- 108 rows (52 groups) share identical features; 56 are extra copies, and 21 groups disagree on DEFAULT, so they are distinct clients.
- 89.8% of these look-alikes have all six bills at zero, i.e. near-empty accounts.
- They default at 29.6% vs the 22.1% base, so we keep them.
- Story use: background only.

### 1.3 Undocumented `EDUCATION` codes (0, 5, 6) and the "Other" fold
The dictionary defines 1 = graduate school, 2 = university, 3 = high school, 4 = others. Codes 0, 5, 6 appear with no definition
(5 and 6 are listed as "unknown" on the UCI page). For each raw code: count, default rate with Wilson 95% CI (confidence interval: the plausible range for the true rate), and lift vs base (rate ÷ 22.1% base rate).

In [ ]:
edu_raw = rate_table(df, "EDUCATION")
edu_raw["documented"] = edu_raw["EDUCATION"].isin([1, 2, 3, 4])
edu_raw["label"] = edu_raw["EDUCATION"].map({0: "0 (undoc.)", 1: "1 Graduate", 2: "2 University", 3: "3 High school",
                                             4: "4 Others", 5: "5 Unknown", 6: "6 Unknown"})
display(edu_raw[["EDUCATION", "label", "documented", "n", "defaults", "rate", "ci_lo", "ci_hi", "lift", "share_clients"]])

fig, ax = plt.subplots(figsize=(10, 4.5))
x = np.arange(len(edu_raw))
cols = [C_NEU if d else C_ACC for d in edu_raw["documented"]]
ax.bar(x, edu_raw["rate"], color=[C_DEF if d else C_ACC for d in edu_raw["documented"]], alpha=.75)
ax.errorbar(x, edu_raw["rate"], yerr=[edu_raw["rate"] - edu_raw["ci_lo"], edu_raw["ci_hi"] - edu_raw["rate"]], fmt="none", color="k", capsize=4)
for xi, (r, n) in enumerate(zip(edu_raw["rate"], edu_raw["n"])):
    ax.text(xi, r + 0.01, f"{r:.1%}\nn={n:,}", ha="center", fontsize=8)
ax.axhline(BASE_RATE, ls="--", color="k", lw=1)
ax.set_xticks(x); ax.set_xticklabels(edu_raw["label"])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title("Default rate by raw EDUCATION code (green = undocumented / 'others', 95% Wilson CI)")
savefig("s01_education_raw_codes"); plt.show()

**Is folding 0/4/5/6 into "Other/Unknown" safe?** We test (a) homogeneity *within* the fold (chi-square / Fisher exact tests, which ask whether rates differ more than chance allows, on 0, 4, 5, 6),
(b) odd codes (0, 5, 6) vs the documented "others" (4), and (c) the fold vs each documented level. If (a)/(b) are non-significant the fold
is statistically harmless; (c) tells us whether the fold is a genuinely distinct, low-risk group.

In [ ]:
def two_group_test(mask_a, mask_b, name):
    a, b = df.loc[mask_a, TARGET], df.loc[mask_b, TARGET]
    tab = np.array([[a.sum(), len(a) - a.sum()], [b.sum(), len(b) - b.sum()]])
    chi2, p_chi, _, exp = stats.chi2_contingency(tab, correction=True)
    p_fish = stats.fisher_exact(tab)[1]
    return {"comparison": name, "n_a": len(a), "rate_a": a.mean(), "n_b": len(b), "rate_b": b.mean(),
            "diff_pp": (a.mean() - b.mean()) * 100, "chi2": chi2, "p_chi2": p_chi, "p_fisher": p_fish,
            "min_expected": exp.min()}

E = df["EDUCATION"]
odd_edu = E.isin([0, 5, 6])
fold = E.isin([0, 4, 5, 6])
tests = [two_group_test(odd_edu, E == 4, "odd (0,5,6) vs 4 others"),
         two_group_test(E == 5, E == 4, "5 vs 4"),
         two_group_test(E == 6, E == 4, "6 vs 4"),
         two_group_test(E == 0, E == 4, "0 vs 4"),
         two_group_test(E == 6, E == 5, "6 vs 5"),
         two_group_test(fold, E.isin([1, 2, 3]), "fold (0,4,5,6) vs documented 1-3")]
for lvl, nm in [(1, "graduate"), (2, "university"), (3, "high school")]:
    tests.append(two_group_test(fold, E == lvl, f"fold vs {lvl} {nm}"))
edu_tests = pd.DataFrame(tests)
display(edu_tests)

ct = pd.crosstab(E[fold], df.loc[fold, TARGET])
chi2, p, dof, exp = stats.chi2_contingency(ct)
print(f"Homogeneity within fold (codes 0,4,5,6): chi2={chi2:.2f}, dof={dof}, p={p:.4f}, min expected={exp.min():.1f}")
print(f"Fold default rate {df.loc[fold, TARGET].mean():.2%} vs base {BASE_RATE:.2%} "
      f"-> the 'Other/Unknown' group is a LOW-risk group ({fold.sum()} clients, {fold.mean():.2%} of book).")
print(f"Striking: EDUCATION 0 has {int(df.loc[E == 0, TARGET].sum())} defaults out of {int((E == 0).sum())} clients; "
      f"codes 5 and 4 default at ~{df.loc[E == 5, TARGET].mean():.1%} / {df.loc[E == 4, TARGET].mean():.1%}.")
dq["edu_undocumented"] = odd_edu

**Observations:**
- Codes 0, 5 and 6 cover 345 clients; folded with code 4 the Other/Unknown group has 468 clients (1.56% of the book).
- This group defaults at only 7.05% vs 22.36% for codes 1-3 (chi-square p < 0.0001).
- The four folded codes do not differ from each other (p = 0.061), so folding is safe; code 0 has 0 defaults in 14 clients.
- Story use: footnote on charts by education; Other/Unknown is a small, low-risk group, not a data error to drop.

### 1.4 Undocumented `MARRIAGE` code 0
Dictionary: 1 = married, 2 = single, 3 = others. Code 0 is undocumented and is folded into "Other" in setup.

In [ ]:
mar_raw = rate_table(df, "MARRIAGE")
mar_raw["documented"] = mar_raw["MARRIAGE"].isin([1, 2, 3])
display(mar_raw[["MARRIAGE", "documented", "n", "defaults", "rate", "ci_lo", "ci_hi", "lift"]])
M = df["MARRIAGE"]
mar_tests = pd.DataFrame([two_group_test(M == 0, M == 3, "0 vs 3 others"),
                          two_group_test(M == 0, M == 1, "0 vs 1 married"),
                          two_group_test(M == 0, M == 2, "0 vs 2 single"),
                          two_group_test(M.isin([0, 3]), M.isin([1, 2]), "fold (0,3) vs married+single")])
display(mar_tests)
dq["mar_undocumented"] = M == 0

**Observations:**
- MARRIAGE code 0 has only 54 clients and defaults at 9.3%, well below code 3 (26.0%, Fisher p = 0.006).
- Folding 0 and 3 into Other (377 clients) gives 23.6%, no different from married + single (22.1%, p = 0.52).
- Story use: background only; Other is too small to carry a message.

### 1.5 Repayment status codes −2, −1, 0
The dictionary only documents −1 (pay duly) and 1..9 (months of delay), yet **−2 and 0 dominate the data**. The dataset author
(I-Cheng Yeh, via Kaggle discussion) clarified:

| Code | Meaning |
|---|---|
| −2 | **No consumption** — no balance to pay / card not used that cycle |
| −1 | **Paid in full** — whole statement balance repaid |
| 0 | **Revolving credit** — minimum payment made, balance carried forward (the normal revolving state, *not* "late") |
| 1..8 | months of delay (9 = 9+ months) |

Below: code-by-month counts (chronological Apr→Sep) and default rate by code per month.

In [ ]:
chron_idx = IDX[::-1]   # 6 (Apr) .. 1 (Sep)
codes = sorted(np.unique(df[PAY_COLS].values))
cnt = pd.DataFrame({MONTH[i]: df[f"PAY_{i}"].value_counts().reindex(codes, fill_value=0) for i in chron_idx})
cnt.index.name = "status"
rate = pd.DataFrame({MONTH[i]: df.groupby(f"PAY_{i}")[TARGET].mean().reindex(codes) for i in chron_idx})
rate.index.name = "status"
print("Counts by status code and month:"); display(cnt)
print("Share of clients (%) by status code and month:"); display((cnt / len(df) * 100).round(2))
print("Default rate (%) by status code and month (NaN = code absent):"); display((rate * 100).round(1))
print("NOTE code 1 ('late 1m') is essentially absent Apr-Aug and appears only in Sep "
      f"({int(cnt.loc[1, 'Sep']):,} clients): in earlier months a first delay seems to be recorded directly as 2.")
print("Undocumented share of all status cells: -2 = "
      f"{(df[PAY_COLS] == -2).values.mean():.1%}, 0 = {(df[PAY_COLS] == 0).values.mean():.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for code in [-2, -1, 0, 1, 2, 3]:
    ax.plot(MONTH_ORDER, rate.loc[code, MONTH_ORDER], marker="o", label=STATUS_LABEL[code])
ax.axhline(BASE_RATE, ls="--", color="k", lw=1)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title("Default rate (Oct) by repayment status code in each month: -2/-1/0 are all low-risk, 1+ is the cliff")
ax.legend(ncol=3, fontsize=8); savefig("s01_status_code_rates"); plt.show()
p_tests = []
for i in IDX:
    s = df[f"PAY_{i}"]
    for a, b in [(-2, -1), (-2, 0), (-1, 0)]:
        r = two_group_test(s == a, s == b, f"{MONTH[i]}: {a} vs {b}")
        p_tests.append(r)
pt = pd.DataFrame(p_tests)[["comparison", "n_a", "rate_a", "n_b", "rate_b", "diff_pp", "p_chi2"]]
print("Are -2, -1 and 0 distinguishable in risk? (pairwise tests per month)"); display(pt)

**Observations:**
- Code 0 (revolving: minimum paid) fills 53.3% of all status cells and code -2 (no consumption) 13.6%; neither is in the official codebook.
- Code 1 (late 1 month) is almost absent Apr-Aug and appears only in Sep (3,688 clients, 12.3%), so a first delay is recorded as 2 in earlier months.
- Codes -2, -1 and 0 all default at 13-20%, while code 2 (late 2 months) defaults at 51-69%; the big step is from 'on time' to 'late'.
- -2, -1 and 0 differ by only 2-4 percentage points (pp) even when p < 0.01.
- Story use: justifies the status groups (Inactive / Paid in full / Revolving / Late) and the caveat that code 1 exists only in September.

### 1.6 Consistency between status codes and money columns
Timing convention: `PAY_i` is the status observed in month *i*; `PAY_AMTi` is paid in month *i* and settles statement `BILL_AMT(i+1)`.
Checks (per month, then "any month"):

* **A. −2 with a non-zero bill** — "no consumption" yet a balance exists (positive = carried balance, could be legit; still worth sizing).
* **B. −1 but not paid** — "paid in full" yet `PAY_AMTi < BILL_AMT(i+1)` with a positive prior bill.
* **C. Late (≥1) but fully paid** — status says delayed while `PAY_AMTi ≥ BILL_AMT(i+1) > 0`.
* **D. Impossible jump** — delay grows by more than 1 month in one month (`PAY_i > max(PAY_(i+1), 0) + 1`), e.g. Aug = 0 then Sep = 3.

In [ ]:
cons_rows = []
A_any = np.zeros(len(df), bool); B_any = A_any.copy(); C_any = A_any.copy(); D_any = A_any.copy()
for i in IDX:
    s, b = df[f"PAY_{i}"], df[f"BILL_AMT{i}"]
    A = (s == -2) & (b != 0)
    rec = {"month": MONTH[i], "A_-2_nonzero_bill": int(A.sum()), "A_-2_positive_bill": int(((s == -2) & (b > 0)).sum())}
    A_any |= A.values
    if i <= 5:
        prev, pay = df[f"BILL_AMT{i+1}"], df[f"PAY_AMT{i}"]
        Bm = (s == -1) & (prev > 0) & (pay < prev)
        Cm = (s >= 1) & (prev > 0) & (pay >= prev)
        Dm = s > np.maximum(df[f"PAY_{i+1}"], 0) + 1
        rec.update({"B_-1_underpaid": int(Bm.sum()), "C_late_but_paid_full": int(Cm.sum()), "D_jump_>1m": int(Dm.sum())})
        B_any |= Bm.values; C_any |= Cm.values; D_any |= Dm.values
    cons_rows.append(rec)
cons = pd.DataFrame(cons_rows).set_index("month").loc[MONTH_ORDER]
display(cons)
dq["A_minus2_nonzero_bill"] = A_any; dq["B_minus1_underpaid"] = B_any
dq["C_late_but_paid_full"] = C_any; dq["D_impossible_jump"] = D_any
flag_sum = pd.DataFrame({f: {"clients": int(dq[f].sum()), "share": dq[f].mean(),
                             "default_rate": df.loc[dq[f], TARGET].mean() if dq[f].any() else np.nan}
                         for f in ["A_minus2_nonzero_bill", "B_minus1_underpaid", "C_late_but_paid_full", "D_impossible_jump"]}).T
display(flag_sum)

Example rows for the impossible-jump check (D), shown chronologically, so the pattern is visible.

In [ ]:
jump_ex = df.loc[dq["D_impossible_jump"], ["ID"] + [f"PAY_{i}" for i in chron_idx] + [TARGET]].head(8)
jump_ex.columns = ["ID"] + [f"{MONTH[i]}" for i in chron_idx] + [TARGET]
display(jump_ex)
# which (previous -> current) transitions create the jumps, all months pooled
pairs = []
for i in IDX[:-1]:
    prv, cur = df[f"PAY_{i+1}"], df[f"PAY_{i}"]
    m = cur > np.maximum(prv, 0) + 1
    pairs.append(pd.DataFrame({"to_month": MONTH[i], "prev": prv[m].values, "curr": cur[m].values}))
pairs = pd.concat(pairs)
print("Most frequent impossible transitions (prev status -> current status), all months:")
print(pairs.groupby(["prev", "curr"]).size().sort_values(ascending=False).head(10).to_string())
print(f"Share of jump cells that are '<=0 -> 2': {((pairs['prev'] <= 0) & (pairs['curr'] == 2)).mean():.1%} "
      "-> jumps are the missing-code-1 artefact (first delay coded 2), not true multi-month leaps.")

**Observations:**
- 5,948 clients (19.8%) show an 'impossible' delay jump of more than one month; 100% of these are moves from 0 or less straight to 2, the missing-code-1 artefact.
- Clients with such a jump default at 44.5% (lift 2.0, i.e. twice the base rate), so the artefact still marks real first delays.
- 3,302 clients (11.0%) have status -2 with a non-zero bill and 5,159 (17.2%) have status -1 without paying the prior bill in full; status follows bank rules, not exact cash.
- Story use: background only (caveat: status codes and money columns are not perfectly consistent).

### 1.7 Over-limit, negative and dormant balances
* **Over-limit:** bill above the credit line (possible with interest/fees or a temporary line increase).
* **Negative bills:** credit balance (overpayment / refund).
* **Dormant:** all six bills equal to zero.

In [ ]:
ol = pd.DataFrame({"month": [MONTH[i] for i in chron_idx],
                   "n_overlimit": [int((df[f"BILL_AMT{i}"] > df["LIMIT_BAL"]).sum()) for i in chron_idx],
                   "n_over_110pct": [int((df[f"BILL_AMT{i}"] > 1.1 * df["LIMIT_BAL"]).sum()) for i in chron_idx],
                   "default_rate_overlimit": [df.loc[df[f"BILL_AMT{i}"] > df["LIMIT_BAL"], TARGET].mean() for i in chron_idx],
                   "n_negative": [int((df[f"BILL_AMT{i}"] < 0).sum()) for i in chron_idx],
                   "min_bill": [df[f"BILL_AMT{i}"].min() for i in chron_idx],
                   "median_negative": [df.loc[df[f"BILL_AMT{i}"] < 0, f"BILL_AMT{i}"].median() for i in chron_idx],
                   "n_zero": [int((df[f"BILL_AMT{i}"] == 0).sum()) for i in chron_idx]})
display(ol)
dq["overlimit_any"] = df["OVERLIMIT_ANY"].astype(bool)
dq["negative_bill_any"] = df["NEG_BILL_ANY"].astype(bool)
dq["dormant_all_zero_bill"] = (df[BILL_COLS] == 0).all(1)
dq["dormant_no_bill_no_pay"] = dq["dormant_all_zero_bill"] & (df[PAMT_COLS] == 0).all(1)
dq["all_status_minus2"] = (df[PAY_COLS] == -2).all(1)
neg_mag = pd.Series(df[BILL_COLS].values[df[BILL_COLS].values < 0])
print(f"Negative bill cells: {len(neg_mag):,} | magnitude median {neg_mag.median():,.0f}, p5 {neg_mag.quantile(.05):,.0f}, min {neg_mag.min():,.0f}")
grp = pd.DataFrame({k: {"clients": int(dq[k].sum()), "share": dq[k].mean(), "default_rate": df.loc[dq[k], TARGET].mean(),
                        "rate_rest": df.loc[~dq[k], TARGET].mean()}
                    for k in ["overlimit_any", "negative_bill_any", "dormant_all_zero_bill", "dormant_no_bill_no_pay", "all_status_minus2"]}).T
lo_, hi_ = wilson_ci(grp["default_rate"] * grp["clients"], grp["clients"])
grp["ci_lo"], grp["ci_hi"] = lo_, hi_
display(grp)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(ol["month"], ol["n_overlimit"], color=C_DEF, alpha=.7, label="bill > limit")
axes[0].bar(ol["month"], ol["n_negative"], color=C_OK, alpha=.7, bottom=0, width=.4, label="bill < 0")
axes[0].set_title("Over-limit and negative statements per month"); axes[0].legend()
g2 = grp.reset_index()
axes[1].barh(g2["index"], g2["default_rate"], color=C_DEF, alpha=.7,
             xerr=[g2["default_rate"] - g2["ci_lo"], g2["ci_hi"] - g2["default_rate"]], capsize=3)
axes[1].axvline(BASE_RATE, ls="--", color="k", lw=1)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("Default rate of anomaly groups")
plt.tight_layout(); savefig("s01_balance_anomalies"); plt.show()

**Observations:**
- 3,931 clients (13.1%) exceed their limit in at least one month; they default at 30.1% vs 20.9% for the rest.
- Over-limit accounts rise every month, from 798 in Apr to 2,115 in Sep.
- Dormant accounts (all six bills zero) are 866 clients (2.9%) and default at 36.6%, far above base.
- Clients with a negative bill (credit balance) at any time default at only 16.5% (1,930 clients).
- Story use: over-limit and dormancy are two simple risk flags for the dashboard; the risky dormant group is a surprise worth one callout.

### 1.8 "Paradoxical defaulters": paid in full every month, still defaulted
Two definitions: (i) status = −1 in all six months; (ii) **money-based** — every positive prior statement in the window was fully covered
(`PAY_AMTi ≥ BILL_AMT(i+1)` whenever `BILL_AMT(i+1) > 0`, with at least one positive bill). We profile the defaulters inside these groups.

In [ ]:
dq["status_paid_full_6m"] = (df[PAY_COLS] == -1).all(1)
prevB = df[[f"BILL_AMT{i+1}" for i in IDX[:-1]]].to_numpy(float)
payA = df[[f"PAY_AMT{i}" for i in IDX[:-1]]].to_numpy(float)
covered = (prevB <= 0) | (payA >= prevB)
dq["money_paid_full_5m"] = covered.all(1) & (prevB > 0).any(1)
for k in ["status_paid_full_6m", "money_paid_full_5m"]:
    m = dq[k]
    print(f"{k}: clients {m.sum():,} ({m.mean():.1%}) | defaulters {int(df.loc[m, TARGET].sum()):,} | default rate {df.loc[m, TARGET].mean():.2%}")
dq["paradox_defaulter"] = dq["money_paid_full_5m"] & (df[TARGET] == 1)

pm = dq["money_paid_full_5m"]
prof_cols = ["LIMIT_BAL", "AGE", "AVG_BILL", "AVG_PAY", "AVG_UTIL", "N_INACTIVE", "N_PAIDFULL", "N_REVOLVE", "N_LATE", "MAX_DELAY"]
prof = pd.concat({"paradox defaulters": df.loc[pm & (df[TARGET] == 1), prof_cols].median(),
                  "full payers, no default": df.loc[pm & (df[TARGET] == 0), prof_cols].median(),
                  "all other defaulters": df.loc[~pm & (df[TARGET] == 1), prof_cols].median(),
                  "everyone": df[prof_cols].median()}, axis=1)
print("Median profile:"); display(prof)
mix = pd.concat({"paradox defaulters": df.loc[pm & (df[TARGET] == 1)],
                 "full payers, no default": df.loc[pm & (df[TARGET] == 0)]})
for c in ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "STATUS_GRP_SEP"]:
    t = pd.crosstab(mix.index.get_level_values(0), mix[c], normalize="index")
    print(f"\n{c} mix (row %):"); display((t * 100).round(1))

**Observations:**
- 1,992 clients (6.6%) were 'paid in full' (status -1) in all six months, yet 284 of them defaulted (14.3%).
- Using money columns, 5,339 clients (17.8%) paid every prior bill in full and 725 defaulted (13.6%).
- These paradox defaulters look like non-defaulting full payers: median limit NT$ 200k vs 210k, median age 36, tiny utilisation (0.8% vs 1.4%).
- The main tell is September: 21.4% of them were late in Sep vs 13.2% of non-defaulting full payers.
- Story use: a 'no one is risk-free' caveat; about 1 in 7 perfect payers still defaults, so watch the latest month.

### 1.9 Outliers in the money columns
Two lenses: Tukey IQR fences (1.5×IQR; IQR = spread of the middle 50% of values, and a point more than 1.5×IQR beyond the quartiles is flagged) and robust z via the median absolute deviation (|z| > 3.5, Iglewicz–Hoaglin).
Heavy right tails are expected in credit data, so the question is *not* "remove them?" but "do extreme clients behave differently?".

In [ ]:
MONEY = ["LIMIT_BAL"] + BILL_COLS + PAMT_COLS
out_rows = []
iqr_any = np.zeros(len(df), bool); mad_any = iqr_any.copy()
for c in MONEY:
    s = df[c].astype(float)
    q1, q3 = s.quantile([.25, .75]); iqr = q3 - q1
    f_iqr = (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)
    med = s.median(); mad = (s - med).abs().median()
    rz = 0.6745 * (s - med) / mad if mad > 0 else pd.Series(0.0, index=s.index)
    f_mad = rz.abs() > 3.5
    thr = s.quantile(.999); f_top = s > thr
    iqr_any |= f_iqr.values; mad_any |= f_mad.values
    out_rows.append({"column": c, "upper_IQR_fence": q3 + 1.5 * iqr, "n_IQR": int(f_iqr.sum()), "pct_IQR": f_iqr.mean(),
                     "rate_IQR": df.loc[f_iqr, TARGET].mean(), "rate_not_IQR": df.loc[~f_iqr, TARGET].mean(),
                     "n_MAD": int(f_mad.sum()), "rate_MAD": df.loc[f_mad, TARGET].mean(),
                     "p99.9": thr, "max": s.max(), "n_top0.1%": int(f_top.sum()), "rate_top0.1%": df.loc[f_top, TARGET].mean()})
outl = pd.DataFrame(out_rows)
display(outl)
dq["outlier_IQR_any"] = iqr_any; dq["outlier_MAD_any"] = mad_any
print(f"Clients with >=1 IQR outlier: {iqr_any.mean():.1%} (default rate {df.loc[iqr_any, TARGET].mean():.2%} vs {df.loc[~iqr_any, TARGET].mean():.2%})")
print(f"Clients with >=1 MAD outlier: {mad_any.mean():.1%} (default rate {df.loc[mad_any, TARGET].mean():.2%} vs {df.loc[~mad_any, TARGET].mean():.2%})")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))
x = np.arange(len(outl)); w = .38
ax.bar(x - w/2, outl["rate_IQR"], w, color=C_DEF, alpha=.75, label="IQR outlier")
ax.bar(x + w/2, outl["rate_not_IQR"], w, color=C_OK, alpha=.75, label="rest")
ax.axhline(BASE_RATE, ls="--", color="k", lw=1)
ax.set_xticks(x); ax.set_xticklabels(outl["column"], rotation=45, ha="right")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.legend()
ax.set_title("Default rate: IQR outliers vs the rest, per money column (high payers default less)")
savefig("s01_outlier_rates"); plt.show()
print("Top 10 clients by LIMIT_BAL / max bill / max payment:")
top = df.assign(MAX_BILL=df[BILL_COLS].max(1), MAX_PAY=df[PAMT_COLS].max(1))
display(top.nlargest(10, "MAX_PAY")[["ID", "LIMIT_BAL", "AGE", "MAX_BILL", "MAX_PAY", "N_LATE", TARGET]])

**Observations:**
- 34.2% of clients have at least one IQR outlier (a value far beyond the middle 50%) in a money column.
- Outlier clients default less, 14.9% vs 25.9%; big payments and big limits signal wealth, not risk.
- LIMIT_BAL outliers (> NT$ 525k, 167 clients) default at 11.4%, and 9 of the 10 largest payers never defaulted.
- Story use: we keep outliers and use log axes; 'big money is safe money' supports the limit and payment findings.

### 1.10 Benford's law: first-digit integrity check
Naturally generated monetary amounts spanning several orders of magnitude tend to follow Benford's law, P(d) = log10(1 + 1/d).
Large deviations can flag rounding policies, caps or fabricated values. Conformity is judged with Nigrini's mean absolute deviation
(MAD, the average gap between observed and expected digit shares; < 0.006 close, < 0.012 acceptable, < 0.015 marginal, else non-conformity) — chi-square is reported but is over-powered at this n.

In [ ]:
def first_digit(s):
    s = s[s > 0].astype(float)
    return (s / 10 ** np.floor(np.log10(s))).astype(int).clip(1, 9)

digits = np.arange(1, 10)
benford = np.log10(1 + 1 / digits)
ben_rows = []
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, ["BILL_AMT1", "PAY_AMT1", "LIMIT_BAL"]):
    fd = first_digit(df[c])
    obs = fd.value_counts(normalize=True).reindex(digits, fill_value=0).values
    cnts = fd.value_counts().reindex(digits, fill_value=0).values
    chi2, p = stats.chisquare(cnts, benford * cnts.sum())
    mad = np.abs(obs - benford).mean()
    verdict = "close" if mad < .006 else "acceptable" if mad < .012 else "marginal" if mad < .015 else "non-conformity"
    ben_rows.append({"column": c, "n_positive": int(cnts.sum()), "chi2": chi2, "p": p, "MAD": mad, "Nigrini": verdict})
    ax.bar(digits, obs, color=C_OK, alpha=.7, label="observed")
    ax.plot(digits, benford, "o-", color=C_DEF, label="Benford")
    ax.set_xticks(digits); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(f"{c}: MAD={mad:.4f} ({verdict})"); ax.legend()
plt.tight_layout(); savefig("s01_benford"); plt.show()
ben = pd.DataFrame(ben_rows); display(ben)
print("LIMIT_BAL is included as a contrast: it is a policy-set, rounded quantity and is NOT expected to follow Benford.")

**Observations:**
- BILL_AMT1 follows Benford's law closely (MAD 0.005); Benford's law is the natural first-digit pattern of real financial amounts.
- PAY_AMT1 does not conform (MAD 0.019), likely because many payments are round numbers.
- LIMIT_BAL is far off (MAD 0.041), as expected for a policy-set, rounded amount.
- Story use: background only (no sign of fabricated bills).

### 1.11 Granularity: `LIMIT_BAL` rounding and `AGE` heaping
Credit lines are set by policy, so they cluster on round numbers. Ages self-reported or keyed by hand can heap on digits 0 and 5
(Whipple's index: 100 = no heaping, >105 = measurable heaping, computed on ages 23–62).

In [ ]:
lb = df["LIMIT_BAL"]
gran = pd.Series({"distinct values": lb.nunique(), "multiple of 10k": (lb % 10_000 == 0).mean(),
                  "multiple of 50k": (lb % 50_000 == 0).mean(), "multiple of 100k": (lb % 100_000 == 0).mean(),
                  "min": lb.min(), "max": lb.max()})
print(gran.to_string())
top_lim = rate_table(df, "LIMIT_BAL").sort_values("n", ascending=False).head(15)
print("\nTop 15 LIMIT_BAL values:"); display(top_lim[["LIMIT_BAL", "n", "share_clients", "rate", "ci_lo", "ci_hi"]])
print(f"Top 15 values cover {top_lim['share_clients'].sum():.1%} of clients.")

a = df["AGE"]; w = a[(a >= 23) & (a <= 62)]
whipple = 100 * (w % 5 == 0).sum() / (len(w) / 5)
last_digit = a.value_counts(normalize=True).groupby(a.value_counts().index % 10).sum().sort_index()
print(f"\nAGE: min {a.min()}, max {a.max()}, distinct {a.nunique()} | Whipple index {whipple:.1f}")
print("Terminal digit shares (uniform = 10%):"); print(last_digit.map(pct).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
vc = a.value_counts().sort_index()
axes[0].bar(vc.index, vc.values, color=[C_DEF if v % 5 == 0 else C_NEU for v in vc.index])
axes[0].set_title("Clients by single year of age (red = ages ending 0/5)"); axes[0].set_xlabel("age")
lv = lb.value_counts().sort_index()
axes[1].scatter(lv.index / 1000, lv.values, s=10, color=C_OK)
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_title("LIMIT_BAL: frequency of each distinct value (log-log)"); axes[1].set_xlabel("limit (NT$ '000)")
plt.tight_layout(); savefig("s01_granularity"); plt.show()

**Observations:**
- LIMIT_BAL takes only 81 distinct values; 99.99% are multiples of NT$ 10k and the top 15 values cover 61.9% of clients.
- NT$ 50k is the most common limit (3,365 clients, 11.2%); the NT$ 20k and 30k limits default at about 35%.
- AGE shows no heaping: the Whipple index is 100.5 (100 = no rounding to 0/5) and every last digit holds 8-11% of clients.
- Story use: treat limit as ordered bands; age is reliable at single-year level.

### 1.12 Data-quality scorecard
One row per check: clients flagged, share of the book, default rate among flagged (with 95% CI) vs the rest, and a recommended action.
Exported as `s01_quality_scorecard.csv` (Tableau: a "data health" table / bar of lift by check).

In [ ]:
ACTION = {
    "dup_features": "keep (distinct clients with identical, mostly inactive histories)",
    "edu_undocumented": "fold into Other/Unknown (see 1.3); keep raw code for audit",
    "mar_undocumented": "fold into Other",
    "A_minus2_nonzero_bill": "keep; -2 means no new consumption, not zero balance",
    "B_minus1_underpaid": "keep; status reflects bank rule, not exact cash",
    "C_late_but_paid_full": "keep; late status can lag the cash flow",
    "D_impossible_jump": "keep; artefact of code 1 unused Apr-Aug (first delay coded 2)",
    "overlimit_any": "keep; strong risk signal",
    "negative_bill_any": "keep; credit balance (overpayment)",
    "dormant_all_zero_bill": "keep; separate 'inactive' segment",
    "dormant_no_bill_no_pay": "keep; separate 'inactive' segment",
    "all_status_minus2": "keep; separate 'inactive' segment",
    "status_paid_full_6m": "keep; transactor segment",
    "money_paid_full_5m": "keep; transactor segment",
    "paradox_defaulter": "investigate; label noise or shock events",
    "outlier_IQR_any": "keep; use log/symlog scales, robust stats",
    "outlier_MAD_any": "keep; use log/symlog scales, robust stats",
}
CHECK_NAME = {
    "dup_features": "Duplicate feature rows (excl. ID)", "edu_undocumented": "EDUCATION code 0/5/6",
    "mar_undocumented": "MARRIAGE code 0", "A_minus2_nonzero_bill": "Status -2 with non-zero bill (any month)",
    "B_minus1_underpaid": "Status -1 but prior bill not fully paid", "C_late_but_paid_full": "Status late but prior bill fully paid",
    "D_impossible_jump": "Delay jumps >1 month in one month", "overlimit_any": "Bill > credit limit (any month)",
    "negative_bill_any": "Negative bill (any month)", "dormant_all_zero_bill": "All 6 bills = 0",
    "dormant_no_bill_no_pay": "All 6 bills = 0 and no payments", "all_status_minus2": "Status -2 in all 6 months",
    "status_paid_full_6m": "Status -1 in all 6 months", "money_paid_full_5m": "Every prior bill fully paid (money-based)",
    "paradox_defaulter": "Full payer who still defaulted", "outlier_IQR_any": "Any money column IQR outlier",
    "outlier_MAD_any": "Any money column robust-z outlier",
}
sc = []
for k in dq.columns:
    if k == "paradox_defaulter":       # conditioned on the target (rate = 100% by construction) - profiled in 1.8 instead
        continue
    m = dq[k].astype(bool)
    n = int(m.sum())
    d = int(df.loc[m, TARGET].sum())
    lo, hi = wilson_ci(d, n) if n else (np.nan, np.nan)
    sc.append({"check_id": k, "check": CHECK_NAME.get(k, k), "clients_flagged": n, "pct_rows": m.mean(),
               "default_rate_flagged": d / n if n else np.nan, "ci_lo": float(lo), "ci_hi": float(hi),
               "default_rate_rest": df.loc[~m, TARGET].mean(),
               "lift": (d / n) / BASE_RATE if n else np.nan, "action": ACTION.get(k, "")})
scorecard = pd.DataFrame(sc)
export(scorecard, "s01_quality_scorecard")
display(scorecard.round(4))

In [ ]:
s2 = scorecard.dropna(subset=["default_rate_flagged"]).sort_values("default_rate_flagged")
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(s2["check"], s2["default_rate_flagged"], color=[C_DEF if r > BASE_RATE else C_OK for r in s2["default_rate_flagged"]], alpha=.75,
        xerr=[s2["default_rate_flagged"] - s2["ci_lo"], s2["ci_hi"] - s2["default_rate_flagged"]], capsize=3)
for y, (r, n) in enumerate(zip(s2["default_rate_flagged"], s2["clients_flagged"])):
    ax.text(min(r + .02, .95), y, f"n={n:,}", va="center", fontsize=8)
ax.axvline(BASE_RATE, ls="--", color="k", lw=1)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title("Data-quality scorecard: default rate among flagged clients (dashed = base rate)")
plt.tight_layout(); savefig("s01_quality_scorecard"); plt.show()

**Observations:**
- No rows are dropped; every flag is kept as a segment.
- The riskiest flags are the delay jump (44.5%, lift 2.0), dormant with no bills and no payments (37.6%, lift 1.70) and over-limit (30.1%, lift 1.36).
- The safest flags are undocumented education codes (7.5%), status -2 with a bill (12.4%) and status -2 in all six months (13.5%).
- Story use: data-quality appendix; dormant and over-limit flags feed the risk dashboard.

## 2. Univariate profiles
One variable at a time: shape, centre, spread, tails, zeros and negatives. Money columns are heavily right-skewed, so they are shown
on log / symmetric-log scales. The deep "vs default" analysis is Section 3; here default only appears as a light lead-in (2.8).

Tableau mapping: 2.1–2.4 → histogram / bar sheets with a filter on segment; 2.6 → 100% stacked bar of status by month;
2.9 → a data-dictionary style summary table.

**Section guide**

- **What it does:** Describes each variable on its own: how values are spread, where the middle sits, and how long the tails are. Default appears only as a light preview at the end (2.8).
- **Question it answers:** What does a typical client look like, and which variables need special handling (log axes, bands)?
- **Inputs / new columns:** the target `DEFAULT`, demographics, `LIMIT_BAL`, `BILL_AMT1-6`, `PAY_AMT1-6`, `PAY_1-6` and the engineered features from setup. No new columns.
- **How to read the charts:** bars = number or share of clients; log axes spread out skewed money values (a "symlog" axis also handles negative bills); stacked bars show the status mix per month, Apr → Sep.
- **Key takeaways:**
  - 6,636 of 30,000 clients defaulted: base rate 22.12%, the reference line for every later chart.
  - Women are 60.4% of clients; median age is 34 and 72.4% are 40 or younger.
  - Median limit is NT$ 140k; median bills rise 31% from Apr to Sep while median payments stay near NT$ 1,500-2,100.
  - Defaulters' bills are about the same size (0.92x), but they pay less (0.59x) on smaller limits (0.60x).
- **Where it feeds:** denominators for later rates; `s02_univariate_summary.csv` backs the Tableau data dictionary, and `s02_status_mix_by_month.csv` the status-mix sheet.

### 2.1 Target: default next month
The base rate every later section compares against. Imbalance matters for how rates and lifts are read (not for modelling here).

In [ ]:
tgt = df["DEFAULT_LABEL"].value_counts().rename_axis("class").reset_index(name="n")
tgt["share"] = tgt["n"] / tgt["n"].sum()
display(tgt)
lo, hi = wilson_ci(df[TARGET].sum(), len(df))
print(f"Base default rate {BASE_RATE:.2%} (95% CI {lo:.2%}-{hi:.2%}) | imbalance ratio non-default:default = "
      f"{(1 - BASE_RATE) / BASE_RATE:.2f}:1 | roughly 1 in {1/BASE_RATE:.1f} clients defaults")
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].bar(tgt["class"], tgt["n"], color=[PAL_TARGET[c] for c in tgt["class"]])
for xi, (n, s) in enumerate(zip(tgt["n"], tgt["share"])):
    axes[0].text(xi, n, f"{n:,}\n{s:.1%}", ha="center", va="bottom")
axes[0].set_title("Clients by outcome (Oct 2005)"); axes[0].set_ylim(0, tgt["n"].max() * 1.18)
axes[1].pie(tgt["n"], labels=tgt["class"], colors=[PAL_TARGET[c] for c in tgt["class"]], autopct="%1.1f%%",
            startangle=90, wedgeprops={"width": .45})
axes[1].set_title("Share of clients")
plt.tight_layout(); savefig("s02_target"); plt.show()

**Observations:**
- 6,636 of 30,000 clients defaulted: a base rate of 22.12% (95% CI 21.65-22.59%).
- That is about 1 in 4.5 clients, an imbalance of 3.52 non-defaulters per defaulter.
- Story use: 22.1% is the reference line on every rate chart.

### 2.2 Demographics: sex, education, marital status
Counts and shares, with the raw codes shown next to the folded labels so the effect of the fold (Section 1.3–1.4) is visible.

In [ ]:
def share_table(col, order=None):
    t = df[col].value_counts().rename_axis("level").reset_index(name="n")
    if order is not None:
        t = t.set_index("level").reindex([o for o in order if o in set(t["level"])]).reset_index()
    t["share"] = t["n"] / t["n"].sum()
    t.insert(0, "variable", col)
    return t
demo_tabs = pd.concat([share_table("SEX_LABEL", ["Male", "Female"]), share_table("EDU_LABEL", EDU_ORDER),
                       share_table("EDUCATION", sorted(df["EDUCATION"].unique())),
                       share_table("MAR_LABEL", MAR_ORDER), share_table("MARRIAGE", sorted(df["MARRIAGE"].unique()))])
demo_tabs["level"] = demo_tabs["level"].astype(str)
display(demo_tabs.reset_index(drop=True))
export(demo_tabs, "s02_demographic_shares")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
specs = [("SEX_LABEL", ["Male", "Female"]), ("EDU_LABEL", EDU_ORDER), ("MAR_LABEL", MAR_ORDER),
         ("EDUCATION", sorted(df["EDUCATION"].unique())), ("MARRIAGE", sorted(df["MARRIAGE"].unique())), ("AGE_BAND", AGE_LABELS)]
for ax, (col, order) in zip(axes.flat, specs):
    t = df[col].value_counts().reindex(order).fillna(0)
    bars = ax.bar([str(o) for o in order], t.values, color=C_OK, alpha=.75)
    for b, v in zip(bars, t.values):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v/len(df):.1%}", ha="center", va="bottom", fontsize=8)
    ax.set_title(col + (" (raw codes)" if col in ("EDUCATION", "MARRIAGE") else ""))
    ax.tick_params(axis="x", rotation=20)
    ax.set_ylim(0, t.max() * 1.15)
plt.tight_layout(); savefig("s02_demographics"); plt.show()

**Observations:**
- Women are 60.4% of clients (18,112) and men 39.6%.
- University (46.8%) and graduate school (35.3%) dominate; high school is 16.4% and Other/Unknown just 1.6%.
- Singles (53.2%) slightly outnumber married clients (45.5%); Other is 1.3%.
- Story use: background only; gives the denominators behind later demographic rates.

### 2.3 Age
Histogram with KDE (kernel density estimate, a smoothed outline of the histogram; single-year bins), band counts, and summary statistics. Age is bounded at 21 (card eligibility) and right-skewed.

In [ ]:
a = df["AGE"]
print(a.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).to_string())
print(f"skew {a.skew():.3f} | excess kurtosis {a.kurt():.3f} | mode {a.mode().iat[0]}")
band = df["AGE_BAND"].value_counts().reindex(AGE_LABELS).rename_axis("AGE_BAND").reset_index(name="n")
band["share"] = band["n"] / len(df); band["cum_share"] = band["share"].cumsum()
display(band)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
sns.histplot(a, bins=np.arange(20.5, 80.5, 1), kde=True, color=C_OK, ax=axes[0])
for q, ls in [(a.median(), "-"), (a.mean(), "--")]:
    axes[0].axvline(q, color=C_DEF, ls=ls, lw=1)
axes[0].set_title(f"AGE distribution (median {a.median():.0f}, mean {a.mean():.1f})")
sns.histplot(data=df, x="AGE", hue="SEX_LABEL", element="step", stat="density", common_norm=False, bins=30, ax=axes[1])
axes[1].set_title("AGE density by sex")
plt.tight_layout(); savefig("s02_age"); plt.show()

**Observations:**
- Age runs from 21 to 79 with median 34 and mean 35.5 (right skew 0.73).
- 26-30 is the largest band (23.8%) and 72.4% of clients are 40 or younger.
- Only 272 clients (0.9%) are 61+, so rates there are noisy.
- Story use: background only; a young book.

### 2.4 Credit limit (`LIMIT_BAL`)
Four views: linear histogram, log-scale histogram, ECDF (empirical cumulative distribution: share of clients at or below each value) with key quantiles, and the most common values (policy points).

In [ ]:
lb = df["LIMIT_BAL"]
qs = [.01, .05, .10, .25, .50, .75, .90, .95, .99]
lim_q = pd.DataFrame({"quantile": ["min"] + [f"p{int(q*100)}" for q in qs] + ["max", "mean", "std"],
                      "LIMIT_BAL": [lb.min()] + [lb.quantile(q) for q in qs] + [lb.max(), lb.mean(), lb.std()]})
display(lim_q.T)
print(f"skew {lb.skew():.3f} | kurtosis {lb.kurt():.3f} | skew of log10 {np.log10(lb).skew():.3f} | distinct {lb.nunique()}")
print("Most common values:"); print(lb.value_counts().head(10).to_string())
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(lb / 1000, bins=60, color=C_OK, alpha=.8); axes[0].set_title("LIMIT_BAL (NT$ '000)")
axes[1].hist(np.log10(lb), bins=40, color=C_OK, alpha=.8); axes[1].set_title("log10(LIMIT_BAL)")
axes[1].set_xticks([4, 4.5, 5, 5.5, 6]); axes[1].set_xticklabels(["10k", "32k", "100k", "316k", "1M"])
xs = np.sort(lb.values) / 1000
axes[2].plot(xs, np.arange(1, len(xs) + 1) / len(xs), color=C_OK, lw=2)
for q in [.25, .5, .75, .9]:
    axes[2].axvline(lb.quantile(q) / 1000, ls=":", color=C_NEU); axes[2].text(lb.quantile(q) / 1000, q, f" p{int(q*100)}", fontsize=8)
axes[2].set_xscale("log"); axes[2].set_title("ECDF of LIMIT_BAL (log x)"); axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1))
plt.tight_layout(); savefig("s02_limit_bal"); plt.show()
lband = df["LIMIT_BAND"].value_counts().reindex(LIMIT_LABELS).rename_axis("LIMIT_BAND").reset_index(name="n")
lband["share"] = lband["n"] / len(df); display(lband)

**Observations:**
- The median limit is NT$ 140k and the mean NT$ 167k; the middle 50% sit between 50k and 240k.
- A quarter of clients (25.6%) hold limits of NT$ 50k or less; only 206 (0.7%) exceed 500k.
- The log of the limit is close to symmetric (skew -0.51), so log axes suit it.
- Story use: the limit bands used across the dashboard.

### 2.5 Monthly bill and payment amounts
Small multiples (chronological Apr→Sep). Bills can be negative, so they use a **symmetric-log** axis (log-like, but handles zero and negatives); payments are ≥ 0 and use **log1p** (log of 1 + value, so zeros are allowed).
Then quantile, shape (skew / kurtosis) and zero / negative share tables.

In [ ]:
chron_idx = IDX[::-1]
fig, axes = plt.subplots(2, 6, figsize=(20, 6.5), sharey="row")
for k, i in enumerate(chron_idx):
    b = df[f"BILL_AMT{i}"].values.astype(float)
    sb = np.sign(b) * np.log10(1 + np.abs(b))
    axes[0, k].hist(sb, bins=60, color=C_OK, alpha=.8)
    axes[0, k].set_title(f"BILL {MONTH[i]}"); axes[0, k].set_xlabel("sign·log10(1+|x|)")
    p = df[f"PAY_AMT{i}"].values.astype(float)
    axes[1, k].hist(np.log10(1 + p), bins=60, color=C_ACC, alpha=.8)
    axes[1, k].set_title(f"PAY_AMT {MONTH[i]}"); axes[1, k].set_xlabel("log10(1+x)")
fig.suptitle("Bill statements (top, symlog) and payments (bottom, log1p), Apr -> Sep. Spikes at 0 = no bill / no payment", fontweight="bold")
plt.tight_layout(); savefig("s02_bill_pay_small_multiples"); plt.show()

In [ ]:
money_cols = [f"BILL_AMT{i}" for i in chron_idx] + [f"PAY_AMT{i}" for i in chron_idx]
qlist = [.01, .05, .25, .5, .75, .95, .99]
mq = pd.DataFrame({c: [df[c].min()] + [df[c].quantile(q) for q in qlist] + [df[c].max()] for c in money_cols},
                  index=["min"] + [f"p{int(q*100)}" for q in qlist] + ["max"]).T
mq.insert(0, "month", [MONTH[int(c[-1])] for c in money_cols])
print("Quantiles (NT$):"); display(mq.round(0))
shape = pd.DataFrame({"month": [MONTH[int(c[-1])] for c in money_cols],
                      "mean": [df[c].mean() for c in money_cols], "median": [df[c].median() for c in money_cols],
                      "skew": [df[c].skew() for c in money_cols], "kurtosis": [df[c].kurt() for c in money_cols],
                      "skew_symlog": [pd.Series(np.sign(df[c]) * np.log1p(np.abs(df[c]))).skew() for c in money_cols],
                      "pct_zero": [(df[c] == 0).mean() for c in money_cols], "pct_negative": [(df[c] < 0).mean() for c in money_cols]},
                     index=money_cols)
print("Shape, zeros and negatives:"); display(shape)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for blk, col_, lab in [("BILL_AMT", C_OK, "bill"), ("PAY_AMT", C_ACC, "payment")]:
    sub = shape.loc[[f"{blk}{i}" for i in chron_idx]]
    axes[0].plot(MONTH_ORDER, sub["median"], marker="o", color=col_, label=f"median {lab}")
    axes[0].plot(MONTH_ORDER, sub["mean"], marker="o", ls="--", color=col_, label=f"mean {lab}")
    axes[1].plot(MONTH_ORDER, sub["pct_zero"], marker="o", color=col_, label=f"zero {lab}")
axes[1].plot(MONTH_ORDER, shape.loc[[f"BILL_AMT{i}" for i in chron_idx], "pct_negative"], marker="s", color=C_DEF, label="negative bill")
axes[0].set_title("Mean vs median per month (NT$): mean >> median = heavy tail"); axes[0].legend(fontsize=8)
axes[1].set_title("Share of zero / negative values per month"); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].legend(fontsize=8)
axes[2].bar(np.arange(12), shape.loc[money_cols, "skew"], color=[C_OK] * 6 + [C_ACC] * 6)
axes[2].set_xticks(np.arange(12)); axes[2].set_xticklabels([f"{c[:4]}_{MONTH[int(c[-1])]}" for c in money_cols], rotation=60, fontsize=8)
axes[2].set_title("Skewness by column (raw scale)")
plt.tight_layout(); savefig("s02_bill_pay_trends"); plt.show()

**Observations:**
- Median bills grow every month, from NT$ 17,071 in Apr to 22,382 in Sep (+31%).
- Median payments are far smaller, NT$ 1,500 to 2,100, and very skewed (skew 10-30).
- Zero-payment months fall from 23.9% (Apr) to 17.5% (Sep); zero bills fall from 13.4% to 6.7%; about 2% of bills are negative.
- Story use: rising balances set the scene; payments are a small share of bills.

### 2.6 Repayment status by month
100% stacked bars of `STATUS_GRP_i` in chronological order (Apr→Sep) show how the book's repayment mix evolves before the default month.
The code-level table keeps every raw value for reference.

In [ ]:
sg = pd.DataFrame({MONTH[i]: df[f"STATUS_GRP_{i}"].value_counts(normalize=True).reindex(STATUS_GROUP_ORDER).fillna(0)
                   for i in chron_idx})
display((sg * 100).round(2))
colors = ["#bdbdbd", C_OK, "#9ecae1", "#fdae6b", "#f16913", "#a50f15"]
fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(6)
for g, c in zip(STATUS_GROUP_ORDER, colors):
    v = sg.loc[g, MONTH_ORDER].values
    ax.bar(MONTH_ORDER, v, bottom=bottom, color=c, label=g, edgecolor="white")
    for xi, (vv, bb) in enumerate(zip(v, bottom)):
        if vv > .035:
            ax.text(xi, bb + vv / 2, f"{vv:.0%}", ha="center", va="center", fontsize=8)
    bottom += v
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_ylim(0, 1)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); ax.set_title("Repayment status mix by month (chronological)")
plt.tight_layout(); savefig("s02_status_mix"); plt.show()
status_long = (df.melt(value_vars=PAY_COLS, var_name="col", value_name="status")
                 .assign(month=lambda d: d["col"].str[-1].astype(int).map(MONTH)))
code_tab = pd.crosstab(status_long["status"], status_long["month"])[MONTH_ORDER]
code_tab.index = [STATUS_LABEL[s] for s in code_tab.index]
print("Code-level counts:"); display(code_tab)
st_export = (sg.reset_index().rename(columns={"index": "status_group"})
               .melt(id_vars="status_group", var_name="month", value_name="share"))
export(st_export, "s02_status_mix_by_month")

**Observations:**
- Revolving (minimum paid) is the largest group every month, 49-56% of clients.
- Inactive clients shrink from 16.3% in Apr to 9.2% in Sep.
- Clients late in Sep total 22.7% vs 10.3% in Apr, but part of that jump is the code-1 artefact from 1.5.
- Story use: the status mix chart, with a caveat on the September code change.

### 2.7 Engineered behavioural features (from setup 0.3)
Grid of histograms of every `FEATURE_COLS` entry. For display, continuous features are clipped to [p1, p99] (the clip is noted in
each title); counts features are shown as bars. Summary statistics follow.

In [ ]:
COUNT_FEATS = [c for c in FEATURE_COLS if df[c].nunique() <= 15]
ncol = 6; nrow = int(np.ceil(len(FEATURE_COLS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(22, 3.2 * nrow))
for ax, c in zip(axes.flat, FEATURE_COLS):
    s = df[c].dropna()
    if c in COUNT_FEATS:
        vc = s.value_counts().sort_index()
        ax.bar(vc.index.astype(str), vc.values / len(s), color=C_ACC, alpha=.8)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title(c, fontsize=10)
    else:
        lo_, hi_ = s.quantile([.01, .99])
        ax.hist(s.clip(lo_, hi_), bins=50, color=C_OK, alpha=.8)
        ax.axvline(s.median(), color=C_DEF, lw=1)
        ax.set_title(f"{c}\n(clipped p1-p99, NaN {df[c].isna().mean():.0%})", fontsize=9)
    ax.tick_params(labelsize=7)
for ax in list(axes.flat)[len(FEATURE_COLS):]:
    ax.axis("off")
fig.suptitle("Engineered features: distributions (red line = median)", fontweight="bold")
plt.tight_layout(); savefig("s02_feature_grid"); plt.show()
fs = df[FEATURE_COLS].describe(percentiles=[.01, .25, .5, .75, .99]).T
fs["skew"] = df[FEATURE_COLS].skew(); fs["pct_nan"] = df[FEATURE_COLS].isna().mean()
display(fs)

**Observations:**
- Median average utilisation (bill / limit) is 28.5%; a quarter of clients use over 68.8%, and the maximum is 536%.
- Most clients are never late: median N_LATE is 0 and 75% have at most one late month.
- Clients revolve in 3.2 of 6 months on average and repay a median 9.6% of their six-month bills.
- 13.1% go over limit in at least one month; AVG_PAY_RATIO is missing for 4.7% (no bills to pay).
- Story use: background only; defines the features used later.

### 2.8 Lead-in: money variables split by outcome
Box (log1p scale) and violin views of the main money/usage variables by `DEFAULT_LABEL`. Light preview only; Section 3 carries the
formal tests, binned rates and WoE/IV (Weight of Evidence / Information Value, credit-scoring measures of how well a variable separates defaulters).

In [ ]:
LEAD = ["LIMIT_BAL", "AVG_BILL", "AVG_PAY", "TOTAL_PAY", "AVG_UTIL", "PAY_TO_BILL_6M"]
fig, axes = plt.subplots(2, len(LEAD), figsize=(22, 8))
order = ["No default", "Default"]
for k, c in enumerate(LEAD):
    s = df[c]
    if c in ("AVG_UTIL", "PAY_TO_BILL_6M"):
        v = s.clip(s.quantile(.01), s.quantile(.99)); lab = f"{c} (clipped p1-p99)"
    else:
        v = np.sign(s) * np.log10(1 + np.abs(s)); lab = f"{c} (sign·log10(1+|x|))"
    tmp = pd.DataFrame({"v": v, "y": df["DEFAULT_LABEL"]}).dropna()
    sns.boxplot(data=tmp, x="y", y="v", order=order, palette=PAL_TARGET, ax=axes[0, k], showfliers=False)
    sns.violinplot(data=tmp, x="y", y="v", order=order, palette=PAL_TARGET, ax=axes[1, k], cut=0, inner="quartile")
    axes[0, k].set_title(lab, fontsize=9); axes[0, k].set_xlabel(""); axes[1, k].set_xlabel(""); axes[0, k].set_ylabel(""); axes[1, k].set_ylabel("")
plt.tight_layout(); savefig("s02_money_by_outcome"); plt.show()
lead = df.groupby("DEFAULT_LABEL")[LEAD].median().T
lead["ratio_def_to_nodef"] = lead["Default"] / lead["No default"]
print("Medians by outcome:"); display(lead)

**Observations:**
- Defaulters have lower limits: median NT$ 90k vs 150k (0.60x).
- They pay less: median monthly payment NT$ 1,612 vs 2,754 (0.59x), and repay 6.0% of bills vs 12.1%.
- Their bills are almost the same size: median NT$ 19,781 vs 21,443 (0.92x), so their utilisation is double (46% vs 24%).
- Story use: a headline, 'defaulters do not owe more, they pay less on smaller limits'.

### 2.9 Univariate summary table (all numeric columns)
One row per numeric column of `df` (raw + engineered): mean, median, std, IQR, skew, kurtosis, % zero, % negative, % missing, n unique.
Exported as `s02_univariate_summary.csv` (Tableau: data dictionary / profile sheet).

In [ ]:
num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != "ID"]
def _block(c):
    if c in RAW_NUM or c in PAY_COLS or c in ("SEX", "EDUCATION", "MARRIAGE", TARGET): return "raw"
    return "engineered"
rows = []
for c in num_cols:
    s = df[c]
    q1, q3 = s.quantile([.25, .75])
    rows.append({"column": c, "block": _block(c), "n_unique": int(s.nunique()), "pct_missing": s.isna().mean(),
                 "mean": s.mean(), "median": s.median(), "std": s.std(), "min": s.min(), "max": s.max(),
                 "IQR": q3 - q1, "skew": s.skew(), "kurtosis": s.kurt(),
                 "pct_zero": (s == 0).mean(), "pct_negative": (s < 0).mean()})
uni = pd.DataFrame(rows)
export(uni, "s02_univariate_summary")
with pd.option_context("display.max_rows", 200):
    display(uni.round(4))
print("Most skewed columns:"); print(uni.reindex(uni["skew"].abs().sort_values(ascending=False).index)[["column", "skew", "kurtosis"]].head(8).to_string(index=False))

**Observations:**
- All 64 numeric columns are complete except the ratio features (no bill means no ratio).
- PAY_TO_BILL_6M is the most skewed column (skew 107), followed by PAY_AMT2 (30) and the spend columns (15-16).
- Story use: background only; the exported table backs the Tableau data dictionary.

## 3. Bivariate: every feature vs default
Every raw and engineered variable is set against the target, one at a time. The section moves from **rates by level** (the shape of the relationship), to **separation statistics** (how far apart defaulters and non-defaulters sit), to **WoE/IV** (credit-scoring view), to a single **driver leaderboard** that ranks all variables on six association metrics.

Conventions: red line = default rate with 95% Wilson CI; grey bars = client count; dashed line = portfolio base rate (≈22.1%).
**Tableau mapping:** every rate-by-level chart here is one bar+line dual-axis sheet; the tidy table `s03_rate_by_level.csv` drives all of them with a `variable` filter.

New `df` columns: **none** (section works on a local copy `d3`).

**Section guide**

- **What it does:** Puts every variable next to default, one at a time. It shows default rate by level, then measures how far apart defaulters and non-defaulters sit, and finally ranks all variables in one leaderboard.
- **Question it answers:** Which single variables are linked to default, and how strongly?
- **Inputs / new columns:** all raw columns plus setup features (`N_LATE`, `MAX_DELAY`, `UTIL_i`, `PAY_RATIO_i`, bands). No new `df` columns (works on a copy `d3`).
- **How to read the charts:** red line = default rate with 95% CI band; grey bars = number of clients; dashed line = 22.1% base rate. WoE bars: red/negative = riskier than average.
- **Terms:** IV (Information Value) scores how well a variable's bins split defaulters from others (above 0.5 = very strong). AUC = chance a random defaulter scores above a random non-defaulter (0.5 = coin flip). KS = biggest gap between the two groups' cumulative curves.
- **Key takeaways:**
  - Delinquency features lead: PAY_1 IV 0.877, N_LATE AUC 0.726; the top 11 drivers are all repayment-history variables.
  - Sep 2 months late = 69.1% default vs 12.8% for revolvers.
  - Limit is the strongest profile variable: 35.9% in the lowest decile vs 11.9% in the top.
  - Bill size is almost flat (19.3-26.8% across bins); what you pay matters, not what you owe.
  - Men default 3.4 pp more than women, and the gap holds in most strata.
- **Where it feeds:** the "who defaults" and driver-ranking charts; `s03_rate_by_level.csv` drives one Tableau bar+line sheet with a `variable` filter.

In [ ]:
from scipy.stats import spearmanr, ks_2samp, mannwhitneyu, pointbiserialr
from sklearn.metrics import roc_auc_score
from sklearn.feature_selection import mutual_info_classif
import statsmodels.api as sm

d3 = df.copy()                                    # local working copy, df untouched
y3 = d3[TARGET].to_numpy()

def slog(x):
    # signed log1p: keeps sign of negative bills/spend, compresses the long right tail
    x = np.asarray(x, float)
    return np.sign(x) * np.log1p(np.abs(x))

def bin_series(s, q=10):
    # few-level integer -> own levels (ordered categorical); else quantile bins; NaN -> 'Missing'
    if pd.api.types.is_numeric_dtype(s) and s.nunique() <= 12:
        lv = sorted(s.dropna().unique())
        out = pd.Series(pd.Categorical(s, categories=lv, ordered=True), index=s.index)
    elif pd.api.types.is_numeric_dtype(s):
        out = pd.qcut(s, q, duplicates="drop")
    else:
        out = s.astype("category") if not isinstance(s.dtype, pd.CategoricalDtype) else s
    if out.isna().any():
        out = out.cat.add_categories("Missing").fillna("Missing")
    return out

RATE_ROWS = []                                     # collects tidy rate tables for Tableau
def keep(t, var):
    t = t.rename(columns={var: "level"}).copy()
    t.insert(0, "variable", var)
    t["level"] = t["level"].astype(str)
    RATE_ROWS.append(t)
    return t

def compact(t, col):
    return t[[col, "n", "rate", "ci_lo", "ci_hi", "lift"]].round(4).to_string(index=False)
print("section 3 helpers ready")

### 3.1 Demographics and credit line
Default rate by sex, education (labelled and raw codes), marital status, age band and limit band. Raw `EDUCATION` codes 0/4/5/6 are shown separately to see whether folding them into *Other/Unknown* hides a difference.

In [ ]:
DEMO_PLOTS = [("SEX_LABEL", None, 0), ("EDU_LABEL", EDU_ORDER, 15), ("EDUCATION", None, 0),
              ("MAR_LABEL", MAR_ORDER, 0), ("AGE_BAND", AGE_LABELS, 0), ("LIMIT_BAND", LIMIT_LABELS, 30)]
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, (c, order, rot) in zip(axes.ravel(), DEMO_PLOTS):
    t = plot_rate(d3, c, ax=ax, order=order, rotate=rot)
    keep(t, c)
fig.suptitle("3.1 Default rate by demographic / credit-line variable", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s03_demographics_rate"); plt.show()
for c, *_ in DEMO_PLOTS:
    print(f"--- {c}\n{compact(rate_table(d3, c), c)}\n")

**Observations:**
- Credit limit gives the widest demographic spread: 31.8% default at NT$ 50k or less vs 11.2% above 500k.
- Men default more than women, 24.2% vs 20.8%.
- High school (25.2%) > university (23.7%) > graduate school (19.2%); Other/Unknown is only 7.1%.
- Age is U-shaped: 26.7% at 21-25, lowest 19.4% at 31-35, back to 26.8% at 61+; married clients default more than singles (23.5% vs 20.9%).
- Story use: the 'who defaults' profile chart, with limit as the main demographic lever.

### 3.2 Single-year AGE: rate line with CI and LOWESS smooth
Bands can hide the shape. Here every single year of age gets its own rate (ages with n < 30 greyed out), with a LOWESS smooth (a locally weighted trend line) fitted on the 30,000 individual 0/1 outcomes. Tests the commonly reported U-shape (young and old riskier).

In [ ]:
ta = rate_table(d3, "AGE")
ok = ta["n"] >= 30
lw = sm.nonparametric.lowess(d3[TARGET].astype(float), d3["AGE"].astype(float), frac=0.3, it=0, delta=1.0)
lw = pd.DataFrame(lw, columns=["AGE", "smooth"]).drop_duplicates("AGE")
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(ta["AGE"], ta["n"], color=C_NEU, alpha=.35); ax.set_ylabel("clients"); ax.set_xlabel("age (years)")
ax2 = ax.twinx()
ax2.fill_between(ta.loc[ok, "AGE"], ta.loc[ok, "ci_lo"], ta.loc[ok, "ci_hi"], color=C_DEF, alpha=.15, label="95% CI")
ax2.plot(ta.loc[ok, "AGE"], ta.loc[ok, "rate"], "o-", color=C_DEF, ms=3, lw=1, label="rate (n>=30)")
ax2.scatter(ta.loc[~ok, "AGE"], ta.loc[~ok, "rate"], color=C_NEU, s=12, label="rate (n<30)")
ax2.plot(lw["AGE"], lw["smooth"], color="k", lw=2.5, label="LOWESS (frac=0.3)")
ax2.axhline(BASE_RATE, ls="--", color="k", lw=1)
ax2.set_ylim(0, 0.5); ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax2.grid(False); ax2.set_ylabel("default rate")
ax2.legend(loc="upper center", ncol=4, fontsize=9)
ax.set_title("3.2 Default rate by single year of age (LOWESS on individual outcomes)")
savefig("s03_age_single_year"); plt.show()
amin = lw.loc[lw["smooth"].idxmin()]
print(f"LOWESS minimum at age {amin.AGE:.0f}: {amin.smooth:.1%} | at 22: {lw.set_index('AGE').smooth.get(22.0, np.nan):.1%} "
      f"| at 60: {lw.set_index('AGE').smooth.get(60.0, np.nan):.1%}")
keep(ta, "AGE")
print(ta.loc[ok, ["AGE", "n", "rate"]].assign(rate=lambda t: t["rate"].round(3)).set_index("AGE").T.astype(object).to_string())

**Observations:**
- The smoothed (LOWESS) curve bottoms out at age 31 with 19.3% default.
- Risk is higher at both ends: 29.0% at age 22 and 26.3% at 60.
- Story use: the U-shaped age curve, young and older clients both riskier.

### 3.3 Finer LIMIT_BAL view: deciles
The business bands are uneven in size; deciles give ten equal-sized groups and show where along the credit line the risk gradient is steepest.

In [ ]:
d3["LIMIT_DECILE"] = pd.qcut(d3["LIMIT_BAL"], 10, duplicates="drop")
d3["LIMIT_DECILE"] = d3["LIMIT_DECILE"].cat.rename_categories(
    [f"{int(iv.left/1000)}-{int(iv.right/1000)}k" for iv in d3["LIMIT_DECILE"].cat.categories])
t = plot_rate(d3, "LIMIT_DECILE", rotate=30, title="3.3 Default rate by LIMIT_BAL decile (NT$ thousands)")
savefig("s03_limit_deciles"); plt.show()
keep(t, "LIMIT_DECILE")
print(compact(t, "LIMIT_DECILE"))
print(f"\nratio lowest-decile rate / highest-decile rate = {t['rate'].iloc[0] / t['rate'].iloc[-1]:.2f}x")

**Observations:**
- Default falls from 35.9% in the lowest limit decile (NT$ 9-30k) to 11.9% in the top decile (360k-1M), a 3.0x ratio.
- The fall is steady, with one small bump at 50-70k (28.5% vs 27.2% just below).
- Story use: the limit decile chart; small limits carry triple the risk.

### 3.4 Repayment status code by month (6 small multiples)
Each panel is one month's status code (−2 … 8). Levels with fewer than 20 clients are suppressed to keep the y-axis readable. Watch the step at code 0 → 1 and 1 → 2, and whether −2/−1 (inactive / paid in full) actually differ from 0 (revolving).

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9), sharey=False)
pay_rates = []
for ax, i in zip(axes.ravel(), IDX):
    c = f"PAY_{i}"
    t = plot_rate(d3, c, ax=ax, min_n=20, title=f"{c} status ({MONTH[i]})")
    keep(rate_table(d3, c), c)
    pay_rates.append(t.assign(month=MONTH[i]).rename(columns={c: "code"})[["month", "code", "n", "rate"]])
fig.suptitle("3.4 Default rate by repayment status code, per month (levels n>=20)", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s03_pay_status_small_multiples"); plt.show()
pr = pd.concat(pay_rates).pivot(index="code", columns="month", values="rate")[MONTH_ORDER[::-1]]
print("default rate by status code x month (Sep is most recent)"); print(pr.round(3).to_string())

**Observations:**
- In Sep, being 2 months late means 69.1% default and 3 months late 75.8%, vs 12.8% for revolving (code 0).
- The same codes are weaker in older months: code 2 in Apr means 50.7%.
- On-time codes are surprisingly ordered: revolving (12.8%) is safer than paid in full (16.8%) in Sep.
- Story use: the September status chart, the single strongest signal.

### 3.5 Behavioural bands and counts
Default rate across the engineered behavioural variables from Section 0: latest status group, utilisation band, number of late months, payment-to-bill band, worst delay, recent late streak, months inactive / paid-in-full / revolving / zero-payment, and the over-limit and negative-bill flags.

In [ ]:
BEH_PLOTS = [("STATUS_GRP_SEP", STATUS_GROUP_ORDER, 20), ("UTIL_BAND", None, 30), ("N_LATE_BAND", None, 0),
             ("PAYRATIO_BAND", None, 30), ("MAX_DELAY", None, 0), ("LATE_STREAK_RECENT", None, 0),
             ("N_INACTIVE", None, 0), ("N_PAIDFULL", None, 0), ("N_REVOLVE", None, 0),
             ("N_ZERO_PAY", None, 0), ("OVERLIMIT_ANY", None, 0), ("NEG_BILL_ANY", None, 0)]
fig, axes = plt.subplots(4, 3, figsize=(18, 18))
for ax, (c, order, rot) in zip(axes.ravel(), BEH_PLOTS):
    t = plot_rate(d3, c, ax=ax, order=order, rotate=rot, min_n=20)
    keep(rate_table(d3, c), c)
fig.suptitle("3.5 Default rate by behavioural variable (levels n>=20)", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s03_behaviour_rate_grid"); plt.show()
summ = []
for c, *_ in BEH_PLOTS:
    t = rate_table(d3, c, min_n=20)
    summ.append({"variable": c, "levels": len(t), "min_rate": t["rate"].min(), "max_rate": t["rate"].max(),
                 "spread_pp": 100 * (t["rate"].max() - t["rate"].min()),
                 "riskiest_level": str(t.loc[t["rate"].idxmax(), c]), "safest_level": str(t.loc[t["rate"].idxmin(), c])})
print(pd.DataFrame(summ).sort_values("spread_pp", ascending=False).round(3).to_string(index=False))

**Observations:**
- Worst-ever delay (MAX_DELAY) spans 72.8 pp, from 10.8% (worst status 0, revolving) to 83.6% (7 months late).
- September status ranges from 12.8% (revolving) to 71.9% (late 3m+); late-month count from 11.7% (none) to 64.1% (4-6).
- Paying nothing (share of bills repaid over 5 statements, `PAYRATIO_BAND` = 0) means 39.7% default vs 14.5% for paying over 100% of the bill (number from the pre-fix run; refresh after the next Kaggle run).
- Utilisation spans only 18.9 pp and is safest at 10-30%; over-limit adds 9.2 pp (30.1% vs 20.9%).
- Story use: the behavioural driver chart; delinquency history beats every other band.

Export: one tidy long table of every rate-by-level view above (`variable`, `level`, `n`, `defaults`, `rate`, CI, shares, lift) — a single Tableau data source with a `variable` parameter/filter reproduces all of 3.1–3.5.

In [ ]:
rate_by_level = pd.concat(RATE_ROWS, ignore_index=True)
export(rate_by_level, "s03_rate_by_level")
rate_by_level.groupby("variable").size().rename("levels").to_frame().T

### 3.6 Continuous variables: default rate by quantile bin
For money amounts and continuous engineered features, the default rate is plotted against the **median value of each vigintile** (20 equal-count bins; fewer when ties collapse bins). Shaded = 95% CI. Money axes are on a signed-log scale where skewed. This is the honest, non-parametric shape of each relationship.

In [ ]:
CONT = ["LIMIT_BAL", "AGE", "BILL_AMT1", "BILL_AMT6", "PAY_AMT1", "PAY_AMT6",
        "AVG_BILL", "AVG_PAY", "TOTAL_PAY", "AVG_UTIL", "MAX_UTIL", "UTIL_TREND",
        "AVG_PAY_RATIO", "PAY_TO_BILL_6M", "AVG_SPEND", "BILL_TREND", "PAY_TREND", "DELAY_TREND",
        "UTIL_1", "PAY_RATIO_1"]
LOGX = {"LIMIT_BAL", "BILL_AMT1", "BILL_AMT6", "PAY_AMT1", "PAY_AMT6", "AVG_BILL", "AVG_PAY", "TOTAL_PAY",
        "AVG_SPEND", "BILL_TREND", "PAY_TREND"}
def qcurve(col, q=20):
    s = d3[col]
    b = pd.qcut(s, q, duplicates="drop")
    g = d3.groupby(b, observed=True).agg(x=(col, "median"), n=(TARGET, "size"), k=(TARGET, "sum")).reset_index(drop=True)
    g["rate"] = g["k"] / g["n"]; g["lo"], g["hi"] = wilson_ci(g["k"], g["n"])
    return g
fig, axes = plt.subplots(5, 4, figsize=(18, 20))
QCURVES = {}
for ax, c in zip(axes.ravel(), CONT):
    g = qcurve(c); QCURVES[c] = g
    x = slog(g["x"]) if c in LOGX else g["x"]
    ax.fill_between(x, g["lo"], g["hi"], color=C_DEF, alpha=.15)
    ax.plot(x, g["rate"], "o-", color=C_DEF, ms=4)
    ax.axhline(BASE_RATE, ls="--", color="k", lw=1)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(f"{c} ({len(g)} bins)", fontsize=10)
    ax.set_xlabel("signed log1p(median of bin)" if c in LOGX else "median of bin", fontsize=8)
fig.suptitle("3.6 Default rate by quantile bin (vigintiles) — continuous variables", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s03_quantile_curves"); plt.show()
print(pd.DataFrame({c: {"bins": len(g), "rate_first_bin": g["rate"].iloc[0], "rate_last_bin": g["rate"].iloc[-1],
                        "min_rate": g["rate"].min(), "max_rate": g["rate"].max()} for c, g in QCURVES.items()}).T.round(3).to_string())

**Observations:**
- Payment amounts have a strong gradient: AVG_PAY falls from 37.7% in the lowest bin to 9.1% in the highest.
- Payment ratio for Sep (PAY_RATIO_1) drops from 49.1% to 12.1%; limit from 36.2% to 12.0%.
- Bill size is almost flat: BILL_AMT1 stays within 19.3-26.8% across 20 bins.
- Story use: supports 'what you pay matters, what you owe does not'.

### 3.7 Distribution overlays: defaulters vs non-defaulters
KDE (densities normalised within each class) and ECDF for twelve key continuous variables. Skewed money variables are shown as signed log1p; ratios are clipped at the 99th percentile for display only. The vertical gap between the two ECDFs at its widest is exactly the KS statistic of 3.8.

In [ ]:
OVL = ["LIMIT_BAL", "AGE", "BILL_AMT1", "PAY_AMT1", "TOTAL_PAY", "AVG_UTIL",
       "MAX_UTIL", "AVG_PAY_RATIO", "PAY_TO_BILL_6M", "AVG_SPEND", "BILL_TREND", "UTIL_TREND"]
def ovl_values(c):
    s = d3[c].astype(float)
    if c in LOGX or c in ("BILL_AMT1",):
        return pd.Series(slog(s), index=s.index), f"slog1p({c})"
    lo, hi = s.quantile([.005, .99])
    return s.clip(lo, hi), f"{c} (clipped p0.5-p99)"
for kind in ("kde", "ecdf"):
    fig, axes = plt.subplots(3, 4, figsize=(18, 11))
    for ax, c in zip(axes.ravel(), OVL):
        v, lab = ovl_values(c)
        tmp = pd.DataFrame({"v": v, "cls": d3["DEFAULT_LABEL"]}).dropna()
        if kind == "kde":
            sns.kdeplot(data=tmp, x="v", hue="cls", common_norm=False, fill=True, alpha=.25, palette=PAL_TARGET, ax=ax, bw_adjust=.8, warn_singular=False)
        else:
            sns.ecdfplot(data=tmp, x="v", hue="cls", palette=PAL_TARGET, ax=ax)
        ax.set_title(c, fontsize=10); ax.set_xlabel(lab, fontsize=8)
    fig.suptitle(f"3.7 {'KDE' if kind == 'kde' else 'ECDF'} overlay: default vs no default", fontsize=14, fontweight="bold")
    plt.tight_layout(); savefig(f"s03_overlay_{kind}"); plt.show()

**Observations:**
- The overlays are images; the matching numbers come from 3.8.
- Defaulters sit left on limit (median NT$ 90k vs 150k, KS 0.18), Sep payment (1,636 vs 2,460) and total payment (9,675 vs 16,524, KS 0.19).
- The curves still overlap heavily: no money variable exceeds KS 0.22 (KS = biggest gap between the two cumulative curves, 0 to 1).
- Story use: background only; shows why no single money variable separates the classes.

### 3.8 Separation statistics for every numeric variable
For each numeric variable (raw and engineered): two-sample **KS** statistic (largest gap between the two groups' cumulative curves, 0 to 1), **Mann-Whitney U** (rank-based test of whether one group tends to have higher values) with **rank-biserial** effect size (r > 0 ⇒ defaulters tend to have *higher* values), **Cohen's d** (difference in means measured in standard deviations) on signed-log1p values, and class medians. With n = 30,000 almost every p-value is tiny, so rank by **effect size**, not p.

In [ ]:
UTIL_COLS = [f"UTIL_{i}" for i in IDX]
PR_COLS = [f"PAY_RATIO_{i}" for i in IDX[:-1]]
NUM_ALL = list(dict.fromkeys(RAW_NUM + PAY_COLS + FEATURE_COLS + UTIL_COLS + PR_COLS + SPEND_COLS))
rows = []
for c in NUM_ALL:
    s = d3[c].astype(float)
    m = s.notna().to_numpy()
    x1, x0 = s[m & (y3 == 1)].to_numpy(), s[m & (y3 == 0)].to_numpy()
    ks = ks_2samp(x1, x0)
    mw = mannwhitneyu(x1, x0, alternative="two-sided")
    n1, n0 = len(x1), len(x0)
    rrb = 2 * mw.statistic / (n1 * n0) - 1
    l1, l0 = slog(x1), slog(x0)
    sp = np.sqrt(((n1 - 1) * l1.var(ddof=1) + (n0 - 1) * l0.var(ddof=1)) / (n1 + n0 - 2))
    rows.append({"variable": c, "n": n1 + n0, "median_default": np.median(x1), "median_no_default": np.median(x0),
                 "KS": ks.statistic, "KS_p": ks.pvalue, "MWU_p": mw.pvalue, "rank_biserial": rrb,
                 "cohens_d_slog": (l1.mean() - l0.mean()) / sp if sp > 0 else np.nan})
sep = pd.DataFrame(rows).sort_values("KS", ascending=False).reset_index(drop=True)
sep["direction"] = np.where(sep["rank_biserial"] > 0, "higher in defaulters", "lower in defaulters")
export(sep, "s03_separation_stats")
print(sep.drop(columns=["KS_p", "MWU_p"]).round(3).to_string())
fig, ax = plt.subplots(figsize=(12, 12))
top = sep.head(40).iloc[::-1]
ax.barh(top["variable"], top["KS"], color=np.where(top["rank_biserial"] > 0, C_DEF, C_OK))
ax.set_xlabel("KS statistic (max ECDF gap)"); ax.set_title("3.8 Top 40 numeric variables by KS (red = higher in defaulters, blue = lower)")
savefig("s03_ks_ranking"); plt.show()

**Observations:**
- Delinquency features separate best: N_LATE, MAX_DELAY and LATE_STREAK_MAX all reach KS 0.40; PAY_1 reaches 0.37.
- The best money feature is average spend (KS 0.215; median NT$ 1,651 for defaulters vs 4,317).
- LIMIT_BAL reaches KS 0.18; defaulters revolve less often (median 2 months vs 4).
- Story use: ranks the drivers; repayment history first, money second.

### 3.9 Monotonicity check
For every numeric variable the default rate is computed per decile (or per level for ≤12-level integers, levels with n ≥ 50). Spearman ρ (rank correlation, -1 to 1) of bin index vs rate measures monotonicity (whether the rate only moves in one direction). A **U-shape / inverted-U** is flagged when the extreme rate sits in an interior bin and *both* ends sit ≥ 2 pp and ≥ 25% of the range away from it. Non-monotone variables matter for Tableau (do not show them as a single trend line) and for any scorecard (need non-linear binning).

In [ ]:
def shape_of(col, q=10):
    b = bin_series(d3[col], q)
    g = d3.groupby(b, observed=True)[TARGET].agg(["size", "mean"])
    g = g[(g["size"] >= 50) & (g.index.astype(str) != "Missing")]
    r = g["mean"].to_numpy()
    k = len(r)
    if k < 4:
        return {"variable": col, "bins": k, "spearman": np.nan, "shape": "too few bins"}
    rho = spearmanr(np.arange(k), r)[0]
    rng_ = r.max() - r.min(); thr = max(0.02, 0.25 * rng_)
    imin, imax = int(np.argmin(r)), int(np.argmax(r))
    if 0 < imin < k - 1 and r[0] - r[imin] >= thr and r[-1] - r[imin] >= thr:
        shp = "U-shape"
    elif 0 < imax < k - 1 and r[imax] - r[0] >= thr and r[imax] - r[-1] >= thr:
        shp = "inverted-U"
    elif rho >= 0.9: shp = "monotone increasing"
    elif rho <= -0.9: shp = "monotone decreasing"
    elif rho >= 0.6: shp = "mostly increasing"
    elif rho <= -0.6: shp = "mostly decreasing"
    else: shp = "irregular / flat"
    return {"variable": col, "bins": k, "spearman": rho, "shape": shp, "rate_min": r.min(), "rate_max": r.max(),
            "range_pp": 100 * rng_, "argmin_bin": imin, "argmax_bin": imax}
mono = pd.DataFrame([shape_of(c) for c in NUM_ALL]).sort_values("spearman").reset_index(drop=True)
print(mono["shape"].value_counts().to_string()); print()
print(mono.round(3).to_string())
SHAPE_COL = {"U-shape": "#756bb1", "inverted-U": "#dd1c77", "monotone increasing": C_DEF, "mostly increasing": "#fc9272",
             "monotone decreasing": C_OK, "mostly decreasing": "#9ecae1", "irregular / flat": C_NEU, "too few bins": "#dddddd"}
mm = mono.dropna(subset=["spearman"])
fig, ax = plt.subplots(figsize=(12, 13))
ax.barh(mm["variable"], mm["spearman"], color=mm["shape"].map(SHAPE_COL))
for s_, col_ in SHAPE_COL.items():
    ax.barh([], [], color=col_, label=s_)
ax.legend(loc="lower right", fontsize=9); ax.axvline(0, color="k", lw=.8)
ax.set_xlabel("Spearman rho (bin index vs default rate)"); ax.set_title("3.9 Monotonicity of the default-rate curve, coloured by detected shape")
savefig("s03_monotonicity"); plt.show()
print("\nNon-monotone (U / inverted-U):", mono.loc[mono["shape"].isin(["U-shape", "inverted-U"]), "variable"].tolist())

**Observations:**
- 11 variables fall steadily as values rise, including LIMIT_BAL, AVG_PAY and every PAY_AMT (rank correlation near -1).
- 24 are not monotone (14 U-shaped, 10 inverted-U), including AGE, every utilisation measure and bills.
- Utilisation is U-shaped: 34.4% at zero or negative use, 15.5% at 10-30%, rising again at high use.
- Story use: use banded bar charts, not straight trend lines, for utilisation and age.

### 3.10 Weight of Evidence and Information Value for every variable
Numeric variables are decile-binned (integers with ≤ 12 levels keep their own levels; missing values get their own bin). WoE = ln(%good / %bad) per bin, so **negative WoE = riskier than average**. IV strength bands (Siddiqi): < 0.02 useless, 0.02–0.1 weak, 0.1–0.3 medium, 0.3–0.5 strong, > 0.5 *suspicious* (too good — in a scorecard this flags leakage or a near-definitional link; here it reflects how directly current delinquency foreshadows next-month default).

In [ ]:
CAT_VARS = ["SEX_LABEL", "EDU_LABEL", "EDUCATION", "MAR_LABEL", "MARRIAGE", "AGE_BAND", "LIMIT_BAND",
            "STATUS_GRP_SEP", "UTIL_BAND", "N_LATE_BAND", "PAYRATIO_BAND"]
IV_VARS = list(dict.fromkeys(CAT_VARS + NUM_ALL))
def iv_strength(v):
    return ("useless" if v < 0.02 else "weak" if v < 0.1 else "medium" if v < 0.3 else "strong" if v < 0.5 else "suspicious (>0.5)")
woe_rows, iv_rows = [], []
for c in IV_VARS:
    tmp = pd.DataFrame({c: bin_series(d3[c]), TARGET: d3[TARGET]})
    t, iv = woe_iv(tmp, c)
    t = t.reset_index().rename(columns={c: "bin"})
    t["n"] = t["good"] + t["bad"]
    t = t[t["n"] > 0]
    t.insert(0, "variable", c); t["bin"] = t["bin"].astype(str); t["bin_order"] = np.arange(len(t))
    woe_rows.append(t[["variable", "bin_order", "bin", "n", "bad", "rate", "woe", "iv_part"]].rename(columns={"bad": "defaults", "iv_part": "iv"}))
    iv_rows.append({"variable": c, "group": "raw" if c in raw.columns or c in PAY_COLS else "engineered/label",
                    "bins": len(t), "IV": iv, "strength": iv_strength(iv)})
woe_tab = pd.concat(woe_rows, ignore_index=True)
iv_rank = pd.DataFrame(iv_rows).sort_values("IV", ascending=False).reset_index(drop=True)
IV_MAP = iv_rank.set_index("variable")["IV"].to_dict()
export(woe_tab, "s03_woe_bins"); export(iv_rank, "s03_iv_ranking")
print(iv_rank["strength"].value_counts().to_string()); print()
print(iv_rank.round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 18))
ir = iv_rank.iloc[::-1]
ax.barh(ir["variable"], ir["IV"], color=np.where(ir["group"] == "raw", C_ACC, "#636363"), alpha=.85)
bands = [(0, .02, "useless"), (.02, .1, "weak"), (.1, .3, "medium"), (.3, .5, "strong"), (.5, max(.6, ir["IV"].max() * 1.05), "suspicious")]
for (a, b, lab), col_ in zip(bands, ["#f7f7f7", "#fee8c8", "#fdbb84", "#fc8d59", "#d7301f"]):
    ax.axvspan(a, b, color=col_, alpha=.25, zorder=0)
    ax.text((a + b) / 2, len(ir) + .3, lab, ha="center", fontsize=9)
ax.set_xlim(0, bands[-1][1]); ax.set_ylim(-1, len(ir) + 1.5)
ax.set_xlabel("Information Value"); ax.set_title("3.10 Information Value of every variable (green = raw column, grey = engineered/label)")
savefig("s03_iv_bar"); plt.show()

WoE profile of the eight highest-IV variables. Bars = WoE per bin (red = riskier than average); black line = default rate. A clean monotone WoE staircase is what a scorecard wants; reversals show where coarse-classing would be needed.

In [ ]:
top8 = iv_rank["variable"].head(8).tolist()
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, c in zip(axes.ravel(), top8):
    w = woe_tab[woe_tab["variable"] == c]
    x = np.arange(len(w))
    ax.bar(x, w["woe"], color=np.where(w["woe"] < 0, C_DEF, C_OK), alpha=.8)
    ax.set_xticks(x); ax.set_xticklabels([s[:14] for s in w["bin"]], rotation=60, ha="right", fontsize=7)
    ax.axhline(0, color="k", lw=.8); ax.set_ylabel("WoE")
    ax2 = ax.twinx(); ax2.plot(x, w["rate"], "k.-"); ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax2.grid(False)
    ax.set_title(f"{c}  (IV={IV_MAP[c]:.3f})", fontsize=10)
fig.suptitle("3.10 WoE profiles — top 8 variables by IV", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s03_woe_top8"); plt.show()

**Observations:**
- Information Value (IV) scores how well a variable's bins split defaulters from others; above 0.5 is usually called 'suspicious' (too good).
- 8 variables exceed 0.5, all delinquency-based: PAY_1 0.877, STATUS_GRP_SEP 0.874, N_LATE 0.862, PAY_2 0.549.
- IV then decays with older months: PAY_3 0.416 down to PAY_6 0.294; LIMIT_BAL is only 0.178.
- No demographic variable makes the top 50 of 71; 10 variables are useless (IV below 0.02).
- Story use: the driver ranking chart.

### 3.11 Driver leaderboard: six association metrics in one table
For every variable:
* **Cramér's V** — categorical association, 0 = none to 1 = perfect (numerics are decile-binned first; `PAY_i` codes treated as categories);
* **point-biserial r** — linear correlation with the 0/1 target (numerics / ordinal codes only);
* **mutual information** (how much knowing the variable reduces uncertainty about default; `sklearn.mutual_info_classif`, nats; kNN estimator for continuous, exact for discrete);
* **univariate AUC** (chance a random defaulter ranks above a random non-defaulter; 0.5 = coin flip) and **Gini = 2·AUC − 1** (oriented so AUC ≥ 0.5; categoricals scored by their in-sample level rate, which is mildly optimistic);
* **IV** from 3.10.

`mean_rank` averages the ranks across V, |r|, MI, Gini and IV. This is a descriptive ranking of single-variable signal, not a model.

In [ ]:
t0 = time.time()
LB_VARS = IV_VARS
num_set = set(NUM_ALL)
lb = []
for c in LB_VARS:
    s = d3[c]
    is_num = c in num_set
    binned = bin_series(s)
    v = cramers_v(binned, d3[TARGET])
    if is_num:
        xs = s.astype(float); m = xs.notna()
        pb = pointbiserialr(y3[m], xs[m])[0]
        score = xs.fillna(xs.median())
    else:
        pb = np.nan
        score = d3.groupby(binned, observed=True)[TARGET].transform("mean")
    auc = roc_auc_score(y3, score)
    lb.append({"variable": c, "kind": "numeric" if is_num else "categorical", "cramers_v": v,
               "point_biserial": pb, "auc_raw": auc, "AUC": max(auc, 1 - auc)})
lb = pd.DataFrame(lb)
# mutual information: numeric -> continuous estimator unless few-level integer; categoricals -> integer codes, discrete
X = pd.DataFrame({c: (d3[c].astype(float).fillna(d3[c].median()) if c in num_set else bin_series(d3[c]).cat.codes)
                  for c in LB_VARS})
disc = np.array([(c not in num_set) or d3[c].nunique() <= 12 for c in LB_VARS])
lb["mutual_info"] = mutual_info_classif(X.to_numpy(), y3, discrete_features=disc, random_state=42)
lb["Gini"] = 2 * lb["AUC"] - 1
lb["IV"] = lb["variable"].map(IV_MAP)
lb["direction"] = np.where(lb["kind"] == "categorical", "n/a (categorical)", np.where(lb["auc_raw"] >= .5, "higher = riskier", "higher = safer"))
METRICS = ["cramers_v", "point_biserial", "mutual_info", "Gini", "IV"]
ranks = pd.DataFrame({m: (lb[m].abs() if m == "point_biserial" else lb[m]).rank(ascending=False) for m in METRICS})
lb["mean_rank"] = ranks.mean(1)
lb = lb.sort_values("mean_rank").reset_index(drop=True)
lb.insert(0, "rank", np.arange(1, len(lb) + 1))
export(lb.drop(columns="auc_raw"), "s03_driver_leaderboard")
print(f"computed in {time.time() - t0:.1f}s")
print(lb.drop(columns="auc_raw").round(4).to_string())

Dot-plot of the top 35 drivers: each metric is scaled to its own maximum (1 = strongest variable on that metric), so agreement between metrics is visible at a glance. A Tableau version: rows = variable, columns = metric, mark = circle, colour = metric.

In [ ]:
top = lb.head(35).copy()
fig, ax = plt.subplots(figsize=(12, 13))
ypos = np.arange(len(top))[::-1]
mk = {"cramers_v": ("o", "#1b9e77"), "point_biserial": ("s", "#d95f02"), "mutual_info": ("^", "#7570b3"),
      "Gini": ("D", "#e7298a"), "IV": ("P", "#66a61e")}
for m, (mark, col_) in mk.items():
    vals = lb[m].abs()
    ax.scatter(top[m].abs() / vals.max(), ypos, marker=mark, color=col_, s=55, alpha=.85, label=m)
ax.set_yticks(ypos); ax.set_yticklabels(top["variable"])
ax.set_xlabel("metric / max(metric) across all variables"); ax.legend(loc="lower right")
ax.set_title("3.11 Driver leaderboard (top 35 by mean rank)")
savefig("s03_driver_leaderboard"); plt.show()
print("metric rank agreement (Spearman between metric ranks):")
print(ranks.corr(method="spearman").round(2).to_string())

**Observations:**
- N_LATE ranks first across six metrics (AUC 0.726, IV 0.862); AUC is the chance a random defaulter scores above a random non-defaulter (0.5 = coin flip).
- September status and PAY_1 come next; the top 11 are all delinquency features.
- LIMIT_BAL is the first non-delinquency profile variable, at rank 20 (AUC 0.618).
- The metrics agree: rank correlations between them are 0.74 to 1.00.
- Story use: the leaderboard backs the 'behaviour beats demographics' message.

### 3.12 Recency decay: does the signal fade with the month's age?
The same metric is computed for each monthly snapshot (Apr → Sep, Sep = most recent before the October outcome): Gini of `PAY_i`, `BILL_AMTi`, `PAY_AMTi`, `UTIL_i`, `PAY_RATIO_i`, `SPEND_i`, plus Cramér's V and IV of `PAY_i`. A steep rise towards Sep = strong recency effect, i.e. the latest month dominates and older history adds less.

In [ ]:
def gini(x):
    x = pd.Series(x).astype(float); m = x.notna().to_numpy()
    a = roc_auc_score(y3[m], x[m]); return 2 * max(a, 1 - a) - 1
rec_rows = []
for i in IDX:
    row = {"month": MONTH[i], "idx": i,
           "Gini PAY status": gini(d3[f"PAY_{i}"]), "Gini BILL_AMT": gini(d3[f"BILL_AMT{i}"]),
           "Gini PAY_AMT": gini(d3[f"PAY_AMT{i}"]), "Gini UTIL": gini(d3[f"UTIL_{i}"]),
           "CramersV PAY status": cramers_v(d3[f"PAY_{i}"], d3[TARGET]),
           "IV PAY status": IV_MAP[f"PAY_{i}"], "IV PAY_AMT": IV_MAP[f"PAY_AMT{i}"], "IV UTIL": IV_MAP[f"UTIL_{i}"],
           "IV BILL_AMT": IV_MAP[f"BILL_AMT{i}"]}
    if i <= 5:
        row["Gini PAY_RATIO"] = gini(d3[f"PAY_RATIO_{i}"]); row["Gini SPEND"] = gini(d3[f"SPEND_{i}"])
        row["IV PAY_RATIO"] = IV_MAP[f"PAY_RATIO_{i}"]
    rec_rows.append(row)
recency = pd.DataFrame(rec_rows).set_index("month").loc[MONTH_ORDER]
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for c in [c for c in recency.columns if c.startswith("Gini")]:
    axes[0].plot(recency.index, recency[c], "o-", label=c.replace("Gini ", ""))
axes[0].set_title("Univariate Gini by month"); axes[0].legend(fontsize=8)
for c in [c for c in recency.columns if c.startswith("IV")]:
    axes[1].plot(recency.index, recency[c], "o-", label=c.replace("IV ", ""))
axes[1].set_title("Information Value by month"); axes[1].legend(fontsize=8)
axes[2].bar(recency.index, recency["CramersV PAY status"], color=C_DEF, alpha=.8)
axes[2].set_title("Cramér's V of PAY status by month")
for a in axes: a.set_xlabel("month (chronological; Sep = latest)")
fig.suptitle("3.12 Recency decay of the default signal", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s03_recency_decay"); plt.show()
print(recency.drop(columns="idx").round(4).to_string())
export(recency.reset_index(), "s03_recency_decay")

**Observations:**
- Status signal fades with age: Gini rises from 0.180 (Apr) to 0.379 (Sep) and IV triples, 0.294 to 0.877 (Gini = 2 x AUC - 1).
- Payment amounts also gain value toward Sep (Gini 0.168 to 0.223).
- Bill amounts carry almost no signal in any month (Gini 0.000-0.035).
- Utilisation is the exception: older months are slightly more informative (Gini 0.132 Apr vs 0.102 Sep).
- Story use: 'the latest month matters most' chart.

### 3.13 Demographic gaps with CIs: the gender "paradox" and its context
Women are the majority of clients yet men default more often. Is that gap a gender effect or a composition effect (men differ in age, education, credit line and behaviour)? Below: the Male − Female default-rate gap (pp, 95% CI, normal approximation) overall and **within** each level of age band, education, marital status, limit band and latest status group; plus the same for Single − Married. If the gap persists inside strata, it is not just composition. (Formal confounding tests — logit, CMH, Simpson scan — are in Section 10.)

In [ ]:
def gap_ci(a, b):
    # a, b: 0/1 arrays. returns diff (a-b), CI, n's
    n1, n2 = len(a), len(b)
    if n1 < 30 or n2 < 30: return None
    p1, p2 = a.mean(), b.mean(); se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    return {"n_a": n1, "n_b": n2, "rate_a": p1, "rate_b": p2, "gap_pp": 100 * (p1 - p2), "lo_pp": 100 * (p1 - p2 - 1.96 * se), "hi_pp": 100 * (p1 - p2 + 1.96 * se)}
def gaps(col, a, b, strata):
    out = []
    r = gap_ci(d3.loc[d3[col] == a, TARGET].to_numpy(), d3.loc[d3[col] == b, TARGET].to_numpy())
    out.append({"contrast": f"{a} - {b}", "stratifier": "(overall)", "stratum": "all", **r})
    for st in strata:
        for lev, g in d3.groupby(st, observed=True):
            r = gap_ci(g.loc[g[col] == a, TARGET].to_numpy(), g.loc[g[col] == b, TARGET].to_numpy())
            if r: out.append({"contrast": f"{a} - {b}", "stratifier": st, "stratum": str(lev), **r})
    return pd.DataFrame(out)
STRATA = ["AGE_BAND", "EDU_LABEL", "MAR_LABEL", "LIMIT_BAND", "STATUS_GRP_SEP"]
gsex = gaps("SEX_LABEL", "Male", "Female", STRATA)
gmar = gaps("MAR_LABEL", "Single", "Married", ["AGE_BAND", "SEX_LABEL", "EDU_LABEL", "LIMIT_BAND", "STATUS_GRP_SEP"])
demo_gaps = pd.concat([gsex, gmar], ignore_index=True)
export(demo_gaps, "s03_demographic_gaps")
for nm, gg in (("Male - Female", gsex), ("Single - Married", gmar)):
    fig, ax = plt.subplots(figsize=(10, 0.32 * len(gg) + 1.5))
    lab = (gg["stratifier"] + ": " + gg["stratum"]).tolist()
    yy = np.arange(len(gg))[::-1]
    sig = (gg["lo_pp"] > 0) | (gg["hi_pp"] < 0)
    ax.errorbar(gg["gap_pp"], yy, xerr=[gg["gap_pp"] - gg["lo_pp"], gg["hi_pp"] - gg["gap_pp"]], fmt="none", ecolor=C_NEU)
    ax.scatter(gg["gap_pp"], yy, c=np.where(sig, C_DEF, C_NEU), zorder=3)
    ax.axvline(0, color="k", lw=.8); ax.axvline(gg["gap_pp"].iloc[0], color=C_DEF, ls=":", lw=1)
    ax.set_yticks(yy); ax.set_yticklabels(lab, fontsize=8); ax.set_xlabel("default-rate gap (percentage points, 95% CI)")
    ax.set_title(f"3.13 {nm} default-rate gap, overall and within strata (red = CI excludes 0)")
    savefig(f"s03_gap_{nm.replace(' - ', '_vs_').lower()}"); plt.show()
print(demo_gaps.round(3).to_string())

Composition check: how do men and women differ on the variables that drive default? If men have lower limits, worse status and are older, part of their higher rate is borrowed from those variables.

In [ ]:
comp = d3.groupby("SEX_LABEL").agg(n=(TARGET, "size"), default_rate=(TARGET, "mean"), mean_age=("AGE", "mean"),
                                     median_limit=("LIMIT_BAL", "median"), share_late_sep=("PAY_1", lambda s: (s >= 1).mean()),
                                     mean_n_late=("N_LATE", "mean"), mean_util=("AVG_UTIL", "mean"),
                                     share_grad=("EDU_LABEL", lambda s: (s == "Graduate school").mean()),
                                     share_married=("MAR_LABEL", lambda s: (s == "Married").mean())).T
print(comp.round(3).to_string())

**Observations:**
- Men default 3.4 pp more than women overall (95% CI 2.4-4.4 pp).
- The gap holds in most strata (5.5 pp at ages 36-40, 6.7 pp among late-1m clients) but vanishes at limits of NT$ 50k or less (0.4 pp) and among clients 2+ months late.
- Men are older (36.5 vs 34.8), have lower limits (median 130k vs 150k), higher utilisation (40% vs 35%) and more late months (0.92 vs 0.78).
- Singles default 2.5 pp less than married clients (CI -3.5 to -1.6), and the gap holds in every limit band.
- Story use: the gender gap slide; part of the gap is behaviour, but it persists.

## 4. Interactions: two- and three-way default-rate surfaces

Section 3 looked at one feature at a time. Here we ask **where features combine**: does the effect of age depend on the credit limit, does high utilisation hurt only when the client is already late, do demographics matter once behaviour is known?

**Method in one line.** For every pair (A, B) we compare the observed default rate in each cell with the rate an **additive-logit, no-interaction** model predicts from the two marginals (`logit p = a_i + b_j`; each variable shifts the log-odds of default on its own, with no joint effect). The residual (observed − expected, in percentage points) and the likelihood-ratio statistic G² (additive model vs. the saturated cell model, which gives every cell its own rate, i.e. the deviance gain from adding the A×B term) quantify the interaction. Large residuals in *specific cells* are the **hidden interactions**: patterns that neither variable shows on its own.

| Sub-section | Content | Tableau mapping |
|---|---|---|
| 4.1 | Helpers: `rate_heatmap`, `interaction_lift`, `facet_heatmap`, `binned_surface` | — |
| 4.2 | Interaction line plots (non-parallel lines = interaction) | line chart, colour = 2nd variable |
| 4.3 | 18 curated two-way default-rate + lift heatmaps | highlight table (rows × cols, colour = rate, label = n) |
| 4.4 | Observed vs. additive-expected residual maps (hidden interactions) | highlight table on `interaction_resid_pp` |
| 4.5 | Automated scan over all 91 pairs of 14 variables, ranking | bar chart of `aic_gain` |
| 4.6 | Reversal cells (risky + risky = safe, safe + safe = risky) | filtered table |
| 4.7 | Three-way small multiples + three-way LR tests | highlight table with page / column facet |
| 4.8 | Continuous × continuous default-rate surfaces | density/hex map, quantile grid |
| 4.9 | Pair plot coloured by default | scatter matrix |
| 4.10 | Demographic risk vs. behavioural risk: who is safe though risky, who defaults though safe | 3×3 highlight table + profile table |

Exports: `s04_pair_interaction_ranking`, `s04_pairwise_rates_long` (one generic long table → **one Tableau sheet with two parameters** can show any pair), `s04_threeway_rates_long`, `s04_quantile_surfaces_long`, `s04_demo_vs_behaviour`, `s04_paradox_segments`.

**New columns:** none are added to `df`. All extra bands live in a working copy `d4` (prefixed `S4_`).

**Section guide**

- **What it does:** Looks at two (sometimes three) variables together. For each pair it compares the actual default rate in each cell with the rate expected if the two effects simply added up. Big gaps are "hidden interactions".
- **Question it answers:** Does the effect of one variable depend on another, and do demographics still matter once behaviour is known?
- **Inputs / new columns:** demographics, limit, September status, late-month count, utilisation, payment ratio, zero-payment months. No new `df` columns; extra bands live in a copy `d4` (prefix `S4_`).
- **How to read the charts:** heatmap cell colour = default rate (or lift vs 22.1% base: red = riskier, blue = safer); numbers in cells = rate and client count; grey = too few clients. Residual maps show observed minus expected, in pp. Non-parallel lines in 4.2 signal an interaction.
- **Terms:** G² = likelihood-ratio test of whether two variables act jointly; AIC gain = fit improvement after a penalty for extra parameters.
- **Key takeaways:**
  - Demographic pairs just add up (age x limit p = 0.49).
  - Strongest interaction: September status x utilisation (AIC gain 154).
  - Hidden risk: 300-500k limit with no bill defaults at 49.5% vs 28.0% expected.
  - Clean clients in risky segments default at 11-18%; Sep-late clients in safe segments at 34-50%.
  - Low/low risk cell 6.0% vs high/high 50.6%.
- **Where it feeds:** the "behaviour beats demographics" 3x3 matrix and the "idle premium card" callout; `s04_pairwise_rates_long.csv` lets one Tableau sheet show any pair via two parameters.

### 4.1 Working copy, extra bands and helper functions
`d4` is a copy of `df` with a few extra coarse bands used in the scan: zero-payment months, utilisation trend direction, implied-spend quintile, and a 3-level limit band for faceting.

In [ ]:
import statsmodels.api as sm
import patsy
from scipy.special import xlogy

d4 = df.copy()
d4["S4_LIMIT3"] = pd.cut(d4["LIMIT_BAL"], [0, 100_000, 300_000, np.inf], labels=["<=100k", "100-300k", ">300k"])
d4["S4_ZEROPAY_BAND"] = pd.cut(d4["N_ZERO_PAY"], [-1, 0, 2, 5, 6], labels=["0", "1-2", "3-5", "6 (never paid)"])
d4["S4_UTIL_TREND_BAND"] = pd.cut(d4["UTIL_TREND"], [-np.inf, -0.02, 0.02, np.inf], labels=["Falling", "Flat", "Rising"])
d4["S4_SPEND_BAND"] = pd.qcut(d4["AVG_SPEND"].rank(method="first"), 5, labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"])
d4["STATUS_GRP_AUG"] = d4["STATUS_GRP_2"]
d4["STATUS_GRP_APR"] = d4["STATUS_GRP_6"]

S4_ORDERS = {"SEX_LABEL": ["Male", "Female"], "EDU_LABEL": EDU_ORDER, "MAR_LABEL": MAR_ORDER,
             "STATUS_GRP_SEP": STATUS_GROUP_ORDER, "STATUS_GRP_AUG": STATUS_GROUP_ORDER, "STATUS_GRP_APR": STATUS_GROUP_ORDER}
for i in IDX:
    S4_ORDERS[f"STATUS_GRP_{i}"] = STATUS_GROUP_ORDER
NICE = {"SEX_LABEL": "Sex", "EDU_LABEL": "Education", "MAR_LABEL": "Marital status", "AGE_BAND": "Age band",
        "LIMIT_BAND": "Credit limit", "S4_LIMIT3": "Credit limit (3 bands)", "STATUS_GRP_SEP": "Status Sep",
        "STATUS_GRP_AUG": "Status Aug", "STATUS_GRP_APR": "Status Apr", "STATUS_GRP_6": "Status Apr (6m ago)",
        "UTIL_BAND": "Avg utilisation", "N_LATE_BAND": "# late months", "PAYRATIO_BAND": "Bills repaid (5 statements, aligned)",
        "S4_ZEROPAY_BAND": "# zero-payment months", "S4_UTIL_TREND_BAND": "Utilisation trend",
        "S4_SPEND_BAND": "Implied spend quintile", "PAY_1": "PAY code Sep", "PAY_2": "PAY code Aug",
        "S4_DEMO_TIER": "Demographic risk tier", "S4_BEHAV_TIER": "Behavioural risk tier"}
def nice(c):
    return NICE.get(c, c)

def levels_of(data, col):
    s = data[col]
    present = set(s.dropna().unique())
    if col in S4_ORDERS:
        base = S4_ORDERS[col]
    elif isinstance(s.dtype, pd.CategoricalDtype):
        base = list(s.cat.categories)
    else:
        base = sorted(present)
    return [l for l in base if l in present]

def cell_table(data, a, b, target=TARGET):
    g = data.groupby([a, b], observed=True)[target].agg(n="size", defaults="sum").reset_index()
    g[a] = g[a].astype(object); g[b] = g[b].astype(object)
    g["rate"] = g["defaults"] / g["n"]
    g["lift"] = g["rate"] / data[target].mean()
    return g

def _pv(g, a, b, val, ra, cb):
    return g.pivot(index=a, columns=b, values=val).reindex(index=ra, columns=cb)

def _heat(ax, V, N, min_n, cmap, title, fmt_fn, center=None, vmin=None, vmax=None, cbar_label="", show_n=True, pct_cbar=False):
    mask = N.fillna(0) < min_n
    Vm = V.where(~mask).astype(float)
    annot = np.empty(Vm.shape, dtype=object)
    for i in range(Vm.shape[0]):
        for j in range(Vm.shape[1]):
            if mask.iat[i, j] or pd.isna(Vm.iat[i, j]):
                annot[i, j] = ""
            else:
                annot[i, j] = fmt_fn(Vm.iat[i, j]) + (f"\n{int(N.iat[i, j]):,}" if show_n else "")
    fs = 9 if Vm.size <= 30 else 8 if Vm.size <= 56 else 6.5 if Vm.size <= 100 else 5.5
    cbk = {"label": cbar_label, "shrink": .8}
    if pct_cbar:
        cbk["format"] = mtick.PercentFormatter(1, decimals=0)
    sns.heatmap(Vm, ax=ax, cmap=cmap, center=center, vmin=vmin, vmax=vmax, annot=annot, fmt="",
                annot_kws={"fontsize": fs}, linewidths=.5, linecolor="white", cbar_kws=cbk)
    ax.set_facecolor("#e6e6e6")
    ax.set_title(title, fontsize=11)
    ax.tick_params(axis="x", rotation=35); ax.tick_params(axis="y", rotation=0)
    for t in ax.get_xticklabels():
        t.set_ha("right")

def rate_heatmap(data, row, col, min_n=50, twin=True, title=None, figsize=None, name=None, verbose=True):
    # Annotated default-rate heatmap (rate + cell n; cells with n < min_n greyed out) and a twin lift-vs-base map.
    g = cell_table(data, row, col)
    ra, cb = levels_of(data, row), levels_of(data, col)
    R, N = _pv(g, row, col, "rate", ra, cb), _pv(g, row, col, "n", ra, cb)
    keep_r, keep_c = (N.fillna(0) >= min_n).any(axis=1), (N.fillna(0) >= min_n).any(axis=0)
    R, N = R.loc[keep_r, keep_c], N.loc[keep_r, keep_c]
    ra, cb = list(R.index), list(R.columns)
    base = data[TARGET].mean()
    w = max(5.5, 0.85 * len(cb) + 2.5)
    h = max(3.2, 0.55 * len(ra) + 1.6)
    fig, axes = plt.subplots(1, 2 if twin else 1, figsize=figsize or ((2 * w if twin else w), h))
    axes = np.atleast_1d(axes)
    vmax = float(R.where(N >= min_n).max().max())
    _heat(axes[0], R, N, min_n, CMAP_RATE, "Default rate (label: rate / n)", lambda v: f"{v:.0%}",
          vmin=0, vmax=max(vmax, base * 1.2), cbar_label="default rate", pct_cbar=True)
    axes[0].set_xlabel(nice(col)); axes[0].set_ylabel(nice(row))
    if twin:
        LL = np.log2(R / base)
        lim = float(np.nanmin([2.0, np.nanmax(np.abs(LL.where(N >= min_n).values))])) if (N >= min_n).any().any() else 1
        _heat(axes[1], LL, N, min_n, CMAP_DIV, "Lift vs base rate (colour: log2 lift)", lambda v: f"x{2 ** v:.2f}",
              center=0, vmin=-lim, vmax=lim, cbar_label="log2(rate / base)", show_n=False)
        axes[1].set_xlabel(nice(col)); axes[1].set_ylabel("")
    fig.suptitle(title or f"{nice(row)} x {nice(col)}: default rate (base {base:.1%}, grey = n < {min_n})", fontweight="bold")
    plt.tight_layout()
    if name:
        savefig(name, fig)
    plt.show()
    if verbose:
        ok = g[g["n"] >= min_n]
        hi, lo = ok.loc[ok["rate"].idxmax()], ok.loc[ok["rate"].idxmin()]
        print(f"{nice(row)} x {nice(col)} | cells shown {len(ok)}/{len(g)} | "
              f"highest: {hi[row]} & {hi[col]} = {hi['rate']:.1%} (n={int(hi['n']):,}) | "
              f"lowest: {lo[row]} & {lo[col]} = {lo['rate']:.1%} (n={int(lo['n']):,}) | spread {100 * (hi['rate'] - lo['rate']):.1f} pp")
        print((R.where(N >= min_n) * 100).to_string(na_rep=".", float_format=lambda v: f"{v:.1f}"))
    return g

def _g2(d, n, p):
    p = np.clip(np.asarray(p, float), 1e-12, 1 - 1e-12)
    d, n = np.asarray(d, float), np.asarray(n, float)
    return float(2 * np.sum(xlogy(d, d / (n * p)) + xlogy(n - d, (n - d) / (n * (1 - p)))))

def _fit_cells(g, formula):
    # Grouped binomial GLM on cell counts; returns fitted p and model rank.
    X = patsy.dmatrix(formula, g, return_type="dataframe")
    y = np.column_stack([g["defaults"], g["n"] - g["defaults"]])
    res = sm.GLM(y, X, family=sm.families.Binomial()).fit()
    return np.asarray(res.predict(X)), int(np.linalg.matrix_rank(X.values))

def interaction_stats(data, a, b, min_n=50):
    # Observed cell rate vs additive-logit (no interaction) expectation. G2 = LR test of A x B term.
    g = cell_table(data, a, b)
    h = g.rename(columns={a: "A", b: "B"}).copy()
    h["A"] = h["A"].astype(str); h["B"] = h["B"].astype(str)
    try:
        p, k = _fit_cells(h, "C(A) + C(B)")
        model = "additive logit"
    except Exception:
        ra = data.groupby(a, observed=True)[TARGET].mean(); rb = data.groupby(b, observed=True)[TARGET].mean()
        base = data[TARGET].mean()
        p = np.clip(g[a].map(ra).values * g[b].map(rb).values / base, 1e-6, 1 - 1e-6)
        k = 1 + (g[a].nunique() - 1) + (g[b].nunique() - 1)
        model = "multiplicative"
    g["expected"] = p
    g["resid_pp"] = 100 * (g["rate"] - g["expected"])
    g["z"] = (g["defaults"] - g["n"] * p) / np.sqrt(g["n"] * p * (1 - p))
    G2 = _g2(g["defaults"], g["n"], p)
    dfree = max(1, len(g) - k)
    G2null = _g2(g["defaults"], g["n"], np.full(len(g), g["defaults"].sum() / g["n"].sum()))
    ok = g[g["n"] >= min_n]
    if len(ok):
        j = ok["resid_pp"].abs().idxmax()
        top = ok.loc[j]
        cell, maxres, zmax = f"{top[a]} & {top[b]}", float(top["resid_pp"]), float(ok["z"].abs().max())
    else:
        cell, maxres, zmax = "", np.nan, np.nan
    st = {"var_a": a, "var_b": b, "cells": len(g), "G2": G2, "df": dfree, "p_value": float(stats.chi2.sf(G2, dfree)),
          "aic_gain": G2 - 2 * dfree, "G2_per_df": G2 / dfree, "interaction_share": G2 / G2null if G2null > 0 else np.nan,
          "max_abs_resid_pp": abs(maxres) if pd.notna(maxres) else np.nan, "resid_pp_at_max": maxres,
          "max_resid_cell": cell, "max_abs_z": zmax, "model": model}
    return st, g

def interaction_lift(data, a, b, min_n=50, name=None, verbose=True):
    # Three panels: observed rate | additive-logit expected rate | residual (pp). Stars mark |z| >= 3.
    st, g = interaction_stats(data, a, b, min_n)
    ra, cb = levels_of(data, a), levels_of(data, b)
    R, E, Z, N = (_pv(g, a, b, v, ra, cb) for v in ("rate", "expected", "z", "n"))
    Rp = _pv(g, a, b, "resid_pp", ra, cb)
    w = max(4.5, 0.75 * len(cb) + 2.2); h = max(3.2, 0.5 * len(ra) + 1.6)
    fig, axes = plt.subplots(1, 3, figsize=(3 * w, h))
    vmax = float(np.nanmax(R.where(N >= min_n).values))
    _heat(axes[0], R, N, min_n, CMAP_RATE, "Observed rate (/ n)", lambda v: f"{v:.0%}", vmin=0, vmax=vmax, pct_cbar=True)
    _heat(axes[1], E, N, min_n, CMAP_RATE, "Expected if no interaction", lambda v: f"{v:.0%}", vmin=0, vmax=vmax, show_n=False, pct_cbar=True)
    lim = float(np.nanmax(np.abs(Rp.where(N >= min_n).values))) if (N >= min_n).any().any() else 1
    _heat(axes[2], Rp, N, min_n, CMAP_DIV, "Residual = observed - expected (pp), * |z|>=3", lambda v: f"{v:+.1f}",
          center=0, vmin=-lim, vmax=lim, show_n=False, cbar_label="pp")
    for i in range(Rp.shape[0]):
        for j in range(Rp.shape[1]):
            if N.iat[i, j] >= min_n and pd.notna(Z.iat[i, j]) and abs(Z.iat[i, j]) >= 3:
                axes[2].text(j + .5, i + .82, "*", ha="center", va="center", fontsize=12, fontweight="bold")
    for ax in axes:
        ax.set_xlabel(nice(b)); ax.set_ylabel("")
    axes[0].set_ylabel(nice(a))
    fig.suptitle(f"Hidden interaction {nice(a)} x {nice(b)}: G2 = {st['G2']:.1f} on {st['df']} df "
                 f"(p = {st['p_value']:.1e}), interaction share of between-cell deviance = {st['interaction_share']:.1%}",
                 fontweight="bold")
    plt.tight_layout()
    if name:
        savefig(name, fig)
    plt.show()
    if verbose:
        ok = g[g["n"] >= min_n].sort_values("resid_pp")
        cols = [a, b, "n", "rate", "expected", "resid_pp", "z"]
        print(f"Most NEGATIVE residuals (safer than the marginals predict) and most POSITIVE (riskier):")
        print(pd.concat([ok.head(3), ok.tail(3)])[cols].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
    return st, g

def facet_heatmap(data, row, col, facet, min_n=30, name=None, verbose=True):
    # Small multiples: one rate heatmap per facet level, shared colour scale.
    fl = levels_of(data, facet)
    ra, cb = levels_of(data, row), levels_of(data, col)
    w = max(4.0, 0.7 * len(cb) + 1.8); h = max(3.2, 0.5 * len(ra) + 1.6)
    fig, axes = plt.subplots(1, len(fl), figsize=(w * len(fl), h), squeeze=False)
    tabs = []
    for k, lv in enumerate(fl):
        sub = data[data[facet] == lv]
        g = cell_table(sub, row, col); g["facet_var"] = facet; g["facet_level"] = str(lv)
        tabs.append(g)
    allg = pd.concat(tabs)
    vmax = float(allg.loc[allg["n"] >= min_n, "rate"].max())
    for k, (lv, g) in enumerate(zip(fl, tabs)):
        R, N = _pv(g, row, col, "rate", ra, cb), _pv(g, row, col, "n", ra, cb)
        _heat(axes[0, k], R, N, min_n, CMAP_RATE, f"{nice(facet)} = {lv}", lambda v: f"{v:.0%}", vmin=0, vmax=vmax, pct_cbar=True)
        axes[0, k].set_xlabel(nice(col)); axes[0, k].set_ylabel(nice(row) if k == 0 else "")
    fig.suptitle(f"{nice(row)} x {nice(col)}, faceted by {nice(facet)} (rate / n; grey = n < {min_n})", fontweight="bold")
    plt.tight_layout()
    if name:
        savefig(name, fig)
    plt.show()
    # three-way LR test: all two-way model vs saturated
    h3 = data.groupby([row, col, facet], observed=True)[TARGET].agg(n="size", defaults="sum").reset_index()
    h3.columns = ["A", "B", "F", "n", "defaults"]
    for c in "ABF":
        h3[c] = h3[c].astype(str)
    out = {"row": row, "col": col, "facet": facet}
    try:
        p2, k2 = _fit_cells(h3, "C(A)*C(B) + C(A)*C(F) + C(B)*C(F)")
        G2 = _g2(h3["defaults"], h3["n"], p2); d3 = max(1, len(h3) - k2)
        out.update({"G2_threeway": G2, "df": d3, "p_threeway": float(stats.chi2.sf(G2, d3))})
    except Exception as e:
        out.update({"G2_threeway": np.nan, "df": np.nan, "p_threeway": np.nan})
    per = []
    for lv in fl:
        st, _ = interaction_stats(data[data[facet] == lv], row, col, min_n)
        per.append({"facet_level": lv, "n": int((data[facet] == lv).sum()), "G2_rowxcol": st["G2"], "df": st["df"],
                    "p": st["p_value"], "max_abs_resid_pp": st["max_abs_resid_pp"], "max_resid_cell": st["max_resid_cell"]})
    if verbose:
        print(f"Three-way test (does {nice(facet)} change the {nice(row)} x {nice(col)} pattern?): "
              f"G2 = {out['G2_threeway']:.1f} on {out['df']} df, p = {out['p_threeway']:.2e}")
        print("Two-way interaction within each facet:")
        print(pd.DataFrame(per).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
    return allg, out

print("Section 4 helpers ready | d4:", d4.shape)

### 4.2 Interaction plots: non-parallel lines are interactions
Each line is one level of the second variable; if the two variables acted independently (on the logit scale), the lines would be roughly parallel. Crossing or fanning lines are the first visual hint of an interaction. Points with n < 50 are dropped.

In [ ]:
def interaction_plot(ax, data, x, line, min_n=50, palette="viridis"):
    g = cell_table(data, x, line)
    xl = levels_of(data, x); ll = levels_of(data, line)
    cols = sns.color_palette(palette, len(ll))
    for c, lv in zip(cols, ll):
        s = g[(g[line] == lv) & (g["n"] >= min_n)].set_index(x).reindex(xl)
        ax.plot(range(len(xl)), s["rate"], marker="o", color=c, lw=2, label=str(lv))
    ax.axhline(BASE_RATE, ls="--", color="k", lw=1)
    ax.set_xticks(range(len(xl))); ax.set_xticklabels([str(v) for v in xl], rotation=30, ha="right")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(f"{nice(x)} by {nice(line)}"); ax.set_ylabel("default rate")
    ax.legend(title=nice(line), fontsize=7, title_fontsize=8, ncol=2)

fig, axes = plt.subplots(2, 3, figsize=(18, 9.5))
pairs_ip = [("AGE_BAND", "LIMIT_BAND"), ("AGE_BAND", "SEX_LABEL"), ("EDU_LABEL", "MAR_LABEL"),
            ("UTIL_BAND", "STATUS_GRP_SEP"), ("LIMIT_BAND", "STATUS_GRP_SEP"), ("PAYRATIO_BAND", "N_LATE_BAND")]
for ax, (x, ln) in zip(axes.ravel(), pairs_ip):
    interaction_plot(ax, d4, x, ln)
plt.suptitle("Interaction plots: default rate by X, one line per level of a second variable", fontweight="bold")
plt.tight_layout(); savefig("s04_interaction_lineplots"); plt.show()

**Observations:**
- These plots are images; the numbers come from the 4.3 tables.
- Age x limit lines run nearly parallel: at every age, default falls about 15-20 pp from NT$ 50k-or-less to 300-500k, so the two add up.
- Utilisation x September status lines are not parallel: utilisation moves late-1m clients from 22.7% to 50.4% but revolvers only from 8.9% to 18.5%.
- Story use: introduces interactions; demographic effects add, behavioural ones interact.

### 4.3 Two-way default-rate heatmaps (18 curated pairs)
Left panel: default rate with cell count. Right panel: lift vs. the 22.1% base rate on a log2 diverging scale (red = riskier than average, blue = safer). Grey cells have n < 50 and are suppressed. A compact rate table (in %) is printed under each figure so insights can be written from text alone.

#### 4.3.1 Demographics × demographics / credit line

In [ ]:
TWO_WAY_DEMO = [("AGE_BAND", "LIMIT_BAND"), ("SEX_LABEL", "AGE_BAND"), ("SEX_LABEL", "EDU_LABEL"),
                ("EDU_LABEL", "MAR_LABEL"), ("MAR_LABEL", "AGE_BAND"), ("SEX_LABEL", "MAR_LABEL"),
                ("EDU_LABEL", "LIMIT_BAND")]
for a, b in TWO_WAY_DEMO:
    rate_heatmap(d4, a, b, min_n=50, name=f"s04_heat_{a}_x_{b}".lower())

**Observations:**
- Limit dominates age: every age band under 61 defaults at 31-34% at NT$ 50k or less vs 11-16% at 300-500k.
- Men are riskier than women in 7 of 8 age bands; male 21-25 is the worst cell (29.3%).
- Other/Unknown education stays under 13% at every limit, while high school at NT$ 50k or less reaches 33.8%.
- Demographic pairs spread only 9-34 pp, far less than behavioural pairs below.
- Story use: the demographic heatmap; limit is the main profile axis.

#### 4.3.2 Demographics / credit line × repayment behaviour
Does the same repayment status in September carry different risk for different ages, limits, education levels?

In [ ]:
TWO_WAY_MIX = [("AGE_BAND", "STATUS_GRP_SEP"), ("LIMIT_BAND", "STATUS_GRP_SEP"), ("EDU_LABEL", "STATUS_GRP_SEP"),
               ("N_LATE_BAND", "LIMIT_BAND"), ("AGE_BAND", "UTIL_BAND"), ("UTIL_BAND", "LIMIT_BAND")]
for a, b in TWO_WAY_MIX:
    rate_heatmap(d4, a, b, min_n=50, name=f"s04_heat_{a}_x_{b}".lower())

**Observations:**
- September status overrides profile: late-2m clients default at 65-77% in every age, limit and education band.
- Limit still matters for revolvers: 19.2% at NT$ 50k or less vs 7.1% at 300-500k.
- With 4-6 late months, every limit band defaults at 62-67%, so a high limit no longer protects.
- Zero or negative utilisation at a 300-500k limit is risky (46.8%).
- Story use: the core 'behaviour beats profile' heatmap.

#### 4.3.3 Behaviour × behaviour
Utilisation, payment ratio and delinquency against each other: which behavioural combinations are toxic, and does high utilisation still matter when the client pays in full?

In [ ]:
TWO_WAY_BEH = [("UTIL_BAND", "STATUS_GRP_SEP"), ("N_LATE_BAND", "UTIL_BAND"), ("PAYRATIO_BAND", "STATUS_GRP_SEP")]
for a, b in TWO_WAY_BEH:
    rate_heatmap(d4, a, b, min_n=50, name=f"s04_heat_{a}_x_{b}".lower())

**Observations:**
- The safest cell is no late months with 10-30% utilisation (7.0%, n = 3,149); the worst is 4-6 late months with 50-70% use (72.1%).
- Paying nothing while 2 months late gives 80.2%; among late-2m clients, paying 50-100% of bills cuts this to 48.4% (number from the pre-fix run; refresh after the next Kaggle run).
- High utilisation still matters for full payers: 26.5% at 70-90% use vs 11.3% at 10-30%.
- Zero or negative use is risky even with a clean record (30.3%).
- Story use: 'toxic combinations' chart (late + no payment).

#### 4.3.4 The "last two months" matrix and "now vs. six months ago"
`PAY_1 × PAY_2` at code level (rows = Sep, columns = Aug): the diagonal is persistence, below-diagonal is deterioration (worse in Sep than in Aug), above is cure. `STATUS_GRP_SEP × STATUS_GRP_6` asks whether April behaviour still matters once September is known.

*Footnote on the move counts:* In September, code 1 mixes clients who were paid-up or inactive in August with clients who were 2+ months late in August; it is not a uniform first delay. So an Aug 2 -> Sep 1 "Better in Sep" move is not necessarily a cure, and an Aug -1/-2 -> Sep 1 "Worse in Sep" move is the bank's Sep-only first-delay code.

In [ ]:
rate_heatmap(d4, "PAY_1", "PAY_2", min_n=30, name="s04_heat_pay1_x_pay2",
             title="Sep code (rows) x Aug code (cols): default rate (grey = n < 30)")
rate_heatmap(d4, "STATUS_GRP_SEP", "STATUS_GRP_6", min_n=50, name="s04_heat_statussep_x_statusapr",
             title="Status now (Sep, rows) x status 6 months ago (Apr, cols)")

# compact numeric: persistence vs deterioration vs cure in the PAY_1 x PAY_2 matrix
tr = d4[["PAY_1", "PAY_2", TARGET]].copy()
tr["move"] = np.select([tr["PAY_1"] > tr["PAY_2"], tr["PAY_1"] < tr["PAY_2"]], ["Worse in Sep", "Better in Sep"], "Same code")
tr["sep_late"] = np.where(tr["PAY_1"] >= 1, "Late in Sep", "Not late in Sep")
display(rate_table(tr, ["sep_late", "move"])[["sep_late", "move", "n", "defaults", "rate", "ci_lo", "ci_hi", "lift"]])

**Observations:**
- Persistence is the danger: among clients late in Sep, keeping the same code as in Aug gives 70.8% default (n = 1,675), vs about 44% when the code improved or worsened.
- Sep code 3 after Aug code 2 reaches 75.7%; inactive in Sep after paid-in-full in Aug is only 8.8%.
- April still matters for mild cases: late 1m in Sep defaults at 21.9% if paid in full in Apr vs 50.7% if late 2m in Apr.
- Clients not late in Sep whose code improved since Aug still default at 21.6% vs 13.4% for those with an unchanged code.
- Story use: the 'late twice in a row' message and a recovery-is-not-full-cure caveat.

### 4.4 Hidden interactions: observed vs. additive expectation
For each pair the middle panel shows what the default rate *would be* if the two variables simply added up on the logit scale. The right panel is the gap. Blue cells (negative residuals) are safer than their two marginals predict; red cells are riskier. Stars mark |z| ≥ 3 (Pearson residual), i.e. gaps very unlikely to be noise. The suptitle gives the likelihood-ratio G² for the interaction term and the share of between-cell deviance that is interaction rather than main effects.

In [ ]:
HIDDEN_PAIRS = [("AGE_BAND", "LIMIT_BAND"), ("SEX_LABEL", "AGE_BAND"), ("EDU_LABEL", "MAR_LABEL"),
                ("UTIL_BAND", "STATUS_GRP_SEP"), ("LIMIT_BAND", "STATUS_GRP_SEP"), ("PAYRATIO_BAND", "STATUS_GRP_SEP"),
                ("N_LATE_BAND", "UTIL_BAND"), ("AGE_BAND", "STATUS_GRP_SEP")]
hidden_stats = []
for a, b in HIDDEN_PAIRS:
    st, _ = interaction_lift(d4, a, b, min_n=50, name=f"s04_resid_{a}_x_{b}".lower())
    hidden_stats.append(st)
hs = pd.DataFrame(hidden_stats)[["var_a", "var_b", "G2", "df", "p_value", "interaction_share", "max_abs_resid_pp", "max_resid_cell", "max_abs_z"]]
hs

**Observations:**
- G2 is a likelihood-ratio test of whether two variables act jointly rather than simply adding up.
- Demographic pairs show no interaction: age x limit p = 0.49, sex x age p = 0.30, education x marital p = 0.47.
- All behavioural pairs interact (p < 0.001); utilisation x September status is strongest (G2 = 222).
- Late 1m with 50-70% use defaults at 50.4% vs 38.4% expected (+12.1 pp); inactive at NT$ 50k or less defaults at 10.9% vs 21.3% expected.
- Story use: the 'hidden cells' callout; demographics add, behaviour interacts.

### 4.5 Automated scan: every pair of 14 variables (91 pairs)
For each pair we fit the additive-logit model on cell counts and compute:

* **G²** (LR test of the A×B term, df = cells − main-effect parameters) and its p-value. With 30,000 rows almost everything is "significant", so we rank by **AIC gain = G² − 2·df** (AIC = Akaike information criterion, model fit penalised for extra parameters) (how much the interaction improves fit after paying for its parameters).
* **interaction_share** = G² / between-cell deviance: what fraction of the pair's explanatory power is interaction rather than main effects.
* **max |residual|** (pp) among cells with n ≥ 50, and where it occurs; **max |z|**.
* **Cramér's V** between A and B: high values warn that the two variables are nearly redundant (e.g. Sep vs Aug status), so their "interaction" partly reflects sparse off-diagonal cells.

Pairs are tagged *profile × profile*, *profile × behaviour*, *behaviour × behaviour*.

In [ ]:
SCAN_VARS = ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND",
             "STATUS_GRP_SEP", "STATUS_GRP_AUG", "STATUS_GRP_APR", "UTIL_BAND", "N_LATE_BAND",
             "PAYRATIO_BAND", "S4_ZEROPAY_BAND", "S4_UTIL_TREND_BAND", "S4_SPEND_BAND"]
PROFILE = {"SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND"}
t0 = time.time()
scan_rows, long_parts = [], []
for a, b in itertools.combinations(SCAN_VARS, 2):
    st, g = interaction_stats(d4, a, b, min_n=50)
    st["cramers_v_ab"] = cramers_v(d4[a].astype(str), d4[b].astype(str))
    fa, fb = a in PROFILE, b in PROFILE
    st["family"] = "profile x profile" if fa and fb else ("behaviour x behaviour" if not (fa or fb) else "profile x behaviour")
    scan_rows.append(st)
    # tidy long rows (both directions so Tableau parameters work in either order)
    oa = {v: i for i, v in enumerate(levels_of(d4, a))}; ob = {v: i for i, v in enumerate(levels_of(d4, b))}
    ma = d4.groupby(a, observed=True)[TARGET].mean(); mb = d4.groupby(b, observed=True)[TARGET].mean()
    base_rows = pd.DataFrame({"n": g["n"], "defaults": g["defaults"], "rate": g["rate"], "lift": g["lift"],
                              "expected_rate_additive": g["expected"], "interaction_resid_pp": g["resid_pp"], "z": g["z"]})
    fwd = base_rows.assign(var_a=nice(a), level_a=g[a].astype(str), order_a=g[a].map(oa), rate_a=g[a].map(ma).astype(float),
                           var_b=nice(b), level_b=g[b].astype(str), order_b=g[b].map(ob), rate_b=g[b].map(mb).astype(float),
                           col_a=a, col_b=b, scan_direction=1)
    rev = base_rows.assign(var_a=nice(b), level_a=g[b].astype(str), order_a=g[b].map(ob), rate_a=g[b].map(mb).astype(float),
                           var_b=nice(a), level_b=g[a].astype(str), order_b=g[a].map(oa), rate_b=g[a].map(ma).astype(float),
                           col_a=b, col_b=a, scan_direction=0)
    long_parts += [fwd, rev]
scan = pd.DataFrame(scan_rows).sort_values("aic_gain", ascending=False).reset_index(drop=True)
scan["rank"] = np.arange(1, len(scan) + 1)
scan["pair"] = scan["var_a"].map(nice) + " x " + scan["var_b"].map(nice)
long4 = pd.concat(long_parts, ignore_index=True)
ci = wilson_ci(long4["defaults"], long4["n"]); long4["ci_lo"], long4["ci_hi"] = ci
long4["reliable_n50"] = (long4["n"] >= 50).astype(int)
long4 = long4[["var_a", "level_a", "order_a", "var_b", "level_b", "order_b", "n", "defaults", "rate", "ci_lo", "ci_hi", "lift",
               "rate_a", "rate_b", "expected_rate_additive", "interaction_resid_pp", "z", "reliable_n50", "col_a", "col_b", "scan_direction"]]
print(f"scanned {len(scan)} pairs in {time.time() - t0:.1f}s | p<0.001: {(scan['p_value'] < 1e-3).sum()} | "
      f"AIC gain > 0: {(scan['aic_gain'] > 0).sum()}")
SHOW = ["rank", "pair", "family", "G2", "df", "p_value", "aic_gain", "interaction_share", "max_abs_resid_pp",
        "resid_pp_at_max", "max_resid_cell", "max_abs_z", "cramers_v_ab"]
display(scan[SHOW].head(15))
export(scan[["rank", "pair", "family"] + [c for c in scan.columns if c not in ("rank", "pair", "family")]], "s04_pair_interaction_ranking")
export(long4, "s04_pairwise_rates_long")

**Tableau note.** `s04_pairwise_rates_long` has one row per (pair, cell) in both orientations. Create two string parameters `Var A`, `Var B` (values = the readable `var_a` names), filter `[var_a] = [Var A] AND [var_b] = [Var B]`, put `level_a` (sorted by `order_a`) on Rows and `level_b` (sorted by `order_b`) on Columns, colour by `rate` or `interaction_resid_pp`, label with `n`. One sheet then shows any of the 91 pairs; filter `reliable_n50 = 1` to hide thin cells.

Top 15 by AIC gain, and the best pairs within each family (the overall ranking is dominated by behaviour × behaviour pairs, so the per-family view surfaces the demographic interactions):

In [ ]:
fam_col = {"profile x profile": C_ACC, "profile x behaviour": "#756bb1","behaviour x behaviour": C_DEF}
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
top = scan.head(15).iloc[::-1]
axes[0].barh(top["pair"], top["aic_gain"], color=top["family"].map(fam_col))
axes[0].set_xscale("log"); axes[0].set_xlabel("AIC gain of A x B term (G2 - 2 df, log scale)")
axes[0].set_title("Top 15 interacting pairs (all families)")
for y, (v, s) in enumerate(zip(top["aic_gain"], top["interaction_share"])):
    axes[0].text(v, y, f"  share {s:.0%}", va="center", fontsize=8)
best = scan[scan["aic_gain"] > 0].groupby("family", group_keys=False).head(6)
best = best.sort_values(["family", "aic_gain"]).reset_index(drop=True)
axes[1].barh(best["pair"], best["interaction_share"], color=best["family"].map(fam_col))
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_xlabel("interaction share of between-cell deviance"); axes[1].set_title("Top 6 per family: how much of the pair's signal is interaction")
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(color=c, label=k) for k, c in fam_col.items()], loc="lower right", fontsize=8)
plt.tight_layout(); savefig("s04_pair_interaction_ranking"); plt.show()
print(best[["family", "pair", "G2", "df", "aic_gain", "interaction_share", "max_resid_cell", "resid_pp_at_max"]].to_string(index=False, float_format=lambda v: f"{v:.3g}"))

**Observations:**
- 48 of 91 pairs interact at p < 0.001 and 54 improve AIC (model fit after a penalty for extra parameters).
- The top three are September status x utilisation (AIC gain 154), late months x zero-payment months (147) and limit x payment ratio (139) (number from the pre-fix run; refresh after the next Kaggle run).
- The only profile x profile pair with any gain is sex x limit (gain 7.65).
- Interaction adds at most 15% of total association (limit x utilisation 0.149); most pairs add under 5%.
- Story use: ranks interactions for the dashboard's 'combo' view.

Interaction-strength matrix over all 14 variables: colour = log10(AIC gain), so the whole landscape of pairwise interactions is visible at once (upper triangle mirrored).

In [ ]:
lab = [nice(v) for v in SCAN_VARS]
M = pd.DataFrame(np.nan, index=lab, columns=lab)
S = M.copy()
for _, r in scan.iterrows():
    i, j = nice(r["var_a"]), nice(r["var_b"])
    v = np.log10(max(r["aic_gain"], 0) + 1)
    M.loc[i, j] = M.loc[j, i] = v
    S.loc[i, j] = S.loc[j, i] = r["interaction_share"]
fig, axes = plt.subplots(1, 2, figsize=(19, 7.5))
sns.heatmap(M, ax=axes[0], cmap="mako_r", annot=True, fmt=".1f", annot_kws={"fontsize": 7}, linewidths=.4,
            cbar_kws={"label": "log10(AIC gain + 1)"})
axes[0].set_title("Interaction strength (absolute)")
sns.heatmap(S, ax=axes[1], cmap="mako_r", annot=True, fmt=".0%", annot_kws={"fontsize": 7}, linewidths=.4,
            cbar_kws={"label": "interaction share"})
axes[1].set_title("Interaction share of between-cell deviance (relative)")
for ax in axes:
    ax.tick_params(axis="x", rotation=45)
    for t in ax.get_xticklabels(): t.set_ha("right")
plt.tight_layout(); savefig("s04_interaction_matrix"); plt.show()

**Observations:**
- The matrix is an image of the same scan; strong cells cluster in the behaviour rows.
- The demographic block (sex, education, marital status, age) is faint; its best pair gains only 7.65 AIC vs 154 for the top behavioural pair.
- Story use: background only.

Residual maps for the **top 6 pairs of the scan that are not already shown in 4.4** (automatically chosen; profile × behaviour preferred so the result is not dominated by month-to-month status pairs).

In [ ]:
shown = {frozenset(p) for p in HIDDEN_PAIRS}
cand = scan[~scan.apply(lambda r: frozenset((r["var_a"], r["var_b"])) in shown, axis=1)]
cand = pd.concat([cand[cand["family"] == "profile x behaviour"].head(3), cand[cand["family"] == "profile x profile"].head(1),
                  cand[cand["family"] == "behaviour x behaviour"].head(2)])
for _, r in cand.iterrows():
    interaction_lift(d4, r["var_a"], r["var_b"], min_n=50, name=f"s04_resid_scan_{r['var_a']}_x_{r['var_b']}".lower())

**Observations:**
- Dormant high-limit accounts are a hidden risk: 300-500k limit with no bill defaults at 49.5% vs 28.0% expected (+21.5 pp, z = 7.0).
- The same holds for 300-500k with zero or negative use (46.8% vs 26.4%), while low-limit no-bill accounts are safer than expected (32.0% vs 52.2%).
- Late 2m in Apr but never skipping a payment gives 52.8% vs 31.7% expected; one late month with no zero-payment months gives 39.4% vs 26.6% (z = 9.6).
- Story use: a standout insight, 'the idle premium card' worth its own callout.

### 4.6 Reversal cells: risky + risky = safe, safe + safe = risky
From the long table, cells (n ≥ 100) where **both** marginal levels are riskier than average (marginal lift ≥ 1.1) yet the cell itself is at or below the base rate (lift ≤ 1.0), and the mirror image. These are the most counter-intuitive two-way combinations in the data, and the most persuasive dashboard call-outs. Also listed: the largest negative and positive residuals vs. the additive expectation (n ≥ 200).

In [ ]:
L1 = long4[(long4["scan_direction"] == 1) & (long4["n"] >= 100)].copy()
L1["lift_a"], L1["lift_b"] = L1["rate_a"] / BASE_RATE, L1["rate_b"] / BASE_RATE
cols6 = ["var_a", "level_a", "var_b", "level_b", "n", "rate", "lift", "lift_a", "lift_b", "expected_rate_additive", "interaction_resid_pp"]
safe_though_risky = L1[(L1["lift_a"] >= 1.1) & (L1["lift_b"] >= 1.1) & (L1["lift"] <= 1.0)].sort_values("lift")
risky_though_safe = L1[(L1["lift_a"] <= 0.9) & (L1["lift_b"] <= 0.9) & (L1["lift"] >= 1.0)].sort_values("lift", ascending=False)
print(f"Risky x risky -> safe cells: {len(safe_though_risky)}")
display(safe_though_risky[cols6].head(15))
print(f"Safe x safe -> risky cells: {len(risky_though_safe)}")
display(risky_though_safe[cols6].head(15))
L2 = L1[L1["n"] >= 200].sort_values("interaction_resid_pp")
print("Largest NEGATIVE residuals vs additive expectation (n >= 200):")
display(L2[cols6 + ["z"]].head(10))
print("Largest POSITIVE residuals vs additive expectation (n >= 200):")
display(L2[cols6 + ["z"]].tail(10).iloc[::-1])

**Observations:**
- Reversals are rare: only one risky x risky cell turns safe and one safe x safe cell turns risky.
- The safe reversal: late 2m in Apr with just 1 late month in total defaults at 18.3% (n = 497), below base; a one-off delay that was cured.
- Late 2m in Aug after paid in full in Apr gives 33.3% vs 46.4% expected, so a good earlier record softens a recent slip.
- The biggest positive surprise again is 300-500k limit with no bill (49.5% vs 28.0%).
- Story use: supports the 'one-off delay vs habit' message.

### 4.7 Three-way views (small multiples)
Each figure repeats a two-way heatmap for each level of a third variable on a **shared colour scale**. Below each figure: a three-way likelihood-ratio test (model with all two-way interactions vs. the saturated three-way cell model — significant means the A×B pattern itself changes across the facet) and the A×B interaction strength within each facet. Thin cells (n < 30) greyed.

In [ ]:
THREE_WAY = [("AGE_BAND", "LIMIT_BAND", "SEX_LABEL"),
             ("UTIL_BAND", "STATUS_GRP_SEP", "S4_LIMIT3"),
             ("EDU_LABEL", "MAR_LABEL", "SEX_LABEL"),
             ("AGE_BAND", "STATUS_GRP_SEP", "SEX_LABEL"),
             ("PAYRATIO_BAND", "N_LATE_BAND", "S4_LIMIT3"),
             ("AGE_BAND", "MAR_LABEL", "S4_LIMIT3")]
three_parts, three_tests = [], []
for r_, c_, f_ in THREE_WAY:
    allg, out = facet_heatmap(d4, r_, c_, f_, min_n=30, name=f"s04_facet_{r_}_x_{c_}_by_{f_}".lower())
    allg = allg.rename(columns={r_: "level_a", c_: "level_b"})
    allg["var_a"], allg["var_b"], allg["facet_var"] = nice(r_), nice(c_), nice(f_)
    allg["level_a"] = allg["level_a"].astype(str); allg["level_b"] = allg["level_b"].astype(str)
    three_parts.append(allg[["var_a", "level_a", "var_b", "level_b", "facet_var", "facet_level", "n", "defaults", "rate", "lift"]])
    three_tests.append(out)

**Observations:**
- Demographic modifiers do nothing: sex does not change age x limit (p = 0.25), education x marital (p = 0.22) or age x status (p = 0.11).
- Limit changes how utilisation and status combine: the interaction is strong below NT$ 300k (p < 0.00001) but absent above (p = 0.115).
- The payment ratio x late months test reports G2 = 428,318; the design was rank-deficient (warning above), so this p-value is a numerical artefact (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: background only; two-way views are enough.

In [ ]:
tw = pd.DataFrame(three_tests)
tw["triple"] = tw["row"].map(nice) + " x " + tw["col"].map(nice) + " | " + tw["facet"].map(nice)
display(tw[["triple", "G2_threeway", "df", "p_threeway"]])
three_long = pd.concat(three_parts, ignore_index=True)
export(three_long, "s04_threeway_rates_long")

**Observations:**
- 5 of 6 three-way tests have p above 0.11; the remaining one is the rank-deficient artefact.
- Story use: background only.

### 4.8 Continuous × continuous default-rate surfaces
Two views per pair. **Left:** hexbin where colour = mean default rate in the hexagon (hexes with < 25 clients hidden; axes clipped at the 0.5/99.5 percentiles, money axes on signed log1p). **Right:** quantile grid (≈ deciles of each variable, so every cell is well populated) with the default rate printed; this is the version to rebuild in Tableau. Discrete variables (e.g. `N_LATE`) use their own values instead of quantiles.

In [ ]:
def slog(x):
    return np.sign(x) * np.log1p(np.abs(x))

def _fmt(v):
    a = abs(v)
    if a >= 1e6: return f"{v / 1e6:.1f}M"
    if a >= 1e3: return f"{v / 1e3:.0f}k"
    if a >= 10: return f"{v:.0f}"
    return f"{v:.2f}"

def _qbin(s, q):
    if s.nunique() <= 12:
        return s.astype(int).astype(str) if (s % 1 == 0).all() else s.astype(str), None
    b = pd.qcut(s, q, duplicates="drop")
    labels = {iv: f"{_fmt(iv.left)}..{_fmt(iv.right)}" for iv in b.cat.categories}
    return b.map(labels), [labels[iv] for iv in b.cat.categories]

def binned_surface(data, x, y, tx=None, ty=None, q=10, min_n=40, name=None, xlab=None, ylab=None):
    sub = data[[x, y, TARGET]].replace([np.inf, -np.inf], np.nan).dropna()
    X = tx(sub[x]) if tx else sub[x]; Y = ty(sub[y]) if ty else sub[y]
    fig, axes = plt.subplots(1, 2, figsize=(17, 6.2))
    xlo, xhi = np.percentile(X, [0.5, 99.5]); ylo, yhi = np.percentile(Y, [0.5, 99.5])
    keep = X.between(xlo, xhi) & Y.between(ylo, yhi)
    Yj = Y[keep] + (RNG.uniform(-.3, .3, keep.sum()) if sub[y].nunique() <= 12 else 0)
    hb = axes[0].hexbin(X[keep], Yj, C=sub.loc[keep, TARGET], reduce_C_function=np.mean, gridsize=32,
                        mincnt=25, cmap=CMAP_RATE, vmin=0, vmax=0.75)
    cb = fig.colorbar(hb, ax=axes[0]); cb.ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); cb.set_label("default rate")
    axes[0].set_xlabel(xlab or (f"{x} (transformed)" if tx else x)); axes[0].set_ylabel(ylab or (f"{y} (transformed)" if ty else y))
    axes[0].set_title("Hexbin: mean default rate (hexes n >= 25)")
    bx, ox = _qbin(sub[x], q); by, oy = _qbin(sub[y], q)
    t = pd.DataFrame({"bx": bx, "by": by, TARGET: sub[TARGET]})
    g = t.groupby(["by", "bx"], observed=True)[TARGET].agg(n="size", defaults="sum").reset_index()
    g["rate"] = g["defaults"] / g["n"]
    ox = ox or sorted(g["bx"].unique(), key=lambda v: float(v)); oy = oy or sorted(g["by"].unique(), key=lambda v: float(v))
    R = g.pivot(index="by", columns="bx", values="rate").reindex(index=oy[::-1], columns=ox)
    N = g.pivot(index="by", columns="bx", values="n").reindex(index=oy[::-1], columns=ox)
    _heat(axes[1], R, N, min_n, CMAP_RATE, f"Quantile grid: default rate (grey = n < {min_n})", lambda v: f"{v:.0%}",
          vmin=0, vmax=max(0.5, float(R.max().max())), show_n=False, pct_cbar=True)
    axes[1].set_xlabel(f"{x} bin"); axes[1].set_ylabel(f"{y} bin")
    fig.suptitle(f"Default-rate surface: {x} x {y}", fontweight="bold")
    plt.tight_layout()
    if name: savefig(name, fig)
    plt.show()
    ok = g[g["n"] >= min_n]
    hi, lo = ok.loc[ok["rate"].idxmax()], ok.loc[ok["rate"].idxmin()]
    print(f"{x} x {y}: grid {len(ox)}x{len(oy)} | max cell {hi['bx']} / {hi['by']} = {hi['rate']:.1%} (n={int(hi['n'])}) | "
          f"min cell {lo['bx']} / {lo['by']} = {lo['rate']:.1%} (n={int(lo['n'])})")
    g["var_x"], g["var_y"] = x, y
    g["order_x"] = g["bx"].map({v: i for i, v in enumerate(ox)}); g["order_y"] = g["by"].map({v: i for i, v in enumerate(oy)})
    return g.rename(columns={"bx": "bin_x", "by": "bin_y"})

surf_parts = []
surf_parts.append(binned_surface(d4, "LIMIT_BAL", "AGE", tx=np.log1p, name="s04_surface_limit_x_age", xlab="log1p(LIMIT_BAL)"))

**Observations:**
- Default ranges from 41.5% for NT$ 10-30k limits at age 34-37 (n = 287) to 6.9% for 360k-1M limits at age 27-29 (n = 216), a 6x spread.
- The surface changes mainly along the limit axis.
- Story use: optional Tableau surface; repeats the limit message.

Utilisation × payment ratio (the two core behavioural axes), limit × average bill, the September bill × September payment (signed log1p), and utilisation × number of late months.

In [ ]:
d4["S4_AVG_UTIL_CLIP"] = d4["AVG_UTIL"].clip(-0.2, 1.5)
d4["S4_AVG_PAY_RATIO_CLIP"] = d4["AVG_PAY_RATIO"].clip(0, 2)
surf_parts.append(binned_surface(d4, "S4_AVG_UTIL_CLIP", "S4_AVG_PAY_RATIO_CLIP", ty=np.log1p, name="s04_surface_util_x_payratio",
                                 xlab="AVG_UTIL (clipped -0.2..1.5)", ylab="log1p(AVG_PAY_RATIO, clipped 0..2)"))
surf_parts.append(binned_surface(d4, "LIMIT_BAL", "AVG_BILL", tx=np.log1p, ty=slog, name="s04_surface_limit_x_avgbill",
                                 xlab="log1p(LIMIT_BAL)", ylab="signed log1p(AVG_BILL)"))
surf_parts.append(binned_surface(d4, "BILL_AMT1", "PAY_AMT1", tx=slog, ty=np.log1p, name="s04_surface_bill1_x_payamt1",
                                 xlab="signed log1p(BILL_AMT1, Sep bill)", ylab="log1p(PAY_AMT1, Sep payment)"))
surf_parts.append(binned_surface(d4, "S4_AVG_UTIL_CLIP", "N_LATE", name="s04_surface_util_x_nlate",
                                 xlab="AVG_UTIL (clipped)", ylab="N_LATE (jittered)"))
surf_long = pd.concat(surf_parts, ignore_index=True)
surf_long["var_x"] = surf_long["var_x"].str.replace("S4_", "").str.replace("_CLIP", "")
surf_long["var_y"] = surf_long["var_y"].str.replace("S4_", "").str.replace("_CLIP", "")
export(surf_long[["var_x", "bin_x", "order_x", "var_y", "bin_y", "order_y", "n", "defaults", "rate"]], "s04_quantile_surfaces_long")

**Observations:**
- Utilisation x late months: near-zero use with 6 late months gives 81.7% (n = 60) vs 7.0% for 13-28% use with no late months (n = 2,231).
- A Sep bill of NT$ 83-142k with zero Sep payment defaults at 52.5%, vs 5.7% for a 22-37k bill with a payment of 10k or more.
- Limit x bill: 39.7% for small limits and small bills vs 2.0% for 360k+ limits with NT$ 46-73k average bills.
- Low utilisation with near-zero payment ratio gives 76.9% (n = 52) vs 7.1% for moderate use with a high payment ratio (n = 508).
- Story use: the 'bill paid vs bill ignored' heatmap.

### 4.9 Pair plot of key features, coloured by default
4,000 sampled clients; money variables on log1p (signed for bills), utilisation and payment ratio clipped. Lower triangle only. The diagonal shows class-normalised densities, so the shapes compare defaulters with non-defaulters irrespective of the 22/78 imbalance.

In [ ]:
samp = d4.sample(4000, random_state=42)
pp = pd.DataFrame({
    "log LIMIT": np.log1p(samp["LIMIT_BAL"]),
    "AGE": samp["AGE"],
    "AVG_UTIL": samp["AVG_UTIL"].clip(-0.1, 1.3),
    "slog AVG_BILL": slog(samp["AVG_BILL"]),
    "log AVG_PAY": np.log1p(samp["AVG_PAY"]),
    "PAY_TO_BILL_6M": samp["PAY_TO_BILL_6M"].clip(0, 2),
    "N_LATE": samp["N_LATE"] + RNG.uniform(-.25, .25, len(samp)),
    "Default": samp["DEFAULT_LABEL"],
})
gpp = sns.pairplot(pp, hue="Default", palette=PAL_TARGET, corner=True, diag_kind="hist", height=2.1,
                   plot_kws={"s": 6, "alpha": .35, "linewidth": 0},
                   diag_kws={"stat": "density", "common_norm": False, "element": "step", "bins": 30})
gpp.fig.suptitle("Pair plot of key features (4,000 sampled clients), coloured by default", y=1.01, fontweight="bold")
savefig("s04_pairplot", gpp.fig); plt.show()
print("Spearman correlations within each class (sample):")
for lab_, grp in pp.groupby("Default"):
    print(lab_); print(grp.drop(columns="Default").corr(method="spearman").to_string(float_format=lambda v: f"{v:.2f}"))

**Observations:**
- Utilisation and late months move together for defaulters (rank correlation 0.27) but not for non-defaulters (0.04).
- Payment amount and payment ratio are unrelated for defaulters (-0.03) but positive for non-defaulters (0.13).
- Utilisation and average bill are tightly linked in both groups (0.80-0.81); age barely links to anything (at most 0.19).
- Story use: background only.

### 4.10 Demographic risk vs. behavioural risk: who is safe though "risky", who defaults though "safe"?
Two purely empirical risk scores (no fitted model):

* **Demographic risk** = smoothed default rate of the client's profile cell *Sex × Education × Marital × Age band × Limit (3 bands)* (empirical-Bayes shrinkage towards the base rate with prior weight 50: small cells are pulled towards 22.1%, as if 50 average clients were added).
* **Behavioural risk** = smoothed default rate of the cell *Status Sep × # late months × Avg utilisation band*.

Each score is cut into terciles (three equal-sized groups: Low / Mid / High). The 3×3 matrix shows how far behaviour overrides demographics; the off-diagonal corners are the two groups of interest.

In [ ]:
def eb_cell_rate(data, keys, m=50):
    g = data.groupby(keys, observed=True)[TARGET].agg(n="size", d="sum").reset_index()
    g["r"] = (g["d"] + m * BASE_RATE) / (g["n"] + m)
    return data[keys].merge(g[keys + ["r"]], on=keys, how="left")["r"].to_numpy()

DEMO_KEYS = ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "S4_LIMIT3"]
BEH_KEYS = ["STATUS_GRP_SEP", "N_LATE_BAND", "UTIL_BAND"]
d4["S4_DEMO_RISK"] = eb_cell_rate(d4, DEMO_KEYS)
d4["S4_BEHAV_RISK"] = eb_cell_rate(d4, BEH_KEYS)
TIERS = ["Low", "Mid", "High"]
d4["S4_DEMO_TIER"] = pd.qcut(d4["S4_DEMO_RISK"].rank(method="first"), 3, labels=TIERS)
d4["S4_BEHAV_TIER"] = pd.qcut(d4["S4_BEHAV_RISK"].rank(method="first"), 3, labels=TIERS)
print("Score ranges by tier:")
print(d4.groupby("S4_DEMO_TIER", observed=True)["S4_DEMO_RISK"].agg(["min", "max", "mean"]).to_string(float_format=lambda v: f"{v:.3f}"))
print(d4.groupby("S4_BEHAV_TIER", observed=True)["S4_BEHAV_RISK"].agg(["min", "max", "mean"]).to_string(float_format=lambda v: f"{v:.3f}"))
print(f"Spearman(demo risk, behav risk) = {d4[['S4_DEMO_RISK', 'S4_BEHAV_RISK']].corr(method='spearman').iloc[0, 1]:.3f}")
dvb = rate_heatmap(d4, "S4_DEMO_TIER", "S4_BEHAV_TIER", min_n=50, name="s04_demo_vs_behaviour",
                   title="Demographic risk tier (rows) x behavioural risk tier (cols): actual default rate")
dvb_out = dvb.rename(columns={"S4_DEMO_TIER": "demo_tier", "S4_BEHAV_TIER": "behav_tier"})
dvb_out["share_clients"] = dvb_out["n"] / dvb_out["n"].sum()
export(dvb_out, "s04_demo_vs_behaviour")

**Observations:**
- Demographic and behavioural risk scores are only weakly linked (rank correlation 0.20).
- Low/low clients default at 6.0% and high/high at 50.6%.
- Behaviour moves the rate far more: within the low-demographic tier it goes from 6.0% to 33.5%; demographics move it only from 6.0% to 14.0% within the low-behaviour tier.
- Story use: the headline 3x3 matrix, 'behaviour beats demographics'.

Profiles of the four corners: what does a "risky-profile but clean" client look like compared with a "safe-profile but delinquent" one?

In [ ]:
corner = np.select(
    [(d4["S4_DEMO_TIER"] == "High") & (d4["S4_BEHAV_TIER"] == "Low"),
     (d4["S4_DEMO_TIER"] == "Low") & (d4["S4_BEHAV_TIER"] == "High"),
     (d4["S4_DEMO_TIER"] == "High") & (d4["S4_BEHAV_TIER"] == "High"),
     (d4["S4_DEMO_TIER"] == "Low") & (d4["S4_BEHAV_TIER"] == "Low")],
    ["Risky profile, clean behaviour", "Safe profile, bad behaviour", "Risky profile, bad behaviour", "Safe profile, clean behaviour"],
    "Middle")
cp = d4.assign(corner=corner).groupby("corner").agg(
    n=(TARGET, "size"), default_rate=(TARGET, "mean"), age_mean=("AGE", "mean"), limit_median=("LIMIT_BAL", "median"),
    female_share=("SEX", lambda s: (s == 2).mean()), grad_share=("EDUCATION", lambda s: (s == 1).mean()),
    single_share=("MARRIAGE", lambda s: (s == 2).mean()), avg_util=("AVG_UTIL", "mean"), n_late=("N_LATE", "mean"),
    pay_to_bill=("PAY_TO_BILL_6M", "median"), n_zero_pay=("N_ZERO_PAY", "mean"), late_in_sep=("PAY_1", lambda s: (s >= 1).mean()))
cp = cp.reindex(["Safe profile, clean behaviour", "Risky profile, clean behaviour", "Middle",
                 "Safe profile, bad behaviour", "Risky profile, bad behaviour"])
display(cp)

**Observations:**
- Risky profile with clean behaviour defaults at 14.0% (n = 2,454); safe profile with bad behaviour at 33.5% (n = 2,316), 2.4 times higher.
- The risky profile is a low-limit client (median NT$ 50k, only 12% graduates); the safe profile has NT$ 210-260k limits, about 80% women and half graduates.
- Bad behaviour means 1.8-2.9 late months and 60-69% late in Sep, whatever the profile.
- Story use: two persona cards contrasting 'looks risky, pays well' with 'looks safe, pays late'.

Segment-level view at the Age × Limit grain (the most intuitive demographic grid):
* **Risky segments that are safe when behaviour is clean** — segments whose overall rate is ≥ 1.15× base, split by "clean" (no late month in 6 months) vs. "dirty".
* **Safe segments that still default** — segments whose overall rate is ≤ 0.85× base, looking only at clients late in September.

In [ ]:
seg = d4.assign(clean=(d4["N_LATE"] == 0), late_sep=(d4["PAY_1"] >= 1))
def _seg(g):
    return pd.Series({"n": len(g), "rate": g[TARGET].mean(),
                      "n_clean": int(g["clean"].sum()), "rate_clean": g.loc[g["clean"], TARGET].mean(),
                      "n_dirty": int((~g["clean"]).sum()), "rate_dirty": g.loc[~g["clean"], TARGET].mean(),
                      "n_late_sep": int(g["late_sep"].sum()), "rate_late_sep": g.loc[g["late_sep"], TARGET].mean()})
S4seg = seg.groupby(["AGE_BAND", "LIMIT_BAND"], observed=True).apply(_seg).reset_index()
S4seg["lift"] = S4seg["rate"] / BASE_RATE
for c_ in ("n", "n_clean", "n_dirty", "n_late_sep"):
    S4seg[c_] = S4seg[c_].astype(int)
risky_safe = S4seg[(S4seg["lift"] >= 1.15) & (S4seg["n_clean"] >= 100)].sort_values("rate_clean")
safe_risky = S4seg[(S4seg["lift"] <= 0.85) & (S4seg["n_late_sep"] >= 50)].sort_values("rate_late_sep", ascending=False)
print(f"Overall: clean clients default at {d4.loc[d4['N_LATE'] == 0, TARGET].mean():.1%}, late-in-Sep clients at {d4.loc[d4['PAY_1'] >= 1, TARGET].mean():.1%}")
print("RISKY segments (lift >= 1.15) - default rate among their CLEAN clients:")
display(risky_safe.head(12))
print("SAFE segments (lift <= 0.85) - default rate among their clients LATE in Sep:")
display(safe_risky.head(12))
para = pd.concat([risky_safe.assign(kind="Risky segment, clean clients"), safe_risky.assign(kind="Safe segment, late-in-Sep clients")])
para["AGE_BAND"] = para["AGE_BAND"].astype(str); para["LIMIT_BAND"] = para["LIMIT_BAND"].astype(str)
export(para, "s04_paradox_segments")

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(S4seg["rate"], S4seg["rate_clean"], s=S4seg["n_clean"] / 15, color=C_OK, alpha=.6, label="clean clients (no late month)")
ax.scatter(S4seg["rate"], S4seg["rate_late_sep"], s=S4seg["n_late_sep"] / 15, color=C_DEF, alpha=.6, label="clients late in Sep")
lim_ = [0, max(.8, float(np.nanmax(S4seg["rate_late_sep"])))]
ax.plot([0, .5], [0, .5], ls=":", color="k", lw=1)
ax.axvline(BASE_RATE, ls="--", color=C_NEU)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("segment default rate (Age band x Limit band)"); ax.set_ylabel("default rate within behaviour group")
ax.set_title("Behaviour overrides segment: within every Age x Limit segment, clean vs late-in-Sep clients")
ax.legend(); plt.tight_layout(); savefig("s04_segment_vs_behaviour_scatter"); plt.show()

**Observations:**
- Overall, clean clients (never late) default at 11.7% and clients late in Sep at 50.3%.
- In every risky age x limit segment, clean clients default at only 11.0-17.9%, close to the clean average.
- In every safe segment, Sep-late clients still default at 34.4-50.0%; for 26-30 at 300-500k it is 36.5% vs 6.9% for clean peers.
- In 46-50 at NT$ 50k or less, clean clients default at 17.0% vs 61.1% for those late in Sep.
- Story use: the closing slide, 'judge the behaviour, not the segment'.

### 4.11 Section summary
* Strongest interactions overall (4.5): September status x utilisation (AIC gain 154), late months x zero-payment months (147), limit x payment ratio (139) (number from the pre-fix run; refresh after the next Kaggle run).
* Strongest demographic interactions: none significant (age x limit p = 0.49, sex x age p = 0.30); demographic effects simply add up.
* Hidden cells (largest residuals): 300-500k limit with no bill (49.5% vs 28.0% expected) and late 2m in Apr with no zero-payment months (52.8% vs 31.7%) (number from the pre-fix run; refresh after the next Kaggle run).
* Three-way modifiers: none among demographics; limit only weakens the utilisation x status interaction above NT$ 300k.
* Behaviour vs. demographics: clean clients in risky segments default at 11-18%, while Sep-late clients in safe segments default at 34-50%.

## 5. Temporal behaviour: trajectories, transitions and roll rates
The six monthly blocks (Apr → Sep 2005) are a short **panel** (the same clients observed repeatedly over time). This section treats them as time series per client:
level trends by outcome, status composition, month-to-month **transition matrices**, credit-risk **roll / cure rates**,
a **Markov-chain** view, **trajectory archetypes** (6-letter sequences), a **Sankey** of flows into default, recency and
onset of delinquency, trend and volatility features, minimum-payer behaviour and **lag correlations**.

**Reminder:** column index 1 = Sep … 6 = Apr. Everything below is re-ordered and plotted **chronologically Apr → Sep**.

**Tableau:** `s05_client_month_panel.csv` (one row per client × month) is the main time-series source; the other
`s05_*` exports are pre-aggregated sheets (transitions, roll rates, sequences, Sankey flows).

**Section guide**

- **What it does:** Treats the six months (Apr → Sep 2005) as a time series for each client. It tracks trends, month-to-month moves between statuses, roll and cure rates, typical 6-month paths, and how recent the last late payment was.
- **Question it answers:** How do future defaulters behave over time, and when do they start to differ?
- **Inputs / new columns:** `PAY_1-6`, `BILL_AMT1-6`, `PAY_AMT1-6`, `UTIL_i`, `PAY_RATIO_i`. Adds `TRAJ_SEQ`, `TRAJ_N_CHANGES`, `TRAJ_ENTROPY`, `MONTHS_SINCE_LATE`, `FIRST_LATE_MONTH`, `*_TREND_BAND`, `BILL_CV`, `PAY_CV`, `*_CV_BAND`, `N_MINPAY`.
- **How to read the charts:** x-axis runs Apr → Sep (index 6 → 1). Red = defaulters, blue = non-defaulters. Transition heatmaps: row = this month's status, column = next month's, cell = share who move. Roll rate = share moving to a worse bucket; cure rate = share returning to current. Markov chain = a model where next month depends only on this month.
- **Key takeaways:**
  - Defaulters already differ in April (median utilisation 0.35 vs 0.15).
  - Once 2 months late, defaulters stay late 68.6% of the time vs 46.2% for others.
  - `RRRRRR` (always revolving) defaults at 10.4%; `RRRRR2` (late only in Sep) at 65.6%.
  - Never late: 11.7%; late in Sep: 50.3%. Lateness fades but never fully (18.7% after 5 months).
  - The Sep jump in "late 1m" is a coding quirk, not a real wave.
- **Where it feeds:** the trend, Sankey and "most recent move" charts; `s05_client_month_panel.csv` is the main Tableau time-series source.

### 5.0 Building blocks
Chronological status matrix (`SC`, columns Apr..Sep), status-group codes (`GC`), a 1-letter code per group used for
sequences (`N` inactive, `P` paid in full, `R` revolving, `1/2/3` late 1m/2m/3m+), and a fixed colour per group.

In [ ]:
CHRON_IDX = IDX[::-1]                                   # [6,5,4,3,2,1] -> Apr..Sep
MONTH_NUM_NAME = dict(enumerate(MONTH_ORDER, start=1))  # 1=Apr ... 6=Sep (chronological month_num)
GRP_LETTER = {"Inactive": "N", "Paid in full": "P", "Revolving": "R", "Late 1m": "1", "Late 2m": "2", "Late 3m+": "3"}
STATUS_COLORS = {"Inactive": "#bdbdbd", "Paid in full": "#2b8cbe", "Revolving": "#74c476",
                 "Late 1m": "#fdae6b", "Late 2m": "#e6550d", "Late 3m+": "#a50f15",
                 "Default": C_DEF, "No default": C_OK}
GRP_ARR = np.array(STATUS_GROUP_ORDER)

SC = df[[f"PAY_{i}" for i in CHRON_IDX]].to_numpy(int)                       # raw status, Apr..Sep
GC = np.select([SC <= -2, SC == -1, SC == 0, SC == 1, SC == 2], [0, 1, 2, 3, 4], 5)   # group code, Apr..Sep
YD = df[TARGET].to_numpy()
N_CLI = len(df)

def grp_heat(M, ax, title, fmt=".0%", cmap="Blues", annot=True, center=None, vmin=None, vmax=None,
             xlabel="to (next month)", ylabel="from"):
    sns.heatmap(pd.DataFrame(M, STATUS_GROUP_ORDER, STATUS_GROUP_ORDER), annot=annot, fmt=fmt, cmap=cmap,
                cbar=False, ax=ax, center=center, vmin=vmin, vmax=vmax, linewidths=.5, annot_kws={"fontsize": 8})
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=35); ax.tick_params(axis="y", rotation=0)

print("Group code legend:", {g: GRP_LETTER[g] for g in STATUS_GROUP_ORDER})
print("Status-group mix by month (chronological):")
mix = pd.DataFrame([np.bincount(GC[:, k], minlength=6) / N_CLI for k in range(6)], index=MONTH_ORDER, columns=STATUS_GROUP_ORDER)
display(mix.round(4))

#### 5.0.1 Coding quirk: status `1` exists (almost) only in September
Counts of each raw status code by month. If `PAY_2..PAY_6` (Aug..Apr) almost never use code `1`, then the *first* month
of delinquency is recorded as `2` in Apr–Aug but as `1` in Sep. Consequences for this section: (a) the `Late 1m`
group is essentially a Sep-only state, (b) the Aug→Sep transition looks artificially different, (c) roll rates and the
Markov chain below therefore merge `Late 1m` and `Late 2m` into **Late 1-2m**.

In [ ]:
raw_codes = pd.DataFrame({m: pd.Series(SC[:, k]).value_counts() for k, m in enumerate(MONTH_ORDER)}).fillna(0).astype(int).sort_index()
raw_codes.index.name = "status"
display(raw_codes)
print("Share of late accounts (status>=1) coded exactly 1, by month:",
      {m: f"{(SC[:, k] == 1).sum() / max(1, (SC[:, k] >= 1).sum()):.1%}" for k, m in enumerate(MONTH_ORDER)})

**Observations:**
- Status code `1` (one month late) almost never appears before September: 0 clients in Apr and May, 28 in Aug, then 3,688 in Sep.
- In Sep, 54.1% of late accounts carry code 1. In Apr–Aug it is 0.6% or less, so earlier months record lateness only as code 2 or higher.
- Because of this, the "Late 1m" share jumps from 0.09% (Aug) to 12.3% (Sep) and Inactive drops from 12.6% to 9.2%. The jump comes from the coding, not from a real wave of new delinquency.
- Story use: caveat note. It explains why later analyses merge late 1m and 2m into "Late 1-2m", and why the Sep spike should not be read as a shock.

### 5.1 Long client × month panel (main Tableau time-series source)
One row per client per month (30,000 × 6 = 180,000 rows). `MONTH_NUM` is chronological (1 = Apr … 6 = Sep).
`SPEND` and `PAY_RATIO` are undefined for Apr (they need the previous statement). `PAY_AMT` in a month settles the
previous month's bill. Static labels are repeated so Tableau needs no join.

In [ ]:
_parts = []
for m, i in enumerate(CHRON_IDX, start=1):
    _parts.append(pd.DataFrame({
        "ID": df["ID"].to_numpy(),
        "MONTH_NUM": m,
        "STATUS": df[f"PAY_{i}"].to_numpy(),
        "BILL": df[f"BILL_AMT{i}"].to_numpy(),
        "PAY_AMT": df[f"PAY_AMT{i}"].to_numpy(),
        "UTIL": df[f"UTIL_{i}"].to_numpy(),
        "SPEND": df[f"SPEND_{i}"].to_numpy() if i <= 5 else np.nan,
        "PAY_RATIO": df[f"PAY_RATIO_{i}"].to_numpy() if i <= 5 else np.nan,
    }))
panel = pd.concat(_parts, ignore_index=True)
_static = df[["ID", "LIMIT_BAL", "AGE", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND", TARGET, "DEFAULT_LABEL"]]
panel = panel.merge(_static, on="ID", how="left")
panel["MONTH"] = pd.Categorical(panel["MONTH_NUM"].map(MONTH_NUM_NAME), MONTH_ORDER, ordered=True)
panel["STATUS_GRP"] = pd.Categorical(GRP_ARR[np.select([panel["STATUS"] <= -2, panel["STATUS"] == -1, panel["STATUS"] == 0,
                                                       panel["STATUS"] == 1, panel["STATUS"] == 2], [0, 1, 2, 3, 4], 5)],
                                     STATUS_GROUP_ORDER, ordered=True)
panel["IS_LATE"] = (panel["STATUS"] >= 1).astype("int8")
panel = panel.sort_values(["ID", "MONTH_NUM"]).reset_index(drop=True)
# compact dtypes
for c, t in {"MONTH_NUM": "int8", "STATUS": "int8", TARGET: "int8", "AGE": "int16", "ID": "int32",
             "LIMIT_BAL": "int32", "BILL": "int32", "PAY_AMT": "int32"}.items():
    panel[c] = panel[c].astype(t)
for c in ("UTIL", "PAY_RATIO", "SPEND"):
    panel[c] = panel[c].astype("float32")
panel = panel[["ID", "MONTH", "MONTH_NUM", "STATUS", "STATUS_GRP", "IS_LATE", "BILL", "PAY_AMT", "UTIL", "SPEND", "PAY_RATIO",
               TARGET, "DEFAULT_LABEL", "LIMIT_BAL", "LIMIT_BAND", "AGE", "AGE_BAND", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL"]]
print(panel.shape, f"{panel.memory_usage(deep=True).sum() / 1e6:.1f} MB in memory")
_exp = panel.copy()
_exp["UTIL"] = _exp["UTIL"].round(4); _exp["PAY_RATIO"] = _exp["PAY_RATIO"].round(4); _exp["SPEND"] = _exp["SPEND"].round(0)
export(_exp, "s05_client_month_panel")
del _exp, _parts
panel.head(8)

### 5.2 Monthly level trends by outcome (mean with 95% CI band, median dashed)
Do future defaulters look different *throughout* the window, or do they diverge only near the end? Means are heavily
influenced by large accounts, so medians are overlaid.

In [ ]:
S5_METRICS = {"BILL": "Bill (NT$)", "PAY_AMT": "Payment (NT$)", "UTIL": "Utilisation", "SPEND": "Implied spend (NT$)",
              "PAY_RATIO": "Payment ratio (pay / prev bill)", "IS_LATE": "Share late (status >= 1)"}
S5_PCT = {"UTIL", "PAY_RATIO", "IS_LATE"}

def s5_monthly_stats(data, by=()):
    by = list(by)
    g = data.groupby(by + ["MONTH_NUM", TARGET], observed=True)
    out = []
    for m in S5_METRICS:
        a = g[m].agg(["mean", "median", "std", "count"]).reset_index()
        a["metric"] = m
        out.append(a)
    t = pd.concat(out, ignore_index=True)
    t["se"] = t["std"] / np.sqrt(t["count"].clip(lower=1))
    t["ci_lo"], t["ci_hi"] = t["mean"] - 1.96 * t["se"], t["mean"] + 1.96 * t["se"]
    t["MONTH"] = t["MONTH_NUM"].map(MONTH_NUM_NAME)
    t["DEFAULT_LABEL"] = t[TARGET].map({0: "No default", 1: "Default"})
    return t

mstats = s5_monthly_stats(panel)
fig, axes = plt.subplots(2, 3, figsize=(17, 8.5))
for ax, (m, lab) in zip(axes.ravel(), S5_METRICS.items()):
    t = mstats[mstats["metric"] == m]
    for d, col in ((0, C_OK), (1, C_DEF)):
        u = t[t[TARGET] == d].dropna(subset=["mean"]).sort_values("MONTH_NUM")
        ax.plot(u["MONTH_NUM"], u["mean"], color=col, marker="o", lw=2, label=("Default" if d else "No default") + " mean")
        ax.fill_between(u["MONTH_NUM"], u["ci_lo"], u["ci_hi"], color=col, alpha=.18)
        if m != "IS_LATE":
            ax.plot(u["MONTH_NUM"], u["median"], color=col, ls="--", lw=1.3, label=("Default" if d else "No default") + " median")
    ax.set_xticks(range(1, 7)); ax.set_xticklabels(MONTH_ORDER); ax.set_title(lab)
    if m in S5_PCT: ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    else: ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v/1000:,.0f}k"))
axes[0, 0].legend(fontsize=8)
fig.suptitle("Monthly trajectories Apr → Sep by outcome (solid = mean ± 95% CI, dashed = median)", fontweight="bold")
plt.tight_layout(); savefig("s05_monthly_trends_by_default"); plt.show()

wide = mstats.pivot_table(index=["metric", "DEFAULT_LABEL"], columns="MONTH", values="mean")[MONTH_ORDER]
print("Mean by month:"); display(wide.round(3))
print("Median by month:"); display(mstats.pivot_table(index=["metric", "DEFAULT_LABEL"], columns="MONTH", values="median")[MONTH_ORDER].round(3))

**Observations:**
- Future defaulters already look different in April: mean utilisation (bill ÷ limit) is 0.40 vs 0.30, and the median is 0.35 vs 0.15. The gap exists from the start of the window.
- The share of defaulters who are late climbs from 24.3% (Apr) to 51.7% (Sep). For non-defaulters it goes from 6.3% to 14.5%; part of the Sep jump is the code-1 quirk.
- Defaulters pay less: median payment NT$1,000 vs NT$1,706 in Apr. Their median payment ratio (payment ÷ previous bill) is 4–6% vs 6–9%.
- Bills are similar. The mean Sep bill is even slightly lower for defaulters (NT$48.5k vs NT$52.0k), while their median implied spend is less than half (NT$1,044 vs NT$2,505).
- Story use: opening trend chart. Defaulters owe similar amounts but use more of their line, repay less and spend less on new purchases.

### 5.3 Same trajectories by credit-limit band and age band (small multiples)
Rows = metric, columns = segment, red = defaulters, blue = non-defaulters. Shows whether the defaulter/non-defaulter gap
is uniform or concentrated in particular segments (a natural Tableau trellis with a segment filter).

In [ ]:
SM_METRICS = ["UTIL", "PAY_RATIO", "IS_LATE", "BILL", "PAY_AMT"]

def s5_small_multiples(band, order, name):
    t = s5_monthly_stats(panel, [band])
    levels = [o for o in order if o in set(t[band].astype(str))]
    fig, axes = plt.subplots(len(SM_METRICS), len(levels), figsize=(2.35 * len(levels), 2.0 * len(SM_METRICS)),
                             sharey="row", sharex=True)
    for r, m in enumerate(SM_METRICS):
        for c, lev in enumerate(levels):
            ax = axes[r, c]
            u0 = t[(t["metric"] == m) & (t[band].astype(str) == lev)]
            for d, col in ((0, C_OK), (1, C_DEF)):
                u = u0[u0[TARGET] == d].dropna(subset=["mean"]).sort_values("MONTH_NUM")
                ax.plot(u["MONTH_NUM"], u["mean"], color=col, lw=1.6, marker=".")
                ax.fill_between(u["MONTH_NUM"], u["ci_lo"], u["ci_hi"], color=col, alpha=.15)
            if r == 0: ax.set_title(lev, fontsize=10)
            if c == 0: ax.set_ylabel(S5_METRICS[m].split(" (")[0], fontsize=9)
            if m in S5_PCT: ax.yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=0))
            else: ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v/1000:,.0f}k"))
            ax.set_xticks(range(1, 7)); ax.set_xticklabels([mm[0] for mm in MONTH_ORDER], fontsize=8)
    fig.suptitle(f"Monthly means by {band} (red = default, blue = no default; bands = 95% CI)", fontweight="bold")
    plt.tight_layout(); savefig(name); plt.show()
    return t

ms_limit = s5_small_multiples("LIMIT_BAND", LIMIT_LABELS, "s05_trends_by_limit_band")

In [ ]:
ms_age = s5_small_multiples("AGE_BAND", AGE_LABELS, "s05_trends_by_age_band")

# compact print: Sep-minus-Apr change in share late, by band and outcome
def _delta(t, band):
    u = t[t["metric"] == "IS_LATE"].pivot_table(index=[band, "DEFAULT_LABEL"], columns="MONTH", values="mean", observed=True)
    return (u["Sep"] - u["Apr"]).unstack().round(3)
print("Change in share late (Sep − Apr), by limit band:"); display(_delta(ms_limit, "LIMIT_BAND"))
print("Change in share late (Sep − Apr), by age band:"); display(_delta(ms_age, "AGE_BAND"))

_m = []
for seg_type, t, band in (("All", mstats, None), ("LIMIT_BAND", ms_limit, "LIMIT_BAND"), ("AGE_BAND", ms_age, "AGE_BAND")):
    u = t.copy()
    u["segment_type"] = seg_type
    u["segment"] = "All" if band is None else u[band].astype(str)
    _m.append(u[["segment_type", "segment", "MONTH", "MONTH_NUM", TARGET, "DEFAULT_LABEL", "metric", "mean", "median", "ci_lo", "ci_hi", "count"]])
export(pd.concat(_m, ignore_index=True), "s05_monthly_metrics")

**Observations:**
- Between Apr and Sep, the share of defaulters who are late rises by 23–35 pp in every limit band; for non-defaulters it rises by only 6–9 pp. The gap opens in every segment, not just one.
- The biggest rise among defaulters is in the >500k (+34.8 pp) and 300-500k (+31.3 pp) bands: defaulters with large lines deteriorate late and fast.
- Across age bands, defaulters rise 25–29 pp, except 61+ at +42.5 pp (a small group of 272 clients).
- Story use: supporting evidence. The trend holds in every segment, so a single trend line with a segment filter works in Tableau.

### 5.4 Status composition per month by outcome (100% stacked)
How the mix of inactive / paid-in-full / revolving / late accounts evolves, separately for future defaulters and non-defaulters.

In [ ]:
comp = (panel.groupby([TARGET, "MONTH_NUM", "STATUS_GRP"], observed=False).size()
        .groupby(level=[0, 1]).transform(lambda s: s / s.sum()).rename("share").reset_index())
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, d in zip(axes, (0, 1)):
    w = comp[comp[TARGET] == d].pivot(index="MONTH_NUM", columns="STATUS_GRP", values="share")[STATUS_GROUP_ORDER]
    ax.stackplot(w.index, w.T.values, labels=STATUS_GROUP_ORDER, colors=[STATUS_COLORS[g] for g in STATUS_GROUP_ORDER], alpha=.9)
    cum = w.cumsum(axis=1)
    for g in STATUS_GROUP_ORDER:
        for mnum in w.index:
            v = w.loc[mnum, g]
            if v >= .06:
                ax.text(mnum, cum.loc[mnum, g] - v / 2, f"{v:.0%}", ha="center", va="center", fontsize=7)
    ax.set_xticks(range(1, 7)); ax.set_xticklabels(MONTH_ORDER); ax.set_xlim(1, 6)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(("Defaulters" if d else "Non-defaulters") + " — status-group mix")
axes[1].legend(loc="center left", bbox_to_anchor=(1, .5), fontsize=8)
plt.tight_layout(); savefig("s05_status_composition_by_default"); plt.show()

comp["MONTH"] = comp["MONTH_NUM"].map(MONTH_NUM_NAME)
comp["DEFAULT_LABEL"] = comp[TARGET].map({0: "No default", 1: "Default"})
display(comp.pivot_table(index=["DEFAULT_LABEL", "STATUS_GRP"], columns="MONTH", values="share", observed=True)[MONTH_ORDER].round(3))
export(comp[["MONTH", "MONTH_NUM", "DEFAULT_LABEL", "STATUS_GRP", "share"]], "s05_status_composition")

**Observations:**
- Among future defaulters, "Late 2m" grows from 21.1% (Apr) to 32.9% (Aug). Among non-defaulters it stays at 5–8%.
- The defaulters' revolving share falls from 46.2% to 28.5% by Sep, while non-defaulters stay at 55–59%. Defaulters leave normal revolving and become late.
- The paid-in-full share is steady in both groups (14–15% for defaulters vs 20–22%), so paying in full is common even among future defaulters.
- Story use: 100% stacked bar per month, a clear picture of the slide into delinquency.

### 5.5 Month-to-month transition matrices (status group, row-normalised)
Each cell = P(group next month | group this month). *Pooled* stacks the five consecutive pairs (Apr→May … Aug→Sep,
150,000 transitions). Comparing defaulters with non-defaulters shows *where* the dynamics differ (persistence of
lateness, cure probability). Note (5.0.1): the `Late 1m` *row* rests on only ~34 transitions (status 1 is Sep-only), so
read it as empty; the `Late 1m` *column* is effectively "Aug → Sep first delinquency".

In [ ]:
def trans_counts(mask=None, pairs=range(5)):
    M = np.zeros((6, 6))
    for k in pairs:
        a, b = GC[:, k], GC[:, k + 1]
        if mask is not None:
            a, b = a[mask], b[mask]
        np.add.at(M, (a, b), 1)
    return M

def row_norm(M):
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.nan_to_num(M / M.sum(1, keepdims=True))

SEGS = {"All": None, "No default": YD == 0, "Default": YD == 1}
TC = {s: trans_counts(m) for s, m in SEGS.items()}
TP = {s: row_norm(M) for s, M in TC.items()}

fig, axes = plt.subplots(1, 4, figsize=(24, 5.6))
for ax, s in zip(axes[:3], SEGS):
    grp_heat(TP[s], ax, f"Pooled transitions — {s}", vmin=0, vmax=1)
grp_heat(TP["Default"] - TP["No default"], axes[3], "Default − No default (pp)", fmt="+.0%", cmap=CMAP_DIV, center=0, vmin=-.4, vmax=.4)
plt.tight_layout(); savefig("s05_transition_matrices_pooled"); plt.show()

print("Diagonal persistence (P(stay in same group)):")
display(pd.DataFrame({s: np.diag(TP[s]) for s in SEGS}, index=STATUS_GROUP_ORDER).round(4))

**Observations:**
- Revolving is the most stable state: 88.9% of revolving clients are still revolving the next month (90.8% of non-defaulters, 79.9% of defaulters).
- Being 2 months late is far stickier for defaulters: 68.6% stay Late 2m vs 46.2% of non-defaulters. Non-defaulters get out of arrears more often.
- Late 1m shows 100% persistence only because code 1 exists almost only in Sep (5.0.1). Ignore that cell.
- Story use: transition heatmap with the message "once late, defaulters stay late".

#### 5.5.1 Is the chain time-homogeneous? One matrix per consecutive month pair (all clients)
If the five matrices look alike the pooled matrix (and the Markov model in 5.8) is a fair summary; differences reveal
seasonality or reporting artefacts.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(27, 5.2))
PAIR_P = []
for k, ax in enumerate(axes):
    Pk = row_norm(trans_counts(pairs=[k]))
    PAIR_P.append(Pk)
    grp_heat(Pk, ax, f"{MONTH_ORDER[k]} → {MONTH_ORDER[k+1]}", vmin=0, vmax=1,
             ylabel="from" if k == 0 else "", xlabel="to")
plt.tight_layout(); savefig("s05_transition_matrices_by_pair"); plt.show()

# total-variation distance of each pair's rows from the pooled rows, weighted by row mass
tvd = []
for k, Pk in enumerate(PAIR_P):
    w = np.bincount(GC[:, k], minlength=6) / N_CLI
    row_tvd = 0.5 * np.abs(Pk - TP["All"]).sum(1)
    tvd.append({"pair": f"{MONTH_ORDER[k]}→{MONTH_ORDER[k+1]}", "weighted_TVD_vs_pooled": float((w * row_tvd).sum()),
                **{f"TVD_{g}": float(r) for g, r in zip(STATUS_GROUP_ORDER, row_tvd)}})
display(pd.DataFrame(tvd).round(3))

# tidy export: segment x pair (incl. pooled) x from x to
_rows = []
for s, m in SEGS.items():
    for k in list(range(5)) + ["pooled"]:
        M = trans_counts(m, pairs=range(5) if k == "pooled" else [k])
        R = row_norm(M)
        lab = "Pooled" if k == "pooled" else f"{MONTH_ORDER[k]}→{MONTH_ORDER[k+1]}"
        for a in range(6):
            for b in range(6):
                _rows.append({"segment": s, "pair": lab, "from_month": "All" if k == "pooled" else MONTH_ORDER[k],
                              "to_month": "All" if k == "pooled" else MONTH_ORDER[k + 1],
                              "from_grp": STATUS_GROUP_ORDER[a], "to_grp": STATUS_GROUP_ORDER[b],
                              "n": int(M[a, b]), "from_total": int(M[a].sum()), "row_share": float(R[a, b])})
transitions_tidy = pd.DataFrame(_rows)
export(transitions_tidy, "s05_transitions")

**Observations:**
- The Apr→May through Jul→Aug matrices are close to the pooled matrix. Their weighted TVD (total variation distance, where 0 = identical) is 0.039–0.048.
- Aug→Sep is the outlier at TVD 0.111. It is driven by the Inactive (0.266) and Late 2m (0.348) rows, again because of the code-1 quirk.
- Story use: background only. It justifies the pooled Markov model, with a caveat about September.

### 5.6 Default rate by the last observed transition (Aug group → Sep group)
The most recent move is the freshest signal. Cells show default rate (and n); cells with n < 30 are blanked.

In [ ]:
a, b = GC[:, 4], GC[:, 5]
last_tr = pd.DataFrame({"from_grp": GRP_ARR[a], "to_grp": GRP_ARR[b], TARGET: YD})
lt = rate_table(last_tr, ["from_grp", "to_grp"])
rate_mat = lt.pivot(index="from_grp", columns="to_grp", values="rate").reindex(index=STATUS_GROUP_ORDER, columns=STATUS_GROUP_ORDER)
n_mat = lt.pivot(index="from_grp", columns="to_grp", values="n").reindex(index=STATUS_GROUP_ORDER, columns=STATUS_GROUP_ORDER).fillna(0)
ann = rate_mat.copy().astype(object)
for r in STATUS_GROUP_ORDER:
    for c in STATUS_GROUP_ORDER:
        ann.loc[r, c] = "" if n_mat.loc[r, c] < 30 else f"{rate_mat.loc[r, c]:.0%}\n(n={int(n_mat.loc[r, c]):,})"
fig, ax = plt.subplots(figsize=(9, 6.5))
sns.heatmap(rate_mat.where(n_mat >= 30), annot=ann, fmt="", cmap=CMAP_RATE, vmin=0, vmax=.8, ax=ax, linewidths=.5,
            annot_kws={"fontsize": 8}, cbar_kws={"format": mtick.PercentFormatter(1), "label": "default rate"})
ax.set_title("Default rate by Aug → Sep status-group transition"); ax.set_xlabel("Sep group"); ax.set_ylabel("Aug group")
plt.tight_layout(); savefig("s05_default_by_last_transition"); plt.show()
_gk = {g: k for k, g in enumerate(STATUS_GROUP_ORDER)}
lt["direction"] = np.sign(lt["to_grp"].map(_gk) - lt["from_grp"].map(_gk)).map({-1: "improved", 0: "same", 1: "worsened"})
display(lt[lt["n"] >= 30].sort_values("rate", ascending=False)[["from_grp", "to_grp", "direction", "n", "defaults", "rate", "ci_lo", "ci_hi", "lift"]].head(20))
export(lt, "s05_default_by_last_transition")

**Observations:**
- The worst last move is Late 2m → Late 3m+ (n=272): 75.7% default, 3.4x the 22.1% base rate. Staying Late 2m (n=1,591) gives 71.8%.
- A fresh slip from Revolving to Late 2m in Sep (n=865) already defaults at 68.8%. One bad month after steady revolving is almost as bad as long delinquency.
- Recovering helps, but only partly: Late 2m → Paid in full still defaults at 35.4%, and Late 2m → Late 1m at 41.6%.
- The safest moves are Paid in full → Inactive (8.8%) and Revolving → Revolving (12.8%, n=14,259).
- Story use: key "most recent move" chart. It supports an early-warning rule based on the Aug→Sep transition.

### 5.7 Roll-rate and cure-rate analysis (credit-risk standard)
Because status `1` is (almost) only used in September (5.0.1), the buckets are **Current** (status ≤ 0),
**Late 1-2m** (status 1 or 2) and **Late 3m+**. For each month pair and starting bucket: *roll-forward* = moves to a
worse bucket, *stay*, *partial cure* = better but still late (3m+ → 1-2m), *cure* = back to Current.
Sep is also shown on the raw 4-bucket scale in the artefact check at the end.

In [ ]:
BK = np.select([SC <= 0, SC <= 2], [0, 1], 2)        # 0 current, 1 late 1-2m, 2 late 3m+  (Apr..Sep)
BK_LAB = ["Current", "Late 1-2m", "Late 3m+"]
rr_rows = []
for seg, m in SEGS.items():
    for k in list(range(5)) + ["pooled"]:
        ks = range(5) if k == "pooled" else [k]
        fr = np.concatenate([BK[:, j] if m is None else BK[m, j] for j in ks])
        to = np.concatenate([BK[:, j + 1] if m is None else BK[m, j + 1] for j in ks])
        for bk in range(3):
            sel = fr == bk
            n = int(sel.sum())
            if n == 0: continue
            t = to[sel]
            rr_rows.append({"segment": seg, "pair": "Pooled" if k == "pooled" else f"{MONTH_ORDER[k]}→{MONTH_ORDER[k+1]}",
                            "pair_num": 6 if k == "pooled" else k + 1, "from_bucket": BK_LAB[bk], "n": n,
                            "roll_forward": float((t > bk).mean()), "roll_next_bucket": float((t == bk + 1).mean()) if bk < 2 else np.nan,
                            "stay": float((t == bk).mean()), "partial_cure": float(((t > 0) & (t < bk)).mean()),
                            "cure": float((t == 0).mean()) if bk > 0 else np.nan})
roll = pd.DataFrame(rr_rows)
print("Pooled roll / cure rates by segment:")
display(roll[roll["pair"] == "Pooled"].set_index(["segment", "from_bucket"])[["n", "roll_forward", "stay", "partial_cure", "cure"]].round(4))
print("All clients, per month pair:")
display(roll[(roll["segment"] == "All") & (roll["pair"] != "Pooled")].pivot(index="from_bucket", columns="pair_num",
        values=["roll_forward", "cure"]).reindex(BK_LAB).round(3))

fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
pair_labels = [f"{MONTH_ORDER[k]}→{MONTH_ORDER[k+1]}" for k in range(5)]
for seg, ls in (("No default", "--"), ("Default", "-")):
    ra = roll[(roll["segment"] == seg) & (roll["pair"] != "Pooled")]
    for bk, col in zip(BK_LAB, ["#31a354", "#e6550d", "#a50f15"]):
        u = ra[ra["from_bucket"] == bk].sort_values("pair_num")
        if bk != "Late 3m+":
            axes[0].plot(u["pair_num"], u["roll_forward"], marker="o", color=col, ls=ls, label=f"{bk} · {seg}")
        if bk != "Current":
            axes[1].plot(u["pair_num"], u["cure"], marker="o", color=col, ls=ls, label=f"{bk} · {seg}")
for ax, t in zip(axes[:2], ["Roll-forward rate (to a worse bucket)", "Cure rate (back to Current)"]):
    ax.set_xticks(range(1, 6)); ax.set_xticklabels(pair_labels); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(t + "\nsolid = defaulters, dashed = non-defaulters"); ax.legend(fontsize=7)
rp = roll[roll["pair"] == "Pooled"].pivot(index="from_bucket", columns="segment", values="cure").reindex(BK_LAB[1:])[["No default", "Default"]]
rp.plot.bar(ax=axes[2], color=[C_OK, C_DEF], rot=0)
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[2].set_title("Pooled cure rate by outcome"); axes[2].set_xlabel("")
plt.tight_layout(); savefig("s05_roll_cure_rates"); plt.show()

# flow-rate chain: P(current -> late 1-2m) * P(late 1-2m -> 3m+)
pr = roll[(roll["segment"] == "All") & (roll["pair"] == "Pooled")].set_index("from_bucket")
chain = pr.loc["Current", "roll_next_bucket"] * pr.loc["Late 1-2m", "roll_next_bucket"]
print(f"Flow-rate chain Current → Late 1-2m → Late 3m+ (pooled, consecutive months): {chain:.3%}")

# artefact check on raw codes: first delinquency is recorded as 2 (skipping 1) in Apr-Aug, but as 1 in Sep
jumps = pd.DataFrame({"pair": pair_labels,
                      "current_to_status1": [float(((SC[:, k] <= 0) & (SC[:, k + 1] == 1)).mean()) for k in range(5)],
                      "current_to_status2": [float(((SC[:, k] <= 0) & (SC[:, k + 1] == 2)).mean()) for k in range(5)],
                      "current_to_status3plus": [float(((SC[:, k] <= 0) & (SC[:, k + 1] >= 3)).mean()) for k in range(5)],
                      "status2_to_status4plus": [float(((SC[:, k] == 2) & (SC[:, k + 1] >= 4)).mean()) for k in range(5)]})
print("Raw-code jumps (share of all clients) — in true delinquency counting, Current can only move to 1:"); display(jumps.round(4))
export(roll, "s05_roll_rates")

**Observations:**
- Pooled roll-forward (moving one bucket worse) is 6.1% from Current and 6.3% from Late 1-2m. The cure rate (returning to Current) is 25.3% from Late 1-2m but only 10.7% from Late 3m+.
- Future defaulters roll forward from Current 3x as often (14.0% vs 4.4%). They cure from Late 1-2m less than half as often (15.7% vs 36.1%).
- Roll-forward from Current rises every month, from 3.2% (Apr→May) to 11.1% (Aug→Sep). Part of the Sep figure is the code-1 recoding: 6.1% of clients jump straight to status 1.
- Each month, 2.9–5.4% of clients jump from current straight to status 2, which true months-past-due counting does not allow. The status codes are not strict delinquency counters.
- Story use: standard credit-risk roll/cure chart with the message "defaulters fall faster and recover slower".

### 5.8 Markov-chain view
Treat the pooled transition matrix $P$ as a first-order, time-homogeneous Markov chain. To avoid the Sep-only
`Late 1m` coding (which makes that state spuriously absorbing), the chain uses **five states**: Inactive, Paid in full,
Revolving, **Late 1-2m**, Late 3m+.
* **Stationary distribution** $\pi = \pi P$: the long-run mix the portfolio drifts towards if behaviour stays as observed.
* **k-step projections** from the April mix ($\pi_{Apr} P^k$) compared with observed monthly mixes; one-step from August vs observed September.
* **Markov-implied default**: chain each April state forward to September, then apply the observed default rate per September state; compare with the observed default rate by April state. Gaps = memory beyond one month (the process is not truly Markov).

In [ ]:
MK_STATES = ["Inactive", "Paid in full", "Revolving", "Late 1-2m", "Late 3m+"]
MK_COL = [STATUS_COLORS["Inactive"], STATUS_COLORS["Paid in full"], STATUS_COLORS["Revolving"], STATUS_COLORS["Late 2m"], STATUS_COLORS["Late 3m+"]]
G5 = np.array([0, 1, 2, 3, 3, 4])[GC]                                   # merged state codes, Apr..Sep

def trans5(pairs=range(5)):
    M = np.zeros((5, 5))
    for k in pairs:
        np.add.at(M, (G5[:, k], G5[:, k + 1]), 1)
    return row_norm(M)
P_ALL = trans5()
PAIR_P5 = [trans5([k]) for k in range(5)]
P_PRE = trans5(range(4))                                                # Apr..Aug only (no Sep coding break)
print("Pooled 5-state transition matrix:"); display(pd.DataFrame(P_ALL, MK_STATES, MK_STATES).round(3))

def stationary(P):
    w, v = np.linalg.eig(P.T)
    s = np.real(v[:, np.argmin(np.abs(w - 1))]); return s / s.sum(), w
st, w_eig = stationary(P_ALL)
st_pre, _ = stationary(P_PRE)
OBS_MIX = np.array([np.bincount(G5[:, k], minlength=5) / N_CLI for k in range(6)])        # Apr..Sep
proj = np.array([OBS_MIX[0] @ np.linalg.matrix_power(P_ALL, k) for k in range(6)])
fwd = np.array([OBS_MIX[5] @ np.linalg.matrix_power(P_ALL, k) for k in range(1, 13)])     # beyond Sep (hypothetical)

cmp = pd.DataFrame({"observed_Apr": OBS_MIX[0], "observed_Sep": OBS_MIX[5], "one_step_from_Aug": OBS_MIX[4] @ P_ALL,
                    "five_step_from_Apr": proj[5], "Sep+12_projection": fwd[-1], "stationary": st,
                    "stationary_AprAug_fit": st_pre}, index=MK_STATES)
display(cmp.round(4))
tv = pd.Series({MONTH_ORDER[k]: 0.5 * np.abs(proj[k] - OBS_MIX[k]).sum() for k in range(6)}, name="TVD projected vs observed")
print("Total-variation distance, projection from Apr vs observed:", tv.round(4).to_dict())
lam = np.sort(np.abs(w_eig))[::-1]
print("Eigenvalues of P (|λ|):", np.round(lam, 3), "-> relaxation time ~", f"{-1/np.log(lam[1]):.1f} months")

fig, axes = plt.subplots(1, 2, figsize=(17, 5.2), gridspec_kw={"width_ratios": [1.5, 1]})
xx = np.arange(1, 7); xf = np.arange(7, 19)
for g_i, g in enumerate(MK_STATES):
    c = MK_COL[g_i]
    axes[0].plot(xx, OBS_MIX[:, g_i], color=c, marker="o", lw=2, label=g)
    axes[0].plot(xx, proj[:, g_i], color=c, ls="--", lw=1.3)
    axes[0].plot(np.r_[6, xf], np.r_[OBS_MIX[5, g_i], fwd[:, g_i]], color=c, ls=":", lw=1.3)
    axes[0].scatter([19.5], [st[g_i]], color=c, marker="D")
axes[0].axvline(6.5, color="k", lw=.8)
axes[0].set_xticks(list(xx) + [12, 18, 19.5]); axes[0].set_xticklabels(MONTH_ORDER + ["Sep+6", "Sep+12", "π"])
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].legend(fontsize=8, ncol=2)
axes[0].set_title("Observed mix (solid) vs Markov from Apr (dashed); forward from Sep (dotted); ◆ stationary", fontsize=10)
cmp[["observed_Sep", "one_step_from_Aug", "five_step_from_Apr", "stationary"]].plot.bar(ax=axes[1], rot=30,
    color=["#252525", "#6baed6", "#9ecae1", "#fd8d3c"])
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("September mix: observed vs Markov")
plt.tight_layout(); savefig("s05_markov_projection"); plt.show()

# Markov-implied default by April state
d_sep = np.array([YD[G5[:, 5] == g].mean() for g in range(5)])
P5_h = np.linalg.matrix_power(P_ALL, 5)
P5_inh = PAIR_P5[0] @ PAIR_P5[1] @ PAIR_P5[2] @ PAIR_P5[3] @ PAIR_P5[4]
mk = pd.DataFrame({"n_Apr": np.bincount(G5[:, 0], minlength=5),
                   "observed_default_rate": [YD[G5[:, 0] == g].mean() for g in range(5)],
                   "markov_homogeneous": P5_h @ d_sep,
                   "markov_month_specific": P5_inh @ d_sep}, index=MK_STATES)
mk["gap_obs_minus_markov"] = mk["observed_default_rate"] - mk["markov_month_specific"]
print("Default rate by April state: observed vs Markov-implied (observed default rate per Sep state applied at the end):")
display(mk.round(4))

**Observations:**
- The 5-state chain settles quickly: the second eigenvalue is 0.824, so the relaxation time (time to forget the starting mix) is about 5.2 months.
- The stationary (long-run) late share is 20.4%, against 22.7% observed in Sep. Projections from April track the observed mix within TVD 0.03–0.045.
- A memoryless chain badly underestimates clients who were late in April. Late 1-2m in Apr defaults at 50.7% observed vs 27.0% implied, and Late 3m+ at 67.1% vs 31.2%. Past lateness adds risk beyond the current state.
- For clients current in April, the Markov estimate is close (Revolving: 18.8% observed vs 19.7% implied).
- Story use: technical support for the message "history matters: past lateness raises risk beyond today's status".

### 5.9 Trajectory archetypes (6-month status sequences)
Each client's Apr→Sep path is encoded as a 6-character string (e.g. `RRRRRR` = always revolving, `PPPP11` = paid in
full then late in Aug and Sep). New columns: `TRAJ_SEQ`, `TRAJ_N_CHANGES` (number of month-to-month group changes),
`TRAJ_ENTROPY` (Shannon entropy, bits, of the group distribution over the six months: 0 = same group every month, higher = more mixed path).

In [ ]:
LETTERS = np.array([GRP_LETTER[g] for g in STATUS_GROUP_ORDER])
df["TRAJ_SEQ"] = ["".join(r) for r in LETTERS[GC]]
df["TRAJ_N_CHANGES"] = (np.diff(GC, axis=1) != 0).sum(1)
_frac = np.stack([(GC == c).sum(1) for c in range(6)], 1) / 6
df["TRAJ_ENTROPY"] = np.round(-(np.where(_frac > 0, _frac * np.log2(np.where(_frac > 0, _frac, 1)), 0)).sum(1), 3)

seq_tab = rate_table(df, "TRAJ_SEQ").sort_values("n", ascending=False).reset_index(drop=True)
seq_tab["rank"] = np.arange(1, len(seq_tab) + 1)
seq_tab["cum_share_clients"] = seq_tab["share_clients"].cumsum()
seq_tab["n_changes"] = seq_tab["TRAJ_SEQ"].map(lambda s: sum(x != y for x, y in zip(s, s[1:])))
seq_tab["n_late_months"] = seq_tab["TRAJ_SEQ"].map(lambda s: sum(ch in "123" for ch in s))
seq_tab["worst"] = seq_tab["TRAJ_SEQ"].map(lambda s: max(s, key=lambda ch: "NPR123".index(ch)))
print(f"Distinct sequences: {len(seq_tab):,} | top 25 cover {seq_tab['cum_share_clients'].iloc[24]:.1%} of clients | "
      f"singletons: {(seq_tab['n'] == 1).sum():,}")
top25 = seq_tab.head(25)
display(top25[["rank", "TRAJ_SEQ", "n", "share_clients", "cum_share_clients", "defaults", "rate", "ci_lo", "ci_hi", "lift", "n_changes"]])

fig, ax = plt.subplots(figsize=(11, 9))
norm = plt.Normalize(0, max(.6, top25["rate"].max()))
cols = plt.get_cmap(CMAP_RATE)(norm(top25["rate"]))
ax.barh(np.arange(25), top25["n"], color=cols)
for y, (n_, r_) in enumerate(zip(top25["n"], top25["rate"])):
    ax.text(n_, y, f"  {r_:.0%}", va="center", fontsize=8)
ax.set_yticks(np.arange(25)); ax.set_yticklabels(top25["TRAJ_SEQ"], family="monospace"); ax.invert_yaxis()
ax.set_xlabel("clients"); ax.set_title("Top 25 status trajectories Apr→Sep (colour & label = default rate)\nN inactive · P paid full · R revolving · 1/2/3 late months")
sm = plt.cm.ScalarMappable(norm=norm, cmap=CMAP_RATE); plt.colorbar(sm, ax=ax, format=mtick.PercentFormatter(1), label="default rate")
plt.tight_layout(); savefig("s05_top25_sequences"); plt.show()

print("Riskiest sequences with n >= 50:")
display(seq_tab[seq_tab["n"] >= 50].sort_values("rate", ascending=False).head(15)[["TRAJ_SEQ", "n", "rate", "ci_lo", "ci_hi", "lift"]])
export(seq_tab.head(100)[["rank", "TRAJ_SEQ", "n", "defaults", "rate", "ci_lo", "ci_hi", "lift", "share_clients",
                          "cum_share_clients", "n_changes", "n_late_months", "worst"]], "s05_top_sequences")

**Observations:**
- Clients follow 986 distinct 6-month paths, and the top 25 cover 67.9% of clients. `RRRRRR` (always revolving) alone covers 32.7% of clients and defaults at 10.4%.
- Stable paths are safe: `NNNNNN` 13.5%, `PPPPPP` 14.3%. Always late (`222222`, n=530) defaults at 77.6%.
- One late month at the end changes the picture: `RRRRR2` (n=517) defaults at 65.6% vs 10.4% for `RRRRRR`, more than 6x higher.
- The riskiest path with n ≥ 50 is `333223` at 86.5% (n=52). `NNNNN1` (inactive, then code 1 in Sep, n=651) defaults at 35.9%.
- Story use: archetype bar chart with the headline "same five months, one bad month = 6x the risk".

#### 5.9.1 Sequence index plot and instability vs default
Left: a sample of 4,000 clients drawn as coloured rows (one cell per month), sorted by sequence, split by outcome — the
"texture" of the two populations. Right: default rate by number of state changes and by sequence entropy.

In [ ]:
from matplotlib.colors import ListedColormap
cm_grp = ListedColormap([STATUS_COLORS[g] for g in STATUS_GROUP_ORDER])
fig = plt.figure(figsize=(18, 9))
gs = fig.add_gridspec(2, 3, width_ratios=[1, 1, 1.6])
samp = RNG.choice(N_CLI, 4000, replace=False)
for j, d in enumerate((0, 1)):
    ax = fig.add_subplot(gs[:, j])
    idx = samp[YD[samp] == d]
    order = np.lexsort(GC[idx][:, ::-1].T)
    ax.imshow(GC[idx][order], aspect="auto", cmap=cm_grp, vmin=-.5, vmax=5.5, interpolation="nearest")
    ax.set_xticks(range(6)); ax.set_xticklabels(MONTH_ORDER); ax.set_yticks([])
    ax.set_title(f"{'Defaulters' if d else 'Non-defaulters'} (n={len(idx):,} sampled)")
handles = [plt.Rectangle((0, 0), 1, 1, color=STATUS_COLORS[g]) for g in STATUS_GROUP_ORDER]
fig.legend(handles, STATUS_GROUP_ORDER, loc="lower left", ncol=6, fontsize=8)
ax = fig.add_subplot(gs[0, 2]); t_ch = plot_rate(df, "TRAJ_N_CHANGES", ax=ax, title="Default rate by number of status-group changes")
ax = fig.add_subplot(gs[1, 2]); t_en = plot_rate(df, "TRAJ_ENTROPY", ax=ax, title="Default rate by sequence entropy (bits)", rotate=45, min_n=30)
plt.tight_layout(); savefig("s05_sequence_index_and_instability"); plt.show()
display(t_ch[["TRAJ_N_CHANGES", "n", "rate", "ci_lo", "ci_hi", "lift"]])
print("Spearman corr with DEFAULT:", {c: round(stats.spearmanr(df[c], df[TARGET])[0], 3) for c in ["TRAJ_N_CHANGES", "TRAJ_ENTROPY"]})
# instability among clients who were ever late vs never late
df["_EVER_LATE"] = np.where(df["N_LATE"] > 0, "Ever late", "Never late")
display(rate_table(df, ["_EVER_LATE", "TRAJ_N_CHANGES"], min_n=30).pivot(index="TRAJ_N_CHANGES", columns="_EVER_LATE", values="rate").round(3))
df.drop(columns="_EVER_LATE", inplace=True)

**Observations:**
- Clients whose status never changes default at 14.0% (n=14,478). One change raises this to 30.3%, and five changes to 52.9% (n=51).
- This is mostly a lateness effect. Among never-late clients the rate stays at 11–15% whatever the number of changes. Among ever-late clients, zero changes (stuck late) is the worst case at 76.6%.
- The Spearman correlation with default is modest: 0.179 for the number of changes and 0.189 for entropy (a measure of how mixed the path is).
- Story use: background only. Instability is mostly a proxy for having been late.

### 5.10 Sankey: status-group flows Apr → Jun → Aug → Sep → outcome
Interactive Plotly Sankey (if Plotly is available; also saved as HTML) plus a static matplotlib alluvial fallback.
The flow table `s05_sankey_flows.csv` is in the source/target/value shape Tableau Sankey templates expect.

In [ ]:
SK_STAGES = [(0, "Apr"), (2, "Jun"), (4, "Aug"), (5, "Sep")]
stage_codes = [GRP_ARR[GC[:, k]] for k, _ in SK_STAGES] + [np.where(YD == 1, "Default", "No default")]
stage_names = [n for _, n in SK_STAGES] + ["Oct outcome"]
node_orders = [STATUS_GROUP_ORDER] * 4 + [["No default", "Default"]]

flows = []
for s in range(len(stage_codes) - 1):
    f = pd.crosstab(stage_codes[s], stage_codes[s + 1])
    for a_ in f.index:
        for b_ in f.columns:
            if f.loc[a_, b_] > 0:
                flows.append({"step": s + 1, "source_stage": stage_names[s], "source": a_, "target_stage": stage_names[s + 1],
                              "target": b_, "value": int(f.loc[a_, b_]),
                              "default_rate_of_flow": float(YD[(stage_codes[s] == a_) & (stage_codes[s + 1] == b_)].mean())})
flows = pd.DataFrame(flows)
export(flows, "s05_sankey_flows")

try:
    import plotly.graph_objects as go
    labels, node_id, node_col = [], {}, []
    for s, nm in enumerate(stage_names):
        for g in node_orders[s]:
            node_id[(s, g)] = len(labels); labels.append(f"{nm}: {g}"); node_col.append(STATUS_COLORS[g])
    def _rgba(h, a=.35):
        h = h.lstrip("#"); return f"rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{a})"
    src = [node_id[(r.step - 1, r.source)] for r in flows.itertuples()]
    tgt = [node_id[(r.step, r.target)] for r in flows.itertuples()]
    fig_sk = go.Figure(go.Sankey(arrangement="snap",
        node=dict(label=labels, color=node_col, pad=12, thickness=16),
        link=dict(source=src, target=tgt, value=flows["value"].tolist(),
                  color=[_rgba(STATUS_COLORS[s_]) for s_ in flows["source"]],
                  customdata=(flows["default_rate_of_flow"] * 100).round(1).tolist(),
                  hovertemplate="%{source.label} → %{target.label}<br>n=%{value:,}<br>default rate of flow: %{customdata}%<extra></extra>")))
    fig_sk.update_layout(title="Status-group flows Apr → Jun → Aug → Sep → default outcome", height=650, font_size=11)
    fig_sk.write_html(str(FIG_DIR / "s05_sankey.html"))
    fig_sk.show()
except Exception as e:
    print("Plotly Sankey skipped:", repr(e))

In [ ]:
def alluvial(stage_codes, stage_names, node_orders, ax, gap_frac=.012, w=.16):
    n = len(stage_codes[0]); gap = gap_frac * n; xs = np.arange(len(stage_codes), dtype=float)
    pos = []
    for s, codes in enumerate(stage_codes):
        vc = pd.Series(codes).value_counts(); y = 0.0; d = {}
        for lab in node_orders[s]:
            if lab in vc.index:
                d[lab] = (y, y + vc[lab]); y += vc[lab] + gap
        pos.append(d)
        for lab, (y0, y1) in d.items():
            ax.add_patch(plt.Rectangle((xs[s] - w / 2, y0), w, y1 - y0, color=STATUS_COLORS[lab], ec="k", lw=.4, zorder=3))
            if (y1 - y0) > .02 * n:
                ax.text(xs[s], (y0 + y1) / 2, f"{lab}\n{(y1 - y0) / n:.0%}", ha="center", va="center", fontsize=7, zorder=4)
    tt = np.linspace(0, 1, 60); sm_ = 3 * tt**2 - 2 * tt**3
    for s in range(len(stage_codes) - 1):
        f = pd.crosstab(stage_codes[s], stage_codes[s + 1])
        out_off = {k: v[0] for k, v in pos[s].items()}; in_off = {k: v[0] for k, v in pos[s + 1].items()}
        for a_ in node_orders[s]:
            for b_ in node_orders[s + 1]:
                v = f.loc[a_, b_] if (a_ in f.index and b_ in f.columns) else 0
                if v == 0: continue
                ya, yb = out_off[a_], in_off[b_]; out_off[a_] += v; in_off[b_] += v
                x = xs[s] + w / 2 + tt * (1 - w)
                lo = ya + (yb - ya) * sm_
                ax.fill_between(x, lo, lo + v, color=STATUS_COLORS[a_], alpha=.35, lw=0, zorder=1)
    ax.set_xlim(-.4, xs[-1] + .4); ax.set_ylim(max(max(v[1] for v in p.values()) for p in pos) + gap, -gap)
    ax.set_xticks(xs); ax.set_xticklabels(stage_names); ax.set_yticks([]); ax.grid(False)
    for sp in ax.spines.values(): sp.set_visible(False)

fig, ax = plt.subplots(figsize=(16, 8.5))
alluvial(stage_codes, stage_names, node_orders, ax)
ax.set_title("Alluvial (static fallback): status-group flows Apr → Jun → Aug → Sep → outcome")
plt.tight_layout(); savefig("s05_sankey_static"); plt.show()

print("Where do defaulters come from? Sep group composition of defaulters vs default rate by Sep group:")
display(flows[flows["step"] == 4].pivot(index="source", columns="target", values="value").reindex(STATUS_GROUP_ORDER)
        .assign(rate=lambda t: t["Default"] / t.sum(1), share_of_defaulters=lambda t: t["Default"] / t["Default"].sum()).round(3))

**Observations:**
- The late states in Sep hold 51.7% of all defaulters (Late 1m 18.9%, Late 2m 27.8%, Late 3m+ 5.0%) but only 22.7% of clients.
- Revolving in Sep is still the largest single source of defaulters: 1,888 (28.5%) at a low 12.8% rate, because 14,737 clients sit there.
- Paid in full in Sep contributes 954 defaulters (14.4%) at a 16.8% rate.
- Story use: the Sankey is a strong visual. Half of the defaulters were visibly late; the other half came from normal-looking accounts.

### 5.11 Recency and onset of delinquency
* `MONTHS_SINCE_LATE`: months between the last late month (status ≥ 1) and Sep (0 = late in Sep … 5 = last late in Apr; **7 = never late**).
* `FIRST_LATE_MONTH`: first month in Apr→Sep with status ≥ 1 (`Never` if none).
The heatmap crosses onset with recency: early-onset-then-recovered vs recent-onset clients.

In [ ]:
late_c = SC >= 1                                              # Apr..Sep
ever = late_c.any(1)
df["MONTHS_SINCE_LATE"] = np.where(ever, np.argmax(late_c[:, ::-1], axis=1), 7)
df["FIRST_LATE_MONTH"] = pd.Categorical(np.where(ever, np.array(MONTH_ORDER)[np.argmax(late_c, axis=1)], "Never"),
                                        MONTH_ORDER + ["Never"], ordered=True)
fig, axes = plt.subplots(1, 3, figsize=(20, 4.8))
t_ms = plot_rate(df, "MONTHS_SINCE_LATE", ax=axes[0], title="Default rate by months since last late month (7 = never)")
t_fl = plot_rate(df, "FIRST_LATE_MONTH", ax=axes[1], order=MONTH_ORDER + ["Never"], title="Default rate by first late month")
rt = rate_table(df, ["FIRST_LATE_MONTH", "MONTHS_SINCE_LATE"], min_n=30)
hm = rt.pivot(index="FIRST_LATE_MONTH", columns="MONTHS_SINCE_LATE", values="rate")
sns.heatmap(hm, annot=True, fmt=".0%", cmap=CMAP_RATE, vmin=0, vmax=.8, ax=axes[2], cbar=False, linewidths=.5)
axes[2].set_title("Default rate: onset × recency (n ≥ 30)"); axes[2].set_xlabel("months since last late"); axes[2].set_ylabel("first late month")
plt.tight_layout(); savefig("s05_recency_onset"); plt.show()
display(t_ms[["MONTHS_SINCE_LATE", "n", "rate", "ci_lo", "ci_hi", "lift"]])
display(t_fl[["FIRST_LATE_MONTH", "n", "rate", "ci_lo", "ci_hi", "lift"]])
export(rt, "s05_onset_recency_rates")

**Observations:**
- Clients late in Sep default at 50.3% (n=6,818), 2.3x the base rate. Risk fades with time since the last late month: 38.0% after 1 month, 28.7% after 2, 18.7% after 5. Never-late clients default at 11.7% (n=19,931).
- Earlier onset means higher risk: clients first late in Apr default at 52.3% (n=3,079), those first late in Sep at 36.1%. Earlier onset means more months in trouble. Caveat: Sep onset is mostly code 1 and Apr onset is code 2. In September, code 1 mixes clients who were paid-up or inactive in August with clients who were 2+ months late in August; it is not a uniform first delay. Compare Apr-Aug onsets with each other for a clean onset effect.
- A client last late 5 months ago (18.7%) is below base but still well above never-late (11.7%).
- Story use: recency-decay line chart with the message "lateness fades, but never fully".

### 5.12 Deterioration vs improvement: trend features
`DELAY_TREND` (status Sep − Apr) and OLS slopes (fitted straight-line trends) `UTIL_TREND`, `BILL_TREND`, `PAY_TREND` (per month, Apr→Sep) are banded
into *strong down / down / flat / up / strong up* (flat = within a small tolerance; up/down split at the median of the
non-flat side). New columns: `DELAY_TREND_BAND`, `UTIL_TREND_BAND`, `BILL_TREND_BAND`, `PAY_TREND_BAND`.

In [ ]:
TREND_ORDER = ["Strong down", "Down", "Flat", "Up", "Strong up"]
def trend_band(x, tol):
    x = pd.Series(x, index=df.index, dtype=float)
    neg_med, pos_med = x[x < -tol].median(), x[x > tol].median()
    out = np.select([x.abs() <= tol, x < neg_med, x < -tol, x <= pos_med], ["Flat", "Strong down", "Down", "Up"], "Strong up")
    return pd.Categorical(out, TREND_ORDER, ordered=True)

df["DELAY_TREND_BAND"] = pd.Categorical(np.clip(df["DELAY_TREND"], -3, 3).map({-3: "<=-3", -2: "-2", -1: "-1", 0: "0", 1: "+1", 2: "+2", 3: ">=+3"}),
                                        ["<=-3", "-2", "-1", "0", "+1", "+2", ">=+3"], ordered=True)
df["UTIL_TREND_BAND"] = trend_band(df["UTIL_TREND"], 0.005)
df["BILL_TREND_BAND"] = trend_band(df["BILL_TREND"], 500)
df["PAY_TREND_BAND"] = trend_band(df["PAY_TREND"], 100)

fig, axes = plt.subplots(2, 2, figsize=(16, 9))
tr_tabs = {}
for ax, (c, ttl) in zip(axes.ravel(), [("DELAY_TREND_BAND", "status Sep − Apr (positive = deteriorating)"),
                                        ("UTIL_TREND_BAND", "utilisation slope (tol ±0.5pp/month)"),
                                        ("BILL_TREND_BAND", "bill slope (tol ±NT$500/month)"),
                                        ("PAY_TREND_BAND", "payment slope (tol ±NT$100/month)")]):
    tr_tabs[c] = plot_rate(df, c, ax=ax, title=f"Default rate by {ttl}")
plt.tight_layout(); savefig("s05_trend_bands"); plt.show()
trend_long = pd.concat([t.rename(columns={c: "band"}).assign(feature=c) for c, t in tr_tabs.items()], ignore_index=True)
display(trend_long[["feature", "band", "n", "rate", "ci_lo", "ci_hi", "lift"]])
print("Band thresholds (medians of non-flat side):",
      {c: (round(df[c][df[c] < -t].median(), 4), round(df[c][df[c] > t].median(), 4)) for c, t in
       [("UTIL_TREND", .005), ("BILL_TREND", 500), ("PAY_TREND", 100)]})
export(trend_long, "s05_trend_band_rates")

**Observations:**
- Clients whose status did not change default at 16.0%. Both large moves are risky: worsening by 3+ notches gives 42.9%, and improving by 3+ notches gives 43.4%. Large moves only happen to clients who were late at some point.
- Utilisation trends work against intuition: clients with falling utilisation default more (28–29%) than those with rising utilisation (16–21%). Falling bills behave the same way (Down: 32.9%).
- Flat payments are the riskiest payment trend (28.9%); strongly rising or strongly falling payments sit near 16%.
- Story use: background only. Trends add little beyond the current levels, and "rising balances" is not the warning sign.

### 5.13 Volatility: coefficient of variation of bills and payments
`BILL_CV` = std / mean of the six bills (undefined when mean bill ≤ 0); `PAY_CV` = std / mean of the six payments
(undefined when nothing was paid). Erratic repayment can signal liquidity stress; perfectly flat payments can signal
autopay / fixed instalments.

In [ ]:
_mb, _sb = B.mean(1), B.std(1)
_mp, _sp = P.mean(1), P.std(1)
df["BILL_CV"] = np.where(_mb > 0, _sb / np.where(_mb > 0, _mb, 1), np.nan)
df["PAY_CV"] = np.where(_mp > 0, _sp / np.where(_mp > 0, _mp, 1), np.nan)

def cv_band(s, label_undef):
    q = pd.qcut(s, 5, duplicates="drop")
    q = q.cat.rename_categories([f"Q{k+1} {max(iv.left, 0):.2f}–{iv.right:.2f}" for k, iv in enumerate(q.cat.categories)])
    return q.cat.add_categories(label_undef).fillna(label_undef)
df["BILL_CV_BAND"] = cv_band(df["BILL_CV"], "Undefined (mean bill<=0)")
df["PAY_CV_BAND"] = cv_band(df["PAY_CV"], "No payments")
fig, axes = plt.subplots(1, 2, figsize=(17, 4.8))
t_bcv = plot_rate(df, "BILL_CV_BAND", ax=axes[0], title="Default rate by bill CV quintile", rotate=25)
t_pcv = plot_rate(df, "PAY_CV_BAND", ax=axes[1], title="Default rate by payment CV quintile", rotate=25)
plt.tight_layout(); savefig("s05_volatility"); plt.show()
display(t_bcv[["BILL_CV_BAND", "n", "rate", "lift"]]); display(t_pcv[["PAY_CV_BAND", "n", "rate", "lift"]])
print("Median CV by outcome:"); display(df.groupby("DEFAULT_LABEL")[["BILL_CV", "PAY_CV"]].median().round(3))
print("Zero-variance payers (identical payment every month, >0):", int(((_sp == 0) & (_mp > 0)).sum()),
      "| default rate:", pct(YD[(_sp == 0) & (_mp > 0)].mean()))

**Observations:**
- Steady bills are riskier. The lowest bill-CV quintile (CV = std ÷ mean, 0–0.06) defaults at 32.2% vs 15.4% in Q4, and the median bill CV is 0.19 for defaulters vs 0.39 for non-defaulters. A flat bill often means a maxed-out or frozen balance.
- Clients who made no payments in six months (1,432) default at 38.0%; clients with a mean bill ≤ 0 default at 34.4%.
- Payment volatility runs the other way: the steadiest payers (Q1) are the safest at 14.4%.
- 218 clients pay exactly the same amount every month, and they default at 24.3%, close to base.
- Story use: supports the "stuck balance" message (a stable high balance with low or zero payments).

### 5.14 Minimum-payer detection
A "minimum-payer month" = 0 < payment ratio < 10% of the previous statement (pays something, but only a token amount).
`N_MINPAY` (0..5 over May–Sep) is crossed with the number of zero-payment months.

In [ ]:
PR = df[[f"PAY_RATIO_{i}" for i in IDX[:-1]]].to_numpy(float)
df["N_MINPAY"] = ((PR > 0) & (PR < 0.1)).sum(1)
fig, axes = plt.subplots(1, 2, figsize=(17, 4.8))
t_mp = plot_rate(df, "N_MINPAY", ax=axes[0], title="Default rate by number of minimum-payer months (0<ratio<10%)")
rt_mz = rate_table(df, ["N_MINPAY", "N_ZERO_PAY"], min_n=30)
sns.heatmap(rt_mz.pivot(index="N_MINPAY", columns="N_ZERO_PAY", values="rate"), annot=True, fmt=".0%", cmap=CMAP_RATE,
            vmin=0, vmax=.7, ax=axes[1], cbar=False, linewidths=.5)
axes[1].set_title("Default rate: minimum-payer months × zero-payment months (n ≥ 30)")
plt.tight_layout(); savefig("s05_minimum_payer"); plt.show()
display(t_mp[["N_MINPAY", "n", "rate", "ci_lo", "ci_hi", "lift"]])
mm = panel[panel["PAY_RATIO"].notna()].assign(MINPAY=lambda t: ((t["PAY_RATIO"] > 0) & (t["PAY_RATIO"] < .1)).astype(int))
print("Share of minimum-payer months by month and outcome:")
display(mm.pivot_table(index="DEFAULT_LABEL", columns="MONTH", values="MINPAY", observed=True)[MONTH_ORDER[1:]].round(3))
export(rt_mz, "s05_minpay_zeropay_rates")

**Observations:**
- Minimum-payer months (payment under 10% of the previous bill) are very common: about half of all months in both groups (Sep: 45.9% for defaulters vs 47.7% for non-defaulters).
- Default rises from 20.0% (0 such months) to 29.4% (4 months), but falls to 17.5% for all 5 months (n=7,103). Clients who always pay a token amount behave like steady revolvers.
- The signal is weak overall, with lift (rate ÷ base rate) between 0.79 and 1.33.
- Story use: background only. The payer taxonomy in 6.4 covers this better.

### 5.15 Lag correlation (persistence) of status, bills, payments and utilisation
6 × 6 Spearman correlation matrices in chronological order, then the average correlation at each lag, then the
month-by-month correlation of each variable with `DEFAULT` (does recency dominate?).

In [ ]:
LAG_VARS = {"Status (PAY_i)": "PAY_", "Bill (BILL_AMTi)": "BILL_AMT", "Payment (PAY_AMTi)": "PAY_AMT", "Utilisation (UTIL_i)": "UTIL_"}
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
lag_rows, corr_store = [], {}
for ax, (lab, pre) in zip(axes, LAG_VARS.items()):
    X = df[[f"{pre}{i}" for i in CHRON_IDX]].copy(); X.columns = MONTH_ORDER
    Cm = X.corr(method="spearman"); corr_store[lab] = Cm
    sns.heatmap(Cm, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax, cbar=False, annot_kws={"fontsize": 8})
    ax.set_title(lab)
    for lag in range(1, 6):
        lag_rows.append({"variable": lab, "lag": lag, "mean_corr": float(np.mean([Cm.iloc[k, k + lag] for k in range(6 - lag)]))})
plt.tight_layout(); savefig("s05_lag_corr_heatmaps"); plt.show()
lagdf = pd.DataFrame(lag_rows)

dcor = []
for lab, pre in LAG_VARS.items():
    for m, i in zip(MONTH_ORDER, CHRON_IDX):
        dcor.append({"variable": lab, "MONTH": m, "spearman_with_default": float(stats.spearmanr(df[f"{pre}{i}"], YD)[0])})
dcor = pd.DataFrame(dcor)
fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
for lab in LAG_VARS:
    u = lagdf[lagdf["variable"] == lab]; axes[0].plot(u["lag"], u["mean_corr"], marker="o", label=lab)
    u = dcor[dcor["variable"] == lab]; axes[1].plot(range(6), u["spearman_with_default"], marker="o", label=lab)
axes[0].set_title("Average autocorrelation by lag (months)"); axes[0].set_xlabel("lag"); axes[0].legend(fontsize=8)
axes[1].set_xticks(range(6)); axes[1].set_xticklabels(MONTH_ORDER); axes[1].axhline(0, color="k", lw=.8)
axes[1].set_title("Spearman correlation with DEFAULT, by month"); axes[1].legend(fontsize=8)
plt.tight_layout(); savefig("s05_lag_profiles"); plt.show()
display(lagdf.pivot(index="variable", columns="lag", values="mean_corr").round(3))
display(dcor.pivot(index="variable", columns="MONTH", values="spearman_with_default")[MONTH_ORDER].round(3))
_lc = pd.concat([Cm.stack().rename("spearman").rename_axis(["month_a", "month_b"]).reset_index().assign(variable=lab)
                 for lab, Cm in corr_store.items()], ignore_index=True)
export(_lc, "s05_lag_correlation")

**Observations:**
- Bills and utilisation are very persistent: the lag-1 Spearman correlation is 0.91, and still 0.70–0.73 at lag 5. Status falls from 0.77 (lag 1) to 0.46 (lag 5), and payments are the least persistent (0.53 at lag 1).
- The correlation of status with default grows every month, from 0.143 (Apr) to 0.292 (Sep). The most recent status is the best signal.
- The payment–default correlation is negative and strengthens over time (−0.12 to −0.16). Bills barely relate to default (about 0).
- Story use: supports the recency message and justifies centring the dashboard on the latest month.

### 5.16 Section 5 headline numbers
A compact numeric digest for writing the insights.

**New `df` columns added in this section:** `TRAJ_SEQ`, `TRAJ_N_CHANGES`, `TRAJ_ENTROPY`, `MONTHS_SINCE_LATE`,
`FIRST_LATE_MONTH`, `DELAY_TREND_BAND`, `UTIL_TREND_BAND`, `BILL_TREND_BAND`, `PAY_TREND_BAND`, `BILL_CV`, `PAY_CV`,
`BILL_CV_BAND`, `PAY_CV_BAND`, `N_MINPAY`. Also defined: `panel` (client × month long table).

In [ ]:
_pr = roll[(roll["segment"] == "All") & (roll["pair"] == "Pooled")].set_index("from_bucket")
headline = {
    "share late, Apr → Sep (all)": f"{OBS_MIX[0, 3:].sum():.1%} → {OBS_MIX[5, 3:].sum():.1%}",
    "share of Sep late accounts coded status 1 (Late 1m)": f"{(SC[:, 5] == 1).sum() / (SC[:, 5] >= 1).sum():.1%}",
    "P(stay late 2m | late 2m), default vs no default": f"{TP['Default'][4, 4]:.1%} vs {TP['No default'][4, 4]:.1%}",
    "pooled cure rate from Late 1-2m / 3m+": f"{_pr.loc['Late 1-2m', 'cure']:.1%} / {_pr.loc['Late 3m+', 'cure']:.1%}",
    "pooled roll-forward from Current / Late 1-2m": f"{_pr.loc['Current', 'roll_forward']:.1%} / {_pr.loc['Late 1-2m', 'roll_forward']:.1%}",
    "share late Sep observed vs Markov stationary": f"{OBS_MIX[5, 3:].sum():.1%} vs {st[3:].sum():.1%}",
    "distinct sequences / top-25 coverage": f"{len(seq_tab):,} / {seq_tab['cum_share_clients'].iloc[24]:.1%}",
    "default rate never late vs late in Sep": f"{t_ms.set_index('MONTHS_SINCE_LATE').loc[7, 'rate']:.1%} vs {t_ms.set_index('MONTHS_SINCE_LATE').loc[0, 'rate']:.1%}",
    "default rate 0 vs >=3 min-payer months": f"{YD[df['N_MINPAY'] == 0].mean():.1%} vs {YD[df['N_MINPAY'] >= 3].mean():.1%}",
    "lag-1 autocorrelation status / bill / payment": " / ".join(f"{lagdf[(lagdf['variable'] == v) & (lagdf['lag'] == 1)]['mean_corr'].iloc[0]:.2f}" for v in list(LAG_VARS)[:3]),
}
display(pd.Series(headline, name="value").to_frame())

**Observations:**
- The late share doubles from 10.3% (Apr) to 22.7% (Sep); part of this is the code-1 quirk.
- Defaulters stay Late 2m 68.6% of the time vs 46.2% for non-defaulters. The cure rate is 25.3% from Late 1-2m and only 10.7% from Late 3m+.
- Never late (11.7%) vs late in Sep (50.3%) is the strongest single temporal split.
- Counting minimum-payer months barely helps (20.0% vs 23.0%).
- Story use: section summary. Use never-late vs late-in-Sep and `RRRRRR` (10.4%) vs `RRRRR2` (65.6%) as the headline numbers.

## 6. Share of Wallet

### 6.0 Framing: what "share of wallet" can and cannot mean in this dataset

**True share of wallet (SoW)** = the bank's share of a customer's *total* spending or borrowing across all providers:

$$\text{SoW}_c = \frac{\text{balance or spend held with this bank}}{\text{customer's total wallet (all lenders / all spend)}}$$

In practice issuers estimate the denominator from **credit-bureau tradelines** (balances and limits on other lenders' cards), card-network or aggregator data, and income/demographic models; the numerator is the bank's own "on-us" balance or spend (Equifax, *Share of Wallet Analysis*; ProSight FA, *Redefining the top-of-wallet customer*, 2016). When the wallet is unobserved, a standard trick is to **estimate it as a high quantile of what similar customers spend/borrow** (the "realistic wallet" idea in IBM Research's wallet-estimation work, Rosset et al., *Wallet estimation models*, 2005, which uses quantile regression), and patent literature on SoW models lists outputs such as *spend capacity, revolving balance and average pay-down %* as risk attributes (US20070067209A1).

**This dataset has no external wallet data**: no income, no bureau tradelines, no off-us cards. We therefore build explicit **proxies**, each labelled with its formula and its caveat, and test each one against default.

| Proxy | Name | Formula (per client) | Interpretation | Main caveat |
|---|---|---|---|---|
| **A** | Credit-line SoW / wallet capture | `UTIL_i = BILL_AMTi / LIMIT_BAL`; `SOW_UTIL_AVG` = mean of max(UTIL_i, 0); headroom = `LIMIT_BAL − BILL_AMT1` (signed: negative when over the limit, credit balances add headroom) | How much of the capacity this bank granted is actually drawn | `LIMIT_BAL` is the bank's own estimate of capacity and **includes family/supplementary cards** — utilisation is SoW of the *granted line*, not of the wallet |
| **B** | Spend SoW | `SOW_SPEND_SHARE_i = max(SPEND_i,0) / LIMIT_BAL`; Herfindahl of spend across 5 months; spend trend | Share of granted capacity consumed by **new** purchases each month | `SPEND_i` is implied (bill change + payment) and includes interest/fees; refunds make it negative (clipped at 0) |
| **C** | Repayment share | `PAY_AMTi / BILL_AMT(i+1)` capped to [0,1]; `PAY_AMT / LIMIT_BAL`; payer type | How much of the card wallet the client chooses to clear each cycle | Minimum-due amount is not given; the 6% "minimum payer" threshold is an **assumption** set from the empirical spike of repayment shares at 3.5–4.5% (6.4a) and sensitivity-tested (6.4b) |
| **D** | Relative wallet (peer) | `LIMIT_BAL / median LIMIT_BAL of peer cell` (AGE_BAND × EDU × SEX × MAR); same for avg bill; quantile-regression wallet estimate and limit residual | Is the client given / using a large line **relative to similar people**? | Demographics explain only a small part of limits; peer cells < 30 clients fall back to AGE_BAND × EDU |
| **E** | Portfolio share of wallet | segment share of exposure (`max(BILL_AMT1,0)`), of clients, of defaults, of exposure-at-default | Where the bank's money — and its at-risk money — sits | EAD proxy = Sep balance of defaulters; no LGD, no recoveries, no undrawn-line conversion |

**Assumptions stated up front**
1. `LIMIT_BAL` is read as the **bank's own estimate of the client's (household) wallet/capacity**. Because it includes supplementary cards, any "per-person" reading over-states capacity for family accounts.
2. Negative bills (credit balances) mean the client holds money with the bank: they count as **zero capture** for SoW purposes.
3. Exposure at default (EAD) = `max(BILL_AMT1, 0)` of clients who default in Oct-2005. No LGD is observed; the expected-loss view in 6.9 shows LGD = 100% (upper bound) and LGD = 45% (Basel foundation-IRB unsecured benchmark) for scale only.
4. Everything here is **descriptive**: associations with default, not causal effects of increasing wallet share.

All new client-level columns are prefixed `SOW_` (listed in 6.11) and exported for Tableau.

### 6.1 Build the SoW feature layer
One cell computes every proxy so that definitions are identical in the notebook and in the Tableau extracts. Bands are fixed, human-readable cut points (not quantiles) so they stay stable across refreshes.

In [ ]:
t0_s6 = time.time()
def _fmt(frame, f):
    return frame.apply(lambda s_: s_.map(f))
Lim = df["LIMIT_BAL"].to_numpy(float)
Bm = df[BILL_COLS].to_numpy(float)          # col 0 = Sep ... col 5 = Apr
Pm = df[PAMT_COLS].to_numpy(float)
Um = Bm / Lim[:, None]
Uc = np.clip(Um, 0, None)                    # credit balances -> zero capture

# ---------- Proxy A: credit-line SoW / wallet capture ----------
df["SOW_UTIL_SEP"] = Uc[:, 0]
df["SOW_UTIL_AVG"] = Uc.mean(1)
df["SOW_UTIL_PEAK"] = Uc.max(1)
df["SOW_HEADROOM_SEP"] = Lim - Bm[:, 0]
df["SOW_HEADROOM_PCT"] = 1 - Um[:, 0]
CAP_BINS = [-np.inf, 0, .1, .3, .5, .7, .9, 1.0, np.inf]
CAP_LABELS = ["0 (none/credit)", "0-10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%", ">100%"]
df["SOW_CAPTURE_BAND"] = pd.cut(df["SOW_UTIL_AVG"], CAP_BINS, labels=CAP_LABELS)
df["SOW_CAPTURE_SEP_BAND"] = pd.cut(df["SOW_UTIL_SEP"], CAP_BINS, labels=CAP_LABELS)
HEAD_BINS = [-np.inf, 0, 10_000, 50_000, 100_000, 200_000, np.inf]
HEAD_LABELS = ["Over limit (<0)", "0-10k", "10-50k", "50-100k", "100-200k", ">200k"]
df["SOW_HEADROOM_BAND"] = pd.cut(df["SOW_HEADROOM_SEP"], HEAD_BINS, labels=HEAD_LABELS, right=False)

# ---------- Proxy B: spend SoW ----------
SPm = df[SPEND_COLS].to_numpy(float)          # SPEND_1..5 = Sep..May
SPp = np.clip(SPm, 0, None)
SSH = SPp / Lim[:, None]
for k, i in enumerate(IDX[:-1]):
    df[f"SOW_SPEND_SHARE_{i}"] = SSH[:, k]
df["SOW_SPEND_SHARE_AVG"] = SSH.mean(1)
tot_sp = SPp.sum(1)
df["SOW_SPEND_HHI"] = np.where(tot_sp > 0, ((SPp / np.where(tot_sp > 0, tot_sp, 1)[:, None]) ** 2).sum(1), np.nan)
df["SOW_SPEND_ACTIVE_M"] = (SPm > 0).sum(1)
t5 = np.arange(5) - 2.0
_ch = SSH[:, ::-1]                            # May -> Sep
df["SOW_SPEND_TREND"] = ((_ch - _ch.mean(1, keepdims=True)) * t5).sum(1) / (t5 ** 2).sum()
SPS_BINS = [-np.inf, 0, .02, .05, .10, .20, np.inf]
SPS_LABELS = ["0 (no spend)", "0-2%", "2-5%", "5-10%", "10-20%", ">20%"]
df["SOW_SPEND_BAND"] = pd.cut(df["SOW_SPEND_SHARE_AVG"], SPS_BINS, labels=SPS_LABELS)
df["SOW_SPEND_CONC_BAND"] = pd.cut(df["SOW_SPEND_HHI"], [0, .3, .5, .8, 1.0001],
                                   labels=["Even (HHI<=0.3)", "Moderate (0.3-0.5)", "Concentrated (0.5-0.8)", "Single-month (>0.8)"])
df["SOW_SPEND_CONC_BAND"] = df["SOW_SPEND_CONC_BAND"].cat.add_categories("No spend").fillna("No spend")

# ---------- Proxy C: repayment share and payer type ----------
prev_bill = Bm[:, 1:]                          # BILL_AMT2..6, settled by PAY_AMT1..5
R = np.where(prev_bill > 0, Pm[:, :5] / np.where(prev_bill > 0, prev_bill, 1), np.nan)
Rc = np.clip(R, 0, 1)                          # monthly repayment share, Sep..May
MIN_T, FULL_T = 0.06, 0.95                     # ASSUMPTION: <6% ~ minimum-due payer (spike at 3.5-4.5%, see 6.4a); >=95% ~ pays in full
PAYER_ORDER = ["Transactor", "Partial payer", "Minimum payer", "Non-payer", "No balance due"]
def classify_share(r, min_t=MIN_T, full_t=FULL_T):
    r = np.asarray(r, float)
    return np.select([np.isnan(r), r >= full_t, r >= min_t, r > 0],
                     ["No balance due", "Transactor", "Partial payer", "Minimum payer"], "Non-payer")
for k, i in enumerate(IDX[:-1]):
    df[f"SOW_PAYSHARE_{i}"] = Rc[:, k]
df["SOW_PAYSHARE_MED"] = np.nanmedian(Rc, axis=1)
df["SOW_PAYSHARE_MEAN"] = np.nanmean(Rc, axis=1)
df["SOW_PAY_LIMIT_AVG"] = Pm.mean(1) / Lim
df["SOW_N_TRANSACTOR_M"] = (Rc >= FULL_T).sum(1)
df["SOW_PAYER_TYPE"] = pd.Categorical(classify_share(df["SOW_PAYSHARE_MED"]), categories=PAYER_ORDER, ordered=True)
PS_BINS = [-np.inf, 1e-9, .06, .10, .25, .50, .95, np.inf]
PS_LABELS = ["0%", "0-6% (~min due)", "6-10%", "10-25%", "25-50%", "50-95%", ">=95%"]
df["SOW_PAYSHARE_BAND"] = pd.cut(df["SOW_PAYSHARE_MED"], PS_BINS, labels=PS_LABELS, right=False)
df["SOW_PAYSHARE_BAND"] = df["SOW_PAYSHARE_BAND"].cat.add_categories("No balance due").fillna("No balance due")

# ---------- Proxy E base: exposure ----------
df["SOW_EXPOSURE_SEP"] = np.clip(Bm[:, 0], 0, None)
df["SOW_EAD"] = df["SOW_EXPOSURE_SEP"] * df[TARGET]

# ---------- Dynamics ----------
df["SOW_UTIL_CHG"] = Um[:, 0] - Um[:, 5]                      # Sep - Apr (raw utilisation)
df["SOW_STRETCH_M"] = (Um > 0.9).sum(1)
util_late, util_early = Uc[:, 0:2].mean(1), Uc[:, 4:6].mean(1)    # Sep/Aug vs May/Apr
pay_late, pay_early = np.nanmean(Rc[:, 0:2], 1), np.nanmean(Rc[:, 3:5], 1)   # Sep/Aug vs Jun/May
df["SOW_D_UTIL"] = util_late - util_early
df["SOW_D_PAYSHARE"] = pay_late - pay_early
up = df["SOW_D_UTIL"] > 0.05
down = df["SOW_D_PAYSHARE"] < -0.05
DYN_ORDER = ["Stable / improving", "Capture up, paying", "Paying less, capture flat", "Stress: capture up + pay down", "Insufficient history"]
dyn = np.select([df["SOW_D_PAYSHARE"].isna(), up & down, up & ~down, ~up & down],
                [DYN_ORDER[4], DYN_ORDER[3], DYN_ORDER[1], DYN_ORDER[2]], DYN_ORDER[0])
df["SOW_DYN_QUADRANT"] = pd.Categorical(dyn, categories=DYN_ORDER, ordered=True)
df["SOW_STRESS_FLAG"] = (dyn == DYN_ORDER[3]).astype(int)
df["SOW_UTIL_CHG_BAND"] = pd.cut(df["SOW_UTIL_CHG"], [-np.inf, -.2, -.05, .05, .2, np.inf],
                                 labels=["Falling >20pp", "Falling 5-20pp", "Stable (+/-5pp)", "Rising 5-20pp", "Rising >20pp"])
df["SOW_STRETCH_BAND"] = pd.cut(df["SOW_STRETCH_M"], [-1, 0, 2, 5, 6], labels=["0 months", "1-2 months", "3-5 months", "All 6 months"])

summary_cols = ["SOW_UTIL_AVG", "SOW_UTIL_SEP", "SOW_HEADROOM_SEP", "SOW_SPEND_SHARE_AVG", "SOW_SPEND_HHI", "SOW_SPEND_TREND",
                "SOW_PAYSHARE_MED", "SOW_PAY_LIMIT_AVG", "SOW_EXPOSURE_SEP", "SOW_UTIL_CHG", "SOW_STRETCH_M", "SOW_D_UTIL", "SOW_D_PAYSHARE"]
display(df[summary_cols].describe(percentiles=[.1, .25, .5, .75, .9]).T)
print(f"SoW layer built in {time.time()-t0_s6:.1f}s")

**Observations:**
- Average capture (utilisation = bill ÷ limit) has a median of 0.29 and a mean of 0.37. One client in ten averages above 0.89, and the maximum is 5.4x the limit.
- Spend share is small: the median client puts 3.1% of the line through new purchases each month (90th percentile 16.5%).
- Repayment share is split in two: the median is 7.4%, but the 75th percentile is already 100%. Clients either pay near the minimum or clear the bill.
- Sep exposure is skewed: median NT$22.4k vs mean NT$51.2k.
- Story use: background only (definitions and ranges of the SoW proxies).

### 6.2 Proxy A — Credit-line share of wallet (wallet capture)
**6.2a** Monthly capture (utilisation) trajectory by outcome, and the distribution of average capture. If the bank's line is the wallet, defaulters should be drawing a visibly larger share of it — and the gap should widen toward September.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
chron_idx = IDX[::-1]                                     # Apr..Sep
traj = (df.groupby("DEFAULT_LABEL")[[f"UTIL_{i}" for i in chron_idx]].agg(["mean", "median"]))
for lab in ["No default", "Default"]:
    m = [df.loc[df.DEFAULT_LABEL == lab, f"UTIL_{i}"].clip(lower=0).mean() for i in chron_idx]
    md_ = [df.loc[df.DEFAULT_LABEL == lab, f"UTIL_{i}"].clip(lower=0).median() for i in chron_idx]
    axes[0].plot(MONTH_ORDER, m, marker="o", color=PAL_TARGET[lab], lw=2, label=f"{lab} - mean")
    axes[0].plot(MONTH_ORDER, md_, marker="s", ls="--", color=PAL_TARGET[lab], lw=1.5, label=f"{lab} - median")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].set_title("Wallet capture (bill / limit) by month")
axes[0].legend(fontsize=8); axes[0].set_ylabel("utilisation of granted line")
for lab in ["No default", "Default"]:
    sns.kdeplot(df.loc[df.DEFAULT_LABEL == lab, "SOW_UTIL_AVG"].clip(upper=1.3), ax=axes[1], fill=True, alpha=.3,
                color=PAL_TARGET[lab], label=lab, common_norm=False, bw_adjust=.6)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("Distribution of 6-month average capture (clipped at 130%)")
axes[1].legend()
plt.tight_layout(); savefig("s06_capture_trajectory"); plt.show()
cap_month = pd.DataFrame({lab: [df.loc[df.DEFAULT_LABEL == lab, f"UTIL_{i}"].clip(lower=0).mean() for i in chron_idx]
                          for lab in ["No default", "Default"]}, index=MONTH_ORDER)
cap_month["gap_pp"] = (cap_month["Default"] - cap_month["No default"]) * 100
print("Mean capture by month (clipped at 0):"); display(cap_month)

**Observations:**
- Defaulters draw more of their line in every month: mean capture is 39.9% vs 29.6% in Apr and 49.0% vs 40.5% in Sep.
- The gap is about 10 pp throughout and narrows to 8.5 pp in Sep, as non-defaulters' usage also rises (+11 pp over six months).
- Story use: opening SoW chart. Future defaulters were already using more of their line six months before default.

**6.2b** Default rate by capture band (6-month average and September snapshot) and by NT$ headroom. Bars = clients, red line = default rate with 95% Wilson CI, dashed = base rate. *Tableau: bar-and-line dual-axis sheet.*

*Note:* headroom here (`SOW_HEADROOM_SEP = LIMIT_BAL − BILL_AMT1`) is **signed**: negative when the client is over the limit, and a credit (negative) bill adds headroom. It differs from Section 18's clipped `EX_HEADROOM = max(LIMIT_BAL − max(BILL_AMT1, 0), 0)`, so do not quote NT$ headroom totals from this section next to Section 18's.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
tA1 = plot_rate(df, "SOW_CAPTURE_BAND", ax=axes[0], title="Default rate by avg capture (Apr-Sep)", rotate=30)
tA2 = plot_rate(df, "SOW_CAPTURE_SEP_BAND", ax=axes[1], title="Default rate by Sep capture", rotate=30)
tA3 = plot_rate(df, "SOW_HEADROOM_BAND", ax=axes[2], title="Default rate by Sep headroom (limit - bill, NT$)", rotate=30)
plt.tight_layout(); savefig("s06_capture_bands"); plt.show()
cols = ["n", "defaults", "rate", "ci_lo", "ci_hi", "share_clients", "lift"]
for t, c in [(tA1, "SOW_CAPTURE_BAND"), (tA2, "SOW_CAPTURE_SEP_BAND"), (tA3, "SOW_HEADROOM_BAND")]:
    display(t[[c] + cols])

**Observations:**
- Default rises steadily with average capture: 15.6% at 10–30%, 32.2% at 90–100% and 34.2% above 100% (over the limit).
- Clients with zero or credit balances (966, 3.2%) are also risky at 35.9%, so the curve is U-shaped: both ends are bad, and 10–30% is the sweet spot.
- The 6-month average separates risk better than the Sep snapshot (the Sep 90–100% band is only 24.3%).
- NT$ headroom (limit − Sep bill) also orders risk: under NT$50k of headroom gives about 29% default, over NT$200k gives 14.2%, and over-limit gives 30.1%.
- Story use: core bar-and-line chart for Proxy A with the message "the more of the line used, the higher the risk".

**6.2c** Does headroom matter *after* controlling for line size? Headroom in NT$ is mechanically larger for big lines, so we show default rate by headroom % (1 − Sep utilisation) **within** each limit band. A heatmap with counts; cells with < 50 clients are blanked.

In [ ]:
tmp = df.copy()
tmp["HEADROOM_PCT_BAND"] = pd.cut(tmp["SOW_HEADROOM_PCT"], [-np.inf, 0, .1, .3, .5, .7, .9, np.inf],
                                  labels=["<0 (over)", "0-10%", "10-30%", "30-50%", "50-70%", "70-90%", ">90%"])
g = rate_table(tmp, ["LIMIT_BAND", "HEADROOM_PCT_BAND"])
rate_p = g.pivot(index="LIMIT_BAND", columns="HEADROOM_PCT_BAND", values="rate")
n_p = g.pivot(index="LIMIT_BAND", columns="HEADROOM_PCT_BAND", values="n")
annot = _fmt(rate_p, lambda v: "" if pd.isna(v) else f"{v:.0%}") + "\n" + _fmt(n_p, lambda v: "" if pd.isna(v) else f"n={int(v)}")
fig, ax = plt.subplots(figsize=(12, 5.5))
sns.heatmap(rate_p.where(n_p >= 50), annot=annot.where(n_p >= 50, ""), fmt="", cmap=CMAP_RATE, ax=ax,
            cbar_kws={"format": mtick.PercentFormatter(1), "label": "default rate"}, annot_kws={"fontsize": 8})
ax.set_title("Default rate: Sep headroom % x limit band (n>=50)"); ax.set_xlabel("headroom % of limit (Sep)"); ax.set_ylabel("limit band")
plt.tight_layout(); savefig("s06_headroom_by_limit"); plt.show()
display(rate_p.round(3))

**Observations:**
- In large lines, headroom matters a lot. In the 300-500k band, default falls from 34.8% (over the limit) to 5.5% (70–90% headroom); in 200-300k it falls from 27.6% to 8.2%.
- In small lines (≤50k), headroom barely matters: 27–36% at every level. Small-line clients are risky whatever their usage.
- Almost-unused lines (>90% headroom) tick back up in every band (for example 14.3% in 300-500k), which matches the dormant-account risk.
- Story use: Tableau heatmap. For lines above 150k, headroom is a real signal and not just a limit effect.

### 6.3 Proxy B — Spend share of wallet
Implied monthly spend (new purchases + charges) as a share of the granted line. A client who puts a large share of their capacity through the card each month is "top-of-wallet" for this bank; a client whose balance grows without new spend is revolving debt, not spending. **6.3a** Monthly spend share by outcome, and default by average spend share.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sp_months = ["May", "Jun", "Jul", "Aug", "Sep"]
sp_idx = [5, 4, 3, 2, 1]
sp_tab = pd.DataFrame({lab: [df.loc[df.DEFAULT_LABEL == lab, f"SOW_SPEND_SHARE_{i}"].mean() for i in sp_idx]
                       for lab in ["No default", "Default"]}, index=sp_months)
sp_tab_med = pd.DataFrame({lab: [df.loc[df.DEFAULT_LABEL == lab, f"SOW_SPEND_SHARE_{i}"].median() for i in sp_idx]
                           for lab in ["No default", "Default"]}, index=sp_months)
for lab in ["No default", "Default"]:
    axes[0].plot(sp_months, sp_tab[lab], marker="o", color=PAL_TARGET[lab], lw=2, label=f"{lab} - mean")
    axes[0].plot(sp_months, sp_tab_med[lab], marker="s", ls="--", color=PAL_TARGET[lab], label=f"{lab} - median")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].legend(fontsize=8)
axes[0].set_title("Monthly spend as % of granted line")
tB1 = plot_rate(df, "SOW_SPEND_BAND", ax=axes[1], title="Default rate by avg monthly spend share", rotate=20)
plt.tight_layout(); savefig("s06_spend_share"); plt.show()
print("Mean / median monthly spend share:"); display(sp_tab.join(sp_tab_med, lsuffix="_mean", rsuffix="_median"))
display(tB1[["SOW_SPEND_BAND", "n", "defaults", "rate", "ci_lo", "ci_hi", "share_clients", "lift"]])

**Observations:**
- Defaulters spend a slightly smaller share of the line: the median monthly spend share is 1.3–1.8% vs 1.4–2.2%, and the mean gap widens by Sep (4.6% vs 6.3%).
- No spending at all over 5 months (1,158 clients) gives 40.1% default, 1.8x base.
- The curve is U-shaped: risk falls from 25.4% (0–2% spend share) to 16.2% (5–10%), then rises to 22.4% above 20%.
- Story use: "an active card is a healthy card". Low spend is the warning sign, not high spend.

**6.3b** Spend concentration and spend trend. HHI of positive spend across the 5 months (0.2 = perfectly even, 1 = all spend in a single month) separates habitual users from one-off big-ticket users; the spend-share trend shows whether the card is gaining or losing share of the client's capacity.

In [ ]:
tmp = df.copy()
tmp["SPEND_TREND_BAND"] = pd.cut(tmp["SOW_SPEND_TREND"], [-np.inf, -.02, -.005, .005, .02, np.inf],
                                 labels=["Falling fast", "Falling", "Flat", "Rising", "Rising fast"])
tmp["SPEND_ACTIVE_BAND"] = tmp["SOW_SPEND_ACTIVE_M"].astype(str) + " of 5"
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
tB2 = plot_rate(tmp, "SOW_SPEND_CONC_BAND", ax=axes[0], title="Default by spend concentration (HHI)", rotate=20)
tB3 = plot_rate(tmp, "SPEND_TREND_BAND", ax=axes[1], title="Default by spend-share trend (per month)")
tB4 = plot_rate(tmp, "SPEND_ACTIVE_BAND", ax=axes[2], title="Default by months with positive spend")
plt.tight_layout(); savefig("s06_spend_conc_trend"); plt.show()
for t, c in [(tB2, "SOW_SPEND_CONC_BAND"), (tB3, "SPEND_TREND_BAND"), (tB4, "SPEND_ACTIVE_BAND")]:
    display(t[[c, "n", "defaults", "rate", "share_clients", "lift"]])

**Observations:**
- Spend spread evenly across months (HHI ≤ 0.3) defaults at 25.0%, vs 17.5% for moderate concentration. HHI, the Herfindahl index, is 0.2 when spend is the same every month and 1 when it all falls in one month. Small, steady "spend" is often just interest and fees on a stuck balance.
- Rising spend is safer (14.9% rising, 16.3% rising fast) than flat spend (26.1%, the largest group at 51% of clients).
- Spend in 0 of 5 months gives 40.1% default; any activity (1–5 months) sits near base at 18–23%.
- Story use: supports "active card = safer". Flat spend flags a frozen account.

**6.3c** Spend vs capture: is the balance driven by **spending** (transactional wallet) or by **carried debt** (revolving wallet)? Default rate on a spend-share × capture grid. High capture with low spend = balance is old debt, not new purchases.

In [ ]:
g = rate_table(df, ["SOW_SPEND_BAND", "SOW_CAPTURE_BAND"])
rp = g.pivot(index="SOW_SPEND_BAND", columns="SOW_CAPTURE_BAND", values="rate")
npv = g.pivot(index="SOW_SPEND_BAND", columns="SOW_CAPTURE_BAND", values="n")
ann = _fmt(rp, lambda v: "" if pd.isna(v) else f"{v:.0%}") + "\n" + _fmt(npv, lambda v: "" if pd.isna(v) else f"{int(v)}")
fig, ax = plt.subplots(figsize=(13, 5.5))
sns.heatmap(rp.where(npv >= 50), annot=ann.where(npv >= 50, ""), fmt="", cmap=CMAP_RATE, ax=ax,
            cbar_kws={"format": mtick.PercentFormatter(1)}, annot_kws={"fontsize": 8})
ax.set_title("Default rate: spend share (rows) x capture (cols); n>=50"); ax.set_xlabel("avg capture (bill/limit)"); ax.set_ylabel("avg monthly spend / limit")
plt.tight_layout(); savefig("s06_spend_x_capture"); plt.show()
display(rp.round(3))

**Observations:**
- High capture with low spend is the danger zone: 0–2% spend share with capture of 50% or more defaults at 43–46%. That balance is carried debt, not shopping.
- High capture with high spend is much safer: 10–20% spend at 70–90% capture gives 22.8%, and >20% spend at 90–100% capture gives 22.5%.
- Low capture with moderate spend (2–10%) is the safest corner at 9–12%.
- Story use: key SoW insight. What matters is whether the line is used for spending or for carrying debt, not how much of it is used.

### 6.4 Proxy C — Repayment share and payer-type taxonomy
Repayment share = `PAY_AMTi / BILL_AMT(i+1)`, capped to [0, 1]. Monthly types: **Transactor** (≥ 95% cleared), **Partial payer** (6–95%), **Minimum payer** (0–6%, at or near the minimum due), **Non-payer** (0), **No balance due** (previous bill ≤ 0). The client type uses the **median** monthly share. (The 6–95% band is called *Partial payer*, not "Revolver": elsewhere in the notebook, e.g. Sections 0, 18 and 20, "Revolver" means September status code 0, whose typical repayment is about 4%, i.e. this section's *Minimum payer*.)

**6.4a** First, where is the minimum due? The dataset does not give it, so we read it off the distribution of monthly repayment shares (all client-months with a positive previous bill): a sharp spike marks the contractual minimum and justifies the 6% cut-off. Then type mix and default rate.

In [ ]:
pooled = Rc[~np.isnan(Rc)]
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.hist(pooled[pooled <= .3], bins=np.arange(0, .3001, .0025), color=C_OK, alpha=.8)
ax.axvline(MIN_T, color=C_DEF, ls="--", label=f"min-payer cut-off {MIN_T:.0%}")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.legend()
ax.set_title("Monthly repayment share (client-months, share <= 30% shown)"); ax.set_xlabel("PAY_AMTi / BILL_AMT(i+1)")
plt.tight_layout(); savefig("s06_repay_share_hist"); plt.show()
bins_tab = pd.cut(pd.Series(pooled), [-1e-9, 0, .03, .035, .045, .06, .10, .25, .5, .95, 1.0], include_lowest=True).value_counts(normalize=True).sort_index()
print(f"client-months with a positive previous bill: {len(pooled):,}"); display(bins_tab.rename("share").to_frame().round(4))

**Observations:**
- The analysis covers 130,561 client-months with a positive previous bill.
- 25.5% of months clear 95% or more of the bill, and 20.4% sit in a narrow spike at 3.5–4.5%: a clear minimum-due cluster.
- 8.5% of months pay nothing. Only 5.9% fall between 25% and 95%, so clients rarely pay "part" of the bill.
- Story use: justifies the 6% minimum-payer cut-off. The histogram is a good explainer visual.

Payer-type mix and default rate (client type = median monthly repayment share).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
tC1 = plot_rate(df, "SOW_PAYER_TYPE", ax=axes[0], order=PAYER_ORDER, title="Default rate by payer type (median repayment share)")
tC2 = plot_rate(df, "SOW_PAYSHARE_BAND", ax=axes[1], title="Default rate by median repayment share", rotate=20)
plt.tight_layout(); savefig("s06_payer_type"); plt.show()
display(tC1[["SOW_PAYER_TYPE", "n", "defaults", "rate", "ci_lo", "ci_hi", "share_clients", "share_defaults", "lift"]])
display(tC2[["SOW_PAYSHARE_BAND", "n", "defaults", "rate", "share_clients", "lift"]])
print("Cross-check vs September status group (row %):")
display(pd.crosstab(df["SOW_PAYER_TYPE"], df["STATUS_GRP_SEP"], normalize="index").reindex(columns=STATUS_GROUP_ORDER).round(3))

**Observations:**
- Minimum payers are the largest group: 40.2% of clients and 45.7% of defaulters, with a 25.2% default rate.
- Transactors (who clear ≥ 95%) are 26.7% of clients and default at 14.6%; partial payers (6–95% repaid) default at 19.9%.
- Non-payers (781 clients, 2.6%) default at 58.1%, 2.6x base. Clients with no balance due default at 31.2% (many are dormant).
- Cross-check: 51.6% of transactors are "paid in full" in the Sep status codes, and 70.6% of minimum payers are "revolving". The taxonomy agrees with the bank's own codes.
- Story use: payer-type bar chart, the main Proxy C chart.

**6.4b** Threshold sensitivity. The 6% / 95% cut-offs are assumptions; if the ranking of payer types survives other reasonable thresholds, the taxonomy is robust.

In [ ]:
rows = []
for mt, ft in [(0.045, 0.95), (0.06, 0.95), (0.08, 0.95), (0.10, 0.95), (0.06, 0.90), (0.06, 0.99), (0.15, 0.90)]:
    typ = classify_share(df["SOW_PAYSHARE_MED"], mt, ft)
    t = pd.DataFrame({"type": typ, "y": df[TARGET]}).groupby("type")["y"].agg(["size", "mean"])
    for k, r_ in t.iterrows():
        rows.append({"thresholds": f"min<{mt:.1%}, full>={ft:.0%}", "type": k, "n": int(r_["size"]), "rate": r_["mean"]})
sens = pd.DataFrame(rows)
display(sens.pivot(index="type", columns="thresholds", values="rate").reindex(PAYER_ORDER).round(3))
display(sens.pivot(index="type", columns="thresholds", values="n").reindex(PAYER_ORDER))

**Observations:**
- The default rates of transactors (14.6–14.7%), non-payers (58.1%) and no-balance clients (31.2%) do not move across 7 threshold settings.
- The minimum-payer rate stays at 24.5–26.2% and is always above the partial-payer rate (15.7–20.8%), so the ranking holds.
- Group sizes do shift: minimum payers range from 8,429 (4.5% cut-off) to 16,585 (15% cut-off).
- Story use: background only (robustness note).

**6.4c** Payer-type mix **month by month** (May → Sep) for defaulters vs non-defaulters, and repayment as a share of the *limit* (cash returned relative to capacity). *Tableau: 100% stacked bar per month, filter on outcome.*

In [ ]:
mix_rows = []
for i, mname in zip([5, 4, 3, 2, 1], ["May", "Jun", "Jul", "Aug", "Sep"]):
    typ = classify_share(df[f"SOW_PAYSHARE_{i}"].where(df[f"BILL_AMT{i+1}"] > 0))
    for lab in ["No default", "Default"]:
        m = (df["DEFAULT_LABEL"] == lab).to_numpy()
        vc = pd.Series(typ[m]).value_counts(normalize=True)
        for k, v in vc.items():
            mix_rows.append({"month": mname, "outcome": lab, "type": k, "share": v})
mix = pd.DataFrame(mix_rows)
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8), gridspec_kw={"width_ratios": [1, 1, 1.1]})
type_cols = {"Transactor": C_ACC, "Partial payer": C_OK, "Minimum payer": "#fdae6b", "Non-payer": C_DEF, "No balance due": C_NEU}
for ax, lab in zip(axes[:2], ["No default", "Default"]):
    pv = mix[mix.outcome == lab].pivot(index="month", columns="type", values="share").reindex(["May", "Jun", "Jul", "Aug", "Sep"])
    pv = pv.reindex(columns=PAYER_ORDER).fillna(0)
    pv.plot(kind="bar", stacked=True, ax=ax, color=[type_cols[c] for c in pv.columns], width=.8, legend=(lab == "Default"))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title(f"Monthly payer-type mix - {lab}"); ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
axes[1].legend(bbox_to_anchor=(1.0, 1), fontsize=8)
tmp = df.copy()
tmp["PAY_LIMIT_BAND"] = pd.cut(tmp["SOW_PAY_LIMIT_AVG"], [-np.inf, 0, .01, .02, .05, .10, np.inf],
                               labels=["0", "0-1%", "1-2%", "2-5%", "5-10%", ">10%"])
tC3 = plot_rate(tmp, "PAY_LIMIT_BAND", ax=axes[2], title="Default by avg monthly payment / limit")
plt.tight_layout(); savefig("s06_payer_mix_monthly"); plt.show()
display(mix.pivot_table(index=["outcome", "type"], columns="month", values="share").reindex(columns=["May", "Jun", "Jul", "Aug", "Sep"]).round(3))
display(tC3[["PAY_LIMIT_BAND", "n", "defaults", "rate", "share_clients", "lift"]])

**Observations:**
- Non-payers are 13.7–17.6% of defaulters each month vs 4.4–6.0% of non-defaulters, about 3x as common.
- Transactors make up 14–15% of defaulters vs 23–26% of non-defaulters.
- From May to Sep the minimum-payer share falls in both groups (defaulters: 37% → 29%) while the partial-payer share rises (18% → 28%): repayment shares drift up across the portfolio.
- Payment ÷ limit: zero payment gives 38.0% default, payments above 10% of the limit give 16.4%, and the middle bands are flat at 19–24%.
- Story use: monthly 100% stacked bar in Tableau; the non-payer gap is the clearest signal.

**6.4d** Consistency of transacting: number of months (out of 5) in which the client cleared ≥ 95%. A client who *always* clears is a pure transactor; one who clears sometimes is switching between using the card as a payment tool and as a credit line.

In [ ]:
tmp = df.copy()
tmp["N_TRANSACTOR_M"] = tmp["SOW_N_TRANSACTOR_M"].astype(str)
fig, ax = plt.subplots(figsize=(10, 4.5))
tC4 = plot_rate(tmp, "N_TRANSACTOR_M", ax=ax, order=[str(k) for k in range(6)], title="Default rate by # months paid >= 95% of previous bill")
plt.tight_layout(); savefig("s06_transactor_months"); plt.show()
display(tC4[["N_TRANSACTOR_M", "n", "defaults", "rate", "share_clients", "lift"]])

**Observations:**
- 61.8% of clients never clear their bill in any of the 5 months, and they default at 26.3%.
- Default falls steadily with each month cleared: 18.1% (1 month), 14.0% (4 months), 11.2% (all 5).
- Story use: a simple, monotone line: "every month a client clears the bill cuts risk".

### 6.4c Proxy C2 — Quarterly paydown: how much of three months of balances did the client pay back?
**Definition (your formula, mapped to our column names).** A payment settles the *previous* month's statement, so the
payment window and the balance window are shifted by one month:

$$\text{QPD} = \frac{\text{Pay}_0 + \text{Pay}_1 + \text{Pay}_2}{\text{Bal}_1 + \text{Bal}_2 + \text{Bal}_3}
= \frac{\texttt{PAY\_AMT1} + \texttt{PAY\_AMT2} + \texttt{PAY\_AMT3}}{\texttt{BILL\_AMT2} + \texttt{BILL\_AMT3} + \texttt{BILL\_AMT4}}$$

| Your name | Our column | Month | Meaning |
|---|---|---|---|
| Pay0 | `PAY_AMT1` | paid in Sep | settles the Aug statement |
| Pay1 | `PAY_AMT2` | paid in Aug | settles the Jul statement |
| Pay2 | `PAY_AMT3` | paid in Jul | settles the Jun statement |
| Bal1 | `BILL_AMT2` | Aug statement | |
| Bal2 | `BILL_AMT3` | Jul statement | |
| Bal3 | `BILL_AMT4` | Jun statement | |

**Columns built here (prefix `SOW_QPD`).**
* `SOW_QPD_1` = the latest quarter (payments Jul-Sep / statements Jun-Aug). **This is the headline paydown.**
* `SOW_QPD_2`, `SOW_QPD_3` = the same window moved back one and two months (payments Jun-Aug / statements May-Jul; payments May-Jul / statements Apr-Jun). Rolling windows, so they overlap.
* `SOW_QPD_PRIOR` = the earlier, non-overlapping window. Only **two** months are possible: payments `PAY_AMT4 + PAY_AMT5` (paid in Jun and May) / statements `BILL_AMT5 + BILL_AMT6` (May and Apr), because the March statement that `PAY_AMT6` settles is not in the data.
* `SOW_QPD_CHG` = `SOW_QPD_1 − SOW_QPD_PRIOR` (is paydown speeding up or slowing down?). `SOW_QPD_BAND`, `SOW_QPD_CHG_BAND` = readable bands.

**Rules and caveats.**
1. A negative statement (credit balance) counts as 0 in the denominator: a credit is not debt to pay down.
2. If the three statements sum to 0 or less there is nothing to pay down: QPD is missing and the band is "No balance".
3. QPD above 100% is possible (paying off older debt, or prepaying new purchases). The column is capped at 5 (500%) to stop tiny balances from making huge ratios.
4. How it differs from what exists: `PAY_RATIO_i` is the same idea for **one** month (noisy); `PAY_TO_BILL_6M` sums six months **without** the one-month shift. QPD is the smoothed, correctly lagged version.

In [ ]:
S6Q_P = df[PAMT_COLS].to_numpy(float)                 # col 0 = PAY_AMT1 (Sep) ... col 5 = PAY_AMT6 (Apr)
S6Q_B = np.clip(df[BILL_COLS].to_numpy(float), 0, None)  # col 0 = BILL_AMT1 (Sep) ... col 5 = BILL_AMT6 (Apr); credits -> 0

def s6q_ratio(pay_cols, bill_cols):
    pay = S6Q_P[:, pay_cols].sum(1); bal = S6Q_B[:, bill_cols].sum(1)
    return np.where(bal > 0, np.minimum(pay / np.where(bal > 0, bal, 1), 5), np.nan)

df["SOW_QPD_1"] = s6q_ratio([0, 1, 2], [1, 2, 3])        # pays Sep+Aug+Jul / bills Aug+Jul+Jun
df["SOW_QPD_2"] = s6q_ratio([1, 2, 3], [2, 3, 4])        # pays Aug+Jul+Jun / bills Jul+Jun+May
df["SOW_QPD_3"] = s6q_ratio([2, 3, 4], [3, 4, 5])        # pays Jul+Jun+May / bills Jun+May+Apr
df["SOW_QPD_PRIOR"] = s6q_ratio([3, 4], [4, 5])          # pays Jun+May / bills May+Apr (2-month window)
df["SOW_QPD_CHG"] = df["SOW_QPD_1"] - df["SOW_QPD_PRIOR"]

S6Q_EDGES = [-np.inf, 0, 0.05, 0.10, 0.25, 0.50, 0.90, 1.10, np.inf]
S6Q_LABELS = ["0 paid", "0-5%", "5-10%", "10-25%", "25-50%", "50-90%", "90-110% (paid off)", ">110%"]
S6Q_ORDER = S6Q_LABELS + ["No balance"]
df["SOW_QPD_BAND"] = pd.cut(df["SOW_QPD_1"], S6Q_EDGES, labels=S6Q_LABELS, right=True).cat.add_categories("No balance").fillna("No balance")
S6Q_CHG_ORDER = ["Fell > 25 pp", "Fell 5-25 pp", "Stable (±5 pp)", "Rose 5-25 pp", "Rose > 25 pp", "n/a"]
df["SOW_QPD_CHG_BAND"] = pd.cut(df["SOW_QPD_CHG"], [-np.inf, -0.25, -0.05, 0.05, 0.25, np.inf], labels=S6Q_CHG_ORDER[:-1]).cat.add_categories("n/a").fillna("n/a")

# self-check: recompute the headline ratio for every client the slow way
_chk = (df["PAY_AMT1"] + df["PAY_AMT2"] + df["PAY_AMT3"]) / (df[["BILL_AMT2", "BILL_AMT3", "BILL_AMT4"]].clip(lower=0).sum(1))
_ok = np.allclose(np.minimum(_chk.where(df[["BILL_AMT2", "BILL_AMT3", "BILL_AMT4"]].clip(lower=0).sum(1) > 0), 5), df["SOW_QPD_1"], equal_nan=True)
print("formula self-check passed:", _ok)

# worked examples: one client per behaviour, chosen deterministically (lowest ID)
S6Q_EX = {"pays it all off": df["SOW_QPD_1"].between(0.98, 1.02),
          "pays a small slice": df["SOW_QPD_1"].between(0.03, 0.06),
          "pays nothing": df["SOW_QPD_1"] == 0,
          "no balance to pay": df["SOW_QPD_1"].isna()}
for lab, m in S6Q_EX.items():
    r = df.loc[m].sort_values("ID").iloc[0]
    print(f"\n--- ID {int(r['ID'])}: {lab} (default = {int(r[TARGET])})")
    print(f"  payments  Jul {r['PAY_AMT3']:>9,.0f} | Aug {r['PAY_AMT2']:>9,.0f} | Sep {r['PAY_AMT1']:>9,.0f}  -> sum {r['PAY_AMT1']+r['PAY_AMT2']+r['PAY_AMT3']:>10,.0f}")
    print(f"  bills     Jun {r['BILL_AMT4']:>9,.0f} | Jul {r['BILL_AMT3']:>9,.0f} | Aug {r['BILL_AMT2']:>9,.0f}  -> sum {max(r['BILL_AMT2'],0)+max(r['BILL_AMT3'],0)+max(r['BILL_AMT4'],0):>10,.0f} (credits as 0)")
    print(f"  SOW_QPD_1 = {r['SOW_QPD_1']:.3f}  band = {r['SOW_QPD_BAND']}   earlier 2-month paydown = {r['SOW_QPD_PRIOR']:.3f}")
display(df[["SOW_QPD_1", "SOW_QPD_2", "SOW_QPD_3", "SOW_QPD_PRIOR", "SOW_QPD_CHG"]].describe(percentiles=[.1, .25, .5, .75, .9]).T.round(3))

**Does a higher quarterly paydown mean lower default?** Default rate by paydown band (bars = clients, line = default rate with 95% CI),
and how much signal it carries next to the one-month and six-month repayment measures (Information Value, IV: below 0.1 weak, 0.1-0.3 medium, above 0.3 strong).

In [ ]:
s6q_tab = rate_table(df, "SOW_QPD_BAND")
s6q_tab = s6q_tab.set_index("SOW_QPD_BAND").reindex([b for b in S6Q_ORDER if b in set(s6q_tab["SOW_QPD_BAND"])]).reset_index()
display(s6q_tab.round(4))
fig, ax = plt.subplots(figsize=(12, 4.8))
plot_rate(df, "SOW_QPD_BAND", ax=ax, order=S6Q_ORDER, title="Default rate by quarterly paydown (payments Jul-Sep / statements Jun-Aug)", rotate=20)
plt.tight_layout(); savefig("s06_qpd_band"); plt.show()

s6q_iv_rows = []
for c, lab in [("SOW_QPD_1", "Quarterly paydown (3 m, lagged)"), ("PAY_RATIO_1", "Monthly paydown, Sep (1 m)"),
               ("AVG_PAY_RATIO", "Mean of 5 monthly paydowns"), ("PAY_TO_BILL_6M", "6-month pay / bill (not lagged)"),
               ("SOW_QPD_PRIOR", "Earlier paydown (Apr-May bills)"), ("SOW_QPD_CHG", "Change in paydown")]:
    _d = df[[c, TARGET]].copy(); _d[c] = _d[c].fillna(-1)          # missing = own bin
    s6q_iv_rows.append(dict(measure=lab, column=c, iv=woe_iv(_d, c)[1], share_missing=float(df[c].isna().mean())))
s6q_iv = pd.DataFrame(s6q_iv_rows).sort_values("iv", ascending=False)
display(s6q_iv.round(4))
export(s6q_tab.assign(SOW_QPD_BAND=lambda t: t["SOW_QPD_BAND"].astype(str)), "s06_qpd_band_rates")

**Observations:**
- The formula check passes for all 30,000 clients. Example: client 4 paid NT$ 5,219 against NT$ 125,838 of statements, so the quarterly paydown is 4.1%.
- Half of all clients pay back less than 9.4% of their balances in a quarter. 28% sit at 0-5%, which matches a minimum payment of about 4% a month.
- Risk falls steadily as paydown rises. Paying nothing gives 67.9% default (330 clients). Paying 0-5% gives 26.2%. Paying 25-50% gives 15.4%. Paying it all off gives 13.6%.
- "No balance" clients (1,858) default at 28.1%, above the base rate. An empty card is not a safe card (sections 16 and 19 say the same).
- The quarterly paydown carries about the same signal as the one-month ratio (IV 0.17 against 0.18). Its value is that it is smoother and easier to explain, not that it is stronger.
- Story use: chapter 5 (the wallet). "How much of the debt do they pay down?" is a clean, simple measure for a Tableau bar chart.

**Does paydown still matter once we know the September status?** Default rate on the grid September status × paydown band
(cells under 50 clients blanked). If the colours change along each row, paydown adds information beyond status.

In [ ]:
s6q_grid = rate_table(df, ["STATUS_GRP_SEP", "SOW_QPD_BAND"])
s6q_piv = s6q_grid.pivot(index="STATUS_GRP_SEP", columns="SOW_QPD_BAND", values="rate")
s6q_n = s6q_grid.pivot(index="STATUS_GRP_SEP", columns="SOW_QPD_BAND", values="n")
s6q_piv = s6q_piv.where(s6q_n >= 50)
s6q_piv = s6q_piv.reindex(index=[g for g in STATUS_GROUP_ORDER if g in s6q_piv.index], columns=[b for b in S6Q_ORDER if b in s6q_piv.columns])
s6q_n = s6q_n.reindex(index=s6q_piv.index, columns=s6q_piv.columns)
fig, ax = plt.subplots(figsize=(13, 4.8))
sns.heatmap(s6q_piv, annot=s6q_piv.map(lambda v: "" if pd.isna(v) else f"{v:.0%}") if hasattr(s6q_piv, "map") else True,
            fmt="", cmap=CMAP_RATE, vmin=0, vmax=0.8, ax=ax, cbar_kws={"format": mtick.PercentFormatter(1)})
ax.set_title("Default rate: September status x quarterly paydown band (blank = fewer than 50 clients)"); ax.set_xlabel(""); ax.set_ylabel("")
plt.xticks(rotation=20, ha="right"); plt.tight_layout(); savefig("s06_qpd_status_grid"); plt.show()
print("default rate (rows = Sep status, columns = paydown band):"); display(s6q_piv.round(3))
print("clients per cell:"); display(s6q_n)
s6q_g = s6q_grid.copy(); s6q_g[["STATUS_GRP_SEP", "SOW_QPD_BAND"]] = s6q_g[["STATUS_GRP_SEP", "SOW_QPD_BAND"]].astype(str)
export(s6q_g, "s06_qpd_status_grid")

**Observations:**
- For revolvers, paydown changes little: default stays between 8% and 14% in every band.
- For clients who are already late, paydown matters a lot:
  - **1 month late:** 40.6% default at 0-5% paydown, against 18.2% if they paid the quarter off.
  - **2 months late:** 69.0% default at 0-5% paydown, against 40.9% if they paid it off.
- So paydown adds real information after the September status, but mainly for late clients. A late client who still pays down hard is half as risky.
- Story use: a strong "two clients, same status, different ending" slide.

**Is the client paying down faster or slower than before, and does the direction matter?** Left: default rate by the change from
the earlier paydown (Apr-May statements) to the latest quarter. Right: median paydown in each rolling window, defaulters vs others.

In [ ]:
s6q_chg = rate_table(df, "SOW_QPD_CHG_BAND")
s6q_chg = s6q_chg.set_index("SOW_QPD_CHG_BAND").reindex([b for b in S6Q_CHG_ORDER if b in set(s6q_chg["SOW_QPD_CHG_BAND"])]).reset_index()
display(s6q_chg.round(4))
s6q_roll = pd.DataFrame([dict(window=w, group=g, median_qpd=float(df.loc[df[TARGET] == k, c].median()))
                         for w, c in [("Apr-May bills (prior)", "SOW_QPD_PRIOR"), ("Apr-Jun bills", "SOW_QPD_3"), ("May-Jul bills", "SOW_QPD_2"), ("Jun-Aug bills (latest)", "SOW_QPD_1")]
                         for k, g in [(1, "Defaulters"), (0, "Others")]])
display(s6q_roll.pivot(index="window", columns="group", values="median_qpd").reindex(s6q_roll["window"].unique()).round(3))
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
plot_rate(df, "SOW_QPD_CHG_BAND", ax=axes[0], order=S6Q_CHG_ORDER, title="Default rate by change in paydown (latest vs earlier)", rotate=15)
for g, col in [("Defaulters", C_DEF), ("Others", C_OK)]:
    t = s6q_roll[s6q_roll["group"] == g]
    axes[1].plot(t["window"], t["median_qpd"], marker="o", color=col, label=g, lw=2)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].legend(); axes[1].tick_params(axis="x", rotation=15)
axes[1].set_title("Median paydown per rolling window")
plt.tight_layout(); savefig("s06_qpd_change"); plt.show()
export(s6q_chg.assign(SOW_QPD_CHG_BAND=lambda t: t["SOW_QPD_CHG_BAND"].astype(str)), "s06_qpd_change_rates")
export(s6q_roll, "s06_qpd_rolling_medians")

# story facts (computed, never typed)
_r = s6q_tab.set_index("SOW_QPD_BAND")["rate"]; _n = s6q_tab.set_index("SOW_QPD_BAND")["n"]
_lo = df["SOW_QPD_1"] < 0.05; _hi = df["SOW_QPD_1"] >= 0.9
add_fact("6.4c", "wallet repayment share", f"Quarterly paydown: clients who repaid under 5% of three months of balances default at {df.loc[_lo, TARGET].mean():.1%} ({int(_lo.sum()):,} clients) vs {df.loc[_hi, TARGET].mean():.1%} for those who paid 90% or more ({int(_hi.sum()):,})", float(df.loc[_lo, TARGET].mean()), "s06_qpd_band", 3)
_best = s6q_iv.iloc[0]
add_fact("6.4c", "wallet repayment share", f"Among repayment measures, '{_best['measure']}' carries the most signal (IV {_best['iv']:.2f}); the quarterly paydown has IV {float(s6q_iv.set_index('column').loc['SOW_QPD_1', 'iv']):.2f} vs {float(s6q_iv.set_index('column').loc['PAY_RATIO_1', 'iv']):.2f} for the one-month ratio", float(_best["iv"]), "s06_qpd_band", 2)

**Observations:**
- Defaulters pay down about half as much as others in every window. Their median is 4.8% of balances in the earliest window and 6.3% in the latest. Others go from 8.0% to 11.4%.
- The direction of change is a weak signal. A paydown that rose by more than 25 points gives 14.2% default. A stable one gives 23.3%. A paydown that fell by more than 25 points gives 18.7%, which is lower than stable, not higher.
- Most clients (55%) keep the same paydown within ±5 points. Paydown is a habit, so its level matters more than its change.
- Story use: "the habit matters, not the wobble". Use the level (SOW_QPD_1) in the dashboard. Leave the change as a tooltip.

### 6.5 Proxy D — Relative wallet within the peer group
Peer cell = `AGE_BAND × EDU_LABEL × SEX_LABEL × MAR_LABEL` (cells with < 30 clients fall back to `AGE_BAND × EDU_LABEL`).
* `SOW_PEER_LIMIT_IDX` = client limit / peer median limit — "how big a line did the bank give relative to similar people?"
* `SOW_PEER_BILL_IDX` = client avg positive bill / peer median — "how much of the peer-typical wallet does the client actually carry here?"

**6.5a** Build the indices and test: is a client with a large line *relative to peers* safer?

In [ ]:
PEER_KEYS = ["AGE_BAND", "EDU_LABEL", "SEX_LABEL", "MAR_LABEL"]
MIN_CELL = 30
avg_pos_bill = np.clip(Bm, 0, None).mean(1)
df["SOW_AVG_POS_BILL"] = avg_pos_bill
gfine = df.groupby(PEER_KEYS, observed=True)
gcoarse = df.groupby(["AGE_BAND", "EDU_LABEL"], observed=True)
n_cell = gfine["LIMIT_BAL"].transform("size")
use_fine = n_cell >= MIN_CELL
peer_lim = np.where(use_fine, gfine["LIMIT_BAL"].transform("median"), gcoarse["LIMIT_BAL"].transform("median"))
peer_bill = np.where(use_fine, gfine["SOW_AVG_POS_BILL"].transform("median"), gcoarse["SOW_AVG_POS_BILL"].transform("median"))
df["SOW_PEER_CELL_N"] = n_cell
df["SOW_PEER_LIMIT_MED"] = peer_lim
df["SOW_PEER_LIMIT_IDX"] = df["LIMIT_BAL"] / peer_lim
df["SOW_PEER_BILL_IDX"] = avg_pos_bill / np.maximum(peer_bill, 1_000)
IDX_BINS = [0, .5, .8, 1.25, 2, np.inf]
IDX_LABELS = ["<0.5x", "0.5-0.8x", "0.8-1.25x", "1.25-2x", ">2x"]
df["SOW_PEER_LIMIT_BAND"] = pd.cut(df["SOW_PEER_LIMIT_IDX"], IDX_BINS, labels=IDX_LABELS)
df["SOW_PEER_BILL_BAND"] = pd.cut(df["SOW_PEER_BILL_IDX"], [-np.inf, .1] + IDX_BINS[1:], labels=["<0.1x (dormant)", "0.1-0.5x"] + IDX_LABELS[1:])
print(f"peer cells: {gfine.ngroups} | clients in cells >= {MIN_CELL}: {use_fine.mean():.1%} (rest use AGE x EDU)")
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
tD1 = plot_rate(df, "SOW_PEER_LIMIT_BAND", ax=axes[0], title="Default by limit relative to peer median")
tD2 = plot_rate(df, "SOW_PEER_BILL_BAND", ax=axes[1], title="Default by avg bill relative to peer median", rotate=15)
plt.tight_layout(); savefig("s06_peer_index"); plt.show()
display(tD1[["SOW_PEER_LIMIT_BAND", "n", "defaults", "rate", "ci_lo", "ci_hi", "share_clients", "lift"]])
display(tD2[["SOW_PEER_BILL_BAND", "n", "defaults", "rate", "share_clients", "lift"]])

**Observations:**
- The data forms 177 peer cells, and 97.3% of clients sit in cells with at least 30 members.
- A small line relative to peers is risky: under 0.5x the peer median gives 31.8% default (24.6% of clients), and over 2x gives 14.3%. The pattern is cleanly monotone.
- Relative bill size says little: 19.7–26.3% across bands, with dormant clients (<0.1x) highest at 26.3%.
- Story use: the bank's limit decision already encodes risk beyond demographics. The peer-relative limit is a Tableau-friendly KPI.

**6.5b** Relative line × relative usage. The quadrant "large line vs peers, small usage vs peers" is the bank's *under-penetrated* wallet; "small line, heavy usage" is where the client may be borrowing elsewhere too (a classic low-SoW / high-total-debt signature).

In [ ]:
g = rate_table(df, ["SOW_PEER_LIMIT_BAND", "SOW_PEER_BILL_BAND"])
rp = g.pivot(index="SOW_PEER_LIMIT_BAND", columns="SOW_PEER_BILL_BAND", values="rate")
npv = g.pivot(index="SOW_PEER_LIMIT_BAND", columns="SOW_PEER_BILL_BAND", values="n")
ann = _fmt(rp, lambda v: "" if pd.isna(v) else f"{v:.0%}") + "\n" + _fmt(npv, lambda v: "" if pd.isna(v) else f"{int(v)}")
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(rp.where(npv >= 50), annot=ann.where(npv >= 50, ""), fmt="", cmap=CMAP_RATE, ax=ax,
            cbar_kws={"format": mtick.PercentFormatter(1)}, annot_kws={"fontsize": 8})
ax.set_title("Default rate: limit vs peers (rows) x bill vs peers (cols); n>=50")
ax.set_xlabel("avg bill / peer median"); ax.set_ylabel("limit / peer median")
plt.tight_layout(); savefig("s06_peer_grid"); plt.show()
display(rp.round(3))

**Observations:**
- Line size dominates: clients with under 0.5x the peer line default at 30–34% at any usage level.
- A large line with modest usage (>2x line, 0.5–0.8x bill) is the safest cell at 5.4%. A large line with heavy usage (>2x bill) rises to 14.9%.
- The dormant column sits at 24–33% in every row, so a large unused line is not safe.
- Story use: the "under-penetrated" wallet (large line, low but non-zero usage) is the low-risk growth target.

**6.5c** Model-based wallet estimate (quantile regression on demographics, following the "realistic wallet = high quantile among similar customers" idea).
* **Limit residual**: median (q = 0.5) regression of `log(LIMIT_BAL)` on peer dummies; `SOW_LIMIT_RESID` = actual − predicted (log points). Positive = the bank granted more than demographics alone justify (**over-limited vs peers**), negative = under-limited.
* **Wallet estimate**: q = 0.9 regression of `log(1 + peak positive bill)`; `SOW_WALLET_EST` = the balance that the top-decile of similar clients carries — a proxy for the client's borrowing capacity on cards; `SOW_WALLET_SHARE` = avg positive bill / wallet estimate.

Falls back to peer-cell quantiles if `statsmodels` is unavailable.

In [ ]:
Xd = pd.get_dummies(df[PEER_KEYS].astype(str), drop_first=True).astype(float)
Xd.insert(0, "const", 1.0)
y_lim = np.log(df["LIMIT_BAL"].astype(float))
y_wal = np.log1p(np.clip(Bm, 0, None).max(1))
try:
    import statsmodels.api as sm
    qr50 = sm.QuantReg(y_lim, Xd).fit(q=0.5, max_iter=5000)
    qr90 = sm.QuantReg(y_wal, Xd).fit(q=0.9, max_iter=5000)
    pred_lim, pred_wal = qr50.predict(Xd), qr90.predict(Xd)
    WALLET_METHOD = "statsmodels QuantReg (q=0.5 limit, q=0.9 wallet)"
    coef = pd.DataFrame({"limit_multiplier_q50": np.exp(qr50.params), "wallet_multiplier_q90": np.exp(qr90.params)})
    print("Top drivers (multiplicative effect vs reference cell):")
    display(coef.drop("const").assign(absdev=lambda d: (np.log(d["limit_multiplier_q50"])).abs())
            .sort_values("absdev", ascending=False).drop(columns="absdev").head(12).round(3))
    ss_res = np.abs(y_lim - pred_lim).sum(); ss_tot = np.abs(y_lim - np.median(y_lim)).sum()
    print(f"limit model pseudo-R1 (Koenker-Machado): {1 - ss_res/ss_tot:.3f}  -> demographics explain little of the line")
except Exception as e:
    print("QuantReg unavailable, falling back to peer-cell quantiles:", e)
    tmpq = pd.DataFrame({"yl": y_lim, "yw": y_wal}).join(df[PEER_KEYS])
    pred_lim = tmpq.groupby(PEER_KEYS, observed=True)["yl"].transform("median")
    pred_wal = tmpq.groupby(PEER_KEYS, observed=True)["yw"].transform(lambda s: s.quantile(.9))
    WALLET_METHOD = "peer-cell quantiles"
df["SOW_LIMIT_RESID"] = np.asarray(y_lim - pred_lim)
df["SOW_WALLET_EST"] = np.expm1(np.asarray(pred_wal))
df["SOW_WALLET_SHARE"] = avg_pos_bill / np.maximum(df["SOW_WALLET_EST"], 1_000)
df["SOW_LIMIT_RESID_BAND"] = pd.cut(df["SOW_LIMIT_RESID"], [-np.inf, -.5, -.2, .2, .5, np.inf],
                                    labels=["Under-limited (<-0.5)", "Mildly under", "In line (+/-0.2)", "Mildly over", "Over-limited (>0.5)"])
df["SOW_WALLET_SHARE_BAND"] = pd.cut(df["SOW_WALLET_SHARE"], [-np.inf, .01, .1, .25, .5, 1.0, np.inf],
                                     labels=["~0 (<1%)", "1-10%", "10-25%", "25-50%", "50-100%", ">100%"])
print("method:", WALLET_METHOD)
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
for lab in ["No default", "Default"]:
    sns.kdeplot(df.loc[df.DEFAULT_LABEL == lab, "SOW_LIMIT_RESID"], ax=axes[0], fill=True, alpha=.3, color=PAL_TARGET[lab], label=lab)
axes[0].axvline(0, color="k", lw=1); axes[0].set_title("Limit residual vs demographic peers (log points)"); axes[0].legend()
tD3 = plot_rate(df, "SOW_LIMIT_RESID_BAND", ax=axes[1], title="Default by over/under-limiting vs peers", rotate=20)
tD4 = plot_rate(df, "SOW_WALLET_SHARE_BAND", ax=axes[2], title="Default by estimated wallet share (avg bill / P90 peer wallet)", rotate=20)
plt.tight_layout(); savefig("s06_wallet_model"); plt.show()
display(tD3[["SOW_LIMIT_RESID_BAND", "n", "defaults", "rate", "ci_lo", "ci_hi", "share_clients", "lift"]])
display(tD4[["SOW_WALLET_SHARE_BAND", "n", "defaults", "rate", "share_clients", "lift"]])

**Observations:**
- Demographics explain little of the limit (pseudo-R1 0.134). Age is the main driver: clients aged 31–40 get about 3x the line of 21–25 year-olds, and a high-school education means 0.46x.
- The limit residual compares the actual line with the line predicted from demographics. Under-limited clients (<−0.5) default at 30.9% vs 15.2% for over-limited ones, so the bank's own judgement adds risk information.
- The estimated wallet share (Sep bill ÷ 90th-percentile peer wallet) is flat at 19–26%, a weak signal.
- Story use: supports "the bank already prices risk into limits". Leave the wallet-share estimate out of the story.

**6.5d** Is the limit-residual effect just the limit effect in disguise? Default rate by limit residual band **within** limit bands (heatmap). If the residual still orders risk inside a limit band, then "being given more than your peers" carries information beyond the absolute line.

In [ ]:
g = rate_table(df, ["LIMIT_BAND", "SOW_LIMIT_RESID_BAND"])
rp = g.pivot(index="LIMIT_BAND", columns="SOW_LIMIT_RESID_BAND", values="rate")
npv = g.pivot(index="LIMIT_BAND", columns="SOW_LIMIT_RESID_BAND", values="n")
ann = _fmt(rp, lambda v: "" if pd.isna(v) else f"{v:.0%}") + "\n" + _fmt(npv, lambda v: "" if pd.isna(v) else f"{int(v)}")
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(rp.where(npv >= 50), annot=ann.where(npv >= 50, ""), fmt="", cmap=CMAP_RATE, ax=ax,
            cbar_kws={"format": mtick.PercentFormatter(1)}, annot_kws={"fontsize": 8})
ax.set_title("Default rate: limit band x limit residual vs peers (n>=50)"); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); savefig("s06_resid_within_limit"); plt.show()
display(rp.round(3))

**Observations:**
- Within each limit band the residual still orders risk, but only mildly: in 100-150k, 25.8% under-limited vs 19.9% over-limited; in ≤50k, 32.4% under-limited vs 25.1% in line.
- Many cells are empty because small lines are almost always "under" and big lines "over". The residual and the limit are tangled together.
- Story use: background only. Most of the residual's signal is the limit itself.

### 6.6 Proxy E — Portfolio share of wallet: where the money and the at-risk money sit
Exposure = `max(BILL_AMT1, 0)` (Sep balance). EAD = exposure of clients who default in October. **6.6a** Concentration: Lorenz curves and Gini for exposure, limits and EAD, and a Pareto (clients ranked by exposure, descending) showing how fast exposure, EAD and defaulter counts accumulate.

In [ ]:
def lorenz(v):
    v = np.sort(np.asarray(v, float))
    cum = np.concatenate([[0], np.cumsum(v)]) / v.sum()
    x = np.linspace(0, 1, len(cum))
    gini = 1 - 2 * float(((cum[1:] + cum[:-1]) / 2 * np.diff(x)).sum())
    return x, cum, gini
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
gini_rows = []
for col, lab, c in [("SOW_EXPOSURE_SEP", "Sep exposure", C_OK), ("LIMIT_BAL", "Credit limit", C_ACC), ("SOW_EAD", "Exposure at default", C_DEF)]:
    x, cum, gi = lorenz(df[col])
    axes[0].plot(x, cum, lw=2, color=c, label=f"{lab} (Gini {gi:.2f})")
    gini_rows.append({"measure": lab, "gini": gi})
axes[0].plot([0, 1], [0, 1], ls="--", color="k", lw=1)
axes[0].set_title("Lorenz curves (clients sorted ascending)"); axes[0].legend()
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
o = df.sort_values("SOW_EXPOSURE_SEP", ascending=False)
xr = np.arange(1, len(o) + 1) / len(o)
for col, lab, c in [("SOW_EXPOSURE_SEP", "exposure", C_OK), ("SOW_EAD", "EAD (NT$)", C_DEF), (TARGET, "defaulters (count)", "#fdae6b")]:
    axes[1].plot(xr, o[col].cumsum() / o[col].sum(), lw=2, color=c, label=f"cum. share of {lab}")
axes[1].plot([0, 1], [0, 1], ls="--", color="k", lw=1)
axes[1].set_title("Pareto: clients ranked by Sep exposure (largest first)"); axes[1].legend()
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_xlabel("top x% of clients by exposure")
plt.tight_layout(); savefig("s06_lorenz_pareto"); plt.show()
display(pd.DataFrame(gini_rows).round(3))
top_rows = []
tot_e, tot_ead, tot_d = o["SOW_EXPOSURE_SEP"].sum(), o["SOW_EAD"].sum(), o[TARGET].sum()
for q in [.01, .05, .10, .20, .50]:
    h = o.head(int(round(q * len(o))))
    top_rows.append({"top clients": f"{q:.0%}", "share_exposure": h["SOW_EXPOSURE_SEP"].sum() / tot_e,
                     "share_EAD": h["SOW_EAD"].sum() / tot_ead, "share_defaulters": h[TARGET].sum() / tot_d,
                     "default_rate": h[TARGET].mean(), "min_exposure_NT$": h["SOW_EXPOSURE_SEP"].min()})
print(f"Total Sep exposure NT$ {tot_e:,.0f} | total EAD NT$ {tot_ead:,.0f} ({tot_ead/tot_e:.1%} of exposure) | zero-exposure clients {(o.SOW_EXPOSURE_SEP==0).mean():.1%}")
display(pd.DataFrame(top_rows).round(3))

**Observations:**
- Total Sep exposure is NT$1.54bn. Exposure at default (EAD, the Sep balance of clients who default) is NT$322m, or 20.9% of exposure, close to the 22.1% client default rate.
- Exposure is concentrated (Gini 0.65): the top 10% of clients hold 44.8% of it, and the top 20% hold 66.1%. EAD has a Gini of 0.925 because most clients carry no EAD.
- Big balances do not default more: the top 1% default at 25.0% and the top 20% at 19.7%, both near base. The top 10% hold 44% of EAD but only 9% of defaulters.
- 8.7% of clients have zero exposure.
- Story use: Lorenz curve. NT$ at risk is driven by a few big balances, not by the number of defaulters.

**6.6b** Segment share table: for each segment, share of clients vs share of exposure vs share of defaults vs share of EAD, the count-based default rate and the **exposure-weighted** default rate (EAD / exposure). `risk_concentration` = share of EAD / share of clients (> 1 means the segment holds more at-risk money than its headcount). *Tableau: this long table drives a "share comparison" bar sheet and a treemap.*

In [ ]:
SEG_DIMS = ["LIMIT_BAND", "AGE_BAND", "EDU_LABEL", "SEX_LABEL", "MAR_LABEL", "STATUS_GRP_SEP", "UTIL_BAND",
            "SOW_PAYER_TYPE", "SOW_CAPTURE_BAND", "SOW_PEER_LIMIT_BAND", "SOW_DYN_QUADRANT"]
tot_n = len(df)
def seg_share(dim_cols, label=None):
    dim_cols = [dim_cols] if isinstance(dim_cols, str) else list(dim_cols)
    g = df.groupby(dim_cols, observed=True).agg(n=(TARGET, "size"), defaults=(TARGET, "sum"),
                                                exposure=("SOW_EXPOSURE_SEP", "sum"), ead=("SOW_EAD", "sum"),
                                                limit_total=("LIMIT_BAL", "sum")).reset_index()
    out = pd.DataFrame({"dimension": label or " > ".join(dim_cols),
                        "segment": g[dim_cols].astype(str).agg(" > ".join, axis=1)})
    for k, c in enumerate(dim_cols[:2]):
        out[f"level_{k+1}"] = g[c].astype(str)
    out = pd.concat([out, g[["n", "defaults", "exposure", "ead", "limit_total"]]], axis=1)
    out["share_clients"] = out["n"] / tot_n
    out["share_defaults"] = out["defaults"] / df[TARGET].sum()
    out["share_exposure"] = out["exposure"] / df["SOW_EXPOSURE_SEP"].sum()
    out["share_ead"] = out["ead"] / df["SOW_EAD"].sum()
    out["share_limit"] = out["limit_total"] / df["LIMIT_BAL"].sum()
    out["default_rate"] = out["defaults"] / out["n"]
    out["ew_default_rate"] = np.where(out["exposure"] > 0, out["ead"] / out["exposure"].where(out["exposure"] > 0, 1), np.nan)
    out["portfolio_util"] = out["exposure"] / out["limit_total"]
    out["risk_concentration"] = out["share_ead"] / out["share_clients"]
    return out
seg = pd.concat([seg_share(d) for d in SEG_DIMS], ignore_index=True)
show = ["segment", "n", "share_clients", "share_exposure", "share_defaults", "share_ead", "default_rate", "ew_default_rate", "portfolio_util", "risk_concentration"]
for d in ["LIMIT_BAND", "STATUS_GRP_SEP", "UTIL_BAND", "SOW_PAYER_TYPE", "AGE_BAND", "EDU_LABEL"]:
    print(f"--- {d} ---"); display(seg.loc[seg.dimension == d, show].round(3))

**Observations:**
- Small lines (≤50k) are 25.6% of clients and 36.8% of defaults but only 16.0% of EAD (risk concentration 0.63): many defaults, little money.
- Clients Late 2m in Sep are 8.9% of clients and 11.3% of exposure, but 37.2% of EAD. That is a risk concentration (share of EAD ÷ share of exposure) of 4.2x.
- Minimum payers are 40.2% of clients, 76.6% of exposure and 82.9% of EAD, so nearly all of the bank's at-risk money sits with them. Transactors hold only 4.9% of exposure and 2.3% of EAD.
- Utilisation of 70% or more covers 24% of clients and 59% of EAD.
- Story use: central SoW slide. Counting defaults points to small lines; NT$ at risk points to minimum payers and high-utilisation revolvers.

**6.6c** Visual share comparison for six key dimensions: grey = share of clients, blue = share of exposure, red = share of EAD. Where red towers over grey, at-risk money is concentrated.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(19, 9))
for ax, d in zip(axes.ravel(), ["LIMIT_BAND", "AGE_BAND", "EDU_LABEL", "STATUS_GRP_SEP", "UTIL_BAND", "SOW_PAYER_TYPE"]):
    s = seg[seg.dimension == d].reset_index(drop=True)
    if d == "STATUS_GRP_SEP":
        s = s.set_index("segment").reindex([x for x in STATUS_GROUP_ORDER if x in set(s.segment)]).reset_index()
    if d == "SOW_PAYER_TYPE":
        s = s.set_index("segment").reindex([x for x in PAYER_ORDER if x in set(s.segment)]).reset_index()
    x = np.arange(len(s)); w = .27
    ax.bar(x - w, s["share_clients"], w, color=C_NEU, label="clients")
    ax.bar(x, s["share_exposure"], w, color=C_OK, label="exposure")
    ax.bar(x + w, s["share_ead"], w, color=C_DEF, label="EAD")
    ax.set_xticks(x); ax.set_xticklabels(s["segment"], rotation=30, ha="right", fontsize=8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title(d)
axes[0, 0].legend()
plt.suptitle("Portfolio share of wallet: clients vs exposure vs exposure-at-default", fontweight="bold")
plt.tight_layout(); savefig("s06_segment_shares"); plt.show()

**Observations:**
- The chart shows the 6.6b table visually. Red (EAD share) towers over grey (client share) for Sep Late 2m (37.2% vs 8.9%), minimum payers (82.9% vs 40.2%) and utilisation of 70% or more (59% vs 24%).
- Grey towers over red for transactors (26.7% vs 2.3%), paid in full (19.0% vs 2.2%) and ≤50k lines (25.6% vs 16.0%).
- Age and education bars are nearly equal: demographics do not concentrate at-risk money.
- Story use: a ready-made Tableau design (grouped bars with three shares per segment).

**6.6d** Treemap: limit band → September status, sized by exposure, coloured by exposure-weighted default rate (interactive, Plotly; skipped gracefully if Plotly is absent). The same hierarchy is in the export with `level_1`/`level_2` columns for a Tableau treemap.

In [ ]:
tree = seg_share(["LIMIT_BAND", "STATUS_GRP_SEP"])
tree = tree[tree["exposure"] > 0]
top_tree = tree.sort_values("ead", ascending=False).head(10)
display(top_tree[["segment", "n", "share_exposure", "share_ead", "default_rate", "ew_default_rate"]].round(3))
try:
    import plotly.express as px
    figp = px.treemap(tree, path=[px.Constant("Portfolio"), "level_1", "level_2"], values="exposure",
                      color="ew_default_rate", color_continuous_scale="Reds",
                      hover_data={"n": True, "default_rate": ":.1%", "share_ead": ":.1%"},
                      title="Exposure treemap: limit band > Sep status (colour = exposure-weighted default rate)")
    figp.update_layout(height=520, margin=dict(t=50, l=10, r=10, b=10))
    figp.show()
except Exception as e:
    print("plotly treemap skipped:", e)

**Observations:**
- The largest exposure blocks are big-line revolvers: 300-500k Revolving holds 18.1% of exposure and 200-300k Revolving 16.9%, and both default at only 7–8%.
- Late 2m blocks are small in exposure (1.6–2.3%) but heavy in EAD: 50-100k Late 2m holds 7.8% of EAD at a 71.7% default rate.
- Big-line revolvers still carry 8.7% and 8.1% of EAD, because a low rate on a big balance adds up.
- Story use: the treemap is a strong Tableau visual of size vs risk.

### 6.7 SoW dynamics: is the card capturing more or less of the wallet, and is repayment keeping up?
**6.7a** Utilisation change Apr → Sep, and "wallet stretch" (months with utilisation > 90%).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
tY1 = plot_rate(df, "SOW_UTIL_CHG_BAND", ax=axes[0], title="Default by capture change Apr -> Sep", rotate=15)
tY2 = plot_rate(df, "SOW_STRETCH_BAND", ax=axes[1], title="Default by # months with capture > 90% (wallet stretch)")
plt.tight_layout(); savefig("s06_dynamics_bands"); plt.show()
display(tY1[["SOW_UTIL_CHG_BAND", "n", "defaults", "rate", "ci_lo", "ci_hi", "share_clients", "lift"]])
display(tY2[["SOW_STRETCH_BAND", "n", "defaults", "rate", "ci_lo", "ci_hi", "share_clients", "lift"]])

**Observations:**
- The change in capture from Apr to Sep is not monotone: clients whose capture fell 5–20 pp default most (31.7%), and those whose capture rose 5–20 pp are the safest (15.8%). Falling usage often means spending stopped (see 6.3), not that the balance was paid down.
- Wallet stretch (number of months with utilisation above 90%) rises cleanly: 0 months gives 20.2%, 3–5 months 29.3%, and all 6 months 32.3%.
- Story use: stretch months is a simple, monotone dashboard KPI. The capture-change chart needs careful wording.

**6.7b** The **stress signature**: capture rising (Δ util > +5 pp, Sep/Aug vs May/Apr) **while** repayment share falls (Δ < −5 pp, Sep/Aug vs Jun/May). Default rate by dynamic quadrant, plus a scatter of the two deltas coloured by default rate (hexbin). Also checked inside each September status group — the signature is only useful if it adds information beyond the current delinquency status.
Section 20 also reports a payment-amount version (UTIL_TREND > 0 and PAY_TREND < 0): 20.9%; both agree the pattern is not riskier than average.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
tY3 = plot_rate(df, "SOW_DYN_QUADRANT", ax=axes[0], order=DYN_ORDER, title="Default by SoW dynamic quadrant", rotate=20)
h = df.dropna(subset=["SOW_D_PAYSHARE"])
hb = axes[1].hexbin(h["SOW_D_UTIL"].clip(-.6, .6), h["SOW_D_PAYSHARE"].clip(-1, 1), C=h[TARGET], reduce_C_function=np.mean,
                    gridsize=28, cmap=CMAP_RATE, mincnt=25)
plt.colorbar(hb, ax=axes[1], format=mtick.PercentFormatter(1), label="default rate")
axes[1].axvline(.05, color="k", lw=.8, ls="--"); axes[1].axhline(-.05, color="k", lw=.8, ls="--")
axes[1].set_xlabel("change in capture (late - early)"); axes[1].set_ylabel("change in repayment share (late - early)")
axes[1].set_title("Capture change vs repayment-share change (hex >= 25 clients)")
plt.tight_layout(); savefig("s06_stress_signature"); plt.show()
display(tY3[["SOW_DYN_QUADRANT", "n", "defaults", "rate", "ci_lo", "ci_hi", "share_clients", "lift"]])
within = rate_table(df, ["STATUS_GRP_SEP", "SOW_STRESS_FLAG"])
wp = within.pivot(index="STATUS_GRP_SEP", columns="SOW_STRESS_FLAG", values="rate").reindex(STATUS_GROUP_ORDER)
wn = within.pivot(index="STATUS_GRP_SEP", columns="SOW_STRESS_FLAG", values="n").reindex(STATUS_GROUP_ORDER)
wp.columns = ["rate_no_stress", "rate_stress"]; wn.columns = ["n_no_stress", "n_stress"]
out = wp.join(wn); out["uplift_pp"] = (out["rate_stress"] - out["rate_no_stress"]) * 100
print("Stress signature within September status group:"); display(out.round(3))

**Observations:**
- The assumed stress signature (capture up while repayment goes down) does not predict default: 17.4% vs 24.1% for stable/improving clients.
- Within each Sep status group it is neutral or even protective (Late 2m: 56.5% with the signature vs 69.8% without).
- The "stable / improving" group looks riskier because it includes stuck, maxed-out accounts whose usage cannot rise further.
- Story use: an honest negative finding. Drop the stress flag, or mention it only as a myth-buster.

### 6.8 The SoW risk grid: capture × repayment share
The key two-dimensional view of share of wallet: **how much of the line is drawn** (rows) against **how much of the bill is cleared** (columns). Left: count-based default rate (with n). Right: share of total exposure-at-default sitting in each cell — the cells a collections team should look at first. *Tableau: highlight table with the same bands; the export `s06_sow_grid` is ready.*

In [ ]:
cap_order = CAP_LABELS
ps_order = PS_LABELS + ["No balance due"]
grid = df.groupby(["SOW_CAPTURE_BAND", "SOW_PAYSHARE_BAND"], observed=True).agg(
    n=(TARGET, "size"), defaults=(TARGET, "sum"), exposure=("SOW_EXPOSURE_SEP", "sum"), ead=("SOW_EAD", "sum")).reset_index()
grid["rate"] = grid["defaults"] / grid["n"]
grid["ci_lo"], grid["ci_hi"] = wilson_ci(grid["defaults"], grid["n"])
grid["ew_default_rate"] = np.where(grid["exposure"] > 0, grid["ead"] / grid["exposure"].where(grid["exposure"] > 0, 1), np.nan)
grid["share_clients"] = grid["n"] / len(df)
grid["share_exposure"] = grid["exposure"] / df["SOW_EXPOSURE_SEP"].sum()
grid["share_ead"] = grid["ead"] / df["SOW_EAD"].sum()
grid["lift"] = grid["rate"] / BASE_RATE
piv = lambda v: grid.pivot(index="SOW_CAPTURE_BAND", columns="SOW_PAYSHARE_BAND", values=v).reindex(index=cap_order, columns=ps_order)
R_, N_, SE_, EW_ = piv("rate"), piv("n"), piv("share_ead"), piv("ew_default_rate")
fig, axes = plt.subplots(1, 2, figsize=(20, 6.5))
ann = _fmt(R_, lambda v: "" if pd.isna(v) else f"{v:.0%}") + "\n" + _fmt(N_, lambda v: "" if pd.isna(v) else f"n={int(v)}")
sns.heatmap(R_.where(N_ >= 30), annot=ann.where(N_ >= 30, ""), fmt="", cmap=CMAP_RATE, ax=axes[0],
            cbar_kws={"format": mtick.PercentFormatter(1), "label": "default rate"}, annot_kws={"fontsize": 7.5})
axes[0].set_title("Default rate (n>=30): capture x repayment share"); axes[0].set_xlabel("median repayment share"); axes[0].set_ylabel("avg capture (bill/limit)")
ann2 = _fmt(SE_, lambda v: "" if pd.isna(v) else f"{v:.1%}") + "\n" + _fmt(EW_, lambda v: "" if pd.isna(v) else f"ew {v:.0%}")
sns.heatmap(SE_, annot=ann2, fmt="", cmap="Reds", ax=axes[1], cbar_kws={"format": mtick.PercentFormatter(1), "label": "share of total EAD"},
            annot_kws={"fontsize": 7.5})
axes[1].set_title("Share of total exposure-at-default (and exposure-weighted default rate)"); axes[1].set_xlabel("median repayment share"); axes[1].set_ylabel("")
plt.tight_layout(); savefig("s06_sow_risk_grid"); plt.show()
print("Default rate:"); display(R_.round(3))
print("Clients:"); display(N_)
print("Share of total EAD:"); display(SE_.round(3))
print("Top 8 cells by share of EAD:")
display(grid.sort_values("share_ead", ascending=False).head(8)[["SOW_CAPTURE_BAND", "SOW_PAYSHARE_BAND", "n", "rate", "ew_default_rate", "share_clients", "share_exposure", "share_ead"]].round(3))

**Observations:**
- Paying nothing is bad at every capture level: 51–67% default.
- Among minimum payers (0–6% repaid), risk climbs with capture: 16.4% at 10–30% capture, 31.9% at 90–100% and 33.3% above 100%.
- Low capture with full repayment is the safest large group: 0–10% capture with ≥ 95% repaid (n=7,105) defaults at 15.2%, and 10–30% capture with ≥ 95% repaid at 9.9%.
- The money at risk sits in one column: minimum payers with capture of 30% or more hold 80.1% of EAD. The single cell of 70–90% capture × minimum due has 3,512 clients (11.7%) and 30.4% of EAD.
- Story use: the headline SoW grid. High capture with minimum payments is where the bank's money is at risk.

### 6.9 Expected-loss style view: NT$ at risk by segment
`EL_proxy = Σ exposure × segment default rate × LGD`. No LGD or recovery data exist, so we show **LGD = 100%** (upper bound, = realised EAD in expectation) and **LGD = 45%** (Basel F-IRB benchmark for senior unsecured claims) purely for scale; rankings do not depend on the LGD choice. Undrawn limits are ignored (no credit-conversion factor), which understates exposure for low-utilisation segments. Two segmentations: limit band × September status, and age band × payer type.

In [ ]:
def el_table(dims):
    g = df.groupby(dims, observed=True).agg(n=(TARGET, "size"), pd_=(TARGET, "mean"), exposure=("SOW_EXPOSURE_SEP", "sum"),
                                            realised_ead=("SOW_EAD", "sum")).reset_index()
    g["segment"] = g[dims].astype(str).agg(" | ".join, axis=1)
    g["EL_LGD100"] = g["exposure"] * g["pd_"]
    g["EL_LGD45"] = g["EL_LGD100"] * 0.45
    g["share_EL"] = g["EL_LGD100"] / g["EL_LGD100"].sum()
    g["share_clients"] = g["n"] / len(df)
    g["EL_per_client"] = g["EL_LGD100"] / g["n"]
    return g.sort_values("EL_LGD100", ascending=False).reset_index(drop=True)
el1 = el_table(["LIMIT_BAND", "STATUS_GRP_SEP"])
el2 = el_table(["AGE_BAND", "SOW_PAYER_TYPE"])
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
for ax, e, ttl in [(axes[0], el1, "Limit band | Sep status"), (axes[1], el2, "Age band | payer type")]:
    top = e[e["n"] >= 50].head(15).iloc[::-1]
    ax.barh(top["segment"], top["EL_LGD100"] / 1e6, color=C_DEF, alpha=.85, label="LGD 100%")
    ax.barh(top["segment"], top["EL_LGD45"] / 1e6, color="#67000d", alpha=.9, label="LGD 45%")
    for yy, (v, s) in enumerate(zip(top["EL_LGD100"] / 1e6, top["share_EL"])):
        ax.text(v, yy, f" {s:.1%}", va="center", fontsize=8)
    ax.set_xlabel("expected loss proxy, NT$ million"); ax.set_title(f"Top 15 segments by NT$ at risk: {ttl} (n>=50)"); ax.tick_params(axis="y", labelsize=8)
axes[0].legend(loc="lower right")
plt.tight_layout(); savefig("s06_expected_loss"); plt.show()
cols_el = ["segment", "n", "share_clients", "pd_", "exposure", "EL_LGD100", "EL_LGD45", "share_EL", "EL_per_client"]
display(el1[cols_el].head(12).round(3)); display(el2[cols_el].head(12).round(3))
print(f"Portfolio EL proxy (LGD100): NT$ {el1['EL_LGD100'].sum()/1e6:,.1f}m | realised EAD: NT$ {df['SOW_EAD'].sum()/1e6:,.1f}m")
k = (el1["share_EL"].cumsum() <= 0.5).sum() + 1
print(f"{k} of {len(el1)} limit x status segments hold 50% of NT$ at risk")

**Observations:**
- The portfolio expected-loss proxy (exposure × segment default rate, with LGD 100%, i.e. nothing recovered) is NT$301m vs NT$322m of realised EAD. The segment default rates reproduce the actual loss well.
- By limit × status, the top segment is 50-100k Late 2m: NT$25.1m of expected loss (8.3%) from only 635 clients, about NT$39.5k each.
- Big-line revolvers rank high in money despite low rates: 200-300k Revolving NT$20.1m, 300-500k Revolving NT$19.7m. Just 8 of 42 segments hold 50% of NT$ at risk.
- By age × payer type, minimum payers aged 26–45 hold 58% of expected loss (17.9 + 15.8 + 14.3 + 10.7%), while no partial-payer segment exceeds 3.1%.
- Story use: closing "where to act" slide, targeting minimum payers aged 26–45 and mid-size lines that are 2 months late.

### 6.10 Which SoW proxy carries the most default signal?
Information Value (decile-binned WoE), Spearman correlation with default, and the default rate in the bottom vs top decile of each continuous proxy. This ranks the proxies against each other and against two familiar reference features (`PAY_1`, `LIMIT_BAL`).

In [ ]:
proxies = {"A capture avg": "SOW_UTIL_AVG", "A capture Sep": "SOW_UTIL_SEP", "A capture peak": "SOW_UTIL_PEAK", "A headroom NT$": "SOW_HEADROOM_SEP",
           "B spend share": "SOW_SPEND_SHARE_AVG", "B spend HHI": "SOW_SPEND_HHI", "B spend trend": "SOW_SPEND_TREND",
           "C repay share med": "SOW_PAYSHARE_MED", "C repay share mean": "SOW_PAYSHARE_MEAN", "C pay / limit": "SOW_PAY_LIMIT_AVG",
           "C # transactor months": "SOW_N_TRANSACTOR_M",
           "D peer limit idx": "SOW_PEER_LIMIT_IDX", "D peer bill idx": "SOW_PEER_BILL_IDX", "D limit residual": "SOW_LIMIT_RESID",
           "D est. wallet share": "SOW_WALLET_SHARE", "E Sep exposure": "SOW_EXPOSURE_SEP",
           "Dyn capture change": "SOW_UTIL_CHG", "Dyn stretch months": "SOW_STRETCH_M", "Dyn d-util": "SOW_D_UTIL", "Dyn d-repay": "SOW_D_PAYSHARE",
           "ref: PAY_1": "PAY_1", "ref: LIMIT_BAL": "LIMIT_BAL"}
rows = []
for name, c in proxies.items():
    s = df[[c, TARGET]].dropna()
    _, iv = woe_iv(s, c)
    rho = stats.spearmanr(s[c], s[TARGET])[0]
    try:
        dec = pd.qcut(s[c].rank(method="first"), 10, labels=False)
        lo, hi = s.loc[dec == 0, TARGET].mean(), s.loc[dec == 9, TARGET].mean()
    except Exception:
        lo = hi = np.nan
    rows.append({"proxy": name, "column": c, "coverage": len(s) / len(df), "IV": iv, "spearman": rho,
                 "rate_bottom_decile": lo, "rate_top_decile": hi})
sig = pd.DataFrame(rows).sort_values("IV", ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 7))
colors = [C_NEU if p.startswith("ref") else C_ACC for p in sig["proxy"]]
ax.barh(sig["proxy"][::-1], sig["IV"][::-1], color=colors[::-1])
for v in [.02, .1, .3]:
    ax.axvline(v, ls=":", color="k", lw=.8)
ax.set_title("Information Value of each SoW proxy (dotted: 0.02 weak, 0.1 medium, 0.3 strong)"); ax.set_xlabel("IV")
plt.tight_layout(); savefig("s06_proxy_iv"); plt.show()
display(sig.round(4))
print(f"Section 6 runtime so far: {time.time()-t0_s6:.0f}s")

**Observations:**
- No SoW proxy beats the current status: PAY_1 has an IV of 0.87 (information value; above 0.3 is strong, 0.1–0.3 medium). The best SoW proxy, mean repayment share, scores 0.167, close to LIMIT_BAL at 0.178.
- Next come NT$ headroom (0.157), peer limit index (0.142), limit residual (0.140), spend HHI (0.132) and average capture (0.129), all of medium strength.
- The bottom decile of repayment share defaults at 37.3% vs 11.7% in the top decile. Capture runs the other way (26.2% → 32.5%).
- The weakest proxies are Sep exposure (0.012), peer bill index (0.023) and estimated wallet share (0.025). How big the balance is says almost nothing about default.
- Story use: proxy ranking chart with the message "how much you repay matters more than how much you owe".

### 6.10b Size of wallet: how big is the client's total wallet, and how much of it does this card get?

**What "size of wallet" means.** Share of wallet is a fraction: *what this bank gets* ÷ *what the client has in total*. Sections 6.2-6.10 worked on the top of that fraction. **Size of wallet** is the bottom: the client's total capacity to spend or borrow across *all* banks and cards. In industry a bank estimates it from **outside data**: credit-bureau files (the limits and balances on every other card and loan the client holds), declared income, and sometimes spending data bought from card networks. A client with NT$ 1 million of total card limits who uses our NT$ 100k card has a big wallet and gives us a small share of it; that client is a growth target. A client whose whole wallet is our NT$ 100k card is fully "on us".

**Why we cannot observe it here.** This dataset has no income, no bureau data and no information on other cards. So the size of wallet can only be **estimated** from what the client does on this one card and from what similar clients do. We build five estimates (all columns start with `SOW_WALLET_`), each a different answer to "how big could this client's card wallet be?".

| # | Estimate | Column | Formula | Watch out |
|---|---|---|---|---|
| 1 | **Bank's view** | `SOW_WALLET_BANK` | `LIMIT_BAL` | The bank's own judgement, so it is not independent of our data. It also covers supplementary (family) cards, so it can over-state one person's capacity. |
| 2 | **Demonstrated payment capacity** | `SOW_WALLET_PAY_MAX`, `SOW_WALLET_PAY_P90`, `SOW_WALLET_PAYCAP` | largest monthly `PAY_AMT`; 90th percentile of the 6 payments; `PAYCAP = 12 × P90` (a year of cash at that rate) | Shows cash the client *can* find, not their income. One large payment (a bonus, or a loan used to clear the card) can inflate the maximum, which is why the 90th percentile is the main version. |
| 3 | **Demonstrated borrowing capacity** | `SOW_WALLET_BORROW` | largest positive `BILL_AMT` over Apr-Sep | A lower bound only: the client was able to carry at least this much. It is capped by our own limit, so it can never show a wallet bigger than our line. |
| 4 | **Peer "realistic wallet"** | `SOW_WALLET_PEER` (+ `SOW_WALLET_PEER_P80`, `SOW_WALLET_PEER_CELL`) | the 90th percentile of the peak balance among similar clients (age band × education × sex × marital status), from quantile regression | Based on the idea of Rosset et al. (IBM Research, 2005, *Wallet estimation models*): most clients use only part of their wallet with any one bank, so the top 10% of similar clients show what a full wallet looks like. It only knows demographics, which explain little (see 6.5c). It is the same model as 6.5c's `SOW_WALLET_EST`; here we build a full size-of-wallet view on top of it. Falls back to grouped peer-cell quantiles (cells under 50 clients use age band only) if `statsmodels` is missing. |
| 5 | **Combined estimate** | `SOW_WALLET_COMBINED` | `max(SOW_WALLET_BORROW, SOW_WALLET_PEER)` | The wallet is at least what the client has shown (3) and at least what similar people realistically carry (4). Taking the larger of the two respects both. Payment capacity (2) is kept apart because it is a yearly flow of cash, not a balance. |

From the combined estimate we build:
* **Implied share of wallet** `SOW_WALLET_CAPTURE` = average positive balance (Apr-Sep) ÷ `SOW_WALLET_COMBINED`. Because the combined wallet is never below the client's peak balance, this stays between 0 and 1.
* **Wallet-to-line ratio** `SOW_WALLET_TO_LINE` = `SOW_WALLET_COMBINED` ÷ `LIMIT_BAL`. A value of 2 means similar people carry twice this client's whole line.
* **Bands**: `SOW_WALLET_SIZE_BAND` (Small < 100k, Medium 100-175k, Large 175-300k, Very large ≥ 300k NT$) and `SOW_WALLET_SOW_BAND` (< 1%, 1-10%, 10-30%, 30-60%, ≥ 60%).
* **Segments** `SOW_WALLET_SEGMENT`: **Over-extended** = peak balance above the peer realistic wallet *and* implied share ≥ 60% (borrowing more than 90% of similar people, and keeping it up); **Under-served** = wallet ≥ 2 × the line (similar people carry at least twice what this bank allows); **Matched** = everyone else.

**Worked example (client ID 7, from the cell below).** A 26-30 year-old single man with a graduate degree.
1. **Bank's view:** his line is NT$ 500,000.
2. **Payment capacity:** his six payments run from NT$ 13,750 to NT$ 55,000. The 90th percentile is NT$ 47,500, so a year at that pace is 12 × 47,500 = NT$ 570,000.
3. **Borrowing capacity:** his highest statement was NT$ 542,653 (Jun).
4. **Peer wallet:** among clients like him, the top 10% carry a peak balance of about NT$ 167,388.
5. **Combined:** max(542,653, 167,388) = **NT$ 542,653**. He has shown a wallet three times bigger than his peers', so his own record wins.

His average balance is NT$ 454,099, so the implied share of wallet is 454,099 ÷ 542,653 = 83.7%: this card holds nearly all of the wallet we can see. He is "over-extended" by our rule, yet he did not default (his payments grew from NT$ 13,770 in Apr to NT$ 55,000 in Sep).
For contrast, client ID 1 (21-25, female, line NT$ 20,000, peak balance NT$ 3,913) gets a peer wallet of NT$ 85,402, which is 4.3 × her line: she is "under-served". She defaulted.

In [ ]:
t0_s6w = time.time()
S6W_B = np.clip(df[BILL_COLS].to_numpy(float), 0, None)     # col 0 = Sep ... col 5 = Apr; credit balances -> 0
S6W_P = df[PAMT_COLS].to_numpy(float)
S6W_L = df["LIMIT_BAL"].to_numpy(float)

# 1. bank's view
df["SOW_WALLET_BANK"] = S6W_L
# 2. demonstrated payment capacity (monthly) and its annualised version
df["SOW_WALLET_PAY_MAX"] = S6W_P.max(1)
df["SOW_WALLET_PAY_P90"] = np.quantile(S6W_P, 0.9, axis=1)
df["SOW_WALLET_PAYCAP"] = 12 * df["SOW_WALLET_PAY_P90"]
# 3. demonstrated borrowing capacity
df["SOW_WALLET_BORROW"] = S6W_B.max(1)

# 4. peer realistic wallet: high quantile of peak balance among similar clients
S6W_KEYS = ["AGE_BAND", "EDU_LABEL", "SEX_LABEL", "MAR_LABEL"]
S6W_MIN_CELL = 50
_gf = df.groupby(S6W_KEYS, observed=True)["SOW_WALLET_BORROW"]
_gc = df.groupby("AGE_BAND", observed=True)["SOW_WALLET_BORROW"]
_fine = (_gf.transform("size") >= S6W_MIN_CELL).to_numpy()
S6W_CELLQ = {q: np.where(_fine, _gf.transform(lambda s_, q=q: s_.quantile(q)), _gc.transform(lambda s_, q=q: s_.quantile(q)))
             for q in (0.8, 0.9)}
df["SOW_WALLET_PEER_CELL"] = S6W_CELLQ[0.9]
S6W_y = np.log1p(df["SOW_WALLET_BORROW"].to_numpy(float))
S6W_X = pd.get_dummies(df[S6W_KEYS].astype(str), drop_first=True).astype(float)
S6W_X.insert(0, "const", 1.0)
try:
    import statsmodels.api as sm
    S6W_Q = {q: np.expm1(np.asarray(sm.QuantReg(S6W_y, S6W_X).fit(q=q, max_iter=5000).predict(S6W_X))) for q in (0.8, 0.9)}
    S6W_METHOD = "statsmodels QuantReg on demographic dummies (q=0.8, q=0.9)"
except Exception as e:
    print("QuantReg unavailable, using grouped peer-cell quantiles:", e)
    S6W_Q = S6W_CELLQ
    S6W_METHOD = f"grouped peer-cell quantiles (cells < {S6W_MIN_CELL} clients fall back to age band)"
df["SOW_WALLET_PEER"] = S6W_Q[0.9]
df["SOW_WALLET_PEER_P80"] = S6W_Q[0.8]

# 5. combined estimate
df["SOW_WALLET_COMBINED"] = np.maximum(df["SOW_WALLET_BORROW"], df["SOW_WALLET_PEER"])
df["SOW_WALLET_SRC"] = np.where(df["SOW_WALLET_BORROW"] > df["SOW_WALLET_PEER"], "Own peak balance", "Peer realistic wallet")

# implied share of wallet, wallet-to-line ratio, bands, segments
S6W_AVGB = S6W_B.mean(1)
df["SOW_WALLET_CAPTURE"] = S6W_AVGB / np.maximum(df["SOW_WALLET_COMBINED"], 1_000)
df["SOW_WALLET_TO_LINE"] = df["SOW_WALLET_COMBINED"] / S6W_L
S6W_SIZE_ORDER = ["Small (<100k)", "Medium (100-175k)", "Large (175-300k)", "Very large (>=300k)"]
df["SOW_WALLET_SIZE_BAND"] = pd.cut(df["SOW_WALLET_COMBINED"], [-np.inf, 100_000, 175_000, 300_000, np.inf], labels=S6W_SIZE_ORDER, right=False)
S6W_SOW_ORDER = ["<1% (dormant)", "1-10%", "10-30%", "30-60%", ">=60%"]
df["SOW_WALLET_SOW_BAND"] = pd.cut(df["SOW_WALLET_CAPTURE"], [-np.inf, .01, .10, .30, .60, np.inf], labels=S6W_SOW_ORDER, right=False)
S6W_SEG_ORDER = ["Under-served (wallet >= 2x line)", "Matched", "Over-extended (beyond peer wallet)"]
_over = (df["SOW_WALLET_BORROW"] > df["SOW_WALLET_PEER"]) & (df["SOW_WALLET_CAPTURE"] >= 0.6)
_under = (df["SOW_WALLET_TO_LINE"] >= 2) & ~_over
df["SOW_WALLET_SEGMENT"] = pd.Categorical(np.select([_under, _over], [S6W_SEG_ORDER[0], S6W_SEG_ORDER[2]], S6W_SEG_ORDER[1]),
                                          categories=S6W_SEG_ORDER, ordered=True)

print("peer wallet method:", S6W_METHOD)
print(f"clients in fine peer cells (>= {S6W_MIN_CELL}): {_fine.mean():.1%} | QR vs grouped P90 Spearman: "
      f"{stats.spearmanr(df['SOW_WALLET_PEER'], df['SOW_WALLET_PEER_CELL'])[0]:.2f} | vs 6.5c SOW_WALLET_EST: "
      f"{stats.spearmanr(df['SOW_WALLET_PEER'], df['SOW_WALLET_EST'])[0]:.2f}")
print(f"combined wallet set by the client's own peak balance for {(df['SOW_WALLET_SRC'] == 'Own peak balance').mean():.1%} of clients, by the peer wallet for the rest")

# worked examples: lowest ID in each segment
S6W_SHOW = ["LIMIT_BAL", "SOW_WALLET_PAY_MAX", "SOW_WALLET_PAY_P90", "SOW_WALLET_PAYCAP", "SOW_WALLET_BORROW", "SOW_WALLET_PEER",
            "SOW_WALLET_COMBINED", "SOW_WALLET_CAPTURE", "SOW_WALLET_TO_LINE"]
for seg_ in S6W_SEG_ORDER:
    r = df.loc[df["SOW_WALLET_SEGMENT"] == seg_].sort_values("ID").iloc[0]
    print(f"\n--- ID {int(r['ID'])}: {seg_} | {r['AGE_BAND']}, {r['EDU_LABEL']}, {r['SEX_LABEL']}, {r['MAR_LABEL']} | default = {int(r[TARGET])}")
    print("  bills Apr..Sep:", " ".join(f"{r[f'BILL_AMT{i}']:,.0f}" for i in IDX[::-1]))
    print("  pays  Apr..Sep:", " ".join(f"{r[f'PAY_AMT{i}']:,.0f}" for i in IDX[::-1]))
    print(f"  (1) bank line {r['LIMIT_BAL']:,.0f} | (2) max pay {r['SOW_WALLET_PAY_MAX']:,.0f}, P90 pay {r['SOW_WALLET_PAY_P90']:,.0f} -> x12 = {r['SOW_WALLET_PAYCAP']:,.0f}"
          f" | (3) peak balance {r['SOW_WALLET_BORROW']:,.0f} | (4) peer P90 {r['SOW_WALLET_PEER']:,.0f}")
    print(f"  (5) combined = max({r['SOW_WALLET_BORROW']:,.0f}, {r['SOW_WALLET_PEER']:,.0f}) = {r['SOW_WALLET_COMBINED']:,.0f} | avg balance {S6W_AVGB[df.index.get_loc(r.name)]:,.0f}"
          f" -> implied share {r['SOW_WALLET_CAPTURE']:.1%} | wallet / line = {r['SOW_WALLET_TO_LINE']:.2f}")
display(df[S6W_SHOW].describe(percentiles=[.1, .25, .5, .75, .9]).T.round(2))

**Observations:**
- Quantile regression worked, and the peer wallet ranks clients exactly like 6.5c's `SOW_WALLET_EST` (Spearman 1.00), as expected from the same model. The simpler grouped peer-cell version agrees less (0.78).
- The peer wallet sets the combined estimate for 90% of clients. Only 10% (3,006) have carried a peak balance above what the top 10% of their peers carry.
- The typical combined wallet is NT$ 170k (median), against a median line of NT$ 140k. The middle half of clients sit in a narrow NT$ 152k-178k range, because demographics are all the peer model has to go on.
- Payment capacity is small next to the line: the median client could repay about NT$ 48k a year at their own 90th-percentile pace, about a third of the median line.
- Story use: background. The worked examples (ID 1, 2, 7) make the five estimates concrete for a slide.

**How big are the wallets, and do the estimates agree?** Left: the spread of each estimate in NT$ (log scale, box = middle half of clients). Right: rank correlation (Spearman) between the estimates. If two estimates rank clients the same way the cell is near 1; near 0 means they measure different things.

In [ ]:
S6W_EST = {"1 Bank line": "SOW_WALLET_BANK", "2 Pay capacity (12 x P90)": "SOW_WALLET_PAYCAP", "2 Max monthly pay": "SOW_WALLET_PAY_MAX",
           "3 Peak balance": "SOW_WALLET_BORROW", "4 Peer P90 wallet": "SOW_WALLET_PEER", "5 Combined": "SOW_WALLET_COMBINED"}
fig, axes = plt.subplots(1, 2, figsize=(17, 5.2), gridspec_kw={"width_ratios": [1.2, 1]})
_long = pd.concat([pd.DataFrame({"estimate": k, "NT$": df[c].clip(lower=1_000)}) for k, c in S6W_EST.items()])
sns.boxplot(data=_long, y="estimate", x="NT$", ax=axes[0], color=C_OK, fliersize=0, whis=(5, 95))
axes[0].set_xscale("log"); axes[0].set_title("Size-of-wallet estimates (NT$, log scale; whiskers 5th-95th pct)"); axes[0].set_ylabel("")
S6W_RHO = df[list(S6W_EST.values())].rename(columns={v: k for k, v in S6W_EST.items()}).corr(method="spearman")
sns.heatmap(S6W_RHO, annot=True, fmt=".2f", cmap=CMAP_DIV, vmin=-1, vmax=1, ax=axes[1], annot_kws={"fontsize": 8})
axes[1].set_title("Spearman correlation between estimates")
plt.tight_layout(); savefig("s06_wallet_estimates"); plt.show()
S6W_MED = pd.DataFrame({k: df[c].quantile([.1, .25, .5, .75, .9]) for k, c in S6W_EST.items()}).T
S6W_MED.columns = ["p10", "p25", "median", "p75", "p90"]
S6W_MED["median / median line"] = S6W_MED["median"] / df["LIMIT_BAL"].median()
display(S6W_MED.round(2)); display(S6W_RHO.round(2))
display(df.groupby("SOW_WALLET_SRC")[[TARGET, "LIMIT_BAL", "SOW_WALLET_COMBINED", "SOW_WALLET_CAPTURE"]].agg(["size", "mean"]).round(3).iloc[:, [0, 1, 3, 5, 7]])

**Observations:**
- The estimates measure different things. Payment capacity and peak balance agree moderately (Spearman 0.58). The peer wallet hardly relates to the client's own behaviour (0.03 with peak balance, 0.13 with payment capacity).
- The bank's line sits in between: it correlates 0.39 with payment capacity and 0.37 with the peer wallet. It already mixes behaviour and profile.
- Clients whose own peak beats the peer wallet have big lines (mean NT$ 287k) and use 77% of the wallet. They default *less* (19.4%) than the rest (22.4%).
- Story use: one honest chart showing that "size of wallet" is not one number here. Each estimate tells a different story.

**Does a bigger wallet, or a bigger share of it, mean more default?** Default rate by wallet-size band (left) and by implied share-of-wallet band (right). Bars = clients, red line = default rate with 95% CI, dashed = base rate. *Tableau: two bar-and-line sheets.*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
S6W_T1 = plot_rate(df, "SOW_WALLET_SIZE_BAND", ax=axes[0], order=S6W_SIZE_ORDER, title="Default rate by estimated size of wallet (combined)")
S6W_T2 = plot_rate(df, "SOW_WALLET_SOW_BAND", ax=axes[1], order=S6W_SOW_ORDER, title="Default rate by implied share of wallet (avg balance / wallet)")
plt.tight_layout(); savefig("s06_wallet_size_bands"); plt.show()
_cols = ["n", "defaults", "rate", "ci_lo", "ci_hi", "share_clients", "lift"]
display(S6W_T1[["SOW_WALLET_SIZE_BAND"] + _cols].round(3)); display(S6W_T2[["SOW_WALLET_SOW_BAND"] + _cols].round(3))
_med = df.groupby("SOW_WALLET_SIZE_BAND", observed=True)[["LIMIT_BAL", "SOW_WALLET_PAYCAP", "SOW_WALLET_CAPTURE", "SOW_WALLET_TO_LINE"]].median()
print("medians by wallet size band:"); display(_med.round(2))
export(pd.concat([S6W_T1.rename(columns={"SOW_WALLET_SIZE_BAND": "band"}).assign(view="wallet size"),
                  S6W_T2.rename(columns={"SOW_WALLET_SOW_BAND": "band"}).assign(view="implied share of wallet")]).assign(band=lambda t: t["band"].astype(str)),
       "s06_wallet_band_rates")

**Observations:**
- The estimated wallet size barely moves default: 20.6% to 26.9% across the four bands. Only the small-wallet band (under NT$ 100k, 12% of clients) is clearly above base.
- The implied share of wallet is flat too: 19.4% to 26.4%. Dormant cards (under 1% of the wallet used) are the riskiest band at 26.4%, the same "empty card" pattern seen in 6.3 and 6.4c.
- Small wallets come with small lines (median NT$ 50k vs NT$ 200k for large wallets). Much of the small-wallet risk is the small-line risk we already know.
- Story use: background only. Wallet size by itself is a weak story.

**Wallet size × share of wallet together.** Each cell is the default rate for clients with that wallet size (rows) and that implied share (columns); cells with fewer than 50 clients are blank. Read across a row to see whether using more of the wallet is riskier for a given wallet size. *Tableau: highlight table, export `s06_wallet_grid`.*

In [ ]:
S6W_G = rate_table(df, ["SOW_WALLET_SIZE_BAND", "SOW_WALLET_SOW_BAND"])
_rp = S6W_G.pivot(index="SOW_WALLET_SIZE_BAND", columns="SOW_WALLET_SOW_BAND", values="rate").reindex(index=S6W_SIZE_ORDER, columns=S6W_SOW_ORDER)
_np = S6W_G.pivot(index="SOW_WALLET_SIZE_BAND", columns="SOW_WALLET_SOW_BAND", values="n").reindex(index=S6W_SIZE_ORDER, columns=S6W_SOW_ORDER)
_ann = _fmt(_rp, lambda v: "" if pd.isna(v) else f"{v:.0%}") + "\n" + _fmt(_np, lambda v: "" if pd.isna(v) else f"n={int(v)}")
fig, ax = plt.subplots(figsize=(11, 4.8))
sns.heatmap(_rp.where(_np >= 50), annot=_ann.where(_np >= 50, ""), fmt="", cmap=CMAP_RATE, ax=ax,
            cbar_kws={"format": mtick.PercentFormatter(1), "label": "default rate"}, annot_kws={"fontsize": 8})
ax.set_title("Default rate: size of wallet (rows) x implied share of wallet (cols); blank = n<50")
ax.set_xlabel("avg balance / estimated wallet"); ax.set_ylabel("estimated wallet (combined)")
plt.tight_layout(); savefig("s06_wallet_grid"); plt.show()
print("default rate:"); display(_rp.where(_np >= 50).round(3)); print("clients:"); display(_np)
_ge = S6W_G.copy(); _ge[["SOW_WALLET_SIZE_BAND", "SOW_WALLET_SOW_BAND"]] = _ge[["SOW_WALLET_SIZE_BAND", "SOW_WALLET_SOW_BAND"]].astype(str)
_ge["size_order"] = _ge["SOW_WALLET_SIZE_BAND"].map({k: i for i, k in enumerate(S6W_SIZE_ORDER)})
_ge["sow_order"] = _ge["SOW_WALLET_SOW_BAND"].map({k: i for i, k in enumerate(S6W_SOW_ORDER)})
export(_ge, "s06_wallet_grid")

**Observations:**
- No cell stands out. Rates stay between 16% and 29% across the grid.
- In medium and large wallets the 1-10% share column is the safest (19.8% and 17.1%). Dormant cards are the riskiest column in every row with enough clients (25.9% to 29.2%).
- Very large wallets that are mostly used (≥ 60%, 499 clients) default at 24.6%, slightly above base. These are the over-extended big borrowers.
- Story use: background only. The grid confirms that wallet size and share add little on their own.

**Does wallet size add anything once we know the September status?** Default rate by wallet-size band *within* each September status group (left), next to the same view for the bank's own line (right, limit band). The table below compares the spread (highest minus lowest rate, cells with n ≥ 50) inside each status: if the wallet spread is much smaller than the limit spread, the wallet estimate adds little beyond what the bank already knows.

In [ ]:
def s6w_within(col, order):
    t = rate_table(df, ["STATUS_GRP_SEP", col])
    rp = t.pivot(index="STATUS_GRP_SEP", columns=col, values="rate").reindex(index=STATUS_GROUP_ORDER, columns=order)
    n_ = t.pivot(index="STATUS_GRP_SEP", columns=col, values="n").reindex(index=STATUS_GROUP_ORDER, columns=order)
    return t, rp.where(n_ >= 50), n_
S6W_WS, S6W_WS_R, S6W_WS_N = s6w_within("SOW_WALLET_SIZE_BAND", S6W_SIZE_ORDER)
_, S6W_WL_R, _ = s6w_within("LIMIT_BAND", LIMIT_LABELS)
_, S6W_WC_R, _ = s6w_within("SOW_WALLET_SOW_BAND", S6W_SOW_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(18, 4.8), gridspec_kw={"width_ratios": [1, 1.5]})
for ax, rp, ttl in [(axes[0], S6W_WS_R, "Sep status x estimated wallet size"), (axes[1], S6W_WL_R, "Sep status x bank's limit band")]:
    sns.heatmap(rp, annot=_fmt(rp, lambda v: "" if pd.isna(v) else f"{v:.0%}"), fmt="", cmap=CMAP_RATE, vmin=0, vmax=.8, ax=ax,
                cbar_kws={"format": mtick.PercentFormatter(1)}, annot_kws={"fontsize": 8})
    ax.set_title(f"Default rate: {ttl} (blank = n<50)"); ax.set_xlabel(""); ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=20)
plt.tight_layout(); savefig("s06_wallet_within_status"); plt.show()
S6W_SPREAD = pd.DataFrame({"n": df["STATUS_GRP_SEP"].value_counts().reindex(STATUS_GROUP_ORDER),
                           "spread_wallet_size_pp": (S6W_WS_R.max(1) - S6W_WS_R.min(1)) * 100,
                           "spread_wallet_share_pp": (S6W_WC_R.max(1) - S6W_WC_R.min(1)) * 100,
                           "spread_limit_band_pp": (S6W_WL_R.max(1) - S6W_WL_R.min(1)) * 100})
print("default rate within status by wallet size:"); display(S6W_WS_R.round(3))
print("spread of default rate inside each status (pp):"); display(S6W_SPREAD.round(1))
_w = S6W_WS.copy(); _w[["STATUS_GRP_SEP", "SOW_WALLET_SIZE_BAND"]] = _w[["STATUS_GRP_SEP", "SOW_WALLET_SIZE_BAND"]].astype(str)
export(_w, "s06_wallet_within_status")

**Observations:**
- Inside each September status, wallet size moves default by only 2.7 to 6.0 points. The bank's limit band moves it by up to 16.6 points (paid in full) and 15.6 points (late 1 month).
- So once we know the status, the wallet-size estimate adds very little, and much less than the line the bank already sets.
- The implied share band spreads more (9 to 19 points), but with no steady direction: overall its rates go 26.4%, 19.4%, 23.3%, 21.8%, 21.0% from dormant to ≥ 60%.
- Story use: a myth-buster. "We tried to estimate the wallet; once you know who is late, it adds almost nothing".

**Which wallet estimate carries the most default signal, and does any beat the bank's own line?** Information Value (IV, decile bins: below 0.1 weak, 0.1-0.3 medium, above 0.3 strong), Spearman correlation with default, and the default rate in the bottom and top fifth of clients for each estimate. Grey bar = `LIMIT_BAL`, the benchmark to beat.

In [ ]:
S6W_IVCOLS = {"1 Bank line (LIMIT_BAL)": "SOW_WALLET_BANK", "2 Max monthly pay": "SOW_WALLET_PAY_MAX", "2 P90 monthly pay": "SOW_WALLET_PAY_P90",
              "2 Pay capacity (12 x P90)": "SOW_WALLET_PAYCAP", "3 Peak balance": "SOW_WALLET_BORROW", "4 Peer P90 wallet (QR)": "SOW_WALLET_PEER",
              "4 Peer P80 wallet": "SOW_WALLET_PEER_P80", "4 Peer P90 wallet (cells)": "SOW_WALLET_PEER_CELL", "5 Combined wallet": "SOW_WALLET_COMBINED",
              "Implied share of wallet": "SOW_WALLET_CAPTURE", "Wallet / line": "SOW_WALLET_TO_LINE"}
_rows = []
for k, c in S6W_IVCOLS.items():
    s_ = df[[c, TARGET]]
    q5 = pd.qcut(s_[c].rank(method="first"), 5, labels=False)
    _rows.append({"estimate": k, "column": c, "IV": woe_iv(s_, c)[1], "spearman": stats.spearmanr(s_[c], s_[TARGET])[0],
                  "rate_bottom_fifth": s_.loc[q5 == 0, TARGET].mean(), "rate_top_fifth": s_.loc[q5 == 4, TARGET].mean(),
                  "n_distinct": s_[c].nunique()})
S6W_IV = pd.DataFrame(_rows).sort_values("IV", ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 5.5))
ax.barh(S6W_IV["estimate"][::-1], S6W_IV["IV"][::-1], color=[C_NEU if c == "SOW_WALLET_BANK" else C_ACC for c in S6W_IV["column"][::-1]])
for v in [.02, .1, .3]:
    ax.axvline(v, ls=":", color="k", lw=.8)
ax.set_title("Information Value of each size-of-wallet estimate (grey = bank's line)"); ax.set_xlabel("IV")
plt.tight_layout(); savefig("s06_wallet_iv"); plt.show()
display(S6W_IV.round(4))
export(S6W_IV, "s06_wallet_iv")

**Observations:**
- No wallet estimate beats the bank's line on its own. `LIMIT_BAL` has an IV of 0.178. Payment capacity comes close (0.157): the bottom fifth of payers default at 30.1%, the top fifth at 12.7%.
- The peer wallet (0.029), peak balance (0.028), combined wallet (0.024) and implied share (0.024) are all weak, below 0.03.
- The only measure above the line is the wallet-to-line ratio (0.185). It is mostly the line seen upside down, as the ratio is built by dividing by `LIMIT_BAL`.
- The annualised and monthly payment capacities score the same, because multiplying by 12 does not change the ranking.
- Story use: the IV bar chart answers "is size of wallet feasible here?" Partly: the one estimate that works (payment capacity) is behaviour, not wallet.

**Who is under-served and who is over-extended?** *Under-served* = similar clients carry at least twice this client's whole line (big wallet, small line): a growth target if they are safe. *Over-extended* = the client has carried more than 90% of similar people and keeps ≥ 60% of that peak on the card (small wallet, big balance). Left: default rate per segment. Right: the same split inside each limit band, to check that "under-served" is not simply "small line" under another name.

In [ ]:
S6W_T3 = rate_table(df, "SOW_WALLET_SEGMENT")
_prof = df.groupby("SOW_WALLET_SEGMENT", observed=True).agg(median_limit=("LIMIT_BAL", "median"), median_wallet=("SOW_WALLET_COMBINED", "median"),
                                                          median_paycap=("SOW_WALLET_PAYCAP", "median"), median_capture=("SOW_WALLET_CAPTURE", "median"),
                                                          median_age=("AGE", "median"), share_late_sep=("PAY_1", lambda s_: (s_ >= 1).mean()))
S6W_T3 = S6W_T3.merge(_prof.reset_index(), on="SOW_WALLET_SEGMENT")
_t = rate_table(df, ["LIMIT_BAND", "SOW_WALLET_SEGMENT"])
_rp = _t.pivot(index="LIMIT_BAND", columns="SOW_WALLET_SEGMENT", values="rate").reindex(index=LIMIT_LABELS, columns=S6W_SEG_ORDER)
_np = _t.pivot(index="LIMIT_BAND", columns="SOW_WALLET_SEGMENT", values="n").reindex(index=LIMIT_LABELS, columns=S6W_SEG_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={"width_ratios": [1, 1.2]})
plot_rate(df, "SOW_WALLET_SEGMENT", ax=axes[0], order=S6W_SEG_ORDER, title="Default rate by wallet segment", rotate=10)
_ann = _fmt(_rp, lambda v: "" if pd.isna(v) else f"{v:.0%}") + "\n" + _fmt(_np, lambda v: "" if pd.isna(v) else f"n={int(v)}")
sns.heatmap(_rp.where(_np >= 50), annot=_ann.where(_np >= 50, ""), fmt="", cmap=CMAP_RATE, ax=axes[1],
            cbar_kws={"format": mtick.PercentFormatter(1)}, annot_kws={"fontsize": 8})
axes[1].set_title("Default rate: limit band x wallet segment (blank = n<50)"); axes[1].set_xlabel(""); axes[1].set_ylabel("")
axes[1].tick_params(axis="x", rotation=10)
plt.tight_layout(); savefig("s06_wallet_segments"); plt.show()
display(S6W_T3.round(3))
print("default rate inside each limit band:"); display(_rp.where(_np >= 50).round(3)); print("clients:"); display(_np)
_e = S6W_T3.copy(); _e["SOW_WALLET_SEGMENT"] = _e["SOW_WALLET_SEGMENT"].astype(str)
export(_e, "s06_wallet_segments")
_x = _t.copy(); _x[["LIMIT_BAND", "SOW_WALLET_SEGMENT"]] = _x[["LIMIT_BAND", "SOW_WALLET_SEGMENT"]].astype(str)
export(_x, "s06_wallet_segments_by_limit")

# story facts (computed from the data)
_ivs = S6W_IV.set_index("column")["IV"]
_best = S6W_IV[S6W_IV["column"] != "SOW_WALLET_BANK"].iloc[0]
add_fact("6.10b", "size of wallet", f"Size of wallet can only be estimated here: the best estimate, '{_best['estimate']}', has IV {_best['IV']:.2f} vs {_ivs['SOW_WALLET_BANK']:.2f} for the bank's own line, while the peer-based wallet scores only {_ivs['SOW_WALLET_PEER']:.2f}", float(_best["IV"]), "s06_wallet_iv", 2)
_s3 = S6W_T3.set_index("SOW_WALLET_SEGMENT")
add_fact("6.10b", "size of wallet", f"'Under-served' clients (similar people carry 2x+ their line; {int(_s3.loc[S6W_SEG_ORDER[0], 'n']):,} clients) default at {_s3.loc[S6W_SEG_ORDER[0], 'rate']:.1%} vs {_s3.loc['Matched', 'rate']:.1%} for matched and {_s3.loc[S6W_SEG_ORDER[2], 'rate']:.1%} for 'over-extended' clients: a small line, not a big wallet, drives the risk", float(_s3.loc[S6W_SEG_ORDER[0], 'rate']), "s06_wallet_segments", 2)
_r1 = S6W_T1.set_index("SOW_WALLET_SIZE_BAND")["rate"]
_rl = rate_table(df, "LIMIT_BAND")["rate"]
add_fact("6.10b", "size of wallet", f"Default barely moves with estimated wallet size: {_r1.min():.1%} to {_r1.max():.1%} across the four wallet-size bands, against {_rl.min():.1%} to {_rl.max():.1%} across limit bands", float(_r1.max() - _r1.min()), "s06_wallet_size_bands", 1)
print(f"6.10b runtime: {time.time()-t0_s6w:.1f}s")

**Observations:**
- Under-served clients (8,769, 29%) default at 31.4%, against 17.9% for matched clients. They are mostly small lines: their median line is NT$ 50k against a wallet of NT$ 160k, and 32% are already late in September.
- Within the same limit band the gap shrinks but stays: 32.6% vs 25.7% at ≤50k and 27.6% vs 24.2% at 50-100k. Being given a small line *relative to similar people* adds a little risk beyond the line itself (as 6.5 found).
- Over-extended clients (2,412, 8%) default at 21.1%, close to base overall. Inside each limit band, though, they are riskier than matched clients: 19.1% vs 7.2% above NT$ 500k, 18.4% vs 12.2% at 300-500k and 24.3% vs 17.0% at 150-200k.
- So "big wallet, small line" is not a safe growth pool here; it is the bank's risk-based small line at work. "Small wallet, big balance" is the warning group once the line is held fixed.
- Story use: a two-segment slide. Under-served = small lines given for a reason; over-extended = big-line clients who carry far more than their peers, and default 3-12 points more than matched clients with the same line.

### 6.11 Tableau exports and new columns
**New `df` columns added in this section (all prefixed `SOW_`):**

| Group | Columns |
|---|---|
| A capture | `SOW_UTIL_SEP`, `SOW_UTIL_AVG`, `SOW_UTIL_PEAK`, `SOW_HEADROOM_SEP`, `SOW_HEADROOM_PCT`, `SOW_CAPTURE_BAND`, `SOW_CAPTURE_SEP_BAND`, `SOW_HEADROOM_BAND` |
| B spend | `SOW_SPEND_SHARE_1..5`, `SOW_SPEND_SHARE_AVG`, `SOW_SPEND_HHI`, `SOW_SPEND_ACTIVE_M`, `SOW_SPEND_TREND`, `SOW_SPEND_BAND`, `SOW_SPEND_CONC_BAND` |
| C repayment | `SOW_PAYSHARE_1..5`, `SOW_PAYSHARE_MED`, `SOW_PAYSHARE_MEAN`, `SOW_PAY_LIMIT_AVG`, `SOW_N_TRANSACTOR_M`, `SOW_PAYER_TYPE`, `SOW_PAYSHARE_BAND` |
| D peer / wallet | `SOW_AVG_POS_BILL`, `SOW_PEER_CELL_N`, `SOW_PEER_LIMIT_MED`, `SOW_PEER_LIMIT_IDX`, `SOW_PEER_BILL_IDX`, `SOW_PEER_LIMIT_BAND`, `SOW_PEER_BILL_BAND`, `SOW_LIMIT_RESID`, `SOW_LIMIT_RESID_BAND`, `SOW_WALLET_EST`, `SOW_WALLET_SHARE`, `SOW_WALLET_SHARE_BAND` |
| C2 quarterly paydown | `SOW_QPD_1`, `SOW_QPD_2`, `SOW_QPD_3`, `SOW_QPD_PRIOR`, `SOW_QPD_CHG`, `SOW_QPD_BAND`, `SOW_QPD_CHG_BAND` |
| Size of wallet (6.10b) | `SOW_WALLET_BANK`, `SOW_WALLET_PAY_MAX`, `SOW_WALLET_PAY_P90`, `SOW_WALLET_PAYCAP`, `SOW_WALLET_BORROW`, `SOW_WALLET_PEER`, `SOW_WALLET_PEER_P80`, `SOW_WALLET_PEER_CELL`, `SOW_WALLET_COMBINED`, `SOW_WALLET_SRC`, `SOW_WALLET_CAPTURE`, `SOW_WALLET_TO_LINE`, `SOW_WALLET_SIZE_BAND`, `SOW_WALLET_SOW_BAND`, `SOW_WALLET_SEGMENT` |
| E exposure | `SOW_EXPOSURE_SEP`, `SOW_EAD` |
| Dynamics | `SOW_UTIL_CHG`, `SOW_UTIL_CHG_BAND`, `SOW_STRETCH_M`, `SOW_STRETCH_BAND`, `SOW_D_UTIL`, `SOW_D_PAYSHARE`, `SOW_DYN_QUADRANT`, `SOW_STRESS_FLAG` |

**Exports:** `s06_sow_client` (one row per client: ID, labels, target, all `SOW_` columns), `s06_sow_segments` (long segment-share table incl. the limit × status treemap hierarchy), `s06_sow_grid` (capture × repayment-share risk grid), `s06_sow_el` (expected-loss ranking), `s06_sow_proxy_iv` (proxy signal ranking).

In [ ]:
SOW_COLS = [c for c in df.columns if c.startswith("SOW_")]
client = df[["ID", "DEFAULT", "DEFAULT_LABEL", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND", "LIMIT_BAL",
             "STATUS_GRP_SEP", "UTIL_BAND"] + SOW_COLS].copy()
for c in client.columns:
    if isinstance(client[c].dtype, pd.CategoricalDtype):
        client[c] = client[c].astype(str)
export(client, "s06_sow_client")
seg_all = pd.concat([seg, tree.assign(dimension="LIMIT_BAND > STATUS_GRP_SEP (treemap)")], ignore_index=True)
export(seg_all, "s06_sow_segments")
grid_out = grid.copy()
grid_out[["SOW_CAPTURE_BAND", "SOW_PAYSHARE_BAND"]] = grid_out[["SOW_CAPTURE_BAND", "SOW_PAYSHARE_BAND"]].astype(str)
grid_out["capture_order"] = grid_out["SOW_CAPTURE_BAND"].map({k: i for i, k in enumerate(cap_order)})
grid_out["payshare_order"] = grid_out["SOW_PAYSHARE_BAND"].map({k: i for i, k in enumerate(ps_order)})
export(grid_out, "s06_sow_grid")
export(pd.concat([el1.assign(view="LIMIT_BAND | STATUS_GRP_SEP"), el2.assign(view="AGE_BAND | SOW_PAYER_TYPE")], ignore_index=True)
       .drop(columns=["LIMIT_BAND", "STATUS_GRP_SEP", "AGE_BAND", "SOW_PAYER_TYPE"], errors="ignore"), "s06_sow_el")
export(sig, "s06_sow_proxy_iv")
print(f"{len(SOW_COLS)} SOW_ columns added:", ", ".join(SOW_COLS))
print(f"Section 6 total runtime: {time.time()-t0_s6:.0f}s")

## 7. Engineered features, correlation and redundancy
The raw data has three six-month blocks (status, bill, payment) that are heavily inter-correlated, plus the engineered layer from Section 0.
This section (a) maps linear, rank and nonlinear dependence, (b) quantifies redundancy (block collinearity, VIF, feature clusters),
(c) adds a second layer of behavioural features (`FE_*`) and ranks them by Information Value, and (d) ends with a **recommended
non-redundant feature set** for the Tableau dashboard (one representative per redundant family).

New `df` columns added here (all prefixed `FE_`): `FE_LIMIT_PER_AGE, FE_UTIL_ACCEL, FE_PAY_CV, FE_MAX_PAY_SHARE, FE_MONTHS_SINCE_FULLPAY,
FE_REVOLVE_BAL, FE_REVOLVE_UTIL, FE_MIN_PAYER_MONTHS, FE_OVERPAY_MONTHS, FE_HEADROOM, FE_HEADROOM_MIN, FE_BILL_CV, FE_PAY_MOMENTUM,
FE_DELAY_SEVERITY, FE_LATE_RECENT3, FE_UTIL_VOL`.

**Section guide**

- **What it does:** Measures how much the columns overlap with each other and with default. It adds a second layer of behaviour features, ranks them, and ends with a short, non-redundant field list for the dashboard.
- **Question it answers:** Which columns say the same thing, which new features add real signal, and which few fields should Tableau use?
- **Inputs / new columns:** all raw money and status columns plus the setup features. Adds 16 columns with prefix `FE_` (e.g. `FE_LIMIT_PER_AGE`, `FE_LATE_RECENT3`, `FE_HEADROOM`).
- **How to read the charts:** correlation heatmaps: red = move together, blue = move opposite, darker = stronger; blocks of colour = redundant families. Quantile curves: line = default rate per decile, red dashed line = 22.1% base rate. The dendrogram (tree) joins similar features low down.
- **Terms:** VIF (variance inflation factor) = how much a column is a copy of the others (above 10 = severe). PCA = compresses many correlated columns into a few summary axes. Mutual information = any kind of dependence, not just straight-line.
- **Key takeaways:**
  - The six bill columns are almost one number per client (mean r 0.89, VIF 14-26).
  - Lateness leads every ranking: `N_LATE` r = 0.40, IV 0.86.
  - The most original new feature is `FE_LIMIT_PER_AGE` (IV 0.18, novelty 0.57).
  - Nine fields cover the useful signal, led by `PAY_1` and `PAY_2`.
- **Where it feeds:** the drivers dashboard field list (`s07_feature_families.csv`) and the driver-ranking bar (`s07_corr_with_default.csv`).

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

RAW_CORR_COLS = ["LIMIT_BAL", "AGE"] + PAY_COLS + BILL_COLS + PAMT_COLS

def abs_corr_linkage(cm, method="average"):
    # hierarchical linkage on distance = 1 - |corr|
    d = 1 - np.abs(np.asarray(cm, float))
    d = np.clip((d + d.T) / 2, 0, None)
    np.fill_diagonal(d, 0)
    return linkage(squareform(d, checks=False), method=method)

def top_pairs(cm, k=15, other=None, other_name="other"):
    # strongest off-diagonal pairs of a correlation matrix
    m = cm.where(np.triu(np.ones(cm.shape, bool), 1)).stack().rename("r").reset_index()
    m.columns = ["var_a", "var_b", "r"]
    if other is not None:
        m[other_name] = [other.loc[a, b] for a, b in zip(m.var_a, m.var_b)]
    return m.reindex(m["r"].abs().sort_values(ascending=False).index).head(k).reset_index(drop=True)

def clustered_corr(cm, title, name, annot=True, size=(12, 11)):
    Z = abs_corr_linkage(cm)
    g = sns.clustermap(cm, row_linkage=Z, col_linkage=Z, cmap=CMAP_DIV, vmin=-1, vmax=1, center=0,
                       annot=annot, fmt=".2f", annot_kws={"size": 6}, figsize=size, dendrogram_ratio=.1,
                       cbar_pos=(0.01, 0.86, 0.02, 0.1), linewidths=.3)
    g.fig.suptitle(title, y=1.01, fontweight="bold")
    savefig(name, g.fig)
    plt.show()
print("section 7 helpers ready")

### 7.1 Raw numeric columns: Pearson vs Spearman (clustered)
Rows/columns are reordered by hierarchical clustering on 1−|r|, so redundant families appear as blocks. Spearman (rank correlation, not swayed by extreme values) is shown alongside
Pearson (straight-line correlation) because bills and payments are extremely right-skewed: Pearson is dominated by a few very large accounts.

In [ ]:
corr_raw_p = df[RAW_CORR_COLS].corr("pearson")
corr_raw_s = df[RAW_CORR_COLS].corr("spearman")
clustered_corr(corr_raw_p, "Raw numeric columns - Pearson correlation (clustered on 1-|r|)", "s07_clustermap_raw_pearson")
clustered_corr(corr_raw_s, "Raw numeric columns - Spearman correlation (clustered on 1-|rho|)", "s07_clustermap_raw_spearman")
tp = top_pairs(corr_raw_s, 15, corr_raw_p, "pearson").rename(columns={"r": "spearman"})
tp["spearman_minus_pearson"] = tp["spearman"] - tp["pearson"]
display(tp)

**Observations:**
- Bill amounts are near-duplicates month to month: adjacent `BILL_AMT` pairs correlate at Pearson 0.92-0.95 (Spearman 0.90-0.91), and even three months apart they stay above 0.80.
- Repayment status codes form a second family: adjacent months such as `PAY_4`/`PAY_5` and `PAY_5`/`PAY_6` correlate at about 0.82.
- Pearson runs 0.02-0.06 above Spearman (a rank-based correlation that ignores extreme values) for bill pairs, so a few very large balances inflate the linear numbers a little.
- Story use: background only; it justifies showing each six-month block as one level plus a trend.

### 7.2 Engineered features (Section 0 layer): Pearson vs Spearman (clustered)
Shows which engineered features are re-expressions of the same signal (e.g. counts of status codes that sum to 6, totals vs averages).

In [ ]:
corr_fe_p = df[FEATURE_COLS].corr("pearson")
corr_fe_s = df[FEATURE_COLS].corr("spearman")
clustered_corr(corr_fe_p, "Engineered features - Pearson correlation (clustered)", "s07_clustermap_eng_pearson", size=(13, 12))
clustered_corr(corr_fe_s, "Engineered features - Spearman correlation (clustered)", "s07_clustermap_eng_spearman", size=(13, 12))
tp = top_pairs(corr_fe_s, 20, corr_fe_p, "pearson").rename(columns={"r": "spearman"})
display(tp)

**Observations:**
- Some engineered pairs are exact copies: `TOTAL_BILL`/`AVG_BILL` and `TOTAL_PAY`/`AVG_PAY` correlate at 1.00.
- `N_LATE` and `LATE_STREAK_MAX` measure almost the same thing (Spearman 0.998).
- `AVG_PAY_RATIO` vs `PAY_TO_BILL_6M` has Spearman 0.93 but Pearson only 0.11, so a few extreme ratios distort Pearson; ratio features need rank-based or banded views.
- Clients who use more of their limit repay a smaller share of the bill (`AVG_UTIL` vs `AVG_PAY_RATIO`, Spearman -0.75).
- Story use: background only; keep one field from each duplicate pair in Tableau.

### 7.3 Correlation of every numeric column with `DEFAULT`
Point-biserial (Pearson) and Spearman correlation with the target, raw and engineered side by side. This is the simplest Tableau
"driver ranking" bar chart; note that correlation only captures monotone association (Section 7.10 checks nonlinear dependence).

In [ ]:
tgt_cols = RAW_CORR_COLS + FEATURE_COLS
tc = pd.DataFrame({
    "pearson": [df[c].corr(df[TARGET]) for c in tgt_cols],
    "spearman": [df[c].corr(df[TARGET], method="spearman") for c in tgt_cols],
}, index=tgt_cols)
tc["block"] = ["raw"] * len(RAW_CORR_COLS) + ["engineered"] * len(FEATURE_COLS)
tc = tc.sort_values("spearman")
fig, axes = plt.subplots(1, 2, figsize=(14, 11), sharey=True)
for ax, m in zip(axes, ["pearson", "spearman"]):
    cols = [C_DEF if v > 0 else C_OK for v in tc[m]]
    ax.barh(tc.index, tc[m], color=cols, alpha=.85)
    for y, (v, b) in enumerate(zip(tc[m], tc["block"])):
        if b == "engineered":
            ax.text(v + (0.005 if v >= 0 else -0.005), y, "E", va="center", ha="left" if v >= 0 else "right", fontsize=7, color="k")
    ax.axvline(0, color="k", lw=.8)
    ax.set_title(f"{m.title()} correlation with DEFAULT")
    ax.set_xlabel("correlation")
axes[0].tick_params(axis="y", labelsize=8)
fig.suptitle("Correlation with default (E = engineered feature)", fontweight="bold")
plt.tight_layout(); savefig("s07_corr_with_default"); plt.show()
print(tc.reindex(tc["spearman"].abs().sort_values(ascending=False).index).head(20).round(3).to_string())
export(tc.rename_axis("feature").reset_index(), "s07_corr_with_default")

**Observations:**
- Lateness counts lead the ranking: `N_LATE` r = 0.40, `LATE_STREAK_MAX` 0.39, `LATE_STREAK_RECENT` 0.37. The best raw column is September status `PAY_1` at 0.33.
- The raw status signal fades with the age of the month: `PAY_1` 0.33, `PAY_2` 0.26, down to `PAY_6` 0.19.
- Money variables are weak and negative: `LIMIT_BAL` -0.15, `AVG_SPEND` -0.10 (Spearman -0.19). Higher limits and more spending go with less default.
- Engineered features take 4 of the top 5 places.
- Story use: the driver-ranking bar chart; lateness behaviour beats money and demographics.

### 7.4 Block structure: within-block collinearity
Each monthly block is a 6-column family. If within-block correlations exceed ~0.9 (as expected for `BILL_AMT1..6`), the six columns carry
essentially one number per client, so the dashboard should show one summary (level + trend) rather than six separate fields.
The lag-decay plot shows how fast month-to-month correlation fades with distance (memory of each behaviour).

In [ ]:
BLOCKS = {"BILL_AMT": BILL_COLS, "PAY_AMT": PAMT_COLS, "PAY status": PAY_COLS,
          "UTIL": [f"UTIL_{i}" for i in IDX], "PAY_RATIO": [f"PAY_RATIO_{i}" for i in IDX[:-1]], "SPEND": SPEND_COLS}
rows, decay = [], []
for name, cols in BLOCKS.items():
    for meth in ["pearson", "spearman"]:
        cm = df[cols].corr(meth).values
        off = cm[np.triu_indices(len(cols), 1)]
        rows.append({"block": name, "method": meth, "mean_r": off.mean(), "min_r": off.min(), "max_r": off.max(),
                     "share_pairs_r>0.9": (off > 0.9).mean(), "share_pairs_r>0.7": (off > 0.7).mean()})
    cm = df[cols].corr("spearman").values
    for lag in range(1, len(cols)):
        decay.append({"block": name, "lag_months": lag, "mean_spearman": np.mean([cm[i, i + lag] for i in range(len(cols) - lag)])})
block_tbl = pd.DataFrame(rows)
display(block_tbl.round(3))

fig, axes = plt.subplots(1, 4, figsize=(20, 4.6), gridspec_kw={"width_ratios": [1, 1, 1, 1.2]})
for ax, (name, cols) in zip(axes[:3], [("BILL_AMT", BILL_COLS), ("PAY_AMT", PAMT_COLS), ("PAY status", PAY_COLS)]):
    cm = df[cols].corr("pearson")
    lab = [f"{c} ({MONTH[int(c[-1])]})" for c in cols]
    sns.heatmap(cm, annot=True, fmt=".2f", cmap=CMAP_DIV, vmin=-1, vmax=1, ax=ax, cbar=False, xticklabels=lab, yticklabels=lab,
                annot_kws={"size": 8})
    ax.set_title(f"{name} within-block Pearson"); ax.tick_params(labelsize=7)
dd = pd.DataFrame(decay)
for name, g in dd.groupby("block", sort=False):
    axes[3].plot(g["lag_months"], g["mean_spearman"], marker="o", label=name)
axes[3].set_title("Lag decay: mean Spearman between months k apart"); axes[3].set_xlabel("lag (months)"); axes[3].set_ylabel("mean rho")
axes[3].legend(fontsize=8)
plt.tight_layout(); savefig("s07_block_structure"); plt.show()
print(dd.pivot(index="block", columns="lag_months", values="mean_spearman").round(3).to_string())

**Observations:**
- Bills are one number per client: the `BILL_AMT` block has mean Pearson 0.89 and all 15 pairs above 0.7. Utilisation behaves the same (mean 0.83).
- Payments are far less regular: the `PAY_AMT` block has mean Pearson only 0.19 (Spearman 0.51), so clients pay very different amounts from month to month.
- Persistence fades with distance: bills correlate 0.91 at a one-month lag and 0.73 at five months; status codes drop faster, from 0.77 to 0.46.
- Story use: supports the "lateness is sticky but fades" point; show bills as level plus trend, payments month by month.

### 7.5 Variance Inflation Factors (raw numerics vs a reduced set)
VIF_j = 1/(1−R²_j) from regressing feature *j* on all others (standardised, with intercept). VIF > 10 = severe collinearity (the feature is largely a copy of the others).
The engineered layer is also checked: some features are exact linear combinations (e.g. `N_LATE + N_INACTIVE + N_PAIDFULL + N_REVOLVE = 6`,
`TOTAL_BILL = 6 × AVG_BILL`) and therefore have infinite VIF. The reduced set is a candidate dashboard set with one field per family.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def vif_table(cols, data=df):
    X = data[cols].astype(float).replace([np.inf, -np.inf], np.nan).dropna()
    X = (X - X.mean()) / X.std().replace(0, 1)
    Xv = np.column_stack([X.values, np.ones(len(X))])
    out = []
    for i, c in enumerate(cols):
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                v = variance_inflation_factor(Xv, i)
        except Exception:
            v = np.inf
        out.append((c, v))
    t = pd.DataFrame(out, columns=["feature", "VIF"])
    t["VIF"] = t["VIF"].replace([np.inf], 1e6).clip(upper=1e6)
    t["flag"] = pd.cut(t["VIF"], [0, 5, 10, np.inf], labels=["ok (<5)", "moderate (5-10)", "severe (>10)"])
    return t.sort_values("VIF", ascending=False).reset_index(drop=True), len(X)

REDUCED_SET = ["LIMIT_BAL", "AGE", "PAY_1", "N_LATE", "AVG_UTIL", "UTIL_TREND", "PAY_TO_BILL_6M", "AVG_PAY",
               "BILL_TREND", "N_INACTIVE", "N_ZERO_PAY", "DELAY_TREND"]
vif_raw, n_raw = vif_table(RAW_CORR_COLS)
vif_red, n_red = vif_table(REDUCED_SET)
vif_eng, n_eng = vif_table(FEATURE_COLS)
print(f"rows used: raw {n_raw:,} | reduced {n_red:,} | engineered {n_eng:,}  (1e6 = infinite / exact collinearity)")
fig, axes = plt.subplots(1, 3, figsize=(18, 6.5))
for ax, (t, ttl) in zip(axes, [(vif_raw, "Raw numerics"), (vif_eng, "Engineered layer (Section 0)"), (vif_red, "Reduced dashboard set")]):
    tt = t.iloc[::-1]
    ax.barh(tt["feature"], tt["VIF"], color=[C_DEF if v > 10 else (C_ACC if v < 5 else "#fdae61") for v in tt["VIF"]])
    ax.set_xscale("log"); ax.axvline(5, ls="--", color="k", lw=.8); ax.axvline(10, ls="--", color=C_DEF, lw=.8)
    ax.set_title(f"VIF - {ttl}"); ax.tick_params(axis="y", labelsize=8)
plt.tight_layout(); savefig("s07_vif"); plt.show()
display(pd.concat({"raw": vif_raw, "reduced": vif_red}, axis=1).round(2))
print("Engineered layer VIF (top 12):"); print(vif_eng.head(12).round(2).to_string())

**Observations:**
- VIF (variance inflation factor) shows how much a feature overlaps with the others; above 10 is severe. All six `BILL_AMT` columns score 14-26, while status codes and payments stay below 5.
- The reduced 12-feature set has a maximum VIF of 5.5 (`PAY_1`), and every other feature is below 3.
- Eight engineered features are exact linear combinations (VIF shown as 1e6). For example `N_LATE`, `N_REVOLVE`, `N_INACTIVE` and `N_PAIDFULL` always sum to 6.
- Story use: background only; a method note for why the dashboard uses a reduced field list.

### 7.6 Association among categorical / banded variables (Cramér's V, Phi-K)
Bias-corrected Cramér's V for every pair of labelled/banded variables, including `DEFAULT_LABEL`. This is the categorical analogue of the
correlation matrix and shows, for instance, how strongly limit band is tied to education/age, and which bands carry target signal.
Phi-K (handles interval variables and is comparable to Pearson for bivariate normal) is computed if the `phik` package is installed.

In [ ]:
CAT_COLS = ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND", "STATUS_GRP_SEP", "UTIL_BAND",
            "N_LATE_BAND", "PAYRATIO_BAND", "DEFAULT_LABEL"]
cv = pd.DataFrame(np.eye(len(CAT_COLS)), index=CAT_COLS, columns=CAT_COLS)
for a, b in itertools.combinations(CAT_COLS, 2):
    cv.loc[a, b] = cv.loc[b, a] = cramers_v(df[a].astype(str), df[b].astype(str))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cv, annot=True, fmt=".2f", cmap="mako_r", vmin=0, vmax=1, ax=ax, mask=np.triu(np.ones_like(cv, bool), 1))
ax.set_title("Cramér's V (bias-corrected) between categorical / banded variables")
plt.tight_layout(); savefig("s07_cramers_v"); plt.show()
print("Association with DEFAULT_LABEL:")
print(cv["DEFAULT_LABEL"].drop("DEFAULT_LABEL").sort_values(ascending=False).round(3).to_string())

try:
    import phik  # noqa: F401
    pk_cols = CAT_COLS + ["LIMIT_BAL", "AGE", "AVG_UTIL", "N_LATE"]
    samp = df[pk_cols].sample(min(len(df), 10000), random_state=42)
    for c in CAT_COLS:
        samp[c] = samp[c].astype(str)
    pkm = samp.phik_matrix(interval_cols=["LIMIT_BAL", "AGE", "AVG_UTIL"])
    fig, ax = plt.subplots(figsize=(11, 9))
    sns.heatmap(pkm, annot=True, fmt=".2f", cmap="mako_r", vmin=0, vmax=1, ax=ax)
    ax.set_title("Phi-K correlation (10k sample)"); plt.tight_layout(); savefig("s07_phik"); plt.show()
except ImportError:
    print("phik not installed - Phi-K matrix skipped (pip install phik to enable). Cramér's V above covers the same question.")
except Exception as e:
    print("Phi-K failed, skipped:", e)

**Observations:**
- Cramér's V measures association between two categorical variables (0 = none, 1 = perfect). September status group (V = 0.42) and late-month band (0.40) are by far the strongest links to default.
- Limit band (0.16), pay-ratio band (0.15) and utilisation band (0.15) form a middle tier.
- Demographics are weak: education 0.07, age band 0.06, sex 0.04, marital status 0.03.
- Story use: core evidence for the headline "behaviour, not demographics, predicts default".

### 7.7 Second-layer engineered features (`FE_*`)
Each feature encodes a distinct, explainable behaviour that the Section 0 layer does not capture directly.

| Feature | Definition | Hypothesis |
|---|---|---|
| `FE_LIMIT_PER_AGE` | `LIMIT_BAL / AGE` | credit granted relative to life stage (bank's own risk view) |
| `FE_UTIL_ACCEL` | recent utilisation velocity (Sep−Jul)/2 minus earlier velocity (Jun−Apr)/2 | accelerating draw-down precedes stress |
| `FE_PAY_CV` | std / mean of the six payments (NaN if never paid) | erratic payers are less disciplined |
| `FE_MAX_PAY_SHARE` | largest single payment / total paid | one-off lump payments vs steady repayment |
| `FE_MONTHS_SINCE_FULLPAY` | months since the last payment ≥ previous bill (6 = never in window) | recency of a "clean" month |
| `FE_REVOLVE_BAL` / `FE_REVOLVE_UTIL` | mean carried balance max(bill(i+1) − pay(i), 0), NT$ and as share of limit | true revolving debt, not just billed amount |
| `FE_MIN_PAYER_MONTHS` | months paying >0 but <10% of previous bill | minimum-payment trap |
| `FE_OVERPAY_MONTHS` | months where payment > previous (positive) bill | transactor / prepaying behaviour |
| `FE_HEADROOM` / `FE_HEADROOM_MIN` | `LIMIT_BAL − BILL_AMT1` / minimum over 6 months (NT$) | absolute room left on the line. **Signed**: negative when over the limit, and a credit (negative) bill adds headroom; differs from Section 18's clipped `EX_HEADROOM = max(LIMIT_BAL − max(BILL_AMT1, 0), 0)` |
| `FE_BILL_CV` | std / mean(|bill|) over 6 months | bill stability |
| `FE_PAY_MOMENTUM` | log1p(mean pay Sep–Jul) − log1p(mean pay Jun–Apr) | payments shrinking recently |
| `FE_DELAY_SEVERITY` | sum of positive delay codes over 6 months | months-of-delay burden (count × depth) |
| `FE_LATE_RECENT3` | late months in Sep–Jul | recent delinquency only |
| `FE_UTIL_VOL` | std of monthly utilisation | volatility of usage |

In [ ]:
B7 = df[BILL_COLS].to_numpy(float); P7 = df[PAMT_COLS].to_numpy(float); S7 = df[PAY_COLS].to_numpy(int)
L7 = df["LIMIT_BAL"].to_numpy(float)[:, None]; U7 = B7 / L7
prevB, P5 = B7[:, 1:], P7[:, :5]                 # column k: payment in month k+1 vs bill of month k+2 (previous statement)

df["FE_LIMIT_PER_AGE"] = df["LIMIT_BAL"] / df["AGE"]
df["FE_UTIL_ACCEL"] = (U7[:, 0] - U7[:, 2]) / 2 - (U7[:, 3] - U7[:, 5]) / 2
mp, sp = P7.mean(1), P7.sum(1)
df["FE_PAY_CV"] = np.where(mp > 0, P7.std(1) / np.where(mp > 0, mp, 1), np.nan)
df["FE_MAX_PAY_SHARE"] = np.where(sp > 0, P7.max(1) / np.where(sp > 0, sp, 1), np.nan)
full = (prevB > 0) & (P5 >= prevB)
df["FE_MONTHS_SINCE_FULLPAY"] = np.where(full.any(1), full.argmax(1), 6)
carried = np.clip(prevB - P5, 0, None)
df["FE_REVOLVE_BAL"] = carried.mean(1)
df["FE_REVOLVE_UTIL"] = df["FE_REVOLVE_BAL"] / df["LIMIT_BAL"]
df["FE_MIN_PAYER_MONTHS"] = ((prevB > 0) & (P5 > 0) & (P5 < 0.1 * prevB)).sum(1)
df["FE_OVERPAY_MONTHS"] = ((P5 > np.clip(prevB, 0, None)) & (P5 > 0)).sum(1)
df["FE_HEADROOM"] = df["LIMIT_BAL"] - df["BILL_AMT1"]
df["FE_HEADROOM_MIN"] = (L7 - B7).min(1)
mb = np.abs(B7).mean(1)
df["FE_BILL_CV"] = np.where(mb > 0, B7.std(1) / np.where(mb > 0, mb, 1), np.nan)
df["FE_PAY_MOMENTUM"] = np.log1p(P7[:, :3].mean(1)) - np.log1p(P7[:, 3:].mean(1))
df["FE_DELAY_SEVERITY"] = np.clip(S7, 0, None).sum(1)
df["FE_LATE_RECENT3"] = (S7[:, :3] >= 1).sum(1)
df["FE_UTIL_VOL"] = U7.std(1)

FE_COLS = [c for c in df.columns if c.startswith("FE_")]
print(len(FE_COLS), "new features")
desc = df[FE_COLS].describe(percentiles=[.05, .5, .95]).T
desc["coverage"] = df[FE_COLS].notna().mean()
display(desc.round(3))

**Observations:**
- The second layer adds 16 `FE_` features. Most cover every client; the payment-shape features (`FE_PAY_CV`, `FE_MAX_PAY_SHARE`) are blank for 4.8% of clients who made no payments.
- `FE_MONTHS_SINCE_FULLPAY` has a median of 6, so at least half of clients never paid a bill in full during the six months.
- The median revolving balance is NT$16,335, and the 95th percentile is NT$160,797.
- Headroom can be negative: the 5th percentile of `FE_HEADROOM` is -NT$859, so some clients are over their limit.
- Story use: background only; definitions for dashboard tooltips.

#### 7.7.1 Default rate by quantile for every new feature
Deciles (fewer when ties collapse bins; raw values for count features). x-axis = bin median. Red dashed = base rate.
Monotone curves are dashboard-friendly (a single banded colour scale); U-shapes need explicit bands.

In [ ]:
def rate_by_quantile(s, y, q=10):
    s = pd.Series(s); ok = s.notna()
    s, y = s[ok], pd.Series(y)[ok]
    b = s if s.nunique() <= 12 else pd.qcut(s, q, duplicates="drop")
    g = pd.DataFrame({"x": s, "y": y, "b": b}).groupby("b", observed=True).agg(med=("x", "median"), n=("y", "size"), rate=("y", "mean"))
    lo, hi = wilson_ci(g["rate"] * g["n"], g["n"])
    return g.assign(lo=lo, hi=hi).reset_index(drop=True)

ncol = 4; nrow = int(np.ceil(len(FE_COLS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(18, 3.3 * nrow))
qcurves = []
for ax, c in zip(axes.ravel(), FE_COLS):
    g = rate_by_quantile(df[c], df[TARGET])
    x = np.arange(len(g))
    ax.fill_between(x, g["lo"], g["hi"], color=C_DEF, alpha=.15)
    ax.plot(x, g["rate"], marker="o", color=C_DEF)
    ax.axhline(BASE_RATE, ls="--", color="k", lw=.8)
    ax.set_xticks(x); ax.set_xticklabels([f"{v:,.2g}" for v in g["med"]], rotation=45, fontsize=7)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title(c, fontsize=10)
    qcurves.append(g.assign(feature=c, bin=x))
for ax in axes.ravel()[len(FE_COLS):]:
    ax.axis("off")
fig.suptitle("Default rate by quantile bin of each FE_ feature (95% Wilson band)", fontweight="bold", y=1.0)
plt.tight_layout(); savefig("s07_fe_quantile_curves"); plt.show()
qcurves = pd.concat(qcurves, ignore_index=True)
export(qcurves[["feature", "bin", "med", "n", "rate", "lo", "hi"]], "s07_fe_quantile_curves")

**Observations:**
- The lateness features rise steeply: `FE_LATE_RECENT3` goes from about 12% default (no late month in the last three) to about 64% (late in all three). `FE_DELAY_SEVERITY` follows the same pattern.
- `FE_LIMIT_PER_AGE` is cleanly monotone: default falls from about 36% in the lowest decile to about 12% in the highest.
- `FE_HEADROOM` falls from about 29% to 13% across deciles. `FE_REVOLVE_UTIL` rises from about 17% to 34%.
- Some curves are not monotone: `FE_MIN_PAYER_MONTHS` peaks near 29% at 3-4 months, and `FE_PAY_CV` peaks mid-range. These need explicit bands, not one colour scale.
- Story use: candidate banded fields for Tableau; limit-per-age is the easiest new one to explain.

#### 7.7.2 Ranking: new vs existing features (IV, Spearman, redundancy)
IV from `woe_iv` (deciles). `max_abs_rho_existing` = strongest |Spearman| with any Section 0 feature, so a high-IV feature that is also
highly redundant adds little. IV bands: <0.02 useless, 0.02–0.1 weak, 0.1–0.3 medium, 0.3–0.5 strong, >0.5 very strong.

In [ ]:
rank_cols = FE_COLS + FEATURE_COLS
rho_all = df[rank_cols + [TARGET]].corr("spearman")
rows = []
for c in rank_cols:
    iv = woe_iv(df, c)[1]
    others = [f for f in FEATURE_COLS if f != c]
    r = rho_all.loc[c, others].abs()
    rows.append({"feature": c, "source": "FE (Section 7)" if c in FE_COLS else "Section 0",
                 "coverage": df[c].notna().mean(), "IV": iv, "spearman_default": rho_all.loc[c, TARGET],
                 "max_abs_rho_existing": r.max(), "most_similar_existing": r.idxmax()})
feat_rank = pd.DataFrame(rows)
feat_rank["IV_band"] = pd.cut(feat_rank["IV"], [-1, .02, .1, .3, .5, 99], labels=["useless", "weak", "medium", "strong", "very strong"])
feat_rank["novelty"] = 1 - feat_rank["max_abs_rho_existing"]
feat_rank = feat_rank.sort_values("IV", ascending=False).reset_index(drop=True)
feat_rank["IV_rank"] = np.arange(1, len(feat_rank) + 1)
IV_LOOKUP = dict(zip(feat_rank["feature"], feat_rank["IV"]))
display(feat_rank.round(3))

fe_only = feat_rank[feat_rank["source"].str.startswith("FE")]
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.scatter(feat_rank["max_abs_rho_existing"], feat_rank["IV"], s=40,
           c=[C_DEF if s.startswith("FE") else C_NEU for s in feat_rank["source"]])
for _, r in feat_rank.iterrows():
    ax.annotate(r["feature"], (r["max_abs_rho_existing"], r["IV"]), fontsize=7, xytext=(3, 2), textcoords="offset points")
ax.axhline(0.1, ls=":", color="k"); ax.axhline(0.3, ls=":", color="k")
ax.set_xlabel("max |Spearman| with an existing Section 0 feature (redundancy)"); ax.set_ylabel("Information Value")
ax.set_title("New (red) vs existing (grey) features: signal vs redundancy\n(top-left = informative AND new)")
plt.tight_layout(); savefig("s07_fe_iv_vs_redundancy"); plt.show()
export(feat_rank, "s07_feature_ranking")

**Observations:**
- IV (information value) scores how well a feature separates defaulters from non-defaulters; above 0.3 is strong. The top four are all lateness measures: `N_LATE` 0.86, `LATE_STREAK_MAX` 0.85, `FE_LATE_RECENT3` 0.82, `FE_DELAY_SEVERITY` 0.80.
- Most new features repeat old signals: `FE_DELAY_SEVERITY` has |rho| 0.998 with `N_LATE` (novelty 0.002).
- The most original useful feature is `FE_LIMIT_PER_AGE` (credit line divided by age): IV 0.18 with novelty 0.57.
- Story use: lateness is the hero driver; limit-per-age is a supporting "credit vs life stage" angle.

### 7.8 Hierarchical clustering of all features → recommended non-redundant set
Distance = 1 − |Spearman| over raw, Section 0 and `FE_` features. Cutting the tree at distance 0.5 (features in a family share |ρ| of roughly ≥ 0.5 on average)
groups redundant features; within each family the feature with the highest IV is kept, and families whose best IV is < 0.05 are dropped.
This is the proposed (compact) field list for the dashboard. The 0.3 cut is also reported for a finer view.

In [ ]:
all_feats = RAW_CORR_COLS + FEATURE_COLS + FE_COLS
for c in RAW_CORR_COLS:
    if c not in IV_LOOKUP:
        IV_LOOKUP[c] = woe_iv(df, c)[1]
samp = df[all_feats].sample(12000, random_state=42)
samp = samp.fillna(samp.median())
rho = samp.corr("spearman").fillna(0)
Z = abs_corr_linkage(rho)
CUT = 0.5
fam_fine = fcluster(Z, 0.3, criterion="distance")
print(f"families at cut 0.3: {len(set(fam_fine))} | at cut {CUT}: {len(set(fcluster(Z, CUT, criterion='distance')))}")
fig, ax = plt.subplots(figsize=(10, 15))
dendrogram(Z, labels=rho.index.tolist(), orientation="left", color_threshold=CUT, leaf_font_size=8, ax=ax)
ax.axvline(CUT, ls="--", color=C_DEF); ax.set_xlabel("1 - |Spearman|")
ax.set_title(f"Feature dendrogram (average linkage); red line = cut at {CUT}")
plt.tight_layout(); savefig("s07_feature_dendrogram"); plt.show()

fam = pd.DataFrame({"feature": rho.index, "family": fcluster(Z, CUT, criterion="distance"), "family_fine_0.3": fam_fine})
fam["IV"] = fam["feature"].map(IV_LOOKUP)
fam_tbl = (fam.sort_values("IV", ascending=False).groupby("family")
           .agg(representative=("feature", "first"), rep_IV=("IV", "first"), n_members=("feature", "size"),
                members=("feature", lambda s: ", ".join(s)))
           .sort_values("rep_IV", ascending=False).reset_index())
display(fam_tbl.round(3))
RECOMMENDED_FEATURES = fam_tbl.loc[fam_tbl["rep_IV"] >= 0.05, "representative"].tolist()
print(f"{len(all_feats)} features -> {len(fam_tbl)} families -> {len(RECOMMENDED_FEATURES)} recommended (rep IV >= 0.05):")
print(RECOMMENDED_FEATURES)
export(fam.merge(fam_tbl[["family", "representative"]], on="family"), "s07_feature_families")

**Observations:**
- The 60 features fall into 14 families at a 0.5 distance cut (28 families at 0.3).
- The largest family has 26 members around `PAY_2`: older status months, revolving counts, bills and utilisation all move together.
- Nine representatives with IV of at least 0.05 remain: `PAY_1` (IV 0.87), `PAY_2` (0.54), `N_ZERO_PAY` (0.26), `DELAY_TREND` (0.25), `FE_LIMIT_PER_AGE` (0.18), `FE_BILL_CV`, `PAY_TREND`, `BILL_TREND`, `FE_OVERPAY_MONTHS`.
- `AGE` forms its own family but has IV only 0.02, so it is almost useless alone.
- Story use: the field list for the drivers dashboard.

### 7.9 PCA on money features (signed log1p, standardised)
`LIMIT_BAL`, `BILL_AMT1..6`, `PAY_AMT1..6` are transformed with sign(x)·log1p(|x|) and standardised. The scree plot (variance explained by each component) shows how many
independent "money dimensions" exist; loadings (each column's weight in a component) name them (expected: overall size, payment intensity, and a bill trend/contrast).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
MONEY = ["LIMIT_BAL"] + BILL_COLS + PAMT_COLS
Xm = df[MONEY].astype(float)
Xm = np.sign(Xm) * np.log1p(Xm.abs())
Xs = StandardScaler().fit_transform(Xm)
pca = PCA().fit(Xs)
scores = pca.transform(Xs)
load = pd.DataFrame(pca.components_.T, index=MONEY, columns=[f"PC{i+1}" for i in range(len(MONEY))])
for k in range(len(MONEY)):                      # sign convention: largest |loading| positive
    j = np.abs(load.iloc[:, k]).idxmax()
    if load.loc[j, load.columns[k]] < 0:
        load.iloc[:, k] *= -1; scores[:, k] *= -1
ev = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [1, 1.3]})
axes[0].bar(range(1, len(ev) + 1), ev, color=C_OK, alpha=.7)
axes[0].plot(range(1, len(ev) + 1), np.cumsum(ev), marker="o", color=C_DEF)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].set_xlabel("component")
axes[0].set_title("Scree: explained variance (bars) and cumulative (line)")
sns.heatmap(load.iloc[:, :5], annot=True, fmt=".2f", cmap=CMAP_DIV, center=0, vmin=-.6, vmax=.6, ax=axes[1])
axes[1].set_title("Loadings of the first 5 components")
plt.tight_layout(); savefig("s07_pca_scree_loadings"); plt.show()
print("explained variance:", " | ".join(f"PC{i+1} {v:.1%}" for i, v in enumerate(ev[:6])))
for k in range(4):
    col = load.columns[k]; top = load[col].reindex(load[col].abs().sort_values(ascending=False).index)[:5]
    print(f"{col}: " + ", ".join(f"{i} {v:+.2f}" for i, v in top.items()))

In [ ]:
pcs = pd.DataFrame(scores[:, :3], columns=["PC1", "PC2", "PC3"], index=df.index).assign(DEFAULT=df[TARGET].values)
s = pcs.sample(6000, random_state=42)
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
for lab, col in [(0, C_OK), (1, C_DEF)]:
    ss = s[s["DEFAULT"] == lab]
    axes[0].scatter(ss["PC1"], ss["PC2"], s=5, alpha=.35, color=col, label="Default" if lab else "No default")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2"); axes[0].legend(markerscale=4)
axes[0].set_title("PC1 vs PC2 (6k sample) by default")
hb = axes[1].hexbin(pcs["PC1"], pcs["PC2"], C=pcs["DEFAULT"], reduce_C_function=np.mean, gridsize=35, mincnt=15, cmap=CMAP_RATE)
plt.colorbar(hb, ax=axes[1], format=mtick.PercentFormatter(1)); axes[1].set_title("Default rate surface over PC1 x PC2 (hex, n>=15)")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
pc_dec = []
for k in ["PC1", "PC2", "PC3"]:
    g = pcs.groupby(pd.qcut(pcs[k], 10, labels=False))["DEFAULT"].mean()
    axes[2].plot(g.index + 1, g.values, marker="o", label=k); pc_dec.append(g.rename(k))
axes[2].axhline(BASE_RATE, ls="--", color="k"); axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[2].set_xlabel("decile of component score"); axes[2].set_title("Default rate by component decile"); axes[2].legend()
plt.tight_layout(); savefig("s07_pca_scatter"); plt.show()
print(pd.concat(pc_dec, axis=1).rename_axis("decile").round(3).T.to_string())

**Observations:**
- PCA (principal component analysis) compresses correlated columns into a few summary axes. PC1 explains 51.8% of the money variance and is the overall bill size; five components are needed for 80%.
- PC2 (10.3%) contrasts high limits and payments against high bills. Default falls across PC2 deciles from 39.1% to 12.8%.
- Bigger balances are not riskier: default falls from 26.3% in the lowest PC1 decile to 13.3% in the highest, mainly in the top half.
- Story use: supports "generous limits and steady payers are safer"; background chart.

### 7.10 Nonlinear dependence: mutual information vs correlation
Pairwise mutual information (kNN estimator, 4k sample) among the top-IV features, and MI of each with `DEFAULT`. Pairs with high MI
but low |ρ| have a non-monotone relationship that a correlation heatmap would hide.

In [ ]:
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif
top_feats = []
for f in feat_rank["feature"]:
    if all(abs(rho_all.loc[f, g]) < 0.97 for g in top_feats):
        top_feats.append(f)
    if len(top_feats) == 12:
        break
sm = df[top_feats + [TARGET]].sample(4000, random_state=7)
Xmi = sm[top_feats].fillna(sm[top_feats].median()).to_numpy(float)
Xmi = (Xmi - Xmi.mean(0)) / Xmi.std(0).clip(1e-9)
Xmi = Xmi + RNG.normal(0, 1e-6, Xmi.shape)       # break ties for the kNN estimator
k = len(top_feats); MI = np.zeros((k, k))
for i in range(k):
    MI[i, :] = mutual_info_regression(Xmi, Xmi[:, i], n_neighbors=3, random_state=0)
MI = (MI + MI.T) / 2; np.fill_diagonal(MI, np.nan)
MI = pd.DataFrame(MI, index=top_feats, columns=top_feats)
rho_top = sm[top_feats].corr("spearman")

fig, axes = plt.subplots(1, 3, figsize=(21, 6.5))
sns.heatmap(MI, annot=True, fmt=".2f", cmap="viridis", ax=axes[0], annot_kws={"size": 7}); axes[0].set_title("Pairwise mutual information (nats)")
sns.heatmap(rho_top.abs(), annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=axes[1], annot_kws={"size": 7}); axes[1].set_title("|Spearman|")
iu = np.triu_indices(k, 1)
pairs = pd.DataFrame({"a": [top_feats[i] for i in iu[0]], "b": [top_feats[j] for j in iu[1]],
                      "MI": MI.values[iu], "abs_rho": rho_top.abs().values[iu]})
axes[2].scatter(pairs["abs_rho"], pairs["MI"], color=C_OK)
# Gaussian reference: MI = -0.5 ln(1 - rho^2)
rr = np.linspace(0, .99, 100); axes[2].plot(rr, -0.5 * np.log(1 - rr ** 2), ls="--", color="k", label="Gaussian MI for that rho")
pairs["excess_MI"] = pairs["MI"] - (-0.5 * np.log(1 - pairs["abs_rho"].clip(upper=.99) ** 2))
for _, r in pairs.nlargest(6, "excess_MI").iterrows():
    axes[2].annotate(f"{r.a} x {r.b}", (r.abs_rho, r.MI), fontsize=7)
axes[2].set_xlabel("|Spearman|"); axes[2].set_ylabel("MI"); axes[2].legend(); axes[2].set_title("MI vs |rho| (points above line = nonlinear)")
for ax in axes[:2]:
    ax.tick_params(labelsize=7)
plt.tight_layout(); savefig("s07_mutual_information"); plt.show()
print("Pairs with most MI beyond what |rho| implies:")
print(pairs.sort_values("excess_MI", ascending=False).head(8).round(3).to_string(index=False))

sm2 = df[top_feats + [TARGET]].sample(10000, random_state=11)
Xt = sm2[top_feats].fillna(sm2[top_feats].median()).to_numpy(float)
Xt = Xt + RNG.normal(0, 1e-6, Xt.shape) * Xt.std(0).clip(1e-9)
mi_t = mutual_info_classif(Xt, sm2[TARGET], n_neighbors=3, random_state=0)
mt = pd.DataFrame({"feature": top_feats, "MI_default": mi_t,
                   "abs_spearman_default": [abs(rho_all.loc[f, TARGET]) for f in top_feats],
                   "IV": [IV_LOOKUP[f] for f in top_feats]})
for c in ["MI_default", "abs_spearman_default", "IV"]:
    mt[f"rank_{c}"] = mt[c].rank(ascending=False).astype(int)
display(mt.sort_values("MI_default", ascending=False).round(4))

**Observations:**
- Mutual information (MI) captures any dependence, not just straight-line ones. `MAX_DELAY` and `N_REVOLVE` share MI 0.65 but |rho| only 0.06, a strong non-monotone link that a correlation heatmap hides.
- MI, |Spearman| and IV agree on the top four drivers of default: `N_LATE`, `FE_LATE_RECENT3`, `MAX_DELAY`, `LATE_STREAK_RECENT`.
- `DELAY_TREND` ranks 5th by MI but 12th by Spearman, so its effect on default is non-linear.
- Story use: background only; show `DELAY_TREND` as bands, not as a continuous axis.

### 7.11 Section summary table
Compact recap for writing insights: strongest single drivers, redundancy findings, and the recommended dashboard fields.

In [ ]:
summary = {
    "BILL_AMT mean within-block Pearson": block_tbl.query("block == 'BILL_AMT' and method == 'pearson'")["mean_r"].iloc[0],
    "PAY_AMT mean within-block Spearman": block_tbl.query("block == 'PAY_AMT' and method == 'spearman'")["mean_r"].iloc[0],
    "PAY status mean within-block Spearman": block_tbl.query("block == 'PAY status' and method == 'spearman'")["mean_r"].iloc[0],
    "raw numerics with VIF > 10": int((vif_raw["VIF"] > 10).sum()),
    "reduced set max VIF": vif_red["VIF"].max(),
    "money PCA: PCs for 80% variance": int(np.searchsorted(np.cumsum(ev), .8) + 1),
    f"feature families (cut {CUT})": len(fam_tbl),
    "best new FE_ feature (IV)": f"{fe_only.iloc[0]['feature']} ({fe_only.iloc[0]['IV']:.3f})",
    "best novel FE_ (novelty > 0.3, by IV)": (lambda t: f"{t.iloc[0]['feature']} (IV {t.iloc[0]['IV']:.3f}, novelty {t.iloc[0]['novelty']:.2f})" if len(t) else "none")(fe_only[fe_only["novelty"] > 0.3]),
}
for k_, v in summary.items():
    print(f"{k_:42s} {v:.3f}" if isinstance(v, float) else f"{k_:42s} {v}")
print("\nRecommended dashboard features:", ", ".join(RECOMMENDED_FEATURES))

**Observations:**
- Redundancy is high: bills have within-block r = 0.886, six raw numerics have VIF above 10, and the reduced set brings the maximum VIF down to 5.5.
- The best new feature is `FE_LATE_RECENT3` (IV 0.823). The best genuinely new one is `FE_LIMIT_PER_AGE` (IV 0.183, novelty 0.57).
- Nine fields cover the useful signal: `PAY_1`, `PAY_2`, `N_ZERO_PAY`, `DELAY_TREND`, `FE_LIMIT_PER_AGE`, `FE_BILL_CV`, `PAY_TREND`, `BILL_TREND`, `FE_OVERPAY_MONTHS`.
- Story use: sets the field list for the drivers dashboard.

## 8. Segmentation and personas
Four complementary ways to cut the portfolio, each a natural Tableau filter / colour dimension:

1. **Rule-based risk tiers** (transparent, business-readable) → `RISK_TIER`
2. **Behavioural clusters** (KMeans on standardised behaviour, target not used; GMM for comparison) → `CLUSTER`, `PERSONA`
3. **Demographic-only cells** (age × education × sex × marriage) — to test how much demographics alone separate risk
4. **RFM-style credit score** (Recency of delinquency, Frequency of late months, Monetary size) → `RFM_R, RFM_F, RFM_M, RFM_SCORE, RFM_TOTAL`

A final comparison (8.6) puts all segmentations on one scale (in-sample AUC / IV of the segment default rate).

New `df` columns: `RISK_TIER, CLUSTER, PERSONA, GMM_COMP, RFM_R, RFM_F, RFM_M, RFM_SCORE, RFM_TOTAL`.

**Section guide**

- **What it does:** Splits the clients into groups in four ways: simple rule-based risk tiers, behaviour clusters found by an algorithm, demographic cells, and an RFM score. It then compares which split separates risk best.
- **Question it answers:** Can we sort clients into a few clear groups with very different default rates, and do behaviour groups beat demographic groups?
- **Inputs / new columns:** `STATUS_GRP_SEP`, `N_LATE`, `AVG_UTIL`, payment ratios, bills, payments, limit, demographics. Adds `RISK_TIER`, `CLUSTER`, `PERSONA`, `GMM_COMP`, `RFM_R`, `RFM_F`, `RFM_M`, `RFM_SCORE`, `RFM_TOTAL`.
- **How to read the charts:** bars = default rate per segment with 95% CI whiskers; dashed line = 22.1% base rate; radar and parallel-coordinate lines = each persona's average behaviour in standard units (0 = portfolio average).
- **Terms:** KMeans = groups clients around k centre points without using default. Silhouette = how well separated clusters are (1 = perfect). RFM = Recency, Frequency and size (Monetary) of lateness. AUC = chance a random defaulter is ranked above a random non-defaulter (0.5 = coin flip).
- **Key takeaways:**
  - Risk tiers run from 10.9% (T1) to 69.6% (T5); T5 is 10.4% of clients but 32.8% of defaulters.
  - C4 Chronic late: 17.7% of clients, 53.7% default, 42.9% of defaulters (number from the pre-fix run; refresh after the next Kaggle run).
  - C1 Moderate revolver: 14.0% default but 55.2% of defaulted exposure; most money is lost here (number from the pre-fix run; refresh after the next Kaggle run).
  - RFM score separates best (AUC 0.751); demographic cells reach only 0.581.
- **Where it feeds:** persona cards and the tier colour in Tableau; `s08_client_segments.csv` joins to the main extract on `ID`.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score, roc_auc_score

def seg_summary(data, col, order=None):
    # n, share, default rate + CI, exposure (positive Sep bill), limits, share of defaults
    t = rate_table(data, col)
    ex = data.assign(_exp=data["BILL_AMT1"].clip(lower=0),
                     _exp_def=data["BILL_AMT1"].clip(lower=0) * data[TARGET]).groupby(col, observed=True).agg(
        exposure_NT=("_exp", "sum"), exposure_defaulted_NT=("_exp_def", "sum"), total_limit_NT=("LIMIT_BAL", "sum"),
        avg_limit_NT=("LIMIT_BAL", "mean")).reset_index()
    t = t.merge(ex, on=col)
    t["share_exposure"] = t["exposure_NT"] / t["exposure_NT"].sum()
    t["share_exposure_defaulted"] = t["exposure_defaulted_NT"] / t["exposure_defaulted_NT"].sum()
    if order is not None:
        t = t.set_index(col).reindex([o for o in order if o in set(t[col])]).reset_index()
    return t
print("section 8 helpers ready")

### 8.1 Rule-based risk tiers
Rules are applied top-down (first match wins). They use only three intuitive inputs: September status (`STATUS_GRP_SEP`),
number of late months in six (`N_LATE`) and average utilisation (`AVG_UTIL`).

| Tier | Rule |
|---|---|
| **T5 Severe** | late ≥ 2 months in Sep |
| **T4 High** | late 1 month in Sep, **or** ≥ 3 late months in the window |
| **T3 Elevated** | 1–2 late months earlier in the window (current in Sep), **or** average utilisation ≥ 90% |
| **T2 Watch** | never late, average utilisation 50–90% |
| **T1 Low** | never late, utilisation < 50% (includes full payers and inactive cards) |

Exposure = positive September bill (NT$). `share_exposure_defaulted` shows where the defaulted balance actually sits.

In [ ]:
TIER_ORDER = ["T1 Low", "T2 Watch", "T3 Elevated", "T4 High", "T5 Severe"]
sep = df["STATUS_GRP_SEP"]
conds = [sep.isin(["Late 2m", "Late 3m+"]),
         (sep == "Late 1m") | (df["N_LATE"] >= 3),
         (df["N_LATE"] >= 1) | (df["AVG_UTIL"] >= 0.9),
         df["AVG_UTIL"] >= 0.5]
df["RISK_TIER"] = pd.Categorical(np.select(conds, TIER_ORDER[::-1][:4], default="T1 Low"), categories=TIER_ORDER, ordered=True)
tier_tbl = seg_summary(df, "RISK_TIER", TIER_ORDER)
display(tier_tbl[["RISK_TIER", "n", "share_clients", "defaults", "rate", "ci_lo", "ci_hi", "lift", "share_defaults",
                  "exposure_NT", "share_exposure", "share_exposure_defaulted", "avg_limit_NT"]].round(4))

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
plot_rate(df, "RISK_TIER", ax=axes[0], order=TIER_ORDER, title="Default rate by rule-based risk tier")
x = np.arange(len(tier_tbl)); w = .27
for j, (c, lab, col) in enumerate([("share_clients", "clients", C_NEU), ("share_exposure", "exposure (Sep bill)", C_OK),
                                   ("share_defaults", "defaulters", C_DEF)]):
    axes[1].bar(x + (j - 1) * w, tier_tbl[c], w, label=lab, color=col)
axes[1].set_xticks(x); axes[1].set_xticklabels(tier_tbl["RISK_TIER"]); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].legend(); axes[1].set_title("Share of clients vs exposure vs defaulters by tier")
plt.tight_layout(); savefig("s08_risk_tiers"); plt.show()
export(tier_tbl, "s08_risk_tiers")

**Observations:**
- The tiers rank risk cleanly: T1 10.9%, T2 12.8%, T3 21.5%, T4 34.6%, T5 69.6% (base rate 22.1%).
- T5 Severe holds 10.4% of clients but 32.8% of defaulters and 42.7% of defaulted exposure.
- T1 Low holds 44.7% of clients and still produces 22% of defaulters (1,463), so low risk does not mean no risk.
- Story use: the headline segmentation and Tableau colour; "one client in ten produces a third of defaults".

### 8.2 Behavioural clustering (KMeans, target not used)
Inputs (behaviour + credit line, **no demographics, no target**): utilisation level/trend, repayment ratio, status-code counts,
worst delay, zero-payment months, log average bill/payment, log limit. Values are clipped at the 1st/99th percentile and standardised.
`PAY_TO_BILL_5M` (share of bills repaid, 5 statements, aligned: `PAY_AMT1..5` over `BILL_AMT2..6`) is undefined for clients with no positive bill; it is set to 1 (nothing owed = nothing unpaid).

**Choosing k:** elbow (inertia = total spread inside clusters; look for the bend) and silhouette (how well separated the clusters are, -1 to 1) on a 6k sample for k = 3..8. Rule: take the k with the best silhouette among k ≥ 4
unless k = 3 is clearly better (> 0.03 higher), because ≥ 4 personas are more useful on a dashboard. Override with `K_OVERRIDE`.

In [ ]:
CL_FEATS = ["AVG_UTIL", "UTIL_TREND", "PAY_TO_BILL_5M", "N_LATE", "MAX_DELAY", "N_INACTIVE", "N_PAIDFULL", "N_REVOLVE",
            "N_ZERO_PAY", "LOG_AVG_BILL", "LOG_AVG_PAY", "LOG_LIMIT"]
Xc = pd.DataFrame({
    "AVG_UTIL": df["AVG_UTIL"], "UTIL_TREND": df["UTIL_TREND"],
    "PAY_TO_BILL_5M": df["PAY_TO_BILL_5M"].fillna(1.0).clip(0, 2),
    "N_LATE": df["N_LATE"], "MAX_DELAY": df["MAX_DELAY"], "N_INACTIVE": df["N_INACTIVE"], "N_PAIDFULL": df["N_PAIDFULL"],
    "N_REVOLVE": df["N_REVOLVE"], "N_ZERO_PAY": df["N_ZERO_PAY"],
    "LOG_AVG_BILL": np.sign(df["AVG_BILL"]) * np.log1p(df["AVG_BILL"].abs()),
    "LOG_AVG_PAY": np.log1p(df["AVG_PAY"]), "LOG_LIMIT": np.log(df["LIMIT_BAL"]),
})[CL_FEATS]
Xc = Xc.clip(Xc.quantile(.01), Xc.quantile(.99), axis=1)
scaler = StandardScaler().fit(Xc)
Z_all = scaler.transform(Xc)
idx_s = RNG.choice(len(Z_all), 6000, replace=False)
Zs = Z_all[idx_s]
ks, inertia, sil = list(range(3, 9)), [], []
for k in ks:
    km = KMeans(k, n_init=10, random_state=42).fit(Zs)
    inertia.append(km.inertia_); sil.append(silhouette_score(Zs, km.labels_, sample_size=4000, random_state=42))
sel = pd.DataFrame({"k": ks, "inertia": inertia, "silhouette": sil})
K_OVERRIDE = None
best4 = sel[sel["k"] >= 4].sort_values("silhouette", ascending=False).iloc[0]
K_BEST = int(K_OVERRIDE or (3 if sel.loc[sel.k == 3, "silhouette"].iloc[0] > best4["silhouette"] + 0.03 else best4["k"]))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(ks, inertia, marker="o"); axes[0].set_title("Elbow: within-cluster SS (6k sample)"); axes[0].set_xlabel("k")
axes[1].plot(ks, sil, marker="o", color=C_ACC); axes[1].set_title("Silhouette (6k sample)"); axes[1].set_xlabel("k")
for ax in axes:
    ax.axvline(K_BEST, ls="--", color=C_DEF)
plt.tight_layout(); savefig("s08_k_selection"); plt.show()
display(sel.round(4)); print("chosen k =", K_BEST)

**Observations:**
- Silhouette measures how well separated clusters are (1 = perfect). It peaks at k = 5 (0.345), against 0.29 at k = 3 and 0.27 at k = 8 (number from the pre-fix run; refresh after the next Kaggle run).
- A score of 0.35 means moderate separation: behaviour forms overlapping groups, not sharp islands (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: background only.

#### 8.2.1 Fit, order clusters by risk, profile and name personas
KMeans is fitted on all 30k clients. Cluster ids are re-numbered by ascending default rate (C0 = safest), which uses the target only
for *ordering/labelling*, not for fitting. Persona names are assigned by transparent rules on the cluster means
(first match wins; edit the thresholds or names freely):
`Dormant (no payments)` / `Inactive prepayer` (≥ 3 inactive months, split by ≥ 3 zero-payment months) → `Chronic late` (≥ 2.5 late months or mean worst delay ≥ 2) → `Transactor` (≥ 2.5 paid-in-full months or
repays ≥ 70% of bills, `PAY_TO_BILL_5M`) → `Heavy revolver` (utilisation ≥ 60%) → `Occasional late` (≥ 0.8 late months) → `Light revolver` (utilisation < 25%)
→ `Moderate revolver`. Duplicate names get a qualifier from the centroid feature that separates them most (e.g. "high/low utilisation").

In [ ]:
km = KMeans(K_BEST, n_init=20, random_state=42).fit(Z_all)
raw_lab = km.labels_
order = pd.Series(df[TARGET].values).groupby(raw_lab).mean().sort_values().index
remap = {old: new for new, old in enumerate(order)}
df["CLUSTER"] = pd.Series(raw_lab).map(remap).values
centroids_z = pd.DataFrame(Z_all, columns=CL_FEATS).groupby(df["CLUSTER"].values).mean()

PROF_COLS = ["LIMIT_BAL", "AGE", "AVG_UTIL", "UTIL_TREND", "PAY_TO_BILL_5M", "N_LATE", "MAX_DELAY", "N_INACTIVE", "N_PAIDFULL",
             "N_REVOLVE", "N_ZERO_PAY", "AVG_BILL", "AVG_PAY", "AVG_SPEND"]
prof = df.groupby("CLUSTER")[PROF_COLS].mean()

def persona_rule(r):
    if r["N_INACTIVE"] >= 3: return "Dormant (no payments)" if r["N_ZERO_PAY"] >= 3 else "Inactive prepayer"
    if r["N_LATE"] >= 2.5 or r["MAX_DELAY"] >= 2: return "Chronic late"
    if r["N_PAIDFULL"] >= 2.5 or r["PAY_TO_BILL_5M"] >= 0.7: return "Transactor"
    if r["AVG_UTIL"] >= 0.6: return "Heavy revolver"
    if r["N_LATE"] >= 0.8: return "Occasional late"
    if r["AVG_UTIL"] < 0.25: return "Light revolver"
    return "Moderate revolver"

base_names = prof.apply(persona_rule, axis=1)
QUAL = {"AVG_UTIL": "utilisation", "UTIL_TREND": "utilisation trend", "PAY_TO_BILL_5M": "share of bills repaid", "N_LATE": "late months",
        "MAX_DELAY": "delay", "N_INACTIVE": "inactivity", "N_PAIDFULL": "full payment", "N_REVOLVE": "revolving",
        "N_ZERO_PAY": "zero-pay months", "LOG_AVG_BILL": "balance", "LOG_AVG_PAY": "payments", "LOG_LIMIT": "limit"}
names = dict(base_names)
for nm in base_names[base_names.duplicated(keep=False)].unique():
    grp = base_names[base_names == nm].index
    cz_g = centroids_z.loc[grp]
    f = (cz_g.max() - cz_g.min()).idxmax()                  # feature that separates the duplicates most
    for c in grp:
        hi = cz_g.loc[c, f] >= cz_g[f].median()
        names[c] = f"{nm} ({'high' if hi else 'low'} {QUAL[f]})"
dup = pd.Series(names)
for nm in dup[dup.duplicated(keep=False)].unique():          # still duplicated -> number them
    for j, c in enumerate(dup[dup == nm].index):
        names[c] = f"{nm} {j + 1}"
PERSONA_NAMES = {c: f"C{c} {nm}" for c, nm in names.items()}
PERSONA_ORDER = [PERSONA_NAMES[c] for c in sorted(PERSONA_NAMES)]
df["PERSONA"] = pd.Categorical(df["CLUSTER"].map(PERSONA_NAMES), categories=PERSONA_ORDER, ordered=True)

clu = seg_summary(df, "CLUSTER")
cluster_profile = (prof.reset_index().merge(clu[["CLUSTER", "n", "share_clients", "defaults", "rate", "ci_lo", "ci_hi", "lift",
                                                  "share_defaults", "share_exposure", "share_exposure_defaulted"]], on="CLUSTER"))
cluster_profile.insert(1, "PERSONA", cluster_profile["CLUSTER"].map(PERSONA_NAMES))
display(cluster_profile.round(3))
export(cluster_profile, "s08_cluster_profiles")

**Observations:**
- C4 Chronic late is 17.7% of clients but has a 53.7% default rate and 42.9% of all defaulters. It also has the lowest average limit (NT$82,781) and 3.6 late months on average (number from the pre-fix run; refresh after the next Kaggle run).
- C1 Moderate revolver is the largest group (47.1%) with only 14.0% default, yet it carries 78.8% of exposure and 55.2% of defaulted exposure. Most of the money lost sits here (number from the pre-fix run; refresh after the next Kaggle run).
- C3 Dormant (no payments) is 5.8% of clients at 30.7% default: near-zero bills, but paying nothing is a warning sign (number from the pre-fix run; refresh after the next Kaggle run).
- C0 Inactive prepayer is the safest group (10.5%) (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: persona cards; contrast "who defaults most" (C4) with "where the money is lost" (C1).

#### 8.2.2 Centroid shapes: radar and parallel coordinates
Standardised (z-score) centroid of each cluster on the clustering inputs. The radar is good for a persona card; the parallel-coordinates
view makes the contrasts between clusters easier to compare.

In [ ]:
from pandas.plotting import parallel_coordinates
cz = centroids_z.copy(); cz.index = [PERSONA_NAMES[c] for c in cz.index]
pal = sns.color_palette("tab10", len(cz))
ang = np.linspace(0, 2 * np.pi, len(CL_FEATS), endpoint=False).tolist(); ang += ang[:1]
fig = plt.figure(figsize=(19, 7.5))
ax = fig.add_subplot(1, 2, 1, polar=True)
for (nm, row), col in zip(cz.iterrows(), pal):
    v = row.tolist() + row.tolist()[:1]
    ax.plot(ang, v, color=col, lw=2, label=nm); ax.fill(ang, v, color=col, alpha=.06)
ax.set_xticks(ang[:-1]); ax.set_xticklabels(CL_FEATS, fontsize=8)
ax.set_title("Cluster centroids (z-scores) - radar", y=1.08)
ax.legend(loc="upper left", bbox_to_anchor=(-0.35, 1.12), fontsize=8)
ax2 = fig.add_subplot(1, 2, 2)
parallel_coordinates(cz.reset_index().rename(columns={"index": "persona"}), "persona", color=pal, ax=ax2, lw=2.5)
ax2.axhline(0, color="k", lw=.8); ax2.set_ylabel("z-score of cluster mean"); ax2.tick_params(axis="x", rotation=45, labelsize=8)
ax2.set_title("Cluster centroids - parallel coordinates"); ax2.legend(fontsize=8)
plt.tight_layout(); savefig("s08_cluster_centroids"); plt.show()
fig, ax = plt.subplots(figsize=(12, 0.6 * len(cz) + 1.5))
sns.heatmap(cz, annot=True, fmt=".2f", cmap=CMAP_DIV, center=0, ax=ax)
ax.set_title("Centroid z-scores (heatmap form, Tableau highlight-table equivalent)")
plt.tight_layout(); savefig("s08_cluster_centroid_heatmap"); plt.show()

**Observations:**
- C4 Chronic late stands out on lateness (3.6 late months vs at most 0.6 elsewhere, `MAX_DELAY` 2.4) and has the highest utilisation (0.60) (number from the pre-fix run; refresh after the next Kaggle run).
- C0 and C3 are both mostly inactive (5.5 and 4.7 inactive months out of 6). C0 still pays (average NT$6,477 per month), while C3 pays almost nothing (NT$91) (number from the pre-fix run; refresh after the next Kaggle run).
- C2 Transactor paid in full in 4.4 of 6 months and uses under 5% of its limit (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: the persona-card radar visual.

#### 8.2.3 Gaussian Mixture comparison
GMM (Gaussian mixture model, full covariance) is fitted on an 8k sample for k = 3..8 (BIC curve; BIC = Bayesian information criterion, lower = better fit after a penalty for complexity), then at the chosen k on all clients. Agreement with KMeans is
measured by ARI / NMI (adjusted Rand index / normalised mutual information: 0 = random, 1 = identical), and the cross-tab shows which KMeans personas the GMM merges or splits. GMM handles elongated clusters
(e.g. a utilisation gradient) better than KMeans' spherical assumption.

In [ ]:
Zg = Z_all[RNG.choice(len(Z_all), 8000, replace=False)]
bic = []
for k in ks:
    g = GaussianMixture(k, covariance_type="full", random_state=42, n_init=2, reg_covar=1e-4).fit(Zg)
    bic.append(g.bic(Zg))
gmm = GaussianMixture(K_BEST, covariance_type="full", random_state=42, n_init=3, reg_covar=1e-4).fit(Zg)
gl = gmm.predict(Z_all)
g_order = pd.Series(df[TARGET].values).groupby(gl).mean().sort_values().index
df["GMM_COMP"] = pd.Series(gl).map({o: n for n, o in enumerate(g_order)}).values
ari = adjusted_rand_score(df["CLUSTER"], df["GMM_COMP"]); nmi = normalized_mutual_info_score(df["CLUSTER"], df["GMM_COMP"])
print(f"KMeans vs GMM (k={K_BEST}): ARI {ari:.3f} | NMI {nmi:.3f}")

fig, axes = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw={"width_ratios": [0.8, 1.3, 1]})
axes[0].plot(ks, bic, marker="o", color=C_ACC); axes[0].axvline(K_BEST, ls="--", color=C_DEF)
axes[0].set_title("GMM BIC by k (8k sample, lower = better)"); axes[0].set_xlabel("k")
ct = pd.crosstab(df["PERSONA"], df["GMM_COMP"], normalize="index")
sns.heatmap(ct, annot=True, fmt=".0%", cmap="Blues", ax=axes[1]); axes[1].set_title("KMeans persona (rows) -> GMM component share")
axes[1].set_xlabel("GMM component (ordered by default rate)")
gr = rate_table(df, "GMM_COMP"); kr = rate_table(df, "CLUSTER")
axes[2].errorbar(kr["CLUSTER"] - .1, kr["rate"], yerr=[kr["rate"] - kr["ci_lo"], kr["ci_hi"] - kr["rate"]], fmt="o", label="KMeans", color=C_OK)
axes[2].errorbar(gr["GMM_COMP"] + .1, gr["rate"], yerr=[gr["rate"] - gr["ci_lo"], gr["ci_hi"] - gr["rate"]], fmt="s", label="GMM", color=C_DEF)
axes[2].axhline(BASE_RATE, ls="--", color="k"); axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[2].set_xlabel("segment id (ordered by default rate)"); axes[2].legend(); axes[2].set_title("Default-rate spread: KMeans vs GMM")
plt.tight_layout(); savefig("s08_gmm_compare"); plt.show()
print("GMM components:"); print(gr[["GMM_COMP", "n", "share_clients", "rate", "ci_lo", "ci_hi"]].round(3).to_string(index=False))

**Observations:**
- GMM (Gaussian mixture, a clustering method that allows oval, overlapping groups) agrees only moderately with KMeans: ARI 0.50 and NMI 0.54 (both 0 = random, 1 = identical) (number from the pre-fix run; refresh after the next Kaggle run).
- GMM also isolates a high-risk group: component 4 holds 21.9% of clients at 45.5% default, while the others range from 10.4% to 24.9% (number from the pre-fix run; refresh after the next Kaggle run).
- GMM ranks risk about as well as KMeans (AUC 0.70 vs 0.695 in 8.6), so we keep the easier-to-name KMeans personas (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: background only; a robustness check on the persona split.

### 8.3 Personas: default rate and demographic composition
Default rate per persona (with 95% Wilson CIs), then composition heatmaps: within each persona, the share of clients in each
demographic category (rows sum to 100%). Compare with the portfolio mix (last row) to see over-representation.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.8))
plot_rate(df, "PERSONA", ax=ax, order=PERSONA_ORDER, rotate=20, title="Default rate by behavioural persona (bars = clients)")
plt.tight_layout(); savefig("s08_persona_default_rate"); plt.show()

DEMO_DIMS = [("SEX_LABEL", ["Male", "Female"]), ("EDU_LABEL", EDU_ORDER), ("MAR_LABEL", MAR_ORDER),
             ("AGE_BAND", AGE_LABELS), ("LIMIT_BAND", LIMIT_LABELS), ("RISK_TIER", TIER_ORDER)]
fig, axes = plt.subplots(2, 3, figsize=(22, 10))
comp_rows = []
for ax, (col, order_) in zip(axes.ravel(), DEMO_DIMS):
    ct = pd.crosstab(df["PERSONA"], df[col].astype(str), normalize="index")
    ct = ct.reindex(columns=[o for o in order_ if o in ct.columns])
    ct.loc["ALL clients"] = df[col].astype(str).value_counts(normalize=True).reindex(ct.columns).values
    sns.heatmap(ct, annot=True, fmt=".0%", cmap="Blues", ax=ax, cbar=False, annot_kws={"size": 8})
    ax.set_title(f"Persona composition by {col}"); ax.set_ylabel(""); ax.set_xlabel(""); ax.tick_params(labelsize=8)
    comp_rows.append(ct.drop(index="ALL clients").stack().rename("share").reset_index().rename(columns={col: "category"}).assign(dimension=col))
plt.tight_layout(); savefig("s08_persona_composition"); plt.show()
persona_comp = pd.concat(comp_rows, ignore_index=True)[["PERSONA", "dimension", "category", "share"]]
export(persona_comp, "s08_persona_composition")

**Observations:**
- Default rate by persona runs from 10.5% (C0) to 53.7% (C4) (number from the pre-fix run; refresh after the next Kaggle run).
- C4 Chronic late is young and on small limits: about 20% are aged 21-25 (vs about 4% in C0) and about 54% have limits of NT$50k or less (number from the pre-fix run; refresh after the next Kaggle run).
- C0, C2 and C3 lean to graduate school (about 47-51%), while C1 and C4 lean to university (about 52-55%) (number from the pre-fix run; refresh after the next Kaggle run).
- The sex mix barely moves: the female share is about 55-67% in every persona (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: shows demographics follow behaviour rather than drive it; persona composition chart.

#### 8.3.1 Persona × risk tier default-rate grid
Does the behavioural persona add information *within* a rule-based tier? Cells with n < 50 are blanked.

In [ ]:
g = rate_table(df, ["PERSONA", "RISK_TIER"])
piv_r = g.pivot(index="PERSONA", columns="RISK_TIER", values="rate")
piv_n = g.pivot(index="PERSONA", columns="RISK_TIER", values="n")
fig, ax = plt.subplots(figsize=(12, 0.6 * len(piv_r) + 2))
sns.heatmap(piv_r.where(piv_n >= 50), annot=piv_r.where(piv_n >= 50).map(lambda v: "" if pd.isna(v) else f"{v:.0%}") + "\n" + piv_n.fillna(0).astype(int).astype(str).radd("n="),
            fmt="", cmap=CMAP_RATE, vmin=0, vmax=.8, ax=ax, annot_kws={"size": 8})
ax.set_title("Default rate: persona x risk tier (n >= 50)")
plt.tight_layout(); savefig("s08_persona_x_tier"); plt.show()
export(g, "s08_persona_x_tier")

**Observations:**
- Personas add information inside a tier. In T1 Low, C3 Dormant defaults at about 23% vs about 8% for C1 revolvers (number from the pre-fix run; refresh after the next Kaggle run).
- In T4 High, C4 Chronic late defaults at about 43% vs about 19% for C2 Transactors (number from the pre-fix run; refresh after the next Kaggle run).
- In T5 Severe, rates range from about 44% (C2) to about 72% (C4) (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: Tableau highlight table; argues for combining simple rules with behavioural personas.

### 8.4 Demographic-only segmentation
Every age band × education × sex × marital-status cell with n ≥ 100 is ranked by default rate. If demographics were strong
drivers, the top and bottom cells would be far apart *and* large; compare the spread with the behavioural segmentations in 8.6.

In [ ]:
DEMO_KEY = ["AGE_BAND", "EDU_LABEL", "SEX_LABEL", "MAR_LABEL"]
cells = rate_table(df, DEMO_KEY, min_n=100)
cells["cell"] = cells[DEMO_KEY].astype(str).agg(" | ".join, axis=1)
cells = cells.sort_values("rate", ascending=False).reset_index(drop=True)
all_cells = rate_table(df, DEMO_KEY)
print(f"cells total {len(all_cells)} | with n>=100: {len(cells)} covering {cells['n'].sum() / len(df):.1%} of clients")
print(f"rate range among n>=100 cells: {cells['rate'].min():.1%} - {cells['rate'].max():.1%}; "
      f"n-weighted std of cell rates {np.sqrt(np.average((cells['rate'] - BASE_RATE) ** 2, weights=cells['n'])):.3f}")
top, bot = cells.head(15), cells.tail(15).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(20, 7), sharex=True)
for ax, t, ttl, col in [(axes[0], top, "Top 15 demographic cells (highest default)", C_DEF), (axes[1], bot, "Bottom 15 demographic cells (lowest default)", C_OK)]:
    t = t.iloc[::-1]
    ax.barh(t["cell"], t["rate"], xerr=[t["rate"] - t["ci_lo"], t["ci_hi"] - t["rate"]], color=col, alpha=.8, capsize=2)
    for y, (r, n) in enumerate(zip(t["rate"], t["n"])):
        ax.text(r + .01, y, f"n={n}", va="center", fontsize=7)
    ax.axvline(BASE_RATE, ls="--", color="k"); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(ttl); ax.tick_params(axis="y", labelsize=8)
plt.tight_layout(); savefig("s08_demo_cells"); plt.show()
display(pd.concat([top.assign(group="top"), bot.assign(group="bottom")])[["group", "cell", "n", "rate", "ci_lo", "ci_hi", "lift"]].round(3))
export(cells, "s08_demographic_cells")

**Observations:**
- The 68 cells with at least 100 clients cover 93.2% of clients, but their rates span only 12.4%-32.2% (weighted SD 4.3 pp).
- The riskiest demographic cell (married university women aged 21-25, n = 264) reaches only 32.2% (lift 1.46), far below the 69.6% of risk tier T5.
- The safest cells are mostly graduate-school women aged 26-45 (about 16-19%).
- Story use: the "demographics alone barely separate risk" contrast panel.

### 8.5 RFM-style credit score (Recency, Frequency, Monetary)
Adapted from marketing RFM to delinquency (higher digit = riskier for R and F; M is size):

| Digit | Definition |
|---|---|
| **R** Recency of delinquency | 0 never late in window, 1 latest late month in Apr–Jun, 2 in Jul–Aug, 3 late in Sep |
| **F** Frequency of late months | 0 = none, 1 = one, 2 = two–three, 3 = four–six |
| **M** Monetary (average bill) | 1/2/3 = tertile of `AVG_BILL` (low / mid / high balance) |

`RFM_SCORE` is the 3-digit string (e.g. `332`), `RFM_TOTAL` the digit sum. Note R = 0 ⇔ F = 0 by construction.

In [ ]:
late = df[PAY_COLS].to_numpy() >= 1                     # col 0 = Sep
last_late = np.where(late.any(1), late.argmax(1), -1)    # 0 = Sep, 1 = Aug, ... 5 = Apr
df["RFM_R"] = np.select([last_late == -1, last_late == 0, last_late <= 2], [0, 3, 2], default=1)
df["RFM_F"] = pd.cut(df["N_LATE"], [-1, 0, 1, 3, 6], labels=[0, 1, 2, 3]).astype(int)
df["RFM_M"] = pd.qcut(df["AVG_BILL"].rank(method="first"), 3, labels=[1, 2, 3]).astype(int)
df["RFM_SCORE"] = df["RFM_R"].astype(str) + df["RFM_F"].astype(str) + df["RFM_M"].astype(str)
df["RFM_TOTAL"] = df[["RFM_R", "RFM_F", "RFM_M"]].sum(1)
print("AVG_BILL tertile cut points (NT$):", df.groupby("RFM_M")["AVG_BILL"].agg(["min", "max"]).round(0).to_dict("index"))

rfm = rate_table(df, "RFM_SCORE", min_n=50).sort_values("rate", ascending=False)
fig, axes = plt.subplots(1, 4, figsize=(22, 4.8), gridspec_kw={"width_ratios": [1, 1, 1, 1.3]})
for ax, m in zip(axes[:3], [1, 2, 3]):
    sub = rate_table(df[df["RFM_M"] == m], ["RFM_R", "RFM_F"])
    pr = sub.pivot(index="RFM_R", columns="RFM_F", values="rate"); pn = sub.pivot(index="RFM_R", columns="RFM_F", values="n")
    ann = pr.map(lambda v: "" if pd.isna(v) else f"{v:.0%}") + pn.map(lambda v: "" if pd.isna(v) else f"\nn={int(v)}")
    sns.heatmap(pr.where(pn >= 30), annot=ann, fmt="", cmap=CMAP_RATE, vmin=0, vmax=.8, ax=ax, cbar=m == 3, annot_kws={"size": 8})
    ax.set_title(f"M = {m} ({['low', 'mid', 'high'][m - 1]} balance): rate by R x F"); ax.invert_yaxis()
plot_rate(df, "RFM_TOTAL", ax=axes[3], title="Default rate by RFM total (0-9)")
plt.tight_layout(); savefig("s08_rfm"); plt.show()
display(rfm[["RFM_SCORE", "n", "share_clients", "rate", "ci_lo", "ci_hi", "lift"]].round(3))
export(rate_table(df, ["RFM_SCORE", "RFM_R", "RFM_F", "RFM_M"]), "s08_rfm_scores")

**Observations:**
- RFM here means Recency, Frequency and size (Monetary) of lateness. The worst code, 331 (late in September, frequently late, small balance), defaults at 72.2% (n = 363); 333 is at 66.8% (n = 984).
- Never-late clients (codes 00x, 66.4% of clients) default at only 9.1%-15.8%. Among them, smaller balances are riskier (001 15.8% vs 003 9.1%).
- Recency dominates: every 3xx code except 311 is above 43%.
- Story use: a simple score card; "recent lateness is the alarm bell".

### 8.6 Which segmentation separates risk best?
Each segmentation is turned into a score = in-sample default rate of the client's segment. AUC (ranking quality), IV, and the
n-weighted spread of segment rates are compared. Demographic-only cells use all cells (no minimum n, so their AUC is if anything
optimistic). This quantifies the claim that **behaviour, not demographics, drives default**.

In [ ]:
df["_DEMO_CELL"] = df[DEMO_KEY].astype(str).agg("|".join, axis=1)
seg_defs = {"Demographic cells (age x edu x sex x mar)": "_DEMO_CELL", "Education only": "EDU_LABEL", "Age band only": "AGE_BAND",
            "Limit band": "LIMIT_BAND", "Rule-based risk tier": "RISK_TIER", f"KMeans personas (k={K_BEST})": "PERSONA",
            f"GMM components (k={K_BEST})": "GMM_COMP", "RFM score (3-digit)": "RFM_SCORE", "RFM total": "RFM_TOTAL",
            "Sep status group": "STATUS_GRP_SEP"}
rows = []
for name, col in seg_defs.items():
    r = df.groupby(col, observed=True)[TARGET].transform("mean")
    g = df.groupby(col, observed=True)[TARGET].agg(["mean", "size"])
    rows.append({"segmentation": name, "column": col, "n_segments": len(g), "AUC": roc_auc_score(df[TARGET], r),
                 "IV": woe_iv(df.assign(_s=df[col].astype(str)), "_s")[1],
                 "rate_min": g["mean"].min(), "rate_max": g["mean"].max(),
                 "weighted_sd_rate": np.sqrt(np.average((g["mean"] - BASE_RATE) ** 2, weights=g["size"]))})
df.drop(columns="_DEMO_CELL", inplace=True)
seg_cmp = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)
display(seg_cmp.round(3))
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.barh(seg_cmp["segmentation"][::-1], seg_cmp["AUC"][::-1],
        color=[C_NEU if ("Demo" in s or "only" in s) else C_OK for s in seg_cmp["segmentation"][::-1]])
for y, v in enumerate(seg_cmp["AUC"][::-1]):
    ax.text(v + .003, y, f"{v:.3f}", va="center", fontsize=8)
ax.axvline(.5, ls="--", color="k"); ax.set_xlim(.45, max(.85, seg_cmp["AUC"].max() + .03))
ax.set_title("Risk separation by segmentation (in-sample AUC of segment default rate); grey = demographic")
plt.tight_layout(); savefig("s08_segmentation_compare"); plt.show()
export(seg_cmp, "s08_segmentation_compare")

**Observations:**
- AUC is the chance that a random defaulter is ranked above a random non-defaulter (0.5 = coin flip). The RFM score is best (0.751), then RFM total (0.744) and the 5-group risk tier (0.738).
- KMeans personas reach 0.695 without ever seeing the target (number from the pre-fix run; refresh after the next Kaggle run).
- Demographic cells use 177 segments and still reach only 0.581. Education alone gives 0.541 and age band 0.540.
- Story use: the key comparison chart, behaviour segments vs demographic segments.

### 8.7 Client-level segment export
One row per client with every segment label, ready to join to the main Tableau extract on `ID`.

In [ ]:
client_segments = df[["ID", "RISK_TIER", "CLUSTER", "PERSONA", "GMM_COMP", "RFM_SCORE", "RFM_TOTAL"]].copy()
client_segments["RISK_TIER"] = client_segments["RISK_TIER"].astype(str)
client_segments["PERSONA"] = client_segments["PERSONA"].astype(str)
export(client_segments, "s08_client_segments")
print(pd.crosstab(df["PERSONA"], df["RISK_TIER"], margins=True).to_string())

**Observations:**
- Personas and tiers mostly agree: 95% of C0 (2,036 of 2,137) is in T1, and all but one C4 client is in T3-T5 (number from the pre-fix run; refresh after the next Kaggle run).
- C3 Dormant splits almost evenly between T1 (838) and T4 (800), so the rules disagree about dormant clients (number from the pre-fix run; refresh after the next Kaggle run).
- 628 C1 revolvers are already in T5 Severe: normal spenders who have just slipped (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: background only; this is the join file for Tableau segment filters.

## 9. Hidden-pattern discovery (GPU-accelerated)
**Purpose: discovery, not prediction.** Sections 3–8 looked at default rates through lenses *we* chose. Here we let algorithms search the space for structure we did not think to look for:

| Lens | Question it answers | Tableau use |
|---|---|---|
| 9.1 XGBoost + native SHAP (main effects + **pairwise interactions**) | Which variables carry signal, and which *pairs* behave non-additively? | importance bar, interaction heatmap |
| 9.2 Partial dependence / ICE, 2-way PDP | What is the *shape* of each effect, and how do interacting pairs combine? | line charts, 2-D heatmaps |
| 9.3 Rule mining (shallow tree, tree ensemble, exhaustive subgroup search) | Which readable conjunctions of conditions isolate high-risk pockets? | ranked rule table |
| 9.4 Association rules (Apriori) | Which banded profiles co-occur with DEFAULT = 1 / DEFAULT = 0? | rule table, support–confidence scatter |
| 9.5 UMAP embedding + density clusters | Does behaviour form natural "islands" with distinct default rates? | 2-D scatter coloured by segment |
| 9.6 Isolation Forest | Are *unusual* clients riskier, or is risk mainstream? | rate by anomaly decile |

The model is a **lens**: it is fitted out-of-fold only so that SHAP values are not memorised noise. **No accuracy leaderboard is reported** — one AUC line exists purely to confirm the lens has learned something real.

Every block auto-detects the GPU (`GPU` flag from setup). On a Kaggle T4: XGBoost trains with `device="cuda"`, SHAP interactions are computed for **all 30,000 rows**, the subgroup search runs on CuPy and the embedding uses cuML UMAP on the full data. On CPU the same code runs with smaller samples (settings below).

**Section guide**

- **What it does:** Lets algorithms search for patterns we did not think to look for. A model (XGBoost) is used only as a lens, not to predict. We also mine readable rules, find frequent combinations, draw a 2-D map of behaviour, and score unusual clients.
- **Question it answers:** Are there drivers, combinations or risky pockets that the hand-built charts missed?
- **Inputs / new columns:** raw columns plus setup features; banded variables for rules. No `df` columns; results go to exports (e.g. `EMB_X`, `EMB_Y` in the embedding file).
- **How to read the charts:** SHAP bars = average push of each feature on predicted risk; beeswarm dots = clients (right = more risk, red = high feature value). PDP line = average predicted risk as one feature changes. Subgroup scatter: x = share of clients, y = default rate. UMAP map: each dot is a client, nearby dots behave alike.
- **Terms:** SHAP = splits each prediction into a contribution per feature. UMAP = squeezes many columns into a 2-D map that keeps neighbours close. WRAcc = rule score that rewards pockets that are both large and risky. Isolation Forest = scores how unusual a client is.
- **Key takeaways:**
  - Lateness leads in every lens: `PAY_1`, `MAX_DELAY`, `N_LATE` carry 27.6% of SHAP attribution.
  - The "late-streak, low-repayment" pocket covers 15.3% of clients at 60.8% default.
  - Warning signs before lateness: 0-5% repayment with 1-2 zero-payment months = 43.8% default.
  - Most unusual decile defaults at 39.9% vs 7.6% for the most typical.
- **Where it feeds:** the driver-importance chart, the rules table (`s09_risk_rules.csv`) and a "risk map" scatter in Tableau.

### 9.0 Device detection and settings
Probes each accelerator individually (a GPU can exist while a given library lacks CUDA support), then picks a `SETTINGS` profile.

In [ ]:
import xgboost as xgb
from contextlib import contextmanager
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import IsolationForest

S9_TIMES = []
@contextmanager
def s9_timer(name, device):
    t0 = time.time()
    yield
    dt = time.time() - t0
    S9_TIMES.append({"block": name, "seconds": round(dt, 1), "device": device})
    print(f"[timing] {name}: {dt:.1f}s on {device}")

def _xgb_cuda_ok():
    if not GPU:
        return False
    try:
        d = xgb.DMatrix(np.random.rand(256, 4), label=np.random.randint(0, 2, 256))
        xgb.train({"device": "cuda", "tree_method": "hist", "verbosity": 0}, d, 2)
        return True
    except Exception as e:
        print("xgboost CUDA not usable -> CPU:", str(e)[:150])
        return False
XGB_DEVICE = "cuda" if _xgb_cuda_ok() else "cpu"

# array backend for the subgroup / itemset counting (CuPy if a working GPU build exists)
xp, ARRAY_DEVICE = np, "cpu (numpy)"
if GPU:
    try:
        import cupy as cp
        _ = float(cp.ones(8).sum())
        xp, ARRAY_DEVICE = cp, "cuda (cupy)"
    except Exception as e:
        print("cupy not usable -> numpy:", str(e)[:120])
def to_np(a):
    return a.get() if hasattr(a, "get") else np.asarray(a)

ON_GPU_XGB = XGB_DEVICE == "cuda"
SETTINGS = dict(
    xgb_rounds      = 900 if ON_GPU_XGB else 300,
    xgb_eta         = 0.03 if ON_GPU_XGB else 0.06,
    xgb_depth       = 5 if ON_GPU_XGB else 4,
    shap_inter_rows = None if ON_GPU_XGB else 3000,   # None = every row (out-of-fold)
    beeswarm_rows   = 8000 if GPU else 5000,
    pdp_rows        = 3000 if GPU else 800,
    pdp_grid        = 30 if GPU else 20,
    pdp2_grid       = 20 if GPU else 12,
    rule_trees      = 200 if GPU else 40,
    sg_min_n        = 200,                            # min clients in a subgroup
    apriori_minsup  = 0.01,
    embed_rows      = None if GPU else 8000,          # None = all 30,000 clients
    iforest_trees   = 400 if GPU else 200,
)
print(f"GPU present: {GPU} | XGBoost device: {XGB_DEVICE} | array backend: {ARRAY_DEVICE}")
pd.DataFrame([SETTINGS]).T.rename(columns={0: "value"})

### 9.1 XGBoost lens with native SHAP (main effects and interactions)
Feature set = raw columns (limit, demographics, 6 statuses, 6 bills, 6 payments) + the core behavioural layer from 0.3 (minus `AVG_BILL`/`AVG_PAY`, which are exact multiples of `TOTAL_BILL`/`TOTAL_PAY` and would only split SHAP credit). Five-fold **out-of-fold** fitting: every client's SHAP values come from a model that never saw that client. SHAP (each feature's share of the push on a client's predicted risk) is computed with XGBoost's own TreeSHAP (`pred_contribs`, `pred_interactions`) — on GPU this is GPUTreeShap and takes seconds even for all rows.

In [ ]:
S9_FEATS = (["LIMIT_BAL", "AGE", "SEX", "EDUCATION", "MARRIAGE"] + PAY_COLS + BILL_COLS + PAMT_COLS
            + [c for c in FEATURE_COLS if c not in ("AVG_BILL", "AVG_PAY")])
X9 = df[S9_FEATS].astype(float)
y9 = df[TARGET].to_numpy()
N9, P9 = X9.shape
XGB_PARAMS = {"objective": "binary:logistic", "eval_metric": "logloss", "eta": SETTINGS["xgb_eta"],
              "max_depth": SETTINGS["xgb_depth"], "subsample": 0.8, "colsample_bytree": 0.8,
              "min_child_weight": 5, "lambda": 1.0, "seed": 42, "device": XGB_DEVICE,
              "tree_method": "hist", "verbosity": 0}

inter_rows = (np.arange(N9) if SETTINGS["shap_inter_rows"] is None
              else np.sort(RNG.choice(N9, SETTINGS["shap_inter_rows"], replace=False)))
oof = np.zeros(N9)
shap_oof = np.zeros((N9, P9 + 1), dtype=np.float32)
inter_abs = np.zeros((P9 + 1, P9 + 1))
inter_signed = np.zeros((P9 + 1, P9 + 1))
n_inter = 0
skf = StratifiedKFold(5, shuffle=True, random_state=42)
with s9_timer("9.1 XGBoost 5-fold OOF + SHAP + SHAP interactions", XGB_DEVICE):
    for k, (tr, te) in enumerate(skf.split(X9, y9)):
        bst = xgb.train(XGB_PARAMS, xgb.DMatrix(X9.iloc[tr], label=y9[tr]), SETTINGS["xgb_rounds"])
        dte = xgb.DMatrix(X9.iloc[te])
        oof[te] = bst.predict(dte)
        shap_oof[te] = bst.predict(dte, pred_contribs=True)
        te_s = np.intersect1d(te, inter_rows)
        for s0 in range(0, len(te_s), 4000):                       # chunk to bound memory
            blk = te_s[s0:s0 + 4000]
            I = bst.predict(xgb.DMatrix(X9.iloc[blk]), pred_interactions=True)
            inter_abs += np.abs(I).sum(0); inter_signed += I.sum(0); n_inter += len(blk)
    # one model on all rows: used only for PDP/ICE curves in 9.2
    bst_full = xgb.train(XGB_PARAMS, xgb.DMatrix(X9, label=y9), SETTINGS["xgb_rounds"])

auc = roc_auc_score(y9, oof)
print(f"Sanity check only (not a deliverable): out-of-fold AUC = {auc:.3f} -> the lens captures real structure.")
print(f"SHAP main effects: {N9:,} rows x {P9} features | SHAP interactions: {n_inter:,} rows")

#### 9.1a Global importance (mean |SHAP|)
Mean absolute SHAP value (log-odds units; log-odds = log of p / (1 − p)) per feature, plus the direction of the effect (Spearman correlation between the feature value and its SHAP value: + means higher value → higher default risk).

In [ ]:
SHAP_VALS = shap_oof[:, :-1]
mean_abs = np.abs(SHAP_VALS).mean(0)
direction = [stats.spearmanr(X9[c].fillna(X9[c].median()), SHAP_VALS[:, j])[0] for j, c in enumerate(S9_FEATS)]
imp = pd.DataFrame({"feature": S9_FEATS, "mean_abs_shap": mean_abs,
                    "mean_shap_defaulters": SHAP_VALS[y9 == 1].mean(0),
                    "mean_shap_nondefaulters": SHAP_VALS[y9 == 0].mean(0),
                    "direction_spearman": direction})
imp["share"] = imp["mean_abs_shap"] / imp["mean_abs_shap"].sum()
imp = imp.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
imp.insert(0, "rank", np.arange(1, len(imp) + 1))
imp["cum_share"] = imp["share"].cumsum()
export(imp, "s09_shap_importance")
TOP_FEATS = imp["feature"].tolist()

top = imp.head(25).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top["feature"], top["mean_abs_shap"], color=[C_DEF if d > 0 else C_OK for d in top["direction_spearman"]])
ax.set_xlabel("mean |SHAP| (log-odds)"); ax.set_title("Global SHAP importance (red = higher value raises risk, blue = lowers)")
savefig("s09_shap_importance"); plt.show()
print(f"Top 5 features carry {imp['cum_share'].iloc[4]:.1%} of total |SHAP|; top 10 carry {imp['cum_share'].iloc[9]:.1%}.")
display(imp.head(20).round(4))

**Observations:**
- SHAP splits each prediction into a push from every feature. The XGBoost lens is a fair mirror of the data (out-of-fold AUC 0.782).
- September status `PAY_1` (mean |SHAP| 0.30), `MAX_DELAY` (0.29) and `N_LATE` (0.17) lead; these three lateness features carry 27.6% of all attribution.
- Spending comes next: `AVG_SPEND` (0.14), where more spending means lower risk. `LIMIT_BAL` ranks 8th, and a higher limit also means lower risk.
- Only marital status reaches the top 20 among demographics (rank 13). Sex, education and age do not.
- Story use: the driver-importance chart.

#### 9.1b SHAP summary (beeswarm)
Each dot is a client: horizontal position = contribution to log-odds of default, colour = feature value (red high, blue low). Spread shows heterogeneity; colour gradients show monotonicity.

In [ ]:
bs_idx = RNG.choice(N9, min(SETTINGS["beeswarm_rows"], N9), replace=False)
try:
    import shap
    shap.summary_plot(SHAP_VALS[bs_idx], X9.iloc[bs_idx], max_display=20, show=False, plot_size=(10, 8))
    plt.title("SHAP beeswarm (out-of-fold, sample)"); savefig("s09_shap_beeswarm"); plt.show()
except Exception as e:
    print("shap plotting unavailable -> manual beeswarm:", str(e)[:100])
    feats = TOP_FEATS[:20][::-1]
    fig, ax = plt.subplots(figsize=(10, 8))
    for r, f in enumerate(feats):
        j = S9_FEATS.index(f)
        col = X9[f].iloc[bs_idx].rank(pct=True).fillna(0.5)
        ax.scatter(SHAP_VALS[bs_idx, j], r + RNG.normal(0, 0.12, len(bs_idx)), c=col, cmap="coolwarm", s=3, alpha=.5)
    ax.set_yticks(range(len(feats))); ax.set_yticklabels(feats); ax.axvline(0, color="k", lw=.8)
    ax.set_xlabel("SHAP value (log-odds)"); ax.set_title("SHAP beeswarm (manual)")
    savefig("s09_shap_beeswarm"); plt.show()

**Observations:**
- For `PAY_1`, defaulters average +0.19 log-odds of risk and non-defaulters -0.17, the widest gap of any feature.
- The direction is consistent: lateness features push risk up as they rise (`N_LATE` direction +0.83, `MAX_DELAY` +0.70). Money features push it down (`LIMIT_BAL` -0.85, `AVG_PAY_RATIO` -0.83, `AVG_SPEND` -0.81).
- Story use: background or appendix visual.

#### 9.1c SHAP interaction matrix → ranked feature pairs
`pred_interactions` decomposes each prediction into main effects (diagonal) and pairwise interaction effects (off-diagonal). The mean |interaction| matrix tells us which **pairs** act non-additively — a direct, data-driven shortlist for the 2-way heatmaps in Section 4 and Tableau.

In [ ]:
M_abs = inter_abs[:-1, :-1] / n_inter
M_sig = inter_signed[:-1, :-1] / n_inter
main_share = np.trace(M_abs) / M_abs.sum()
print(f"Share of total |SHAP-interaction mass| on the diagonal (main effects): {main_share:.1%}; pairwise interactions: {1 - main_share:.1%}")

iu, ju = np.triu_indices(P9, 1)
pairs = pd.DataFrame({"feature_a": np.array(S9_FEATS)[iu], "feature_b": np.array(S9_FEATS)[ju],
                      "mean_abs_interaction": M_abs[iu, ju] + M_abs[ju, iu],     # both symmetric halves
                      "mean_signed_interaction": M_sig[iu, ju] + M_sig[ju, iu]})
pairs = pairs.sort_values("mean_abs_interaction", ascending=False).reset_index(drop=True)
pairs.insert(0, "rank", np.arange(1, len(pairs) + 1))
pairs["share_of_interaction_mass"] = pairs["mean_abs_interaction"] / pairs["mean_abs_interaction"].sum()
export(pairs, "s09_shap_interactions")

hm_feats = TOP_FEATS[:20]
hi = [S9_FEATS.index(f) for f in hm_feats]
Hm = M_abs[np.ix_(hi, hi)] * 2
np.fill_diagonal(Hm, np.nan)                # hide main effects so interactions are readable
H = pd.DataFrame(Hm, index=hm_feats, columns=hm_feats)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(H, cmap=CMAP_RATE, ax=ax, cbar_kws={"label": "mean |SHAP interaction| (log-odds)"}, linewidths=.3)
ax.set_title("SHAP interaction strength among top-20 features (diagonal = main effect, hidden)")
savefig("s09_shap_interaction_heatmap"); plt.show()
print("Top 15 feature-pair interactions:")
display(pairs.head(15).round(5))

**Observations:**
- Off-diagonal (pairwise) terms hold 63% of the absolute SHAP-interaction mass, so effects are far from purely additive.
- The strongest pair is `PAY_1` x `MAX_DELAY` (0.074), more than twice the next one (`PAY_1` x `N_LATE`, 0.033).
- The top pair without a lateness feature is `BILL_AMT1` x `MAX_UTIL` (0.030): balance size matters differently depending on how full the card is.
- Story use: shortlist for the interaction heatmaps.

#### 9.1d Dependence plots for the top 6 features, coloured by the strongest interaction partner
x = feature value, y = its SHAP value. Colour = percentile of the partner with the largest mean |interaction| for that feature. A vertical colour gradient at a fixed x is the visual signature of an interaction.

In [ ]:
def strongest_partner(f):
    j = S9_FEATS.index(f)
    row = M_abs[j].copy() + M_abs[:, j]; row[j] = -1
    return S9_FEATS[int(np.argmax(row))]

dep_idx = RNG.choice(N9, min(8000, N9), replace=False)
fig, axes = plt.subplots(2, 3, figsize=(17, 9.5))
dep_rows = []
for ax, f in zip(axes.ravel(), TOP_FEATS[:6]):
    p = strongest_partner(f); j = S9_FEATS.index(f)
    x = X9[f].iloc[dep_idx].to_numpy(); sv = SHAP_VALS[dep_idx, j]
    col = X9[p].iloc[dep_idx].rank(pct=True).fillna(0.5).to_numpy()
    if X9[f].nunique() <= 15:
        x = x + RNG.normal(0, 0.08, len(x))
    lo, hi_ = np.nanpercentile(x, [1, 99]); keep = (x >= lo) & (x <= hi_)
    sc = ax.scatter(x[keep], sv[keep], c=col[keep], cmap="coolwarm", s=5, alpha=.6)
    ax.axhline(0, color="k", lw=.7); ax.set_xlabel(f); ax.set_ylabel(f"SHAP({f})")
    ax.set_title(f"{f}  | colour: {p}", fontsize=10)
    dep_rows.append({"feature": f, "strongest_partner": p})
fig.colorbar(sc, ax=axes, shrink=.6, label="partner percentile (blue low → red high)")
fig.suptitle("SHAP dependence plots (top 6 features)", fontweight="bold")
savefig("s09_shap_dependence"); plt.show()
display(pd.DataFrame(dep_rows))

**Observations:**
- `PAY_1` is the strongest interaction partner for three of the top six features (`MAX_DELAY`, `N_LATE`, `AVG_SPEND`), so September status changes how other signals work.
- `BILL_AMT1` and `MAX_UTIL` partner each other, which forms a separate balance/utilisation pair.
- Story use: background only.

### 9.2 Partial dependence and ICE curves
PDP = average predicted default probability as one feature is swept over its range (others held at observed values); ICE = the same curve per client. Fanning ICE lines = the effect depends on other variables (interaction); parallel lines = additive effect. Computed manually by batch-predicting the full-data lens model on a replicated sample.

In [ ]:
pdp_samp = X9.sample(SETTINGS["pdp_rows"], random_state=1).reset_index(drop=True)
def feat_grid(col, n):
    v = X9[col].dropna().to_numpy()
    u = np.unique(v)
    if len(u) <= n:
        return u
    return np.unique(np.quantile(v, np.linspace(0.01, 0.99, n)))

def ice_curves(col, grid, base=pdp_samp):
    big = pd.concat([base] * len(grid), ignore_index=True)
    big[col] = np.repeat(grid, len(base))
    return bst_full.predict(xgb.DMatrix(big)).reshape(len(grid), len(base)).T   # rows = clients

PDP_FEATS = TOP_FEATS[:6]
pdp_rows, pdp_summary = [], []
with s9_timer("9.2 PDP / ICE (1-way)", XGB_DEVICE):
    fig, axes = plt.subplots(2, 3, figsize=(17, 9))
    for ax, f in zip(axes.ravel(), PDP_FEATS):
        g = feat_grid(f, SETTINGS["pdp_grid"])
        ice = ice_curves(f, g)
        pdp = ice.mean(0)
        show = RNG.choice(len(ice), min(120, len(ice)), replace=False)
        for r in show:
            ax.plot(g, ice[r], color=C_NEU, lw=.4, alpha=.35)
        ax.plot(g, pdp, color=C_DEF, lw=2.8, label="PDP")
        ax.axhline(BASE_RATE, ls="--", color="k", lw=1)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_ylim(0, 1)
        ax.set_title(f); ax.set_xlabel(f); ax.set_ylabel("predicted P(default)")
        centred = ice - ice[:, [0]]
        pdp_summary.append({"feature": f, "pdp_min": pdp.min(), "pdp_max": pdp.max(), "pdp_range": pdp.max() - pdp.min(),
                            "x_at_pdp_max": g[int(np.argmax(pdp))],
                            "ice_spread_at_end (sd of centred ICE)": centred[:, -1].std()})
        pdp_rows += [{"feature": f, "value": gv, "pdp": pv} for gv, pv in zip(g, pdp)]
    fig.suptitle("Partial dependence (red) and ICE (grey) for the top 6 SHAP features", fontweight="bold")
    plt.tight_layout(); savefig("s09_pdp_ice"); plt.show()
export(pd.DataFrame(pdp_rows), "s09_pdp")
display(pd.DataFrame(pdp_summary).round(4))

**Observations:**
- PDP (partial dependence) shows average predicted risk as one feature changes. `PAY_1` has the largest effect: predicted default jumps from about 19% to 36% (a 17 pp range) when September status reaches two months late.
- `MAX_DELAY` (9 pp range), `AVG_SPEND` (9 pp, falling as spend rises) and `N_LATE` (7 pp) follow. `MAX_UTIL` moves risk by only 3.4 pp.
- ICE (individual curves) spread most for `PAY_1` (SD 0.10), so the size of the status effect differs a lot between clients.
- Story use: the "two-month cliff" annotation.

#### 9.2b Two-way PDP for the top 3 interaction pairs vs the observed default rate
Top row: model-implied joint effect (2-way PDP). Bottom row: raw observed default rate in quantile cells (cells < 50 clients blanked). Agreement between rows means the interaction is visible in plain data, not just in the model — and is therefore safe to put in a Tableau heatmap. Caveat: PDP also evaluates *infeasible* combinations (e.g. `MAX_DELAY` < `PAY_1`, which cannot occur since MAX_DELAY is the max of the statuses); read the PDP only where the observed panel has data.

In [ ]:
def obs_bins(s):
    s = s.copy()
    if s.nunique() <= 12:
        lo, hi_ = s.quantile([.01, .99])
        return s.clip(lo, hi_).round(2)
    return pd.qcut(s.rank(method="first"), 5, labels=[f"Q{i}" for i in range(1, 6)])

TOP_PAIRS = pairs.head(3)[["feature_a", "feature_b"]].values.tolist()
with s9_timer("9.2b PDP (2-way)", XGB_DEVICE):
    fig, axes = plt.subplots(2, 3, figsize=(19, 11))
    for c, (fa, fb) in enumerate(TOP_PAIRS):
        ga, gb = feat_grid(fa, SETTINGS["pdp2_grid"]), feat_grid(fb, SETTINGS["pdp2_grid"])
        base = pdp_samp
        big = pd.concat([base] * (len(ga) * len(gb)), ignore_index=True)
        big[fa] = np.repeat(np.repeat(ga, len(gb)), len(base))
        big[fb] = np.repeat(np.tile(gb, len(ga)), len(base))
        Z2 = bst_full.predict(xgb.DMatrix(big)).reshape(len(ga) * len(gb), len(base)).mean(1).reshape(len(ga), len(gb))
        fmt = lambda v: f"{v:,.0f}" if abs(v) >= 100 else f"{v:.2g}"
        sns.heatmap(pd.DataFrame(Z2, index=[fmt(v) for v in ga], columns=[fmt(v) for v in gb]), ax=axes[0, c],
                    cmap=CMAP_RATE, vmin=0, vmax=1, cbar_kws={"label": "PDP P(default)"})
        axes[0, c].invert_yaxis(); axes[0, c].set_xlabel(fb); axes[0, c].set_ylabel(fa)
        axes[0, c].set_title(f"2-way PDP: {fa} x {fb}", fontsize=10)
        tmp = pd.DataFrame({"A": obs_bins(df[fa]), "B": obs_bins(df[fb]), TARGET: df[TARGET]})
        rt = rate_table(tmp, ["A", "B"])
        piv = rt.pivot(index="A", columns="B", values="rate").where(rt.pivot(index="A", columns="B", values="n") >= 50)
        sns.heatmap(piv, ax=axes[1, c], cmap=CMAP_RATE, vmin=0, vmax=1, annot=piv.shape[1] <= 8, fmt=".0%",
                    annot_kws={"size": 7}, cbar_kws={"label": "observed default rate"})
        axes[1, c].invert_yaxis(); axes[1, c].set_xlabel(fb); axes[1, c].set_ylabel(fa)
        axes[1, c].set_title(f"Observed rate: {fa} x {fb}", fontsize=10)
        axes[0, c].grid(False); axes[1, c].grid(False)
    plt.tight_layout(); savefig("s09_pdp_2way"); plt.show()

**Observations:**
- The three pairs shown are `PAY_1` x `MAX_DELAY`, `PAY_1` x `N_LATE` and `MAX_DELAY` x `LATE_STREAK_RECENT` (interaction strength 0.074, 0.033 and 0.031 in 9.1c).
- The raw data shows the same stacking: being 2 months late in September gives 69.1% default, and also having 4-6 late months raises it to 73.7% (9.3c).
- Story use: background only.

### 9.3 Rule mining: readable high-risk pockets
Three complementary rule sources:
1. **One shallow tree** (depth 4, ≥ 300 clients per leaf) — a compact, human-readable partition.
2. **A tree ensemble sweep** — many depth-3 trees on bootstrap samples and random feature subsets; every leaf becomes a candidate rule (RuleFit-style), re-scored on the full data.
3. **Exhaustive subgroup discovery** — every conjunction of 1–3 conditions over banded variables, scored by **WRAcc = coverage × (rate − base rate)**, which rewards pockets that are both large and risky. Counts for all ~50k triples come from matrix products (CuPy on GPU).

Note: the tree uses `PAY_TO_BILL_5M` = share of bills repaid (5 statements, aligned: `PAY_AMT1..5` over `BILL_AMT2..6`); in tree rules `PAY_TO_BILL_5M = -1` encodes "no positive bill Apr-Aug".

In [ ]:
TREE_FEATS = ["PAY_1", "PAY_2", "PAY_3", "N_LATE", "MAX_DELAY", "LATE_STREAK_RECENT", "LIMIT_BAL", "AGE",
              "AVG_UTIL", "UTIL_TREND", "PAY_TO_BILL_5M", "N_ZERO_PAY", "N_PAIDFULL", "N_INACTIVE",
              "DELAY_TREND", "PAY_AMT1", "BILL_AMT1", "TOTAL_PAY"]
XT = df[TREE_FEATS].astype(float).copy()
XT["PAY_TO_BILL_5M"] = XT["PAY_TO_BILL_5M"].fillna(-1)
INT_FEATS = {c for c in TREE_FEATS if (XT[c] % 1 == 0).all()}

def _num(v):
    return f"{v:,.0f}" if abs(v) >= 1000 else f"{v:.3g}"

def fmt_bounds(bounds):
    parts = []
    for f, (lo, hi) in bounds.items():
        if f in INT_FEATS:
            lo_i = math.floor(lo) + 1 if np.isfinite(lo) else None
            hi_i = math.floor(hi) if np.isfinite(hi) else None
            if lo_i is not None and hi_i is not None:
                parts.append(f"{f} = {lo_i}" if lo_i == hi_i else f"{lo_i} <= {f} <= {hi_i}")
            elif lo_i is not None: parts.append(f"{f} >= {lo_i}")
            else: parts.append(f"{f} <= {hi_i}")
        else:
            if np.isfinite(lo) and np.isfinite(hi): parts.append(f"{_num(lo)} < {f} <= {_num(hi)}")
            elif np.isfinite(lo): parts.append(f"{f} > {_num(lo)}")
            else: parts.append(f"{f} <= {_num(hi)}")
    return " AND ".join(parts) if parts else "(all)"

def tree_rules(clf, cols, Xf, yv):
    t = clf.tree_; leaf_ids = clf.apply(Xf.to_numpy()); out = []
    def rec(node, bounds):
        if t.children_left[node] == -1:
            m = leaf_ids == node; n = int(m.sum())
            if n:
                out.append({"rule": fmt_bounds(bounds), "n_conditions": len(bounds), "n": n,
                            "defaults": int(yv[m].sum()), "mask": m})
            return
        f, thr = cols[t.feature[node]], t.threshold[node]
        lo, hi = bounds.get(f, (-np.inf, np.inf))
        b1 = dict(bounds); b1[f] = (lo, min(hi, thr)); rec(t.children_left[node], b1)
        b2 = dict(bounds); b2[f] = (max(lo, thr), hi); rec(t.children_right[node], b2)
    rec(0, {})
    return out

def score_rules(rows, yv):
    r = pd.DataFrame(rows).drop(columns="mask", errors="ignore")
    r["coverage"] = r["n"] / len(yv); r["rate"] = r["defaults"] / r["n"]
    r["lift"] = r["rate"] / BASE_RATE; r["wracc"] = r["coverage"] * (r["rate"] - BASE_RATE)
    return r

with s9_timer("9.3a shallow decision tree", "cpu"):
    dt4 = DecisionTreeClassifier(max_depth=4, min_samples_leaf=300, random_state=42).fit(XT, y9)
    leaves = score_rules(tree_rules(dt4, TREE_FEATS, XT, y9), y9).sort_values("rate", ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(dt4, feature_names=TREE_FEATS, class_names=["No default", "Default"], filled=True, proportion=True,
          impurity=False, rounded=True, fontsize=7, ax=ax)
ax.set_title("Depth-4 rule tree (min 300 clients per leaf); 'value' = [share non-default, share default]")
savefig("s09_rule_tree"); plt.show()
with pd.option_context("display.max_colwidth", 200):
    display(leaves[["rule", "n", "coverage", "defaults", "rate", "lift", "wracc"]].round(4))

**Observations:**
- Every first split asks one question: is September status 2 or more months late (`PAY_1` >= 2)?
- The riskiest leaf combines `PAY_1` >= 2, repaying 13.8% of bills or less, a non-worsening delay trend and utilisation up to 0.80: 82.9% default (701 clients, lift 3.7) (number from the pre-fix run; refresh after the next Kaggle run).
- Long-standing lateness is worse than new lateness: inside `PAY_1` >= 2, a stable delay trend gives 71-83% default vs 64-74% for a worsening one (number from the pre-fix run; refresh after the next Kaggle run).
- The safest leaf (never late, paid more than NT$6,795 in six months, limit above NT$75k) holds 41.7% of clients at 8.5% (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: a decision-tree rule-of-thumb graphic.

#### 9.3b Tree-ensemble rule sweep (RuleFit-style candidate generation)
Different trees see different features and bootstrap samples, so they surface alternative descriptions of risky pockets that a single greedy tree hides. All leaves are re-scored on the full data and de-duplicated.

In [ ]:
with s9_timer("9.3b tree-ensemble rule sweep", "cpu"):
    ens_rows, seen = [], set()
    rs = np.random.RandomState(7)
    for b in range(SETTINGS["rule_trees"]):
        cols = list(rs.choice(TREE_FEATS, 6, replace=False))
        boot = rs.choice(N9, int(0.7 * N9), replace=True)
        clf = DecisionTreeClassifier(max_depth=3, min_samples_leaf=300, random_state=b).fit(XT[cols].iloc[boot], y9[boot])
        for r in tree_rules(clf, cols, XT[cols], y9):
            if r["rule"] not in seen and r["n"] >= 300:
                seen.add(r["rule"]); ens_rows.append(r)
    ens = score_rules(ens_rows, y9).sort_values("wracc", ascending=False).reset_index(drop=True)
print(f"{SETTINGS['rule_trees']} trees -> {len(ens):,} distinct leaf rules (n >= 300)")
print("Top 20 ensemble rules by WRAcc:")
with pd.option_context("display.max_colwidth", 200):
    display(ens.head(20)[["rule", "n", "coverage", "rate", "lift", "wracc"]].round(4))
print("Top 10 ensemble rules by lift (n >= 300):")
with pd.option_context("display.max_colwidth", 200):
    display(ens.sort_values("lift", ascending=False).head(10)[["rule", "n", "rate", "lift"]].round(4))

**Observations:**
- 200 shallow trees produced 1,323 distinct candidate rules (number from the pre-fix run; refresh after the next Kaggle run).
- The best broad rule is "currently in a late streak, paid at most about 26% of bills, September bill above NT$109": 4,592 clients (15.3%) at 60.8% default, about 42% of all defaulters (number from the pre-fix run; refresh after the next Kaggle run).
- The sharpest rule is `PAY_1` >= 2, late in all 6 months, utilisation at most 0.58: 82.6% default (n = 391) (number from the pre-fix run; refresh after the next Kaggle run).
- The top 10 rules are near-copies with slightly different thresholds, so the pocket is robust.
- Story use: "the 15% late-streak pocket" headline.

#### 9.3c Exhaustive subgroup discovery on banded variables (1–3 conditions)
Items = `variable = band` for 13 banded variables. For every conjunction of up to three items from *different* variables we count clients and defaulters via matrix products (`X.T @ X`, then one product per conditioning item for triples). A 2- or 3-condition pocket is kept only if it beats every one of its sub-pockets by ≥ 2 pp (non-redundancy filter).

In [ ]:
def _trend_band(s, tol):
    return pd.Series(np.select([s > tol, s < -tol], ["Rising", "Falling"], "Flat"), index=s.index)
BANDS = {
    "Sep status": df["STATUS_GRP_1"], "Aug status": df["STATUS_GRP_2"],
    "Limit": df["LIMIT_BAND"].astype(str), "Age": df["AGE_BAND"].astype(str), "Sex": df["SEX_LABEL"],
    "Education": df["EDU_LABEL"], "Marital": df["MAR_LABEL"], "Avg util": df["UTIL_BAND"].astype(str),
    "# late months": df["N_LATE_BAND"].astype(str), "Bills repaid 5m (aligned)": df["PAYRATIO_BAND"].astype(str),
    "# zero-pay months": pd.cut(df["N_ZERO_PAY"], [-1, 0, 2, 5, 6], labels=["0", "1-2", "3-5", "6"]).astype(str),
    "Util trend": _trend_band(df["UTIL_TREND"], 0.02),
    "Delay trend": pd.Series(np.select([df["DELAY_TREND"] > 0, df["DELAY_TREND"] < 0], ["Worsening", "Improving"], "Stable"), index=df.index),
}
item_cols, item_var = {}, []
for v, s in BANDS.items():
    for lvl in pd.Series(s).dropna().unique():
        m = (s == lvl).to_numpy()
        if m.sum() >= 30:
            item_cols[f"{v}={lvl}"] = m; item_var.append(v)
ITEMS = pd.DataFrame(item_cols)
ITEM_NAMES = np.array(ITEMS.columns); ITEM_VAR = np.array(item_var); m_items = len(ITEM_NAMES)
print(f"{m_items} items from {len(BANDS)} banded variables")

with s9_timer("9.3c subgroup counting (all 1-3 conjunctions)", ARRAY_DEVICE):
    Xb = xp.asarray(ITEMS.to_numpy(np.float32)); yb = xp.asarray(y9.astype(np.float32))
    Xy = Xb * yb[:, None]
    n1, k1 = Xb.sum(0), Xy.sum(0)
    n2, k2 = Xb.T @ Xb, Xb.T @ Xy
    n3 = xp.empty((m_items,) * 3, dtype=xp.float32); k3 = xp.empty_like(n3)
    for i in range(m_items):
        Xi = Xb * Xb[:, i:i + 1]
        n3[i] = Xi.T @ Xb; k3[i] = Xi.T @ Xy
    n1, k1, n2, k2, n3, k3 = map(to_np, (n1, k1, n2, k2, n3, k3))

with np.errstate(divide="ignore", invalid="ignore"):
    r1, r2 = k1 / n1, k2 / n2
MIN_N = SETTINGS["sg_min_n"]
recs = [pd.DataFrame({"items": [(i,) for i in range(m_items)], "n": n1, "defaults": k1, "best_parent_rate": BASE_RATE,
                      "worst_parent_rate": BASE_RATE})]
I2, J2 = np.triu_indices(m_items, 1)
ok = (ITEM_VAR[I2] != ITEM_VAR[J2]) & (n2[I2, J2] >= MIN_N); I2, J2 = I2[ok], J2[ok]
recs.append(pd.DataFrame({"items": list(zip(I2, J2)), "n": n2[I2, J2], "defaults": k2[I2, J2],
                          "best_parent_rate": np.maximum(r1[I2], r1[J2]), "worst_parent_rate": np.minimum(r1[I2], r1[J2])}))
ar = np.arange(m_items)
I3, J3, K3 = np.meshgrid(ar, ar, ar, indexing="ij")
ok = ((I3 < J3) & (J3 < K3) & (ITEM_VAR[I3] != ITEM_VAR[J3]) & (ITEM_VAR[I3] != ITEM_VAR[K3])
      & (ITEM_VAR[J3] != ITEM_VAR[K3]) & (n3 >= MIN_N))
I3, J3, K3 = I3[ok], J3[ok], K3[ok]
par = np.stack([r2[I3, J3], r2[I3, K3], r2[J3, K3]], 1)
recs.append(pd.DataFrame({"items": list(zip(I3, J3, K3)), "n": n3[I3, J3, K3], "defaults": k3[I3, J3, K3],
                          "best_parent_rate": par.max(1), "worst_parent_rate": par.min(1)}))
SG = pd.concat(recs, ignore_index=True)
SG = SG[SG["n"] >= MIN_N].copy()
SG["n"], SG["defaults"] = SG["n"].round().astype(int), SG["defaults"].round().astype(int)
SG["n_conditions"] = SG["items"].map(len)
SG["rule"] = SG["items"].map(lambda t: " AND ".join(ITEM_NAMES[list(t)]))
SG["vars"] = SG["items"].map(lambda t: set(ITEM_VAR[list(t)]))
SG["coverage"] = SG["n"] / N9; SG["rate"] = SG["defaults"] / SG["n"]
SG["lift"] = SG["rate"] / BASE_RATE; SG["wracc"] = SG["coverage"] * (SG["rate"] - BASE_RATE)
SG["gain_vs_parents"] = SG["rate"] - SG["best_parent_rate"]
SG["drop_vs_parents"] = SG["worst_parent_rate"] - SG["rate"]
print(f"Evaluated {len(SG):,} subgroups with n >= {MIN_N} (sizes: {SG['n_conditions'].value_counts().sort_index().to_dict()})")

risk = SG[(SG["n_conditions"] == 1) | (SG["gain_vs_parents"] >= 0.02)]
safe = SG[(SG["n_conditions"] == 1) | (SG["drop_vs_parents"] >= 0.02)]
DELINQ_VARS = {"Sep status", "Aug status", "# late months", "Delay trend"}
hidden = risk[risk["vars"].map(lambda s: not (s & DELINQ_VARS))]
cols_show = ["rule", "n_conditions", "n", "coverage", "rate", "lift", "wracc", "gain_vs_parents"]
with pd.option_context("display.max_colwidth", 200):
    print("A) Top 20 high-risk pockets by WRAcc (non-redundant):"); display(risk.nlargest(20, "wracc")[cols_show].round(4))
    print("B) Top 15 'hidden' pockets - defined WITHOUT any repayment-status variable:"); display(hidden.nlargest(15, "wracc")[cols_show].round(4))
    print("C) Top 15 highest-lift pockets with n >= 300:"); display(risk[risk["n"] >= 300].nlargest(15, "lift")[cols_show].round(4))
    print("D) Top 10 low-risk pockets (most negative WRAcc):"); display(safe.nsmallest(10, "wracc")[cols_show[:-1] + ["drop_vs_parents"]].round(4))

**Observations:**
- WRAcc scores a rule by both size and lift. Of 11,068 subgroups tested, the best are August status Late 2m (n = 3,927, 55.6%) and September status Late 2m (n = 2,667, 69.1%) (number from the pre-fix run; refresh after the next Kaggle run).
- The highest lift is August Late 2m + repaying 0-5% + stable delay trend: 78.2% default (n = 486, lift 3.5) (number from the pre-fix run; refresh after the next Kaggle run).
- Hidden pockets without any status variable: 1-2 zero-payment months (29.5% of clients, 30.8% default); limit up to NT$50k (31.8%). Combining 0-5% repayment with 1-2 zero-payment months gives 43.8% (n = 2,549, lift 2.0) (number from the pre-fix run; refresh after the next Kaggle run).
- The safest pocket is never-late clients (66.4% of clients, 11.7%); add graduate school and no zero-payment month and it falls to 7.8% (n = 4,733) (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: "warning signs before lateness": skipped payments and tiny repayments.

#### 9.3d Subgroup landscape and rule export
Each dot is a non-redundant subgroup: x = coverage, y = default rate. The top 20 by WRAcc are numbered (see table A). Dots up and to the right are the pockets worth a dedicated Tableau filter.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6.5))
for nc, mk in zip([1, 2, 3], ["o", "s", "^"]):
    s = risk[risk["n_conditions"] == nc]
    ax.scatter(s["coverage"], s["rate"], c=s["wracc"], cmap=CMAP_DIV, vmin=-0.05, vmax=0.05, s=14, marker=mk,
               alpha=.55, label=f"{nc} condition(s)")
topw = risk.nlargest(20, "wracc").reset_index(drop=True)
for r, row in topw.iterrows():
    ax.annotate(str(r + 1), (row["coverage"], row["rate"]), fontsize=8, fontweight="bold")
ax.axhline(BASE_RATE, ls="--", color="k", lw=1)
ax.set_xscale("log"); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("coverage (share of clients, log scale)"); ax.set_ylabel("default rate")
ax.set_title("Subgroup landscape: coverage vs default rate (colour = WRAcc)"); ax.legend()
savefig("s09_subgroup_landscape"); plt.show()

keep_cols = ["source", "rule", "n_conditions", "n", "coverage", "defaults", "rate", "lift", "wracc"]
rules_out = pd.concat([
    leaves.assign(source="tree_depth4"),
    ens.head(50).assign(source="tree_ensemble_top50_wracc"),
    risk.nlargest(100, "wracc").assign(source="subgroup_high_risk_top100"),
    hidden.nlargest(30, "wracc").assign(source="subgroup_hidden_no_status_top30"),
    safe.nsmallest(30, "wracc").assign(source="subgroup_low_risk_top30"),
], ignore_index=True)[keep_cols]
rules_out["defaults"] = rules_out["defaults"].astype(int); rules_out["n"] = rules_out["n"].astype(int)
export(rules_out, "s09_risk_rules")

**Observations:**
- The landscape has two zones. Broad pockets (10-30% of clients) sit at about 30-35% default, while pockets of 70% or more cover under 9% of clients (September Late 2m is 8.9%).
- 223 non-redundant rules are exported for a Tableau coverage-vs-rate scatter.
- Story use: the "size vs severity" scatter in Tableau.

### 9.4 Association rules (Apriori) on banded variables + DEFAULT
Classic market-basket view: each client is a "basket" of items (the bands from 9.3c plus `DEFAULT=1` or `DEFAULT=0`). A level-wise Apriori (candidate join + subset pruning, support counted on boolean matrices in chunks) finds all itemsets of size ≤ 3 with support ≥ 1%. Rules `antecedent → DEFAULT=1` (risk profiles) and `antecedent → DEFAULT=0` (safe profiles) are ranked by lift. Subgroup search (9.3c) asks "where is risk concentrated?"; Apriori adds the *frequency* constraint, so it surfaces common profiles, not niche pockets.

In [ ]:
from collections import defaultdict
AR_ITEMS = ITEMS.copy()
AR_ITEMS["DEFAULT=1"] = y9 == 1; AR_ITEMS["DEFAULT=0"] = y9 == 0
AR_NAMES = np.array(AR_ITEMS.columns); AR_VAR = np.array(list(ITEM_VAR) + ["DEFAULT", "DEFAULT"])

def apriori(B, var, min_support, max_len=3, chunk=2000):
    Bx = xp.asarray(B); n = B.shape[0]
    sup1 = to_np(Bx.sum(0)) / n
    levels = [{(i,): float(s) for i, s in enumerate(sup1) if s >= min_support}]
    for k in range(2, max_len + 1):
        prev = sorted(levels[-1]); prev_set = set(prev)
        by_prefix = defaultdict(list)
        for it in prev:
            by_prefix[it[:-1]].append(it[-1])
        cands = []
        for pre, lasts in by_prefix.items():
            lasts = sorted(lasts)
            for a in range(len(lasts)):
                for b in range(a + 1, len(lasts)):
                    c = pre + (lasts[a], lasts[b])
                    if len({var[i] for i in c}) < k:                          # one band per variable
                        continue
                    if all(c[:j] + c[j + 1:] in prev_set for j in range(k)):   # Apriori pruning
                        cands.append(c)
        if not cands:
            break
        C = np.array(cands); sup = []
        for s0 in range(0, len(C), chunk):
            cc = C[s0:s0 + chunk]
            M = Bx[:, cc[:, 0]]
            for j in range(1, k):
                M = M & Bx[:, cc[:, j]]
            sup.append(to_np(M.sum(0)) / n)
        sup = np.concatenate(sup)
        levels.append({tuple(c): float(s) for c, s in zip(cands, sup) if s >= min_support})
        print(f"  level {k}: {len(cands):,} candidates -> {len(levels[-1]):,} frequent")
    return levels

with s9_timer("9.4 Apriori (size <= 3)", ARRAY_DEVICE):
    LEVELS = apriori(AR_ITEMS.to_numpy(bool), AR_VAR, SETTINGS["apriori_minsup"])
print("frequent itemsets by size:", {k + 1: len(L) for k, L in enumerate(LEVELS)})
try:
    from mlxtend.frequent_patterns import apriori as mlx_apriori
    fi = mlx_apriori(AR_ITEMS, min_support=SETTINGS["apriori_minsup"], max_len=3)
    print(f"mlxtend cross-check: {len(fi):,} frequent itemsets vs ours {sum(len(L) for L in LEVELS):,}")
except Exception:
    print("mlxtend not installed - using the in-house Apriori only")

SUP = {}
for L in LEVELS:
    SUP.update(L)
d1, d0 = len(AR_NAMES) - 2, len(AR_NAMES) - 1
rule_rows = []
for its, s in SUP.items():
    for cons, base in ((d1, BASE_RATE), (d0, 1 - BASE_RATE)):
        if cons in its and len(its) > 1:
            ant = tuple(i for i in its if i != cons)
            conf = s / SUP[ant]
            rule_rows.append({"antecedent": " AND ".join(AR_NAMES[list(ant)]), "consequent": AR_NAMES[cons],
                              "n_items": len(ant), "support": s, "antecedent_support": SUP[ant], "confidence": conf,
                              "lift": conf / base, "leverage": s - SUP[ant] * base,
                              "conviction": (1 - base) / max(1e-9, 1 - conf)})
AR = pd.DataFrame(rule_rows)
AR1, AR0 = AR[AR["consequent"] == "DEFAULT=1"], AR[AR["consequent"] == "DEFAULT=0"]
ar_cols = ["antecedent", "support", "antecedent_support", "confidence", "lift", "leverage", "conviction"]
with pd.option_context("display.max_colwidth", 200):
    print(f"{len(AR1):,} rules -> DEFAULT=1 | {len(AR0):,} rules -> DEFAULT=0")
    print("Top 15 rules -> DEFAULT=1 by lift:"); display(AR1.nlargest(15, "lift")[ar_cols].round(4))
    print("Top 10 rules -> DEFAULT=1 by support (common risky profiles, lift >= 1.5):"); display(AR1[AR1["lift"] >= 1.5].nlargest(10, "support")[ar_cols].round(4))
    print("Top 15 rules -> DEFAULT=0 by lift (safe profiles):"); display(AR0.nlargest(15, "lift")[ar_cols].round(4))
export(pd.concat([AR1.nlargest(200, "lift"), AR0.nlargest(200, "lift")]), "s09_assoc_rules")

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
for ax, R, ttl in ((axes[0], AR1, "rules -> DEFAULT=1"), (axes[1], AR0, "rules -> DEFAULT=0")):
    sc = ax.scatter(R["support"], R["confidence"], c=R["lift"], cmap=CMAP_DIV, s=10 + 6 * R["n_items"], alpha=.6,
                    norm=plt.matplotlib.colors.TwoSlopeNorm(vcenter=1.0, vmin=min(0.3, R["lift"].min()), vmax=max(R["lift"].max(), 1.01)))
    ax.set_xscale("log"); ax.set_xlabel("support (log)"); ax.set_ylabel("confidence")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title(f"Association {ttl} (colour = lift)")
    fig.colorbar(sc, ax=ax, label="lift")
savefig("s09_assoc_rules"); plt.show()

**Observations:**
- Apriori is the market-basket method: it finds item combinations that often occur together. It found 10,312 frequent itemsets, exactly matching mlxtend, which yield 653 rules leading to default and 1,121 leading to no default.
- The top default rule is 4-6 late months + stable delay trend: 77.5% confidence (default rate), lift 3.5.
- The most common risky profile is a worsening delay trend: 9.0% of all clients have it and default (confidence 34%, lift 1.5).
- Safe rules are capped by the 77.9% base repayment rate: the best (limit NT$300-500k + utilisation 10-30%) repays 94.8% (lift 1.22).
- Story use: background only; confirms the subgroup findings.

### 9.5 Behavioural embedding (UMAP) and density clusters
Inputs: the six monthly statuses, utilisations, payment ratios, log payments and log limit (standardised) — i.e. *behaviour only*, no target. UMAP squeezes these 24 dimensions into 2 while keeping neighbourhoods. If default concentrates on specific islands, risk has a behavioural "shape" that Tableau can show as a scatter. Backend: cuML UMAP (GPU) → umap-learn (CPU) → t-SNE (fallback).

In [ ]:
EMB_COLS = PAY_COLS + [f"UTIL_{i}" for i in IDX] + [f"PAY_RATIO_{i}" for i in IDX[:-1]] + PAMT_COLS + ["LIMIT_BAL"]
E = df[EMB_COLS].astype(float).copy()
for i in IDX:
    E[f"UTIL_{i}"] = E[f"UTIL_{i}"].clip(-0.5, 1.5)
for i in IDX[:-1]:
    E[f"PAY_RATIO_{i}"] = E[f"PAY_RATIO_{i}"].fillna(0).clip(0, 2)       # NaN = no previous bill
for c in PAMT_COLS + ["LIMIT_BAL"]:
    E[c] = np.log1p(E[c].clip(lower=0))
Z9 = StandardScaler().fit_transform(E)

emb_idx = (np.arange(N9) if SETTINGS["embed_rows"] is None
           else np.sort(RNG.choice(N9, SETTINGS["embed_rows"], replace=False)))
EMB_BACKEND, emb_device = None, "cpu"
if GPU:
    try:
        from cuml.manifold import UMAP as _UMAP
        EMB_BACKEND, emb_device = "cuML UMAP", "cuda"
    except Exception as e:
        print("cuML not available:", str(e)[:100])
if EMB_BACKEND is None:
    try:
        from umap import UMAP as _UMAP
        EMB_BACKEND = "umap-learn"
    except Exception as e:
        print("umap-learn not available:", str(e)[:100])
if EMB_BACKEND is None:
    from sklearn.manifold import TSNE
    EMB_BACKEND = "sklearn t-SNE"
    emb_idx = np.sort(RNG.choice(N9, min(5000, N9), replace=False))

with s9_timer(f"9.5 embedding ({EMB_BACKEND}, {len(emb_idx):,} rows)", emb_device):
    Zs = Z9[emb_idx].astype(np.float32)
    if EMB_BACKEND == "sklearn t-SNE":
        EMB = TSNE(n_components=2, perplexity=40, init="pca", random_state=42).fit_transform(Zs)
    else:
        EMB = to_np(_UMAP(n_neighbors=30, min_dist=0.0, n_components=2, random_state=42).fit_transform(Zs))
print(f"Embedding backend: {EMB_BACKEND} | rows embedded: {len(emb_idx):,} of {N9:,}")

# density clusters on the 2-D embedding
CL_BACKEND, min_cs = None, max(50, len(emb_idx) // 100)
with s9_timer("9.5 density clustering", "cuda" if GPU else "cpu"):
    labels = None
    if GPU:
        try:
            from cuml.cluster import HDBSCAN as _HDB
            labels = to_np(_HDB(min_cluster_size=min_cs).fit_predict(EMB.astype(np.float32))); CL_BACKEND = "cuML HDBSCAN"
        except Exception:
            labels = None
    if labels is None:
        try:
            from sklearn.cluster import HDBSCAN as _HDB
            labels = _HDB(min_cluster_size=min_cs).fit_predict(EMB); CL_BACKEND = "sklearn HDBSCAN"
        except Exception:
            try:
                import hdbscan
                labels = hdbscan.HDBSCAN(min_cluster_size=min_cs).fit_predict(EMB); CL_BACKEND = "hdbscan"
            except Exception:
                from sklearn.cluster import DBSCAN
                labels = DBSCAN(eps=0.3, min_samples=20).fit_predict(StandardScaler().fit_transform(EMB)); CL_BACKEND = "sklearn DBSCAN"
print(f"Clustering: {CL_BACKEND} (min_cluster_size={min_cs}) -> {len(set(labels)) - (-1 in labels)} clusters, noise share {np.mean(labels == -1):.1%}")

emb_df = df.iloc[emb_idx][["ID", TARGET, "DEFAULT_LABEL", "STATUS_GRP_SEP", "LIMIT_BAND", "AGE_BAND", "SEX_LABEL",
                           "EDU_LABEL", "N_LATE", "AVG_UTIL", "PAY_TO_BILL_5M", "LIMIT_BAL", "AGE", "PAY_1"]].copy()
emb_df["EMB_X"], emb_df["EMB_Y"] = EMB[:, 0], EMB[:, 1]
emb_df["CLUSTER"] = np.where(labels == -1, "noise", "C" + pd.Series(labels).astype(str).str.zfill(2)).astype(str)
emb_df["EMB_BACKEND"] = EMB_BACKEND
export(emb_df[["ID", "EMB_X", "EMB_Y", "CLUSTER", TARGET, "DEFAULT_LABEL", "STATUS_GRP_SEP", "LIMIT_BAND",
               "AGE_BAND", "SEX_LABEL", "EDU_LABEL", "EMB_BACKEND"]], "s09_embedding")

#### 9.5a The embedding coloured four ways, plus a local default-rate surface
Top: by default, by September status, by limit band. Bottom: density clusters (labelled with their default rate) and a hexbin of the **local default rate** — the most direct "where does risk live" view.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(19, 12))
ax = axes[0, 0]
for v, c in ((0, C_OK), (1, C_DEF)):
    s = emb_df[emb_df[TARGET] == v]
    ax.scatter(s["EMB_X"], s["EMB_Y"], s=2, alpha=.35 if v else .2, color=c, label="Default" if v else "No default")
ax.legend(markerscale=6); ax.set_title("Coloured by default")
ax = axes[0, 1]
pal = dict(zip(STATUS_GROUP_ORDER, ["#bdbdbd", "#2b8cbe", "#31a354", "#fdae6b", "#e6550d", "#7f0000"]))
for g in STATUS_GROUP_ORDER:
    s = emb_df[emb_df["STATUS_GRP_SEP"] == g]
    ax.scatter(s["EMB_X"], s["EMB_Y"], s=2, alpha=.4, color=pal[g], label=g)
ax.legend(markerscale=6, fontsize=8); ax.set_title("Coloured by September status")
ax = axes[0, 2]
lpal = dict(zip(LIMIT_LABELS, sns.color_palette("mako_r", len(LIMIT_LABELS))))
for g in LIMIT_LABELS:
    s = emb_df[emb_df["LIMIT_BAND"].astype(str) == g]
    ax.scatter(s["EMB_X"], s["EMB_Y"], s=2, alpha=.4, color=lpal[g], label=g)
ax.legend(markerscale=6, fontsize=8); ax.set_title("Coloured by limit band")

cl_tab = rate_table(emb_df, "CLUSTER").sort_values("rate", ascending=False).reset_index(drop=True)
ax = axes[1, 0]
cls = [c for c in emb_df["CLUSTER"].unique() if c != "noise"]
cpal = dict(zip(sorted(cls), sns.color_palette("tab20", max(1, len(cls)))))
nz = emb_df[emb_df["CLUSTER"] == "noise"]
ax.scatter(nz["EMB_X"], nz["EMB_Y"], s=1, color="lightgrey", alpha=.3)
for c in sorted(cls):
    s = emb_df[emb_df["CLUSTER"] == c]
    ax.scatter(s["EMB_X"], s["EMB_Y"], s=2, alpha=.5, color=cpal[c])
    rr = cl_tab.set_index("CLUSTER").loc[c, "rate"]
    ax.annotate(f"{c}\n{rr:.0%}", (s["EMB_X"].median(), s["EMB_Y"].median()), fontsize=8, fontweight="bold", ha="center",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", alpha=.7, lw=0))
ax.set_title(f"{CL_BACKEND} clusters (label = default rate)")
ax = axes[1, 1]
hb = ax.hexbin(emb_df["EMB_X"], emb_df["EMB_Y"], C=emb_df[TARGET], reduce_C_function=np.mean, gridsize=40,
               mincnt=max(5, len(emb_df) // 2000), cmap=CMAP_RATE, vmin=0, vmax=1)
fig.colorbar(hb, ax=ax, format=mtick.PercentFormatter(1), label="local default rate"); ax.set_title("Local default-rate surface (hexbin)")
ax = axes[1, 2]
sc = ax.scatter(emb_df["EMB_X"], emb_df["EMB_Y"], c=emb_df["AVG_UTIL"].clip(0, 1.1), s=2, alpha=.4, cmap="viridis")
fig.colorbar(sc, ax=ax, label="avg utilisation"); ax.set_title("Coloured by average utilisation")
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.suptitle(f"Behavioural embedding ({EMB_BACKEND}, {len(emb_df):,} clients)", fontweight="bold")
plt.tight_layout(); savefig("s09_embedding"); plt.show()

prof = emb_df.groupby("CLUSTER").agg(median_PAY_1=("PAY_1", "median"), mean_N_LATE=("N_LATE", "mean"),
                                      median_AVG_UTIL=("AVG_UTIL", "median"), median_BILLS_REPAID_5M=("PAY_TO_BILL_5M", "median"),
                                      median_LIMIT=("LIMIT_BAL", "median"), median_AGE=("AGE", "median"),
                                      top_status=("STATUS_GRP_SEP", lambda s: s.mode().iat[0])).reset_index()
cl_prof = cl_tab[["CLUSTER", "n", "defaults", "rate", "ci_lo", "ci_hi", "share_clients", "lift"]].merge(prof, on="CLUSTER")
export(cl_prof, "s09_embedding_clusters")
display(cl_prof.round(3))

**Observations:**
- UMAP squeezes 24 behaviour columns into a 2-D map that keeps similar clients close; HDBSCAN finds dense groups on it. It found 16 clusters plus 17% noise, with default rates from 7.7% (C03, inactive, median limit NT$250k) to 67.3% (C00, Late 2m).
- The four riskiest clusters (C00, C08, C14, C13) all show late status, median utilisation 0.69-0.76, repayment of about 4-5% of bills and limits of NT$50-80k. Together they hold 4,305 clients (14.3%) and about 36% of defaulters (number from the pre-fix run; refresh after the next Kaggle run).
- The largest cluster, C11 (42.3% of clients), is ordinary revolvers at 11.1%.
- Story use: a "risk map" scatter in Tableau showing that risk lives in one region.

### 9.6 Anomaly lens: Isolation Forest
Isolation Forest scores how easy each client is to isolate in behavioural space (same standardised inputs as 9.5, all 30,000 clients, no target used). Question: are the *unusual* clients the risky ones, or is default a mainstream behaviour? (Isolation Forest is CPU-only in scikit-learn; it is fast enough.)

In [ ]:
with s9_timer("9.6 Isolation Forest", "cpu"):
    iso = IsolationForest(n_estimators=SETTINGS["iforest_trees"], random_state=42, n_jobs=-1).fit(Z9)
    anom = -iso.score_samples(Z9)                      # higher = more anomalous
an = df[["ID", TARGET, "N_LATE", "AVG_UTIL", "LIMIT_BAL", "TOTAL_PAY", "TOTAL_BILL", "N_ZERO_PAY", "NEG_BILL_ANY"]].copy()
an["anomaly_score"] = anom
an["anomaly_decile"] = pd.qcut(an["anomaly_score"].rank(method="first"), 10, labels=[f"D{i}" for i in range(1, 11)])
dec = rate_table(an, "anomaly_decile")
dec["mean_score"] = an.groupby("anomaly_decile", observed=True)["anomaly_score"].mean().values
export(dec, "s09_anomaly_deciles")
fig, ax = plt.subplots(figsize=(11, 4.5))
plot_rate(an, "anomaly_decile", ax=ax, title="Default rate by Isolation-Forest anomaly decile (D10 = most unusual)")
savefig("s09_anomaly_deciles"); plt.show()
display(dec[["anomaly_decile", "n", "rate", "ci_lo", "ci_hi", "lift", "mean_score"]].round(4))
rho = stats.spearmanr(anom, oof)[0]
print(f"Spearman(anomaly score, XGB lens out-of-fold risk) = {rho:.3f} -> "
      + ("anomaly is largely a separate axis from modelled risk" if abs(rho) < 0.3 else "anomaly partly overlaps modelled risk" if abs(rho) < 0.6 else "anomaly mostly re-discovers modelled risk"))
top1 = an["anomaly_score"] >= an["anomaly_score"].quantile(0.98)
cmp = pd.DataFrame({"top 2% anomalies (median)": an[top1].median(numeric_only=True),
                    "rest (median)": an[~top1].median(numeric_only=True)}).drop(["ID", "anomaly_score", TARGET, "NEG_BILL_ANY"])
cmp.loc["default rate"] = [an.loc[top1, TARGET].mean(), an.loc[~top1, TARGET].mean()]
cmp.loc["share NEG_BILL_ANY"] = [an.loc[top1, "NEG_BILL_ANY"].mean(), an.loc[~top1, "NEG_BILL_ANY"].mean()]
display(cmp.round(3))

**Observations:**
- Isolation Forest flags clients whose behaviour is unusual. Default rises steadily with unusualness: 7.6% in the most typical decile vs 39.9% in the most unusual.
- The top 2% of anomalies have a median of 5 late months, 4 zero-payment months and a NT$30k limit, and default at 51.2% vs 21.5% for the rest.
- The anomaly score correlates only 0.37 with model risk, so it is a partly separate signal.
- Story use: a side note that "unusual is risky"; background only.

### 9.7 Optional: RAPIDS cuDF for groupby-heavy work
Every Tableau-style rate table in this notebook is a groupby. On a GPU runtime with RAPIDS, `cudf` (or the zero-code-change `%load_ext cudf.pandas`, which must run **before** `import pandas`) can execute those groupbys on the GPU. The notebook does **not** depend on it; the cell below only benchmarks it if available.

In [ ]:
keys = ["LIMIT_BAND", "AGE_BAND", "STATUS_GRP_SEP", "EDU_LABEL", "SEX_LABEL"]
bench = df[keys + [TARGET, "AVG_UTIL"]].copy()
for k in keys:
    bench[k] = bench[k].astype(str)
bench = pd.concat([bench] * 20, ignore_index=True)       # 600k rows so timing is meaningful
t0 = time.time(); g_pd = bench.groupby(keys)[[TARGET, "AVG_UTIL"]].mean(); t_pd = time.time() - t0
print(f"pandas groupby on {len(bench):,} rows: {t_pd*1000:.0f} ms -> {len(g_pd):,} groups")
try:
    if not GPU:
        raise RuntimeError("no GPU")
    import cudf
    gb = cudf.from_pandas(bench)
    gb.groupby(keys)[[TARGET, "AVG_UTIL"]].mean()        # warm-up
    t0 = time.time(); g_cu = gb.groupby(keys)[[TARGET, "AVG_UTIL"]].mean(); t_cu = time.time() - t0
    print(f"cuDF groupby: {t_cu*1000:.0f} ms ({t_pd / max(t_cu, 1e-6):.1f}x vs pandas)")
except Exception as e:
    print(f"cuDF not used ({str(e)[:80]}). On a RAPIDS GPU image: put '%load_ext cudf.pandas' in the first cell to accelerate pandas transparently.")

### 9.8 Discovery digest and timings
Compact cross-lens summary (for writing insights) and the wall-clock time / device of each block.

In [ ]:
print("Devices -> XGBoost:", XGB_DEVICE, "| arrays:", ARRAY_DEVICE, "| embedding:", EMB_BACKEND, "| clustering:", CL_BACKEND)
print("\nTop 10 SHAP features:", ", ".join(imp["feature"].head(10)))
print("\nTop 5 SHAP interaction pairs:")
for _, r in pairs.head(5).iterrows():
    print(f"  {r['feature_a']} x {r['feature_b']}: {r['mean_abs_interaction']:.4f}")
print("\nTop 5 high-risk pockets (WRAcc):")
for _, r in risk.nlargest(5, "wracc").iterrows():
    print(f"  {r['rule']}  | n={int(r['n']):,} rate={r['rate']:.1%} lift={r['lift']:.2f}")
print("\nTop 5 hidden pockets (no repayment-status variable):")
for _, r in hidden.nlargest(5, "wracc").iterrows():
    print(f"  {r['rule']}  | n={int(r['n']):,} rate={r['rate']:.1%} lift={r['lift']:.2f}")
print(f"\nEmbedding clusters: {len(cl_prof)} (incl. noise); default-rate range {cl_prof['rate'].min():.1%} - {cl_prof['rate'].max():.1%}")
print(f"Anomaly D10 vs D1 default rate: {dec['rate'].iloc[-1]:.1%} vs {dec['rate'].iloc[0]:.1%}")
tim = pd.DataFrame(S9_TIMES)
display(tim)
print(f"Total timed compute in Section 9: {tim['seconds'].sum():.0f}s")

**Observations:**
- All lenses agree: lateness (`PAY_1`, `MAX_DELAY`, `N_LATE`) leads in SHAP, the tree rules, the subgroups and the embedding.
- Without status variables, the best warning signs are zero-payment months, limits of NT$50k or less, and repaying 0-5% of the bill (up to 43.8% default) (number from the pre-fix run; refresh after the next Kaggle run).
- Embedding clusters span 7.7%-67.3% default; the anomaly deciles span 7.6%-39.9%.
- All timed compute took 35 s on GPU, and cuDF ran groupbys 27x faster than pandas.
- Story use: the summary slide for the discovery section.

## 10. Statistical robustness checks
Sections 3–8 describe patterns; this section asks which of them survive formal scrutiny:
1. are the categorical associations real after multiple-testing correction,
2. how precise are the headline rate gaps (bootstrap) and could they be chance (permutation),
3. do demographic effects **survive once credit line and repayment behaviour are controlled** (logit for inference, Cochran-Mantel-Haenszel, Simpson scan),
4. what effect size this sample can even detect (power / MDE).

The logistic regressions here are **inferential only** (odds ratios + CIs); no accuracy metrics are reported.
New `df` columns: **none** (works on a local copy `d10`).

**Section guide**

- **What it does:** Tests whether the headline gaps are real and not chance. It corrects for running many tests, puts confidence ranges on gaps, and checks whether demographic gaps survive once limit and repayment status are held fixed.
- **Question it answers:** Which findings are solid, and are demographic effects really just behaviour in disguise?
- **Inputs / new columns:** categorical bands, `PAY_i`, `LIMIT_BAL`, `STATUS_GRP_SEP`, sex, education, marital status, age band. No new `df` columns (works on a copy `d10`).
- **How to read the charts:** forest plot: dot = gap between two groups, whisker = 95% CI, vertical line at 0 (or 1 for ratios) = no difference. Odds-ratio charts: right of 1 = higher odds of default.
- **Terms:** bootstrap = re-sample the data many times to see how much a number varies. Odds ratio (OR) = how many times the odds of default are multiplied (1 = no effect). Confounding = a gap that is really caused by a third variable. Simpson's paradox = a gap that reverses inside subgroups. MDE = smallest gap the data can reliably detect.
- **Key takeaways:**
  - All 30 categorical variables stay significant, but effect sizes differ ten-fold (`PAY_1` V = 0.42 vs sex 0.04).
  - September 2+ months late vs on time: +55.7 pp (risk ratio 5.0).
  - The young-age effect is mostly a limit effect (OR 1.59 → 1.06); the male effect survives (1.25 → 1.14).
  - No Simpson reversals in 72 checks.
- **Where it feeds:** the "behaviour vs demographics" forest plot and method credibility; captions should avoid rates for Tableau cells under about 300 clients.

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import StratifiedTable, Table2x2
from statsmodels.stats.power import NormalIndPower

d10 = df.copy()
for c in ["AGE_BAND", "LIMIT_BAND", "UTIL_BAND", "N_LATE_BAND", "PAYRATIO_BAND"]:
    d10[c] = d10[c].astype(str)
y10 = d10[TARGET].to_numpy()
print("section 10 ready")

### 10.1 Chi-square tests of independence, every categorical vs default, with Benjamini-Hochberg FDR
One test per categorical variable (labels, bands, every monthly `PAY_i` code and status group, count variables). P-values are adjusted with **BH-FDR** (Benjamini-Hochberg false discovery rate: keeps the share of false alarms low when many tests run at once; q = 0.05). Because many p-values underflow to 0 at n = 30,000, the table also shows −log10(p) computed from the log survival function, and Cramér's V as the effect size. `pct_cells_exp<5` flags where the chi-square approximation is shaky (sparse late-status codes).

In [ ]:
CHI_VARS = (["SEX_LABEL", "EDU_LABEL", "EDUCATION", "MAR_LABEL", "MARRIAGE", "AGE_BAND", "LIMIT_BAND", "UTIL_BAND",
             "N_LATE_BAND", "PAYRATIO_BAND", "MAX_DELAY", "LATE_STREAK_RECENT", "N_INACTIVE", "N_PAIDFULL",
             "N_REVOLVE", "N_ZERO_PAY", "OVERLIMIT_ANY", "NEG_BILL_ANY"]
            + PAY_COLS + [f"STATUS_GRP_{i}" for i in IDX])
def neg_log10_chi2_p(chi2, dof):
    v = stats.chi2.logsf(chi2, dof)
    if not np.isfinite(v):   # underflow: leading term of the upper incomplete gamma asymptotic
        k = dof / 2
        v = (k - 1) * np.log(chi2 / 2) - chi2 / 2 - math.lgamma(k)
    return -v / np.log(10)
rows = []
for c in CHI_VARS:
    ct = pd.crosstab(d10[c], d10[TARGET])
    chi2, p, dof, exp = stats.chi2_contingency(ct, correction=False)
    rows.append({"variable": c, "levels": ct.shape[0], "dof": dof, "chi2": chi2, "p": p,
                 "neg_log10_p": neg_log10_chi2_p(chi2, dof),
                 "cramers_v": cramers_v(d10[c], d10[TARGET]), "pct_cells_exp<5": 100 * (exp < 5).mean()})
chi = pd.DataFrame(rows)
chi["p_adj_BH"] = multipletests(chi["p"], alpha=0.05, method="fdr_bh")[1]
chi["significant_BH"] = chi["p_adj_BH"] < 0.05
chi = chi.sort_values("cramers_v", ascending=False).reset_index(drop=True)
export(chi, "s10_chi_square_bh")
print(f"{chi['significant_BH'].sum()} of {len(chi)} variables significant after BH-FDR")
print(chi.to_string(formatters={"p": "{:.2e}".format, "p_adj_BH": "{:.2e}".format, "chi2": "{:,.1f}".format,
                                "neg_log10_p": "{:,.1f}".format, "cramers_v": "{:.3f}".format, "pct_cells_exp<5": "{:.1f}".format}))
fig, ax = plt.subplots(figsize=(11, 9))
cc = chi.iloc[::-1]
ax.barh(cc["variable"], cc["cramers_v"], color=np.where(cc["significant_BH"], C_DEF, C_NEU))
for yv, (v, nl) in enumerate(zip(cc["cramers_v"], cc["neg_log10_p"])):
    ax.text(v + .004, yv, f"-log10p={nl:,.0f}", va="center", fontsize=7)
ax.set_xlabel("Cramér's V (effect size)"); ax.set_title("10.1 Chi-square tests vs default: effect size (red = significant after BH-FDR)")
savefig("s10_chi_square_bh"); plt.show()

**Observations:**
- BH-FDR corrects p-values when many tests run at once. All 30 variables stay significant after it, even marital status (p = 7.8e-7).
- Effect sizes differ about ten-fold: `PAY_1` Cramér's V = 0.42 vs sex 0.04 and marital status 0.03.
- 14-27% of cells in the monthly `PAY_i` tables have expected counts below 5, so the grouped status variables are the safer test.
- Story use: the "significant is not the same as important" message.

### 10.2 Bootstrap confidence intervals for key rate differences (B = 2,000)
Bootstrap = re-sample the clients many times and recompute the gap, to see how much it could vary by chance. Vectorised: resampling *n* Bernoulli outcomes with replacement and taking the mean is distributionally identical to drawing Binomial(*n*, *p̂*)/*n*, so each group's 2,000 bootstrap rates come from a single `RNG.binomial` call. Percentile CIs for both the **difference** (pp) and the **risk ratio**. A brute-force index-resampling bootstrap on the first contrast confirms the shortcut.

In [ ]:
B_BOOT = 2000
CONTRASTS = [
    ("Female vs Male",               d10["SEX_LABEL"] == "Female",          d10["SEX_LABEL"] == "Male"),
    ("Single vs Married",            d10["MAR_LABEL"] == "Single",          d10["MAR_LABEL"] == "Married"),
    ("Graduate vs High school",      d10["EDU_LABEL"] == "Graduate school", d10["EDU_LABEL"] == "High school"),
    ("Age 21-25 vs 31-35",           d10["AGE_BAND"] == "21-25",            d10["AGE_BAND"] == "31-35"),
    ("Limit <=50k vs >500k",         d10["LIMIT_BAND"] == "<=50k",          d10["LIMIT_BAND"] == ">500k"),
    ("PAY_1 >=2 vs PAY_1 <=0",       d10["PAY_1"] >= 2,                     d10["PAY_1"] <= 0),
]
boot_rows, BOOT_DRAWS = [], {}
for lab, ma, mb in CONTRASTS:
    ya, yb = y10[ma.to_numpy()], y10[mb.to_numpy()]
    na, nb = len(ya), len(yb); pa, pb = ya.mean(), yb.mean()
    ra = RNG.binomial(na, pa, B_BOOT) / na
    rb = RNG.binomial(nb, pb, B_BOOT) / nb
    dd, rr = ra - rb, ra / rb
    BOOT_DRAWS[lab] = dd
    se = np.sqrt(pa * (1 - pa) / na + pb * (1 - pb) / nb)
    boot_rows.append({"contrast": lab, "n_A": na, "n_B": nb, "rate_A": pa, "rate_B": pb,
                      "diff_pp": 100 * (pa - pb), "boot_lo_pp": 100 * np.percentile(dd, 2.5), "boot_hi_pp": 100 * np.percentile(dd, 97.5),
                      "wald_lo_pp": 100 * (pa - pb - 1.96 * se), "wald_hi_pp": 100 * (pa - pb + 1.96 * se),
                      "risk_ratio": pa / pb, "rr_lo": np.percentile(rr, 2.5), "rr_hi": np.percentile(rr, 97.5)})
boot = pd.DataFrame(boot_rows)
export(boot, "s10_bootstrap_gaps")
print(boot.round(4).to_string())

# brute-force check on the first contrast (index resampling, chunked to bound memory)
lab, ma, mb = CONTRASTS[0]
ya, yb = y10[ma.to_numpy()].astype(np.int8), y10[mb.to_numpy()].astype(np.int8)
bf = []
for _ in range(B_BOOT // 250):
    ia = RNG.integers(0, len(ya), (250, len(ya))); ib = RNG.integers(0, len(yb), (250, len(yb)))
    bf.append(ya[ia].mean(1) - yb[ib].mean(1))
bf = np.concatenate(bf)
print(f"\ncheck [{lab}] index-resampling CI: [{100*np.percentile(bf, 2.5):.2f}, {100*np.percentile(bf, 97.5):.2f}] pp "
      f"vs binomial shortcut [{boot.loc[0, 'boot_lo_pp']:.2f}, {boot.loc[0, 'boot_hi_pp']:.2f}] pp")

Forest plot: left = rate difference (pp), right = risk ratio (log scale). Demographic contrasts sit within a few pp of zero; credit-line and especially repayment-status contrasts are an order of magnitude larger.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
yy = np.arange(len(boot))[::-1]
axes[0].errorbar(boot["diff_pp"], yy, xerr=[boot["diff_pp"] - boot["boot_lo_pp"], boot["boot_hi_pp"] - boot["diff_pp"]],
                 fmt="o", color=C_DEF, ecolor="k", capsize=4)
axes[0].axvline(0, color="k", lw=.8); axes[0].set_xlabel("rate difference A − B (pp), bootstrap 95% CI")
axes[0].set_yticks(yy); axes[0].set_yticklabels(boot["contrast"])
axes[1].errorbar(boot["risk_ratio"], yy, xerr=[boot["risk_ratio"] - boot["rr_lo"], boot["rr_hi"] - boot["risk_ratio"]],
                 fmt="s", color=C_ACC, ecolor="k", capsize=4)
axes[1].axvline(1, color="k", lw=.8); axes[1].set_xscale("log"); axes[1].set_xlabel("risk ratio A / B (log scale), bootstrap 95% CI")
for i_, r in boot.iterrows():
    axes[0].text(r["boot_hi_pp"] + 1, yy[i_], f"{r['diff_pp']:+.1f} pp", va="center", fontsize=8)
fig.suptitle("10.2 Bootstrap forest plot of key default-rate contrasts (B = 2,000)", fontsize=13, fontweight="bold")
plt.tight_layout(); savefig("s10_bootstrap_forest"); plt.show()

**Observations:**
- The repayment contrast dwarfs everything: `PAY_1` >= 2 vs <= 0 differs by +55.7 pp (95% CI 54.1-57.4), a risk ratio of 5.0.
- Credit line comes next: limit up to NT$50k vs above NT$500k differs by +20.6 pp (16.1-24.8), risk ratio 2.8.
- Demographic gaps are small: age 21-25 vs 31-35 +7.2 pp, graduate vs high school -5.9 pp, female vs male -3.4 pp, single vs married -2.5 pp.
- Bootstrap and Wald intervals agree to within about 0.1 pp.
- Story use: the forest plot for the "behaviour vs demographics" slide.

### 10.3 Permutation tests for the three largest demographic gaps
For each demographic variable (sex, education, marital status, age band) the highest- and lowest-rate levels with n ≥ 500 are compared; the three largest gaps are tested by shuffling the default labels between the two groups 5,000 times (vectorised in chunks). The p-value is (#|null| ≥ |observed| + 1)/(B + 1), so the smallest reportable value is ≈ 2e-4.

In [ ]:
cands = []
for c in ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND"]:
    t = rate_table(d10, c, min_n=500).sort_values("rate")
    cands.append({"variable": c, "A": t[c].iloc[-1], "B": t[c].iloc[0], "gap": t["rate"].iloc[-1] - t["rate"].iloc[0]})
cands = pd.DataFrame(cands).sort_values("gap", ascending=False).head(3).reset_index(drop=True)
N_PERM, CHUNK = 5000, 500
perm_rows = []
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for ax, (_, r) in zip(axes, cands.iterrows()):
    ya = y10[(d10[r["variable"]] == r["A"]).to_numpy()]; yb = y10[(d10[r["variable"]] == r["B"]).to_numpy()]
    pool = np.concatenate([ya, yb]).astype(np.int8); na = len(ya)
    obs = ya.mean() - yb.mean()
    null = []
    for _ in range(N_PERM // CHUNK):
        P_ = RNG.permuted(np.tile(pool, (CHUNK, 1)), axis=1)
        null.append(P_[:, :na].mean(1) - P_[:, na:].mean(1))
    null = np.concatenate(null)
    pval = ((np.abs(null) >= abs(obs)).sum() + 1) / (N_PERM + 1)
    perm_rows.append({"variable": r["variable"], "A (riskier)": r["A"], "B (safer)": r["B"], "n_A": na, "n_B": len(yb),
                      "obs_gap_pp": 100 * obs, "null_sd_pp": 100 * null.std(), "z_vs_null": obs / null.std(), "perm_p": pval})
    ax.hist(100 * null, bins=60, color=C_NEU, alpha=.8)
    ax.axvline(100 * obs, color=C_DEF, lw=2.5, label=f"observed {100*obs:+.1f} pp")
    ax.set_title(f"{r['variable']}: {r['A']} vs {r['B']}", fontsize=10); ax.set_xlabel("gap under H0 (pp)"); ax.legend(fontsize=8)
fig.suptitle(f"10.3 Permutation null distributions ({N_PERM:,} shuffles)", fontsize=13, fontweight="bold")
plt.tight_layout(); savefig("s10_permutation_tests"); plt.show()
perm = pd.DataFrame(perm_rows)
print(perm.round(4).to_string())

**Observations:**
- The three largest demographic gaps are age 21-25 vs 31-35 (7.2 pp), high school vs graduate school (5.9 pp) and male vs female (3.4 pp).
- None of the 5,000 label shuffles matched any of them (p = 0.0002, the floor), and the z-scores run 6.9-8.5.
- These gaps are real but modest next to the behavioural gaps.
- Story use: background only; supports the demographic panel.

### 10.4 Logistic regression for inference: how much of the demographic effect is really behaviour?
Three nested `statsmodels` Logit models, reported **only as odds ratios with 95% CIs** (OR = how many times the odds of default are multiplied; 1 = no effect) (no accuracy metrics):
* **M1** demographics only (sex, education, marital status, age band);
* **M2** M1 + log2(LIMIT_BAL) (OR per doubling of the credit line);
* **M3** M2 + September repayment-status group.

References: Female, University, Married, age 31–35, status Revolving. If a demographic OR moves towards 1 from M1 → M3, its raw gap was largely **confounded** by credit line and repayment behaviour.

In [ ]:
d10["LOG2_LIMIT"] = np.log2(d10["LIMIT_BAL"])
DEMO_F = ("C(SEX_LABEL, Treatment('Female')) + C(EDU_LABEL, Treatment('University')) + "
          "C(MAR_LABEL, Treatment('Married')) + C(AGE_BAND, Treatment('31-35'))")
FORMULAS = {"M1 demographics": f"{TARGET} ~ {DEMO_F}",
            "M2 + log2 limit": f"{TARGET} ~ {DEMO_F} + LOG2_LIMIT",
            "M3 + Sep status": f"{TARGET} ~ {DEMO_F} + LOG2_LIMIT + C(STATUS_GRP_SEP, Treatment('Revolving'))"}
def clean_term(t):
    import re
    m = re.match(r"C\((\w+), Treatment\('[^']*'\)\)\[T\.(.+)\]", t)
    return f"{m.group(1)}: {m.group(2)}" if m else t
or_rows, fit_rows = [], []
for nm, f in FORMULAS.items():
    res = smf.logit(f, data=d10).fit(disp=0)
    ci = res.conf_int()
    for t in res.params.index.drop("Intercept"):
        or_rows.append({"model": nm, "term": clean_term(t), "OR": np.exp(res.params[t]),
                        "OR_lo": np.exp(ci.loc[t, 0]), "OR_hi": np.exp(ci.loc[t, 1]), "p": res.pvalues[t]})
    fit_rows.append({"model": nm, "n": int(res.nobs), "df_model": int(res.df_model), "LL": res.llf,
                     "McFadden_pseudoR2": res.prsquared, "LLR_p": res.llr_pvalue, "AIC": res.aic})
ors = pd.DataFrame(or_rows)
export(ors, "s10_logit_odds_ratios")
print(pd.DataFrame(fit_rows).round(4).to_string(index=False)); print()
wide = ors.pivot(index="term", columns="model", values="OR")
wide["shrink_M1_to_M3_%"] = 100 * (1 - (np.log(wide["M3 + Sep status"]) / np.log(wide["M1 demographics"])))
order = ors.drop_duplicates("term")["term"].tolist()
print("Odds ratios by model (shrink % = reduction of log-OR from M1 to M3; >100% = sign flipped):")
print(wide.reindex(order).round(3).to_string())

In [ ]:
demo_terms = [t for t in order if t.split(":")[0] in ("SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND")]
other_terms = [t for t in order if t not in demo_terms]
fig, axes = plt.subplots(1, 2, figsize=(17, 7.5), gridspec_kw={"width_ratios": [1.3, 1]})
cols_m = dict(zip(FORMULAS, [C_NEU, "#fdae6b", C_DEF]))
for ax, terms, ttl in ((axes[0], demo_terms, "Demographic odds ratios across models"),
                       (axes[1], other_terms, "Credit line and Sep status (M2 / M3)")):
    base = np.arange(len(terms))[::-1]
    for k, (m, col_) in enumerate(cols_m.items()):
        s = ors[ors["model"] == m].set_index("term").reindex(terms)
        yy = base + (k - 1) * 0.25
        ok = s["OR"].notna().to_numpy()
        ax.errorbar(s["OR"][ok], yy[ok], xerr=[(s["OR"] - s["OR_lo"])[ok], (s["OR_hi"] - s["OR"])[ok]],
                    fmt="o", color=col_, ecolor=col_, capsize=2, ms=5, label=m)
    ax.axvline(1, color="k", lw=.8); ax.set_xscale("log")
    ax.set_yticks(base); ax.set_yticklabels(terms, fontsize=8); ax.set_xlabel("odds ratio (log scale, 95% CI)")
    ax.set_title(ttl, fontsize=11); ax.legend(fontsize=8, loc="lower right")
fig.suptitle("10.4 Odds-ratio forest plot: demographic effects shrink once behaviour is controlled", fontsize=13, fontweight="bold")
plt.tight_layout(); savefig("s10_logit_or_forest"); plt.show()

**Observations:**
- McFadden pseudo-R² is the share of uncertainty a model explains. Demographics alone reach only 0.011, adding the credit limit gives 0.034, and adding September status gives 0.159.
- The young-age effect is mostly a limit effect: ages 21-25 have an odds ratio of 1.59 alone but 1.06 once the limit is added (87% shrink). Graduate school moves from 0.80 to 0.98 (91% shrink).
- The male effect survives: OR 1.25 falls only to 1.14 (39% shrink).
- Being 2 months late in September multiplies the odds by 14.4.
- Story use: key evidence that age and education gaps mostly reflect credit limits.

### 10.5 Cochran-Mantel-Haenszel: SEX vs default, stratified
2×2 tables (Male/Female × Default/No default) inside each stratum are pooled into a Mantel-Haenszel common odds ratio. Compare the **crude OR** with the **MH-adjusted OR**: a big move means the stratifier confounds the sex effect. The **Breslow-Day** test checks whether the OR is homogeneous across strata (if rejected, the sex effect differs by stratum = effect modification). Repeated for Single vs Married.

In [ ]:
def cmh(col, a, b, strat):
    sub = d10[d10[col].isin([a, b])]
    tabs, used = [], []
    for lev, g in sub.groupby(strat, observed=True):
        t = np.array([[((g[col] == a) & (g[TARGET] == 1)).sum(), ((g[col] == a) & (g[TARGET] == 0)).sum()],
                      [((g[col] == b) & (g[TARGET] == 1)).sum(), ((g[col] == b) & (g[TARGET] == 0)).sum()]], float)
        if (t.sum(1) >= 20).all() and (t > 0).all():
            tabs.append(t); used.append(str(lev))
    st = StratifiedTable(tabs)
    crude = Table2x2(np.array([[((sub[col] == a) & (sub[TARGET] == 1)).sum(), ((sub[col] == a) & (sub[TARGET] == 0)).sum()],
                               [((sub[col] == b) & (sub[TARGET] == 1)).sum(), ((sub[col] == b) & (sub[TARGET] == 0)).sum()]], float))
    lo, hi = st.oddsratio_pooled_confint()
    return {"contrast": f"{a} vs {b}", "stratifier": strat, "strata_used": len(tabs),
            "crude_OR": crude.oddsratio, "MH_OR": st.oddsratio_pooled, "MH_lo": lo, "MH_hi": hi,
            "CMH_p": st.test_null_odds(correction=True).pvalue, "BreslowDay_p": st.test_equal_odds().pvalue,
            "OR_change_%": 100 * (st.oddsratio_pooled / crude.oddsratio - 1)}
CMH_STRATA = ["STATUS_GRP_SEP", "LIMIT_BAND", "AGE_BAND", "EDU_LABEL", "MAR_LABEL", "N_LATE_BAND", "UTIL_BAND"]
cmh_rows = [cmh("SEX_LABEL", "Male", "Female", s) for s in CMH_STRATA if s != "SEX_LABEL"]
cmh_rows += [cmh("MAR_LABEL", "Single", "Married", s) for s in ["STATUS_GRP_SEP", "LIMIT_BAND", "AGE_BAND", "SEX_LABEL", "EDU_LABEL"]]
cmh_tab = pd.DataFrame(cmh_rows)
export(cmh_tab, "s10_cmh")
print(cmh_tab.to_string(formatters={"CMH_p": "{:.2e}".format, "BreslowDay_p": "{:.3g}".format,
                                    **{k: "{:.3f}".format for k in ["crude_OR", "MH_OR", "MH_lo", "MH_hi"]}, "OR_change_%": "{:+.1f}".format}))
fig, ax = plt.subplots(figsize=(10, 6))
yy = np.arange(len(cmh_tab))[::-1]
ax.errorbar(cmh_tab["MH_OR"], yy, xerr=[cmh_tab["MH_OR"] - cmh_tab["MH_lo"], cmh_tab["MH_hi"] - cmh_tab["MH_OR"]],
            fmt="o", color=C_DEF, capsize=3, label="MH-adjusted OR (95% CI)")
ax.scatter(cmh_tab["crude_OR"], yy, marker="|", s=250, color="k", label="crude OR")
ax.axvline(1, color="k", lw=.8)
ax.set_yticks(yy); ax.set_yticklabels(cmh_tab["contrast"] + "  | by " + cmh_tab["stratifier"], fontsize=8)
ax.set_xlabel("odds ratio"); ax.legend(fontsize=8); ax.set_title("10.5 Crude vs Mantel-Haenszel adjusted odds ratios")
savefig("s10_cmh"); plt.show()

**Observations:**
- CMH (Cochran-Mantel-Haenszel) compares groups inside strata to remove a confounder. The male vs female crude OR of 1.215 stays between 1.15 and 1.22 under all seven stratifiers; the largest drop is 5.7% (late-month band).
- Single vs married gets stronger once limit band is controlled: OR 0.863 becomes 0.792.
- Breslow-Day p is below 0.01 for sex within September status (0.009) and limit band (0.003), so the male gap varies in size across those groups.
- Story use: background only; the sex gap is robust but small.

### 10.6 Simpson's paradox scan
For every demographic contrast A (e.g. Male − Female) and every stratifier B, the crude rate gap is compared with the **stratum-adjusted gap** (Mantel-Haenszel-weighted average of within-stratum gaps, weights nₐn_b/(nₐ+n_b); strata need ≥ 30 per arm). Flags:
* **REVERSAL** — adjusted gap has the opposite sign of the crude gap (both ≥ 0.5 pp in size): Simpson's paradox;
* **attenuated** — same sign but < 50% of the crude gap (B explains most of A's effect);
* **amplified** — > 150% of the crude gap; otherwise **stable**;
* when the crude gap is < 0.5 pp the ratio is undefined: **masked** if the adjusted gap is ≥ 1 pp (a gap hidden by composition), else **no gap either way**.
`reversed_weight_%` = share of the stratum weight in which the within-stratum gap points the other way.

In [ ]:
SIMP_CONTRASTS = [("SEX_LABEL", "Male", "Female"), ("MAR_LABEL", "Single", "Married"), ("MAR_LABEL", "Other", "Married"),
                  ("EDU_LABEL", "Graduate school", "University"), ("EDU_LABEL", "University", "High school"),
                  ("EDU_LABEL", "Graduate school", "High school"), ("EDU_LABEL", "Other/Unknown", "University"),
                  ("AGE_BAND", "21-25", "31-35"), ("AGE_BAND", "51-60", "31-35")]
SIMP_STRATA = ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND", "STATUS_GRP_SEP", "UTIL_BAND",
               "N_LATE_BAND", "PAYRATIO_BAND"]
sim_rows = []
for col, a, b in SIMP_CONTRASTS:
    sub = d10[d10[col].isin([a, b])]
    crude = sub.loc[sub[col] == a, TARGET].mean() - sub.loc[sub[col] == b, TARGET].mean()
    for st in SIMP_STRATA:
        if st == col: continue
        g = sub.groupby([st, col], observed=True)[TARGET].agg(["size", "mean"]).unstack(col)
        g = g.dropna()
        if g.empty: continue
        na, nb = g[("size", a)], g[("size", b)]
        g = g[(na >= 30) & (nb >= 30)]
        if len(g) < 2: continue
        na, nb = g[("size", a)], g[("size", b)]
        dlt = g[("mean", a)] - g[("mean", b)]
        w = na * nb / (na + nb)
        adj = float((w * dlt).sum() / w.sum())
        rev_w = float(w[np.sign(dlt) != np.sign(crude)].sum() / w.sum())
        if abs(crude) < .005:
            flag = "masked (crude~0, adjusted>=1pp)" if abs(adj) >= .01 else "no gap either way"
        elif abs(adj) >= .005 and np.sign(adj) != np.sign(crude): flag = "REVERSAL"
        elif abs(adj) < 0.5 * abs(crude): flag = "attenuated"
        elif abs(adj) > 1.5 * abs(crude): flag = "amplified"
        else: flag = "stable"
        sim_rows.append({"contrast": f"{a} - {b}", "stratifier": st, "strata": len(g), "crude_pp": 100 * crude,
                         "adjusted_pp": 100 * adj, "adj/crude": adj / crude if abs(crude) >= .005 else np.nan,
                         "reversed_weight_%": 100 * rev_w, "flag": flag})
simp = pd.DataFrame(sim_rows)
export(simp, "s10_simpson_scan")
print(simp["flag"].value_counts().to_string()); print()
print(simp.round(3).to_string())
hm = simp.pivot(index="contrast", columns="stratifier", values="adj/crude").reindex(columns=SIMP_STRATA)
fl = simp.pivot(index="contrast", columns="stratifier", values="flag").reindex(columns=SIMP_STRATA)
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(hm.clip(-1.5, 2.5), cmap=CMAP_DIV, center=1, annot=hm.round(2), fmt="", ax=ax, cbar_kws={"label": "adjusted gap / crude gap"})
for (i_, j_), v in np.ndenumerate(fl.to_numpy()):
    if v == "REVERSAL":
        ax.add_patch(plt.Rectangle((j_, i_), 1, 1, fill=False, ec="k", lw=3))
ax.set_title("10.6 Simpson scan: ratio of stratum-adjusted to crude gap (1 = unchanged, <0 = REVERSAL, boxed)")
savefig("s10_simpson_scan"); plt.show()

**Observations:**
- Simpson's paradox is a gap that reverses inside subgroups. Among 72 contrast x stratifier checks there are no reversals: 54 stable, 9 attenuated, 1 amplified, 1 masked, 7 no gap.
- The male - female gap shrinks from 3.4 pp to 2.0 pp within late-month bands but keeps its sign.
- The single - married gap grows from -2.5 pp to -3.9 pp within limit bands (amplified).
- "Other" marital status vs married looks equal overall (0.1 pp) but is -4.4 pp within limit bands (masked).
- Story use: background only; adds method credibility.

### 10.7 Power and minimum detectable effect (MDE)
With n = 30,000, statistical significance is cheap: the question is whether a gap is *large enough to matter*. For each contrast in 10.2, the MDE is the smallest true rate difference detectable with 80% power at α = 0.05 (two-sided; Cohen's h, an effect-size scale for two proportions, converted back to percentage points around group B's observed rate). The curve shows MDE vs per-group n at the portfolio base rate. Rule of thumb for this notebook: with ~70 tests in Section 3 alone, treat gaps below ≈ 2× MDE as fragile even when p < 0.05.

In [ ]:
pw = NormalIndPower()
def mde_pp(n_a, n_b, p0, alpha=0.05, power=0.8):
    h = pw.solve_power(effect_size=None, nobs1=n_a, ratio=n_b / n_a, alpha=alpha, power=power, alternative="two-sided")
    p1 = np.sin(np.arcsin(np.sqrt(p0)) + h / 2) ** 2
    return 100 * (p1 - p0)
mde = boot[["contrast", "n_A", "n_B", "rate_B", "diff_pp"]].copy()
mde["MDE_pp"] = [mde_pp(r.n_A, r.n_B, r.rate_B) for r in mde.itertuples()]
mde["observed/MDE"] = mde["diff_pp"].abs() / mde["MDE_pp"]
mde["power_for_observed"] = [pw.power(effect_size=abs(2 * np.arcsin(np.sqrt(r.rate_B + r.diff_pp / 100)) - 2 * np.arcsin(np.sqrt(r.rate_B))),
                                      nobs1=r.n_A, ratio=r.n_B / r.n_A, alpha=0.05) for r in mde.itertuples()]
print(mde.round(3).to_string(index=False))
ns = np.unique(np.geomspace(50, 15000, 60).astype(int))
curve = [mde_pp(n, n, BASE_RATE) for n in ns]
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(ns, curve, color=C_DEF, lw=2)
for r in mde.itertuples():
    ax.scatter(2 / (1 / r.n_A + 1 / r.n_B), r.MDE_pp, color="k", s=25)
    ax.annotate(r.contrast, (2 / (1 / r.n_A + 1 / r.n_B), r.MDE_pp), fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set_xscale("log"); ax.set_xlabel("clients per group (harmonic mean for unequal groups)")
ax.set_ylabel("MDE (pp)"); ax.set_title(f"10.7 Minimum detectable default-rate difference (80% power, α=0.05, base {BASE_RATE:.1%})")
savefig("s10_power_mde"); plt.show()
print("\nMDE at equal group sizes:", {int(n): round(float(mde_pp(n, n, BASE_RATE)), 2) for n in (100, 300, 1000, 3000, 10000, 15000)})

**Observations:**
- MDE (minimum detectable effect) is the smallest gap the data can reliably detect. All six observed gaps exceed their MDE: sex 3.4 vs 1.4 pp, and September status 55.7 vs 1.9 pp (29x).
- The over-NT$500k limit group has only 206 clients, so its MDE is 6.9 pp.
- Equal groups of 1,000 detect 5.4 pp, of 300 detect 10.1 pp, and of 15,000 detect 1.4 pp. With this much data, significance is cheap; judge gaps by their size.
- Story use: caption guidance; avoid showing rates for Tableau cells under about 300 clients.

## 11. Snapshot or panel? Data-structure audit and deep time-series / panel analysis

**Why this section matters for the story.** Before any chart puts "time" on an axis we must know what kind of time this
file contains. The answer decides which Tableau views are legitimate: a *snapshot* supports comparisons between clients,
a *panel* supports stories about change within a client. This section proves that the file is a **hybrid** - a
cross-sectional snapshot (one row per client, one outcome in Oct 2005, static limit and demographics) that **embeds a
short balanced panel** (6 monthly periods Apr-Sep 2005 x 3 series: status, bill, payment) - and then exploits the panel
far beyond Section 5 (variance decomposition, calendar effects, rhythm and spectra, lead-lag, event studies, person-month hazards).

Convention used throughout this section: arrays are **chronological** (column 0 = Apr ... column 5 = Sep), the reverse of the raw index.

**Section guide**

- **What it does:** Checks what kind of "time" this file holds. It proves the six months belong to the same account, measures how much clients change month to month, and then studies order of events: what moves first, what happens around a first late month, and how the chance of turning late or curing changes by month.
- **Question it answers:** Is this a one-off snapshot or a true monthly panel, and which time-based charts are honest to show in Tableau?
- **Inputs / new columns:** `PAY_1-6`, `BILL_AMT1-6`, `PAY_AMT1-6`, `UTIL_i`, `PAY_RATIO_i`, `LIMIT_BAL`, `AGE`. Adds `PN_N_PAY_MONTHS`, `PN_PAY_RUNS`, `PN_RHYTHM`, `PN_BILL_AC1`, `PN_PAY_AC1`, `PN_DOM_PERIOD`, `PN_ACTIVITY_PATTERN`, `PN_FIRST_ACTIVE_MONTH` (prefix `PN_`).
- **How to read the charts:** arrays run Apr → Sep (left to right). Lines = monthly averages or rates; event-study curves are centred on month 0 = first late month; red = defaulters, blue = non-defaulters; dashed line = 22.1% base rate.
- **Terms:** panel = the same clients observed each month. ICC = share of variation that is stable differences between clients. Hazard = chance of an event this month among clients still at risk. Fixed effects = compare each client only with themselves.
- **Key takeaways:** the file is a *snapshot with a memory*: one outcome per client plus a six-month ledger. It can show "what came before", not "when they defaulted" or seasonality.
  - The months are one linked ledger: 16.2% of payments equal the previous month's bill exactly, vs 0.03% against a random client's bill.
  - Bills and utilisation are client traits (ICC 0.88 and 0.82); payments and spend change month to month (ICC 0.19 and 0.09).
  - Steady payers (51.5% of clients) default at 13.9%; a broken payment rhythm pushes default to 34-52%.
  - Delinquency hardens: the monthly chance of curing falls from 36% after one late month to 7% after four or more.
  - Same September status, different past: revolvers with a clean Apr-Aug record default at 10.7%, with 3-5 earlier late months at 37.7%.
- **Where it feeds:** decides the legitimate time axes for Tableau (`s11_recommended_time_axes.csv`, `s11_structure_verdict.csv`); event-study and hazard exports feed the "path to default" story.

### 11.1 What is the grain of this file, and which columns move through time?
One row per client or one row per client-month? Which columns repeat by month and which exist only once?

In [ ]:
s11_t0 = time.time()
s11_rng = np.random.default_rng(11)
s11_CH = IDX[::-1]                                   # raw index in chronological order: 6 (Apr) ... 1 (Sep)
s11_B = df[[f"BILL_AMT{i}" for i in s11_CH]].to_numpy(float)     # bills, Apr..Sep
s11_P = df[[f"PAY_AMT{i}" for i in s11_CH]].to_numpy(float)      # payments, Apr..Sep
s11_S = df[[f"PAY_{i}" for i in s11_CH]].to_numpy(int)           # status codes, Apr..Sep
s11_L = df["LIMIT_BAL"].to_numpy(float)
s11_Y = df[TARGET].to_numpy(int)
s11_ID = df["ID"].to_numpy()
s11_N = len(df)
s11_U = s11_B / s11_L[:, None]                       # utilisation (uses the single, static limit)
s11_R = np.full((s11_N, 6), np.nan)                  # pay ratio: payment in m / bill of m-1 (May..Sep only)
s11_SP = np.full((s11_N, 6), np.nan)                 # implied spend in m (May..Sep only)
for _m in range(1, 6):
    s11_R[:, _m] = df[f"PAY_RATIO_{6 - _m}"].to_numpy(float)
    s11_SP[:, _m] = df[f"SPEND_{6 - _m}"].to_numpy(float)
s11_SPL = s11_SP / s11_L[:, None]                    # spend as share of limit
s11_LATE = s11_S >= 1
s11_KEY = {}                                         # headline numbers reused in the verdict table (11.12)

s11_time_cols = PAY_COLS + BILL_COLS + PAMT_COLS
s11_grain = pd.DataFrame([
    ("rows", len(df)),
    ("distinct client IDs", df["ID"].nunique()),
    ("duplicate IDs", int(df["ID"].duplicated().sum())),
    ("rows per client (max)", int(df["ID"].value_counts().max())),
    ("time-indexed columns (3 series x 6 months)", len(s11_time_cols)),
    ("static columns (limit + 4 demographics)", 5),
    ("outcome columns", 1),
    ("missing cells in time-indexed columns", int(df[s11_time_cols].isna().sum().sum())),
    ("explicit date / timestamp columns", 0),
], columns=["check", "value"])
display(s11_grain)
s11_KEY["one_row_per_id"] = bool(df["ID"].nunique() == len(df))
print("Grain = one row per client:", s11_KEY["one_row_per_id"])

**Observations:**
- The file has 30,000 rows and 30,000 distinct client IDs with no duplicates, so each row is one client.
- 18 of the 24 data columns are three monthly series (status, bill, payment) over six months, with zero missing cells. There is no date column at all.
- Story use: background only. It sets the rule "count clients, not rows" for every Tableau sheet.

Which column families actually vary inside a client? A family is *time-varying* if clients show more than one distinct
value across the six months. Static columns have a single value by construction, so change is unobservable, not absent.

In [ ]:
def s11_nvar(M):
    Ms = np.sort(M, axis=1)
    nd = 1 + (np.diff(Ms, axis=1) != 0).sum(1)
    return float((nd > 1).mean()), float(nd.mean())

s11_roles = []
for _name, _cols, _M in [("Repayment status", "PAY_1..PAY_6", s11_S), ("Bill statement", "BILL_AMT1..6", s11_B),
                         ("Payment amount", "PAY_AMT1..6", s11_P)]:
    _sh, _nd = s11_nvar(_M)
    s11_roles.append((_name, _cols, 6, "time-varying (monthly, Apr-Sep 2005)", _sh, _nd))
for _c in ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"]:
    s11_roles.append((_c, _c, 1, "static / point-in-time (date of measurement unknown)", np.nan, 1.0))
s11_roles.append(("DEFAULT", "default.payment.next.month", 1, "outcome, one month only (Oct 2005)", np.nan, 1.0))
s11_roles = pd.DataFrame(s11_roles, columns=["family", "columns", "n_columns", "role", "share_clients_with_change", "mean_distinct_values"])
display(s11_roles)
s11_KEY["status_changes"] = s11_roles.loc[0, "share_clients_with_change"]
s11_KEY["bill_changes"] = s11_roles.loc[1, "share_clients_with_change"]
add_fact("11.1", "data structure", f"One row per client ({len(df):,} IDs, no duplicates) but 18 of 24 columns are 3 series x 6 months; "
         f"{s11_KEY['bill_changes']:.0%} of clients change bill and {s11_KEY['status_changes']:.0%} change status inside the window", s11_KEY["bill_changes"], None, 2)

**Observations:**
- Bills change inside 95.9% of clients and payments inside 94.5%, with about 5.4 and 4.7 different values over six months.
- Repayment status changes for only 51.8% of clients (1.7 different codes on average). Status is sticky; money flows are not.
- Limit, sex, education, marriage and age appear once per client, so any change in them is invisible. The outcome covers one month only (Oct 2005).
- Story use: background. It tells the reader which fields can go on a time axis (the 18 monthly ones) and which cannot.

**Wide to long.** The file is stored *wide* (one row, 18 time columns). The embedded panel becomes visible when it is
melted to *long* (one row per client-month). `s11_long` is that table (180,000 rows) and is the base for everything below.
A panel is **balanced** when every client has every period.

In [ ]:
s11_long = pd.DataFrame({
    "ID": np.repeat(s11_ID, 6),
    "m": np.tile(np.arange(6), s11_N),
    "MONTH": np.tile(np.array(MONTH_ORDER), s11_N),
    "STATUS": s11_S.ravel(), "BILL": s11_B.ravel(), "PAY_AMT": s11_P.ravel(),
    "UTIL": s11_U.ravel(), "PAY_RATIO": s11_R.ravel(), "SPEND": s11_SP.ravel(),
    "LATE": s11_LATE.ravel().astype(int),
    "DEFAULT": np.repeat(s11_Y, 6),
})
print("WIDE (as delivered), two clients:")
display(df.loc[:1, ["ID", "LIMIT_BAL", "AGE"] + [f"PAY_{i}" for i in s11_CH] + [f"BILL_AMT{i}" for i in s11_CH[:3]] + [TARGET]])
print("LONG (panel view), the same two clients:")
display(s11_long[s11_long["ID"].isin(s11_ID[:2])])

s11_per_id = s11_long.groupby("ID").size()
s11_balance = s11_long.groupby("MONTH", observed=True).agg(clients=("ID", "nunique"), status_missing=("STATUS", lambda v: int(v.isna().sum())),
        bill_missing=("BILL", lambda v: int(v.isna().sum())), pay_missing=("PAY_AMT", lambda v: int(v.isna().sum())),
        pay_ratio_undefined=("PAY_RATIO", lambda v: int(v.isna().sum())), spend_undefined=("SPEND", lambda v: int(v.isna().sum()))).reindex(MONTH_ORDER)
display(s11_balance)
s11_KEY["balanced"] = bool((s11_per_id == 6).all() and (s11_balance["clients"] == s11_N).all())
print(f"Panel shape: {s11_N:,} clients x 6 months = {len(s11_long):,} client-months | balanced: {s11_KEY['balanced']}")
print("Derived series (pay ratio, spend) need the previous month, so they exist for 5 periods (May-Sep) only.")

**Observations:**
- Reshaping to long format (one row per client per month) gives 30,000 x 6 = 180,000 client-months. The panel is balanced: every client has all six months.
- Pay ratio and spend need the previous month's bill, so they exist only for May-Sep. The pay ratio is undefined (previous bill zero or negative) for 4,708 clients in May and 3,175 in Sep.
- Story use: background. The long table becomes the second Tableau data source, joined on ID.

**What is censored?** A six-month window cuts every client's history on both sides.
* *Left*: account open date and all history before April are unknown. A status of *k* in April means the delay began about *k* months earlier, so the codes leak a little pre-window history.
* *Right*: only one outcome month (Oct). A client late in September is an unfinished spell.
* *Interval*: no dates inside a month - only the month-end code and amounts.

In [ ]:
s11_apr = pd.Series(s11_S[:, 0])
s11_left = s11_apr[s11_apr >= 1].value_counts().sort_index().rename("clients").reset_index()
s11_left.columns = ["april_status", "clients"]
s11_left["share_of_clients"] = s11_left["clients"] / s11_N
s11_left["implied_first_missed_month"] = [(pd.Period("2005-04", "M") - int(k)).strftime("%b %Y") for k in s11_left["april_status"]]
print("Clients already late in April (delinquency spell started before the window = left-censored):")
display(s11_left)

s11_first_late = np.where(s11_LATE.any(1), s11_LATE.argmax(1), -1)          # chronological index of first late month, -1 = never
s11_cens = pd.DataFrame([
    ("late already in April (left-censored spell)", (s11_first_late == 0).mean()),
    ("first late month observed inside May-Sep (onset observed)", (s11_first_late >= 1).mean()),
    ("never late in the window (right-censored for 'first late')", (s11_first_late == -1).mean()),
    ("late in September (spell still open when the window closes)", s11_LATE[:, 5].mean()),
    ("late in every one of the six months (spell censored on both sides)", s11_LATE.all(1).mean()),
    ("earliest month implied by April codes", np.nan),
], columns=["item", "share_of_clients"])
display(s11_cens)
s11_KEY["left_cens"] = float((s11_first_late == 0).mean())
s11_KEY["sep_late"] = float(s11_LATE[:, 5].mean())
print("Earliest pre-window month implied by an April status code:", (pd.Period("2005-04", "M") - int(s11_apr.max())).strftime("%b %Y"))
add_fact("11.1", "data structure", f"{s11_KEY['left_cens']:.1%} of clients are already late in April (left-censored spells) and "
         f"{s11_KEY['sep_late']:.1%} are late in September (spell still open); the outcome is a single month", s11_KEY["left_cens"], None, 1)

**Observations:**
- 10.3% of clients are already late in April, so their delay began before the window. The oldest April code (8) points back to Aug 2004.
- Most of these are code 2 (2,766 clients, 9.2%), meaning the delay started around Feb 2005.
- 22.7% are late in September with the delay still running when the data stops, and 4.5% are late in all six months. 66.4% are never late in the window.
- Story use: caveat for onset and duration charts. Flag or drop the clients already late in April.

### 11.2 Are the six months really a linked sequence for the same account?
A file could hold six unrelated monthly columns. If the months are a true sequence, (a) adjacent bills must be more alike
than distant bills, (b) payments must line up with the *previous* statement and with no other month, and (c) destroying
the order must destroy these regularities.

First: correlation by distance in months. A linked series decays with lag; `within-client` removes each client's own mean.

In [ ]:
def s11_acf(X, lag=1):
    # lag-k autocorrelation of each row (client); NaN when the series is constant
    xc = X - X.mean(1, keepdims=True)
    den = (xc ** 2).sum(1)
    num = (xc[:, :-lag] * xc[:, lag:]).sum(1)
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(den > 0, num / np.where(den > 0, den, 1), np.nan)

s11_lagc = []
for _name, _M in [("bill", s11_B), ("payment", s11_P), ("status", s11_S.astype(float)), ("utilisation", s11_U)]:
    for _d in range(1, 6):
        _pear = np.mean([np.corrcoef(_M[:, a], _M[:, a + _d])[0, 1] for a in range(6 - _d)])
        _spear = np.mean([stats.spearmanr(_M[:, a], _M[:, a + _d])[0] for a in range(6 - _d)])
        s11_lagc.append((_name, _d, _pear, _spear, np.nanmean(s11_acf(_M, _d))))
s11_lagc = pd.DataFrame(s11_lagc, columns=["series", "lag_months", "pearson_between_clients", "spearman_between_clients", "mean_within_client_autocorr"])
display(s11_lagc.pivot(index="series", columns="lag_months", values="pearson_between_clients").round(3))
display(s11_lagc.pivot(index="series", columns="lag_months", values="mean_within_client_autocorr").round(3))
s11_KEY["bill_lag1"] = float(s11_lagc.query("series=='bill' and lag_months==1")["pearson_between_clients"].iloc[0])
s11_KEY["bill_lag5"] = float(s11_lagc.query("series=='bill' and lag_months==5")["pearson_between_clients"].iloc[0])

**Observations:**
- Bills hardly change from one month to the next: the correlation across clients is 0.94 one month apart and still 0.80 five months apart.
- Status also persists (0.77 at one month, 0.48 at five), but payments barely do (0.21 at one month).
- Inside each client's own six months (second table) the one-month correlation is small (bill 0.18) and turns negative at 2-4 months. With only six points this estimate leans toward about -0.17 even for pure noise, so read the sign, not the size.
- Story use: background. Bills and status describe who a client is; payments describe what they did that month.

**Alignment placebo.** If `PAY_AMT` in month *m* settles the statement of month *m-1*, exact matches (payment = bill to the
dollar) should pile up at offset -1 and almost nowhere else, and never against another client's bill.

In [ ]:
s11_align_rows = []
for _off in (-2, -1, 0, 1):
    _hit = _n = 0
    for _m in range(6):
        if 0 <= _m + _off <= 5:
            _b = s11_B[:, _m + _off]; _p = s11_P[:, _m]; _ok = _b > 0
            _hit += int((_p[_ok] == _b[_ok]).sum()); _n += int(_ok.sum())
    s11_align_rows.append((f"bill of month m{_off:+d}" if _off else "bill of the same month m", _n, _hit, _hit / _n))
_perm = s11_rng.permutation(s11_N); _hit = _n = 0
for _m in range(1, 6):
    _b = s11_B[_perm, _m - 1]; _p = s11_P[:, _m]; _ok = _b > 0
    _hit += int((_p[_ok] == _b[_ok]).sum()); _n += int(_ok.sum())
s11_align_rows.append(("bill of month m-1 of a RANDOM other client", _n, _hit, _hit / _n))
s11_align = pd.DataFrame(s11_align_rows, columns=["payment in month m compared with", "client_months", "exact_matches", "share_exact"])
display(s11_align)

s11_exact_m = pd.DataFrame({"MONTH": MONTH_ORDER[1:],
    "share_payment_equals_previous_bill": [float((s11_P[:, m][s11_B[:, m - 1] > 0] == s11_B[:, m - 1][s11_B[:, m - 1] > 0]).mean()) for m in range(1, 6)],
    "share_spend_non_negative": [float((s11_SP[:, m] >= 0).mean()) for m in range(1, 6)]})
display(s11_exact_m)
s11_KEY["exact_true"] = float(s11_align.loc[1, "share_exact"]); s11_KEY["exact_other"] = float(s11_align.loc[4, "share_exact"])
add_fact("11.2", "data structure", f"{s11_KEY['exact_true']:.1%} of payments equal the previous month's bill to the dollar, against "
         f"{s11_align.loc[2, 'share_exact']:.2%} for the same month and {s11_KEY['exact_other']:.3%} for another client's bill: the months are a linked ledger",
         s11_KEY["exact_true"], "s11_shuffle_test", 2)

**Observations:**
- A payment equals the previous month's bill to the dollar in 16.3% of client-months. Against the same month's bill it is 2.4%, against the bill two months back 2.6%, and against a random other client's bill only 0.03%.
- The match share eases from 18.3% in May to 15.2% in Sep. Implied spend is zero or positive in about 99.6% of cases every month.
- Story use: proves the months are one linked account ledger and backs the note "PAY_AMT settles the previous bill" in the dashboard.

**Shuffle test.** Two nulls: (1) *month order scrambled inside each client* (keeps each client's level, destroys the
sequence) and (2) *clients scrambled inside each month* (keeps each month's distribution, destroys the account link).
If the data were unordered snapshots, the observed metrics would sit on null 1; if months belonged to unrelated accounts, on null 2.

In [ ]:
def s11_link_metrics(Bm, Pm, Sm):
    prev, pay = Bm[:, :-1], Pm[:, 1:]
    ok = prev > 0
    spend = Bm[:, 1:] - Bm[:, :-1] + Pm[:, 1:]
    dS = np.diff(Sm, axis=1)
    return {"adjacent-month bill correlation": np.corrcoef(Bm[:, :-1].ravel(), Bm[:, 1:].ravel())[0, 1],
            "within-client lag-1 autocorr of bill (mean)": np.nanmean(s11_acf(Bm, 1)),
            "payment = previous bill exactly (share)": (pay[ok] == prev[ok]).mean(),
            "implied spend >= 0 (share)": (spend >= 0).mean(),
            "mean |change in status|": np.abs(dS).mean(),
            "status unchanged month to month (share)": (dS == 0).mean()}

s11_obs = s11_link_metrics(s11_B, s11_P, s11_S)
s11_null1, s11_null2 = [], []
for _rep in range(5):
    _o = np.argsort(s11_rng.random((s11_N, 6)), axis=1)                       # null 1: scramble month order per client
    s11_null1.append(s11_link_metrics(np.take_along_axis(s11_B, _o, 1), np.take_along_axis(s11_P, _o, 1), np.take_along_axis(s11_S, _o, 1)))
    _B2, _P2, _S2 = s11_B.copy(), s11_P.copy(), s11_S.copy()                  # null 2: scramble clients per month
    for _m in range(6):
        _pm = s11_rng.permutation(s11_N)
        _B2[:, _m], _P2[:, _m], _S2[:, _m] = s11_B[_pm, _m], s11_P[_pm, _m], s11_S[_pm, _m]
    s11_null2.append(s11_link_metrics(_B2, _P2, _S2))
s11_shuffle = pd.DataFrame({"observed": s11_obs,
                            "null 1: month order scrambled": pd.DataFrame(s11_null1).mean(),
                            "null 2: clients scrambled": pd.DataFrame(s11_null2).mean()})
display(s11_shuffle)

fig, axes = plt.subplots(2, 3, figsize=(15, 6.5))
for ax, (_k, _row) in zip(axes.ravel(), s11_shuffle.iterrows()):
    ax.bar(["observed", "order\nscrambled", "clients\nscrambled"], _row.values, color=[C_DEF, C_NEU, C_OK])
    ax.set_title(_k, fontsize=10); ax.axhline(0, color="k", lw=.6)
fig.suptitle("Shuffle test: the observed regularities need BOTH the month order and the client link", fontweight="bold")
plt.tight_layout(); savefig("s11_shuffle_test"); plt.show()
s11_KEY["ac_obs"] = float(s11_shuffle.iloc[1, 0]); s11_KEY["ac_null"] = float(s11_shuffle.iloc[1, 1])
add_fact("11.2", "data structure", f"Within-client lag-1 autocorrelation of bills is {s11_KEY['ac_obs']:.2f} but {s11_KEY['ac_null']:.2f} once the month order is scrambled; "
         f"adjacent-bill correlation falls from {s11_shuffle.iloc[0, 0]:.2f} to {s11_shuffle.iloc[0, 2]:.2f} when clients are scrambled", s11_KEY["ac_obs"], "s11_shuffle_test", 2)

**Observations:**
- Shuffling clients inside each month drops the bill correlation between neighbouring months from 0.94 to 0.005, and exact "payment = previous bill" matches from 16.3% to 0.03%.
- Shuffling the month order inside each client keeps the bill correlation high (0.88), but the within-client correlation flips from +0.18 to -0.17 and exact matches fall to 4.2%.
- In the real data status stays the same from one month to the next 81% of the time, against 73% (months shuffled) and 36% (clients shuffled).
- Story use: one "the months are real" chart for a methods page. Background for the main story.

### 11.3 Which parts are snapshot-only, and does that bias the time view?
`LIMIT_BAL` and the demographics exist once. A limit can be raised or cut during six months, but we only see one value
(most likely the latest). Every monthly utilisation therefore divides a *dated* bill by an *undated* limit: April
utilisation may use September's limit (look-ahead). First the symptoms: over-limit shares and utilisation by month.

In [ ]:
s11_over = pd.DataFrame({
    "MONTH": MONTH_ORDER,
    "mean_util": s11_U.mean(0), "median_util": np.median(s11_U, axis=0),
    "share_over_limit": (s11_U > 1).mean(0), "share_over_110pct": (s11_U > 1.1).mean(0),
    "share_util_90_100": ((s11_U > 0.9) & (s11_U <= 1)).mean(0),
    "default_rate_if_over_limit": [s11_Y[s11_U[:, m] > 1].mean() for m in range(6)]})
display(s11_over)
_oa, _os = s11_U[:, 0] > 1, s11_U[:, 5] > 1
s11_over_flow = pd.DataFrame([
    ("over limit in April only (not in Sep)", int((_oa & ~_os).sum()), s11_Y[_oa & ~_os].mean()),
    ("over limit in Sep only (not in April)", int((~_oa & _os).sum()), s11_Y[~_oa & _os].mean()),
    ("over limit in both", int((_oa & _os).sum()), s11_Y[_oa & _os].mean()),
    ("max bill / limit > 1.0 in any month", int((s11_U.max(1) > 1).sum()), s11_Y[s11_U.max(1) > 1].mean()),
    ("max bill / limit > 1.5 in any month", int((s11_U.max(1) > 1.5).sum()), s11_Y[s11_U.max(1) > 1.5].mean()),
], columns=["group", "clients", "default_rate"])
display(s11_over_flow)
print(f"LIMIT_BAL: {df['LIMIT_BAL'].nunique()} distinct values; {(df['LIMIT_BAL'] % 10000 == 0).mean():.1%} are multiples of 10,000 "
      f"(administrative grid, one value per client, no history).")
s11_KEY["over_apr"] = float(s11_over.loc[0, "share_over_limit"]); s11_KEY["over_sep"] = float(s11_over.loc[5, "share_over_limit"])

**Observations:**
- Average utilisation (bill / limit) rises every month, from 31.9% in April to 42.4% in September. The share of clients over their limit rises from 2.7% to 7.1%.
- Over-limit clients default at 30-36% in every month, well above the 22.1% base.
- 3,931 clients exceed their limit in at least one month (default 30.1%). Only 294 are over the limit in both April and September.
- The limit has 81 distinct values, all multiples of 10,000, and one value per client, so any limit change during the window is hidden.
- Story use: caveat. Label utilisation as "bill / current limit".

**How much could an unseen limit change matter?** Rank-based findings are immune to a *uniform* change, so the risk
is a *selective* one. Scenario: banks raised limits during the window for clients who were never late (a common
practice), so their true April limit was lower by *k*%. We recompute April utilisation and its gap between defaulters and non-defaulters.

In [ ]:
_never = ~s11_LATE.any(1)
s11_scen = []
for _k in (0, 10, 20, 30):
    _lim_apr = np.where(_never, s11_L / (1 + _k / 100), s11_L)
    _ua = s11_B[:, 0] / _lim_apr
    _trend = s11_U[:, 5] - _ua
    s11_scen.append((f"+{_k}% raise for never-late clients", _ua.mean(), (_ua > 1).mean(), _ua[s11_Y == 1].mean(), _ua[s11_Y == 0].mean(),
                     _ua[s11_Y == 1].mean() - _ua[s11_Y == 0].mean(), (_trend > 0.05).mean(),
                     stats.spearmanr(_ua, s11_Y)[0]))
s11_scen = pd.DataFrame(s11_scen, columns=["scenario (true April limit lower)", "mean_april_util", "share_over_limit_april", "april_util_defaulters",
                                           "april_util_non_defaulters", "gap_def_minus_nondef", "share_util_rising_gt5pts", "spearman_april_util_vs_default"])
display(s11_scen)
s11_KEY["gap0"] = float(s11_scen.loc[0, "gap_def_minus_nondef"]); s11_KEY["gap30"] = float(s11_scen.loc[3, "gap_def_minus_nondef"])
add_fact("11.3", "data structure", f"Utilisation divides a dated bill by an undated limit: over-limit share is {s11_KEY['over_apr']:.1%} in April vs {s11_KEY['over_sep']:.1%} in September; "
         f"a hidden 30% limit raise for good payers would shrink the April defaulter gap from {s11_KEY['gap0']:.3f} to {s11_KEY['gap30']:.3f}", s11_KEY["over_apr"], None, 1)

**Observations:**
- Suppose the bank quietly raised limits by 30% for never-late clients. Then the true April utilisation gap between defaulters and non-defaulters would shrink from 10.3 to 7.0 percentage points.
- The rank correlation of April utilisation with default would drop from 0.095 to 0.063.
- The utilisation signal survives this scenario but shrinks by about a third.
- Story use: caveat. Present utilisation findings as a direction, not an exact size.

`AGE` is an integer measured once. Over a six-month window roughly half of all clients have a birthday, so clients on
the top edge of an age band could sit in the next band depending on the (unknown) measurement date. How many are exposed?

In [ ]:
_edges = [25, 30, 35, 40, 45, 50, 60]
s11_age_edge = pd.DataFrame({"upper_edge_age": _edges,
                             "clients_at_edge": [int((df["AGE"] == a).sum()) for a in _edges],
                             "default_rate_at_edge": [df.loc[df["AGE"] == a, TARGET].mean() for a in _edges]})
s11_age_edge["share_of_all_clients"] = s11_age_edge["clients_at_edge"] / s11_N
display(s11_age_edge)
print(f"Clients on a band edge: {s11_age_edge['clients_at_edge'].sum():,} ({s11_age_edge['share_of_all_clients'].sum():.1%}); "
      f"about half of them would change AGE_BAND with one birthday inside the window.")
print("Static demographics per client -> no within-client variation, so they can never enter a within-client (fixed-effect) analysis.")

**Observations:**
- 5,659 clients (18.9%) are at the top age of a band (25, 30, 35, ...), so a birthday inside the window could move them to the next band.
- Default rates at these edges range from 18.8% (age 45) to 34.3% (age 60, only 67 clients).
- Age and the other demographics never change within a client, so they cannot be used in a within-client comparison.
- Story use: background only, with a small caveat on age-band charts.

### 11.4 How "periodic" is it really? Within- vs between-client variance (ICC)
If almost all variation is *between* clients, six months behave like six noisy copies of one snapshot. If much is *within*
clients, the panel carries real time information. Two-way decomposition on the balanced panel:
total = between clients + common month effect + residual (client-specific movement). ICC(1) = share of variance due to
stable client differences; `reliability_of_mean` = how stable the 6-month average is (Spearman-Brown formula: reliability of an average of several measurements).

In [ ]:
def s11_icc(X):
    X = np.asarray(X, float)
    X = X[~np.isnan(X).any(1)]
    n, T = X.shape; gm = X.mean()
    ssb = T * ((X.mean(1) - gm) ** 2).sum()
    ssm = n * ((X.mean(0) - gm) ** 2).sum()
    sst = ((X - gm) ** 2).sum()
    sse = sst - ssb - ssm
    msb, msw = ssb / (n - 1), (sst - ssb) / (n * (T - 1))
    icc = (msb - msw) / (msb + (T - 1) * msw)
    return {"clients": n, "periods": T, "between_client_share": ssb / sst, "month_share": ssm / sst, "within_residual_share": sse / sst,
            "icc1": icc, "reliability_of_mean": T * icc / (1 + (T - 1) * icc)}

def s11_slog(X):
    return np.sign(X) * np.log1p(np.abs(X))

s11_icc_vars = {
    "Status code (-2..8)": s11_S.astype(float),
    "Late flag (status >= 1)": s11_LATE.astype(float),
    "Months of delay (status clipped at 0)": np.clip(s11_S, 0, None).astype(float),
    "Bill (NTD)": s11_B, "Bill (signed log)": s11_slog(s11_B),
    "Payment (NTD)": s11_P, "Payment (log1p)": np.log1p(s11_P),
    "Paid anything (flag)": (s11_P > 0).astype(float),
    "Utilisation": s11_U,
    "Pay ratio (May-Sep)": s11_R[:, 1:],
    "Implied spend (May-Sep)": s11_SP[:, 1:],
    "Spend / limit (May-Sep)": s11_SPL[:, 1:],
}
s11_icc_tab = pd.DataFrame([{"variable": k, **s11_icc(v)} for k, v in s11_icc_vars.items()])
for _lab, _mask in [("icc1_non_default", s11_Y == 0), ("icc1_default", s11_Y == 1)]:
    s11_icc_tab[_lab] = [s11_icc(v[_mask])["icc1"] for v in s11_icc_vars.values()]
s11_icc_tab = s11_icc_tab.sort_values("between_client_share", ascending=False).reset_index(drop=True)
display(s11_icc_tab.round(3))
export(s11_icc_tab, "s11_icc_variance_decomposition")

fig, ax = plt.subplots(figsize=(11, 5.5))
_y = np.arange(len(s11_icc_tab))[::-1]
ax.barh(_y, s11_icc_tab["between_client_share"], color=C_OK, label="between clients (snapshot-like)")
ax.barh(_y, s11_icc_tab["month_share"], left=s11_icc_tab["between_client_share"], color=C_ACC, label="common month effect (calendar)")
ax.barh(_y, s11_icc_tab["within_residual_share"], left=s11_icc_tab["between_client_share"] + s11_icc_tab["month_share"], color=C_DEF, label="within client over time (panel-like)")
ax.set_yticks(_y); ax.set_yticklabels(s11_icc_tab["variable"]); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("share of total variance"); ax.set_title("How periodic is the data? Variance split: between clients vs within clients over time")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, -0.3), ncol=3, frameon=False)
plt.tight_layout(); savefig("s11_icc_decomposition"); plt.show()

_g = s11_icc_tab.set_index("variable")
s11_KEY["icc_bill"] = float(_g.loc["Bill (NTD)", "icc1"]); s11_KEY["icc_status"] = float(_g.loc["Status code (-2..8)", "icc1"])
s11_KEY["icc_pay"] = float(_g.loc["Payment (NTD)", "icc1"]); s11_KEY["icc_spend"] = float(_g.loc["Implied spend (May-Sep)", "icc1"])
s11_KEY["month_share_max"] = float(s11_icc_tab["month_share"].max())
add_fact("11.4", "data structure", f"Bills are {s11_KEY['icc_bill']:.0%} between-client (ICC) and status {s11_KEY['icc_status']:.0%}, but payments only {s11_KEY['icc_pay']:.0%} "
         f"and implied spend {s11_KEY['icc_spend']:.0%}: levels are snapshot-like, flows are panel-like", s11_KEY["icc_bill"], "s11_icc_decomposition", 3)
add_fact("11.4", "calendar", f"A common month effect never explains more than {s11_KEY['month_share_max']:.1%} of any series' variance: there is no strong portfolio-wide calendar swing",
         s11_KEY["month_share_max"], "s11_icc_decomposition", 2)

**Observations:**
- ICC is the share of variation that comes from stable differences between clients. Bills (ICC 0.88) and utilisation (0.82) are mostly client traits.
- Status is in the middle (ICC 0.65). Payments (0.19) and spend / limit (0.09) mostly change month to month within the same client.
- The calendar month explains at most 1.6% of the variance (late flag) and under 1% for every other series.
- The late flag is more consistent for defaulters (ICC 0.48) than for non-defaulters (0.37): their lateness is a habit, not a one-off.
- Story use: show bills and utilisation as client segments, and payments and spend as monthly behaviour.

### 11.5 Is there a calendar effect, and is the portfolio deteriorating month by month?
Portfolio totals by calendar month: billing, payments, implied spend, late share, the inflow of *new* delinquency and the
outflow of cures. Only six points and no second year, so **seasonality cannot be separated from trend** - we can only
describe the Apr-Sep path.

**Macro backdrop (Taiwan's 2005-06 card and cash-card debt crisis).** The window sits right at the turn of the cycle:
* Fitch (quoted by FinanceAsia, 28 Mar 2006) dates the first rise in card delinquency to **Q3 2005** and reports annualised charge-off rates of **14% (credit cards) and 17% (cash cards) in Jan 2006**, against about 6% in earlier years; more than **400,000** borrowers were over three months past due. [FinanceAsia](https://www.financeasia.com/article/taiwans-increasingly-dangerous-credit-card-problem/36228)
* Card debt **peaked in Q3 2005**; banks wrote off about US\$4.9bn of bad loans in 2006, roughly 70% of it credit-card debt. [Electronic Payments International, Feb 2007](https://www.electronicpaymentsinternational.com/country-surveys/taiwans-credit-crisis-the-calm-after-the-storm/)
* From **Nov-Dec 2005** the "card slave" controversy led the Financial Supervisory Commission and the Bankers Association to open a debt negotiation mechanism, stop new issuance at banks above a 2.5% bad-loan ratio and raise the minimum payment to 10%; by 24 Aug 2006 about 188,000 restructurings (NTD 278bn, about a third of peak balances) had been agreed. [National Policy Foundation review](https://www.npf.org.tw/2/3558)
* By **Apr 2006** the bad-loan ratio was 3.38% on credit cards and a record 6.75% on cash cards. [Taipei Times, 1 Jun 2006](https://www.taipeitimes.com/News/biz/archives/2006/06/01/2003311182) The BIS notes card receivables fell about 30% in the two years after the peak. [BIS Quarterly Review, Jun 2007](https://www.bis.org/publications/recent-episodes-credit-card-distress-asia_1.pdf)

So these 30,000 accounts are observed in the last months **before** the crisis became public. Caution on the status
series: code 1 exists almost only in September (Section 1), so "late" shares are shown at thresholds >= 1 and >= 2.

In [ ]:
s11_mrows = []
for _m in range(6):
    _r = {"MONTH": MONTH_ORDER[_m], "month_num": _m + 1,
          "total_bill_bn": s11_B[:, _m].sum() / 1e9, "total_pay_bn": s11_P[:, _m].sum() / 1e9,
          "total_spend_bn": np.nan if _m == 0 else s11_SP[:, _m].sum() / 1e9,
          "pay_to_prev_bill": np.nan if _m == 0 else s11_P[:, _m].sum() / np.clip(s11_B[:, _m - 1], 0, None).sum(),
          "active_share": (s11_B[:, _m] > 0).mean(), "payer_share": (s11_P[:, _m] > 0).mean(),
          "mean_util": s11_U[:, _m].mean(), "median_bill": np.median(s11_B[:, _m]),
          "late_share_ge1": (s11_S[:, _m] >= 1).mean(), "late_share_ge2": (s11_S[:, _m] >= 2).mean(), "late_share_ge3": (s11_S[:, _m] >= 3).mean()}
    if _m > 0:
        _risk = s11_S[:, _m - 1] <= 0; _wl = s11_S[:, _m - 1] >= 1
        _r.update({"at_risk_current": int(_risk.sum()), "new_late_ge1": int((_risk & (s11_S[:, _m] >= 1)).sum()),
                   "inflow_rate_ge1": (s11_S[_risk, _m] >= 1).mean(), "inflow_rate_ge2": (s11_S[_risk, _m] >= 2).mean(),
                   "cures": int((_wl & (s11_S[:, _m] <= 0)).sum()), "cure_rate": (s11_S[_wl, _m] <= 0).mean(),
                   "share_clients_bill_up": (s11_B[:, _m] > s11_B[:, _m - 1]).mean(), "share_clients_pay_up": (s11_P[:, _m] > s11_P[:, _m - 1]).mean(),
                   "share_clients_status_worse": (s11_S[:, _m] > s11_S[:, _m - 1]).mean(), "share_clients_status_better": (s11_S[:, _m] < s11_S[:, _m - 1]).mean()})
    s11_mrows.append(_r)
s11_monthly = pd.DataFrame(s11_mrows)
for _c in ["total_bill_bn", "total_pay_bn", "total_spend_bn"]:
    s11_monthly[_c.replace("_bn", "_mom_growth")] = s11_monthly[_c].pct_change()
s11_monthly["net_new_late"] = s11_monthly["new_late_ge1"] - s11_monthly["cures"]
display(s11_monthly.set_index("MONTH").T)
export(s11_monthly, "s11_monthly_aggregate")

**Observations:**
- Total billed balances rise every month, from 1.17 bn NTD in April to 1.54 bn in September. Total payments stay around 0.14-0.18 bn.
- Payments cover only 11.5-12.6% of the previous month's total bill in every month.
- The share 2+ months late rises from 10.3% (Apr) to 14.8% (Aug). New lateness among current clients rises from 3.2% in May to 6.1% in July.
- September breaks the pattern because the 1-month code starts there. New lateness jumps to 11.1% while the 2+ share falls to 10.4%, and only 10.1% of late clients cure (24-32% before).
- Story use: the opening "debt was building up" timeline (Apr-Aug), with a footnote on the September coding break.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
_x = np.arange(6)
ax = axes[0, 0]
ax.plot(_x, s11_monthly["total_bill_bn"], marker="o", color=C_DEF, label="total billed")
ax.set_ylabel("NTD bn (bills)"); ax2 = ax.twinx(); ax2.grid(False)
ax2.plot(_x, s11_monthly["total_pay_bn"], marker="s", color=C_OK, label="total paid")
ax2.plot(_x, s11_monthly["total_spend_bn"], marker="^", color=C_ACC, label="implied spend")
ax2.set_ylabel("NTD bn (payments, spend)"); ax.set_title("Portfolio totals by month")
ax.legend(loc="upper left", frameon=False); ax2.legend(loc="lower right", frameon=False)
ax = axes[0, 1]
for _c, _col, _lab in [("total_bill_mom_growth", C_DEF, "bills"), ("total_pay_mom_growth", C_OK, "payments"), ("total_spend_mom_growth", C_ACC, "spend")]:
    ax.plot(_x, s11_monthly[_c], marker="o", color=_col, label=_lab)
ax.axhline(0, color="k", lw=.7); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title("Month-over-month growth"); ax.legend(frameon=False)
ax = axes[0, 2]
ax.plot(_x, s11_monthly["pay_to_prev_bill"], marker="o", color=C_OK, label="paid / previous bill (aggregate)")
ax.plot(_x, s11_monthly["payer_share"], marker="s", color=C_NEU, label="share of clients paying anything")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title("Repayment intensity"); ax.legend(frameon=False)
ax = axes[1, 0]
for _c, _col, _lab in [("late_share_ge1", C_DEF, "status >= 1 (Sep adds code 1)"), ("late_share_ge2", "#7f0000", "status >= 2 (comparable)"), ("late_share_ge3", C_NEU, "status >= 3")]:
    ax.plot(_x, s11_monthly[_c], marker="o", color=_col, label=_lab)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title("Stock of late accounts"); ax.legend(frameon=False)
ax = axes[1, 1]
ax.plot(_x, s11_monthly["inflow_rate_ge1"], marker="o", color=C_DEF, label="inflow: current -> late (>=1)")
ax.plot(_x, s11_monthly["inflow_rate_ge2"], marker="o", ls="--", color="#7f0000", label="inflow: current -> late (>=2)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_ylabel("inflow rate")
ax2 = ax.twinx(); ax2.grid(False); ax2.plot(_x, s11_monthly["cure_rate"], marker="s", color=C_ACC, label="cure: late -> current")
ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax2.set_ylabel("cure rate")
ax.set_title("Flows: new delinquency vs cures"); ax.legend(loc="upper left", frameon=False); ax2.legend(loc="lower right", frameon=False)
ax = axes[1, 2]
for _c, _col, _lab in [("share_clients_bill_up", C_DEF, "bill higher than last month"), ("share_clients_pay_up", C_OK, "payment higher than last month"),
                       ("share_clients_status_worse", "#7f0000", "status worse"), ("share_clients_status_better", C_ACC, "status better")]:
    ax.plot(_x, s11_monthly[_c], marker="o", color=_col, label=_lab)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title("Within-client direction of change (share of clients)"); ax.legend(frameon=False, fontsize=8)
for ax in axes.ravel():
    ax.set_xticks(_x); ax.set_xticklabels(MONTH_ORDER)
fig.suptitle("Calendar view of the portfolio, Apr-Sep 2005 (six points: trend and season cannot be separated)", fontweight="bold")
plt.tight_layout(); savefig("s11_calendar_aggregates"); plt.show()

_ms = s11_monthly.set_index("MONTH")
s11_KEY["bill_growth"] = float(_ms.loc["Sep", "total_bill_bn"] / _ms.loc["Apr", "total_bill_bn"] - 1)
s11_KEY["late2_apr"] = float(_ms.loc["Apr", "late_share_ge2"]); s11_KEY["late2_aug"] = float(_ms.loc["Aug", "late_share_ge2"]); s11_KEY["late2_sep"] = float(_ms.loc["Sep", "late_share_ge2"])
s11_KEY["peak_inflow_month"] = str(_ms["inflow_rate_ge2"].iloc[1:5].idxmax())
s11_KEY["net_new_total"] = int(_ms["net_new_late"].sum())
print(f"Total billed Apr -> Sep: {s11_KEY['bill_growth']:+.1%} | late (>=2) share Apr {s11_KEY['late2_apr']:.1%} -> Aug {s11_KEY['late2_aug']:.1%} -> Sep {s11_KEY['late2_sep']:.1%}")
print(f"Highest comparable inflow rate (May-Aug, status >= 2): {s11_KEY['peak_inflow_month']} | net new late accounts over the window: {s11_KEY['net_new_total']:+,}")
add_fact("11.5", "calendar", f"Total billed balances grew {s11_KEY['bill_growth']:+.1%} from April to September while the share of accounts 2+ months late moved "
         f"{s11_KEY['late2_apr']:.1%} -> {s11_KEY['late2_aug']:.1%} (Aug) -> {s11_KEY['late2_sep']:.1%} (Sep)", s11_KEY["bill_growth"], "s11_calendar_aggregates", 2)
add_fact("11.5", "calendar", f"Net inflow into delinquency over the window is {s11_KEY['net_new_total']:+,} accounts (new late minus cures), on the eve of Taiwan's 2005-06 card crisis",
         s11_KEY["net_new_total"], "s11_calendar_aggregates", 2)

**Observations:**
- This chart is an image. Numbers come from the printed lines below it and the monthly table above.
- Total billed balances grew 31.8% from April to September.
- The share 2+ months late goes from 10.3% in Apr to 14.7% in Aug, then 10.4% in Sep. The Sep drop is a coding break, not a recovery.
- July has the highest comparable rate of new lateness (6.1%). New late accounts minus cures add up to +3,739 over the window.
- Story use: main timeline chart for the "build-up" act. Annotate the September break.

### 11.6 Does each client have a rhythm? Frequency and periodicity with only six points
Six observations allow only a crude spectrum (periods of 6, 3 and 2 months) and noisy autocorrelations - stated up front.
Still, the *pattern of paying months* is a clean, robust descriptor. Rhythm types from the paid / not-paid sequence:

| Type | Rule |
|---|---|
| Every month | paid in all 6 months |
| Misses one month | paid in exactly 5 |
| Alternating | strict on/off alternation (6 runs) |
| One block | 2-4 paying months, all contiguous |
| Sporadic | 2-4 paying months, scattered |
| One lump | a single payment |
| Never pays | no payment in the window |

New columns: `PN_N_PAY_MONTHS`, `PN_PAY_RUNS`, `PN_RHYTHM`.

In [ ]:
s11_paid = s11_P > 0
s11_npay = s11_paid.sum(1)
s11_runs = 1 + (s11_paid[:, 1:] != s11_paid[:, :-1]).sum(1)
s11_blocks = s11_paid[:, 0].astype(int) + (s11_paid[:, 1:] & ~s11_paid[:, :-1]).sum(1)
S11_RHYTHM_ORDER = ["Every month", "Misses one month", "Alternating", "One block", "Sporadic", "One lump", "Never pays"]
s11_rhythm = np.select([s11_npay == 6, s11_npay == 5, s11_npay == 0, s11_npay == 1, s11_runs == 6, s11_blocks == 1],
                       ["Every month", "Misses one month", "Never pays", "One lump", "Alternating", "One block"], default="Sporadic")
df["PN_N_PAY_MONTHS"] = s11_npay
df["PN_PAY_RUNS"] = s11_runs
df["PN_RHYTHM"] = pd.Categorical(s11_rhythm, categories=S11_RHYTHM_ORDER, ordered=True)

s11_rh = rate_table(df, "PN_RHYTHM")
_extra = df.groupby("PN_RHYTHM", observed=True).agg(mean_n_late=("N_LATE", "mean"), share_never_late=("N_LATE", lambda v: (v == 0).mean()),
            share_no_bill_at_all=("TOTAL_BILL", lambda v: (v == 0).mean()), median_limit=("LIMIT_BAL", "median"), mean_util=("AVG_UTIL", "mean")).reset_index()
s11_rh = s11_rh.merge(_extra, on="PN_RHYTHM")
_nl = rate_table(df[df["N_LATE"] == 0], "PN_RHYTHM")[["PN_RHYTHM", "n", "rate"]].rename(columns={"n": "n_never_late", "rate": "rate_among_never_late"})
s11_rh = s11_rh.merge(_nl, on="PN_RHYTHM", how="left")
display(s11_rh[["PN_RHYTHM", "n", "share_clients", "rate", "ci_lo", "ci_hi", "lift", "mean_n_late", "share_never_late", "share_no_bill_at_all", "mean_util", "n_never_late", "rate_among_never_late"]])
export(s11_rh, "s11_rhythm_type_rates")

fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
plot_rate(df, "PN_RHYTHM", ax=axes[0], order=S11_RHYTHM_ORDER, title="Default rate by payment rhythm", rotate=25)
_t = rate_table(df, "PN_PAY_RUNS")
axes[1].bar(_t["PN_PAY_RUNS"].astype(str), _t["rate"], yerr=[_t["rate"] - _t["ci_lo"], _t["ci_hi"] - _t["rate"]], color=C_NEU, capsize=3)
axes[1].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_xlabel("number of runs in the paid / not-paid sequence (1 = constant, 6 = alternating)"); axes[1].set_title("Default rate by number of payment runs")
plt.tight_layout(); savefig("s11_rhythm_default"); plt.show()
display(_t[["PN_PAY_RUNS", "n", "rate", "ci_lo", "ci_hi", "lift"]])
_rr = s11_rh.set_index("PN_RHYTHM")
s11_KEY["rate_every"] = float(_rr.loc["Every month", "rate"]); s11_KEY["rate_sporadic"] = float(_rr.loc["Sporadic", "rate"]); s11_KEY["share_every"] = float(_rr.loc["Every month", "share_clients"])
add_fact("11.6", "rhythm", f"{s11_KEY['share_every']:.0%} of clients pay in all six months and default at {s11_KEY['rate_every']:.1%}; sporadic payers default at {s11_KEY['rate_sporadic']:.1%}",
         s11_KEY["rate_sporadic"], "s11_rhythm_default", 2)

**Observations:**
- 51.5% of clients pay something every month, and they default at 13.9%, well below the 22.1% base.
- Irregular payers default far more: alternating payers 52.0% (298 clients), "never pays" 38.0%, sporadic 34.3%.
- Default rises steadily with the number of separate paying stretches: 16.0% with one stretch, 24.0% with two, 29.9% with three, up to 52.0% with six.
- Even among never-late clients, rhythm matters: 10.1% default for every-month payers vs 25.1% for those who never pay.
- Story use: strong Tableau segment (`PN_RHYTHM` as colour or filter): steady payers are safe, a broken rhythm is risky.

**Per-client autocorrelation.** Lag-1 and lag-2 autocorrelation of each client's own six-point series (bill, payment,
status, utilisation). With T = 6 the estimator is biased towards about -1/T = -0.17 for pure noise, so read the sign and the
gap between groups, not the level. New columns: `PN_BILL_AC1`, `PN_PAY_AC1`.

In [ ]:
s11_ac_rows = []
s11_ac_store = {}
for _name, _M in [("bill", s11_B), ("payment", s11_P), ("status", s11_S.astype(float)), ("utilisation", s11_U)]:
    for _lag in (1, 2):
        _r = s11_acf(_M, _lag); s11_ac_store[(_name, _lag)] = _r
        for _lab, _mask in [("All", np.ones(s11_N, bool)), ("No default", s11_Y == 0), ("Default", s11_Y == 1)]:
            _v = _r[_mask]
            s11_ac_rows.append((_name, _lag, _lab, np.isnan(_v).mean(), np.nanmean(_v), np.nanmedian(_v), np.nanmean(_v > 0) if np.isfinite(_v).any() else np.nan))
s11_ac_tab = pd.DataFrame(s11_ac_rows, columns=["series", "lag", "group", "share_constant_series", "mean_autocorr", "median_autocorr", "share_positive"])
s11_ac_tab["share_positive"] = [np.nanmean(s11_ac_store[(a, b)][(s11_Y == 0) if g == "No default" else (s11_Y == 1) if g == "Default" else np.ones(s11_N, bool)][
    ~np.isnan(s11_ac_store[(a, b)][(s11_Y == 0) if g == "No default" else (s11_Y == 1) if g == "Default" else np.ones(s11_N, bool)])] > 0)
    for a, b, g in zip(s11_ac_tab["series"], s11_ac_tab["lag"], s11_ac_tab["group"])]
display(s11_ac_tab.round(3))
df["PN_BILL_AC1"] = s11_ac_store[("bill", 1)]
df["PN_PAY_AC1"] = s11_ac_store[("payment", 1)]

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, _name in zip(axes, ["bill", "payment", "status", "utilisation"]):
    _r = s11_ac_store[(_name, 1)]
    for _lab, _mask, _col in [("No default", s11_Y == 0, C_OK), ("Default", s11_Y == 1, C_DEF)]:
        ax.hist(_r[_mask & ~np.isnan(_r)], bins=40, range=(-1, 1), density=True, alpha=.5, color=_col, label=_lab)
    ax.axvline(-1 / 6, color="k", ls=":", lw=1); ax.set_title(f"Lag-1 autocorrelation: {_name}"); ax.set_xlabel("r(1) per client")
axes[0].legend(frameon=False)
plt.tight_layout(); savefig("s11_client_autocorr"); plt.show()

s11_ac_band = df.assign(_b=pd.cut(df["PN_BILL_AC1"], [-1.01, -0.3, 0, 0.3, 0.6, 1.01], labels=["< -0.3", "-0.3-0", "0-0.3", "0.3-0.6", "> 0.6"]).astype(object).fillna("constant series"))
_t = rate_table(s11_ac_band, "_b").rename(columns={"_b": "bill_lag1_autocorr_band"})
display(_t[["bill_lag1_autocorr_band", "n", "rate", "ci_lo", "ci_hi", "lift"]])
_a = s11_ac_tab.query("series=='bill' and lag==1").set_index("group")["mean_autocorr"]
s11_KEY["ac1_def"] = float(_a["Default"]); s11_KEY["ac1_nondef"] = float(_a["No default"])
add_fact("11.6", "rhythm", f"Defaulters' bills are more persistent month to month (mean within-client lag-1 autocorrelation {s11_KEY['ac1_def']:.2f}) than non-defaulters' ({s11_KEY['ac1_nondef']:.2f}): "
         f"a balance that only drifts is a debt, one that saw-tooths is spending that gets repaid", s11_KEY["ac1_def"], "s11_client_autocorr", 2)

**Observations:**
- Defaulters' bills drift more smoothly: their average one-month autocorrelation (how much a month resembles the one before) is 0.20, against 0.17 for non-defaulters.
- Defaulters' payments jump around more: one-month autocorrelation -0.14 vs -0.05.
- Clients whose bill never changes (1,227) default at 39.8%. Clients whose bill zig-zags (autocorrelation below -0.3) default at only 16.4%.
- Story use: supporting detail. A balance that never moves is a warning sign, probably because nothing is being paid off.

**Zig-zag count.** Number of direction reversals in the month-to-month changes (0 = monotone, 4 = saw-tooth). A monotone
bill path means steady build-up or pay-down; a saw-tooth is a transactor's signature.

In [ ]:
def s11_sign_changes(M):
    d = np.sign(np.diff(M, axis=1))
    return ((d[:, 1:] * d[:, :-1]) < 0).sum(1)

s11_zz = []
for _name, _M in [("bill", s11_B), ("payment", s11_P), ("utilisation", s11_U)]:
    _sc = s11_sign_changes(_M)
    _t = rate_table(df.assign(_z=_sc), "_z").rename(columns={"_z": "direction_reversals"}); _t.insert(0, "series", _name)
    s11_zz.append(_t)
s11_zz = pd.concat(s11_zz, ignore_index=True)
display(s11_zz.pivot(index="direction_reversals", columns="series", values="rate").round(4))
display(s11_zz.pivot(index="direction_reversals", columns="series", values="n"))
export(s11_zz[["series", "direction_reversals", "n", "defaults", "rate", "ci_lo", "ci_hi", "lift"]], "s11_zigzag_rates")
_mono_up = (np.diff(s11_B, axis=1) > 0).all(1); _mono_dn = (np.diff(s11_B, axis=1) < 0).all(1)
print(f"Bill strictly rising all 5 steps: {_mono_up.sum():,} clients, default {s11_Y[_mono_up].mean():.1%} | strictly falling: {_mono_dn.sum():,} clients, default {s11_Y[_mono_dn].mean():.1%}")

**Observations:**
- Bill paths are spread almost evenly across 0-4 changes of direction (20-25% of clients each), so there is no single typical path.
- 1,654 clients whose bills rise in all five steps default at 15.7%. 1,990 whose bills fall in all five steps default at 23.2%.
- This matches the earlier finding that rising balances alone are not a danger sign.
- Story use: background. A counter-intuitive side note: rising bills are not the risk.

**A tiny spectrum.** The discrete Fourier transform of a 6-point series has only three non-trivial frequencies: periods of
6 months (one slow swing, trend-like), 3 months and 2 months (alternation). For white noise (pure random variation) the expected power shares are
40% / 40% / 20%. Excess power at period 2 would indicate a bi-monthly rhythm. New column: `PN_DOM_PERIOD` (payments).

In [ ]:
def s11_spectrum(M):
    xc = M - M.mean(1, keepdims=True)
    F = np.abs(np.fft.rfft(xc, axis=1)) ** 2            # k = 0..3
    pw = F[:, 1:] * np.array([2.0, 2.0, 1.0])           # one-sided weights (Parseval)
    tot = pw.sum(1)
    with np.errstate(divide="ignore", invalid="ignore"):
        sh = np.where(tot[:, None] > 0, pw / np.where(tot[:, None] > 0, tot[:, None], 1), np.nan)
    return sh

S11_PERIODS = ["6 months (slow swing)", "3 months", "2 months (alternating)"]
s11_spec_rows = []
s11_spec = {}
for _name, _M in [("payment", s11_P), ("bill", s11_B), ("status", s11_S.astype(float))]:
    _sh = s11_spectrum(_M); s11_spec[_name] = _sh
    for _lab, _mask in [("All", np.ones(s11_N, bool)), ("No default", s11_Y == 0), ("Default", s11_Y == 1)]:
        _mm = np.nanmean(_sh[_mask], axis=0)
        s11_spec_rows.append((_name, _lab, int((~np.isnan(_sh[_mask, 0])).sum()), _mm[0], _mm[1], _mm[2]))
s11_spec_rows.append(("white-noise benchmark", "-", 0, 0.4, 0.4, 0.2))
s11_spec_tab = pd.DataFrame(s11_spec_rows, columns=["series", "group", "n_non_constant"] + S11_PERIODS)
display(s11_spec_tab.round(3))

_sh = s11_spec["payment"]
_dom = np.where(np.isnan(_sh[:, 0]), "Flat (constant payments)", np.array(S11_PERIODS)[np.nan_to_num(_sh, nan=0).argmax(1)])
df["PN_DOM_PERIOD"] = pd.Categorical(_dom, categories=S11_PERIODS + ["Flat (constant payments)"])
s11_dom = rate_table(df, "PN_DOM_PERIOD")
display(s11_dom[["PN_DOM_PERIOD", "n", "share_clients", "rate", "ci_lo", "ci_hi", "lift"]])
export(s11_dom, "s11_dominant_period_rates")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.3))
_w = 0.25; _xx = np.arange(3)
for _j, (_lab, _col) in enumerate([("No default", C_OK), ("Default", C_DEF)]):
    _row = s11_spec_tab[(s11_spec_tab["series"] == "payment") & (s11_spec_tab["group"] == _lab)][S11_PERIODS].values[0]
    axes[0].bar(_xx + (_j - 0.5) * _w, _row, _w, color=_col, label=_lab)
axes[0].plot(_xx, [0.4, 0.4, 0.2], "k_", ms=40, mew=2, label="white noise")
axes[0].set_xticks(_xx); axes[0].set_xticklabels(S11_PERIODS); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].set_title("Payments: mean share of spectral power by period"); axes[0].legend(frameon=False)
plot_rate(df, "PN_DOM_PERIOD", ax=axes[1], title="Default rate by dominant payment period", rotate=15)
plt.tight_layout(); savefig("s11_payment_spectrum"); plt.show()
s11_KEY["p2_share"] = float(s11_spec_tab.query("series=='payment' and group=='All'")[S11_PERIODS[2]].iloc[0])
add_fact("11.6", "rhythm", f"Payments put {s11_KEY['p2_share']:.0%} of their spectral power at the 2-month period against 20% expected for noise: "
         f"with six points there is no detectable client-level cycle worth charting", s11_KEY["p2_share"], "s11_payment_spectrum", 1)

**Observations:**
- With six points a series can only show three cycle lengths: 6, 3 and 2 months. Random noise would split its power 40% / 40% / 20%.
- Payments put 18.3% of their power on the 2-month cycle, slightly below noise. Bills put 59.6% on the 6-month cycle, which is just a slow trend.
- Clients with flat (constant) payments default at 36.2% and those with a 2-month back-and-forth at 28.5%, vs 18.5% for a slow 6-month swing.
- Story use: background only. There are no real cycles to chart; use rhythm types instead.

### 11.7 What moves first? Cross-lagged panel analysis
Does utilisation rise *before* status worsens, or after? Does a falling pay ratio lead a delinquency or follow it?
Cross-lagged correlations compare `corr(X at t, Y at t+1)` with `corr(Y at t, X at t+1)`; the asymmetry hints at temporal
precedence. Four usable transitions (May->Jun ... Aug->Sep) for status, utilisation, pay ratio and spend / limit.
**Not causal**: precedence in a 6-month window with left-censoring is only suggestive, and the Aug->Sep step includes the status code regime change.

In [ ]:
s11_cl = pd.concat([pd.DataFrame({
    "ID": s11_ID, "m": _m,
    "status_t": s11_S[:, _m], "util_t": s11_U[:, _m], "payratio_t": s11_R[:, _m], "spend_t": s11_SPL[:, _m],
    "status_n": s11_S[:, _m + 1], "util_n": s11_U[:, _m + 1], "payratio_n": s11_R[:, _m + 1], "spend_n": s11_SPL[:, _m + 1]}) for _m in range(1, 5)], ignore_index=True)
S11_CLV = ["status", "util", "payratio", "spend"]
S11_CLN = {"status": "Status", "util": "Utilisation", "payratio": "Pay ratio", "spend": "Spend / limit"}

def s11_crosslag(frame):
    sp = frame[[v + "_t" for v in S11_CLV] + [v + "_n" for v in S11_CLV]].corr(method="spearman")
    M = sp.loc[[v + "_t" for v in S11_CLV], [v + "_n" for v in S11_CLV]].to_numpy()
    lead = pd.DataFrame(M, index=[S11_CLN[v] + " (t)" for v in S11_CLV], columns=[S11_CLN[v] + " (t+1)" for v in S11_CLV])
    asym = pd.DataFrame(M - M.T, index=[S11_CLN[v] for v in S11_CLV], columns=[S11_CLN[v] for v in S11_CLV])
    return lead, asym

s11_lead, s11_asym = s11_crosslag(s11_cl)
s11_lead_noSep, s11_asym_noSep = s11_crosslag(s11_cl[s11_cl["m"] <= 3])
print("Spearman corr( row at t , column at t+1 ), all four transitions pooled:"); display(s11_lead.round(3))
print("Asymmetry = corr(row_t, col_t+1) - corr(col_t, row_t+1). Positive -> the ROW variable tends to move first:"); display(s11_asym.round(3))
print("Same asymmetry without the Aug->Sep transition (status coding changes in Sep):"); display(s11_asym_noSep.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
sns.heatmap(s11_lead, annot=True, fmt=".2f", cmap=CMAP_DIV, center=0, ax=axes[0], cbar=False)
axes[0].set_title("Cross-lagged Spearman correlation (t -> t+1)")
sns.heatmap(s11_asym, annot=True, fmt="+.3f", cmap=CMAP_DIV, center=0, ax=axes[1], cbar=False)
axes[1].set_title("Lead-lag asymmetry (positive: row leads column)")
plt.tight_layout(); savefig("s11_crosslag_heatmaps"); plt.show()
s11_KEY["asym_util_status"] = float(s11_asym.loc["Utilisation", "Status"]); s11_KEY["asym_pay_status"] = float(s11_asym.loc["Pay ratio", "Status"])
s11_KEY["util_to_status"] = float(s11_lead.loc["Utilisation (t)", "Status (t+1)"]); s11_KEY["status_to_util"] = float(s11_lead.loc["Status (t)", "Utilisation (t+1)"])

**Observations:**
- Status this month predicts next month's pay ratio (-0.75) more strongly than pay ratio predicts next month's status (-0.50). The gap of 0.25 stays the same without the Aug-Sep step.
- In plain terms, being late tends to come before paying a smaller share, more than the other way round.
- Status also slightly leads utilisation (gap +0.11, or +0.07 without September).
- Story use: supports "lateness comes first, money behaviour follows". Say "tends to precede", not "causes".

**Cross-lagged regressions.** Each variable at *t+1* is regressed on all four at *t* (standardised, winsorised at 1-99%, i.e. extreme values capped at the 1st and 99th percentiles).
Controlling for a variable's own past, what does each of the others add? Pooled OLS, standard errors clustered by client.

In [ ]:
try:
    import statsmodels.api as s11_sm
    S11_HAS_SM = True
except Exception as _e:
    S11_HAS_SM = False
    print("statsmodels not available:", _e)

def s11_z(s):
    s = s.astype(float)
    lo, hi = s.quantile(0.01), s.quantile(0.99)
    s = s.clip(lo, hi)
    return (s - s.mean()) / s.std()

def s11_ols(y, X, groups=None):
    # returns (coef Series, se Series); cluster-robust when statsmodels is present, plain least squares otherwise
    if S11_HAS_SM:
        kw = {"cov_type": "cluster", "cov_kwds": {"groups": np.asarray(groups)}} if groups is not None else {}
        r = s11_sm.OLS(np.asarray(y, float), np.asarray(X, float)).fit(**kw)
        return pd.Series(r.params, index=X.columns), pd.Series(r.bse, index=X.columns), float(r.rsquared)
    beta = np.linalg.lstsq(np.asarray(X, float), np.asarray(y, float), rcond=None)[0]
    return pd.Series(beta, index=X.columns), pd.Series(np.nan, index=X.columns), np.nan

s11_clz = s11_cl.dropna().copy()
for _c in [v + s for v in S11_CLV for s in ("_t", "_n")]:
    s11_clz[_c] = s11_z(s11_clz[_c])
_X = s11_clz[[v + "_t" for v in S11_CLV]].copy(); _X.insert(0, "const", 1.0)
s11_clreg = {}
for _v in S11_CLV:
    _b, _se, _r2 = s11_ols(s11_clz[_v + "_n"], _X, s11_clz["ID"])
    s11_clreg[S11_CLN[_v] + " (t+1)"] = _b.drop("const").rename(lambda c: S11_CLN[c[:-2]] + " (t)")
    s11_clreg[S11_CLN[_v] + " (t+1)"]["R2"] = _r2
s11_clreg = pd.DataFrame(s11_clreg)
print(f"Standardised coefficients, rows = predictor at t, columns = outcome at t+1 (n = {len(s11_clz):,} client-transitions):")
display(s11_clreg.round(3))

**Observations:**
- After controlling for each variable's own past, status this month moves next month's pay ratio by -0.35 SD, while pay ratio moves next month's status by only -0.12 SD.
- Utilisation adds +0.08 SD to next month's status; spend / limit adds almost nothing (-0.01).
- Utilisation is easy to predict from its own past (R2 0.87); spend / limit is not (R2 0.04).
- Story use: background. It confirms the cross-lag reading on 99,962 client-month steps.

**Client fixed effects.** Pooled regressions mix "risky clients have high utilisation" (between) with "a client's status
worsens after *his own* utilisation rises" (within). The within transformation (demeaning every variable by client)
removes all stable client traits, including everything static (limit, demographics). Outcome: status at *t*; predictors:
utilisation, pay ratio and spend / limit at *t-1* (standardised), plus month dummies. Three estimators side by side:
pooled OLS (ordinary least-squares regression on all client-months), fixed effects (within), first differences (month-to-month changes). No lagged dependent variable (it would be biased with T this short).

In [ ]:
s11_fe = pd.concat([pd.DataFrame({
    "ID": s11_ID, "m": _m, "status": s11_S[:, _m].astype(float), "late": s11_LATE[:, _m].astype(float),
    "util_l1": s11_U[:, _m - 1], "payratio_l1": s11_R[:, _m - 1], "spend_l1": s11_SPL[:, _m - 1]}) for _m in range(2, 6)], ignore_index=True).dropna()
S11_FEX = ["util_l1", "payratio_l1", "spend_l1"]
for _c in S11_FEX:
    s11_fe[_c] = s11_z(s11_fe[_c])
_md = pd.get_dummies(s11_fe["m"], prefix="m", drop_first=True, dtype=float)
s11_fe = pd.concat([s11_fe, _md], axis=1)
_mdc = list(_md.columns)

s11_fe_rows = []
def s11_add(model, outcome, b, se):
    for c in S11_FEX:
        s11_fe_rows.append((outcome, model, c, b[c], se[c], b[c] / se[c] if se[c] and np.isfinite(se[c]) else np.nan))

for _out in ("status", "late"):
    # pooled OLS
    _X = s11_fe[S11_FEX + _mdc].copy(); _X.insert(0, "const", 1.0)
    _b, _se, _ = s11_ols(s11_fe[_out], _X, s11_fe["ID"]); s11_add("Pooled OLS", _out, _b, _se)
    # fixed effects: within transformation
    _cols = [_out] + S11_FEX + _mdc
    _dm = s11_fe[_cols] - s11_fe.groupby("ID")[_cols].transform("mean")
    _b, _se, _ = s11_ols(_dm[_out], _dm[S11_FEX + _mdc], s11_fe["ID"]); s11_add("Client fixed effects (within)", _out, _b, _se)
    # first differences
    _srt = s11_fe.sort_values(["ID", "m"])
    _d = _srt[[_out] + S11_FEX].diff()
    _ok = (_srt["ID"].values == _srt["ID"].shift().values) & (_srt["m"].values - _srt["m"].shift().values == 1)
    _d = _d[_ok]; _dX = _d[S11_FEX].copy()
    _mm = pd.get_dummies(_srt.loc[_ok, "m"], prefix="m", drop_first=True, dtype=float); _dX = pd.concat([_dX, _mm], axis=1); _dX.insert(0, "const", 1.0)
    _b, _se, _ = s11_ols(_d[_out], _dX, _srt.loc[_ok, "ID"]); s11_add("First differences", _out, _b, _se)
s11_fe_tab = pd.DataFrame(s11_fe_rows, columns=["outcome", "model", "predictor (t-1, per 1 SD)", "coef", "cluster_se", "t"])
print(f"n = {len(s11_fe):,} client-months from {s11_fe['ID'].nunique():,} clients")
display(s11_fe_tab.pivot_table(index=["outcome", "predictor (t-1, per 1 SD)"], columns="model", values="coef").round(4))
display(s11_fe_tab.pivot_table(index=["outcome", "predictor (t-1, per 1 SD)"], columns="model", values="t").round(1))
export(s11_fe_tab, "s11_panel_regressions")

fig, ax = plt.subplots(figsize=(10, 4.2))
_pt = s11_fe_tab[s11_fe_tab["outcome"] == "status"]
_models = ["Pooled OLS", "Client fixed effects (within)", "First differences"]
for _j, (_mod, _col) in enumerate(zip(_models, [C_NEU, C_DEF, C_OK])):
    _s = _pt[_pt["model"] == _mod].set_index("predictor (t-1, per 1 SD)").reindex(S11_FEX)
    ax.bar(np.arange(3) + (_j - 1) * 0.27, _s["coef"], 0.27, yerr=1.96 * _s["cluster_se"].fillna(0), color=_col, capsize=3, label=_mod)
ax.axhline(0, color="k", lw=.7); ax.set_xticks(np.arange(3)); ax.set_xticklabels(["utilisation (t-1)", "pay ratio (t-1)", "spend / limit (t-1)"])
ax.set_ylabel("change in status code per 1 SD"); ax.set_title("Status at t on lagged behaviour: between-client vs within-client effects"); ax.legend(frameon=False)
plt.tight_layout(); savefig("s11_fixed_effects"); plt.show()
_q = s11_fe_tab[s11_fe_tab["outcome"] == "status"].set_index(["model", "predictor (t-1, per 1 SD)"])["coef"]
s11_KEY["util_pooled"] = float(_q[("Pooled OLS", "util_l1")]); s11_KEY["util_fe"] = float(_q[("Client fixed effects (within)", "util_l1")])
s11_KEY["pay_pooled"] = float(_q[("Pooled OLS", "payratio_l1")]); s11_KEY["pay_fe"] = float(_q[("Client fixed effects (within)", "payratio_l1")])
add_fact("11.7", "lead-lag", f"A 1-SD higher utilisation last month goes with a {s11_KEY['util_pooled']:+.3f} status change across clients but only {s11_KEY['util_fe']:+.3f} within the same client: "
         f"utilisation mostly marks WHO is risky, not WHEN a client turns", s11_KEY["util_fe"], "s11_fixed_effects", 3)

**Observations:**
- Across all clients (pooled), 1 SD more utilisation last month goes with +0.216 in status. Comparing each client only with themselves (fixed effects), it is +0.112, and using month-to-month changes only +0.027.
- Pay ratio flips sign: across clients a higher pay ratio goes with better status (-0.431), but within a client the coefficient is +0.096.
- So most of the utilisation and pay-ratio effect shows which clients are risky, not when a client turns late.
- Story use: important caveat. Utilisation marks risky people more than risky moments.

**First-difference correlations and a direct precedence check.** (a) Correlation of month-to-month *changes*, same month
and one month ahead. (b) Among clients current in both *t-1* and *t*, compare those who turn late at *t+1* with those who
stay current: did their utilisation, pay ratio or spending already move at *t*?

In [ ]:
_dS, _dU, _dR, _dP = np.diff(s11_S, axis=1).astype(float), np.diff(s11_U, axis=1), np.diff(s11_R, axis=1), np.diff(s11_SPL, axis=1)
s11_dnames = ["d Status", "d Utilisation", "d Pay ratio", "d Spend / limit"]
_same = pd.DataFrame({n: a[:, 1:].ravel() for n, a in zip(s11_dnames, [_dS, _dU, _dR, _dP])})          # changes ending Jun..Sep
print("Same-month correlation of changes (Spearman):"); display(_same.corr(method="spearman").round(3))
_leadf = pd.DataFrame({n + " (t)": a[:, 1:-1].ravel() for n, a in zip(s11_dnames, [_dS, _dU, _dR, _dP])})
for _n, _a in zip(s11_dnames, [_dS, _dU, _dR, _dP]):
    _leadf[_n + " (t+1)"] = _a[:, 2:].ravel()
_lc = _leadf.corr(method="spearman").loc[[n + " (t)" for n in s11_dnames], [n + " (t+1)" for n in s11_dnames]]
print("Lead correlation of changes: change at t (rows) vs change at t+1 (columns):"); display(_lc.round(3))

s11_prec = []
for _m in range(1, 5):                      # t = _m, outcome at _m+1; need t-1 >= 0
    _cur = (s11_S[:, _m - 1] <= 0) & (s11_S[:, _m] <= 0)
    _turn = _cur & s11_LATE[:, _m + 1]; _stay = _cur & ~s11_LATE[:, _m + 1]
    for _lab, _mask in [("turns late at t+1", _turn), ("stays current at t+1", _stay)]:
        s11_prec.append((MONTH_ORDER[_m], _lab, int(_mask.sum()), s11_U[_mask, _m].mean(), (s11_U[_mask, _m] - s11_U[_mask, _m - 1]).mean(),
                         np.nanmean(s11_R[_mask, _m]), np.nanmean(np.clip(s11_SPL[_mask, _m], -1, 2)), (s11_P[_mask, _m] == 0).mean(), s11_Y[_mask].mean()))
s11_prec = pd.DataFrame(s11_prec, columns=["month_t", "group", "clients", "util_t", "change_in_util_t", "pay_ratio_t", "spend_over_limit_t", "share_paid_nothing_t", "default_rate"])
display(s11_prec.round(4))
export(s11_prec, "s11_precedence_before_first_late")
_pp = s11_prec.groupby("group")[["util_t", "change_in_util_t", "pay_ratio_t", "share_paid_nothing_t"]].mean()
s11_KEY["pre_dutil_turn"] = float(_pp.loc["turns late at t+1", "change_in_util_t"]); s11_KEY["pre_dutil_stay"] = float(_pp.loc["stays current at t+1", "change_in_util_t"])
s11_KEY["pre_nopay_turn"] = float(_pp.loc["turns late at t+1", "share_paid_nothing_t"]); s11_KEY["pre_nopay_stay"] = float(_pp.loc["stays current at t+1", "share_paid_nothing_t"])
add_fact("11.7", "lead-lag", f"In the month before a client first turns late, utilisation moves {s11_KEY['pre_dutil_turn']:+.3f} (vs {s11_KEY['pre_dutil_stay']:+.3f} for those staying current) and "
         f"{s11_KEY['pre_nopay_turn']:.0%} pay nothing (vs {s11_KEY['pre_nopay_stay']:.0%})", s11_KEY["pre_nopay_turn"], "s11_event_first_late", 3)

**Observations:**
- Changes in status and in utilisation in the same month are barely related (-0.03).
- A rise in status is followed by a fall in pay ratio the next month (-0.52): a late month is usually followed by a lower payment share.
- Clients who turn late next month already have higher utilisation (47% vs 29% in May; 52% vs 35% in July) and lower pay ratios (20% vs 40% in May). Their utilisation change in the month before is about the same as for others.
- August is the exception: clients turning late in September have lower utilisation (25% vs 39%) and 40% paid nothing. Many of them are the new September code-1 cases.
- Story use: the warning sign is the level, not a sudden jump. Feeds the "path to default" story.

### 11.8 What happens around the first late month? An event study
Calendar time hides the story because clients turn late in different months. Here clients are aligned on **event time**:
0 = the first month their status shows late. Only clients whose first late month is **Jun, Jul or Aug** are used, so that
at least two earlier months and one later month exist. Control group: never-late clients, re-weighted to the same
credit-limit band and the same calendar month as the treated clients. Curves are split by eventual default.

Caveat: in Apr-Aug a first delay is coded 2, so the true missed payment probably sits at event time -1 - expect "pre-trends".

In [ ]:
s11_K = np.arange(-2, 3)
S11_EV_METRICS = {
    "Utilisation (bill / limit)": s11_U,
    "Bill (NTD)": s11_B,
    "Payment (NTD)": s11_P,
    "Paid nothing (share)": (s11_P == 0).astype(float),
    "Pay ratio (payment / previous bill, capped at 2)": np.clip(s11_R, None, 2),
    "Implied spend / limit": np.clip(s11_SPL, -1, 2),
}

def s11_align_event(X, rows, ev):
    idx = ev[:, None] + s11_K[None, :]
    ok = (idx >= 0) & (idx <= 5)
    return np.where(ok, X[rows[:, None], np.clip(idx, 0, 5)], np.nan)

def s11_curve(V, w=None):
    w = np.ones(len(V)) if w is None else np.asarray(w, float)
    W = np.where(np.isnan(V), 0.0, w[:, None]); Vz = np.nan_to_num(V)
    sw = W.sum(0)
    with np.errstate(divide="ignore", invalid="ignore"):
        mean = (W * Vz).sum(0) / sw
        var = (W * (Vz - mean) ** 2).sum(0) / sw
        neff = sw ** 2 / (W ** 2).sum(0)
        se = np.sqrt(var / neff)
    return mean, se, (W > 0).sum(0)

def s11_event_curves(event, groups):
    # groups: list of (label, rows, event_index, weights or None) -> tidy curves
    out = []
    for label, rows, ev, w in groups:
        for metric, X in S11_EV_METRICS.items():
            mean, se, n = s11_curve(s11_align_event(X, rows, ev), w)
            out.append(pd.DataFrame({"event": event, "group": label, "metric": metric, "event_time": s11_K, "mean": mean,
                                     "ci_lo": mean - 1.96 * se, "ci_hi": mean + 1.96 * se, "n_obs": n}))
    return pd.concat(out, ignore_index=True)

def s11_plot_event(curves, title, name, styles):
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for ax, metric in zip(axes.ravel(), S11_EV_METRICS):
        for label, (col, ls) in styles.items():
            c = curves[(curves["metric"] == metric) & (curves["group"] == label) & (curves["n_obs"] > 0)]
            ax.plot(c["event_time"], c["mean"], marker="o", color=col, ls=ls, label=f"{label}")
            ax.fill_between(c["event_time"], c["ci_lo"], c["ci_hi"], color=col, alpha=.15)
        ax.axvline(0, color="k", ls=":", lw=1); ax.set_title(metric, fontsize=10); ax.set_xticks(s11_K); ax.set_xlabel("event time (months)")
        if "share" in metric or "Utilisation" in metric or "ratio" in metric or "limit" in metric:
            ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    axes[0, 0].legend(frameon=False, fontsize=8)
    fig.suptitle(title, fontweight="bold"); plt.tight_layout(); savefig(name); plt.show()

# ---- event 1: first late month
_band = df["LIMIT_BAND"].cat.codes.to_numpy(); _nb = int(_band.max()) + 1
_tr = np.where(np.isin(s11_first_late, [2, 3, 4]))[0]
_never_rows = np.where(s11_first_late == -1)[0]
_cr, _ce, _cw = [], [], []
for _e in (2, 3, 4):
    _tb = np.bincount(_band[s11_first_late == _e], minlength=_nb).astype(float)
    _cb = np.bincount(_band[_never_rows], minlength=_nb).astype(float)
    _cr.append(_never_rows); _ce.append(np.full(len(_never_rows), _e)); _cw.append(_tb[_band[_never_rows]] / np.where(_cb[_band[_never_rows]] > 0, _cb[_band[_never_rows]], 1))
_cr, _ce, _cw = np.concatenate(_cr), np.concatenate(_ce), np.concatenate(_cw)
_trd, _trn = _tr[s11_Y[_tr] == 1], _tr[s11_Y[_tr] == 0]
s11_ev1 = s11_event_curves("First late month", [
    ("First late, later defaults", _trd, s11_first_late[_trd], None),
    ("First late, no default", _trn, s11_first_late[_trn], None),
    ("Never late (matched control)", _cr, _ce, _cw)])
print("Treated clients by first late month:", {MONTH_ORDER[e]: int((s11_first_late == e).sum()) for e in (2, 3, 4)},
      f"| default rate of treated {s11_Y[_tr].mean():.1%} | never-late controls {len(_never_rows):,} (default {s11_Y[_never_rows].mean():.1%})")
s11_plot_event(s11_ev1, "Event study around the FIRST LATE month (event time 0), first late in Jun-Aug", "s11_event_first_late",
               {"First late, later defaults": (C_DEF, "-"), "First late, no default": (C_OK, "-"), "Never late (matched control)": (C_NEU, "--")})
for _mt in ["Utilisation (bill / limit)", "Paid nothing (share)", "Pay ratio (payment / previous bill, capped at 2)"]:
    print(_mt); display(s11_ev1[s11_ev1["metric"] == _mt].pivot(index="group", columns="event_time", values="mean").round(3))
print("Observations per event-time cell:"); display(s11_ev1[s11_ev1["metric"] == "Utilisation (bill / limit)"].pivot(index="group", columns="event_time", values="n_obs"))

**Observations:**
- 3,759 clients turn late for the first time in Jun-Aug. 39.2% of them later default, vs 11.7% of 19,931 never-late controls.
- Two months before the event, future defaulters already run 45.9% utilisation vs 35.3% for controls; at the event it is 52.8% vs 43.5%. The gap exists before the event.
- In the month after the first late month, 79.5% of future defaulters pay nothing (73.4% of those who do not default), vs 10.0% of controls. Their pay ratio drops to 1.5%.
- Story use: key "path to default" chart, a line over event time with 0 = first late month.

**The cure event.** Same alignment for the first *return to current* (status <= 0 after a late month), again for cures in
Jun-Aug. Comparison group: client-months that were late in the previous month and **stayed late**, weighted to the same
calendar months. What does a cure look like in money terms - a large payment, lower spending, or just a reclassification?

In [ ]:
_cure_m = np.full(s11_N, -1)
for _m in range(5, 0, -1):                                   # first cure month wins (loop backwards, overwrite)
    _c = s11_LATE[:, _m - 1] & (s11_S[:, _m] <= 0)
    _cure_m[_c] = _m
s11_first_cure = _cure_m
_cu = np.where(np.isin(s11_first_cure, [2, 3, 4]))[0]
_sr, _se_, _sw = [], [], []
for _e in (2, 3, 4):
    _rows = np.where(s11_LATE[:, _e - 1] & s11_LATE[:, _e])[0]
    _sr.append(_rows); _se_.append(np.full(len(_rows), _e)); _sw.append(np.full(len(_rows), (s11_first_cure == _e).sum() / max(len(_rows), 1)))
_sr, _se_, _sw = np.concatenate(_sr), np.concatenate(_se_), np.concatenate(_sw)
_cud, _cun = _cu[s11_Y[_cu] == 1], _cu[s11_Y[_cu] == 0]
s11_ev2 = s11_event_curves("First cure (return to current)", [
    ("Cured, later defaults", _cud, s11_first_cure[_cud], None),
    ("Cured, no default", _cun, s11_first_cure[_cun], None),
    ("Stayed late (comparison)", _sr, _se_, _sw)])
print("Cured clients by cure month:", {MONTH_ORDER[e]: int((s11_first_cure == e).sum()) for e in (2, 3, 4)},
      f"| default rate after cure {s11_Y[_cu].mean():.1%} | stayed-late client-months {len(_sr):,} (default {s11_Y[_sr].mean():.1%})")
s11_plot_event(s11_ev2, "Event study around the FIRST CURE (event time 0 = back to current), cures in Jun-Aug", "s11_event_cure",
               {"Cured, later defaults": (C_DEF, "-"), "Cured, no default": (C_OK, "-"), "Stayed late (comparison)": (C_NEU, "--")})
for _mt in ["Utilisation (bill / limit)", "Paid nothing (share)", "Pay ratio (payment / previous bill, capped at 2)"]:
    print(_mt); display(s11_ev2[s11_ev2["metric"] == _mt].pivot(index="group", columns="event_time", values="mean").round(3))
s11_event_long = pd.concat([s11_ev1, s11_ev2], ignore_index=True)
export(s11_event_long, "s11_event_study_curves")

def s11_evget(curves, group, metric, k):
    v = curves[(curves["group"] == group) & (curves["metric"] == metric) & (curves["event_time"] == k)]["mean"]
    return float(v.iloc[0])
_u = "Utilisation (bill / limit)"; _pn = "Paid nothing (share)"; _pr = "Pay ratio (payment / previous bill, capped at 2)"
s11_KEY["ev_util_pre"] = np.mean([s11_evget(s11_ev1, g, _u, -2) for g in ("First late, later defaults", "First late, no default")])
s11_KEY["ev_util_0"] = np.mean([s11_evget(s11_ev1, g, _u, 0) for g in ("First late, later defaults", "First late, no default")])
s11_KEY["ev_ctrl_pre"] = s11_evget(s11_ev1, "Never late (matched control)", _u, -2); s11_KEY["ev_ctrl_0"] = s11_evget(s11_ev1, "Never late (matched control)", _u, 0)
s11_KEY["ev_nopay_m1_def"] = s11_evget(s11_ev1, "First late, later defaults", _pn, -1); s11_KEY["ev_nopay_m1_ctrl"] = s11_evget(s11_ev1, "Never late (matched control)", _pn, -1)
s11_KEY["cure_pr_0"] = s11_evget(s11_ev2, "Cured, no default", _pr, 0); s11_KEY["cure_pr_stay"] = s11_evget(s11_ev2, "Stayed late (comparison)", _pr, 0)
s11_KEY["cure_default"] = float(s11_Y[_cu].mean()); s11_KEY["stay_default"] = float(s11_Y[_sr].mean())
add_fact("11.8", "event study", f"Clients about to turn late already run utilisation of about {s11_KEY['ev_util_pre']:.0%} two months before (matched never-late controls: {s11_KEY['ev_ctrl_pre']:.0%}) "
         f"and reach {s11_KEY['ev_util_0']:.0%} at the event: the level gap precedes the event, the jump is small", s11_KEY["ev_util_pre"], "s11_event_first_late", 3)
add_fact("11.8", "event study", f"A cure coincides with a pay ratio of {s11_KEY['cure_pr_0']:.0%} of the previous bill (stayed-late: {s11_KEY['cure_pr_stay']:.0%}); "
         f"cured clients still default at {s11_KEY['cure_default']:.1%} vs {s11_KEY['stay_default']:.1%} for those who stayed late", s11_KEY["cure_default"], "s11_event_cure", 2)

**Observations:**
- 2,787 clients cure (return to current) in Jun-Aug. 33.3% of them still default later, vs 61.3% for client-months that stayed late.
- In the cure month 65-67% of cured clients pay nothing and the pay ratio is about 2%. The month after, almost all pay (only 0.1-0.3% pay nothing) and the pay ratio jumps to 36-39%. The payment seems to show up one month after the status change.
- Cured clients who later default carry higher utilisation (50-53%) than cured clients who do not (44-49%).
- Story use: curing helps but does not erase the risk (33% vs 61%). Good second-act chart.

### 11.9 When do clients turn late, and when do they cure? Person-month hazard tables
The panel can be re-cut as a **person-month** table: one row per client per month *at risk*. Risk set for "first late" =
clients current in every month so far; event = status >= 1 this month. The discrete-time hazard is events / at risk.
Clients already late in April are left-truncated (their history starts before the window) and never enter. `cloglog` = log(-log(1 - hazard)), the scale on which
discrete-time hazards are proportional. (Kaplan-Meier curves and information horizons are in Section 17; here only the hazard tables.)

Remember the coding break: September is the only month where a one-month delay (code 1) is recorded, so the September
hazard at threshold >= 1 is not comparable with May-Aug; `hazard_ge2` is shown next to it.

In [ ]:
_parts = []
_static = ["LIMIT_BAND", "AGE_BAND", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL"]
for _m in range(1, 6):
    _risk = ~s11_LATE[:, :_m].any(1)
    _p = df.loc[_risk, ["ID", TARGET] + _static].reset_index(drop=True)
    _p["m"] = _m; _p["MONTH"] = MONTH_ORDER[_m]
    _p["event"] = s11_LATE[_risk, _m].astype(int); _p["event_ge2"] = (s11_S[_risk, _m] >= 2).astype(int)
    _p["prev_status"] = s11_S[_risk, _m - 1]; _p["prev_util"] = s11_U[_risk, _m - 1]; _p["prev_payratio"] = s11_R[_risk, _m - 1]
    _p["prev_paid_any"] = np.where(s11_P[_risk, _m - 1] > 0, "Paid something", "Paid nothing")
    _p["april_status"] = s11_S[_risk, 0]
    _parts.append(_p)
s11_pm = pd.concat(_parts, ignore_index=True)
s11_pm["MONTH"] = pd.Categorical(s11_pm["MONTH"], categories=MONTH_ORDER[1:], ordered=True)
_smap = {-2: "Inactive", -1: "Paid in full", 0: "Revolving"}
s11_pm["PREV_STATUS_GRP"] = pd.Categorical(s11_pm["prev_status"].map(_smap), categories=["Inactive", "Paid in full", "Revolving"])
s11_pm["APRIL_STATUS_GRP"] = pd.Categorical(s11_pm["april_status"].map(_smap), categories=["Inactive", "Paid in full", "Revolving"])
s11_pm["PREV_UTIL_BAND"] = pd.cut(s11_pm["prev_util"], [-np.inf, 0, 0.3, 0.6, 0.9, np.inf], labels=["<=0", "0-30%", "30-60%", "60-90%", ">90%"])
s11_pm["PREV_PAYRATIO_BAND"] = pd.cut(s11_pm["prev_payratio"], [-np.inf, 0, 0.05, 0.2, 0.99, np.inf], labels=["0", "0-5%", "5-20%", "20-99%", "full+"]).astype(object).fillna("n/a (no prior bill)")
print(f"Person-month table for first-late: {len(s11_pm):,} rows from {s11_pm['ID'].nunique():,} clients (current in April), {s11_pm['event'].sum():,} events")

def s11_hazard(pm, by, event="event"):
    by = [by] if isinstance(by, str) else list(by)
    g = pm.groupby(by, observed=True)[event].agg(at_risk="size", events="sum").reset_index()
    g["hazard"] = g["events"] / g["at_risk"]
    g["ci_lo"], g["ci_hi"] = wilson_ci(g["events"], g["at_risk"])
    g["cloglog"] = np.log(-np.log(1 - g["hazard"].clip(1e-9, 1 - 1e-9)))
    return g

s11_haz = s11_hazard(s11_pm, "MONTH")
s11_haz["hazard_ge2"] = s11_hazard(s11_pm, "MONTH", "event_ge2")["hazard"].values
s11_haz["survival_still_current"] = (1 - s11_haz["hazard"]).cumprod()
s11_haz["default_rate_of_events"] = [s11_pm[(s11_pm["MONTH"] == mm) & (s11_pm["event"] == 1)][TARGET].mean() for mm in s11_haz["MONTH"]]
s11_haz["default_rate_of_survivors"] = [s11_pm[(s11_pm["MONTH"] == mm) & (s11_pm["event"] == 0)][TARGET].mean() for mm in s11_haz["MONTH"]]
display(s11_haz)
export(s11_haz, "s11_hazard_first_late_by_month")
s11_KEY["haz_may_aug"] = float(s11_haz["events"].iloc[:4].sum() / s11_haz["at_risk"].iloc[:4].sum()); s11_KEY["haz_sep"] = float(s11_haz["hazard"].iloc[4])
s11_KEY["surv_end"] = float(s11_haz["survival_still_current"].iloc[-1])
add_fact("11.9", "hazard", f"Among clients current so far, the monthly hazard of a first late month is {s11_KEY['haz_may_aug']:.1%} in May-Aug and {s11_KEY['haz_sep']:.1%} in September "
         f"(when 1-month delays start being recorded); {s11_KEY['surv_end']:.0%} of April-current clients are never late", s11_KEY["haz_may_aug"], "s11_hazard_segments", 2)

**Observations:**
- Among 26,921 clients current in April, the monthly chance of a first late month (the hazard) rises from 3.2% in May to 5.5% in July, then 4.9% in August.
- September jumps to 10.6% only because 1-month delays are recorded there. Counting only 2+ month delays it is 3.2%.
- By September 74.0% of April-current clients have still never been late. 36-42% of those who turn late go on to default.
- Story use: hazard line for the timeline act, with a footnote on the September break.

Hazard by segment and month: static segments (limit band, age band, sex, education, April status) and - the real gain of
the person-month view - **time-varying** covariates measured one month earlier (previous status, utilisation band, pay-ratio band).

In [ ]:
S11_SEGVARS = {"LIMIT_BAND": "Credit limit band", "AGE_BAND": "Age band", "SEX_LABEL": "Sex", "EDU_LABEL": "Education", "MAR_LABEL": "Marital status",
               "APRIL_STATUS_GRP": "Status in April", "PREV_STATUS_GRP": "Status in previous month", "PREV_UTIL_BAND": "Utilisation in previous month",
               "PREV_PAYRATIO_BAND": "Pay ratio in previous month", "prev_paid_any": "Payment in previous month"}
_hs = []
for _v, _lab in S11_SEGVARS.items():
    _h = s11_hazard(s11_pm, [_v, "MONTH"]).rename(columns={_v: "level"}); _h["level"] = _h["level"].astype(str); _h.insert(0, "segment_variable", _lab); _hs.append(_h)
    _h = s11_hazard(s11_pm, [_v]).rename(columns={_v: "level"}); _h["level"] = _h["level"].astype(str); _h.insert(0, "segment_variable", _lab); _h["MONTH"] = "All (May-Sep)"; _hs.append(_h)
s11_haz_seg = pd.concat(_hs, ignore_index=True)
export(s11_haz_seg, "s11_hazard_first_late_by_segment")
_all = s11_haz_seg[s11_haz_seg["MONTH"] == "All (May-Sep)"]
display(_all[["segment_variable", "level", "at_risk", "events", "hazard", "ci_lo", "ci_hi", "cloglog"]].reset_index(drop=True))

fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for ax, _lab in zip(axes.ravel(), ["Credit limit band", "Age band", "Education", "Status in previous month", "Utilisation in previous month", "Pay ratio in previous month"]):
    _d = s11_haz_seg[(s11_haz_seg["segment_variable"] == _lab) & (s11_haz_seg["MONTH"] != "All (May-Sep)")]
    _pv = _d.pivot(index="level", columns="MONTH", values="hazard").reindex(columns=MONTH_ORDER[1:])
    _order = [l for l in pd.unique(s11_haz_seg[(s11_haz_seg["segment_variable"] == _lab) & (s11_haz_seg["MONTH"] == "All (May-Sep)")]["level"])]
    sns.heatmap(_pv.reindex(_order), annot=True, fmt=".1%", cmap=CMAP_RATE, ax=ax, cbar=False)
    ax.set_title(f"Hazard of first late month: {_lab}", fontsize=10); ax.set_ylabel(""); ax.set_xlabel("")
fig.suptitle("Discrete-time hazard of turning late (risk set: current in every earlier month)", fontweight="bold")
plt.tight_layout(); savefig("s11_hazard_segments"); plt.show()
_ps = _all[_all["segment_variable"] == "Status in previous month"].set_index("level")["hazard"]
s11_KEY["haz_rev"] = float(_ps.get("Revolving", np.nan)); s11_KEY["haz_full"] = float(_ps.get("Paid in full", np.nan))
add_fact("11.9", "hazard", f"A revolver's monthly hazard of turning late is {s11_KEY['haz_rev']:.1%} against {s11_KEY['haz_full']:.1%} for a client who paid in full last month",
         s11_KEY["haz_rev"], "s11_hazard_segments", 2)

**Observations:**
- The monthly chance of turning late falls steadily as the credit limit rises: 8.7% for limits up to 50k, 3.5% for 300-500k and 2.7% above 500k.
- Last month's behaviour matters: clients who paid nothing last month have a 14.0% hazard, and those above 90% utilisation 9.1%, vs 4.3% at 0-30% utilisation.
- Demographics barely matter: female 5.5% vs male 5.9%. The youngest band (21-25) is highest at 7.1%.
- Story use: a "who turns late next month" bar chart, with limit band and last month's payment as the main splits.

A descriptive discrete-time hazard model (complementary log-log link) on the person-month table, only to read the
**relative hazards net of each other**. `exp(coef)` is a hazard ratio. Not a predictive model; no validation. Skipped if statsmodels is missing.

In [ ]:
s11_hr = None
if S11_HAS_SM:
    try:
        _d = s11_pm[["event", "ID", "MONTH", "PREV_STATUS_GRP", "PREV_UTIL_BAND", "prev_paid_any", "LIMIT_BAND", "SEX_LABEL"]].copy()
        _X = pd.concat([pd.get_dummies(_d[c].astype(str), prefix=c, dtype=float) for c in ["MONTH", "PREV_STATUS_GRP", "PREV_UTIL_BAND", "prev_paid_any", "LIMIT_BAND", "SEX_LABEL"]], axis=1)
        _ref = ["MONTH_May", "PREV_STATUS_GRP_Paid in full", "PREV_UTIL_BAND_0-30%", "prev_paid_any_Paid something", "LIMIT_BAND_100-150k", "SEX_LABEL_Female"]
        _X = _X.drop(columns=[c for c in _ref if c in _X.columns]); _X.insert(0, "const", 1.0)
        _links = s11_sm.families.links
        _link = _links.CLogLog() if hasattr(_links, "CLogLog") else _links.cloglog()
        _fit = s11_sm.GLM(_d["event"].astype(float), _X, family=s11_sm.families.Binomial(link=_link)).fit(cov_type="cluster", cov_kwds={"groups": _d["ID"].to_numpy()})
        s11_hr = pd.DataFrame({"term": _X.columns, "coef": np.asarray(_fit.params), "se": np.asarray(_fit.bse)})
        s11_hr["hazard_ratio"] = np.exp(s11_hr["coef"]); s11_hr["hr_lo"] = np.exp(s11_hr["coef"] - 1.96 * s11_hr["se"]); s11_hr["hr_hi"] = np.exp(s11_hr["coef"] + 1.96 * s11_hr["se"])
        s11_hr = s11_hr[s11_hr["term"] != "const"].reset_index(drop=True)
        print("Reference levels:", ", ".join(_ref))
        display(s11_hr.round(3))
        export(s11_hr, "s11_hazard_ratios_cloglog")
        fig, ax = plt.subplots(figsize=(9, 7))
        _yy = np.arange(len(s11_hr))[::-1]
        ax.errorbar(s11_hr["hazard_ratio"], _yy, xerr=[s11_hr["hazard_ratio"] - s11_hr["hr_lo"], s11_hr["hr_hi"] - s11_hr["hazard_ratio"]], fmt="o", color=C_DEF, capsize=3)
        ax.axvline(1, color="k", ls="--", lw=1); ax.set_xscale("log"); ax.set_yticks(_yy); ax.set_yticklabels(s11_hr["term"], fontsize=8)
        ax.set_xlabel("hazard ratio (log scale)"); ax.set_title("Discrete-time hazard of first late month: hazard ratios (cloglog, descriptive)")
        plt.tight_layout(); savefig("s11_hazard_ratios"); plt.show()
    except Exception as _e:
        print("cloglog model skipped:", repr(_e)[:300])
else:
    print("statsmodels missing - hazard-ratio model skipped (tables above are complete without it).")

**Observations:**
- A hazard ratio says how many times higher the monthly risk is, holding the other factors fixed. A limit up to 50k gives 1.62 vs the 100-150k reference; a limit above 500k gives 0.56.
- Utilisation above 90% last month gives 1.69, and paying nothing last month gives 1.61.
- Revolving last month gives a lower risk (0.81) than paying in full. Sex has no effect (1.02, interval includes 1).
- The September dummy is 3.68, which reflects the coding break, not a real shock.
- Story use: supports the segment chart in a "what raises next-month risk" panel.

**Cure hazard.** Risk set = client-months that were late in the previous month; event = back to current (status <= 0).
New angle vs the roll rates of Section 5: **duration dependence** - does the chance of curing fall with every extra month
spent late? Duration is counted inside the window, so spells already open in April are flagged as left-censored (true duration is longer).

In [ ]:
_parts = []
for _m in range(1, 6):
    _rows = np.where(s11_LATE[:, _m - 1])[0]
    _dur = np.zeros(len(_rows), int); _alive = np.ones(len(_rows), bool)
    for _j in range(_m - 1, -1, -1):
        _alive &= s11_LATE[_rows, _j]; _dur += _alive
    _parts.append(pd.DataFrame({"ID": s11_ID[_rows], "m": _m, "MONTH": MONTH_ORDER[_m], "event": (s11_S[_rows, _m] <= 0).astype(int),
                                "prev_status": s11_S[_rows, _m - 1], "months_late_so_far": _dur, "left_censored": _dur == _m,
                                "prev_paid_any": np.where(s11_P[_rows, _m - 1] > 0, "Paid something", "Paid nothing"), TARGET: s11_Y[_rows]}))
s11_pmc = pd.concat(_parts, ignore_index=True)
s11_pmc["MONTH"] = pd.Categorical(s11_pmc["MONTH"], categories=MONTH_ORDER[1:], ordered=True)
s11_pmc["PREV_DEPTH"] = pd.Categorical(np.select([s11_pmc["prev_status"] == 1, s11_pmc["prev_status"] == 2, s11_pmc["prev_status"] == 3], ["1 month", "2 months", "3 months"], "4+ months"),
                                       categories=["1 month", "2 months", "3 months", "4+ months"])
s11_pmc["DURATION"] = np.where(s11_pmc["months_late_so_far"] >= 4, "4-5", s11_pmc["months_late_so_far"].astype(str))
s11_pmc["SPELL_START"] = np.where(s11_pmc["left_censored"], "open since April or earlier", "started inside window")
print(f"Person-month table for cure: {len(s11_pmc):,} late client-months, {s11_pmc['event'].sum():,} cures")
_tabs = []
for _v, _lab in [("MONTH", "Calendar month"), ("PREV_DEPTH", "Delay depth in previous month"), ("DURATION", "Months late so far (in window)"),
                 ("SPELL_START", "Spell start"), ("prev_paid_any", "Payment in previous month")]:
    _h = s11_hazard(s11_pmc, _v).rename(columns={_v: "level", "hazard": "cure_hazard"}); _h["level"] = _h["level"].astype(str); _h.insert(0, "segment_variable", _lab)
    _dd = s11_pmc.groupby(_v, observed=True)[TARGET].mean(); _h["default_rate_of_risk_set"] = _h["level"].map({str(k): v for k, v in _dd.items()})
    _tabs.append(_h)
s11_cure_haz = pd.concat(_tabs, ignore_index=True)
display(s11_cure_haz[["segment_variable", "level", "at_risk", "events", "cure_hazard", "ci_lo", "ci_hi", "cloglog", "default_rate_of_risk_set"]])
_dd2 = s11_hazard(s11_pmc, ["DURATION", "SPELL_START"]).rename(columns={"hazard": "cure_hazard"})
display(_dd2.pivot(index="DURATION", columns="SPELL_START", values="cure_hazard").round(4))
export(s11_cure_haz, "s11_hazard_cure")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for ax, _lab in zip(axes, ["Calendar month", "Months late so far (in window)", "Delay depth in previous month"]):
    _d = s11_cure_haz[s11_cure_haz["segment_variable"] == _lab]
    ax.bar(_d["level"], _d["cure_hazard"], yerr=[_d["cure_hazard"] - _d["ci_lo"], _d["ci_hi"] - _d["cure_hazard"]], color=C_ACC, capsize=3)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title(f"Cure hazard by {_lab.lower()}", fontsize=10)
plt.tight_layout(); savefig("s11_cure_hazard"); plt.show()
_cd = s11_cure_haz[s11_cure_haz["segment_variable"] == "Months late so far (in window)"].set_index("level")["cure_hazard"]
s11_KEY["cure_d1"] = float(_cd.get("1", np.nan)); s11_KEY["cure_d4"] = float(_cd.get("4-5", np.nan))
add_fact("11.9", "hazard", f"The monthly chance of curing falls from {s11_KEY['cure_d1']:.0%} after one late month to {s11_KEY['cure_d4']:.0%} after four or more: delinquency hardens with duration",
         s11_KEY["cure_d1"], "s11_cure_hazard", 3)

**Observations:**
- The monthly chance of curing falls with time spent late: 36.0% after one late month, 19.1% after two, 13.0% after three, 6.9% after four or five.
- Deeper delays cure less: 25.3% at 2 months behind, 15.9% at 3 and 3.1% at 4 or more.
- Paying something last month nearly doubles the cure chance (27.9% vs 15.2%). Late spells that began before April cure at 17.6%, vs 31.2% for new ones.
- Story use: strong "delinquency hardens" chart. The longer a client stays late, the less likely they come back.

### 11.10 Do accounts enter or leave the panel? Attrition-like checks
The panel is balanced on paper (every client has six rows), but an account with a zero bill is economically absent.
"Active" = positive statement balance. Entries (inactive -> active) and exits (active -> inactive) by month show whether
the population behind the charts changes over time, and whether late arrivals behave differently.
New columns: `PN_ACTIVITY_PATTERN`, `PN_FIRST_ACTIVE_MONTH`.

In [ ]:
s11_act = s11_B > 0
s11_ee = []
for _m in range(6):
    _r = {"MONTH": MONTH_ORDER[_m], "active_accounts": int(s11_act[:, _m].sum()), "active_share": s11_act[:, _m].mean(),
          "zero_bill": int((s11_B[:, _m] == 0).sum()), "negative_bill_credit_balance": int((s11_B[:, _m] < 0).sum()),
          "default_rate_active": s11_Y[s11_act[:, _m]].mean(), "default_rate_inactive": s11_Y[~s11_act[:, _m]].mean()}
    if _m > 0:
        _en = s11_act[:, _m] & ~s11_act[:, _m - 1]; _ex = ~s11_act[:, _m] & s11_act[:, _m - 1]
        _r.update({"entries": int(_en.sum()), "exits": int(_ex.sum()), "net": int(_en.sum() - _ex.sum()),
                   "default_rate_entrants": s11_Y[_en].mean(), "default_rate_exiters": s11_Y[_ex].mean(),
                   "exits_with_status_paid_or_inactive": (s11_S[_ex, _m] <= -1).mean()})
    s11_ee.append(_r)
s11_ee = pd.DataFrame(s11_ee)
display(s11_ee)
export(s11_ee, "s11_entry_exit_by_month")

_nact = s11_act.sum(1); _aruns = 1 + (s11_act[:, 1:] != s11_act[:, :-1]).sum(1)
S11_ACT_ORDER = ["Always active", "Activated during window", "Went dormant during window", "Intermittent", "Never active"]
_pat = np.select([_nact == 6, _nact == 0, (_aruns == 2) & s11_act[:, 5], (_aruns == 2) & s11_act[:, 0]],
                 ["Always active", "Never active", "Activated during window", "Went dormant during window"], default="Intermittent")
df["PN_ACTIVITY_PATTERN"] = pd.Categorical(_pat, categories=S11_ACT_ORDER)
_fa = np.where(s11_act.any(1), np.array(MONTH_ORDER)[s11_act.argmax(1)], "Never active")
df["PN_FIRST_ACTIVE_MONTH"] = pd.Categorical(_fa, categories=MONTH_ORDER + ["Never active"])
s11_act_rates = rate_table(df, "PN_ACTIVITY_PATTERN")
s11_fa_rates = rate_table(df, "PN_FIRST_ACTIVE_MONTH")
_la = np.where(s11_act.any(1), np.array(MONTH_ORDER)[5 - s11_act[:, ::-1].argmax(1)], "Never active")
s11_la_rates = rate_table(df.assign(_la=pd.Categorical(_la, categories=MONTH_ORDER + ["Never active"])), "_la").rename(columns={"_la": "last_active_month"})
display(s11_act_rates[["PN_ACTIVITY_PATTERN", "n", "share_clients", "rate", "ci_lo", "ci_hi", "lift"]])
display(s11_fa_rates[["PN_FIRST_ACTIVE_MONTH", "n", "share_clients", "rate", "ci_lo", "ci_hi", "lift"]])
display(s11_la_rates[["last_active_month", "n", "share_clients", "rate", "ci_lo", "ci_hi", "lift"]])
export(s11_act_rates, "s11_activity_pattern_rates")
export(s11_fa_rates, "s11_first_active_month_rates")

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
_x = np.arange(1, 6)
axes[0].bar(_x - 0.2, s11_ee["entries"].iloc[1:], 0.4, color=C_ACC, label="entries (inactive -> active)")
axes[0].bar(_x + 0.2, s11_ee["exits"].iloc[1:], 0.4, color=C_NEU, label="exits (active -> inactive)")
axes[0].set_xticks(_x); axes[0].set_xticklabels(MONTH_ORDER[1:]); axes[0].set_title("Accounts entering / leaving the active set"); axes[0].legend(frameon=False)
plot_rate(df, "PN_ACTIVITY_PATTERN", ax=axes[1], order=S11_ACT_ORDER, title="Default rate by activity pattern", rotate=20)
plot_rate(df, "PN_FIRST_ACTIVE_MONTH", ax=axes[2], order=MONTH_ORDER + ["Never active"], title="Default rate by first active month", rotate=20)
plt.tight_layout(); savefig("s11_entry_exit"); plt.show()
_ar = s11_act_rates.set_index("PN_ACTIVITY_PATTERN")
s11_KEY["always_active_share"] = float(_ar.loc["Always active", "share_clients"]); s11_KEY["never_active_rate"] = float(_ar.loc["Never active", "rate"]) if "Never active" in _ar.index else np.nan
s11_KEY["activated_rate"] = float(_ar.loc["Activated during window", "rate"]); s11_KEY["dormant_rate"] = float(_ar.loc["Went dormant during window", "rate"])
add_fact("11.10", "data structure", f"Only {s11_KEY['always_active_share']:.0%} of clients carry a positive bill in all six months; accounts that activate inside the window default at "
         f"{s11_KEY['activated_rate']:.1%}, those that go dormant at {s11_KEY['dormant_rate']:.1%}, never-active ones at {s11_KEY['never_active_rate']:.1%}",
         s11_KEY["always_active_share"], "s11_entry_exit", 2)

**Observations:**
- The share of active accounts (positive bill) grows from 84.3% in April to 91.3% in September. Each month about 1,250-1,490 accounts become active and 800-1,000 go quiet.
- Active and inactive accounts default at similar rates (about 22% vs 23-25%).
- 74.6% of clients are active all six months. The 966 never-active clients (3.2%) default at 35.9%; accounts that become active during the window default at 20.9%.
- Story use: caveat on denominators (all clients vs active accounts), plus a side note that "no bill at all" is a risk flag.

### 11.11 How much does history add beyond the latest snapshot?
The snapshot analyst would keep only September. The panel analyst can also use the 6-month average, the maximum, the
trend (slope) and the volatility. For each family (status, bill, payment, utilisation, pay ratio, spend) we compare the
**Information Value** and the single-variable AUC of each view, then ask how much a family's *history* (mean + slope +
volatility) lifts a descriptive in-sample AUC over the September value alone. (Month-by-month horizon AUCs are in Section 17.)

In [ ]:
def s11_slope(X):
    X = np.asarray(X, float); T = X.shape[1]; tt = np.arange(T, dtype=float)
    ok = ~np.isnan(X); n = ok.sum(1); nn = np.maximum(n, 1)
    tm = np.where(ok, tt, 0).sum(1) / nn; xm = np.nansum(X, 1) / nn
    num = np.where(ok, (tt - tm[:, None]) * (np.nan_to_num(X) - xm[:, None]), 0).sum(1)
    den = np.where(ok, (tt - tm[:, None]) ** 2, 0).sum(1)
    return np.where((n >= 3) & (den > 0), num / np.where(den > 0, den, 1), np.nan)

def s11_bin_codes(x, bins=10):
    s = pd.Series(np.asarray(x, float))
    if s.nunique() > bins:
        return pd.qcut(s, bins, duplicates="drop").cat.codes.to_numpy()          # NaN -> -1 (own bin)
    return (s.rank(method="dense").fillna(0).astype(int) - 1).to_numpy()

def s11_iv(x, y, eps=0.5):
    ct = pd.crosstab(s11_bin_codes(x), y)
    g, b = ct[0], ct[1]
    dg, db = (g + eps) / (g.sum() + eps), (b + eps) / (b.sum() + eps)
    return float(((dg - db) * np.log(dg / db)).sum())

def s11_auc(x, y):
    x = np.asarray(x, float); ok = ~np.isnan(x)
    r = stats.rankdata(x[ok]); yy = y[ok]; n1 = yy.sum(); n0 = len(yy) - n1
    a = (r[yy == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)
    return float(max(a, 1 - a))

S11_FAMS = {"Status": s11_S.astype(float), "Bill": s11_B, "Payment": s11_P, "Utilisation": s11_U, "Pay ratio": s11_R[:, 1:], "Spend / limit": s11_SPL[:, 1:]}
S11_VIEWS = ["Sep snapshot", "Aug snapshot", "Earliest month", "Window mean", "Window max", "Window min", "Slope", "Volatility (std)"]
s11_views = {}
for _f, _M in S11_FAMS.items():
    s11_views[_f] = {"Sep snapshot": _M[:, -1], "Aug snapshot": _M[:, -2], "Earliest month": _M[:, 0], "Window mean": np.nanmean(_M, axis=1),
                     "Window max": np.nanmax(np.where(np.isnan(_M), -np.inf, _M), axis=1), "Window min": np.nanmin(np.where(np.isnan(_M), np.inf, _M), axis=1),
                     "Slope": s11_slope(_M), "Volatility (std)": np.nanstd(_M, axis=1)}
    for _k in ("Window max", "Window min"):
        s11_views[_f][_k] = np.where(np.isfinite(s11_views[_f][_k]), s11_views[_f][_k], np.nan)
s11_svh = pd.DataFrame([{"family": f, "view": v, "iv": s11_iv(x, s11_Y), "auc": s11_auc(x, s11_Y), "share_missing": float(np.isnan(x).mean())}
                        for f, vs in s11_views.items() for v, x in vs.items()])
display(s11_svh.pivot(index="family", columns="view", values="iv").reindex(index=list(S11_FAMS), columns=S11_VIEWS).round(3))
display(s11_svh.pivot(index="family", columns="view", values="auc").reindex(index=list(S11_FAMS), columns=S11_VIEWS).round(3))

**Observations:**
- IV (information value) measures how well one column separates defaulters from others. For status, the September value alone has IV 0.87, more than the six-month mean or maximum (0.78 each).
- For payments and pay ratio, history helps more: the six-month minimum pay ratio has IV 0.43 vs 0.26 for September alone, and the minimum payment 0.28 vs 0.16.
- Bills carry almost nothing in any view (IV 0.01-0.15).
- Story use: sets the story order: lead with September status, then add payment history.

In [ ]:
s11_up = []
try:
    from sklearn.linear_model import LogisticRegression as s11_LR
    from sklearn.metrics import roc_auc_score as s11_roc
    def s11_fit_auc(cols):
        X = pd.concat([pd.get_dummies(pd.Series(s11_bin_codes(x)), prefix=f"v{j}", dtype=float) for j, x in enumerate(cols)], axis=1).to_numpy()
        mdl = s11_LR(max_iter=400).fit(X, s11_Y)
        return float(s11_roc(s11_Y, mdl.predict_proba(X)[:, 1]))
    for _f, _vs in s11_views.items():
        _a_snap = s11_fit_auc([_vs["Sep snapshot"]])
        _a_mean = s11_fit_auc([_vs["Window mean"]])
        _a_hist = s11_fit_auc([_vs["Sep snapshot"], _vs["Window mean"], _vs["Slope"], _vs["Volatility (std)"]])
        s11_up.append((_f, _a_snap, _a_mean, _a_hist, _a_hist - _a_snap))
    _a_all_snap = s11_fit_auc([vs["Sep snapshot"] for vs in s11_views.values()])
    _a_all_hist = s11_fit_auc([vs[v] for vs in s11_views.values() for v in ("Sep snapshot", "Window mean", "Slope", "Volatility (std)")])
    s11_up.append(("ALL SIX FAMILIES", _a_all_snap, np.nan, _a_all_hist, _a_all_hist - _a_all_snap))
except Exception as _e:
    print("scikit-learn step skipped:", repr(_e)[:200])
s11_up = pd.DataFrame(s11_up, columns=["family", "auc_sep_snapshot_only", "auc_window_mean_only", "auc_snapshot_plus_history", "history_uplift"])
display(s11_up.round(4))
s11_svh_x = s11_svh.merge(s11_up, on="family", how="left")
export(s11_svh_x, "s11_snapshot_vs_history")

fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={"width_ratios": [1.6, 1]})
_pv = s11_svh.pivot(index="family", columns="view", values="iv").reindex(index=list(S11_FAMS), columns=["Earliest month", "Aug snapshot", "Sep snapshot", "Window mean", "Window max", "Slope", "Volatility (std)"])
_pv.plot(kind="bar", ax=axes[0], width=0.85, colormap="viridis")
axes[0].set_title("Information Value of each view of a family: snapshot vs history"); axes[0].set_ylabel("IV"); axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0); axes[0].legend(frameon=False, ncol=4, fontsize=8)
if len(s11_up):
    _yy = np.arange(len(s11_up))[::-1]
    axes[1].barh(_yy, s11_up["auc_snapshot_plus_history"], color=C_ACC, label="Sep + mean + slope + volatility")
    axes[1].barh(_yy, s11_up["auc_sep_snapshot_only"], color=C_NEU, label="Sep snapshot only")
    axes[1].set_yticks(_yy); axes[1].set_yticklabels(s11_up["family"]); axes[1].set_xlim(0.5, None)
    axes[1].set_xlabel("in-sample AUC (descriptive)"); axes[1].set_title("How much does history add?"); axes[1].legend(frameon=False, fontsize=8, loc="lower right")
plt.tight_layout(); savefig("s11_snapshot_vs_history"); plt.show()

_iv = s11_svh.set_index(["family", "view"])["iv"]
s11_KEY["iv_status_sep"] = float(_iv[("Status", "Sep snapshot")]); s11_KEY["iv_status_mean"] = float(_iv[("Status", "Window mean")]); s11_KEY["iv_status_max"] = float(_iv[("Status", "Window max")])
s11_KEY["iv_pay_sep"] = float(_iv[("Payment", "Sep snapshot")]); s11_KEY["iv_pay_mean"] = float(_iv[("Payment", "Window mean")])
s11_KEY["iv_util_sep"] = float(_iv[("Utilisation", "Sep snapshot")]); s11_KEY["iv_util_mean"] = float(_iv[("Utilisation", "Window mean")])
add_fact("11.11", "snapshot vs history", f"For status, the September snapshot alone carries IV {s11_KEY['iv_status_sep']:.2f} vs {s11_KEY['iv_status_mean']:.2f} for the 6-month mean; "
         f"for payments the 6-month mean ({s11_KEY['iv_pay_mean']:.2f}) beats the September value ({s11_KEY['iv_pay_sep']:.2f}): noisy flows need averaging, sticky states do not",
         s11_KEY["iv_status_sep"], "s11_snapshot_vs_history", 3)
if len(s11_up):
    _al = s11_up.set_index("family").loc["ALL SIX FAMILIES"]
    s11_KEY["auc_all_snap"] = float(_al["auc_sep_snapshot_only"]); s11_KEY["auc_all_hist"] = float(_al["auc_snapshot_plus_history"])
    add_fact("11.11", "snapshot vs history", f"Across all six families, adding history to the September snapshot lifts a descriptive AUC from {s11_KEY['auc_all_snap']:.3f} to {s11_KEY['auc_all_hist']:.3f}: "
             f"the panel adds depth, the snapshot carries most of the signal", s11_KEY["auc_all_hist"], "s11_snapshot_vs_history", 3)

**Observations:**
- AUC runs from 0.5 (coin toss) to 1 (perfect separation). Adding history to the September value raises it for every family: status 0.711 to 0.748, bills 0.531 to 0.636, utilisation 0.578 to 0.642.
- With all six families together, the AUC goes from 0.766 (September only) to 0.791 with history, a gain of 0.025.
- Story use: the snapshot carries most of the signal and history adds depth. One line in the story, not a chart.

A concrete reading of "history beyond the snapshot": hold the September status fixed and split by how many of the
**five earlier months** were late. If the snapshot were sufficient, the rows of each column would be flat.

In [ ]:
_prior = s11_LATE[:, :5].sum(1)
_pb = pd.Categorical(np.select([_prior == 0, _prior == 1, _prior == 2], ["0", "1", "2"], "3-5"), categories=["0", "1", "2", "3-5"])
_tmp = df.assign(_prior_late=_pb)
s11_cond = rate_table(_tmp, ["STATUS_GRP_SEP", "_prior_late"]).rename(columns={"_prior_late": "late_months_apr_aug"})
_pv_r = s11_cond.pivot(index="late_months_apr_aug", columns="STATUS_GRP_SEP", values="rate").reindex(columns=[c for c in STATUS_GROUP_ORDER if c in set(s11_cond["STATUS_GRP_SEP"])])
_pv_n = s11_cond.pivot(index="late_months_apr_aug", columns="STATUS_GRP_SEP", values="n").reindex(columns=_pv_r.columns)
fig, ax = plt.subplots(figsize=(10, 3.8))
sns.heatmap(_pv_r, annot=_pv_r.applymap(lambda v: "" if pd.isna(v) else f"{v:.0%}") if False else True, fmt=".0%", cmap=CMAP_RATE, ax=ax, cbar=False)
ax.set_title("Default rate: September status (snapshot) x late months in Apr-Aug (history)"); ax.set_xlabel("status in September"); ax.set_ylabel("late months Apr-Aug")
plt.tight_layout(); savefig("s11_snapshot_x_history"); plt.show()
display(_pv_r.round(3)); display(_pv_n)
export(s11_cond[["STATUS_GRP_SEP", "late_months_apr_aug", "n", "defaults", "rate", "ci_lo", "ci_hi", "lift"]], "s11_sep_status_x_prior_history")
_cs = s11_cond[s11_cond["STATUS_GRP_SEP"] == "Revolving"].set_index("late_months_apr_aug")["rate"]
s11_KEY["rev_clean"] = float(_cs.get("0", np.nan)); s11_KEY["rev_dirty"] = float(_cs.get("3-5", np.nan))
add_fact("11.11", "snapshot vs history", f"Two clients both 'revolving' in September: with a clean Apr-Aug record the default rate is {s11_KEY['rev_clean']:.1%}, "
         f"with 3-5 earlier late months it is {s11_KEY['rev_dirty']:.1%}: the snapshot hides the scar", s11_KEY["rev_dirty"], "s11_snapshot_x_history", 3)

**Observations:**
- Among clients revolving in September, default is 10.7% with a clean Apr-Aug record and 37.7% with 3-5 earlier late months.
- The same holds for paid-in-full in September (13.8% vs 45.4%) and for 1 month late (25.1% vs 51.4%).
- Story use: strong chart "same status today, different past", a heatmap of September status by number of earlier late months.

### 11.12 Verdict: what this data is and is not, and what that means for the Tableau story
Every row below is filled from numbers computed in this section.

In [ ]:
def s11_f(key, fmt="{:.1%}"):
    v = s11_KEY.get(key, np.nan)
    try:
        return fmt.format(v) if v is not None and np.isfinite(v) else "n/a"
    except Exception:
        return str(v)

s11_verdict = pd.DataFrame([
    ("Grain", "Cross-section: one row per client", f"{s11_N:,} rows, {df['ID'].nunique():,} IDs, no duplicates",
     "Count clients with COUNTD(ID); never sum client-level fields after pivoting to long"),
    ("Embedded panel", "Yes: balanced, 6 monthly periods x 3 series", f"{s11_N * 6:,} client-months, balanced = {s11_KEY.get('balanced')}; derived ratios exist for 5 periods",
     "Pivot the 18 time columns to long (client x month) as a second Tableau source related on ID"),
    ("Months are a true sequence", "Yes, same account, ordered", f"payment = previous bill exactly in {s11_f('exact_true')} of client-months vs {s11_f('exact_other', '{:.3%}')} for another client; "
     f"within-client bill autocorr {s11_f('ac_obs', '{:.2f}')} vs {s11_f('ac_null', '{:.2f}')} scrambled",
     "Line charts over Apr-Sep are legitimate; use a discrete, ordered month axis (not a continuous date)"),
    ("Outcome", "Snapshot only: one binary flag for Oct 2005", "no default date, no outcome history, right-censored",
     "Default cannot be put on a time axis; use it as colour / split, never as a monthly series"),
    ("Limit and demographics", "Snapshot only, undated", f"over-limit share {s11_f('over_apr')} in April vs {s11_f('over_sep')} in September with one limit per client",
     "Label monthly utilisation as 'bill / current limit'; avoid claims about limit changes"),
    ("Left / right censoring", "Both", f"{s11_f('left_cens')} already late in April; {s11_f('sep_late')} late in September (spell open)",
     "Onset and duration charts must exclude or flag April-late clients"),
    ("How periodic is it?", "Levels are client traits, flows move within client",
     f"ICC bill {s11_f('icc_bill', '{:.2f}')}, status {s11_f('icc_status', '{:.2f}')}, payment {s11_f('icc_pay', '{:.2f}')}, spend {s11_f('icc_spend', '{:.2f}')}",
     "Show levels as client segments (bars / heatmaps); show payments and spend as monthly trends"),
    ("Calendar / seasonality", "Not identifiable: six points, one year", f"month effect explains at most {s11_f('month_share_max')} of variance; billed total {s11_f('bill_growth', '{:+.1%}')} Apr->Sep",
     "No seasonality claims; annotate the timeline with the 2005-06 Taiwan card-crisis context instead"),
    ("Status series break", "Code 1 recorded (almost) only in September", f"first-late hazard {s11_f('haz_may_aug')} in May-Aug vs {s11_f('haz_sep')} in Sep",
     "Use status >= 2 for any month-over-month late-share line; footnote the September break"),
    ("Client-level cycles", "Not detectable with T = 6", f"payment power at period 2: {s11_f('p2_share', '{:.0%}')} vs 20% for noise",
     "Use rhythm TYPES (PN_RHYTHM) as a categorical dimension instead of frequency charts"),
    ("Lead-lag", "Suggestive only", f"lagged utilisation effect on status: pooled {s11_f('util_pooled', '{:+.3f}')} vs within-client {s11_f('util_fe', '{:+.3f}')} per SD",
     "Phrase as 'tends to precede', never 'causes'; event-time charts (t-2..t+2) are the honest visual"),
    ("Snapshot vs history", "Snapshot carries most signal, history adds a scar", f"status IV Sep {s11_f('iv_status_sep', '{:.2f}')} vs mean {s11_f('iv_status_mean', '{:.2f}')}; "
     f"all-family AUC {s11_f('auc_all_snap', '{:.3f}')} -> {s11_f('auc_all_hist', '{:.3f}')} with history",
     "Lead the story with September status; use history as the second act (same status, different past)"),
    ("Effective population", "Changes inside the window", f"only {s11_f('always_active_share', '{:.0%}')} of clients have a positive bill in all six months",
     "Monthly averages should state the denominator (all clients vs active accounts)"),
], columns=["aspect", "verdict", "evidence_from_this_section", "implication_for_tableau"])
pd.set_option("display.max_colwidth", 200)
display(s11_verdict)
pd.reset_option("display.max_colwidth")
export(s11_verdict, "s11_structure_verdict")

s11_axes = pd.DataFrame([
    ("Calendar month (Apr..Sep, discrete, ordered)", "portfolio totals, status mix, hazards", "legitimate", "6 points; no seasonality; Sep status break"),
    ("Event time (months relative to first late / cure)", "behaviour around a turning point", "legitimate", "restricted to events in Jun-Aug; unequal window lengths"),
    ("Months late so far (duration)", "cure hazard, hardening of delinquency", "legitimate with flag", "spells open in April are left-censored"),
    ("Lag in months (1..5)", "persistence, lead-lag", "legitimate", "pooled over calendar months"),
    ("Continuous date axis / day level", "anything", "NOT legitimate", "no dates inside months exist"),
    ("Time axis for DEFAULT", "default 'trend'", "NOT legitimate", "single outcome month"),
    ("Client age / account age as time", "lifecycle", "cross-sectional only", "AGE is a cohort comparison, not ageing; account age unknown"),
], columns=["time_axis", "good_for", "status", "caveat"])
display(s11_axes)
export(s11_axes, "s11_recommended_time_axes")

**Observations:**
- The file is a cross-section of 30,000 clients with a balanced six-month panel inside. Payments match the previous bill in 16.2% of cases vs 0.025% for a random client, so the months are one ledger.
- Levels behave like client traits (bill ICC 0.88); flows behave like monthly behaviour (payments 0.19, spend 0.18).
- Default is a single month, so it can never be a time series, and six months cannot reveal seasonality.
- Status code 1 is recorded almost only in September, so month-over-month late-share lines should use status 2 or more.
- Story use: the methods / "how to read this dashboard" page. It sets the allowed time axes.

**Bottom line.** The file is a *snapshot with a memory*: a cross-section of 30,000 clients observed once for the outcome,
each carrying a six-month ledger. The snapshot view answers "who is risky?"; the panel view answers "what did the months
before look like, and in what order did things move?". It cannot answer "when did they default?" or "is there a season?".

**New `df` columns added in Section 11**

| Column | Meaning |
|---|---|
| `PN_N_PAY_MONTHS` | number of months (0-6) with a positive payment |
| `PN_PAY_RUNS` | number of runs in the paid / not-paid sequence (1 = constant, 6 = alternating) |
| `PN_RHYTHM` | payment rhythm type (Every month, Misses one month, Alternating, One block, Sporadic, One lump, Never pays) |
| `PN_BILL_AC1`, `PN_PAY_AC1` | within-client lag-1 autocorrelation of bills / payments (NaN if constant) |
| `PN_DOM_PERIOD` | dominant spectral period of the client's payment series (6, 3 or 2 months, or flat) |
| `PN_ACTIVITY_PATTERN` | Always active / Activated / Went dormant / Intermittent / Never active (bill > 0) |
| `PN_FIRST_ACTIVE_MONTH` | first month with a positive bill |

Also defined (not in `df`): `s11_long` (client-month table), `s11_pm` / `s11_pmc` (person-month hazard tables for first late / cure),
`s11_event_long` (event-study curves), `s11_monthly`, `s11_icc_tab`, `s11_verdict`.

**Tableau exports from this section:** `s11_icc_variance_decomposition`, `s11_monthly_aggregate`, `s11_rhythm_type_rates`,
`s11_zigzag_rates`, `s11_dominant_period_rates`, `s11_panel_regressions`, `s11_precedence_before_first_late`,
`s11_event_study_curves`, `s11_hazard_first_late_by_month`, `s11_hazard_first_late_by_segment`, `s11_hazard_ratios_cloglog`,
`s11_hazard_cure`, `s11_entry_exit_by_month`, `s11_activity_pattern_rates`, `s11_first_active_month_rates`,
`s11_snapshot_vs_history`, `s11_sep_status_x_prior_history`, `s11_structure_verdict`, `s11_recommended_time_axes`.

In [ ]:
s11_facts = pd.DataFrame([f for f in STORY_FACTS if f["section"].startswith("11")])
pd.set_option("display.max_colwidth", 250)
display(s11_facts)
pd.reset_option("display.max_colwidth")
print(f"Section 11 runtime: {time.time() - s11_t0:.0f}s")

## 12. Column forensics: what does each code and amount really mean?

**Why this section matters for the story.** Every chart in a dashboard silently assumes that "−1" means *paid in full*,
"2" means *two months late* and `BILL_AMT` is *what the client owes*. If those readings are wrong, the story is wrong.
This section checks each code and each amount against the money it should agree with, sizes the places where they
disagree, and estimates how much of the target is noise that no chart can ever explain.

Questions answered:
* 12.1 How are bills and payments timed against each other? (proof + three real clients)
* 12.2 What does each status code mean in money terms, and where do codes contradict the money?
* 12.3 What does the missing code `1` in Apr–Aug do to the month columns?
* 12.4 Which payment habits exist (exact full, round, minimum-like, zero, over-payment) and how risky is each?
* 12.5 How large is the label-noise floor?
* 12.6 Verdicts, story facts and new columns

Conventions: index 1 = Sep … 6 = Apr. For month *i*, `BILL_AMTi` is the statement balance, `PAY_AMTi` is the payment made
in month *i*, and it settles the **previous** statement `BILL_AMT(i+1)`. New `df` columns carry the prefix `CF_`;
section globals use `S12_` / `s12_`.

**Section guide**

- **What it does:** Checks what each status code and money column really means by comparing codes with the money behind them. It proves the payment timing, decodes each status code, sorts payments into habits, and sizes the defaults that nothing in the file can explain.
- **Question it answers:** Can we trust the usual readings ("-1 = paid in full", "2 = two months late"), and how much of default is unexplainable noise?
- **Inputs / new columns:** `PAY_1-6`, `BILL_AMT1-6`, `PAY_AMT1-6`, `LIMIT_BAL`, `DEFAULT`. Adds 10 columns with prefix `CF_` (e.g. `CF_PAY_HABIT`, `CF_N_ZERO_PAY_DUE`, `CF_SEP1_ORIGIN`, `CF_NOISE_FLAGS`).
- **How to read the charts:** "money fingerprint" heatmap: rows = status codes, columns = money measures scaled 0-100%; ECDF curves = share of months at or below each paid share; bars = default rate per habit with 95% CI; dashed line = 22.1% base rate.
- **Key takeaways:**
  - `PAY_AMTi` pays the previous statement: exact matches are 6.8x more common there than in the same month.
  - Code `-2` means "nothing due", not "no use"; code `0` revolvers pay about 4% of the bill (the minimum).
  - At the same 0-5% paid share, default runs from 12.2% (revolving) to 73.3% (late 3m+): the code carries more than the money.
  - Missing a due payment once lifts default from 15.7% to 36.0%.
  - 1,274 defaulters (19.2%) show no visible trouble; the noise floor is about 10-14%.
- **Where it feeds:** the status-code legend and data dictionary in Tableau (`s12_column_reading_guide.csv`), the "meet the clients" panel, and the "quiet card" persona.

### 12.0 Building blocks: a long client × month panel with the money attached to each status
One row per client-month (180,000 rows). Each row holds the status code, the statement (`BILL`), the payment made that
month (`PAY`), the statement it settles (`PREV_BILL`), the next statement (`NEXT_BILL`), the previous status, and derived
quantities: paid share `RATIO = PAY / PREV_BILL`, utilisation, implied new charges `SPEND = BILL − PREV_BILL + PAY`
and next-month balance change as a share of the limit.

In [ ]:
S12_T0 = time.time()
s12_parts = []
for i in IDX:
    s12_parts.append(pd.DataFrame({
        "ID": df["ID"].to_numpy(), "IDX": i, "MONTH": MONTH[i],
        "STATUS": df[f"PAY_{i}"].to_numpy(),
        "BILL": df[f"BILL_AMT{i}"].to_numpy(float), "PAY": df[f"PAY_AMT{i}"].to_numpy(float),
        "PREV_BILL": df[f"BILL_AMT{i+1}"].to_numpy(float) if i < 6 else np.full(len(df), np.nan),
        "NEXT_BILL": df[f"BILL_AMT{i-1}"].to_numpy(float) if i > 1 else np.full(len(df), np.nan),
        "PREV_STATUS": df[f"PAY_{i+1}"].to_numpy(float) if i < 6 else np.full(len(df), np.nan),
        "LIMIT": df["LIMIT_BAL"].to_numpy(float), "DEFAULT": df[TARGET].to_numpy()}))
S12_PANEL = pd.concat(s12_parts, ignore_index=True)
del s12_parts
s12p = S12_PANEL
s12_has_prev = s12p["PREV_BILL"] > 0
s12p["RATIO"] = np.where(s12_has_prev, s12p["PAY"] / s12p["PREV_BILL"].where(s12_has_prev), np.nan)
s12p["UTIL"] = s12p["BILL"] / s12p["LIMIT"]
s12p["SPEND"] = s12p["BILL"] - s12p["PREV_BILL"] + s12p["PAY"]
s12p["NEXT_CHG"] = (s12p["NEXT_BILL"] - s12p["BILL"]) / s12p["LIMIT"]
s12p["BILL0"] = (s12p["BILL"] == 0).astype(float)
s12p["PREV0"] = np.where(s12p["PREV_BILL"].notna(), (s12p["PREV_BILL"] == 0).astype(float), np.nan)
s12p["NEGB"] = (s12p["BILL"] < 0).astype(float)
s12p["PAY0"] = (s12p["PAY"] == 0).astype(float)
s12p["PAY_GE_PREV"] = np.where(s12_has_prev, (s12p["PAY"] >= s12p["PREV_BILL"]).astype(float), np.nan)
s12p["EXACT"] = np.where(s12_has_prev, (s12p["PAY"] == s12p["PREV_BILL"]).astype(float), np.nan)
s12p["SPEND_POS"] = np.where(s12p["SPEND"].notna(), (s12p["SPEND"] > 0).astype(float), np.nan)
s12p["NEXT_UP"] = np.where(s12p["NEXT_BILL"].notna(), (s12p["NEXT_BILL"] > s12p["BILL"]).astype(float), np.nan)
S12_CODE_ORDER = [-2, -1, 0, 1, 2, 3, 4, 5, 6, 7, 8]
s12p["CODE_GRP"] = s12p["STATUS"].clip(upper=4).map(lambda v: "4-8" if v >= 4 else str(v))
S12_CODEGRP_ORDER = ["-2", "-1", "0", "1", "2", "3", "4-8"]
print(S12_PANEL.shape, "| rows with a previous statement > 0:", f"{s12_has_prev.mean():.1%}")
S12_PANEL.head()

### 12.1 Which statement does a payment settle? Proving the timing from exact matches
If `PAY_AMTi` settles `BILL_AMT(i+1)`, clients who clear their card should pay **exactly** that statement to the dollar.
For every payment month we count how often a positive payment equals, to the dollar, the statement of the same month,
the month before, two months before, or the month after. The column that lights up shows the true timing.

In [ ]:
s12_lag_rows = []
for i in IDX:
    pay = df[f"PAY_AMT{i}"].to_numpy(float)
    pos = pay > 0
    rec = {"payment month": MONTH[i], "positive payments": int(pos.sum())}
    for lag, lab in [(-1, "= next statement (i-1)"), (0, "= same-month statement (i)"),
                     (1, "= previous statement (i+1)"), (2, "= two statements back (i+2)")]:
        j = i + lag
        if 1 <= j <= 6:
            b = df[f"BILL_AMT{j}"].to_numpy(float)
            rec[lab] = float(((pay == b) & pos & (b > 0)).sum() / pos.sum())
        else:
            rec[lab] = np.nan
    s12_lag_rows.append(rec)
S12_LAG = pd.DataFrame(s12_lag_rows).set_index("payment month").loc[[MONTH[i] for i in IDX[::-1]]]
display((S12_LAG.iloc[:, 1:] * 100).round(2))
s12_lag_prev = np.nanmean(S12_LAG["= previous statement (i+1)"].to_numpy(float))
s12_lag_same = np.nanmean(S12_LAG["= same-month statement (i)"].to_numpy(float))
print(f"Mean share of positive payments equal to the PREVIOUS statement: {s12_lag_prev:.1%} "
      f"vs SAME-month statement: {s12_lag_same:.1%} (ratio {s12_lag_prev / max(s12_lag_same, 1e-9):.1f}x)")

fig, ax = plt.subplots(figsize=(10, 4))
S12_LAG.iloc[:, 1:].T.plot(kind="bar", ax=ax, colormap="viridis", width=.8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=8)
ax.set_ylabel("share of positive payments"); ax.legend(title="payment month", ncol=6, fontsize=8)
ax.set_title("A payment matches the PREVIOUS statement to the dollar far more often than any other")
savefig("s12_payment_timing"); plt.show()
add_fact("12.1", "timing", f"{s12_lag_prev:.0%} of positive payments equal the previous statement to the dollar, vs {s12_lag_same:.1%} for the same-month statement: PAY_AMTi settles BILL_AMT(i+1)",
         round(s12_lag_prev, 4), "s12_payment_timing", 2)

**Observations:**
- 16.4% to 19.8% of positive payments each month equal the **previous** statement to the dollar (mean 17.7%).
- The same-month statement matches only 2.4% to 2.8%. The next statement matches 2.1% and the statement two months back 2.7% to 2.9%. That 2-3% background comes from clients whose balance repeats month after month.
- The previous-statement match is 6.8x the same-month match. So `PAY_AMTi` settles `BILL_AMT(i+1)`: the Sep payment pays the Aug statement.
- Story use: one line on the dashboard's data note: "payment in month t pays the bill issued in month t-1". Every paid-share metric uses this lag.

#### 12.1.1 Three real clients, month by month
Each table runs chronologically (Apr → Sep). Read a row as: *in this month the client had status X, the statement
closed at BILL, and the client paid PAY towards the statement shown in "settles"*. `new charges` is the implied
spending plus interest and fees (`BILL − previous BILL + PAY`). Clients: (a) a textbook full payer, (b) a revolver,
(c) a client who slides into delay and defaults.

In [ ]:
s12_B = df[BILL_COLS].to_numpy(float); s12_P = df[PAMT_COLS].to_numpy(float); s12_S = df[PAY_COLS].to_numpy(int)
s12_exact5 = ((s12_P[:, :5] == s12_B[:, 1:]) & (s12_B[:, 1:] > 0)).all(1)
s12_pick_a = df.index[s12_exact5 & (df["N_PAIDFULL"] == 6) & (s12_B.min(1) >= 3000) & (s12_B.std(1) > 1000)]
if len(s12_pick_a) == 0:
    s12_pick_a = df.index[s12_exact5 & (df["N_PAIDFULL"] == 6)]
s12_pick_b = df.index[(df["N_REVOLVE"] == 6) & (df[TARGET] == 0) & df["AVG_UTIL"].between(.6, .9)
                      & (df["AVG_PAY_RATIO"] < .2)]
s12_pick_c = df.index[(df["PAY_6"] <= 0) & (df["PAY_1"] >= 3) & (df[TARGET] == 1) & (df["BILL_AMT6"] > 0)]

def s12_client_table(ix):
    r = df.loc[ix]
    rows = []
    for i in IDX[::-1]:
        prev = r[f"BILL_AMT{i+1}"] if i < 6 else np.nan
        rows.append({"month": MONTH[i], "status": STATUS_LABEL.get(int(r[f"PAY_{i}"]), r[f"PAY_{i}"]),
                     "statement BILL": int(r[f"BILL_AMT{i}"]), "paid PAY": int(r[f"PAY_AMT{i}"]),
                     "settles": f"{MONTH[i+1]} statement" if i < 6 else "Mar statement (not in data)",
                     "settled bill": prev, "paid share": (r[f"PAY_AMT{i}"] / prev) if (i < 6 and prev > 0) else np.nan,
                     "new charges": (r[f"BILL_AMT{i}"] - prev + r[f"PAY_AMT{i}"]) if i < 6 else np.nan,
                     "utilisation": r[f"BILL_AMT{i}"] / r["LIMIT_BAL"]})
    return pd.DataFrame(rows)

S12_EXAMPLES = {}
for tag, pool in [("(a) exact full payer", s12_pick_a), ("(b) revolver", s12_pick_b), ("(c) slides into delay", s12_pick_c)]:
    if len(pool) == 0:
        print(tag, ": no client matches the filter"); continue
    ix = pool[0]
    r = df.loc[ix]
    print(f"\n{tag}: ID {int(r['ID'])} | limit {int(r['LIMIT_BAL']):,} | {r['SEX_LABEL']}, {r['AGE']}, {r['EDU_LABEL']} | "
          f"Oct outcome: {r['DEFAULT_LABEL']} | pool size {len(pool):,}")
    S12_EXAMPLES[tag] = s12_client_table(ix)
    display(S12_EXAMPLES[tag])

**Observations:**
- (a) ID 940 pays exactly the previous statement every month (May pays 11,193 = the Apr statement, and so on), keeps utilisation near 5% and does not default. This is a textbook transactor.
- (b) ID 4 pays 3.6% to 4.2% of the previous statement every month and is coded `0` throughout. A NT$22,177 purchase in Jul lifts utilisation from 57% to 99%. This is a textbook minimum-payment revolver, and it does not default.
- (c) ID 270 pays the Apr statement in full in May but is coded `0`, not `-1`. It pays 3.5% in Jun (coded `0`) and 3.0% in Jul (coded `2`). The money is almost the same but the code jumps straight from 0 to 2. Aug brings a zero payment, Sep is coded `3`, and the client defaults.
- Case (c) shows that the code is not a function of this month's paid share. It tracks a missed-minimum or days-past-due rule that we cannot see in the data.
- Story use: these three tables suit a "meet the clients" panel that explains the columns before any aggregate chart.

### 12.2 What does each status code mean in money terms?
For every code we pool all client-months and summarise the money attached to it: statement, payment, paid share of the
previous statement, share with zero statement, share paying at least the previous statement, share paying it exactly,
utilisation, the next month's balance move and the client's October default rate. This is the *money fingerprint* of
each code.

In [ ]:
def s12_code_profile(p, by):
    g = p.groupby(by, observed=True)
    out = g.agg(cells=("STATUS", "size"), med_bill=("BILL", "median"), med_prev_bill=("PREV_BILL", "median"),
                med_pay=("PAY", "median"), med_paid_share=("RATIO", "median"), sh_bill_zero=("BILL0", "mean"),
                sh_prev_zero=("PREV0", "mean"), sh_bill_neg=("NEGB", "mean"), sh_pay_zero=("PAY0", "mean"),
                sh_pay_ge_prev=("PAY_GE_PREV", "mean"), sh_exact_prev=("EXACT", "mean"), med_util=("UTIL", "median"),
                sh_new_charges=("SPEND_POS", "mean"), med_next_chg_pct_limit=("NEXT_CHG", "median"),
                sh_next_bill_up=("NEXT_UP", "mean"), default_rate=("DEFAULT", "mean"))
    return out.reset_index()

S12_CODE_PROF = s12_code_profile(s12p, "STATUS")
S12_CODE_PROF.insert(1, "label", S12_CODE_PROF["STATUS"].map(STATUS_LABEL))
display(S12_CODE_PROF)
export(S12_CODE_PROF, "s12_code_money_profile")

The same fingerprint as a heatmap (shares only, coarse code groups). Each column is scaled 0–100% so the pattern
across codes is visible at a glance.

In [ ]:
s12_prof_g = s12_code_profile(s12p, "CODE_GRP").set_index("CODE_GRP").loc[S12_CODEGRP_ORDER]
s12_share_cols = {"sh_bill_zero": "statement = 0", "sh_bill_neg": "statement < 0", "sh_prev_zero": "prev. statement = 0",
                  "sh_pay_zero": "payment = 0", "sh_pay_ge_prev": "pays >= prev. bill", "sh_exact_prev": "pays prev. bill exactly",
                  "sh_new_charges": "new charges > 0", "sh_next_bill_up": "next bill higher", "default_rate": "Oct default rate"}
s12_hm = s12_prof_g[list(s12_share_cols)].rename(columns=s12_share_cols)
s12_hm.index = [STATUS_LABEL.get(int(c), c) if c != "4-8" else "4-8 Late 4m+" for c in s12_hm.index]
fig, ax = plt.subplots(figsize=(13, 4.8))
sns.heatmap(s12_hm, annot=True, fmt=".0%", cmap="mako_r", vmin=0, vmax=1, cbar=False, linewidths=.5, ax=ax)
ax.set_title("Money fingerprint of each status code (all client-months pooled)"); ax.set_ylabel("")
plt.xticks(rotation=25, ha="right"); savefig("s12_code_fingerprint"); plt.show()
display(s12_prof_g[["cells", "med_bill", "med_pay", "med_paid_share", "med_util", "med_next_chg_pct_limit"]])

**Observations:**
- `-2` has a median statement of 0, but only 51.5% of these months have a zero statement and 9.9% have a negative one. 43% show new charges. When a bill was due, 95% paid it in full (57% to the dollar).
- `-1` has small balances (median statement 2,160, utilisation 1.4%). 90% show new charges, so this is an active **transactor** state. Only 76% paid the previous bill in full.
- `0` is the big-balance state: median statement 42,377, utilisation 51%, median paid share 4.9%. Only 4.8% paid in full.
- `2` looks almost the same in money: utilisation 62%, paid share 4.3%. The difference is that 27% paid nothing, against 7.5% for `0`. The default rate is 55% against 17%.
- Codes 3 to 8 pay nothing in 63% to 99% of months, and the default rate is 55% to 80%. Code `7` is odd: 218 cells with a median statement of only 2,400 (8% utilisation) and an 80% default rate, which looks like a small written-off cluster.
- `-2`, `-1` and `0` default at a similar 16% to 19%. The money behind them differs hugely, but the risk does not.
- Story use: the heatmap is the "decoder ring" for the status codes on the dashboard.

#### 12.2.1 How much does each code repay? Distribution of the paid share
Paid share of the previous statement (`PAY / PREV_BILL`, statements > 0 only, clipped at 150%), as an ECDF (curve of the share of months at or below each value) per code group.
A code that means "paid in full" should jump to 1 at 100%; a revolving code should sit far left of 100%.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
s12_cols = sns.color_palette("tab10", len(S12_CODEGRP_ORDER))
s12_q = []
for k, cg in enumerate(S12_CODEGRP_ORDER):
    v = s12p.loc[(s12p["CODE_GRP"] == cg) & s12p["RATIO"].notna(), "RATIO"].clip(upper=1.5).sort_values().to_numpy()
    if len(v) < 20:
        continue
    axes[0].plot(v, np.arange(1, len(v) + 1) / len(v), color=s12_cols[k], lw=2, label=f"{cg} (n={len(v):,})")
    s12_q.append({"code": cg, "n": len(v), "p10": np.quantile(v, .1), "p25": np.quantile(v, .25), "median": np.median(v),
                  "p75": np.quantile(v, .75), "p90": np.quantile(v, .9), "share_lt_10pct": (v < .1).mean(),
                  "share_ge_100pct": (v >= 1).mean()})
axes[0].axvline(1, ls="--", color="k", lw=1); axes[0].legend(fontsize=8, title="code")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].set_xlabel("paid share of previous statement (clipped 150%)"); axes[0].set_title("ECDF of paid share by status code")
v0 = s12p.loc[(s12p["STATUS"] == 0) & s12p["RATIO"].between(0, .3, inclusive="right"), "RATIO"]
axes[1].hist(v0, bins=np.arange(0, .3001, .005), color=C_ACC, alpha=.8)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_title("Code 0 (revolving): paid share between 0 and 30% (0.5pp bins)"); axes[1].set_xlabel("paid share")
plt.tight_layout(); savefig("s12_paid_share_by_code"); plt.show()
S12_RATIO_Q = pd.DataFrame(s12_q)
display(S12_RATIO_Q)
s12_hist, s12_edges = np.histogram(v0, bins=np.arange(0, .3001, .005))
S12_MINPAY_MODE = float(s12_edges[np.argmax(s12_hist)])
print(f"Code 0: most common paid-share bin starts at {S12_MINPAY_MODE:.1%} "
      f"(bin holds {s12_hist.max() / max(1, s12_hist.sum()):.1%} of 0-30% payers).")

**Observations:**
- `-2` and `-1` pile up at 100% paid: the median is 100% for both. But 15% of `-1` months paid under 10% of the previous bill.
- `0` is tightly bunched: p25 3.7%, median 4.9%, and 74% of months pay under 10%. The single most common paid-share bin is 3.5% to 4.0%, which holds 22.5% of the 0-30% payers. That spike looks like the bank's minimum-payment rule.
- `2` has nearly the same paid-share curve as `0` (median 4.3%), but more mass at exactly 0. Codes 3+ have medians of 0.
- Story use: "the minimum payment is about 4% of the bill" is a concrete, relatable number for the revolver persona.

#### 12.2.2 Do the usual readings hold? A verdict per code
Each common reading is turned into a money test and scored as the share of that code's client-months that pass.
"k months late" is tested two ways: the payment does not cover the statement, and the previous month was at least
k − 1 (a client cannot be 3 months late unless last month was at least 2 months late).

In [ ]:
st, has = s12p["STATUS"], s12_has_prev
s12_tests = []
def s12_add(code_txt, reading, test, mask, cond):
    sh = float(cond[mask].mean()) if mask.sum() else np.nan
    s12_tests.append({"code": code_txt, "usual reading": reading, "money test": test, "cells tested": int(mask.sum()), "share passing": sh})

s12_add("-2", "no consumption / no bill", "statement <= 0", st == -2, (s12p["BILL"] <= 0))
s12_add("-2", "no consumption / no bill", "no new charges this month (SPEND <= 0)", (st == -2) & s12p["SPEND"].notna(), (s12p["SPEND"] <= 0))
s12_add("-2", "no consumption / no bill", "statement, previous statement and payment all 0", (st == -2) & s12p["PREV_BILL"].notna(),
        (s12p["BILL"] == 0) & (s12p["PREV_BILL"] == 0) & (s12p["PAY"] == 0))
s12_add("-1", "paid in full", "payment >= previous statement (prev > 0)", (st == -1) & has, s12p["PAY"] >= s12p["PREV_BILL"])
s12_add("-1", "paid in full", "payment >= 95% of previous statement (prev > 0)", (st == -1) & has, s12p["RATIO"] >= .95)
s12_add("-1", "paid in full", "previous statement > 0 (there was something to pay)", (st == -1) & s12p["PREV_BILL"].notna(), s12p["PREV_BILL"] > 0)
s12_add("0", "revolving, minimum paid", "0 < paid share < 100%", (st == 0) & has, (s12p["RATIO"] > 0) & (s12p["RATIO"] < 1))
s12_add("0", "revolving, minimum paid", "payment > 0 (prev > 0)", (st == 0) & has, s12p["PAY"] > 0)
s12_add("0", "revolving, minimum paid", "statement > 0 (a balance revolves)", st == 0, s12p["BILL"] > 0)
for k in [2, 3]:
    s12_add(str(k), f"{k} months late", "payment < previous statement (prev > 0)", (st == k) & has, s12p["PAY"] < s12p["PREV_BILL"])
    s12_add(str(k), f"{k} months late", f"previous month status >= {k-1}", (st == k) & s12p["PREV_STATUS"].notna(), s12p["PREV_STATUS"] >= k - 1)
s12_add("4-8", "k months late", "previous month status >= k-1", (st >= 4) & s12p["PREV_STATUS"].notna(), s12p["PREV_STATUS"] >= st - 1)
s12_add("4-8", "k months late", "payment = 0", (st >= 4), s12p["PAY"] == 0)
s12_add("1", "1 month late", "payment < previous statement (prev > 0)", (st == 1) & has, s12p["PAY"] < s12p["PREV_BILL"])
s12_add("1", "1 month late", "statement > 0 (there is a balance to be late on)", st == 1, s12p["BILL"] > 0)
S12_VERDICT = pd.DataFrame(s12_tests)
S12_VERDICT["verdict"] = pd.cut(S12_VERDICT["share passing"], [-.01, .5, .8, .95, 1.01],
                                labels=["contradicted", "partly true", "mostly true", "holds"])
display(S12_VERDICT)
export(S12_VERDICT, "s12_code_reading_verdicts")
s12_v = S12_VERDICT.set_index(["code", "money test"])["share passing"]
add_fact("12.2", "status semantics", f"Only {s12_v[('-2', 'statement <= 0')]:.0%} of '-2 no consumption' months have a zero or negative statement; "
         f"{1 - s12_v[('-2', 'no new charges this month (SPEND <= 0)')]:.0%} show new charges", round(float(s12_v[('-2', 'statement <= 0')]), 4), "s12_code_fingerprint", 3)
add_fact("12.2", "status semantics", f"{s12_v[('-1', 'payment >= previous statement (prev > 0)')]:.0%} of '-1 paid in full' months really pay the previous statement in full",
         round(float(s12_v[('-1', 'payment >= previous statement (prev > 0)')]), 4), "s12_code_fingerprint", 2)

**Observations:**
- "-2 = no consumption / no bill" is **contradicted**. Only 61% of `-2` months have a zero or negative statement, 57% show no new charges, and just 40% are fully empty (bill, previous bill and payment all 0). A better reading of `-2` is "nothing was due and nothing needed paying".
- "-1 = paid in full" is **partly true**: 76% pay at least the previous statement. Another 15% of `-1` months had no previous bill at all.
- "0 = revolving, minimum paid" **holds**: 92% pay part of the bill, 96% pay something and 98% carry a balance.
- "2 = two months late" is half wrong. 93% underpay, but only 62% of `2` months follow a late month. The other 38% are first delays.
- "3+ = k months late" **holds** perfectly: 100% follow a status of at least k-1.
- Code `1` has a positive statement in only 54% of cases.
- Story use: the dashboard legend should read -2 "nothing due", -1 "paid in full (mostly)", 0 "revolving", 1 "Sep-only late flag", 2+ "late".

#### 12.2.3 Where do codes contradict the money, and do the contradictions matter?
Five cell-level contradictions. For each, we compare the default rate of clients with a contradicting month against
clients with the *same code* in the *same month* whose money agrees. If contradicting cells default differently, the code
is hiding information; if not, the contradiction is cosmetic.

| Flag | Contradiction |
|---|---|
| X1 | `-1` paid in full, but paid < 50% of the previous statement |
| X2 | `0` revolving, but statement and previous statement are both 0 (nothing revolves) |
| X3 | `-2` no consumption, but the statement is positive and new charges were made |
| X4 | late (`>= 1`), yet the previous statement was paid in full |
| X5 | `0` revolving, yet the previous statement was paid in full |

In [ ]:
S12_X = {
    "X1 '-1' but paid <50%": ((st == -1) & has & (s12p["RATIO"] < .5), (st == -1)),
    "X2 '0' with no balance": ((st == 0) & (s12p["BILL"] == 0) & (s12p["PREV_BILL"] == 0), (st == 0) & s12p["PREV_BILL"].notna()),
    "X3 '-2' with new charges": ((st == -2) & (s12p["BILL"] > 0) & (s12p["SPEND"] > 0), (st == -2) & s12p["SPEND"].notna()),
    "X4 late but paid in full": ((st >= 1) & has & (s12p["PAY"] >= s12p["PREV_BILL"]), (st >= 1) & s12p["PREV_BILL"].notna()),
    "X5 '0' but paid in full": ((st == 0) & has & (s12p["PAY"] >= s12p["PREV_BILL"]), (st == 0) & s12p["PREV_BILL"].notna()),
}
s12_xrows, s12_xmonth = [], {}
for name, (flag, base) in S12_X.items():
    flag = flag.fillna(False).astype(bool); base = base.fillna(False).astype(bool)
    ok = base & ~flag
    y_flag, y_ok = s12p.loc[flag, "DEFAULT"], s12p.loc[ok, "DEFAULT"]
    lo, hi = wilson_ci(y_flag.sum(), len(y_flag))
    tab = np.array([[y_flag.sum(), len(y_flag) - y_flag.sum()], [y_ok.sum(), len(y_ok) - y_ok.sum()]])
    p = stats.chi2_contingency(tab)[1] if tab.min() > 0 else np.nan
    s12_xrows.append({"contradiction": name, "cells": int(flag.sum()), "share of code cells": flag.sum() / max(1, base.sum()),
                      "clients": int(s12p.loc[flag, "ID"].nunique()), "default rate (contradicting cells)": y_flag.mean(),
                      "ci_lo": lo, "ci_hi": hi, "default rate (same code, money agrees)": y_ok.mean(), "p_chi2": p})
    s12_xmonth[name] = s12p.loc[flag].groupby("MONTH").size().reindex(MONTH_ORDER).fillna(0).astype(int)
S12_XTAB = pd.DataFrame(s12_xrows)
display(S12_XTAB)
print("Contradicting cells by month (Apr..Sep):")
display(pd.DataFrame(s12_xmonth).T[MONTH_ORDER])
export(S12_XTAB, "s12_code_money_contradictions")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2))
y = np.arange(len(S12_XTAB))
ax.barh(y - .2, S12_XTAB["default rate (contradicting cells)"], .4, color=C_DEF, alpha=.8, label="contradicting cells",
        xerr=[S12_XTAB["default rate (contradicting cells)"] - S12_XTAB["ci_lo"], S12_XTAB["ci_hi"] - S12_XTAB["default rate (contradicting cells)"]], capsize=3)
ax.barh(y + .2, S12_XTAB["default rate (same code, money agrees)"], .4, color=C_OK, alpha=.8, label="same code, money agrees")
ax.set_yticks(y); ax.set_yticklabels([f"{c}  (n={n:,})" for c, n in zip(S12_XTAB["contradiction"], S12_XTAB["cells"])])
ax.axvline(BASE_RATE, ls="--", color="k", lw=1); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.invert_yaxis(); ax.legend(); ax.set_title("Do code-money contradictions change risk? (client default rate in Oct)")
savefig("s12_contradictions"); plt.show()
s12_x1 = S12_XTAB.set_index("contradiction").loc["X1 '-1' but paid <50%"]
add_fact("12.2", "status semantics", f"{int(s12_x1['cells']):,} '-1 paid in full' months paid under half the previous statement; those clients default at "
         f"{s12_x1['default rate (contradicting cells)']:.1%} vs {s12_x1['default rate (same code, money agrees)']:.1%} when the money agrees",
         round(float(s12_x1['default rate (contradicting cells)']), 4), "s12_contradictions", 2)

**Observations:**
- X1: 4,696 `-1` months (13.6%) paid under half the previous bill. These clients default at 20.8% vs 15.5% when the money agrees (p < 0.001). The contradiction carries risk.
- X3: 7,948 `-2` months (41% of those with a flow) show a positive balance and new charges. These are the **safest** cells in the table: 9.2% vs 24.3% for "truly empty" `-2` months. An actively used card that needs no payment is a good sign.
- X4: 1,752 late-coded months follow a full payment. They default at 27.5%, half the 55.2% of other late months. 798 of them are in Sep, mostly code `1`.
- X2 (839 `0` months with no balance, 19.9% vs 16.7%) and X5 (3,696 `0` months paid in full, 14.9% vs 16.9%) are small and cosmetic.
- Story use: "the code is not the whole truth". An empty, dormant card (`-2`, no use) is riskier than an active card that is paid off.

#### 12.2.4 Is the status code just the money in disguise?
Default rate on a grid of **September status** × **September paid share** (`PAY_RATIO_1`, share of the Aug statement
paid in Sep). If rows still differ inside one paid-share column, the code carries information the money does not.

In [ ]:
s12_rb = pd.cut(df["PAY_RATIO_1"], [-0.01, 0, .05, .1, .3, .7, .999, 5.01],
                labels=["0", "0-5%", "5-10%", "10-30%", "30-70%", "70-<100%", ">=100%"])
s12_rb = s12_rb.cat.add_categories("no Aug bill").fillna("no Aug bill")
s12_tmp = df.assign(S12_RB=s12_rb)
S12_GRID = rate_table(s12_tmp, ["STATUS_GRP_SEP", "S12_RB"], min_n=1)
s12_rate = S12_GRID.pivot(index="STATUS_GRP_SEP", columns="S12_RB", values="rate").reindex(STATUS_GROUP_ORDER)
s12_n = S12_GRID.pivot(index="STATUS_GRP_SEP", columns="S12_RB", values="n").reindex(STATUS_GROUP_ORDER)
s12_rate_m = s12_rate.where(s12_n >= 30)
s12_annot = s12_rate_m.copy().astype(object)
for r_ in s12_rate_m.index:
    for c_ in s12_rate_m.columns:
        v, n = s12_rate_m.loc[r_, c_], s12_n.loc[r_, c_]
        s12_annot.loc[r_, c_] = "" if pd.isna(v) else f"{v:.0%}\nn={int(n):,}"
fig, ax = plt.subplots(figsize=(12, 4.8))
sns.heatmap(s12_rate_m.astype(float), annot=s12_annot.values, fmt="", cmap=CMAP_RATE, vmin=0, vmax=.75, linewidths=.5, ax=ax,
            cbar_kws={"format": mtick.PercentFormatter(1)})
ax.set_title("Default rate: Sep status × Sep paid share of Aug statement (cells n >= 30)"); ax.set_xlabel("paid share"); ax.set_ylabel("")
savefig("s12_status_vs_paidshare"); plt.show()
s12_spread = (s12_rate_m.max() - s12_rate_m.min()).rename("rate spread across status (pp)") * 100
display(s12_spread.round(1).to_frame().T)
s12_iv_s = woe_iv(df, "STATUS_GRP_SEP")[1]
s12_iv_r = woe_iv(s12_tmp, "S12_RB")[1]
s12_tmp["S12_CROSS"] = s12_tmp["STATUS_GRP_SEP"].astype(str) + "|" + s12_tmp["S12_RB"].astype(str)
s12_iv_x = woe_iv(s12_tmp, "S12_CROSS")[1]
print(f"IV Sep status group {s12_iv_s:.3f} | IV Sep paid-share band {s12_iv_r:.3f} | IV of the cross {s12_iv_x:.3f}")
export(S12_GRID, "s12_status_x_paidshare_rates")
add_fact("12.2", "status semantics", f"Sep status carries IV {s12_iv_s:.2f} vs {s12_iv_r:.2f} for the Sep paid share; inside one paid-share band the status still moves default by up to {np.nanmax(s12_spread.to_numpy()):.0f}pp",
         round(float(s12_iv_s), 3), "s12_status_vs_paidshare", 3)

**Observations:**
- The Sep status group has an IV of 0.87, against 0.25 for the Sep paid share. The cross reaches 0.96, so money adds a little on top of status, but status dominates.
- Inside the 0-5% paid-share column, default runs from 12.2% (Revolving) to 14.8% (Paid in full), 39.1% (Late 1m), 68.2% (Late 2m) and 73.3% (Late 3m+). That is a 61pp spread at identical money.
- Even among clients who paid at least 100% of the Aug bill, Late 2m defaults at 42.5% vs 8.1% for Inactive and 12.6% for Revolving.
- Inside each status row, paid share barely moves risk. Revolving stays at 11.5% to 14.9% across all bands.
- Story use: a strong hero-chart candidate. "It is not how much you pay, it is whether the bank has already marked you late".

#### 12.2.5 What happens to the balance after each code?
The month after a status, does the statement grow or shrink? Next-month balance change as a share of the limit
(months Apr–Aug, which have a following statement), split by code group and by October outcome.

In [ ]:
s12_nx = s12p[s12p["NEXT_CHG"].notna()].copy()
s12_nx["NEXT_CHG_C"] = s12_nx["NEXT_CHG"].clip(-.6, .6)
s12_nx["outcome"] = s12_nx["DEFAULT"].map({0: "No default", 1: "Default"})
fig, ax = plt.subplots(figsize=(12, 4.5))
sns.boxplot(data=s12_nx, x="CODE_GRP", y="NEXT_CHG_C", hue="outcome", order=S12_CODEGRP_ORDER, palette=PAL_TARGET,
            showfliers=False, ax=ax)
ax.axhline(0, color="k", lw=1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("status code this month"); ax.set_ylabel("next statement − this statement (% of limit)")
ax.set_title("Balance move in the month after each code (Apr-Aug)")
savefig("s12_next_balance_by_code"); plt.show()
S12_NEXT = s12_nx.groupby(["CODE_GRP", "outcome"], observed=True).agg(cells=("NEXT_CHG", "size"), median_chg=("NEXT_CHG", "median"),
                                                                    share_up=("NEXT_UP", "mean")).unstack("outcome").reindex(S12_CODEGRP_ORDER)
display(S12_NEXT)

**Observations:**
- The median balance hardly moves the month after any code (within ±1.5% of the limit, except code 1).
- Future defaulters grow their balance **less** often after the same code. After `-2`, 12% vs 25% see a higher next bill. After `0` it is 43% vs 50%, and after `2` it is 48% vs 64%.
- This fits section 0's note that rising utilisation does not mean more defaults. Defaulters look constrained or disengaged, not on a spending spree.
- Story use: supports the "quiet card, not the shopaholic" thread.

### 12.3 The missing code `1`: what does a "first late month" look like in each column?
Sections 1 and 5 showed that code `1` is almost only used in September. Here we look at what that means for reading the
month columns: (a) how a *first* delay (previous month ≤ 0, this month ≥ 1) is coded in each month, (b) where the
September code-1 clients came from, (c) whether a September "1" and an August "2" after a clean month are the same thing.

In [ ]:
s12_fd = s12p[s12p["PREV_STATUS"].notna()].copy()
s12_fd["first_delay"] = (s12_fd["PREV_STATUS"] <= 0) & (s12_fd["STATUS"] >= 1)
s12_fd_rows = []
for m in MONTH_ORDER[1:]:
    x = s12_fd[(s12_fd["MONTH"] == m)]
    f = x[x["first_delay"]]
    late = x[x["STATUS"] >= 1]
    s12_fd_rows.append({"month": m, "late cells": len(late), "late coded 1": (late["STATUS"] == 1).mean(),
                        "first-delay cells": len(f), "first delay coded 1": (f["STATUS"] == 1).mean(),
                        "first delay coded 2": (f["STATUS"] == 2).mean(), "first delay coded 3+": (f["STATUS"] >= 3).mean(),
                        "default rate after first delay": f["DEFAULT"].mean()})
S12_FIRST = pd.DataFrame(s12_fd_rows)
display(S12_FIRST)
s12_fd_apraug = S12_FIRST[S12_FIRST["month"] != "Sep"]
s12_first2 = float((s12_fd_apraug["first delay coded 2"] * s12_fd_apraug["first-delay cells"]).sum() / s12_fd_apraug["first-delay cells"].sum())
s12_first1_sep = float(S12_FIRST.set_index("month").loc["Sep", "first delay coded 1"])
print(f"May-Aug: {s12_first2:.1%} of first delays are coded 2 | Sep: {s12_first1_sep:.1%} of first delays are coded 1")
add_fact("12.3", "coding quirk", f"In May-Aug {s12_first2:.1%} of first delays are coded '2'; in Sep {s12_first1_sep:.0%} are coded '1': the same event has different codes by month",
         round(s12_first2, 4), "s12_first_delay_codes", 3)
fig, ax = plt.subplots(figsize=(10, 4))
S12_FIRST.set_index("month")[["first delay coded 1", "first delay coded 2", "first delay coded 3+"]].plot(kind="bar", stacked=True, ax=ax,
                                                                                                       color=[C_ACC, C_DEF, "#7f0000"])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.set_title("How a first delay (previous month <= 0) is coded, by month"); ax.set_ylabel("share of first-delay cells")
savefig("s12_first_delay_codes"); plt.show()

**Observations:**
- May to Aug: 98.5% to 100% of first delays (previous month ≤ 0) are coded `2` (pooled 99.5%). Almost none are coded `1`.
- Sep: 64.9% of first delays are coded `1` and 35.1% are coded `2`. The coding rule changes in the last month.
- No first delay is ever coded 3+. Delays grow one step at a time (as 12.2.2 showed).
- The default rate after a first delay is stable: 39.7% to 43.1% in every month, Sep included. The event means the same thing; only its code changes.
- Story use: a must-have caveat. "Late 1 month" exists only for Sep, so a month-by-month status chart must use late yes/no.

#### 12.3.1 Where did September's code-1 clients come from, and what money do they carry?
Split the 3,688-ish September code-1 clients by their August status. A "1" after an August "2" is a partial **cure**
(2 → 1); a "1" after August −2/−1 is a **new** delay. The money tells us whether these are the same kind of client.

In [ ]:
s12_m1 = df["PAY_1"] == 1
df["CF_SEP1_ORIGIN"] = np.select([s12_m1 & (df["PAY_2"] <= -1), s12_m1 & (df["PAY_2"] == 0), s12_m1 & (df["PAY_2"] >= 1)],
                                 ["from Aug -2/-1", "from Aug 0", "from Aug late (cure 2->1 etc.)"], "not code 1 in Sep")
s12_o = df[s12_m1].copy()
s12_o["Aug status"] = s12_o["PAY_2"].map(STATUS_LABEL)
S12_SEP1 = s12_o.groupby("Aug status").agg(clients=("ID", "size"), default_rate=(TARGET, "mean"),
                                          sep_bill_le0=("BILL_AMT1", lambda v: (v <= 0).mean()),
                                          sep_bill_neg=("BILL_AMT1", lambda v: (v < 0).mean()),
                                          med_sep_bill=("BILL_AMT1", "median"), med_aug_bill=("BILL_AMT2", "median"),
                                          sep_pay_zero=("PAY_AMT1", lambda v: (v == 0).mean()),
                                          med_util_sep=("UTIL_1", "median")).sort_values("clients", ascending=False)
S12_SEP1["share of code-1"] = S12_SEP1["clients"] / S12_SEP1["clients"].sum()
display(S12_SEP1)
export(S12_SEP1.reset_index(), "s12_sep_code1_origin")
s12_new = s12_m1 & (df["PAY_2"] <= -1)
print(f"Sep code 1 from Aug -2/-1: {int(s12_new.sum()):,} clients; Sep statement <= 0 for {(df.loc[s12_new, 'BILL_AMT1'] <= 0).mean():.1%}; "
      f"default rate {df.loc[s12_new, TARGET].mean():.1%}")
add_fact("12.3", "coding quirk", f"{(df.loc[s12_new, 'BILL_AMT1'] <= 0).mean():.0%} of Sep '1 month late' clients coming from -2/-1 owe nothing on their Sep statement, yet they default at {df.loc[s12_new, TARGET].mean():.1%}",
         round(float((df.loc[s12_new, 'BILL_AMT1'] <= 0).mean()), 4), "s12_sep1_origin", 2)

In [ ]:
s12_cmp = []
for lab, mask in [("Sep code 1 after Aug <=0", (df["PAY_1"] == 1) & (df["PAY_2"] <= 0)),
                  ("Sep code 2 after Aug <=0", (df["PAY_1"] == 2) & (df["PAY_2"] <= 0)),
                  ("Aug code 2 after Jul <=0", (df["PAY_2"] == 2) & (df["PAY_3"] <= 0)),
                  ("Jul code 2 after Jun <=0", (df["PAY_3"] == 2) & (df["PAY_4"] <= 0)),
                  ("Sep code 1 after Aug 2 (cure)", (df["PAY_1"] == 1) & (df["PAY_2"] == 2)),
                  ("Sep code 2 after Aug 2 (stays)", (df["PAY_1"] == 2) & (df["PAY_2"] == 2)),
                  ("Sep code 3 after Aug 2 (rolls)", (df["PAY_1"] == 3) & (df["PAY_2"] == 2))]:
    k, n = int(df.loc[mask, TARGET].sum()), int(mask.sum())
    lo, hi = wilson_ci(k, n)
    s12_cmp.append({"cohort": lab, "clients": n, "default_rate": k / max(n, 1), "ci_lo": lo, "ci_hi": hi})
S12_FIRSTCMP = pd.DataFrame(s12_cmp)
display(S12_FIRSTCMP)
fig, ax = plt.subplots(figsize=(11, 4.2))
yy = np.arange(len(S12_FIRSTCMP))
ax.barh(yy, S12_FIRSTCMP["default_rate"], color=[C_ACC, C_DEF, C_DEF, C_DEF, C_NEU, C_NEU, C_NEU], alpha=.8,
        xerr=[S12_FIRSTCMP["default_rate"] - S12_FIRSTCMP["ci_lo"], S12_FIRSTCMP["ci_hi"] - S12_FIRSTCMP["default_rate"]], capsize=3)
ax.set_yticks(yy); ax.set_yticklabels([f"{c} (n={n:,})" for c, n in zip(S12_FIRSTCMP["cohort"], S12_FIRSTCMP["clients"])])
ax.axvline(BASE_RATE, ls="--", color="k", lw=1); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.invert_yaxis()
ax.set_title("Is Sep '1' the same event as an earlier '2'? Default rate by first-delay and follow-up cohort")
savefig("s12_sep1_origin"); plt.show()

**Observations:**
- Of Sep's code-1 clients, 45% came from Aug `2` (a partial cure, 41.6% default), 33% from `-2` (28.3%) and 17% from `-1` (20.6%).
- 92% of the 1,833 clients coded `1` after `-2/-1` have a Sep statement ≤ 0, and 16% to 29% have a credit balance. They owe nothing, yet 25.8% default.
- Sep code 1 after a clean Aug defaults at 25.7%. Sep code 2 after a clean Aug defaults at 65.7%, and Aug code 2 after a clean Jul at 43.5%. So Sep `1` is a much weaker signal than an earlier `2`, while Sep `2` from clean is stronger.
- After an Aug `2`, Sep moves split cleanly: 2→1 at 41.6%, 2→2 at 71.8%, 2→3 at 75.7%.
- Story use: Sep code 1 is a mixed bag (cures plus zero-balance flags). Label it on the dashboard rather than calling it "1 month late".

#### 12.3.2 What this means for reading the month columns
A practical consequence: the same numeric value means different things in different columns. The table counts how
many clients a naive rule "status ≥ 2 = seriously late" flags in each month, versus a code-robust rule "status ≥ 1 =
late", and how the default rate of the flagged group differs.

In [ ]:
s12_rule = []
for i in IDX[::-1]:
    s = df[f"PAY_{i}"]
    for rule, m in [("status >= 1", s >= 1), ("status >= 2", s >= 2)]:
        s12_rule.append({"month": MONTH[i], "rule": rule, "flagged": int(m.sum()), "share": m.mean(), "default_rate": df.loc[m, TARGET].mean()})
S12_RULE = pd.DataFrame(s12_rule).pivot(index="month", columns="rule", values=["flagged", "default_rate"]).loc[MONTH_ORDER]
display(S12_RULE)
s12_sep_gap = S12_RULE.loc["Sep", ("flagged", "status >= 1")] - S12_RULE.loc["Sep", ("flagged", "status >= 2")]
s12_aug_gap = S12_RULE.loc["Aug", ("flagged", "status >= 1")] - S12_RULE.loc["Aug", ("flagged", "status >= 2")]
print(f"Clients that only 'status >= 1' catches: Sep {int(s12_sep_gap):,} vs Aug {int(s12_aug_gap):,}. "
      "Use 'status >= 1' (late yes/no) when comparing months; never compare code values across Sep and earlier months.")

**Observations:**
- In Apr to Aug the rules "status ≥ 1" and "status ≥ 2" flag the same clients (they differ by at most 28 clients).
- In Sep, "≥ 1" flags 6,818 clients (50.3% default) and "≥ 2" flags 3,130 (69.6% default). The 3,688 code-1 clients make the difference.
- A naive "≥ 2" rule makes Sep look like it has fewer late clients than Aug (3,130 vs 4,410), while "≥ 1" shows a jump (6,818 vs 4,438). Neither is clean, so trend charts should show both or use late yes/no with a note.
- Story use: caption for the monthly delinquency trend chart.

### 12.4 Which payment habits exist, and how risky is each?
Each client-month with a positive previous statement (May–Sep payments) is put into exactly one habit:

| Habit | Rule (paid share = PAY / previous statement) |
|---|---|
| Zero payment | PAY = 0 |
| Minimum-like | 0 < paid share ≤ 10% |
| Partial | 10% < paid share < 90% |
| Near full | 90% ≤ paid share < 100% |
| Exact full | PAY = previous statement to the dollar |
| Over-payment | PAY > previous statement |

Separately, a payment is **round** if it is positive, a multiple of NT$1,000 and not an exact full payment (a chosen
amount rather than a copied statement).

In [ ]:
s12_h = s12p[s12_has_prev].copy()
s12_r = s12_h["RATIO"]
s12_h["HABIT"] = np.select([s12_h["PAY"] == 0, s12_h["PAY"] == s12_h["PREV_BILL"], s12_h["PAY"] > s12_h["PREV_BILL"],
                            s12_r <= .1, s12_r < .9],
                           ["Zero payment", "Exact full", "Over-payment", "Minimum-like", "Partial"], "Near full")
S12_HABIT_ORDER = ["Zero payment", "Minimum-like", "Partial", "Near full", "Exact full", "Over-payment"]
s12_h["ROUND"] = (s12_h["PAY"] > 0) & (s12_h["PAY"] % 1000 == 0) & (s12_h["PAY"] != s12_h["PREV_BILL"])
S12_HAB_CELL = s12_h.groupby("HABIT").agg(cells=("PAY", "size"), client_default_rate=("DEFAULT", "mean"),
                                         med_prev_bill=("PREV_BILL", "median"), med_pay=("PAY", "median"),
                                         share_round=("ROUND", "mean"), med_util=("UTIL", "median")).reindex(S12_HABIT_ORDER)
S12_HAB_CELL["share of cells"] = S12_HAB_CELL["cells"] / S12_HAB_CELL["cells"].sum()
display(S12_HAB_CELL)
S12_HAB_MONTH = pd.crosstab(s12_h["HABIT"], s12_h["MONTH"], normalize="columns").reindex(S12_HABIT_ORDER)[MONTH_ORDER[1:]]
print("Habit mix by month (column %):"); display((S12_HAB_MONTH * 100).round(1))
print("Habit by status code (row %):")
display((pd.crosstab(s12_h["CODE_GRP"], s12_h["HABIT"], normalize="index").reindex(index=S12_CODEGRP_ORDER, columns=S12_HABIT_ORDER) * 100).round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
(S12_HAB_MONTH.T).plot(kind="bar", stacked=True, ax=axes[0], colormap="RdYlGn", width=.8)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].legend(fontsize=8, ncol=2, loc="lower right"); axes[0].set_title("Payment habit mix per month (statements > 0)")
hh = S12_HAB_CELL.reset_index()
lo, hi = wilson_ci(hh["client_default_rate"] * hh["cells"], hh["cells"])
axes[1].bar(hh["HABIT"], hh["client_default_rate"], color=sns.color_palette("RdYlGn", len(hh)), yerr=[hh["client_default_rate"] - lo, hi - hh["client_default_rate"]], capsize=3)
axes[1].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_title("Client default rate by habit of a payment month"); axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); savefig("s12_payment_habits"); plt.show()
s12_hc = S12_HAB_CELL["client_default_rate"]
add_fact("12.4", "payment habits", f"Exact-full-payment months belong to clients defaulting at {s12_hc['Exact full']:.1%}; zero-payment months (with a bill due) at {s12_hc['Zero payment']:.1%}",
         round(float(s12_hc['Zero payment']), 4), "s12_payment_habits", 2)

**Observations:**
- Of all due months: minimum-like 50.4%, exact full 16.3%, partial 15.4%, over-payment 8.9%, zero payment 8.5%, near full 0.6%.
- Client default rate by habit falls from zero payment (46.5%) to minimum-like (22.5%), partial (19.5%), exact full (15.9%) and over-payment (10.9%).
- Over the window, minimum-like falls from 52.1% to 47.3% and partial rises from 13.2% to 18.3%. Over-payment rises from 7.2% to 10.3% and exact full falls from 18.3% to 15.2%.
- The habits map onto the codes: `-2` months are 57% exact and 38% over-payment, `0` months are 71% minimum-like, and `4-8` months are 86% zero payment.
- Story use: payment habit is an intuitive dimension for a Tableau filter or colour.

#### 12.4.1 Round numbers: do people pick amounts, and are pickers riskier?
If amounts were random, about 0.1% of positive payments would be multiples of 1,000 and 1% multiples of 100. Heaping
far above that means many clients choose a round amount instead of paying the statement. Among *partial* payers we then
compare default rates of round vs non-round payments.

In [ ]:
s12_allpay = df[PAMT_COLS].to_numpy(float).ravel()
s12_allpay = s12_allpay[s12_allpay > 0]
S12_HEAP = pd.DataFrame({"multiple of": [10, 100, 1000, 5000, 10000],
                         "observed share": [np.mean(s12_allpay % m == 0) for m in [10, 100, 1000, 5000, 10000]],
                         "share if random": [1 / m for m in [10, 100, 1000, 5000, 10000]]})
S12_HEAP["heaping factor"] = S12_HEAP["observed share"] / S12_HEAP["share if random"]
display(S12_HEAP)
s12_part = s12_h[s12_h["HABIT"].isin(["Minimum-like", "Partial", "Near full"])]
S12_ROUND = s12_part.groupby(["HABIT", "ROUND"]).agg(cells=("PAY", "size"), client_default_rate=("DEFAULT", "mean")).unstack("ROUND").reindex(S12_HABIT_ORDER).dropna(how="all")
display(S12_ROUND)
s12_last3 = pd.Series((s12_allpay % 1000).astype(int))
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(S12_HEAP["multiple of"].astype(str), S12_HEAP["observed share"], color=C_ACC, alpha=.8, label="observed")
axes[0].plot(S12_HEAP["multiple of"].astype(str), S12_HEAP["share if random"], "k--o", label="if random")
axes[0].set_yscale("log"); axes[0].set_title("Payments heap on round amounts"); axes[0].legend(); axes[0].set_xlabel("multiple of NT$")
top_rem = s12_last3.value_counts().head(12)
axes[1].bar(top_rem.index.astype(str), top_rem.values / len(s12_last3), color=C_NEU)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("Most common last-3-digit endings of payments")
plt.tight_layout(); savefig("s12_round_payments"); plt.show()
s12_r1000 = float(S12_HEAP.loc[S12_HEAP["multiple of"] == 1000, "observed share"].iloc[0])
add_fact("12.4", "payment habits", f"{s12_r1000:.0%} of positive payments are exact multiples of NT$1,000 (vs 0.1% if random): clients pick amounts, they do not copy statements",
         round(s12_r1000, 4), "s12_round_payments", 1)

**Observations:**
- 24.8% of positive payments are multiples of NT$1,000, which is 248x what random amounts would give. 42% are multiples of 100 and 5.9% are multiples of 5,000.
- Round payers are slightly **safer** than non-round payers at the same habit: partial 17.3% vs 21.5%, minimum-like 20.7% vs 23.2%.
- Story use: a human-behaviour aside. People type round numbers, and those who choose an amount above the minimum tend to be in control.

#### 12.4.2 Client-level habits: counts per client and the dominant habit
New columns (May–Sep payment months, 0..5): `CF_N_EXACT_FULL`, `CF_N_ROUND_PAY`, `CF_N_MINLIKE`, `CF_N_ZERO_PAY_DUE`
(zero payment while a statement was due), `CF_N_OVERPAY`, and `CF_N_CREDIT_BAL` (months with a negative statement,
Apr–Sep, 0..6). `CF_PAY_HABIT` is the client's most frequent habit (ties go to the riskier habit; "No bills due" when no
statement was ever positive).

In [ ]:
s12_cnt = s12_h.pivot_table(index="ID", columns="HABIT", values="PAY", aggfunc="size", fill_value=0).reindex(columns=S12_HABIT_ORDER, fill_value=0)
s12_round_n = s12_h.groupby("ID")["ROUND"].sum()
s12_idx = df["ID"]
df["CF_N_EXACT_FULL"] = s12_idx.map(s12_cnt["Exact full"]).fillna(0).astype(int).to_numpy()
df["CF_N_MINLIKE"] = s12_idx.map(s12_cnt["Minimum-like"]).fillna(0).astype(int).to_numpy()
df["CF_N_ZERO_PAY_DUE"] = s12_idx.map(s12_cnt["Zero payment"]).fillna(0).astype(int).to_numpy()
df["CF_N_OVERPAY"] = s12_idx.map(s12_cnt["Over-payment"]).fillna(0).astype(int).to_numpy()
df["CF_N_ROUND_PAY"] = s12_idx.map(s12_round_n).fillna(0).astype(int).to_numpy()
df["CF_N_CREDIT_BAL"] = (df[BILL_COLS] < 0).sum(1).astype(int)
# dominant habit: max count, ties broken towards the riskier (earlier in order) habit
s12_dom = s12_cnt.to_numpy().argmax(1)
s12_dom_map = pd.Series(np.array(S12_HABIT_ORDER)[s12_dom], index=s12_cnt.index)
df["CF_PAY_HABIT"] = s12_idx.map(s12_dom_map).fillna("No bills due").to_numpy()
S12_HAB_ORDER_CL = S12_HABIT_ORDER + ["No bills due"]
fig, ax = plt.subplots(figsize=(12, 4.5))
S12_HAB_CL = plot_rate(df, "CF_PAY_HABIT", ax=ax, order=S12_HAB_ORDER_CL, title="Default rate by dominant payment habit (May-Sep)")
savefig("s12_dominant_habit"); plt.show()
display(S12_HAB_CL[["CF_PAY_HABIT", "n", "rate", "ci_lo", "ci_hi", "share_clients", "lift"]])
export(S12_HAB_CL, "s12_habit_rates")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 7.5), sharey=True)
S12_CNT_COLS = ["CF_N_EXACT_FULL", "CF_N_ROUND_PAY", "CF_N_MINLIKE", "CF_N_ZERO_PAY_DUE", "CF_N_OVERPAY", "CF_N_CREDIT_BAL"]
s12_cnt_tabs = []
for ax, c in zip(axes.ravel(), S12_CNT_COLS):
    t = rate_table(df, c)
    ax.bar(t[c].astype(str), t["rate"], color=C_DEF, alpha=.75,
           yerr=[t["rate"] - t["ci_lo"], t["ci_hi"] - t["rate"]], capsize=2)
    for xi, n in enumerate(t["n"]):
        ax.text(xi, 0.01, f"{n:,}", ha="center", fontsize=7, rotation=90, color="w")
    ax.axhline(BASE_RATE, ls="--", color="k", lw=1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(c.replace("CF_N_", "months: ").lower()); ax.set_xlabel("number of months")
    s12_cnt_tabs.append(t.rename(columns={c: "months"}).assign(feature=c))
plt.suptitle("Default rate by how often each habit occurs (white = clients)", y=1.01)
plt.tight_layout(); savefig("s12_habit_counts"); plt.show()
S12_CNT_TAB = pd.concat(s12_cnt_tabs, ignore_index=True)
export(S12_CNT_TAB[["feature", "months", "n", "defaults", "rate", "ci_lo", "ci_hi", "lift"]], "s12_habit_count_rates")
s12_iv = pd.Series({c: woe_iv(df, c)[1] for c in S12_CNT_COLS + ["CF_PAY_HABIT"]}).sort_values(ascending=False)
print("Information value of each habit column:"); print(s12_iv.round(3).to_string())
s12_ex0 = df.loc[df["CF_N_EXACT_FULL"] == 0, TARGET].mean(); s12_ex5 = df.loc[df["CF_N_EXACT_FULL"] >= 3, TARGET].mean()
add_fact("12.4", "payment habits", f"Clients with 3+ exact full payments default at {s12_ex5:.1%} vs {s12_ex0:.1%} with none",
         round(float(s12_ex5), 4), "s12_habit_counts", 2)

**Observations:**
- By dominant habit, default rates are: zero payment 48.0% (2,209 clients), "no bills due" 31.2% (1,406), minimum-like 22.9% (15,002, half the book), exact full 16.1%, partial 15.7% and over-payment 10.5%.
- Zero payments while a bill is due show a steep dose-response: 15.7% at 0 months, 36.0% at 1, 49.6% at 2 and 79.9% at 5. It has the highest IV of the habit columns (0.42).
- Over-payment months protect: 24.5% at 0 and 7% to 8% at 4-5. Exact full payments help once (24.5% at 0 → 16.3% at 1), then the line goes flat.
- Minimum-like months form an inverted U. Clients who pay the minimum in all 5 months default at 17.5%, less than those with 2-4 such months (28-29%). Steady revolvers are not the problem; switching is.
- Clients with 5 round payments default at only 9.7%.
- Story use: "missing a payment once doubles the risk" (15.7% → 36.0%) is a headline-ready number.

#### 12.4.3 What code does a credit balance get, and what does it mean?
A negative statement means the bank owes the client (over-payment or refund). Which status code does the data give
such a month, and how do these clients behave next?

In [ ]:
s12_neg = s12p[s12p["BILL"] < 0]
S12_NEG_CODE = pd.crosstab(s12_neg["MONTH"], s12_neg["STATUS"]).reindex(MONTH_ORDER)
S12_NEG_CODE.columns = [STATUS_LABEL.get(c, c) for c in S12_NEG_CODE.columns]
display(S12_NEG_CODE)
s12_neg_share = s12_neg["STATUS"].value_counts(normalize=True)
print("Share of negative-statement months by code:", {STATUS_LABEL.get(k, k): f"{v:.1%}" for k, v in s12_neg_share.items()})
S12_NEG_RATE = rate_table(df.assign(S12_NEGB=pd.cut(df["CF_N_CREDIT_BAL"], [-1, 0, 1, 3, 6], labels=["0", "1", "2-3", "4-6"])), "S12_NEGB")
display(S12_NEG_RATE[["S12_NEGB", "n", "rate", "ci_lo", "ci_hi", "lift"]])
s12_neg_late = float((s12_neg["STATUS"] >= 1).mean())
add_fact("12.4", "status semantics", f"{s12_neg_late:.1%} of months with a credit balance (negative statement) still carry a late code; {s12_neg_share.get(-1, 0) + s12_neg_share.get(-2, 0):.0%} are coded -1/-2",
         round(s12_neg_late, 4), None, 1)

**Observations:**
- Of the negative-statement months, 61% are coded `-2`, 17% `-1`, 9% `0` and 10% code `1`. All the code-1 ones are in Sep (377 cells). In Sep, a credit balance usually gets code `1`, not `-2`.
- One month of credit balance goes with 14.2% default. Four to six such months go with 25.3%, which suggests long-dormant cards with a leftover credit.
- Story use: minor. It confirms that Sep code 1 often flags a zero or credit balance, not a real delay.

### 12.5 How large is the label-noise floor?
Some defaults cannot be explained by anything in the file. We size five groups of **implausible-looking defaulters**,
then ask how low the default rate gets in the "cleanest" clients. This is the floor any dashboard or rule must live with.

| Flag | Definition |
|---|---|
| N1 | member of a feature-duplicate group whose rows carry conflicting `DEFAULT` labels |
| N2 | all six statements and all six payments are 0 (no activity at all), yet defaulted |
| N3 | Sep statement ≤ 0 (owes nothing at the last observation), yet defaulted |
| N4 | every positive previous statement May–Sep was fully paid (money-based full payer), yet defaulted |
| N5 | status ≤ −1 in all six months (codes say "never owed / always paid"), yet defaulted |

In [ ]:
S12_FEAT = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"] + PAY_COLS + BILL_COLS + PAMT_COLS
s12_dupmask = df.duplicated(subset=S12_FEAT, keep=False)
s12_gid = df[s12_dupmask].groupby(S12_FEAT).ngroup()
s12_conf_g = df.loc[s12_dupmask, TARGET].groupby(s12_gid).nunique()
s12_conf_ids = s12_gid[s12_gid.isin(s12_conf_g[s12_conf_g > 1].index)].index
print(f"Feature-duplicate rows {int(s12_dupmask.sum())} in {s12_gid.nunique()} groups; groups with conflicting labels "
      f"{int((s12_conf_g > 1).sum())} ({len(s12_conf_ids)} rows)")
print(f"Of the conflicting rows, all-zero statements: {(df.loc[s12_conf_ids, BILL_COLS] == 0).all(1).mean():.0%}; "
      f"all statuses -2: {(df.loc[s12_conf_ids, PAY_COLS] == -2).all(1).mean():.0%}")
display(df.loc[s12_conf_ids, ["ID", "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE", "PAY_1", "BILL_AMT1", "PAY_AMT1", TARGET]]
        .sort_values(["LIMIT_BAL", "AGE", TARGET]).head(10))

s12_prevB = df[[f"BILL_AMT{i+1}" for i in IDX[:-1]]].to_numpy(float)
s12_payA = df[[f"PAY_AMT{i}" for i in IDX[:-1]]].to_numpy(float)
s12_full_money = ((s12_prevB <= 0) | (s12_payA >= s12_prevB)).all(1) & (s12_prevB > 0).any(1)
S12_NFLAGS = pd.DataFrame({
    "N1 conflicting duplicate": df.index.isin(s12_conf_ids),
    "N2 no activity at all": ((df[BILL_COLS] == 0).all(1) & (df[PAMT_COLS] == 0).all(1)).to_numpy(),
    "N3 owes nothing in Sep": (df["BILL_AMT1"] <= 0).to_numpy(),
    "N4 money full payer": s12_full_money,
    "N5 codes always <= -1": (df[PAY_COLS] <= -1).all(1).to_numpy()}, index=df.index)
s12_y = df[TARGET].to_numpy()
s12_nrows = []
for c in S12_NFLAGS.columns:
    m = S12_NFLAGS[c].to_numpy()
    lo, hi = wilson_ci(s12_y[m].sum(), m.sum())
    s12_nrows.append({"flag": c, "clients": int(m.sum()), "defaulters": int(s12_y[m].sum()), "default_rate": s12_y[m].mean(),
                      "ci_lo": lo, "ci_hi": hi, "share of all defaulters": s12_y[m].sum() / s12_y.sum()})
s12_any = S12_NFLAGS.any(axis=1).to_numpy()
s12_nrows.append({"flag": "ANY of N1-N5", "clients": int(s12_any.sum()), "defaulters": int(s12_y[s12_any].sum()),
                  "default_rate": s12_y[s12_any].mean(), "ci_lo": wilson_ci(s12_y[s12_any].sum(), s12_any.sum())[0],
                  "ci_hi": wilson_ci(s12_y[s12_any].sum(), s12_any.sum())[1], "share of all defaulters": s12_y[s12_any].sum() / s12_y.sum()})
S12_NOISE = pd.DataFrame(s12_nrows)
display(S12_NOISE)
export(S12_NOISE, "s12_noise_flags")
df["CF_NOISE_FLAGS"] = S12_NFLAGS.sum(1).astype(int)
df["CF_IMPLAUSIBLE_DEFAULT"] = ((df["CF_NOISE_FLAGS"] > 0) & (df[TARGET] == 1)).astype(int)
s12_anyrow = S12_NOISE.iloc[-1]
s12_n2 = S12_NOISE.set_index("flag").loc["N2 no activity at all"]
add_fact("12.5", "label noise", f"{int(s12_n2['clients']):,} clients with zero statements and zero payments in all six months still default at {s12_n2['default_rate']:.1%}",
         round(float(s12_n2['default_rate']), 4), "s12_noise_floor", 2)
add_fact("12.5", "label noise", f"{int(s12_anyrow['defaulters']):,} defaulters ({s12_anyrow['share of all defaulters']:.1%} of all) show no sign of trouble in the data (nothing owed, always paid, or conflicting duplicates); their group defaults at {s12_anyrow['default_rate']:.1%}",
         round(float(s12_anyrow['share of all defaulters']), 4), "s12_noise_floor", 3)

**Observations:**
- 108 rows form 52 feature-duplicate groups. 21 groups (46 rows) carry conflicting labels, and 93% of those rows have all-zero statements, so identical empty records get both outcomes.
- 795 clients with no activity at all (zero bills and zero payments for six months) default at 37.6%, well above the 22.1% base.
- 2,598 clients owe nothing in Sep and default at 24.8%. The 5,339 money-based full payers default at 13.6% (725 defaulters), and the 5,195 always-`≤ -1` clients at 14.2%.
- In total, 1,274 defaulters (19.2% of all) show no visible trouble in the data.
- Story use: an honest limit. About one default in five cannot be explained by this file, and the dashboard should say so.

#### 12.5.1 How do the noise flags overlap?
Clients can trip several flags at once (e.g. an inactive client owes nothing in Sep and has codes ≤ −1). Counts of each
flag combination among **defaulters**.

In [ ]:
s12_def_flags = S12_NFLAGS[df[TARGET] == 1]
s12_def_flags = s12_def_flags[s12_def_flags.any(axis=1)]
s12_combo = s12_def_flags.apply(lambda r: " + ".join([c.split(" ")[0] for c in s12_def_flags.columns if r[c]]), axis=1).value_counts()
S12_COMBO = s12_combo.rename_axis("flag combination").reset_index(name="defaulters")
display(S12_COMBO.head(15))
fig, ax = plt.subplots(figsize=(10, 4.2))
top = S12_COMBO.head(12)
ax.barh(top["flag combination"], top["defaulters"], color=C_DEF, alpha=.75); ax.invert_yaxis()
ax.set_title("Implausible-looking defaulters: flag combinations"); ax.set_xlabel("defaulters")
savefig("s12_noise_combos"); plt.show()

**Observations:**
- The largest combination is N4 + N5 (487 defaulters): full payers by both codes and money. Together with N4 or N5 alone this is the "paid everything" group.
- N2 + N3 (179) and N2 + N3 + N5 (100) are the empty-card defaulters. N3 alone adds 153 clients who owed nothing in Sep but had activity earlier.
- Conflicting duplicates (N1) are tiny: 22 defaulters.
- Story use: the noise has two faces, "paid everything" and "never used the card", and they need different explanations.

#### 12.5.2 How low can the default rate go? The floor under ever-cleaner filters
Filters are applied cumulatively, each one removing a source of visible risk. The rate at the end is the share of
clients who default with no warning sign at all in this file: the practical noise floor.

In [ ]:
s12_steps = [("all clients", np.ones(len(df), bool)),
             ("never late (status <= 0 all 6 months)", (df[PAY_COLS] <= 0).all(1).to_numpy()),
             ("+ never a zero payment while a bill was due", (df["CF_N_ZERO_PAY_DUE"] == 0).to_numpy()),
             ("+ max utilisation < 50%", (df["MAX_UTIL"] < .5).to_numpy()),
             ("+ no revolving months (codes -2/-1 only)", (df["N_REVOLVE"] == 0).to_numpy()),
             ("+ paid every due statement in full", s12_full_money | (s12_prevB <= 0).all(1))]
s12_cum = np.ones(len(df), bool); s12_fl = []
for lab, m in s12_steps:
    s12_cum = s12_cum & m
    k, n = int(s12_y[s12_cum].sum()), int(s12_cum.sum())
    lo, hi = wilson_ci(k, n)
    s12_fl.append({"filter": lab, "clients": n, "defaulters": k, "default_rate": k / max(n, 1), "ci_lo": lo, "ci_hi": hi})
S12_FLOOR = pd.DataFrame(s12_fl)
display(S12_FLOOR)
fig, ax = plt.subplots(figsize=(11, 4.3))
xx = np.arange(len(S12_FLOOR))
ax.plot(xx, S12_FLOOR["default_rate"], marker="o", color=C_DEF, lw=2)
ax.fill_between(xx, S12_FLOOR["ci_lo"], S12_FLOOR["ci_hi"], color=C_DEF, alpha=.15)
for xi, (r, n) in enumerate(zip(S12_FLOOR["default_rate"], S12_FLOOR["clients"])):
    ax.text(xi, r + .01, f"{r:.1%}\nn={n:,}", ha="center", fontsize=8)
ax.set_xticks(xx); ax.set_xticklabels([f.replace("+ ", "+\n") for f in S12_FLOOR["filter"]], fontsize=8)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_ylim(0, max(.3, S12_FLOOR["ci_hi"].max() * 1.15))
ax.axhline(BASE_RATE, ls="--", color="k", lw=1); ax.set_title("The noise floor: default rate never reaches zero, however clean the history")
savefig("s12_noise_floor"); plt.show()
export(S12_FLOOR, "s12_noise_floor")
s12_floor = S12_FLOOR.iloc[-1]
add_fact("12.5", "label noise", f"Even among {int(s12_floor['clients']):,} clients who were never late, never skipped a due payment, kept utilisation under 50% and paid every statement in full, {s12_floor['default_rate']:.1%} defaulted",
         round(float(s12_floor['default_rate']), 4), "s12_noise_floor", 3)

**Observations:**
- Never late: 11.7%. Adding "never skipped a due payment" gives 11.4%, and adding "max utilisation < 50%" gives 10.6% (10,410 clients). That is the lowest point.
- Adding "no revolving months" pushes the rate back **up** to 14.2% (4,995 clients). Full transactors and inactive cards are riskier than low-utilisation revolvers.
- The floor sits around 10% to 14%. No combination of clean-history filters gets close to zero.
- Story use: the "noise floor" chart. Roughly one in ten spotless clients still defaults.

#### 12.5.3 Who are the clean defaulters? Profile against clean non-defaulters
If clean defaulters looked like everyone else, the noise is random. If they differ (e.g. younger, smaller limits, larger
recent spending), part of the "noise" is a signal the status codes do not capture.

In [ ]:
s12_clean = s12_cum
s12_prof_cols = ["LIMIT_BAL", "AGE", "AVG_BILL", "AVG_PAY", "AVG_UTIL", "MAX_UTIL", "N_INACTIVE", "N_PAIDFULL", "BILL_AMT1", "PAY_AMT1",
                 "CF_N_EXACT_FULL", "CF_N_ROUND_PAY", "CF_N_CREDIT_BAL"]
S12_CLEAN_PROF = pd.concat({"clean defaulters": df.loc[s12_clean & (s12_y == 1), s12_prof_cols].median(),
                            "clean non-defaulters": df.loc[s12_clean & (s12_y == 0), s12_prof_cols].median(),
                            "all defaulters": df.loc[s12_y == 1, s12_prof_cols].median()}, axis=1)
s12_mw = {c: stats.mannwhitneyu(df.loc[s12_clean & (s12_y == 1), c], df.loc[s12_clean & (s12_y == 0), c]).pvalue
          if (s12_clean & (s12_y == 1)).sum() > 5 else np.nan for c in s12_prof_cols}
S12_CLEAN_PROF["p (MWU clean def vs clean non-def)"] = pd.Series(s12_mw)
display(S12_CLEAN_PROF)
for c in ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND"]:
    t = pd.crosstab(df.loc[s12_clean, c], df.loc[s12_clean, TARGET], normalize="columns")
    t.columns = ["clean non-def", "clean def"]
    print(f"\n{c} mix (column %):"); print((t * 100).round(1).to_string())

**Observations:**
- Clean defaulters have the same limit (median 200k vs 210k) and age (36 vs 36) as clean non-defaulters, and almost identical sex, education and marriage mix.
- What separates them is how little they use the card. The average bill is 760 vs 2,197, the Sep statement 724 vs 1,886 and the Sep payment 390 vs 1,406 (all p < 0.001). Utilisation is 0.5% vs 1.2%.
- The "noise" is partly a low-engagement signal: near-dormant cards default more, even with perfect history.
- Story use: links to the dormant-card finding in 12.2.3 and 12.5. "Quiet cards" deserve their own persona.

### 12.6 What should every later chart assume? Verdict table, facts and new columns
A compact reading guide built from the numbers above: for each column family, the reading the dashboard should use.

In [ ]:
s12_vv = S12_VERDICT.set_index(["code", "money test"])["share passing"]
S12_GUIDE = pd.DataFrame([
    {"column": "BILL_AMTi", "reading to use": "statement balance closing in month i (can be 0 or negative = credit)",
     "evidence": f"{(s12p['BILL'] < 0).mean():.1%} of statements negative, {(s12p['BILL'] == 0).mean():.1%} zero"},
    {"column": "PAY_AMTi", "reading to use": "payment made in month i towards statement BILL_AMT(i+1)",
     "evidence": f"{s12_lag_prev:.0%} of payments equal the previous statement exactly vs {s12_lag_same:.1%} the same-month one"},
    {"column": "PAY_i = -2", "reading to use": "no new activity / nothing due, but NOT always a zero balance",
     "evidence": f"statement <= 0 in only {s12_vv[('-2', 'statement <= 0')]:.0%} of -2 months"},
    {"column": "PAY_i = -1", "reading to use": "paid in full (mostly true)",
     "evidence": f"pays >= previous statement in {s12_vv[('-1', 'payment >= previous statement (prev > 0)')]:.0%} of cases"},
    {"column": "PAY_i = 0", "reading to use": "revolving: paid part of the statement",
     "evidence": f"0 < paid share < 100% in {s12_vv[('0', '0 < paid share < 100%')]:.0%} of cases"},
    {"column": "PAY_i = 1", "reading to use": "Sep only; mixes new small delays with 2->1 cures; do not compare with code 1/2 in other months",
     "evidence": f"first delays coded 2 in May-Aug: {s12_first2:.1%}; coded 1 in Sep: {s12_first1_sep:.1%}"},
    {"column": "PAY_i >= 2", "reading to use": "late; in Apr-Aug '2' already means the first late month",
     "evidence": f"'2' preceded by status >= 1 in only {s12_vv[('2', 'previous month status >= 1')]:.0%} of cases"},
    {"column": "DEFAULT", "reading to use": "label with a visible noise floor",
     "evidence": f"{s12_floor['default_rate']:.1%} default among the cleanest {int(s12_floor['clients']):,} clients"},
])
display(S12_GUIDE)
export(S12_GUIDE, "s12_column_reading_guide")

**Observations:**
- The guide sets the readings every later chart should use. `BILL_AMT` is the statement balance (2.2% negative, 10.1% zero), and `PAY_AMTi` pays the previous statement.
- Codes: `-2` means "nothing due", `-1` means mostly paid in full, `0` means revolving. `1` is a Sep-only mixed flag, and in Apr to Aug `2` is already the first late month.
- The target carries a noise floor of about 10% to 14% among the cleanest clients.
- Story use: copy this table into the Tableau data dictionary (section 21).

#### 12.6.1 Story facts registered by this section

In [ ]:
S12_FACTS = pd.DataFrame([f for f in STORY_FACTS if f["section"].startswith("12")])
display(S12_FACTS)
s12_new_cols = [c for c in df.columns if c.startswith("CF_")]
print("New df columns:", s12_new_cols)
print(f"Section 12 runtime: {time.time() - S12_T0:.1f}s")

#### 12.6.2 New `df` columns created in Section 12

| Column | Definition |
|---|---|
| `CF_SEP1_ORIGIN` | for clients with Sep status 1: where they came from in Aug (`from Aug -2/-1`, `from Aug 0`, `from Aug late`); else `not code 1 in Sep` |
| `CF_N_EXACT_FULL` | months (May–Sep) where the payment equals the previous statement to the dollar |
| `CF_N_MINLIKE` | months with 0 < paid share of the previous statement ≤ 10% |
| `CF_N_ZERO_PAY_DUE` | months with a zero payment while the previous statement was positive |
| `CF_N_OVERPAY` | months paying more than the previous statement |
| `CF_N_ROUND_PAY` | months with a positive, round (multiple of NT$1,000) payment that is not an exact full payment |
| `CF_N_CREDIT_BAL` | months (Apr–Sep) with a negative statement (credit balance) |
| `CF_PAY_HABIT` | most frequent payment habit May–Sep (ties go to the riskier habit; `No bills due` if no positive statement) |
| `CF_NOISE_FLAGS` | number of noise flags N1–N5 the client trips (0–5) |
| `CF_IMPLAUSIBLE_DEFAULT` | 1 if the client defaulted and trips at least one noise flag |

Section globals all start with `S12_` / `s12_`; the long panel is `S12_PANEL`. Exports: `s12_code_money_profile`,
`s12_code_reading_verdicts`, `s12_code_money_contradictions`, `s12_status_x_paidshare_rates`, `s12_sep_code1_origin`,
`s12_habit_rates`, `s12_habit_count_rates`, `s12_noise_flags`, `s12_noise_floor`, `s12_column_reading_guide`.

## 13. The full association atlas: every column against every column

**Why this section matters for the story.** Sections 3-4 asked "what goes with default?". A dashboard, however, shows columns
*next to each other*, and a reader will assume that two charts are two pieces of evidence. This atlas maps the dependence between
**every pair of columns** (not only with the target), so that we know which fields are one signal seen six times, which
relationships are hidden from a correlation coefficient, which relationships *change* for defaulters, and how fast knowledge about
default saturates as columns are added.

| # | Question |
|---|---|
| 13.1 | How strongly is every column tied to every other column, on one common 0-1 scale, whatever the data types? |
| 13.2 | Which numeric relationships are invisible to Pearson (nonlinear or non-monotone)? |
| 13.3 | Does the relationship between two columns itself change for defaulters, or across status / credit-line strata? |
| 13.4 | Which links survive once everything else is held fixed (partial correlation, sparse graphical model)? Which columns touch `DEFAULT` directly? |
| 13.5 | Across months, which block leads and which follows (status, bill, payment)? |
| 13.6 | Which pairs tell more together than apart (synergy), and which merely repeat each other (redundancy)? |
| 13.7 | Into how many information "families" do the 47 columns collapse? |
| 13.8 | How much uncertainty about default is left after one column, two, ... eight? |
| 13.9 | Do extreme bills and extreme payments move together more than a correlation suggests (tail dependence)? |

Conventions: Section 7 already showed Pearson/Spearman clustermaps of numeric columns, VIF, a kNN mutual-information panel on a
subset and hierarchical feature families. Nothing of that is repeated; here every measure covers **all** columns of **all** types
and every pair is exported. No `df` columns are added by this section. All globals are prefixed `S13_` / `s13_`.

**Section guide**

- **What it does:** Measures the link between every pair of columns, not just each column and default. It finds columns that repeat each other, links that a plain correlation misses, links that change for defaulters, and how quickly extra columns stop adding knowledge about default.
- **Question it answers:** Which dashboard fields are really separate pieces of evidence, and which are one signal shown many times?
- **Inputs / new columns:** 23 raw columns, 24 setup features and `DEFAULT` (48 columns). No new `df` columns; results live in `s13_*` tables.
- **How to read the charts:** heatmaps: darker cell = stronger link on a 0-1 scale; blocks along the diagonal = families of redundant columns. Network: dots = columns, lines = direct links (red = positive, blue = negative, thicker = stronger). Staircase: bars = share of uncertainty about default removed as columns are added.
- **Terms:** mutual information (MI) = how much knowing one column tells you about another, for any shape of link. Cramér's V = link strength between categories (0-1). Partial correlation = link left after holding all other columns fixed. Synergy = a pair says more together than the sum of its parts. Tail dependence = whether extreme values occur together.
- **Key takeaways:**
  - Many fields repeat one signal: the 47 columns fall into 6 information families, and bill columns are the biggest hubs yet almost unrelated to default (0.00-0.03).
  - Once other columns are held fixed, only N_LATE, LATE_STREAK_RECENT and PAY_2 stay directly linked to default.
  - PAY_1 alone removes 14.2% of the uncertainty about default; the best three columns reach only 16.6%, so about 83% is left unexplained. Demographics plus limit reach 2.6%.
  - Pearson misses a lot: 134 numeric pairs look unrelated (|r| < 0.1) but have distance correlation of 0.2 or more, mostly ratio fields with extreme values.
  - The risky money corner is a top-10% August bill with a zero September payment: 57.7% default (142 clients).
- **Where it feeds:** the Tableau "any column vs any column" explorer (`s13_association_long.csv`), the network sheet, and the **information staircase** story chart (`s13_information_staircase.csv`).

### 13.0 Atlas helpers
Shared machinery: column typing and blocks, banding, a vectorised contingency engine (one-hot Gram matrix gives *all* pairwise
contingency tables in a single matrix product, on GPU through cupy when available), mutual information, bias-corrected Cramér's V,
rank-based correlation ratio, Benjamini-Hochberg.

In [ ]:
import networkx as nx
from scipy.cluster.hierarchy import linkage as s13_linkage_fn
from scipy.spatial.distance import squareform as s13_squareform
from matplotlib.patches import Patch as S13_Patch
from matplotlib.lines import Line2D as S13_Line2D

S13_T0 = time.time()
s13_rng = np.random.default_rng(13)

# ---------- optional GPU backend (guarded; CPU fallback everywhere) ----------
s13_xp, S13_GPU = np, False
if GPU:
    try:
        import cupy as s13_cp
        s13_tmp = float(s13_cp.zeros(2).sum())
        s13_xp, S13_GPU = s13_cp, True
    except Exception as s13_e:
        print("cupy not usable, falling back to numpy:", str(s13_e)[:120])
def s13_np(a):
    return a.get() if hasattr(a, "get") else np.asarray(a)
print("atlas backend:", "cupy (GPU)" if S13_GPU else "numpy (CPU)")

# ---------- column sets, types and blocks ----------
S13_DEMO = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"]
S13_RAW = S13_DEMO + PAY_COLS + BILL_COLS + PAMT_COLS                 # the 23 raw features
S13_ENG = list(FEATURE_COLS)                                          # 24 core engineered features (Section 0)
S13_RAWT = S13_RAW + [TARGET]
S13_ALL = S13_RAW + S13_ENG + [TARGET]
S13_NOMINAL = ["SEX", "EDUCATION", "MARRIAGE", TARGET]
S13_NUM = [c for c in S13_ALL if c not in S13_NOMINAL]
S13_BLOCK_COL = {"demographic": "#7b3294", "status": C_DEF, "bill": C_OK, "payment": C_ACC, "engineered": "#f1a340", "target": "#111111"}
def s13_block(c):
    if c == TARGET: return "target"
    if c in S13_DEMO or c in ("SEX_FEMALE", "EDU_GRAD", "EDU_HIGHSCHOOL", "MARRIED"): return "demographic"
    if c in PAY_COLS: return "status"
    if c in BILL_COLS: return "bill"
    if c in PAMT_COLS: return "payment"
    return "engineered"
def s13_type(c):
    return "nominal" if c in S13_NOMINAL else "numeric"

# working frame: numeric columns as they are, nominal columns with the readable folded labels
s13_X = df[S13_ALL].copy()
s13_X["SEX"] = df["SEX_LABEL"].astype(str)
s13_X["EDUCATION"] = df["EDU_LABEL"].astype(str)
s13_X["MARRIAGE"] = df["MAR_LABEL"].astype(str)
s13_y = df[TARGET].to_numpy().astype(int)
S13_N = len(df)

# ---------- banding ----------
def s13_band(s, q=10, status_groups=False):
    # integer codes 0..L-1. nominal -> levels; low-cardinality numeric -> own values; else quantile bands; NaN -> own level
    if status_groups and s.name in PAY_COLS:
        codes = pd.Categorical(s.map(status_group), categories=STATUS_GROUP_ORDER).codes
    elif s.dtype == object or s.nunique(dropna=True) <= (12 if q >= 10 else 7):
        codes = pd.factorize(s, sort=True)[0]
    else:
        codes = pd.qcut(s, q, labels=False, duplicates="drop").to_numpy(float)
        codes = np.where(np.isnan(codes), -1, codes)
    codes = np.asarray(codes).astype(int)
    if (codes < 0).any():
        codes = np.where(codes < 0, codes.max() + 1, codes)
    return np.unique(codes, return_inverse=True)[1].reshape(-1)       # compact (drops empty levels)

def s13_onehot(K):
    nlev = K.max(0) + 1
    off = np.concatenate([[0], np.cumsum(nlev)]).astype(int)
    O = np.zeros((K.shape[0], off[-1]), np.float32)
    rows = np.arange(K.shape[0])
    for j in range(K.shape[1]):
        O[rows, off[j] + K[:, j]] = 1.0
    return O, off

def s13_gram(O):
    # all pairwise contingency tables at once: block (i, j) of O'O is the crosstab of column i and column j
    if S13_GPU:
        try:
            Og = s13_xp.asarray(O)
            return s13_np(Og.T @ Og).astype(np.float64)
        except Exception:
            pass
    return (O.T @ O).astype(np.float64)

def s13_entropy(counts):
    c = np.asarray(counts, float).ravel(); c = c[c > 0]; p = c / c.sum()
    return float(-(p * np.log(p)).sum())

def s13_mi_ct(ct):
    # plug-in mutual information (nats) of a contingency table
    ct = np.asarray(ct, float); n = ct.sum()
    p = ct / n
    e = p.sum(1, keepdims=True) @ p.sum(0, keepdims=True)
    nz = p > 0
    return float((p[nz] * np.log(p[nz] / e[nz])).sum())

def s13_cramers_ct(ct):
    # bias-corrected Cramer's V (Bergsma 2013) from a contingency table
    ct = np.asarray(ct, float)
    ct = ct[ct.sum(1) > 0][:, ct.sum(0) > 0]
    n = ct.sum(); r, k = ct.shape
    if min(r, k) < 2: return np.nan
    e = ct.sum(1, keepdims=True) @ ct.sum(0, keepdims=True) / n
    chi2 = ((ct - e) ** 2 / e).sum()
    phi2 = max(0.0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return float(np.sqrt(phi2 / max(1e-12, min(kc - 1, rc - 1))))

def s13_eta(x, g):
    # correlation ratio of the RANKS of numeric x across the groups g (rank-based, so it is comparable with Spearman)
    x = np.asarray(x, float); m = ~np.isnan(x)
    r = stats.rankdata(x[m]); gg = np.asarray(g)[m]
    tot = ((r - r.mean()) ** 2).sum()
    cnt = np.bincount(gg); sm = np.bincount(gg, weights=r); ok = cnt > 0
    ssb = (cnt[ok] * (sm[ok] / cnt[ok] - r.mean()) ** 2).sum()
    return float(np.sqrt(ssb / tot)) if tot > 0 else np.nan

def s13_spearman(frame):
    # Spearman matrix = Pearson on ranks. With missing values pandas uses pairwise-complete rows on the pre-computed ranks
    # (only AVG_PAY_RATIO and PAY_TO_BILL_6M have missing values, so the approximation touches two columns).
    R = frame.rank()
    if not R.isna().any().any():
        return pd.DataFrame(np.corrcoef(R.to_numpy().T), index=frame.columns, columns=frame.columns)
    return R.corr()

def s13_bh(p):
    # Benjamini-Hochberg adjusted p-values
    p = np.asarray(p, float); n = len(p); o = np.argsort(p)
    adj = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1]
    out = np.empty(n); out[o] = np.clip(adj, 0, 1)
    return out

def s13_cluster_order(A):
    D = 1 - np.asarray(A, float); D = np.clip((D + D.T) / 2, 0, None); np.fill_diagonal(D, 0)
    return s13_linkage_fn(s13_squareform(D, checks=False), method="average")

def s13_clustermap(A, title, name, annot=False, size=(12, 11), cmap="mako_r", vmax=1.0):
    Z = s13_cluster_order(A)
    rc = pd.Series({c: S13_BLOCK_COL[s13_block(c)] for c in A.index}, name="block")
    g = sns.clustermap(A, row_linkage=Z, col_linkage=Z, cmap=cmap, vmin=0, vmax=vmax, annot=annot, fmt=".2f",
                       annot_kws={"size": 5.5}, figsize=size, dendrogram_ratio=.09, row_colors=rc, col_colors=rc,
                       cbar_pos=(0.01, 0.87, 0.02, 0.1), linewidths=.2 if len(A) < 30 else 0,
                       xticklabels=True, yticklabels=True)
    g.ax_heatmap.tick_params(labelsize=7 if len(A) < 30 else 6)
    g.ax_heatmap.legend(handles=[S13_Patch(color=v, label=k) for k, v in S13_BLOCK_COL.items()], ncol=6, fontsize=7,
                        loc="upper center", bbox_to_anchor=(0.5, -0.13), frameon=False)
    g.fig.suptitle(title, y=1.01, fontweight="bold")
    savefig(name, g.fig); plt.show()

S13_HD = s13_entropy(np.bincount(s13_y))            # entropy of DEFAULT in nats
print(f"columns: {len(S13_RAW)} raw + {len(S13_ENG)} engineered + target = {len(S13_ALL)} | numeric {len(S13_NUM)}, nominal {len(S13_NOMINAL)}")
print(f"H(DEFAULT) = {S13_HD:.4f} nats = {S13_HD / np.log(2):.4f} bits (base rate {BASE_RATE:.2%})")

### 13.1 How strongly is every column tied to every other column?
One matrix, one 0-1 scale, the right measure for each pair of types:

| Pair | Measure | Note |
|---|---|---|
| numeric - numeric | \|Spearman ρ\| | status codes `PAY_i` are treated as ordinal numerics |
| nominal - nominal | bias-corrected Cramér's V | `SEX`, `EDUCATION` (folded to 4 labels), `MARRIAGE` (3 labels), `DEFAULT` |
| numeric - nominal | correlation ratio η on ranks | share of rank variance between groups, square-rooted |
| every pair | normalised mutual information | numerics cut into deciles (≤ 12 distinct values kept as is, missing = own band); NMI = I / √(H_a·H_b) |

The three "classical" measures are not perfectly interchangeable (η and V are unsigned and tend to be a little smaller than a
Spearman of the same strength), which is exactly why NMI is computed for **all** pairs as a type-free cross-check.
Computed on all 30,000 rows.

In [ ]:
def s13_assoc_matrices(cols):
    X = s13_X[cols]; p = len(cols)
    num = [c for c in cols if c not in S13_NOMINAL]
    sp = s13_spearman(X[num]); pe = X[num].corr()
    K = np.column_stack([s13_band(X[c], 10) for c in cols])
    O, off = s13_onehot(K); G = s13_gram(O)
    H = np.array([s13_entropy(np.diag(G[off[i]:off[i + 1], off[i]:off[i + 1]])) for i in range(p)])
    MI = np.zeros((p, p)); V = np.eye(p); A = np.eye(p); ETA = np.full((p, p), np.nan)
    kind = np.full((p, p), "self", dtype=object)
    for i in range(p):
        MI[i, i] = H[i]
        for j in range(i + 1, p):
            ct = G[off[i]:off[i + 1], off[j]:off[j + 1]]
            MI[i, j] = MI[j, i] = s13_mi_ct(ct)
            V[i, j] = V[j, i] = s13_cramers_ct(ct)
            a, b = cols[i], cols[j]
            na, nb = a in S13_NOMINAL, b in S13_NOMINAL
            if not na and not nb:
                val, kd = abs(sp.loc[a, b]), "abs_spearman"
            elif na and nb:
                val, kd = V[i, j], "cramers_v"
            else:
                xn, gc = (b, i) if na else (a, j)
                val = s13_eta(X[xn].to_numpy(float), K[:, gc]); kd = "eta_rank"
                ETA[i, j] = ETA[j, i] = val
            A[i, j] = A[j, i] = val; kind[i, j] = kind[j, i] = kd
    NMI = MI / np.sqrt(np.outer(H, H))
    mk = lambda M: pd.DataFrame(M, index=cols, columns=cols)
    return {"assoc": mk(A), "nmi": mk(NMI), "mi": mk(MI), "v_banded": mk(V), "eta": mk(ETA), "kind": mk(kind),
            "spearman": sp, "pearson": pe, "H": pd.Series(H, index=cols), "K": K, "levels": pd.Series(K.max(0) + 1, index=cols)}

s13_t = time.time()
s13_M = s13_assoc_matrices(S13_ALL)
s13_A, s13_NMI, s13_MI = s13_M["assoc"], s13_M["nmi"], s13_M["mi"]
s13_A_raw = s13_A.loc[S13_RAWT, S13_RAWT]; s13_NMI_raw = s13_NMI.loc[S13_RAWT, S13_RAWT]
print(f"{len(S13_ALL)} columns -> {len(S13_ALL) * (len(S13_ALL) - 1) // 2:,} pairs, all four measures in {time.time() - s13_t:.1f}s")
print("bands per column (deciles unless fewer distinct values):")
print(s13_M["levels"].to_frame("bands").T.to_string())
s13_kind_counts = pd.Series(s13_M["kind"].to_numpy()[np.triu_indices(len(S13_ALL), 1)]).value_counts()
print("\npairs by measure used in the unified matrix:"); print(s13_kind_counts.to_string())

#### 13.1.1 Unified association matrix, 23 raw features + `DEFAULT` (clustered)
Rows and columns are ordered by average-linkage clustering on 1 − association; the side strip shows the block of each column.
Tableau: a highlight table on `s13_association_long` (columns `var_a`, `var_b`, colour = `association`).

In [ ]:
s13_clustermap(s13_A_raw, "13.1 Unified association (|Spearman| / Cramer's V / eta) - 23 raw features + DEFAULT",
               "s13_assoc_raw", annot=True, size=(13, 12))
s13_iu = np.triu_indices(len(S13_RAWT), 1)
s13_raw_pairs = pd.DataFrame({"var_a": np.array(S13_RAWT)[s13_iu[0]], "var_b": np.array(S13_RAWT)[s13_iu[1]],
                              "association": s13_A_raw.to_numpy()[s13_iu], "nmi": s13_NMI_raw.to_numpy()[s13_iu]})
s13_raw_pairs["block_pair"] = [" x ".join(sorted([s13_block(a), s13_block(b)])) for a, b in zip(s13_raw_pairs.var_a, s13_raw_pairs.var_b)]
s13_bp = s13_raw_pairs.groupby("block_pair").agg(pairs=("association", "size"), mean_assoc=("association", "mean"),
                                                 median_assoc=("association", "median"), max_assoc=("association", "max"),
                                                 mean_nmi=("nmi", "mean")).sort_values("mean_assoc", ascending=False)
print("Raw columns: association by block pair (276 pairs)")
print(s13_bp.round(3).to_string())
print(f"\nshare of raw pairs with association >= 0.5: {(s13_raw_pairs.association >= .5).mean():.1%} | >= 0.3: {(s13_raw_pairs.association >= .3).mean():.1%} | < 0.05: {(s13_raw_pairs.association < .05).mean():.1%}")

**Observations:**
- Bills move together almost perfectly (average link 0.84 inside the bill block); status codes come next (0.67).
- Across blocks, bills link to status (0.51) and to payments (0.51), but payments barely link to status (0.13).
- Links to DEFAULT are weak: status block 0.20 on average (max 0.29), payments 0.14, demographics 0.06, bills 0.01.
- 29% of raw pairs have a link of 0.5 or more, and 22% are below 0.05.
- Story use: background. It warns that many dashboard charts would repeat the same bill information.

#### 13.1.2 Unified association matrix, raw + core engineered features (48 columns)
Adds the 24 Section-0 engineered features. Too dense to annotate; the numbers are in the export. Look for engineered features
that sit *between* raw blocks (they bridge blocks) versus those that simply join an existing block.

In [ ]:
s13_clustermap(s13_A, "13.1 Unified association - raw + engineered + DEFAULT (48 columns, 1,128 pairs)",
               "s13_assoc_all", annot=False, size=(15, 14))
s13_bridge = []
for c in S13_ENG:
    row = s13_A.loc[c, S13_RAW]
    byb = row.groupby([s13_block(k) for k in row.index]).max()
    s13_bridge.append({"feature": c, **byb.to_dict(), "strongest_raw_partner": row.idxmax(), "assoc": row.max(),
                       "blocks_touched_>=0.4": int((byb >= .4).sum()), "assoc_with_DEFAULT": s13_A.loc[c, TARGET]})
s13_bridge = pd.DataFrame(s13_bridge).sort_values(["blocks_touched_>=0.4", "assoc_with_DEFAULT"], ascending=False)
print("Engineered features: strongest association with each raw block (max over the block)")
print(s13_bridge.round(3).to_string(index=False))

**Observations:**
- TOTAL_BILL and AVG_BILL are near-copies of single bill months (0.94 with BILL_AMT3) and say almost nothing about default (0.03).
- The lateness counts (N_LATE, LATE_STREAK_MAX, LATE_STREAK_RECENT) link only to the status block, yet have the strongest link to default (0.38-0.39), above PAY_1's 0.29.
- Ratio features such as AVG_UTIL and AVG_PAY_RATIO bridge blocks: they link to bills (0.72-0.77) and to status (0.66-0.71) at the same time.
- Story use: choose lateness counts over bill totals for the dashboard, because the bill fields are redundant.

#### 13.1.3 Normalised mutual information, every pair
NMI makes no assumption about type or shape. Left: 24 raw columns annotated; right panel (second figure): all 48.
Because deciles lose resolution, NMI values are numerically smaller than correlations; compare ranks, not levels.

In [ ]:
s13_clustermap(s13_NMI_raw, "13.1 Normalised mutual information (deciles) - 23 raw features + DEFAULT", "s13_nmi_raw",
               annot=True, size=(13, 12), cmap="rocket_r", vmax=float(np.nanquantile(s13_NMI_raw.to_numpy()[s13_iu], .99)))
s13_iu_all = np.triu_indices(len(S13_ALL), 1)
s13_clustermap(s13_NMI, "13.1 Normalised mutual information - all 48 columns", "s13_nmi_all", annot=False, size=(15, 14),
               cmap="rocket_r", vmax=float(np.nanquantile(s13_NMI.to_numpy()[s13_iu_all], .97)))
s13_bias = ((s13_M["levels"].to_numpy()[:, None] - 1) * (s13_M["levels"].to_numpy()[None, :] - 1)) / (2 * S13_N)
print(f"finite-sample MI bias (levels-1)(levels-1)/2n: median {np.median(s13_bias[s13_iu_all]):.5f} nats, max {s13_bias[s13_iu_all].max():.5f} nats "
      f"(vs median observed MI {np.median(s13_MI.to_numpy()[s13_iu_all]):.4f}) -> negligible at n = {S13_N:,}")

**Observations:**
- The heatmaps are images. The printed check shows that small-sample bias in mutual information is at most 0.0017 nats, against a median observed value of 0.112, so the pattern is real.
- From the block table in 13.1.1: mutual information is highest inside the status block (0.39) and the bill block (0.39), and tiny between demographics and default (0.004).
- Story use: background only.

#### 13.1.4 Tidy export of all pairs with every measure (`s13_association_long`)
Both orders of every pair (48 × 47 rows), so a Tableau parameter "column A" plus a sorted bar of `association` gives an
"any column versus any column" explorer without any data blending. `share_of_a_explained_by_b` is the asymmetric uncertainty
coefficient I(a;b)/H(a). The printed table lists the strongest **cross-block** pairs (same-block pairs are trivially strong).

In [ ]:
s13_rows = []
s13_levels = s13_M["levels"]
for a in S13_ALL:
    for b in S13_ALL:
        if a == b: continue
        both_num = (a in S13_NUM) and (b in S13_NUM)
        s13_rows.append({
            "var_a": a, "var_b": b, "block_a": s13_block(a), "block_b": s13_block(b), "type_a": s13_type(a), "type_b": s13_type(b),
            "same_block": s13_block(a) == s13_block(b), "in_raw_set": (a in S13_RAWT) and (b in S13_RAWT),
            "measure": s13_M["kind"].loc[a, b], "association": s13_A.loc[a, b],
            "spearman": s13_M["spearman"].loc[a, b] if both_num else np.nan,
            "pearson": s13_M["pearson"].loc[a, b] if both_num else np.nan,
            "eta_rank": s13_M["eta"].loc[a, b], "cramers_v_banded": s13_M["v_banded"].loc[a, b],
            "mi_nats": s13_MI.loc[a, b], "nmi": s13_NMI.loc[a, b],
            "share_of_a_explained_by_b": s13_MI.loc[a, b] / s13_M["H"][a]})
s13_long = pd.DataFrame(s13_rows)
s13_long["rank_for_a"] = s13_long.groupby("var_a")["association"].rank(ascending=False, method="first").astype(int)
export(s13_long, "s13_association_long")

s13_half = s13_long[s13_long.var_a < s13_long.var_b]
s13_cross = s13_half[~s13_half.same_block & (s13_half.var_a != TARGET) & (s13_half.var_b != TARGET)]
print("Top 25 cross-block pairs (raw x raw only):")
print(s13_cross[s13_cross.in_raw_set].nlargest(25, "association")[["var_a", "var_b", "block_a", "block_b", "measure", "association", "spearman", "nmi"]].round(3).to_string(index=False))
print("\nTop 15 pairs involving a nominal column (excluding DEFAULT):")
s13_nom = s13_half[((s13_half.type_a == "nominal") | (s13_half.type_b == "nominal")) & (s13_half.var_a != TARGET) & (s13_half.var_b != TARGET)]
print(s13_nom.nlargest(15, "association")[["var_a", "var_b", "measure", "association", "nmi"]].round(3).to_string(index=False))
s13_top_cross = s13_cross[s13_cross.in_raw_set].nlargest(1, "association").iloc[0]
add_fact("13.1", "association atlas", f"Across {len(S13_ALL)} columns ({len(s13_half):,} pairs) {(s13_half.association >= .5).mean():.0%} of pairs have association >= 0.5; "
         f"the strongest cross-block raw pair is {s13_top_cross.var_a} x {s13_top_cross.var_b} ({s13_top_cross.association:.2f})",
         float((s13_half.association >= .5).mean()), "s13_assoc_all", 2)

**Observations:**
- The strongest links across blocks are bill-status and bill-payment pairs, between 0.55 and 0.67 (top: BILL_AMT5 x PAY_6 at 0.67).
- Bill-payment pairs peak one month apart (for example BILL_AMT6 x PAY_AMT5 at 0.67), matching the rule that a payment settles last month's bill.
- Among category columns the only notable link is AGE x MARRIAGE (0.50). Education links to limit at 0.28 and to everything else below 0.23.
- Story use: background. The CSV powers a "pick any column" explorer sheet in Tableau.

#### 13.1.5 Which columns are hubs and which are loners?
For each column: mean association with all other feature columns (connectedness), the strongest partner, and the association
with `DEFAULT`. A hub that is also tied to `DEFAULT` is a headline field; a loner tied to `DEFAULT` carries *unique* information;
a loner not tied to `DEFAULT` is dashboard decoration.

In [ ]:
s13_feat = [c for c in S13_ALL if c != TARGET]
s13_Af = s13_A.loc[s13_feat, s13_feat].copy()
s13_Af = s13_Af.mask(np.eye(len(s13_Af), dtype=bool))
s13_hub = pd.DataFrame({"block": [s13_block(c) for c in s13_feat], "mean_assoc": s13_Af.mean(1), "n_partners_>=0.5": (s13_Af >= .5).sum(1),
                        "strongest_partner": s13_Af.idxmax(1), "max_assoc": s13_Af.max(1),
                        "assoc_with_DEFAULT": s13_A.loc[s13_feat, TARGET], "nmi_with_DEFAULT": s13_NMI.loc[s13_feat, TARGET]})
s13_hub = s13_hub.sort_values("mean_assoc", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(16, 10), gridspec_kw={"width_ratios": [1, 1.1]})
s13_h = s13_hub.iloc[::-1]
axes[0].barh(s13_h.index, s13_h["mean_assoc"], color=[S13_BLOCK_COL[b] for b in s13_h["block"]])
axes[0].set_title("Connectedness: mean association with the other 46 columns"); axes[0].tick_params(axis="y", labelsize=7)
axes[0].legend(handles=[S13_Patch(color=v, label=k) for k, v in S13_BLOCK_COL.items() if k != "target"], fontsize=8, loc="lower right")
axes[1].scatter(s13_hub["mean_assoc"], s13_hub["assoc_with_DEFAULT"], c=[S13_BLOCK_COL[b] for b in s13_hub["block"]], s=60, edgecolor="white")
for c, r in s13_hub.iterrows():
    axes[1].annotate(c, (r["mean_assoc"], r["assoc_with_DEFAULT"]), fontsize=6.5, xytext=(3, 2), textcoords="offset points")
axes[1].axvline(s13_hub["mean_assoc"].median(), ls=":", color="k", lw=1); axes[1].axhline(s13_hub["assoc_with_DEFAULT"].median(), ls=":", color="k", lw=1)
axes[1].set_xlabel("mean association with other columns (hub-ness)"); axes[1].set_ylabel("association with DEFAULT")
axes[1].set_title("Hub or loner, relevant or not (dotted = medians)")
plt.tight_layout(); savefig("s13_hubs_loners"); plt.show()
print(s13_hub.round(3).to_string())
export(s13_hub.rename_axis("column").reset_index(), "s13_column_connectedness")

**Observations:**
- Bill columns are the biggest hubs (average link 0.43-0.48 with all other columns) but have almost no link to default (0.00-0.03).
- Lateness counts are the opposite: few partners (average 0.20-0.23) but the strongest links to default (N_LATE 0.39, LATE_STREAK_RECENT 0.38).
- LIMIT_BAL is nearly a loner (average 0.21, no partner above 0.5) with a moderate default link (0.17), so it adds information no other column has.
- SEX is the most isolated column overall.
- Story use: a guide for choosing dashboard fields: lateness and limit, not bill totals.

#### 13.1.6 Do the classical measures and mutual information agree?
Each dot is one of the 1,128 pairs. Pairs far **above** the trend have more mutual information than their rank correlation
suggests: candidates for nonlinear or non-monotone dependence, examined properly in 13.2.

In [ ]:
s13_ag = s13_half.copy()
s13_ag["pair_type"] = s13_ag["measure"].map({"abs_spearman": "numeric-numeric", "eta_rank": "numeric-nominal", "cramers_v": "nominal-nominal"})
s13_fit_x = s13_ag["association"].clip(0, .999)
s13_ag["nmi_expected"] = np.interp(s13_fit_x, *[v.to_numpy() for v in (lambda g: (g["association"], g["nmi"]))(
    s13_ag.assign(b=pd.qcut(s13_fit_x, 25, duplicates="drop")).groupby("b", observed=True)[["association", "nmi"]].median())])
s13_ag["nmi_excess"] = s13_ag["nmi"] - s13_ag["nmi_expected"]
fig, ax = plt.subplots(figsize=(10, 6))
for k, colr in [("numeric-numeric", C_OK), ("numeric-nominal", "#f1a340"), ("nominal-nominal", C_DEF)]:
    g = s13_ag[s13_ag.pair_type == k]
    ax.scatter(g["association"], g["nmi"], s=12, alpha=.55, color=colr, label=f"{k} ({len(g)})")
s13_o = np.argsort(s13_ag["association"].to_numpy())
ax.plot(s13_ag["association"].to_numpy()[s13_o], s13_ag["nmi_expected"].to_numpy()[s13_o], color="k", lw=1.5, label="median trend")
for _, r in s13_ag.nlargest(8, "nmi_excess").iterrows():
    ax.annotate(f"{r.var_a} x {r.var_b}", (r.association, r.nmi), fontsize=6.5, xytext=(4, 2), textcoords="offset points")
ax.set_xlabel("unified association (|Spearman| / eta / V)"); ax.set_ylabel("normalised mutual information"); ax.legend(fontsize=8)
ax.set_title("13.1 Agreement of classical association and mutual information (all pairs)")
plt.tight_layout(); savefig("s13_measure_agreement"); plt.show()
print(f"Spearman rank agreement between the two measures over all pairs: {stats.spearmanr(s13_ag.association, s13_ag.nmi)[0]:.3f}")
print("Pairs with most MI beyond what their rank correlation implies:")
print(s13_ag.nlargest(15, "nmi_excess")[["var_a", "var_b", "measure", "association", "nmi", "nmi_expected", "nmi_excess"]].round(3).to_string(index=False))

**Observations:**
- The two measures mostly agree on which pairs are strongly linked (rank agreement 0.84).
- MAX_DELAY x N_REVOLVE has almost no rank correlation (0.04) but clear mutual information (0.39): a real link that is not a straight-line trend.
- PAY_1 x PAY_2 has mutual information 0.61 where its correlation would predict 0.20, because status codes behave like categories rather than numbers.
- Story use: background. It leads into the non-linear checks in 13.2.

### 13.2 Which numeric relationships are invisible to Pearson?
For all 946 pairs of the 44 numeric columns we compare four lenses:

* **Pearson r** (linear), **Spearman ρ** (monotone), both on all 30,000 rows;
* **distance correlation** (dCor, Székely 2007), which is zero only under independence. Implemented in numpy
  (double-centred distance matrices, batched; cupy if a GPU is present) on a **2,500-row random sample** of rows that are
  complete on all 44 columns, computed on percentile ranks so that a handful of huge balances cannot dominate;
* **mutual information** on deciles, converted to the correlation scale with Linfoot's r_MI = √(1 − e^(−2I)).

Two gaps separate two different failures of Pearson: `gap_monotone` = |ρ| − |r| (monotone but curved or heavy-tailed), and
`gap_nonmonotone` = dCor − |ρ| (dependence that is not even monotone: U-shapes, fans, mixtures).
`nonlinear_score` = dCor − |r| is the overall "hidden from Pearson" score.

In [ ]:
S13_DC_N = 2500
def s13_centred(x, xp):
    x = xp.asarray(x, dtype=xp.float32)
    d = xp.abs(x[:, None] - x[None, :])
    d -= d.mean(0, keepdims=True)
    d -= d.mean(1, keepdims=True)
    return d.ravel()

def s13_dcor_matrix(Z, xp=np, batch=8):
    # Z: n x p (float). Returns p x p distance-correlation matrix. Memory: batch * n^2 * 4 bytes (200 MB at n=2500, batch=8).
    n, p = Z.shape
    dcov2 = np.zeros((p, p))
    for b0 in range(0, p, batch):
        idx = list(range(b0, min(p, b0 + batch)))
        M = xp.stack([s13_centred(Z[:, i], xp) for i in idx])
        for j in range(b0, p):
            v = s13_np(M @ s13_centred(Z[:, j], xp)) / (n * n)
            dcov2[idx, j] = v; dcov2[j, idx] = v
        del M
    dvar = np.sqrt(np.clip(np.diag(dcov2), 1e-30, None))
    return np.sqrt(np.clip(dcov2, 0, None) / np.outer(dvar, dvar))

s13_t = time.time()
s13_complete = s13_X[S13_NUM].notna().all(1).to_numpy()
s13_dc_rows = s13_rng.choice(np.flatnonzero(s13_complete), S13_DC_N, replace=False)
s13_Zdc = s13_X.iloc[s13_dc_rows][S13_NUM].rank(pct=True).to_numpy(np.float32)
try:
    s13_DC = s13_dcor_matrix(s13_Zdc, s13_xp)
except Exception as s13_e:                       # GPU out of memory or similar -> CPU
    print("GPU dCor failed, using numpy:", str(s13_e)[:100]); s13_DC = s13_dcor_matrix(s13_Zdc, np)
s13_DC = pd.DataFrame(s13_DC, index=S13_NUM, columns=S13_NUM)
print(f"distance correlation: {len(S13_NUM)} columns, {len(S13_NUM) * (len(S13_NUM) - 1) // 2} pairs, n = {S13_DC_N:,} "
      f"(of {int(s13_complete.sum()):,} complete rows) in {time.time() - s13_t:.1f}s on {'GPU' if S13_GPU else 'CPU'}")

s13_iun = np.triu_indices(len(S13_NUM), 1)
s13_nl = pd.DataFrame({"var_a": np.array(S13_NUM)[s13_iun[0]], "var_b": np.array(S13_NUM)[s13_iun[1]],
                       "pearson": s13_M["pearson"].loc[S13_NUM, S13_NUM].to_numpy()[s13_iun],
                       "spearman": s13_M["spearman"].loc[S13_NUM, S13_NUM].to_numpy()[s13_iun],
                       "dcor": s13_DC.to_numpy()[s13_iun],
                       "mi_nats": s13_MI.loc[S13_NUM, S13_NUM].to_numpy()[s13_iun],
                       "nmi": s13_NMI.loc[S13_NUM, S13_NUM].to_numpy()[s13_iun]})
s13_nl["r_mi"] = np.sqrt(1 - np.exp(-2 * s13_nl["mi_nats"]))
s13_nl["gap_monotone"] = s13_nl["spearman"].abs() - s13_nl["pearson"].abs()
s13_nl["gap_nonmonotone"] = s13_nl["dcor"] - s13_nl["spearman"].abs()
s13_nl["nonlinear_score"] = s13_nl["dcor"] - s13_nl["pearson"].abs()
s13_nl["block_a"] = s13_nl["var_a"].map(s13_block); s13_nl["block_b"] = s13_nl["var_b"].map(s13_block)
export(s13_nl, "s13_linear_vs_nonlinear_pairs")
print("\nHow the four lenses relate over all numeric pairs (Spearman rank correlation between measures):")
print(s13_nl[["pearson", "spearman", "dcor", "r_mi"]].abs().corr("spearman").round(3).to_string())
print(f"\npairs with |pearson| < 0.1: {(s13_nl.pearson.abs() < .1).sum()} | of those with dCor >= 0.2: {((s13_nl.pearson.abs() < .1) & (s13_nl.dcor >= .2)).sum()} "
      f"| with r_MI >= 0.3: {((s13_nl.pearson.abs() < .1) & (s13_nl.r_mi >= .3)).sum()}")

#### 13.2.1 The pairs Pearson misses
Three rankings: (a) overall hidden dependence (`nonlinear_score`), (b) monotone but badly non-linear (`gap_monotone`),
(c) genuinely non-monotone (`gap_nonmonotone`, requiring dCor ≥ 0.15 so that noise is not ranked).

In [ ]:
s13_show = ["var_a", "var_b", "pearson", "spearman", "dcor", "r_mi", "gap_monotone", "gap_nonmonotone", "nonlinear_score"]
print("(a) Top 20 by nonlinear_score = dCor - |Pearson|")
print(s13_nl.nlargest(20, "nonlinear_score")[s13_show].round(3).to_string(index=False))
print("\n(b) Top 12 monotone-but-nonlinear: |Spearman| - |Pearson|")
print(s13_nl.nlargest(12, "gap_monotone")[s13_show].round(3).to_string(index=False))
print("\n(c) Top 15 non-monotone: dCor - |Spearman| (dCor >= 0.15)")
s13_nonmono = s13_nl[s13_nl.dcor >= .15].nlargest(15, "gap_nonmonotone")
print(s13_nonmono[s13_show].round(3).to_string(index=False))
print("\n(d) Pearson overstates (|Pearson| - |Spearman| largest): driven by a few extreme accounts")
print(s13_nl.nsmallest(8, "gap_monotone")[s13_show].round(3).to_string(index=False))
s13_n_hidden = int(((s13_nl.pearson.abs() < .1) & (s13_nl.dcor >= .2)).sum())
s13_top_nl = s13_nl.nlargest(1, "nonlinear_score").iloc[0]
add_fact("13.2", "nonlinearity", f"{s13_n_hidden} of {len(s13_nl)} numeric pairs look unrelated to Pearson (|r| < 0.1) yet have distance correlation >= 0.2; "
         f"the most hidden is {s13_top_nl.var_a} x {s13_top_nl.var_b} (r = {s13_top_nl.pearson:.2f}, rho = {s13_top_nl.spearman:.2f}, dCor = {s13_top_nl.dcor:.2f})",
         s13_n_hidden, "s13_nonlinear_top9", 2)

**Observations:**
- 134 of 946 numeric pairs look unrelated to Pearson (|r| below 0.1) but have distance correlation of 0.2 or more. Distance correlation catches any shape of link.
- The extreme case is AVG_PAY_RATIO x PAY_TO_BILL_6M: Pearson 0.11 but Spearman 0.93. A few huge ratio values hide a near-perfect rank link.
- PAY_TO_BILL_6M appears in 18 of the top 20 pairs. With BILL_AMT1, for example, Pearson is -0.05 but Spearman -0.72.
- MAX_DELAY x N_REVOLVE is truly non-monotone (it rises and then falls): Spearman 0.04, distance correlation 0.53.
- Story use: method note. In Tableau, use ranked or banded views for ratio fields, not raw scatter plots.

#### 13.2.2 Gap matrices
Lower triangle: `gap_monotone` (|ρ| − |r|); upper triangle: `gap_nonmonotone` (dCor − |ρ|). Blue cells in the lower triangle are
pairs where Pearson *over*states (outlier-driven); red cells in the upper triangle are non-monotone relationships.

In [ ]:
s13_pn = len(S13_NUM)
s13_gapM = np.zeros((s13_pn, s13_pn))
s13_gapM[s13_iun] = s13_nl["gap_nonmonotone"].to_numpy()
s13_gapM[(s13_iun[1], s13_iun[0])] = s13_nl["gap_monotone"].to_numpy()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(pd.DataFrame(s13_gapM, index=S13_NUM, columns=S13_NUM), cmap=CMAP_DIV, center=0, vmin=-.5, vmax=.5, ax=ax,
            xticklabels=True, yticklabels=True, cbar_kws={"shrink": .6, "label": "gap"})
ax.tick_params(labelsize=7)
ax.set_title("13.2 Lower triangle: |Spearman| - |Pearson|   |   Upper triangle: dCor - |Spearman|")
plt.tight_layout(); savefig("s13_gap_matrix"); plt.show()
s13_gap_by_block = s13_nl.assign(bp=[" x ".join(sorted([a, b])) for a, b in zip(s13_nl.block_a, s13_nl.block_b)]).groupby("bp")[
    ["gap_monotone", "gap_nonmonotone", "nonlinear_score"]].agg(["mean", "max"]).round(3)
print(s13_gap_by_block.to_string())

**Observations:**
- Pearson understates bill-payment links by 0.32 on average (Spearman minus Pearson), because a few huge payments dominate.
- Payment-payment pairs show the same gap (0.32); bill-bill pairs show none (-0.05).
- Truly non-monotone links are rare. The largest average gap of that kind is bill x demographic, at 0.12.
- Story use: background. Supports using log or banded money axes.

#### 13.2.3 The nine most "hidden" relationships, drawn
Top pairs by `nonlinear_score`, each column allowed at most twice so that one column does not fill the grid. Axes are percentile
ranks (the copula scale); hexbin = density on all rows, line = mean percentile of *y* in 20 equal-count bins of *x*.
A flat line with a structured cloud means a non-monotone link; a bent line means monotone but non-linear.

In [ ]:
def s13_pick_diverse(tbl, k, max_use=2):
    used, out = {}, []
    for _, r in tbl.iterrows():
        if used.get(r.var_a, 0) >= max_use or used.get(r.var_b, 0) >= max_use: continue
        out.append(r); used[r.var_a] = used.get(r.var_a, 0) + 1; used[r.var_b] = used.get(r.var_b, 0) + 1
        if len(out) == k: break
    return pd.DataFrame(out)

s13_top9 = s13_pick_diverse(s13_nl.sort_values("nonlinear_score", ascending=False), 9)
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
for ax, (_, r) in zip(axes.ravel(), s13_top9.iterrows()):
    xy = s13_X[[r.var_a, r.var_b]].dropna()
    u, v = xy[r.var_a].rank(pct=True), xy[r.var_b].rank(pct=True)
    ax.hexbin(u, v, gridsize=35, cmap="Greys", bins="log", mincnt=1)
    bins = pd.qcut(u.rank(method="first"), 20, labels=False)
    bm = pd.DataFrame({"u": u, "v": v, "b": bins}).groupby("b").agg(u=("u", "mean"), v=("v", "mean"))
    ax.plot(bm["u"], bm["v"], color=C_DEF, lw=2.5, marker="o", ms=3)
    ax.set_xlabel(f"{r.var_a} (percentile)"); ax.set_ylabel(f"{r.var_b} (percentile)")
    ax.set_title(f"{r.var_a} x {r.var_b}\nr={r.pearson:.2f}  rho={r.spearman:.2f}  dCor={r.dcor:.2f}  r_MI={r.r_mi:.2f}", fontsize=9)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
fig.suptitle("13.2 Nine relationships Pearson does not see (percentile scale; red = binned mean)", fontweight="bold", y=1.0)
plt.tight_layout(); savefig("s13_nonlinear_top9"); plt.show()
print(s13_top9[s13_show].round(3).to_string(index=False))

**Observations:**
- The drawn pairs include bill-payment pairs such as BILL_AMT6 x PAY_AMT6 (Pearson 0.12, Spearman 0.53) and N_ZERO_PAY x PAY_AMT6 (-0.12 vs -0.57).
- MAX_DELAY x N_REVOLVE (distance correlation 0.53, Spearman 0.04) and PAY_1 x N_REVOLVE (0.59 vs 0.21) are hump-shaped rather than straight.
- A likely reason: revolving (code 0) sits between paying in full (-1) and being late (1+), so months of revolving peak at middle delay values.
- Story use: background. One hump chart could illustrate it.

### 13.3 Does the relationship itself change for defaulters?
So far `DEFAULT` was one more column. Here it is a **switch**: the 44 × 44 Spearman matrix is computed separately for defaulters
and non-defaulters. A pair whose correlation differs by outcome is a relationship whose *meaning* depends on the outcome
(statistically, an interaction that no single-column chart can show).
Test: Fisher z on each of the 946 pairs (variance factor 1.06 for Spearman), Benjamini-Hochberg FDR.
Near-deterministic pairs (pooled |ρ| > 0.98, such as `TOTAL_BILL` and `AVG_BILL`) are excluded.
Caveat: defaulters are a selected group, so a difference is descriptive, not causal.

In [ ]:
s13_m1, s13_m0 = s13_y == 1, s13_y == 0
s13_R1 = s13_spearman(s13_X.loc[s13_m1, S13_NUM]); s13_R0 = s13_spearman(s13_X.loc[s13_m0, S13_NUM])
s13_Rd = s13_R1 - s13_R0
s13_Rpool = s13_M["spearman"].loc[S13_NUM, S13_NUM]
s13_ord = s13_cluster_order(s13_Rpool.abs().fillna(0))
from scipy.cluster.hierarchy import leaves_list as s13_leaves
s13_o = [S13_NUM[i] for i in s13_leaves(s13_ord)]
fig, axes = plt.subplots(1, 3, figsize=(24, 8.5))
for ax, M, ttl in [(axes[0], s13_R0, f"Non-defaulters (n={int(s13_m0.sum()):,})"), (axes[1], s13_R1, f"Defaulters (n={int(s13_m1.sum()):,})")]:
    sns.heatmap(M.loc[s13_o, s13_o], cmap=CMAP_DIV, vmin=-1, vmax=1, center=0, ax=ax, cbar=False, xticklabels=True, yticklabels=True)
    ax.set_title(f"Spearman - {ttl}"); ax.tick_params(labelsize=5.5)
sns.heatmap(s13_Rd.loc[s13_o, s13_o], cmap="PuOr_r", vmin=-.4, vmax=.4, center=0, ax=axes[2], xticklabels=True, yticklabels=True,
            cbar_kws={"shrink": .6, "label": "rho(default) - rho(no default)"})
axes[2].set_title("Difference: defaulters minus non-defaulters"); axes[2].tick_params(labelsize=5.5)
fig.suptitle("13.3 Differential association: the correlation structure by outcome (same column order in all panels)", fontweight="bold")
plt.tight_layout(); savefig("s13_differential_matrices"); plt.show()

s13_n1, s13_n0 = int(s13_m1.sum()), int(s13_m0.sum())
s13_r1v, s13_r0v, s13_rpv = s13_R1.to_numpy()[s13_iun], s13_R0.to_numpy()[s13_iun], s13_Rpool.to_numpy()[s13_iun]
s13_z = (np.arctanh(np.clip(s13_r1v, -.9999, .9999)) - np.arctanh(np.clip(s13_r0v, -.9999, .9999))) / np.sqrt(1.06 / (s13_n1 - 3) + 1.06 / (s13_n0 - 3))
s13_diff = pd.DataFrame({"var_a": np.array(S13_NUM)[s13_iun[0]], "var_b": np.array(S13_NUM)[s13_iun[1]], "rho_pooled": s13_rpv,
                         "rho_no_default": s13_r0v, "rho_default": s13_r1v, "delta_rho": s13_r1v - s13_r0v, "fisher_z": s13_z})
s13_diff = s13_diff[(s13_diff.rho_pooled.abs() <= .98) & s13_diff.fisher_z.notna()].copy()
s13_diff["p_value"] = 2 * stats.norm.sf(s13_diff["fisher_z"].abs())
s13_diff["p_fdr"] = s13_bh(s13_diff["p_value"].to_numpy())
s13_diff["sign_flip"] = (np.sign(s13_diff.rho_default) != np.sign(s13_diff.rho_no_default)) & (s13_diff.rho_default.abs() > .05) & (s13_diff.rho_no_default.abs() > .05)
s13_diff["block_a"] = s13_diff.var_a.map(s13_block); s13_diff["block_b"] = s13_diff.var_b.map(s13_block)
s13_diff = s13_diff.sort_values("delta_rho", key=np.abs, ascending=False).reset_index(drop=True)
export(s13_diff, "s13_differential_association")
print(f"pairs tested: {len(s13_diff)} | FDR < 0.01: {(s13_diff.p_fdr < .01).sum()} ({(s13_diff.p_fdr < .01).mean():.0%}) | "
      f"|delta rho| >= 0.10 and FDR < 0.01: {((s13_diff.delta_rho.abs() >= .1) & (s13_diff.p_fdr < .01)).sum()} | sign flips: {int(s13_diff.sign_flip.sum())}")
print(f"mean |rho| among non-defaulters {np.nanmean(np.abs(s13_r0v)):.3f} vs defaulters {np.nanmean(np.abs(s13_r1v)):.3f}")
print("\nTop 20 pairs whose correlation differs most by outcome:")
print(s13_diff.head(20)[["var_a", "var_b", "rho_pooled", "rho_no_default", "rho_default", "delta_rho", "fisher_z", "p_fdr", "sign_flip"]].round(3).to_string(index=False))
print("\nSign flips by outcome (both |rho| > 0.05), strongest first:")
print(s13_diff[s13_diff.sign_flip & (s13_diff.p_fdr < .01)].head(12)[["var_a", "var_b", "rho_no_default", "rho_default", "delta_rho", "p_fdr"]].round(3).to_string(index=False))
s13_d0 = s13_diff.iloc[0]
add_fact("13.3", "differential association", f"{((s13_diff.delta_rho.abs() >= .1) & (s13_diff.p_fdr < .01)).sum()} of {len(s13_diff)} numeric pairs have a correlation that differs by at least 0.10 "
         f"between defaulters and non-defaulters (FDR < 1%); the largest shift is {s13_d0.var_a} x {s13_d0.var_b}: rho {s13_d0.rho_no_default:.2f} (no default) vs {s13_d0.rho_default:.2f} (default)",
         float(s13_d0.delta_rho), "s13_differential_top6", 2)

**Observations:**
- 280 of 942 pairs change their correlation by 0.10 or more between defaulters and non-defaulters (FDR below 1%, meaning still significant after correcting for many tests). 35 pairs flip sign.
- The biggest change: PAY_2 x N_REVOLVE is 0.60 for non-defaulters but 0.03 for defaulters.
- For defaulters, earlier status codes match late streaks much more closely (PAY_2 x LATE_STREAK_RECENT 0.69 vs 0.18).
- Overall link strength is the same in both groups (average |rho| 0.328 vs 0.327). What changes is which pairs are linked.
- Story use: supporting. For defaulters an old late month is part of a streak; for others it is a one-off.

#### 13.3.1 The six largest shifts, drawn
For each pair: clients are cut into deciles of *x* (pooled); the line is the median of *y* in each decile, separately for
defaulters and non-defaulters, with the inter-quartile band. Diverging or crossing lines are the visual form of a differing
correlation. Tableau: dual-line chart on `s13_differential_top6_curves`.

In [ ]:
s13_top6 = s13_pick_diverse(s13_diff[s13_diff.p_fdr < .01], 6)
s13_curves = []
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (_, r) in zip(axes.ravel(), s13_top6.iterrows()):
    d = pd.DataFrame({"x": s13_X[r.var_a], "y": s13_X[r.var_b], "g": df["DEFAULT_LABEL"]}).dropna()
    d["bin"] = pd.qcut(d["x"].rank(method="first"), 10, labels=False) + 1
    g = d.groupby(["g", "bin"], observed=True)["y"].quantile([.25, .5, .75]).unstack().reset_index()
    g.columns = ["outcome", "x_decile", "q25", "median", "q75"]
    g["var_x"], g["var_y"] = r.var_a, r.var_b
    s13_curves.append(g)
    for lab in ["No default", "Default"]:
        s = g[g.outcome == lab]
        ax.plot(s["x_decile"], s["median"], marker="o", color=PAL_TARGET[lab], lw=2, label=lab)
        ax.fill_between(s["x_decile"], s["q25"], s["q75"], color=PAL_TARGET[lab], alpha=.12)
    ax.set_xlabel(f"decile of {r.var_a}"); ax.set_ylabel(f"{r.var_b} (median, IQR band)")
    ax.set_title(f"{r.var_a} x {r.var_b}\nrho: no default {r.rho_no_default:.2f} | default {r.rho_default:.2f} | delta {r.delta_rho:+.2f}", fontsize=9)
    ax.legend(fontsize=8)
fig.suptitle("13.3 The six relationships that change most with the outcome", fontweight="bold", y=1.0)
plt.tight_layout(); savefig("s13_differential_top6"); plt.show()
s13_curves = pd.concat(s13_curves, ignore_index=True)
export(s13_curves, "s13_differential_top6_curves")

**Observations:**
- This chart is an image. Numbers come from the table in 13.3 above.
- For non-defaulters, status codes rise with N_REVOLVE (rho about 0.60-0.65 for PAY_2 to PAY_5). For defaulters they stay flat (0.03-0.10).
- PAY_1 x N_REVOLVE even reverses: +0.38 for non-defaulters, -0.07 for defaulters.
- Story use: background. A candidate "two lines pull apart" chart.

#### 13.3.2 Which blocks are rewired for defaulters?
Mean correlation shift by block pair: is it the status block, the money blocks or the engineered ratios whose internal structure
differs for defaulters?

In [ ]:
s13_diff["block_pair"] = [" x ".join(sorted([a, b])) for a, b in zip(s13_diff.block_a, s13_diff.block_b)]
s13_bshift = s13_diff.groupby("block_pair").agg(pairs=("delta_rho", "size"), mean_delta=("delta_rho", "mean"), mean_abs_delta=("delta_rho", lambda v: v.abs().mean()),
                                                share_fdr_1pct=("p_fdr", lambda v: (v < .01).mean()), mean_abs_rho_no_default=("rho_no_default", lambda v: v.abs().mean()),
                                                mean_abs_rho_default=("rho_default", lambda v: v.abs().mean())).sort_values("mean_abs_delta", ascending=False)
print(s13_bshift.round(3).to_string())
fig, ax = plt.subplots(figsize=(10, 4.8))
s13_bs = s13_bshift.iloc[::-1]
ax.barh(s13_bs.index, s13_bs["mean_delta"], color=[C_DEF if v > 0 else C_OK for v in s13_bs["mean_delta"]])
ax.axvline(0, color="k", lw=1); ax.set_xlabel("mean rho(default) - rho(no default) over pairs of the block pair")
ax.set_title("13.3 Where correlation is stronger (red) or weaker (blue) among defaulters")
plt.tight_layout(); savefig("s13_differential_by_block"); plt.show()

**Observations:**
- Status x engineered pairs change most between the groups (average shift 0.15, 78% of pairs significant).
- Payments are less consistent with each other for defaulters (payment x payment 0.43 vs 0.52), and bills link less to status (0.46 vs 0.54).
- Bills link more tightly to each other for defaulters (0.89 vs 0.83). Links involving demographics barely change (shift 0.01-0.04).
- Story use: background. Defaulters pay less consistently from month to month.

#### 13.3.3 Relationships that flip sign across strata
Same idea with two other switches: September status group (`STATUS_GRP_SEP`, 6 levels) and a coarse credit-line band
(≤100k, 100-300k, >300k). A pair is a **flip** when its Spearman is significantly negative (< −0.10) in at least one stratum and
significantly positive (> +0.10) in another (each with p < 0.001, strata with n ≥ 300 only). A pooled correlation of such a pair
is an average of opposites and should not be shown unsegmented.

In [ ]:
s13_lim3 = pd.cut(df["LIMIT_BAL"], [0, 100_000, 300_000, 1e9], labels=["<=100k", "100-300k", ">300k"])
def s13_strata_flips(strat, levels, min_n=300):
    rs, ns = {}, {}
    for lev in levels:
        m = (strat == lev).to_numpy()
        if m.sum() < min_n: continue
        rs[lev] = s13_spearman(s13_X.loc[m, S13_NUM]).to_numpy()[s13_iun]; ns[lev] = int(m.sum())
    levs = list(rs)
    R = np.stack([rs[l] for l in levs]); Nn = np.array([ns[l] for l in levs])[:, None]
    sig = np.abs(R) * np.sqrt(Nn - 3) > 3.29
    Rs = np.where(sig, R, np.nan)
    out = pd.DataFrame({"var_a": np.array(S13_NUM)[s13_iun[0]], "var_b": np.array(S13_NUM)[s13_iun[1]], "rho_pooled": s13_rpv})
    for k, l in enumerate(levs):
        out[str(l)] = R[k]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        out["rho_min"], out["rho_max"] = np.nanmin(Rs, 0), np.nanmax(Rs, 0)
    out["range"] = out["rho_max"] - out["rho_min"]
    out["flip"] = (out["rho_min"] < -.1) & (out["rho_max"] > .1)
    nsig = sig.sum(0); allpos = (np.where(sig, R, 1) > 0).all(0); allneg = (np.where(sig, R, -1) < 0).all(0)
    out["simpson"] = (nsig >= max(2, len(levs) - 1)) & (((out["rho_pooled"] < -.02) & allpos) | ((out["rho_pooled"] > .02) & allneg))
    return out[out.rho_pooled.abs() <= .98].sort_values("range", ascending=False).reset_index(drop=True), levs, ns

s13_flip_status, s13_lev_s, s13_ns_s = s13_strata_flips(df["STATUS_GRP_SEP"], STATUS_GROUP_ORDER)
s13_flip_limit, s13_lev_l, s13_ns_l = s13_strata_flips(s13_lim3, ["<=100k", "100-300k", ">300k"])
for nm, tb, ns in [("STATUS_GRP_SEP", s13_flip_status, s13_ns_s), ("LIMIT band (3)", s13_flip_limit, s13_ns_l)]:
    print(f"\n=== strata of {nm}: n = {ns}")
    print(f"pairs: {len(tb)} | sign flips: {int(tb.flip.sum())} ({tb.flip.mean():.1%}) | Simpson-type (pooled sign opposite to the strata): {int(tb.simpson.sum())}")
    print(tb[tb.flip].head(15).drop(columns=["rho_min", "rho_max", "flip"]).round(2).to_string(index=False))
    if tb.simpson.any():
        print("Simpson-type pairs:"); print(tb[tb.simpson].head(10).drop(columns=["rho_min", "rho_max", "flip", "simpson"]).round(2).to_string(index=False))
s13_fs = s13_flip_status.assign(stratifier="STATUS_GRP_SEP").melt(id_vars=["stratifier", "var_a", "var_b", "rho_pooled", "flip", "simpson", "range"],
                                                                  value_vars=[str(l) for l in s13_lev_s], var_name="stratum", value_name="rho")
s13_fl = s13_flip_limit.assign(stratifier="LIMIT_BAND_3").melt(id_vars=["stratifier", "var_a", "var_b", "rho_pooled", "flip", "simpson", "range"],
                                                               value_vars=[str(l) for l in s13_lev_l], var_name="stratum", value_name="rho")
export(pd.concat([s13_fs, s13_fl], ignore_index=True), "s13_strata_correlations")

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5), gridspec_kw={"width_ratios": [1.5, 1]})
for ax, tb, levs, nm in [(axes[0], s13_flip_status, s13_lev_s, "September status group"), (axes[1], s13_flip_limit, s13_lev_l, "credit-line band")]:
    top = tb[tb.flip].head(14) if tb.flip.sum() >= 3 else tb.head(14)
    H = top.set_index(top.var_a + " x " + top.var_b)[["rho_pooled"] + [str(l) for l in levs]].rename(columns={"rho_pooled": "POOLED"})
    sns.heatmap(H, annot=True, fmt=".2f", cmap=CMAP_DIV, center=0, vmin=-.8, vmax=.8, ax=ax, cbar=False, annot_kws={"size": 8})
    ax.set_title(f"Spearman by {nm}: {'sign-flipping pairs' if tb.flip.sum() >= 3 else 'pairs with the widest range'}"); ax.set_ylabel(""); ax.tick_params(labelsize=8)
plt.tight_layout(); savefig("s13_strata_flips"); plt.show()
add_fact("13.3", "differential association", f"{int(s13_flip_status.flip.sum())} numeric pairs ({s13_flip_status.flip.mean():.0%}) change the SIGN of their correlation across September status groups, "
         f"and {int(s13_flip_limit.flip.sum())} across credit-line bands", int(s13_flip_status.flip.sum()), "s13_strata_flips", 2)

**Observations:**
- Across September status groups, 218 of 942 pairs (23%) change sign. Across credit-limit bands only 26 (3%) do.
- Example: PAY_5 x N_REVOLVE is +0.75 for inactive clients but -0.71 for clients 3+ months late.
- 6 pairs show a Simpson-type reversal, where the pooled sign is the opposite of the sign inside the groups. N_LATE x DELAY_TREND is +0.20 pooled but negative in all six status groups (-0.20 to -0.77).
- Story use: caveat. A chart that pools all status groups can show a misleading trend, so filter by September status first.

### 13.4 Which links survive once everything else is held fixed?
A marginal association can be second-hand (A-B only because both follow C). Partial correlation removes everything else.

* **Variables (35 nodes):** `LIMIT_BAL`, `AGE`, four demographic indicators (`SEX_FEMALE`, `EDU_GRAD`, `EDU_HIGHSCHOOL`, `MARRIED`),
  the 18 monthly raw columns, ten engineered features that are *not* exact linear functions of the raw columns, and `DEFAULT`.
* **Transformation:** every non-binary column is rank-gaussianised (normal scores of ranks: a Gaussian-copula or "nonparanormal"
  model), binaries are standardised. Missing `PAY_TO_BILL_6M` is set to the median.
* **Full partial correlation:** from the inverse of the correlation matrix (ridge 0.01 on the diagonal for stability), all rows.
* **Sparse network:** `GraphicalLassoCV` on an 8,000-row sample, which shrinks weak partial links to exactly zero.

Caveat: with a binary target and tied status codes the Gaussian copula is an approximation; read the network as a map of
*direct linear-in-normal-scores* links, not as a causal graph.

In [ ]:
S13_PC_ENG = ["AVG_UTIL", "UTIL_TREND", "PAY_TO_BILL_6M", "AVG_SPEND", "N_LATE", "LATE_STREAK_RECENT", "N_INACTIVE", "BILL_TREND", "N_ZERO_PAY", "N_OVERLIMIT"]
s13_pcX = pd.DataFrame({"LIMIT_BAL": df["LIMIT_BAL"], "AGE": df["AGE"], "SEX_FEMALE": (df["SEX"] == 2).astype(float),
                        "EDU_GRAD": (df["EDUCATION"] == 1).astype(float), "EDU_HIGHSCHOOL": (df["EDUCATION"] == 3).astype(float),
                        "MARRIED": (df["MARRIAGE"] == 1).astype(float)})
for c in PAY_COLS + BILL_COLS + PAMT_COLS + S13_PC_ENG:
    s13_pcX[c] = df[c].astype(float)
s13_pcX["PAY_TO_BILL_6M"] = s13_pcX["PAY_TO_BILL_6M"].fillna(s13_pcX["PAY_TO_BILL_6M"].median())
s13_pcX[TARGET] = df[TARGET].astype(float)
S13_PC_COLS = list(s13_pcX.columns)
def s13_gaussianise(frame):
    out = {}
    for c in frame.columns:
        s = frame[c]
        z = s if s.nunique() <= 2 else pd.Series(stats.norm.ppf((s.rank() - .5) / len(s)), index=s.index)
        out[c] = (z - z.mean()) / z.std()
    return pd.DataFrame(out)
s13_Zg = s13_gaussianise(s13_pcX)
s13_Rg = np.corrcoef(s13_Zg.to_numpy().T)
def s13_partial_from_cov(R, ridge=0.0):
    Pm = np.linalg.inv((1 - ridge) * R + ridge * np.eye(len(R)))
    d = np.sqrt(np.diag(Pm)); pc = -Pm / np.outer(d, d); np.fill_diagonal(pc, 1.0)
    return pc
s13_PCfull = pd.DataFrame(s13_partial_from_cov(s13_Rg, ridge=0.01), index=S13_PC_COLS, columns=S13_PC_COLS)
s13_Rg = pd.DataFrame(s13_Rg, index=S13_PC_COLS, columns=S13_PC_COLS)
print(f"condition number of the normal-score correlation matrix: {np.linalg.cond(s13_Rg.to_numpy()):,.0f}")

fig, axes = plt.subplots(1, 2, figsize=(22, 10))
sns.heatmap(s13_Rg, cmap=CMAP_DIV, vmin=-1, vmax=1, center=0, ax=axes[0], cbar=False, xticklabels=True, yticklabels=True)
axes[0].set_title("Marginal correlation (normal scores)")
sns.heatmap(s13_PCfull, cmap=CMAP_DIV, vmin=-1, vmax=1, center=0, ax=axes[1], xticklabels=True, yticklabels=True, cbar_kws={"shrink": .6})
axes[1].set_title("Partial correlation (each pair given the other 33 columns)")
for ax in axes:
    ax.tick_params(labelsize=7)
    for k in [6, 12, 18, 24, 34]:
        ax.axhline(k, color="k", lw=.6); ax.axvline(k, color="k", lw=.6)
fig.suptitle("13.4 What is left of each association after conditioning on everything else", fontweight="bold")
plt.tight_layout(); savefig("s13_partial_matrix"); plt.show()

s13_iup = np.triu_indices(len(S13_PC_COLS), 1)
s13_pc_pairs = pd.DataFrame({"var_a": np.array(S13_PC_COLS)[s13_iup[0]], "var_b": np.array(S13_PC_COLS)[s13_iup[1]],
                             "marginal_r": s13_Rg.to_numpy()[s13_iup], "partial_r": s13_PCfull.to_numpy()[s13_iup]})
s13_pc_pairs["block_a"] = s13_pc_pairs.var_a.map(s13_block); s13_pc_pairs["block_b"] = s13_pc_pairs.var_b.map(s13_block)
s13_pc_pairs["sign_reversal"] = (np.sign(s13_pc_pairs.marginal_r) != np.sign(s13_pc_pairs.partial_r)) & (s13_pc_pairs.marginal_r.abs() > .1) & (s13_pc_pairs.partial_r.abs() > .05)
print(f"pairs: {len(s13_pc_pairs)} | |marginal| >= 0.3: {(s13_pc_pairs.marginal_r.abs() >= .3).sum()} | |partial| >= 0.3: {(s13_pc_pairs.partial_r.abs() >= .3).sum()} "
      f"| |partial| < 0.05: {(s13_pc_pairs.partial_r.abs() < .05).mean():.0%} of pairs | sign reversals after conditioning: {int(s13_pc_pairs.sign_reversal.sum())}")
print("\nTop 20 direct links (|partial|):")
print(s13_pc_pairs.sort_values("partial_r", key=np.abs, ascending=False).head(20).round(3).to_string(index=False))
print("\nStrong marginal links that vanish (|marginal| >= 0.5, |partial| < 0.05):")
print(s13_pc_pairs[(s13_pc_pairs.marginal_r.abs() >= .5) & (s13_pc_pairs.partial_r.abs() < .05)].sort_values("marginal_r", key=np.abs, ascending=False).head(12).round(3).to_string(index=False))
print("\nSign reversals (marginal and partial of opposite sign):")
print(s13_pc_pairs[s13_pc_pairs.sign_reversal].sort_values("partial_r", key=np.abs, ascending=False).head(12).round(3).to_string(index=False))

**Observations:**
- Of 217 pairs with a marginal correlation of 0.3 or more, only 27 stay at 0.3 once all other columns are held fixed. 65% of pairs drop below 0.05.
- The strongest direct link is LIMIT_BAL x AVG_UTIL (-0.67): for the same bill, a bigger limit means lower utilisation.
- Status and the next payment turn clearly negative once other columns are held fixed (PAY_4 x PAY_AMT3: +0.03 marginal, -0.38 partial), so being late goes with paying less.
- Many strong links are second-hand. BILL_AMT1 x BILL_AMT3 drops from 0.83 to 0.03 because the month in between explains it.
- Story use: background. Supports showing one bill month and one status month per chart.

#### 13.4.1 Sparse dependency network (Graphical Lasso)
`GraphicalLassoCV` chooses the L1 penalty by 3-fold cross-validated likelihood on an 8,000-row sample; edges whose partial
correlation is shrunk to zero disappear. If the solver fails to converge (ill-conditioned bill block), the code falls back to a
fixed penalty and finally to thresholding the full partial matrix, and prints which path was used.
Node colour = block, edge width = |partial correlation|, red = positive, blue = negative; edges to `DEFAULT` are drawn in black outline.

In [ ]:
from sklearn.covariance import GraphicalLassoCV, GraphicalLasso
s13_t = time.time()
S13_GL_N = 8000
s13_gl_rows = s13_rng.choice(S13_N, S13_GL_N, replace=False)
s13_Zs = s13_Zg.to_numpy()[s13_gl_rows]
s13_Zs = (s13_Zs - s13_Zs.mean(0)) / s13_Zs.std(0)
s13_gl_path = "GraphicalLassoCV"
try:
    s13_gl = GraphicalLassoCV(alphas=[0.2, 0.1, 0.05, 0.03, 0.02], n_refinements=1, cv=3, max_iter=300, tol=1e-3).fit(s13_Zs)
    s13_prec, s13_alpha = s13_gl.precision_, float(s13_gl.alpha_)
except Exception as s13_e:
    try:
        s13_gl_path = "GraphicalLasso(alpha=0.05) fallback"
        s13_gl = GraphicalLasso(alpha=0.05, max_iter=300, tol=1e-3).fit(s13_Zs)
        s13_prec, s13_alpha = s13_gl.precision_, 0.05
    except Exception as s13_e2:
        s13_gl_path = "thresholded full partial matrix (|partial| >= 0.05) fallback"
        s13_prec = np.linalg.inv(0.99 * s13_Rg.to_numpy() + 0.01 * np.eye(len(S13_PC_COLS)))
        s13_prec[np.abs(s13_PCfull.to_numpy()) < .05] = 0; s13_alpha = np.nan
s13_dg = np.sqrt(np.diag(s13_prec))
s13_PCgl = -s13_prec / np.outer(s13_dg, s13_dg); np.fill_diagonal(s13_PCgl, 0)
s13_PCgl = pd.DataFrame(s13_PCgl, index=S13_PC_COLS, columns=S13_PC_COLS)
S13_EDGE_MIN = 0.02
s13_G = nx.Graph(); s13_G.add_nodes_from(S13_PC_COLS)
for i, j in zip(*s13_iup):
    w = s13_PCgl.iat[i, j]
    if abs(w) >= S13_EDGE_MIN:
        s13_G.add_edge(S13_PC_COLS[i], S13_PC_COLS[j], weight=abs(w), partial=w)
print(f"{s13_gl_path}: alpha = {s13_alpha}, n = {S13_GL_N:,}, {time.time() - s13_t:.1f}s")
print(f"edges kept (|partial| >= {S13_EDGE_MIN}): {s13_G.number_of_edges()} of {len(s13_iup[0])} possible ({s13_G.number_of_edges() / len(s13_iup[0]):.0%} density)")

s13_pos = nx.spring_layout(s13_G, weight="weight", seed=13, k=2.2 / np.sqrt(len(S13_PC_COLS)), iterations=300)
fig, ax = plt.subplots(figsize=(16, 12))
s13_edges = list(s13_G.edges(data=True))
nx.draw_networkx_edges(s13_G, s13_pos, ax=ax, edgelist=[(a, b) for a, b, _ in s13_edges], width=[0.4 + 9 * e["weight"] for _, _, e in s13_edges],
                       edge_color=[C_DEF if e["partial"] > 0 else C_OK for _, _, e in s13_edges], alpha=.55)
s13_tedges = [(a, b) for a, b, _ in s13_edges if TARGET in (a, b)]
nx.draw_networkx_edges(s13_G, s13_pos, ax=ax, edgelist=s13_tedges, width=1.2, edge_color="k", style="dashed")
nx.draw_networkx_nodes(s13_G, s13_pos, ax=ax, node_color=[S13_BLOCK_COL[s13_block(c)] for c in s13_G.nodes],
                       node_size=[900 if c == TARGET else 420 for c in s13_G.nodes], edgecolors="white", linewidths=1.2)
nx.draw_networkx_labels(s13_G, {k: (v[0], v[1] + .035) for k, v in s13_pos.items()}, ax=ax, font_size=7.5)
ax.legend(handles=[S13_Patch(color=v, label=k) for k, v in S13_BLOCK_COL.items()] +
          [S13_Line2D([0], [0], color=C_DEF, lw=3, label="positive partial"), S13_Line2D([0], [0], color=C_OK, lw=3, label="negative partial"),
           S13_Line2D([0], [0], color="k", lw=1.2, ls="--", label="edge to DEFAULT")], fontsize=8, loc="lower left")
ax.set_title(f"13.4 Sparse conditional-dependence network ({s13_gl_path}, {s13_G.number_of_edges()} edges)"); ax.axis("off")
plt.tight_layout(); savefig("s13_glasso_network"); plt.show()

s13_edge_tbl = pd.DataFrame([{"source": a, "target": b, "block_source": s13_block(a), "block_target": s13_block(b), "partial_glasso": e["partial"],
                              "abs_partial": e["weight"], "partial_full": s13_PCfull.loc[a, b], "marginal_r": s13_Rg.loc[a, b]} for a, b, e in s13_edges])
s13_edge_tbl = s13_edge_tbl.sort_values("abs_partial", ascending=False).reset_index(drop=True)
export(s13_edge_tbl, "s13_network_edges")
s13_node_tbl = pd.DataFrame({"node": S13_PC_COLS, "block": [s13_block(c) for c in S13_PC_COLS], "x": [s13_pos[c][0] for c in S13_PC_COLS],
                             "y": [s13_pos[c][1] for c in S13_PC_COLS], "degree": [s13_G.degree(c) for c in S13_PC_COLS],
                             "strength": [s13_G.degree(c, weight="weight") for c in S13_PC_COLS]})
export(s13_node_tbl, "s13_network_nodes")
s13_census = s13_edge_tbl.assign(bp=[" x ".join(sorted([a, b])) for a, b in zip(s13_edge_tbl.block_source, s13_edge_tbl.block_target)]).groupby("bp").agg(
    edges=("abs_partial", "size"), mean_abs_partial=("abs_partial", "mean"), max_abs_partial=("abs_partial", "max")).sort_values("edges", ascending=False)
print("\nEdge census by block pair:"); print(s13_census.round(3).to_string())
print("\nNode degree (top 10):"); print(s13_node_tbl.sort_values("strength", ascending=False).head(10)[["node", "block", "degree", "strength"]].round(3).to_string(index=False))

**Observations:**
- The sparse network keeps 193 of 595 possible links (32%).
- LIMIT_BAL (26 links) and PAY_TO_BILL_6M (21) are the main connectors. DEFAULT has only 3 direct links.
- Status-status links are few (5) but strong (average partial 0.33). Demographic-status links are almost nil (0.04).
- Story use: one network chart could open a methods page; otherwise background.

#### 13.4.2 Which columns connect directly to `DEFAULT`?
The neighbours of `DEFAULT` in the sparse network are its **Markov-blanket-like set** (the small set of columns that, once known, make the rest add little): given them, the remaining columns add
(approximately) nothing linear. The bar chart compares, for every column, the marginal correlation with `DEFAULT`, the full
partial correlation and the graphical-lasso partial: what survives conditioning, what vanishes, what flips sign.

In [ ]:
s13_mb = pd.DataFrame({"block": [s13_block(c) for c in S13_PC_COLS], "marginal_r": s13_Rg[TARGET], "partial_full": s13_PCfull[TARGET],
                       "partial_glasso": s13_PCgl[TARGET]}).drop(index=TARGET)
s13_mb["in_markov_blanket"] = s13_mb["partial_glasso"].abs() >= S13_EDGE_MIN
s13_mb["retained_share"] = (s13_mb["partial_full"].abs() / s13_mb["marginal_r"].abs()).clip(upper=9.99)
s13_mb["sign_flip"] = (np.sign(s13_mb.marginal_r) != np.sign(s13_mb.partial_full)) & (s13_mb.partial_full.abs() > .01) & (s13_mb.marginal_r.abs() > .02)
s13_mb = s13_mb.sort_values("marginal_r", key=np.abs, ascending=False)
fig, ax = plt.subplots(figsize=(11, 11))
s13_yy = np.arange(len(s13_mb))[::-1]
ax.barh(s13_yy + .27, s13_mb["marginal_r"], height=.27, color=C_NEU, label="marginal")
ax.barh(s13_yy, s13_mb["partial_full"], height=.27, color=C_DEF, label="partial (all 33 others)")
ax.barh(s13_yy - .27, s13_mb["partial_glasso"], height=.27, color="#111111", label="partial (graphical lasso)")
ax.set_yticks(s13_yy); ax.set_yticklabels([f"{c}{'  *' if b else ''}" for c, b in zip(s13_mb.index, s13_mb.in_markov_blanket)], fontsize=8)
for lab, c in zip(ax.get_yticklabels(), s13_mb.index):
    lab.set_color(S13_BLOCK_COL[s13_block(c)])
ax.axvline(0, color="k", lw=1); ax.legend(fontsize=9); ax.set_xlabel("correlation with DEFAULT (normal-score scale)")
ax.set_title("13.4 Marginal vs partial association with DEFAULT (* = direct neighbour in the sparse network)")
plt.tight_layout(); savefig("s13_marginal_vs_partial_default"); plt.show()
print(s13_mb.round(3).to_string())
export(s13_mb.rename_axis("column").reset_index(), "s13_default_marginal_vs_partial")
s13_blanket = s13_mb[s13_mb.in_markov_blanket].sort_values("partial_glasso", key=np.abs, ascending=False)
print(f"\nMarkov-blanket-like set of DEFAULT ({len(s13_blanket)} of {len(s13_mb)} columns): {', '.join(s13_blanket.index)}")
print("by block:", s13_blanket.groupby("block").size().to_dict(), "| all columns by block:", s13_mb.groupby("block").size().to_dict())
s13_strong_marg = s13_mb[s13_mb.marginal_r.abs() >= .1]
s13_lost = s13_strong_marg[s13_strong_marg.partial_full.abs() < .02]
add_fact("13.4", "conditional independence", f"Of {len(s13_strong_marg)} columns with a marginal correlation of at least 0.10 with DEFAULT, {len(s13_lost)} fall below 0.02 once the other columns are held fixed; "
         f"the strongest direct link is {s13_blanket.index[0]} (partial {s13_blanket.partial_glasso.iloc[0]:+.2f} vs marginal {s13_blanket.marginal_r.iloc[0]:+.2f})",
         len(s13_lost), "s13_marginal_vs_partial_default", 3)
add_fact("13.4", "conditional independence", f"The sparse dependency network keeps {s13_G.number_of_edges()} of {len(s13_iup[0])} possible links; DEFAULT is directly connected to {len(s13_blanket)} columns "
         f"({', '.join(s13_blanket.index[:5])}{'...' if len(s13_blanket) > 5 else ''})", len(s13_blanket), "s13_glasso_network", 2)

**Observations:**
- Once other columns are held fixed, only N_LATE (partial 0.09 vs marginal 0.40), LATE_STREAK_RECENT (0.09 vs 0.39) and PAY_2 stay directly linked to default.
- PAY_1's marginal 0.31 falls to about 0, because N_LATE and LATE_STREAK_RECENT already contain its information.
- Payment amounts (-0.12 to -0.16 marginal) and limit (-0.17) shrink to about -0.02 or less.
- Story use: key message. "How many months late" sums up most of what the other columns say about default.

### 13.5 Across months, which block leads and which follows?
Section 7.4 looked *inside* each monthly block (lag decay). Here we look **across** blocks: the full 18 × 18 Spearman matrix of
status, bill and payment in chronological order (Apr → Sep), then each 6 × 6 cross-block panel.
In a cross panel the diagonal is the same-month link; cells above the diagonal pair an earlier row-month with a later column-month.
Reminder: `PAY_AMT` of month *m* settles the bill of month *m − 1*, and status code 1 occurs almost only in September, so the
Aug → Sep transition is partly a coding artefact.

In [ ]:
S13_CHRON = IDX[::-1]                                              # 6 (Apr) ... 1 (Sep)
S13_LAG_BLOCKS = {"status": [f"PAY_{i}" for i in S13_CHRON], "bill": [f"BILL_AMT{i}" for i in S13_CHRON], "payment": [f"PAY_AMT{i}" for i in S13_CHRON]}
s13_lag_cols = sum(S13_LAG_BLOCKS.values(), [])
s13_lag_lab = [f"{b[:4]} {MONTH[i]}" for b in S13_LAG_BLOCKS for i in S13_CHRON]
s13_L18 = s13_spearman(df[s13_lag_cols])
fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(pd.DataFrame(s13_L18.to_numpy(), index=s13_lag_lab, columns=s13_lag_lab), annot=True, fmt=".2f", annot_kws={"size": 6.5},
            cmap=CMAP_DIV, vmin=-1, vmax=1, center=0, ax=ax, cbar_kws={"shrink": .6})
for k in [6, 12]:
    ax.axhline(k, color="k", lw=1.5); ax.axvline(k, color="k", lw=1.5)
ax.set_title("13.5 Lag atlas: Spearman of status / bill / payment, months in chronological order (Apr -> Sep)")
plt.tight_layout(); savefig("s13_lag_atlas_18"); plt.show()

**Observations:**
- This chart is an image. Numbers come from the lag table in 13.5.1 below.
- In the same month, status and bills link at 0.54 and bills and payments at 0.51.
- Status and payments barely link (0.10 in the same month).
- Story use: background only.

#### 13.5.1 Cross-block lag panels and lag profiles
Top row: the three 6 × 6 cross panels (rows = first block's month, columns = second block's month).
Bottom row: mean Spearman by lag *k* = column month − row month (k > 0: the row block is observed **earlier**).
An asymmetric profile means one block anticipates the other.

In [ ]:
S13_CROSS = [("status", "bill"), ("status", "payment"), ("bill", "payment")]
s13_mlab = [MONTH[i] for i in S13_CHRON]
s13_lag_rows = []
fig, axes = plt.subplots(2, 3, figsize=(19, 10.5), gridspec_kw={"height_ratios": [1.25, 1]})
for k, (ba, bb) in enumerate(S13_CROSS):
    M = s13_L18.loc[S13_LAG_BLOCKS[ba], S13_LAG_BLOCKS[bb]].to_numpy()
    sns.heatmap(pd.DataFrame(M, index=s13_mlab, columns=s13_mlab), annot=True, fmt=".2f", cmap=CMAP_DIV, center=0,
                vmin=-np.abs(M).max(), vmax=np.abs(M).max(), ax=axes[0, k], cbar=False, annot_kws={"size": 9})
    axes[0, k].set_ylabel(f"{ba} month"); axes[0, k].set_xlabel(f"{bb} month"); axes[0, k].set_title(f"{ba} (rows) x {bb} (columns)")
    for ra in range(6):
        for rb in range(6):
            s13_lag_rows.append({"block_a": ba, "block_b": bb, "month_a": s13_mlab[ra], "month_b": s13_mlab[rb], "lag_b_minus_a": rb - ra, "spearman": M[ra, rb]})
s13_lag_long = pd.DataFrame(s13_lag_rows)
s13_lag_prof = s13_lag_long.groupby(["block_a", "block_b", "lag_b_minus_a"], sort=False)["spearman"].agg(["mean", "min", "max", "size"]).reset_index()
for k, (ba, bb) in enumerate(S13_CROSS):
    g = s13_lag_prof[(s13_lag_prof.block_a == ba) & (s13_lag_prof.block_b == bb)].sort_values("lag_b_minus_a")
    axes[1, k].plot(g["lag_b_minus_a"], g["mean"], marker="o", color=C_DEF, lw=2)
    axes[1, k].fill_between(g["lag_b_minus_a"], g["min"], g["max"], color=C_DEF, alpha=.15)
    axes[1, k].axvline(0, color="k", lw=1, ls=":"); axes[1, k].axhline(0, color="k", lw=.8)
    best = g.loc[g["mean"].abs().idxmax()]
    axes[1, k].scatter([best["lag_b_minus_a"]], [best["mean"]], s=160, facecolor="none", edgecolor="k", lw=2, zorder=5)
    axes[1, k].set_xlabel(f"lag k (months): {bb} month - {ba} month   [k>0: {ba} earlier]"); axes[1, k].set_ylabel("mean Spearman (band = min..max)")
    axes[1, k].set_title(f"lag profile {ba} -> {bb}: strongest at k = {int(best['lag_b_minus_a'])} ({best['mean']:.2f})")
fig.suptitle("13.5 Cross-block lag structure", fontweight="bold", y=1.0)
plt.tight_layout(); savefig("s13_lag_cross_panels"); plt.show()
export(s13_lag_long, "s13_lag_atlas_long")
print(s13_lag_prof.pivot_table(index=["block_a", "block_b"], columns="lag_b_minus_a", values="mean", sort=False).round(3).to_string())

**Observations:**
- Status links most to the bill one month later (0.62), and bills link most to the payment one month later (0.64).
- Links fade slowly with distance: status-bill falls from 0.62 one month ahead to 0.29 five months back.
- Status vs payment is weak everywhere, peaking at 0.25 two months later.
- Story use: background. Supports "bill first, payment next month" in the ledger explanation.

#### 13.5.2 Lead/lag table: who anticipates whom, beyond persistence?
Raw lag correlations are inflated by persistence (a client with a high bill has a high bill every month). The sharper question:
*does X in month m tell us about Y in month m+1 beyond what Y in month m already tells?* That is the partial Spearman
ρ(X_m, Y_{m+1} | Y_m), averaged over the five month-to-month transitions, for both directions of each cross pair
(a cross-sectional, Granger-flavoured lead test, i.e. does X's past help forecast Y beyond Y's own past; no causal claim). `asymmetry` > 0 means the first block leads.

In [ ]:
def s13_partial3(r_xy, r_xz, r_yz):
    return (r_xy - r_xz * r_yz) / np.sqrt(max(1e-12, (1 - r_xz ** 2) * (1 - r_yz ** 2)))
s13_ll_rows = []
for ba, bb in S13_CROSS:
    for lead, follow in [(ba, bb), (bb, ba)]:
        for m in range(5):
            x, y1, y0 = S13_LAG_BLOCKS[lead][m], S13_LAG_BLOCKS[follow][m + 1], S13_LAG_BLOCKS[follow][m]
            s13_ll_rows.append({"pair": f"{ba} x {bb}", "leader": lead, "follower": follow, "transition": f"{s13_mlab[m]}->{s13_mlab[m + 1]}",
                                "raw_lag1": s13_L18.loc[x, y1], "same_month": s13_L18.loc[x, y0],
                                "partial_lead": s13_partial3(s13_L18.loc[x, y1], s13_L18.loc[x, y0], s13_L18.loc[y1, y0])})
s13_ll = pd.DataFrame(s13_ll_rows)
s13_ll_sum = s13_ll.groupby(["pair", "leader", "follower"], sort=False).agg(raw_lag1=("raw_lag1", "mean"), same_month=("same_month", "mean"),
                                                                           partial_lead=("partial_lead", "mean"), partial_min=("partial_lead", "min"),
                                                                           partial_max=("partial_lead", "max")).reset_index()
s13_best_lag = (s13_lag_prof.assign(a=s13_lag_prof["mean"].abs()).sort_values("a", ascending=False).groupby(["block_a", "block_b"], sort=False).head(1)
                .assign(pair=lambda g: g.block_a + " x " + g.block_b)[["pair", "lag_b_minus_a", "mean"]].rename(columns={"lag_b_minus_a": "strongest_raw_lag", "mean": "rho_at_strongest_lag"}))
s13_leadlag = []
for pr, g in s13_ll_sum.groupby("pair", sort=False):
    a, b = g.iloc[0], g.iloc[1]
    s13_leadlag.append({"pair": pr, f"first_leads_partial": a.partial_lead, "second_leads_partial": b.partial_lead,
                        "asymmetry": abs(a.partial_lead) - abs(b.partial_lead), "leader": a.leader if abs(a.partial_lead) > abs(b.partial_lead) else b.leader})
s13_leadlag = pd.DataFrame(s13_leadlag).merge(s13_best_lag, on="pair")
print("Lead/lag summary (partial Spearman of leader in month m with follower in month m+1, given follower in month m; mean of 5 transitions):")
print(s13_ll_sum.round(3).to_string(index=False))
print("\nWho leads:"); print(s13_leadlag.round(3).to_string(index=False))
print("\nBy transition (partial lead):")
print(s13_ll.pivot_table(index=["leader", "follower"], columns="transition", values="partial_lead", sort=False).round(3).to_string())
export(s13_ll, "s13_lead_lag")
fig, ax = plt.subplots(figsize=(11, 4.8))
s13_lbl = s13_ll_sum["leader"] + " (m) -> " + s13_ll_sum["follower"] + " (m+1)"
ax.barh(s13_lbl[::-1], s13_ll_sum["partial_lead"][::-1], color=[C_DEF if v > 0 else C_OK for v in s13_ll_sum["partial_lead"][::-1]],
        xerr=[(s13_ll_sum["partial_lead"] - s13_ll_sum["partial_min"])[::-1], (s13_ll_sum["partial_max"] - s13_ll_sum["partial_lead"])[::-1]], capsize=3)
ax.axvline(0, color="k", lw=1); ax.set_xlabel("partial Spearman given the follower's own previous month (bar = mean, whisker = range over 5 transitions)")
ax.set_title("13.5 Lead/lag: what this month's X adds about next month's Y")
plt.tight_layout(); savefig("s13_lead_lag"); plt.show()
s13_lead_top = s13_ll_sum.loc[s13_ll_sum.partial_lead.abs().idxmax()]
add_fact("13.5", "lag structure", f"Strongest cross-block lead beyond persistence: {s13_lead_top.leader} in month m vs {s13_lead_top.follower} in month m+1 "
         f"(partial rho {s13_lead_top.partial_lead:+.2f} given the follower's own previous month)", float(s13_lead_top.partial_lead), "s13_lead_lag", 2)

**Observations:**
- After allowing for each column's own past, this month's bill says the most about next month's payment (partial 0.51). The reverse is only 0.07.
- Status leads bills (0.25) more than bills lead status (0.10).
- Status and payments do not lead each other (both about 0). The payment-to-status link turns negative only for Aug to Sep (-0.15), when the September coding changes.
- Story use: background. "The bill drives the payment" is a clean ledger fact.

### 13.6 Which pairs tell more together than apart?
For two columns X, Y and the target D, the **interaction information** is

  syn(X, Y) = I(X,Y ; D) − I(X ; D) − I(Y ; D)

Positive = **synergy** (the pair jointly says more about default than the sum of the parts: a true interaction).
Negative = **redundancy** (the two columns repeat the same message).

* 26 curated columns, banded coarsely to keep joint tables well filled: nominal levels, the six status groups for `PAY_i`,
  own values when ≤ 7 distinct, otherwise quintiles; missing = own band.
* All 325 pair tables come from two matrix products (one-hot Gram per outcome class).
* **Honesty check:** plug-in MI is biased upwards in finite samples, and the bias of the joint term is larger than the bias of
  the two single terms, so *every* pair shows spurious positive synergy of about (L_x−1)(L_y−1)/2n nats. A permutation baseline
  (100 shuffles of `DEFAULT`) gives the chance level; `synergy_excess` = observed − permutation mean, with an empirical p-value.
  Shuffling the target tests "no relation to default at all"; it calibrates the bias, it is not a test of additivity.
Values are reported as % of H(DEFAULT).

In [ ]:
S13_SYN_COLS = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE", "PAY_1", "PAY_2", "PAY_3", "PAY_6", "BILL_AMT1", "PAY_AMT1", "PAY_AMT2",
                "AVG_UTIL", "UTIL_TREND", "PAY_TO_BILL_6M", "AVG_PAY_RATIO", "AVG_SPEND", "N_LATE", "MAX_DELAY", "LATE_STREAK_RECENT",
                "N_INACTIVE", "N_PAIDFULL", "N_REVOLVE", "DELAY_TREND", "BILL_TREND", "N_ZERO_PAY"]
def s13_band5(c):
    s = s13_X[c]
    if c in ("MAX_DELAY", "DELAY_TREND"):                         # many sparse integer levels -> coarse ordered bands
        s = pd.cut(s, [-np.inf, -1, 0, 1, 2, np.inf], labels=False).astype(float) if c == "MAX_DELAY" else pd.cut(s, [-np.inf, -2, -1, 0, 1, 2, np.inf], labels=False).astype(float)
        s.name = c
    return s13_band(s, 5, status_groups=True)
s13_K5_all = pd.DataFrame({c: s13_band5(c) for c in S13_ALL if c != TARGET})      # coarse bands for every feature (reused in 13.7 / 13.8)
s13_K5 = s13_K5_all[S13_SYN_COLS].to_numpy()
s13_O5, s13_off5 = s13_onehot(s13_K5)
s13_Gall5 = s13_gram(s13_O5)

def s13_pair_info(G1, Gall, off):
    # I(X;D) for every column and I(X,Y;D) for every pair, from the class-1 Gram and the overall Gram
    p = len(off) - 1; G0 = Gall - G1
    I1 = np.zeros(p); IJ = np.zeros((p, p))
    for i in range(p):
        si = slice(off[i], off[i + 1])
        I1[i] = s13_mi_ct(np.stack([np.diag(G0[si, si]), np.diag(G1[si, si])], 1))
        for j in range(i + 1, p):
            sj = slice(off[j], off[j + 1])
            IJ[i, j] = IJ[j, i] = s13_mi_ct(np.stack([G0[si, sj].ravel(), G1[si, sj].ravel()], 1))
    return I1, IJ

s13_t = time.time()
s13_I1, s13_IJ = s13_pair_info(s13_gram(s13_O5[s13_y == 1]), s13_Gall5, s13_off5)
s13_SYN = s13_IJ - s13_I1[:, None] - s13_I1[None, :]
S13_NPERM = 100
s13_perm_syn = np.zeros((S13_NPERM,) + s13_SYN.shape, np.float32); s13_perm_I1 = np.zeros((S13_NPERM, len(S13_SYN_COLS)))
for b in range(S13_NPERM):
    yp = s13_rng.permutation(s13_y)
    i1, ij = s13_pair_info(s13_gram(s13_O5[yp == 1]), s13_Gall5, s13_off5)
    s13_perm_syn[b] = ij - i1[:, None] - i1[None, :]; s13_perm_I1[b] = i1
print(f"synergy for {len(S13_SYN_COLS)} columns ({len(S13_SYN_COLS) * (len(S13_SYN_COLS) - 1) // 2} pairs) + {S13_NPERM} permutations in {time.time() - s13_t:.1f}s")

s13_ius = np.triu_indices(len(S13_SYN_COLS), 1)
s13_lv5 = s13_K5.max(0) + 1
s13_syn = pd.DataFrame({"var_a": np.array(S13_SYN_COLS)[s13_ius[0]], "var_b": np.array(S13_SYN_COLS)[s13_ius[1]],
                        "levels_a": s13_lv5[s13_ius[0]], "levels_b": s13_lv5[s13_ius[1]],
                        "info_a": s13_I1[s13_ius[0]], "info_b": s13_I1[s13_ius[1]], "info_joint": s13_IJ[s13_ius], "synergy": s13_SYN[s13_ius],
                        "perm_mean": s13_perm_syn.mean(0)[s13_ius], "perm_sd": s13_perm_syn.std(0)[s13_ius], "perm_p95": np.quantile(s13_perm_syn, .95, axis=0)[s13_ius],
                        "p_perm": ((s13_perm_syn >= s13_SYN[None]).sum(0)[s13_ius] + 1) / (S13_NPERM + 1)})
s13_syn["analytic_bias"] = (s13_syn.levels_a - 1) * (s13_syn.levels_b - 1) / (2 * S13_N)
s13_syn["synergy_excess"] = s13_syn["synergy"] - s13_syn["perm_mean"]
s13_syn["z_vs_perm"] = s13_syn["synergy_excess"] / s13_syn["perm_sd"]
s13_syn["redundant_share_of_weaker"] = (-s13_syn["synergy"] / s13_syn[["info_a", "info_b"]].min(1)).clip(lower=0)
for c in ["info_a", "info_b", "info_joint", "synergy", "perm_mean", "perm_p95", "synergy_excess", "analytic_bias"]:
    s13_syn[c + "_pct"] = 100 * s13_syn[c] / S13_HD
s13_syn["kind"] = np.where((s13_syn.synergy_excess > 0) & (s13_syn.p_perm <= .01), "synergy (beyond chance)", np.where(s13_syn.synergy < 0, "redundancy", "within chance"))
export(s13_syn, "s13_synergy_pairs")
print(f"\nchance level of synergy (permutation mean): median {s13_syn.perm_mean_pct.median():.3f}% of H(D), max {s13_syn.perm_mean_pct.max():.3f}%; "
      f"analytic (L-1)(L-1)/2n agrees: corr = {np.corrcoef(s13_syn.perm_mean, s13_syn.analytic_bias)[0, 1]:.3f}")
print(f"single-column MI chance level: mean {100 * s13_perm_I1.mean() / S13_HD:.4f}% of H(D)")
print(s13_syn["kind"].value_counts().to_string())
print(f"raw synergy > 0: {(s13_syn.synergy > 0).sum()} pairs | still > 0 after removing the chance level and p <= 0.01: {(s13_syn.kind == 'synergy (beyond chance)').sum()} pairs")

#### 13.6.1 Top synergistic and top redundant pairs
Left table: pairs whose joint information exceeds the sum of parts by the largest margin **after** subtracting the permutation
baseline. Right table: the most redundant pairs; `redundant_share_of_weaker` is the share of the weaker column's information
that is already contained in the stronger one.

In [ ]:
s13_sc = ["var_a", "var_b", "info_a_pct", "info_b_pct", "info_joint_pct", "synergy_pct", "perm_mean_pct", "synergy_excess_pct", "z_vs_perm", "p_perm"]
s13_top_syn = s13_syn[s13_syn.kind == "synergy (beyond chance)"].nlargest(15, "synergy_excess")
s13_top_red = s13_syn.nsmallest(15, "synergy")
print("Top 15 SYNERGISTIC pairs (all values in % of H(DEFAULT)):")
print(s13_top_syn[s13_sc].round(3).to_string(index=False))
print("\nTop 15 REDUNDANT pairs:")
print(s13_top_red[["var_a", "var_b", "info_a_pct", "info_b_pct", "info_joint_pct", "synergy_pct", "redundant_share_of_weaker"]].round(3).to_string(index=False))
print("\nRelative synergy (excess as a share of the sum of the two single informations), top 10 with p <= 0.01:")
s13_rel = s13_syn[s13_syn.kind == "synergy (beyond chance)"].assign(relative=lambda g: g.synergy_excess / (g.info_a + g.info_b)).nlargest(10, "relative")
print(s13_rel[["var_a", "var_b", "info_a_pct", "info_b_pct", "synergy_excess_pct", "relative"]].round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
if len(s13_top_syn):
    g = s13_top_syn.iloc[::-1]
    axes[0].barh(g.var_a + " + " + g.var_b, g["synergy_pct"], color=C_NEU, label="raw synergy (plug-in)")
    axes[0].barh(g.var_a + " + " + g.var_b, g["synergy_excess_pct"], color=C_ACC, label="beyond permutation baseline")
    axes[0].legend(fontsize=8)
axes[0].set_xlabel("% of H(DEFAULT)"); axes[0].set_title("Most synergistic pairs: more together than apart"); axes[0].tick_params(axis="y", labelsize=8)
g = s13_top_red.iloc[::-1]
axes[1].barh(g.var_a + " + " + g.var_b, g["synergy_pct"], color=C_DEF)
axes[1].set_xlabel("% of H(DEFAULT) (negative = overlap)"); axes[1].set_title("Most redundant pairs: the same message twice"); axes[1].tick_params(axis="y", labelsize=8)
plt.tight_layout(); savefig("s13_synergy_top_pairs"); plt.show()
if len(s13_top_syn):
    s13_s0 = s13_top_syn.iloc[0]
    add_fact("13.6", "synergy", f"Only {(s13_syn.kind == 'synergy (beyond chance)').sum()} of {len(s13_syn)} pairs are synergistic beyond the permutation baseline; the largest, {s13_s0.var_a} + {s13_s0.var_b}, "
             f"adds {s13_s0.synergy_excess_pct:.2f}% of H(DEFAULT) on top of the two columns separately ({s13_s0.info_a_pct:.1f}% and {s13_s0.info_b_pct:.1f}%)",
             float(s13_s0.synergy_excess_pct), "s13_synergy_top_pairs", 2)
s13_r0 = s13_top_red.iloc[0]
add_fact("13.6", "redundancy", f"{(s13_syn.synergy < 0).mean():.0%} of pairs are redundant; the worst, {s13_r0.var_a} + {s13_r0.var_b}, shares {s13_r0.redundant_share_of_weaker:.0%} "
         f"of the weaker column's information about default", float(s13_r0.redundant_share_of_weaker), "s13_synergy_matrix", 2)

**Observations:**
- Synergy means two columns together tell more about default than the two separately. Only 85 of 325 pairs are synergistic beyond chance; 184 are redundant (they repeat each other).
- The top pair, AVG_PAY_RATIO + N_REVOLVE, removes 13.2% of the uncertainty about default together, vs 2.2% and 4.3% alone (6.6 points extra).
- The lateness columns repeat each other: N_LATE + MAX_DELAY share 98% of the weaker one's information, and PAY_1 + LATE_STREAK_RECENT 95%.
- Story use: choose pairs for two-way heatmaps (pay ratio x months revolving) and show only one lateness field.

#### 13.6.2 Synergy matrix and the chance band
Left: synergy minus permutation mean for all 325 pairs (green = synergy, red = redundancy; a dot marks pairs that exceed chance
at p ≤ 0.01). Right: observed synergy against its permutation baseline; points inside the grey band are indistinguishable from
finite-sample bias and must not be reported as interactions.

In [ ]:
s13_EX = s13_SYN - s13_perm_syn.mean(0)
np.fill_diagonal(s13_EX, np.nan)
s13_EXd = pd.DataFrame(100 * s13_EX / S13_HD, index=S13_SYN_COLS, columns=S13_SYN_COLS)
fig, axes = plt.subplots(1, 2, figsize=(22, 10), gridspec_kw={"width_ratios": [1.35, 1]})
s13_lim = float(np.nanquantile(np.abs(s13_EXd.to_numpy()), .97))
sns.heatmap(s13_EXd, cmap="PiYG", center=0, vmin=-s13_lim, vmax=s13_lim, ax=axes[0], annot=True, fmt=".1f", annot_kws={"size": 5.5},
            cbar_kws={"shrink": .6, "label": "synergy beyond chance, % of H(DEFAULT)"})
s13_sigM = np.zeros_like(s13_SYN, bool)
s13_sig_idx = (s13_syn.kind == "synergy (beyond chance)").to_numpy()
s13_sigM[s13_ius[0][s13_sig_idx], s13_ius[1][s13_sig_idx]] = True; s13_sigM |= s13_sigM.T
for i, j in zip(*np.where(s13_sigM)):
    axes[0].plot(j + .85, i + .2, marker="o", ms=2.5, color="k")
axes[0].set_title("Interaction information with DEFAULT (dot = synergy beyond chance, p <= 0.01)"); axes[0].tick_params(labelsize=7.5)
axes[1].scatter(s13_syn["perm_mean_pct"], s13_syn["synergy_pct"], c=s13_syn["kind"].map({"synergy (beyond chance)": C_ACC, "redundancy": C_DEF, "within chance": C_NEU}), s=22, alpha=.8)
s13_xs = np.linspace(0, s13_syn["perm_mean_pct"].max() * 1.05, 50)
axes[1].plot(s13_xs, s13_xs, color="k", lw=1, label="chance level (permutation mean)")
s13_o = np.argsort(s13_syn["perm_mean_pct"].to_numpy())
axes[1].fill_between(s13_syn["perm_mean_pct"].to_numpy()[s13_o], 0, s13_syn["perm_p95_pct"].to_numpy()[s13_o], color=C_NEU, alpha=.25, label="below permutation 95th percentile")
axes[1].set_yscale("symlog", linthresh=.05); axes[1].axhline(0, color="k", lw=.8)
axes[1].set_xlabel("synergy expected by chance (% of H(D)); grows with the number of cells"); axes[1].set_ylabel("observed synergy (% of H(D), symlog)")
axes[1].legend(handles=[S13_Patch(color=C_ACC, label="synergy beyond chance"), S13_Patch(color=C_NEU, label="within chance"), S13_Patch(color=C_DEF, label="redundancy"),
                        S13_Line2D([0], [0], color="k", lw=1, label="chance level")], fontsize=8)
axes[1].set_title("Observed vs chance synergy: the honesty plot")
plt.tight_layout(); savefig("s13_synergy_matrix"); plt.show()
s13_colsum = pd.DataFrame({"info_alone_pct": 100 * s13_I1 / S13_HD, "mean_synergy_excess_pct": np.nanmean(s13_EXd.to_numpy(), 1),
                           "n_synergy_partners": s13_sigM.sum(1), "most_synergistic_partner": s13_EXd.idxmax(1), "most_redundant_partner": s13_EXd.idxmin(1)}, index=S13_SYN_COLS)
print(s13_colsum.sort_values("info_alone_pct", ascending=False).round(3).to_string())

**Observations:**
- BILL_AMT1 is useless alone (0.17% of the uncertainty) but has 23 synergy partners. A bill only means something next to a payment or a ratio.
- N_REVOLVE (12 partners), N_PAIDFULL and N_INACTIVE (11 each) also gain from pairing.
- The strong status columns lose on average when paired (PAY_1 -2.8 points, N_LATE -3.2) because their partners repeat them.
- Story use: background. Supports showing bills only in context (bill vs payment), never alone.

#### 13.6.3 What a synergy looks like: the two strongest synergistic pairs as default-rate grids
A synergy is a default-rate surface that cannot be written as "row effect + column effect". Cells show the default rate
(cell count in brackets); bands are the coarse bands used above (band index, low → high; for status columns 0 = Inactive ... 5 = Late 3m+).

In [ ]:
s13_show_pairs = s13_top_syn.head(2) if len(s13_top_syn) >= 2 else s13_syn.nlargest(2, "synergy_excess")
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
for ax, (_, r) in zip(axes, s13_show_pairs.iterrows()):
    d = pd.DataFrame({"a": s13_K5_all[r.var_a], "b": s13_K5_all[r.var_b], "y": s13_y})
    rate = d.pivot_table(index="a", columns="b", values="y", aggfunc="mean"); cnt = d.pivot_table(index="a", columns="b", values="y", aggfunc="size")
    lab = rate.copy().astype(object)
    for i in rate.index:
        for j in rate.columns:
            lab.loc[i, j] = "" if pd.isna(rate.loc[i, j]) else f"{rate.loc[i, j]:.0%}\n({int(cnt.loc[i, j])})"
    sns.heatmap(rate.where(cnt >= 30), annot=lab, fmt="", cmap=CMAP_RATE, vmin=0, vmax=.8, ax=ax, annot_kws={"size": 7.5}, cbar_kws={"format": mtick.PercentFormatter(1)})
    ax.set_ylabel(f"{r.var_a} band"); ax.set_xlabel(f"{r.var_b} band")
    ax.set_title(f"{r.var_a} x {r.var_b}: joint {r.info_joint_pct:.1f}% vs {r.info_a_pct:.1f}% + {r.info_b_pct:.1f}% (excess {r.synergy_excess_pct:+.2f}%)", fontsize=10)
plt.tight_layout(); savefig("s13_synergy_examples"); plt.show()

**Observations:**
- These grids are images. Numbers come from the top-synergy table in 13.6.1.
- The two drawn pairs, AVG_PAY_RATIO x N_REVOLVE and PAY_6 x DELAY_TREND, remove 13.2% and 15.4% of the uncertainty together, vs 6.5% and 9.2% for the two parts added up.
- So the effect of one column on default depends on the band of the other; it cannot be read as a row effect plus a column effect.
- Story use: a candidate two-way heatmap for Tableau (pay ratio band x months revolving).

### 13.7 Into how many information "families" do the columns collapse?
Section 7.8 cut a dendrogram of numeric features. Here the grouping uses the **mixed-type** association matrix of 13.1 (so the
nominal columns take part) and a graph method: columns are nodes, an edge joins two columns whose association is ≥ 0.30
(weight = association), and greedy modularity maximisation (networkx; an algorithm that splits a network into tightly linked groups) finds the communities.

For each family: the **representative** (member with the highest mutual information with `DEFAULT`), the **hub** (member most
associated with the rest of the family), and the family's **joint information** about `DEFAULT`, estimated honestly: the top
three members (coarse bands) define cells on a random half of the data and are scored on the other half (held-out log-loss gain, where log-loss measures how wrong probability guesses are,
so that large joint tables cannot inflate the figure).

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities
S13_FAM_THR = 0.30
s13_FG = nx.Graph(); s13_FG.add_nodes_from(s13_feat)
for i, a in enumerate(s13_feat):
    for b in s13_feat[i + 1:]:
        w = s13_A.loc[a, b]
        if w >= S13_FAM_THR:
            s13_FG.add_edge(a, b, weight=float(w))
s13_comms = sorted([sorted(c) for c in greedy_modularity_communities(s13_FG, weight="weight")], key=lambda c: -len(c))
s13_fam_of = {c: k + 1 for k, cm in enumerate(s13_comms) for c in cm}

# honest (held-out) information of a set of banded columns about DEFAULT
s13_half_mask = np.zeros(S13_N, bool); s13_half_mask[s13_rng.choice(S13_N, S13_N // 2, replace=False)] = True
def s13_cells(K):
    cid = np.zeros(len(K), np.int64)
    for j in range(K.shape[1]):
        cid = cid * (int(K[:, j].max()) + 1) + K[:, j]
    return np.unique(cid, return_inverse=True)[1].reshape(-1)
def s13_hb(p):
    p = np.clip(p, 1e-12, 1 - 1e-12); return -(p * np.log(p) + (1 - p) * np.log(1 - p))
def s13_info_split(K, tr, y=s13_y, m=2.0):
    # returns (plug-in info on the training half, Miller-Madow corrected, held-out info on the other half, occupied cells); nats
    cid = s13_cells(K); nc = cid.max() + 1; te = ~tr
    nn = np.bincount(cid[tr], minlength=nc); n1 = np.bincount(cid[tr], weights=y[tr], minlength=nc)
    base = y[tr].mean(); occ = nn > 0
    info_in = s13_hb(base) - (nn[occ] / nn.sum() * s13_hb(n1[occ] / nn[occ])).sum()
    info_mm = info_in - (occ.sum() - 1) / (2 * nn.sum())
    pc = (n1 + m * base) / (nn + m)
    pt = np.clip(pc[cid[te]], 1e-6, 1 - 1e-6); yt = y[te]
    ce = -(yt * np.log(pt) + (1 - yt) * np.log(1 - pt)).mean()
    ce0 = -(yt * np.log(base) + (1 - yt) * np.log(1 - base)).mean()
    return float(info_in), float(info_mm), float(ce0 - ce), int(occ.sum())

s13_mi_def = s13_MI.loc[s13_feat, TARGET]
s13_fam_rows = []
for k, cm in enumerate(s13_comms, 1):
    sub = s13_A.loc[cm, cm].to_numpy()
    inner = sub[np.triu_indices(len(cm), 1)] if len(cm) > 1 else np.array([np.nan])
    rep = s13_mi_def[cm].idxmax()
    hub = cm[int(np.argmax(sub.sum(1)))] if len(cm) > 1 else cm[0]
    top3 = list(s13_mi_def[cm].sort_values(ascending=False).index[:3])
    _, _, out3, cells3 = s13_info_split(s13_K5_all[top3].to_numpy(), s13_half_mask)
    _, _, out1, _ = s13_info_split(s13_K5_all[[rep]].to_numpy(), s13_half_mask)
    outside = [c for c in s13_feat if c not in cm]
    s13_fam_rows.append({"family": k, "n_members": len(cm), "representative": rep, "hub": hub, "blocks": ", ".join(sorted({s13_block(c) for c in cm})),
                         "mean_assoc_inside": np.nanmean(inner), "mean_assoc_outside": s13_A.loc[cm, outside].to_numpy().mean() if outside else np.nan,
                         "rep_info_pct": 100 * out1 / S13_HD, "family_joint_info_pct": 100 * max(out3, out1) / S13_HD, "joint_from": " + ".join(top3),
                         "members": ", ".join(cm)})
s13_fam = pd.DataFrame(s13_fam_rows).sort_values("family_joint_info_pct", ascending=False).reset_index(drop=True)
s13_fam["gain_beyond_rep_pct"] = s13_fam["family_joint_info_pct"] - s13_fam["rep_info_pct"]
with pd.option_context("display.max_colwidth", 200):
    print(f"{len(s13_feat)} columns -> {len(s13_comms)} families (edge threshold {S13_FAM_THR}, modularity communities); {sum(len(c) == 1 for c in s13_comms)} singletons")
    print(s13_fam.drop(columns=["members"]).round(3).to_string(index=False))
    print(); print(s13_fam[["family", "members"]].to_string(index=False))
s13_fam_members = pd.DataFrame({"column": s13_feat, "family": [s13_fam_of[c] for c in s13_feat], "block": [s13_block(c) for c in s13_feat],
                                "nmi_with_default": s13_NMI.loc[s13_feat, TARGET].to_numpy(), "is_representative": [c in set(s13_fam.representative) for c in s13_feat]})
export(s13_fam_members, "s13_family_members"); export(s13_fam, "s13_families")

**Observations:**
- The 47 columns collapse into 6 families. Three of them are single columns (EDUCATION, SEX, NEG_BILL_ANY).
- The status family (21 members, led by PAY_1) removes 14.2% of the uncertainty with PAY_1 alone and 16.0% with its best three members.
- The money family (21 members: bills, payments, spend) reaches only 5.9% even with three members.
- AGE + MARRIAGE form a small family worth 0.28%.
- Story use: a dashboard design rule. One field per family is enough: status family first, money second.

#### 13.7.1 The redundancy map
The association graph coloured by family. Node size = mutual information with `DEFAULT`; a black ring marks each family's
representative. Tight, isolated clusters are fields the dashboard can summarise with one number; nodes bridging two clusters
are the ratio-type features that mix blocks.

In [ ]:
s13_fpal = sns.color_palette("tab20", max(len(s13_comms), 3))
s13_fpos = nx.spring_layout(s13_FG, weight="weight", seed=7, k=2.4 / np.sqrt(len(s13_feat)), iterations=300)
fig, ax = plt.subplots(figsize=(16, 12))
s13_fe = list(s13_FG.edges(data=True))
nx.draw_networkx_edges(s13_FG, s13_fpos, ax=ax, width=[.3 + 3.5 * (e["weight"] - S13_FAM_THR) / (1 - S13_FAM_THR) for _, _, e in s13_fe],
                       edge_color=["#444444" if s13_fam_of[a] == s13_fam_of[b] else "#bbbbbb" for a, b, _ in s13_fe], alpha=.45)
s13_sizes = [150 + 4500 * float(s13_NMI.loc[c, TARGET]) / float(s13_NMI.loc[s13_feat, TARGET].max()) * .35 for c in s13_FG.nodes]
s13_reps = set(s13_fam.representative)
nx.draw_networkx_nodes(s13_FG, s13_fpos, ax=ax, node_color=[s13_fpal[s13_fam_of[c] - 1] for c in s13_FG.nodes], node_size=s13_sizes,
                       edgecolors=["k" if c in s13_reps else "white" for c in s13_FG.nodes], linewidths=[2.2 if c in s13_reps else .8 for c in s13_FG.nodes])
nx.draw_networkx_labels(s13_FG, {k: (v[0], v[1] + .04) for k, v in s13_fpos.items()}, ax=ax, font_size=7.5)
ax.legend(handles=[S13_Patch(color=s13_fpal[k], label=f"family {k + 1} ({len(cm)}): {s13_fam.set_index('family').loc[k + 1, 'representative']}") for k, cm in enumerate(s13_comms)],
          fontsize=7.5, loc="lower left", ncol=2, title="family (size): representative")
ax.set_title(f"13.7 Redundancy map: {len(s13_feat)} columns in {len(s13_comms)} information families (edge = association >= {S13_FAM_THR}; size = MI with DEFAULT)")
ax.axis("off"); plt.tight_layout(); savefig("s13_family_graph"); plt.show()
s13_fam_nodes = s13_fam_members.assign(x=[s13_fpos[c][0] for c in s13_feat], y=[s13_fpos[c][1] for c in s13_feat])
export(s13_fam_nodes, "s13_family_graph_nodes")
s13_big = s13_fam.sort_values("n_members", ascending=False).iloc[0]
s13_bestf = s13_fam.iloc[0]
add_fact("13.7", "redundancy", f"The {len(s13_feat)} columns collapse into {len(s13_comms)} information families ({sum(len(c) == 1 for c in s13_comms)} singletons); "
         f"the largest has {int(s13_big.n_members)} members (representative {s13_big.representative}); the most informative family (around {s13_bestf.representative}) "
         f"removes {s13_bestf.family_joint_info_pct:.1f}% of the uncertainty about default (held-out)", len(s13_comms), "s13_family_graph", 3)

**Observations:**
- This graph is an image. Numbers come from the family table in 13.7.
- The status and money families each hold 21 columns. Links inside them average 0.40 and 0.49, against about 0.22 to columns outside.
- Demographics sit apart (average link to other columns 0.04-0.11).
- Story use: background. One picture explains why the dashboard needs only a few fields.

### 13.8 How much uncertainty about default is left after one column, two, ... eight?
The conditional-entropy view. H(DEFAULT) is the uncertainty before looking at anything (a 22% base rate is 0.76 bits). Knowing a
column X leaves H(D | X) = H(D) − I(X ; D).

1. **One column:** share of uncertainty removed by each of the 47 columns (coarse bands), in-sample and held-out.
2. **Best pairs:** the 15 most informative pairs among the 26 columns of 13.6.
3. **Greedy forward selection (information staircase):** starting from nothing, repeatedly add the column with the largest
   conditional mutual information I(X ; D | already chosen), up to 8 columns. Selection uses a random half of the clients
   (with a Miller-Madow bias correction, a small-sample fix for entropy estimates, so that many-level columns are not favoured); the staircase is then read on the
   **other half** as held-out log-loss gain. The in-sample curve keeps climbing (it memorises cells); the held-out curve shows
   where knowledge really saturates. The procedure is repeated with the halves swapped to check stability.

In [ ]:
s13_single = []
for c in s13_feat:
    i_in, i_mm, i_out, cells = s13_info_split(s13_K5_all[[c]].to_numpy(), s13_half_mask)
    s13_single.append({"column": c, "block": s13_block(c), "family": s13_fam_of[c], "bands": cells, "removed_in_sample_pct": 100 * i_in / S13_HD,
                       "removed_heldout_pct": 100 * i_out / S13_HD})
s13_single = pd.DataFrame(s13_single).sort_values("removed_heldout_pct", ascending=False).reset_index(drop=True)
s13_single["remaining_pct"] = 100 - s13_single["removed_heldout_pct"]
print(f"H(DEFAULT) = {S13_HD / np.log(2):.3f} bits. Uncertainty removed by each single column (% of H; held-out half):")
print(s13_single.round(2).to_string(index=False))
export(s13_single, "s13_uncertainty_single")
s13_pairs_top = s13_syn.nlargest(15, "info_joint")[["var_a", "var_b", "info_a_pct", "info_b_pct", "info_joint_pct", "synergy_pct"]].copy()
s13_pairs_top["heldout_joint_pct"] = [100 * s13_info_split(s13_K5_all[[a, b]].to_numpy(), s13_half_mask)[2] / S13_HD for a, b in zip(s13_pairs_top.var_a, s13_pairs_top.var_b)]
print("\nTop 15 pairs by joint information (% of H; in-sample on all rows, and held-out):")
print(s13_pairs_top.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 8.5))
s13_sg = s13_single.head(25).iloc[::-1]
ax.barh(s13_sg["column"], s13_sg["removed_heldout_pct"], color=[S13_BLOCK_COL[b] for b in s13_sg["block"]])
ax.set_xlabel("% of the uncertainty about DEFAULT removed by knowing this one column (held-out)")
ax.set_title("13.8 One column at a time: how much does each column tell? (top 25)")
ax.legend(handles=[S13_Patch(color=v, label=k) for k, v in S13_BLOCK_COL.items() if k != "target"], fontsize=8, loc="lower right")
plt.tight_layout(); savefig("s13_uncertainty_single"); plt.show()

**Observations:**
- H(DEFAULT) = 0.76 bits is the starting uncertainty about default. The best single column, PAY_1, removes 14.2% of it on held-out data.
- Lateness counts come close (N_LATE 13.6%, LATE_STREAK_MAX 13.3%). Older status codes fall from 9.1% (PAY_2) to 4.9% (PAY_6).
- The best non-status column removes only 4.4% (N_ZERO_PAY), LIMIT_BAL 2.6% and single payment months 1.2-2.8%.
- Story use: ranking bar for the "what matters" chart: recent status first, everything else far behind.

#### 13.8.1 The information staircase
Each step adds the column with the highest conditional information given the columns already chosen. Bars: held-out share of
uncertainty removed (cumulative). Dashed line: the same sets scored in-sample (optimistic). Grey line: number of occupied cells
(log scale, right axis), which shows why the two curves separate.

In [ ]:
def s13_greedy(tr, max_k=8, cands=None):
    cands = list(cands or s13_feat); chosen, rows = [], []
    prev_mm = 0.0; prev_out = 0.0
    for step in range(1, max_k + 1):
        best = None
        for c in cands:
            if c in chosen: continue
            i_in, i_mm, i_out, cells = s13_info_split(s13_K5_all[chosen + [c]].to_numpy(), tr)
            if best is None or i_mm > best[1]:
                best = (c, i_mm, i_in, i_out, cells)
        c, i_mm, i_in, i_out, cells = best
        chosen.append(c)
        rows.append({"step": step, "added": c, "block": s13_block(c), "family": s13_fam_of[c], "cells": cells,
                     "in_sample_pct": 100 * i_in / S13_HD, "in_sample_corrected_pct": 100 * i_mm / S13_HD, "heldout_pct": 100 * i_out / S13_HD,
                     "cond_info_gain_corrected_pct": 100 * (i_mm - prev_mm) / S13_HD, "heldout_gain_pct": 100 * (i_out - prev_out) / S13_HD})
        prev_mm, prev_out = i_mm, i_out
    return pd.DataFrame(rows)

s13_t = time.time()
s13_stair = s13_greedy(s13_half_mask)
s13_stair_b = s13_greedy(~s13_half_mask)
print(f"greedy forward selection over {len(s13_feat)} candidate columns, 8 steps, two folds: {time.time() - s13_t:.1f}s")
print("Fold A (select on half A, score on half B):"); print(s13_stair.round(2).to_string(index=False))
print("\nFold B (halves swapped):"); print(s13_stair_b[["step", "added", "cells", "in_sample_pct", "heldout_pct", "heldout_gain_pct"]].round(2).to_string(index=False))
s13_same = sum(a == b for a, b in zip(s13_stair.added, s13_stair_b.added))
print(f"\nsame column chosen at the same step in both folds: {s13_same} of 8 | common columns among the 8: {len(set(s13_stair.added) & set(s13_stair_b.added))}")
s13_peak = int(s13_stair.loc[s13_stair.heldout_pct.idxmax(), "step"])
s13_step90 = int(s13_stair.loc[(s13_stair.heldout_pct >= .9 * s13_stair.heldout_pct.max()).idxmax(), "step"])
print(f"held-out information peaks at step {s13_peak} ({s13_stair.heldout_pct.max():.1f}% of H removed); 90% of that peak is reached at step {s13_step90}")

fig, ax = plt.subplots(figsize=(13, 6.5))
s13_xs = np.arange(0, 9)
s13_ho = np.concatenate([[0], s13_stair["heldout_pct"]]); s13_ins = np.concatenate([[0], s13_stair["in_sample_pct"]])
ax.step(s13_xs, s13_ho, where="post", color=C_DEF, lw=2.5, label="held-out (honest)")
ax.fill_between(s13_xs, 0, s13_ho, step="post", color=C_DEF, alpha=.15)
ax.plot(s13_xs, s13_ins, color=C_NEU, ls="--", marker="o", lw=1.5, label="in-sample plug-in (optimistic)")
ax.plot(s13_xs, np.concatenate([[0], s13_stair_b["heldout_pct"]]), color=C_OK, ls=":", marker="s", ms=4, lw=1.5, label="held-out, halves swapped")
for _, r in s13_stair.iterrows():
    ax.annotate(f"+{r.added}\n{r.heldout_gain_pct:+.1f} pts", (r.step, r.heldout_pct), xytext=(r.step - .45, r.heldout_pct + 2.2 + (1.6 if r.step % 2 else 0)), fontsize=8,
                arrowprops={"arrowstyle": "-", "color": "k", "lw": .6})
ax.set_xticks(s13_xs); ax.set_xlabel("number of columns known"); ax.set_ylabel("% of uncertainty about DEFAULT removed")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(100)); ax.set_ylim(0, max(s13_ins.max(), s13_ho.max()) * 1.25)
ax2 = ax.twinx(); ax2.plot(s13_stair["step"], s13_stair["cells"], color="#bbbbbb", lw=1, marker=".")
ax2.set_yscale("log"); ax2.set_ylabel("occupied cells (log)"); ax2.grid(False)
ax.legend(loc="upper left", fontsize=9)
ax.set_title(f"13.8 Information staircase: {s13_stair.heldout_pct.iloc[0]:.0f}% of the uncertainty goes with the first column, "
             f"the held-out total peaks at {s13_stair.heldout_pct.max():.0f}% - {100 - s13_stair.heldout_pct.max():.0f}% is never explained")
plt.tight_layout(); savefig("s13_information_staircase"); plt.show()
export(pd.concat([s13_stair.assign(fold="A"), s13_stair_b.assign(fold="B")], ignore_index=True), "s13_information_staircase")
add_fact("13.8", "information saturation", f"Knowing only {s13_stair.added.iloc[0]} removes {s13_stair.heldout_pct.iloc[0]:.1f}% of the uncertainty about default; the best {s13_peak} columns together "
         f"reach {s13_stair.heldout_pct.max():.1f}% (held-out), so about {100 - s13_stair.heldout_pct.max():.0f}% of the uncertainty is not explained by any combination of banded columns",
         float(s13_stair.heldout_pct.max()), "s13_information_staircase", 3)
add_fact("13.8", "information saturation", f"Knowledge saturates fast: 90% of the attainable information is reached with {s13_step90} column(s); "
         f"columns added after step {s13_peak} lower held-out information (cells become too thin)", s13_step90, "s13_information_staircase", 2)

**Observations:**
- Adding the best column one at a time, held-out information peaks at 3 columns (PAY_1, N_LATE, TOTAL_PAY), which remove 16.6% of the uncertainty.
- After step 3, held-out information falls (to 7.3% by step 8) while in-sample keeps rising to 48%. The cells get too thin and start fitting noise.
- The two halves of the data agree on only 3 of the 8 steps, but both start with PAY_1 and then N_LATE.
- Story use: key message. About 83% of the uncertainty about default is not explained by any combination of these columns.

#### 13.8.2 Does the staircase depend on where you start?
Remove the dominant column (`PAY_1`) and its family from the candidates and rerun: how much of the story can the *other*
columns tell without the September status? And how far do demographics alone get?

In [ ]:
s13_fam_first = s13_fam_of[s13_stair.added.iloc[0]]
s13_alt = {"all columns": s13_stair,
           f"without {s13_stair.added.iloc[0]}": s13_greedy(s13_half_mask, 6, [c for c in s13_feat if c != s13_stair.added.iloc[0]]),
           "without its whole family": s13_greedy(s13_half_mask, 6, [c for c in s13_feat if s13_fam_of[c] != s13_fam_first]),
           "no status information at all": s13_greedy(s13_half_mask, 6, [c for c in s13_feat if s13_block(c) != "status" and c not in
                                                    ("N_LATE", "MAX_DELAY", "LATE_STREAK_MAX", "LATE_STREAK_RECENT", "N_INACTIVE", "N_PAIDFULL", "N_REVOLVE", "DELAY_TREND")]),
           "demographics + credit line only": s13_greedy(s13_half_mask, 5, S13_DEMO)}
fig, ax = plt.subplots(figsize=(12, 6))
s13_alt_rows = []
for (nm, tb), colr in zip(s13_alt.items(), [C_DEF, "#f1a340", C_ACC, C_OK, "#7b3294"]):
    ax.plot(np.arange(len(tb) + 1), np.concatenate([[0], tb["heldout_pct"]]), marker="o", lw=2, color=colr, label=f"{nm}: {' > '.join(tb.added.iloc[:3])} ...")
    s13_alt_rows.append({"candidate_set": nm, "first": tb.added.iloc[0], "first_pct": tb.heldout_pct.iloc[0], "peak_pct": tb.heldout_pct.max(),
                         "peak_step": int(tb.loc[tb.heldout_pct.idxmax(), "step"]), "order": " > ".join(tb.added)})
ax.set_xlabel("number of columns known"); ax.set_ylabel("% of uncertainty removed (held-out)"); ax.yaxis.set_major_formatter(mtick.PercentFormatter(100))
ax.legend(fontsize=8, loc="center right"); ax.set_title("13.8 Staircases from different starting sets")
plt.tight_layout(); savefig("s13_staircase_alternatives"); plt.show()
s13_alt_tbl = pd.DataFrame(s13_alt_rows)
with pd.option_context("display.max_colwidth", 200):
    print(s13_alt_tbl.round(2).to_string(index=False))
export(s13_alt_tbl, "s13_staircase_alternatives")
s13_nostat = s13_alt_tbl[s13_alt_tbl.candidate_set == "no status information at all"].iloc[0]
s13_demo = s13_alt_tbl[s13_alt_tbl.candidate_set == "demographics + credit line only"].iloc[0]
add_fact("13.8", "information saturation", f"Without any repayment-status information the best columns remove {s13_nostat.peak_pct:.1f}% of the uncertainty (vs {s13_stair.heldout_pct.max():.1f}% with it); "
         f"demographics plus credit line alone reach {s13_demo.peak_pct:.1f}%", float(s13_nostat.peak_pct), "s13_staircase_alternatives", 2)

**Observations:**
- Without PAY_1, N_LATE takes its place and reaches 15.6%, almost the same as with it.
- Without any status information the best set reaches only 9.2% (led by N_ZERO_PAY). Demographics plus credit limit reach only 2.6%.
- Story use: strong comparison bar: behaviour 16.6% vs demographics 2.6%.

### 13.9 Do extremes move together? Quantile and tail dependence between money columns
A correlation is an average over the whole distribution. For exposure the tails matter: when the bill is in its top 5%, how often
is the payment (or next month's bill, or the credit line) also in its top 5%?

χ(q) is the conditional probability that *v* is beyond its q-quantile given that *u* is: lower tail P(U<q, V<q)/q for q < 0.5,
upper tail P(U>q, V>q)/(1−q) for q ≥ 0.5. Independence gives q (lower) or 1−q (upper). The dashed reference is a **Gaussian
copula with the same Spearman ρ** (the tail behaviour a bell-curve-shaped dependence of the same strength would give): an empirical curve above it means the tails are more tightly coupled than the correlation implies.
Ranks break ties at random (payments have many exact zeros, so their *lower* tail is uninformative by construction).

In [ ]:
S13_TAIL_PAIRS = [("BILL_AMT2", "PAY_AMT1", "Aug bill vs Sep payment (settles it)"), ("BILL_AMT1", "PAY_AMT1", "Sep bill vs Sep payment"),
                  ("BILL_AMT1", "BILL_AMT2", "Sep bill vs Aug bill"), ("PAY_AMT1", "PAY_AMT2", "Sep payment vs Aug payment"),
                  ("LIMIT_BAL", "BILL_AMT1", "credit line vs Sep bill"), ("LIMIT_BAL", "PAY_AMT1", "credit line vs Sep payment")]
def s13_pobs(x, rng):
    # pseudo-observations in (0,1) with random tie-breaking
    x = np.asarray(x, float); o = np.lexsort((rng.random(len(x)), x))
    r = np.empty(len(x)); r[o] = np.arange(1, len(x) + 1)
    return r / (len(x) + 1)
def s13_chi(u, v, qs):
    return np.array([((u < q) & (v < q)).mean() / q if q < .5 else ((u > q) & (v > q)).mean() / (1 - q) for q in qs])
def s13_chi_gauss(rho_s, qs):
    rho = 2 * np.sin(np.pi * rho_s / 6); out = []
    mvn = stats.multivariate_normal(mean=[0, 0], cov=[[1, rho], [rho, 1]])
    for q in qs:
        cqq = float(mvn.cdf([stats.norm.ppf(q), stats.norm.ppf(q)]))
        out.append(cqq / q if q < .5 else (1 - 2 * q + cqq) / (1 - q))
    return np.array(out)

s13_qs = np.r_[0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.98, 0.99]
s13_trng = np.random.default_rng(139)
s13_U = {c: s13_pobs(df[c].to_numpy(), s13_trng) for c in sorted({c for p in S13_TAIL_PAIRS for c in p[:2]})}
s13_tail_rows, s13_tail_sum = [], []
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, (a, b, lab) in zip(axes.ravel(), S13_TAIL_PAIRS):
    u, v = s13_U[a], s13_U[b]
    rho_s = stats.spearmanr(df[a], df[b])[0]
    emp, gau = s13_chi(u, v, s13_qs), s13_chi_gauss(rho_s, s13_qs)
    ind = np.where(s13_qs < .5, s13_qs, 1 - s13_qs)
    e1, e0 = s13_chi(u[s13_m1], v[s13_m1], s13_qs), s13_chi(u[s13_m0], v[s13_m0], s13_qs)     # pooled ranks, conditional on outcome
    ax.plot(s13_qs, emp, marker="o", color="k", lw=2, label="empirical (all)")
    ax.plot(s13_qs, gau, ls="--", color=C_NEU, lw=2, label="Gaussian copula, same rho")
    ax.plot(s13_qs, ind, ls=":", color=C_NEU, lw=1, label="independence")
    ax.plot(s13_qs, e1, color=C_DEF, lw=1.2, alpha=.9, label="defaulters"); ax.plot(s13_qs, e0, color=C_OK, lw=1.2, alpha=.9, label="non-defaulters")
    ax.axvline(.5, color="k", lw=.6); ax.set_ylim(0, 1.02); ax.set_xlabel("q   (left half: lower tail, right half: upper tail)"); ax.set_ylabel("chi(q)")
    ax.set_title(f"{lab}\n{a} x {b}, Spearman {rho_s:.2f}", fontsize=9.5)
    for q, e, g_, i_, d1, d0 in zip(s13_qs, emp, gau, ind, e1, e0):
        s13_tail_rows.append({"var_a": a, "var_b": b, "pair": lab, "q": q, "tail": "lower" if q < .5 else "upper", "chi_empirical": e, "chi_gaussian": g_,
                              "chi_independence": i_, "chi_defaulters": d1, "chi_non_defaulters": d0})
    pick = lambda arr, q: float(arr[np.argmin(np.abs(s13_qs - q))])
    s13_tail_sum.append({"pair": f"{a} x {b}", "spearman": rho_s, "upper_0.90": pick(emp, .9), "upper_0.95": pick(emp, .95), "upper_0.99": pick(emp, .99),
                         "gauss_0.95": pick(gau, .95), "ratio_to_gauss_0.95": pick(emp, .95) / pick(gau, .95), "lift_vs_indep_0.95": pick(emp, .95) / .05,
                         "lower_0.05": pick(emp, .05), "gauss_lower_0.05": pick(gau, .05), "upper_0.95_default": pick(e1, .95), "upper_0.95_no_default": pick(e0, .95)})
axes[0, 0].legend(fontsize=7.5, loc="upper center")
fig.suptitle("13.9 Quantile dependence chi(q): are the tails more tightly coupled than the correlation implies?", fontweight="bold", y=1.0)
plt.tight_layout(); savefig("s13_tail_dependence"); plt.show()
s13_tail = pd.DataFrame(s13_tail_rows); s13_tail_sum = pd.DataFrame(s13_tail_sum)
export(s13_tail, "s13_tail_dependence")
print(s13_tail_sum.round(3).to_string(index=False))
s13_tl = s13_tail_sum.loc[s13_tail_sum["ratio_to_gauss_0.95"].idxmax()]
add_fact("13.9", "tail dependence", f"Extremes co-move more than correlation implies: for {s13_tl.pair}, {s13_tl['upper_0.95']:.0%} of clients in the top 5% of one are in the top 5% of the other "
         f"({s13_tl['lift_vs_indep_0.95']:.0f}x independence, {s13_tl['ratio_to_gauss_0.95']:.1f}x a Gaussian copula with the same Spearman {s13_tl.spearman:.2f})",
         float(s13_tl["upper_0.95"]), "s13_tail_dependence", 2)

**Observations:**
- When the August bill is in its top 5%, the September payment is also in its top 5% only 15.5% of the time. That is less than half what a normal (Gaussian) link of the same strength would give (35%).
- LIMIT_BAL x BILL_AMT1 is the opposite: almost no overall correlation (0.05) but 20.7% share the top 5%, 3.3 times the Gaussian value.
- Neighbouring bills share extremes strongly (87.9% at the top 5%).
- For defaulters, a top-5% August bill comes with a top-5% payment only 9.6% of the time, vs 17.2% for non-defaulters.
- Story use: supporting. Big bills are not paid back in big amounts, especially by defaulters.

#### 13.9.1 Copula-style rank scatter for four key pairs
The same pairs on the percentile-rank scale, where the marginal distributions are flattened and only the dependence remains
(6,000 random clients, coloured by outcome). Points massing in the top-right corner = upper-tail dependence; a band along an
edge = zero-inflation; a sharp diagonal ridge = exact settlement (payment equals the previous bill).

In [ ]:
s13_sc_rows = s13_rng.choice(S13_N, 6000, replace=False)
fig, axes = plt.subplots(1, 4, figsize=(22, 5.8))
for ax, (a, b, lab) in zip(axes, S13_TAIL_PAIRS[:4]):
    u, v = s13_U[a][s13_sc_rows], s13_U[b][s13_sc_rows]; yy = s13_y[s13_sc_rows]
    ax.scatter(u[yy == 0], v[yy == 0], s=4, alpha=.25, color=C_OK, label="No default")
    ax.scatter(u[yy == 1], v[yy == 1], s=4, alpha=.35, color=C_DEF, label="Default")
    ax.add_patch(plt.Rectangle((.9, .9), .1, .1, fill=False, ec="k", lw=1.5))
    both = (s13_U[a] > .9) & (s13_U[b] > .9)
    ax.set_title(f"{lab}\ntop-10% x top-10% box: {both.mean():.1%} of clients (1% if independent), default rate {s13_y[both].mean():.1%}", fontsize=9)
    ax.set_xlabel(f"{a} (percentile rank)"); ax.set_ylabel(f"{b} (percentile rank)"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
axes[0].legend(fontsize=8, markerscale=3, loc="upper left")
fig.suptitle("13.9 Rank (copula) scatter of four money pairs", fontweight="bold", y=1.02)
plt.tight_layout(); savefig("s13_copula_scatter"); plt.show()

**Observations:**
- This chart is an image. Numbers come from the tail table in 13.9 above.
- On the rank scale, bill-to-bill points lie along the diagonal (Spearman 0.91, top-5% overlap 0.88), while bill-to-payment points spread widely (Spearman 0.64, top-5% overlap 0.16).
- Story use: background only.

#### 13.9.2 Co-movement of extremes across all twelve money columns
For each client: in how many of the 12 monthly money columns (6 bills, 6 payments) is the client in that column's top 5%?
If extremes were independent across columns the count would be Binomial(12, 0.05). The comparison shows how concentrated the
extremes are in the same clients, and the default rate by count shows whether "extreme everywhere" clients are risky or safe.
Also: the 2 × 2 corner table of extreme bill × extreme payment.

In [ ]:
s13_money = BILL_COLS + PAMT_COLS
s13_ext = np.column_stack([s13_pobs(df[c].to_numpy(), s13_trng) > .95 for c in s13_money])
s13_cnt = s13_ext.sum(1)
s13_ext_tbl = pd.DataFrame({"n_extreme_columns": np.arange(13), "clients": np.bincount(s13_cnt, minlength=13),
                            "expected_if_independent": S13_N * stats.binom.pmf(np.arange(13), 12, .05)})
s13_ext_tbl["default_rate"] = [s13_y[s13_cnt == k].mean() if (s13_cnt == k).any() else np.nan for k in range(13)]
s13_ext_tbl["ratio_obs_to_expected"] = s13_ext_tbl["clients"] / s13_ext_tbl["expected_if_independent"]
print(s13_ext_tbl.round(3).to_string(index=False))
s13_grp = pd.cut(s13_cnt, [-1, 0, 1, 3, 6, 12], labels=["0", "1", "2-3", "4-6", "7-12"])
s13_ext_rate = rate_table(pd.DataFrame({"extreme_columns": s13_grp, TARGET: s13_y}), "extreme_columns")
print(); print(s13_ext_rate.round(3).to_string(index=False))
export(s13_ext_rate, "s13_extreme_comovement")
s13_jac = np.zeros((12, 12))
for i in range(12):
    for j in range(12):
        s13_jac[i, j] = (s13_ext[:, i] & s13_ext[:, j]).sum() / s13_ext[:, i].sum()
fig, axes = plt.subplots(1, 3, figsize=(22, 6.2), gridspec_kw={"width_ratios": [1, 1, 1.25]})
axes[0].bar(s13_ext_tbl["n_extreme_columns"] - .2, s13_ext_tbl["clients"], width=.4, color=C_DEF, label="observed")
axes[0].bar(s13_ext_tbl["n_extreme_columns"] + .2, s13_ext_tbl["expected_if_independent"], width=.4, color=C_NEU, label="Binomial(12, 5%)")
axes[0].set_yscale("log"); axes[0].set_xlabel("number of money columns in which the client is in the top 5%"); axes[0].set_ylabel("clients (log)")
axes[0].set_title("Extremes cluster in the same clients"); axes[0].legend(fontsize=8)
axes[1].bar(s13_ext_rate["extreme_columns"].astype(str), s13_ext_rate["rate"], color=C_DEF, alpha=.85,
            yerr=[s13_ext_rate["rate"] - s13_ext_rate["ci_lo"], s13_ext_rate["ci_hi"] - s13_ext_rate["rate"]], capsize=4)
axes[1].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
for x, (r, n) in enumerate(zip(s13_ext_rate["rate"], s13_ext_rate["n"])):
    axes[1].text(x, r + .012, f"n={n:,}", ha="center", fontsize=8)
axes[1].set_xlabel("number of top-5% money columns"); axes[1].set_title("Default rate by number of extreme columns")
sns.heatmap(pd.DataFrame(s13_jac, index=s13_money, columns=s13_money), annot=True, fmt=".2f", cmap="rocket_r", vmin=0, vmax=1, ax=axes[2], annot_kws={"size": 7})
axes[2].set_title("P(column in top 5% | row in top 5%)   (independence = 0.05)"); axes[2].tick_params(labelsize=7.5)
plt.tight_layout(); savefig("s13_extreme_comovement"); plt.show()

s13_corner = pd.DataFrame({"bill_top10": np.where(s13_U["BILL_AMT2"] > .9, "Aug bill top 10%", "Aug bill other"),
                           "pay": np.select([s13_U["PAY_AMT1"] > .9, df["PAY_AMT1"].to_numpy() == 0], ["Sep payment top 10%", "Sep payment zero"], "Sep payment other"), TARGET: s13_y})
s13_corner_tbl = rate_table(s13_corner, ["bill_top10", "pay"])
print("\nCorner table: extreme August bill x extreme September payment")
print(s13_corner_tbl[["bill_top10", "pay", "n", "rate", "ci_lo", "ci_hi", "lift"]].round(3).to_string(index=False))
export(s13_corner_tbl, "s13_extreme_corner_rates")
s13_multi = (s13_cnt >= 4).mean(); s13_multi_exp = 1 - stats.binom.cdf(3, 12, .05)
add_fact("13.9", "tail dependence", f"{s13_multi:.1%} of clients are in the top 5% of four or more of the 12 monthly money columns ({s13_multi / s13_multi_exp:,.0f}x what independent extremes would give); "
         f"their default rate is {s13_y[s13_cnt >= 4].mean():.1%} vs {BASE_RATE:.1%} overall", float(s13_multi), "s13_extreme_comovement", 2)

**Observations:**
- 6.0% of clients are in the top 5% of four or more of the 12 money columns, far more than independent extremes would give. 684 clients are extreme in exactly six.
- These "big account" clients default less (16.8% for 4-6 columns) than clients with no extreme column (24.5%).
- The risky corner is a top-10% August bill followed by a zero September payment: 142 clients, 57.7% default (2.6 times the base). A zero payment without a big bill gives 35.3%.
- Story use: strong "big bill, no payment" callout for the dashboard.

### 13.10 Atlas summary
Key numbers of this section in one table (all computed above), the story facts registered, and the Tableau exports.

**New `df` columns added by this section:** none (the atlas works on copies; all results are in `s13_*` tables).

| Export | Content | Suggested Tableau sheet |
|---|---|---|
| `s13_association_long` | every ordered pair of the 48 columns with all measures | "any column vs any column" explorer (parameter + highlight table / ranked bar) |
| `s13_column_connectedness` | hub-ness and link to `DEFAULT` per column | scatter: hub vs relevance |
| `s13_linear_vs_nonlinear_pairs` | Pearson, Spearman, dCor, MI per numeric pair | scatter Pearson vs dCor, filter on gap |
| `s13_differential_association`, `s13_differential_top6_curves` | correlation by outcome, Fisher z, FDR; binned curves | diverging bar of Δρ; dual-line small multiples |
| `s13_strata_correlations` | Spearman per status group / credit-line band | highlight table of sign flips |
| `s13_network_nodes`, `s13_network_edges` | sparse conditional-dependence graph with layout | network (dual-axis line + circle) |
| `s13_default_marginal_vs_partial` | marginal vs partial link with `DEFAULT` | dumbbell chart "what survives conditioning" |
| `s13_lag_atlas_long`, `s13_lead_lag` | cross-block lag correlations, lead tests | lag heatmap, lead/lag bars |
| `s13_synergy_pairs` | interaction information with permutation baseline | synergy matrix, top pairs |
| `s13_families`, `s13_family_members`, `s13_family_graph_nodes` | information families and representatives | family map |
| `s13_uncertainty_single`, `s13_information_staircase`, `s13_staircase_alternatives` | conditional-entropy view | **information staircase** (story chart) |
| `s13_tail_dependence`, `s13_extreme_comovement`, `s13_extreme_corner_rates` | quantile dependence and extremes | χ(q) lines, extreme-count bars |

In [ ]:
s13_summary = pd.DataFrame([
    ("13.1", "columns / pairs mapped", f"{len(S13_ALL)} / {len(s13_half):,}"),
    ("13.1", "share of pairs with association >= 0.5", f"{(s13_half.association >= .5).mean():.1%}"),
    ("13.1", "share of pairs with association < 0.05", f"{(s13_half.association < .05).mean():.1%}"),
    ("13.1", "most connected column / most isolated column", f"{s13_hub.index[0]} / {s13_hub.index[-1]}"),
    ("13.2", "numeric pairs with |Pearson| < 0.1 but dCor >= 0.2", f"{s13_n_hidden} of {len(s13_nl)}"),
    ("13.2", "median |Spearman| - |Pearson| over numeric pairs", f"{s13_nl.gap_monotone.median():+.3f}"),
    ("13.3", "pairs with |delta rho| >= 0.1 by outcome (FDR < 1%)", f"{((s13_diff.delta_rho.abs() >= .1) & (s13_diff.p_fdr < .01)).sum()} of {len(s13_diff)}"),
    ("13.3", "pairs flipping sign across status groups / limit bands", f"{int(s13_flip_status.flip.sum())} / {int(s13_flip_limit.flip.sum())}"),
    ("13.4", "network edges kept / possible", f"{s13_G.number_of_edges()} / {len(s13_iup[0])}"),
    ("13.4", "direct neighbours of DEFAULT", ", ".join(s13_blanket.index)),
    ("13.5", "strongest lead beyond persistence", f"{s13_lead_top.leader} -> {s13_lead_top.follower} ({s13_lead_top.partial_lead:+.2f})"),
    ("13.6", "pairs synergistic beyond chance / redundant / within chance", f"{(s13_syn.kind == 'synergy (beyond chance)').sum()} / {(s13_syn.kind == 'redundancy').sum()} / {(s13_syn.kind == 'within chance').sum()}"),
    ("13.7", "information families (singletons)", f"{len(s13_comms)} ({sum(len(c) == 1 for c in s13_comms)})"),
    ("13.8", "uncertainty removed by best single column (held-out)", f"{s13_stair.added.iloc[0]}: {s13_stair.heldout_pct.iloc[0]:.1f}%"),
    ("13.8", "peak of the information staircase (held-out)", f"{s13_stair.heldout_pct.max():.1f}% at {s13_peak} columns: {' > '.join(s13_stair.added.iloc[:s13_peak])}"),
    ("13.9", "largest upper-tail excess over Gaussian copula (q = 0.95)", f"{s13_tl.pair}: {s13_tl['ratio_to_gauss_0.95']:.2f}x"),
], columns=["subsection", "metric", "value"])
with pd.option_context("display.max_colwidth", 200):
    print(s13_summary.to_string(index=False))
export(s13_summary, "s13_summary")
print(f"\nSection 13 runtime: {time.time() - S13_T0:.0f}s on {'GPU' if S13_GPU else 'CPU'} "
      f"(samples: dCor n = {S13_DC_N:,}; graphical lasso n = {S13_GL_N:,}; everything else all {S13_N:,} rows)")

#### Story facts registered by Section 13

In [ ]:
s13_facts = pd.DataFrame([f for f in STORY_FACTS if f["section"].startswith("13")])
with pd.option_context("display.max_colwidth", 400):
    display(s13_facts)

**Observations:**
- The section registers 15 story facts. The strongest (strength 3) are the direct-links-to-default, information-family and information-staircase results.
- Only N_LATE, LATE_STREAK_RECENT and PAY_2 stay directly tied to default, and about 83% of the uncertainty about default remains after the best columns.
- Without status information the best columns reach 9.2%, and demographics plus limit only 2.6%.
- Story use: feeds the synthesis section (20) and the "what matters" dashboard page.

## 14. Conditional and higher-order interactions: what still matters once we know X?

**Why this section matters for the story.** Section 3 ranked variables one at a time and Section 4 mapped pairs. But the September repayment status is so dominant (IV ≈ 0.87) that most "important" variables might simply be *PAY_1 in disguise*. A dashboard story needs to know which signals carry **new** information, **for whom** a driver works, which signal to **believe when two disagree**, and whether the world is basically additive (so simple sheets suffice) or full of deep conjunctions.

Section 4 already did: two-way heatmaps, the additive-logit gap, a 91-pair scan, a few three-way facets and continuous surfaces. Section 9 mined ≤3-condition subgroups (no replication). Section 10 ran a regression adjustment and Simpson scan. **Nothing below repeats those**; everything here is *conditional* or *higher-order*.

| Sub-section | Question | Tableau mapping |
|---|---|---|
| 14.1 | Which variables still matter **within** a status / limit / late-count stratum? How much of each variable's IV is PAY_1 in disguise? | highlight table variable × stratum; retention bar |
| 14.2 | How much *incremental* information (conditional mutual information) does each variable add beyond status, and beyond status + late count? | ranked bars with permutation baseline |
| 14.3 | Effect modification: does a continuous driver act differently for revolvers vs late payers, men vs women, young vs old, low vs high limit? | small-multiple line charts; slope forest |
| 14.4 | Systematic 3-way scan over 13 banded variables (286 triples): is the world mostly 2-way? | ranked bar; faceted highlight tables |
| 14.5 | 4- and 5-condition pockets by beam search, **with split-half replication** | pocket table; discovery-vs-verification scatter |
| 14.6 | Dominance: when two signals disagree, which one wins? | 2×2 disagreement tables; dumbbell |
| 14.7 | Mediation-style decomposition: how much of each demographic gap passes through limit, utilisation, status? | waterfall per gap |
| 14.8 | The credit line as a universal moderator | moderation matrix |
| 14.9 | Non-additivity summary: main effects vs pairwise vs higher-order share of the pattern | one stacked bar |

**Exports:** `s14_conditional_iv`, `s14_iv_retention`, `s14_conditional_mi`, `s14_effect_mod_curves`, `s14_effect_mod_slopes`, `s14_threeway_ranking`, `s14_threeway_top_cells`, `s14_replicated_pockets`, `s14_disagreement_tables`, `s14_dominance_summary`, `s14_gap_mediation`, `s14_limit_moderation`, `s14_nonadditivity`.

**New `df` columns** (prefix `CI_`, coarse bands only) are listed in 14.10.

**Section guide**

- **What it does:** Asks what still matters once we already know the September status (the strongest signal). It re-measures every variable inside status, limit and late-count groups, checks whether drivers work differently for different people, scans three-way and deeper combinations (verified on a hold-out half), and splits demographic gaps into the parts explained by limit, utilisation and status.
- **Question it answers:** Which signals add new information beyond September status, for whom, and which signal to believe when two disagree?
- **Inputs / new columns:** `STATUS_GRP_SEP`, `N_LATE_BAND`, limit, utilisation, payment ratio, spend, demographics and setup features. Adds coarse bands with prefix `CI_` (e.g. `CI_SEP_CLEAN`, `CI_LIMIT3`, `CI_UTIL4`, `CI_PAYR5`; full list in 14.10).
- **How to read the charts:** heatmaps: row = variable, column = stratum, colour = strength (IV or AUC) inside that stratum. Line charts: one line per group; parallel lines = the driver works the same for everyone. Waterfalls: each step shows how much of a gap disappears after adjusting for one more factor.
- **Terms:** IV = how well a variable separates defaulters (above 0.3 = strong). Conditional mutual information = extra information a variable gives once status is known. Retention = share of a variable's IV that survives inside status groups. Mediation = how much of a gap "travels through" another variable. Hold-out half = data not used to find a pattern, used to check it.
- **Key takeaways:**
  - Once September status is known, past lateness, spending and payment amounts still add information; demographics add almost none (education 3.2 vs `N_LATE` 17.8 millibits).
  - A driver can flip direction by status: higher utilisation raises risk for revolvers (logit slope +1.30) but lowers it for inactive clients (-2.84).
  - Risk is mostly additive: a three-way term matters in only 10% of 286 triples and carries 0.8% of the deviance.
  - When signals disagree, believe recent lateness: late in Sep with a clean history defaults at 36.0% vs 26.9% for the reverse.
  - Demographic gaps travel mainly through the credit limit: the 7.2 pp age gap and 78% of the 5.9 pp education gap vanish once limits are equalised; the marriage gap does not.
- **Where it feeds:** the "beyond September status" and "when signals conflict" sheets, replicated pocket cards (`s14_replicated_pockets.csv`), gap waterfalls (`s14_gap_mediation.csv`) and the single non-additivity bar.

### 14.0 Coarse bands and shared helpers
Higher-order tables explode quickly, so this section uses **coarse** versions of the s00 bands (3–5 levels). They are added to `df` with the `CI_` prefix so Tableau can reuse them. Helpers: ordered integer codes for any banded column, an aggregated-cell binomial GLM (deviance on the contingency table, not on 30,000 rows), and a fast association triple (IV, Cramér's V, AUC) from integer codes.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from scipy.stats import rankdata
from scipy.special import xlogy

S14_T0 = time.time()
S14_N = len(df)
s14_y = df[TARGET].to_numpy(int)

# ---------- coarse bands (new df columns, prefix CI_) ----------
df["CI_SEP_CLEAN"] = pd.Categorical(np.where(df["PAY_1"] >= 1, "Late in Sep", "Clean in Sep"),
                                    categories=["Clean in Sep", "Late in Sep"], ordered=True)
df["CI_STATUS4"] = pd.Categorical(np.select([df["PAY_1"] <= -2, df["PAY_1"] == -1, df["PAY_1"] == 0],
                                            ["Inactive", "Paid in full", "Revolving"], default="Late"),
                                  categories=["Inactive", "Paid in full", "Revolving", "Late"], ordered=True)
df["CI_STATUS_AUG4"] = pd.Categorical(np.select([df["PAY_2"] <= -2, df["PAY_2"] == -1, df["PAY_2"] == 0],
                                                ["Inactive", "Paid in full", "Revolving"], default="Late"),
                                      categories=["Inactive", "Paid in full", "Revolving", "Late"], ordered=True)
df["CI_HIST_NLATE"] = (df[["PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]] >= 1).sum(axis=1).astype(int)
df["CI_HIST_LATE3"] = pd.cut(df["CI_HIST_NLATE"], [-1, 0, 2, 5], labels=["0", "1-2", "3-5"])
df["CI_LIMIT3"] = pd.cut(df["LIMIT_BAL"], [0, 100_000, 300_000, np.inf], labels=["<=100k", "100-300k", ">300k"])
S14_LIMIT_MED = float(df["LIMIT_BAL"].median())
s14_lim_lab = [f"Low limit (<={S14_LIMIT_MED/1000:.0f}k)", f"High limit (>{S14_LIMIT_MED/1000:.0f}k)"]
df["CI_LIMIT_HALF"] = pd.Categorical(np.where(df["LIMIT_BAL"] <= S14_LIMIT_MED, s14_lim_lab[0], s14_lim_lab[1]),
                                     categories=s14_lim_lab, ordered=True)
df["CI_AGE3"] = pd.cut(df["AGE"], [20, 30, 45, 80], labels=["21-30", "31-45", "46+"])
df["CI_UTIL4"] = pd.cut(df["AVG_UTIL"], [-np.inf, 0.1, 0.5, 0.9, np.inf], labels=["<=10%", "10-50%", "50-90%", ">90%"])
df["CI_PAYR5"] = pd.cut(df["PAY_TO_BILL_5M"], [-np.inf, 0.05, 0.25, 1.0, np.inf], labels=["<=5%", "5-25%", "25-100%", ">100%"])
df["CI_PAYR5"] = df["CI_PAYR5"].cat.add_categories("No bill").fillna("No bill")
df["CI_ZEROPAY3"] = pd.cut(df["N_ZERO_PAY"], [-1, 0, 2, 6], labels=["0", "1-2", "3-6"])
df["CI_UTILTREND3"] = pd.cut(df["UTIL_TREND"], [-np.inf, -0.02, 0.02, np.inf], labels=["Falling", "Flat", "Rising"])
df["CI_SPEND3"] = pd.qcut(df["AVG_SPEND"].rank(method="first"), 3, labels=["Low spend", "Mid spend", "High spend"])
df["CI_OVERLIMIT"] = pd.Categorical(np.where(df["OVERLIMIT_ANY"] == 1, "Over limit >=1m", "Never over limit"),
                                    categories=["Never over limit", "Over limit >=1m"], ordered=True)
df["CI_LOG_PAY_AMT1"] = np.log1p(df["PAY_AMT1"].astype(float))

S14_NEW_COLS = ["CI_SEP_CLEAN", "CI_STATUS4", "CI_STATUS_AUG4", "CI_HIST_NLATE", "CI_HIST_LATE3", "CI_LIMIT3", "CI_LIMIT_HALF",
                "CI_AGE3", "CI_UTIL4", "CI_PAYR5", "CI_ZEROPAY3", "CI_UTILTREND3", "CI_SPEND3", "CI_OVERLIMIT", "CI_LOG_PAY_AMT1"]

S14_ORDER = {"SEX_LABEL": ["Male", "Female"], "EDU_LABEL": EDU_ORDER, "MAR_LABEL": MAR_ORDER,
             "STATUS_GRP_SEP": STATUS_GROUP_ORDER, "STATUS_GRP_2": STATUS_GROUP_ORDER}
S14_NICE = {"SEX_LABEL": "Sex", "EDU_LABEL": "Education", "MAR_LABEL": "Marriage", "AGE_BAND": "Age band", "CI_AGE3": "Age (3)",
            "LIMIT_BAND": "Credit limit", "CI_LIMIT3": "Limit (3)", "CI_LIMIT_HALF": "Limit half", "STATUS_GRP_SEP": "Status Sep",
            "CI_STATUS4": "Status Sep (4)", "CI_STATUS_AUG4": "Status Aug (4)", "CI_SEP_CLEAN": "Sep clean/late",
            "N_LATE_BAND": "Late months (6m)", "CI_HIST_LATE3": "Late months Apr-Aug", "UTIL_BAND": "Utilisation",
            "CI_UTIL4": "Utilisation (4)", "PAYRATIO_BAND": "Share of bills repaid (5 statements, aligned)", "CI_PAYR5": "Bills repaid, 5 statements (4 bands)",
            "CI_ZEROPAY3": "Zero-payment months", "CI_UTILTREND3": "Utilisation trend", "CI_SPEND3": "Spend tercile",
            "CI_OVERLIMIT": "Over limit"}

def s14_nice(v):
    return S14_NICE.get(v, v)

def s14_cat_codes(col):
    # ordered integer codes (0..L-1) + labels for a banded / label column; empty levels dropped, NaN -> "Missing"
    s = df[col]
    if isinstance(s.dtype, pd.CategoricalDtype):
        levels = [l for l in s.cat.categories if bool((s == l).any())]
    elif col in S14_ORDER:
        levels = [l for l in S14_ORDER[col] if bool((s == l).any())]
    else:
        levels = sorted(s.dropna().unique().tolist())
    codes = np.array(pd.Categorical(s, categories=levels).codes, dtype=int)
    if (codes < 0).any():
        levels = list(levels) + ["Missing"]
        codes[codes < 0] = len(levels) - 1
    return codes, [str(l) for l in levels]

S14_BC = {}
for _v in ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND", "STATUS_GRP_SEP", "UTIL_BAND", "N_LATE_BAND",
           "PAYRATIO_BAND", "CI_SEP_CLEAN", "CI_STATUS4", "CI_STATUS_AUG4", "CI_HIST_LATE3", "CI_LIMIT3", "CI_LIMIT_HALF",
           "CI_AGE3", "CI_UTIL4", "CI_PAYR5", "CI_ZEROPAY3", "CI_UTILTREND3", "CI_SPEND3", "CI_OVERLIMIT"]:
    S14_BC[_v] = s14_cat_codes(_v)

def s14_fmt(v):
    if not np.isfinite(v): return "na"
    a = abs(v)
    if a >= 1000: return f"{v/1000:.0f}k"
    if a >= 10: return f"{v:.0f}"
    return f"{v:.2f}"

# ---------- aggregated-cell binomial GLM ----------
def s14_cells(cols):
    # non-empty cells of the cross-classification of banded columns: level index matrix, n, defaults, levels per factor
    cd = [S14_BC[c][0] for c in cols]
    nl = [len(S14_BC[c][1]) for c in cols]
    flat = np.ravel_multi_index(cd, nl)
    tot = int(np.prod(nl))
    n = np.bincount(flat, minlength=tot).astype(float)
    k = np.bincount(flat, weights=s14_y, minlength=tot).astype(float)
    keep = n > 0
    idx = np.array(np.unravel_index(np.where(keep)[0], nl)).T
    return idx, n[keep], k[keep], nl

def s14_design(idx, nl, order):
    # dummy design with intercept + all terms up to the given interaction order (reference = first level)
    m, f = idx.shape
    dums = []
    for j in range(f):
        D = np.zeros((m, nl[j] - 1))
        for l in range(1, nl[j]):
            D[:, l - 1] = idx[:, j] == l
        dums.append(D)
    cols = [np.ones((m, 1))]
    for r in range(1, order + 1):
        for sub in itertools.combinations(range(f), r):
            M = dums[sub[0]]
            for j in sub[1:]:
                M = (M[:, :, None] * dums[j][:, None, :]).reshape(m, -1)
            cols.append(M)
    X = np.hstack(cols)
    return X[:, np.abs(X).sum(0) > 0]

def s14_binom_dev(k, n, mu):
    mu = np.clip(mu, 1e-12, 1 - 1e-12)
    return float(2 * (xlogy(k, k / (n * mu)) + xlogy(n - k, (n - k) / (n * (1 - mu)))).sum())

def s14_irls(X, k, n, iters=60):
    # small ridge-stabilised IRLS, used only if statsmodels fails on a sparse table
    beta = np.zeros(X.shape[1]); beta[0] = np.log(k.sum() / (n.sum() - k.sum()))
    for _ in range(iters):
        eta = np.clip(X @ beta, -30, 30)
        mu = 1 / (1 + np.exp(-eta))
        w = np.clip(n * mu * (1 - mu), 1e-9, None)
        z = eta + (k - n * mu) / w
        beta_new = np.linalg.solve(X.T @ (X * w[:, None]) + 1e-6 * np.eye(X.shape[1]), X.T @ (w * z))
        if np.max(np.abs(beta_new - beta)) < 1e-8:
            beta = beta_new; break
        beta = beta_new
    return 1 / (1 + np.exp(-np.clip(X @ beta, -30, 30)))

def s14_fit(X, k, n):
    # binomial GLM on aggregated cells -> (deviance, rank of design, fitted cell rates)
    rank = int(np.linalg.matrix_rank(X))
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            res = sm.GLM(np.column_stack([k, n - k]), X, family=sm.families.Binomial()).fit(maxiter=100)
        mu = np.asarray(res.fittedvalues, float)
        dev = s14_binom_dev(k, n, mu)
        if not np.isfinite(dev): raise ValueError("non-finite deviance")
    except Exception:
        mu = s14_irls(X, k, n)
        dev = s14_binom_dev(k, n, mu)
    return dev, rank, mu

def s14_null_dev(k, n):
    return s14_binom_dev(k, n, np.full(len(k), k.sum() / n.sum()))

# ---------- association triple from integer codes ----------
def s14_assoc(codes, yv, score, eps=0.5):
    # IV (same smoothing as woe_iv), small-sample bias-adjusted IV, bias-corrected Cramer's V, directional AUC
    n = len(yv); nb = float(yv.sum()); ng = n - nb
    out = {"n": n, "default_rate": nb / n if n else np.nan, "n_levels": 0, "iv": np.nan, "iv_adj": np.nan,
           "cramers_v": np.nan, "auc": np.nan}
    if n < 50 or nb < 5 or ng < 5:
        return out
    K = int(codes.max()) + 1
    tot = np.bincount(codes, minlength=K).astype(float)
    bad = np.bincount(codes, weights=yv, minlength=K).astype(float)
    keep = tot > 0
    tot, bad = tot[keep], bad[keep]; good = tot - bad
    kk = int(keep.sum()); out["n_levels"] = kk
    if kk < 2:
        out.update({"iv": 0.0, "iv_adj": 0.0, "cramers_v": 0.0, "auc": 0.5}); return out
    dg, db = (good + eps) / (ng + eps), (bad + eps) / (nb + eps)
    iv = float(((dg - db) * np.log(dg / db)).sum())
    # under independence  n_g*n_b/n * IV ~ chi2(k-1)  ->  E[IV] = (k-1) * (1/n_g + 1/n_b)
    out["iv"], out["iv_adj"] = iv, max(0.0, iv - (kk - 1) * (1 / ng + 1 / nb))
    eb, eg = tot * nb / n, tot * ng / n
    chi2 = float((((bad - eb) ** 2) / eb + ((good - eg) ** 2) / eg).sum())
    phi2 = max(0.0, chi2 / n - (kk - 1) / (n - 1))
    kc, rc = kk - (kk - 1) ** 2 / (n - 1), 2 - 1 / (n - 1)
    out["cramers_v"] = float(np.sqrt(phi2 / max(1e-12, min(kc - 1, rc - 1))))
    r = rankdata(score)
    out["auc"] = float((r[yv == 1].sum() - nb * (nb + 1) / 2) / (nb * ng))
    return out

print("new df columns:", S14_NEW_COLS)
display(pd.DataFrame({c: df[c].value_counts(sort=False) for c in ["CI_SEP_CLEAN", "CI_STATUS4", "CI_HIST_LATE3", "CI_LIMIT3", "CI_AGE3",
                                                                  "CI_UTIL4", "CI_PAYR5", "CI_ZEROPAY3", "CI_UTILTREND3"]}.items(),
                     columns=["band", "counts"]).assign(counts=lambda t: t["counts"].map(lambda s: dict(s))))

### 14.1 Conditional importance: which variables still matter *inside* a stratum?
For every variable V (23 raw, 6 banded, 27 engineered) we recompute its **IV**, **Cramér's V** and **univariate AUC** *within each stratum* of three conditioning variables: the dominant driver `STATUS_GRP_SEP` (plus the 2-level clean / late split), the coarse credit limit `CI_LIMIT3`, and `N_LATE_BAND`.

Method notes. (i) Each V is discretised **once, globally** (deciles for continuous, own levels otherwise), so strata are comparable. (ii) IV is biased upward in small strata; `iv_adj = IV − (k−1)(1/n_good + 1/n_bad)` removes its expectation under independence. (iii) AUC is directional: for numeric V it is P(defaulter has the higher value); for nominal V levels are scored by their *marginal* default rate (so AUC < 0.5 inside a stratum = the usual ordering reverses there). `auc_strength = |AUC − 0.5| + 0.5`.

In [ ]:
S14_RAW = ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"] + PAY_COLS + BILL_COLS + PAMT_COLS
S14_BANDED = ["AGE_BAND", "LIMIT_BAND", "UTIL_BAND", "N_LATE_BAND", "PAYRATIO_BAND", "CI_HIST_LATE3"]
S14_ENG = [("PAY_TO_BILL_5M" if c == "PAY_TO_BILL_6M" else c) for c in FEATURE_COLS] + ["UTIL_1", "PAY_RATIO_1", "SPEND_1"]   # aligned repaid share replaces the legacy 6M ratio
S14_NOMINAL = {"SEX", "EDUCATION", "MARRIAGE"}
S14_ALLV = S14_RAW + S14_BANDED + S14_ENG

def s14_family(v):
    if v in ("SEX", "EDUCATION", "MARRIAGE", "AGE", "AGE_BAND"): return "Demographics"
    if v in ("LIMIT_BAL", "LIMIT_BAND"): return "Credit line"
    if v.startswith("PAY_AMT") or v.startswith("PAY_RATIO") or v in ("AVG_PAY_RATIO", "PAY_TO_BILL_6M", "PAY_TO_BILL_5M", "TOTAL_PAY", "AVG_PAY",
                                                                     "PAY_TREND", "N_ZERO_PAY", "PAYRATIO_BAND"): return "Payments"
    if v.startswith("PAY_") or v in ("N_LATE", "MAX_DELAY", "LATE_STREAK_MAX", "LATE_STREAK_RECENT", "N_INACTIVE", "N_PAIDFULL",
                                     "N_REVOLVE", "DELAY_TREND", "N_LATE_BAND", "CI_HIST_LATE3"): return "Repayment status"
    if "SPEND" in v: return "Spend"
    return "Bills / utilisation"
S14_FAM_COL = {"Demographics": "#756bb1", "Credit line": "#31a354", "Repayment status": "#e34a33", "Payments": "#2b8cbe",
               "Bills / utilisation": "#fd8d3c", "Spend": "#969696"}

def s14_disc(s, bins=10):
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 12:
        codes = np.array(pd.qcut(s, bins, duplicates="drop").cat.codes, dtype=int)
    else:
        codes = np.array(pd.factorize(s, sort=True)[0], dtype=int)
    if (codes < 0).any():
        codes[codes < 0] = codes.max() + 1
    return codes

s14_vcodes, s14_vscore = {}, {}
for v in S14_ALLV:
    s = df[v]
    cd = s14_disc(s)
    s14_vcodes[v] = cd
    if pd.api.types.is_numeric_dtype(s) and v not in S14_NOMINAL:
        s14_vscore[v] = s.astype(float).fillna(float(s.median())).to_numpy()
    else:
        lvl_rate = np.bincount(cd, weights=s14_y) / np.maximum(np.bincount(cd), 1)
        s14_vscore[v] = lvl_rate[cd]
s14_kind = {v: ("raw" if v in S14_RAW else "banded" if v in S14_BANDED else "engineered") for v in S14_ALLV}

S14_STRATIFIERS = ["STATUS_GRP_SEP", "CI_SEP_CLEAN", "CI_LIMIT3", "N_LATE_BAND"]
rows = []
for strat in ["ALL"] + S14_STRATIFIERS:
    zc, zl = (np.zeros(S14_N, int), ["All clients"]) if strat == "ALL" else S14_BC[strat]
    for j, lab in enumerate(zl):
        m = zc == j
        ym = s14_y[m]
        for v in S14_ALLV:
            a = s14_assoc(s14_vcodes[v][m], ym, s14_vscore[v][m])
            a.update({"stratifier": strat, "stratum": lab, "stratum_order": j, "variable": v, "kind": s14_kind[v], "family": s14_family(v)})
            rows.append(a)
s14_civ = pd.DataFrame(rows)
s14_civ["auc_strength"] = (s14_civ["auc"] - 0.5).abs() + 0.5
s14_civ = s14_civ[["stratifier", "stratum", "stratum_order", "variable", "kind", "family", "n", "default_rate", "n_levels",
                   "iv", "iv_adj", "cramers_v", "auc", "auc_strength"]]
export(s14_civ, "s14_conditional_iv")

print("strata sizes and default rates")
display(s14_civ.drop_duplicates(["stratifier", "stratum"])[["stratifier", "stratum", "n", "default_rate"]].reset_index(drop=True))

**Heatmap: variable × September-status stratum.** Left = bias-adjusted IV, right = AUC strength. Rows are the 28 variables with the largest marginal IV; the first column is the unconditional value, the last the n-weighted average across the six strata.

In [ ]:
s14_marg = s14_civ[s14_civ["stratifier"] == "ALL"].set_index("variable")

def s14_wavg(strat, col):
    t = s14_civ[(s14_civ["stratifier"] == strat) & s14_civ[col].notna()].copy()
    t["w"] = t[col] * t["n"]
    g = t.groupby("variable")[["w", "n"]].sum()
    return g["w"] / g["n"]

def s14_strat_matrix(strat, col, variables):
    t = s14_civ[s14_civ["stratifier"] == strat]
    w = t.pivot(index="variable", columns="stratum", values=col).reindex(variables)[S14_BC[strat][1]]
    w.insert(0, "All clients", s14_marg[col].reindex(variables))
    w["Weighted avg"] = s14_wavg(strat, col).reindex(variables)
    return w

s14_top_vars = s14_marg["iv_adj"].sort_values(ascending=False).head(28).index.tolist()
fig, axes = plt.subplots(1, 2, figsize=(19, 11))
m_iv = s14_strat_matrix("STATUS_GRP_SEP", "iv_adj", s14_top_vars)
sns.heatmap(m_iv, annot=True, fmt=".2f", cmap=CMAP_RATE, vmin=0, vmax=0.6, ax=axes[0], cbar_kws={"label": "bias-adjusted IV"}, annot_kws={"size": 8})
axes[0].set_title("IV of each variable WITHIN each Sep-status stratum"); axes[0].set_xlabel(""); axes[0].set_ylabel("")
m_auc = s14_strat_matrix("STATUS_GRP_SEP", "auc_strength", s14_top_vars)
sns.heatmap(m_auc, annot=True, fmt=".2f", cmap="mako_r", vmin=0.5, vmax=0.75, ax=axes[1], cbar_kws={"label": "AUC strength"}, annot_kws={"size": 8})
axes[1].set_title("Univariate AUC strength within each Sep-status stratum"); axes[1].set_xlabel(""); axes[1].set_ylabel("")
for ax in axes: ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); savefig("s14_cond_iv_by_status"); plt.show()
print("bias-adjusted IV by Sep-status stratum (top 28 variables by marginal IV)")
display(m_iv.round(3))

**Observations:**
- Inside each September status group, `PAY_1` (the September status itself) carries almost nothing: its IV falls from 0.875 overall to 0.001 on average. That is expected, because the group already fixes it.
- Lateness history keeps some power: `N_LATE` (number of late months) falls from IV 0.86 to 0.17 on average, still "medium" strength (0.1-0.3).
- Money variables matter most for Inactive and Paid-in-full clients: `AVG_SPEND` has IV 0.31 in both groups and `PAY_TO_BILL_5M` (share of bills repaid, 5 statements, aligned) 0.34 among Inactive clients. Among Late 2m clients (69% default anyway) almost every IV is near zero (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: the "beyond September status" heatmap; it shows which signals still sort clients once status is known.

**Same question inside credit-limit and late-count strata.** If a variable's IV collapses inside `N_LATE_BAND` strata it is mostly a delinquency proxy; if it survives inside limit strata it is not a credit-line proxy.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 11), gridspec_kw={"width_ratios": [5, 7]})
m_lim = s14_strat_matrix("CI_LIMIT3", "iv_adj", s14_top_vars)
sns.heatmap(m_lim, annot=True, fmt=".2f", cmap=CMAP_RATE, vmin=0, vmax=0.9, ax=axes[0], cbar=False, annot_kws={"size": 8})
axes[0].set_title("IV within credit-limit strata"); axes[0].set_xlabel(""); axes[0].set_ylabel("")
m_nl = s14_strat_matrix("N_LATE_BAND", "iv_adj", s14_top_vars)
m_nl.columns = ["All clients"] + [f"late months = {c}" for c in S14_BC["N_LATE_BAND"][1]] + ["Weighted avg"]
sns.heatmap(m_nl, annot=True, fmt=".2f", cmap=CMAP_RATE, vmin=0, vmax=0.9, ax=axes[1], cbar_kws={"label": "bias-adjusted IV"}, annot_kws={"size": 8})
axes[1].set_title("IV within late-month-count strata"); axes[1].set_xlabel(""); axes[1].set_ylabel("")
for ax in axes: ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); savefig("s14_cond_iv_by_limit_nlate"); plt.show()
display(m_lim.round(3).head(15)); display(m_nl.round(3).head(15))

**Observations:**
- Inside credit-limit bands nothing collapses: `PAY_1` keeps IV 0.78-0.86 in all three bands, so status is not just a stand-in for the credit line.
- Inside late-month bands, `PAY_1` still has IV 0.32-0.42 among clients with 1 to 6 late months, but 0.016 among never-late clients (none of them is late in Sep by definition).
- Payment signals mostly vanish once the late count is known: `N_ZERO_PAY` 0.26 -> 0.07, `PAY_RATIO_1` 0.26 -> 0.05.
- For high-limit clients (>300k) not paying is a sharper warning: `N_ZERO_PAY` and `PAY_RATIO_1` reach IV 0.50 and 0.49 there, vs about 0.20 below 100k.
- Story use: background; supports the order "lateness first, payments second".

**Who matters for whom?** Rank of every variable among clients who look **clean in September** (status ≤ 0: inactive, paid in full, revolving) versus among those **already late**. The rank shift column is the story: variables that jump up among clean clients are the early-warning candidates.

In [ ]:
t = s14_civ[s14_civ["stratifier"] == "CI_SEP_CLEAN"]
s14_cl = t.pivot(index="variable", columns="stratum", values="iv_adj")
s14_cl_auc = t.pivot(index="variable", columns="stratum", values="auc")
s14_rank = pd.DataFrame({"family": [s14_family(v) for v in s14_cl.index], "iv_all": s14_marg["iv_adj"].reindex(s14_cl.index),
                         "iv_clean": s14_cl["Clean in Sep"], "iv_late": s14_cl["Late in Sep"],
                         "auc_clean": s14_cl_auc["Clean in Sep"], "auc_late": s14_cl_auc["Late in Sep"]}, index=s14_cl.index)
for c in ["all", "clean", "late"]:
    s14_rank[f"rank_{c}"] = s14_rank[f"iv_{c}"].rank(ascending=False, method="min").astype(int)
s14_rank["rank_shift_clean_minus_late"] = s14_rank["rank_late"] - s14_rank["rank_clean"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
for ax, col, ttl in [(axes[0], "iv_clean", "Clean in Sep"), (axes[1], "iv_late", "Late in Sep")]:
    tt = s14_rank.sort_values(col, ascending=False).head(15).iloc[::-1]
    ax.barh(tt.index, tt[col], color=[S14_FAM_COL[f] for f in tt["family"]])
    for yy, (vv, rr) in enumerate(zip(tt[col], tt["rank_all"])):
        ax.text(vv, yy, f" {vv:.3f} (overall #{rr})", va="center", fontsize=8)
    nn = int(t[t["stratum"] == ttl]["n"].iloc[0]); rr0 = float(t[t["stratum"] == ttl]["default_rate"].iloc[0])
    ax.set_title(f"Top 15 variables among '{ttl}' clients (n={nn:,}, default {rr0:.1%})"); ax.set_xlabel("bias-adjusted IV within the stratum")
    ax.set_xlim(0, tt[col].max() * 1.35)
axes[1].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in S14_FAM_COL.values()], labels=list(S14_FAM_COL), fontsize=8, loc="lower right")
plt.tight_layout(); savefig("s14_rank_clean_vs_late"); plt.show()

print("Top 15 among clean-in-Sep clients"); display(s14_rank.sort_values("iv_clean", ascending=False).head(15).round(3))
print("Top 15 among late-in-Sep clients"); display(s14_rank.sort_values("iv_late", ascending=False).head(15).round(3))
print("Largest rank gains among clean clients (vs rank among late clients)")
display(s14_rank[s14_rank["iv_clean"] > 0.02].sort_values("rank_shift_clean_minus_late", ascending=False).head(10).round(3))

s14_top_clean = s14_rank["iv_clean"].idxmax(); s14_top_late = s14_rank["iv_late"].idxmax()
add_fact("14.1", "conditional importance",
         f"Among clients who look clean in September the most informative variable is {s14_top_clean} (IV {s14_rank.loc[s14_top_clean, 'iv_clean']:.2f}); "
         f"among clients already late it is {s14_top_late} (IV {s14_rank.loc[s14_top_late, 'iv_late']:.2f})",
         float(s14_rank.loc[s14_top_clean, "iv_clean"]), "s14_rank_clean_vs_late", 3)

**Observations:**
- Among clients clean in September (23,182 clients, 13.8% default) the top signals are past lateness (`MAX_DELAY` IV 0.18, `N_LATE` 0.175) and spending (`AVG_SPEND` 0.18).
- Spend and payment amounts climb the ranking among clean clients: `AVG_SPEND` moves from rank 44 (late clients) to rank 2 (clean clients), `PAY_AMT2` from 45 to 10.
- Among clients late in September (6,818 clients, 50.3% default) the depth of the delay (`PAY_1`, IV 0.53) and the share of bills repaid (`PAY_TO_BILL_5M`, 5 statements, aligned, 0.41) lead (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: early-warning sheet. For clean clients watch falling spend and payments; for late clients watch how much of the bill they still pay.

**How much of each variable's signal is just PAY_1 in disguise?** Retention ratio = (n-weighted average IV within strata) ÷ (marginal IV). A ratio near 0 means the variable only *sorted clients into status groups*; near 1 means its information is independent of the conditioner; above 1 means it works *better* once the conditioner is fixed (an interaction). Bars = retention given Sep status; markers = retention given late-month count and given credit limit.

In [ ]:
s14_ret = pd.DataFrame({"family": [s14_family(v) for v in S14_ALLV], "kind": [s14_kind[v] for v in S14_ALLV],
                        "iv_marginal": s14_marg["iv_adj"].reindex(S14_ALLV).to_numpy()}, index=S14_ALLV)
for strat, tag in [("STATUS_GRP_SEP", "status"), ("CI_SEP_CLEAN", "cleanlate"), ("N_LATE_BAND", "nlate"), ("CI_LIMIT3", "limit")]:
    s14_ret[f"civ_{tag}"] = s14_wavg(strat, "iv_adj").reindex(S14_ALLV)
    s14_ret[f"ret_{tag}"] = np.where(s14_ret["iv_marginal"] >= 0.01, s14_ret[f"civ_{tag}"] / s14_ret["iv_marginal"], np.nan)
s14_ret["pay1_in_disguise_share"] = (1 - s14_ret["ret_status"]).clip(lower=0)
s14_ret.index.name = "variable"
export(s14_ret.reset_index(), "s14_iv_retention")

tt = s14_ret[s14_ret["iv_marginal"] >= 0.02].sort_values("iv_marginal", ascending=False).head(34).sort_values("ret_status")
fig, axes = plt.subplots(1, 2, figsize=(17, 10), gridspec_kw={"width_ratios": [3, 2]})
ax = axes[0]
yy = np.arange(len(tt))
ax.barh(yy, tt["ret_status"].clip(upper=2), color=[S14_FAM_COL[f] for f in tt["family"]], alpha=.85)
ax.scatter(tt["ret_nlate"].clip(upper=2), yy, marker="D", color="k", s=22, zorder=3, label="given late-month count")
ax.scatter(tt["ret_limit"].clip(upper=2), yy, marker="o", facecolor="white", edgecolor="k", s=28, zorder=3, label="given credit limit (3 bands)")
ax.axvline(1, color="k", ls="--", lw=1)
ax.set_yticks(yy); ax.set_yticklabels(tt.index, fontsize=8)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("share of marginal IV retained after conditioning")
ax.set_title("IV retention: bars = given Sep status")
h1 = [plt.Rectangle((0, 0), 1, 1, color=c) for c in S14_FAM_COL.values()]
hh, ll = ax.get_legend_handles_labels()
ax.legend(handles=h1 + hh, labels=list(S14_FAM_COL) + ll, fontsize=8, loc="lower right")
ax = axes[1]
ax.scatter(tt["iv_marginal"], tt["civ_status"], c=[S14_FAM_COL[f] for f in tt["family"]], s=45)
lim = float(tt["iv_marginal"].max()) * 1.05
ax.plot([0, lim], [0, lim], "k--", lw=1); ax.plot([0, lim], [0, lim / 2], color=C_NEU, ls=":", lw=1)
for v, r in tt.sort_values("iv_marginal", ascending=False).head(14).iterrows():
    ax.annotate(v, (r["iv_marginal"], r["civ_status"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("marginal IV"); ax.set_ylabel("conditional IV given Sep status (weighted avg)")
ax.set_title("Below the dashed line = signal absorbed by Sep status (dotted = 50%)")
plt.tight_layout(); savefig("s14_iv_retention"); plt.show()

display(s14_ret.sort_values("iv_marginal", ascending=False).head(34).round(3))
s14_fam_ret = s14_ret[s14_ret["iv_marginal"] >= 0.02].groupby("family")[["ret_status", "ret_nlate", "ret_limit"]].median()
print("median retention by variable family (variables with marginal IV >= 0.02)"); display(s14_fam_ret.round(3))

_p2 = s14_ret.loc["PAY_2"]; _lb = s14_ret.loc["LIMIT_BAL"]
add_fact("14.1", "PAY_1 in disguise",
         f"Once September status is fixed, PAY_2 keeps only {_p2['ret_status']:.0%} of its IV ({_p2['iv_marginal']:.2f} -> {_p2['civ_status']:.2f}) and LIMIT_BAL keeps {_lb['ret_status']:.0%} "
         f"({_lb['iv_marginal']:.2f} -> {_lb['civ_status']:.2f}); median retention is {s14_fam_ret.loc['Repayment status', 'ret_status']:.0%} for status variables vs "
         f"{s14_fam_ret.loc['Payments', 'ret_status']:.0%} for payment variables",
         float(_p2["ret_status"]), "s14_iv_retention", 3)

**Observations:**
- Retention = share of a variable's IV that survives inside the September status groups. Median retention is 22% for status variables, 67% for payment variables, 82% for bill/utilisation and 92% for demographics.
- `PAY_2` keeps only 11% of its IV (0.55 -> 0.06): most of its signal is September status in disguise. `LIMIT_BAL` keeps 75% (0.18 -> 0.13).
- Inside late-count bands `LIMIT_BAL` keeps only 30%, so much of the limit's signal overlaps with past lateness.
- Utilisation gains inside limit bands (`AVG_UTIL` 115%, `MAX_UTIL` 135%): it says more when clients with the same limit are compared.
- Story use: the "PAY_1 in disguise" bar chart (`s14_iv_retention.csv`).

### 14.2 Incremental information: what does V tell us about default that status does not?
Conditional mutual information `I(V; D | Z) = Σ_z p(z) · I(V; D | Z = z)` (plug-in estimate, in **millibits**) for three conditioning sets: nothing, `Z₁ = STATUS_GRP_SEP` (6 groups), `Z₂ = STATUS_GRP_SEP × N_LATE_BAND` (30 cells). The plug-in estimator is biased upward, more so with many cells, so each estimate comes with a **permutation baseline**: default labels are shuffled *within* Z-cells 30 times (this preserves `D|Z` and `V|Z` but breaks `V–D|Z`). `excess = MI − permutation mean` is the honest incremental information; it is also shown as a share of the remaining uncertainty `H(D|Z)`.

In [ ]:
S14_PERM_RNG = np.random.default_rng(1402)
S14_NPERM = 30

def s14_cmi_bits(vc, K, zc, Z, yv):
    cnt = np.bincount((zc * K + vc) * 2 + yv, minlength=Z * K * 2).reshape(Z, K, 2).astype(float)
    nz = cnt.sum(axis=(1, 2), keepdims=True); nzv = cnt.sum(axis=2, keepdims=True); nzd = cnt.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        term = cnt * np.log2(cnt * nz / (nzv * nzd))
    return float(np.nansum(term) / cnt.sum())

def s14_cond_entropy(zc, Z, yv):
    cnt = np.bincount(zc * 2 + yv, minlength=Z * 2).reshape(Z, 2).astype(float)
    nz = cnt.sum(1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        h = -np.nansum(cnt * np.log2(cnt / nz))
    return float(h / cnt.sum())

def s14_perm_within(yv, zc, rng):
    order = np.argsort(zc, kind="stable")
    shuf = np.lexsort((rng.random(len(zc)), zc))
    out = np.empty_like(yv)
    out[order] = yv[shuf]
    return out

_sc, _sl = S14_BC["STATUS_GRP_SEP"]; _nc, _nl = S14_BC["N_LATE_BAND"]
S14_COND = {"none": (np.zeros(S14_N, int), 1),
            "status": (_sc, len(_sl)),
            "status_x_nlate": (_sc * len(_nl) + _nc, len(_sl) * len(_nl))}
rows = []
for tag, (zc, Z) in S14_COND.items():
    hdz = s14_cond_entropy(zc, Z, s14_y)
    perms = [s14_perm_within(s14_y, zc, S14_PERM_RNG) for _ in range(S14_NPERM)]
    for v in S14_ALLV:
        vc = s14_vcodes[v]; K = int(vc.max()) + 1
        mi = s14_cmi_bits(vc, K, zc, Z, s14_y)
        pm = np.array([s14_cmi_bits(vc, K, zc, Z, yp) for yp in perms])
        rows.append({"conditioning": tag, "variable": v, "family": s14_family(v), "kind": s14_kind[v], "H_D_given_Z_bits": hdz,
                     "mi_mbits": 1000 * mi, "perm_mean_mbits": 1000 * pm.mean(), "perm_p95_mbits": 1000 * np.percentile(pm, 95),
                     "perm_max_mbits": 1000 * pm.max(), "excess_mbits": 1000 * (mi - pm.mean()),
                     "beats_all_perms": bool(mi > pm.max()), "share_of_remaining_uncertainty": (mi - pm.mean()) / hdz})
s14_cmi = pd.DataFrame(rows)
export(s14_cmi, "s14_conditional_mi")
s14_cmi_w = s14_cmi.pivot(index="variable", columns="conditioning", values="excess_mbits")[list(S14_COND)]
s14_cmi_w["retained_given_status"] = s14_cmi_w["status"] / s14_cmi_w["none"].where(s14_cmi_w["none"] > 0.5)
s14_cmi_w["retained_given_status_nlate"] = s14_cmi_w["status_x_nlate"] / s14_cmi_w["none"].where(s14_cmi_w["none"] > 0.5)
print("remaining uncertainty about default, H(D|Z) in bits:",
      {k: round(float(s14_cmi[s14_cmi['conditioning'] == k]['H_D_given_Z_bits'].iloc[0]), 4) for k in S14_COND})
print("variables beating every permutation:", s14_cmi.groupby("conditioning")["beats_all_perms"].sum().to_dict(), "of", len(S14_ALLV))
display(s14_cmi_w.sort_values("status", ascending=False).head(25).round(2))

In [ ]:
tt = s14_cmi_w.sort_values("status", ascending=False).head(28).iloc[::-1]
p95 = s14_cmi.pivot(index="variable", columns="conditioning", values="perm_p95_mbits").reindex(tt.index)
pmean = s14_cmi.pivot(index="variable", columns="conditioning", values="perm_mean_mbits").reindex(tt.index)
fig, axes = plt.subplots(1, 2, figsize=(18, 10), gridspec_kw={"width_ratios": [3, 2]})
ax = axes[0]; yy = np.arange(len(tt)); hgt = 0.27
for off, tag, col, lab in [(hgt, "none", C_NEU, "I(V; D)  unconditional"), (0, "status", C_DEF, "I(V; D | Sep status)"),
                           (-hgt, "status_x_nlate", C_OK, "I(V; D | Sep status, late-month count)")]:
    ax.barh(yy + off, tt[tag].clip(lower=0), height=hgt, color=col, label=lab)
    ax.scatter((p95[tag] - pmean[tag]).to_numpy(), yy + off, marker="|", color="k", s=60, zorder=3,
               label="95th pct of permutation noise" if tag == "none" else None)
ax.set_yticks(yy); ax.set_yticklabels(tt.index, fontsize=8); ax.set_xscale("symlog", linthresh=1)
ax.set_xlabel("excess mutual information over permutation mean (millibits, symlog axis)")
ax.set_title("Incremental information about default: top 28 by I(V; D | Sep status)"); ax.legend(fontsize=8, loc="lower right")
ax = axes[1]
allv = s14_cmi_w[s14_cmi_w["none"] > 0.5]
ax.scatter(allv["none"], allv["status"].clip(lower=0.05), c=[S14_FAM_COL[s14_family(v)] for v in allv.index], s=40)
lim = float(allv["none"].max()) * 1.3
ax.plot([0.3, lim], [0.3, lim], "k--", lw=1)
for v, r in allv.sort_values("status", ascending=False).head(12).iterrows():
    ax.annotate(v, (r["none"], max(r["status"], 0.05)), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("I(V; D) unconditional (millibits)"); ax.set_ylabel("I(V; D | Sep status) (millibits)")
ax.set_title("Distance below the diagonal = information already in Sep status")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in S14_FAM_COL.values()], labels=list(S14_FAM_COL), fontsize=8, loc="upper left")
plt.tight_layout(); savefig("s14_conditional_mi"); plt.show()

_best = s14_cmi_w["status"].idxmax(); _best2 = s14_cmi_w["status_x_nlate"].idxmax()
_hd = {k: float(s14_cmi[s14_cmi["conditioning"] == k]["H_D_given_Z_bits"].iloc[0]) for k in S14_COND}
print(f"status removes {(1 - _hd['status'] / _hd['none']):.1%} of the uncertainty about default; status x late-count removes {(1 - _hd['status_x_nlate'] / _hd['none']):.1%}")
print("demographics, excess millibits:"); display(s14_cmi_w.loc[["SEX", "EDUCATION", "MARRIAGE", "AGE", "LIMIT_BAL"]].round(2))
add_fact("14.2", "incremental information",
         f"Beyond September status the single most informative extra variable is {_best} ({s14_cmi_w.loc[_best, 'status']:.1f} millibits, "
         f"{s14_cmi_w.loc[_best, 'retained_given_status']:.0%} of its unconditional information); beyond status and late-month count it is {_best2} "
         f"({s14_cmi_w.loc[_best2, 'status_x_nlate']:.1f} millibits)", float(s14_cmi_w.loc[_best, "status"]), "s14_conditional_mi", 2)
add_fact("14.2", "incremental information",
         f"September status alone removes {(1 - _hd['status'] / _hd['none']):.1%} of the uncertainty about default (entropy {_hd['none']:.3f} -> {_hd['status']:.3f} bits); "
         f"adding the late-month count takes it to {(1 - _hd['status_x_nlate'] / _hd['none']):.1%}", float(1 - _hd["status"] / _hd["none"]), "s14_conditional_mi", 2)

**Observations:**
- Knowing the September status removes 14.4% of the uncertainty about default (0.762 -> 0.653 bits; a bit is the unit of information). Adding the late-month count only reaches 16.7%.
- After status, the best extra variables are lateness history (`N_LATE` 17.8 millibits), then spend, payments and limit (`AVG_SPEND` 14.5, `AVG_PAY` 12.6, `LIMIT_BAL` 12.4).
- After status and late count, lateness variables drop to about 0, but money variables keep 30-50% of their information (`AVG_PAY` 10.5, `AVG_SPEND` 10.3); `UTIL_1` keeps 98%.
- Demographics add very little: education 3.2 millibits after status, sex 0.9, marriage 0.5.
- Story use: "what adds information beyond status" bar chart (`s14_conditional_mi.csv`).

### 14.3 Effect modification: does a continuous driver work the same way for everyone?
Seven continuous drivers (average utilisation, paid/billed over 6 months, credit limit, age, average implied spend, bill trend, September payment) × six modifiers (Sep status, sex, education, marriage, age band, limit band).

* **Curves:** default rate by global sextile of the driver, one line per modifier level (points with n < 40 hidden). Parallel lines = no effect modification.
* **Slopes:** within each modifier level a logit `D ~ a + b·q`, where `q` is the driver's **percentile rank** (0–1). So `b` is the log-odds change from the bottom to the top of the driver's distribution: comparable across drivers and immune to outliers.
* **Test:** likelihood-ratio test of `modifier × q` against `modifier + q` (df = levels − 1), BH-FDR across all driver × modifier combinations.

In [ ]:
S14_DRIVERS = {"AVG_UTIL": "Avg utilisation", "PAY_TO_BILL_5M": "Share of bills repaid (5 statements, aligned)", "LIMIT_BAL": "Credit limit (NT$)", "AGE": "Age",
               "AVG_SPEND": "Avg implied spend (NT$)", "BILL_TREND": "Bill trend (NT$/month)", "CI_LOG_PAY_AMT1": "log(1 + Sep payment)"}
S14_MODS = ["STATUS_GRP_SEP", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "CI_AGE3", "CI_LIMIT3"]
S14_EM_SKIP = {("LIMIT_BAL", "CI_LIMIT3"), ("AGE", "CI_AGE3")}
S14_NBIN = 6

rows = []
for drv in S14_DRIVERS:
    x = df[drv].astype(float)
    b = pd.qcut(x, S14_NBIN, duplicates="drop")
    bc = np.array(b.cat.codes, dtype=int)
    med = x.groupby(bc).median()
    for mod in S14_MODS:
        if (drv, mod) in S14_EM_SKIP: continue
        mc, ml = S14_BC[mod]
        ok = bc >= 0
        g = pd.DataFrame({"m": mc[ok], "b": bc[ok], "y": s14_y[ok]}).groupby(["m", "b"])["y"].agg(n="size", defaults="sum").reset_index()
        g["rate"] = g["defaults"] / g["n"]
        g["ci_lo"], g["ci_hi"] = wilson_ci(g["defaults"], g["n"])
        g["driver"], g["modifier"] = drv, mod
        g["level"] = [ml[i] for i in g["m"]]
        g["bin_median"] = g["b"].map(med)
        rows.append(g)
s14_em_curves = pd.concat(rows, ignore_index=True).rename(columns={"m": "level_order", "b": "driver_bin"})
s14_em_curves["driver_label"] = s14_em_curves["driver"].map(S14_DRIVERS); s14_em_curves["modifier_label"] = s14_em_curves["modifier"].map(s14_nice)
export(s14_em_curves, "s14_effect_mod_curves")

def s14_em_figure(drv):
    fig, axes = plt.subplots(1, len(S14_MODS), figsize=(24, 4.2), sharey=True)
    sub = s14_em_curves[s14_em_curves["driver"] == drv]
    for ax, mod in zip(axes, S14_MODS):
        t = sub[sub["modifier"] == mod]
        if t.empty:
            ax.text(0.5, 0.5, "(modifier is a band of\nthe driver itself)", ha="center", va="center", transform=ax.transAxes, color=C_NEU)
            ax.set_title(s14_nice(mod), fontsize=10); continue
        levels = S14_BC[mod][1]
        pal = sns.color_palette("viridis" if mod in ("CI_AGE3", "CI_LIMIT3") else "tab10", len(levels))
        for j, lab in enumerate(levels):
            tl = t[(t["level_order"] == j) & (t["n"] >= 40)].sort_values("driver_bin")
            if tl.empty: continue
            ax.plot(tl["driver_bin"], tl["rate"], marker="o", ms=4, lw=1.8, color=pal[j], label=lab)
            ax.fill_between(tl["driver_bin"], tl["ci_lo"], tl["ci_hi"], color=pal[j], alpha=.10)
        ax.axhline(BASE_RATE, color="k", ls="--", lw=.8)
        meds = sub.drop_duplicates("driver_bin").sort_values("driver_bin")
        ax.set_xticks(meds["driver_bin"]); ax.set_xticklabels([s14_fmt(m) for m in meds["bin_median"]], fontsize=7, rotation=30)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title(s14_nice(mod), fontsize=10)
        ax.legend(fontsize=6.5, loc="best", framealpha=.6)
    axes[0].set_ylabel("default rate")
    fig.suptitle(f"{S14_DRIVERS[drv]}: default rate by sextile (x = bin median), one line per modifier level", fontweight="bold", y=1.03)
    plt.tight_layout(); savefig(f"s14_em_{drv.lower()}"); plt.show()

for drv in ["AVG_UTIL", "PAY_TO_BILL_5M", "CI_LOG_PAY_AMT1", "AVG_SPEND"]:
    s14_em_figure(drv)

**Observations:**
- These curves are image-only; the numbers come from the slope tables printed further down (14.3 slope test and the focused slopes).
- Utilisation works in opposite directions by status: higher utilisation raises risk for revolvers (logit slope +1.30) and Late 1m clients (+0.80), but lowers it for Inactive (-2.84) and Paid-in-full clients (-0.92).
- Paying a larger share of the bill always helps, most for Late 1m clients: their default falls from 43.6% (bottom fifth of paid/billed) to 19.9% (top fifth) (number from the pre-fix run; refresh after the next Kaggle run).
- The September payment amount and spend also change slope by status (LR chi2 97 and 100 on 5 df); the September payment even changes sign (slopes -1.68 to +0.43).
- Story use: background for the effect-modification sheet.

In [ ]:
for drv in ["BILL_TREND", "LIMIT_BAL", "AGE"]:
    s14_em_figure(drv)

**Observations:**
- Image-only cell; numbers come from the slope table below.
- Rising bills go with lower risk overall (pooled slope -0.67), and how much depends on the credit line: slope -1.05 to -0.08 across the three limit bands (LR chi2 52 on 2 df, the second-strongest modification).
- A higher limit protects in most status groups (pooled slope -1.07) but not in all: the slope ranges from -1.58 to +0.51 across status groups.
- Age has almost no slope overall, but inside limit bands older clients are a bit riskier (slopes +0.13 to +0.64).
- Story use: background.

**Slope heterogeneity, quantified.** One logit slope per (driver, modifier level) with a 95% CI, plus the interaction LR test per (driver, modifier). `slope` is log-odds from the lowest to the highest percentile of the driver; `odds_ratio_top_vs_bottom = exp(slope)`; `top_minus_bottom_pp` is the model-free check (default rate in the top fifth minus the bottom fifth of the driver, within the level).

In [ ]:
def s14_glm(X, yv):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return sm.GLM(yv, X, family=sm.families.Binomial()).fit()

slope_rows, test_rows = [], []
for drv in S14_DRIVERS:
    x = df[drv].astype(float)
    okx = x.notna().to_numpy()
    q = np.full(S14_N, np.nan); q[okx] = x[okx].rank(pct=True).to_numpy()
    for mod in S14_MODS:
        if (drv, mod) in S14_EM_SKIP: continue
        mc, ml = S14_BC[mod]
        kept = []
        for j, lab in enumerate(ml):
            m = okx & (mc == j)
            nn, dd = int(m.sum()), int(s14_y[m].sum())
            if nn < 200 or dd < 20 or nn - dd < 20: continue
            try:
                r = s14_glm(np.column_stack([np.ones(nn), q[m]]), s14_y[m])
                bq, se = float(r.params[1]), float(r.bse[1])
            except Exception:
                continue
            top, bot = m & (q > 0.8), m & (q <= 0.2)
            tb = (s14_y[top].mean() - s14_y[bot].mean()) if top.sum() >= 30 and bot.sum() >= 30 else np.nan
            kept.append(j)
            slope_rows.append({"driver": drv, "modifier": mod, "level": lab, "level_order": j, "n": nn, "default_rate": dd / nn, "slope": bq,
                               "se": se, "slope_lo": bq - 1.96 * se, "slope_hi": bq + 1.96 * se, "odds_ratio_top_vs_bottom": float(np.exp(bq)),
                               "top_minus_bottom_pp": 100 * tb})
        if len(kept) < 2: continue
        m = okx & np.isin(mc, kept)
        D = np.column_stack([(mc[m] == j).astype(float) for j in kept[1:]])
        X0 = np.column_stack([np.ones(int(m.sum())), D, q[m]])
        X1 = np.column_stack([X0, D * q[m][:, None]])
        r0, r1 = s14_glm(X0, s14_y[m]), s14_glm(X1, s14_y[m])
        lr = float(2 * (r1.llf - r0.llf)); dfi = len(kept) - 1
        sl = [r_["slope"] for r_ in slope_rows if r_["driver"] == drv and r_["modifier"] == mod]
        test_rows.append({"driver": drv, "modifier": mod, "levels": len(kept), "pooled_slope": float(r0.params[-1]), "lr_chi2": lr, "df": dfi,
                          "p_value": float(stats.chi2.sf(lr, dfi)), "lr_per_df": lr / dfi, "slope_min": min(sl), "slope_max": max(sl),
                          "slope_range": max(sl) - min(sl), "sign_flip": bool(min(sl) < 0 < max(sl))})
s14_em_slopes = pd.DataFrame(slope_rows)
s14_em_tests = pd.DataFrame(test_rows)
s14_em_tests["p_fdr"] = multipletests(s14_em_tests["p_value"], method="fdr_bh")[1]
s14_em_tests = s14_em_tests.sort_values("lr_per_df", ascending=False).reset_index(drop=True)
s14_em_tests["rank"] = np.arange(1, len(s14_em_tests) + 1)
s14_em_export = s14_em_slopes.merge(s14_em_tests[["driver", "modifier", "pooled_slope", "lr_chi2", "df", "p_value", "p_fdr", "slope_range", "sign_flip", "rank"]],
                                    on=["driver", "modifier"], how="left")
s14_em_export["driver_label"] = s14_em_export["driver"].map(S14_DRIVERS); s14_em_export["modifier_label"] = s14_em_export["modifier"].map(s14_nice)
export(s14_em_export, "s14_effect_mod_slopes")
print(f"{int((s14_em_tests['p_fdr'] < 0.05).sum())} of {len(s14_em_tests)} driver x modifier combinations show slope heterogeneity at FDR 5%; "
      f"{int(s14_em_tests['sign_flip'].sum())} have a sign flip across levels")
display(s14_em_tests.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
piv = s14_em_tests.pivot(index="driver", columns="modifier", values="lr_per_df").reindex(index=list(S14_DRIVERS), columns=S14_MODS)
pv = s14_em_tests.pivot(index="driver", columns="modifier", values="p_fdr").reindex(index=list(S14_DRIVERS), columns=S14_MODS)
rg = s14_em_tests.pivot(index="driver", columns="modifier", values="slope_range").reindex(index=list(S14_DRIVERS), columns=S14_MODS)
ann = np.empty(piv.shape, dtype=object)
for i in range(piv.shape[0]):
    for j in range(piv.shape[1]):
        if np.isnan(piv.iloc[i, j]): ann[i, j] = ""
        else:
            star = "***" if pv.iloc[i, j] < 0.001 else "**" if pv.iloc[i, j] < 0.01 else "*" if pv.iloc[i, j] < 0.05 else ""
            ann[i, j] = f"{piv.iloc[i, j]:.0f}{star}\nrange {rg.iloc[i, j]:.2f}"
sns.heatmap(np.log10(piv.clip(lower=1)), annot=ann, fmt="", cmap="YlOrRd", ax=axes[0], cbar_kws={"label": "log10(LR chi2 per df)"}, annot_kws={"size": 8})
axes[0].set_yticklabels([S14_DRIVERS[d] for d in piv.index], rotation=0); axes[0].set_xticklabels([s14_nice(m) for m in piv.columns], rotation=25)
axes[0].set_title("Effect-modification matrix: LR chi2 per df (stars = FDR), slope range"); axes[0].set_xlabel("modifier"); axes[0].set_ylabel("driver")

ax = axes[1]
top = s14_em_tests.head(6)
ypos, ylab, y0 = [], [], 0
pal6 = sns.color_palette("tab10", 6)
for gi, (_, tr) in enumerate(top.iterrows()):
    sl = s14_em_slopes[(s14_em_slopes["driver"] == tr["driver"]) & (s14_em_slopes["modifier"] == tr["modifier"])].sort_values("level_order")
    for _, r in sl.iterrows():
        ax.errorbar(r["slope"], y0, xerr=1.96 * r["se"], fmt="o", color=pal6[gi], capsize=3, ms=5)
        ypos.append(y0); ylab.append(f"{S14_DRIVERS[tr['driver']].split(' (')[0]} | {r['level']}"); y0 += 1
    y0 += 0.8
ax.axvline(0, color="k", lw=1)
ax.set_yticks(ypos); ax.set_yticklabels(ylab, fontsize=7.5); ax.invert_yaxis()
ax.set_xlabel("logit slope (bottom -> top percentile of the driver), 95% CI"); ax.set_title("Slopes by level for the 6 strongest effect modifications")
plt.tight_layout(); savefig("s14_effect_modification_matrix"); plt.show()

**Observations:**
- Slope = change in log-odds of default (the scale a logistic model uses) from the lowest to the highest percentile of the driver.
- 19 of 40 driver x modifier pairs have clearly different slopes across groups (FDR 5%, i.e. after correcting for running many tests), and 12 even flip sign.
- September status is the strongest modifier: it tops the list for utilisation (chi2 168), credit limit (125), spend (100) and September payment (97).
- Sex and marriage modify almost nothing: no driver x sex or driver x marriage test passes FDR 5% (closest: limit x sex, p 0.051).
- Story use: effect-modification matrix (`s14_effect_mod_slopes.csv`). Message: the same number means different things for different status groups.

**Two focused questions.** (a) Does utilisation matter differently for revolvers vs late payers? (b) Does a higher credit limit protect at *every* age? Here the limit slope is re-estimated in all eight s00 age bands, and the age slope in all seven limit bands.

In [ ]:
def s14_slopes_by(drv, mod_col, labels=None):
    x = df[drv].astype(float); q = x.rank(pct=True).to_numpy()
    mc, ml = s14_cat_codes(mod_col)
    out = []
    for j, lab in enumerate(ml):
        m = (mc == j) & ~np.isnan(q)          # skip clients with no value (e.g. no bill to repay)
        nn, dd = int(m.sum()), int(s14_y[m].sum())
        if nn < 150 or dd < 15: continue
        r = s14_glm(np.column_stack([np.ones(nn), q[m]]), s14_y[m])
        top, bot = m & (q > 0.8), m & (q <= 0.2)
        out.append({"driver": drv, "modifier": mod_col, "level": lab, "n": nn, "default_rate": dd / nn, "slope": float(r.params[1]), "se": float(r.bse[1]),
                    "rate_bottom_fifth": s14_y[bot].mean() if bot.sum() >= 30 else np.nan, "rate_top_fifth": s14_y[top].mean() if top.sum() >= 30 else np.nan})
    return pd.DataFrame(out)

s14_f_util = s14_slopes_by("AVG_UTIL", "STATUS_GRP_SEP")
s14_f_ptb = s14_slopes_by("PAY_TO_BILL_5M", "STATUS_GRP_SEP")
s14_f_lim = s14_slopes_by("LIMIT_BAL", "AGE_BAND")
s14_f_age = s14_slopes_by("AGE", "LIMIT_BAND")
fig, axes = plt.subplots(1, 4, figsize=(22, 4.6))
for ax, t, ttl in [(axes[0], s14_f_util, "Utilisation slope by Sep status"), (axes[1], s14_f_ptb, "Share of bills repaid (5m) slope by Sep status"),
                   (axes[2], s14_f_lim, "Credit-limit slope by age band"), (axes[3], s14_f_age, "Age slope by credit-limit band")]:
    xx = np.arange(len(t))
    ax.bar(xx, t["slope"], color=[C_DEF if s > 0 else C_OK for s in t["slope"]], alpha=.8)
    ax.errorbar(xx, t["slope"], yerr=1.96 * t["se"], fmt="none", color="k", capsize=3)
    ax.axhline(0, color="k", lw=1); ax.set_xticks(xx); ax.set_xticklabels(t["level"], rotation=30, ha="right", fontsize=8)
    ax.set_title(ttl, fontsize=10); ax.set_ylabel("logit slope (bottom -> top percentile)")
plt.tight_layout(); savefig("s14_focus_slopes"); plt.show()
for t in (s14_f_util, s14_f_ptb, s14_f_lim, s14_f_age):
    display(t.round(3))

_u = s14_f_util.set_index("level")
_parts = [f"{l} {_u.loc[l, 'slope']:+.2f}" for l in _u.index]
add_fact("14.3", "effect modification",
         f"Utilisation is not a universal risk signal: its logit slope (bottom to top percentile) is {', '.join(_parts)}; among revolvers the top fifth by utilisation "
         f"defaults at {_u.loc['Revolving', 'rate_top_fifth']:.1%} vs {_u.loc['Revolving', 'rate_bottom_fifth']:.1%} for the bottom fifth",
         float(_u.loc["Revolving", "slope"]), "s14_focus_slopes", 3)
_l = s14_f_lim.set_index("level")
add_fact("14.3", "effect modification",
         f"A higher credit limit protects in {int((_l['slope'] + 1.96 * _l['se'] < 0).sum())} of {len(_l)} age bands (logit slope from {_l['slope'].min():+.2f} in {_l['slope'].idxmin()} "
         f"to {_l['slope'].max():+.2f} in {_l['slope'].idxmax()})", float(_l["slope"].min()), "s14_focus_slopes", 2)
_t1 = s14_em_tests.iloc[0]
add_fact("14.3", "effect modification",
         f"Strongest effect modification: {S14_DRIVERS[_t1['driver']]} x {s14_nice(_t1['modifier'])} (LR chi2 {_t1['lr_chi2']:.0f} on {int(_t1['df'])} df; slope ranges from "
         f"{_t1['slope_min']:+.2f} to {_t1['slope_max']:+.2f}); {int((s14_em_tests['p_fdr'] < 0.05).sum())} of {len(s14_em_tests)} driver-modifier pairs are heterogeneous at FDR 5%",
         float(_t1["lr_chi2"]), "s14_effect_modification_matrix", 2)

**Observations:**
- Among revolvers the top fifth by utilisation defaults at 17.4% vs 12.3% for the bottom fifth; among Late 1m clients 41.9% vs 28.8%.
- A higher limit protects in 7 of 8 age bands. Between 21 and 60 the top fifth by limit defaults at 11.6-16.8% vs 30.8-34.4% for the bottom fifth.
- The protection weakens with age: slope -1.71 at 26-30, -1.02 at 51-60 and -0.61 (not significant, n=272) at 61+.
- Inside each limit band age raises risk a little (slopes +0.10 to +0.66); in the 200-300k band the oldest fifth defaults at 19.9% vs 12.1% for the youngest.
- Story use: two annotated line charts; "a high limit protects, but less after 50".

### 14.4 Systematic three-way scan: is the world mostly two-way?
Section 4.7 tested three hand-picked triples. Here **all 286 triples of 13 coarse banded variables** are tested. For each triple the 30,000 rows are collapsed to the contingency table (≤ 80 cells) and three binomial GLMs are fitted on the cell counts: null, main effects, all two-way terms. Because the saturated model reproduces every cell, the **deviance of the all-two-way model is the likelihood-ratio statistic G² for the three-way term** (df = non-empty cells − rank of the design). BH-FDR across triples.

With n = 30,000 almost anything is "significant", so each triple also gets an **effect size**: the share of the cell-rate deviance carried by main effects / two-way / three-way terms, and the n-weighted mean absolute gap (pp) between observed cell rates and the two-way model. The same scan over the 78 pairs provides the yardstick.

In [ ]:
S14_SCAN = ["CI_STATUS4", "CI_STATUS_AUG4", "CI_HIST_LATE3", "CI_UTIL4", "CI_PAYR5", "CI_ZEROPAY3", "CI_UTILTREND3", "CI_SPEND3",
            "CI_LIMIT3", "CI_AGE3", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL"]
_t0 = time.time()
rows = []
for trip in itertools.combinations(S14_SCAN, 3):
    idx, n_c, k_c, nl = s14_cells(list(trip))
    d0 = s14_null_dev(k_c, n_c)
    d1, r1, _ = s14_fit(s14_design(idx, nl, 1), k_c, n_c)
    d2, r2, mu2 = s14_fit(s14_design(idx, nl, 2), k_c, n_c)
    df3 = len(n_c) - r2
    rows.append({"var_a": trip[0], "var_b": trip[1], "var_c": trip[2], "triple": " x ".join(s14_nice(v) for v in trip), "cells": len(n_c),
                 "min_cell_n": int(n_c.min()), "dev_null": d0, "dev_main": d1, "dev_2way": d2, "G2_3way": d2, "df_3way": df3,
                 "p_value": float(stats.chi2.sf(d2, df3)) if df3 > 0 else np.nan, "share_main": (d0 - d1) / d0, "share_2way": (d1 - d2) / d0,
                 "share_3way": d2 / d0, "mean_abs_gap_pp": 100 * float(np.average(np.abs(k_c / n_c - mu2), weights=n_c))})
s14_tri = pd.DataFrame(rows)
s14_tri["G2_per_df"] = s14_tri["G2_3way"] / s14_tri["df_3way"]
_ok = s14_tri["p_value"].notna()
s14_tri.loc[_ok, "p_fdr"] = multipletests(s14_tri.loc[_ok, "p_value"], method="fdr_bh")[1]
s14_tri["significant_fdr05"] = s14_tri["p_fdr"] < 0.05
s14_tri = s14_tri.sort_values("G2_per_df", ascending=False).reset_index(drop=True)
s14_tri["rank"] = np.arange(1, len(s14_tri) + 1)

rows = []
for pair in itertools.combinations(S14_SCAN, 2):
    idx, n_c, k_c, nl = s14_cells(list(pair))
    d0 = s14_null_dev(k_c, n_c)
    d1, r1, mu1 = s14_fit(s14_design(idx, nl, 1), k_c, n_c)
    df2 = len(n_c) - r1
    rows.append({"pair": " x ".join(s14_nice(v) for v in pair), "G2_2way": d1, "df_2way": df2, "p_value": float(stats.chi2.sf(d1, df2)),
                 "share_2way": d1 / d0, "mean_abs_gap_pp": 100 * float(np.average(np.abs(k_c / n_c - mu1), weights=n_c))})
s14_pairs = pd.DataFrame(rows)
s14_pairs["p_fdr"] = multipletests(s14_pairs["p_value"], method="fdr_bh")[1]
export(s14_tri, "s14_threeway_ranking")

S14_TRI_SIG = int(s14_tri["significant_fdr05"].sum()); S14_PAIR_SIG = int((s14_pairs["p_fdr"] < 0.05).sum())
print(f"scan time {time.time() - _t0:.1f}s")
print(f"three-way term significant at FDR 5%: {S14_TRI_SIG} of {len(s14_tri)} triples ({S14_TRI_SIG / len(s14_tri):.0%})")
print(f"two-way term   significant at FDR 5%: {S14_PAIR_SIG} of {len(s14_pairs)} pairs ({S14_PAIR_SIG / len(s14_pairs):.0%})")
print("deviance shares across the 286 triples (median / 90th pct):")
display(s14_tri[["share_main", "share_2way", "share_3way", "mean_abs_gap_pp"]].describe(percentiles=[.5, .9]).T.round(4))
print("pair yardstick: mean abs gap between observed and additive rates (pp):", s14_pairs["mean_abs_gap_pp"].describe(percentiles=[.5, .9]).round(3).to_dict())
print("Top 15 triples by G2 per df")
display(s14_tri.head(15)[["rank", "triple", "cells", "min_cell_n", "G2_3way", "df_3way", "G2_per_df", "p_fdr", "share_main", "share_2way", "share_3way", "mean_abs_gap_pp"]].round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6.5), gridspec_kw={"width_ratios": [3, 2, 2]})
ax = axes[0]
tt = s14_tri.head(15).iloc[::-1]
ax.barh(tt["triple"], tt["G2_per_df"], color=[C_DEF if s else C_NEU for s in tt["significant_fdr05"]])
for yy, (g, sh, gp) in enumerate(zip(tt["G2_per_df"], tt["share_3way"], tt["mean_abs_gap_pp"])):
    ax.text(g, yy, f" 3-way share {sh:.1%}, gap {gp:.2f}pp", va="center", fontsize=7.5)
ax.axvline(1, color="k", ls="--", lw=1); ax.set_xlim(0, tt["G2_per_df"].max() * 1.5)
ax.set_xlabel("G2 per df of the three-way term (1 = pure noise)"); ax.set_title("Top 15 three-way interactions"); ax.tick_params(axis="y", labelsize=8)
ax = axes[1]
ax.hist(s14_tri["p_value"].dropna(), bins=20, color=C_OK, alpha=.8, label="286 triples (3-way term)")
ax.hist(s14_pairs["p_value"], bins=20, color=C_DEF, alpha=.5, label="78 pairs (2-way term)")
ax.set_xlabel("p-value"); ax.set_ylabel("count"); ax.set_title("p-value distribution (uniform = nothing there)"); ax.legend(fontsize=8)
ax = axes[2]
sh = s14_tri[["share_main", "share_2way", "share_3way"]].mean()
ax.bar(["Main effects", "Two-way", "Three-way"], sh.to_numpy(), color=[C_OK, C_ACC, C_DEF])
for i, v in enumerate(sh.to_numpy()): ax.text(i, v, f"{v:.1%}", ha="center", va="bottom", fontweight="bold")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title("Average share of cell-rate deviance, 286 triples"); ax.set_ylim(0, 1.08)
plt.tight_layout(); savefig("s14_threeway_scan"); plt.show()

_tp = s14_tri.iloc[0]
add_fact("14.4", "higher-order",
         f"Across all {len(s14_tri)} triples of 13 banded variables the three-way term is significant (FDR 5%) in {S14_TRI_SIG} ({S14_TRI_SIG / len(s14_tri):.0%}), "
         f"versus {S14_PAIR_SIG / len(s14_pairs):.0%} of pairs for the two-way term; on average main effects carry {sh['share_main']:.1%} of the cell-rate deviance, "
         f"two-way terms {sh['share_2way']:.1%} and the three-way term {sh['share_3way']:.1%}", float(sh["share_3way"]), "s14_threeway_scan", 3)
add_fact("14.4", "higher-order",
         f"Strongest genuine three-way interaction: {_tp['triple']} (G2 {_tp['G2_3way']:.0f} on {int(_tp['df_3way'])} df, {_tp['share_3way']:.1%} of deviance, "
         f"mean cell gap {_tp['mean_abs_gap_pp']:.2f}pp vs the two-way model)", float(_tp["G2_per_df"]), "s14_threeway_top4", 2)

**Observations:**
- Of 286 triples of 13 banded variables, only 28 (10%) have a significant three-way term, vs 44 of 78 pairs (56%) for two-way terms.
- On average main effects carry 94.8% of the deviance (how far a model's cell rates miss the real ones), two-way terms 4.3% and the three-way term 0.8%.
- The three-way term moves cell rates by only 0.49 pp on average, vs 0.85 pp for two-way terms.
- The strongest triple is Sep status x Apr-Aug late months x zero-payment months (G2 57 on 11 df), and even there it is only 1.2% of the deviance.
- Story use: one line, "risk is mostly additive"; the dashboard does not need three-way filters.

**The top four triples as faceted heatmaps.** Facet = the variable with the fewest levels. Each cell shows the observed default rate, the residual versus the all-two-way model in pp (in brackets: this is *where* the three-way term lives) and n. Tableau: highlight table with the facet variable on Columns/Pages (`s14_threeway_top_cells`).

In [ ]:
def s14_triple_arrays(trip):
    idx, n_c, k_c, nl = s14_cells(list(trip))
    _, _, mu2 = s14_fit(s14_design(idx, nl, 2), k_c, n_c)
    R, Nn, E = np.full(nl, np.nan), np.zeros(nl), np.full(nl, np.nan)
    for (a, b, c_), n_, k_, m_ in zip(idx, n_c, k_c, mu2):
        R[a, b, c_], Nn[a, b, c_], E[a, b, c_] = k_ / n_, n_, m_
    return R, Nn, E, nl

cell_rows = []
for ti in range(4):
    r = s14_tri.iloc[ti]; trip = [r["var_a"], r["var_b"], r["var_c"]]
    R, Nn, E, nl = s14_triple_arrays(trip)
    f = int(np.argmin(nl)); rest = [i for i in range(3) if i != f]
    fig, axes = plt.subplots(1, nl[f], figsize=(5.2 * nl[f] + 1, 0.55 * nl[rest[0]] + 2.6), sharey=True)
    axes = np.atleast_1d(axes)
    vmax = float(np.nanmin([0.9, np.nanmax(np.where(Nn >= 30, R, np.nan))]))
    for j, ax in enumerate(axes):
        Rj, Nj, Ej = np.take(R, j, axis=f), np.take(Nn, j, axis=f), np.take(E, j, axis=f)
        ann = np.empty(Rj.shape, dtype=object)
        for a in range(Rj.shape[0]):
            for b in range(Rj.shape[1]):
                ann[a, b] = "" if Nj[a, b] == 0 else f"{Rj[a, b]:.0%} ({100 * (Rj[a, b] - Ej[a, b]):+.1f})\nn={int(Nj[a, b]):,}"
        sns.heatmap(pd.DataFrame(Rj, index=S14_BC[trip[rest[0]]][1], columns=S14_BC[trip[rest[1]]][1]), annot=ann, fmt="", cmap=CMAP_RATE,
                    vmin=0, vmax=vmax, ax=ax, cbar=(j == len(axes) - 1), annot_kws={"size": 7.5}, mask=(Nj == 0))
        ax.set_title(f"{s14_nice(trip[f])} = {S14_BC[trip[f]][1][j]}", fontsize=10)
        ax.set_xlabel(s14_nice(trip[rest[1]])); ax.set_ylabel(s14_nice(trip[rest[0]]) if j == 0 else "")
    fig.suptitle(f"#{ti + 1}  {r['triple']}   |   G2 = {r['G2_3way']:.0f} on {int(r['df_3way'])} df, FDR p = {r['p_fdr']:.1e}, 3-way share = {r['share_3way']:.1%}",
                 fontweight="bold", y=1.04)
    plt.tight_layout(); savefig(f"s14_threeway_top{ti + 1}"); plt.show()
    for a in range(nl[0]):
        for b in range(nl[1]):
            for c_ in range(nl[2]):
                if Nn[a, b, c_] > 0:
                    cell_rows.append({"triple_rank": ti + 1, "triple": r["triple"], "var_a": s14_nice(trip[0]), "level_a": S14_BC[trip[0]][1][a], "order_a": a,
                                      "var_b": s14_nice(trip[1]), "level_b": S14_BC[trip[1]][1][b], "order_b": b, "var_c": s14_nice(trip[2]),
                                      "level_c": S14_BC[trip[2]][1][c_], "order_c": c_, "n": int(Nn[a, b, c_]), "rate": R[a, b, c_],
                                      "rate_2way_model": E[a, b, c_], "threeway_resid_pp": 100 * (R[a, b, c_] - E[a, b, c_])})
s14_tri_cells = pd.DataFrame(cell_rows)
export(s14_tri_cells, "s14_threeway_top_cells")
print("largest three-way residual cells (n >= 100) in the top-4 triples")
display(s14_tri_cells[s14_tri_cells["n"] >= 100].assign(a=lambda t: t["threeway_resid_pp"].abs()).sort_values("a", ascending=False).drop(columns="a").head(12).round(3))

**Observations:**
- All three-way residuals in cells with n >= 100 stay within about 8.5 pp of the two-way model.
- Clients late in Sep with 1-2 earlier late months but no zero-payment month default at 42.1% (n=337), 8.5 pp below the model's 50.6%: still paying something helps.
- Clients late in Sep with a clean Apr-Aug but rising utilisation default at 61.6% (n=401), 8.0 pp above the model.
- Revolvers with 1-2 earlier late months and no zero-payment month default at 21.0% vs 15.1% expected (+5.8 pp).
- Story use: optional drill-down; the "still paying something" cell is a good annotation.

### 14.5 Four- and five-condition pockets, found by beam search and verified on a hold-out half
Section 9.3c enumerated every conjunction of ≤ 3 conditions, on all the data, without replication. Deeper conjunctions cannot be enumerated (62 conditions → millions of 5-sets), and deep pockets found on the full sample are exactly where noise mining happens. So:

1. **Split** the 30,000 clients randomly into half A (discovery) and half B (verification).
2. **Beam search on A only** (a search that keeps only the best 50 candidates at each step; width 50, depth 2 → 5; depth 2 is exhaustive). Four beams: highest Wilson-lower-bound default rate ("lift" beam), highest WRAcc `= coverage × (rate − base)`, lowest Wilson-upper-bound rate, and "**surprisingly safe**" = lowest rate among conjunctions that contain at least one condition that is risky on its own (solo rate ≥ 1.25 × base). A new condition must come from an unused variable, keep ≥ 75 clients in A and remove ≥ 5% of the parent pocket.
3. **Verify on B**: pockets with 4–5 conditions and n ≥ 150 overall are re-scored on half B. A high-risk pocket *replicates* if the 95% Wilson interval on B lies entirely above the base rate (below, for safe pockets). The shortlist is diversified (Jaccard overlap, the share of clients two pockets have in common, < 0.6) and each pocket must beat its best sub-pocket with one condition removed.

In [ ]:
S14_ITEM_VARS = ["STATUS_GRP_SEP", "CI_STATUS_AUG4", "CI_HIST_LATE3", "N_LATE_BAND", "UTIL_BAND", "PAYRATIO_BAND", "CI_ZEROPAY3",
                 "CI_UTILTREND3", "CI_SPEND3", "LIMIT_BAND", "CI_AGE3", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "CI_OVERLIMIT"]
s14_items, _cols = [], []
for vi, v in enumerate(S14_ITEM_VARS):
    cd, lv = S14_BC[v]
    for j, lab in enumerate(lv):
        m = cd == j
        if m.sum() >= 300:
            s14_items.append((vi, v, lab)); _cols.append(m)
S14_M = np.column_stack(_cols)
s14_item_var = np.array([it[0] for it in s14_items])
s14_item_lab = [f"{s14_nice(it[1])} = {it[2]}" for it in s14_items]
S14_NI = len(s14_items)

s14_halfA = np.random.default_rng(1405).permutation(S14_N) < S14_N // 2
S14_MA = S14_M[s14_halfA]; S14_MAf = S14_MA.astype(np.float32)
s14_yA = s14_y[s14_halfA].astype(np.float32); s14_yB = s14_y[~s14_halfA]
S14_BASE_A, S14_BASE_B = float(s14_yA.mean()), float(s14_yB.mean())
s14_item_rate = np.array([s14_y[S14_M[:, i]].mean() for i in range(S14_NI)])
s14_risky_items = set(np.where(s14_item_rate >= 1.25 * BASE_RATE)[0].tolist())
print(f"{S14_NI} conditions from {len(S14_ITEM_VARS)} variables | half A n={int(s14_halfA.sum()):,} base {S14_BASE_A:.2%} | half B n={int((~s14_halfA).sum()):,} base {S14_BASE_B:.2%}")
print("conditions risky on their own (solo rate >= 1.25 x base):"); print([f"{s14_item_lab[i]} ({s14_item_rate[i]:.0%})" for i in sorted(s14_risky_items)])

def s14_score_hi(nn, kk): return wilson_ci(kk, nn)[0]
def s14_score_wracc(nn, kk): return nn / len(s14_yA) * (kk / nn - S14_BASE_A)
def s14_score_lo(nn, kk): return -wilson_ci(kk, nn)[1]

def s14_beam(name, score_fn, width=50, max_depth=5, min_nA=75, need_risky=False, keep=300):
    current = {frozenset([i]): S14_MA[:, i] for i in range(S14_NI)}
    rec = []
    for depth in range(2, max_depth + 1):
        cand = {}
        for key, memb in current.items():
            mf = memb.astype(np.float32)
            nn = mf @ S14_MAf; kk = (mf * s14_yA) @ S14_MAf
            used = {int(s14_item_var[i]) for i in key}
            n_par = float(mf.sum())
            for j in np.where((nn >= min_nA) & (nn <= 0.95 * n_par))[0]:
                if int(s14_item_var[j]) in used: continue
                nk = key | {int(j)}
                if nk not in cand: cand[nk] = (float(nn[j]), float(kk[j]))
        if not cand: break
        keys = list(cand)
        nn = np.array([cand[q][0] for q in keys]); kk = np.array([cand[q][1] for q in keys])
        sc = score_fn(nn, kk)
        if need_risky:
            sc = np.where(np.array([len(q & s14_risky_items) > 0 for q in keys]), sc, -np.inf)
        order = np.argsort(-sc); order = order[np.isfinite(sc[order])]
        for t_ in order[:keep]:
            rec.append({"beam": name, "depth": depth, "key": tuple(sorted(keys[t_])), "nA": nn[t_], "kA": kk[t_], "score_A": sc[t_]})
        current = {keys[t_]: S14_MA[:, sorted(keys[t_])].all(axis=1) for t_ in order[:width]}
        print(f"  {name:<16} depth {depth}: {len(cand):>6,} candidates evaluated, best score {sc[order[0]]:.4f}")
    return pd.DataFrame(rec)

_t0 = time.time()
s14_beams = pd.concat([s14_beam("high lift", s14_score_hi), s14_beam("high WRAcc", s14_score_wracc),
                       s14_beam("safest", s14_score_lo), s14_beam("surprisingly safe", s14_score_lo, need_risky=True)], ignore_index=True)
print(f"beam search {time.time() - _t0:.1f}s, {len(s14_beams):,} recorded candidates, {s14_beams['key'].nunique():,} distinct pockets")

In [ ]:
def s14_eval_key(key):
    cols = list(key)
    m = S14_M[:, cols].all(axis=1)
    mA, mB = m & s14_halfA, m & ~s14_halfA
    n, k = int(m.sum()), int(s14_y[m].sum())
    nA, kA, nB, kB = int(mA.sum()), int(s14_y[mA].sum()), int(mB.sum()), int(s14_y[mB].sum())
    par = []
    for i in cols:
        mp = S14_M[:, [c_ for c_ in cols if c_ != i]].all(axis=1)
        par.append(s14_y[mp].mean())
    lo, hi = wilson_ci(k, n); loB, hiB = wilson_ci(kB, max(nB, 1))
    return {"key": key, "conditions": " AND ".join(s14_item_lab[i] for i in cols), "n_conditions": len(cols), "n": n, "defaults": k, "rate": k / n,
            "ci_lo": float(lo), "ci_hi": float(hi), "lift": k / n / BASE_RATE, "wracc": n / S14_N * (k / n - BASE_RATE),
            "nA": nA, "rate_A": kA / max(nA, 1), "nB": nB, "rate_B": kB / max(nB, 1), "ci_lo_B": float(loB), "ci_hi_B": float(hiB),
            "shrink_pp": 100 * (kB / max(nB, 1) - kA / max(nA, 1)), "parent_rate_max": float(max(par)), "parent_rate_min": float(min(par)),
            "risky_conditions": "; ".join(f"{s14_item_lab[i]} ({s14_item_rate[i]:.0%} alone)" for i in cols if i in s14_risky_items)}

deep = s14_beams[s14_beams["depth"] >= 4]
_ev = {key: s14_eval_key(key) for key in deep["key"].unique()}
s14_pk = deep.merge(pd.DataFrame(list(_ev.values())), on="key", how="left")
s14_pk = s14_pk[(s14_pk["n"] >= 150) & (s14_pk["nB"] >= 50)].reset_index(drop=True)
s14_pk["gain_vs_best_parent_pp"] = np.where(s14_pk["beam"].isin(["high lift", "high WRAcc"]), 100 * (s14_pk["rate"] - s14_pk["parent_rate_max"]),
                                            100 * (s14_pk["rate"] - s14_pk["parent_rate_min"]))
s14_pk["replicated"] = np.where(s14_pk["beam"].isin(["high lift", "high WRAcc"]), s14_pk["ci_lo_B"] > BASE_RATE, s14_pk["ci_hi_B"] < BASE_RATE)

def s14_select(cands, k=15, max_jacc=0.6):
    chosen, masks = [], []
    for _, r in cands.iterrows():
        if any(r["key"] == c_["key"] for c_ in chosen): continue
        m = S14_M[:, list(r["key"])].all(axis=1)
        if any((m & mm).sum() / max((m | mm).sum(), 1) > max_jacc for mm in masks): continue
        chosen.append(r); masks.append(m)
        if len(chosen) == k: break
    return pd.DataFrame(chosen).reset_index(drop=True)

# high-risk shortlist: interleave the lift ranking and the WRAcc ranking (both on half A), each condition must add >= 1pp
hi_l = s14_pk[(s14_pk["beam"] == "high lift") & (s14_pk["gain_vs_best_parent_pp"] >= 1)].sort_values("score_A", ascending=False)
hi_w = s14_pk[(s14_pk["beam"] == "high WRAcc") & (s14_pk["gain_vs_best_parent_pp"] >= 1)].sort_values("score_A", ascending=False)
s14_hi = pd.concat([s14_select(hi_l, 9).assign(source="lift beam"), s14_select(hi_w, 9).assign(source="WRAcc beam")], ignore_index=True)
s14_hi = s14_select(s14_hi.sort_values("rate_A", ascending=False), 15)
sf = s14_pk[(s14_pk["beam"] == "surprisingly safe") & (s14_pk["gain_vs_best_parent_pp"] <= -0.5) & (s14_pk["rate_A"] < S14_BASE_A)].sort_values("score_A", ascending=False)
s14_safe = s14_select(sf, 15).assign(source="surprisingly-safe beam")
s14_safest = s14_select(s14_pk[s14_pk["beam"] == "safest"].sort_values("score_A", ascending=False), 5).assign(source="safest beam")

S14_PCOLS = ["conditions", "n_conditions", "n", "rate", "ci_lo", "ci_hi", "lift", "wracc", "rate_A", "rate_B", "ci_lo_B", "ci_hi_B", "shrink_pp",
             "gain_vs_best_parent_pp", "replicated", "source"]
pd.set_option("display.max_colwidth", 200)
print(f"TOP 15 HIGH-RISK POCKETS (4-5 conditions, discovered on half A, verified on half B; base rate {BASE_RATE:.1%})")
display(s14_hi[S14_PCOLS].round(3))
print("TOP 15 SURPRISINGLY SAFE POCKETS (contain at least one condition that is risky on its own)")
display(s14_safe[S14_PCOLS + ["risky_conditions"]].round(3))
print("For reference: the 5 safest pockets overall")
display(s14_safest[S14_PCOLS].round(3))

s14_pockets_out = pd.concat([s14_hi.assign(pocket_type="high risk"), s14_safe.assign(pocket_type="surprisingly safe"),
                             s14_safest.assign(pocket_type="safest")], ignore_index=True)
s14_pockets_out["base_rate"] = BASE_RATE
export(s14_pockets_out.drop(columns=["key", "beam", "depth", "nA_x", "kA", "score_A"], errors="ignore"), "s14_replicated_pockets")

**Does discovery survive verification?** Left: every 4–5-condition pocket the beams recorded, rate on half A (where it was found) vs rate on half B (never seen by the search). Points off the diagonal toward the base rate = winner's curse (pockets picked by a search look more extreme than they really are). Right: the shortlisted pockets, A → B, with the 95% interval on B.

In [ ]:
s14_pku = s14_pk.drop_duplicates("key")
fig, axes = plt.subplots(1, 2, figsize=(19, 8), gridspec_kw={"width_ratios": [2, 3]})
ax = axes[0]
bcol = {"high lift": C_DEF, "high WRAcc": "#fd8d3c", "safest": C_OK, "surprisingly safe": C_ACC}
for b_, g in s14_pk.groupby("beam"):
    ax.scatter(g["rate_A"], g["rate_B"], s=np.clip(g["n"] / 25, 6, 60), alpha=.35, color=bcol[b_], label=f"{b_} (n={len(g)})")
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.axhline(BASE_RATE, color=C_NEU, lw=.8); ax.axvline(BASE_RATE, color=C_NEU, lw=.8)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("default rate on half A (discovery)"); ax.set_ylabel("default rate on half B (verification)")
ax.set_title("Split-half replication of all recorded 4-5 condition pockets"); ax.legend(fontsize=8)
ax = axes[1]
both = pd.concat([s14_hi.assign(t="high risk"), s14_safe.assign(t="surprisingly safe")], ignore_index=True)
yy = np.arange(len(both))[::-1]
for y_, (_, r) in zip(yy, both.iterrows()):
    col = C_DEF if r["t"] == "high risk" else C_ACC
    ax.plot([r["ci_lo_B"], r["ci_hi_B"]], [y_, y_], color=col, lw=2, alpha=.6)
    ax.scatter(r["rate_A"], y_, marker="o", facecolor="white", edgecolor=col, s=45, zorder=3)
    ax.scatter(r["rate_B"], y_, marker="o", color=col, s=45, zorder=3)
    ax.text(1.01, y_, ("ok" if r["replicated"] else "NOT replicated") + f"  n={int(r['n'])}", va="center", fontsize=7, transform=ax.get_yaxis_transform())
ax.axvline(BASE_RATE, color="k", ls="--", lw=1)
ax.set_yticks(yy); ax.set_yticklabels([(c_[:95] + "...") if len(c_) > 98 else c_ for c_ in both["conditions"]], fontsize=6.5)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("default rate: hollow = half A, solid = half B with 95% CI")
ax.set_title("Shortlisted pockets: discovery vs verification")
plt.tight_layout(); savefig("s14_pocket_replication"); plt.show()

rep = []
for b_, g in s14_pku.groupby("beam"):
    rep.append({"beam": b_, "pockets": len(g), "corr_A_B": float(np.corrcoef(g["rate_A"], g["rate_B"])[0, 1]) if len(g) > 2 else np.nan,
                "mean_rate_A": g["rate_A"].mean(), "mean_rate_B": g["rate_B"].mean(), "mean_shrink_pp": g["shrink_pp"].mean(),
                "share_replicated": g["replicated"].mean()})
s14_rep = pd.DataFrame(rep); display(s14_rep.round(3))
print(f"shortlist: high-risk replicated {int(s14_hi['replicated'].sum())}/{len(s14_hi)}, mean shrink {s14_hi['shrink_pp'].mean():+.1f}pp | "
      f"surprisingly safe replicated {int(s14_safe['replicated'].sum())}/{len(s14_safe)}, mean shrink {s14_safe['shrink_pp'].mean():+.1f}pp")

if len(s14_hi):
    _h = s14_hi.sort_values("rate_B", ascending=False).iloc[0]
    add_fact("14.5", "pockets",
             f"Deep pockets replicate: {int(s14_hi['replicated'].sum())} of {len(s14_hi)} high-risk 4-5 condition pockets found on a random half keep a default rate significantly above base on the "
             f"other half (mean shrinkage {s14_hi['shrink_pp'].mean():+.1f}pp). Worst verified pocket: [{_h['conditions']}] with {_h['rate_B']:.0%} on the hold-out half (n={int(_h['n'])} overall, {_h['rate']:.0%})",
             float(_h["rate_B"]), "s14_pocket_replication", 3)
if len(s14_safe):
    _s = s14_safe[s14_safe["replicated"]].sort_values("rate_B").head(1)
    _s = _s.iloc[0] if len(_s) else s14_safe.sort_values("rate_B").iloc[0]
    add_fact("14.5", "pockets",
             f"Surprisingly safe pocket: [{_s['conditions']}] defaults at {_s['rate']:.1%} (hold-out half {_s['rate_B']:.1%}, n={int(_s['n'])}) although it contains {_s['risky_conditions']}; "
             f"{int(s14_safe['replicated'].sum())} of {len(s14_safe)} such pockets replicate below the base rate", float(_s["rate"]), "s14_pocket_replication", 2)

**Observations:**
- Beam search (keep the best few candidates at each step, then add one more condition) found 4-5 condition pockets on one random half; all 15 shortlisted high-risk and all 15 surprisingly-safe pockets held up on the other half.
- The worst pockets default at about 80%, 3.5-3.7 times the base rate. Example: Late 2m in Sep + pays 5-10% of bills + low spend + married + never over limit = 82.7% (n=173), 82% on the hold-out half (number from the pre-fix run; refresh after the next Kaggle run).
- Surprisingly safe: never-late clients with 1-2 zero-payment months plus a high limit or high spend default at 5-10% (e.g. 6.7%, n=210), although 1-2 zero-payment months alone means 31%.
- The safest pockets (never late, utilisation 10-30%, high spend) default at 1-4% (n=194-440).
- Safe pockets drift a little toward the base on the hold-out half (8.2% -> 11.0% on average); high-risk pockets do not (76.9% -> 76.1%).
- Story use: pocket cards (`s14_replicated_pockets.csv`).

### 14.6 Dominance: when two signals disagree, which one should we believe?
Each top driver is reduced to a binary risk flag. For a pair (A, B) the 2×2 table has two **agreement** cells (both safe, both risky) and two **disagreement** cells (only A risky, only B risky). The signal whose "risky alone" cell has the higher default rate **wins** the disagreement; the margin (pp, with a two-proportion test) says by how much. Twelve pairs, including the three asked for: clean Sep but bad history vs bad Sep but clean history; high utilisation but strong payer; low limit but never late.

In [ ]:
S14_SIG = {
    "SEP_LATE":   (df["PAY_1"].to_numpy() >= 1,                "late in Sep",                  "clean in Sep"),
    "HIST_LATE":  (df["CI_HIST_NLATE"].to_numpy() >= 1,        "late >=1x in Apr-Aug",         "clean Apr-Aug"),
    "AUG_LATE":   (df["PAY_2"].to_numpy() >= 1,                "late in Aug",                  "clean in Aug"),
    "EVER_LATE":  (df["N_LATE"].to_numpy() >= 1,               "late at least once (6m)",      "never late (6m)"),
    "HIGH_UTIL":  (df["AVG_UTIL"].to_numpy() > 0.7,            "avg utilisation > 70%",        "avg utilisation <= 70%"),
    "LOW_PAYER":  ((df["PAY_TO_BILL_5M"] < 0.10).to_numpy(),   "repays < 10% of bills (5 statements, aligned)", "repays >= 10% (or no bill)"),
    "LOW_LIMIT":  (df["LIMIT_BAL"].to_numpy() <= 50_000,       "limit <= 50k",                 "limit > 50k"),
    "ZERO_PAY":   (df["N_ZERO_PAY"].to_numpy() >= 2,           ">= 2 zero-payment months",     "< 2 zero-payment months"),
    "YOUNG":      (df["AGE"].to_numpy() <= 25,                 "age <= 25",                    "age > 25"),
    "OVERLIMIT":  (df["OVERLIMIT_ANY"].to_numpy() == 1,        "over limit at least once",     "never over limit"),
}
S14_SIG_NICE = {"SEP_LATE": "Sep status", "HIST_LATE": "Apr-Aug history", "AUG_LATE": "Aug status", "EVER_LATE": "any lateness", "HIGH_UTIL": "utilisation",
                "LOW_PAYER": "repayment share", "LOW_LIMIT": "credit limit", "ZERO_PAY": "zero-payment months", "YOUNG": "age", "OVERLIMIT": "over-limit"}
S14_SIG_PAIRS = [("SEP_LATE", "HIST_LATE"), ("SEP_LATE", "AUG_LATE"), ("SEP_LATE", "HIGH_UTIL"), ("SEP_LATE", "LOW_PAYER"), ("SEP_LATE", "LOW_LIMIT"),
                 ("EVER_LATE", "LOW_LIMIT"), ("HIGH_UTIL", "LOW_PAYER"), ("HIGH_UTIL", "LOW_LIMIT"), ("HIST_LATE", "HIGH_UTIL"), ("HIST_LATE", "ZERO_PAY"),
                 ("LOW_LIMIT", "LOW_PAYER"), ("LOW_LIMIT", "YOUNG"), ("HIGH_UTIL", "OVERLIMIT")]

print("solo rates of each risk flag")
display(pd.DataFrame([{"signal": s, "risky_state": v[1], "n_risky": int(v[0].sum()), "share_risky": v[0].mean(), "rate_risky": s14_y[v[0]].mean(),
                       "rate_safe": s14_y[~v[0]].mean(), "gap_pp": 100 * (s14_y[v[0]].mean() - s14_y[~v[0]].mean())} for s, v in S14_SIG.items()]).round(3))

cell_rows, sum_rows = [], []
for a, b in S14_SIG_PAIRS:
    fa, fb = S14_SIG[a][0], S14_SIG[b][0]
    cells = {}
    for sa in (0, 1):
        for sb in (0, 1):
            m = (fa == bool(sa)) & (fb == bool(sb))
            n, k = int(m.sum()), int(s14_y[m].sum())
            lo, hi = wilson_ci(k, max(n, 1))
            cells[(sa, sb)] = (n, k)
            cell_rows.append({"pair": f"{S14_SIG_NICE[a]} vs {S14_SIG_NICE[b]}", "signal_a": a, "signal_b": b, "a_state": S14_SIG[a][1 if sa else 2],
                              "b_state": S14_SIG[b][1 if sb else 2], "a_risky": sa, "b_risky": sb,
                              "cell": ["both safe", "only B risky", "only A risky", "both risky"][2 * sa + sb], "n": n, "defaults": k,
                              "rate": k / max(n, 1), "ci_lo": float(lo), "ci_hi": float(hi), "share_clients": n / S14_N})
    (nA_, kA_), (nB_, kB_) = cells[(1, 0)], cells[(0, 1)]
    rA, rB = kA_ / max(nA_, 1), kB_ / max(nB_, 1)
    pp = (kA_ + kB_) / max(nA_ + nB_, 1)
    z = (rA - rB) / np.sqrt(max(pp * (1 - pp) * (1 / max(nA_, 1) + 1 / max(nB_, 1)), 1e-12))
    win = a if rA > rB else b
    sum_rows.append({"pair": f"{S14_SIG_NICE[a]} vs {S14_SIG_NICE[b]}", "signal_a": a, "signal_b": b,
                     "disagreement_a": f"{S14_SIG[a][1]} BUT {S14_SIG[b][2]}", "n_only_a": nA_, "rate_only_a": rA,
                     "disagreement_b": f"{S14_SIG[b][1]} BUT {S14_SIG[a][2]}", "n_only_b": nB_, "rate_only_b": rB,
                     "rate_both_safe": cells[(0, 0)][1] / max(cells[(0, 0)][0], 1), "rate_both_risky": cells[(1, 1)][1] / max(cells[(1, 1)][0], 1),
                     "n_both_risky": cells[(1, 1)][0], "disagree_share": (nA_ + nB_) / S14_N, "margin_pp": 100 * abs(rA - rB), "z": float(z),
                     "p_value": float(2 * stats.norm.sf(abs(z))), "believe": S14_SIG_NICE[win], "winner": win})
s14_dis = pd.DataFrame(cell_rows); s14_dom = pd.DataFrame(sum_rows)
export(s14_dis, "s14_disagreement_tables"); export(s14_dom, "s14_dominance_summary")
display(s14_dom.drop(columns=["signal_a", "signal_b", "winner"]).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8), gridspec_kw={"width_ratios": [4, 1.4]})
ax = axes[0]
tt = s14_dom.iloc[::-1].reset_index(drop=True)
for y_, r in tt.iterrows():
    ax.plot([r["rate_both_safe"], r["rate_both_risky"]], [y_, y_], color="#d9d9d9", lw=6, zorder=1, solid_capstyle="round")
    ax.scatter(r["rate_both_safe"], y_, color=C_OK, s=70, zorder=3, marker="s")
    ax.scatter(r["rate_both_risky"], y_, color=C_DEF, s=70, zorder=3, marker="s")
    ax.scatter(r["rate_only_a"], y_, color="#54278f", s=110, zorder=4, marker=">")
    ax.scatter(r["rate_only_b"], y_, color="#fd8d3c", s=110, zorder=4, marker="<")
    sig = "" if r["p_value"] < 0.05 else " (n.s.)"
    ax.text(1.01, y_, f"believe {r['believe']}  (+{r['margin_pp']:.1f}pp{sig})", transform=ax.get_yaxis_transform(), va="center", fontsize=9, fontweight="bold")
    ax.text(r["rate_only_a"], y_ + 0.27, f"{r['rate_only_a']:.0%} n={r['n_only_a']:,}", ha="center", fontsize=7, color="#54278f")
    ax.text(r["rate_only_b"], y_ - 0.38, f"{r['rate_only_b']:.0%} n={r['n_only_b']:,}", ha="center", fontsize=7, color="#d95f0e")
ax.axvline(BASE_RATE, color="k", ls="--", lw=1)
ax.set_yticks(range(len(tt))); ax.set_yticklabels([f"A: {S14_SIG[a][1]}\nB: {S14_SIG[b][1]}" for a, b in zip(tt["signal_a"], tt["signal_b"])], fontsize=8)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("default rate")
ax.scatter([], [], color=C_OK, marker="s", label="both safe"); ax.scatter([], [], color="#54278f", marker=">", label="only A risky")
ax.scatter([], [], color="#fd8d3c", marker="<", label="only B risky"); ax.scatter([], [], color=C_DEF, marker="s", label="both risky")
ax.legend(loc="lower right", fontsize=8, ncol=4); ax.set_title("When signals conflict, believe ...")
ax = axes[1]
lg = pd.DataFrame({"appearances": pd.concat([s14_dom["signal_a"], s14_dom["signal_b"]]).value_counts(), "wins": s14_dom["winner"].value_counts()}).fillna(0)
lg["win_rate"] = lg["wins"] / lg["appearances"]; lg = lg.sort_values(["win_rate", "appearances"])
ax.barh([S14_SIG_NICE[i] for i in lg.index], lg["win_rate"], color=C_ACC)
for y_, (w, n_) in enumerate(zip(lg["wins"], lg["appearances"])): ax.text(0.02, y_, f"{int(w)}/{int(n_)}", va="center", fontsize=9)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title("Disagreements won"); ax.set_xlim(0, 1.05)
plt.subplots_adjust(wspace=0.75); savefig("s14_dominance"); plt.show()

_d = s14_dom.set_index(["signal_a", "signal_b"])
_r1 = _d.loc[("SEP_LATE", "HIST_LATE")]; _r2 = _d.loc[("HIGH_UTIL", "LOW_PAYER")]; _r3 = _d.loc[("EVER_LATE", "LOW_LIMIT")]
add_fact("14.6", "dominance",
         f"When recency and history disagree: late in Sep with a clean Apr-Aug record defaults at {_r1['rate_only_a']:.1%} (n={int(_r1['n_only_a']):,}) vs {_r1['rate_only_b']:.1%} "
         f"(n={int(_r1['n_only_b']):,}) for clean in Sep with earlier lateness; believe {_r1['believe']} (margin {_r1['margin_pp']:.1f}pp)",
         float(_r1["margin_pp"]), "s14_dominance", 3)
add_fact("14.6", "dominance",
         f"High utilisation with a healthy repayment share defaults at {_r2['rate_only_a']:.1%} vs {_r2['rate_only_b']:.1%} for low utilisation with a weak repayment share (believe {_r2['believe']}); "
         f"low limit but never late defaults at {_r3['rate_only_b']:.1%} vs {_r3['rate_only_a']:.1%} for a higher limit with some lateness (believe {_r3['believe']})",
         float(_r2["margin_pp"]), "s14_dominance", 2)

**Observations:**
- When September status and Apr-Aug history disagree, believe September: late in Sep with a clean history = 36.0% (n=2,369); clean in Sep but late before = 26.9% (n=3,251).
- Lateness beats balance signals every time: late in Sep with utilisation <= 70% = 46.3%, vs 17.1% for high utilisation but clean in Sep.
- A low limit without any lateness is fairly safe (16.7%, n=3,990); some lateness with a higher limit is 39.6%.
- Money signals tie with each other: high utilisation with a healthy repayment share = 25.2% vs low utilisation with a weak repayment share = 24.1% (p 0.50) (number from the pre-fix run; refresh after the next Kaggle run).
- Story use: "when signals conflict" sheet. Order of trust: lateness, then balances, then demographics.

### 14.7 Mediation-style decomposition: how much of each demographic gap travels *through* limit, utilisation and status?
Purely descriptive, no regression. For a gap between two groups (e.g. male vs female) we **directly standardise** both groups (re-weight them to the same mix) to the pooled two-group distribution of a set of mediators and re-measure the gap:

`raw gap → standardised for credit limit (7 bands) → + utilisation (4 bands) → + Sep status (6 groups)`.

What disappears at each step "passes through" that mediator; what is left is the direct (unexplained) gap. Cells present in only one group are dropped (coverage is reported). Because the order of adjustment matters, the table also gives the **Shapley** (order-averaged) contribution of each mediator. Intervals: 150 bootstrap resamples. *Not causal*: limit, utilisation and status are themselves outcomes of unobserved income and the bank's own decisions.

In [ ]:
S14_MED = {"limit": "LIMIT_BAND", "util": "CI_UTIL4", "status": "STATUS_GRP_SEP"}
S14_MED_NICE = {"limit": "credit limit", "util": "utilisation", "status": "Sep status"}
S14_GAPS = [("Sex: Male - Female", "SEX_LABEL", "Male", "Female"),
            ("Education: High school - Graduate school", "EDU_LABEL", "High school", "Graduate school"),
            ("Marriage: Married - Single", "MAR_LABEL", "Married", "Single"),
            ("Age: 21-25 - 31-35", "AGE_BAND", "21-25", "31-35"),
            ("Age: 51-60 - 31-35", "AGE_BAND", "51-60", "31-35")]
_mn = list(S14_MED)
S14_SUBSETS = [s for r in range(len(_mn) + 1) for s in itertools.combinations(_mn, r)]
s14_sub_code = {}
for sub in S14_SUBSETS:
    if not sub:
        s14_sub_code[sub] = (np.zeros(S14_N, int), 1)
    else:
        cd = [S14_BC[S14_MED[m]][0] for m in sub]; nl = [len(S14_BC[S14_MED[m]][1]) for m in sub]
        s14_sub_code[sub] = (np.ravel_multi_index(cd, nl), int(np.prod(nl)))

def s14_std_gap(ga, yv, code, K):
    # direct standardisation to the pooled two-group mediator distribution, common support only
    na = np.bincount(code[ga], minlength=K).astype(float); nb = np.bincount(code[~ga], minlength=K).astype(float)
    ka = np.bincount(code[ga], weights=yv[ga], minlength=K); kb = np.bincount(code[~ga], weights=yv[~ga], minlength=K)
    sup = (na > 0) & (nb > 0)
    w = (na + nb)[sup]; cov = w.sum() / len(yv); w = w / w.sum()
    ra, rb = float((w * ka[sup] / na[sup]).sum()), float((w * kb[sup] / nb[sup]).sum())
    return ra - rb, ra, rb, cov

def s14_decompose(ga, yv, codes):
    g = {sub: s14_std_gap(ga, yv, codes[sub][0], codes[sub][1]) for sub in S14_SUBSETS}
    gap = {sub: v[0] for sub, v in g.items()}
    seq = {"raw": gap[()], "via_limit": gap[()] - gap[("limit",)], "via_util": gap[("limit",)] - gap[("limit", "util")],
           "via_status": gap[("limit", "util")] - gap[("limit", "util", "status")], "remaining": gap[("limit", "util", "status")]}
    shap = {}
    for m in _mn:
        others = [o for o in _mn if o != m]; tot = 0.0
        for r in range(len(others) + 1):
            for sub in itertools.combinations(others, r):
                wgt = math.factorial(r) * math.factorial(len(_mn) - r - 1) / math.factorial(len(_mn))
                with_m = tuple(x for x in _mn if x in sub or x == m)
                tot += wgt * (gap[sub] - gap[with_m])
        shap[f"via_{m}"] = tot
    return seq, shap, g

_rng = np.random.default_rng(1407)
rows, s14_med_detail = [], []
for name, col, a, b in S14_GAPS:
    sel = df[col].isin([a, b]).to_numpy()
    ga = (df.loc[sel, col] == a).to_numpy(); yv = s14_y[sel].astype(float)
    codes = {sub: (s14_sub_code[sub][0][sel], s14_sub_code[sub][1]) for sub in S14_SUBSETS}
    seq, shap, g = s14_decompose(ga, yv, codes)
    boot = []
    for _ in range(150):
        ii = _rng.integers(0, len(yv), len(yv))
        sq, shp, _g = s14_decompose(ga[ii], yv[ii], {sub: (codes[sub][0][ii], codes[sub][1]) for sub in S14_SUBSETS})
        boot.append({**sq, **{f"shap_{k_}": v_ for k_, v_ in shp.items()}})
    boot = pd.DataFrame(boot)
    for comp in ["raw", "via_limit", "via_util", "via_status", "remaining"]:
        rows.append({"gap": name, "group_a": a, "group_b": b, "n_a": int(ga.sum()), "n_b": int((~ga).sum()), "component": comp,
                     "sequential_pp": 100 * seq[comp], "seq_lo_pp": 100 * boot[comp].quantile(.025), "seq_hi_pp": 100 * boot[comp].quantile(.975),
                     "shapley_pp": 100 * shap[comp] if comp in shap else 100 * seq[comp],
                     "share_of_raw_gap_seq": seq[comp] / seq["raw"] if abs(seq["raw"]) > 1e-9 else np.nan,
                     "share_of_raw_gap_shapley": (shap[comp] if comp in shap else seq[comp]) / seq["raw"] if abs(seq["raw"]) > 1e-9 else np.nan})
    for sub in S14_SUBSETS:
        s14_med_detail.append({"gap": name, "standardised_for": " + ".join(S14_MED_NICE[m] for m in sub) or "(nothing: raw)", "rate_a": g[sub][1], "rate_b": g[sub][2],
                               "gap_pp": 100 * g[sub][0], "common_support_coverage": g[sub][3]})
s14_med = pd.DataFrame(rows); s14_med_detail = pd.DataFrame(s14_med_detail)
export(s14_med, "s14_gap_mediation")
display(s14_med.pivot(index="gap", columns="component", values="sequential_pp").reindex([g_[0] for g_ in S14_GAPS])[["raw", "via_limit", "via_util", "via_status", "remaining"]].round(2))
print("Shapley (order-averaged) contributions, pp")
display(s14_med.pivot(index="gap", columns="component", values="shapley_pp").reindex([g_[0] for g_ in S14_GAPS])[["raw", "via_limit", "via_util", "via_status", "remaining"]].round(2))
print("every adjustment set (gap after standardising for ...)"); display(s14_med_detail.round(4))

In [ ]:
fig, axes = plt.subplots(1, len(S14_GAPS), figsize=(4.6 * len(S14_GAPS), 5.6))
for ax, (name, col, a, b) in zip(axes, S14_GAPS):
    t = s14_med[s14_med["gap"] == name].set_index("component")
    raw = t.loc["raw", "sequential_pp"]; rem = t.loc["remaining", "sequential_pp"]
    ax.bar(0, raw, color=C_NEU); ax.text(0, raw, f"{raw:+.2f}", ha="center", va="bottom" if raw >= 0 else "top", fontsize=9, fontweight="bold")
    cur = raw
    for i, comp in enumerate(["via_limit", "via_util", "via_status"], start=1):
        d = t.loc[comp, "sequential_pp"]
        # a positive contribution removes that much of the gap
        shrink = (d > 0) == (raw > 0)
        ax.bar(i, -d, bottom=cur, color=C_ACC if shrink else C_DEF)
        ax.plot([i - 1.4, i + .4], [cur, cur], color="k", lw=.6)
        ax.text(i, cur - d, f"{-d:+.2f}", ha="center", va="top" if d > 0 else "bottom", fontsize=9)
        cur -= d
    ax.bar(4, rem, color="#54278f")
    ax.errorbar(4, rem, yerr=[[rem - t.loc["remaining", "seq_lo_pp"]], [t.loc["remaining", "seq_hi_pp"] - rem]], color="k", capsize=4)
    ax.text(4, rem, f" {rem:+.2f}", ha="left", va="bottom" if rem >= 0 else "top", fontsize=9, fontweight="bold")
    ax.axhline(0, color="k", lw=1)
    ax.set_xticks(range(5)); ax.set_xticklabels(["raw gap", "via\nlimit", "via\nutilisation", "via\nSep status", "remaining\n(direct)"], fontsize=8)
    ax.set_title(name, fontsize=10); ax.set_ylabel("default-rate gap (pp)")
fig.suptitle("Sequential standardisation waterfalls: green = step shrinks the gap, red = step widens it, purple = left after all three (95% bootstrap CI)", fontweight="bold", y=1.02)
plt.tight_layout(); savefig("s14_gap_mediation"); plt.show()

_m = s14_med.set_index(["gap", "component"])
_g1 = S14_GAPS[1][0]; _g0 = S14_GAPS[0][0]
add_fact("14.7", "demographics",
         f"Education gap (high school minus graduate school) is {_m.loc[(_g1, 'raw'), 'sequential_pp']:+.1f}pp raw; {_m.loc[(_g1, 'via_limit'), 'share_of_raw_gap_seq']:.0%} passes through the credit limit, "
         f"{_m.loc[(_g1, 'via_util'), 'share_of_raw_gap_seq']:.0%} through utilisation, {_m.loc[(_g1, 'via_status'), 'share_of_raw_gap_seq']:.0%} through Sep status, leaving {_m.loc[(_g1, 'remaining'), 'sequential_pp']:+.1f}pp direct",
         float(_m.loc[(_g1, "remaining"), "sequential_pp"]), "s14_gap_mediation", 3)
add_fact("14.7", "demographics",
         f"Sex gap (male minus female) is {_m.loc[(_g0, 'raw'), 'sequential_pp']:+.1f}pp raw and {_m.loc[(_g0, 'remaining'), 'sequential_pp']:+.1f}pp after standardising for limit, utilisation and Sep status "
         f"(95% CI {_m.loc[(_g0, 'remaining'), 'seq_lo_pp']:+.1f} to {_m.loc[(_g0, 'remaining'), 'seq_hi_pp']:+.1f}); Sep status alone accounts for {_m.loc[(_g0, 'via_status'), 'share_of_raw_gap_shapley']:.0%} (Shapley)",
         float(_m.loc[(_g0, "remaining"), "sequential_pp"]), "s14_gap_mediation", 2)

**Observations:**
- Standardising = re-weighting both groups to the same mix of a variable, then comparing default rates.
- The age gap (21-25 vs 31-35, +7.2 pp) disappears once credit limits are equalised (+0.27 pp): young clients default more mainly because they get small lines.
- The education gap (high school minus graduate school, +5.9 pp) drops to +1.3 pp after the limit and to +0.6 pp after limit, utilisation and status; 78% travels through the limit.
- The sex gap (+3.4 pp) only shrinks to +1.9 pp after all three; about half stays.
- The marriage gap (+2.5 pp) grows to +3.9 pp after the limit and stays +2.5 pp after all three: married clients have larger lines yet still default more.
- Story use: gap waterfalls (`s14_gap_mediation.csv`).

### 14.8 The credit line as a moderator of everything
Clients are split at the median limit. For every banded variable we compare the **lift pattern** of its levels in the low-limit half and in the high-limit half (lift is relative to each half's own base rate, so the main effect of the limit is removed). `log2(lift_high / lift_low)` > 0 means the level is relatively riskier among high-limit clients. Per variable: IV in each half, correlation of the two lift patterns, and the LR test of `variable × limit half` (G² of the additive model on the k × 2 table).

In [ ]:
S14_MODVARS = ["STATUS_GRP_SEP", "CI_STATUS_AUG4", "N_LATE_BAND", "CI_HIST_LATE3", "UTIL_BAND", "PAYRATIO_BAND", "CI_ZEROPAY3", "CI_UTILTREND3",
               "CI_SPEND3", "CI_OVERLIMIT", "AGE_BAND", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL"]
hc, hl = S14_BC["CI_LIMIT_HALF"]
lowm, highm = hc == 0, hc == 1
S14_BASE_LOW, S14_BASE_HIGH = float(s14_y[lowm].mean()), float(s14_y[highm].mean())
lvl_rows, var_rows = [], []
for v in S14_MODVARS:
    vc, vl = S14_BC[v]
    for j, lab in enumerate(vl):
        ml_, mh_ = lowm & (vc == j), highm & (vc == j)
        nl_, nh_ = int(ml_.sum()), int(mh_.sum())
        rl = s14_y[ml_].mean() if nl_ else np.nan; rh = s14_y[mh_].mean() if nh_ else np.nan
        lvl_rows.append({"variable": v, "variable_label": s14_nice(v), "level": lab, "level_order": j, "n_low": nl_, "rate_low": rl, "lift_low": rl / S14_BASE_LOW,
                         "n_high": nh_, "rate_high": rh, "lift_high": rh / S14_BASE_HIGH,
                         "log2_lift_ratio": np.log2((rh / S14_BASE_HIGH) / (rl / S14_BASE_LOW)) if nl_ >= 50 and nh_ >= 50 and rl > 0 and rh > 0 else np.nan})
    idx, n_c, k_c, nl2 = s14_cells([v, "CI_LIMIT_HALF"])
    d1, r1, _ = s14_fit(s14_design(idx, nl2, 1), k_c, n_c)
    dfi = len(n_c) - r1
    sc = np.zeros(S14_N)
    a_lo = s14_assoc(vc[lowm], s14_y[lowm], sc[lowm]); a_hi = s14_assoc(vc[highm], s14_y[highm], sc[highm])
    lv = pd.DataFrame(lvl_rows); lv = lv[(lv["variable"] == v) & lv["log2_lift_ratio"].notna()]
    var_rows.append({"variable": v, "variable_label": s14_nice(v), "levels": len(vl), "iv_low_limit": a_lo["iv_adj"], "iv_high_limit": a_hi["iv_adj"],
                     "iv_ratio_high_over_low": a_hi["iv_adj"] / a_lo["iv_adj"] if a_lo["iv_adj"] > 0.005 else np.nan,
                     "lift_pattern_corr": float(np.corrcoef(np.log(lv["lift_low"]), np.log(lv["lift_high"]))[0, 1]) if len(lv) > 2 else np.nan,
                     "max_abs_log2_lift_ratio": float(lv["log2_lift_ratio"].abs().max()) if len(lv) else np.nan,
                     "G2_interaction": d1, "df": dfi, "p_value": float(stats.chi2.sf(d1, dfi)) if dfi > 0 else np.nan})
s14_modl = pd.DataFrame(lvl_rows); s14_modv = pd.DataFrame(var_rows)
s14_modv["p_fdr"] = multipletests(s14_modv["p_value"].fillna(1), method="fdr_bh")[1]
s14_modv["G2_per_df"] = s14_modv["G2_interaction"] / s14_modv["df"]
s14_modv = s14_modv.sort_values("G2_per_df", ascending=False).reset_index(drop=True)
export(s14_modl.merge(s14_modv[["variable", "iv_low_limit", "iv_high_limit", "G2_interaction", "df", "p_fdr", "lift_pattern_corr"]], on="variable"), "s14_limit_moderation")
print(f"base rate: low-limit half {S14_BASE_LOW:.1%} (n={int(lowm.sum()):,}) | high-limit half {S14_BASE_HIGH:.1%} (n={int(highm.sum()):,})")
display(s14_modv.round(4))

In [ ]:
order = s14_modv["variable"].tolist()
maxl = int(s14_modl.groupby("variable")["level"].size().max())
Mx = np.full((len(order), maxl), np.nan); lab = np.full((len(order), maxl), "", dtype=object)
for i, v in enumerate(order):
    t = s14_modl[s14_modl["variable"] == v].sort_values("level_order")
    for j, (_, r) in enumerate(t.iterrows()):
        Mx[i, j] = r["log2_lift_ratio"]
        lab[i, j] = f"{str(r['level'])[:13]}\n{r['rate_low']:.0%} | {r['rate_high']:.0%}" if r["n_low"] and r["n_high"] else str(r["level"])[:13]
fig, axes = plt.subplots(1, 2, figsize=(21, 9), gridspec_kw={"width_ratios": [4, 1.3]})
ax = axes[0]
lim = float(np.nanmin([1.0, np.nanmax(np.abs(Mx))]))
im = ax.imshow(np.ma.masked_invalid(Mx), cmap=CMAP_DIV, vmin=-lim, vmax=lim, aspect="auto")
for i in range(Mx.shape[0]):
    for j in range(Mx.shape[1]):
        if lab[i, j]:
            ax.text(j, i, lab[i, j], ha="center", va="center", fontsize=6.8, color="k" if not np.isnan(Mx[i, j]) else C_NEU)
ax.set_yticks(range(len(order)))
ax.set_yticklabels([f"{s14_nice(v)}  (G2/df {g:.1f}{'*' if p < 0.05 else ''})" for v, g, p in zip(order, s14_modv["G2_per_df"], s14_modv["p_fdr"])], fontsize=8.5)
ax.set_xticks(range(maxl)); ax.set_xticklabels([f"level {j + 1}" for j in range(maxl)]); ax.grid(False)
plt.colorbar(im, ax=ax, label="log2(lift in high-limit half / lift in low-limit half)", fraction=0.03)
ax.set_title("Moderation matrix: each cell = level, default rate low-limit | high-limit half; colour = relative lift shift (* = interaction FDR < 5%)")
ax = axes[1]
yy = np.arange(len(order))[::-1]
ax.barh(yy + 0.2, s14_modv["iv_low_limit"], height=0.4, color=C_DEF, label="low-limit half")
ax.barh(yy - 0.2, s14_modv["iv_high_limit"], height=0.4, color=C_OK, label="high-limit half")
ax.set_yticks(yy); ax.set_yticklabels([s14_nice(v) for v in order], fontsize=8); ax.set_xlabel("bias-adjusted IV within the half"); ax.legend(fontsize=8)
ax.set_title("Signal strength by limit half")
plt.tight_layout(); savefig("s14_limit_moderation"); plt.show()

print("levels whose relative risk changes most between limit halves")
display(s14_modl.dropna(subset=["log2_lift_ratio"]).assign(a=lambda t: t["log2_lift_ratio"].abs()).sort_values("a", ascending=False).drop(columns="a").head(12).round(3))
_nsig = int((s14_modv["p_fdr"] < 0.05).sum()); _mv = s14_modv.iloc[0]
_lv = s14_modl[s14_modl["variable"] == _mv["variable"]].dropna(subset=["log2_lift_ratio"])
_lv = _lv.loc[_lv["log2_lift_ratio"].abs().idxmax()]
add_fact("14.8", "credit line as moderator",
         f"The credit limit moderates {_nsig} of {len(s14_modv)} banded variables (FDR 5%); the strongest is {_mv['variable_label']} (G2 {_mv['G2_interaction']:.0f} on {int(_mv['df'])} df): "
         f"level '{_lv['level']}' defaults at {_lv['rate_low']:.1%} in the low-limit half vs {_lv['rate_high']:.1%} in the high-limit half (lift {_lv['lift_low']:.2f} vs {_lv['lift_high']:.2f})",
         float(_mv["G2_per_df"]), "s14_limit_moderation", 2)
_st = s14_modv.set_index("variable").loc["STATUS_GRP_SEP"]
add_fact("14.8", "credit line as moderator",
         f"September status separates risk in both halves of the credit-limit distribution (IV {_st['iv_low_limit']:.2f} low-limit half vs {_st['iv_high_limit']:.2f} high-limit half; "
         f"lift patterns correlate {_st['lift_pattern_corr']:.2f})", float(_st["iv_high_limit"]), "s14_limit_moderation", 1)

**Observations:**
- Base rates: low-limit half 28.3% (n=15,290), high-limit half 15.7% (n=14,710). The limit changes the effect of 8 of 14 banded variables (FDR 5%) but none of the four demographics (p_fdr >= 0.19).
- Falling utilisation means opposite things: 34.5% default in the low-limit half vs 11.5% in the high-limit half.
- Zero or negative balances are much more alarming with a high limit: "No bill" 38.1% and utilisation <= 0 36.1%, about 2.3-2.4 times that half's base, vs about 1.1 times in the low-limit half.
- September status works in both halves (IV 0.86 vs 0.78); its relative lift is even larger in the high-limit half (Late 3m+: 5.0x vs 2.5x).
- Story use: "credit line as moderator" sheet; limit half works well as a Tableau filter.

### 14.9 Non-additivity summary: how much of the pattern is main effects, pairwise, or higher-order?
One number for the story. Take a rich four-way grid, e.g. `STATUS_GRP_SEP × N_LATE_BAND × utilisation (4) × limit (3)`, and fit nested logit models **on the cell table** (weights = cell n): null → main effects → + all two-way → + all three-way → saturated. The **analysis of deviance** (the logistic version of ANOVA) splits the total variation of cell default rates (on the logit scale) into layers.

Raw shares overstate higher-order terms because every extra parameter soaks up about one unit of deviance from pure sampling noise. The **noise-adjusted** share subtracts each layer's parameter count (its expected deviance under "no real effect"). Four grids are shown to check that the answer is not an artefact of one choice.

In [ ]:
S14_GRIDS = {"Behaviour grid: Sep status x late months x utilisation x limit": ["STATUS_GRP_SEP", "N_LATE_BAND", "CI_UTIL4", "CI_LIMIT3"],
             "Money grid: utilisation x paid/billed x zero-pay months x spend": ["CI_UTIL4", "CI_PAYR5", "CI_ZEROPAY3", "CI_SPEND3"],
             "Mixed grid: Sep status (4) x limit x age x education": ["CI_STATUS4", "CI_LIMIT3", "CI_AGE3", "EDU_LABEL"],
             "Demographic grid: sex x education x marriage x age": ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "CI_AGE3"]}
S14_LAYERS = ["Main effects", "Two-way", "Three-way", "Four-way"]
rows = []
for gname, cols in S14_GRIDS.items():
    idx, n_c, k_c, nl = s14_cells(cols)
    devs, ranks = [s14_null_dev(k_c, n_c)], [1]
    n_fail = 0
    for order in (1, 2, 3):
        d, r, _ = s14_fit(s14_design(idx, nl, order), k_c, n_c)
        # nested models cannot fit worse than the smaller model; a higher deviance means the optimiser failed
        # (complete separation in sparse cells), so we keep the smaller model's deviance and flag the grid
        if d > devs[-1] + 1e-6:
            n_fail += 1; d = devs[-1]
        devs.append(d); ranks.append(r)
    devs.append(0.0); ranks.append(len(n_c))
    ddev = -np.diff(devs); ddf = np.diff(ranks)
    adj = np.clip(ddev - ddf, 0, None)
    for i, layer in enumerate(S14_LAYERS):
        rows.append({"grid": gname, "variables": " x ".join(s14_nice(c_) for c_ in cols), "cells": len(n_c), "cells_n_lt_30": int((n_c < 30).sum()),
                     "layer": layer, "layer_order": i + 1, "deviance": float(ddev[i]), "df": int(ddf[i]), "p_value": float(stats.chi2.sf(ddev[i], ddf[i])) if ddf[i] > 0 else np.nan,
                     "share_raw": float(ddev[i] / devs[0]), "share_noise_adjusted": float(adj[i] / adj.sum()), "null_deviance": devs[0],
                     "fits_repaired": n_fail})
s14_nonadd = pd.DataFrame(rows)
export(s14_nonadd, "s14_nonadditivity")
display(s14_nonadd.drop(columns=["variables", "null_deviance"]).round(4))

fig, axes = plt.subplots(1, 2, figsize=(18, 5.2), sharey=True)
lcol = [C_OK, C_ACC, "#fd8d3c", C_DEF]
for ax, col, ttl in [(axes[0], "share_raw", "Raw share of cell-rate deviance"), (axes[1], "share_noise_adjusted", "Noise-adjusted share (layer deviance minus its df)")]:
    piv = s14_nonadd.pivot(index="grid", columns="layer", values=col).reindex(index=list(S14_GRIDS)[::-1], columns=S14_LAYERS)
    left = np.zeros(len(piv))
    for li, layer in enumerate(S14_LAYERS):
        ax.barh(range(len(piv)), piv[layer], left=left, color=lcol[li], label=layer)
        for y_, (v_, l_) in enumerate(zip(piv[layer], left)):
            if v_ >= 0.025: ax.text(l_ + v_ / 2, y_, f"{v_:.1%}", ha="center", va="center", fontsize=9, color="white", fontweight="bold")
        left += piv[layer].to_numpy()
    ax.set_yticks(range(len(piv))); ax.set_yticklabels([g.replace(": ", ":\n") for g in piv.index], fontsize=8.5)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlim(0, 1); ax.set_title(ttl)
axes[0].legend(ncol=4, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, -0.22))
plt.tight_layout(); savefig("s14_nonadditivity"); plt.show()

S14_FACT_GRID = list(S14_GRIDS)[2]          # Mixed grid: fits cleanly; the sparse behaviour/money grids need repaired fits
_g = s14_nonadd[s14_nonadd["grid"] == S14_FACT_GRID].set_index("layer")
print("grids whose nested fits needed repair:", s14_nonadd.groupby("grid")["fits_repaired"].first().to_dict())
add_fact("14.9", "non-additivity",
         f"On the {int(_g['cells'].iloc[0])}-cell grid Sep status x limit x age x education, {_g.loc['Main effects', 'share_raw']:.0%} of the variation in cell default rates is main effects, "
         f"{_g.loc['Two-way', 'share_raw']:.0%} pairwise and {_g.loc['Three-way', 'share_raw'] + _g.loc['Four-way', 'share_raw']:.0%} higher-order "
         f"(noise-adjusted: {_g.loc['Main effects', 'share_noise_adjusted']:.0%} / {_g.loc['Two-way', 'share_noise_adjusted']:.0%} / "
         f"{_g.loc['Three-way', 'share_noise_adjusted'] + _g.loc['Four-way', 'share_noise_adjusted']:.0%})",
         float(_g.loc["Main effects", "share_raw"]), "s14_nonadditivity", 3)

**Observations:**
- Main effects carry most of the pattern in every grid: 95% (Mixed), 91% (Behaviour), 87% (Money) and 80% (Demographic) of the variation in cell default rates.
- In the Mixed grid (Sep status x limit x age x education, 143 cells) two-way terms add 2.9% and three- and four-way terms add only 1.5% and 0.5% (p 0.31 and 0.97). After the noise correction, main effects hold 98%.
- The Behaviour and Money grids have many tiny cells (147 of 251 and 40 of 133 have fewer than 30 clients). Some of their bigger models could not be fitted (the optimiser failed), so the code keeps the smaller model's fit there. Their two-way and three-way layers then show 0%, and the leftover lands in the four-way layer (8.6% and 4.1%). Read those two bars as "main effects about 90%, rest not well measured".
- In the Demographic grid no interaction layer is significant (p 0.13-0.55).
- Story use: "risk mostly adds up": one stacked bar from the Mixed grid. The few true interactions are the pockets in 14.5.

### 14.10 Section wrap-up: facts for the story, new columns, exports

**New `df` columns added by this section (all prefixed `CI_`):**

| Column | Definition |
|---|---|
| `CI_SEP_CLEAN` | `Clean in Sep` (PAY_1 ≤ 0) / `Late in Sep` (PAY_1 ≥ 1) |
| `CI_STATUS4`, `CI_STATUS_AUG4` | Sep / Aug status collapsed to Inactive, Paid in full, Revolving, Late |
| `CI_HIST_NLATE`, `CI_HIST_LATE3` | number of late months in Apr–Aug (excludes Sep) and its band 0 / 1-2 / 3-5 |
| `CI_LIMIT3`, `CI_LIMIT_HALF` | credit limit ≤100k / 100-300k / >300k; below-or-at vs above the median limit |
| `CI_AGE3` | age 21-30 / 31-45 / 46+ |
| `CI_UTIL4` | average utilisation ≤10% / 10-50% / 50-90% / >90% |
| `CI_PAYR5` | 6-month paid/billed ≤5% / 5-25% / 25-100% / >100% / No bill |
| `CI_ZEROPAY3` | zero-payment months 0 / 1-2 / 3-6 |
| `CI_UTILTREND3` | utilisation trend Falling (< −2pp/month) / Flat / Rising (> +2pp/month) |
| `CI_SPEND3` | tercile of average implied spend |
| `CI_OVERLIMIT` | never over limit / over limit in at least one month |
| `CI_LOG_PAY_AMT1` | log(1 + PAY_AMT1) |

**Tableau sheets suggested by this section:** (1) highlight table variable × stratum from `s14_conditional_iv` with a stratifier parameter; (2) retention bar from `s14_iv_retention`; (3) small-multiple lines from `s14_effect_mod_curves` (driver and modifier parameters); (4) slope forest from `s14_effect_mod_slopes`; (5) faceted highlight tables from `s14_threeway_top_cells`; (6) pocket cards with discovery/verification rates from `s14_replicated_pockets`; (7) "when signals conflict" dumbbell from `s14_disagreement_tables` / `s14_dominance_summary`; (8) waterfalls from `s14_gap_mediation`; (9) the single stacked bar from `s14_nonadditivity`.

In [ ]:
s14_facts = pd.DataFrame([f for f in STORY_FACTS if f["section"].startswith("14")])
pd.set_option("display.max_colwidth", 400)
display(s14_facts)
pd.set_option("display.max_colwidth", 50)
print("new df columns present:", all(c_ in df.columns for c_ in S14_NEW_COLS), "| df shape", df.shape)
print(f"Section 14 runtime: {time.time() - S14_T0:.0f}s")

**Observations:**
- The section registers 18 facts. The strongest: lateness history is the best extra signal after September status; utilisation flips sign by status group; risk is mostly additive (three-way terms 0.8% of deviance); deep pockets replicate; recent lateness beats older history (36.0% vs 26.9%); most of the education gap runs through the credit limit.
- Fact 14.9 shows impossible shares (-6062% / 6070%) from the failed grid fit above; drop or fix it before Section 20 uses it.
- Story use: feeds the Section 20 synthesis.

## 15. Demographics in depth: lifecycle, credit-line assignment and gap decomposition

**Why this section matters for the story.** Round 1 showed that demographics are *weak* drivers of default (Sections 3.13, 8.4, 10.4). A dashboard viewer will still ask "but who are the risky people?". This section answers that properly: it shows **how** the demographic gaps arise (who gets which credit line, who behaves how), **decomposes** every headline gap into a composition part (different behaviour mix) and a remaining rate part, and delivers the Tableau story sheet *"demographics fade once behaviour is held equal"*.

Sections 3.13 (gaps within strata), 8.4 (raw 4-way cells) and 10.x (bootstrap, permutation, logit, CMH, Simpson) are **not repeated**; here we go deeper: lifecycle, the bank's line-assignment rule, Kitagawa / Oaxaca-Blinder / Fairlie decompositions, standardisation, a descriptive fairness lens and empirical-Bayes shrinkage of small cells.

| Sub-section | Question |
|---|---|
| 15.1 | Who is in the portfolio, and where are they in the life cycle? |
| 15.2 | How does the bank hand out credit lines, and is the line matched to realised risk? |
| 15.3 | What is the true shape of the age effect, and is it just behaviour? |
| 15.4 | Why do men default more than women? |
| 15.5 | Do the other demographic gaps decompose the same way? |
| 15.6 | Why does the Other/Unknown education group default so rarely? |
| 15.7 | Does the household story (marriage x sex x age) hold? |
| 15.8 | What is left of each demographic gap once behaviour is held equal? |
| 15.9 | Would a behaviour-only rule treat demographic groups evenly? |
| 15.10 | Which small demographic cells are really different, and which are noise? |

**Section guide**

- **What it does:** Explains *how* demographic gaps in default arise. It profiles the client base, studies how the bank sets credit lines by age, sex and education, and splits each gap (men vs women, married vs single, young vs prime age, education) into a part due to different behaviour mix and a part left over.
- **Question it answers:** "Who are the risky people?" - and how much of each demographic gap disappears once behaviour and credit line are held equal?
- **Inputs / new columns:** `SEX`, `EDUCATION`, `MARRIAGE`, `AGE`, `LIMIT_BAL`, `STATUS_GRP_SEP`, `LIMIT_BAND`, `N_LATE`, utilisation and payment ratios. Adds columns with prefix `DM_` (e.g. `DM_LIFE_STAGE`, `DM_LIMIT_RESID`, `DM_BEH_EXPECTED`, `DM_BEH_TIER`, `DM_CELL_RATE_EB`; full list in 15.11).
- **How to read the charts:** pyramid: bars = clients per age and sex, dark part = defaulters. Age curves: line = smoothed default rate, band = bootstrap range. Dumbbell: grey dot = raw rate, red dot = rate after adjusting for behaviour, dashed line = 22.1% base rate. Waterfalls and stacked bars split each gap into "explained" and "unexplained" parts.
- **Terms:** Kitagawa / Oaxaca-Blinder / Fairlie = methods that split a gap between two groups into a *composition* part (the groups behave differently) and a *rate* part (same behaviour, different default). Standardisation = re-weight groups to the same behaviour mix. Empirical-Bayes shrinkage = pull small, noisy cells toward the average.
- **Key takeaways:**
  - Demographic gaps are small and mostly behaviour: holding status and limit equal removes 56% of the age spread, 45% of the sex gap and 44% of the education gap, but none of the marriage gap.
  - Age is U-shaped (19.8% at 31 vs 29.0% at 22 and 27.2% at 60); behaviour mix reproduces 66% of it, because young clients get small lines and use more of them.
  - The bank's private judgement in the credit line is the real signal: clients far below their demographic peers' line default at 32.4% vs 14.5% far above.
  - Men default 3.4 pp more than women; about half is behaviour mix and 1.5-1.9 pp remains. Married clients default 2.5-3.5 pp more than singles even at the same behaviour.
  - Other/Unknown education (468 clients) defaults at 7.1% although its behaviour predicts 17.5%: a small, distinct low-risk segment.
- **Where it feeds:** the story sheet "demographics fade once behaviour is held equal" (`s15_crude_vs_adjusted.csv`), gap bars (`s15_gap_decomposition.csv`), age U-shape curves and the risk-vs-line bubble map.

### 15.0 Section helpers
OLS wrapper (statsmodels, HC1 robust errors, i.e. standard errors that stay valid when noise is uneven across clients; numpy fallback), a weighted local-linear smoother for single-year age curves, and the shared covariate blocks used by every regression and decomposition below. All globals carry the `S15_` / `s15_` prefix; new `df` columns carry `DM_`.

In [ ]:
try:
    import statsmodels.api as sm
    S15_HAS_SM = True
except Exception:
    S15_HAS_SM = False
from scipy.special import expit as s15_expit
S15_RNG = np.random.default_rng(1515)
S15_T0 = time.time()

def s15_ols(X, y, cov="HC1"):
    # OLS with robust (HC1) standard errors. Returns (coef table, R2, adjusted R2).
    X = pd.DataFrame(X).astype(float)
    y = np.asarray(y, float)
    names = ["const"] + [str(v) for v in X.columns]
    if S15_HAS_SM:
        m = sm.OLS(y, sm.add_constant(X, has_constant="add")).fit(cov_type=cov)
        return pd.DataFrame({"coef": np.asarray(m.params), "se": np.asarray(m.bse)}, index=names), float(m.rsquared), float(m.rsquared_adj)
    A = np.column_stack([np.ones(len(X)), X.to_numpy()])
    beta = np.linalg.lstsq(A, y, rcond=None)[0]
    res = y - A @ beta
    bread = np.linalg.pinv(A.T @ A)
    V = bread @ ((A * (res ** 2)[:, None]).T @ A) @ bread * len(y) / (len(y) - A.shape[1])
    r2 = 1 - (res ** 2).sum() / ((y - y.mean()) ** 2).sum()
    return pd.DataFrame({"coef": beta, "se": np.sqrt(np.diag(V))}, index=names), float(r2), float(1 - (1 - r2) * (len(y) - 1) / (len(y) - A.shape[1]))

def s15_smooth(k, n, ages, grid, bw=7.0, min_eff=0):
    # Local-linear (LOWESS-style, tricube kernel) smooth of the mean k/n over age, weighted by n.
    # k, n: arrays (A,) or (B, A) of sums and counts per single-year age. Returns (B, G).
    dist = ages[None, :] - grid[:, None]
    w0 = np.clip(1 - np.abs(dist / bw) ** 3, 0, None) ** 3
    k = np.atleast_2d(np.asarray(k, float)); n = np.atleast_2d(np.asarray(n, float))
    W = n[:, None, :] * w0[None]
    KW = k[:, None, :] * w0[None]
    S0 = W.sum(-1); S1 = (W * dist).sum(-1); S2 = (W * dist * dist).sum(-1)
    T0 = KW.sum(-1); T1 = (KW * dist).sum(-1)
    den = S0 * S2 - S1 ** 2
    with np.errstate(divide="ignore", invalid="ignore"):
        est = np.where(den > 1e-6, (S2 * T0 - S1 * T1) / den, T0 / S0)
        est = np.where(S0 >= max(min_eff, 1e-9), est, np.nan)
    return est

def s15_dummies(series, ref, prefix):
    dm = pd.get_dummies(series.astype(str), dtype=float)
    dm = dm.drop(columns=[ref])
    dm.columns = [f"{prefix}: {v}" for v in dm.columns]
    return dm

df["DM_LOG_LIMIT"] = np.log(df["LIMIT_BAL"])
s15_stat = df[PAY_COLS].to_numpy()
s15_blocks = {
    "Sex": pd.DataFrame({"Sex: Female": (df["SEX"] == 2).astype(float)}),
    "Age": s15_dummies(df["AGE_BAND"], "26-30", "Age"),
    "Education": s15_dummies(df["EDU_LABEL"], "University", "Education"),
    "Marital status": s15_dummies(df["MAR_LABEL"], "Single", "Marital"),
    "Sep status": s15_dummies(df["STATUS_GRP_SEP"], "Revolving", "Sep status"),
    "Status history Apr-Aug": pd.DataFrame({"Late months Apr-Aug": (s15_stat[:, 1:] >= 1).sum(1), "Worst status Apr-Aug": s15_stat[:, 1:].max(1),
                                            "Inactive months": df["N_INACTIVE"].to_numpy(), "Paid-in-full months": df["N_PAIDFULL"].to_numpy()}, index=df.index).astype(float),
    "Credit line": pd.DataFrame({"log limit": df["DM_LOG_LIMIT"]}),
    "Utilisation": pd.DataFrame({"Avg utilisation": df["AVG_UTIL"].clip(0, 1.5), "Utilisation trend": df["UTIL_TREND"].clip(-0.2, 0.2),
                                 "Over limit (any month)": df["OVERLIMIT_ANY"].astype(float)}),
    "Repayment share": pd.DataFrame({"Share of bills repaid (5 statements, aligned)": df["PAY_TO_BILL_5M"].clip(0, 2).fillna(0), "No positive bill": df["PAY_TO_BILL_5M"].isna().astype(float),
                                     "Zero-payment months": df["N_ZERO_PAY"].astype(float)}),
    "Bill size": pd.DataFrame({"log avg bill": np.log1p(df["AVG_BILL"].clip(lower=0)), "log avg payment": np.log1p(df["AVG_PAY"])}),
}
S15_DEMO_BLOCKS = ["Sex", "Age", "Education", "Marital status"]
S15_BEH_BLOCKS = [b for b in s15_blocks if b not in S15_DEMO_BLOCKS]
S15_X = pd.concat(list(s15_blocks.values()), axis=1).astype(float)
S15_COLBLOCK = {col: blk for blk, fr in s15_blocks.items() for col in fr.columns}
assert S15_X.notna().all().all() and len(S15_X) == len(df)
print(f"statsmodels: {S15_HAS_SM} | design matrix {S15_X.shape} | blocks: " + ", ".join(f"{b} ({fr.shape[1]})" for b, fr in s15_blocks.items()))

### 15.1 Who is in the portfolio? Demography and life cycle
#### 15.1.1 Population pyramid (age x sex), defaulters shaded
A pyramid is the standard demographic view (Tableau: two mirrored bar charts sharing the age axis). The dark part of each bar is the number of defaulters, so both the size and the risk of every age-sex cohort are visible at once.

In [ ]:
s15_pyr = df.groupby(["AGE", "SEX_LABEL"], observed=True)[TARGET].agg(n="size", defaults="sum").reset_index()
s15_pyr["rate"] = s15_pyr["defaults"] / s15_pyr["n"]
fig, ax = plt.subplots(figsize=(11, 8))
for s15_sx, s15_sign, s15_col in (("Male", -1, C_OK), ("Female", 1, C_ACC)):
    s15_g = s15_pyr[s15_pyr["SEX_LABEL"] == s15_sx]
    ax.barh(s15_g["AGE"], s15_sign * s15_g["n"], color=s15_col, alpha=.45, height=.9, label=f"{s15_sx} (all)")
    ax.barh(s15_g["AGE"], s15_sign * s15_g["defaults"], color=s15_col, height=.9, label=f"{s15_sx} defaulters")
ax.axvline(0, color="k", lw=.8)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: f"{abs(v):,.0f}"))
ax.set_xlabel("clients (men to the left, women to the right)"); ax.set_ylabel("age"); ax.set_ylim(20, 76)
ax.set_title("15.1.1 Population pyramid of the portfolio (dark = defaulters)"); ax.legend(loc="upper right", fontsize=8)
savefig("s15_pyramid"); plt.show()
export(s15_pyr, "s15_pyramid")
s15_demo_sum = df.groupby("SEX_LABEL").agg(n=(TARGET, "size"), share=(TARGET, lambda s: len(s) / len(df)), median_age=("AGE", "median"), mean_age=("AGE", "mean"),
                                           share_under30=("AGE", lambda s: (s < 30).mean()), share_45plus=("AGE", lambda s: (s >= 45).mean()), default_rate=(TARGET, "mean"))
print(s15_demo_sum.round(3).to_string())
s15_fem_band = df.groupby("AGE_BAND", observed=True)["SEX"].agg(n="size", share_female=lambda s: (s == 2).mean())
print("\nShare of women by age band:"); print(s15_fem_band.round(3).T.to_string())
s15_fem_young, s15_fem_old = s15_fem_band["share_female"].iloc[0], s15_fem_band["share_female"].iloc[-2:].mean()
add_fact("15.1", "demographics", f"Women are {(df['SEX'] == 2).mean():.0%} of clients; their share falls from {s15_fem_young:.0%} at 21-25 to about {s15_fem_old:.0%} past 50, so the male book is older", float(s15_fem_young - s15_fem_old), "s15_pyramid", 1)

**Observations:**
- Women are 60% of clients (18,112) and default less: 20.8% vs 24.2% for men.
- Women are younger: median age 33 vs 35. Their share falls from 70% at 21-25 to 53% at 51-60 and 45% at 61+.
- So part of any male-female comparison is really an age comparison.
- Story use: opening "who are the clients" sheet (`s15_pyramid.csv`).

#### 15.1.2 Education by age and sex
Is education a cohort effect? If younger cohorts are better educated, any education gap in default is partly an age gap (and vice versa). 100% stacked bars per age band, one panel per sex.

In [ ]:
s15_edu_age = (df.groupby(["SEX_LABEL", "AGE_BAND", "EDU_LABEL"], observed=True).size().rename("n").reset_index())
s15_edu_age["share"] = s15_edu_age["n"] / s15_edu_age.groupby(["SEX_LABEL", "AGE_BAND"], observed=True)["n"].transform("sum")
s15_edu_cols = dict(zip(EDU_ORDER, ["#08519c", "#6baed6", "#fd8d3c", "#969696"]))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, s15_sx in zip(axes, ["Male", "Female"]):
    s15_p = s15_edu_age[s15_edu_age["SEX_LABEL"] == s15_sx].pivot(index="AGE_BAND", columns="EDU_LABEL", values="share").reindex(index=AGE_LABELS, columns=EDU_ORDER).fillna(0)
    s15_bot = np.zeros(len(s15_p))
    for s15_e in EDU_ORDER:
        ax.bar(s15_p.index.astype(str), s15_p[s15_e], bottom=s15_bot, color=s15_edu_cols[s15_e], label=s15_e)
        s15_bot += s15_p[s15_e].to_numpy()
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title(f"15.1.2 Education mix by age band: {s15_sx}"); ax.set_xlabel("age band")
axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout(); savefig("s15_edu_by_age"); plt.show()
export(s15_edu_age, "s15_edu_by_age_sex")
s15_edu_prof = df.groupby("EDU_LABEL").agg(n=(TARGET, "size"), mean_age=("AGE", "mean"), median_age=("AGE", "median"), share_female=("SEX", lambda s: (s == 2).mean()),
                                           share_married=("MAR_LABEL", lambda s: (s == "Married").mean()), default_rate=(TARGET, "mean")).reindex(EDU_ORDER)
print(s15_edu_prof.round(3).to_string())
s15_grad_young = (df.loc[df["AGE"] <= 30, "EDU_LABEL"] == "Graduate school").mean(); s15_grad_old = (df.loc[df["AGE"] > 45, "EDU_LABEL"] == "Graduate school").mean()
s15_hs_young = (df.loc[df["AGE"] <= 30, "EDU_LABEL"] == "High school").mean(); s15_hs_old = (df.loc[df["AGE"] > 45, "EDU_LABEL"] == "High school").mean()
print(f"\nGraduate-school share: age<=30 {s15_grad_young:.1%} vs age>45 {s15_grad_old:.1%} | High-school share: {s15_hs_young:.1%} vs {s15_hs_old:.1%}")
print(f"Cramer's V education x age band: {cramers_v(df['EDU_LABEL'], df['AGE_BAND']):.3f} | education x sex: {cramers_v(df['EDU_LABEL'], df['SEX_LABEL']):.3f}")
add_fact("15.1", "demographics", f"Education is a cohort effect: high-school share is {s15_hs_young:.0%} among clients up to 30 but {s15_hs_old:.0%} past 45", float(s15_hs_old - s15_hs_young), "s15_edu_by_age", 1)

**Observations:**
- Education is a cohort effect: the graduate-school share is 39.1% for clients up to 30 but 24.1% past 45; the high-school share rises from 9.8% to 34.3%.
- High-school clients are older (mean age 40.3 vs 34.2 for graduate school) and default more (25.2% vs 19.2%).
- Education is linked to age (Cramér's V 0.156, a weak-to-moderate association) but not to sex (0.026).
- Other/Unknown (468 clients) defaults at only 7.1%.
- Story use: background; a reminder that the education gap is partly an age gap.

#### 15.1.3 The marriage transition: share married by single-year age, by sex
A lifecycle transition curve: at which age does half of each sex report "married"? It tells us how strongly marital status is just a proxy for age, and whether the proxy differs for men and women.

In [ ]:
s15_mar_age = df.groupby(["SEX_LABEL", "AGE"], observed=True).agg(n=("MAR_LABEL", "size"), married=("MAR_LABEL", lambda s: (s == "Married").sum()),
                                                                  other=("MAR_LABEL", lambda s: (s == "Other").sum())).reset_index()
s15_mar_age["share_married"] = s15_mar_age["married"] / s15_mar_age["n"]
s15_mar_age["ci_lo"], s15_mar_age["ci_hi"] = wilson_ci(s15_mar_age["married"], s15_mar_age["n"])
s15_mar_age["share_other"] = s15_mar_age["other"] / s15_mar_age["n"]
fig, ax = plt.subplots(figsize=(11, 5))
s15_cross = {}
for s15_sx, s15_col in (("Male", C_OK), ("Female", C_ACC)):
    s15_g = s15_mar_age[(s15_mar_age["SEX_LABEL"] == s15_sx) & (s15_mar_age["n"] >= 30)].sort_values("AGE")
    ax.plot(s15_g["AGE"], s15_g["share_married"], color=s15_col, marker="o", ms=3, label=f"{s15_sx}: married")
    ax.fill_between(s15_g["AGE"], s15_g["ci_lo"], s15_g["ci_hi"], color=s15_col, alpha=.18)
    ax.plot(s15_g["AGE"], s15_g["share_other"], color=s15_col, ls=":", label=f"{s15_sx}: other status")
    s15_sm = s15_g["share_married"].rolling(3, center=True, min_periods=1).mean().to_numpy(); s15_ag = s15_g["AGE"].to_numpy()
    s15_i = int(np.argmax(s15_sm >= 0.5))
    s15_cross[s15_sx] = float(s15_ag[s15_i]) if s15_i == 0 else float(s15_ag[s15_i - 1] + (0.5 - s15_sm[s15_i - 1]) / (s15_sm[s15_i] - s15_sm[s15_i - 1]) * (s15_ag[s15_i] - s15_ag[s15_i - 1]))
    ax.axvline(s15_cross[s15_sx], color=s15_col, ls="--", lw=1)
ax.axhline(0.5, color="k", lw=.8); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("age (single years with n >= 30 per sex)"); ax.set_ylabel("share of clients")
ax.set_title("15.1.3 Marriage transition curve by sex (band = 95% Wilson CI, dashed = age where 50% are married)"); ax.legend(fontsize=8)
savefig("s15_marriage_transition"); plt.show()
export(s15_mar_age, "s15_marriage_transition")
print("Age at which half are married (3-year smoothed):", {k: round(v, 1) for k, v in s15_cross.items()})
print(df.groupby("AGE_BAND", observed=True)["MAR_LABEL"].value_counts(normalize=True).unstack().reindex(columns=MAR_ORDER).round(3).to_string())
print(f"Cramer's V marital status x age band: {cramers_v(df['MAR_LABEL'], df['AGE_BAND']):.3f}")
add_fact("15.1", "lifecycle", f"Half of women are married by age {s15_cross['Female']:.1f} and half of men by {s15_cross['Male']:.1f}: marital status is largely an age proxy (Cramer's V {cramers_v(df['MAR_LABEL'], df['AGE_BAND']):.2f})",
         float(s15_cross["Male"] - s15_cross["Female"]), "s15_marriage_transition", 2)

**Observations:**
- Half of women are married by age 32.9, half of men by 35.4.
- The married share rises from 11% at 21-25 to 72-74% from 41 on, and 83% at 61+.
- Marital status is strongly tied to age (Cramér's V 0.353), so "married vs single" is largely "older vs younger".
- Story use: background for the household sheet.

#### 15.1.4 Life-stage variable `DM_LIFE_STAGE`
Rule-based life stages combining age and marital status (rules in the code, applied top to bottom): Senior (60+) → Other marital status → Young (<30) single / married → Mid-life (30-44) single / married → Mature (45-59) single / married. One readable dimension for Tableau instead of an age x marriage grid.

In [ ]:
s15_a, s15_m = df["AGE"], df["MAR_LABEL"]
S15_STAGE_ORDER = ["Young single (<30)", "Young married (<30)", "Mid-life single (30-44)", "Mid-life married (30-44)",
                   "Mature single (45-59)", "Mature married (45-59)", "Senior (60+)", "Other marital status (<60)"]
df["DM_LIFE_STAGE"] = np.select(
    [s15_a >= 60, s15_m == "Other", (s15_a < 30) & (s15_m == "Single"), (s15_a < 30) & (s15_m == "Married"),
     (s15_a < 45) & (s15_m == "Single"), (s15_a < 45) & (s15_m == "Married"), s15_m == "Single", s15_m == "Married"],
    ["Senior (60+)", "Other marital status (<60)", "Young single (<30)", "Young married (<30)",
     "Mid-life single (30-44)", "Mid-life married (30-44)", "Mature single (45-59)", "Mature married (45-59)"], default="Unclassified")
fig, ax = plt.subplots(figsize=(12, 4.8))
s15_stage = plot_rate(df, "DM_LIFE_STAGE", ax=ax, order=S15_STAGE_ORDER, title="15.1.4 Default rate by life stage (bars = clients, line = rate with 95% CI)", rotate=25)
savefig("s15_life_stage"); plt.show()
s15_stage_prof = df.groupby("DM_LIFE_STAGE").agg(median_limit=("LIMIT_BAL", "median"), mean_util=("AVG_UTIL", "mean"), share_late_sep=("PAY_1", lambda s: (s >= 1).mean()),
                                                  share_female=("SEX", lambda s: (s == 2).mean()), share_grad=("EDU_LABEL", lambda s: (s == "Graduate school").mean())).reset_index()
s15_stage = s15_stage.merge(s15_stage_prof, on="DM_LIFE_STAGE")
print(s15_stage[["DM_LIFE_STAGE", "n", "share_clients", "rate", "ci_lo", "ci_hi", "lift", "median_limit", "mean_util", "share_late_sep", "share_female", "share_grad"]].round(3).to_string(index=False))
export(s15_stage, "s15_life_stage")
s15_st_hi, s15_st_lo = s15_stage.loc[s15_stage["rate"].idxmax()], s15_stage.loc[s15_stage["rate"].idxmin()]
add_fact("15.1", "lifecycle", f"Life stages span only {s15_st_lo['rate']:.1%} ({s15_st_lo['DM_LIFE_STAGE']}) to {s15_st_hi['rate']:.1%} ({s15_st_hi['DM_LIFE_STAGE']}) in default rate",
         float(s15_st_hi["rate"] - s15_st_lo["rate"]), "s15_life_stage", 1)

**Observations:**
- Default rates by life stage sit in a narrow band: 19.4% (mid-life single, 30-44) to 28.3% (seniors 60+, n=339).
- Married clients default more at every stage: young 26.1% vs 22.2% single, mid-life 22.0% vs 19.4%, mature 25.2% vs 20.6%.
- Mid-life clients get about double the line of young clients (median NT$180k vs 80-90k) and use less of it (33-35% vs 41-46% utilisation).
- Story use: one life-stage bar chart (`s15_life_stage.csv`); easier to read than age x marriage.

### 15.2 How does the bank hand out credit lines?
`LIMIT_BAL` is the only variable the **bank** chose. Reading it against demographics reveals the underwriting rule; reading it against realised default shows where that rule was right or wrong.

#### 15.2.1 Credit line by every demographic (log scale)
Violins on a log10 axis (the limit is right-skewed and set in round steps), with a medians table. Tableau: box plots of `LIMIT_BAL` (log axis) by dimension.

In [ ]:
df["DM_LOG10_LIMIT"] = np.log10(df["LIMIT_BAL"])
s15_dims = [("SEX_LABEL", ["Male", "Female"]), ("EDU_LABEL", EDU_ORDER), ("MAR_LABEL", MAR_ORDER), ("AGE_BAND", AGE_LABELS), ("DM_LIFE_STAGE", S15_STAGE_ORDER)]
s15_kfmt = mtick.FuncFormatter(lambda v, p: f"{10 ** v / 1000:,.0f}k")
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for ax, (s15_c, s15_o) in zip(axes.ravel(), s15_dims[:4]):
    sns.violinplot(data=df, x=s15_c, y="DM_LOG10_LIMIT", order=s15_o, color="#c6dbef", cut=0, inner="quartile", ax=ax)
    s15_med = df.groupby(s15_c, observed=True)["DM_LOG10_LIMIT"].median().reindex(s15_o)
    ax.plot(range(len(s15_o)), s15_med.to_numpy(), color=C_DEF, marker="D", lw=1.5, label="median")
    ax.yaxis.set_major_formatter(s15_kfmt); ax.set_yticks(np.log10([10e3, 20e3, 50e3, 100e3, 200e3, 500e3, 1e6]))
    ax.set_ylabel("credit line NT$ (log scale)"); ax.set_xlabel(""); ax.set_title(f"15.2.1 Credit line by {s15_c}"); ax.legend(fontsize=8)
plt.tight_layout(); savefig("s15_limit_violin"); plt.show()
s15_rows = []
for s15_c, s15_o in s15_dims:
    for s15_lev in s15_o:
        s15_g = df[df[s15_c] == s15_lev]
        if len(s15_g) == 0: continue
        s15_rows.append({"demographic": s15_c, "level": s15_lev, "n": len(s15_g), "median_limit": s15_g["LIMIT_BAL"].median(), "p25": s15_g["LIMIT_BAL"].quantile(.25),
                         "p75": s15_g["LIMIT_BAL"].quantile(.75), "mean_limit": s15_g["LIMIT_BAL"].mean(), "geo_mean_limit": float(np.exp(s15_g["DM_LOG_LIMIT"].mean())),
                         "share_le_50k": (s15_g["LIMIT_BAL"] <= 50_000).mean(), "share_ge_300k": (s15_g["LIMIT_BAL"] >= 300_000).mean(), "default_rate": s15_g[TARGET].mean()})
s15_limit_demo = pd.DataFrame(s15_rows)
print(s15_limit_demo.round(3).to_string(index=False))
export(s15_limit_demo, "s15_limit_by_demo")
for s15_c in ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND"]:
    s15_kw = stats.kruskal(*[g["LIMIT_BAL"].to_numpy() for _, g in df.groupby(s15_c, observed=True)])
    s15_eps = (s15_kw.statistic - df[s15_c].nunique() + 1) / (len(df) - df[s15_c].nunique())
    print(f"Kruskal-Wallis {s15_c:10s}: H = {s15_kw.statistic:9.1f}, p = {s15_kw.pvalue:.1e}, epsilon^2 = {s15_eps:.3f}")

**Observations:**
- Graduate-school clients get the largest lines (median NT$200k) vs 110k for university and 80k for high school.
- Clients aged 21-25 get a median of 50k and 57% of them sit at or below 50k; lines peak at 180k for ages 31-40.
- Women get larger lines than men (median 150k vs 130k; 22% vs 31% at or below 50k), even though men are older.
- Groups with smaller lines default more: age 21-25 (50k) 26.7%, graduate school (200k) 19.2%.
- Story use: "how the bank sets lines" sheet.

#### 15.2.2 The credit-line life cycle: median limit by single-year age, per sex and per education
Does the bank raise lines with age (seniority / income proxy), and where does it stop? Shaded band = inter-quartile range for the sex panel.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
s15_lim_age = []
for s15_sx, s15_col in (("Male", C_OK), ("Female", C_ACC)):
    s15_g = df[df["SEX_LABEL"] == s15_sx].groupby("AGE")["LIMIT_BAL"].agg(n="size", median="median", p25=lambda s: s.quantile(.25), p75=lambda s: s.quantile(.75)).reset_index()
    s15_g = s15_g[s15_g["n"] >= 30]
    axes[0].plot(s15_g["AGE"], s15_g["median"] / 1000, color=s15_col, marker="o", ms=3, label=s15_sx)
    axes[0].fill_between(s15_g["AGE"], s15_g["p25"] / 1000, s15_g["p75"] / 1000, color=s15_col, alpha=.12)
    s15_lim_age.append(s15_g.assign(split="Sex", level=s15_sx))
for s15_e in EDU_ORDER:
    s15_g = df[df["EDU_LABEL"] == s15_e].groupby("AGE")["LIMIT_BAL"].agg(n="size", median="median", p25=lambda s: s.quantile(.25), p75=lambda s: s.quantile(.75)).reset_index()
    s15_g = s15_g[s15_g["n"] >= 30]
    axes[1].plot(s15_g["AGE"], s15_g["median"] / 1000, color=s15_edu_cols[s15_e], marker="o", ms=3, label=s15_e)
    s15_lim_age.append(s15_g.assign(split="Education", level=s15_e))
for ax, s15_ttl in zip(axes, ["by sex (band = IQR)", "by education"]):
    ax.set_xlabel("age (single years, n >= 30)"); ax.set_title(f"15.2.2 Median credit line by age, {s15_ttl}"); ax.legend(fontsize=8)
axes[0].set_ylabel("median credit line (NT$ thousand)")
plt.tight_layout(); savefig("s15_limit_by_age"); plt.show()
s15_lim_age = pd.concat(s15_lim_age, ignore_index=True)
export(s15_lim_age, "s15_limit_by_age")
s15_all_age = df.groupby("AGE")["LIMIT_BAL"].agg(n="size", median="median"); s15_all_age = s15_all_age[s15_all_age["n"] >= 50]
s15_peak_age = int(s15_all_age["median"].rolling(3, center=True, min_periods=1).mean().idxmax())
print(f"Median line: age 21-23 {df.loc[df['AGE'] <= 23, 'LIMIT_BAL'].median():,.0f} | age 30 {df.loc[df['AGE'] == 30, 'LIMIT_BAL'].median():,.0f} | "
      f"peak around age {s15_peak_age} ({s15_all_age['median'].rolling(3, center=True, min_periods=1).mean().max():,.0f}) | age 55-60 {df.loc[df['AGE'].between(55, 60), 'LIMIT_BAL'].median():,.0f}")
print(f"Spearman(AGE, LIMIT) all: {stats.spearmanr(df['AGE'], df['LIMIT_BAL'])[0]:.3f} | age<=35: {stats.spearmanr(df.loc[df['AGE'] <= 35, 'AGE'], df.loc[df['AGE'] <= 35, 'LIMIT_BAL'])[0]:.3f} "
      f"| age>35: {stats.spearmanr(df.loc[df['AGE'] > 35, 'AGE'], df.loc[df['AGE'] > 35, 'LIMIT_BAL'])[0]:.3f}")
add_fact("15.2", "credit line", f"The median credit line climbs from NT${df.loc[df['AGE'] <= 23, 'LIMIT_BAL'].median() / 1000:,.0f}k at 21-23 to a plateau near NT${s15_all_age['median'].rolling(3, center=True, min_periods=1).mean().max() / 1000:,.0f}k around age {s15_peak_age}, then stops growing",
         float(s15_peak_age), "s15_limit_by_age", 2)

**Observations:**
- The median line climbs from NT$50k at 21-23 to 180k at age 30, peaks around 31, then falls to 100k by 55-60.
- Age and limit move together up to 35 (Spearman rank correlation 0.41) and slightly apart after 35 (-0.11).
- Story use: credit-line life-cycle chart (`s15_limit_by_age.csv`).

#### 15.2.3 What explains the line? A descriptive regression of log(LIMIT_BAL) on demographics
OLS with robust errors; coefficients are shown as **% difference in credit line** versus the reference client (Male, 26-30, University, Single), holding the other demographics fixed. This is a description of the bank's allocation, not a causal model.

In [ ]:
s15_demo_cols = [c for c in S15_X.columns if S15_COLBLOCK[c] in S15_DEMO_BLOCKS]
s15_co_demo, S15_R2_DEMO, s15_r2a = s15_ols(S15_X[s15_demo_cols], df["DM_LOG_LIMIT"])
s15_eff = s15_co_demo.drop("const").copy()
s15_eff["pct_effect"] = np.exp(s15_eff["coef"]) - 1
s15_eff["pct_lo"] = np.exp(s15_eff["coef"] - 1.96 * s15_eff["se"]) - 1
s15_eff["pct_hi"] = np.exp(s15_eff["coef"] + 1.96 * s15_eff["se"]) - 1
s15_eff["block"] = [S15_COLBLOCK[c] for c in s15_eff.index]
s15_eff = s15_eff.reset_index().rename(columns={"index": "term"})
fig, ax = plt.subplots(figsize=(10, 6))
s15_y = np.arange(len(s15_eff))[::-1]
ax.errorbar(s15_eff["pct_effect"], s15_y, xerr=[s15_eff["pct_effect"] - s15_eff["pct_lo"], s15_eff["pct_hi"] - s15_eff["pct_effect"]], fmt="o", color=C_OK, ecolor=C_NEU, capsize=3)
ax.axvline(0, color="k", lw=.8); ax.set_yticks(s15_y); ax.set_yticklabels(s15_eff["term"]); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("credit line vs reference client (Male, 26-30, University, Single), 95% CI")
ax.set_title(f"15.2.3 Demographic 'pricing' of the credit line (OLS on log limit, R2 = {S15_R2_DEMO:.1%})")
savefig("s15_limit_ols_demo"); plt.show()
print(f"n = {len(df):,} | R2 = {S15_R2_DEMO:.4f} | adj R2 = {s15_r2a:.4f} | reference line = NT${np.exp(s15_co_demo.loc['const', 'coef']):,.0f}")
print(s15_eff[["block", "term", "coef", "se", "pct_effect", "pct_lo", "pct_hi"]].round(4).to_string(index=False))
export(s15_eff, "s15_limit_ols_demo")
# contribution of each demographic block: R2 alone and R2 lost when dropped
s15_rows = []
for s15_b in S15_DEMO_BLOCKS:
    s15_alone = s15_ols(S15_X[[c for c in s15_demo_cols if S15_COLBLOCK[c] == s15_b]], df["DM_LOG_LIMIT"])[1]
    s15_wo = s15_ols(S15_X[[c for c in s15_demo_cols if S15_COLBLOCK[c] != s15_b]], df["DM_LOG_LIMIT"])[1]
    s15_rows.append({"block": s15_b, "r2_alone": s15_alone, "r2_unique (drop-one loss)": S15_R2_DEMO - s15_wo})
s15_r2_blocks = pd.DataFrame(s15_rows)
print("\nWhich demographic carries the line?"); print(s15_r2_blocks.round(4).to_string(index=False))
s15_top_blk = s15_r2_blocks.sort_values("r2_alone", ascending=False).iloc[0]
s15_hs_eff = float(s15_eff.loc[s15_eff["term"] == "Education: High school", "pct_effect"].iloc[0]); s15_gs_eff = float(s15_eff.loc[s15_eff["term"] == "Education: Graduate school", "pct_effect"].iloc[0])
add_fact("15.2", "credit line", f"Demographics alone explain {S15_R2_DEMO:.0%} of the variance of log credit line; {s15_top_blk['block'].lower()} is the strongest block (R2 {s15_top_blk['r2_alone']:.0%} alone). "
         f"Graduate school carries {s15_gs_eff:+.0%} and high school {s15_hs_eff:+.0%} line vs university at equal age, sex and marital status", float(S15_R2_DEMO), "s15_limit_ols_demo", 2)

**Observations:**
- Demographics explain 20% of the variation in log credit line (R² 0.20). The reference client (male, 26-30, university, single) gets about NT$89k.
- At equal age, sex and marriage: graduate school +57%, high school -21%, women +21%, age 21-25 -54%.
- Age is the strongest block (R² 0.117 alone), then education (0.075); sex and marriage add little (0.015 or less).
- Story use: background; the line mostly follows age and education.

#### 15.2.4 Does behaviour explain the line better than demographics?
Nested models. **Status history** (Sep status group, Apr-Aug lateness, inactive / paid-in-full months) is behaviour that is *not* mechanically tied to the limit. **Utilisation, repayment share and bill size** are added last and flagged: utilisation is `BILL / LIMIT`, so its extra R² is partly an accounting identity, not insight. Note also the direction is ambiguous: the snapshot limit may already reflect past behaviour (line increases for good payers).

In [ ]:
s15_hist_cols = [c for c in S15_X.columns if S15_COLBLOCK[c] in ("Sep status", "Status history Apr-Aug")]
s15_mech_cols = [c for c in S15_X.columns if S15_COLBLOCK[c] in ("Utilisation", "Repayment share", "Bill size")]
s15_specs = [("Demographics only", s15_demo_cols), ("Status history only", s15_hist_cols), ("Demographics + status history", s15_demo_cols + s15_hist_cols),
             ("All behaviour only (incl. utilisation, bills)", s15_hist_cols + s15_mech_cols), ("Demographics + all behaviour", s15_demo_cols + s15_hist_cols + s15_mech_cols)]
s15_r2_tab = pd.DataFrame([{"model": nm, "n_terms": len(cols), "r2": s15_ols(S15_X[cols], df["DM_LOG_LIMIT"])[1]} for nm, cols in s15_specs])
s15_r2_tab["mechanical_terms"] = s15_r2_tab["model"].str.contains("All|all")
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.barh(s15_r2_tab["model"][::-1], s15_r2_tab["r2"][::-1], color=[C_NEU if m else C_OK for m in s15_r2_tab["mechanical_terms"][::-1]])
for s15_i, s15_v in enumerate(s15_r2_tab["r2"][::-1]):
    ax.text(s15_v + .005, s15_i, f"{s15_v:.1%}", va="center")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("R2 of log(LIMIT_BAL)")
ax.set_title("15.2.4 What explains the credit line? (grey = includes terms mechanically tied to the limit)")
savefig("s15_limit_r2_nested"); plt.show()
print(s15_r2_tab.round(4).to_string(index=False))
export(s15_r2_tab, "s15_limit_r2_nested")
s15_co_full = s15_ols(S15_X[s15_demo_cols + s15_hist_cols], df["DM_LOG_LIMIT"])[0].drop("const")
s15_co_full["pct_effect"] = np.exp(s15_co_full["coef"]) - 1
print("\nStatus-history terms in the 'demographics + status history' model (% line per unit):")
print(s15_co_full.loc[s15_hist_cols, ["coef", "se", "pct_effect"]].round(4).to_string())
s15_r2_dh = float(s15_r2_tab.loc[s15_r2_tab["model"] == "Demographics + status history", "r2"].iloc[0])
add_fact("15.2", "credit line", f"Adding six months of status history to demographics lifts the explained variance of the credit line only from {S15_R2_DEMO:.0%} to {s15_r2_dh:.0%}: "
         f"most of the line is set by something we do not observe (income, tenure, collateral)", float(s15_r2_dh - S15_R2_DEMO), "s15_limit_r2_nested", 2)

**Observations:**
- Status history alone explains 14% of the log line; demographics plus status history 28%.
- Adding utilisation and bills lifts it to 55%, but that is partly mechanical (utilisation = bill / limit).
- Clients inactive in September hold 41% larger lines; each step worse in Apr-Aug status means about 10% less line.
- Story use: background; most of the line comes from information we cannot see (income, tenure, collateral).

#### 15.2.5 Line relative to demographic peers (`DM_LIMIT_RESID`) and default
The residual of the demographic model is the client's line **relative to people of the same sex, age band, education and marital status** (log points; +0.69 = double the peer line). It is the bank's private judgement about the client beyond demographics. Does that judgement predict default, and does it still do so among clients who look the same in September?

In [ ]:
s15_Xd = np.column_stack([np.ones(len(df)), S15_X[s15_demo_cols].to_numpy()])
df["DM_LIMIT_PEER"] = np.exp(s15_Xd @ s15_co_demo["coef"].to_numpy())
df["DM_LIMIT_RESID"] = df["DM_LOG_LIMIT"] - np.log(df["DM_LIMIT_PEER"])
S15_RESID_LABELS = ["Q1 far below peers", "Q2 below peers", "Q3 near peers", "Q4 above peers", "Q5 far above peers"]
df["DM_LIMIT_RESID_BAND"] = pd.qcut(df["DM_LIMIT_RESID"], 5, labels=S15_RESID_LABELS)
df["DM_RESID_VENTILE"] = pd.qcut(df["DM_LIMIT_RESID"], 20, labels=False, duplicates="drop")
def s15_resid_curve(data, by="DM_RESID_VENTILE"):
    g = data.groupby(by, observed=True).agg(n=(TARGET, "size"), defaults=(TARGET, "sum"), resid=("DM_LIMIT_RESID", "mean"), ratio=("DM_LIMIT_RESID", lambda s: float(np.exp(s.mean())))).reset_index()
    g["rate"] = g["defaults"] / g["n"]; g["ci_lo"], g["ci_hi"] = wilson_ci(g["defaults"], g["n"])
    return g
s15_rc_all = s15_resid_curve(df)
s15_rc_split = {nm: s15_resid_curve(df[m]) for nm, m in (("Sep status clean (<= 0)", df["PAY_1"] <= 0), ("Sep status late (>= 1)", df["PAY_1"] >= 1))}
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(s15_rc_all["ratio"], s15_rc_all["rate"], color=C_DEF, marker="o"); axes[0].fill_between(s15_rc_all["ratio"], s15_rc_all["ci_lo"], s15_rc_all["ci_hi"], color=C_DEF, alpha=.15)
axes[0].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[0].axvline(1, color=C_NEU, lw=1)
axes[0].set_title("15.2.5 Default rate vs line relative to demographic peers (ventiles)")
for (nm, g), s15_col in zip(s15_rc_split.items(), (C_OK, C_DEF)):
    axes[1].plot(g["ratio"], g["rate"], color=s15_col, marker="o", label=nm); axes[1].fill_between(g["ratio"], g["ci_lo"], g["ci_hi"], color=s15_col, alpha=.15)
axes[1].axvline(1, color=C_NEU, lw=1); axes[1].legend(fontsize=8); axes[1].set_title("... same, split by September status")
for ax in axes:
    ax.set_xscale("log"); ax.set_xticks([0.1, 0.2, 0.5, 1, 2, 4]); ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: f"{v:g}x"))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("own line / line predicted from demographics (log scale)"); ax.set_ylabel("default rate")
plt.tight_layout(); savefig("s15_limit_resid_default"); plt.show()
s15_rb = rate_table(df, "DM_LIMIT_RESID_BAND").merge(df.groupby("DM_LIMIT_RESID_BAND", observed=True).agg(median_limit=("LIMIT_BAL", "median"), median_peer_line=("DM_LIMIT_PEER", "median"),
                                                     share_late_sep=("PAY_1", lambda s: (s >= 1).mean()), mean_util=("AVG_UTIL", "mean")).reset_index(), on="DM_LIMIT_RESID_BAND")
print(s15_rb[["DM_LIMIT_RESID_BAND", "n", "rate", "ci_lo", "ci_hi", "lift", "median_limit", "median_peer_line", "share_late_sep", "mean_util"]].round(3).to_string(index=False))
export(s15_rb, "s15_limit_resid_bands")
export(pd.concat([s15_rc_all.assign(split="All")] + [g.assign(split=nm) for nm, g in s15_rc_split.items()], ignore_index=True), "s15_limit_resid_curve")
s15_rho_raw = stats.spearmanr(df["LIMIT_BAL"], df[TARGET])[0]; s15_rho_res = stats.spearmanr(df["DM_LIMIT_RESID"], df[TARGET])[0]; s15_rho_peer = stats.spearmanr(df["DM_LIMIT_PEER"], df[TARGET])[0]
print(f"\nSpearman with default: raw limit {s15_rho_raw:.3f} | peer-relative residual {s15_rho_res:.3f} | demographic-predicted line {s15_rho_peer:.3f}")
s15_q1, s15_q5 = s15_rb["rate"].iloc[0], s15_rb["rate"].iloc[-1]
add_fact("15.2", "credit line", f"Clients whose line is far below their demographic peers default at {s15_q1:.1%} vs {s15_q5:.1%} for those far above: the risk signal in the limit lives in the bank's private judgement "
         f"(Spearman {s15_rho_res:.2f}), not in the demographic part of the line ({s15_rho_peer:.2f})", float(s15_q1 - s15_q5), "s15_limit_resid_default", 3)

**Observations:**
- The peer residual compares a client's line with people of the same sex, age band, education and marital status.
- Clients far below their peers (bottom fifth) default at 32.4%; clients far above (top fifth) at 14.5%.
- Rank correlation with default: raw limit -0.170, peer-relative part -0.154, demographic-predicted part only -0.074.
- So the bank's private judgement (beyond age, sex and education) carries most of the limit's risk signal.
- Story use: strong sheet, "the bank knew something" (`s15_limit_resid_bands.csv`).

#### 15.2.6 Risk-vs-line mismatch map: who gets "too much" or "too little" line for their realised risk?
Each bubble is a demographic segment (life stage x education x sex, n ≥ 150). A sensible lender gives larger lines to safer segments, so bubbles should fall along a downward trend (weighted fit of default rate on log median line). Segments **above** the trend default more than their line implies ("over-lined for their risk"); segments **below** default less ("under-lined", room to grow). Descriptive only: realised risk was not known when lines were set.

In [ ]:
s15_seg_key = ["DM_LIFE_STAGE", "EDU_LABEL", "SEX_LABEL"]
s15_seg = rate_table(df, s15_seg_key, min_n=150)
s15_seg = s15_seg.merge(df.groupby(s15_seg_key, observed=True).agg(median_limit=("LIMIT_BAL", "median"), geo_mean_limit=("DM_LOG_LIMIT", lambda s: float(np.exp(s.mean()))),
                                                                   mean_util=("AVG_UTIL", "mean"), exposure=("LIMIT_BAL", "sum")).reset_index(), on=s15_seg_key)
s15_seg["segment"] = s15_seg[s15_seg_key].astype(str).agg(" | ".join, axis=1)
s15_xx = np.log10(s15_seg["geo_mean_limit"])
s15_fit = np.polyfit(s15_xx, s15_seg["rate"], 1, w=np.sqrt(s15_seg["n"]))
s15_seg["trend_rate"] = np.polyval(s15_fit, s15_xx)
s15_seg["gap_pp"] = 100 * (s15_seg["rate"] - s15_seg["trend_rate"])
s15_seg["mismatch"] = np.where((s15_seg["ci_lo"] > s15_seg["trend_rate"]), "Over-lined for its risk",
                               np.where(s15_seg["ci_hi"] < s15_seg["trend_rate"], "Under-lined for its risk", "On trend"))
s15_mm_cols = {"Over-lined for its risk": C_DEF, "Under-lined for its risk": C_ACC, "On trend": C_NEU}
fig, ax = plt.subplots(figsize=(13, 7))
for s15_k, s15_g in s15_seg.groupby("mismatch"):
    ax.scatter(s15_g["geo_mean_limit"], s15_g["rate"], s=s15_g["n"] / 6, color=s15_mm_cols[s15_k], alpha=.6, edgecolor="k", lw=.5, label=f"{s15_k} ({len(s15_g)})")
s15_gx = np.linspace(s15_xx.min(), s15_xx.max(), 50)
ax.plot(10 ** s15_gx, np.polyval(s15_fit, s15_gx), color="k", ls="--", lw=1.2, label="n-weighted trend")
for _, s15_r in pd.concat([s15_seg.nlargest(5, "gap_pp"), s15_seg.nsmallest(5, "gap_pp")]).iterrows():
    ax.annotate(s15_r["segment"], (s15_r["geo_mean_limit"], s15_r["rate"]), fontsize=7, xytext=(5, 4), textcoords="offset points")
ax.set_xscale("log"); ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: f"{v / 1000:,.0f}k")); ax.xaxis.set_minor_formatter(mtick.FuncFormatter(lambda v, p: f"{v / 1000:,.0f}k"))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("segment geometric-mean credit line (NT$, log scale)"); ax.set_ylabel("segment default rate")
ax.set_title("15.2.6 Risk-vs-line mismatch map (bubble = clients; red = defaults more than its line implies)")
s15_h, s15_l = ax.get_legend_handles_labels(); s15_lg = ax.legend(s15_h, s15_l, fontsize=8)
for s15_hd in s15_lg.legend_handles if hasattr(s15_lg, "legend_handles") else s15_lg.legendHandles:
    try: s15_hd.set_sizes([40])
    except Exception: pass
savefig("s15_risk_line_mismatch"); plt.show()
s15_seg = s15_seg.sort_values("gap_pp", ascending=False).reset_index(drop=True)
s15_show = ["segment", "n", "rate", "ci_lo", "ci_hi", "median_limit", "geo_mean_limit", "trend_rate", "gap_pp", "mismatch"]
print(f"Trend: default rate changes by {s15_fit[0] * np.log10(2) * 100:+.2f} pp per doubling of the segment line | weighted corr = {np.corrcoef(s15_xx, s15_seg['rate'])[0, 1]:.2f} | segments: {len(s15_seg)}")
print("\nMost over-lined for realised risk:"); print(s15_seg[s15_show].head(8).round(3).to_string(index=False))
print("\nMost under-lined for realised risk:"); print(s15_seg[s15_show].tail(8).iloc[::-1].round(3).to_string(index=False))
print("\n", s15_seg["mismatch"].value_counts().to_string())
export(s15_seg, "s15_risk_line_mismatch")
s15_over, s15_under = s15_seg.iloc[0], s15_seg.iloc[-1]
add_fact("15.2", "credit line", f"Across demographic segments each doubling of the credit line goes with {s15_fit[0] * np.log10(2) * 100:+.1f} pp default; the largest mismatch is '{s15_over['segment']}' "
         f"({s15_over['rate']:.1%}, {s15_over['gap_pp']:+.1f} pp above the trend for its line) while '{s15_under['segment']}' sits {s15_under['gap_pp']:+.1f} pp below", float(s15_over["gap_pp"]), "s15_risk_line_mismatch", 2)

**Observations:**
- Across 31 segments, default falls 4.2 pp for each doubling of the median line; 20 segments sit on the trend, 7 are over-lined and 4 under-lined.
- Over-lined segments are mostly married men: mature married graduate-school men default at 24.6% on a 250k median line, 6.2 pp above the trend.
- Under-lined segments are mostly single women: young single graduate women default at 18.8% on 130k, 3.9 pp below the trend.
- Story use: risk-vs-line bubble map (`s15_risk_line_mismatch.csv`).

### 15.3 Age deep-dive: is the U-shape real, where is its minimum, and is it just behaviour?
Section 3.2 drew one overall single-year age line. Here: a GAM-like smooth (n-weighted local-linear fit with a tricube kernel, ±7 years, i.e. LOWESS on the age-level rates) with a **client-level bootstrap band** (B = 400), separately by sex, marital status and education; then a bootstrap CI for the **age of minimum risk**.

#### 15.3.1 Smoothed default rate by age, by sex

In [ ]:
S15_AGES = np.arange(21, 80)
S15_GRID = np.arange(21, 65.5, 0.5)
def s15_age_curve(mask=None, n_boot=400, bw=7.0, lo_age=22, hi_age=60, min_eff=150):
    g = df if mask is None else df[mask]
    a = g["AGE"].to_numpy(); y = g[TARGET].to_numpy(); nA = len(S15_AGES)
    c1 = np.bincount(a[y == 1] - 21, minlength=nA)[:nA].astype(float)
    c0 = np.bincount(a[y == 0] - 21, minlength=nA)[:nA].astype(float)
    eff = s15_smooth(c0 + c1, np.ones(nA), S15_AGES, S15_GRID, bw)[0]          # kernel-smoothed local n per year
    ok = eff * 2 * bw * 0.6 >= min_eff                                         # enough clients inside the window
    est = np.where(ok, s15_smooth(c1, c0 + c1, S15_AGES, S15_GRID, bw)[0], np.nan)
    draws = S15_RNG.multinomial(len(a), np.concatenate([c0, c1]) / len(a), size=n_boot)   # exact client-level bootstrap of the age x outcome counts
    bs = s15_smooth(draws[:, nA:], draws[:, :nA] + draws[:, nA:], S15_AGES, S15_GRID, bw)
    bs[:, ~ok] = np.nan
    lo, hi = np.nanpercentile(bs, [2.5, 97.5], axis=0)
    sel = ok & (S15_GRID >= lo_age) & (S15_GRID <= hi_age)
    amin = float(S15_GRID[sel][np.nanargmin(est[sel])]); amin_b = S15_GRID[sel][np.nanargmin(bs[:, sel], axis=1)]
    raw = pd.DataFrame({"AGE": S15_AGES, "n": c0 + c1, "rate": np.where(c0 + c1 > 0, c1 / np.maximum(c0 + c1, 1), np.nan)})
    return {"n": len(a), "est": est, "lo": lo, "hi": hi, "raw": raw, "age_min": amin, "age_min_lo": float(np.percentile(amin_b, 2.5)), "age_min_hi": float(np.percentile(amin_b, 97.5)),
            "rate_min": float(np.nanmin(est[sel])), "rate_22": float(est[S15_GRID == 22][0]) if ok[S15_GRID == 22][0] else np.nan,
            "rate_60": float(est[S15_GRID == 60][0]) if ok[S15_GRID == 60][0] else np.nan, "p_min_le_30": float((amin_b <= 30).mean()), "max_age_shown": float(S15_GRID[ok].max())}

def s15_plot_curves(ax, curves, colors, title, show_raw=True):
    for (nm, cv), col in zip(curves.items(), colors):
        ax.plot(S15_GRID, cv["est"], color=col, lw=2, label=f"{nm} (min at {cv['age_min']:.0f})")
        ax.fill_between(S15_GRID, cv["lo"], cv["hi"], color=col, alpha=.15)
        if show_raw:
            r = cv["raw"][cv["raw"]["n"] >= 40]
            ax.scatter(r["AGE"], r["rate"], s=10, color=col, alpha=.45)
        ax.scatter([cv["age_min"]], [cv["rate_min"]], marker="v", s=70, color=col, edgecolor="k", zorder=5)
    ax.axhline(BASE_RATE, ls="--", color="k", lw=1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_xlabel("age"); ax.set_ylabel("default rate"); ax.set_title(title); ax.legend(fontsize=8)

s15_curves = {"All clients": s15_age_curve()}
s15_curves_sex = {sx: s15_age_curve(df["SEX_LABEL"] == sx) for sx in ["Male", "Female"]}
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
s15_plot_curves(axes[0], s15_curves, [C_DEF], "15.3.1 Default rate by age: smooth + 95% bootstrap band (dots = raw single-year rates)")
s15_plot_curves(axes[1], s15_curves_sex, [C_OK, C_ACC], "... by sex (triangle = age of minimum risk)")
plt.tight_layout(); savefig("s15_age_curve_sex"); plt.show()
s15_cv = s15_curves["All clients"]
print(f"All clients: minimum smoothed rate {s15_cv['rate_min']:.1%} at age {s15_cv['age_min']:.1f} (95% CI {s15_cv['age_min_lo']:.1f}-{s15_cv['age_min_hi']:.1f}); rate at 22: {s15_cv['rate_22']:.1%}; at 60: {s15_cv['rate_60']:.1%}")

**Observations:**
- The smoothed default rate is lowest at age 31 (95% CI 30-32.5): 19.8%, vs 29.0% at 22 and 27.2% at 60.
- Both arms of the U are similar in size: +9.2 pp on the young side, +7.4 pp on the old side.
- Story use: headline age chart.

#### 15.3.2 The same curve by marital status and by education
If the U-shape is a genuine lifecycle effect it should appear inside each group; if it is a composition artefact (young = single, old = high school) the within-group curves will be flatter than the overall one.

In [ ]:
s15_curves_mar = {m: s15_age_curve(df["MAR_LABEL"] == m) for m in ["Married", "Single"]}
s15_curves_edu = {e: s15_age_curve(df["EDU_LABEL"] == e) for e in ["Graduate school", "University", "High school"]}
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
s15_plot_curves(axes[0], s15_curves_mar, ["#756bb1", "#fd8d3c"], "15.3.2 Default rate by age, by marital status", show_raw=False)
s15_plot_curves(axes[1], s15_curves_edu, [s15_edu_cols[e] for e in s15_curves_edu], "... by education", show_raw=False)
plt.tight_layout(); savefig("s15_age_curve_mar_edu"); plt.show()
s15_all_curves = {**s15_curves, **{f"Sex: {k}": v for k, v in s15_curves_sex.items()}, **{f"Marital: {k}": v for k, v in s15_curves_mar.items()}, **{f"Education: {k}": v for k, v in s15_curves_edu.items()}}
s15_curve_long = pd.concat([pd.DataFrame({"group": nm, "age": S15_GRID, "smooth_rate": cv["est"], "ci_lo": cv["lo"], "ci_hi": cv["hi"]}) for nm, cv in s15_all_curves.items()], ignore_index=True).dropna(subset=["smooth_rate"])
export(s15_curve_long, "s15_age_curves")

**Observations:**
- Image-only cell; numbers come from the minimum-age table in the next cell.
- The U appears inside every marital and education group, so it is not only a mix effect: the minimum is at 35.5 for married (21.8%) and 31 for single clients (18.5%).
- The young arm is similar everywhere (7.8-10.8 pp); the old arm is not (1.2 pp for high school vs 11.4 pp for university).
- Story use: background.

#### 15.3.3 Where is the minimum? Bootstrap CI for the age of minimum risk
For each group: the age (22-60) at which the smoothed rate is lowest, its 95% bootstrap interval, the depth of the U on the young side (rate at 22 minus minimum) and on the old side (rate at 60 minus minimum).

In [ ]:
s15_min_tab = pd.DataFrame([{"group": nm, "n": cv["n"], "age_min": cv["age_min"], "ci_lo": cv["age_min_lo"], "ci_hi": cv["age_min_hi"], "rate_at_min": cv["rate_min"], "rate_at_22": cv["rate_22"],
                             "rate_at_60": cv["rate_60"], "young_arm_pp": 100 * (cv["rate_22"] - cv["rate_min"]), "old_arm_pp": 100 * (cv["rate_60"] - cv["rate_min"]),
                             "P(min <= 30)": cv["p_min_le_30"]} for nm, cv in s15_all_curves.items()])
fig, ax = plt.subplots(figsize=(10, 4.2))
s15_y = np.arange(len(s15_min_tab))[::-1]
ax.errorbar(s15_min_tab["age_min"], s15_y, xerr=[s15_min_tab["age_min"] - s15_min_tab["ci_lo"], s15_min_tab["ci_hi"] - s15_min_tab["age_min"]], fmt="o", color=C_DEF, ecolor=C_NEU, capsize=4)
for s15_yy, s15_r in zip(s15_y, s15_min_tab.itertuples()):
    ax.text(s15_r.ci_hi + .6, s15_yy, f"{s15_r.rate_at_min:.1%} at minimum", va="center", fontsize=8)
ax.set_yticks(s15_y); ax.set_yticklabels(s15_min_tab["group"]); ax.set_xlabel("age of minimum default risk (95% bootstrap CI)"); ax.set_xlim(21, 64)
ax.set_title("15.3.3 Where is the bottom of the U?")
savefig("s15_age_of_min_risk"); plt.show()
print(s15_min_tab.round(3).to_string(index=False))
export(s15_min_tab, "s15_age_of_min_risk")
s15_r0 = s15_min_tab.iloc[0]
add_fact("15.3", "age", f"Default risk is U-shaped in age with its minimum at about {s15_r0['age_min']:.0f} (95% CI {s15_r0['ci_lo']:.0f}-{s15_r0['ci_hi']:.0f}): {s15_r0['rate_at_min']:.1%} there vs {s15_r0['rate_at_22']:.1%} at 22 and {s15_r0['rate_at_60']:.1%} at 60",
         float(s15_r0["age_min"]), "s15_age_curve_sex", 3)
s15_rm, s15_rf = s15_min_tab[s15_min_tab["group"] == "Sex: Male"].iloc[0], s15_min_tab[s15_min_tab["group"] == "Sex: Female"].iloc[0]
add_fact("15.3", "age", f"The U bottoms out at {s15_rm['age_min']:.0f} for men (CI {s15_rm['ci_lo']:.0f}-{s15_rm['ci_hi']:.0f}) and {s15_rf['age_min']:.0f} for women (CI {s15_rf['ci_lo']:.0f}-{s15_rf['ci_hi']:.0f}); "
         f"the old-age arm is {s15_rm['old_arm_pp']:.1f} pp for men and {s15_rf['old_arm_pp']:.1f} pp for women", float(s15_rm["age_min"] - s15_rf["age_min"]), "s15_age_of_min_risk", 1)

**Observations:**
- Women's minimum is at 33 (CI 31-35.5). Men's is at 30.5 but very uncertain (CI 29.5-60), because their curve stays flat after 30.
- The old arm is much larger for women (12.7 pp) than for men (2.1 pp): at 60, women default at 31.1% vs 23.7% for men.
- Graduate-school clients reach their minimum earliest (29.5), high-school clients latest (35).
- Story use: annotate the age chart by sex.

#### 15.3.4 Age x behaviour: how do utilisation, repayment and lateness move over the life cycle?
Six behaviour curves by single-year age (dots = raw means for ages with n ≥ 50, line = same smoother). If the behaviour curves are themselves U-shaped (or inverted), the age effect on default may be nothing more than age-specific behaviour.

In [ ]:
s15_beh = pd.DataFrame({"AGE": df["AGE"], "Mean utilisation": df["AVG_UTIL"].clip(0, 1.5), "Mean share of bills repaid (5 statements, aligned, capped 2)": df["PAY_TO_BILL_5M"].clip(0, 2),
                        "Share late in Sep (status >= 1)": (df["PAY_1"] >= 1).astype(float), "Mean late months (of 6)": df["N_LATE"].astype(float),
                        "Share paid in full or inactive in Sep (status < 0)": (df["PAY_1"] < 0).astype(float), "Geometric-mean credit line (NT$k)": df["DM_LOG_LIMIT"]})
s15_beh_curves = []
fig, axes = plt.subplots(2, 3, figsize=(17, 8.5))
for ax, s15_c in zip(axes.ravel(), [c for c in s15_beh.columns if c != "AGE"]):
    s15_v = s15_beh[["AGE", s15_c]].dropna()
    s15_g = s15_v.groupby("AGE")[s15_c].agg(n="size", tot="sum").reindex(S15_AGES).fillna(0)
    s15_sm = s15_smooth(s15_g["tot"].to_numpy(), s15_g["n"].to_numpy(), S15_AGES, S15_GRID, 7.0, min_eff=20)[0]
    s15_rawm = (s15_g["tot"] / s15_g["n"].where(s15_g["n"] >= 50))
    if "credit line" in s15_c:
        s15_sm, s15_rawm = np.exp(s15_sm) / 1000, np.exp(s15_rawm) / 1000
    ax.scatter(S15_AGES, s15_rawm, s=12, color=C_NEU); ax.plot(S15_GRID[S15_GRID <= 62], s15_sm[S15_GRID <= 62], color=C_OK, lw=2)
    if "Share" in s15_c or "utilisation" in s15_c or "paid" in s15_c: ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(s15_c, fontsize=10); ax.set_xlabel("age"); ax.set_xlim(20, 63)
    s15_beh_curves.append(pd.DataFrame({"measure": s15_c, "age": S15_GRID, "smooth": s15_sm}))
    s15_pick = {int(a): float(s15_sm[S15_GRID == a][0]) for a in (22, 26, 30, 35, 40, 50, 60)}
    print(f"{s15_c:50s} " + " | ".join(f"{a}: {v:.3f}" if v < 10 else f"{a}: {v:,.0f}" for a, v in s15_pick.items()))
fig.suptitle("15.3.4 Behaviour over the life cycle (single-year age)", fontweight="bold"); plt.tight_layout()
savefig("s15_age_behaviour"); plt.show()
s15_beh_curves = pd.concat(s15_beh_curves, ignore_index=True)
export(s15_beh_curves[s15_beh_curves["age"] <= 62], "s15_age_behaviour_curves")

**Observations:**
- Young clients use more of their line: mean utilisation 56% at 22 vs 33% at 35.
- They repay a smaller share of their bills: 19% at 22 vs 40% at 35 (number from the pre-fix run; refresh after the next Kaggle run).
- Lateness is also U-shaped: 28.8% late in Sep at 22, 21.5% at 30, 29.7% at 60. The credit line is an upside-down U: NT$34k at 22, 146k at 35, 109k at 60.
- Story use: these behaviour curves explain the age U.

#### 15.3.5 Is the age effect explained by behaviour?
Two tests. **(a)** The age curve *inside* each September status group: if age only acted through behaviour, the within-group curves would be flat. **(b)** A behaviour-only expectation: give every client the portfolio default rate of their behaviour cell (`DM_BEH_CELL` = Sep status group x limit band, 42 cells; stored as `DM_BEH_EXPECTED`) and average by age. The gap between observed and expected is the age effect that behaviour does **not** explain (band = client-level bootstrap, B = 300).

In [ ]:
df["DM_BEH_CELL"] = df["STATUS_GRP_SEP"].astype(str) + " | " + df["LIMIT_BAND"].astype(str)
S15_CELL_RATE = df.groupby("DM_BEH_CELL")[TARGET].mean()
df["DM_BEH_EXPECTED"] = df["DM_BEH_CELL"].map(S15_CELL_RATE).astype(float)
print(f"behaviour cells: {df['DM_BEH_CELL'].nunique()} | smallest cell n = {df['DM_BEH_CELL'].value_counts().min()} | cell rates span {S15_CELL_RATE.min():.1%} - {S15_CELL_RATE.max():.1%}")
s15_sg5 = df["STATUS_GRP_SEP"].replace({"Late 2m": "Late 2m+", "Late 3m+": "Late 2m+"})
s15_sg_order = ["Inactive", "Paid in full", "Revolving", "Late 1m", "Late 2m+"]
s15_sg_cols = dict(zip(s15_sg_order, ["#969696", "#31a354", "#2b8cbe", "#fd8d3c", "#e34a33"]))
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
s15_within = []
for s15_k in s15_sg_order:
    s15_m = (s15_sg5 == s15_k).to_numpy()
    s15_g = df[s15_m].groupby("AGE")[TARGET].agg(n="size", k="sum").reindex(S15_AGES).fillna(0)
    s15_sm = s15_smooth(s15_g["k"].to_numpy(), s15_g["n"].to_numpy(), S15_AGES, S15_GRID, 7.0)[0]
    s15_keep = (S15_GRID >= 22) & (S15_GRID <= 60)
    axes[0].plot(S15_GRID[s15_keep], s15_sm[s15_keep], color=s15_sg_cols[s15_k], lw=2, label=f"{s15_k} (n={s15_m.sum():,})")
    s15_within.append({"Sep status group": s15_k, "n": int(s15_m.sum()), "rate": df.loc[s15_m, TARGET].mean(), "smooth_at_22": s15_sm[S15_GRID == 22][0], "smooth_min_22_60": np.nanmin(s15_sm[s15_keep]),
                       "age_of_min": S15_GRID[s15_keep][np.nanargmin(s15_sm[s15_keep])], "smooth_at_60": s15_sm[S15_GRID == 60][0], "range_pp": 100 * (np.nanmax(s15_sm[s15_keep]) - np.nanmin(s15_sm[s15_keep]))})
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].set_xlabel("age"); axes[0].set_ylabel("default rate (smoothed)")
axes[0].set_title("15.3.5a Age curve inside each September status group"); axes[0].legend(fontsize=8)
# (b) observed vs behaviour-expected by age
s15_ag = df.groupby("AGE").agg(n=(TARGET, "size"), k=(TARGET, "sum"), e=("DM_BEH_EXPECTED", "sum")).reindex(S15_AGES).fillna(0)
s15_obs_sm = s15_smooth(s15_ag["k"].to_numpy(), s15_ag["n"].to_numpy(), S15_AGES, S15_GRID, 7.0)[0]
s15_exp_sm = s15_smooth(s15_ag["e"].to_numpy(), s15_ag["n"].to_numpy(), S15_AGES, S15_GRID, 7.0)[0]
s15_age_idx = df["AGE"].to_numpy() - 21; s15_yv = df[TARGET].to_numpy(float); s15_ev = df["DM_BEH_EXPECTED"].to_numpy()
s15_bs = np.empty((300, len(S15_GRID)))
for s15_b in range(300):
    s15_ix = S15_RNG.integers(0, len(df), len(df))
    s15_nb = np.bincount(s15_age_idx[s15_ix], minlength=len(S15_AGES)).astype(float)
    s15_db = np.bincount(s15_age_idx[s15_ix], weights=(s15_yv - s15_ev)[s15_ix], minlength=len(S15_AGES))
    s15_bs[s15_b] = s15_smooth(s15_db, s15_nb, S15_AGES, S15_GRID, 7.0)[0]
s15_res_lo, s15_res_hi = np.percentile(s15_bs, [2.5, 97.5], axis=0)
s15_keep = S15_GRID <= 62
axes[1].plot(S15_GRID[s15_keep], s15_obs_sm[s15_keep], color=C_DEF, lw=2, label="observed")
axes[1].plot(S15_GRID[s15_keep], s15_exp_sm[s15_keep], color=C_OK, lw=2, ls="--", label="expected from behaviour mix only")
axes[1].fill_between(S15_GRID[s15_keep], (s15_exp_sm + s15_res_lo)[s15_keep], (s15_exp_sm + s15_res_hi)[s15_keep], color=C_DEF, alpha=.15, label="expected + bootstrap band of (observed - expected)")
axes[1].axhline(BASE_RATE, color="k", ls=":", lw=1); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_xlabel("age")
axes[1].set_title("15.3.5b Observed age curve vs what the behaviour mix alone predicts"); axes[1].legend(fontsize=8)
plt.tight_layout(); savefig("s15_age_vs_behaviour"); plt.show()
print(pd.DataFrame(s15_within).round(3).to_string(index=False))
s15_w = s15_ag["n"].to_numpy(); s15_ok = s15_w >= 30
s15_o_r, s15_e_r = (s15_ag["k"] / s15_ag["n"]).to_numpy()[s15_ok], (s15_ag["e"] / s15_ag["n"]).to_numpy()[s15_ok]
S15_AGE_R2 = 1 - np.average((s15_o_r - s15_e_r) ** 2, weights=s15_w[s15_ok]) / np.average((s15_o_r - np.average(s15_o_r, weights=s15_w[s15_ok])) ** 2, weights=s15_w[s15_ok])
s15_sel = (S15_GRID >= 22) & (S15_GRID <= 60)
s15_obs_rng, s15_res_rng = 100 * (s15_obs_sm[s15_sel].max() - s15_obs_sm[s15_sel].min()), 100 * ((s15_obs_sm - s15_exp_sm)[s15_sel].max() - (s15_obs_sm - s15_exp_sm)[s15_sel].min())
print(f"\nSingle-year age rates: behaviour-only expectation explains {S15_AGE_R2:.1%} of the n-weighted variance of observed rates (ages with n >= 30)")
print(f"Smoothed range 22-60: observed {s15_obs_rng:.1f} pp | residual (observed - expected) {s15_res_rng:.1f} pp")
for s15_a0 in (22, 25, 30, 35, 45, 55, 60):
    s15_j = np.where(S15_GRID == s15_a0)[0][0]
    print(f"  age {s15_a0}: observed {s15_obs_sm[s15_j]:.1%} | expected {s15_exp_sm[s15_j]:.1%} | residual {100 * (s15_obs_sm[s15_j] - s15_exp_sm[s15_j]):+.2f} pp (CI {100 * s15_res_lo[s15_j]:+.2f} to {100 * s15_res_hi[s15_j]:+.2f})")
export(pd.DataFrame({"age": S15_GRID, "observed_smooth": s15_obs_sm, "expected_from_behaviour": s15_exp_sm, "residual_lo": s15_res_lo, "residual_hi": s15_res_hi})[s15_keep], "s15_age_observed_vs_expected")
add_fact("15.3", "age", f"Behaviour mix alone (Sep status x limit band) reproduces {S15_AGE_R2:.0%} of the variance of single-year age default rates; the smoothed age range shrinks from {s15_obs_rng:.1f} pp to {s15_res_rng:.1f} pp once behaviour is netted out",
         float(S15_AGE_R2), "s15_age_vs_behaviour", 3)

**Observations:**
- Behaviour mix (Sep status x limit band) explains 66% of the variation in single-year age default rates; the smoothed age range falls from 9.2 pp to 2.9 pp.
- What is left is small: -0.9 pp at age 30 and +1.3 pp at 45 (both CIs exclude zero); other ages are within noise.
- Inside a status group age still matters for some: among Late 1m clients the rate goes from 30.9% (age 36.5) to 50.4% at 60.
- Story use: "age matters mostly through behaviour".

### 15.4 Why do men default more than women?
Section 3.13 showed the gap and its persistence within strata; Section 10 tested it. Here we **profile** the two sexes on every behaviour and then **decompose** the gap into a part due to different composition (men hold different lines, statuses, utilisation) and a part due to different default rates at equal characteristics.

#### 15.4.1 Love plot: standardised mean differences, men minus women
SMD (standardised mean difference) = (mean men − mean women) / pooled SD. |SMD| < 0.1 is the usual "balanced" threshold from the matching literature. Highly skewed money variables are compared on a signed-log scale.

In [ ]:
def s15_slog(s):
    return np.sign(s) * np.log1p(np.abs(s))
s15_prof = pd.DataFrame({
    "Credit line (log)": df["DM_LOG_LIMIT"], "Line vs demographic peers (resid)": df["DM_LIMIT_RESID"], "Age": df["AGE"],
    "Graduate school": (df["EDU_LABEL"] == "Graduate school").astype(float), "High school": (df["EDU_LABEL"] == "High school").astype(float), "Married": (df["MAR_LABEL"] == "Married").astype(float),
    "Sep status code (PAY_1)": df["PAY_1"], "Late in Sep (>= 1)": (df["PAY_1"] >= 1).astype(float), "Late 2m+ in Sep": (df["PAY_1"] >= 2).astype(float),
    "Paid in full in Sep": (df["PAY_1"] == -1).astype(float), "Inactive in Sep": (df["PAY_1"] == -2).astype(float), "Revolving in Sep": (df["PAY_1"] == 0).astype(float),
    "Late months (of 6)": df["N_LATE"], "Worst delay in 6m": df["MAX_DELAY"], "Longest late streak": df["LATE_STREAK_MAX"], "Paid-in-full months": df["N_PAIDFULL"], "Inactive months": df["N_INACTIVE"],
    "Revolving months": df["N_REVOLVE"], "Status trend (Sep - Apr)": df["DELAY_TREND"],
    "Avg utilisation": df["AVG_UTIL"].clip(0, 1.5), "Max utilisation": df["MAX_UTIL"].clip(0, 2), "Utilisation trend": df["UTIL_TREND"].clip(-0.2, 0.2), "Over limit (any month)": df["OVERLIMIT_ANY"],
    "Share of bills repaid (5 statements, aligned, capped 2)": df["PAY_TO_BILL_5M"].clip(0, 2), "Avg monthly pay ratio": df["AVG_PAY_RATIO"], "Zero-payment months": df["N_ZERO_PAY"],
    "Avg bill (signed log)": s15_slog(df["AVG_BILL"]), "Avg payment (log)": np.log1p(df["AVG_PAY"]), "Avg implied spend (signed log)": s15_slog(df["AVG_SPEND"]), "Bill trend (signed log)": s15_slog(df["BILL_TREND"]),
    "Negative bill (any month)": df["NEG_BILL_ANY"], "DEFAULT": df[TARGET]}).astype(float)
def s15_smd_table(mask_a, mask_b):
    a, b = s15_prof[mask_a], s15_prof[mask_b]
    sd = np.sqrt((a.var() + b.var()) / 2)
    out = pd.DataFrame({"mean_a": a.mean(), "mean_b": b.mean(), "smd": (a.mean() - b.mean()) / sd})
    out["p_welch"] = [stats.ttest_ind(a[c].dropna(), b[c].dropna(), equal_var=False).pvalue for c in s15_prof.columns]
    return out
s15_smd_sex = s15_smd_table(df["SEX"] == 1, df["SEX"] == 2).rename(columns={"mean_a": "mean_male", "mean_b": "mean_female"})
s15_sv = s15_smd_sex.sort_values("smd")
fig, ax = plt.subplots(figsize=(10, 9.5))
ax.hlines(np.arange(len(s15_sv)), 0, s15_sv["smd"], color=C_NEU, lw=1)
ax.scatter(s15_sv["smd"], np.arange(len(s15_sv)), c=np.where(s15_sv["smd"].abs() >= 0.1, C_DEF, C_OK), s=45, zorder=3)
ax.axvline(0, color="k", lw=.8); ax.axvline(0.1, color="k", ls=":", lw=1); ax.axvline(-0.1, color="k", ls=":", lw=1)
ax.set_yticks(np.arange(len(s15_sv))); ax.set_yticklabels(s15_sv.index, fontsize=8)
ax.set_xlabel("standardised mean difference, men − women (red = |SMD| >= 0.1)"); ax.set_title("15.4.1 Love plot: how do men and women differ?")
savefig("s15_love_plot_sex"); plt.show()
print(s15_smd_sex.reindex(s15_smd_sex["smd"].abs().sort_values(ascending=False).index).round(4).to_string())
export(s15_smd_sex.reset_index().rename(columns={"index": "variable"}), "s15_smd_sex")
s15_big = s15_smd_sex.drop(index="DEFAULT")["smd"].abs().sort_values(ascending=False)
add_fact("15.4", "gender", f"Men and women are nearly balanced on behaviour: only {(s15_big >= 0.1).sum()} of {len(s15_big)} profile variables exceed |SMD| 0.1; the largest difference is '{s15_big.index[0]}' "
         f"(SMD {s15_smd_sex.loc[s15_big.index[0], 'smd']:+.2f}), while late-in-Sep differs by SMD {s15_smd_sex.loc['Late in Sep (>= 1)', 'smd']:+.2f}", float(s15_big.iloc[0]), "s15_love_plot_sex", 2)

**Observations:**
- SMD = standardised mean difference (gap in means divided by the spread; below 0.1 is small). The largest are age (+0.19), max utilisation (+0.18), credit line (-0.15, men lower) and average utilisation (+0.14).
- Men are slightly worse on most behaviours: late in Sep 24.0% vs 21.9%, ever over limit 15.1% vs 11.8%, share of bills repaid (5 statements, aligned) 0.33 vs 0.38 (number from the pre-fix run; refresh after the next Kaggle run).
- No difference is large (all SMD below 0.2 in size).
- Story use: background for the gender decomposition.

#### 15.4.2 Decomposition machinery (used for every gap in 15.4-15.6)
Three complementary decompositions of a default-rate gap between group A and group B:

* **Kitagawa (1955)** over behaviour cells (Sep status group x limit band): gap = Σ (w_A − w_B)·r̄ (**composition**: the groups sit in different cells) + Σ (r_A − r_B)·w̄ (**rate**: different default rates inside the same cell). Exact, model-free.
* **Oaxaca-Blinder**, linear probability model with pooled coefficients (statsmodels OLS, group indicator included as in Jann 2008): *explained* = (X̄_A − X̄_B)·β, split by covariate block; *unexplained* = coefficient of the group indicator. Bootstrap CI (B = 200) for the explained part.
* **Fairlie (2005)**, the non-linear analogue: pooled logit, rank-matched samples, covariate blocks switched one at a time in random order (20 draws).

Covariate blocks: Sep status, status history Apr-Aug, credit line, utilisation, repayment share, bill size, plus the *other* demographics. "Explained" can exceed 100% or be negative: then composition alone predicts a larger (or opposite) gap than observed and the rate effect offsets it.

In [ ]:
def s15_kitagawa(mask_a, mask_b, cell="DM_BEH_CELL"):
    ta = df[mask_a].groupby(cell, observed=True)[TARGET].agg(nA="size", kA="sum")
    tb = df[mask_b].groupby(cell, observed=True)[TARGET].agg(nB="size", kB="sum")
    t = ta.join(tb, how="outer").fillna(0.0)
    pooled = (t["kA"] + t["kB"]) / (t["nA"] + t["nB"])
    t["rA"] = np.where(t["nA"] > 0, t["kA"] / t["nA"].replace(0, np.nan), pooled); t["rB"] = np.where(t["nB"] > 0, t["kB"] / t["nB"].replace(0, np.nan), pooled)
    t["wA"], t["wB"] = t["nA"] / t["nA"].sum(), t["nB"] / t["nB"].sum()
    t["composition"] = (t["wA"] - t["wB"]) * (t["rA"] + t["rB"]) / 2
    t["rate_effect"] = (t["rA"] - t["rB"]) * (t["wA"] + t["wB"]) / 2
    return t, float(t["composition"].sum()), float(t["rate_effect"].sum())

def s15_ob_np(Xn, y, g):
    # pooled-coefficient Oaxaca-Blinder (linear probability); returns per-column explained contributions
    A = np.column_stack([np.ones(len(y)), g, Xn])
    beta = np.linalg.lstsq(A, y, rcond=None)[0][2:]
    return (Xn[g == 1].mean(0) - Xn[g == 0].mean(0)) * beta

def s15_decompose(name, mask_a, mask_b, label_a, label_b, drop_block, n_boot=200, n_draw=20):
    mask_a, mask_b = np.asarray(mask_a), np.asarray(mask_b)
    sub = mask_a | mask_b
    cols = [c for c in S15_X.columns if S15_COLBLOCK[c] != drop_block]
    X = S15_X.loc[sub, cols]; X = X.loc[:, X.std() > 0]; cols = list(X.columns)
    blk = np.array([S15_COLBLOCK[c] for c in cols]); blocks = [b for b in s15_blocks if b in set(blk)]
    y = df.loc[sub, TARGET].to_numpy(float); g = mask_a[sub].astype(float); Xn = X.to_numpy()
    rate_a, rate_b = y[g == 1].mean(), y[g == 0].mean(); gap = rate_a - rate_b
    out = {"contrast": name, "label_a": label_a, "label_b": label_b, "n_a": int(g.sum()), "n_b": int((1 - g).sum()), "rate_a": rate_a, "rate_b": rate_b, "gap": gap}
    # Kitagawa on the joint cell and on each margin
    out["kit_cells"], out["kit_comp"], out["kit_rate"] = s15_kitagawa(mask_a, mask_b)
    out["kit_comp_status"] = s15_kitagawa(mask_a, mask_b, "STATUS_GRP_SEP")[1]; out["kit_comp_limit"] = s15_kitagawa(mask_a, mask_b, "LIMIT_BAND")[1]
    # Oaxaca-Blinder with statsmodels OLS (point estimate) and numpy bootstrap
    co, r2, _ = s15_ols(X.assign(**{"Group A indicator": g}), y)
    contrib = (X[g == 1].mean() - X[g == 0].mean()) * co["coef"].reindex(cols)
    out["ob_block"] = contrib.groupby(blk).sum().reindex(blocks)
    out["ob_expl"] = float(contrib.sum()); out["ob_unexpl"] = float(co.loc["Group A indicator", "coef"]); out["ob_unexpl_se"] = float(co.loc["Group A indicator", "se"]); out["lpm_r2"] = r2
    bs_tot, bs_blk = [], []
    for _ in range(n_boot):
        ix = S15_RNG.integers(0, len(y), len(y))
        cb = s15_ob_np(Xn[ix], y[ix], g[ix])
        bs_tot.append(cb.sum()); bs_blk.append([cb[blk == b].sum() for b in blocks])
    out["ob_expl_lo"], out["ob_expl_hi"] = np.percentile(bs_tot, [2.5, 97.5])
    q = np.percentile(np.array(bs_blk), [2.5, 97.5], axis=0)
    out["ob_block_lo"], out["ob_block_hi"] = pd.Series(q[0], index=blocks), pd.Series(q[1], index=blocks)
    # Fairlie: pooled logit, matched samples, random block order
    const = beta = None
    if S15_HAS_SM:
        try:
            lm = sm.Logit(y, sm.add_constant(X, has_constant="add")).fit(disp=0, maxiter=200)
            pr = np.asarray(lm.params, float); const, beta = pr[0], pr[1:]
            if not np.isfinite(pr).all(): const = beta = None
        except Exception:
            const = beta = None
    if beta is None:
        try:
            from sklearn.linear_model import LogisticRegression
            mu, sd = Xn.mean(0), Xn.std(0)
            lr = LogisticRegression(C=1e4, max_iter=3000).fit((Xn - mu) / sd, y)
            beta = lr.coef_[0] / sd; const = float(lr.intercept_[0] - (mu * beta).sum())
        except Exception:
            beta = None
    if beta is not None:
        XA, XB = Xn[g == 1], Xn[g == 0]
        out["fl_expl"] = float(s15_expit(const + XA @ beta).mean() - s15_expit(const + XB @ beta).mean())
        m = min(len(XA), len(XB)); acc = dict.fromkeys(blocks, 0.0)
        for _ in range(n_draw):
            a = XA[S15_RNG.choice(len(XA), m, replace=False)]; b = XB[S15_RNG.choice(len(XB), m, replace=False)]
            a = a[np.argsort(a @ beta)]; cur = b[np.argsort(b @ beta)].copy()
            prev = s15_expit(const + cur @ beta).mean()
            for bname in S15_RNG.permutation(blocks):
                cur[:, blk == bname] = a[:, blk == bname]
                new = s15_expit(const + cur @ beta).mean(); acc[bname] += (new - prev) / n_draw; prev = new
        fl = pd.Series(acc).reindex(blocks)
        out["fl_block"] = fl * (out["fl_expl"] / fl.sum()) if abs(fl.sum()) > 1e-9 else fl      # rescale matched-sample sum to the full-sample explained part
        out["fl_unexpl"] = gap - out["fl_expl"]
    else:
        out["fl_expl"] = out["fl_unexpl"] = np.nan; out["fl_block"] = pd.Series(np.nan, index=blocks)
    return out

def s15_report(r):
    pp = lambda v: f"{100 * v:+.2f} pp"
    sh = lambda v: f"{v / r['gap']:.0%}" if abs(r["gap"]) > 1e-9 else "n/a"
    print(f"{r['contrast']}: {r['label_a']} {r['rate_a']:.2%} (n={r['n_a']:,}) vs {r['label_b']} {r['rate_b']:.2%} (n={r['n_b']:,}) -> gap {pp(r['gap'])}")
    print(f"  Kitagawa (status x limit cells): composition {pp(r['kit_comp'])} ({sh(r['kit_comp'])}) | rate effect {pp(r['kit_rate'])} ({sh(r['kit_rate'])})"
          f" | composition using status only {pp(r['kit_comp_status'])}, limit band only {pp(r['kit_comp_limit'])}")
    print(f"  Oaxaca-Blinder (LPM, R2 {r['lpm_r2']:.3f}): explained {pp(r['ob_expl'])} ({sh(r['ob_expl'])}; 95% CI {pp(r['ob_expl_lo'])} to {pp(r['ob_expl_hi'])}) | unexplained {pp(r['ob_unexpl'])} (se {100 * r['ob_unexpl_se']:.2f})")
    print(f"  Fairlie (logit): explained {pp(r['fl_expl'])} ({sh(r['fl_expl'])}) | unexplained {pp(r['fl_unexpl'])}")
    tab = pd.DataFrame({"OB_pp": 100 * r["ob_block"], "OB_lo": 100 * r["ob_block_lo"], "OB_hi": 100 * r["ob_block_hi"], "Fairlie_pp": 100 * r["fl_block"]})
    tab["OB_share_of_gap"] = r["ob_block"] / r["gap"]
    print(tab.round(3).to_string())

def s15_plot_decomp(r, figname):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1, 1.25]})
    ax = axes[0]
    steps = [(f"{r['label_b']}\nrate", r["rate_b"], C_NEU), ("composition\n(behaviour mix)", r["kit_comp"], C_OK), ("rate effect\n(same cell)", r["kit_rate"], C_DEF), (f"{r['label_a']}\nrate", r["rate_a"], C_NEU)]
    base = 0.0
    for i, (lab, v, col) in enumerate(steps):
        if i in (0, 3):
            ax.bar(i, v, color=col); ax.text(i, v, f"{v:.2%}", ha="center", va="bottom", fontsize=9)
            base = v if i == 0 else base
        else:
            ax.bar(i, v, bottom=base, color=col); ax.text(i, base + v + (0.001 if v >= 0 else -0.001), f"{100 * v:+.2f} pp", ha="center", va="bottom" if v >= 0 else "top", fontsize=9); base += v
    ax.set_xticks(range(4)); ax.set_xticklabels([s[0] for s in steps], fontsize=9); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    lo = min(r["rate_a"], r["rate_b"]) - abs(r["gap"]) * 1.5 - 0.01; ax.set_ylim(max(0, lo), max(r["rate_a"], r["rate_b"]) + abs(r["gap"]) * 1.0 + 0.01)
    ax.set_title(f"Kitagawa waterfall: {r['contrast']}")
    ax = axes[1]
    blocks = list(r["ob_block"].index) + ["UNEXPLAINED"]
    ob = np.append(100 * r["ob_block"].to_numpy(), 100 * r["ob_unexpl"]); fl = np.append(100 * r["fl_block"].to_numpy(), 100 * r["fl_unexpl"])
    err = np.array([np.append(100 * (r["ob_block"] - r["ob_block_lo"]).to_numpy(), 196 * r["ob_unexpl_se"]), np.append(100 * (r["ob_block_hi"] - r["ob_block"]).to_numpy(), 196 * r["ob_unexpl_se"])])
    yy = np.arange(len(blocks))[::-1]
    ax.barh(yy + .2, ob, height=.38, color=C_OK, xerr=np.abs(err), capsize=2, label="Oaxaca-Blinder (linear), 95% CI")
    ax.barh(yy - .2, fl, height=.38, color=C_ACC, label="Fairlie (logit)")
    ax.axvline(0, color="k", lw=.8); ax.set_yticks(yy); ax.set_yticklabels(blocks); ax.set_xlabel("contribution to the gap (percentage points)")
    ax.set_title(f"Which covariates carry the {100 * r['gap']:+.2f} pp gap?"); ax.legend(fontsize=8)
    plt.tight_layout(); savefig(figname); plt.show()
print("decomposition helpers ready")

#### 15.4.3 The gender gap, decomposed
Left: Kitagawa waterfall from the female rate to the male rate. Right: contribution of each covariate block (linear OB with bootstrap CI vs Fairlie logit), with the unexplained remainder as the last bar.

In [ ]:
S15_DEC = {}
S15_DEC["Male vs Female"] = s15_decompose("Male vs Female", df["SEX"] == 1, df["SEX"] == 2, "Male", "Female", "Sex")
s15_r = S15_DEC["Male vs Female"]
s15_report(s15_r)
s15_plot_decomp(s15_r, "s15_decomp_sex")
s15_kc = s15_r["kit_cells"].copy(); s15_kc["total"] = s15_kc["composition"] + s15_kc["rate_effect"]
print("\nCells contributing most to the gap (pp):")
print((s15_kc.assign(composition=100 * s15_kc["composition"], rate_effect=100 * s15_kc["rate_effect"], total=100 * s15_kc["total"])
       .reindex(s15_kc["total"].abs().sort_values(ascending=False).index)[["nA", "nB", "rA", "rB", "wA", "wB", "composition", "rate_effect", "total"]].head(10).round(3).to_string()))
export(s15_kc.reset_index().rename(columns={"index": "DM_BEH_CELL", "nA": "n_male", "nB": "n_female", "rA": "rate_male", "rB": "rate_female", "wA": "share_male", "wB": "share_female"}), "s15_kitagawa_sex_cells")
s15_topb = s15_r["ob_block"].abs().idxmax()
add_fact("15.4", "gender", f"Men default {100 * s15_r['gap']:.1f} pp more than women ({s15_r['rate_a']:.1%} vs {s15_r['rate_b']:.1%}); Kitagawa attributes {s15_r['kit_comp'] / s15_r['gap']:.0%} to behaviour mix (Sep status x limit) "
         f"and {s15_r['kit_rate'] / s15_r['gap']:.0%} to higher rates inside the same cell", float(s15_r["kit_comp"] / s15_r["gap"]), "s15_decomp_sex", 3)
add_fact("15.4", "gender", f"Oaxaca-Blinder explains {s15_r['ob_expl'] / s15_r['gap']:.0%} of the gender gap with covariates (largest block: {s15_topb}, {100 * s15_r['ob_block'][s15_topb]:+.2f} pp); "
         f"{100 * s15_r['ob_unexpl']:+.2f} pp stays unexplained (Fairlie logit: {100 * s15_r['fl_unexpl']:+.2f} pp)", float(s15_r["ob_unexpl"]), "s15_decomp_sex", 2)

**Observations:**
- Men 24.2% vs women 20.8%: a gap of +3.39 pp.
- Kitagawa (splits a gap into "different mix" and "different rate in the same cell"): 1.47 pp (43%) comes from men sitting in riskier status x limit cells, 1.92 pp (57%) from higher rates in the same cells.
- Oaxaca-Blinder (a regression version of the same split): covariates explain 53% (1.80 pp), mainly Apr-Aug status history (+1.00 pp) and utilisation (+0.46 pp); 1.59 pp stays unexplained. Fairlie (logit version) agrees: 56%.
- The biggest cell is revolvers with lines of 50k or less: 17.5% of men vs 11.7% of women.
- Story use: gender waterfall (`s15_kitagawa_sex_cells.csv`).

### 15.5 Do the other demographic gaps decompose the same way?
Same three decompositions for **Married vs Single**, **High school vs Graduate school** and **Young (21-25) vs Prime (31-40)**. The higher-risk group is always group A, so every gap is positive.

In [ ]:
s15_contrasts = [
    ("Married vs Single", df["MAR_LABEL"] == "Married", df["MAR_LABEL"] == "Single", "Married", "Single", "Marital status", "s15_decomp_marriage"),
    ("High school vs Graduate school", df["EDU_LABEL"] == "High school", df["EDU_LABEL"] == "Graduate school", "High school", "Graduate school", "Education", "s15_decomp_education"),
    ("Young 21-25 vs Prime 31-40", df["AGE"].between(21, 25), df["AGE"].between(31, 40), "Young 21-25", "Prime 31-40", "Age", "s15_decomp_age"),
]
for s15_nm, s15_ma, s15_mb, s15_la, s15_lb, s15_drop, s15_fig in s15_contrasts:
    S15_DEC[s15_nm] = s15_decompose(s15_nm, s15_ma, s15_mb, s15_la, s15_lb, s15_drop)
    s15_report(S15_DEC[s15_nm]); s15_plot_decomp(S15_DEC[s15_nm], s15_fig); print()

**Observations:**
- Married vs single (+2.54 pp): behaviour explains little (Oaxaca-Blinder 18%, Kitagawa -3%); married clients default more at the same behaviour.
- High school vs graduate school (+5.92 pp): fully explained (Oaxaca-Blinder 109%, Kitagawa 89%), mainly status history (+2.84 pp) and utilisation (+1.64 pp).
- Young 21-25 vs prime 31-40 (+6.23 pp): 93% explained, by status history (+2.40 pp), credit line (+1.92 pp) and utilisation (+1.48 pp).
- Story use: gap bars sheet.

#### 15.5.1 One summary chart: explained vs unexplained share of every gap
For each gap and each method, the stacked bar splits the gap (pp) into the explained / composition part and the unexplained / rate part. The heatmap shows which covariate block does the explaining (Oaxaca-Blinder, pp). Tableau: stacked bar from `s15_gap_decomposition`.

In [ ]:
s15_rows, s15_crows = [], []
for s15_nm, s15_r in S15_DEC.items():
    for s15_meth, s15_e, s15_u in (("Kitagawa (status x limit cells)", s15_r["kit_comp"], s15_r["kit_rate"]), ("Oaxaca-Blinder (linear)", s15_r["ob_expl"], s15_r["ob_unexpl"]), ("Fairlie (logit)", s15_r["fl_expl"], s15_r["fl_unexpl"])):
        for s15_comp, s15_v in (("Explained (composition)", s15_e), ("Unexplained (rate)", s15_u)):
            s15_rows.append({"contrast": s15_nm, "group_a": s15_r["label_a"], "group_b": s15_r["label_b"], "rate_a": s15_r["rate_a"], "rate_b": s15_r["rate_b"], "gap_pp": 100 * s15_r["gap"],
                             "method": s15_meth, "component": s15_comp, "pp": 100 * s15_v, "share_of_gap": s15_v / s15_r["gap"]})
    for s15_b in s15_r["ob_block"].index:
        s15_crows.append({"contrast": s15_nm, "block": s15_b, "ob_pp": 100 * s15_r["ob_block"][s15_b], "ob_lo_pp": 100 * s15_r["ob_block_lo"][s15_b], "ob_hi_pp": 100 * s15_r["ob_block_hi"][s15_b],
                          "fairlie_pp": 100 * s15_r["fl_block"][s15_b], "ob_share_of_gap": s15_r["ob_block"][s15_b] / s15_r["gap"]})
s15_gapdec, s15_gapcon = pd.DataFrame(s15_rows), pd.DataFrame(s15_crows)
export(s15_gapdec, "s15_gap_decomposition"); export(s15_gapcon, "s15_gap_contributions")
fig, axes = plt.subplots(1, 2, figsize=(18, 6), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
s15_meths = list(s15_gapdec["method"].unique()); s15_names = list(S15_DEC)
s15_yt, s15_yl = [], []
for s15_i, s15_nm in enumerate(s15_names):
    for s15_j, s15_meth in enumerate(s15_meths):
        s15_g = s15_gapdec[(s15_gapdec["contrast"] == s15_nm) & (s15_gapdec["method"] == s15_meth)].set_index("component")["pp"]
        s15_yv0 = -(s15_i * (len(s15_meths) + 1) + s15_j)
        s15_e, s15_u = s15_g["Explained (composition)"], s15_g["Unexplained (rate)"]
        ax.barh(s15_yv0, s15_e, color=C_OK, label="Explained (composition)" if s15_i + s15_j == 0 else None)
        ax.barh(s15_yv0, s15_u, left=s15_e if s15_e * s15_u >= 0 else 0, color=C_DEF, label="Unexplained (rate)" if s15_i + s15_j == 0 else None)
        ax.text(max(s15_e + s15_u, s15_e, s15_u, 0) + .08, s15_yv0, f"expl. {s15_e / (s15_e + s15_u):.0%}" if np.isfinite(s15_e) else "n/a", va="center", fontsize=8)
        s15_yt.append(s15_yv0); s15_yl.append(f"{s15_nm} | {s15_meth.split(' (')[0]}" )
ax.set_yticks(s15_yt); ax.set_yticklabels(s15_yl, fontsize=8); ax.axvline(0, color="k", lw=.8)
ax.set_xlabel("default-rate gap, percentage points (bars of opposite sign start at 0)"); ax.set_title("15.5.1 Every demographic gap: explained vs unexplained"); ax.legend(fontsize=8, loc="lower right")
s15_hm = s15_gapcon.pivot(index="block", columns="contrast", values="ob_pp").reindex(index=[b for b in s15_blocks], columns=s15_names)
sns.heatmap(s15_hm, annot=True, fmt="+.2f", cmap=CMAP_DIV, center=0, ax=axes[1], cbar_kws={"label": "pp of the gap (Oaxaca-Blinder)"})
axes[1].set_title("Which covariate block explains which gap? (blank = block is the contrast itself)"); axes[1].set_xlabel(""); axes[1].set_ylabel("")
plt.setp(axes[1].get_xticklabels(), rotation=15, ha="right", fontsize=8)
plt.tight_layout(); savefig("s15_gap_decomposition_summary"); plt.show()
s15_sumtab = s15_gapdec[s15_gapdec["component"] == "Explained (composition)"].pivot(index="contrast", columns="method", values="share_of_gap").reindex(s15_names)
s15_sumtab.insert(0, "gap_pp", [100 * S15_DEC[k]["gap"] for k in s15_names])
print("Explained share of each gap by method:"); print(s15_sumtab.round(3).to_string())
for s15_nm in s15_names[1:]:
    s15_r = S15_DEC[s15_nm]; s15_topb = s15_r["ob_block"].abs().idxmax()
    add_fact("15.5", "demographic gaps", f"{s15_nm}: gap {100 * s15_r['gap']:+.1f} pp, of which {s15_r['ob_expl'] / s15_r['gap']:.0%} is explained by behaviour and the other demographics (Oaxaca-Blinder; Kitagawa {s15_r['kit_comp'] / s15_r['gap']:.0%}); "
             f"main carrier: {s15_topb} ({100 * s15_r['ob_block'][s15_topb]:+.2f} pp)", float(s15_r["ob_expl"] / s15_r["gap"]), "s15_gap_decomposition_summary", 2)

**Observations:**
- All three methods agree on the ranking: the education and age gaps are about 90-110% explained, the sex gap 43-56%, the marriage gap only -3% to 40%.
- Story use: one comparison bar (`s15_gap_decomposition.csv`).

### 15.6 The education paradox: why does Other/Unknown default so rarely?
Round 1 found that `EDUCATION` codes 0/4/5/6 default at about 7%, a third of the base rate. Three explanations to separate: (a) these clients simply **behave** better (then the behaviour-expected rate will also be low); (b) the codes mark a **special product or segment** (distinct lines, ages, activity patterns); (c) small-sample noise.

#### 15.6.1 Raw codes one by one: observed vs behaviour-expected default rate

In [ ]:
S15_EDU_RAW = {0: "0 (undocumented)", 1: "1 Graduate school", 2: "2 University", 3: "3 High school", 4: "4 Others", 5: "5 (undocumented)", 6: "6 (undocumented)"}
df["DM_EDU_RAW_LABEL"] = df["EDUCATION"].map(S15_EDU_RAW)
def s15_profile(g):
    k, n, e = g[TARGET].sum(), len(g), g["DM_BEH_EXPECTED"].sum()
    lo, hi = wilson_ci(k, n)
    return pd.Series({"n": n, "defaults": k, "rate": k / n, "ci_lo": float(lo), "ci_hi": float(hi), "expected_rate (behaviour)": e / n, "O/E": k / e if e > 0 else np.nan,
                      "median_limit": g["LIMIT_BAL"].median(), "line_vs_peers (x)": float(np.exp(g["DM_LIMIT_RESID"].mean())), "mean_age": g["AGE"].mean(), "share_female": (g["SEX"] == 2).mean(),
                      "share_married": (g["MAR_LABEL"] == "Married").mean(), "share_marriage_other": (g["MAR_LABEL"] == "Other").mean(),
                      "share_late_sep": (g["PAY_1"] >= 1).mean(), "share_never_late": (g["N_LATE"] == 0).mean(), "share_paidfull_or_inactive_sep": (g["PAY_1"] < 0).mean(), "share_revolving_sep": (g["PAY_1"] == 0).mean(),
                      "mean_util": g["AVG_UTIL"].mean(), "median_share_bills_repaid_5m": g["PAY_TO_BILL_5M"].median(), "share_all_bills_zero": (g[BILL_COLS] == 0).all(axis=1).mean(),
                      "median_avg_bill": g["AVG_BILL"].median(), "median_avg_pay": g["AVG_PAY"].median()})
s15_edu_raw = pd.DataFrame({lab: s15_profile(df[df["EDUCATION"] == code]) for code, lab in S15_EDU_RAW.items()}).T
s15_edu_raw.loc["ALL CLIENTS"] = s15_profile(df)
print(s15_edu_raw.T.round(3).to_string())
export(s15_edu_raw.reset_index().rename(columns={"index": "education_code"}), "s15_education_raw_codes")
fig, ax = plt.subplots(figsize=(11, 4.8))
s15_e2 = s15_edu_raw.drop(index="ALL CLIENTS"); s15_x = np.arange(len(s15_e2))
ax.bar(s15_x, s15_e2["rate"], color=[C_NEU if str(i)[0] in "123" else C_ACC for i in s15_e2.index], yerr=[s15_e2["rate"] - s15_e2["ci_lo"], s15_e2["ci_hi"] - s15_e2["rate"]], capsize=4, label="observed default rate (95% CI)")
ax.scatter(s15_x, s15_e2["expected_rate (behaviour)"], marker="D", color=C_DEF, s=70, zorder=4, label="expected from behaviour cell mix")
for s15_i, s15_n in enumerate(s15_e2["n"]):
    ax.text(s15_i, 0.004, f"n={int(s15_n):,}", ha="center", fontsize=8, color="white" if s15_e2["rate"].iloc[s15_i] > 0.04 else "k")
ax.axhline(BASE_RATE, color="k", ls="--", lw=1); ax.set_xticks(s15_x); ax.set_xticklabels(s15_e2.index, rotation=15, ha="right"); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title("15.6.1 Raw EDUCATION codes: observed vs behaviour-expected default rate"); ax.legend(fontsize=8)
savefig("s15_education_raw_codes"); plt.show()

**Observations:**
- Codes 4 and 5 default at 5.7% and 6.4%, code 0 has 0 defaults in 14 clients, code 6 15.7% (n=51).
- Their behaviour predicts much higher rates: 15.3% for code 4 and 18.1% for code 5 (observed / expected 0.37 and 0.36).
- They are rarely late: 10-13% late in Sep vs 22.7% overall; 84-88% never late vs 66%.
- Story use: footnote on education charts.

#### 15.6.2 Status mix and within-status default rates: Other/Unknown vs the documented levels
If the group's low rate were only a better status mix, its rate **inside** each September status group would match everyone else's. Left: status mix (100% stacked). Right: default rate within each status group, Other/Unknown vs the rest (Wilson CIs).

In [ ]:
s15_mix = pd.crosstab(df["EDU_LABEL"], df["STATUS_GRP_SEP"], normalize="index").reindex(index=EDU_ORDER, columns=STATUS_GROUP_ORDER)
s15_is_oth = df["EDU_LABEL"] == "Other/Unknown"
s15_wi = []
for s15_k in STATUS_GROUP_ORDER:
    for s15_nm, s15_m in (("Other/Unknown", s15_is_oth), ("Documented education", ~s15_is_oth)):
        s15_g = df[s15_m & (df["STATUS_GRP_SEP"] == s15_k)]
        s15_lo, s15_hi = wilson_ci(s15_g[TARGET].sum(), max(len(s15_g), 1))
        s15_wi.append({"STATUS_GRP_SEP": s15_k, "group": s15_nm, "n": len(s15_g), "defaults": int(s15_g[TARGET].sum()), "rate": s15_g[TARGET].mean() if len(s15_g) else np.nan, "ci_lo": float(s15_lo), "ci_hi": float(s15_hi)})
s15_wi = pd.DataFrame(s15_wi)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
s15_stc = ["#bdbdbd", "#31a354", "#2b8cbe", "#fdae6b", "#fd8d3c", "#e34a33"]; s15_bot = np.zeros(len(s15_mix))
for s15_k, s15_col in zip(STATUS_GROUP_ORDER, s15_stc):
    axes[0].bar(s15_mix.index, s15_mix[s15_k], bottom=s15_bot, color=s15_col, label=s15_k); s15_bot += s15_mix[s15_k].to_numpy()
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].legend(fontsize=8, bbox_to_anchor=(1.0, 1)); axes[0].set_title("15.6.2 September status mix by education")
for s15_nm, s15_col, s15_off in (("Other/Unknown", C_ACC, -.12), ("Documented education", C_NEU, .12)):
    s15_g = s15_wi[s15_wi["group"] == s15_nm]
    axes[1].errorbar(np.arange(len(s15_g)) + s15_off, s15_g["rate"], yerr=[s15_g["rate"] - s15_g["ci_lo"], s15_g["ci_hi"] - s15_g["rate"]], fmt="o", color=s15_col, capsize=4, label=s15_nm)
axes[1].set_xticks(range(len(STATUS_GROUP_ORDER))); axes[1].set_xticklabels(STATUS_GROUP_ORDER); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_title("Default rate within each September status group"); axes[1].legend(fontsize=8)
plt.tight_layout(); savefig("s15_education_other_status"); plt.show()
print(s15_mix.round(3).to_string()); print(); print(s15_wi.round(3).to_string(index=False))
export(s15_wi, "s15_education_other_within_status")
s15_cmh = []
for s15_k in STATUS_GROUP_ORDER:
    s15_ct = pd.crosstab(s15_is_oth[df["STATUS_GRP_SEP"] == s15_k], df.loc[df["STATUS_GRP_SEP"] == s15_k, TARGET])
    if s15_ct.shape == (2, 2): s15_cmh.append(s15_ct.to_numpy(float))
s15_num = sum(t[1, 1] * t[0, 0] / t.sum() for t in s15_cmh); s15_den = sum(t[1, 0] * t[0, 1] / t.sum() for t in s15_cmh)
s15_or_crude = (df.loc[s15_is_oth, TARGET].mean() / (1 - df.loc[s15_is_oth, TARGET].mean())) / (df.loc[~s15_is_oth, TARGET].mean() / (1 - df.loc[~s15_is_oth, TARGET].mean()))
S15_OR_MH = s15_num / s15_den
print(f"\nOdds ratio of default, Other/Unknown vs documented: crude {s15_or_crude:.2f} | Mantel-Haenszel within Sep status group {S15_OR_MH:.2f}")

**Observations:**
- Other/Unknown has a slightly safer status mix (13% late in Sep vs 23-26% for the documented levels).
- But they default less inside every status group: Late 1m 0 of 32 vs 34.2%, Revolving 7.3% vs 12.9%, Paid in full 3.3% vs 16.9%.
- Odds ratio of default: 0.26 crude and 0.31 within status groups (Mantel-Haenszel, i.e. averaged inside the groups), so the status mix explains little.
- Story use: footnote.

#### 15.6.3 Is the low risk explained by behaviour? Decomposition of Other/Unknown vs University
The same Kitagawa / Oaxaca-Blinder / Fairlie machinery, with University (the largest level) as the comparison group. A large *unexplained* part means the group defaults less **even at equal behaviour** — consistent with a distinct product / customer type rather than with "just better payers".

In [ ]:
S15_DEC_EDU_OTHER = s15_decompose("University vs Other/Unknown", df["EDU_LABEL"] == "University", df["EDU_LABEL"] == "Other/Unknown", "University", "Other/Unknown", "Education")
s15_report(S15_DEC_EDU_OTHER); s15_plot_decomp(S15_DEC_EDU_OTHER, "s15_decomp_edu_other")

**Observations:**
- University 23.7% vs Other/Unknown 7.1%: a gap of 16.7 pp.
- Behaviour explains only 23% (Kitagawa) to 40% (Fairlie); about 10 pp stays unexplained.
- The explained part comes from status history (+3.37 pp) and September status (+2.65 pp).
- Story use: background.

#### 15.6.4 Are codes 4/5/6 a special product or segment? Evidence table
Indicator by indicator: Other/Unknown vs documented education, with a two-sample test (two-proportion z-test for shares, Mann-Whitney for amounts). Look for a *pattern* (premium lines, activity profile, co-occurrence with the undocumented marriage code) rather than any single row.

In [ ]:
s15_ind = {
    "Default rate": df[TARGET].astype(float), "Behaviour-expected default rate": df["DM_BEH_EXPECTED"],
    "Credit line (NT$)": df["LIMIT_BAL"], "Line >= 300k": (df["LIMIT_BAL"] >= 300_000).astype(float), "Line vs demographic peers (log pts)": df["DM_LIMIT_RESID"],
    "Age": df["AGE"].astype(float), "Female": (df["SEX"] == 2).astype(float), "Married": (df["MAR_LABEL"] == "Married").astype(float), "Marriage code undocumented/other (0 or 3)": (df["MAR_LABEL"] == "Other").astype(float),
    "Never late in 6 months": (df["N_LATE"] == 0).astype(float), "Late in Sep": (df["PAY_1"] >= 1).astype(float), "Paid in full or inactive in Sep (status < 0)": (df["PAY_1"] < 0).astype(float),
    "Inactive all 6 months": (df["N_INACTIVE"] == 6).astype(float), "All six bills zero": (df[BILL_COLS] == 0).all(axis=1).astype(float), "Any negative bill (credit balance)": df["NEG_BILL_ANY"].astype(float),
    "Avg utilisation": df["AVG_UTIL"], "Share of bills repaid (5 statements, aligned)": df["PAY_TO_BILL_5M"], "Avg bill (NT$)": df["AVG_BILL"], "Avg payment (NT$)": df["AVG_PAY"], "Avg implied spend (NT$)": df["AVG_SPEND"],
    "Over limit (any month)": df["OVERLIMIT_ANY"].astype(float)}
s15_rows = []
for s15_nm, s15_s in s15_ind.items():
    s15_a, s15_b = s15_s[s15_is_oth].dropna(), s15_s[~s15_is_oth].dropna()
    s15_binary = set(np.unique(s15_s.dropna())) <= {0.0, 1.0}
    if s15_binary:
        s15_pp = (s15_a.sum() + s15_b.sum()) / (len(s15_a) + len(s15_b)); s15_z = (s15_a.mean() - s15_b.mean()) / np.sqrt(s15_pp * (1 - s15_pp) * (1 / len(s15_a) + 1 / len(s15_b)))
        s15_p = 2 * stats.norm.sf(abs(s15_z)); s15_va, s15_vb, s15_stat = s15_a.mean(), s15_b.mean(), "share"
    else:
        s15_p = stats.mannwhitneyu(s15_a, s15_b, alternative="two-sided").pvalue; s15_va, s15_vb, s15_stat = s15_a.median(), s15_b.median(), "median"
    s15_rows.append({"indicator": s15_nm, "statistic": s15_stat, "Other/Unknown": s15_va, "Documented": s15_vb, "ratio": s15_va / s15_vb if s15_vb not in (0, np.nan) else np.nan, "p_value": s15_p,
                     "reading": "distinct" if s15_p < 0.001 else ("weakly distinct" if s15_p < 0.05 else "no difference")})
s15_evid = pd.DataFrame(s15_rows)
print(f"Other/Unknown n = {int(s15_is_oth.sum())} (codes: " + ", ".join(f"{k}: {v}" for k, v in df.loc[s15_is_oth, "EDUCATION"].value_counts().sort_index().items()) + ")")
print(s15_evid.to_string(index=False, float_format=lambda v: f"{v:,.4f}"))
export(s15_evid, "s15_education_other_evidence")
s15_oth = s15_edu_raw.loc[[S15_EDU_RAW[c] for c in (0, 4, 5, 6)]]
s15_o_rate = df.loc[s15_is_oth, TARGET].mean(); s15_o_exp = df.loc[s15_is_oth, "DM_BEH_EXPECTED"].mean()
s15_beh_share = (BASE_RATE - s15_o_exp) / (BASE_RATE - s15_o_rate)
print(f"\nOther/Unknown: observed {s15_o_rate:.1%} | behaviour-expected {s15_o_exp:.1%} | base {BASE_RATE:.1%} -> behaviour mix explains {s15_beh_share:.0%} of the shortfall vs base; O/E = {s15_o_rate / s15_o_exp:.2f}")
add_fact("15.6", "education paradox", f"Other/Unknown education (n={int(s15_is_oth.sum())}) defaults at {s15_o_rate:.1%}; its behaviour mix predicts {s15_o_exp:.1%}, so behaviour explains only {s15_beh_share:.0%} of the shortfall vs the {BASE_RATE:.1%} base "
         f"(Mantel-Haenszel odds ratio within Sep status {S15_OR_MH:.2f})", float(s15_o_rate / s15_o_exp), "s15_education_raw_codes", 3)
s15_el = s15_evid.set_index("indicator")
add_fact("15.6", "education paradox", f"The Other/Unknown group looks like a distinct segment: median line NT${s15_el.loc['Credit line (NT$)', 'Other/Unknown'] / 1000:,.0f}k vs {s15_el.loc['Credit line (NT$)', 'Documented'] / 1000:,.0f}k, "
         f"{s15_el.loc['Never late in 6 months', 'Other/Unknown']:.0%} never late vs {s15_el.loc['Never late in 6 months', 'Documented']:.0%}, yet utilisation {s15_el.loc['Avg utilisation', 'Other/Unknown']:.0%} vs {s15_el.loc['Avg utilisation', 'Documented']:.0%}",
         float(s15_el.loc["Credit line (NT$)", "ratio"]), "s15_education_other_status", 2)

**Observations:**
- Other/Unknown clients differ on money: larger lines (median 170k vs 140k), bills (27.6k vs 20.9k), payments (3.4k vs 2.4k) and spend (6.5k vs 3.6k), and more are inactive all six months (11.1% vs 7.0%).
- They do not differ on age, sex, marriage, utilisation, share of bills repaid (5 statements) or line vs peers.
- Observed 7.1% vs behaviour-expected 17.5% vs base 22.1%: behaviour explains 31% of the shortfall.
- They look like a real low-risk segment (bigger spenders who rarely fall late), which fits Section 1's reading that this is not a data error; why they got these codes is unknown.
- Story use: footnote.

### 15.7 The household story: marriage x sex x age
The classic reading is that married men carry household debt and single young men are the riskiest. Four groups (married / single x men / women; "Other" marital status left out, n is too small) followed across six age bands: default rate with CIs, median line and mean utilisation.

In [ ]:
S15_AGE6 = ["21-25", "26-30", "31-35", "36-40", "41-50", "51+"]
df["DM_AGE6"] = pd.cut(df["AGE"], [20, 25, 30, 35, 40, 50, 80], labels=S15_AGE6)
s15_hh = df[df["MAR_LABEL"].isin(["Married", "Single"])].copy()
s15_hh["DM_HH_GROUP"] = s15_hh["MAR_LABEL"].astype(str) + " " + s15_hh["SEX_LABEL"].astype(str).str.lower().replace({"male": "men", "female": "women"})
df["DM_HH_GROUP"] = s15_hh["DM_HH_GROUP"].reindex(df.index).fillna("Other marital status")
s15_hh_tab = rate_table(s15_hh, ["DM_HH_GROUP", "DM_AGE6"]).merge(
    s15_hh.groupby(["DM_HH_GROUP", "DM_AGE6"], observed=True).agg(median_limit=("LIMIT_BAL", "median"), mean_util=("AVG_UTIL", "mean"), share_late_sep=("PAY_1", lambda s: (s >= 1).mean()),
                                                                 expected_rate=("DM_BEH_EXPECTED", "mean")).reset_index(), on=["DM_HH_GROUP", "DM_AGE6"])
S15_HH_ORDER = ["Married men", "Single men", "Married women", "Single women"]; s15_hh_cols = dict(zip(S15_HH_ORDER, ["#08519c", "#6baed6", "#a63603", "#fd8d3c"]))
fig, axes = plt.subplots(1, 3, figsize=(20, 5.3))
for s15_i, s15_k in enumerate(S15_HH_ORDER):
    s15_g = s15_hh_tab[s15_hh_tab["DM_HH_GROUP"] == s15_k].set_index("DM_AGE6").reindex(S15_AGE6); s15_x = np.arange(len(S15_AGE6)) + (s15_i - 1.5) * .06
    s15_okn = (s15_g["n"] >= 40).to_numpy()
    axes[0].errorbar(s15_x[s15_okn], s15_g["rate"][s15_okn], yerr=[(s15_g["rate"] - s15_g["ci_lo"])[s15_okn], (s15_g["ci_hi"] - s15_g["rate"])[s15_okn]], color=s15_hh_cols[s15_k], marker="o", capsize=3, lw=2, label=s15_k)
    axes[1].plot(s15_x[s15_okn], s15_g["median_limit"][s15_okn] / 1000, color=s15_hh_cols[s15_k], marker="o", lw=2, label=s15_k)
    axes[2].plot(s15_x[s15_okn], s15_g["mean_util"][s15_okn], color=s15_hh_cols[s15_k], marker="o", lw=2, label=s15_k)
axes[0].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1))
for ax, s15_ttl in zip(axes, ["15.7 Default rate (95% CI)", "Median credit line (NT$ thousand)", "Mean utilisation"]):
    ax.set_xticks(range(len(S15_AGE6))); ax.set_xticklabels(S15_AGE6); ax.set_xlabel("age band"); ax.set_title(s15_ttl + " by marital status x sex"); ax.legend(fontsize=8)
plt.tight_layout(); savefig("s15_household_triple"); plt.show()
for s15_c, s15_f in (("n", "{:,.0f}"), ("rate", "{:.3f}"), ("median_limit", "{:,.0f}"), ("mean_util", "{:.3f}"), ("expected_rate", "{:.3f}")):
    print(f"\n{s15_c}:"); print(s15_hh_tab.pivot(index="DM_HH_GROUP", columns="DM_AGE6", values=s15_c).reindex(index=S15_HH_ORDER, columns=S15_AGE6).to_string(float_format=lambda v, f=s15_f: f.format(v)))
export(s15_hh_tab, "s15_household")
s15_hh_big = s15_hh_tab[s15_hh_tab["n"] >= 100]
s15_hi, s15_lo = s15_hh_big.loc[s15_hh_big["rate"].idxmax()], s15_hh_big.loc[s15_hh_big["rate"].idxmin()]
add_fact("15.7", "household", f"Across marital status x sex x age the riskiest group is {s15_hi['DM_HH_GROUP'].lower()} aged {s15_hi['DM_AGE6']} ({s15_hi['rate']:.1%}, n={int(s15_hi['n']):,}) and the safest {s15_lo['DM_HH_GROUP'].lower()} aged {s15_lo['DM_AGE6']} ({s15_lo['rate']:.1%}, n={int(s15_lo['n']):,})",
         float(s15_hi["rate"] - s15_lo["rate"]), "s15_household_triple", 2)

**Observations:**
- Married men are the riskiest group in most age bands (27.4% at 26-30, 29.3% at 51+; 33.9% at 21-25 but n=62).
- Single women are the safest (15.6% at 31-35).
- Behaviour-expected rates are flatter than observed ones: married men 21-25 expected 24.3% vs observed 33.9%.
- Single men past 40 have small lines (60-80k) and high utilisation (48-53%), yet default only 20-24%.
- Story use: four-line household chart (`s15_household.csv`).

#### 15.7.1 Does the marriage gap in default survive inside each sex x age band?
Married − Single default-rate difference (pp, 95% CI) for men and for women in each age band, next to the same difference for the **behaviour-expected** rate. If the two move together, marriage acts through behaviour.

In [ ]:
s15_rows = []
for s15_sx in ["Male", "Female"]:
    for s15_ab in S15_AGE6:
        s15_g = s15_hh[(s15_hh["SEX_LABEL"] == s15_sx) & (s15_hh["DM_AGE6"] == s15_ab)]
        s15_a, s15_b = s15_g[s15_g["MAR_LABEL"] == "Married"], s15_g[s15_g["MAR_LABEL"] == "Single"]
        if min(len(s15_a), len(s15_b)) < 40: continue
        s15_p1, s15_p2 = s15_a[TARGET].mean(), s15_b[TARGET].mean(); s15_se = np.sqrt(s15_p1 * (1 - s15_p1) / len(s15_a) + s15_p2 * (1 - s15_p2) / len(s15_b))
        s15_rows.append({"sex": s15_sx, "age_band": s15_ab, "n_married": len(s15_a), "n_single": len(s15_b), "rate_married": s15_p1, "rate_single": s15_p2, "gap_pp": 100 * (s15_p1 - s15_p2),
                         "lo_pp": 100 * (s15_p1 - s15_p2 - 1.96 * s15_se), "hi_pp": 100 * (s15_p1 - s15_p2 + 1.96 * s15_se),
                         "expected_gap_pp": 100 * (s15_a["DM_BEH_EXPECTED"].mean() - s15_b["DM_BEH_EXPECTED"].mean()), "limit_ratio_married_single": s15_a["LIMIT_BAL"].median() / s15_b["LIMIT_BAL"].median()})
s15_mgap = pd.DataFrame(s15_rows)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), sharey=True)
for ax, s15_sx in zip(axes, ["Male", "Female"]):
    s15_g = s15_mgap[s15_mgap["sex"] == s15_sx]; s15_x = np.arange(len(s15_g))
    ax.errorbar(s15_x, s15_g["gap_pp"], yerr=[s15_g["gap_pp"] - s15_g["lo_pp"], s15_g["hi_pp"] - s15_g["gap_pp"]], fmt="o", color=C_DEF, capsize=4, label="observed gap")
    ax.scatter(s15_x, s15_g["expected_gap_pp"], marker="D", color=C_OK, label="gap expected from behaviour mix", zorder=4)
    ax.axhline(0, color="k", lw=.8); ax.set_xticks(s15_x); ax.set_xticklabels(s15_g["age_band"]); ax.set_xlabel("age band")
    ax.set_title(f"15.7.1 Married − Single default gap: {'men' if s15_sx == 'Male' else 'women'}"); ax.legend(fontsize=8)
axes[0].set_ylabel("percentage points")
plt.tight_layout(); savefig("s15_marriage_gap_by_sex_age"); plt.show()
print(s15_mgap.round(3).to_string(index=False))
export(s15_mgap, "s15_marriage_gap_by_sex_age")
s15_crude_gap = 100 * (s15_hh.loc[s15_hh["MAR_LABEL"] == "Married", TARGET].mean() - s15_hh.loc[s15_hh["MAR_LABEL"] == "Single", TARGET].mean())
s15_wgap = np.average(s15_mgap["gap_pp"], weights=1 / (1 / s15_mgap["n_married"] + 1 / s15_mgap["n_single"]))
print(f"\nCrude Married − Single gap {s15_crude_gap:+.2f} pp | precision-weighted mean gap within sex x age band {s15_wgap:+.2f} pp | cells with CI excluding 0: {int(((s15_mgap['lo_pp'] > 0) | (s15_mgap['hi_pp'] < 0)).sum())} of {len(s15_mgap)}")
add_fact("15.7", "household", f"The crude Married − Single default gap of {s15_crude_gap:+.1f} pp becomes {s15_wgap:+.1f} pp on average inside sex x age bands", float(s15_wgap), "s15_marriage_gap_by_sex_age", 2)

**Observations:**
- Inside sex x age bands, married clients default 3.45 pp more than singles on average (crude gap +2.54 pp); 5 of 12 cells exclude zero.
- The gap is clearest for women under 36 (+4.7 to +5.4 pp) and for men aged 26-30 (+7.0 pp) and 51+ (+9.3 pp).
- For men the behaviour-expected gap is often negative while the real gap is positive (51+: -1.2 vs +9.3 pp): married men default more than their behaviour predicts.
- Story use: marriage-gap dot plot.

#### 15.7.2 Do married clients hold systematically larger lines at the same age and education?
`LIMIT_BAL` *includes family / supplementary credit*. If married clients' lines embed a spouse's supplementary card, they should be larger than those of single clients **of the same age, sex and education**. Test: OLS of log limit on a married indicator with single-year-age, education and sex fixed effects (robust SE); the coefficient is the "marriage premium" in %. Repeated by sex and by age band, and cross-checked with exact matched cells (age x education x sex).

In [ ]:
def s15_marriage_premium(data):
    fe = [pd.get_dummies(data["AGE"].astype(int).astype(str), dtype=float, prefix="age").iloc[:, 1:]]
    if data["EDU_LABEL"].nunique() > 1: fe.append(pd.get_dummies(data["EDU_LABEL"].astype(str), dtype=float, prefix="edu").iloc[:, 1:])
    if data["SEX"].nunique() > 1: fe.append(pd.DataFrame({"female": (data["SEX"] == 2).astype(float)}, index=data.index))
    X = pd.concat([pd.DataFrame({"married": (data["MAR_LABEL"] == "Married").astype(float)}, index=data.index)] + fe, axis=1)
    X = X.loc[:, X.std() > 0]
    co, r2, _ = s15_ols(X, data["DM_LOG_LIMIT"])
    b, se = co.loc["married", "coef"], co.loc["married", "se"]
    return {"n": len(data), "n_married": int((data["MAR_LABEL"] == "Married").sum()), "premium": np.exp(b) - 1, "lo": np.exp(b - 1.96 * se) - 1, "hi": np.exp(b + 1.96 * se) - 1, "r2": r2,
            "raw_median_ratio": data.loc[data["MAR_LABEL"] == "Married", "LIMIT_BAL"].median() / data.loc[data["MAR_LABEL"] == "Single", "LIMIT_BAL"].median() - 1}
s15_rows = [{"scope": "All (married + single)", "sex": "Both", "age_band": "All", **s15_marriage_premium(s15_hh)}]
for s15_sx in ["Male", "Female"]:
    s15_rows.append({"scope": f"{s15_sx} only", "sex": s15_sx, "age_band": "All", **s15_marriage_premium(s15_hh[s15_hh["SEX_LABEL"] == s15_sx])})
    for s15_ab in S15_AGE6:
        s15_g = s15_hh[(s15_hh["SEX_LABEL"] == s15_sx) & (s15_hh["DM_AGE6"] == s15_ab)]
        if (s15_g["MAR_LABEL"] == "Married").sum() >= 40 and (s15_g["MAR_LABEL"] == "Single").sum() >= 40:
            s15_rows.append({"scope": f"{s15_sx}, {s15_ab}", "sex": s15_sx, "age_band": s15_ab, **s15_marriage_premium(s15_g)})
s15_prem = pd.DataFrame(s15_rows)
fig, ax = plt.subplots(figsize=(11, 4.8))
for s15_sx, s15_col, s15_off in (("Male", C_OK, -.08), ("Female", C_ACC, .08)):
    s15_g = s15_prem[(s15_prem["sex"] == s15_sx) & (s15_prem["age_band"] != "All")]
    s15_x = np.array([S15_AGE6.index(a) for a in s15_g["age_band"]]) + s15_off
    ax.errorbar(s15_x, s15_g["premium"], yerr=[s15_g["premium"] - s15_g["lo"], s15_g["hi"] - s15_g["premium"]], fmt="o-", color=s15_col, capsize=4, label=f"{s15_sx}")
ax.axhline(0, color="k", lw=.8); ax.axhline(s15_prem["premium"].iloc[0], color=C_DEF, ls="--", lw=1, label=f"overall premium {s15_prem['premium'].iloc[0]:+.1%}")
ax.set_xticks(range(len(S15_AGE6))); ax.set_xticklabels(S15_AGE6); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("age band"); ax.set_ylabel("married vs single credit line, same age / education")
ax.set_title("15.7.2 The marriage premium in the credit line (OLS with age-year and education fixed effects, 95% CI)"); ax.legend(fontsize=8)
savefig("s15_marriage_premium"); plt.show()
print(s15_prem.round(4).to_string(index=False))
export(s15_prem, "s15_marriage_premium")
# exact matched cells: same single-year age x education x sex
s15_mc = s15_hh.groupby(["AGE", "EDU_LABEL", "SEX_LABEL", "MAR_LABEL"], observed=True)["LIMIT_BAL"].agg(n="size", median="median", logmean=lambda s: np.log(s).mean()).unstack("MAR_LABEL")
s15_mc = s15_mc[(s15_mc[("n", "Married")] >= 15) & (s15_mc[("n", "Single")] >= 15)]
s15_mw = 1 / (1 / s15_mc[("n", "Married")] + 1 / s15_mc[("n", "Single")])
s15_match_prem = float(np.exp(np.average(s15_mc[("logmean", "Married")] - s15_mc[("logmean", "Single")], weights=s15_mw)) - 1)
s15_match_share = float(np.average(s15_mc[("median", "Married")] > s15_mc[("median", "Single")], weights=s15_mw)); s15_match_tie = float(np.average(s15_mc[("median", "Married")] == s15_mc[("median", "Single")], weights=s15_mw))
print(f"\nExact matched cells (age x education x sex with >= 15 married and >= 15 single): {len(s15_mc)} cells, {int(s15_mc['n'].sum().sum()):,} clients")
print(f"  precision-weighted geometric-mean premium {s15_match_prem:+.1%} | married median higher in {s15_match_share:.0%} of weighted cells, equal in {s15_match_tie:.0%}")
s15_p0 = s15_prem.iloc[0]; s15_pm, s15_pf = s15_prem[s15_prem["scope"] == "Male only"].iloc[0], s15_prem[s15_prem["scope"] == "Female only"].iloc[0]
add_fact("15.7", "household", f"At the same age, sex and education married clients hold a {s15_p0['premium']:+.0%} credit line (95% CI {s15_p0['lo']:+.0%} to {s15_p0['hi']:+.0%}; men {s15_pm['premium']:+.0%}, women {s15_pf['premium']:+.0%}; raw median gap {s15_p0['raw_median_ratio']:+.0%}), "
         f"consistent with LIMIT_BAL including family / supplementary credit", float(s15_p0["premium"]), "s15_marriage_premium", 2)

**Observations:**
- At the same age, sex and education, married clients hold an 8% larger line (CI +6% to +11%).
- This comes from men: +29% overall and +48-49% from age 41. For women it is -5% overall, negative under 36 and positive from 41.
- In exactly matched cells the premium is only +2.8%.
- Story use: background; fits the idea that `LIMIT_BAL` includes family / supplementary cards for older married men.

### 15.8 What is left of each demographic gap once behaviour is held equal? Direct and indirect standardisation
The epidemiologist's answer to confounding, and the cleanest Tableau story sheet of this section.

* **Indirect**: expected defaults of a group if each member defaulted at the portfolio rate of their behaviour cell (Sep status group x limit band). O/E (observed ÷ expected defaults) is the standardised default ratio (SDR); **adjusted rate = SDR x base rate**. CI from the Poisson approximation on observed defaults.
* **Direct**: the group's own cell-specific rates applied to the **portfolio's** cell mix (cells the group does not occupy borrow the portfolio rate; `coverage` reports the share of portfolio weight the group actually covers).
* Sensitivity: a richer cell definition (status x limit band x number of late months).

`s15_crude_vs_adjusted` is exported for a dumbbell chart: crude vs adjusted rate for every demographic level.

In [ ]:
df["DM_BEH_CELL_RICH"] = df["DM_BEH_CELL"] + " | late " + df["N_LATE_BAND"].astype(str)
S15_CELL_RATE_RICH = df.groupby("DM_BEH_CELL_RICH")[TARGET].mean()
df["DM_BEH_EXPECTED_RICH"] = df["DM_BEH_CELL_RICH"].map(S15_CELL_RATE_RICH).astype(float)
S15_CELL_W = df["DM_BEH_CELL"].value_counts(normalize=True)
def s15_standardise(col, order):
    rows = []
    for lev in order:
        g = df[df[col] == lev]
        if len(g) == 0: continue
        n, k = len(g), g[TARGET].sum(); e = g["DM_BEH_EXPECTED"].sum(); e2 = g["DM_BEH_EXPECTED_RICH"].sum()
        own = g.groupby("DM_BEH_CELL")[TARGET].mean().reindex(S15_CELL_W.index)
        direct = float((S15_CELL_W * own.fillna(S15_CELL_RATE.reindex(S15_CELL_W.index))).sum())
        sdr = k / e; lo, hi = wilson_ci(k, n)
        rows.append({"demographic": col, "level": lev, "n": n, "defaults": int(k), "crude_rate": k / n, "crude_lo": float(lo), "crude_hi": float(hi), "expected_rate": e / n, "sdr": sdr,
                     "adjusted_rate": sdr * BASE_RATE, "adj_lo": sdr * np.exp(-1.96 / np.sqrt(k)) * BASE_RATE, "adj_hi": sdr * np.exp(1.96 / np.sqrt(k)) * BASE_RATE,
                     "direct_rate": direct, "direct_coverage": float(S15_CELL_W[own.notna().to_numpy()].sum()), "adjusted_rate_rich": k / e2 * BASE_RATE})
    return pd.DataFrame(rows)
S15_STD_DIMS = [("SEX_LABEL", ["Male", "Female"]), ("EDU_LABEL", EDU_ORDER), ("MAR_LABEL", MAR_ORDER), ("AGE_BAND", AGE_LABELS), ("DM_LIFE_STAGE", S15_STAGE_ORDER)]
s15_std = pd.concat([s15_standardise(c, o) for c, o in S15_STD_DIMS], ignore_index=True)
s15_std["crude_gap_pp"] = 100 * (s15_std["crude_rate"] - BASE_RATE); s15_std["adjusted_gap_pp"] = 100 * (s15_std["adjusted_rate"] - BASE_RATE)
s15_std["label"] = s15_std["demographic"].map({"SEX_LABEL": "Sex", "EDU_LABEL": "Education", "MAR_LABEL": "Marital", "AGE_BAND": "Age", "DM_LIFE_STAGE": "Life stage"}) + ": " + s15_std["level"].astype(str)
s15_std["base_rate"] = BASE_RATE
print(s15_std[["label", "n", "crude_rate", "expected_rate", "sdr", "adjusted_rate", "adj_lo", "adj_hi", "direct_rate", "direct_coverage", "adjusted_rate_rich", "crude_gap_pp", "adjusted_gap_pp"]].round(4).to_string(index=False))
export(s15_std, "s15_crude_vs_adjusted")

Dumbbell chart: grey dot = crude rate, red dot = behaviour-adjusted rate (indirect, with CI), small blue tick = direct standardisation. The shorter the distance of the red dot from the dashed base-rate line, the more of that group's gap was behaviour.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 0.33 * len(s15_std) + 1.5))
s15_y = np.arange(len(s15_std))[::-1]
ax.hlines(s15_y, s15_std["crude_rate"], s15_std["adjusted_rate"], color=C_NEU, lw=2, zorder=1)
ax.errorbar(s15_std["adjusted_rate"], s15_y, xerr=[s15_std["adjusted_rate"] - s15_std["adj_lo"], s15_std["adj_hi"] - s15_std["adjusted_rate"]], fmt="none", ecolor=C_DEF, alpha=.5, capsize=2, zorder=2)
ax.scatter(s15_std["crude_rate"], s15_y, color="#525252", s=55, zorder=3, label="crude rate")
ax.scatter(s15_std["adjusted_rate"], s15_y, color=C_DEF, s=55, zorder=4, label="adjusted to portfolio behaviour mix (indirect, 95% CI)")
ax.scatter(s15_std["direct_rate"], s15_y, color=C_OK, marker="|", s=120, zorder=4, label="direct standardisation")
ax.axvline(BASE_RATE, color="k", ls="--", lw=1)
for s15_yy in s15_y[np.r_[False, (s15_std["demographic"].to_numpy()[1:] != s15_std["demographic"].to_numpy()[:-1])]]:
    ax.axhline(s15_yy + .5, color=C_NEU, lw=.6)
ax.set_yticks(s15_y); ax.set_yticklabels(s15_std["label"], fontsize=8); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("default rate")
ax.set_title("15.8 Demographics fade once behaviour is held equal: crude vs behaviour-adjusted default rates"); ax.legend(fontsize=8, loc="lower right")
savefig("s15_crude_vs_adjusted"); plt.show()
s15_fade = s15_std.groupby("demographic", sort=False).apply(lambda g: pd.Series({
    "levels": len(g), "crude_range_pp": 100 * (g["crude_rate"].max() - g["crude_rate"].min()), "adjusted_range_pp": 100 * (g["adjusted_rate"].max() - g["adjusted_rate"].min()),
    "crude_wsd_pp": 100 * np.sqrt(np.average((g["crude_rate"] - BASE_RATE) ** 2, weights=g["n"])), "adjusted_wsd_pp": 100 * np.sqrt(np.average((g["adjusted_rate"] - BASE_RATE) ** 2, weights=g["n"])),
    "adjusted_rich_wsd_pp": 100 * np.sqrt(np.average((g["adjusted_rate_rich"] - BASE_RATE) ** 2, weights=g["n"]))})).reset_index()
s15_fade["share_of_spread_removed"] = 1 - s15_fade["adjusted_wsd_pp"] / s15_fade["crude_wsd_pp"]
s15_fade["share_removed_rich_cells"] = 1 - s15_fade["adjusted_rich_wsd_pp"] / s15_fade["crude_wsd_pp"]
print("How much of each demographic's spread disappears? (wsd = n-weighted SD of level rates around the base rate)")
print(s15_fade.round(3).to_string(index=False))
export(s15_fade, "s15_demographic_fade")
s15_still = s15_std[(s15_std["adj_lo"] > BASE_RATE) | (s15_std["adj_hi"] < BASE_RATE)]
print(f"\nLevels whose adjusted CI still excludes the base rate ({len(s15_still)} of {len(s15_std)}):")
print(s15_still[["label", "n", "crude_rate", "adjusted_rate", "adj_lo", "adj_hi"]].round(4).to_string(index=False))
s15_f4 = s15_fade[s15_fade["demographic"] != "DM_LIFE_STAGE"]
s15_fmax = s15_f4.loc[s15_f4["share_of_spread_removed"].idxmax()]; s15_fmin = s15_f4.loc[s15_f4["share_of_spread_removed"].idxmin()]
add_fact("15.8", "demographics fade", f"Holding behaviour equal (Sep status x limit band) removes {s15_f4['share_of_spread_removed'].min():.0%}-{s15_f4['share_of_spread_removed'].max():.0%} of the spread in demographic default rates: "
         f"most for {s15_fmax['demographic']} ({s15_fmax['crude_wsd_pp']:.1f} -> {s15_fmax['adjusted_wsd_pp']:.1f} pp), least for {s15_fmin['demographic']} ({s15_fmin['crude_wsd_pp']:.1f} -> {s15_fmin['adjusted_wsd_pp']:.1f} pp)",
         float(s15_f4["share_of_spread_removed"].mean()), "s15_crude_vs_adjusted", 3)
s15_sm_, s15_sf_ = s15_std[s15_std["label"] == "Sex: Male"].iloc[0], s15_std[s15_std["label"] == "Sex: Female"].iloc[0]
add_fact("15.8", "demographics fade", f"The male-female gap shrinks from {100 * (s15_sm_['crude_rate'] - s15_sf_['crude_rate']):.1f} pp crude to {100 * (s15_sm_['adjusted_rate'] - s15_sf_['adjusted_rate']):.1f} pp after behaviour standardisation; "
         f"{len(s15_still)} of {len(s15_std)} demographic levels still differ from the base rate after adjustment", float(100 * (s15_sm_["adjusted_rate"] - s15_sf_["adjusted_rate"])), "s15_crude_vs_adjusted", 2)

**Observations:**
- Holding behaviour equal removes 56% of the spread across age bands, 45% for sex, 44% for education and about 0% for marriage.
- The sex gap shrinks from 3.4 pp to 1.9 pp (adjusted men 23.2% vs women 21.4%).
- Age 21-25 moves from 26.7% crude to 21.7% adjusted, right at the base rate.
- Other/Unknown stays far below base (7.1% crude, 8.9% adjusted). 9 of 25 levels still differ from the base after adjustment, mostly by 1-3 pp.
- Story use: headline dumbbell "demographics fade" (`s15_demographic_fade.csv`).

### 15.9 A fairness-style lens (descriptive only): would a behaviour-only rule treat groups evenly?
No model is built. Take the simplest behaviour-only rule a collections team could use — **flag if September status ≥ 2** (`DM_FLAG_SEP2`) — and ask, per demographic group: how many are flagged (*flag rate*), how many of the flagged default (*hit rate* / precision), how many defaulters are caught (*recall*), and how many good clients are flagged (*false-positive rate*). The **disparate flag-rate ratio** compares each group with the portfolio and the lowest-flagged group in its dimension (the "four-fifths" convention would look at min / max ≥ 0.8). A behaviour-only rule can still produce unequal flag rates when behaviour differs between groups; that is disparity in outcome, not in treatment.

In [ ]:
df["DM_FLAG_SEP2"] = (df["PAY_1"] >= 2).astype(int)
S15_FLAG_RATE = df["DM_FLAG_SEP2"].mean(); S15_HIT_RATE = df.loc[df["DM_FLAG_SEP2"] == 1, TARGET].mean()
s15_rows = []
for s15_c, s15_o in S15_STD_DIMS[:4]:
    for s15_lev in s15_o:
        s15_g = df[df[s15_c] == s15_lev]; s15_f = s15_g["DM_FLAG_SEP2"] == 1; s15_d = s15_g[TARGET] == 1
        s15_lo, s15_hi = wilson_ci((s15_f & s15_d).sum(), max(s15_f.sum(), 1)); s15_flo, s15_fhi = wilson_ci(s15_f.sum(), len(s15_g))
        s15_rows.append({"demographic": s15_c, "level": s15_lev, "n": len(s15_g), "default_rate": s15_d.mean(), "flagged": int(s15_f.sum()), "flag_rate": s15_f.mean(), "flag_lo": float(s15_flo), "flag_hi": float(s15_fhi),
                         "hit_rate": s15_d[s15_f].mean(), "hit_lo": float(s15_lo), "hit_hi": float(s15_hi), "recall": s15_f[s15_d].mean(), "false_positive_rate": s15_f[~s15_d].mean(), "default_rate_unflagged": s15_d[~s15_f].mean()})
s15_fair = pd.DataFrame(s15_rows)
s15_fair["flag_ratio_vs_portfolio"] = s15_fair["flag_rate"] / S15_FLAG_RATE
s15_fair["flag_ratio_vs_lowest"] = s15_fair["flag_rate"] / s15_fair.groupby("demographic")["flag_rate"].transform("min")
s15_fair["min_over_max_in_dimension"] = s15_fair.groupby("demographic")["flag_rate"].transform("min") / s15_fair.groupby("demographic")["flag_rate"].transform("max")
s15_fair["label"] = s15_fair["demographic"].map({"SEX_LABEL": "Sex", "EDU_LABEL": "Education", "MAR_LABEL": "Marital", "AGE_BAND": "Age"}) + ": " + s15_fair["level"].astype(str)
fig, axes = plt.subplots(1, 3, figsize=(19, 6.5), sharey=True)
s15_y = np.arange(len(s15_fair))[::-1]
axes[0].barh(s15_y, s15_fair["flag_rate"], color=C_NEU, xerr=[s15_fair["flag_rate"] - s15_fair["flag_lo"], s15_fair["flag_hi"] - s15_fair["flag_rate"]], capsize=2)
axes[0].axvline(S15_FLAG_RATE, color="k", ls="--", lw=1); axes[0].set_title(f"15.9 Flag rate (Sep status >= 2); portfolio {S15_FLAG_RATE:.1%}")
axes[1].errorbar(s15_fair["hit_rate"], s15_y, xerr=[s15_fair["hit_rate"] - s15_fair["hit_lo"], s15_fair["hit_hi"] - s15_fair["hit_rate"]], fmt="o", color=C_DEF, capsize=3)
axes[1].axvline(S15_HIT_RATE, color="k", ls="--", lw=1); axes[1].set_title(f"Hit rate: default among flagged; portfolio {S15_HIT_RATE:.1%}")
axes[2].scatter(s15_fair["recall"], s15_y, color=C_OK, s=45, label="recall (defaulters flagged)"); axes[2].scatter(s15_fair["false_positive_rate"], s15_y, color=C_ACC, marker="s", s=45, label="false-positive rate")
axes[2].scatter(s15_fair["default_rate_unflagged"], s15_y, color=C_NEU, marker="x", s=45, label="default rate among unflagged"); axes[2].legend(fontsize=8); axes[2].set_title("Recall, false positives and missed risk")
axes[0].set_yticks(s15_y); axes[0].set_yticklabels(s15_fair["label"], fontsize=8)
for ax in axes: ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
plt.tight_layout(); savefig("s15_fairness_flag"); plt.show()
print(s15_fair[["label", "n", "default_rate", "flag_rate", "flag_ratio_vs_portfolio", "flag_ratio_vs_lowest", "hit_rate", "hit_lo", "hit_hi", "recall", "false_positive_rate", "default_rate_unflagged"]].round(3).to_string(index=False))
print("\nmin / max flag-rate ratio per dimension (four-fifths convention: >= 0.80):"); print(s15_fair.groupby("demographic", sort=False)["min_over_max_in_dimension"].first().round(3).to_string())
export(s15_fair, "s15_fairness_flag")
s15_fs = s15_fair[s15_fair["demographic"] == "SEX_LABEL"].set_index("level")
s15_fa = s15_fair[s15_fair["demographic"] == "AGE_BAND"]
add_fact("15.9", "fairness lens", f"A behaviour-only flag (Sep status >= 2) hits {s15_fs.loc['Male', 'flag_rate']:.1%} of men vs {s15_fs.loc['Female', 'flag_rate']:.1%} of women (ratio {s15_fs.loc['Male', 'flag_rate'] / s15_fs.loc['Female', 'flag_rate']:.2f}), "
         f"but the hit rate is almost equal ({s15_fs.loc['Male', 'hit_rate']:.0%} vs {s15_fs.loc['Female', 'hit_rate']:.0%}): unequal flag rates reflect unequal behaviour", float(s15_fs.loc["Male", "flag_rate"] / s15_fs.loc["Female", "flag_rate"]), "s15_fairness_flag", 2)
add_fact("15.9", "fairness lens", f"Flag rates by age band range from {s15_fa['flag_rate'].min():.1%} ({s15_fa.loc[s15_fa['flag_rate'].idxmin(), 'level']}) to {s15_fa['flag_rate'].max():.1%} ({s15_fa.loc[s15_fa['flag_rate'].idxmax(), 'level']}), "
         f"a min/max ratio of {s15_fa['flag_rate'].min() / s15_fa['flag_rate'].max():.2f}", float(s15_fa["flag_rate"].min() / s15_fa["flag_rate"].max()), "s15_fairness_flag", 1)

**Observations:**
- Rule: flag if September status is 2 or worse. It flags 11.5% of men vs 9.8% of women (ratio 0.85, passing the four-fifths rule, which asks the lowest flag rate to be at least 80% of the highest).
- Education (0.45) and age (0.55) fail: 13.6% of high-school vs 7.3% of graduate-school clients are flagged; 15.1% at 61+ vs 8.3% at 31-35.
- The hit rate (share of flagged clients who default) is similar for the main groups, 66-75%; Other/Unknown is the exception at 24%.
- Story use: optional side note; unequal flag rates mirror unequal behaviour.

#### 15.9.1 Calibration within group: does the same behaviour tier mean the same risk for everyone?
Five rule-based behaviour tiers (`DM_BEH_TIER`): T1 paid in full or inactive in Sep and never late · T2 revolving and never late · T3 current in Sep but late earlier · T4 late 1 month in Sep · T5 late 2+ months in Sep. If tiers are *calibrated within group*, each tier's default rate is the same for every sex, age band, education and marital status. The heatmap shows the deviation (pp) of each group's tier rate from the portfolio tier rate; a chi-square test per tier x dimension tells which deviations are beyond noise.

In [ ]:
S15_TIERS = ["T1 Paid in full or inactive in Sep, never late", "T2 Revolving, never late", "T3 Current in Sep, late earlier", "T4 Late 1m in Sep", "T5 Late 2m+ in Sep"]
df["DM_BEH_TIER"] = np.select([df["PAY_1"] >= 2, df["PAY_1"] == 1, df["N_LATE"] > 0, df["PAY_1"] == 0], [S15_TIERS[4], S15_TIERS[3], S15_TIERS[2], S15_TIERS[1]], default=S15_TIERS[0])
s15_tier = rate_table(df, "DM_BEH_TIER").set_index("DM_BEH_TIER").reindex(S15_TIERS)
print(s15_tier[["n", "defaults", "rate", "ci_lo", "ci_hi", "share_clients", "share_defaults", "lift"]].round(3).to_string())
s15_rows, s15_tests = [], []
for s15_c, s15_o in S15_STD_DIMS[:4]:
    for s15_t in S15_TIERS:
        s15_sub = df[df["DM_BEH_TIER"] == s15_t]
        s15_ct = pd.crosstab(s15_sub[s15_c].astype(str), s15_sub[TARGET]); s15_ct = s15_ct[s15_ct.sum(axis=1) >= 30]
        s15_chi = stats.chi2_contingency(s15_ct)
        s15_tests.append({"demographic": s15_c, "tier": s15_t, "chi2": s15_chi[0], "dof": s15_chi[2], "p_value": s15_chi[1], "cramers_v": np.sqrt(s15_chi[0] / s15_ct.to_numpy().sum() / max(1, min(s15_ct.shape) - 1))})
        for s15_lev in s15_o:
            s15_g = s15_sub[s15_sub[s15_c] == s15_lev]
            if len(s15_g) < 30: continue
            s15_lo, s15_hi = wilson_ci(s15_g[TARGET].sum(), len(s15_g))
            s15_rows.append({"demographic": s15_c, "level": s15_lev, "tier": s15_t, "n": len(s15_g), "rate": s15_g[TARGET].mean(), "ci_lo": float(s15_lo), "ci_hi": float(s15_hi), "tier_rate": s15_tier.loc[s15_t, "rate"],
                             "deviation_pp": 100 * (s15_g[TARGET].mean() - s15_tier.loc[s15_t, "rate"]), "outside_ci": bool((s15_lo > s15_tier.loc[s15_t, "rate"]) or (s15_hi < s15_tier.loc[s15_t, "rate"]))})
s15_cal, s15_caltest = pd.DataFrame(s15_rows), pd.DataFrame(s15_tests)
s15_cal["label"] = s15_cal["demographic"].map({"SEX_LABEL": "Sex", "EDU_LABEL": "Education", "MAR_LABEL": "Marital", "AGE_BAND": "Age"}) + ": " + s15_cal["level"].astype(str)
s15_lab_order = list(dict.fromkeys(s15_cal["label"]))
s15_hm = s15_cal.pivot(index="label", columns="tier", values="deviation_pp").reindex(index=s15_lab_order, columns=S15_TIERS)
s15_rt = s15_cal.pivot(index="label", columns="tier", values="rate").reindex(index=s15_lab_order, columns=S15_TIERS)
fig, axes = plt.subplots(1, 2, figsize=(19, 7))
sns.heatmap(s15_rt, annot=True, fmt=".1%", cmap=CMAP_RATE, ax=axes[0], cbar_kws={"label": "default rate"}, annot_kws={"fontsize": 8})
axes[0].set_title("15.9.1 Default rate by behaviour tier within each demographic group (blank: n < 30)")
sns.heatmap(s15_hm, annot=True, fmt="+.1f", cmap=CMAP_DIV, center=0, vmin=-12, vmax=12, ax=axes[1], cbar_kws={"label": "deviation from portfolio tier rate (pp)"}, annot_kws={"fontsize": 8})
axes[1].set_title("Calibration within group: deviation from the tier's portfolio rate (pp)")
for ax in axes:
    ax.set_xlabel(""); ax.set_ylabel(""); ax.set_xticklabels([t.split(" ")[0] + "\n" + " ".join(t.split(" ")[1:]) for t in S15_TIERS], rotation=0, fontsize=8)
plt.tight_layout(); savefig("s15_tier_calibration"); plt.show()
print("\nIs the tier rate equal across groups? chi-square per tier x dimension:")
print(s15_caltest.pivot(index="tier", columns="demographic", values="p_value").reindex(S15_TIERS).to_string(float_format=lambda v: f"{v:.2e}"))
s15_mad = s15_cal.groupby("demographic", sort=False).apply(lambda g: np.average(g["deviation_pp"].abs(), weights=g["n"]))
print("\nn-weighted mean absolute calibration deviation (pp) by dimension:"); print(s15_mad.round(2).to_string())
print(f"Largest deviations (cells with CI excluding the tier rate: {int(s15_cal['outside_ci'].sum())} of {len(s15_cal)}):")
print(s15_cal.reindex(s15_cal["deviation_pp"].abs().sort_values(ascending=False).index)[["label", "tier", "n", "rate", "tier_rate", "deviation_pp", "outside_ci"]].head(12).round(3).to_string(index=False))
export(s15_cal, "s15_tier_calibration"); export(s15_caltest, "s15_tier_calibration_tests")
s15_crude_mad = float(np.average((s15_std[s15_std["demographic"] != "DM_LIFE_STAGE"]["crude_rate"] - BASE_RATE).abs() * 100, weights=s15_std[s15_std["demographic"] != "DM_LIFE_STAGE"]["n"]))
s15_cal_mad = float(np.average(s15_cal["deviation_pp"].abs(), weights=s15_cal["n"]))
s15_worst = s15_cal[s15_cal["outside_ci"]].reindex(s15_cal[s15_cal["outside_ci"]]["deviation_pp"].abs().sort_values(ascending=False).index)
s15_worst_txt = f"; the largest reliable miscalibration is {s15_worst.iloc[0]['label']} in {s15_worst.iloc[0]['tier'].split(' ')[0]} ({s15_worst.iloc[0]['deviation_pp']:+.1f} pp)" if len(s15_worst) else ""
add_fact("15.9", "fairness lens", f"Inside a behaviour tier, demographic groups deviate from the tier's default rate by {s15_cal_mad:.1f} pp on average (vs {s15_crude_mad:.1f} pp crude deviation from the base rate); "
         f"{int(s15_cal['outside_ci'].sum())} of {len(s15_cal)} group x tier cells are reliably off{s15_worst_txt}", s15_cal_mad, "s15_tier_calibration", 2)

**Observations:**
- Tier default rates: T2 revolving never late 10.7%, T1 paid in full or inactive in Sep, never late 13.5%, T3 26.9%, T4 33.9%, T5 69.6%. T5 holds 10.4% of clients but 32.8% of defaults.
- Inside a tier, groups differ from the tier rate by 1.1-1.7 pp on average; 26 of 82 group x tier cells are reliably off.
- The largest gaps: Other/Unknown in T4 (0 of 32, -33.9 pp), age 61+ in T4 (55.6%, +21.6 pp, n=36), age 51-60 in T4 (41.8%, +7.9 pp).
- Story use: background; the tier table (`s15_tier_calibration.csv`) works as a Tableau highlight table.

### 15.10 Small-cell demographic atlas with empirical-Bayes shrinkage
Section 8.4 ranked the raw 4-way cells with n ≥ 100. Here **all** cells (sex x education x marital status x age band) are kept, and each cell's rate is shrunk toward the portfolio rate with a **beta-binomial** model whose prior is estimated by the method of moments from the cells themselves: prior mean μ = base rate; between-cell variance τ² = (Σ nᵢ(p̂ᵢ − μ)² − (K − 1)·μ(1 − μ)) / (N − K); prior strength M = μ(1 − μ)/τ² − 1 "pseudo-clients". Shrunk rate = (k + μM)/(n + M): a cell of size n keeps the weight n/(n + M) on its own data. Small cells can then be shown on a dashboard without being over-read.

In [ ]:
S15_CELL_KEY = ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND"]
s15_cells = rate_table(df, S15_CELL_KEY)
s15_cells["cell"] = s15_cells[S15_CELL_KEY].astype(str).agg(" | ".join, axis=1)
s15_K, s15_N = len(s15_cells), s15_cells["n"].sum(); S15_MU = s15_cells["defaults"].sum() / s15_N
s15_Q = float((s15_cells["n"] * (s15_cells["rate"] - S15_MU) ** 2).sum())
S15_TAU2 = max(1e-8, (s15_Q - (s15_K - 1) * S15_MU * (1 - S15_MU)) / (s15_N - s15_K))
S15_M = S15_MU * (1 - S15_MU) / S15_TAU2 - 1
s15_alpha, s15_beta = S15_MU * S15_M, (1 - S15_MU) * S15_M
s15_cells["shrink_weight_own_data"] = s15_cells["n"] / (s15_cells["n"] + S15_M)
s15_cells["rate_shrunk"] = (s15_cells["defaults"] + s15_alpha) / (s15_cells["n"] + S15_M)
s15_cells["shrunk_lo"] = stats.beta.ppf(0.025, s15_cells["defaults"] + s15_alpha, s15_cells["n"] - s15_cells["defaults"] + s15_beta)
s15_cells["shrunk_hi"] = stats.beta.ppf(0.975, s15_cells["defaults"] + s15_alpha, s15_cells["n"] - s15_cells["defaults"] + s15_beta)
s15_cells["prob_above_base"] = stats.beta.sf(S15_MU, s15_cells["defaults"] + s15_alpha, s15_cells["n"] - s15_cells["defaults"] + s15_beta)
s15_se0 = np.sqrt(S15_MU * (1 - S15_MU) / s15_cells["n"])
s15_cells["z_vs_base"] = (s15_cells["rate"] - S15_MU) / s15_se0
s15_cells["funnel_flag"] = np.where(s15_cells["z_vs_base"].abs() > 3.09, "outside 99.8% limits", np.where(s15_cells["z_vs_base"].abs() > 1.96, "outside 95% limits", "inside limits"))
s15_cells["rank_raw"] = s15_cells["rate"].rank(ascending=False, method="min").astype(int); s15_cells["rank_shrunk"] = s15_cells["rate_shrunk"].rank(ascending=False, method="min").astype(int)
s15_cells["expected_rate_behaviour"] = df.groupby(S15_CELL_KEY, observed=True)["DM_BEH_EXPECTED"].mean().reset_index(drop=True).to_numpy()
print(f"cells: {s15_K} | clients {s15_N:,} | median cell n {s15_cells['n'].median():.0f} | cells with n < 30: {(s15_cells['n'] < 30).sum()} | n < 100: {(s15_cells['n'] < 100).sum()}")
print(f"prior mean {S15_MU:.4f} | between-cell SD of true rates tau = {100 * np.sqrt(S15_TAU2):.2f} pp | prior strength M = {S15_M:,.0f} pseudo-clients (alpha {s15_alpha:.1f}, beta {s15_beta:.1f})")
print(f"observed n-weighted SD of raw cell rates {100 * np.sqrt(s15_Q / s15_N):.2f} pp, of which pure binomial noise would give {100 * np.sqrt((s15_K - 1) * S15_MU * (1 - S15_MU) / s15_N):.2f} pp")
print(s15_cells["funnel_flag"].value_counts().to_string())
df["DM_CELL4"] = df[S15_CELL_KEY].astype(str).agg(" | ".join, axis=1)
df["DM_CELL_RATE_EB"] = df["DM_CELL4"].map(s15_cells.set_index("cell")["rate_shrunk"]).astype(float)
export(s15_cells, "s15_cells_shrunk")

Funnel plot (left): raw cell rate vs cell size with binomial control limits around the base rate (95% and 99.8%; the range a cell of that size would show by pure chance); points inside the funnel are indistinguishable from noise. Shrinkage plot (right): raw vs shrunk rate; small cells are pulled hard toward the base rate, large cells stay put.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5))
ax = axes[0]
s15_nn = np.logspace(0, np.log10(s15_cells["n"].max() * 1.1), 200)
for s15_z, s15_ls, s15_lab in ((1.96, ":", "95% limits"), (3.09, "--", "99.8% limits")):
    ax.plot(s15_nn, np.clip(S15_MU + s15_z * np.sqrt(S15_MU * (1 - S15_MU) / s15_nn), 0, 1), color="k", ls=s15_ls, lw=1, label=s15_lab)
    ax.plot(s15_nn, np.clip(S15_MU - s15_z * np.sqrt(S15_MU * (1 - S15_MU) / s15_nn), 0, 1), color="k", ls=s15_ls, lw=1)
s15_fcol = {"inside limits": C_NEU, "outside 95% limits": "#fd8d3c", "outside 99.8% limits": C_DEF}
for s15_k, s15_g in s15_cells.groupby("funnel_flag"):
    ax.scatter(s15_g["n"], s15_g["rate"], color=s15_fcol[s15_k], s=28, alpha=.8, label=f"{s15_k} ({len(s15_g)})")
for _, s15_r in s15_cells[s15_cells["funnel_flag"] == "outside 99.8% limits"].iterrows():
    ax.annotate(s15_r["cell"], (s15_r["n"], s15_r["rate"]), fontsize=6, xytext=(4, 3), textcoords="offset points")
ax.axhline(S15_MU, color="k", lw=.8); ax.set_xscale("log"); ax.set_ylim(-0.02, 0.8); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("cell size n (log scale)"); ax.set_ylabel("raw default rate"); ax.set_title("15.10 Funnel plot: 4-way demographic cells vs binomial control limits"); ax.legend(fontsize=8)
ax = axes[1]
s15_sc = ax.scatter(s15_cells["rate"], s15_cells["rate_shrunk"], c=np.log10(s15_cells["n"]), cmap="viridis", s=30)
ax.plot([0, .8], [0, .8], color="k", lw=.8, ls="--"); ax.axhline(S15_MU, color=C_NEU, lw=.8)
plt.colorbar(s15_sc, ax=ax, label="log10(cell n)"); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("raw cell default rate"); ax.set_ylabel("empirical-Bayes shrunk rate"); ax.set_title(f"Shrinkage: prior worth {S15_M:,.0f} clients at {S15_MU:.1%}")
plt.tight_layout(); savefig("s15_cells_funnel"); plt.show()

**Observations:**
- The funnel chart is image-only; numbers come from the printed summary above it.
- 177 sex x education x marriage x age cells, median 33 clients; 87 cells have fewer than 30 clients.
- The true between-cell spread is only 4.2 pp. The observed spread is 5.27 pp, and 3.18 pp of that is pure chance.
- 36 cells fall outside the 95% limits (10 outside 99.8%). The prior is worth 96 clients, so a cell of 96 clients gets half its weight from its own rate.
- Story use: funnel chart as a warning against ranking tiny cells.

Top and bottom cells **after** shrinkage, with 95% credible intervals (the Bayesian version of a confidence interval), and how the ranking changed. Cells that topped the raw ranking only because they are tiny drop out.

In [ ]:
s15_top = s15_cells.sort_values("rate_shrunk", ascending=False).head(12); s15_bottom = s15_cells.sort_values("rate_shrunk").head(12)
fig, axes = plt.subplots(1, 2, figsize=(20, 6.5), sharex=True)
for ax, s15_t, s15_ttl, s15_col in ((axes[0], s15_top, "Highest shrunk default rate", C_DEF), (axes[1], s15_bottom, "Lowest shrunk default rate", C_OK)):
    s15_t = s15_t.iloc[::-1]; s15_y = np.arange(len(s15_t))
    ax.barh(s15_y, s15_t["rate_shrunk"], color=s15_col, alpha=.8, xerr=[s15_t["rate_shrunk"] - s15_t["shrunk_lo"], s15_t["shrunk_hi"] - s15_t["rate_shrunk"]], capsize=2, label="shrunk rate (95% credible interval)")
    ax.scatter(s15_t["rate"], s15_y, color="k", marker="x", zorder=4, label="raw rate")
    for s15_yy, (s15_n, s15_rr) in enumerate(zip(s15_t["n"], s15_t["rank_raw"])):
        ax.text(0.005, s15_yy, f"n={s15_n}, raw rank {s15_rr}", va="center", fontsize=7, color="white")
    ax.axvline(S15_MU, color="k", ls="--", lw=1); ax.set_yticks(s15_y); ax.set_yticklabels(s15_t["cell"], fontsize=8); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title(f"15.10 {s15_ttl}"); ax.legend(fontsize=8, loc="lower right")
plt.tight_layout(); savefig("s15_cells_shrunk_top_bottom"); plt.show()
s15_show = ["cell", "n", "defaults", "rate", "rate_shrunk", "shrunk_lo", "shrunk_hi", "prob_above_base", "shrink_weight_own_data", "rank_raw", "rank_shrunk", "expected_rate_behaviour", "funnel_flag"]
print("Top 12 after shrinkage:"); print(s15_top[s15_show].round(3).to_string(index=False))
print("\nBottom 12 after shrinkage:"); print(s15_bottom[s15_show].round(3).to_string(index=False))
s15_raw_top10 = set(s15_cells.nsmallest(10, "rank_raw")["cell"]); s15_shr_top10 = set(s15_cells.nsmallest(10, "rank_shrunk")["cell"])
s15_rho = stats.spearmanr(s15_cells["rate"], s15_cells["rate_shrunk"])[0]
s15_cred = s15_cells[(s15_cells["prob_above_base"] > 0.975) | (s15_cells["prob_above_base"] < 0.025)]
print(f"\nSpearman raw vs shrunk ranking: {s15_rho:.3f} | raw top-10 cells still in the shrunk top-10: {len(s15_raw_top10 & s15_shr_top10)} of 10 (median n of dropped cells: "
      f"{s15_cells[s15_cells['cell'].isin(s15_raw_top10 - s15_shr_top10)]['n'].median() if len(s15_raw_top10 - s15_shr_top10) else float('nan'):.0f})")
print(f"shrunk rates span {s15_cells['rate_shrunk'].min():.1%} - {s15_cells['rate_shrunk'].max():.1%} (raw: {s15_cells['rate'].min():.1%} - {s15_cells['rate'].max():.1%})")
print(f"cells credibly different from the base rate (posterior prob > 97.5% either side): {len(s15_cred)} of {s15_K}, covering {s15_cred['n'].sum() / s15_N:.1%} of clients "
      f"({int((s15_cred['prob_above_base'] > .5).sum())} above, {int((s15_cred['prob_above_base'] < .5).sum())} below)")
s15_t1, s15_b1 = s15_top.iloc[0], s15_bottom.iloc[0]
add_fact("15.10", "demographic atlas", f"Across {s15_K} sex x education x marital x age cells the true between-cell SD of default rates is only {100 * np.sqrt(S15_TAU2):.1f} pp; after empirical-Bayes shrinkage (prior worth {S15_M:,.0f} clients) "
         f"rates span {s15_cells['rate_shrunk'].min():.1%}-{s15_cells['rate_shrunk'].max():.1%} instead of the raw {s15_cells['rate'].min():.0%}-{s15_cells['rate'].max():.0%}", float(100 * np.sqrt(S15_TAU2)), "s15_cells_funnel", 3)
add_fact("15.10", "demographic atlas", f"Only {len(s15_raw_top10 & s15_shr_top10)} of the 10 highest raw-rate cells survive shrinkage; the riskiest reliable cell is '{s15_t1['cell']}' ({s15_t1['rate_shrunk']:.1%} shrunk, n={int(s15_t1['n'])}) "
         f"and the safest '{s15_b1['cell']}' ({s15_b1['rate_shrunk']:.1%}, n={int(s15_b1['n'])}); {len(s15_cred)} cells ({s15_cred['n'].sum() / s15_N:.0%} of clients) differ credibly from the base rate", float(s15_t1["rate_shrunk"] - s15_b1["rate_shrunk"]), "s15_cells_shrunk_top_bottom", 2)

**Observations:**
- After shrinkage the riskiest cells are married women 21-25 at university (29.5%, n=264) and single men 21-25 at university (29.5%, n=661).
- Tiny cells drop out: male / university / other marital / 46-50 had 63.6% raw (7 of 11) and shrinks to 26.4%.
- The safest reliable cells are single graduate women and men aged 26-35 (16.4-16.9%, n=894-1,858).
- Story use: "riskiest demographic cells" table using shrunk rates only (`s15_cells_shrunk.csv`).

#### 15.10.1 Do the reliable demographic cells differ because of behaviour?
Shrunk cell rate against the cell's behaviour-expected rate (cells with n ≥ 100). Points on the diagonal are cells whose demographic "risk" is entirely their behaviour mix; the vertical distance is the demographic residue.

In [ ]:
s15_cb = s15_cells[s15_cells["n"] >= 100].copy()
s15_cb["residue_pp"] = 100 * (s15_cb["rate_shrunk"] - s15_cb["expected_rate_behaviour"])
fig, ax = plt.subplots(figsize=(9, 7))
s15_sc = ax.scatter(s15_cb["expected_rate_behaviour"], s15_cb["rate_shrunk"], s=s15_cb["n"] / 8, c=s15_cb["residue_pp"], cmap=CMAP_DIV, vmin=-8, vmax=8, edgecolor="k", lw=.4)
s15_lim = [min(s15_cb["expected_rate_behaviour"].min(), s15_cb["rate_shrunk"].min()) - .01, max(s15_cb["expected_rate_behaviour"].max(), s15_cb["rate_shrunk"].max()) + .01]
ax.plot(s15_lim, s15_lim, color="k", ls="--", lw=1)
for _, s15_r in pd.concat([s15_cb.nlargest(4, "residue_pp"), s15_cb.nsmallest(4, "residue_pp")]).iterrows():
    ax.annotate(s15_r["cell"], (s15_r["expected_rate_behaviour"], s15_r["rate_shrunk"]), fontsize=7, xytext=(5, 3), textcoords="offset points")
plt.colorbar(s15_sc, ax=ax, label="shrunk rate − behaviour-expected rate (pp)")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("default rate expected from the cell's behaviour mix"); ax.set_ylabel("empirical-Bayes shrunk default rate")
ax.set_title("15.10.1 Demographic cells (n >= 100): how much of their rate is behaviour?")
savefig("s15_cells_vs_behaviour"); plt.show()
S15_CELL_R2 = np.corrcoef(s15_cb["expected_rate_behaviour"], s15_cb["rate_shrunk"])[0, 1] ** 2
print(f"cells with n >= 100: {len(s15_cb)} | R2 of shrunk rate on behaviour-expected rate: {S15_CELL_R2:.2f} | n-weighted mean |residue| {np.average(s15_cb['residue_pp'].abs(), weights=s15_cb['n']):.2f} pp")
print(s15_cb.reindex(s15_cb["residue_pp"].abs().sort_values(ascending=False).index)[["cell", "n", "rate", "rate_shrunk", "expected_rate_behaviour", "residue_pp"]].head(10).round(3).to_string(index=False))
add_fact("15.10", "demographic atlas", f"Behaviour mix alone explains {S15_CELL_R2:.0%} of the variation in shrunk default rates across the {len(s15_cb)} demographic cells with n >= 100", float(S15_CELL_R2), "s15_cells_vs_behaviour", 2)

**Observations:**
- For the 68 cells with n >= 100, the behaviour-expected rate explains 63% of the variation in shrunk rates; the average leftover is 1.5 pp.
- The biggest leftovers: married graduate men 51-60 (+5.8 pp: 25.8% vs 20.0% expected) and single high-school women 51-60 (-4.3 pp).
- Married men show up again and again with positive leftovers (+3.6 to +5.8 pp).
- Story use: background.

### 15.11 New columns, exports and story facts of Section 15

| Column | Definition |
|---|---|
| `DM_LIFE_STAGE` | rule-based life stage from age and marital status (8 levels, 15.1.4) |
| `DM_LOG_LIMIT`, `DM_LOG10_LIMIT` | natural / base-10 log of `LIMIT_BAL` |
| `DM_LIMIT_PEER` | credit line predicted from sex, age band, education, marital status (OLS on log limit) |
| `DM_LIMIT_RESID` | log(own line) − log(peer line): line relative to demographic peers |
| `DM_LIMIT_RESID_BAND`, `DM_RESID_VENTILE` | quintile label / ventile index of `DM_LIMIT_RESID` |
| `DM_BEH_CELL`, `DM_BEH_CELL_RICH` | behaviour cell: Sep status group x limit band (x late-months band) |
| `DM_BEH_EXPECTED`, `DM_BEH_EXPECTED_RICH` | portfolio default rate of the client's behaviour cell (behaviour-only expectation) |
| `DM_EDU_RAW_LABEL` | readable label of the raw `EDUCATION` code (0-6) |
| `DM_AGE6` | six age bands used for the household analysis |
| `DM_HH_GROUP` | marital status x sex household group (Married men, Single women, ...) |
| `DM_FLAG_SEP2` | 1 if September status ≥ 2 (behaviour-only flag of 15.9) |
| `DM_BEH_TIER` | five rule-based behaviour tiers (15.9.1) |
| `DM_CELL4` | 4-way demographic cell label (sex, education, marital status, age band) |
| `DM_CELL_RATE_EB` | empirical-Bayes shrunk default rate of the client's 4-way cell |

**Tableau exports (all `s15_*.csv`)** — story sheets: `s15_crude_vs_adjusted` (dumbbell: demographics fade), `s15_gap_decomposition` + `s15_gap_contributions` (stacked bars: explained vs unexplained), `s15_cells_shrunk` (funnel / atlas), `s15_risk_line_mismatch` (bubble map), `s15_age_curves` + `s15_age_of_min_risk` (U-shape), `s15_pyramid`, `s15_marriage_transition`, `s15_life_stage`, `s15_household`, `s15_marriage_premium`, `s15_fairness_flag`, `s15_tier_calibration`, `s15_education_raw_codes`, `s15_education_other_evidence`, `s15_limit_by_demo`, `s15_limit_ols_demo`, `s15_limit_resid_curve`.

In [ ]:
s15_new_cols = [c for c in df.columns if c.startswith("DM_")]
print(f"Section 15 added {len(s15_new_cols)} columns: {s15_new_cols}")
print(f"Section 15 runtime: {time.time() - S15_T0:.0f}s")
s15_facts = pd.DataFrame([f for f in STORY_FACTS if f["section"].startswith("15")])
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    display(s15_facts)
for s15_f in s15_facts.itertuples():
    print(f"[{s15_f.section} | {s15_f.theme} | strength {s15_f.strength}] {s15_f.fact}")

**Observations:**
- The section registers 31 facts. The strongest: the age U (minimum 19.8% at 31); behaviour explains 66% of the age pattern; clients far below their peers' line default at 32.4% vs 14.5% far above; the gender gap splits 43% mix / 57% rate; behaviour removes 56% of the age spread but none of the marriage spread; Other/Unknown education defaults at 7.1% vs 17.5% expected.
- Story use: feeds the Section 20 synthesis.

## 16. Money-flow dynamics and trajectory shapes

**Why this section matters for the story.** Sections 5 and 11 treated the six months as a sequence of *states*
(status codes, transitions, roll rates). This section follows the **money**: the *shape* of each client's path of
bills, payments, implied spend and utilisation from April to September. A dashboard story needs characters, and the
characters here are shapes: the *ramp-up*, the *pay-down*, the *spike*, the *flat-high revolver*, the *debt spiral*,
the *deleverager*. The uncomfortable question that runs through the whole section: **does a path that looks dangerous
to a banker (rising debt, payments below charges) actually default more?**

Conventions: all arrays are chronological (**Apr → Sep**). Month-on-month flows exist for **May..Sep** (5 points).
The accounting identity used throughout is `bill_t = bill_(t-1) + new charges_t − payment_t`, so *new charges*
(implied spend, incl. interest and fees) are `SPEND_t = bill_t − bill_(t-1) + payment_t`.
New `df` columns carry the prefix `MF_`; section globals the prefix `S16_` / `s16_`.

**Section guide**

- **What it does:** Follows the money rather than the status codes. It describes the *shape* of each client's six-month path of bills, payments, new spending and utilisation, groups clients into shapes and archetypes (debt spiral, deleverager, full payer, ...), and looks at payment shocks, frozen cards, over-limit episodes and credit balances.
- **Question it answers:** Does a path that looks dangerous to a banker (rising debt, payments below new charges) actually default more?
- **Inputs / new columns:** `BILL_AMT1-6`, `PAY_AMT1-6`, `PAY_1-6`, `LIMIT_BAL`, utilisation, payment ratio and implied spend (new charges = bill − previous bill + payment). Adds many columns with prefix `MF_` (e.g. `MF_UTIL_SHAPE`, `MF_FLOW_ARCHETYPE`, `MF_REGIME_SEQ`, `MF_N_PAY_SHOCK`, `MF_COVERAGE_BAND`; full list in 16.14).
- **How to read the charts:** x-axis runs Apr → Sep. Shape panels: line = typical path of a cluster, ribbon = middle 50% of clients. Bars = default rate per group with 95% CI; dashed line = 22.1% base rate. Heatmaps: colour = default rate, number = clients. Regime words use B = balance built up, D = paid down, F = flat, Z = no balance.
- **Terms:** utilisation = bill ÷ credit limit. Coverage = payments ÷ new charges over five months (below 100% = balance grew). Drawdown = largest fall from a running peak. IV = how well a grouping separates defaulters (above 0.3 = strong).
- **Key takeaways:**
  - The banker's intuition is backwards: 'debt spiral' clients (balance up 3+ months) default at 15.5% while 'treading water' clients default at 27.7%; naive vs actual risk rank correlates −0.29.
  - Risk sits in balances that stop moving or shrink: flat-for-five FFFFF 28.6%, steady pay-down shape 29.5%, no build months 35.0%, heavy net repayers (coverage over 150%) 31.3%.
  - An active card is a healthy card: a Sep spending surge defaults at 12.6%, while frozen cards (balance, no purchases, 5 months) default at 36.5%, and frozen-in-Sep stays riskier inside every status group.
  - Paying zero is the line, not paying less: Sep payment = 0 gives 35.2% default, a smaller-than-usual payment only 13.9-15.4%.
  - Money lenses peak at IV 0.20 vs 0.87 for Sep status, but spend and payment levels keep IV 0.24-0.30 inside 'Paid in full', where the status code says nothing.
- **Where it feeds:** persona "characters" for the story (archetypes and client biographies, `s16_client_biography.csv`), the balance waterfall (`s16_waterfall.csv`) and the money-flow lens league table (`s16_lens_league.csv`).

### 16.0 Building blocks
Chronological matrices (bills, payments, status, utilisation, implied spend, payment ratio) and four small helpers:
`s16_tab` (default rate, Wilson CI, exposure by group), `s16_bars`, `s16_curve` (default rate by quantile bin) and
`s16_heat` (two-way default-rate heatmap with counts).

In [ ]:
S16_T0 = time.time()
S16_Y = df[TARGET].to_numpy()
S16_N = len(df)
S16_B = df[BILL_COLS].to_numpy(float)[:, ::-1].copy()        # bills, Apr..Sep
S16_P = df[PAMT_COLS].to_numpy(float)[:, ::-1].copy()        # payments, Apr..Sep
S16_ST = df[PAY_COLS].to_numpy(int)[:, ::-1].copy()          # status codes, Apr..Sep
S16_LIM = df["LIMIT_BAL"].to_numpy(float)
S16_U = S16_B / S16_LIM[:, None]                             # utilisation, Apr..Sep
S16_UC = np.clip(S16_U, -0.5, 2.0)                           # clipped copy used for shape work (tames a few extreme rows)
S16_DB = S16_B[:, 1:] - S16_B[:, :-1]                        # balance change, May..Sep
S16_SP = S16_DB + S16_P[:, 1:]                               # implied new charges, May..Sep
S16_PREV = S16_B[:, :-1]                                     # previous statement, for May..Sep
S16_PR = np.where(S16_PREV > 0, np.minimum(S16_P[:, 1:] / np.where(S16_PREV > 0, S16_PREV, 1), 5), np.nan)   # pay / prev bill
S16_EXPO = np.clip(S16_B[:, -1], 0, None)                    # Sep balance (>= 0) = exposure
S16_M6 = list(MONTH_ORDER)
S16_M5 = list(MONTH_ORDER[1:])
S16_PAIRS = [f"{a}→{b}" for a, b in zip(S16_M6[:-1], S16_M6[1:])]
S16_GRP_SEP = df["STATUS_GRP_SEP"].to_numpy()
assert np.allclose(S16_SP[:, 4], df["SPEND_1"].to_numpy(float)) and np.allclose(S16_SP[:, 0], df["SPEND_5"].to_numpy(float))

def s16_tab(labels, order=None, name="group", mask=None, min_n=0):
    # default-rate table by group, with Wilson CI and exposure (Sep balance) shares
    g = pd.DataFrame({name: np.asarray(labels), "y": S16_Y, "expo": S16_EXPO, "ead": S16_EXPO * S16_Y})
    if mask is not None:
        g = g[np.asarray(mask)]
    t = g.groupby(name, observed=True).agg(n=("y", "size"), defaults=("y", "sum"), exposure=("expo", "sum"),
                                           expo_default=("ead", "sum")).reset_index()
    t["share"] = t["n"] / len(g)
    t["rate"] = t["defaults"] / t["n"]
    t["ci_lo"], t["ci_hi"] = wilson_ci(t["defaults"], t["n"])
    t["lift"] = t["rate"] / BASE_RATE
    t["expo_share"] = t["exposure"] / max(1.0, g["expo"].sum())
    t["expo_default_share"] = t["expo_default"] / max(1.0, g["ead"].sum())
    if order is not None:
        have = set(t[name])
        t = t.set_index(name).reindex([o for o in order if o in have]).reset_index()
    return t[t["n"] >= min_n].reset_index(drop=True)

def s16_bars(t, name, ax, title, rot=0, fs=8):
    x = np.arange(len(t))
    ax.bar(x, t["rate"], color=[C_DEF if r > BASE_RATE else C_OK for r in t["rate"]], alpha=.85)
    ax.errorbar(x, t["rate"], yerr=[t["rate"] - t["ci_lo"], t["ci_hi"] - t["rate"]], fmt="none", ecolor="k", capsize=3, lw=1)
    ax.axhline(BASE_RATE, ls="--", color="k", lw=1)
    for xi, r, hi, n in zip(x, t["rate"], t["ci_hi"], t["n"]):
        ax.text(xi, hi + .008, f"{r:.0%}\nn={n:,}", ha="center", va="bottom", fontsize=fs)
    ax.set_xticks(x)
    ax.set_xticklabels(t[name].astype(str), rotation=rot, ha="right" if rot else "center")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_ylabel("default rate"); ax.set_title(title)
    ax.set_ylim(0, min(1.05, float(t["ci_hi"].max()) * 1.3 + .05))

def s16_curve(x, q=10, name="x"):
    # default rate by quantile bin of a numeric descriptor
    x = np.asarray(x, float)
    ok = ~np.isnan(x)
    b = pd.qcut(pd.Series(x[ok]), q, duplicates="drop")
    g = pd.DataFrame({"bin": b.cat.codes.to_numpy(), "x": x[ok], "y": S16_Y[ok]})
    t = g.groupby("bin").agg(x_med=("x", "median"), x_lo=("x", "min"), x_hi=("x", "max"), n=("y", "size"), defaults=("y", "sum")).reset_index()
    t["rate"] = t["defaults"] / t["n"]
    t["ci_lo"], t["ci_hi"] = wilson_ci(t["defaults"], t["n"])
    t.insert(0, "descriptor", name)
    return t

def s16_plot_curve(t, ax, title, fmt="{:.2f}"):
    x = np.arange(len(t))
    ax.fill_between(x, t["ci_lo"], t["ci_hi"], color=C_DEF, alpha=.15)
    ax.plot(x, t["rate"], marker="o", color=C_DEF, lw=2)
    ax.axhline(BASE_RATE, ls="--", color="k", lw=1)
    ax.set_xticks(x); ax.set_xticklabels([fmt.format(v) for v in t["x_med"]], rotation=45, ha="right", fontsize=8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title(title, fontsize=10)

def s16_heat(r, c, ax, title, r_order=None, c_order=None, min_n=50, mask=None, fs=8):
    # default-rate heatmap for two label arrays; cell text = rate and n; cells below min_n are blanked
    g = pd.DataFrame({"r": np.asarray(r), "c": np.asarray(c), "y": S16_Y})
    if mask is not None:
        g = g[np.asarray(mask)]
    a = g.groupby(["r", "c"], observed=True)["y"].agg(rate="mean", n="size").reset_index()
    R = a.pivot(index="r", columns="c", values="rate")
    Nn = a.pivot(index="r", columns="c", values="n")
    if r_order is not None:
        ro = [o for o in r_order if o in R.index]; R, Nn = R.reindex(ro), Nn.reindex(ro)
    if c_order is not None:
        co = [o for o in c_order if o in R.columns]; R, Nn = R[co], Nn[co]
    Nn = Nn.fillna(0)
    R = R.where(Nn >= min_n).astype(float)
    ann = np.array([["" if np.isnan(R.iat[i, j]) else f"{R.iat[i, j]:.0%}\n{int(Nn.iat[i, j]):,}" for j in range(R.shape[1])]
                    for i in range(R.shape[0])])
    sns.heatmap(R, annot=ann, fmt="", cmap=CMAP_RATE, ax=ax, cbar=False, linewidths=.5, mask=R.isna().to_numpy(),
                annot_kws={"fontsize": fs}, vmin=0, vmax=max(.5, float(np.nanmax(R.to_numpy())) if R.notna().any().any() else .5))
    ax.set_title(title); ax.set_xlabel(""); ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=30); ax.tick_params(axis="y", rotation=0)
    a.columns = ["row", "col", "rate", "n"]
    return a

def s16_max_run(M):
    # longest run of True per row
    best = np.zeros(len(M), int); cur = np.zeros(len(M), int)
    for k in range(M.shape[1]):
        cur = np.where(M[:, k], cur + 1, 0)
        best = np.maximum(best, cur)
    return best

def s16_end_run(M):
    # length of the run of True that ends in the last column
    run = np.zeros(len(M), int)
    for k in range(M.shape[1]):
        still = (run == k) & M[:, -1 - k]
        run[still] += 1
    return run

print(f"clients {S16_N:,} | base rate {BASE_RATE:.2%} | Sep exposure NT$ {S16_EXPO.sum() / 1e9:.2f} bn "
      f"({(S16_EXPO * S16_Y).sum() / S16_EXPO.sum():.1%} of it owed by eventual defaulters)")
print("identity check, bill_t = bill_(t-1) + spend_t - pay_t :", bool(np.allclose(S16_B[:, 1:], S16_B[:, :-1] + S16_SP - S16_P[:, 1:])))

### 16.1 What does a client's utilisation path look like, and which features of its *shape* matter?
Each client has a 6-point utilisation path (Apr → Sep). First, the average path with the **level removed**
(each client's own mean subtracted, and a z-scored version) by outcome: do defaulters *move* differently,
independent of how high they sit?

In [ ]:
S16_UDM = S16_UC - S16_UC.mean(1, keepdims=True)                    # level removed
_sd = S16_UC.std(1, keepdims=True)
S16_UZ = np.where(_sd > 1e-9, S16_UDM / np.where(_sd > 1e-9, _sd, 1), np.nan)   # z-scored per client (pure shape)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for g, lab, col in ((0, "No default", C_OK), (1, "Default", C_DEF)):
    m = S16_Y == g
    axes[0].plot(S16_M6, np.median(S16_UC[m], 0), marker="o", color=col, lw=2, label=lab)
    axes[0].fill_between(S16_M6, np.percentile(S16_UC[m], 25, 0), np.percentile(S16_UC[m], 75, 0), color=col, alpha=.12)
    mu = S16_UDM[m].mean(0); se = S16_UDM[m].std(0) / np.sqrt(m.sum())
    axes[1].plot(S16_M6, mu, marker="o", color=col, lw=2, label=lab)
    axes[1].fill_between(S16_M6, mu - 1.96 * se, mu + 1.96 * se, color=col, alpha=.2)
    axes[2].plot(S16_M6, np.nanmean(S16_UZ[m], 0), marker="o", color=col, lw=2, label=lab)
axes[0].set_title("Raw utilisation: median and IQR"); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_title("Level removed: mean of (util − own 6-month mean), 95% CI"); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=1))
axes[2].set_title("Pure shape: mean z-scored path (flat paths excluded)")
for ax in axes:
    ax.legend(); ax.axhline(0, color="k", lw=.6)
plt.tight_layout(); savefig("s16_norm_trajectories"); plt.show()

S16_NORM = pd.DataFrame([{"DEFAULT_LABEL": lab, "MONTH": mth, "util_median": np.median(S16_UC[S16_Y == g, k]),
                          "util_demeaned_mean": S16_UDM[S16_Y == g, k].mean(), "util_z_mean": np.nanmean(S16_UZ[S16_Y == g, k])}
                         for g, lab in ((0, "No default"), (1, "Default")) for k, mth in enumerate(S16_M6)])
display(S16_NORM.round(4))
export(S16_NORM, "s16_norm_trajectories")

**Observations:**
- Defaulters sit much higher from the start: median utilisation (bill ÷ limit) is 35.2% in Apr for defaulters vs 15.4% for non-defaulters, and 49.3% vs 26.8% in Sep.
- With each client's own average removed, both groups climb by about the same amount (from about −5 to +4/+5 points), so the gap is in level, not in climb.
- On the z-scored path (each client scaled to their own spread) non-defaulters keep rising to +0.24 in Sep, while defaulters peak in Jul (+0.09) and slip to −0.07 by Sep: they stall at the end.
- Story use: opening contrast for the trajectory chapter: 'defaulters are not the climbers, they are the ones already near the top'.

#### 16.1.1 Shape descriptors per client
Nine descriptors of the utilisation path. Slope and curvature come from a **quadratic fit** on centred time
(`util ≈ a + b·t + c·t²`; `c > 0` convex = accelerating upward or bottoming out, `c < 0` concave = saturating or peaking).
**Max drawdown** = largest fall from a running peak; **max run-up** = largest rise from a running trough;
**turning points** = direction reversals, counting only moves larger than 2 points of utilisation.

In [ ]:
_X2 = np.column_stack([np.ones(6), np.arange(6) - 2.5, (np.arange(6) - 2.5) ** 2])
_coef = S16_UC @ np.linalg.pinv(_X2).T                                 # intercept, slope, curvature
_runmax = np.maximum.accumulate(S16_UC, axis=1)
_runmin = np.minimum.accumulate(S16_UC, axis=1)
_d = np.diff(S16_UC, axis=1)
_sg = np.where(np.abs(_d) > 0.02, np.sign(_d), 0)
_turns = np.zeros(S16_N, int); _prev = np.zeros(S16_N)
for k in range(5):
    cur = _sg[:, k]
    _turns += ((cur != 0) & (_prev != 0) & (cur != _prev)).astype(int)
    _prev = np.where(cur != 0, cur, _prev)
_rng = S16_UC.max(1) - S16_UC.min(1)

S16_DESC = pd.DataFrame({
    "UTIL_START": S16_UC[:, 0], "UTIL_END": S16_UC[:, 5], "UTIL_SLOPE": _coef[:, 1], "UTIL_CURV": _coef[:, 2],
    "UTIL_MAXDD": (_runmax - S16_UC).max(1), "UTIL_RUNUP": (S16_UC - _runmin).max(1), "UTIL_RANGE": _rng,
    "UTIL_TURNS": _turns,
    "UTIL_PEAK_MONTH": np.where(_rng < 0.03, "Flat", np.array(S16_M6)[S16_UC.argmax(1)]),
    TARGET: S16_Y})
df["MF_UTIL_CURV"] = S16_DESC["UTIL_CURV"].to_numpy()
df["MF_UTIL_MAXDD"] = S16_DESC["UTIL_MAXDD"].to_numpy()
df["MF_UTIL_RUNUP"] = S16_DESC["UTIL_RUNUP"].to_numpy()
df["MF_UTIL_RANGE"] = S16_DESC["UTIL_RANGE"].to_numpy()
df["MF_UTIL_TURNS"] = S16_DESC["UTIL_TURNS"].to_numpy()
df["MF_UTIL_PEAK_MONTH"] = pd.Categorical(S16_DESC["UTIL_PEAK_MONTH"], ["Flat"] + S16_M6, ordered=True)
print("slope from the quadratic fit equals s00 UTIL_TREND on unclipped rows:",
      bool(np.allclose(_coef[np.abs(S16_U).max(1) < .5, 1], df["UTIL_TREND"].to_numpy()[np.abs(S16_U).max(1) < .5])))
display(S16_DESC.drop(columns=[TARGET, "UTIL_PEAK_MONTH"]).describe(percentiles=[.05, .25, .5, .75, .95]).T.round(4))
display(S16_DESC.groupby(TARGET)[["UTIL_START", "UTIL_END", "UTIL_SLOPE", "UTIL_CURV", "UTIL_MAXDD", "UTIL_RUNUP", "UTIL_RANGE", "UTIL_TURNS"]]
        .agg(["mean", "median"]).T.round(4))

**Observations:**
- The fitted slope matches the s00 `UTIL_TREND` column, so the shape descriptors are consistent with earlier sections.
- Defaulters start higher (median Apr utilisation 35.2% vs 15.4%) but their median slope is 0.0 points per month vs 0.2 for non-defaulters.
- Non-defaulters show a larger median run-up (largest rise from a running low): 6.7 points vs 3.0 for defaulters.
- Curvature and turning points look the same in both groups (median turns = 0 for both).
- Story use: background only (feeds the descriptor ranking in 16.1.2).

#### 16.1.2 Default-rate curve for every shape descriptor
Seven continuous descriptors in deciles (x labels = bin median), two discrete ones as categories. Dashed = base rate.
A flat curve means the descriptor is decoration; a steep or U-shaped one earns a place on the dashboard.

In [ ]:
S16_DESC_CONT = [("UTIL_START", "Start level (Apr)"), ("UTIL_END", "End level (Sep)"), ("UTIL_SLOPE", "Slope per month"),
                 ("UTIL_CURV", "Curvature (quadratic coef.)"), ("UTIL_MAXDD", "Max drawdown"), ("UTIL_RUNUP", "Max run-up"),
                 ("UTIL_RANGE", "Range (max − min)")]
fig, axes = plt.subplots(3, 3, figsize=(17, 12))
_parts = []
for ax, (c, lab) in zip(axes.ravel(), S16_DESC_CONT):
    t = s16_curve(S16_DESC[c], 10, c)
    s16_plot_curve(t, ax, lab, fmt="{:.3f}" if c in ("UTIL_SLOPE", "UTIL_CURV") else "{:.2f}")
    _parts.append(t.assign(bin_label=t["x_med"].round(4).astype(str)))
S16_T_PEAK = s16_tab(S16_DESC["UTIL_PEAK_MONTH"], ["Flat"] + S16_M6, "peak_month")
s16_bars(S16_T_PEAK, "peak_month", axes[2, 1], "Month of peak utilisation")
S16_T_TURN = s16_tab(S16_DESC["UTIL_TURNS"], name="turning_points")
s16_bars(S16_T_TURN, "turning_points", axes[2, 2], "Number of turning points")
for t, c in ((S16_T_PEAK, "peak_month"), (S16_T_TURN, "turning_points")):
    _parts.append(pd.DataFrame({"descriptor": "UTIL_" + c.upper(), "bin": np.arange(len(t)), "bin_label": t[c].astype(str),
                                "n": t["n"], "defaults": t["defaults"], "rate": t["rate"], "ci_lo": t["ci_lo"], "ci_hi": t["ci_hi"]}))
plt.suptitle("Default rate by utilisation-path shape descriptor", y=1.0, fontweight="bold")
plt.tight_layout(); savefig("s16_shape_descriptor_rates"); plt.show()
S16_DESC_RATES = pd.concat(_parts, ignore_index=True)
export(S16_DESC_RATES, "s16_shape_descriptor_rates")

S16_DESC_IV = pd.DataFrame([{"descriptor": c, "IV": woe_iv(S16_DESC, c)[1],
                             "rate_lowest_bin": s16_curve(S16_DESC[c], 10)["rate"].iloc[0],
                             "rate_highest_bin": s16_curve(S16_DESC[c], 10)["rate"].iloc[-1]} for c, _ in S16_DESC_CONT]
                           + [{"descriptor": c, "IV": woe_iv(S16_DESC, c)[1]} for c in ("UTIL_TURNS", "UTIL_PEAK_MONTH")]
                           ).sort_values("IV", ascending=False).reset_index(drop=True)
display(S16_DESC_IV.round(4))
display(S16_T_PEAK.round(4)); display(S16_T_TURN.round(4))
_b = S16_DESC_IV.iloc[0]
add_fact("16.1", "trajectory shape", f"Among nine utilisation-path descriptors the strongest is {_b['descriptor']} (IV {_b['IV']:.2f}); "
         f"slope has IV {S16_DESC_IV.set_index('descriptor').loc['UTIL_SLOPE', 'IV']:.2f} and curvature "
         f"{S16_DESC_IV.set_index('descriptor').loc['UTIL_CURV', 'IV']:.2f}", float(_b["IV"]), "s16_shape_descriptor_rates", 2)
_pk = S16_T_PEAK.set_index("peak_month")["rate"]
add_fact("16.1", "trajectory shape", f"Clients whose utilisation peaks in Sep default at {_pk.get('Sep', np.nan):.1%}, those who peaked in Apr at "
         f"{_pk.get('Apr', np.nan):.1%}, flat paths at {_pk.get('Flat', np.nan):.1%} (base {BASE_RATE:.1%})", float(_pk.get("Sep", np.nan)), "s16_shape_descriptor_rates", 2)

**Observations:**
- Start level is the strongest shape descriptor (IV 0.118; IV = how well a grouping separates defaulters, above 0.3 = strong); run-up and slope follow at about 0.087, end level at 0.077.
- Curvature (IV 0.033), range (0.032) and number of turning points (0.007) are close to decoration.
- Peak month runs against intuition: clients whose utilisation peaked in Apr default at 28.9% and May at 28.3%, while those peaking in Sep (the largest group, 7,127 clients and 41% of exposure) default at only 17.5%.
- Story use: 'an early peak followed by shrinking balances is the warning, not a fresh high'.

#### 16.1.3 Direction × convexity: is an *accelerating* climb worse than a *saturating* one?
Slope class (falling / flat / rising, threshold ±1 point of utilisation per month) crossed with curvature class
(concave / linear / convex, threshold ±0.5 point per month²). Rising + convex = accelerating up; rising + concave =
levelling off; falling + concave = accelerating pay-down; falling + convex = pay-down that is running out of steam.

In [ ]:
S16_SLOPE_CLS = np.select([S16_DESC["UTIL_SLOPE"] < -0.01, S16_DESC["UTIL_SLOPE"] > 0.01], ["Falling", "Rising"], "Flat")
S16_CURV_CLS = np.select([S16_DESC["UTIL_CURV"] < -0.005, S16_DESC["UTIL_CURV"] > 0.005], ["Concave", "Convex"], "Linear")
df["MF_UTIL_PATH_CLASS"] = pd.Series(S16_SLOPE_CLS).str.cat(pd.Series(S16_CURV_CLS), sep=" / ").to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
S16_SC = s16_heat(S16_SLOPE_CLS, S16_CURV_CLS, axes[0], "Default rate: slope class × curvature class (rate, n)",
                  ["Falling", "Flat", "Rising"], ["Concave", "Linear", "Convex"], fs=10)
s16_heat(S16_SLOPE_CLS, pd.cut(df["AVG_UTIL"], [-np.inf, .1, .5, .9, np.inf], labels=["Low <10%", "Mid 10-50%", "High 50-90%", "Maxed >90%"]).astype(str),
         axes[1], "Default rate: slope class × average level", ["Falling", "Flat", "Rising"], ["Low <10%", "Mid 10-50%", "High 50-90%", "Maxed >90%"], fs=10)
plt.tight_layout(); savefig("s16_slope_curvature"); plt.show()
S16_SC.columns = ["slope_class", "curvature_class", "rate", "n"]
export(S16_SC, "s16_slope_curvature")
display(s16_tab(df["MF_UTIL_PATH_CLASS"], name="path_class").sort_values("rate", ascending=False).round(4))

**Observations:**
- Falling / Linear paths (steady pay-down, 3,134 clients) are the riskiest class at 34.8% default (lift 1.57 = 1.57× the base rate).
- Rising paths are the safest: Rising / Linear 16.5% and Rising / Convex (accelerating climb) 17.4%, both below the 22.1% base.
- Rising classes hold 66% of the exposure (money owed in Sep) but only 59% of the money owed by defaulters.
- Story use: a direct counter-example to the 'rising debt is dangerous' intuition; good for a 3×3 heatmap in Tableau.

### 16.2 Are there a handful of recurring trajectory *shapes*, and does shape matter beyond level?
Two clusterings per series, target never used:

* **Shape clustering**: each client's path is z-scored (own mean 0, own sd 1) so KMeans sees only the *shape*.
  Paths with (almost) no movement cannot be z-scored and form explicit **Flat** classes split by level.
* **Level + shape clustering**: KMeans on the raw paths.

`k` is chosen by **silhouette on a 4,000-client sample** (shape: k = 4..8 so that at least four archetypes exist;
raw: k = 3..8). KMeans runs on **cuML** when a GPU is present (guarded), else scikit-learn.
Shape clusters are **named by rules** applied to the centroid (ramp up, pay-down, late jump, spike, V-shape, ...).

In [ ]:
from sklearn.cluster import KMeans as S16_SkKMeans
from sklearn.metrics import silhouette_score as s16_silhouette, adjusted_rand_score as s16_ari

S16_KM_BACKEND = "sklearn"
if GPU:
    try:
        from cuml.cluster import KMeans as S16_CuKMeans
        S16_KM_BACKEND = "cuml"
    except Exception as _e:
        print("cuML not available, using scikit-learn:", type(_e).__name__)

def s16_to_np(a):
    for attr in ("get", "to_numpy"):
        if hasattr(a, attr):
            try:
                return np.asarray(getattr(a, attr)())
            except Exception:
                pass
    return np.asarray(a)

def s16_kmeans(X, k, seed=42):
    global S16_KM_BACKEND
    X = np.ascontiguousarray(X, dtype=np.float32)
    if S16_KM_BACKEND == "cuml":
        try:
            km = S16_CuKMeans(n_clusters=k, random_state=seed, n_init=3, max_iter=300)
            lab = s16_to_np(km.fit_predict(X)).astype(int)
            return lab, s16_to_np(km.cluster_centers_).astype(float)
        except Exception as e:
            print("cuML KMeans failed, falling back to scikit-learn:", type(e).__name__)
            S16_KM_BACKEND = "sklearn"
    km = S16_SkKMeans(n_clusters=k, n_init=3, random_state=seed).fit(X)
    return km.labels_.astype(int), km.cluster_centers_.astype(float)

def s16_pick_k(X, ks, tag, n_s=4000):
    idx = np.random.RandomState(7).choice(len(X), min(n_s, len(X)), replace=False)
    rows = []
    for k in ks:
        lab, _ = s16_kmeans(X, k)
        sil = s16_silhouette(X[idx], lab[idx]) if len(np.unique(lab[idx])) > 1 else np.nan
        rows.append({"series": tag, "k": k, "silhouette": sil, "smallest_cluster_share": np.bincount(lab, minlength=k).min() / len(lab)})
    t = pd.DataFrame(rows)
    return t, int(t.loc[t["silhouette"].idxmax(), "k"])

def s16_name_shape(c, up, down):
    # rule-based name for a centroid path (shape only)
    c = (c - c.mean()) / (c.std() + 1e-9)
    imax, imin, last = int(c.argmax()), int(c.argmin()), len(c) - 1
    if 1 <= imax <= last - 1 and min(c[imax] - c[0], c[imax] - c[-1]) > 1.0:
        return "Spike (mid-window peak)"
    if 1 <= imin <= last - 1 and min(c[0] - c[imin], c[-1] - c[imin]) > 1.0:
        return "V-shape (dip and rebound)"
    tot = c[-1] - c[0]
    early, late = (c[2] - c[0]) / tot, (c[-1] - c[-3]) / tot
    word = up if tot > 0 else down
    if late > 0.65:
        return f"{word} (late {'jump' if tot > 0 else 'drop'})"
    if early > 0.65:
        return f"{word} (early, then {'plateau' if tot > 0 else 'flat'})"
    return f"{word} (steady)"

def s16_unique(names):
    seen, out = {}, []
    for nme in names:
        seen[nme] = seen.get(nme, 0) + 1
        out.append(nme if seen[nme] == 1 else f"{nme} #{seen[nme]}")
    return out

def s16_shape_cluster(X, act, tag, up, down, ks=range(4, 9)):
    # KMeans on per-client z-scored paths for the rows flagged in act; returns labels (object array, '' elsewhere)
    Xa = X[act]
    Z = (Xa - Xa.mean(1, keepdims=True)) / Xa.std(1, keepdims=True)
    sil, k = s16_pick_k(Z, ks, tag + " shape")
    lab, cen = s16_kmeans(Z, k)
    names = np.array(s16_unique([s16_name_shape(c, up, down) for c in cen]), dtype=object)
    out = np.full(len(X), "", dtype=object); out[act] = names[lab]
    lab2, _ = s16_kmeans(Z, k, seed=7)
    return out, sil, k, s16_ari(lab, lab2)

def s16_level_cluster(X, valid, tag, prefix, ks=range(3, 9)):
    # KMeans on raw paths (level + shape); clusters named by rank of mean level and net change
    Xv = X[valid]
    sil, k = s16_pick_k(Xv, ks, tag + " raw")
    lab, cen = s16_kmeans(Xv, k)
    order = np.argsort(cen.mean(1))
    names = np.empty(k, dtype=object)
    for rank, j in enumerate(order):
        chg = cen[j, -1] - cen[j, 0]
        trend = "rising" if chg > 0.05 else ("falling" if chg < -0.05 else "stable")
        names[j] = f"{prefix}{rank + 1}: avg {cen[j].mean():.0%}, {trend}"
    out = np.full(len(X), "", dtype=object); out[valid] = names[lab]
    return out, sil, k

def s16_path_summary(labels, X, months, series, ctype):
    # tidy centroid table: per cluster and month the median / IQR of the raw path and of the level-removed path
    Xd = X - np.nanmean(X, axis=1, keepdims=True)
    rows = []
    for nme in pd.unique(labels):
        m = labels == nme
        n, rate = int(m.sum()), float(S16_Y[m].mean())
        if np.isnan(X[m]).all():
            continue
        for k, mth in enumerate(months):
            rows.append({"series": series, "cluster_type": ctype, "cluster": nme, "MONTH": mth, "n": n, "share": n / len(X), "default_rate": rate,
                         "raw_median": np.nanmedian(X[m, k]), "raw_q25": np.nanpercentile(X[m, k], 25), "raw_q75": np.nanpercentile(X[m, k], 75),
                         "shape_median": np.nanmedian(Xd[m, k]), "shape_q25": np.nanpercentile(Xd[m, k], 25), "shape_q75": np.nanpercentile(Xd[m, k], 75)})
    return pd.DataFrame(rows)

def s16_plot_paths(summ, title, fname, kind="shape", ncols=4):
    cl = summ.drop_duplicates("cluster").sort_values("default_rate", ascending=False)
    nrows = int(np.ceil(len(cl) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 3.3 * nrows), sharey=True, squeeze=False)
    for ax in axes.ravel()[len(cl):]:
        ax.axis("off")
    for ax, (_, r) in zip(axes.ravel(), cl.iterrows()):
        u = summ[summ["cluster"] == r["cluster"]]
        col = C_DEF if r["default_rate"] > BASE_RATE else C_OK
        ax.fill_between(u["MONTH"], u[f"{kind}_q25"], u[f"{kind}_q75"], color=col, alpha=.2)
        ax.plot(u["MONTH"], u[f"{kind}_median"], marker="o", color=col, lw=2)
        ax.axhline(0, color="k", lw=.6)
        ax.set_title(f"{r['cluster']}\nn={r['n']:,} ({r['share']:.0%}) · default {r['default_rate']:.1%}", fontsize=9)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    fig.suptitle(title + "  (line = median, ribbon = IQR; red = above base rate)", y=1.0, fontweight="bold")
    plt.tight_layout(); savefig(fname); plt.show()

print("KMeans backend:", S16_KM_BACKEND)

#### 16.2.1 Utilisation-path shapes
Flat classes first (range < 3 points of utilisation over six months), then shape clusters on the rest. The panels show
the **level-removed** path (utilisation minus the client's own mean), so a ribbon far from zero means a large move.

In [ ]:
_flat = S16_DESC["UTIL_RANGE"].to_numpy() < 0.03
_mean = S16_UC.mean(1)
S16_USHAPE, S16_SIL_U, S16_K_U, _ari = s16_shape_cluster(S16_UC, ~_flat, "utilisation", "Ramp up", "Pay-down")
S16_USHAPE[_flat] = np.select([_mean[_flat] >= .5, _mean[_flat] > .02], ["Flat high", "Flat low"], "Flat zero (dormant)")
df["MF_UTIL_SHAPE"] = S16_USHAPE
display(S16_SIL_U.round(4))
print(f"chosen k = {S16_K_U} | flat paths {_flat.mean():.1%} of clients | seed stability (ARI, 2 seeds) = {_ari:.2f}")
S16_SUM_USHAPE = s16_path_summary(S16_USHAPE, S16_UC, S16_M6, "utilisation", "shape")
s16_plot_paths(S16_SUM_USHAPE, "Utilisation-path SHAPES (level removed)", "s16_util_shapes")
S16_T_USHAPE = s16_tab(S16_USHAPE, name="util_shape").sort_values("rate", ascending=False).reset_index(drop=True)
S16_T_USHAPE = S16_T_USHAPE.merge(pd.DataFrame({"util_shape": S16_USHAPE, "avg_util": _mean, "sep_late": (S16_ST[:, 5] >= 1).astype(float)})
                                  .groupby("util_shape").mean().reset_index(), on="util_shape")
display(S16_T_USHAPE.round(4))
_hi, _lo = S16_T_USHAPE.iloc[0], S16_T_USHAPE.iloc[-1]
add_fact("16.2", "trajectory shape", f"Utilisation paths fall into {len(S16_T_USHAPE)} named shapes; riskiest is '{_hi['util_shape']}' "
         f"({_hi['rate']:.1%} default, {_hi['share']:.0%} of clients), safest '{_lo['util_shape']}' ({_lo['rate']:.1%})",
         float(_hi["rate"]), "s16_util_shapes", 3)

**Observations:**
- k = 4 shape clusters were chosen (silhouette 0.31 = modest separation; very stable across seeds, ARI 0.99), plus flat classes for the 21.4% of clients whose path barely moves.
- 'Pay-down (steady)' is the riskiest shape: 7,520 clients default at 29.5%; they still average 47% utilisation and 32% of them are late in Sep.
- 'Ramp up (late jump)' is the safest at 16.1%, and 'Ramp up (steady)' (7,954 clients, 46% of exposure) defaults at 19.5%.
- Dormant flat-zero cards (4,982 clients, almost no balance) still default at 24.2%.
- Story use: core 'characters' for the story: the shrinking balance, not the growing one, is the risky persona.

#### 16.2.2 Utilisation: level + shape clusters (raw paths)
Same data without the normalisation, so clusters separate mostly by *how high* the client sits and then by drift.
Panels show the raw path.

In [ ]:
S16_ULEVEL, S16_SIL_UL, S16_K_UL = s16_level_cluster(S16_UC, np.ones(S16_N, bool), "utilisation", "U")
df["MF_UTIL_LEVEL_CLUSTER"] = S16_ULEVEL
display(S16_SIL_UL.round(4)); print("chosen k =", S16_K_UL)
S16_SUM_ULEVEL = s16_path_summary(S16_ULEVEL, S16_UC, S16_M6, "utilisation", "level+shape")
s16_plot_paths(S16_SUM_ULEVEL, "Utilisation-path LEVEL + SHAPE clusters (raw utilisation)", "s16_util_level_clusters", kind="raw")
display(s16_tab(S16_ULEVEL, name="util_level_cluster").round(4))

**Observations:**
- Raw paths split cleanly by level (k = 3, silhouette 0.54, far clearer than the 0.31 for shapes).
- U3 (avg 85% utilisation, rising) defaults at 29.8% and holds 52% of exposure and 65% of the money owed by defaulters.
- U1 (avg 6%, stable, 49% of clients) defaults at 18.0%; U2 (avg 46%) sits near base at 21.8%.
- Story use: supports the 'level beats movement' message; U3 is the high-exposure segment for the money view.

#### 16.2.3 Shape × level: does the same shape mean the same thing at 20% and at 95% utilisation?
Default rate for each shape within four bands of average utilisation (cells with n < 50 blank). If rows are flat, level
is everything; if columns are flat, shape is everything.

In [ ]:
S16_LVL4 = pd.cut(df["AVG_UTIL"], [-np.inf, .1, .5, .9, np.inf], labels=["Low <10%", "Mid 10-50%", "High 50-90%", "Maxed >90%"]).astype(str).to_numpy()
fig, ax = plt.subplots(figsize=(10, 0.55 * len(S16_T_USHAPE) + 2))
S16_SHAPE_LEVEL = s16_heat(S16_USHAPE, S16_LVL4, ax, "Default rate: utilisation shape × average level (rate, n)",
                           list(S16_T_USHAPE["util_shape"]), ["Low <10%", "Mid 10-50%", "High 50-90%", "Maxed >90%"], fs=9)
plt.tight_layout(); savefig("s16_shape_x_level"); plt.show()
S16_SHAPE_LEVEL.columns = ["util_shape", "level_band", "rate", "n"]
export(S16_SHAPE_LEVEL, "s16_shape_x_level")
_w = S16_SHAPE_LEVEL[S16_SHAPE_LEVEL["n"] >= 50]
_sp_shape = _w.groupby("level_band")["rate"].agg(lambda s: s.max() - s.min())
_sp_level = _w.groupby("util_shape")["rate"].agg(lambda s: s.max() - s.min())
print("spread of default rate ACROSS SHAPES within a level band (max − min):"); display(_sp_shape.round(4).to_frame("spread"))
print(f"median spread ACROSS LEVELS within a shape: {_sp_level.median():.1%} | median spread across shapes within a level: {_sp_shape.median():.1%}")
add_fact("16.2", "trajectory shape", f"Within one utilisation-level band, default rates differ by a median {_sp_shape.median():.0%} points across shapes; "
         f"within one shape they differ by {_sp_level.median():.0%} points across level bands", float(_sp_shape.median()), "s16_shape_x_level", 2)

**Observations:**
- Inside one utilisation band, default rates across shapes differ by a median 15.5 points (largest gap 25.5 points in the 50-90% band).
- Inside one shape, default rates across utilisation bands differ by a median 16.6 points.
- So shape and level matter about equally; neither one explains the other away (numbers from the printed spread table, the heatmap is image-only).
- Story use: justifies showing shape and level together (a two-way heatmap) rather than either alone.

#### 16.2.4 Payment-ratio path shapes (share of the previous bill repaid, May → Sep)
Only clients with a positive previous bill in all five months have a complete path (the rest are labelled
*No balance due in ≥1 month*). Ratios are capped at 150%. Flat paths (range < 5 points) are split into full payers,
partial payers and minimum payers.

In [ ]:
S16_PRC = np.clip(S16_PR, 0, 1.5)
_valid = ~np.isnan(S16_PRC).any(1)
_rngp = np.where(_valid, np.nanmax(np.where(np.isnan(S16_PRC), -np.inf, S16_PRC), 1) - np.nanmin(np.where(np.isnan(S16_PRC), np.inf, S16_PRC), 1), np.nan)
_flatp = _valid & (_rngp < 0.05)
_meanp = np.nanmean(np.where(_valid[:, None], S16_PRC, 0), 1)
S16_PSHAPE, S16_SIL_P, S16_K_P, _arip = s16_shape_cluster(S16_PRC, _valid & ~_flatp, "payment ratio", "Rising repayment", "Falling repayment")
S16_PSHAPE[_flatp] = np.select([_meanp[_flatp] >= .9, _meanp[_flatp] < .1], ["Flat: full payer", "Flat: minimum payer"], "Flat: partial payer")
S16_PSHAPE[~_valid] = "No balance due in >=1 month"
df["MF_PAYR_SHAPE"] = S16_PSHAPE
S16_PLEVEL, S16_SIL_PL, S16_K_PL = s16_level_cluster(S16_PRC, _valid, "payment ratio", "P")
S16_PLEVEL[~_valid] = "No balance due in >=1 month"
df["MF_PAYR_LEVEL_CLUSTER"] = S16_PLEVEL
display(pd.concat([S16_SIL_P, S16_SIL_PL]).round(4))
print(f"complete paths: {_valid.mean():.1%} of clients | shape k = {S16_K_P} (ARI across seeds {_arip:.2f}) | raw k = {S16_K_PL}")
S16_SUM_PSHAPE = s16_path_summary(S16_PSHAPE, S16_PRC, S16_M5, "payment ratio", "shape")
S16_SUM_PLEVEL = s16_path_summary(S16_PLEVEL, S16_PRC, S16_M5, "payment ratio", "level+shape")
s16_plot_paths(S16_SUM_PSHAPE, "Payment-ratio path SHAPES (level removed)", "s16_payratio_shapes")
s16_plot_paths(S16_SUM_PLEVEL, "Payment-ratio LEVEL + SHAPE clusters (raw ratio)", "s16_payratio_level_clusters", kind="raw")
S16_T_PSHAPE = s16_tab(S16_PSHAPE, name="payratio_shape").sort_values("rate", ascending=False).reset_index(drop=True)
display(S16_T_PSHAPE.round(4))
display(s16_tab(S16_PLEVEL, name="payratio_level_cluster").round(4))

**Observations:**
- Only 75.7% of clients have a complete payment-ratio path (share of the previous bill repaid); the shape clusters are less stable (k = 8, seed ARI 0.73), so treat shape labels as rough.
- Flat full payers (2,157 clients) default at 11.5% and flat minimum payers (6,994 clients, 51% of exposure) at 20.0%; 'Spike #2' is the riskiest shape at 34.3%.
- Level clusters say more: P1 (repays about 7% of the bill, 52% of clients) defaults at 26.4% and carries 87% of the money owed by defaulters; P7 (repays about 98%) defaults at 11.8%.
- Story use: 'how much of the bill you repay' matters more than how that share moves; supports a repayment-level dimension.

#### 16.2.5 Spend-path shapes (implied new charges as a share of the limit, May → Sep)
Spend is scaled by the credit limit so that a NT$ 20k month means the same for a 50k line and a 500k line. Flat paths
(range < 1 point of the limit) are split into *no spend* and *steady spend*.

In [ ]:
S16_SPL = np.clip(S16_SP / S16_LIM[:, None], -0.5, 1.5)
_rngs = S16_SPL.max(1) - S16_SPL.min(1)
_flats = _rngs < 0.01
_means = S16_SPL.mean(1)
S16_SSHAPE, S16_SIL_S, S16_K_S, _aris = s16_shape_cluster(S16_SPL, ~_flats, "spend", "Rising spend", "Fading spend")
S16_SSHAPE[_flats] = np.where(np.abs(_means[_flats]) < 0.005, "Flat: no spend", "Flat: steady spend")
df["MF_SPEND_SHAPE"] = S16_SSHAPE
S16_SLEVEL, S16_SIL_SL, S16_K_SL = s16_level_cluster(S16_SPL, np.ones(S16_N, bool), "spend", "S")
df["MF_SPEND_LEVEL_CLUSTER"] = S16_SLEVEL
display(pd.concat([S16_SIL_S, S16_SIL_SL]).round(4))
print(f"shape k = {S16_K_S} (ARI across seeds {_aris:.2f}) | raw k = {S16_K_SL} | flat spend paths {_flats.mean():.1%}")
S16_SUM_SSHAPE = s16_path_summary(S16_SSHAPE, S16_SPL, S16_M5, "spend / limit", "shape")
S16_SUM_SLEVEL = s16_path_summary(S16_SLEVEL, S16_SPL, S16_M5, "spend / limit", "level+shape")
s16_plot_paths(S16_SUM_SSHAPE, "Spend-path SHAPES (spend / limit, level removed)", "s16_spend_shapes")
s16_plot_paths(S16_SUM_SLEVEL, "Spend-path LEVEL + SHAPE clusters (spend / limit)", "s16_spend_level_clusters", kind="raw")
S16_T_SSHAPE = s16_tab(S16_SSHAPE, name="spend_shape").sort_values("rate", ascending=False).reset_index(drop=True)
display(S16_T_SSHAPE.round(4))
display(s16_tab(S16_SLEVEL, name="spend_level_cluster").round(4))

**Observations:**
- 'Flat: steady spend' is the riskiest spend shape at 38.6% (3,596 clients, 21% of the money owed by defaulters); these near-constant small charges are likely mostly interest and fees.
- 'Flat: no spend' is also above base at 28.0%, while 'Rising spend (late jump)' is the safest at 14.4%.
- Spend level clusters are almost useless: 81% of clients fall in S1 (avg 3% of limit) and class rates range only 15.7% to 23.5%.
- Story use: 'an active, growing card is a healthy card'; feeds the no-last-hurrah message in 16.6.

#### 16.2.6 Which shape vocabulary says the most about default, and do the three overlap?
Information value and Cramér's V (vs default) for all six cluster labels, then the overlap between the three shape
vocabularies and a utilisation-shape × payment-shape default-rate grid. Centroids and client assignments are exported.

In [ ]:
S16_SHAPE_COLS = ["MF_UTIL_SHAPE", "MF_UTIL_LEVEL_CLUSTER", "MF_PAYR_SHAPE", "MF_PAYR_LEVEL_CLUSTER", "MF_SPEND_SHAPE", "MF_SPEND_LEVEL_CLUSTER"]
S16_SHAPE_IV = pd.DataFrame([{"label": c, "n_classes": df[c].nunique(), "IV": woe_iv(df, c)[1], "cramers_v_vs_default": cramers_v(df[c], df[TARGET]),
                              "min_rate": df.groupby(c)[TARGET].mean().min(), "max_rate": df.groupby(c)[TARGET].mean().max()}
                             for c in S16_SHAPE_COLS]).sort_values("IV", ascending=False).reset_index(drop=True)
display(S16_SHAPE_IV.round(4))
_sh3 = ["MF_UTIL_SHAPE", "MF_PAYR_SHAPE", "MF_SPEND_SHAPE"]
display(pd.DataFrame([[cramers_v(df[a], df[b]) for b in _sh3] for a in _sh3], index=_sh3, columns=_sh3).round(3))

fig, ax = plt.subplots(figsize=(15, 0.55 * len(S16_T_USHAPE) + 2.5))
S16_UxP = s16_heat(S16_USHAPE, S16_PSHAPE, ax, "Default rate: utilisation shape (rows) × payment-ratio shape (columns), n ≥ 80",
                   list(S16_T_USHAPE["util_shape"]), list(S16_T_PSHAPE["payratio_shape"]), min_n=80)
plt.tight_layout(); savefig("s16_util_x_pay_shape"); plt.show()
S16_UxP.columns = ["util_shape", "payratio_shape", "rate", "n"]
export(S16_UxP, "s16_util_x_pay_shape")

S16_CENTROIDS = pd.concat([S16_SUM_USHAPE, S16_SUM_ULEVEL, S16_SUM_PSHAPE, S16_SUM_PLEVEL, S16_SUM_SSHAPE, S16_SUM_SLEVEL], ignore_index=True)
export(S16_CENTROIDS.round(5), "s16_shape_centroids")
export(df[["ID"] + S16_SHAPE_COLS + ["MF_UTIL_PATH_CLASS", "AVG_UTIL", TARGET, "DEFAULT_LABEL"]].round(4), "s16_client_shapes")
export(pd.concat([S16_SIL_U, S16_SIL_UL, S16_SIL_P, S16_SIL_PL, S16_SIL_S, S16_SIL_SL], ignore_index=True).round(4), "s16_shape_silhouette")
_b = S16_SHAPE_IV.iloc[0]
add_fact("16.2", "trajectory shape", f"Of six trajectory vocabularies, {_b['label']} separates default best (IV {_b['IV']:.2f}, class rates "
         f"{_b['min_rate']:.0%} to {_b['max_rate']:.0%}); utilisation SHAPE alone has IV "
         f"{S16_SHAPE_IV.set_index('label').loc['MF_UTIL_SHAPE', 'IV']:.2f}", float(_b["IV"]), "s16_util_x_pay_shape", 2)

**Observations:**
- Spend shape is the most informative trajectory vocabulary (IV 0.159, Cramér's V 0.17), ahead of payment-ratio level (0.102), payment-ratio shape (0.081) and utilisation shape (0.081).
- Spend level clusters carry almost nothing (IV 0.006).
- The three shape vocabularies overlap only partly: Cramér's V (0 = unrelated, 1 = identical) is 0.42 for utilisation vs spend, 0.27 for utilisation vs payment ratio and 0.18 for payment ratio vs spend.
- Story use: background for choosing Tableau dimensions; spend shape is the one to keep.

### 16.3 Build, pay down or tread water: what does the month-by-month balance regime say?
Each of the five month-on-month balance changes gets one letter:
**B** build (bill up > 5%), **D** pay-down (bill down > 5%), **F** flat (within ±5%), **Z** no balance (bill ≤ 0 in both
months). The change is measured against `max(|previous bill|, NT$ 1,000)` so tiny balances do not flip regimes.
Five letters give a regime word per client, e.g. `BBBBB` (five straight builds) or `DDDDD` (steady deleveraging).

In [ ]:
_rel = S16_DB / np.maximum(np.abs(S16_PREV), 1000)
_nobal = (S16_B[:, 1:] <= 0) & (S16_PREV <= 0)
S16_REG = np.where(_nobal, "Z", np.where(_rel > .05, "B", np.where(_rel < -.05, "D", "F")))
S16_REG_SEQ = np.array(["".join(r) for r in S16_REG])
S16_BUILD, S16_DOWN, S16_FLAT = S16_REG == "B", S16_REG == "D", S16_REG == "F"
df["MF_REGIME_SEQ"] = S16_REG_SEQ
df["MF_N_BUILD"] = S16_BUILD.sum(1)
df["MF_N_PAYDOWN"] = S16_DOWN.sum(1)
df["MF_N_FLAT"] = S16_FLAT.sum(1)
df["MF_REGIME_LAST"] = pd.Series(S16_REG[:, 4]).map({"B": "Build", "D": "Pay-down", "F": "Flat", "Z": "No balance"}).to_numpy()
S16_REG_NAMES = {"B": "Build", "D": "Pay-down", "F": "Flat", "Z": "No balance"}
S16_REG_COL = {"Build": C_DEF, "Pay-down": C_ACC, "Flat": C_NEU, "No balance": "#d9d9d9"}

_rows = []
for g, lab in ((0, "No default"), (1, "Default")):
    for k, pr in enumerate(S16_PAIRS):
        for L_, nm in S16_REG_NAMES.items():
            _rows.append({"DEFAULT_LABEL": lab, "pair": pr, "regime": nm, "share": float((S16_REG[S16_Y == g, k] == L_).mean())})
S16_REG_MIX = pd.DataFrame(_rows)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6), gridspec_kw={"width_ratios": [1, 1, 1.15]})
for ax, lab in zip(axes[:2], ("No default", "Default")):
    pv = S16_REG_MIX[S16_REG_MIX["DEFAULT_LABEL"] == lab].pivot(index="pair", columns="regime", values="share").reindex(S16_PAIRS)[list(S16_REG_NAMES.values())]
    pv.plot(kind="bar", stacked=True, ax=ax, color=[S16_REG_COL[c] for c in pv.columns], width=.8, legend=(lab == "Default"))
    ax.set_title(f"Balance regime mix by month pair: {lab}"); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
for col, lab, colr in (("MF_N_BUILD", "build months", C_DEF), ("MF_N_PAYDOWN", "pay-down months", C_ACC), ("MF_N_FLAT", "flat months", "k")):
    t = s16_tab(df[col], name="n_months")
    axes[2].plot(t["n_months"], t["rate"], marker="o", lw=2, color=colr, label=lab)
    axes[2].fill_between(t["n_months"], t["ci_lo"], t["ci_hi"], color=colr, alpha=.12)
axes[2].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[2].legend(); axes[2].set_xlabel("number of months (out of 5) in the regime")
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[2].set_title("Default rate by number of months in each regime")
plt.tight_layout(); savefig("s16_regime_mix_counts"); plt.show()
S16_REG_COUNTS = pd.concat([s16_tab(df[c], name="n_months").assign(regime=nm) for c, nm in
                            (("MF_N_BUILD", "Build"), ("MF_N_PAYDOWN", "Pay-down"), ("MF_N_FLAT", "Flat"))], ignore_index=True)
display(S16_REG_COUNTS.pivot(index="n_months", columns="regime", values="rate").round(4))
display(S16_REG_COUNTS.pivot(index="n_months", columns="regime", values="n"))
export(S16_REG_MIX, "s16_regime_mix"); export(S16_REG_COUNTS, "s16_regime_counts")

**Observations:**
- Clients with zero 'build' months (balance never rose more than 5%) default at 35.0% (7,890 clients); with 4 build months only 13.7%.
- Flat months point the other way: 0 flat months 16.8%, 3-5 flat months about 28.5%.
- The number of pay-down months barely matters (23.5% at 1 month down to 19.2% at 5).
- Story use: 'a balance that never grows is a warning sign' (numbers from the printed table; chart is image-only).

#### 16.3.1 Regime words: the most common, the riskiest and the safest 5-letter patterns
`BBBBB` is the textbook "debt spiral" and `DDDDD` the textbook "deleveraging". Top 15 patterns by size, then the
riskiest and safest patterns with at least 150 clients.

In [ ]:
S16_PAT = s16_tab(S16_REG_SEQ, name="pattern").sort_values("n", ascending=False).reset_index(drop=True)
S16_PAT["rank_by_n"] = np.arange(1, len(S16_PAT) + 1)
print(f"{len(S16_PAT)} distinct patterns; top 15 cover {S16_PAT['share'].head(15).sum():.1%} of clients")
display(S16_PAT.head(15)[["pattern", "n", "share", "rate", "ci_lo", "ci_hi", "lift", "expo_share"]].round(4))
_big = S16_PAT[S16_PAT["n"] >= 150].sort_values("rate", ascending=False)
print("riskiest patterns (n >= 150):"); display(_big.head(8)[["pattern", "n", "rate", "lift"]].round(4))
print("safest patterns (n >= 150):"); display(_big.tail(8)[["pattern", "n", "rate", "lift"]].round(4))
_named = S16_PAT.set_index("pattern").reindex(["BBBBB", "DDDDD", "FFFFF", "ZZZZZ"])[["n", "share", "rate", "lift", "expo_share"]]
_named.index = ["BBBBB debt spiral", "DDDDD deleveraging", "FFFFF treading water", "ZZZZZ no balance"]
print("the four pure patterns:"); display(_named.round(4))

fig, ax = plt.subplots(figsize=(14, 5))
_t = S16_PAT.head(25).sort_values("rate", ascending=False)
ax.bar(_t["pattern"], _t["rate"], color=[C_DEF if r > BASE_RATE else C_OK for r in _t["rate"]], alpha=.85)
ax.errorbar(np.arange(len(_t)), _t["rate"], yerr=[_t["rate"] - _t["ci_lo"], _t["ci_hi"] - _t["rate"]], fmt="none", ecolor="k", capsize=2, lw=1)
for i, (r, n) in enumerate(zip(_t["rate"], _t["n"])):
    ax.text(i, .005, f"{n:,}", ha="center", va="bottom", fontsize=7, rotation=90, color="white")
ax.axhline(BASE_RATE, ls="--", color="k", lw=1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title("Default rate of the 25 most common balance-regime patterns (B build, D pay-down, F flat, Z no balance; n inside bars)")
ax.tick_params(axis="x", rotation=60)
plt.tight_layout(); savefig("s16_regime_patterns"); plt.show()
export(S16_PAT, "s16_regime_patterns")
add_fact("16.3", "debt spiral", f"Five straight balance builds (BBBBB, {int(_named.iloc[0]['n']):,} clients) default at {_named.iloc[0]['rate']:.1%}, "
         f"five straight pay-downs (DDDDD, {int(_named.iloc[1]['n']):,}) at {_named.iloc[1]['rate']:.1%}, flat-for-five (FFFFF) at {_named.iloc[2]['rate']:.1%} "
         f"(base {BASE_RATE:.1%})", float(_named.iloc[0]["rate"]), "s16_regime_patterns", 3)

**Observations:**
- 369 distinct five-letter patterns exist; the top 15 cover only 30.3% of clients.
- The textbook 'debt spiral' BBBBB (484 clients) defaults at 16.5%, below base; 'deleveraging' DDDDD (193) at 19.2%; 'treading water' FFFFF (2,699, the most common) at 28.6%; 'no balance' ZZZZZ (966) at 35.9%.
- The riskiest patterns are flat with scattered pay-downs: FDFDF 61.3% (150 clients), FFDFD 52.2%, FFDFF 49.1%.
- The safest patterns mix builds and pay-downs (BBDDB 8.2%, DDBDB 10.1%): an actively used and repaid card.
- Story use: headline counter-intuition: 'the debt spiral is one of the safest patterns'.

#### 16.3.2 Regime persistence and the last two moves
Left: pooled transition matrix between consecutive regimes (is building habit-forming?). Right: default rate by the
last two regimes (Jul→Aug then Aug→Sep), the part of the word a collections team would see most recently.

In [ ]:
_frm, _to = S16_REG[:, :-1].ravel(), S16_REG[:, 1:].ravel()
S16_REG_TRANS = pd.crosstab(pd.Series(_frm).map(S16_REG_NAMES), pd.Series(_to).map(S16_REG_NAMES), normalize="index").reindex(
    index=list(S16_REG_NAMES.values()), columns=list(S16_REG_NAMES.values()))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.3))
sns.heatmap(S16_REG_TRANS, annot=True, fmt=".0%", cmap="Blues", cbar=False, ax=axes[0], linewidths=.5)
axes[0].set_title("Regime transition (row = this month pair, column = next)"); axes[0].set_xlabel(""); axes[0].set_ylabel("")
S16_LAST2 = s16_heat(pd.Series(S16_REG[:, 3]).map(S16_REG_NAMES), pd.Series(S16_REG[:, 4]).map(S16_REG_NAMES), axes[1],
                     "Default rate: Jul→Aug regime (rows) × Aug→Sep regime (columns)", list(S16_REG_NAMES.values()), list(S16_REG_NAMES.values()), fs=9)
plt.tight_layout(); savefig("s16_regime_transitions"); plt.show()
display(S16_REG_TRANS.round(3))
S16_LAST2.columns = ["regime_jul_aug", "regime_aug_sep", "rate", "n"]
export(S16_LAST2, "s16_regime_last2")
display(s16_tab(df["MF_REGIME_LAST"], list(S16_REG_NAMES.values()), "regime_aug_sep").round(4))

**Observations:**
- Regimes are sticky: Flat stays Flat 63.6% of the time, No balance stays 72.0%, Build stays Build 41.9%.
- The latest move (Aug→Sep) matters: Build 15.1%, Pay-down 21.7%, Flat 26.9%, No balance 28.2%.
- Flat in Aug→Sep covers 38% of clients but 69% of the money owed by defaulters.
- Story use: a simple recent-behaviour rule for collections: 'a balance that stopped moving'.

### 16.4 Is the "debt spiral" really the dangerous one? Rule-based flow archetypes
A banker's intuition ranks borrowers roughly: *debt spiral* (worst) > *treading water* > everyone else >
*binge then repay* > *deleverager* > *full payer*. Each client is assigned to the **first** rule that fires:

| Order | Archetype | Rule |
|---|---|---|
| 1 | No balance | bill ≤ 0 in all six months |
| 2 | Debt spiral, token payments | ≥ 3 **consecutive** build months (bill up > 5%) and, in those months, payments averaging < 10% of the previous bill |
| 3 | Debt spiral, pays but charges more | ≥ 3 consecutive build months, payments ≥ 10% of the previous bill (or no previous bill) |
| 4 | Binge then repay | one month (May..Aug) where the bill jumps by > 20% of the limit, and at least half of that jump is gone by Sep |
| 5 | Deleverager | ≥ 3 consecutive pay-down months and Sep bill < 70% of a positive Apr bill |
| 6 | Treading water | ≥ 4 flat months, mean bill > NT$ 5,000, median payment ≤ 10% of the previous bill (interest and little more) |
| 7 | Full payer | payment ≥ 95% of the previous bill in every month with a balance (≥ 3 such months) |
| 8 | Other / mixed | none of the above |

Note on the spiral rule: "payment below new charges" is **the same event** as "bill went up", by the accounting
identity. The cell verifies this, so the rule needs only the consecutive-build condition.

In [ ]:
_same = ((S16_P[:, 1:] < S16_SP) == (S16_DB > 0)).mean()
print(f"'payment < new charges' coincides with 'bill rose' in {_same:.2%} of client-months (accounting identity)")
S16_BUILD_RUN = s16_max_run(S16_BUILD)
S16_DOWN_RUN = s16_max_run(S16_DOWN)
df["MF_BUILD_RUN_MAX"] = S16_BUILD_RUN
df["MF_PAYDOWN_RUN_MAX"] = S16_DOWN_RUN
_okpr = S16_BUILD & ~np.isnan(S16_PR)
_prb = np.where(_okpr.sum(1) > 0, np.where(_okpr, S16_PR, 0).sum(1) / np.maximum(_okpr.sum(1), 1), np.nan)
_spiral = S16_BUILD_RUN >= 3
_token = _spiral & (_prb < 0.10)
_binge = np.zeros(S16_N, bool)
for k in range(4):
    _binge |= (S16_DB[:, k] / S16_LIM > 0.20) & (S16_B[:, 5] <= S16_B[:, k + 1] - 0.5 * S16_DB[:, k])
_delev = (S16_DOWN_RUN >= 3) & (S16_B[:, 0] > 0) & (S16_B[:, 5] < 0.7 * S16_B[:, 0])
_prmed = np.nanmedian(np.where(np.isnan(S16_PR).all(1)[:, None], 0, S16_PR), axis=1)
_tread = (S16_FLAT.sum(1) >= 4) & (S16_B.mean(1) > 5000) & (_prmed <= 0.10) & ~np.isnan(S16_PR).all(1)
_full = ((~np.isnan(S16_PR)).sum(1) >= 3) & (np.where(np.isnan(S16_PR), np.inf, S16_PR).min(1) >= 0.95)
_nobal6 = (S16_B <= 0).all(1)
S16_ARCH_ORDER = ["Debt spiral, token payments", "Debt spiral, pays but charges more", "Treading water", "Other / mixed",
                  "Binge then repay", "Deleverager", "Full payer", "No balance"]       # = naive risk ranking, worst first
S16_ARCH = np.select([_nobal6, _token, _spiral, _binge, _delev, _tread, _full],
                     ["No balance", "Debt spiral, token payments", "Debt spiral, pays but charges more", "Binge then repay",
                      "Deleverager", "Treading water", "Full payer"], "Other / mixed")
df["MF_FLOW_ARCHETYPE"] = pd.Categorical(S16_ARCH, S16_ARCH_ORDER)
S16_T_ARCH = s16_tab(S16_ARCH, S16_ARCH_ORDER, "archetype")
S16_T_ARCH["naive_rank"] = np.arange(1, len(S16_T_ARCH) + 1)
S16_T_ARCH["actual_rank"] = S16_T_ARCH["rate"].rank(ascending=False).astype(int)
_prof = pd.DataFrame({"archetype": S16_ARCH, "avg_util": df["AVG_UTIL"].to_numpy(), "limit_median": S16_LIM, "sep_late_share": (S16_ST[:, 5] >= 1).astype(float),
                      "ever_late_share": (df["N_LATE"].to_numpy() > 0).astype(float)}).groupby("archetype").agg(
    avg_util=("avg_util", "mean"), limit_median=("limit_median", "median"), sep_late_share=("sep_late_share", "mean"), ever_late_share=("ever_late_share", "mean")).reset_index()
S16_T_ARCH = S16_T_ARCH.merge(_prof, on="archetype")
display(S16_T_ARCH[["archetype", "n", "share", "rate", "ci_lo", "ci_hi", "lift", "naive_rank", "actual_rank", "expo_share", "expo_default_share",
                    "avg_util", "limit_median", "sep_late_share", "ever_late_share"]].round(4))
S16_ARCH_RHO = stats.spearmanr(S16_T_ARCH["naive_rank"], S16_T_ARCH["actual_rank"])[0]
print(f"Spearman correlation between the naive risk ranking and the actual one: {S16_ARCH_RHO:+.2f}")

fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={"width_ratios": [1.25, 1]})
s16_bars(S16_T_ARCH, "archetype", axes[0], "Default rate by flow archetype (ordered by NAIVE risk, worst expected on the left)", rot=25)
x = np.arange(len(S16_T_ARCH))
axes[1].bar(x - .2, S16_T_ARCH["share"], .4, color=C_NEU, label="share of clients")
axes[1].bar(x + .2, S16_T_ARCH["expo_share"], .4, color=C_OK, label="share of Sep exposure")
axes[1].plot(x, S16_T_ARCH["expo_default_share"], "o", color=C_DEF, label="share of defaulted exposure")
axes[1].set_xticks(x); axes[1].set_xticklabels(S16_T_ARCH["archetype"], rotation=25, ha="right")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].legend(); axes[1].set_title("Size and exposure by archetype")
plt.tight_layout(); savefig("s16_flow_archetypes"); plt.show()
export(S16_T_ARCH, "s16_flow_archetypes")
_a = S16_T_ARCH.set_index("archetype")
add_fact("16.4", "debt spiral", f"Debt-spiral clients (>=3 consecutive builds) default at {S16_Y[_spiral].mean():.1%} "
         f"(token payers {_a.loc['Debt spiral, token payments', 'rate']:.1%}, active payers {_a.loc['Debt spiral, pays but charges more', 'rate']:.1%}) versus "
         f"{_a.loc['Deleverager', 'rate']:.1%} for deleveragers and {_a.loc['Treading water', 'rate']:.1%} for those treading water; "
         f"naive vs actual risk ranking correlate {S16_ARCH_RHO:+.2f}", float(S16_Y[_spiral].mean()), "s16_flow_archetypes", 3)

**Observations:**
- The banker's ranking is backwards: Spearman correlation between the naive and actual risk rank is −0.29.
- Both debt-spiral archetypes are below base: 'token payments' 17.9%, 'pays but charges more' 12.9%.
- 'Treading water' (6,037 clients, 20%) defaults at 27.7% and holds 43% of the money owed by defaulters; 'No balance' is riskiest at 35.9%.
- Full payers are safest at 13.0% but still not zero.
- Story use: centrepiece of the money chapter: 'the debt spiral is not the danger; the stuck balance is'.

#### 16.4.1 Does the archetype add anything once the September status is known?
Same archetypes within each Sep status group (cells with n < 40 blank). If the archetype ranking survives inside a
column, flows carry information the status code does not; if the columns look uniform, the status code already
contains the story.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
S16_ARCH_STATUS = s16_heat(S16_ARCH, S16_GRP_SEP, ax, "Default rate: flow archetype × Sep status group (rate, n)", S16_ARCH_ORDER, STATUS_GROUP_ORDER, min_n=40, fs=9)
plt.tight_layout(); savefig("s16_archetype_x_status"); plt.show()
S16_ARCH_STATUS.columns = ["archetype", "status_grp_sep", "rate", "n"]
export(S16_ARCH_STATUS, "s16_archetype_x_status")
_w = S16_ARCH_STATUS[S16_ARCH_STATUS["n"] >= 40]
S16_ARCH_SPREAD = _w.groupby("status_grp_sep")["rate"].agg(spread=lambda s: s.max() - s.min(), n_archetypes="size").reindex(STATUS_GROUP_ORDER)
print("spread of default rate across archetypes within each Sep status group:"); display(S16_ARCH_SPREAD.round(4))
_std = (pd.DataFrame({"a": S16_ARCH, "g": S16_GRP_SEP, "y": S16_Y}).groupby(["a", "g"])["y"].mean().unstack()
        .reindex(columns=STATUS_GROUP_ORDER))
_wts = pd.Series(S16_GRP_SEP).value_counts(normalize=True).reindex(STATUS_GROUP_ORDER)
S16_ARCH_ADJ = pd.DataFrame({"raw_rate": S16_T_ARCH.set_index("archetype")["rate"],
                             "status_standardised_rate": (_std.fillna(_std.mean()) * _wts).sum(1)}).reindex(S16_ARCH_ORDER)
print("raw vs status-standardised default rate (each archetype re-weighted to the portfolio's Sep status mix):")
display(S16_ARCH_ADJ.round(4))
add_fact("16.4", "debt spiral", f"After standardising for Sep status, archetype default rates span {S16_ARCH_ADJ['status_standardised_rate'].min():.1%} to "
         f"{S16_ARCH_ADJ['status_standardised_rate'].max():.1%} (raw span {S16_ARCH_ADJ['raw_rate'].min():.1%} to {S16_ARCH_ADJ['raw_rate'].max():.1%})",
         float(S16_ARCH_ADJ["status_standardised_rate"].max() - S16_ARCH_ADJ["status_standardised_rate"].min()), "s16_archetype_x_status", 2)

**Observations:**
- Inside each Sep status group archetypes still differ: by 35.5 points among Late 2m, 29.7 among Late 1m, but only 8.3 among Revolving.
- Re-weighting each archetype to the portfolio's Sep status mix narrows the range from 12.9%-35.9% to 17.3%-24.2%.
- 'No balance' falls from 35.9% to 23.8%: most of its risk came from its status mix (63% late in Sep). Debt-spiral archetypes stay below base (17.3% and 21.4%).
- Story use: caveat bullet: archetypes add some signal, but much less than the status code.

### 16.5 How do payment *sizes* move from month to month, and is a payment shock a warning?
Each payment is compared with the client's own previous payment. Eight mutually exclusive moves: zero→zero,
restart (0 → paid), stop (paid → 0), drop > 50%, drop 10–50%, stable ±10%, rise 10–100%, rise > 100%.
Left: distribution of `log2(payment / previous payment)` when both are positive. Right: composition of moves by outcome.

In [ ]:
_p0, _p1 = S16_P[:, :-1], S16_P[:, 1:]
_ratio = _p1 / np.where(_p0 > 0, _p0, 1)
S16_PMOVE_ORDER = ["Zero → zero", "Stop (paid → 0)", "Drop > 50%", "Drop 10–50%", "Stable ±10%", "Rise 10–100%", "Rise > 100%", "Restart (0 → paid)"]
S16_PMOVE = np.select([(_p0 == 0) & (_p1 == 0), (_p0 == 0) & (_p1 > 0), (_p0 > 0) & (_p1 == 0), _ratio < .5, _ratio < .9, _ratio <= 1.1, _ratio <= 2],
                      ["Zero → zero", "Restart (0 → paid)", "Stop (paid → 0)", "Drop > 50%", "Drop 10–50%", "Stable ±10%", "Rise 10–100%"], "Rise > 100%")
_both = (_p0 > 0) & (_p1 > 0)
fig, axes = plt.subplots(1, 2, figsize=(17, 4.8))
for g, lab, col in ((0, "No default", C_OK), (1, "Default", C_DEF)):
    v = np.log2(_ratio[(S16_Y == g)[:, None] & _both])
    axes[0].hist(np.clip(v, -6, 6), bins=np.linspace(-6, 6, 97), density=True, alpha=.5, color=col, label=f"{lab} (median {np.median(v):+.2f})")
axes[0].set_title("log2(payment / previous payment), both positive, pooled over 5 month pairs"); axes[0].legend(); axes[0].set_xlabel("log2 ratio (0 = same payment)")
S16_PMOVE_MIX = pd.DataFrame({lab: pd.Series(S16_PMOVE[S16_Y == g].ravel()).value_counts(normalize=True) for g, lab in ((0, "No default"), (1, "Default"))}).reindex(S16_PMOVE_ORDER)
S16_PMOVE_MIX.plot(kind="barh", ax=axes[1], color=[C_OK, C_DEF])
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("Share of month-on-month payment moves by outcome"); axes[1].invert_yaxis()
plt.tight_layout(); savefig("s16_payment_growth"); plt.show()
S16_PMOVE_MIX["default_share_of_move"] = [S16_Y[:, None].repeat(5, 1)[S16_PMOVE == m_].mean() for m_ in S16_PMOVE_ORDER]
display(S16_PMOVE_MIX.round(4))
df["MF_PAY_LAST_MOVE"] = pd.Categorical(S16_PMOVE[:, 4], S16_PMOVE_ORDER)
S16_T_LASTMOVE = s16_tab(S16_PMOVE[:, 4], S16_PMOVE_ORDER, "aug_sep_payment_move")
print("default rate by the LAST payment move (Aug → Sep):"); display(S16_T_LASTMOVE[["aug_sep_payment_move", "n", "share", "rate", "ci_lo", "ci_hi", "lift"]].round(4))
export(S16_PMOVE_MIX.reset_index().rename(columns={"index": "payment_move"}), "s16_payment_moves")
export(S16_T_LASTMOVE, "s16_payment_last_move")

**Observations:**
- Defaulters have more zero→zero months (15.5% vs 10.1% of moves), more stops (13.3% vs 7.1%) and more restarts (13.9% vs 8.6%).
- By the last move (Aug→Sep): stopping payment 38.5%, zero→zero 33.5%, restarting 33.2%.
- Cutting a payment by more than half but still paying is safe at 15.8%, below the 18.2% for a stable payment.
- Story use: 'paying something matters more than how much'; MF_PAY_LAST_MOVE is the second-strongest lens (16.13).

#### 16.5.1 Payment shocks: a payment that falls by more than half
A **shock** = this month's payment is less than 50% of last month's positive payment (includes stopping altogether).
How many shocks does a client have, how often does one occur right before default, and does *when* it happened matter?

In [ ]:
S16_SHOCK = (_p0 > 0) & (_p1 < 0.5 * _p0)
df["MF_N_PAY_SHOCK"] = S16_SHOCK.sum(1)
df["MF_PAY_SHOCK_SEP"] = S16_SHOCK[:, 4].astype(int)
_last = np.where(S16_SHOCK.any(1), np.array(S16_M5)[4 - np.argmax(S16_SHOCK[:, ::-1], axis=1)], "No shock")
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
S16_T_NSHOCK = s16_tab(df["MF_N_PAY_SHOCK"], name="n_shocks")
s16_bars(S16_T_NSHOCK, "n_shocks", axes[0], "Default rate by number of payment shocks (5 month pairs)")
S16_T_LASTSHOCK = s16_tab(_last, ["No shock"] + S16_M5, "month_of_last_shock")
s16_bars(S16_T_LASTSHOCK, "month_of_last_shock", axes[1], "Default rate by month of the most recent shock")
_sm = pd.DataFrame({lab: S16_SHOCK[S16_Y == g].mean(0) for g, lab in ((0, "No default"), (1, "Default"))}, index=S16_M5)
_sm.plot(kind="bar", ax=axes[2], color=[C_OK, C_DEF]); axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[2].set_title("Share of clients with a payment shock, by month"); axes[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); savefig("s16_payment_shocks"); plt.show()
display(S16_T_NSHOCK[["n_shocks", "n", "share", "rate", "lift"]].round(4)); display(S16_T_LASTSHOCK[["month_of_last_shock", "n", "share", "rate", "lift"]].round(4))
display(_sm.round(4))
_any = S16_SHOCK.any(1)
print(f"any shock: {_any[S16_Y == 1].mean():.1%} of defaulters vs {_any[S16_Y == 0].mean():.1%} of non-defaulters | "
      f"shock in Sep: {S16_SHOCK[S16_Y == 1, 4].mean():.1%} vs {S16_SHOCK[S16_Y == 0, 4].mean():.1%}")
# shock in Sep, controlling for Sep status
S16_SHOCK_STATUS = (pd.DataFrame({"g": S16_GRP_SEP, "shock_sep": np.where(S16_SHOCK[:, 4], "Shock in Sep", "No shock in Sep"), "y": S16_Y})
                    .groupby(["g", "shock_sep"])["y"].agg(rate="mean", n="size").unstack().reindex(STATUS_GROUP_ORDER))
print("Sep payment shock within Sep status group:"); display(S16_SHOCK_STATUS.round(4))
export(S16_T_NSHOCK, "s16_payment_shock_counts"); export(S16_T_LASTSHOCK, "s16_payment_shock_last_month")
add_fact("16.5", "payment dynamics", f"A payment shock (payment falls >50% m/m) hit {_any[S16_Y == 1].mean():.0%} of defaulters and {_any[S16_Y == 0].mean():.0%} of "
         f"non-defaulters at least once; default rate with a Sep shock is {S16_Y[S16_SHOCK[:, 4]].mean():.1%} vs {S16_Y[~S16_SHOCK[:, 4]].mean():.1%} without",
         float(S16_Y[S16_SHOCK[:, 4]].mean()), "s16_payment_shocks", 2)

**Observations:**
- Clients with no payment shock (payment falling by more than half) default at 18.6%; with 3 shocks at 26.6%.
- 67.4% of defaulters had at least one shock vs 59.3% of non-defaulters; a shock in Sep lifts the rate to 26.2%.
- Inside each Sep status group the shock adds almost nothing (Revolving 13.0% without vs 10.8% with; Late 2m 69.6% vs 67.3%), so it mostly repeats the status code.
- Story use: background only.

#### 16.5.2 One-off big payments: the bonus / settlement signature
A **big payment** = more than 5× the client's own median payment (median > 0) and at least NT$ 5,000. Is a client who
made one safer afterwards, and does it matter whether the windfall was early (Apr) or recent (Sep)?
The lower table looks one month ahead: after a big-payment month, how often is the client late the next month?

In [ ]:
_med = np.median(S16_P, axis=1)
S16_BIG = (S16_P > 5 * _med[:, None]) & (_med[:, None] > 0) & (S16_P >= 5000)
df["MF_BIG_PAY_ANY"] = S16_BIG.any(1).astype(int)
_bm = np.where(S16_BIG.any(1), np.array(S16_M6)[5 - np.argmax(S16_BIG[:, ::-1], axis=1)], np.where(_med > 0, "None", "n/a (median payment 0)"))
df["MF_BIG_PAY_MONTH"] = pd.Categorical(_bm, ["None"] + S16_M6 + ["n/a (median payment 0)"])
S16_T_BIG = s16_tab(_bm, ["None"] + S16_M6 + ["n/a (median payment 0)"], "month_of_last_big_payment")
fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
s16_bars(S16_T_BIG, "month_of_last_big_payment", axes[0], "Default rate by month of the most recent big payment (>5× own median)", rot=20)
_rows = []
for k in range(5):                                   # big payment in Apr..Aug -> status in the following month
    for flag, nm in ((True, "After a big payment"), (False, "After a normal payment")):
        m = (S16_BIG[:, k] == flag) & (_med > 0)
        _rows.append({"month_of_payment": S16_M6[k], "group": nm, "n": int(m.sum()), "late_same_month": float((S16_ST[m, k] >= 1).mean()),
                      "late_next_month": float((S16_ST[m, k + 1] >= 1).mean()), "default_rate": float(S16_Y[m].mean()),
                      "median_bill_change_next_month_pct_limit": float(np.median(S16_DB[m, k] / S16_LIM[m]))})
S16_BIG_NEXT = pd.DataFrame(_rows)
_pv = S16_BIG_NEXT.pivot(index="month_of_payment", columns="group", values="late_next_month").reindex(S16_M6[:5])
_pv.plot(kind="bar", ax=axes[1], color=[C_ACC, C_NEU]); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_title("Share late (status ≥ 1) in the month AFTER the payment"); axes[1].tick_params(axis="x", rotation=0); axes[1].set_xlabel("month of the payment")
plt.tight_layout(); savefig("s16_big_payments"); plt.show()
display(S16_T_BIG[["month_of_last_big_payment", "n", "share", "rate", "ci_lo", "ci_hi", "lift"]].round(4))
display(S16_BIG_NEXT.round(4))
export(S16_T_BIG, "s16_big_payment_month"); export(S16_BIG_NEXT, "s16_big_payment_next_month")
_bt = S16_T_BIG.set_index("month_of_last_big_payment")["rate"]
add_fact("16.5", "payment dynamics", f"{df['MF_BIG_PAY_ANY'].mean():.0%} of clients made a one-off big payment (>5× own median); they default at "
         f"{S16_Y[S16_BIG.any(1)].mean():.1%} vs {S16_Y[(~S16_BIG.any(1)) & (_med > 0)].mean():.1%} for regular payers; big payment in Sep {_bt.get('Sep', np.nan):.1%}, "
         f"in Apr {_bt.get('Apr', np.nan):.1%}", float(S16_Y[S16_BIG.any(1)].mean()), "s16_big_payments", 2)

**Observations:**
- Clients whose last big payment (more than 5× their own median) came in Aug default at 11.5%, in Sep at 14.6%, in Apr at 19.5%, vs 22.2% with no big payment.
- After a big payment in Aug, 14.5% are late the next month vs 20.2% after a normal payment.
- Clients whose median payment is 0 default at 30.8%.
- Story use: a 'windfall' positive signal; fresher windfalls protect more.

#### 16.5.3 Last-month payment collapse: September payment vs the client's own Apr–Aug average
The most recent payment relative to the client's own history. Is "paid much less than usual in September" a distinct
warning, and does it survive inside each Sep status group?

In [ ]:
_prior = S16_P[:, :5].mean(1)
_r = S16_P[:, 5] / np.where(_prior > 0, _prior, 1)
df["MF_PAY_SEP_VS_PRIOR"] = np.where(_prior > 0, np.minimum(_r, 20), np.nan)
S16_COLL_ORDER = ["No prior payments", "Sep = 0", "<25%", "25–75%", "75–125%", "125–300%", ">300%"]
S16_COLL = np.select([_prior <= 0, S16_P[:, 5] == 0, _r < .25, _r < .75, _r <= 1.25, _r <= 3], S16_COLL_ORDER[:6], ">300%")
df["MF_PAY_COLLAPSE_BAND"] = pd.Categorical(S16_COLL, S16_COLL_ORDER, ordered=True)
S16_T_COLL = s16_tab(S16_COLL, S16_COLL_ORDER, "sep_payment_vs_own_mean")
fig, axes = plt.subplots(1, 2, figsize=(18, 4.8), gridspec_kw={"width_ratios": [1, 1.2]})
s16_bars(S16_T_COLL, "sep_payment_vs_own_mean", axes[0], "Default rate by Sep payment / own Apr–Aug mean payment", rot=15)
S16_COLL_STATUS = s16_heat(S16_COLL, S16_GRP_SEP, axes[1], "Same, within Sep status group (rate, n)", S16_COLL_ORDER, STATUS_GROUP_ORDER, min_n=40)
plt.tight_layout(); savefig("s16_payment_collapse"); plt.show()
display(S16_T_COLL[["sep_payment_vs_own_mean", "n", "share", "rate", "ci_lo", "ci_hi", "lift", "expo_share"]].round(4))
S16_COLL_STATUS.columns = ["sep_payment_vs_own_mean", "status_grp_sep", "rate", "n"]
export(S16_T_COLL, "s16_payment_collapse"); export(S16_COLL_STATUS, "s16_payment_collapse_x_status")
_c = S16_T_COLL.set_index("sep_payment_vs_own_mean")["rate"]
add_fact("16.5", "payment dynamics", f"Clients whose Sep payment was zero after paying earlier default at {_c.get('Sep = 0', np.nan):.1%}, those who paid "
         f"75-125% of their usual amount at {_c.get('75–125%', np.nan):.1%}, and those who paid >3× their usual at {_c.get('>300%', np.nan):.1%}",
         float(_c.get("Sep = 0", np.nan)), "s16_payment_collapse", 2)

**Observations:**
- Paying nothing in Sep after paying before is the warning: 35.2% default (3,817 clients); no prior payments at all 34.7%.
- A smaller-than-usual payment is not a warning: paying 25-75% of the usual amount defaults at 13.9% and under 25% at 15.4%, both below the 18.3% for a normal-sized payment.
- Paying more than 3× the usual amount sits near base (22.6%).
- Story use: sharpens the 16.5 message: the line is between zero and some payment.

### 16.6 What happens to *spending* before default: a last hurrah, or a card that has gone quiet?
Implied spend (new charges incl. interest and fees) exists for May..Sep. First the path by outcome, split by whether the
client is late in September: the share of clients with a material new charge (> 2% of the limit) and the mean spend
as a share of the limit.

In [ ]:
_late_sep = S16_ST[:, 5] >= 1
S16_SPEND_GROUPS = [("No default, current in Sep", (S16_Y == 0) & ~_late_sep, C_OK, "-"), ("No default, late in Sep", (S16_Y == 0) & _late_sep, C_OK, "--"),
                    ("Default, current in Sep", (S16_Y == 1) & ~_late_sep, C_DEF, "-"), ("Default, late in Sep", (S16_Y == 1) & _late_sep, C_DEF, "--")]
_rows = []
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
for nm, m, col, ls in S16_SPEND_GROUPS:
    act = (S16_SP[m] > 0.02 * S16_LIM[m, None]).mean(0)
    mean_sl = np.clip(S16_SP[m] / S16_LIM[m, None], -1, 2).mean(0)
    med_sp = np.median(S16_SP[m], 0)
    axes[0].plot(S16_M5, act, marker="o", color=col, ls=ls, lw=2, label=f"{nm} (n={m.sum():,})")
    axes[1].plot(S16_M5, mean_sl, marker="o", color=col, ls=ls, lw=2)
    axes[2].plot(S16_M5, med_sp, marker="o", color=col, ls=ls, lw=2)
    for k, mth in enumerate(S16_M5):
        _rows.append({"group": nm, "MONTH": mth, "n": int(m.sum()), "share_material_spend": act[k], "mean_spend_over_limit": mean_sl[k], "median_spend": med_sp[k]})
axes[0].set_title("Share of clients with new charges > 2% of limit"); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].legend(fontsize=8)
axes[1].set_title("Mean new charges / limit"); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=1))
axes[2].set_title("Median new charges (NT$)")
plt.tight_layout(); savefig("s16_spend_paths"); plt.show()
S16_SPEND_PATH = pd.DataFrame(_rows)
display(S16_SPEND_PATH.pivot(index="group", columns="MONTH", values="share_material_spend")[S16_M5].round(4))
display(S16_SPEND_PATH.pivot(index="group", columns="MONTH", values="mean_spend_over_limit")[S16_M5].round(4))
export(S16_SPEND_PATH, "s16_spend_paths")

**Observations:**
- Among non-defaulters current in Sep, the share with material new charges (more than 2% of limit) rises from 41.8% in May to 54.6% in Sep.
- Defaulters late in Sep start lower (32.3%) and stop climbing after Jul (45.2% → 43.8%).
- Mean new charges as a share of limit fall for late defaulters from 5.4% in Jul to 3.1% in Sep; non-defaulters current in Sep stay at 6.6%.
- Story use: 'defaulters' cards go quiet before default': no spending binge.

#### 16.6.1 Frozen months: a balance but no new purchases
A **frozen month** = the bill is positive but new charges are at most 2% of the previous bill (about the size of one
month's interest), i.e. the client owes money and is not using the card. A run of frozen months ending in September is
the signature of a blocked, cut-up or abandoned card. How common is it and what does it predict? The right panel checks
the "blocked account" reading directly: are frozen months more often delinquent months?

In [ ]:
S16_FROZEN = (S16_B[:, 1:] > 0) & (S16_SP <= 0.02 * np.clip(S16_PREV, 0, None))
df["MF_N_FROZEN"] = S16_FROZEN.sum(1)
df["MF_FROZEN_RECENT"] = s16_end_run(S16_FROZEN)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
S16_T_NFROZEN = s16_tab(df["MF_N_FROZEN"], name="n_frozen_months")
s16_bars(S16_T_NFROZEN, "n_frozen_months", axes[0], "Default rate by number of frozen months (of 5)")
S16_T_FRZRUN = s16_tab(df["MF_FROZEN_RECENT"], name="frozen_run_ending_sep")
s16_bars(S16_T_FRZRUN, "frozen_run_ending_sep", axes[1], "Default rate by length of the frozen run ending in Sep")
_stn = S16_ST[:, 1:]
_bal = S16_B[:, 1:] > 0
S16_FROZEN_STATUS = pd.DataFrame({
    "Frozen month": pd.Series(np.select([_stn[S16_FROZEN] <= -1, _stn[S16_FROZEN] == 0, _stn[S16_FROZEN] <= 2], ["Paid / inactive", "Revolving", "Late 1-2m"], "Late 3m+")).value_counts(normalize=True),
    "Active month with balance": pd.Series(np.select([_stn[_bal & ~S16_FROZEN] <= -1, _stn[_bal & ~S16_FROZEN] == 0, _stn[_bal & ~S16_FROZEN] <= 2],
                                                     ["Paid / inactive", "Revolving", "Late 1-2m"], "Late 3m+")).value_counts(normalize=True)}).reindex(
    ["Paid / inactive", "Revolving", "Late 1-2m", "Late 3m+"])
S16_FROZEN_STATUS.T.plot(kind="barh", stacked=True, ax=axes[2], color=["#2b8cbe", "#74c476", "#e6550d", "#a50f15"])
axes[2].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[2].set_title("Status code in frozen vs active client-months"); axes[2].legend(fontsize=8, ncol=2, loc="lower right")
plt.tight_layout(); savefig("s16_frozen_months"); plt.show()
display(S16_T_NFROZEN[["n_frozen_months", "n", "share", "rate", "ci_lo", "ci_hi", "lift"]].round(4))
display(S16_T_FRZRUN[["frozen_run_ending_sep", "n", "share", "rate", "ci_lo", "ci_hi", "lift"]].round(4))
display(S16_FROZEN_STATUS.round(4))
print(f"frozen share of client-months with a balance: {S16_FROZEN.sum() / _bal.sum():.1%} | by outcome: "
      f"defaulters {S16_FROZEN[S16_Y == 1].sum() / _bal[S16_Y == 1].sum():.1%}, others {S16_FROZEN[S16_Y == 0].sum() / _bal[S16_Y == 0].sum():.1%}")
_fz = S16_FROZEN[:, 4]
S16_FROZEN_X_STATUS = (pd.DataFrame({"g": S16_GRP_SEP, "f": np.where(_fz, "Frozen in Sep", "Not frozen in Sep"), "y": S16_Y})[S16_B[:, 5] > 0]
                       .groupby(["g", "f"])["y"].agg(rate="mean", n="size").unstack().reindex(STATUS_GROUP_ORDER))
print("frozen in Sep vs not (clients with a Sep balance), within Sep status group:"); display(S16_FROZEN_X_STATUS.round(4))
export(S16_T_NFROZEN, "s16_frozen_counts"); export(S16_T_FRZRUN, "s16_frozen_recent_run")
_fs = S16_FROZEN_STATUS
add_fact("16.6", "spend dynamics", f"{_fs.loc[['Late 1-2m', 'Late 3m+'], 'Frozen month'].sum():.0%} of frozen client-months (balance, no new charges) carry a late status vs "
         f"{_fs.loc[['Late 1-2m', 'Late 3m+'], 'Active month with balance'].sum():.0%} of active months; clients frozen in all 5 months default at "
         f"{S16_Y[df['MF_N_FROZEN'].to_numpy() == 5].mean():.1%}, never-frozen at {S16_Y[df['MF_N_FROZEN'].to_numpy() == 0].mean():.1%}",
         float(S16_Y[df["MF_N_FROZEN"].to_numpy() == 5].mean()), "s16_frozen_months", 3)

**Observations:**
- A frozen month (balance owed, no new purchases) occurs in 21.7% of defaulters' months with a balance vs 15.3% for others.
- Never frozen: 18.5% default; two frozen months: 32.0%; frozen all 5 months: 36.5% (515 clients).
- Unlike payment shocks, this survives the status control: frozen in Sep is riskier in every Sep status group (Revolving 17.6% vs 11.7%, Late 3m+ 80.2% vs 69.3%).
- Story use: one of the few money signals that adds to the status code; strong candidate for the dashboard.

#### 16.6.2 Last hurrah or fade-out: late-window spend relative to early-window spend
`late` = mean positive spend in Aug–Sep, `early` = mean positive spend in May–Jul. The ratio is banded, and a **Sep
surge** flag marks a September spend more than 3× the May–Aug mean (and ≥ NT$ 5,000 and ≥ 5% of the limit).
A simple spend-pattern label combines the pieces.

In [ ]:
_spp = np.clip(S16_SP, 0, None)
_early, _latew = _spp[:, :3].mean(1), _spp[:, 3:].mean(1)
_lr = _latew / np.where(_early > 0, _early, 1)
df["MF_SPEND_LATE_RATIO"] = np.where(_early > 0, np.minimum(_lr, 20), np.nan)
S16_HURRAH_ORDER = ["No spend at all", "Stopped (early > 0, late = 0)", "<50%", "50–150%", "150–300%", ">300%", "New spender (early = 0)"]
S16_HURRAH = np.select([(_early <= 0) & (_latew <= 0), _early <= 0, _latew <= 0, _lr < .5, _lr <= 1.5, _lr <= 3],
                       ["No spend at all", "New spender (early = 0)", "Stopped (early > 0, late = 0)", "<50%", "50–150%", "150–300%"], ">300%")
_prior4 = _spp[:, :4].mean(1)
S16_SURGE = (S16_SP[:, 4] > 3 * _prior4) & (S16_SP[:, 4] >= 5000) & (S16_SP[:, 4] >= 0.05 * S16_LIM)
df["MF_SPEND_SURGE_SEP"] = S16_SURGE.astype(int)
S16_SPAT_ORDER = ["No spend", "Stopped using (balance remains)", "Fading", "Regular", "Sep surge"]
S16_SPAT = np.select([(_spp.sum(1) <= 0), df["MF_FROZEN_RECENT"].to_numpy() >= 2, S16_SURGE, (_early > 0) & (_lr < .5)],
                     ["No spend", "Stopped using (balance remains)", "Sep surge", "Fading"], "Regular")
df["MF_SPEND_PATTERN"] = pd.Categorical(S16_SPAT, S16_SPAT_ORDER)
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
S16_T_HURRAH = s16_tab(S16_HURRAH, S16_HURRAH_ORDER, "late_vs_early_spend")
s16_bars(S16_T_HURRAH, "late_vs_early_spend", axes[0], "Default rate by late (Aug–Sep) / early (May–Jul) spend", rot=25)
S16_T_SPAT = s16_tab(S16_SPAT, S16_SPAT_ORDER, "spend_pattern")
s16_bars(S16_T_SPAT, "spend_pattern", axes[1], "Default rate by spend pattern", rot=20)
S16_SPAT_STATUS = s16_heat(S16_SPAT, S16_GRP_SEP, axes[2], "Spend pattern × Sep status group (rate, n)", S16_SPAT_ORDER, STATUS_GROUP_ORDER, min_n=40)
plt.tight_layout(); savefig("s16_last_hurrah"); plt.show()
display(S16_T_HURRAH[["late_vs_early_spend", "n", "share", "rate", "ci_lo", "ci_hi", "lift"]].round(4))
display(S16_T_SPAT[["spend_pattern", "n", "share", "rate", "ci_lo", "ci_hi", "lift", "expo_share", "expo_default_share"]].round(4))
S16_SPAT_STATUS.columns = ["spend_pattern", "status_grp_sep", "rate", "n"]
export(S16_T_HURRAH, "s16_spend_late_vs_early"); export(S16_T_SPAT, "s16_spend_patterns"); export(S16_SPAT_STATUS, "s16_spend_pattern_x_status")
_s = S16_T_SPAT.set_index("spend_pattern")["rate"]
add_fact("16.6", "spend dynamics", f"No 'last hurrah': clients with a Sep spending surge default at {_s.get('Sep surge', np.nan):.1%}, while those who stopped using the card "
         f"with a balance outstanding default at {_s.get('Stopped using (balance remains)', np.nan):.1%} (regular spenders {_s.get('Regular', np.nan):.1%})",
         float(_s.get("Stopped using (balance remains)", np.nan)), "s16_last_hurrah", 3)

**Observations:**
- There is no 'last hurrah': a Sep spending surge (more than 3× earlier spend) goes with only 12.6% default (2,142 clients).
- Clients whose Aug-Sep spend is more than 3× their May-Jul spend default at 15.4%.
- No spend at all is the riskiest group at 40.1%; stopping the card with a balance left gives 22.4%.
- Story use: myth-buster slide: 'defaulters do not max out their cards on the way out'.

### 16.7 Follow the money at portfolio level: are defaulters net borrowers every month?
Sum of new charges, payments and the net balance change per month, split by eventual outcome (per-client means so
the two groups are comparable). Then cumulative net flow: the portfolio sum as a share of the April balance, and the
per-client cumulative balance change scaled by the limit (mean and median).

In [ ]:
_rows = []
for g, lab in ((0, "No default"), (1, "Default")):
    m = S16_Y == g
    for k, mth in enumerate(S16_M5):
        _rows.append({"DEFAULT_LABEL": lab, "MONTH": mth, "clients": int(m.sum()),
                      "charges_sum": S16_SP[m, k].sum(), "payments_sum": S16_P[m, k + 1].sum(), "net_change_sum": S16_DB[m, k].sum(),
                      "balance_end_sum": S16_B[m, k + 1].sum(), "balance_start_sum": S16_B[m, k].sum(),
                      "charges_per_client": S16_SP[m, k].mean(), "payments_per_client": S16_P[m, k + 1].mean(), "net_change_per_client": S16_DB[m, k].mean(),
                      "share_net_borrowers": float((S16_DB[m, k] > 0).mean()), "share_net_repayers": float((S16_DB[m, k] < 0).mean()),
                      "coverage_pay_over_charges": S16_P[m, k + 1].sum() / S16_SP[m, k].sum()})
S16_FLOW = pd.DataFrame(_rows)
S16_FLOW["cum_net_over_apr_balance"] = S16_FLOW.groupby("DEFAULT_LABEL")["net_change_sum"].cumsum() / S16_FLOW["DEFAULT_LABEL"].map(
    {"No default": S16_B[S16_Y == 0, 0].sum(), "Default": S16_B[S16_Y == 1, 0].sum()})
_cum = np.cumsum(S16_DB, axis=1) / S16_LIM[:, None]
fig, axes = plt.subplots(2, 2, figsize=(16, 9.5))
for ax, lab in zip(axes[0], ("No default", "Default")):
    u = S16_FLOW[S16_FLOW["DEFAULT_LABEL"] == lab]; x = np.arange(5)
    ax.bar(x - .2, u["charges_per_client"], .4, color=C_DEF, alpha=.8, label="new charges")
    ax.bar(x + .2, u["payments_per_client"], .4, color=C_ACC, alpha=.8, label="payments")
    ax.plot(x, u["net_change_per_client"], "k-o", lw=2, label="net balance change")
    ax.axhline(0, color="k", lw=.6); ax.set_xticks(x); ax.set_xticklabels(S16_M5); ax.legend()
    ax.set_title(f"{lab}: per-client monthly flows (NT$)"); ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v / 1e3:.0f}k"))
_ymax = max(a.get_ylim()[1] for a in axes[0]); _ymin = min(a.get_ylim()[0] for a in axes[0])
for a in axes[0]:
    a.set_ylim(_ymin, _ymax)
for lab, g, col in (("No default", 0, C_OK), ("Default", 1, C_DEF)):
    u = S16_FLOW[S16_FLOW["DEFAULT_LABEL"] == lab]
    axes[1, 0].plot(["Apr"] + S16_M5, [0] + list(u["cum_net_over_apr_balance"]), marker="o", color=col, lw=2, label=f"{lab}: portfolio sum / Apr balance")
    axes[1, 1].plot(["Apr"] + S16_M5, [0] + list(_cum[S16_Y == g].mean(0)), marker="o", color=col, lw=2, label=f"{lab}: mean")
    axes[1, 1].plot(["Apr"] + S16_M5, [0] + list(np.median(_cum[S16_Y == g], 0)), marker="s", color=col, lw=1.5, ls="--", label=f"{lab}: median")
axes[1, 0].set_title("Cumulative net flow, portfolio level (share of Apr balance)"); axes[1, 0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1, 0].legend()
axes[1, 1].set_title("Cumulative balance change per client (share of own limit)"); axes[1, 1].yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=1)); axes[1, 1].legend(fontsize=8)
for a in axes[1]:
    a.axhline(0, color="k", lw=.6)
plt.tight_layout(); savefig("s16_portfolio_flows"); plt.show()
display(S16_FLOW[["DEFAULT_LABEL", "MONTH", "charges_per_client", "payments_per_client", "net_change_per_client", "share_net_borrowers",
                  "share_net_repayers", "coverage_pay_over_charges", "cum_net_over_apr_balance"]].round(4))
export(S16_FLOW, "s16_portfolio_flows")
_fd = S16_FLOW[S16_FLOW["DEFAULT_LABEL"] == "Default"]; _fn = S16_FLOW[S16_FLOW["DEFAULT_LABEL"] == "No default"]
add_fact("16.7", "money flow", f"As a group, defaulters were net borrowers in {(_fd['net_change_sum'] > 0).sum()} of 5 months and non-defaulters in {(_fn['net_change_sum'] > 0).sum()} of 5; "
         f"by Sep the defaulter book had moved {_fd['cum_net_over_apr_balance'].iloc[-1]:+.1%} vs its Apr balance, the non-defaulter book {_fn['cum_net_over_apr_balance'].iloc[-1]:+.1%}",
         float(_fd["cum_net_over_apr_balance"].iloc[-1]), "s16_portfolio_flows", 3)

**Observations:**
- Both groups are net borrowers in every month: per client, new charges exceed payments by NT$ 1.2-3.9k each month.
- Defaulters charge less (NT$ 4.5-6.5k per month vs 6.7-9.7k) and pay less (3.2-3.4k vs 5.2-6.6k).
- By Sep the defaulter book grew 26.8% over its Apr balance and the non-defaulter book 33.2%: the safe group borrowed more.
- Story use: portfolio-level line chart for the 'follow the money' slide.

#### 16.7.1 Waterfall: April balance → + charges − payments → September balance
Per-client averages for the two outcome groups (NT$) and the same waterfall scaled by the average credit limit, so
the reader sees both the money and the utilisation story. This is a ready-made Tableau waterfall (`s16_waterfall.csv`).

In [ ]:
_rows = []
for g, lab in ((0, "No default"), (1, "Default")):
    m = S16_Y == g
    a, ch, pa, e, lim = S16_B[m, 0].mean(), S16_SP[m].sum(1).mean(), S16_P[m, 1:].sum(1).mean(), S16_B[m, 5].mean(), S16_LIM[m].mean()
    for step, (nm, val, base, kind) in enumerate((("Apr balance", a, 0, "balance"), ("+ New charges (May–Sep)", ch, a, "charges"),
                                                  ("− Payments (May–Sep)", -pa, a + ch, "payments"), ("Sep balance", e, 0, "balance"))):
        _rows.append({"DEFAULT_LABEL": lab, "step": step, "item": nm, "kind": kind, "value_per_client": val, "bar_start": base if kind != "payments" else base + val,
                      "bar_height": abs(val), "value_over_avg_limit": val / lim, "avg_limit": lim})
S16_WF = pd.DataFrame(_rows)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8), sharey=True)
_wc = {"balance": C_NEU, "charges": C_DEF, "payments": C_ACC}
for ax, lab in zip(axes, ("No default", "Default")):
    u = S16_WF[S16_WF["DEFAULT_LABEL"] == lab]
    ax.bar(u["item"], u["bar_height"], bottom=u["bar_start"], color=[_wc[k] for k in u["kind"]], alpha=.85)
    for i, r in enumerate(u.itertuples()):
        ax.text(i, r.bar_start + r.bar_height, f"{r.value_per_client / 1e3:+.1f}k\n({r.value_over_avg_limit:+.0%} of limit)", ha="center", va="bottom", fontsize=9)
    ax.set_title(f"{lab}: average client (limit NT$ {u['avg_limit'].iloc[0] / 1e3:.0f}k)"); ax.tick_params(axis="x", rotation=12)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f"{v / 1e3:.0f}k")); ax.set_ylim(0, S16_WF["bar_start"].add(S16_WF["bar_height"]).max() * 1.18)
plt.tight_layout(); savefig("s16_waterfall"); plt.show()
display(S16_WF.round(3)); export(S16_WF, "s16_waterfall")
_w = S16_WF.set_index(["DEFAULT_LABEL", "item"])["value_per_client"]
S16_WF_COV = {lab: -_w[(lab, "− Payments (May–Sep)")] / _w[(lab, "+ New charges (May–Sep)")] for lab in ("No default", "Default")}
print("5-month coverage (payments / new charges):", {k: f"{v:.1%}" for k, v in S16_WF_COV.items()})
add_fact("16.7", "money flow", f"Over May-Sep the average defaulter charged NT$ {_w[('Default', '+ New charges (May–Sep)')] / 1e3:.0f}k and repaid "
         f"{-_w[('Default', '− Payments (May–Sep)')] / 1e3:.0f}k (coverage {S16_WF_COV['Default']:.0%}); the average non-defaulter charged "
         f"{_w[('No default', '+ New charges (May–Sep)')] / 1e3:.0f}k and repaid {-_w[('No default', '− Payments (May–Sep)')] / 1e3:.0f}k (coverage {S16_WF_COV['No default']:.0%})",
         float(S16_WF_COV["Default"]), "s16_waterfall", 2)

**Observations:**
- Average defaulter: Apr balance NT$ 38.3k + 26.8k new charges − 16.5k payments = 48.5k in Sep, which is 37.3% of an average limit of 130k.
- Average non-defaulter: 39.0k + 42.2k − 29.3k = 52.0k, only 29.2% of an average limit of 178k.
- Payments cover 61.8% of new charges for defaulters vs 69.3% for non-defaulters.
- Story use: ready-made Tableau waterfall: 'same debt in NT$, smaller credit line'.

### 16.8 Do payments cover the charges? The 5-month coverage ratio
`coverage = Σ payments (May..Sep) / Σ new charges (May..Sep)`. By the identity, coverage < 100% means the balance
grew over the window and > 100% means it shrank, so this is the cleanest single "net borrower / net repayer" number.
Clients with no net charges (Σ charges ≤ 0) get their own band. Then the 2-D map against average utilisation.

In [ ]:
_ch, _pa = S16_SP.sum(1), S16_P[:, 1:].sum(1)
S16_COV = np.where(_ch > 0, _pa / np.where(_ch > 0, _ch, 1), np.nan)
df["MF_COVERAGE"] = np.minimum(S16_COV, 10)
S16_COV_ORDER = ["No net charges", "<50%", "50–80%", "80–95%", "95–105% (balanced)", "105–150%", ">150%"]
S16_COVB = np.select([_ch <= 0, S16_COV < .5, S16_COV < .8, S16_COV < .95, S16_COV <= 1.05, S16_COV <= 1.5], S16_COV_ORDER[:6], ">150%")
df["MF_COVERAGE_BAND"] = pd.Categorical(S16_COVB, S16_COV_ORDER, ordered=True)
S16_T_COV = s16_tab(S16_COVB, S16_COV_ORDER, "coverage_band")
fig, axes = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw={"width_ratios": [1, .8, 1.3]})
s16_bars(S16_T_COV, "coverage_band", axes[0], "Default rate by 5-month payment coverage of charges", rot=20)
_t = s16_curve(np.where((S16_COV > 0) & (S16_COV < 5), S16_COV, np.nan), 15, "coverage")
s16_plot_curve(_t, axes[1], "Default rate by coverage (15 quantile bins; x = bin median)")
S16_UTIL_LABELS = [str(c) for c in df["UTIL_BAND"].cat.categories]
S16_COV_UTIL = s16_heat(S16_COVB, df["UTIL_BAND"].astype(str), axes[2], "Coverage band × average utilisation band (rate, n)", S16_COV_ORDER, S16_UTIL_LABELS, min_n=40)
plt.tight_layout(); savefig("s16_coverage"); plt.show()
display(S16_T_COV[["coverage_band", "n", "share", "rate", "ci_lo", "ci_hi", "lift", "expo_share", "expo_default_share"]].round(4))
S16_COV_UTIL.columns = ["coverage_band", "util_band", "rate", "n"]
export(S16_T_COV, "s16_coverage_bands"); export(S16_COV_UTIL, "s16_coverage_x_util")
_c = S16_T_COV.set_index("coverage_band")["rate"]
_w = S16_COV_UTIL[S16_COV_UTIL["n"] >= 40]
print(f"IV of coverage band: {woe_iv(df, 'MF_COVERAGE_BAND')[1]:.3f} | IV of UTIL_BAND: {woe_iv(df, 'UTIL_BAND')[1]:.3f}")
print(f"spread across coverage bands within a utilisation band (median of max − min): {_w.groupby('util_band')['rate'].agg(lambda s: s.max() - s.min()).median():.1%}; "
      f"across utilisation bands within a coverage band: {_w.groupby('coverage_band')['rate'].agg(lambda s: s.max() - s.min()).median():.1%}")
add_fact("16.8", "money flow", f"Net borrowers are not the risky ones: clients covering <50% of their charges default at {_c.get('<50%', np.nan):.1%}, balanced payers "
         f"(95-105%) at {_c.get('95–105% (balanced)', np.nan):.1%}, heavy net repayers (>150%) at {_c.get('>150%', np.nan):.1%}, and clients with no net charges at "
         f"{_c.get('No net charges', np.nan):.1%}", float(_c.get("<50%", np.nan)), "s16_coverage", 3)

**Observations:**
- Coverage (payments ÷ new charges over five months) has a U-shape: 50-80% coverage is safest at 16.2%, under 50% is 20.0%.
- Heavy net repayers (over 150%, balance shrinking) default at 31.3% and clients with no net charges at 39.3%.
- Coverage band (IV 0.131) is about as informative as utilisation band (IV 0.124), and each still varies by about 23-25 points inside the other.
- Story use: 'net borrowers are not the risky ones'; pairs with the pay-down shape finding.

### 16.9 Over-limit episodes: who goes over, for how long, how deep, and what happens next?
A month is **over limit** when the bill exceeds the credit limit. First the dynamics: stock per month by outcome and
month-by-month **entry** (not over → over) and **exit** (over → not over) rates.

In [ ]:
S16_OL = S16_B > S16_LIM[:, None]
_rows = []
for k in range(5):
    a, b = S16_OL[:, k], S16_OL[:, k + 1]
    _rows.append({"pair": S16_PAIRS[k], "over_before": int(a.sum()), "entries": int((~a & b).sum()), "exits": int((a & ~b).sum()), "stay_over": int((a & b).sum()),
                  "entry_rate": (~a & b).sum() / (~a).sum(), "exit_rate": (a & ~b).sum() / max(1, a.sum()), "over_after": int(b.sum()),
                  "default_rate_entrants": S16_Y[~a & b].mean(), "default_rate_stayers": S16_Y[a & b].mean(), "default_rate_exiters": S16_Y[a & ~b].mean(),
                  "default_rate_never_in_pair": S16_Y[~a & ~b].mean()})
S16_OL_FLOW = pd.DataFrame(_rows)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
for g, lab, col in ((0, "No default", C_OK), (1, "Default", C_DEF)):
    axes[0].plot(S16_M6, S16_OL[S16_Y == g].mean(0), marker="o", lw=2, color=col, label=lab)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].set_title("Share of clients over limit, by month"); axes[0].legend()
x = np.arange(5)
axes[1].bar(x - .2, S16_OL_FLOW["entries"], .4, color=C_DEF, alpha=.8, label="entries"); axes[1].bar(x + .2, S16_OL_FLOW["exits"], .4, color=C_ACC, alpha=.8, label="exits")
axes[1].set_xticks(x); axes[1].set_xticklabels(S16_PAIRS); axes[1].legend(); axes[1].set_title("Clients entering / leaving over-limit")
for c, lab, col in (("default_rate_entrants", "entrants", C_DEF), ("default_rate_stayers", "stayers", "#a50f15"), ("default_rate_exiters", "exiters", C_ACC), ("default_rate_never_in_pair", "not over in either month", C_NEU)):
    axes[2].plot(S16_PAIRS, S16_OL_FLOW[c], marker="o", lw=2, color=col, label=lab)
axes[2].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[2].legend(fontsize=8)
axes[2].set_title("Default rate by over-limit move in each month pair")
plt.tight_layout(); savefig("s16_overlimit_dynamics"); plt.show()
display(S16_OL_FLOW.round(4)); export(S16_OL_FLOW, "s16_overlimit_flows")
print(f"ever over limit: {S16_OL.any(1).mean():.1%} of clients | persistence P(over next month | over now) = "
      f"{S16_OL_FLOW['stay_over'].sum() / S16_OL_FLOW['over_before'].sum():.1%} | P(enter | not over) = {S16_OL_FLOW['entries'].sum() / (~S16_OL[:, :5]).sum():.2%}")

**Observations:**
- Over-limit clients (bill above the credit limit) grow from 798 in Apr to 2,115 in Sep; the monthly entry rate doubles from 1.4% to 3.1%.
- 13.1% of clients go over at least once, and 61.6% of over-limit clients are still over the next month.
- Clients who stay over default at 33-38%, vs about 21.5% for clients never over in that pair of months.
- Story use: background for the over-limit sub-story; the Apr→Sep growth is a good small-multiple.

#### 16.9.1 Depth, duration and pattern of over-limit episodes
**Depth** = the worst `bill / limit − 1` over the six months; **duration** = number of over-limit months and the longest
consecutive run; **pattern** = never / past only (back under the limit by Sep) / over in Sep (entered during the window) /
over in all six months.

In [ ]:
_depth = np.where(S16_OL.any(1), (S16_U - 1).max(1), 0.0)
df["MF_OL_MAX_DEPTH"] = _depth
df["MF_OL_RUN_MAX"] = s16_max_run(S16_OL)
S16_OLD_ORDER = ["Never over", "0–2%", "2–5%", "5–10%", "10–20%", ">20%"]
S16_OLD = np.select([~S16_OL.any(1), _depth <= .02, _depth <= .05, _depth <= .10, _depth <= .20], S16_OLD_ORDER[:5], ">20%")
df["MF_OL_DEPTH_BAND"] = pd.Categorical(S16_OLD, S16_OLD_ORDER, ordered=True)
S16_OLP_ORDER = ["Never over", "Past only (under limit in Sep)", "Over in Sep (entered in window)", "Over limit all 6 months"]
S16_OLP = np.select([~S16_OL.any(1), S16_OL.all(1), S16_OL[:, 5]], ["Never over", "Over limit all 6 months", "Over in Sep (entered in window)"], "Past only (under limit in Sep)")
df["MF_OL_PATTERN"] = pd.Categorical(S16_OLP, S16_OLP_ORDER)
_dur = np.select([S16_OL.sum(1) == 0, S16_OL.sum(1) == 1, S16_OL.sum(1) <= 3], ["0", "1 month", "2-3 months"], "4-6 months")
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
S16_T_OLD = s16_tab(S16_OLD, S16_OLD_ORDER, "max_depth_over_limit")
s16_bars(S16_T_OLD, "max_depth_over_limit", axes[0], "Default rate by worst over-limit depth")
S16_T_OLP = s16_tab(S16_OLP, S16_OLP_ORDER, "overlimit_pattern")
s16_bars(S16_T_OLP, "overlimit_pattern", axes[1], "Default rate by over-limit pattern", rot=15)
S16_OL_DD = s16_heat(S16_OLD, _dur, axes[2], "Depth (rows) × duration (columns), over-limit clients (rate, n)", S16_OLD_ORDER[1:], ["1 month", "2-3 months", "4-6 months"],
                     min_n=30, mask=S16_OL.any(1), fs=9)
plt.tight_layout(); savefig("s16_overlimit_depth_duration"); plt.show()
display(S16_T_OLD[["max_depth_over_limit", "n", "share", "rate", "ci_lo", "ci_hi", "lift", "expo_share"]].round(4))
display(S16_T_OLP[["overlimit_pattern", "n", "share", "rate", "ci_lo", "ci_hi", "lift", "expo_share", "expo_default_share"]].round(4))
display(s16_tab(df["MF_OL_RUN_MAX"], name="longest_overlimit_run")[["longest_overlimit_run", "n", "rate", "lift"]].round(4))
S16_OL_DD.columns = ["max_depth_over_limit", "duration", "rate", "n"]
export(S16_T_OLD, "s16_overlimit_depth"); export(S16_T_OLP, "s16_overlimit_pattern"); export(S16_OL_DD, "s16_overlimit_depth_x_duration")
_o = S16_T_OLP.set_index("overlimit_pattern")["rate"]
add_fact("16.9", "over-limit", f"{S16_OL.any(1).mean():.0%} of clients went over their limit at least once; default rate is {_o.get('Never over', np.nan):.1%} never over, "
         f"{_o.get('Past only (under limit in Sep)', np.nan):.1%} past-only, {_o.get('Over in Sep (entered in window)', np.nan):.1%} over in Sep, "
         f"{_o.get('Over limit all 6 months', np.nan):.1%} over in all six months", float(_o.get("Over in Sep (entered in window)", np.nan)), "s16_overlimit_depth_duration", 2)

**Observations:**
- Going 2-20% over the limit lifts default to 34-37%; going more than 20% over is only 27.3% (620 clients).
- Clients over the limit in all six months (183) default at 35.0%; a longest run of 3+ months sits around 34-37%.
- Being over only in the past (30.1%) is as risky as being over in Sep (29.6%): the mark does not fade.
- Story use: 'having gone over once' is the flag; depth matters little beyond a few percent.

#### 16.9.2 What happens the month after an over-limit month?
Client-months (Apr..Aug) grouped by where the bill sits relative to the limit, then the status **one month later**.
"New delinquency" restricts to client-months that were *not* late, so it measures fresh trouble instead of persistence.
The comparison that matters is *just under* the limit (90–100%) against *just over* it: is crossing the line an event
or only a number?

In [ ]:
_ub = np.select([S16_U[:, :5] < .5, S16_U[:, :5] < .9, S16_U[:, :5] <= 1.0, S16_U[:, :5] <= 1.05], ["< 50%", "50–90%", "90–100%", "Over by ≤ 5%"], "Over by > 5%")
_nxt_late = S16_ST[:, 1:] >= 1
_cur_ok = S16_ST[:, :5] <= 0
_yy = np.repeat(S16_Y[:, None], 5, axis=1)
_rows = []
for b in ["< 50%", "50–90%", "90–100%", "Over by ≤ 5%", "Over by > 5%"]:
    m = _ub == b
    _rows.append({"bill_vs_limit_this_month": b, "client_months": int(m.sum()), "late_next_month": _nxt_late[m].mean(),
                  "new_delinquency_next_month (currently not late)": _nxt_late[m & _cur_ok].mean(), "n_currently_not_late": int((m & _cur_ok).sum()),
                  "still_over_limit_next_month": S16_OL[:, 1:][m].mean(), "payment_ratio_next_month_median": np.nanmedian(S16_PR[m]),
                  "eventual_default_rate": _yy[m].mean()})
S16_OL_NEXT = pd.DataFrame(_rows)
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(S16_OL_NEXT))
ax.bar(x - .27, S16_OL_NEXT["late_next_month"], .27, color=C_NEU, label="late next month")
ax.bar(x, S16_OL_NEXT["new_delinquency_next_month (currently not late)"], .27, color=C_DEF, label="NEW delinquency next month (currently not late)")
ax.bar(x + .27, S16_OL_NEXT["eventual_default_rate"], .27, color="#a50f15", alpha=.6, label="eventual default rate")
ax.set_xticks(x); ax.set_xticklabels(S16_OL_NEXT["bill_vs_limit_this_month"]); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.legend()
ax.set_title("What follows a client-month, by bill relative to limit (Apr..Aug months, status one month later)")
plt.tight_layout(); savefig("s16_overlimit_next_month"); plt.show()
display(S16_OL_NEXT.round(4)); export(S16_OL_NEXT, "s16_overlimit_next_month")
_n = S16_OL_NEXT.set_index("bill_vs_limit_this_month")["new_delinquency_next_month (currently not late)"]
add_fact("16.9", "over-limit", f"Among client-months that are not late, a new delinquency follows {_n['90–100%']:.1%} of months at 90-100% utilisation, "
         f"{_n['Over by ≤ 5%']:.1%} of months just over the limit and {_n['Over by > 5%']:.1%} of months more than 5% over (vs {_n['< 50%']:.1%} below 50%)",
         float(_n["Over by > 5%"]), "s16_overlimit_next_month", 2)

**Observations:**
- New delinquency the next month (among clients not yet late) rises from 5.4% below 50% utilisation to 9.2% at 90-100%, then flattens (9.0% just over the limit, 8.9% more than 5% over).
- Over-limit months are sticky: 55% (over by ≤ 5%) and 79% (over by > 5%) are still over the next month.
- Next-month median repayment drops from 23% of the bill below 50% utilisation to about 4% at 50% and above.
- Story use: the risk step is reaching the limit, not crossing it.

### 16.10 Credit balances: who over-pays, for how long, and what follows?
A **negative bill** means the bank owes the client (over-payment or a refund). How common is it per month, how long
does it last, how does a client get there (payment larger than the previous bill, or a negative "spend" = refund /
reversal), and what is the next month's balance?

In [ ]:
S16_NEG = S16_B < 0
df["MF_N_NEG_BILL"] = S16_NEG.sum(1)
df["MF_NEG_MAX_DEPTH"] = np.clip(-S16_U.min(1), 0, None)
S16_NEGP_ORDER = ["Never in credit", "Past only", "In credit in Sep", "In credit all 6 months"]
S16_NEGP = np.select([~S16_NEG.any(1), S16_NEG.all(1), S16_NEG[:, 5]], ["Never in credit", "In credit all 6 months", "In credit in Sep"], "Past only")
df["MF_NEG_PATTERN"] = pd.Categorical(S16_NEGP, S16_NEGP_ORDER)
_ent = ~S16_NEG[:, :5] & S16_NEG[:, 1:]                                # entered credit in May..Sep
_how = np.select([(S16_SP < 0) & (S16_P[:, 1:] <= np.clip(S16_PREV, 0, None)), (S16_SP < 0)], ["Refund / reversal only", "Refund and over-payment"], "Over-payment (payment > previous bill)")
S16_NEG_HOW = pd.Series(_how[_ent]).value_counts(normalize=True)
_after = np.select([S16_B[:, 1:] < 0, S16_B[:, 1:] == 0], ["Still in credit", "Exactly zero"], "Positive bill")
S16_NEG_NEXT = pd.Series(_after[S16_NEG[:, :5]]).value_counts(normalize=True)
_stneg = pd.Series(S16_ST[S16_NEG]).map(lambda v: STATUS_LABEL.get(int(v), str(v))).value_counts(normalize=True)

fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
for g, lab, col in ((0, "No default", C_OK), (1, "Default", C_DEF)):
    axes[0].plot(S16_M6, S16_NEG[S16_Y == g].mean(0), marker="o", lw=2, color=col, label=lab)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1, decimals=1)); axes[0].set_title("Share of clients with a credit balance, by month"); axes[0].legend()
S16_T_NNEG = s16_tab(np.select([S16_NEG.sum(1) == 0, S16_NEG.sum(1) == 1, S16_NEG.sum(1) <= 3], ["0", "1", "2-3"], "4-6"), ["0", "1", "2-3", "4-6"], "months_in_credit")
s16_bars(S16_T_NNEG, "months_in_credit", axes[1], "Default rate by number of credit-balance months")
S16_T_NEGP = s16_tab(S16_NEGP, S16_NEGP_ORDER, "credit_balance_pattern")
s16_bars(S16_T_NEGP, "credit_balance_pattern", axes[2], "Default rate by credit-balance pattern", rot=12)
plt.tight_layout(); savefig("s16_credit_balance"); plt.show()
display(S16_T_NNEG[["months_in_credit", "n", "share", "rate", "ci_lo", "ci_hi", "lift"]].round(4))
display(S16_T_NEGP[["credit_balance_pattern", "n", "share", "rate", "ci_lo", "ci_hi", "lift"]].round(4))
print("how clients ENTER a credit balance:"); display(S16_NEG_HOW.round(4).to_frame("share of entries"))
print("the month AFTER a credit-balance month:"); display(S16_NEG_NEXT.round(4).to_frame("share"))
print("status code recorded in credit-balance months:"); display(_stneg.round(4).to_frame("share").head(6))
print(f"persistence P(in credit next month | in credit now) = {(S16_NEG[:, :5] & S16_NEG[:, 1:]).sum() / S16_NEG[:, :5].sum():.1%} | "
      f"median credit depth among credit clients = {np.median(df.loc[S16_NEG.any(1), 'MF_NEG_MAX_DEPTH']):.2%} of limit")
export(S16_T_NNEG, "s16_credit_balance_months"); export(S16_T_NEGP, "s16_credit_balance_pattern")

**Observations:**
- 6.4% of clients were in credit (negative bill, the bank owes them) at least once; one month in credit defaults at 14.2%, but all six months at 29.6% (88 clients).
- 99.2% of entries come from over-payment, not refunds; 54.5% are still in credit the next month, and the median credit is tiny (0.10% of the limit).
- Credit-balance months are mostly coded −2 (61%) or −1 (17%), yet 10% carry code 1 'Late 1m', a coding quirk that fits code 1 being Sep-specific.
- Story use: background only.

#### 16.10.1 Who are the over-payers, and why do some of them still default?
Share of clients ever in credit by limit band and age band, their default rate against the rest *within* each limit
band, and a profile of the defaulters among them (September status).

In [ ]:
_ever = S16_NEG.any(1)
fig, axes = plt.subplots(1, 3, figsize=(19, 4.6))
_tl = pd.DataFrame({"band": df["LIMIT_BAND"].astype(str), "ever": _ever, "y": S16_Y})
S16_NEG_LIMIT = _tl.groupby("band").agg(n=("y", "size"), share_ever_in_credit=("ever", "mean")).reindex(LIMIT_LABELS)
S16_NEG_LIMIT["default_rate_ever_in_credit"] = _tl[_tl["ever"]].groupby("band")["y"].mean()
S16_NEG_LIMIT["default_rate_never_in_credit"] = _tl[~_tl["ever"]].groupby("band")["y"].mean()
axes[0].bar(S16_NEG_LIMIT.index, S16_NEG_LIMIT["share_ever_in_credit"], color=C_ACC, alpha=.8)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].set_title("Share of clients ever in credit, by limit band"); axes[0].tick_params(axis="x", rotation=25)
S16_NEG_LIMIT[["default_rate_ever_in_credit", "default_rate_never_in_credit"]].plot(kind="bar", ax=axes[1], color=[C_ACC, C_NEU])
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("Default rate: ever vs never in credit, within limit band"); axes[1].tick_params(axis="x", rotation=25); axes[1].set_xlabel("")
_ta = pd.DataFrame({"band": df["AGE_BAND"].astype(str), "ever": _ever}).groupby("band")["ever"].mean().reindex(AGE_LABELS)
axes[2].bar(_ta.index, _ta.values, color=C_ACC, alpha=.8); axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[2].set_title("Share of clients ever in credit, by age band")
plt.tight_layout(); savefig("s16_credit_balance_who"); plt.show()
display(S16_NEG_LIMIT.round(4))
export(S16_NEG_LIMIT.reset_index().rename(columns={"band": "LIMIT_BAND"}), "s16_credit_balance_by_limit")
_dn = _ever & (S16_Y == 1)
print(f"credit-balance clients who default: {_dn.sum():,} ({S16_Y[_ever].mean():.1%} of {_ever.sum():,}); their Sep status mix vs credit-balance non-defaulters:")
display(pd.DataFrame({"defaulters": pd.Series(S16_GRP_SEP[_dn]).value_counts(normalize=True),
                      "non-defaulters": pd.Series(S16_GRP_SEP[_ever & (S16_Y == 0)]).value_counts(normalize=True)}).reindex(STATUS_GROUP_ORDER).round(4))
_n = S16_T_NEGP.set_index("credit_balance_pattern")["rate"]
add_fact("16.10", "credit balance", f"{_ever.mean():.1%} of clients were in credit (negative bill) at least once; they default at {S16_Y[_ever].mean():.1%} vs "
         f"{S16_Y[~_ever].mean():.1%} for the rest, and even clients in credit in Sep default at {_n.get('In credit in Sep', np.nan):.1%}",
         float(S16_Y[_ever].mean()), "s16_credit_balance", 2)

**Observations:**
- Over-paying is more common with bigger limits: 3.9% of clients at ≤50k vs 10.2% at 300-500k.
- Within low limits over-payers are clearly safer (≤50k: 21.3% vs 32.2%), but the edge disappears above 300k (14.4% vs 13.2%).
- The 318 over-payers who still default are more often late in Sep: 32% Late 1m and 9% Late 2m, vs 25% and 1% for over-payers who do not default.
- Story use: background only.

### 16.11 A time-series feature bank: which *kind* of time descriptor carries the signal?
A compact tsfresh-style bank (tsfresh is a library of automatic time-series summary features) computed with numpy: **11 statistics × 5 series = 55 features** per client, kept in a
separate frame (`S16_TS`), not in `df`.

| Family | Statistics |
|---|---|
| Level | `mean` |
| Trend | `slope` (OLS per month), `last_minus_first`, `r2` (share of variance explained by a straight line) |
| Volatility | `std`, `cv` (std / \|mean\|), `max_jump` (largest single month-on-month move, signed) |
| Recency | `last_over_mean` (latest value relative to own mean) |
| Shape | `ac1` (lag-1 autocorrelation), `argmax` (month index of the peak), `n_above_mean` |

Series: bill, payment, utilisation, implied spend (5 months), status code. Every feature is ranked by **IV**
(`woe_iv`, deciles, missing values as their own bin) and by univariate **AUC** (shown as `max(AUC, 1 − AUC)`).

In [ ]:
def s16_ts_feats(M, name):
    n, T = M.shape
    t = np.arange(T) - (T - 1) / 2
    mean = M.mean(1); std = M.std(1)
    dm = M - mean[:, None]; den = (dm ** 2).sum(1)
    slope = (dm * t).sum(1) / (t ** 2).sum()
    dif = np.diff(M, axis=1)
    out = {
        "mean": mean, "std": std,
        "cv": np.where(np.abs(mean) > 1e-9, std / np.where(np.abs(mean) > 1e-9, np.abs(mean), 1), np.nan),
        "slope": slope, "last_minus_first": M[:, -1] - M[:, 0],
        "last_over_mean": np.where(np.abs(mean) > 1e-9, M[:, -1] / np.where(np.abs(mean) > 1e-9, mean, 1), np.nan),
        "ac1": np.where(den > 1e-12, (dm[:, 1:] * dm[:, :-1]).sum(1) / np.where(den > 1e-12, den, 1), np.nan),
        "max_jump": dif[np.arange(n), np.abs(dif).argmax(1)],
        "argmax": M.argmax(1).astype(float), "n_above_mean": (M > mean[:, None]).sum(1).astype(float),
        "r2": np.where(den > 1e-12, slope ** 2 * (t ** 2).sum() / np.where(den > 1e-12, den, 1), np.nan)}
    return pd.DataFrame({f"{name}_{k}": v for k, v in out.items()})

S16_TS_FAMILY = {"mean": "Level", "slope": "Trend", "last_minus_first": "Trend", "r2": "Trend", "std": "Volatility", "cv": "Volatility",
                 "max_jump": "Volatility", "last_over_mean": "Recency", "ac1": "Shape", "argmax": "Shape", "n_above_mean": "Shape"}
S16_TS_SERIES = {"bill": S16_B, "pay": S16_P, "util": S16_U, "spend": S16_SP, "status": S16_ST.astype(float)}
S16_TS = pd.concat([s16_ts_feats(M, nm) for nm, M in S16_TS_SERIES.items()], axis=1)
print("feature bank:", S16_TS.shape, "| features with any missing value:", int(S16_TS.isna().any().sum()))

def s16_iv(x):
    x = pd.Series(np.asarray(x, float))
    b = pd.qcut(x, 10, duplicates="drop").astype(str) if x.nunique(dropna=True) > 10 else x.astype(str)
    return woe_iv(pd.DataFrame({"f": b.to_numpy(), TARGET: S16_Y}), "f")[1]

def s16_auc(x):
    x = np.asarray(x, float); ok = ~np.isnan(x)
    y = S16_Y[ok]; r = stats.rankdata(x[ok]); n1 = y.sum(); n0 = len(y) - n1
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

_rows = []
for c in S16_TS.columns:
    ser = c.split("_")[0]; stat = c[len(ser) + 1:]
    a = s16_auc(S16_TS[c])
    _rows.append({"feature": c, "series": ser, "statistic": stat, "family": S16_TS_FAMILY[stat], "IV": s16_iv(S16_TS[c]), "AUC": a,
                  "AUC_separation": max(a, 1 - a), "direction": "higher = riskier" if a >= .5 else "higher = safer", "missing_share": float(S16_TS[c].isna().mean())})
s16_ts_feature_ranking = pd.DataFrame(_rows)
s16_ts_feature_ranking["rank_IV"] = s16_ts_feature_ranking["IV"].rank(ascending=False).astype(int)
s16_ts_feature_ranking["rank_AUC"] = s16_ts_feature_ranking["AUC_separation"].rank(ascending=False).astype(int)
s16_ts_feature_ranking = s16_ts_feature_ranking.sort_values("IV", ascending=False).reset_index(drop=True)
S16_TS_RANK = s16_ts_feature_ranking
print("TOP 20 by IV:"); display(S16_TS_RANK.head(20).round(4))
print("TOP 10 money-only features (status series excluded):"); display(S16_TS_RANK[S16_TS_RANK["series"] != "status"].head(10).round(4))
print(f"rank agreement IV vs AUC (Spearman): {stats.spearmanr(S16_TS_RANK['IV'], S16_TS_RANK['AUC_separation'])[0]:.2f}")
export(S16_TS_RANK, "s16_ts_feature_ranking")

**Observations:**
- Across 55 time-series features (11 statistics × 5 series), the Sep-and-earlier status mean is by far the strongest (IV 0.78).
- The best money-only feature is spend volatility (spend_std, IV 0.27, AUC 0.63; AUC = chance of ranking a random defaulter above a non-defaulter), and more spend means safer.
- IV and AUC rankings agree only moderately (Spearman 0.65), so feature choice depends a little on the metric.
- Story use: background for model features.

#### 16.11.1 Top-20 chart, the series × statistic IV grid, and the family verdict
Left: the 20 strongest features coloured by family. Middle: IV for every series × statistic cell (the whole bank in one
picture). Right: IV by family, each dot one feature. The family table answers the question in the heading.

In [ ]:
S16_FAM_COL = {"Level": "#2b8cbe", "Trend": "#e34a33", "Volatility": "#fdae6b", "Recency": "#31a354", "Shape": "#756bb1"}
S16_FAM_ORDER = list(S16_FAM_COL)
fig, axes = plt.subplots(1, 3, figsize=(21, 7), gridspec_kw={"width_ratios": [1.1, 1.1, .8]})
_t = S16_TS_RANK.head(20).iloc[::-1]
axes[0].barh(_t["feature"], _t["IV"], color=[S16_FAM_COL[f] for f in _t["family"]])
for i, (iv, au) in enumerate(zip(_t["IV"], _t["AUC_separation"])):
    axes[0].text(iv, i, f" IV {iv:.2f} · AUC {au:.2f}", va="center", fontsize=8)
axes[0].set_xlim(0, _t["IV"].max() * 1.3); axes[0].set_title("Top 20 time-series features by IV")
axes[0].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in S16_FAM_COL.values()], labels=S16_FAM_ORDER, loc="lower right", fontsize=8)
_stat_order = ["mean", "slope", "last_minus_first", "r2", "std", "cv", "max_jump", "last_over_mean", "ac1", "argmax", "n_above_mean"]
_grid = S16_TS_RANK.pivot(index="statistic", columns="series", values="IV").reindex(index=_stat_order, columns=list(S16_TS_SERIES))
sns.heatmap(_grid, annot=True, fmt=".2f", cmap="YlOrRd", cbar=False, ax=axes[1], linewidths=.5)
axes[1].set_yticklabels([f"{s}  [{S16_TS_FAMILY[s]}]" for s in _stat_order], rotation=0); axes[1].set_title("IV by statistic × series"); axes[1].set_xlabel(""); axes[1].set_ylabel("")
sns.stripplot(data=S16_TS_RANK, x="family", y="IV", order=S16_FAM_ORDER, hue="series", ax=axes[2], size=7, jitter=.2)
axes[2].set_title("IV of every feature by family"); axes[2].set_xlabel(""); axes[2].legend(fontsize=8, title="series")
plt.tight_layout(); savefig("s16_ts_feature_bank"); plt.show()

S16_TS_FAMILY_SUM = S16_TS_RANK.groupby("family").agg(n_features=("IV", "size"), IV_mean=("IV", "mean"), IV_median=("IV", "median"), IV_max=("IV", "max"),
                                                    AUC_sep_mean=("AUC_separation", "mean"), best_feature=("feature", "first")).reindex(S16_FAM_ORDER)
S16_TS_FAMILY_MONEY = S16_TS_RANK[S16_TS_RANK["series"] != "status"].groupby("family").agg(IV_mean_money_only=("IV", "mean"), IV_max_money_only=("IV", "max"),
                                                                                           best_money_feature=("feature", "first")).reindex(S16_FAM_ORDER)
S16_TS_FAMILY_SUM = S16_TS_FAMILY_SUM.join(S16_TS_FAMILY_MONEY)
display(S16_TS_FAMILY_SUM.round(4))
S16_TS_SERIES_SUM = S16_TS_RANK.groupby("series").agg(IV_mean=("IV", "mean"), IV_max=("IV", "max"), best_feature=("feature", "first")).reindex(list(S16_TS_SERIES))
display(S16_TS_SERIES_SUM.round(4))
display(_grid.round(3))
export(S16_TS_FAMILY_SUM.reset_index(), "s16_ts_family_summary")
_fb = S16_TS_FAMILY_SUM["IV_mean"].idxmax(); _fm = S16_TS_FAMILY_SUM["IV_mean_money_only"].idxmax()
add_fact("16.11", "time descriptors", f"Across a 55-feature time-series bank, the '{_fb}' family carries the most signal on average (mean IV "
         f"{S16_TS_FAMILY_SUM.loc[_fb, 'IV_mean']:.2f}); for money series alone it is '{_fm}' (mean IV {S16_TS_FAMILY_SUM.loc[_fm, 'IV_mean_money_only']:.2f}). "
         f"Best single feature: {S16_TS_RANK.iloc[0]['feature']} (IV {S16_TS_RANK.iloc[0]['IV']:.2f})", float(S16_TS_FAMILY_SUM.loc[_fb, "IV_mean"]), "s16_ts_feature_bank", 3)
_bm = S16_TS_RANK[S16_TS_RANK["series"] != "status"].iloc[0]
add_fact("16.11", "time descriptors", f"The best money-only time feature is {_bm['feature']} (IV {_bm['IV']:.2f}, AUC {_bm['AUC_separation']:.2f}); status series features average IV "
         f"{S16_TS_SERIES_SUM.loc['status', 'IV_mean']:.2f} vs {S16_TS_RANK[S16_TS_RANK['series'] != 'status']['IV'].mean():.2f} for money series",
         float(_bm["IV"]), "s16_ts_feature_bank", 2)

**Observations:**
- By family, Level averages the highest IV (0.27) but only because of status_mean; Volatility is steadier (mean 0.15), Shape is weakest (0.07).
- By series, status averages IV 0.24 and spend 0.14, ahead of bill (0.10), pay (0.08) and utilisation (0.08).
- Fit quality (r2) and month-to-month autocorrelation (ac1) are near zero for every series (IV ≤ 0.07).
- Story use: background only (numbers from the printed family table; charts are image-only).

#### 16.11.2 Are the winners new information, or the September status in disguise?
The top eight features are added to `df` as `MF_TS_*`. For each: Spearman correlation with `PAY_1` (Sep status) and
with `N_LATE`, plus the IV **inside** the two largest Sep status groups. A feature whose IV collapses within a status
group is a re-statement of the status code; one that keeps its IV is a genuinely different lens.

In [ ]:
S16_TS_TOP8 = list(S16_TS_RANK["feature"].head(8))
_rows = []
for c in S16_TS_TOP8:
    df[f"MF_TS_{c.upper()}"] = S16_TS[c].to_numpy()
    r = {"feature": c, "df_column": f"MF_TS_{c.upper()}", "IV": float(S16_TS_RANK.set_index("feature").loc[c, "IV"]),
         "spearman_with_PAY_1": stats.spearmanr(S16_TS[c], df["PAY_1"], nan_policy="omit")[0],
         "spearman_with_N_LATE": stats.spearmanr(S16_TS[c], df["N_LATE"], nan_policy="omit")[0]}
    for g in ("Revolving", "Paid in full"):
        m = S16_GRP_SEP == g
        x = pd.Series(S16_TS.loc[m, c].to_numpy())
        b = pd.qcut(x, 5, duplicates="drop").astype(str) if x.nunique(dropna=True) > 5 else x.astype(str)
        r[f"IV_within_{g}"] = woe_iv(pd.DataFrame({"f": b.to_numpy(), TARGET: S16_Y[m]}), "f")[1]
    _rows.append(r)
S16_TS_TOP8_TAB = pd.DataFrame(_rows)
display(S16_TS_TOP8_TAB.round(4))
# same check for the best money-only features
_rows = []
for c in list(S16_TS_RANK[S16_TS_RANK["series"] != "status"]["feature"].head(8)):
    r = {"feature": c, "IV": float(S16_TS_RANK.set_index("feature").loc[c, "IV"]), "spearman_with_PAY_1": stats.spearmanr(S16_TS[c], df["PAY_1"], nan_policy="omit")[0]}
    for g in ("Revolving", "Paid in full"):
        m = S16_GRP_SEP == g
        x = pd.Series(S16_TS.loc[m, c].to_numpy())
        b = pd.qcut(x, 5, duplicates="drop").astype(str) if x.nunique(dropna=True) > 5 else x.astype(str)
        r[f"IV_within_{g}"] = woe_iv(pd.DataFrame({"f": b.to_numpy(), TARGET: S16_Y[m]}), "f")[1]
    _rows.append(r)
S16_TS_MONEY_TAB = pd.DataFrame(_rows)
print("best money-only features, same check:"); display(S16_TS_MONEY_TAB.round(4))
export(pd.concat([S16_TS_TOP8_TAB.assign(set="top 8 overall"), S16_TS_MONEY_TAB.assign(set="top 8 money-only")], ignore_index=True), "s16_ts_top_features_vs_status")

**Observations:**
- Status features mostly repeat the Sep status: status_mean correlates 0.69 with PAY_1 and its IV drops to 0.16 inside Revolving; status_slope drops to 0.003.
- Money features are nearly unrelated to PAY_1 (Spearman −0.02 to −0.12) and keep real signal inside 'Paid in full' (spend_mean IV 0.30, pay_mean 0.29, spend_std 0.24).
- So money behaviour is new information mainly for clients whose status looks clean.
- Story use: key argument for adding a money lens next to the status code on the dashboard.

### 16.12 Trajectory gallery: twelve real clients, one "biography" each
Numbers become a story when the audience can see one person. For the six largest non-dormant utilisation shapes, the
gallery shows the **most typical** non-defaulter and defaulter: the client whose utilisation path is closest to the
median path of their shape × outcome group (ties broken toward a typical credit limit). Each panel: bill bars, payment
bars, the credit-limit line, and the status code printed above each month (red when late).
`s16_client_biography.csv` holds the client-month rows for the five most typical clients of every group
(`PICK_RANK` 1 = the one plotted), ready for a Tableau "client biography" sheet with a client selector.

In [ ]:
S16_GAL_SHAPES = [s for s in pd.Series(S16_USHAPE).value_counts().index if s != "Flat zero (dormant)"][:6]
_picks = []
for sh in S16_GAL_SHAPES:
    for g, lab in ((0, "No default"), (1, "Default")):
        idx = np.where((S16_USHAPE == sh) & (S16_Y == g) & (S16_B.max(1) > 0))[0]
        if len(idx) == 0:
            continue
        dist = ((S16_UC[idx] - np.median(S16_UC[idx], 0)) ** 2).sum(1) + 0.02 * (np.log(S16_LIM[idx]) - np.median(np.log(S16_LIM[idx]))) ** 2
        for rk, j in enumerate(idx[np.argsort(dist, kind="stable")][:5], start=1):
            _picks.append({"row": int(j), "ID": int(df["ID"].iloc[j]), "UTIL_SHAPE": sh, "DEFAULT_LABEL": lab, "PICK_RANK": rk, "GROUP_N": len(idx)})
S16_PICKS = pd.DataFrame(_picks)
_bio = []
for r in S16_PICKS.itertuples():
    j = r.row
    for k, mth in enumerate(S16_M6):
        _bio.append({"ID": r.ID, "UTIL_SHAPE": r.UTIL_SHAPE, "DEFAULT_LABEL": r.DEFAULT_LABEL, "PICK_RANK": r.PICK_RANK, "MONTH": mth, "MONTH_NUM": k + 1,
                     "BILL": S16_B[j, k], "PAY_AMT": S16_P[j, k], "LIMIT_BAL": S16_LIM[j], "UTIL": round(float(S16_U[j, k]), 4), "STATUS": int(S16_ST[j, k]),
                     "STATUS_LABEL": STATUS_LABEL.get(int(S16_ST[j, k]), str(S16_ST[j, k])), "SPEND": S16_SP[j, k - 1] if k > 0 else np.nan,
                     "FLOW_ARCHETYPE": S16_ARCH[j], "REGIME_SEQ": S16_REG_SEQ[j], "SEX_LABEL": df["SEX_LABEL"].iloc[j], "EDU_LABEL": df["EDU_LABEL"].iloc[j],
                     "MAR_LABEL": df["MAR_LABEL"].iloc[j], "AGE": int(df["AGE"].iloc[j])})
S16_BIO = pd.DataFrame(_bio)
export(S16_BIO, "s16_client_biography")

_top = S16_PICKS[S16_PICKS["PICK_RANK"] == 1].reset_index(drop=True)
fig, axes = plt.subplots(3, 4, figsize=(21, 12.5))
for ax in axes.ravel()[len(_top):]:
    ax.axis("off")
x = np.arange(6)
for ax, r in zip(axes.ravel(), _top.itertuples()):
    j = r.row
    ax.bar(x - .2, S16_B[j] / 1e3, .4, color="#6baed6", label="bill")
    ax.bar(x + .2, S16_P[j] / 1e3, .4, color=C_ACC, label="payment")
    ax.axhline(S16_LIM[j] / 1e3, color="k", ls="--", lw=1.2, label="credit limit")
    top = max(S16_LIM[j], S16_B[j].max(), S16_P[j].max()) / 1e3
    for k in range(6):
        ax.text(k, top * 1.06, f"{S16_ST[j, k]:+d}" if S16_ST[j, k] != 0 else "0", ha="center", va="bottom", fontsize=11, fontweight="bold",
                color=C_DEF if S16_ST[j, k] >= 1 else "#444444")
    ax.set_ylim(min(0, S16_B[j].min() / 1e3 * 1.1), top * 1.25)
    ax.set_xticks(x); ax.set_xticklabels(S16_M6)
    ax.set_ylabel("NT$ thousand")
    ax.set_title(f"{r.UTIL_SHAPE} · {r.DEFAULT_LABEL.upper()}\nID {r.ID} · {df['SEX_LABEL'].iloc[j]}, {int(df['AGE'].iloc[j])}, {df['EDU_LABEL'].iloc[j]} · {S16_ARCH[j]}",
                 fontsize=9, color=C_DEF if r.DEFAULT_LABEL == "Default" else C_OK)
axes[0, 0].legend(loc="center left", fontsize=8)
fig.suptitle("Trajectory gallery: the most typical client of each utilisation shape × outcome (numbers above the bars = repayment status code)", y=1.0, fontweight="bold")
plt.tight_layout(); savefig("s16_trajectory_gallery"); plt.show()
display(_top.merge(pd.DataFrame({"row": np.arange(S16_N), "LIMIT_BAL": S16_LIM, "status_Apr_to_Sep": [" ".join(str(v) for v in r_) for r_ in S16_ST],
                                 "regime": S16_REG_SEQ, "archetype": S16_ARCH, "avg_util": S16_U.mean(1)}), on="row").drop(columns="row").round(3))

**Observations:**
- The typical non-defaulting 'Ramp up' client (ID 28753, limit 180k) is a textbook BBBBB 'debt spiral' with status 0 in all six months.
- In 'Flat high' the typical defaulter (ID 5220, 98% utilisation) and non-defaulter (ID 6912, 94%) share the same FFFFF word; only one status 2 in Jul separates them.
- The typical 'Flat low' defaulter (ID 11626) sits at only 10% utilisation but carries status 2 in all six months.
- Story use: client biographies for the persona slides (`s16_client_biography.csv`).

### 16.13 Which money-flow lens is the strongest? One league table
Every categorical lens created in this section, ranked by information value, with its class count and the range of
class default rates. This is the shortlist for Tableau dimensions; the client-level export carries all of them.

In [ ]:
S16_LENSES = ["MF_UTIL_SHAPE", "MF_UTIL_LEVEL_CLUSTER", "MF_UTIL_PATH_CLASS", "MF_UTIL_PEAK_MONTH", "MF_PAYR_SHAPE", "MF_PAYR_LEVEL_CLUSTER", "MF_SPEND_SHAPE",
              "MF_SPEND_LEVEL_CLUSTER", "MF_REGIME_LAST", "MF_N_BUILD", "MF_N_PAYDOWN", "MF_FLOW_ARCHETYPE", "MF_PAY_LAST_MOVE", "MF_N_PAY_SHOCK",
              "MF_BIG_PAY_MONTH", "MF_PAY_COLLAPSE_BAND", "MF_N_FROZEN", "MF_FROZEN_RECENT", "MF_SPEND_PATTERN", "MF_COVERAGE_BAND", "MF_OL_DEPTH_BAND",
              "MF_OL_PATTERN", "MF_NEG_PATTERN"]
_rows = []
for c in S16_LENSES:
    s = df[c].astype(str)
    g = pd.DataFrame({"s": s, "y": S16_Y}).groupby("s")["y"].agg(["mean", "size"])
    g = g[g["size"] >= 100]
    _rows.append({"lens": c, "n_classes": int(s.nunique()), "IV": woe_iv(pd.DataFrame({"f": s.to_numpy(), TARGET: S16_Y}), "f")[1],
                  "cramers_v": cramers_v(s, df[TARGET]), "min_class_rate (n>=100)": g["mean"].min(), "max_class_rate (n>=100)": g["mean"].max(),
                  "riskiest_class": g["mean"].idxmax(), "safest_class": g["mean"].idxmin()})
S16_LEAGUE = pd.DataFrame(_rows).sort_values("IV", ascending=False).reset_index(drop=True)
display(S16_LEAGUE.round(4))
print(f"reference: IV of STATUS_GRP_SEP = {woe_iv(df, 'STATUS_GRP_SEP')[1]:.3f}, of UTIL_BAND = {woe_iv(df, 'UTIL_BAND')[1]:.3f}")
fig, ax = plt.subplots(figsize=(11, 7))
_t = S16_LEAGUE.iloc[::-1]
ax.barh(_t["lens"], _t["IV"], color=C_OK, alpha=.85)
for i, (iv, lo, hi) in enumerate(zip(_t["IV"], _t["min_class_rate (n>=100)"], _t["max_class_rate (n>=100)"])):
    ax.text(iv, i, f" {iv:.2f}  ({lo:.0%} – {hi:.0%})", va="center", fontsize=8)
ax.set_xlim(0, S16_LEAGUE["IV"].max() * 1.3); ax.set_title("Money-flow lenses ranked by information value (class default-rate range in brackets)")
plt.tight_layout(); savefig("s16_lens_league"); plt.show()
export(S16_LEAGUE, "s16_lens_league")
S16_MF_COLS = [c for c in df.columns if c.startswith("MF_")]
export(df[["ID", TARGET, "DEFAULT_LABEL"] + S16_MF_COLS].round(5), "s16_client_money_flow")
_b = S16_LEAGUE.iloc[0]
add_fact("16.13", "money flow", f"The strongest money-flow lens is {_b['lens']} (IV {_b['IV']:.2f}; class rates {_b['min_class_rate (n>=100)']:.0%} to "
         f"{_b['max_class_rate (n>=100)']:.0%}), against IV {woe_iv(df, 'STATUS_GRP_SEP')[1]:.2f} for the Sep status group", float(_b["IV"]), "s16_lens_league", 2)

**Observations:**
- The strongest money-flow lens is the number of build months (MF_N_BUILD, IV 0.20), then the last payment move (0.19), spend shape (0.16), Sep payment collapse (0.15) and coverage band (0.13).
- All are far below the Sep status group (IV 0.874) and only on a par with utilisation band (0.124).
- Riskiest classes: no build months 35.0%, stopped paying 38.5%, steady-spend flat path 38.6%.
- Story use: shortlist of Tableau dimensions for the money chapter (`s16_lens_league.csv`).

### 16.14 New columns, exports and story facts

**New `df` columns (prefix `MF_`)**

| Column | Meaning |
|---|---|
| `MF_UTIL_CURV`, `MF_UTIL_MAXDD`, `MF_UTIL_RUNUP`, `MF_UTIL_RANGE`, `MF_UTIL_TURNS`, `MF_UTIL_PEAK_MONTH` | utilisation-path descriptors: quadratic curvature, max drawdown, max run-up, range, turning points, month of peak |
| `MF_UTIL_PATH_CLASS` | slope class / curvature class (e.g. "Rising / Convex") |
| `MF_UTIL_SHAPE`, `MF_UTIL_LEVEL_CLUSTER` | utilisation shape (z-scored KMeans + flat classes) and raw-path (level + shape) cluster |
| `MF_PAYR_SHAPE`, `MF_PAYR_LEVEL_CLUSTER` | same for the payment-ratio path (May..Sep) |
| `MF_SPEND_SHAPE`, `MF_SPEND_LEVEL_CLUSTER` | same for the spend / limit path (May..Sep) |
| `MF_REGIME_SEQ`, `MF_N_BUILD`, `MF_N_PAYDOWN`, `MF_N_FLAT`, `MF_REGIME_LAST` | 5-letter balance-regime word (B/D/F/Z), counts, last regime |
| `MF_BUILD_RUN_MAX`, `MF_PAYDOWN_RUN_MAX`, `MF_FLOW_ARCHETYPE` | longest consecutive build / pay-down run; rule-based flow archetype |
| `MF_PAY_LAST_MOVE`, `MF_N_PAY_SHOCK`, `MF_PAY_SHOCK_SEP` | Aug→Sep payment move; number of payment shocks; shock in Sep |
| `MF_BIG_PAY_ANY`, `MF_BIG_PAY_MONTH` | one-off big payment flag and month of the most recent one |
| `MF_PAY_SEP_VS_PRIOR`, `MF_PAY_COLLAPSE_BAND` | Sep payment / own Apr–Aug mean, and its band |
| `MF_N_FROZEN`, `MF_FROZEN_RECENT` | months with a balance but no new charges; frozen run ending in Sep |
| `MF_SPEND_LATE_RATIO`, `MF_SPEND_SURGE_SEP`, `MF_SPEND_PATTERN` | late / early spend ratio, Sep surge flag, spend-pattern label |
| `MF_COVERAGE`, `MF_COVERAGE_BAND` | 5-month payments / new charges, and its band |
| `MF_OL_MAX_DEPTH`, `MF_OL_RUN_MAX`, `MF_OL_DEPTH_BAND`, `MF_OL_PATTERN` | over-limit depth, longest run, depth band, pattern |
| `MF_N_NEG_BILL`, `MF_NEG_MAX_DEPTH`, `MF_NEG_PATTERN` | credit-balance months, depth, pattern |
| `MF_TS_*` (8 columns) | the eight strongest features of the time-series bank (names depend on the ranking) |

**Key Tableau exports:** `s16_shape_centroids` (centroid paths with IQR ribbons), `s16_client_shapes` and
`s16_client_money_flow` (client-level assignments), `s16_regime_patterns`, `s16_flow_archetypes`, `s16_portfolio_flows`,
`s16_waterfall`, `s16_coverage_x_util`, `s16_overlimit_*`, `s16_ts_feature_ranking`, `s16_client_biography`.

In [ ]:
print(f"{len(S16_MF_COLS)} MF_ columns added:"); print(S16_MF_COLS)
S16_FACTS = pd.DataFrame([f for f in STORY_FACTS if f["section"].startswith("16")])
with pd.option_context("display.max_colwidth", None):
    display(S16_FACTS)
print(f"Section 16 runtime: {time.time() - S16_T0:.0f}s")

**Observations:**
- The section adds 50 `MF_` columns and registers 22 story facts (6 at strength 3).
- The full section runs in 109 s.
- Story use: background only.

## 17. Early warning, thresholds and tipping points

**Why this section matters for the story.** Sections 3-5 established *that* repayment status dominates default. A dashboard audience asks the operational follow-ups: **how early** was the trouble visible, **where exactly** are the lines that separate safe from unsafe (Tableau reference lines), **which simple warning signs** catch most defaulters and how much lead time they give, and **who slips through** with no warning at all. Everything here is descriptive: any model is a *lens* to measure accumulated information, never a deliverable.

Conventions: months are handled **chronologically** (Apr → Sep) in this section. "Late" = status ≥ 1 unless stated; because code 1 exists almost only in September (Section 5.0.1), every delinquency analysis carries a **status ≥ 2 sensitivity**. New `df` columns carry the prefix `EW_`; section globals `S17_` / `s17_`.

**Section guide**

- **What it does:** Turns the findings into practical warning rules. It measures how early default becomes visible, finds natural cut-off lines, tests for "tipping points", catalogues 26 simple warning signs, and studies who defaults with no warning.
- **Question it answers:** How early could we have seen it, where are the lines between safe and unsafe, and which simple signs catch most defaulters?
- **Inputs / new columns:** `PAY_1-6`, `BILL_AMT1-6`, `PAY_AMT1-6`, `LIMIT_BAL`, `AGE`, utilisation and payment ratios (all handled Apr → Sep). Adds columns with prefix `EW_` (e.g. `EW_N_SIGNS`, `EW_SIGN_<NAME>`, `EW_LEAD_TIME`, `EW_GROUP`, `EW_SILENT_DEFAULTER`; full list in 17.11).
- **How to read the charts:** survival curves: line = share of clients still not late (or not cured) after each month. Rate curves: dots = observed default rate (size = clients), red line = fitted segments, vertical lines = tipping points. Bubble chart: x = share of defaulters a sign catches, y = default rate when the sign fires. Dashed line = 22.1% base rate.
- **Terms:** AUC = chance a random defaulter ranks above a random non-defaulter (0.5 = coin flip). Kaplan-Meier = standard method for "share still surviving" curves. Lead time = months between the first alarm and September. Silent defaulter = defaulted with no warning sign.
- **Key takeaways:**
  - Sep alone carries 94% of the six-month signal (AUC 0.770 vs 0.788), but April data already gives 0.693 and 24% of defaulters were late by April.
  - The sharpest cliff is Sep status 1 → 2 (34% → 69% default, +35 points); paying zero vs a token amount in Sep moves default 53% → 22%; crossing the credit limit adds only +5.7 points.
  - A simple sum of Sep status + Aug status + late months puts 50% of defaulters in the top 20%, 97% of what the full model does.
  - Warning signs have limits: 10 of 26 popular signs mark safer clients, 8% of defaulters fire no core sign (20% in 300-500k limits), and 35% were never late.
  - Recovery is false comfort: cured-in-Sep clients still default at 38% (3.2× never-late); a big Sep payment does not reset an Aug-late client (69% with a spike).
  - Because code 1 exists almost only in September, every lateness result is also shown for status ≥ 2.
- **Where it feeds:** Tableau reference lines (`s17_thresholds.csv`), the warning-sign bubble chart (`s17_warning_signs.csv`), survival sheets (`s17_km_curves.csv`) and the "false comfort" bars.

### 17.0 Building blocks
Chronological matrices (Apr → Sep) for status, bill, payment, utilisation, month-level payment ratio and implied spend; small helpers for AUC, IV and gain curves.

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from scipy.special import expit as s17_expit

S17_T0 = time.time()
S17_RNG = np.random.default_rng(17)
S17_N = len(df)
S17_Y = df[TARGET].to_numpy(int)
S17_S = df[PAY_COLS].to_numpy(int)[:, ::-1].copy()        # Apr .. Sep
S17_B = df[BILL_COLS].to_numpy(float)[:, ::-1].copy()
S17_P = df[PAMT_COLS].to_numpy(float)[:, ::-1].copy()
S17_LIM = df["LIMIT_BAL"].to_numpy(float)
S17_U = S17_B / S17_LIM[:, None]

def s17_shift(A, k=1, fill=np.nan):
    # value of the same client k months earlier (first k months get `fill`)
    out = np.full(A.shape, fill, dtype=bool if isinstance(fill, bool) else float)
    out[:, k:] = A[:, :-k]
    return out

S17_BPREV = s17_shift(S17_B)
with np.errstate(divide="ignore", invalid="ignore"):
    S17_PR = np.where(S17_BPREV > 0, np.minimum(S17_P / np.where(S17_BPREV > 0, S17_BPREV, 1), 5), np.nan)   # payment / previous bill
S17_SP = S17_B - S17_BPREV + S17_P                                                                       # implied spend (NaN in Apr)

def s17_auc(x, y=None):
    y = S17_Y if y is None else y
    x = np.asarray(x, float); m = ~np.isnan(x)
    return float(roc_auc_score(y[m], x[m]))

def s17_iv(x):
    return woe_iv(pd.DataFrame({"x": np.asarray(x, float), TARGET: S17_Y}), "x")[1]

def s17_gain(score, y=None):
    # cumulative share of defaulters captured when clients are ranked by score (ties share credit linearly)
    y = S17_Y if y is None else y
    g = pd.DataFrame({"s": np.asarray(score, float), "y": y}).groupby("s")["y"].agg(["size", "sum"]).sort_index(ascending=False)
    return np.r_[0, g["size"].cumsum().to_numpy() / len(y)], np.r_[0, g["sum"].cumsum().to_numpy() / y.sum()]

def s17_smd(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    sd = np.sqrt((np.nanvar(a) + np.nanvar(b)) / 2)
    return float((np.nanmean(a) - np.nanmean(b)) / sd) if sd > 0 else 0.0

print(f"clients {S17_N:,} | base rate {BASE_RATE:.2%} | status matrix {S17_S.shape} (Apr..Sep)")

### 17.1 Information horizon: how early could we have seen it?
For every horizon (Apr only, Apr–May, …, Apr–Sep) features are rebuilt **only from data up to that month**: last status, max delay so far, late months so far, mean status, last / mean / max utilisation, zero-payment months, payment-to-limit, and payment ratio so far (payments ÷ previous bills; needs ≥ 2 months). Each gets a univariate AUC (direction-free, `max(a, 1−a)`) and IV. The **reverse** question uses only the **last k months** (Sep; Aug–Sep; …).

A gradient-boosted **lens** (5-fold out-of-fold AUC; window summaries + raw monthly columns + static columns) measures the *total* information in each window. It is a yardstick for accumulated information, not a model deliverable.

In [ ]:
try:
    import xgboost as s17_xgb
    S17_XGB_OK = True
except Exception:
    S17_XGB_OK = False
S17_DEV = "cpu"
if S17_XGB_OK and GPU:
    try:
        s17_xgb.train({"device": "cuda", "tree_method": "hist", "verbosity": 0},
                      s17_xgb.DMatrix(S17_RNG.random((256, 4)), label=S17_RNG.integers(0, 2, 256)), 2)
        S17_DEV = "cuda"
    except Exception as s17_e:
        print("xgboost CUDA not usable -> CPU:", str(s17_e)[:120])
S17_SKF = StratifiedKFold(5, shuffle=True, random_state=17)

def s17_oof(X):
    X = np.asarray(X, float); oof = np.zeros(S17_N); aucs = []
    for tr, te in S17_SKF.split(X, S17_Y):
        if S17_XGB_OK:
            mdl = s17_xgb.XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.08, subsample=0.8, colsample_bytree=0.8,
                                        min_child_weight=20, tree_method="hist", device=S17_DEV, n_jobs=4,
                                        eval_metric="logloss", random_state=17)
        else:
            from sklearn.ensemble import HistGradientBoostingClassifier
            mdl = HistGradientBoostingClassifier(max_depth=4, max_iter=150, learning_rate=0.08, random_state=17)
        mdl.fit(X[tr], S17_Y[tr]); oof[te] = mdl.predict_proba(X[te])[:, 1]
        aucs.append(roc_auc_score(S17_Y[te], oof[te]))
    return oof, float(np.mean(aucs)), float(np.std(aucs))

def s17_window_summary(lo, hi):
    st, ut, pa = S17_S[:, lo:hi + 1], S17_U[:, lo:hi + 1], S17_P[:, lo:hi + 1]
    out = pd.DataFrame({"status_last": st[:, -1].astype(float), "max_delay": st.max(1).astype(float),
                        "n_late": (st >= 1).sum(1).astype(float), "mean_status": st.mean(1),
                        "util_last": ut[:, -1], "avg_util": ut.mean(1), "max_util": ut.max(1),
                        "n_zero_pay": (pa == 0).sum(1).astype(float), "pay_to_limit": pa.mean(1) / S17_LIM})
    if hi > lo:
        num = S17_P[:, lo + 1:hi + 1].sum(1); den = np.clip(S17_B[:, lo:hi], 0, None).sum(1)
        out["pay_ratio"] = np.where(den > 0, np.minimum(num / np.where(den > 0, den, 1), 5), np.nan)
    else:
        out["pay_ratio"] = np.nan
    return out

S17_STATIC = df[["LIMIT_BAL", "AGE", "SEX", "EDUCATION", "MARRIAGE"]].astype(float).reset_index(drop=True)
def s17_window_matrix(lo, hi):
    raw = {}
    for _m in range(lo, hi + 1):
        raw[f"st_{MONTH_ORDER[_m]}"] = S17_S[:, _m]; raw[f"bill_{MONTH_ORDER[_m]}"] = S17_B[:, _m]
        raw[f"pay_{MONTH_ORDER[_m]}"] = S17_P[:, _m]; raw[f"util_{MONTH_ORDER[_m]}"] = S17_U[:, _m]
    return pd.concat([s17_window_summary(lo, hi), pd.DataFrame(raw), S17_STATIC], axis=1)

s17_uni_rows, s17_lens_rows, S17_OOF = [], [], {}
s17_o, s17_mu, s17_sd = s17_oof(S17_STATIC)
s17_lens_rows.append({"direction": "static only", "window": "no behaviour", "n_months": 0,
                      "auc_oof": roc_auc_score(S17_Y, s17_o), "auc_fold_mean": s17_mu, "auc_fold_sd": s17_sd})
for _dir in ("forward", "reverse"):
    for _k in range(1, 7):
        _lo, _hi = (0, _k - 1) if _dir == "forward" else (6 - _k, 5)
        _lab = MONTH_ORDER[_lo] if _lo == _hi else f"{MONTH_ORDER[_lo]}-{MONTH_ORDER[_hi]}"
        _sm = s17_window_summary(_lo, _hi)
        for _f in _sm.columns:
            _x = _sm[_f].to_numpy(float)
            if np.isnan(_x).all() or np.nanstd(_x) == 0:
                continue
            _a = s17_auc(_x)
            s17_uni_rows.append({"direction": _dir, "window": _lab, "n_months": _k, "feature": _f, "auc": max(_a, 1 - _a),
                                 "higher_is_riskier": _a >= 0.5, "iv": s17_iv(_x), "n_valid": int((~np.isnan(_x)).sum())})
        if _dir == "reverse" and _k == 6:
            s17_o = S17_OOF[("forward", 6)]; s17_mu, s17_sd = S17_LENS_FULL
        else:
            s17_o, s17_mu, s17_sd = s17_oof(s17_window_matrix(_lo, _hi))
        if _dir == "forward" and _k == 6:
            S17_LENS_FULL = (s17_mu, s17_sd)
        S17_OOF[(_dir, _k)] = s17_o
        s17_lens_rows.append({"direction": _dir, "window": _lab, "n_months": _k, "auc_oof": roc_auc_score(S17_Y, s17_o),
                              "auc_fold_mean": s17_mu, "auc_fold_sd": s17_sd})
S17_H_UNI = pd.DataFrame(s17_uni_rows)
S17_H_LENS = pd.DataFrame(s17_lens_rows)
S17_AUC_STATIC = float(S17_H_LENS.loc[0, "auc_oof"])
S17_H_LENS["gain_vs_previous"] = S17_H_LENS.groupby("direction")["auc_oof"].diff()
S17_H_LENS.loc[S17_H_LENS["n_months"] == 1, "gain_vs_previous"] = S17_H_LENS.loc[S17_H_LENS["n_months"] == 1, "auc_oof"] - S17_AUC_STATIC
S17_H_LENS["share_of_full_signal"] = (S17_H_LENS["auc_oof"] - 0.5) / (S17_H_LENS.loc[(S17_H_LENS["direction"] == "forward") & (S17_H_LENS["n_months"] == 6), "auc_oof"].iloc[0] - 0.5)
df["EW_LENS_OOF"] = S17_OOF[("forward", 6)]
print(f"lens = {'xgboost' if S17_XGB_OK else 'sklearn HistGradientBoosting'} on {S17_DEV}; 5-fold out-of-fold AUC")
display(S17_H_LENS.round(4))
export(S17_H_LENS, "s17_info_horizon_lens"); export(S17_H_UNI, "s17_info_horizon_univariate")

**Information accumulation curve.** Left: lens AUC as history is added from April forward (blue) and as it is added backward from September (red). Middle / right: univariate AUC and IV of each "so-far" feature by forward horizon.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 5.6))
ax = axes[0]
for _dir, _col, _lb in (("forward", C_OK, "forward: Apr .. month k (how early?)"), ("reverse", C_DEF, "reverse: last k months ending Sep")):
    _t = S17_H_LENS[S17_H_LENS["direction"] == _dir]
    ax.errorbar(_t["n_months"], _t["auc_oof"], yerr=1.96 * _t["auc_fold_sd"] / np.sqrt(5), marker="o", lw=2, capsize=3, color=_col, label=_lb)
    for _r in _t.itertuples():
        ax.annotate(_r.window, (_r.n_months, _r.auc_oof), textcoords="offset points", xytext=(0, 9 if _dir == "reverse" else -15), ha="center", fontsize=7, color=_col)
ax.axhline(S17_AUC_STATIC, ls="--", color=C_NEU, label=f"static columns only ({S17_AUC_STATIC:.3f})")
ax.set_xlabel("months of history in the window"); ax.set_ylabel("lens out-of-fold AUC"); ax.legend(fontsize=8, loc="lower right")
ax.set_title("Information accumulation: AUC vs months of history")
_fw = S17_H_UNI[S17_H_UNI["direction"] == "forward"]
for _f, _g in _fw.groupby("feature"):
    axes[1].plot(_g["n_months"], _g["auc"], "o-", label=_f); axes[2].plot(_g["n_months"], _g["iv"], "o-", label=_f)
for ax, _ttl in ((axes[1], "Univariate AUC by forward horizon"), (axes[2], "Information Value by forward horizon")):
    ax.set_xticks(range(1, 7)); ax.set_xticklabels(["Apr", "Apr-May", "Apr-Jun", "Apr-Jul", "Apr-Aug", "Apr-Sep"], fontsize=8)
    ax.set_title(_ttl); ax.legend(fontsize=7, ncol=2)
fig.suptitle("17.1 How early could we have seen it?", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s17_info_horizon"); plt.show()

s17_pv = S17_H_UNI.assign(win=S17_H_UNI["direction"].str[:3] + " " + S17_H_UNI["window"])
s17_ord = [f"for {w}" for w in ["Apr", "Apr-May", "Apr-Jun", "Apr-Jul", "Apr-Aug", "Apr-Sep"]] + [f"rev {w}" for w in ["Sep", "Aug-Sep", "Jul-Sep", "Jun-Sep", "May-Sep", "Apr-Sep"]]
fig, axes = plt.subplots(1, 2, figsize=(20, 5))
for ax, _v, _fm in ((axes[0], "auc", ".3f"), (axes[1], "iv", ".2f")):
    sns.heatmap(s17_pv.pivot(index="feature", columns="win", values=_v).reindex(columns=s17_ord), annot=True, fmt=_fm, cmap="viridis", ax=ax, cbar=False, annot_kws={"fontsize": 8})
    ax.set_title(f"Univariate {_v.upper()} by window (for = forward from Apr, rev = last k months)"); ax.set_xlabel(""); ax.set_ylabel("")
    ax.axvline(6, color="w", lw=3)
plt.tight_layout(); savefig("s17_info_horizon_heatmaps"); plt.show()

s17_f = S17_H_LENS[S17_H_LENS["direction"] == "forward"].set_index("n_months"); s17_r = S17_H_LENS[S17_H_LENS["direction"] == "reverse"].set_index("n_months")
print(f"static only {S17_AUC_STATIC:.3f} | Apr only {s17_f.loc[1, 'auc_oof']:.3f} | Apr-Jun {s17_f.loc[3, 'auc_oof']:.3f} | Apr-Aug {s17_f.loc[5, 'auc_oof']:.3f} | all {s17_f.loc[6, 'auc_oof']:.3f}")
print(f"Sep only {s17_r.loc[1, 'auc_oof']:.3f} | Aug-Sep {s17_r.loc[2, 'auc_oof']:.3f} | Jul-Sep {s17_r.loc[3, 'auc_oof']:.3f}")
add_fact("17.1", "early warning", f"With April data alone the lens already reaches AUC {s17_f.loc[1, 'auc_oof']:.3f} ({s17_f.loc[1, 'share_of_full_signal']:.0%} of the full six-month signal above chance); static columns alone give {S17_AUC_STATIC:.3f}", float(s17_f.loc[1, "auc_oof"]), "s17_info_horizon", 3)
add_fact("17.1", "recency", f"September alone carries {s17_r.loc[1, 'share_of_full_signal']:.0%} of the full signal (AUC {s17_r.loc[1, 'auc_oof']:.3f} vs {s17_f.loc[6, 'auc_oof']:.3f}); adding August lifts it by {s17_r.loc[2, 'gain_vs_previous']:+.3f}, months before July add {s17_r.loc[6, 'auc_oof'] - s17_r.loc[3, 'auc_oof']:+.3f} in total", float(s17_r.loc[1, "share_of_full_signal"]), "s17_info_horizon", 3)

**Observations:**
- Static columns alone (limit, age, education ...) give AUC 0.627 (AUC = chance a random defaulter is ranked above a random non-defaulter; 0.5 = coin flip).
- April data alone already lifts this to 0.693; each further month adds only 0.013-0.020 until Sep adds the biggest step (+0.033) to reach 0.788.
- Counted backwards, Sep alone gives 0.770 (94% of the full signal); adding Aug gives 0.776 and everything before Jul adds only about 0.005.
- Story use: the 'recency' slide: the last month says almost everything, but warning signs are visible from April.

#### 17.1b The simplest horizon rule: "ever late so far"
Without any model: flag a client the first time a late status appears. For each month: share of clients flagged so far, hit rate (default rate of the flagged), capture (share of all eventual defaulters already flagged). Both late definitions (≥ 1, ≥ 2). Also: when did eventual defaulters **first** show a late status?

In [ ]:
s17_rows = []
for _thr in (1, 2):
    _cum = np.maximum.accumulate(S17_S >= _thr, axis=1)
    for _m in range(6):
        _fl = _cum[:, _m]
        s17_rows.append({"late_definition": f"status >= {_thr}", "through_month": MONTH_ORDER[_m], "flagged_share": _fl.mean(),
                         "hit_rate": S17_Y[_fl].mean(), "capture_of_defaulters": _fl[S17_Y == 1].mean(),
                         "false_alarm_share_of_good": _fl[S17_Y == 0].mean(), "rate_if_not_flagged": S17_Y[~_fl].mean()})
S17_EVERLATE = pd.DataFrame(s17_rows)
display(S17_EVERLATE.round(4)); export(S17_EVERLATE, "s17_everlate_by_horizon")

s17_first = np.where((S17_S >= 1).any(1), (S17_S >= 1).argmax(1), 6)
s17_fd = pd.crosstab(pd.Categorical.from_codes(s17_first, MONTH_ORDER + ["Never late"]), df["DEFAULT_LABEL"].to_numpy(), normalize="columns")
fig, axes = plt.subplots(1, 2, figsize=(17, 4.8))
for _thr, _ls in ((1, "-"), (2, "--")):
    _t = S17_EVERLATE[S17_EVERLATE["late_definition"] == f"status >= {_thr}"]
    axes[0].plot(_t["through_month"], _t["capture_of_defaulters"], "o" + _ls, color=C_DEF, label=f"defaulters already flagged (>= {_thr})")
    axes[0].plot(_t["through_month"], _t["false_alarm_share_of_good"], "o" + _ls, color=C_OK, label=f"non-defaulters flagged (>= {_thr})")
    axes[0].plot(_t["through_month"], _t["hit_rate"], "s" + _ls, color="k", alpha=.6, label=f"hit rate of the flag (>= {_thr})")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].legend(fontsize=8); axes[0].set_title("Cumulative 'ever late so far' flag by month")
s17_fd.plot(kind="bar", ax=axes[1], color=[C_DEF, C_OK] if s17_fd.columns[0] == "Default" else [C_OK, C_DEF], width=.8)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("Month of FIRST late status (>= 1), share within outcome"); axes[1].tick_params(axis="x", rotation=0); axes[1].set_xlabel("")
plt.tight_layout(); savefig("s17_everlate_horizon"); plt.show()
print(s17_fd.round(4).to_string())
s17_a = S17_EVERLATE[(S17_EVERLATE["late_definition"] == "status >= 1")].set_index("through_month")
add_fact("17.1", "early warning", f"{s17_a.loc['Apr', 'capture_of_defaulters']:.0%} of eventual defaulters were already late in April (hit rate {s17_a.loc['Apr', 'hit_rate']:.0%}); by August the 'ever late' flag covers {s17_a.loc['Aug', 'capture_of_defaulters']:.0%} of defaulters; {1 - s17_a.loc['Sep', 'capture_of_defaulters']:.0%} of defaulters were never late at all", float(s17_a.loc["Apr", "capture_of_defaulters"]), "s17_everlate_horizon", 2)

**Observations:**
- By April 10.3% of clients are already flagged as late, and they hold 24.3% of eventual defaulters (hit rate 52%).
- Capture grows steadily to 52% by Aug; with Sep included it reaches 64.8% under status ≥ 1 but only 58.5% under status ≥ 2, because code 1 exists almost only in Sep.
- Through Aug both definitions are almost identical, as expected; the flag's hit rate falls from 52% to about 43-46% as it widens.
- 35.2% of defaulters were never late in any month: this simple rule can never catch them.
- Story use: the 'how early could we know' timeline for Tableau (`s17_everlate_by_horizon.csv`).

### 17.2 Survival view: how long do clients stay clean, and how long do they stay late?
**Kaplan–Meier** curves (the standard estimate of the share still "surviving", here still not late, after each month), implemented in numpy (Greenwood standard errors; `lifelines` is used only as an optional cross-check). Everyone is observed for all six months, so censoring is purely administrative (end of September).

* **Time to first delinquency.** Clients already late in April are *left-censored* (we do not know when they started), so the risk set is the clients **current in April**; clock = months after April (1 = May … 5 = Sep).
* **Sep quirk.** With "late = status ≥ 1" the September step is inflated because code 1 is recorded (almost) only in September; the ≥ 2 curve is the like-for-like series.
* Groups are compared with a k-sample **log-rank test** (tests whether two or more survival curves differ) (hypergeometric variance, χ² with k−1 df).

(Section 11 holds the person-month hazard tables; here we only draw curves.)

In [ ]:
def s17_km(time_, event_, tmax):
    rows, surv, vs = [], 1.0, 0.0
    for _t in range(1, tmax + 1):
        nr = int((time_ >= _t).sum()); d = int(((time_ == _t) & event_).sum())
        if nr > 0:
            surv *= 1 - d / nr
        if nr > d:
            vs += d / (nr * (nr - d))
        se = surv * np.sqrt(vs)
        rows.append({"t": _t, "at_risk": nr, "events": d, "survival": surv, "ci_lo": max(0.0, surv - 1.96 * se), "ci_hi": min(1.0, surv + 1.96 * se)})
    return pd.DataFrame(rows)

def s17_logrank(time_, event_, codes, tmax):
    k = int(codes.max()) + 1; OE = np.zeros(k); V = np.zeros((k, k))
    for _t in range(1, tmax + 1):
        ar = time_ >= _t; ev = (time_ == _t) & event_
        nj = np.bincount(codes[ar], minlength=k).astype(float); dj = np.bincount(codes[ev], minlength=k).astype(float)
        Nn, D = nj.sum(), dj.sum()
        if Nn < 2 or D == 0:
            continue
        OE += dj - D * nj / Nn
        V += D * (Nn - D) / (Nn ** 2 * (Nn - 1)) * (np.diag(nj * Nn) - np.outer(nj, nj))
    stat = float(OE[:-1] @ np.linalg.pinv(V[:-1, :-1]) @ OE[:-1])
    return stat, float(stats.chi2.sf(stat, k - 1)), k - 1

def s17_first_time(flag):
    # flag: n x T boolean (months after entry). Returns time 1..T and event indicator (censored at T if never)
    ev = flag.any(1)
    return np.where(ev, flag.argmax(1) + 1, flag.shape[1]), ev

S17_KM_ROWS, S17_KM_SUMM = [], []
def s17_km_group(mask, time_, event_, segname, tmax, analysis, definition, months, order=None):
    if segname is None:
        groups = {"All": mask}; segname = "(all)"
    else:
        sv = df[segname].astype(str).to_numpy()
        levels = [o for o in (order or sorted(pd.unique(sv[mask]))) if ((sv == o) & mask).sum() >= 50]
        groups = {lv: mask & (sv == lv) for lv in levels}
    out, chi, pv, dof = {}, np.nan, np.nan, 0
    if len(groups) > 1:
        codes = np.full(S17_N, -1)
        for _j, gm in enumerate(groups.values()):
            codes[gm] = _j
        um = codes >= 0
        chi, pv, dof = s17_logrank(time_[um], event_[um], codes[um], tmax)
    for lv, gm in groups.items():
        km = s17_km(time_[gm], event_[gm], tmax); out[lv] = km
        for _r in km.itertuples():
            S17_KM_ROWS.append({"analysis": analysis, "definition": definition, "segment_var": segname, "segment": lv, "t": _r.t,
                                "month": months[_r.t], "n": int(gm.sum()), "at_risk": _r.at_risk, "events": _r.events,
                                "survival": _r.survival, "ci_lo": _r.ci_lo, "ci_hi": _r.ci_hi})
        S17_KM_ROWS.append({"analysis": analysis, "definition": definition, "segment_var": segname, "segment": lv, "t": 0, "month": months[0],
                            "n": int(gm.sum()), "at_risk": int(gm.sum()), "events": 0, "survival": 1.0, "ci_lo": 1.0, "ci_hi": 1.0})
        S17_KM_SUMM.append({"analysis": analysis, "definition": definition, "segment_var": segname, "segment": lv, "n": int(gm.sum()),
                            "S_before_last": km["survival"].iloc[-2], "S_end": km["survival"].iloc[-1],
                            "mean_event_free_months": 1 + km["survival"].iloc[:-1].sum(), "max_months": tmax,
                            "logrank_chi2": chi, "logrank_df": dof, "logrank_p": pv})
    return out

def s17_km_plot(ax, curves, months, title, ylabel, palette=None):
    cols = sns.color_palette(palette or "viridis", len(curves))
    for (lv, km), _c in zip(curves.items(), cols):
        ax.step(np.r_[0, km["t"]], np.r_[1, km["survival"]], where="post", color=_c, lw=2, label=f"{lv} (n={int(km['at_risk'].iloc[0]):,})")
        ax.fill_between(np.r_[0, km["t"]], np.r_[1, km["ci_lo"]], np.r_[1, km["ci_hi"]], step="post", color=_c, alpha=.12)
    ax.set_xticks(range(len(months))); ax.set_xticklabels(months); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(title, fontsize=10); ax.set_ylabel(ylabel); ax.legend(fontsize=7)

df["EW_UTIL_APR_BAND"] = pd.cut(df["UTIL_6"], [-np.inf, 0, 0.3, 0.7, 1.0, np.inf], labels=["<=0", "0-30%", "30-70%", "70-100%", ">100%"])
S17_UAPR_ORDER = ["<=0", "0-30%", "30-70%", "70-100%", ">100%"]
S17_TTL = {}
for _thr in (1, 2):
    S17_TTL[_thr] = s17_first_time(S17_S[:, 1:] >= _thr) + (S17_S[:, 0] < _thr,)
print("risk sets (current in April):", {f">={k}": int(v[2].sum()) for k, v in S17_TTL.items()}, "| left-censored (late in April):", int((S17_S[:, 0] >= 1).sum()))

**Time to first delinquency** for clients current in April — overall under both late definitions, then by credit-limit band, age band, education, April utilisation band and April status group (late = status ≥ 1).

In [ ]:
S17_SEGS = [("LIMIT_BAND", LIMIT_LABELS), ("AGE_BAND", AGE_LABELS), ("EDU_LABEL", EDU_ORDER), ("EW_UTIL_APR_BAND", S17_UAPR_ORDER),
            ("STATUS_GRP_6", ["Inactive", "Paid in full", "Revolving"])]
fig, axes = plt.subplots(2, 3, figsize=(21, 10.5)); axes = axes.ravel()
s17_all = {}
for _thr in (1, 2):
    _tt, _ev, _mk = S17_TTL[_thr]
    s17_all[f"late = status >= {_thr}"] = s17_km_group(_mk, _tt, _ev, None, 5, "time to first delinquency", f"status >= {_thr}", MONTH_ORDER)["All"]
    for _seg, _ord in S17_SEGS:
        _cur = s17_km_group(_mk, _tt, _ev, _seg, 5, "time to first delinquency", f"status >= {_thr}", MONTH_ORDER, _ord)
        if _thr == 1:
            _ax = axes[1 + [s[0] for s in S17_SEGS].index(_seg)]
            _lr = [r for r in S17_KM_SUMM if r["segment_var"] == _seg and r["definition"] == "status >= 1" and r["analysis"] == "time to first delinquency"][0]
            s17_km_plot(_ax, _cur, MONTH_ORDER, f"by {_seg}  (log-rank chi2 {_lr['logrank_chi2']:.0f}, df {_lr['logrank_df']}, p {_lr['logrank_p']:.1e})", "still never late")
s17_km_plot(axes[0], s17_all, MONTH_ORDER, "All clients current in April: two late definitions", "still never late", "Set1")
fig.suptitle("17.2 Kaplan-Meier: share of April-current clients who have not yet been late", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s17_km_first_late"); plt.show()
S17_KM_SUMM_DF = pd.DataFrame(S17_KM_SUMM)
display(S17_KM_SUMM_DF[S17_KM_SUMM_DF["analysis"] == "time to first delinquency"].round(4))
s17_s1, s17_s2 = s17_all["late = status >= 1"], s17_all["late = status >= 2"]
print(f"never late through Aug: {s17_s1['survival'].iloc[3]:.1%} (>=1) | through Sep: {s17_s1['survival'].iloc[4]:.1%} (>=1) vs {s17_s2['survival'].iloc[4]:.1%} (>=2)")
print(f"September step: {s17_s1['events'].iloc[4] / s17_s1['at_risk'].iloc[4]:.1%} of the still-clean go late in Sep under >=1, vs {s17_s2['events'].iloc[4] / s17_s2['at_risk'].iloc[4]:.1%} under >=2 and {s17_s1['events'].iloc[:4].sum() / s17_s1['at_risk'].iloc[:4].sum():.1%} per month in May-Aug")
add_fact("17.2", "survival", f"Of clients current in April, {s17_s1['survival'].iloc[3]:.0%} are still never-late at end of August and {s17_s1['survival'].iloc[4]:.0%} at end of September (status>=1); the September step ({s17_s1['events'].iloc[4] / s17_s1['at_risk'].iloc[4]:.1%} vs {s17_s2['events'].iloc[4] / s17_s2['at_risk'].iloc[4]:.1%} under status>=2) is mostly the Sep-only code 1", float(s17_s1["survival"].iloc[4]), "s17_km_first_late", 2)

**Observations:**
- Kaplan-Meier (share still never late after each month) for the 26,921 clients current in April: 82.8% are still clean at end of Aug, 74.0% at end of Sep under status ≥ 1, 80.3% under status ≥ 2.
- The Sep drop is mostly the Sep-only code 1: 10.6% of the still-clean go late in Sep under ≥ 1 vs 3.1% under ≥ 2, compared with about 4.6% per month in May-Aug.
- Small limits go late faster: 62.7% of ≤50k clients stay clean to Sep vs 83.2% at 300-500k; high April utilisation (>100%) gives only 59.2%.
- Story use: survival curves by limit band are a clean Tableau line chart; keep the status ≥ 2 version to avoid the Sep artefact.

#### 17.2b Optional cross-check against `lifelines` and segment spread under the stricter definition
If `lifelines` is installed the numpy KM is compared with `KaplanMeierFitter` (max absolute difference). The table then ranks segments by end-of-window survival under **status ≥ 2** (free of the September quirk).

In [ ]:
try:
    from lifelines import KaplanMeierFitter
    _tt, _ev, _mk = S17_TTL[1]
    _kmf = KaplanMeierFitter().fit(_tt[_mk], _ev[_mk])
    _ll = _kmf.survival_function_at_times([1, 2, 3, 4, 5]).to_numpy()
    print(f"lifelines cross-check: max |numpy KM - lifelines| = {np.abs(_ll - s17_s1['survival'].to_numpy()).max():.2e}")
except Exception as s17_e:
    print("lifelines not available (optional) - numpy implementation used:", type(s17_e).__name__)
s17_t = S17_KM_SUMM_DF[(S17_KM_SUMM_DF["analysis"] == "time to first delinquency") & (S17_KM_SUMM_DF["definition"] == "status >= 2") & (S17_KM_SUMM_DF["segment_var"] != "(all)")].copy()
s17_t["went_late_by_Sep"] = 1 - s17_t["S_end"]
display(s17_t.sort_values("went_late_by_Sep", ascending=False)[["segment_var", "segment", "n", "went_late_by_Sep", "mean_event_free_months", "logrank_chi2", "logrank_p"]].round(4))
s17_lr = s17_t.drop_duplicates("segment_var").set_index("segment_var")["logrank_chi2"].sort_values(ascending=False)
print("log-rank chi2 by segmenting variable (status >= 2):"); print(s17_lr.round(1).to_string())
s17_hi, s17_lo = s17_t.sort_values("went_late_by_Sep").iloc[-1], s17_t.sort_values("went_late_by_Sep").iloc[0]
add_fact("17.2", "survival", f"Among April-current clients, {s17_hi['went_late_by_Sep']:.0%} of '{s17_hi['segment_var']} = {s17_hi['segment']}' reach a 2-month delay by September vs {s17_lo['went_late_by_Sep']:.0%} of '{s17_lo['segment_var']} = {s17_lo['segment']}'; the strongest separator of time-to-first-delinquency is {s17_lr.index[0]} (log-rank chi2 {s17_lr.iloc[0]:.0f})", float(s17_hi["went_late_by_Sep"]), "s17_km_first_late", 2)

**Observations:**
- `lifelines` is not installed, so only the numpy Kaplan-Meier is shown (no cross-check).
- Under status ≥ 2, the riskiest April segments are utilisation >100% (40% reach a 2-month delay by Sep), 70-100% (34%) and limit ≤50k (34%); the safest are Inactive in Apr (8%) and limits above 300k (9%).
- Limit band separates time-to-first-delinquency best (log-rank chi2 1,485; chi2 = how strongly the curves differ), ahead of April utilisation (1,083) and April status (819); age (183) and education (254) are weaker.
- Story use: background for the survival chart; supports 'limit and utilisation, not demographics'.

#### 17.2c Time to cure: once late, how long until the account is clean again?
Two entry cohorts: clients **late in April** (onset unknown, so this is time-to-cure *from April*) and clients **newly late in May** (current in April, late in May: a clean incident cohort). Cure = first later month with status ≤ 0. Sensitivity: cure = status ≤ 1 (treats the September code 1 as cured). Curves show the share **still not cured**.

In [ ]:
S17_CURE = {}
S17_CURE["late in Apr"] = (S17_S[:, 0] >= 1, 1, MONTH_ORDER)
S17_CURE["newly late in May"] = ((S17_S[:, 0] <= 0) & (S17_S[:, 1] >= 1), 2, MONTH_ORDER[1:])
df["EW_ENTRY_SEVERITY_APR"] = np.where(S17_S[:, 0] >= 3, "3+ months", np.where(S17_S[:, 0] == 2, "2 months", np.where(S17_S[:, 0] == 1, "1 month", "not late")))
df["EW_ENTRY_SEVERITY_MAY"] = np.where(S17_S[:, 1] >= 3, "3+ months", np.where(S17_S[:, 1] == 2, "2 months", np.where(S17_S[:, 1] == 1, "1 month", "not late")))
fig, axes = plt.subplots(2, 3, figsize=(21, 10)); S17_CURE_T = {}
for _row, (_nm, (_mk, _st, _mon)) in enumerate(S17_CURE.items()):
    _tmax = 6 - _st
    _tt, _ev = s17_first_time(S17_S[:, _st:] <= 0); S17_CURE_T[_nm] = (_tt, _ev, _mk)
    _tt2, _ev2 = s17_first_time(S17_S[:, _st:] <= 1)
    _o = {"cure = status <= 0": s17_km_group(_mk, _tt, _ev, None, _tmax, f"time to cure ({_nm})", "cure: status <= 0", _mon)["All"],
          "cure = status <= 1 (sensitivity)": s17_km_group(_mk, _tt2, _ev2, None, _tmax, f"time to cure ({_nm})", "cure: status <= 1", _mon)["All"]}
    s17_km_plot(axes[_row, 0], _o, _mon, f"Cohort {_nm}: share still not cured", "still not cured", "Set1")
    _sevcol = "EW_ENTRY_SEVERITY_APR" if _st == 1 else "EW_ENTRY_SEVERITY_MAY"
    for _j, (_seg, _ord) in enumerate(((_sevcol, ["1 month", "2 months", "3+ months"]), ("LIMIT_BAND", LIMIT_LABELS))):
        _cur = s17_km_group(_mk, _tt, _ev, _seg, _tmax, f"time to cure ({_nm})", "cure: status <= 0", _mon, _ord)
        _lr = [r for r in S17_KM_SUMM if r["segment_var"] == _seg and r["analysis"] == f"time to cure ({_nm})"][0]
        s17_km_plot(axes[_row, 1 + _j], _cur, _mon, f"{_nm}: by {_seg} (log-rank chi2 {_lr['logrank_chi2']:.0f}, p {_lr['logrank_p']:.1e})", "still not cured")
fig.suptitle("17.2c Kaplan-Meier time to cure", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s17_km_cure"); plt.show()
S17_KM_SUMM_DF = pd.DataFrame(S17_KM_SUMM); S17_KM_DF = pd.DataFrame(S17_KM_ROWS).sort_values(["analysis", "definition", "segment_var", "segment", "t"])
display(S17_KM_SUMM_DF[S17_KM_SUMM_DF["analysis"].str.startswith("time to cure")].round(4))
export(S17_KM_DF, "s17_km_curves"); export(S17_KM_SUMM_DF, "s17_km_summary")

Does the **speed of cure** matter for the October outcome? Default rate by the month of first cure for both cohorts, split by whether the client relapsed (went late again) afterwards.

In [ ]:
s17_rows = []
for _nm, (_mk0, _st, _mon) in S17_CURE.items():
    _tt, _ev, _mk = S17_CURE_T[_nm]
    for _t in list(range(1, 6 - _st + 1)) + [None]:
        _g = _mk & ((~_ev) if _t is None else (_ev & (_tt == _t)))
        if _g.sum() == 0:
            continue
        if _t is None:
            _rel = np.zeros(S17_N, bool)
        else:
            _rel = (S17_S[:, _st + _t:] >= 1).any(1) if _st + _t < 6 else np.zeros(S17_N, bool)
        for _rl, _gm in (("all", _g), ("no relapse", _g & ~_rel), ("relapsed later", _g & _rel)):
            if _gm.sum() >= 30:
                _lo, _hi = wilson_ci(S17_Y[_gm].sum(), _gm.sum())
                s17_rows.append({"cohort": _nm, "first_cure_month": "never cured" if _t is None else _mon[_t], "months_to_cure": np.nan if _t is None else _t,
                                 "relapse": _rl, "n": int(_gm.sum()), "rate": S17_Y[_gm].mean(), "ci_lo": float(_lo), "ci_hi": float(_hi)})
S17_CURE_RATE = pd.DataFrame(s17_rows)
fig, axes = plt.subplots(1, 2, figsize=(17, 4.8))
for ax, _nm in zip(axes, S17_CURE):
    _t = S17_CURE_RATE[(S17_CURE_RATE["cohort"] == _nm) & (S17_CURE_RATE["relapse"] == "all")]
    ax.bar(_t["first_cure_month"], _t["rate"], yerr=[_t["rate"] - _t["ci_lo"], _t["ci_hi"] - _t["rate"]], color=[C_DEF if m == "never cured" else C_ACC for m in _t["first_cure_month"]], capsize=3, alpha=.85)
    for _x, (_r, _n) in enumerate(zip(_t["rate"], _t["n"])):
        ax.text(_x, _r + .03, f"{_r:.0%}\nn={_n:,}", ha="center", fontsize=8)
    ax.axhline(BASE_RATE, ls="--", color="k", lw=1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_ylim(0, 1)
    ax.set_title(f"Cohort {_nm}: default rate by month of first cure")
plt.tight_layout(); savefig("s17_cure_speed_default"); plt.show()
display(S17_CURE_RATE.round(4)); export(S17_CURE_RATE, "s17_cure_speed_default")
s17_c = S17_CURE_RATE[(S17_CURE_RATE["cohort"] == "late in Apr") & (S17_CURE_RATE["relapse"] == "all")].set_index("first_cure_month")
s17_kc = S17_KM_SUMM_DF[(S17_KM_SUMM_DF["analysis"] == "time to cure (late in Apr)") & (S17_KM_SUMM_DF["segment_var"] == "(all)") & (S17_KM_SUMM_DF["definition"] == "cure: status <= 0")].iloc[0]
add_fact("17.2", "cure", f"Of clients late in April, {s17_kc['S_end']:.0%} never return to a clean status by September; those never cured default at {s17_c.loc['never cured', 'rate']:.0%} vs {s17_c.loc['May', 'rate']:.0%} for those cured already in May", float(s17_kc["S_end"]), "s17_km_cure", 2)

**Observations:**
- Of 3,079 clients late in April, 43.6% never return to a clean status (≤ 0) by Sep; for a 3+ month delay in April it is 73%.
- Clients with bigger limits cure faster: among April-late clients 47% of ≤50k never cure vs 33% at 300-500k.
- Late-in-Apr clients who never cure default at 70.3% (1,341 clients); those cured already in May default at 33.3%.
- Relapse (going late again after curing) is what hurts: cured in May with no relapse 18.7% vs relapsed 49.7%; cured in Jul and relapsed 77.5%.
- Story use: 'a quick, lasting cure almost resets the client; a relapse is worse than never curing'.

### 17.3 Natural thresholds: where is the single best line for each variable?
For each important continuous / ordinal variable the best **binary split** `x ≤ cut` vs `x > cut` is located three ways: maximum χ², maximum Youden J (= KS distance) and maximum two-bin IV (each side must hold ≥ 2% of clients). A **bootstrap** (500 multinomial resamples on a ≤ 400-cell grid) gives a 95% interval for the χ² cut location: a tight interval = a real, stable threshold; a wide one = a smooth gradient with no natural line. These cuts are intended as **Tableau reference lines**.

In [ ]:
S17_CUT_VARS = ["PAY_1", "PAY_2", "PAY_3", "MAX_DELAY", "N_LATE", "LATE_STREAK_RECENT", "DELAY_TREND", "LIMIT_BAL", "AGE",
                "UTIL_1", "AVG_UTIL", "MAX_UTIL", "UTIL_TREND", "PAY_RATIO_1", "AVG_PAY_RATIO", "PAY_TO_BILL_6M",
                "PAY_AMT1", "PAY_AMT2", "TOTAL_PAY", "BILL_AMT1", "AVG_BILL", "BILL_TREND", "AVG_SPEND", "N_ZERO_PAY", "N_PAIDFULL", "N_REVOLVE"]

def s17_cells(x, max_cells=400):
    x = np.asarray(x, float); m = ~np.isnan(x); xv, yv = x[m], S17_Y[m]
    edges = np.unique(xv)
    if len(edges) > max_cells:
        edges = np.unique(np.quantile(xv, np.linspace(0, 1, max_cells + 1)[1:]))
    cell = np.searchsorted(edges, xv, side="left")                    # x <= edges[cell]
    return edges, np.bincount(cell, weights=1 - yv, minlength=len(edges)), np.bincount(cell, weights=yv, minlength=len(edges))

def s17_cut_stats(n0, n1, min_frac=0.02):
    # candidate j: left = cells 0..j. Works on (K,) or (B, K) arrays
    c0, c1 = np.cumsum(n0, -1)[..., :-1], np.cumsum(n1, -1)[..., :-1]
    T0, T1 = n0.sum(-1, keepdims=True), n1.sum(-1, keepdims=True); N = T0 + T1
    nl = c0 + c1; nr = N - nl
    with np.errstate(divide="ignore", invalid="ignore"):
        chi = N * (c1 * (T0 - c0) - c0 * (T1 - c1)) ** 2 / (nl * nr * T1 * T0)
        you = np.abs(c1 / T1 - c0 / T0)
        gl, bl = (c0 + .5) / (T0 + .5), (c1 + .5) / (T1 + .5); gr, br = (T0 - c0 + .5) / (T0 + .5), (T1 - c1 + .5) / (T1 + .5)
        iv = (gl - bl) * np.log(gl / bl) + (gr - br) * np.log(gr / br)
    bad = (nl < min_frac * N) | (nr < min_frac * N)
    return np.where(bad, -1, np.nan_to_num(chi)), np.where(bad, -1, np.nan_to_num(you)), np.where(bad, -1, np.nan_to_num(iv))

s17_rows, S17_CUT_PROFILE = [], {}
for _v in S17_CUT_VARS:
    _edges, _n0, _n1 = s17_cells(df[_v])
    if len(_edges) < 2:
        continue
    _chi, _you, _iv = s17_cut_stats(_n0, _n1)
    _j = int(_chi.argmax()); _cut = _edges[_j]
    _N = int(_n0.sum() + _n1.sum())
    _bs = S17_RNG.multinomial(_N, np.r_[_n0, _n1] / _N, size=500).astype(float)
    _bchi = s17_cut_stats(_bs[:, :len(_edges)], _bs[:, len(_edges):])[0]
    _bcut = _edges[_bchi.argmax(1)]
    _nl, _nr = _n0[:_j + 1].sum() + _n1[:_j + 1].sum(), _n0[_j + 1:].sum() + _n1[_j + 1:].sum()
    _rl, _rr = _n1[:_j + 1].sum() / _nl, _n1[_j + 1:].sum() / _nr
    S17_CUT_PROFILE[_v] = (_edges[:-1], _chi, np.percentile(_bcut, [2.5, 97.5]), _cut)
    s17_rows.append({"variable": _v, "n_valid": _N, "n_missing": S17_N - _N, "cut": _cut, "cut_ci_lo": np.percentile(_bcut, 2.5), "cut_ci_hi": np.percentile(_bcut, 97.5),
                     "boot_share_at_cut": float((_bcut == _cut).mean()), "n_below": int(_nl), "n_above": int(_nr), "rate_below": _rl, "rate_above": _rr,
                     "lift_below": _rl / BASE_RATE, "lift_above": _rr / BASE_RATE, "rate_ratio_above_vs_below": _rr / _rl, "gap_pp": 100 * (_rr - _rl),
                     "chi2": float(_chi[_j]), "cut_youden": _edges[int(_you.argmax())], "youden_J": float(_you.max()),
                     "cut_iv": _edges[int(_iv.argmax())], "iv_two_bins": float(_iv.max()),
                     "rule": f"{_v} <= {_cut:,.4g}  vs  > {_cut:,.4g}"})
S17_THR = pd.DataFrame(s17_rows).sort_values("chi2", ascending=False).reset_index(drop=True)
S17_THR["ci_width_pct_of_iqr"] = [100 * (r.cut_ci_hi - r.cut_ci_lo) / max(1e-12, float(df[r.variable].quantile(.75) - df[r.variable].quantile(.25))) for r in S17_THR.itertuples()]
S17_THR["three_criteria_agree"] = (S17_THR["cut"] == S17_THR["cut_youden"]) & (S17_THR["cut"] == S17_THR["cut_iv"])
display(S17_THR[["variable", "cut", "cut_ci_lo", "cut_ci_hi", "boot_share_at_cut", "n_below", "n_above", "rate_below", "rate_above", "lift_above", "gap_pp", "chi2", "cut_youden", "youden_J", "cut_iv", "iv_two_bins"]].round(4))
export(S17_THR, "s17_thresholds")

Left: default rate below vs above the best cut for every variable (sorted by χ²). Right: the χ² profile along candidate cuts for the six strongest variables, with the bootstrap interval of the optimum shaded: a sharp peak is a threshold, a plateau is a gradient.

In [ ]:
fig = plt.figure(figsize=(21, 9)); gs = fig.add_gridspec(2, 5)
ax = fig.add_subplot(gs[:, :2]); _t = S17_THR.iloc[::-1].reset_index(drop=True)
for _i, _r in _t.iterrows():
    ax.plot([_r["rate_below"], _r["rate_above"]], [_i, _i], color=C_NEU, lw=2, zorder=1)
ax.scatter(_t["rate_below"], _t.index, color=C_OK, s=60, label="rate at or below cut", zorder=2)
ax.scatter(_t["rate_above"], _t.index, color=C_DEF, s=60, label="rate above cut", zorder=2)
ax.set_yticks(_t.index); ax.set_yticklabels([f"{r.variable}  (cut {r.cut:,.4g})" for r in _t.itertuples()], fontsize=8)
ax.axvline(BASE_RATE, ls="--", color="k", lw=1); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.legend(fontsize=8, loc="lower right")
ax.set_title("Best single cut per variable: default rate either side")
for _k, _v in enumerate(S17_THR["variable"].head(6)):
    ax = fig.add_subplot(gs[_k // 3, 2 + _k % 3]); _e, _chi, _ci, _cut = S17_CUT_PROFILE[_v]
    _ok = _chi >= 0
    ax.plot(_e[_ok], _chi[_ok], color=C_DEF, lw=1.5, marker="o" if _ok.sum() < 20 else None, ms=4)
    ax.axvspan(_ci[0], _ci[1], color=C_ACC, alpha=.25); ax.axvline(_cut, color="k", ls="--", lw=1)
    if _ok.sum() > 20:
        ax.set_xlim(np.quantile(_e[_ok], .0), np.quantile(_e[_ok], .98))
    ax.set_title(f"{_v}: chi2 by cut (best {_cut:,.4g})", fontsize=10); ax.set_ylabel("chi2")
fig.suptitle("17.3 Natural thresholds", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s17_thresholds"); plt.show()
s17_b = S17_THR.iloc[0]; s17_u = S17_THR.set_index("variable")
add_fact("17.3", "thresholds", f"The sharpest single line in the data is {s17_b['variable']} > {s17_b['cut']:.0f}: {s17_b['rate_above']:.0%} default above vs {s17_b['rate_below']:.0%} at or below ({int(s17_b['n_above']):,} clients above; bootstrap puts the cut at the same value in {s17_b['boot_share_at_cut']:.0%} of resamples)", float(s17_b["rate_above"]), "s17_thresholds", 3)
add_fact("17.3", "thresholds", f"Credit limit splits best at NT${s17_u.loc['LIMIT_BAL', 'cut']:,.0f} ({s17_u.loc['LIMIT_BAL', 'rate_below']:.0%} at or below vs {s17_u.loc['LIMIT_BAL', 'rate_above']:.0%} above; 95% CI {s17_u.loc['LIMIT_BAL', 'cut_ci_lo']:,.0f}-{s17_u.loc['LIMIT_BAL', 'cut_ci_hi']:,.0f}); September payment splits at NT${s17_u.loc['PAY_AMT1', 'cut']:,.0f} ({s17_u.loc['PAY_AMT1', 'rate_below']:.0%} vs {s17_u.loc['PAY_AMT1', 'rate_above']:.0%})", float(s17_u.loc["LIMIT_BAL", "cut"]), "s17_thresholds", 2)

**Observations:**
- The sharpest single line is PAY_1 > 1 (Sep delay of 2+ months): 69.6% default above vs 16.6% below, chosen in 100% of bootstrap resamples (re-samples of the data).
- Recent late streak > 0 (50.3% vs 13.8%) and max delay > 1 (46.3% vs 12.8%) are next; all top cuts are delinquency variables.
- Money cuts are weaker but clear: Sep payment ratio ≤ 2% of the bill gives 50.9% vs 19.0%; credit limit ≤ NT$120k gives 28.9% vs 16.3% (bootstrap range 70k-140k).
- Age (cut at 25: 26.7% vs 21.5%) and Sep bill amount barely matter (numbers from the threshold table above; the chart is image-only).
- Story use: the 'one line per variable' slide; the PAY_1 cut uses status 2, so it avoids the Sep-only code 1.

#### 17.3b Two to four cuts per variable: tree-based optimal binning
A one-variable `DecisionTreeClassifier` (≤ 5 leaves, each ≥ 5% of clients) gives the best small set of bands per variable: the data-driven alternative to the hand-made bands of Section 0. For each band: n, default rate, lift; per variable: IV of the tree bands and whether the rates are monotone.

In [ ]:
s17_rows, s17_sum = [], []
for _v in S17_CUT_VARS:
    _x = df[_v].to_numpy(float); _m = ~np.isnan(_x)
    _tr = DecisionTreeClassifier(max_leaf_nodes=5, min_samples_leaf=0.05, random_state=0).fit(_x[_m].reshape(-1, 1), S17_Y[_m])
    _thr = np.sort(_tr.tree_.threshold[_tr.tree_.feature >= 0])
    if len(_thr) == 0:
        continue
    _bin = np.digitize(_x[_m], _thr, right=True); _ed = np.r_[-np.inf, _thr, np.inf]; _rates = []
    for _b in range(len(_thr) + 1):
        _g = _bin == _b; _k = S17_Y[_m][_g].sum(); _lo, _hi = wilson_ci(_k, _g.sum()); _rates.append(_k / _g.sum())
        _lab = f"<= {_ed[_b + 1]:,.4g}" if _b == 0 else (f"> {_ed[_b]:,.4g}" if _b == len(_thr) else f"({_ed[_b]:,.4g}, {_ed[_b + 1]:,.4g}]")
        s17_rows.append({"variable": _v, "bin": _b, "band": _lab, "lower": _ed[_b], "upper": _ed[_b + 1], "n": int(_g.sum()), "share": _g.mean(),
                         "rate": _k / _g.sum(), "ci_lo": float(_lo), "ci_hi": float(_hi), "lift": _k / _g.sum() / BASE_RATE})
    _d = np.diff(_rates)
    s17_sum.append({"variable": _v, "n_cuts": len(_thr), "cuts": ", ".join(f"{t:,.4g}" for t in _thr), "iv_tree_bands": s17_iv(_bin.astype(float)) if _m.all() else woe_iv(pd.DataFrame({"x": _bin, TARGET: S17_Y[_m]}), "x")[1],
                    "min_rate": min(_rates), "max_rate": max(_rates), "rate_spread_pp": 100 * (max(_rates) - min(_rates)),
                    "monotone": bool((_d >= 0).all() or (_d <= 0).all())})
S17_TREEBINS = pd.DataFrame(s17_rows); S17_TREESUM = pd.DataFrame(s17_sum).sort_values("iv_tree_bands", ascending=False).reset_index(drop=True)
display(S17_TREESUM.round(4))
export(S17_TREEBINS, "s17_tree_bins"); export(S17_TREESUM, "s17_tree_bins_summary")
s17_top = S17_TREESUM["variable"].head(12).tolist()
fig, axes = plt.subplots(3, 4, figsize=(22, 11)); axes = axes.ravel()
for ax, _v in zip(axes, s17_top):
    _t = S17_TREEBINS[S17_TREEBINS["variable"] == _v]
    ax.bar(range(len(_t)), _t["rate"], yerr=[_t["rate"] - _t["ci_lo"], _t["ci_hi"] - _t["rate"]], color=plt.cm.Reds(0.25 + 0.75 * _t["rate"] / max(.01, _t["rate"].max())), capsize=3)
    for _x, (_r, _n) in enumerate(zip(_t["rate"], _t["share"])):
        ax.text(_x, _r + .02, f"{_r:.0%}\n({_n:.0%})", ha="center", fontsize=7)
    ax.set_xticks(range(len(_t))); ax.set_xticklabels(_t["band"], fontsize=7, rotation=20, ha="right"); ax.axhline(BASE_RATE, ls="--", color="k", lw=1)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_ylim(0, min(1, _t["rate"].max() + .15)); ax.set_title(f"{_v} (IV {S17_TREESUM.set_index('variable').loc[_v, 'iv_tree_bands']:.2f})", fontsize=10)
fig.suptitle("17.3b Tree-optimal bands: default rate (share of clients in brackets)", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s17_tree_bins"); plt.show()
s17_nm = S17_TREESUM[~S17_TREESUM["monotone"]]["variable"].tolist()
print("non-monotone variables in tree bands:", s17_nm)
add_fact("17.3", "thresholds", f"With at most five bands, {S17_TREESUM.loc[0, 'variable']} alone spans {S17_TREESUM.loc[0, 'min_rate']:.0%} to {S17_TREESUM.loc[0, 'max_rate']:.0%} default (cuts at {S17_TREESUM.loc[0, 'cuts']}); {len(s17_nm)} of {len(S17_TREESUM)} variables are non-monotone in their optimal bands", float(S17_TREESUM.loc[0, "rate_spread_pp"]), "s17_tree_bins", 2)

**Observations:**
- With up to five data-driven bands, PAY_1 spans 12.8% to 69.6% (IV 0.87; IV above 0.3 = strong) and number of late months spans 11.7% to 64.1% (IV 0.84).
- The best money variable is Sep payment ratio (IV 0.30, 10.4% to 51.0%); age (IV 0.02) and Sep bill (0.015) are near useless.
- 18 of 26 variables are not monotone in their best bands (risk goes up, then down), so simple 'more = riskier' rules often fail.
- Story use: data-driven bands for Tableau filters (`s17_tree_bins.csv`).

### 17.4 Tipping points: is the risk curve a slope, a kink or a cliff?
Fine-grained default-rate curves (2.5-point utilisation / payment-ratio bins, NT$10k limit steps, single years of age, 0.1 log-units of payment). Each curve is fitted **in logit space** with binomial likelihood on the bins:

| Model | Parameters | Reads as |
|---|---|---|
| linear | 2 | one constant slope: no tipping point |
| 1 knot / 2 knots (continuous piecewise-linear, knots by grid search) | 4 / 6 | **kink**: the slope changes at the knot |
| jump (level shift + slope change at a grid-searched threshold) | 5 | **cliff**: a discontinuity |
| cubic / quintic polynomial | 4 / 6 | smooth curve with no special point |

Models are compared by **AIC** (knot locations count as parameters). A cliff is flagged when the jump model beats the continuous 1-knot model by > 2 AIC and the fitted jump is ≥ 2 pp. Knot / jump locations get a parametric-bootstrap 95% interval (80 resamples of the bin counts).

In [ ]:
def s17_irls(X, k, n, iters=30):
    beta = np.zeros(X.shape[1]); beta[0] = np.log((k.sum() + .5) / (n.sum() - k.sum() + .5))
    for _ in range(iters):
        eta = np.clip(X @ beta, -30, 30); p = s17_expit(eta); w = np.maximum(n * p * (1 - p), 1e-9)
        XtW = X.T * w
        new = np.linalg.solve(XtW @ X + 1e-8 * np.eye(X.shape[1]), XtW @ (eta + (k - n * p) / w))
        done = np.max(np.abs(new - beta)) < 1e-8; beta = new
        if done:
            break
    p = np.clip(s17_expit(np.clip(X @ beta, -30, 30)), 1e-12, 1 - 1e-12)
    return beta, float((k * np.log(p) + (n - k) * np.log(1 - p)).sum())

def s17_basis(x, kind, knots=()):
    one = np.ones(len(x))
    if kind == "linear": return np.c_[one, x]
    if kind == "seg":    return np.column_stack([one, x] + [np.clip(x - kn, 0, None) for kn in knots])
    if kind == "jump":   return np.c_[one, x, (x > knots[0]).astype(float), np.clip(x - knots[0], 0, None)]
    z = (x - S17_Z[0]) / S17_Z[1]
    return np.column_stack([one] + [z ** d for d in range(1, (3 if kind == "cubic" else 5) + 1)])

def s17_grid(x, k, n, do2=True):
    K = len(x); best = {}
    cand = x[2:K - 2]
    f1 = [(s17_irls(s17_basis(x, "seg", (kn,)), k, n)[1], kn) for kn in cand]
    best["1 knot"] = max(f1)
    thr = (x[1:K - 2] + x[2:K - 1]) / 2
    best["jump"] = max((s17_irls(s17_basis(x, "jump", (th,)), k, n)[1], th) for th in thr)
    if do2:
        best["2 knots"] = max((s17_irls(s17_basis(x, "seg", (cand[a], cand[b])), k, n)[1], (cand[a], cand[b]))
                              for a in range(len(cand)) for b in range(a + 3, len(cand)))
    return best

def s17_segfit(tab, n_boot=80):
    global S17_Z
    x, k, n = tab["x"].to_numpy(float), tab["k"].to_numpy(float), tab["n"].to_numpy(float)
    S17_Z = (x.mean(), x.std())
    g = s17_grid(x, k, n); res = {}
    spec = {"linear": ("linear", (), 2), "1 knot": ("seg", (g["1 knot"][1],), 4), "2 knots": ("seg", tuple(g["2 knots"][1]), 6),
            "jump": ("jump", (g["jump"][1],), 5), "cubic": ("cubic", (), 4), "quintic": ("quintic", (), 6)}
    for nm, (kind, kn, npar) in spec.items():
        X = s17_basis(x, kind, kn); beta, ll = s17_irls(X, k, n)
        res[nm] = {"kind": kind, "knots": kn, "aic": -2 * ll + 2 * npar, "beta": beta, "fit": s17_expit(X @ beta)}
    b = res["jump"]["beta"]; th = res["jump"]["knots"][0]
    res["jump"]["jump_pp"] = 100 * float(s17_expit(b[0] + b[1] * th + b[2]) - s17_expit(b[0] + b[1] * th))
    b = res["1 knot"]["beta"]; res["1 knot"]["slopes"] = (float(b[1]), float(b[1] + b[2]))
    bk, bj = [], []
    for _ in range(n_boot):
        gb = s17_grid(x, S17_RNG.binomial(n.astype(int), k / n).astype(float), n, do2=False)
        bk.append(gb["1 knot"][1]); bj.append(gb["jump"][1])
    res["1 knot"]["ci"] = tuple(np.percentile(bk, [2.5, 97.5])); res["jump"]["ci"] = tuple(np.percentile(bj, [2.5, 97.5]))
    return res

def s17_bin_curve(x, mask, edges, min_n=40):
    xv = np.clip(np.asarray(x, float)[mask], edges[0], edges[-1] - 1e-9); yv = S17_Y[mask]
    b = np.digitize(xv, edges) - 1
    t = pd.DataFrame({"b": b, "x": xv, "y": yv}).groupby("b").agg(x=("x", "mean"), n=("y", "size"), k=("y", "sum")).reset_index()
    t["lo_edge"] = edges[t["b"]]; t["hi_edge"] = edges[t["b"] + 1]
    t = t[t["n"] >= min_n].reset_index(drop=True); t["rate"] = t["k"] / t["n"]
    return t

s17_fin = np.isfinite
S17_CURVES = [
    {"name": "UTIL_1", "label": "September utilisation (bill / limit), clients with a positive Sep bill", "x": df["UTIL_1"].to_numpy(float), "mask": (df["BILL_AMT1"] > 0).to_numpy(), "edges": np.arange(0, 1.3251, 0.025), "unit": "ratio", "back": lambda v: v},
    {"name": "AVG_UTIL", "label": "Average utilisation over 6 months (positive only)", "x": df["AVG_UTIL"].to_numpy(float), "mask": (df["AVG_UTIL"] > 0).to_numpy(), "edges": np.arange(0, 1.3251, 0.025), "unit": "ratio", "back": lambda v: v},
    {"name": "PAY_RATIO_1", "label": "September payment / August bill (zero payment = own bin)", "x": df["PAY_RATIO_1"].to_numpy(float), "mask": df["PAY_RATIO_1"].notna().to_numpy(), "edges": np.r_[0, 1e-9, np.arange(0.025, 1.5251, 0.025)], "unit": "ratio", "back": lambda v: v},
    {"name": "PAY_TO_BILL_6M", "label": "Six-month payments / bills (zero = own bin)", "x": df["PAY_TO_BILL_6M"].to_numpy(float), "mask": df["PAY_TO_BILL_6M"].notna().to_numpy(), "edges": np.r_[0, 1e-9, np.arange(0.025, 1.5251, 0.025)], "unit": "ratio", "back": lambda v: v},
    {"name": "LIMIT_BAL", "label": "Credit limit (NT$ thousand, 10k steps)", "x": S17_LIM / 1000, "mask": np.ones(S17_N, bool), "edges": np.r_[np.arange(10, 510, 10), 1001.0], "unit": "NT$k", "back": lambda v: v},
    {"name": "AGE", "label": "Age (single years, 61+ pooled)", "x": df["AGE"].to_numpy(float), "mask": np.ones(S17_N, bool), "edges": np.arange(21, 63, 1.0), "unit": "years", "back": lambda v: v},
    {"name": "PAY_AMT1", "label": "September payment, log10(1 + NT$) (zero = own bin; Aug bill > 0)", "x": np.log10(1 + df["PAY_AMT1"].to_numpy(float)), "mask": (df["BILL_AMT2"] > 0).to_numpy(), "edges": np.r_[0, 0.01, np.arange(2.0, 5.61, 0.1)], "unit": "NT$", "back": lambda v: 10 ** v - 1},
    {"name": "BILL_AMT1", "label": "September bill, log10(NT$) (positive bills)", "x": np.log10(np.clip(df["BILL_AMT1"].to_numpy(float), 1, None)), "mask": (df["BILL_AMT1"] > 0).to_numpy(), "edges": np.r_[0, np.arange(2.0, 5.81, 0.1)], "unit": "NT$", "back": lambda v: 10 ** v},
]
s17_rows, s17_curve_rows, S17_SEG = [], [], {}
for _c in S17_CURVES:
    _tab = s17_bin_curve(_c["x"], _c["mask"], _c["edges"]); _res = s17_segfit(_tab); S17_SEG[_c["name"]] = (_tab, _res)
    _aic = {m: r["aic"] for m, r in _res.items()}; _best = min(_aic, key=_aic.get); _bk = _c["back"]
    _cliff = (_aic["jump"] < _aic["1 knot"] - 2) and (abs(_res["jump"]["jump_pp"]) >= 2)
    s17_rows.append({"variable": _c["name"], "curve": _c["label"], "n_bins": len(_tab), "n_clients": int(_tab["n"].sum()), "unit": _c["unit"],
                     **{f"aic_{m.replace(' ', '_')}": a for m, a in _aic.items()}, "best_model": _best,
                     "best_smooth_minus_best_segmented_aic": min(_aic["cubic"], _aic["quintic"]) - min(_aic["1 knot"], _aic["2 knots"], _aic["jump"]),
                     "knot_1": _bk(_res["1 knot"]["knots"][0]), "knot_1_ci_lo": _bk(_res["1 knot"]["ci"][0]), "knot_1_ci_hi": _bk(_res["1 knot"]["ci"][1]),
                     "logit_slope_before": _res["1 knot"]["slopes"][0], "logit_slope_after": _res["1 knot"]["slopes"][1],
                     "knots_2": " | ".join(f"{_bk(v):,.4g}" for v in _res["2 knots"]["knots"]),
                     "jump_at": _bk(_res["jump"]["knots"][0]), "jump_ci_lo": _bk(_res["jump"]["ci"][0]), "jump_ci_hi": _bk(_res["jump"]["ci"][1]),
                     "jump_pp": _res["jump"]["jump_pp"], "cliff_flag": bool(_cliff)})
    for _m, _r in _res.items():
        for _x, _n, _k, _f in zip(_tab["x"], _tab["n"], _tab["k"], _r["fit"]):
            s17_curve_rows.append({"variable": _c["name"], "x": _x, "x_original_units": _bk(_x), "n": int(_n), "observed_rate": _k / _n, "model": _m, "fitted_rate": float(_f)})
S17_TIP = pd.DataFrame(s17_rows); S17_CURVE_DF = pd.DataFrame(s17_curve_rows)
display(S17_TIP.drop(columns=["curve"]).round(3))
export(S17_TIP, "s17_tipping_points"); export(S17_CURVE_DF, "s17_rate_curves")

Each panel: observed bin rates (bubble area ∝ clients), the best **segmented** fit (red), the best **smooth** fit (dashed blue), knots (solid vertical, green band = bootstrap interval of the 1-knot location) and the jump location (dotted).

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 10.5)); axes = axes.ravel()
for ax, _c in zip(axes, S17_CURVES):
    _tab, _res = S17_SEG[_c["name"]]; _row = S17_TIP[S17_TIP["variable"] == _c["name"]].iloc[0]
    _sg = min(("1 knot", "2 knots", "jump"), key=lambda m: _res[m]["aic"]); _sm = min(("cubic", "quintic"), key=lambda m: _res[m]["aic"])
    ax.scatter(_tab["x"], _tab["rate"], s=12 + 260 * _tab["n"] / _tab["n"].max(), color=C_NEU, alpha=.6, edgecolor="w")
    ax.plot(_tab["x"], _res[_sg]["fit"], color=C_DEF, lw=2.2, label=f"{_sg} (AIC {_res[_sg]['aic']:.0f})")
    ax.plot(_tab["x"], _res[_sm]["fit"], color=C_OK, lw=1.5, ls="--", label=f"{_sm} (AIC {_res[_sm]['aic']:.0f})")
    ax.plot(_tab["x"], _res["linear"]["fit"], color="k", lw=.8, ls=":", label=f"linear (AIC {_res['linear']['aic']:.0f})")
    ax.axvspan(_res["1 knot"]["ci"][0], _res["1 knot"]["ci"][1], color=C_ACC, alpha=.18)
    for _kn in _res[_sg]["knots"]:
        ax.axvline(_kn, color=C_DEF, lw=1.2, ls=":" if _sg == "jump" else "-")
    ax.axhline(BASE_RATE, ls="--", color="k", lw=.6); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    _kn_txt = f"1-knot at {_row['knot_1']:,.3g} {_c['unit']}" + (f" | CLIFF {_row['jump_pp']:+.0f}pp at {_row['jump_at']:,.3g}" if _row["cliff_flag"] else "")
    ax.set_title(f"{_c['name']}: {_kn_txt}", fontsize=10); ax.set_xlabel(_c["label"], fontsize=8); ax.legend(fontsize=7)
fig.suptitle("17.4 Tipping points: segmented vs smooth fits of fine-grained default-rate curves", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s17_tipping_points"); plt.show()
print("cliffs flagged:", S17_TIP.loc[S17_TIP["cliff_flag"], ["variable", "jump_at", "jump_pp"]].round(3).to_dict("records"))
print("curves where a smooth fit is at least as good as any segmented fit (no real tipping point):", S17_TIP.loc[S17_TIP["best_smooth_minus_best_segmented_aic"] <= 2, "variable"].tolist())
s17_tp = S17_TIP.set_index("variable")
add_fact("17.4", "tipping points", f"September utilisation has its kink at {s17_tp.loc['UTIL_1', 'knot_1']:.0%} of the limit (95% CI {s17_tp.loc['UTIL_1', 'knot_1_ci_lo']:.0%}-{s17_tp.loc['UTIL_1', 'knot_1_ci_hi']:.0%}): the logit slope changes from {s17_tp.loc['UTIL_1', 'logit_slope_before']:+.2f} to {s17_tp.loc['UTIL_1', 'logit_slope_after']:+.2f} per 100 points of utilisation", float(s17_tp.loc["UTIL_1", "knot_1"]), "s17_tipping_points", 2)
add_fact("17.4", "tipping points", f"Credit limit: risk falls steeply up to about NT${1000 * s17_tp.loc['LIMIT_BAL', 'knot_1']:,.0f} and then flattens (logit slope {s17_tp.loc['LIMIT_BAL', 'logit_slope_before']:+.4f} vs {s17_tp.loc['LIMIT_BAL', 'logit_slope_after']:+.4f} per NT$1k); {int(S17_TIP['cliff_flag'].sum())} of {len(S17_TIP)} curves show a genuine discontinuity", float(1000 * s17_tp.loc["LIMIT_BAL", "knot_1"]), "s17_tipping_points", 2)

**Observations:**
- Sep utilisation has a kink at 71% of the limit: risk rises with utilisation up to there and is flat after (logit slope +0.92 then −0.08), but the bootstrap range is wide (6%-79%) and a smooth curve fits about as well.
- Sep payment ratio has a sharp kink at 4.1% of the bill: risk falls steeply up to that point and is flat after.
- Credit limit risk falls steeply up to about NT$150k and then flattens; age has its minimum near 29.
- Four curves show a real jump (a cliff of 5-8 points): utilisation at 15%, payment ratio at 5%, Sep payment near NT$1,000 and Sep bill near NT$12.6k.
- Story use: 'pay at least the minimum (~5%)' is the clearest money tipping point.

#### 17.4b Named cliffs: local jump tests at a-priori thresholds and between adjacent ordinal levels
(1) For ordinal variables: default-rate jump between **adjacent levels** with a two-proportion z-test: the largest step is that variable's cliff. (2) For candidate business thresholds (zero payment, full payment, 90% / 100% utilisation, limit bands): compare the narrow window just below with the window just above.

In [ ]:
def s17_two_prop(k1, n1, k2, n2):
    p1, p2 = k1 / n1, k2 / n2; se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    z = (p2 - p1) / se if se > 0 else np.nan
    return p1, p2, 100 * (p2 - p1), 100 * (p2 - p1 - 1.96 * se), 100 * (p2 - p1 + 1.96 * se), z

s17_rows = []
for _v in ["PAY_1", "PAY_2", "PAY_3", "MAX_DELAY", "N_LATE", "LATE_STREAK_RECENT", "N_ZERO_PAY"]:
    _t = rate_table(df, _v, min_n=100).sort_values(_v).reset_index(drop=True)
    for _a, _b in zip(_t.index[:-1], _t.index[1:]):
        _p1, _p2, _d, _lo, _hi, _z = s17_two_prop(_t.loc[_a, "defaults"], _t.loc[_a, "n"], _t.loc[_b, "defaults"], _t.loc[_b, "n"])
        s17_rows.append({"variable": _v, "from_level": _t.loc[_a, _v], "to_level": _t.loc[_b, _v], "n_from": int(_t.loc[_a, "n"]), "n_to": int(_t.loc[_b, "n"]),
                         "rate_from": _p1, "rate_to": _p2, "jump_pp": _d, "ci_lo_pp": _lo, "ci_hi_pp": _hi, "z": _z,
                         "share_of_total_range": _d / (100 * (_t["rate"].max() - _t["rate"].min()))})
S17_STEP = pd.DataFrame(s17_rows)
S17_STEP["is_biggest_step"] = S17_STEP.groupby("variable")["jump_pp"].transform("max") == S17_STEP["jump_pp"]
display(S17_STEP.round(4))

def s17_window(label, x, lo1, hi1, lo2, hi2, mask=None):
    x = np.asarray(x, float); mk = np.ones(S17_N, bool) if mask is None else mask
    a = mk & (x >= lo1) & (x < hi1); b = mk & (x >= lo2) & (x < hi2)
    p1, p2, d, lo, hi, z = s17_two_prop(S17_Y[a].sum(), a.sum(), S17_Y[b].sum(), b.sum())
    return {"threshold": label, "below_window": f"[{lo1:g}, {hi1:g})", "above_window": f"[{lo2:g}, {hi2:g})", "n_below": int(a.sum()), "n_above": int(b.sum()),
            "rate_below": p1, "rate_above": p2, "jump_pp": d, "ci_lo_pp": lo, "ci_hi_pp": hi, "z": z}
s17_pb = (df["BILL_AMT1"] > 0).to_numpy(); s17_ab = (df["BILL_AMT2"] > 0).to_numpy()
S17_LOCAL = pd.DataFrame([
    s17_window("Sep utilisation crosses 100% (over limit)", df["UTIL_1"], 0.90, 1.00, 1.00, 1.10, s17_pb),
    s17_window("Sep utilisation crosses 90%", df["UTIL_1"], 0.80, 0.90, 0.90, 1.00, s17_pb),
    s17_window("Sep utilisation crosses 50%", df["UTIL_1"], 0.40, 0.50, 0.50, 0.60, s17_pb),
    s17_window("Sep payment: zero vs tiny (0 vs 1-999 NT$), Aug bill > 0", df["PAY_AMT1"], 0, 1, 1, 1000, s17_ab),
    s17_window("Sep payment: NT$1-999 vs 1,000-1,999, Aug bill > 0", df["PAY_AMT1"], 1, 1000, 1000, 2000, s17_ab),
    s17_window("Sep payment ratio: zero vs 0-5% of Aug bill", df["PAY_RATIO_1"], 0, 1e-9, 1e-9, 0.05),
    s17_window("Sep payment ratio: 3-5% vs 5-7% (minimum-payment zone)", df["PAY_RATIO_1"], 0.03, 0.05, 0.05, 0.07),
    s17_window("Sep payment ratio: 90-100% vs 100-110% (paid in full)", df["PAY_RATIO_1"], 0.90, 0.999, 0.999, 1.10),
    s17_window("Credit limit: 50k vs 60-80k", df["LIMIT_BAL"], 50_000, 50_001, 60_000, 80_001),
    s17_window("Credit limit: 20-30k vs 50k", df["LIMIT_BAL"], 20_000, 30_001, 50_000, 50_001),
    s17_window("Age: 23-25 vs 26-28", df["AGE"], 23, 26, 26, 29),
    s17_window("Sep status: 0 (revolving) vs 1 (late 1m)", df["PAY_1"], 0, 1, 1, 2),
    s17_window("Sep status: 1 (late 1m) vs 2 (late 2m)", df["PAY_1"], 1, 2, 2, 3),
    s17_window("Aug status: 0 (revolving) vs 2 (late 2m)", df["PAY_2"], 0, 1, 2, 3),
]).sort_values("jump_pp", key=np.abs, ascending=False).reset_index(drop=True)
display(S17_LOCAL.round(4))
export(S17_STEP, "s17_ordinal_steps"); export(S17_LOCAL, "s17_local_jumps")
fig, axes = plt.subplots(1, 2, figsize=(21, 6))
_t = S17_LOCAL.iloc[::-1]
axes[0].barh(_t["threshold"], _t["jump_pp"], xerr=[_t["jump_pp"] - _t["ci_lo_pp"], _t["ci_hi_pp"] - _t["jump_pp"]], color=np.where(_t["jump_pp"] > 0, C_DEF, C_OK), capsize=3, alpha=.85)
axes[0].axvline(0, color="k", lw=.8); axes[0].set_xlabel("default-rate jump just above vs just below (pp, 95% CI)"); axes[0].set_title("Local jumps at named thresholds"); axes[0].tick_params(axis="y", labelsize=8)
for _v, _col in (("PAY_1", C_DEF), ("PAY_2", "#fd8d3c"), ("MAX_DELAY", "#756bb1"), ("N_LATE", C_OK)):
    _t = rate_table(df, _v, min_n=100).sort_values(_v)
    axes[1].step(_t[_v], _t["rate"], where="mid", marker="o", color=_col, label=_v)
axes[1].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[1].axhline(.5, ls=":", color=C_NEU); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_title("Ordinal ladders: where is the step? (levels with n >= 100)"); axes[1].set_xlabel("level"); axes[1].legend()
plt.tight_layout(); savefig("s17_named_cliffs"); plt.show()
s17_p1 = S17_STEP[(S17_STEP["variable"] == "PAY_1") & S17_STEP["is_biggest_step"]].iloc[0]; s17_lj = S17_LOCAL.set_index("threshold")
add_fact("17.4", "cliffs", f"The biggest cliff in the file: September status {int(s17_p1['from_level'])} -> {int(s17_p1['to_level'])} lifts default from {s17_p1['rate_from']:.0%} to {s17_p1['rate_to']:.0%} ({s17_p1['jump_pp']:+.0f} pp, {s17_p1['share_of_total_range']:.0%} of the whole PAY_1 range in one step)", float(s17_p1["jump_pp"]), "s17_named_cliffs", 3)
add_fact("17.4", "cliffs", f"Paying nothing vs paying a token amount in September (Aug bill > 0): {s17_lj.loc['Sep payment: zero vs tiny (0 vs 1-999 NT$), Aug bill > 0', 'rate_below']:.0%} vs {s17_lj.loc['Sep payment: zero vs tiny (0 vs 1-999 NT$), Aug bill > 0', 'rate_above']:.0%}; crossing the credit limit (90-100% vs 100-110% utilisation) moves default by only {s17_lj.loc['Sep utilisation crosses 100% (over limit)', 'jump_pp']:+.1f} pp", float(s17_lj.loc["Sep utilisation crosses 100% (over limit)", "jump_pp"]), "s17_named_cliffs", 2)

**Observations:**
- The biggest cliff in the file is Sep status 1 → 2: default jumps from 34.0% to 69.1% (+35 points, 56% of the whole PAY_1 range in one step).
- For Aug and Jul (where code 1 is missing) the step is 0 → 2: +39.7 and +34.1 points.
- Paying nothing vs a token amount in Sep (Aug bill > 0) moves default from 52.5% to 22.1% (−30 points), while crossing the credit limit adds only +5.7 points and crossing 50% or 90% utilisation adds nothing.
- The first late month is the largest step for N_LATE (11.7% → 29.8%) and for zero-payment months (13.9% → 29.0%).
- Story use: two headline cliffs: 'two months late' and 'paying zero'.

### 17.5 Dose–response of delinquency: how much, how recent, and where is the point of no return?
Three "dose" surfaces: (a) **severity × recency** — default rate by status code in each month; (b) **dose × recency** — number of late months × months since the last late month; (c) **current × history** — September status × number of late months in Apr–Aug. Cells at or above 50% are outlined.

In [ ]:
s17_rows = []
for _m in range(6):
    _sv = np.clip(S17_S[:, _m], -2, 4)
    for _lev in range(-2, 5):
        _g = _sv == _lev
        if _g.sum() >= 30:
            s17_rows.append({"month": MONTH_ORDER[_m], "months_before_outcome": 6 - _m, "status": ("4+" if _lev == 4 else str(_lev)), "status_num": _lev, "n": int(_g.sum()), "rate": S17_Y[_g].mean()})
S17_SEVREC = pd.DataFrame(s17_rows)
s17_msl = np.where((S17_S >= 1).any(1), (S17_S[:, ::-1] >= 1).argmax(1), -1)          # months since last late (0 = Sep), -1 = never
s17_nl = (S17_S >= 1).sum(1)
S17_DOSEREC = pd.DataFrame({"n_late": s17_nl, "months_since_last_late": s17_msl, "y": S17_Y})[s17_nl > 0].groupby(["n_late", "months_since_last_late"])["y"].agg(n="size", rate="mean").reset_index()
s17_prior = (S17_S[:, :5] >= 1).sum(1); s17_sep = np.clip(S17_S[:, 5], -2, 4)
S17_CURHIST = pd.DataFrame({"sep_status": s17_sep, "n_late_apr_aug": s17_prior, "y": S17_Y}).groupby(["sep_status", "n_late_apr_aug"])["y"].agg(n="size", rate="mean").reset_index()

def s17_heat(ax, tab, idx, col, title, xl, yl, min_n=30, order=None):
    t = tab[tab["n"] >= min_n]; hm = t.pivot(index=idx, columns=col, values="rate"); nn = t.pivot(index=idx, columns=col, values="n")
    if order is not None:
        hm, nn = hm.reindex(order), nn.reindex(order)
    lab = hm.copy().astype(object)
    for _i in hm.index:
        for _j in hm.columns:
            lab.loc[_i, _j] = "" if pd.isna(hm.loc[_i, _j]) else f"{hm.loc[_i, _j]:.0%}\n{int(nn.loc[_i, _j]):,}"
    sns.heatmap(hm.astype(float), annot=lab.to_numpy(), fmt="", cmap=CMAP_RATE, vmin=0, vmax=.85, ax=ax, cbar=False, linewidths=.5, annot_kws={"fontsize": 7})
    for _a, _i in enumerate(hm.index):
        for _b, _j in enumerate(hm.columns):
            if pd.notna(hm.loc[_i, _j]) and hm.loc[_i, _j] >= .5:
                ax.add_patch(plt.Rectangle((_b, _a), 1, 1, fill=False, edgecolor="lime", lw=2))
    ax.set_title(title, fontsize=10); ax.set_xlabel(xl); ax.set_ylabel(yl)

fig, axes = plt.subplots(1, 3, figsize=(23, 6.2))
S17_SEVREC["month"] = pd.Categorical(S17_SEVREC["month"], MONTH_ORDER, ordered=True)
s17_heat(axes[0], S17_SEVREC, "status", "month", "(a) Severity x recency: status code in each month", "month (Sep = most recent)", "status code", order=["-2", "-1", "0", "1", "2", "3", "4+"])
s17_heat(axes[1], S17_DOSEREC, "n_late", "months_since_last_late", "(b) Dose x recency: late months x months since last late", "months since last late month (0 = late in Sep)", "number of late months (of 6)")
s17_heat(axes[2], S17_CURHIST, "sep_status", "n_late_apr_aug", "(c) Current x history: Sep status x late months in Apr-Aug", "late months in Apr-Aug", "Sep status (4 = 4+)")
fig.suptitle("17.5 Dose-response of delinquency (cell = default rate / n; green outline = 50%+)", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s17_dose_response_heatmaps"); plt.show()
export(S17_SEVREC, "s17_severity_recency"); export(S17_DOSEREC, "s17_dose_recency"); export(S17_CURHIST, "s17_current_history")
s17_sr = S17_SEVREC[S17_SEVREC["status"] == "2"].set_index("month")["rate"]
print("default rate of a 2-month delay by the month it occurred:", {m: round(float(v), 3) for m, v in s17_sr.items()})
s17_any = pd.Series({_mn: S17_Y[S17_S[:, _k] >= 1].mean() for _k, _mn in enumerate(MONTH_ORDER)})   # like-for-like: any delay (status >= 1)
print("default rate of any delay (status >= 1) by the month it was recorded:", {m: round(float(v), 3) for m, v in s17_any.items()})
add_fact("17.5", "dose-response", f"Like for like (any delay, status >= 1), a delay recorded in September goes with {s17_any['Sep']:.1%} default vs {s17_any['Apr']:.1%} for one recorded in April: {'no recency premium' if s17_any['Sep'] <= s17_any['Apr'] + 0.02 else 'a recency premium'} on this comparison. "
         f"The September code-2 group ({s17_sr['Sep']:.0%} vs April {s17_sr['Apr']:.0%}) is a stricter group because only September has a code 1, so it is not recency evidence; the clean-months scar curve (17.10c) is", float(s17_any["Sep"]), "s17_dose_response_heatmaps", 2)

**Observations:**
- Compare like with like. Any delay (status >= 1) recorded in Sep goes with about 50.3% default vs about 52.3% for one recorded in Apr, so this split shows no recency premium.
- A code-2 delay looks worse in Sep (69.1%) than in Apr-Aug (about 51-56%), but that is an artefact of the status codes: code 1 exists only in Sep, so in Sep it siphons off the milder first delays and the Sep code-2 group is stricter. In September, code 1 mixes clients who were paid-up or inactive in August with clients who were 2+ months late in August; it is not a uniform first delay.
- The real recency evidence is the clean-months scar curve in 17.10c (default falls with each consecutive clean month since the last delay) and the months-since-last-late curve in Section 5.11.
- The heatmaps are image-only; the two printed lines (code 2 by month, any delay by month) are the numbers for this cell.
- Story use: qualifies the recency message from 17.1; do not quote the 69% vs 52% code-2 pair as recency (`s17_severity_recency.csv` for the heatmap).

#### 17.5b Marginal effect of each additional late month
Default rate by the number of late months, the increment contributed by each extra month (pp and odds ratio vs the previous count), under both late definitions — and split by whether the client is late **in September** (does history still add once the current state is known?).

In [ ]:
s17_rows = []
for _thr in (1, 2):
    _cnt = (S17_S >= _thr).sum(1); _sepl = S17_S[:, 5] >= _thr
    for _lab, _mk in (("all clients", np.ones(S17_N, bool)), ("late in Sep", _sepl), ("not late in Sep", ~_sepl)):
        _prev = None
        for _c in range(7):
            _g = _mk & (_cnt == _c)
            if _g.sum() < 50:
                _prev = None; continue
            _r = S17_Y[_g].mean(); _lo, _hi = wilson_ci(S17_Y[_g].sum(), _g.sum())
            s17_rows.append({"late_definition": f"status >= {_thr}", "population": _lab, "n_late_months": _c, "n": int(_g.sum()), "rate": _r, "ci_lo": float(_lo), "ci_hi": float(_hi),
                             "marginal_pp": np.nan if _prev is None else 100 * (_r - _prev), "odds_ratio_vs_previous": np.nan if _prev is None else (_r / (1 - _r)) / (_prev / (1 - _prev))})
            _prev = _r
S17_MARG = pd.DataFrame(s17_rows)
fig, axes = plt.subplots(1, 2, figsize=(17, 5), sharey=True)
for ax, _thr in zip(axes, (1, 2)):
    for _lab, _col in (("all clients", "k"), ("late in Sep", C_DEF), ("not late in Sep", C_OK)):
        _t = S17_MARG[(S17_MARG["late_definition"] == f"status >= {_thr}") & (S17_MARG["population"] == _lab)]
        ax.errorbar(_t["n_late_months"], _t["rate"], yerr=[_t["rate"] - _t["ci_lo"], _t["ci_hi"] - _t["rate"]], marker="o", capsize=3, color=_col, label=_lab)
    ax.axhline(.5, ls=":", color=C_NEU); ax.axhline(BASE_RATE, ls="--", color="k", lw=1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_xlabel(f"number of months with status >= {_thr}"); ax.set_title(f"Default rate by late-month count (late = status >= {_thr})"); ax.legend(fontsize=8)
plt.tight_layout(); savefig("s17_marginal_late_month"); plt.show()
display(S17_MARG.round(4)); export(S17_MARG, "s17_marginal_late_month")
s17_m = S17_MARG[(S17_MARG["late_definition"] == "status >= 1") & (S17_MARG["population"] == "all clients")].set_index("n_late_months")
s17_big = s17_m["marginal_pp"].idxmax()
add_fact("17.5", "dose-response", f"The first late month is the expensive one: default goes {s17_m.loc[0, 'rate']:.0%} -> {s17_m.loc[1, 'rate']:.0%} -> {s17_m.loc[2, 'rate']:.0%} for 0 / 1 / 2 late months; the largest single increment is month number {int(s17_big)} ({s17_m.loc[s17_big, 'marginal_pp']:+.0f} pp) and the curve reaches {s17_m['rate'].max():.0%} at {int(s17_m['rate'].idxmax())} late months", float(s17_m.loc[s17_big, "marginal_pp"]), "s17_marginal_late_month", 2)

**Observations:**
- The first late month costs the most: 11.7% → 29.8% (+18 points, odds × 3.2); later months add 6-13 points each, reaching 70.3% at six late months.
- Under status ≥ 2 the pattern is the same (12.8% → 32.2% → 43.4%), so it is not a code-1 artefact.
- History still matters for clients not late in Sep: 1 earlier late month 22.6%, 4 late months 41.8% (status ≥ 1).
- For clients already 2+ months late in Sep, extra history adds little (62% → 70% from 1 to 4 months).
- Story use: 'the first slip is the expensive one'.

#### 17.5c The point of no return: simplest conditions that push default past 50% and 75%
A small vocabulary of transparent delinquency / payment conditions is enumerated as **single conditions and pairs** (AND). Reported: every condition with ≥ 200 clients whose default rate reaches the bar, simplest first; pairs are kept only when *neither* member alone reaches the bar (so the combination is genuinely needed). (Section 9.3 mines rules on banded variables generally; this is the narrow "how bad must delinquency get" question.)

In [ ]:
s17_atoms = {}
for _k in (1, 2, 3):
    s17_atoms[f"Sep status >= {_k}"] = S17_S[:, 5] >= _k
for _k in (2, 3):
    s17_atoms[f"Aug status >= {_k}"] = S17_S[:, 4] >= _k
s17_atoms["Jul status >= 2"] = S17_S[:, 3] >= 2
for _k in range(1, 7):
    s17_atoms[f"late months (>=1) >= {_k}"] = (S17_S >= 1).sum(1) >= _k
for _k in range(1, 6):
    s17_atoms[f"months at 2+ delay >= {_k}"] = (S17_S >= 2).sum(1) >= _k
for _k in (2, 3, 4):
    s17_atoms[f"max delay >= {_k}"] = S17_S.max(1) >= _k
for _k in (2, 3, 4):
    s17_atoms[f"late streak ending Sep >= {_k}"] = df["LATE_STREAK_RECENT"].to_numpy() >= _k
s17_atoms["zero payment in Sep (Aug bill > 0)"] = (S17_P[:, 5] == 0) & (S17_B[:, 4] > 0)
s17_atoms["zero payment in Aug and Sep"] = (S17_P[:, 5] == 0) & (S17_P[:, 4] == 0) & (S17_B[:, 3] > 0)
s17_atoms["Sep payment < 5% of Aug bill"] = S17_PR[:, 5] < 0.05
s17_atoms["Sep utilisation > 90%"] = S17_U[:, 5] > 0.9
s17_atoms["Sep utilisation > 100%"] = S17_U[:, 5] > 1.0
s17_atoms["zero-payment months >= 3"] = (S17_P == 0).sum(1) >= 3
s17_atoms["credit limit <= 50k"] = S17_LIM <= 50_000
s17_rows = []; s17_names = list(s17_atoms)
def s17_cond_row(name, mk, size, parts):
    k, n = int(S17_Y[mk].sum()), int(mk.sum()); lo, hi = wilson_ci(k, n)
    return {"condition": name, "n_conditions": size, "n": n, "share_of_clients": n / S17_N, "defaults": k, "rate": k / n, "ci_lo": float(lo), "ci_hi": float(hi),
            "capture_of_defaulters": k / S17_Y.sum(), "parts": parts}
for _a in s17_names:
    if s17_atoms[_a].sum() >= 200:
        s17_rows.append(s17_cond_row(_a, s17_atoms[_a], 1, (_a,)))
for _a, _b in itertools.combinations(s17_names, 2):
    _mk = s17_atoms[_a] & s17_atoms[_b]; _n = int(_mk.sum())
    if _n >= 200 and _n < min(s17_atoms[_a].sum(), s17_atoms[_b].sum()):
        s17_rows.append(s17_cond_row(f"{_a}  AND  {_b}", _mk, 2, (_a, _b)))
S17_COND = pd.DataFrame(s17_rows)
s17_single_rate = S17_COND[S17_COND["n_conditions"] == 1].set_index("condition")["rate"].to_dict()
s17_out = []
for _bar in (0.50, 0.60, 0.75):
    _t = S17_COND[S17_COND["rate"] >= _bar].copy()
    _t = _t[[(r.n_conditions == 1) or all(s17_single_rate.get(p, 0) < _bar for p in r.parts) for r in _t.itertuples()]]
    _t["bar"] = _bar
    s17_out.append(_t.sort_values(["n_conditions", "n"], ascending=[True, False]))
S17_NORETURN = pd.concat(s17_out, ignore_index=True).drop(columns=["parts"])
for _bar in (0.50, 0.75):
    _t = S17_NORETURN[S17_NORETURN["bar"] == _bar]
    print(f"--- conditions with default rate >= {_bar:.0%} and n >= 200: {(_t['n_conditions'] == 1).sum()} single, {(_t['n_conditions'] == 2).sum()} pairs that need both ---")
    display(_t.groupby("n_conditions", group_keys=False).head(12)[["condition", "n", "rate", "ci_lo", "ci_hi", "capture_of_defaulters"]].round(4))
export(S17_NORETURN, "s17_no_return"); export(S17_COND.drop(columns=["parts"]), "s17_condition_catalogue")
fig, ax = plt.subplots(figsize=(13, 6))
_s1, _s2 = S17_COND[S17_COND["n_conditions"] == 1], S17_COND[S17_COND["n_conditions"] == 2]
ax.scatter(_s2["capture_of_defaulters"], _s2["rate"], s=12, color=C_NEU, alpha=.35, label="pairs")
ax.scatter(_s1["capture_of_defaulters"], _s1["rate"], s=55, color=C_DEF, label="single conditions", zorder=3)
for _r in _s1.itertuples():
    ax.annotate(_r.condition, (_r.capture_of_defaulters, _r.rate), fontsize=6.5, xytext=(4, 2), textcoords="offset points")
ax.axhline(.5, ls=":", color="k"); ax.axhline(.75, ls=":", color="k"); ax.axhline(BASE_RATE, ls="--", color="k", lw=1)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("share of all defaulters covered by the condition"); ax.set_ylabel("default rate under the condition"); ax.legend()
ax.set_title("17.5c Point of no return: default rate vs coverage for simple delinquency conditions (n >= 200)")
savefig("s17_no_return"); plt.show()
s17_50 = S17_NORETURN[(S17_NORETURN["bar"] == .5) & (S17_NORETURN["n_conditions"] == 1)]; s17_75 = S17_NORETURN[S17_NORETURN["bar"] == .75]
if len(s17_50):
    add_fact("17.5", "point of no return", f"The broadest single condition that crosses 50% is '{s17_50.iloc[0]['condition']}': {int(s17_50.iloc[0]['n']):,} clients, {s17_50.iloc[0]['rate']:.0%} default, covering {s17_50.iloc[0]['capture_of_defaulters']:.0%} of all defaulters", float(s17_50.iloc[0]["rate"]), "s17_no_return", 3)
if len(s17_75):
    add_fact("17.5", "point of no return", f"75% is crossed first by '{s17_75.iloc[0]['condition']}' ({int(s17_75.iloc[0]['n']):,} clients, {s17_75.iloc[0]['rate']:.0%}); {len(s17_75)} simple conditions with n >= 200 reach that bar", float(s17_75.iloc[0]["rate"]), "s17_no_return", 2)
else:
    add_fact("17.5", "point of no return", f"No simple one- or two-condition rule with n >= 200 reaches 75% default; the ceiling is {S17_COND['rate'].max():.0%} ('{S17_COND.sort_values('rate').iloc[-1]['condition']}')", float(S17_COND["rate"].max()), "s17_no_return", 2)

**Observations:**
- 22 single conditions reach 50% default; the broadest is 'Sep status ≥ 1' (6,818 clients, 50.3%, catching 51.7% of all defaulters).
- 'Sep status ≥ 2' is sharper: 3,130 clients, 69.6% default, catching 32.8% of defaulters.
- No single condition reaches 75%; the first pair is 'Sep status ≥ 2 AND 5+ late months' (1,108 clients, 75.6%), and the highest is 'Sep status ≥ 3 AND 6 late months' (80.0%, 225 clients).
- Story use: 'point of no return' callout: even the worst simple profile still has 1 in 5 clients who pay.

### 17.6 Warning-sign catalogue: 26 binary signs, what each is worth and how early it fires
Every sign is defined **per month** (so its first firing date is known) and then collapsed to "ever fired". Families: **D** delinquency, **P** payment, **U** utilisation / balance, **X** compound (two families at once), **S** usage shift (included deliberately as candidate signs that may *not* be risk signs).

For each sign: prevalence, default rate when present (hit rate) and absent, lift, **capture** (share of all defaulters showing the sign), false-alarm share of non-defaulters, **lead time** (median months before September at first firing; 0 = fired first in September), and whether a *stale* sign (not active in Aug–Sep) still matters. Signs that need lagged data cannot fire in April (left-truncation).

In [ ]:
S17_SIGNS = {}
def s17_add_sign(name, family, desc, fire):
    S17_SIGNS[name] = {"family": family, "desc": desc, "fire": np.asarray(fire, bool)}

s17_l1, s17_l2, s17_l3 = S17_S >= 1, S17_S >= 2, S17_S >= 3
s17_sprev = s17_shift(S17_S.astype(float)); s17_pprev = s17_shift(S17_P); s17_uprev = s17_shift(S17_U)
s17_everbefore = s17_shift(np.maximum.accumulate(s17_l1, axis=1), 2, False)
s17_zp = (S17_P == 0) & (S17_BPREV > 0)
s17_mp = (S17_PR > 0) & (S17_PR < 0.1)
s17_bupd = (S17_B > S17_BPREV) & (S17_P < s17_pprev)
s17_rev = S17_S == 0
# D: delinquency
s17_add_sign("LATE_ANY", "D", "status >= 1 in the month", s17_l1)
s17_add_sign("LATE_2PLUS", "D", "status >= 2 in the month", s17_l2)
s17_add_sign("LATE_3PLUS", "D", "status >= 3 in the month", s17_l3)
s17_add_sign("NEW_LATE", "D", "late this month, not late the month before", s17_l1 & (s17_sprev <= 0))
s17_add_sign("TWO_CONSEC_LATE", "D", "late in two consecutive months", s17_l1 & s17_shift(s17_l1, 1, False))
s17_add_sign("THREE_CONSEC_LATE", "D", "late in three consecutive months", s17_l1 & s17_shift(s17_l1, 1, False) & s17_shift(s17_l1, 2, False))
s17_add_sign("ROLL_DEEPER", "D", "already late and status rose further", (s17_sprev >= 1) & (S17_S > s17_sprev))
s17_add_sign("RELAPSE", "D", "late again after a clean month following an earlier late month", s17_l1 & (s17_sprev <= 0) & s17_everbefore)
# P: payment
s17_add_sign("ZERO_PAY_POS_BAL", "P", "paid nothing although the previous bill was positive", s17_zp)
s17_add_sign("TWO_ZERO_PAY", "P", "zero payment on a positive bill two months running", s17_zp & s17_shift(s17_zp, 1, False))
s17_add_sign("PAY_DROP_50", "P", "payment fell by more than half (previous payment >= NT$1,000, balance still owed)", (S17_P < 0.5 * s17_pprev) & (s17_pprev >= 1000) & (S17_BPREV > 0) & (S17_B > 0))
s17_add_sign("MIN_PAYER_3M", "P", "paid 0-10% of the previous bill three months running", s17_mp & s17_shift(s17_mp, 1, False) & s17_shift(s17_mp, 2, False))
s17_add_sign("PAY_RATIO_FALL", "P", "payment ratio fell two months in a row and is below 100%", (S17_PR < s17_shift(S17_PR)) & (s17_shift(S17_PR) < s17_shift(S17_PR, 2)) & (S17_PR < 1))
# U: utilisation / balance
s17_add_sign("UTIL_GT_90", "U", "bill above 90% of the limit", S17_U > 0.9)
s17_add_sign("OVER_LIMIT", "U", "bill above the credit limit", S17_U > 1.0)
s17_add_sign("UTIL_JUMP_20PP", "U", "utilisation rose by more than 20 points in a month", (S17_U - s17_uprev) > 0.2)
s17_add_sign("RISING_BILL_3M", "U", "bill rose three months in a row", (S17_B > S17_BPREV) & (S17_BPREV > s17_shift(S17_B, 2)) & (s17_shift(S17_B, 2) > s17_shift(S17_B, 3)) & (S17_B > 0))
s17_add_sign("BILL_UP_PAY_DOWN_2M", "U", "bill up and payment down, two months running", s17_bupd & s17_shift(s17_bupd, 1, False))
s17_add_sign("BALANCE_GROWTH_50", "U", "bill grew by more than 50% (previous bill >= NT$5,000)", (S17_B > 1.5 * S17_BPREV) & (S17_BPREV >= 5000))
# X: compound
s17_add_sign("MAXED_MIN_PAY", "X", "utilisation > 90% and payment below 10% of the previous bill", (S17_U > 0.9) & (S17_PR < 0.1))
s17_add_sign("LATE_AND_ZERO_PAY", "X", "late and paid nothing", s17_l1 & (S17_P == 0))
s17_add_sign("LATE_AND_MAXED", "X", "late with utilisation > 90%", s17_l1 & (S17_U > 0.9))
# S: usage shift
s17_add_sign("SPEND_STOP", "S", "no new spending after an active month, balance still owed", (S17_SP <= 0) & (s17_shift(S17_SP) > 1000) & (S17_B > 0))
s17_add_sign("SPEND_SURGE", "S", "implied spend above 30% of the limit in one month", S17_SP > 0.3 * S17_LIM[:, None])
s17_add_sign("REVOLVE_3M", "S", "revolving (status 0) three months running", s17_rev & s17_shift(s17_rev, 1, False) & s17_shift(s17_rev, 2, False))
s17_add_sign("PAIDFULL_TO_REVOLVE", "S", "switched from paying in full / inactive to revolving", (s17_sprev <= -1) & s17_rev)

S17_SIGN_NAMES = list(S17_SIGNS)
S17_FAM = {"D": "Delinquency", "P": "Payment", "U": "Utilisation / balance", "X": "Compound", "S": "Usage shift"}
S17_EVER = np.column_stack([S17_SIGNS[s]["fire"].any(1) for s in S17_SIGN_NAMES])
S17_FF = np.column_stack([np.where(S17_SIGNS[s]["fire"].any(1), S17_SIGNS[s]["fire"].argmax(1), np.nan) for s in S17_SIGN_NAMES])   # first firing month index
S17_CORE = np.array([S17_SIGNS[s]["family"] != "S" for s in S17_SIGN_NAMES])
s17_rows = []
for _j, _s in enumerate(S17_SIGN_NAMES):
    _f = S17_SIGNS[_s]["fire"]; _ev = S17_EVER[:, _j]; _rec = _f[:, 4:].any(1); _stale = _ev & ~_rec
    df[f"EW_SIGN_{_s}"] = _ev.astype(np.int8)
    _lead = 5 - S17_FF[_ev, _j]; _leadd = 5 - S17_FF[_ev & (S17_Y == 1), _j]
    _lo, _hi = wilson_ci(S17_Y[_ev].sum(), _ev.sum())
    s17_rows.append({"sign": _s, "family": S17_FAM[S17_SIGNS[_s]["family"]], "definition": S17_SIGNS[_s]["desc"], "first_possible_month": MONTH_ORDER[int(np.argmax(_f.any(0)))],
                     "n_fired": int(_ev.sum()), "prevalence": _ev.mean(), "hit_rate": S17_Y[_ev].mean(), "hit_ci_lo": float(_lo), "hit_ci_hi": float(_hi),
                     "rate_when_absent": S17_Y[~_ev].mean(), "lift": S17_Y[_ev].mean() / BASE_RATE, "capture": _ev[S17_Y == 1].mean(),
                     "false_alarm_share": _ev[S17_Y == 0].mean(), "youden_J": _ev[S17_Y == 1].mean() - _ev[S17_Y == 0].mean(),
                     "lead_median_months": float(np.median(_lead)), "lead_mean_months": float(_lead.mean()), "lead_median_defaulters": float(np.median(_leadd)),
                     "share_first_fired_by_Jun": float((S17_FF[_ev, _j] <= 2).mean()),
                     "n_recent": int(_rec.sum()), "rate_recent_AugSep": S17_Y[_rec].mean() if _rec.sum() else np.nan,
                     "n_stale": int(_stale.sum()), "rate_stale": S17_Y[_stale].mean() if _stale.sum() >= 30 else np.nan,
                     "mean_months_fired": float(_f[_ev].sum(1).mean())})
S17_SIGN_TAB = pd.DataFrame(s17_rows).sort_values("lift", ascending=False).reset_index(drop=True)
display(S17_SIGN_TAB[["sign", "family", "prevalence", "hit_rate", "rate_when_absent", "lift", "capture", "false_alarm_share", "youden_J", "lead_median_months", "lead_median_defaulters", "rate_recent_AugSep", "rate_stale"]].round(3))
export(S17_SIGN_TAB, "s17_warning_signs")

**Bubble chart:** capture (x) vs hit rate (y), bubble area = prevalence, colour = family. The top-right is the ideal sign (catches many defaulters, rarely wrong); most real signs trade one for the other. Right: lead time vs lift, and what happens to a sign when it goes stale.

In [ ]:
S17_FAM_COL = {"Delinquency": C_DEF, "Payment": "#fd8d3c", "Utilisation / balance": C_OK, "Compound": "#756bb1", "Usage shift": C_NEU}
fig, axes = plt.subplots(1, 3, figsize=(24, 7.5), gridspec_kw={"width_ratios": [1.5, 1, 1]})
ax = axes[0]
for _fam, _g in S17_SIGN_TAB.groupby("family"):
    ax.scatter(_g["capture"], _g["hit_rate"], s=40 + 3000 * _g["prevalence"], color=S17_FAM_COL[_fam], alpha=.55, edgecolor="k", lw=.5, label=_fam)
for _r in S17_SIGN_TAB.itertuples():
    ax.annotate(_r.sign, (_r.capture, _r.hit_rate), fontsize=7, ha="center", va="center")
ax.axhline(BASE_RATE, ls="--", color="k", lw=1); ax.axhline(.5, ls=":", color=C_NEU)
_xx = np.linspace(0.02, 1, 100)
for _prev in (0.05, 0.15, 0.4):
    ax.plot(_xx, np.clip(_xx * BASE_RATE / _prev, 0, 1), color=C_NEU, lw=.6, ls=":")
    ax.text(min(1, _prev / BASE_RATE) * 0.98, min(1, BASE_RATE / _prev) * 0.98 if _prev < BASE_RATE else 0.98 * BASE_RATE / _prev, f"prev {_prev:.0%}", fontsize=6, color=C_NEU, ha="right")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_ylim(0, 1)
ax.set_xlabel("capture: share of all defaulters showing the sign"); ax.set_ylabel("hit rate: default rate when the sign is present")
_lg = ax.legend(fontsize=8, loc="upper right")
for _h in getattr(_lg, "legend_handles", getattr(_lg, "legendHandles", [])):
    _h.set_sizes([60])
ax.set_title("Warning signs: capture vs hit rate (bubble = prevalence)")
ax = axes[1]
ax.scatter(S17_SIGN_TAB["lead_mean_months"], S17_SIGN_TAB["lift"], s=40 + 3000 * S17_SIGN_TAB["prevalence"], c=[S17_FAM_COL[f] for f in S17_SIGN_TAB["family"]], alpha=.55, edgecolor="k", lw=.5)
for _r in S17_SIGN_TAB.itertuples():
    ax.annotate(_r.sign, (_r.lead_mean_months, _r.lift), fontsize=6.5, ha="center")
ax.axhline(1, ls="--", color="k", lw=1); ax.set_xlabel("mean lead time (months before Sep at first firing)"); ax.set_ylabel("lift"); ax.set_title("Lead time vs lift: early signs are weaker")
ax = axes[2]; _t = S17_SIGN_TAB.dropna(subset=["rate_stale"]).sort_values("rate_recent_AugSep")
ax.hlines(_t["sign"], _t["rate_stale"], _t["rate_recent_AugSep"], color=C_NEU, lw=2)
ax.scatter(_t["rate_recent_AugSep"], _t["sign"], color=C_DEF, s=45, label="sign active in Aug/Sep", zorder=3); ax.scatter(_t["rate_stale"], _t["sign"], color=C_OK, s=45, label="fired earlier only (stale)", zorder=3)
ax.axvline(BASE_RATE, ls="--", color="k", lw=1); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.tick_params(axis="y", labelsize=7); ax.legend(fontsize=8); ax.set_title("Fresh vs stale sign: default rate")
fig.suptitle("17.6 Warning-sign catalogue", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s17_warning_signs_bubble"); plt.show()
s17_bj = S17_SIGN_TAB.sort_values("youden_J", ascending=False).iloc[0]; s17_bh = S17_SIGN_TAB[S17_SIGN_TAB["n_fired"] >= 300].sort_values("hit_rate", ascending=False).iloc[0]
s17_weak = S17_SIGN_TAB[S17_SIGN_TAB["lift"] < 1]["sign"].tolist()
print("signs with lift < 1 (popular 'warning signs' that are not):", s17_weak)
add_fact("17.6", "warning signs", f"Best all-round sign: {s17_bj['sign']} ({s17_bj['definition']}) fires for {s17_bj['prevalence']:.0%} of clients, catches {s17_bj['capture']:.0%} of defaulters at a {s17_bj['hit_rate']:.0%} hit rate; the most precise sign with 300+ clients is {s17_bh['sign']} ({s17_bh['hit_rate']:.0%} hit rate, {s17_bh['capture']:.0%} capture)", float(s17_bj["youden_J"]), "s17_warning_signs_bubble", 3)
add_fact("17.6", "warning signs", f"{len(s17_weak)} of {len(S17_SIGN_TAB)} candidate signs have lift below 1 ({', '.join(s17_weak[:5])}): they mark lower-risk clients, not trouble", len(s17_weak), "s17_warning_signs_bubble", 2)

**Observations:**
- The best all-round sign is LATE_ANY (any late month): it fires for 33.6% of clients, catches 64.8% of defaulters, and 42.7% of flagged clients default.
- The sharpest signs are rare: ROLL_DEEPER (delay getting worse) and LATE_3PLUS hit 63% but catch only 10-11% of defaulters.
- 10 of 26 popular warning signs have lift below 1 (they mark safer clients): e.g. SPEND_SURGE (20.6%), RISING_BILL_3M (18.4%), REVOLVE_3M (17.3%), BALANCE_GROWTH_50 (15.4%).
- Recent signs are sharper: LATE_2PLUS hits 57.8% when it fired in Aug/Sep vs 25.4% when stale (numbers from the table above; bubble chart is image-only).
- Story use: myth-busting catalogue; matches section 16's 'growing balance is not the danger'.

#### 17.6b Do the signs overlap? Co-occurrence (Jaccard, lift) and network
Jaccard = |A ∩ B| / |A ∪ B|; co-occurrence lift = P(A ∩ B) / (P(A)·P(B)). Signs with Jaccard near 1 are the same alarm rung twice. The network links signs with Jaccard ≥ 0.25 (node size = prevalence, colour = hit rate).

In [ ]:
s17_E = S17_EVER.astype(float); s17_inter = s17_E.T @ s17_E; s17_cnt = s17_E.sum(0)
S17_JACC = s17_inter / (s17_cnt[:, None] + s17_cnt[None, :] - s17_inter)
S17_COLIFT = (s17_inter / S17_N) / np.outer(s17_cnt / S17_N, s17_cnt / S17_N)
s17_fo = sorted(range(len(S17_SIGN_NAMES)), key=lambda j: ("DPUXS".index(S17_SIGNS[S17_SIGN_NAMES[j]]["family"]), -s17_cnt[j]))
s17_lab = [S17_SIGN_NAMES[j] for j in s17_fo]
fig, axes = plt.subplots(1, 3, figsize=(27, 8.5))
sns.heatmap(pd.DataFrame(S17_JACC[np.ix_(s17_fo, s17_fo)], s17_lab, s17_lab), cmap="mako_r", vmin=0, vmax=1, annot=True, fmt=".1f", annot_kws={"fontsize": 5.5}, ax=axes[0], cbar=False)
axes[0].set_title("Jaccard similarity of 'ever fired' (ordered by family)"); axes[0].tick_params(labelsize=7)
sns.heatmap(pd.DataFrame(np.log2(np.clip(S17_COLIFT[np.ix_(s17_fo, s17_fo)], 1e-3, None)), s17_lab, s17_lab), cmap=CMAP_DIV, center=0, vmin=-3, vmax=3, ax=axes[1], cbar_kws={"label": "log2 co-occurrence lift", "shrink": .6})
axes[1].set_title("Co-occurrence lift (log2; red = fire together more than chance)"); axes[1].tick_params(labelsize=7)
s17_pairs = []
for _a, _b in itertools.combinations(range(len(S17_SIGN_NAMES)), 2):
    _both = S17_EVER[:, _a] & S17_EVER[:, _b]
    s17_pairs.append({"sign_a": S17_SIGN_NAMES[_a], "sign_b": S17_SIGN_NAMES[_b], "family_a": S17_SIGNS[S17_SIGN_NAMES[_a]]["family"], "family_b": S17_SIGNS[S17_SIGN_NAMES[_b]]["family"],
                      "n_both": int(_both.sum()), "jaccard": S17_JACC[_a, _b], "cooccurrence_lift": S17_COLIFT[_a, _b],
                      "rate_both": S17_Y[_both].mean() if _both.sum() >= 30 else np.nan, "capture_both": _both[S17_Y == 1].mean()})
S17_PAIRS = pd.DataFrame(s17_pairs)
try:
    import networkx as s17_nx
    s17_G = s17_nx.Graph()
    for _r in S17_SIGN_TAB.itertuples():
        s17_G.add_node(_r.sign, prev=_r.prevalence, hit=_r.hit_rate)
    for _r in S17_PAIRS[S17_PAIRS["jaccard"] >= 0.25].itertuples():
        s17_G.add_edge(_r.sign_a, _r.sign_b, weight=_r.jaccard)
    s17_pos = s17_nx.spring_layout(s17_G, seed=17, k=0.9, weight="weight")
    s17_nx.draw_networkx_edges(s17_G, s17_pos, ax=axes[2], width=[6 * d["weight"] for _, _, d in s17_G.edges(data=True)], edge_color=C_NEU, alpha=.6)
    _nc = s17_nx.draw_networkx_nodes(s17_G, s17_pos, ax=axes[2], node_size=[150 + 4000 * s17_G.nodes[n]["prev"] for n in s17_G], node_color=[s17_G.nodes[n]["hit"] for n in s17_G], cmap=CMAP_RATE, vmin=0, vmax=.8, edgecolors="k", linewidths=.5)
    s17_nx.draw_networkx_labels(s17_G, s17_pos, ax=axes[2], font_size=7)
    plt.colorbar(_nc, ax=axes[2], shrink=.6, label="hit rate"); axes[2].set_title("Sign network (edge = Jaccard >= 0.25)"); axes[2].axis("off")
    print(f"network: {s17_G.number_of_edges()} edges, {s17_nx.number_connected_components(s17_G)} components (isolated signs carry unique information)")
except Exception as s17_e:
    axes[2].text(.5, .5, f"networkx unavailable: {type(s17_e).__name__}", ha="center"); axes[2].axis("off")
plt.tight_layout(); savefig("s17_sign_cooccurrence"); plt.show()
export(S17_PAIRS, "s17_sign_cooccurrence")
print("most redundant pairs (Jaccard):"); display(S17_PAIRS.sort_values("jaccard", ascending=False).head(10).round(3))
print("most dangerous co-occurring pairs from DIFFERENT families (n >= 200):")
s17_x = S17_PAIRS[(S17_PAIRS["family_a"] != S17_PAIRS["family_b"]) & (S17_PAIRS["n_both"] >= 200)].sort_values("rate_both", ascending=False)
display(s17_x.head(12).round(3))
s17_rd = S17_PAIRS.sort_values("jaccard", ascending=False).iloc[0]
add_fact("17.6", "warning signs", f"Alarm redundancy: {int((S17_PAIRS['jaccard'] >= 0.5).sum())} sign pairs overlap with Jaccard >= 0.5 (top: {s17_rd['sign_a']} / {s17_rd['sign_b']} at {s17_rd['jaccard']:.2f}); the most dangerous cross-family pair is {s17_x.iloc[0]['sign_a']} + {s17_x.iloc[0]['sign_b']} ({s17_x.iloc[0]['rate_both']:.0%} default, n = {int(s17_x.iloc[0]['n_both']):,})", float(s17_x.iloc[0]["rate_both"]), "s17_sign_cooccurrence", 2)

**Observations:**
- Many alarms repeat each other: UTIL_GT_90 and MAXED_MIN_PAY overlap almost fully (Jaccard 0.93; Jaccard = shared ÷ combined clients), LATE_3PLUS and ROLL_DEEPER 0.88, LATE_ANY and LATE_2PLUS 0.83.
- The network has 63 links in 5 groups, so only a few signs carry unique information.
- The most dangerous cross-family pairs combine a deep delay with minimum payments or a falling payment ratio (about 63-66% default, but only 217-290 clients each).
- Story use: background; argues for a short list of 4-6 non-overlapping signs on the dashboard.

#### 17.6c The warning-sign count: default rate by the NUMBER of signs fired
`EW_N_SIGNS` counts all 26 signs, `EW_N_SIGNS_CORE` the 22 core signs (families D, P, U, X; the usage-shift family is context, not alarm). `EW_N_FAMILIES` counts how many of the three base families (D, P, U) have fired, and `EW_FAMILY_COMBO` names them — does risk come from *many* alarms or from *which* family?

In [ ]:
df["EW_N_SIGNS"] = S17_EVER.sum(1)
df["EW_N_SIGNS_CORE"] = S17_EVER[:, S17_CORE].sum(1)
df["EW_N_SIGNS_RECENT"] = np.column_stack([S17_SIGNS[s]["fire"][:, 4:].any(1) for s in S17_SIGN_NAMES])[:, S17_CORE].sum(1)
S17_FAM_EVER = {f: S17_EVER[:, [j for j, s in enumerate(S17_SIGN_NAMES) if S17_SIGNS[s]["family"] == f]].any(1) for f in "DPUXS"}
df["EW_N_FAMILIES"] = sum(S17_FAM_EVER[f].astype(int) for f in "DPU")
df["EW_FAMILY_COMBO"] = ["+".join([n for n, f in (("Delinq", "D"), ("Payment", "P"), ("Util", "U")) if S17_FAM_EVER[f][i]]) or "None" for i in range(S17_N)]
df["EW_N_SIGNS_CAPPED"] = np.minimum(df["EW_N_SIGNS_CORE"], 12)
fig, axes = plt.subplots(1, 3, figsize=(24, 5.2))
s17_tn = plot_rate(df, "EW_N_SIGNS_CAPPED", ax=axes[0], title="Default rate by number of CORE signs fired (12 = 12+)")
axes[0].set_xlabel("core warning signs fired (of 22)")
s17_tr = plot_rate(df.assign(EW_N_SIGNS_RECENT_C=np.minimum(df["EW_N_SIGNS_RECENT"], 10)), "EW_N_SIGNS_RECENT_C", ax=axes[1], title="... counting only signs active in Aug/Sep (10 = 10+)")
s17_tf = plot_rate(df, "EW_FAMILY_COMBO", ax=axes[2], order=rate_table(df, "EW_FAMILY_COMBO").sort_values("rate")["EW_FAMILY_COMBO"].tolist(), title="Default rate by which base families fired", rotate=25)
plt.tight_layout(); savefig("s17_sign_count_curve"); plt.show()
display(s17_tn[["EW_N_SIGNS_CAPPED", "n", "rate", "ci_lo", "ci_hi", "share_clients", "share_defaults", "lift"]].round(4))
display(s17_tf[["EW_FAMILY_COMBO", "n", "rate", "ci_lo", "ci_hi", "share_clients", "share_defaults", "lift"]].round(4))
export(s17_tn, "s17_sign_count_curve"); export(s17_tf, "s17_sign_family_combo"); export(s17_tr, "s17_sign_count_recent")
s17_rev_cum = s17_tn.sort_values("EW_N_SIGNS_CAPPED", ascending=False); s17_rev_cum["cum_rate"] = s17_rev_cum["defaults"].cumsum() / s17_rev_cum["n"].cumsum()
S17_K50 = int(s17_tn.loc[s17_tn["rate"] >= .5, "EW_N_SIGNS_CAPPED"].min()) if (s17_tn["rate"] >= .5).any() else int(s17_tn["EW_N_SIGNS_CAPPED"].max())
s17_z, s17_k = s17_tn.iloc[0], s17_tn[s17_tn["EW_N_SIGNS_CAPPED"] == S17_K50].iloc[0]
print(f"AUC of the raw counts: all signs {s17_auc(df['EW_N_SIGNS']):.3f} | core {s17_auc(df['EW_N_SIGNS_CORE']):.3f} | core recent {s17_auc(df['EW_N_SIGNS_RECENT']):.3f} | first count with rate >= 50%: {S17_K50}")
add_fact("17.6", "warning signs", f"Counting alarms works: clients with zero core signs default at {s17_z['rate']:.1%} ({s17_z['share_clients']:.0%} of clients), and the rate first passes 50% at {S17_K50} signs; the plain count has AUC {s17_auc(df['EW_N_SIGNS_CORE']):.3f}, {s17_auc(df['EW_N_SIGNS_RECENT']):.3f} if only Aug/Sep signs are counted", float(s17_z["rate"]), "s17_sign_count_curve", 3)

**Observations:**
- The sign count is not a straight line: 0 signs 17.2%, 1-2 signs only 9-10%, and the rate passes 50% only at 11+ signs (51.6% at 12+).
- By family: delinquency + payment signs default at 47.8% and delinquency + payment + utilisation at 42.3%; payment-only or utilisation-only clients sit at 10-12%, below the zero-sign group.
- Counting signs ranks clients with AUC 0.69 (core signs), far below the Sep status alone.
- The first alarm is usually not delinquency: REVOLVE_3M (three revolving months) is 28% of first alarms and PAY_DROP_50 16%, and both have low risk when they ring alone (14% and 13%).
- Story use: 'path to default' sequence chart (`s17_sign_ordered_pairs.csv`, `s17_family_paths.csv`): without a late month, money signals rarely mean danger.

### 17.7 Sign sequences: which alarm rings first, and what is the typical path to default?
Using each sign's **first firing month**: (a) how often a sign is the first (or tied-first) alarm for a client; (b) the pairwise **precedence** matrix P(A fires strictly before B | both fire); (c) the most common **ordered pairs** A → B among defaulters vs non-defaulters; (d) family-level **paths** (order in which Delinquency / Payment / Utilisation / Usage-shift first appear). Caveat: the window opens in April, so anything already present in April is left-truncated ("first" means first *observed*).

In [ ]:
s17_ff = np.where(np.isnan(S17_FF), 99, S17_FF); s17_min = s17_ff.min(1); s17_any = s17_min < 99
df["EW_FIRST_SIGN_MONTH"] = np.where(s17_any, np.array(MONTH_ORDER + ["None"])[np.where(s17_any, s17_min, 6).astype(int)], "None")
df["EW_LEAD_TIME"] = np.where(s17_any, 5 - s17_min, np.nan)
s17_isfirst = (s17_ff == s17_min[:, None]) & s17_any[:, None]; s17_ntied = s17_isfirst.sum(1)
s17_rows = []
for _j, _s in enumerate(S17_SIGN_NAMES):
    _ev = S17_EVER[:, _j]; _fi = s17_isfirst[:, _j]; _solo = _fi & (s17_ntied == 1)
    s17_rows.append({"sign": _s, "family": S17_FAM[S17_SIGNS[_s]["family"]], "n_fired": int(_ev.sum()), "among_first_when_fired": _fi[_ev].mean(),
                     "share_of_first_alarms_fractional": float((_fi[s17_any] / np.maximum(s17_ntied[s17_any], 1)).sum() / s17_any.sum()),
                     "n_sole_first": int(_solo.sum()), "rate_when_sole_first": S17_Y[_solo].mean() if _solo.sum() >= 30 else np.nan,
                     "mean_first_month_index": float(S17_FF[_ev, _j].mean()), "rate_defaulters_among_first": _fi[_ev & (S17_Y == 1)].mean(), "rate_nondefaulters_among_first": _fi[_ev & (S17_Y == 0)].mean()})
S17_ORDER = pd.DataFrame(s17_rows).sort_values("share_of_first_alarms_fractional", ascending=False).reset_index(drop=True)
display(S17_ORDER.round(3)); export(S17_ORDER, "s17_sign_order")
s17_nS = len(S17_SIGN_NAMES); S17_PREC = np.full((s17_nS, s17_nS), np.nan); s17_op = []
for _a in range(s17_nS):
    for _b in range(s17_nS):
        if _a == _b:
            continue
        _both = S17_EVER[:, _a] & S17_EVER[:, _b]; _ab = _both & (S17_FF[:, _a] < S17_FF[:, _b])
        if _both.sum() >= 100:
            S17_PREC[_a, _b] = _ab.sum() / _both.sum()
        if _ab.sum() >= 200:
            s17_op.append({"first": S17_SIGN_NAMES[_a], "then": S17_SIGN_NAMES[_b], "path": f"{S17_SIGN_NAMES[_a]} -> {S17_SIGN_NAMES[_b]}",
                           "cross_family": S17_SIGNS[S17_SIGN_NAMES[_a]]["family"] != S17_SIGNS[S17_SIGN_NAMES[_b]]["family"], "n": int(_ab.sum()),
                           "default_rate": S17_Y[_ab].mean(), "share_of_defaulters": _ab[S17_Y == 1].mean(), "share_of_nondefaulters": _ab[S17_Y == 0].mean(),
                           "mean_gap_months": float((S17_FF[_ab, _b] - S17_FF[_ab, _a]).mean())})
S17_OPAIRS = pd.DataFrame(s17_op); S17_OPAIRS["defaulter_to_nondefaulter_ratio"] = S17_OPAIRS["share_of_defaulters"] / S17_OPAIRS["share_of_nondefaulters"]
_rev = S17_OPAIRS.set_index(["then", "first"])["default_rate"].to_dict()
S17_OPAIRS["rate_reverse_order"] = [_rev.get((r.first, r.then), np.nan) for r in S17_OPAIRS.itertuples()]
S17_OPAIRS["order_effect_pp"] = 100 * (S17_OPAIRS["default_rate"] - S17_OPAIRS["rate_reverse_order"])
export(S17_OPAIRS, "s17_sign_ordered_pairs")
print("ordered pairs most typical of DEFAULTERS (cross-family, by share of defaulters):")
display(S17_OPAIRS[S17_OPAIRS["cross_family"]].sort_values("share_of_defaulters", ascending=False).head(12)[["path", "n", "default_rate", "share_of_defaulters", "share_of_nondefaulters", "defaulter_to_nondefaulter_ratio", "mean_gap_months"]].round(3))
print("ordered pairs most typical of NON-defaulters (cross-family, by share of non-defaulters):")
display(S17_OPAIRS[S17_OPAIRS["cross_family"]].sort_values("share_of_nondefaulters", ascending=False).head(8)[["path", "n", "default_rate", "share_of_defaulters", "share_of_nondefaulters", "defaulter_to_nondefaulter_ratio"]].round(3))
print("where ORDER matters: same two signs, opposite order (both orders n >= 200), largest gaps:")
display(S17_OPAIRS.dropna(subset=["rate_reverse_order"]).sort_values("order_effect_pp", ascending=False).head(8)[["path", "n", "default_rate", "rate_reverse_order", "order_effect_pp"]].round(3))

Left: precedence matrix (row fires before column), signs ordered from earliest to latest "typical position". Middle: the highest-default ordered pairs across families. Right: family-level paths (`>` = strictly earlier, `=` = same month).

In [ ]:
s17_early = np.nanmean(S17_PREC, axis=1) - np.nanmean(S17_PREC, axis=0); s17_eo = np.argsort(-np.nan_to_num(s17_early, nan=-9))
s17_fam_first = {f: np.where(np.isnan(S17_FF[:, [j for j, s in enumerate(S17_SIGN_NAMES) if S17_SIGNS[s]["family"] == f]]), 99, S17_FF[:, [j for j, s in enumerate(S17_SIGN_NAMES) if S17_SIGNS[s]["family"] == f]]).min(1) for f in "DPUS"}
s17_fm = np.column_stack([s17_fam_first[f] for f in "DPUS"]); s17_fn = np.array(["Delinq", "Pay", "Util", "Shift"])
def s17_path(row):
    idx = [j for j in np.argsort(row, kind="stable") if row[j] < 99]
    if not idx:
        return "(no sign)"
    out = s17_fn[idx[0]]
    for a, b in zip(idx[:-1], idx[1:]):
        out += (" = " if row[a] == row[b] else " > ") + s17_fn[b]
    return out
df["EW_FAMILY_PATH"] = [s17_path(r) for r in s17_fm]
S17_PATHS = rate_table(df, "EW_FAMILY_PATH", min_n=150).sort_values("n", ascending=False).reset_index(drop=True)
fig, axes = plt.subplots(1, 3, figsize=(27, 8.5), gridspec_kw={"width_ratios": [1.3, 1, 1]})
_lab = [S17_SIGN_NAMES[j] for j in s17_eo]
sns.heatmap(pd.DataFrame(S17_PREC[np.ix_(s17_eo, s17_eo)], _lab, _lab), cmap=CMAP_DIV, center=.33, vmin=0, vmax=1, annot=True, fmt=".1f", annot_kws={"fontsize": 5.5}, ax=axes[0], cbar=False)
axes[0].set_title("P(row sign fires strictly before column sign | both fire)"); axes[0].tick_params(labelsize=7)
_t = S17_OPAIRS[S17_OPAIRS["cross_family"]].sort_values("default_rate", ascending=False).head(18).iloc[::-1]
axes[1].barh(_t["path"], _t["default_rate"], color=C_DEF, alpha=.85)
for _y, (_r, _n) in enumerate(zip(_t["default_rate"], _t["n"])):
    axes[1].text(_r + .005, _y, f"{_r:.0%} (n={_n:,})", va="center", fontsize=7)
axes[1].axvline(BASE_RATE, ls="--", color="k", lw=1); axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].tick_params(axis="y", labelsize=7); axes[1].set_xlim(0, 1)
axes[1].set_title("Highest-default cross-family ordered pairs (n >= 200)")
_t = S17_PATHS.head(22).sort_values("rate")
axes[2].barh(_t["EW_FAMILY_PATH"], _t["rate"], xerr=[_t["rate"] - _t["ci_lo"], _t["ci_hi"] - _t["rate"]], color=plt.cm.Reds(.2 + .8 * _t["rate"] / _t["rate"].max()), capsize=2)
for _y, (_r, _n) in enumerate(zip(_t["ci_hi"], _t["n"])):
    axes[2].text(_r + .01, _y, f"n={_n:,}", va="center", fontsize=7)
axes[2].axvline(BASE_RATE, ls="--", color="k", lw=1); axes[2].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[2].tick_params(axis="y", labelsize=7)
axes[2].set_title("Family-level paths: 22 most common, by default rate")
fig.suptitle("17.7 Order of appearance of warning signs", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s17_sign_sequences"); plt.show()
display(S17_PATHS[["EW_FAMILY_PATH", "n", "rate", "ci_lo", "ci_hi", "share_clients", "share_defaults", "lift"]].head(22).round(4)); export(S17_PATHS, "s17_family_paths")
s17_fa = S17_ORDER.iloc[0]; s17_dp = S17_OPAIRS[S17_OPAIRS["cross_family"]].sort_values("share_of_defaulters", ascending=False).iloc[0]
s17_d_first = df.loc[S17_FAM_EVER["D"], "EW_FAMILY_PATH"].str.startswith("Delinq").mean(); s17_oe = S17_OPAIRS.dropna(subset=["rate_reverse_order"]).sort_values("order_effect_pp", ascending=False).iloc[0]
add_fact("17.7", "sequences", f"The first alarm is usually not delinquency: {s17_fa['sign']} accounts for {s17_fa['share_of_first_alarms_fractional']:.0%} of first alarms, and among clients who ever go late only {s17_d_first:.0%} have delinquency as their first (or tied-first) sign family", float(s17_d_first), "s17_sign_sequences", 2)
add_fact("17.7", "sequences", f"Most common defaulter path across families: {s17_dp['path']} ({s17_dp['share_of_defaulters']:.0%} of defaulters vs {s17_dp['share_of_nondefaulters']:.0%} of non-defaulters, {s17_dp['default_rate']:.0%} default); order matters most for {s17_oe['path']} ({s17_oe['default_rate']:.0%} vs {s17_oe['rate_reverse_order']:.0%} in the reverse order)", float(s17_dp["share_of_defaulters"]), "s17_sign_sequences", 2)

**Observations:**
- Paths starting with delinquency dominate risk: 'Delinq > Pay' (late, then payment trouble) defaults at 60.2% (1,909 clients, 17% of defaulters); 'Delinq = Util > Pay' at 58.9%.
- Paths with only payment, utilisation or usage-shift signs default at 7-13%, below the 18.7% for clients with no sign at all.
- The most common path, 'Shift > Pay', is among the safest at 9.3%.
- Story use: family-level path chart; reinforces 'without a late month, money signals rarely mean danger'.

### 17.8 Capture curves: how close do simple transparent rules get to the lens?
A descriptive gain (Lorenz) view, **not** a model score: rank clients by a transparent quantity and ask what share of all defaulters sits in the top 10 / 20 / 30 / 50%. Ties share credit linearly (equivalent to random tie-breaking). The lens out-of-fold ranking from 17.1 is the reference for "all the information in the file"; the perfect curve and the diagonal bound the picture.

In [ ]:
S17_SCORES = {
    "N core signs (22)": df["EW_N_SIGNS_CORE"].to_numpy(float),
    "N core signs active Aug/Sep": df["EW_N_SIGNS_RECENT"].to_numpy(float),
    "N signs (all 26)": df["EW_N_SIGNS"].to_numpy(float),
    "Sep status (PAY_1)": df["PAY_1"].to_numpy(float),
    "Max delay in 6 months": df["MAX_DELAY"].to_numpy(float),
    "N late months": df["N_LATE"].to_numpy(float),
    "Sep status, ties by N core signs": df["PAY_1"].to_numpy(float) * 100 + df["EW_N_SIGNS_CORE"].to_numpy(float),
    "Sep status + Aug status + N late (sum)": np.clip(df["PAY_1"], 0, None).to_numpy(float) + np.clip(df["PAY_2"], 0, None).to_numpy(float) + df["N_LATE"].to_numpy(float),
    "Lens OOF: Sep data only": S17_OOF[("reverse", 1)],
    "Lens OOF: all six months": S17_OOF[("forward", 6)],
}
s17_rows, s17_pts = [], []; s17_grid_q = np.linspace(0, 1, 101)
s17_lens_px, s17_lens_py = s17_gain(S17_SCORES["Lens OOF: all six months"])
for _nm, _sc in S17_SCORES.items():
    _px, _py = s17_gain(_sc); _row = {"ranking": _nm, "auc": s17_auc(_sc), "distinct_values": int(len(np.unique(_sc)))}
    for _q in (.05, .10, .20, .30, .50):
        _row[f"capture_top_{int(_q * 100)}pct"] = float(np.interp(_q, _px, _py))
    _row["share_of_lens_capture_top20"] = _row["capture_top_20pct"] / float(np.interp(.2, s17_lens_px, s17_lens_py))
    _row["hit_rate_top_10pct"] = _row["capture_top_10pct"] * BASE_RATE / .10; _row["hit_rate_top_20pct"] = _row["capture_top_20pct"] * BASE_RATE / .20
    s17_rows.append(_row)
    for _q in s17_grid_q:
        s17_pts.append({"ranking": _nm, "top_share_of_clients": _q, "capture_of_defaulters": float(np.interp(_q, _px, _py))})
S17_CAPTURE = pd.DataFrame(s17_rows).sort_values("capture_top_20pct", ascending=False).reset_index(drop=True)
S17_CAPTURE_PTS = pd.concat([pd.DataFrame(s17_pts), pd.DataFrame({"ranking": "Perfect", "top_share_of_clients": s17_grid_q, "capture_of_defaulters": np.minimum(1, s17_grid_q / BASE_RATE)}),
                             pd.DataFrame({"ranking": "Random", "top_share_of_clients": s17_grid_q, "capture_of_defaulters": s17_grid_q})], ignore_index=True)
display(S17_CAPTURE.round(4)); export(S17_CAPTURE, "s17_capture_table"); export(S17_CAPTURE_PTS, "s17_capture_curves")
fig, axes = plt.subplots(1, 2, figsize=(20, 6.5))
_pal = sns.color_palette("tab10", len(S17_SCORES))
for (_nm, _sc), _col in zip(S17_SCORES.items(), _pal):
    _px, _py = s17_gain(_sc); _lens = _nm.startswith("Lens")
    axes[0].plot(_px, _py, color="k" if _nm == "Lens OOF: all six months" else _col, lw=2.6 if _lens else 1.4, ls="-" if not _nm.startswith("Lens OOF: Sep") else "--", marker="o" if len(_px) < 15 else None, ms=3, label=_nm)
axes[0].plot(s17_grid_q, np.minimum(1, s17_grid_q / BASE_RATE), color=C_NEU, ls=":", label="perfect"); axes[0].plot([0, 1], [0, 1], color=C_NEU, ls="--", lw=1, label="random")
for _q in (.1, .2, .3):
    axes[0].axvline(_q, color=C_NEU, lw=.5)
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].legend(fontsize=7.5, loc="lower right")
axes[0].set_xlabel("top share of clients (ranked by the quantity)"); axes[0].set_ylabel("share of all defaulters captured"); axes[0].set_title("Capture (gain) curves")
_t = S17_CAPTURE.set_index("ranking")[["capture_top_10pct", "capture_top_20pct", "capture_top_30pct"]].iloc[::-1]
_t.plot(kind="barh", ax=axes[1], color=["#fdbb84", "#fc8d59", "#d7301f"], width=.8)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_ylabel(""); axes[1].set_title("Share of defaulters in the top 10 / 20 / 30% of clients"); axes[1].tick_params(axis="y", labelsize=8)
fig.suptitle("17.8 How far do transparent rules go?", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s17_capture_curves"); plt.show()
s17_cp = S17_CAPTURE.set_index("ranking"); s17_simple = S17_CAPTURE[~S17_CAPTURE["ranking"].str.startswith("Lens")].iloc[0]
add_fact("17.8", "capture", f"The top 20% of clients by the full lens hold {s17_cp.loc['Lens OOF: all six months', 'capture_top_20pct']:.0%} of defaulters; the best transparent ranking ('{s17_simple['ranking']}') holds {s17_simple['capture_top_20pct']:.0%}, i.e. {s17_simple['share_of_lens_capture_top20']:.0%} of what the lens achieves, and September status alone {s17_cp.loc['Sep status (PAY_1)', 'capture_top_20pct']:.0%}", float(s17_simple["share_of_lens_capture_top20"]), "s17_capture_curves", 3)
add_fact("17.8", "capture", f"Even with all information, the top 10% of clients contain only {s17_cp.loc['Lens OOF: all six months', 'capture_top_10pct']:.0%} of defaulters (perfect would be {min(1, .1 / BASE_RATE):.0%}): more than half of defaults are not concentrated in any small, identifiable group", float(s17_cp.loc["Lens OOF: all six months", "capture_top_10pct"]), "s17_capture_curves", 2)

**Observations:**
- The full six-month model (the 'lens') puts 51.6% of defaulters in its top 20% of clients.
- A transparent sum of Sep status + Aug status + number of late months gets 50.1%, which is 97% of the lens; Sep status alone gets 47.5%.
- Even the lens puts only 31.9% of defaulters in its top 10% (a perfect ranking would hold 45%).
- Silent defaulters differ from clean non-defaulters only weakly (they spend and owe less: log AVG_SPEND −0.41 SD, where |SMD| above 0.2 = noticeable), and even the full model ranks poorly inside the zero-sign group (AUC 0.638 vs 0.788).
- Story use: 'the limits of warning signs' slide: some defaults give no warning at all.

### 17.9 Silent defaulters and false alarms: who defaults with no warning, and who looks terrible but pays?
`EW_GROUP` partitions clients by outcome × core-sign count: **Silent defaulter** (default, zero core signs), **Clean non-defaulter** (no default, zero core signs), **Loud defaulter** / **False alarm** (sign count at or above the level where the rate first passes 50%, with / without default), and the middle. Two nested "silent" definitions are reported: zero of the 22 core signs, and the looser *never late* (no delinquency sign at all).

In [ ]:
s17_nc = df["EW_N_SIGNS_CORE"].to_numpy(); s17_d = S17_Y == 1
df["EW_GROUP"] = np.select([s17_d & (s17_nc == 0), ~s17_d & (s17_nc == 0), s17_d & (s17_nc >= S17_K50), ~s17_d & (s17_nc >= S17_K50), s17_d],
                           ["Silent defaulter", "Clean non-defaulter", "Loud defaulter", "False alarm", "Mid-signal defaulter"], default="Mid-signal non-defaulter")
df["EW_SILENT_DEFAULTER"] = (s17_d & (s17_nc == 0)).astype(int)
df["EW_NEVER_LATE_DEFAULTER"] = (s17_d & ~S17_FAM_EVER["D"]).astype(int)
S17_GROUP_ORDER = ["Clean non-defaulter", "Silent defaulter", "Mid-signal non-defaulter", "Mid-signal defaulter", "False alarm", "Loud defaulter"]
s17_gt = df.groupby("EW_GROUP").agg(n=(TARGET, "size"), median_limit=("LIMIT_BAL", "median"), mean_age=("AGE", "mean"), female=("SEX", lambda s: (s == 2).mean()),
                                    grad_school=("EDUCATION", lambda s: (s == 1).mean()), avg_util=("AVG_UTIL", "mean"), median_bill_sep=("BILL_AMT1", "median"),
                                    median_pay_sep=("PAY_AMT1", "median"), inactive_months=("N_INACTIVE", "mean"), paidfull_months=("N_PAIDFULL", "mean"),
                                    revolve_months=("N_REVOLVE", "mean"), n_core_signs=("EW_N_SIGNS_CORE", "mean"), lens_oof=("EW_LENS_OOF", "mean")).reindex(S17_GROUP_ORDER)
s17_gt.insert(1, "share_of_clients", s17_gt["n"] / S17_N); s17_gt.insert(2, "share_of_defaulters", np.where(s17_gt.index.str.contains("non-def|False|Clean"), 0, s17_gt["n"] / s17_d.sum()))
display(s17_gt.round(3)); export(s17_gt.reset_index(), "s17_ew_groups")
s17_silent = (df["EW_GROUP"] == "Silent defaulter").to_numpy(); s17_zero = s17_nc == 0
print(f"zero core signs: {s17_zero.sum():,} clients ({s17_zero.mean():.1%}); default rate {S17_Y[s17_zero].mean():.2%}  -> silent defaulters: {s17_silent.sum():,} = {s17_silent.sum() / s17_d.sum():.1%} of all defaulters")
print(f"never late (no delinquency sign): {(~S17_FAM_EVER['D']).sum():,} clients; default rate {S17_Y[~S17_FAM_EVER['D']].mean():.2%} -> {df['EW_NEVER_LATE_DEFAULTER'].sum():,} defaulters = {df['EW_NEVER_LATE_DEFAULTER'].sum() / s17_d.sum():.1%} of all defaulters")
print(f"sign threshold for 'loud': >= {S17_K50} core signs")
print("September status of silent defaulters:"); print(df.loc[s17_silent, "STATUS_GRP_SEP"].value_counts(normalize=True).round(3).to_string())

**What distinguishes them?** Standardised mean differences (SMD = difference in means ÷ pooled SD; |SMD| > 0.2 is noticeable) for three contrasts: silent defaulters vs **clean non-defaulters** (same zero-sign profile, different outcome: is there *anything* that separates them?), silent vs **loud defaulters** (two kinds of defaulter), and **false alarms** vs loud defaulters (same alarms, different outcome).

In [ ]:
s17_feat = {"LIMIT_BAL": df["LIMIT_BAL"], "AGE": df["AGE"], "Female": (df["SEX"] == 2).astype(float), "Grad school": (df["EDUCATION"] == 1).astype(float),
            "University": (df["EDUCATION"] == 2).astype(float), "High school": (df["EDUCATION"] == 3).astype(float), "Married": (df["MARRIAGE"] == 1).astype(float),
            "AVG_UTIL": df["AVG_UTIL"], "UTIL_1 (Sep)": df["UTIL_1"], "UTIL_TREND": df["UTIL_TREND"], "log BILL_AMT1": np.log1p(df["BILL_AMT1"].clip(lower=0)),
            "log AVG_BILL": np.log1p(df["AVG_BILL"].clip(lower=0)), "log PAY_AMT1": np.log1p(df["PAY_AMT1"]), "log AVG_PAY": np.log1p(df["AVG_PAY"]),
            "PAY_TO_BILL_6M": df["PAY_TO_BILL_6M"], "AVG_PAY_RATIO": df["AVG_PAY_RATIO"], "log AVG_SPEND": np.log1p(df["AVG_SPEND"].clip(lower=0)),
            "N_INACTIVE": df["N_INACTIVE"], "N_PAIDFULL": df["N_PAIDFULL"], "N_REVOLVE": df["N_REVOLVE"], "PAY_1 (Sep status)": df["PAY_1"], "PAY_2 (Aug status)": df["PAY_2"],
            "N_ZERO_PAY": df["N_ZERO_PAY"], "NEG_BILL_ANY": df["NEG_BILL_ANY"], "N usage-shift signs": pd.Series(S17_EVER[:, ~S17_CORE].sum(1), index=df.index),
            "N core signs": df["EW_N_SIGNS_CORE"], "N core signs Aug/Sep": df["EW_N_SIGNS_RECENT"], "Lens OOF score": df["EW_LENS_OOF"]}
s17_grp = df["EW_GROUP"].to_numpy()
S17_CONTRASTS = [("Silent defaulter", "Clean non-defaulter"), ("Silent defaulter", "Loud defaulter"), ("False alarm", "Loud defaulter")]
s17_rows = []
for _a, _b in S17_CONTRASTS:
    for _nm, _v in s17_feat.items():
        _x = np.asarray(_v, float)
        s17_rows.append({"contrast": f"{_a} vs {_b}", "feature": _nm, "mean_a": np.nanmean(_x[s17_grp == _a]), "mean_b": np.nanmean(_x[s17_grp == _b]), "smd": s17_smd(_x[s17_grp == _a], _x[s17_grp == _b])})
S17_SMD = pd.DataFrame(s17_rows); export(S17_SMD, "s17_silent_smd")
fig, axes = plt.subplots(1, 3, figsize=(24, 8.5))
for ax, (_a, _b) in zip(axes, S17_CONTRASTS):
    _t = S17_SMD[S17_SMD["contrast"] == f"{_a} vs {_b}"].sort_values("smd")
    ax.hlines(_t["feature"], 0, _t["smd"], color=C_NEU); ax.scatter(_t["smd"], _t["feature"], c=np.where(_t["smd"].abs() >= .2, C_DEF, C_NEU), zorder=3)
    ax.axvline(0, color="k", lw=.8); ax.axvline(.2, ls=":", color=C_NEU); ax.axvline(-.2, ls=":", color=C_NEU)
    ax.set_title(f"{_a} (n={(s17_grp == _a).sum():,}) minus {_b} (n={(s17_grp == _b).sum():,})", fontsize=10); ax.set_xlabel("standardised mean difference"); ax.tick_params(axis="y", labelsize=8)
fig.suptitle("17.9 What distinguishes silent defaulters and false alarms?", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s17_silent_smd"); plt.show()
for _a, _b in S17_CONTRASTS:
    _t = S17_SMD[S17_SMD["contrast"] == f"{_a} vs {_b}"]
    print(f"--- {_a} vs {_b}: largest |SMD| ---"); print(_t.reindex(_t["smd"].abs().sort_values(ascending=False).index).head(8).round(3).to_string(index=False))
s17_c1 = S17_SMD[S17_SMD["contrast"] == "Silent defaulter vs Clean non-defaulter"]; s17_c1 = s17_c1[s17_c1["feature"] != "Lens OOF score"]; s17_c1b = s17_c1.reindex(s17_c1["smd"].abs().sort_values(ascending=False).index).iloc[0]
s17_auc_zero = s17_auc(df.loc[s17_zero, "EW_LENS_OOF"].to_numpy(), S17_Y[s17_zero])
print(f"lens AUC inside the zero-sign population: {s17_auc_zero:.3f} (vs {s17_auc(df['EW_LENS_OOF']):.3f} overall)")
add_fact("17.9", "silent defaulters", f"{s17_silent.sum():,} defaulters ({s17_silent.sum() / s17_d.sum():.0%} of all) fired none of the 22 core warning signs, and {df['EW_NEVER_LATE_DEFAULTER'].sum() / s17_d.sum():.0%} of defaulters were never late in six months; the zero-sign population still defaults at {S17_Y[s17_zero].mean():.1%}", float(s17_silent.sum() / s17_d.sum()), "s17_silent_smd", 3)
add_fact("17.9", "silent defaulters", f"Silent defaulters are nearly indistinguishable from clean non-defaulters: the largest standardised difference is {s17_c1b['feature']} at {s17_c1b['smd']:+.2f} SD, and the lens AUC inside the zero-sign population is only {s17_auc_zero:.3f}", float(s17_auc_zero), "s17_silent_smd", 3)

**Observations:**
- Silent defaulters differ from clean non-defaulters only weakly: they spend, owe and pay less (log AVG_SPEND −0.41 SD, log AVG_BILL −0.40 SD; |SMD| above 0.2 = noticeable).
- Inside the zero-sign group even the full model ranks poorly (AUC 0.638 vs 0.788 overall).
- False alarms vs loud defaulters: the main difference is a milder Sep status (−0.41 SD) and a slightly lower model score (−0.61 SD).
- Story use: background; supports 'low activity is the only hint of a silent defaulter'.

#### 17.9b Where do silent defaulters and false alarms live?
Share of each segment's **defaulters that were silent** (how blind the alarms are there) and share of each segment's **loud clients that did not default** (how noisy the alarms are there), by limit band, age band, education, sex and September status group.

In [ ]:
s17_rows = []
s17_loud = s17_nc >= S17_K50
for _seg in ["LIMIT_BAND", "AGE_BAND", "EDU_LABEL", "SEX_LABEL", "MAR_LABEL", "STATUS_GRP_SEP"]:
    for _lev, _g in df.groupby(_seg, observed=True):
        _ix = df.index.isin(_g.index) if False else (df[_seg].astype(str) == str(_lev)).to_numpy()
        _nd = int((_ix & s17_d).sum()); _nl = int((_ix & s17_loud).sum())
        s17_rows.append({"segment_var": _seg, "segment": str(_lev), "n": int(_ix.sum()), "default_rate": S17_Y[_ix].mean(), "defaulters": _nd,
                         "silent_share_of_defaulters": (_ix & s17_silent).sum() / _nd if _nd >= 30 else np.nan,
                         "zero_sign_share_of_clients": (_ix & s17_zero).mean() / _ix.mean(), "zero_sign_default_rate": S17_Y[_ix & s17_zero].mean() if (_ix & s17_zero).sum() >= 30 else np.nan,
                         "loud_clients": _nl, "false_alarm_share_of_loud": 1 - S17_Y[_ix & s17_loud].mean() if _nl >= 30 else np.nan})
S17_BLIND = pd.DataFrame(s17_rows); display(S17_BLIND.round(3)); export(S17_BLIND, "s17_silent_by_segment")
fig, axes = plt.subplots(1, 2, figsize=(21, 7))
_t = S17_BLIND.dropna(subset=["silent_share_of_defaulters"]); _lab = _t["segment_var"] + ": " + _t["segment"]
axes[0].barh(_lab, _t["silent_share_of_defaulters"], color=C_OK, alpha=.8); axes[0].axvline(s17_silent.sum() / s17_d.sum(), ls="--", color="k", lw=1)
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].set_title("Blind spots: share of the segment's defaulters with ZERO core signs"); axes[0].tick_params(axis="y", labelsize=8); axes[0].invert_yaxis()
_t = S17_BLIND.dropna(subset=["false_alarm_share_of_loud"]); _lab = _t["segment_var"] + ": " + _t["segment"]
axes[1].barh(_lab, _t["false_alarm_share_of_loud"], color=C_DEF, alpha=.8); axes[1].axvline(1 - S17_Y[s17_loud].mean(), ls="--", color="k", lw=1)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title(f"Noise: share of the segment's loud clients (>= {S17_K50} core signs) who did NOT default"); axes[1].tick_params(axis="y", labelsize=8); axes[1].invert_yaxis()
plt.tight_layout(); savefig("s17_silent_by_segment"); plt.show()
s17_fa_n = int((df["EW_GROUP"] == "False alarm").sum()); s17_bs = S17_BLIND.dropna(subset=["silent_share_of_defaulters"]); s17_bs = s17_bs[s17_bs["segment_var"] != "STATUS_GRP_SEP"].sort_values("silent_share_of_defaulters").iloc[-1]
s17_fa3 = S17_SMD[(S17_SMD["contrast"] == "False alarm vs Loud defaulter") & (S17_SMD["feature"] != "Lens OOF score")]; s17_fa3 = s17_fa3.reindex(s17_fa3["smd"].abs().sort_values(ascending=False).index).iloc[0]
add_fact("17.9", "false alarms", f"{s17_fa_n:,} clients fired {S17_K50}+ core signs and still did not default ({1 - S17_Y[s17_loud].mean():.0%} of all loud clients); what separates them most from loud defaulters is {s17_fa3['feature']} ({s17_fa3['smd']:+.2f} SD)", float(1 - S17_Y[s17_loud].mean()), "s17_silent_by_segment", 2)
add_fact("17.9", "silent defaulters", f"The alarms are blindest in '{s17_bs['segment_var']} = {s17_bs['segment']}', where {s17_bs['silent_share_of_defaulters']:.0%} of defaulters gave no core warning sign (overall {s17_silent.sum() / s17_d.sum():.0%})", float(s17_bs["silent_share_of_defaulters"]), "s17_silent_by_segment", 2)

**Observations:**
- Alarms are blindest for big limits: 19.6% of defaulters with 300-500k limits were silent vs 2.7% at ≤50k.
- By Sep status, 63.8% of defaulters who were Inactive in Sep and 26.4% of those Paid in full gave no core sign.
- False alarms are about half of loud clients in every limit band (42-52%), and 69% among Revolving-in-Sep loud clients.
- Story use: segment callout: 'wealthy, quiet clients slip through the alarms'.

### 17.10 False comfort: does a recent improvement really reduce risk?
A collector sees a client who was late in August and looks better in September, or who has just made a big payment. Is that client now "safe"? Each comfort signal is compared with (i) the same clients **without** the improvement and (ii) clients with the same September state who were **never in trouble** — the gap to (ii) is the *scar* that the improvement does not erase.

(a) Clients late in August (status ≥ 2) by their September move; (b) the same September status with vs without an August delay.

In [ ]:
s17_aug, s17_sepst = S17_S[:, 4], S17_S[:, 5]
s17_late_aug = s17_aug >= 2
s17_move = np.select([s17_sepst <= 0, s17_sepst < s17_aug, s17_sepst == s17_aug], ["Cured (Sep <= 0)", "Improved, still late", "Unchanged"], default="Worse")
S17_MOVE = pd.DataFrame({"move": s17_move[s17_late_aug], "y": S17_Y[s17_late_aug]}).groupby("move")["y"].agg(n="size", defaults="sum", rate="mean").reindex(["Cured (Sep <= 0)", "Improved, still late", "Unchanged", "Worse"]).reset_index()
S17_MOVE["ci_lo"], S17_MOVE["ci_hi"] = wilson_ci(S17_MOVE["defaults"], S17_MOVE["n"]); S17_MOVE["share"] = S17_MOVE["n"] / S17_MOVE["n"].sum()
s17_never = ~(S17_S >= 1).any(1); S17_RATE_NEVER = float(S17_Y[s17_never].mean())
s17_rows = []
for _lev, _nm in ((-2, "Inactive (-2)"), (-1, "Paid in full (-1)"), (0, "Revolving (0)"), (1, "Late 1m (1)"), (2, "Late 2m (2)")):
    for _lab, _mk in (("late in Aug (>= 2)", s17_late_aug), ("not late in Aug, late earlier (Apr-Jul)", ~s17_late_aug & (S17_S[:, :4] >= 1).any(1)), ("never late before Sep", ~(S17_S[:, :5] >= 1).any(1))):
        _g = (s17_sepst == _lev) & _mk
        if _g.sum() >= 30:
            _lo, _hi = wilson_ci(S17_Y[_g].sum(), _g.sum())
            s17_rows.append({"sep_status": _nm, "history": _lab, "n": int(_g.sum()), "rate": S17_Y[_g].mean(), "ci_lo": float(_lo), "ci_hi": float(_hi)})
S17_SCAR = pd.DataFrame(s17_rows)
fig, axes = plt.subplots(1, 2, figsize=(20, 5.5))
axes[0].bar(S17_MOVE["move"], S17_MOVE["rate"], yerr=[S17_MOVE["rate"] - S17_MOVE["ci_lo"], S17_MOVE["ci_hi"] - S17_MOVE["rate"]], color=[C_ACC, "#fdae6b", "#fd8d3c", C_DEF], capsize=4)
for _x, (_r, _n) in enumerate(zip(S17_MOVE["rate"], S17_MOVE["n"])):
    axes[0].text(_x, _r + .035, f"{_r:.0%}\nn={_n:,}", ha="center", fontsize=9)
axes[0].axhline(BASE_RATE, ls="--", color="k", lw=1, label=f"base rate {BASE_RATE:.0%}"); axes[0].axhline(S17_RATE_NEVER, ls=":", color=C_OK, label=f"never-late clients {S17_RATE_NEVER:.0%}")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].set_ylim(0, 1); axes[0].legend(fontsize=8); axes[0].set_title("(a) Clients late in August (status >= 2): default rate by September move")
sns.barplot(data=S17_SCAR, x="sep_status", y="rate", hue="history", ax=axes[1], palette=[C_DEF, "#fd8d3c", C_OK])
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[1].set_xlabel(""); axes[1].legend(fontsize=8)
axes[1].set_title("(b) Same September status, different history: the scar")
plt.tight_layout(); savefig("s17_false_comfort_status"); plt.show()
display(S17_MOVE.round(4)); display(S17_SCAR.round(4)); export(S17_MOVE, "s17_comfort_aug_sep_move"); export(S17_SCAR, "s17_comfort_scar")
s17_mv = S17_MOVE.set_index("move"); s17_sc = S17_SCAR.set_index(["sep_status", "history"])["rate"]
s17_clean_ref = float(S17_Y[(s17_sepst <= 0) & ~(S17_S[:, :5] >= 1).any(1)].mean())
add_fact("17.10", "false comfort", f"A September cure is only partial comfort: clients late in August who are clean in September still default at {s17_mv.loc['Cured (Sep <= 0)', 'rate']:.0%}, vs {s17_clean_ref:.0%} for clean-in-September clients with no earlier delay ({s17_mv.loc['Cured (Sep <= 0)', 'rate'] / s17_clean_ref:.1f}x) and {s17_mv.loc['Unchanged', 'rate']:.0%} for those who stayed at the same delay", float(s17_mv.loc["Cured (Sep <= 0)", "rate"]), "s17_false_comfort_status", 3)

**Observations:**
- Of 4,410 clients late in Aug, those clean in Sep still default at 38.0%; improved but still late 43.6%; unchanged 71.7%.
- A past delay leaves a scar even with a clean Sep: Revolving in Sep with an earlier delay 24.8% vs 10.7% never late; Paid in full in Sep but late in Aug 37.8% vs 13.8%.
- Story use: 'a September cure is only partial comfort' (3.2× the never-late rate).

#### 17.10b Does a big September payment buy safety?
Among clients with a positive August bill: default rate by the share of the August bill repaid in September, separately for clients late in August, late earlier only, and never late. Plus the "payment spike" (September payment ≥ 2× the Apr–Aug average and ≥ NT$5,000).

In [ ]:
s17_prb = pd.cut(pd.Series(S17_PR[:, 5]), [-np.inf, 0, 0.05, 0.10, 0.25, 0.50, 0.999, np.inf], labels=["0", "0-5%", "5-10%", "10-25%", "25-50%", "50-100%", "100%+"])
s17_hist = np.select([s17_late_aug, (S17_S[:, :4] >= 1).any(1)], ["late in Aug (>= 2)", "late earlier only (Apr-Jul)"], default="never late before Sep")
s17_pb = pd.DataFrame({"sep_pay_ratio": s17_prb.to_numpy(), "history": s17_hist, TARGET: S17_Y})[~np.isnan(S17_PR[:, 5])]
s17_pb["sep_pay_ratio"] = pd.Categorical(s17_pb["sep_pay_ratio"], ["0", "0-5%", "5-10%", "10-25%", "25-50%", "50-100%", "100%+"], ordered=True)
S17_PAYCOMF = rate_table(s17_pb, ["history", "sep_pay_ratio"], min_n=30)
s17_avg_prior = S17_P[:, :5].mean(1); s17_spike = (S17_P[:, 5] >= 2 * s17_avg_prior) & (S17_P[:, 5] >= 5000)
s17_rows = []
for _lab, _mk in (("late in Aug (>= 2)", s17_late_aug), ("late earlier only (Apr-Jul)", ~s17_late_aug & (S17_S[:, :4] >= 1).any(1)), ("never late before Sep", ~(S17_S[:, :5] >= 1).any(1)), ("late in Sep (>= 1)", s17_sepst >= 1)):
    for _sp, _g in (("payment spike in Sep", _mk & s17_spike), ("no spike", _mk & ~s17_spike)):
        _lo, _hi = wilson_ci(S17_Y[_g].sum(), _g.sum())
        s17_rows.append({"population": _lab, "signal": _sp, "n": int(_g.sum()), "rate": S17_Y[_g].mean(), "ci_lo": float(_lo), "ci_hi": float(_hi)})
S17_SPIKE = pd.DataFrame(s17_rows)
fig, axes = plt.subplots(1, 2, figsize=(20, 5.5))
for _h, _col in (("late in Aug (>= 2)", C_DEF), ("late earlier only (Apr-Jul)", "#fd8d3c"), ("never late before Sep", C_OK)):
    _t = S17_PAYCOMF[S17_PAYCOMF["history"] == _h]
    axes[0].errorbar(_t["sep_pay_ratio"].astype(str), _t["rate"], yerr=[_t["rate"] - _t["ci_lo"], _t["ci_hi"] - _t["rate"]], marker="o", capsize=3, color=_col, label=_h)
axes[0].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].legend(fontsize=8)
axes[0].set_xlabel("September payment as share of the August bill"); axes[0].set_title("Default rate by size of the September payment, by delinquency history")
sns.barplot(data=S17_SPIKE, x="population", y="rate", hue="signal", ax=axes[1], palette=[C_ACC, C_NEU])
for _p, _r in zip(axes[1].patches, list(S17_SPIKE[S17_SPIKE["signal"] == "payment spike in Sep"].itertuples()) + list(S17_SPIKE[S17_SPIKE["signal"] == "no spike"].itertuples())):
    axes[1].text(_p.get_x() + _p.get_width() / 2, _p.get_height() + .01, f"{_r.rate:.0%}\nn={_r.n:,}", ha="center", fontsize=8)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[1].set_xlabel(""); axes[1].tick_params(axis="x", labelsize=8)
axes[1].set_title("Payment spike (Sep payment >= 2x Apr-Aug mean and >= NT$5,000)")
plt.tight_layout(); savefig("s17_false_comfort_payment"); plt.show()
display(S17_PAYCOMF[["history", "sep_pay_ratio", "n", "rate", "ci_lo", "ci_hi", "lift"]].round(4)); display(S17_SPIKE.round(4))
export(S17_PAYCOMF, "s17_comfort_payment_size"); export(S17_SPIKE, "s17_comfort_payment_spike")
s17_pc = S17_PAYCOMF.set_index(["history", "sep_pay_ratio"])["rate"]; s17_sk = S17_SPIKE.set_index(["population", "signal"])["rate"]
s17_pr_sep = S17_PR[:, 5]
def s17_rate_of(mask):
    mask = mask & ~np.isnan(s17_pr_sep)
    return (float(S17_Y[mask].mean()) if mask.sum() >= 20 else np.nan), int(mask.sum())
s17_full_late, s17_n_full_late = s17_rate_of(s17_late_aug & (s17_pr_sep >= 0.5))
s17_zero_late, _ = s17_rate_of(s17_late_aug & (s17_pr_sep <= 0))
s17_full_never, _ = s17_rate_of(~(S17_S[:, :5] >= 1).any(1) & (s17_pr_sep >= 0.999))
add_fact("17.10", "false comfort", f"A big September payment does not reset a client who was late in August: those who repaid half or more of the August bill still default at {s17_full_late:.0%} (n={s17_n_full_late:,}), vs {s17_zero_late:.0%} if they paid nothing and {s17_full_never:.0%} for full payers never late before", float(s17_full_late), "s17_false_comfort_payment", 3)
s17_sp_no, s17_sp_yes = s17_sk[("late in Aug (>= 2)", "no spike")], s17_sk[("late in Aug (>= 2)", "payment spike in Sep")]
add_fact("17.10", "false comfort", f"Among clients late in August, a September payment spike goes with {s17_sp_yes:.0%} default vs {s17_sp_no:.0%} without one ({'lower' if s17_sp_yes < s17_sp_no else 'higher'}); either way it is {s17_sp_yes / S17_RATE_NEVER:.1f}x the never-late rate", float(s17_sp_yes), "s17_false_comfort_payment", 2)

**Observations:**
- For clients late in Aug, paying more of the bill in Sep does not lower risk: 52.6% if they paid nothing, 55.8-65.7% if they paid 0-25%, 51.2% at 50-100% (only 41 clients).
- A Sep payment spike (2× the usual payment) even goes with higher default for Aug-late clients: 69.4% vs 55.4% without.
- For never-late clients a spike is good news (9.7% vs 14.8%), and payment size matters only mildly (14.2% at 0-5% vs 10.4% at 50-100%).
- Story use: 'a big payment does not reset a late client', a strong false-comfort callout.

#### 17.10c How long does the scar last? Clean months since the last delay, by how bad the episode was
For clients with at least one late month who are clean at the end: default rate by the number of **consecutive clean months ending in September**, split by the worst status of the earlier episode and by its length. The dashed line is the never-late rate — the level a fully "healed" client would reach. (Section 5.11 shows recency alone; here the question is whether a *severe* or *long* episode heals more slowly.) Closing table: every comfort signal next to its comparators.

In [ ]:
s17_ever = (S17_S >= 1).any(1); s17_clean_run = np.where(s17_ever, (S17_S[:, ::-1] >= 1).argmax(1), -1)       # consecutive clean months ending Sep
s17_sev = np.where(S17_S.max(1) >= 3, "worst 3+ months", "worst 1-2 months"); s17_len = np.where((S17_S >= 1).sum(1) >= 2, "2+ late months", "1 late month")
s17_rows = []
for _dim, _arr in (("episode severity", s17_sev), ("episode length", s17_len), ("all", np.full(S17_N, "all ever-late"))):
    for _lev in pd.unique(_arr):
        for _c in range(0, 6):
            _g = s17_ever & (s17_clean_run == _c) & (_arr == _lev)
            if _g.sum() >= 40:
                _lo, _hi = wilson_ci(S17_Y[_g].sum(), _g.sum())
                s17_rows.append({"split": _dim, "group": _lev, "clean_months_ending_sep": _c, "n": int(_g.sum()), "rate": S17_Y[_g].mean(), "ci_lo": float(_lo), "ci_hi": float(_hi),
                                 "ratio_to_never_late": S17_Y[_g].mean() / S17_RATE_NEVER})
S17_HEAL = pd.DataFrame(s17_rows)
fig, axes = plt.subplots(1, 2, figsize=(18, 5.2), sharey=True)
for ax, _dim in zip(axes, ("episode severity", "episode length")):
    for _lev, _col in zip(sorted(S17_HEAL.loc[S17_HEAL["split"] == _dim, "group"].unique()), (C_OK, C_DEF)):
        _t = S17_HEAL[(S17_HEAL["split"] == _dim) & (S17_HEAL["group"] == _lev)]
        ax.errorbar(_t["clean_months_ending_sep"], _t["rate"], yerr=[_t["rate"] - _t["ci_lo"], _t["ci_hi"] - _t["rate"]], marker="o", capsize=3, color=_col, label=_lev)
    _t = S17_HEAL[S17_HEAL["split"] == "all"]; ax.plot(_t["clean_months_ending_sep"], _t["rate"], color="k", ls=":", label="all ever-late")
    ax.axhline(S17_RATE_NEVER, ls="--", color=C_ACC, label=f"never late ({S17_RATE_NEVER:.0%})"); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.legend(fontsize=8)
    ax.set_xlabel("consecutive clean months ending in September (0 = late in Sep)"); ax.set_title(f"Healing curve by {_dim}")
plt.tight_layout(); savefig("s17_healing_curve"); plt.show()
display(S17_HEAL.round(4)); export(S17_HEAL, "s17_healing_curve")

s17_cured = s17_late_aug & (s17_sepst <= 0); s17_bigpay = s17_late_aug & (S17_PR[:, 5] >= 0.999); s17_impr = (s17_sepst < s17_aug) & (s17_aug >= 1)
def s17_cf(signal, with_mk, without_mk, ref_mk, ref_label):
    return {"comfort_signal": signal, "n_with": int(with_mk.sum()), "rate_with": S17_Y[with_mk].mean(), "n_without": int(without_mk.sum()), "rate_without": S17_Y[without_mk].mean(),
            "reference": ref_label, "rate_reference": S17_Y[ref_mk].mean(), "risk_reduction_pp": 100 * (S17_Y[without_mk].mean() - S17_Y[with_mk].mean()),
            "residual_excess_pp": 100 * (S17_Y[with_mk].mean() - S17_Y[ref_mk].mean()), "ratio_to_reference": S17_Y[with_mk].mean() / S17_Y[ref_mk].mean()}
s17_ref_clean = (s17_sepst <= 0) & ~(S17_S[:, :5] >= 1).any(1)
S17_COMFORT = pd.DataFrame([
    s17_cf("Late in Aug, clean in Sep (cured)", s17_cured, s17_late_aug & (s17_sepst >= 1), s17_ref_clean, "clean in Sep, never late before"),
    s17_cf("Status improved Aug -> Sep (any amount, from late)", s17_impr, (s17_aug >= 1) & ~s17_impr, s17_ref_clean, "clean in Sep, never late before"),
    s17_cf("Late in Aug, paid 100%+ of Aug bill in Sep", s17_bigpay, s17_late_aug & (S17_PR[:, 5] < 0.999), (S17_PR[:, 5] >= 0.999) & ~(S17_S[:, :5] >= 1).any(1), "paid 100%+ in Sep, never late before"),
    s17_cf("Late in Aug, payment spike in Sep", s17_late_aug & s17_spike, s17_late_aug & ~s17_spike, s17_spike & ~(S17_S[:, :5] >= 1).any(1), "payment spike, never late before"),
    s17_cf("Ever late, 3+ clean months to Sep", s17_ever & (s17_clean_run >= 3), s17_ever & (s17_clean_run < 3), s17_never, "never late in 6 months"),
    s17_cf("Ever late, 1-2 clean months to Sep", s17_ever & (s17_clean_run >= 1) & (s17_clean_run <= 2), s17_ever & (s17_clean_run == 0), s17_never, "never late in 6 months"),
])
display(S17_COMFORT.round(4)); export(S17_COMFORT, "s17_false_comfort")
s17_h = S17_HEAL[S17_HEAL["split"] == "all"].set_index("clean_months_ending_sep")
add_fact("17.10", "false comfort", f"The scar fades slowly: ever-late clients default at {s17_h.loc[0, 'rate']:.0%} when late in September, {s17_h.loc[1, 'rate']:.0%} after one clean month and {s17_h.loc[int(s17_h.index.max()), 'rate']:.0%} after {int(s17_h.index.max())} clean months, still {s17_h.loc[int(s17_h.index.max()), 'ratio_to_never_late']:.1f}x the never-late rate of {S17_RATE_NEVER:.0%}", float(s17_h.loc[1, "rate"]), "s17_healing_curve", 2)

**Observations:**
- The scar fades slowly: ever-late clients default at 50.3% when late in Sep, 38.0% after one clean month, 28.7% after two and 18.7% after five, still 1.6× the 11.7% never-late rate.
- Bad episodes scar longer: after a 3+ month delay, two clean months still mean 38.8% default vs 28.0% after a 1-2 month delay.
- Of the comfort signals, 3+ clean months cuts risk most (46.6% → 22.5%), but still 10.8 points above never-late.
- Story use: 'healing curve' line chart (`s17_healing_curve.csv`) for the closing early-warning slide.

### 17.11 Section 17 wrap-up: new columns, exports and story facts

| New `df` column | Meaning |
|---|---|
| `EW_LENS_OOF` | out-of-fold lens score using all six months (descriptive yardstick only) |
| `EW_UTIL_APR_BAND` | April utilisation band (KM segmenting variable) |
| `EW_ENTRY_SEVERITY_APR`, `EW_ENTRY_SEVERITY_MAY` | delay depth in April / May (cure cohorts) |
| `EW_SIGN_<NAME>` (26 columns) | 1 if the warning sign ever fired in Apr–Sep |
| `EW_N_SIGNS`, `EW_N_SIGNS_CORE`, `EW_N_SIGNS_RECENT`, `EW_N_SIGNS_CAPPED` | signs fired: all 26 / 22 core / core active in Aug–Sep / core capped at 12 |
| `EW_N_FAMILIES`, `EW_FAMILY_COMBO`, `EW_FAMILY_PATH` | base families fired (count, names, order of first appearance) |
| `EW_FIRST_SIGN_MONTH`, `EW_LEAD_TIME` | month of the first alarm; months before September |
| `EW_GROUP`, `EW_SILENT_DEFAULTER`, `EW_NEVER_LATE_DEFAULTER` | silent / loud / false-alarm partition and flags |

**Tableau mapping.** `s17_thresholds` → reference lines on any distribution or rate chart; `s17_rate_curves` + `s17_tipping_points` → rate curve with fitted segments and knot annotations; `s17_km_curves` → step-line survival sheets (filter by `analysis`, `segment_var`); `s17_warning_signs` → capture-vs-hit-rate bubble chart; `s17_sign_count_curve` → bar + line; `s17_capture_curves` → gain chart; `s17_info_horizon_lens` → "months of history" line; `s17_false_comfort`, `s17_healing_curve` → comparison bars.

In [ ]:
s17_newcols = [c for c in df.columns if c.startswith("EW_")]
print(f"{len(s17_newcols)} EW_ columns added; section runtime {time.time() - S17_T0:.0f}s")
export(df[["ID", TARGET, "EW_LENS_OOF", "EW_N_SIGNS", "EW_N_SIGNS_CORE", "EW_N_SIGNS_RECENT", "EW_N_FAMILIES", "EW_FAMILY_COMBO", "EW_FAMILY_PATH",
           "EW_FIRST_SIGN_MONTH", "EW_LEAD_TIME", "EW_GROUP"] + [c for c in s17_newcols if c.startswith("EW_SIGN_")]], "s17_client_early_warning")
S17_FACTS = pd.DataFrame([f for f in STORY_FACTS if f["section"].startswith("17")])
with pd.option_context("display.max_colwidth", None):
    display(S17_FACTS)

**Observations:**
- The section adds 42 `EW_` columns, exports the client-level file `s17_client_early_warning.csv` (30,000 × 38) and registers 33 story facts (11 at strength 3).
- Runtime is 153 s.
- Story use: background only.

## 18. Exposure economics: where is the money at risk, and what would a policy change do?

**Why this section matters for the story.** Earlier sections count *clients*. A lender counts *money*. A defaulter with a NT$2,000 balance and one with NT$400,000 are the same "1" in a default rate but very different losses. This section asks where the New Taiwan dollars (NT$) sit, which segments lose money and which earn it under clearly labelled assumptions, and what simple credit-line rules would have done. Section 6.6 and 6.9 already showed segment shares and a first expected-loss view. Here we go sideways: frequency vs severity, money-first vs risk-first ranking, a revenue side, policy simulations and undrawn headroom.

Everything is **descriptive arithmetic on the observed file**, not a model. New `df` columns carry the prefix `EX_`; section globals use `S18_` / `s18_`.

### 18.0 Assumptions (read before any NT$ number)
| Item | Base value | Range tested | Why / caveat |
|---|---|---|---|
| Exposure (drawn balance) | September bill `BILL_AMT1`, clipped at 0 | - | Latest statement before the October default flag. Credit balances count as 0. |
| Exposure at default (EAD) | Sep exposure of clients with `DEFAULT = 1` | + CCF x undrawn headroom | No post-September draws are observed. |
| Undrawn headroom | `max(LIMIT_BAL - exposure, 0)` | - | Over-limit clients have 0 headroom. |
| Loss given default (LGD) | **80%** | 45% - 100% | Unsecured card debt. 45% is the Basel F-IRB senior unsecured benchmark, 100% is the upper bound. **Assumption.** |
| Revolving APR | **18%** | 10% - 20% | Taiwan card rates were capped near 20% at the time. **Illustrative assumption.** |
| Interest-bearing balance | unpaid part of the previous bill, `max(BILL_AMT(i+1) - PAY_AMTi, 0)`, only in months with status >= 0 | - | Months with status -1 (paid in full) or -2 (inactive) earn no interest (grace period). |
| Interchange | **1.5%** of positive implied spend `SPEND_i` | 1% - 2% | Inactive months (status -2) earn nothing. Spend includes interest and fees, so it is a slight overstatement. **Assumption.** |
| Cost of funds | **2%** of the average positive balance | 1% - 3% | Operating costs, fees and late charges are ignored. |
| Annualisation | 5 observed months (May-Sep) scaled by 12/5 | - | Revenue is a run-rate. Loss is the single October default event. The P&L is a scale indicator, not an accounting result. |
| Draw before default (policy CCF) | **50%** of the headroom a rule removes | 25% - 100% | The share of the removed line a defaulter would have drawn before charge-off. **Assumption.** |

Customer type (Sep status): **Inactive** (-2), **Transactor** (-1, pays in full), **Revolver** (0, minimum paid), **Delinquent** (>= 1).

In [ ]:
S18_T0 = time.time()
S18_A = {"APR": 0.18, "IX": 0.015, "LGD": 0.80, "COF": 0.02, "CCF": 0.50, "INT_HAIRCUT": 0.25}
S18_B = df[BILL_COLS].to_numpy(float)      # col 0 = Sep ... col 5 = Apr
S18_P = df[PAMT_COLS].to_numpy(float)
S18_S = df[PAY_COLS].to_numpy(int)
S18_L = df["LIMIT_BAL"].to_numpy(float)
S18_Y = df[TARGET].to_numpy(int)
S18_N = len(df)

df["EX_EXPOSURE"] = np.clip(S18_B[:, 0], 0, None)
df["EX_HEADROOM"] = np.clip(S18_L - df["EX_EXPOSURE"].to_numpy(), 0, None)
df["EX_HEADROOM_PCT"] = df["EX_HEADROOM"] / S18_L
df["EX_EAD"] = df["EX_EXPOSURE"] * S18_Y
S18_EXP = df["EX_EXPOSURE"].to_numpy()
S18_HR = df["EX_HEADROOM"].to_numpy()

# revenue building blocks (rate = 100%)
s18_carry = np.clip(S18_B[:, 1:] - S18_P[:, :5], 0, None)           # unpaid part of the previous statement, months Sep..May
s18_spend = S18_B[:, :5] - S18_B[:, 1:] + S18_P[:, :5]               # implied new charges, months Sep..May
S18_INT1 = (s18_carry * (S18_S[:, :5] >= 0)).mean(1)                  # avg interest-bearing balance; x APR = annual interest
S18_IX1 = (np.clip(s18_spend, 0, None) * (S18_S[:, :5] != -2)).mean(1) * 12   # annualised purchase volume; x rate = interchange
S18_BAL = np.clip(S18_B, 0, None).mean(1)                             # avg drawn balance; x cost of funds

def s18_econ(apr=None, ix=None, lgd=None, cof=None, ccf=0.0):
    apr = S18_A["APR"] if apr is None else apr
    ix = S18_A["IX"] if ix is None else ix
    lgd = S18_A["LGD"] if lgd is None else lgd
    cof = S18_A["COF"] if cof is None else cof
    interest = apr * S18_INT1
    inter = ix * S18_IX1
    fund = cof * S18_BAL
    loss = S18_Y * lgd * (S18_EXP + ccf * S18_HR)
    return {"interest": interest, "interchange": inter, "funding": fund, "loss": loss, "profit": interest + inter - fund - loss}

S18_E = s18_econ()
df["EX_INT_REV"] = S18_E["interest"]
df["EX_IX_REV"] = S18_E["interchange"]
df["EX_REVENUE"] = df["EX_INT_REV"] + df["EX_IX_REV"]
df["EX_FUND_COST"] = S18_E["funding"]
df["EX_LOSS"] = S18_E["loss"]
df["EX_PROFIT"] = S18_E["profit"]
S18_REV = df["EX_REVENUE"].to_numpy()
S18_INTREV = df["EX_INT_REV"].to_numpy()
S18_IXREV = df["EX_IX_REV"].to_numpy()

S18_TYPE_ORDER = ["Inactive", "Transactor", "Revolver", "Delinquent"]
s18_s1 = S18_S[:, 0]
df["EX_CUST_TYPE"] = pd.Categorical(np.where(s18_s1 <= -2, "Inactive", np.where(s18_s1 == -1, "Transactor", np.where(s18_s1 == 0, "Revolver", "Delinquent"))), S18_TYPE_ORDER)
s18_cnt = np.column_stack([(S18_S <= -2).sum(1), (S18_S == -1).sum(1), (S18_S == 0).sum(1), (S18_S >= 1).sum(1)])
df["EX_CUST_TYPE_6M"] = pd.Categorical(np.array(S18_TYPE_ORDER)[3 - np.argmax(s18_cnt[:, ::-1], 1)], S18_TYPE_ORDER)   # ties -> riskier type

s18_sum = pd.Series({
    "clients": S18_N, "defaulters": int(S18_Y.sum()),
    "total credit limit (NT$m)": S18_L.sum() / 1e6,
    "Sep drawn exposure (NT$m)": S18_EXP.sum() / 1e6,
    "undrawn headroom (NT$m)": S18_HR.sum() / 1e6,
    "EAD = exposure of defaulters (NT$m)": df["EX_EAD"].sum() / 1e6,
    "credit loss at LGD 80% (NT$m)": df["EX_LOSS"].sum() / 1e6,
    "interest revenue, annual (NT$m)": df["EX_INT_REV"].sum() / 1e6,
    "interchange revenue, annual (NT$m)": df["EX_IX_REV"].sum() / 1e6,
    "cost of funds, annual (NT$m)": df["EX_FUND_COST"].sum() / 1e6,
    "profit proxy (NT$m)": df["EX_PROFIT"].sum() / 1e6,
    "clients over limit in Sep": int((S18_EXP > S18_L).sum()),
    "clients with zero / credit Sep balance": int((S18_EXP == 0).sum()),
})
display(s18_sum.to_frame("value").round(2))

**Observations:**
- The book has NT$5,025m of credit lines. Only NT$1,537m (31%) is drawn in September; NT$3,513m is undrawn headroom.
- Defaulters' September balances (EAD) total NT$322m. At LGD 80% that is a NT$258m credit loss.
- Under the base assumptions, revenue is NT$244m a year (interest NT$204m, interchange NT$40m), cost of funds is NT$27m, and the profit proxy is **-NT$41m**.
- 2,115 clients (7%) are over their limit in September and 2,598 (9%) have a zero or credit balance.
- Story use: the money frame. One October default event costs more than a year of the book's assumed revenue.

### 18.1 Where is the money?
Five questions. How much of the line do defaulters hold? Which segments hold more NT$ than their defaulter headcount? Do defaulters carry bigger balances than their peers? How concentrated are defaulters' balances? And if we work clients from riskiest to safest, do we reach the money as fast as we reach the defaulters?

#### 18.1.1 How is the NT$ credit line split between drawn and undrawn, and between defaulters and payers?
Left: the total credit line split into drawn balance and headroom, for payers and defaulters. Right: the defaulters' share of clients, limit, drawn balance, headroom and revenue.

In [ ]:
s18_g = df.groupby("DEFAULT_LABEL").agg(n=(TARGET, "size"), limit=("LIMIT_BAL", "sum"), drawn=("EX_EXPOSURE", "sum"),
                                         headroom=("EX_HEADROOM", "sum"), revenue=("EX_REVENUE", "sum"), loss=("EX_LOSS", "sum")).reindex(["No default", "Default"])
s18_over = float(np.clip(S18_EXP - S18_L, 0, None).sum())
fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
s18_parts = [("drawn, payers", s18_g.loc["No default", "drawn"], "#6baed6"), ("drawn, defaulters", s18_g.loc["Default", "drawn"], C_DEF),
             ("headroom, payers", s18_g.loc["No default", "headroom"], "#c6dbef"), ("headroom, defaulters", s18_g.loc["Default", "headroom"], "#fcbba1")]
s18_left = 0
for lab, v, col in s18_parts:
    axes[0].barh([0], [v / 1e9], left=s18_left / 1e9, color=col, label=f"{lab}: NT${v/1e9:.2f}bn")
    s18_left += v
axes[0].set_yticks([]); axes[0].set_xlabel("NT$ billion"); axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, fontsize=8)
axes[0].set_title(f"Credit line NT${S18_L.sum()/1e9:.2f}bn (+ NT${s18_over/1e6:.0f}m drawn above limits)")
s18_sh = (s18_g.loc["Default"] / s18_g.sum())[["n", "limit", "drawn", "headroom", "revenue"]]
axes[1].bar(["clients", "credit limit", "drawn (Sep)", "headroom", "revenue"], s18_sh.values, color=[C_NEU, C_ACC, C_DEF, "#fcbba1", C_OK])
axes[1].axhline(BASE_RATE, ls="--", color="k", lw=1)
for i, v in enumerate(s18_sh.values):
    axes[1].text(i, v, f"{v:.1%}", ha="center", va="bottom")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("Defaulters' share of each NT$ pool (dashed = share of clients)")
plt.tight_layout(); savefig("s18_line_split"); plt.show()
s18_g["util_of_limit"] = s18_g["drawn"] / s18_g["limit"]
display((s18_g / [1, 1e6, 1e6, 1e6, 1e6, 1e6, 1]).round(3).rename(columns=lambda c: c if c in ("n", "util_of_limit") else c + "_NT$m"))
add_fact("18.1", "exposure", f"Defaulters are {s18_sh['n']:.1%} of clients but hold {s18_sh['drawn']:.1%} of the September drawn balance and only {s18_sh['limit']:.1%} of the credit line; they use {s18_g.loc['Default', 'util_of_limit']:.0%} of their lines vs {s18_g.loc['No default', 'util_of_limit']:.0%} for payers", float(s18_sh["drawn"]), "s18_line_split", 3)

**Observations:**
- Defaulters are 22.1% of clients but hold only 17.2% of the credit line and 20.9% of the drawn balance. Their share of money is *below* their headcount.
- They draw harder on their smaller lines: 37% utilisation vs 29% for payers.
- Defaulters still hold NT$548m of headroom, 1.7x their drawn balance (NT$322m).
- Story use: defaulters are small-line, high-utilisation clients. The NT$ story is not the headcount story.

#### 18.1.2 Which segments hold more NT$-at-risk than their share of defaulters suggests?
Each dot is a segment from one of seven dimensions. x = share of all defaulters, y = share of all EAD (NT$). Above the diagonal: the segment's defaulters are richer than average, so the money at risk is bigger than the headcount. `ead_tilt` = share of EAD / share of defaulters. *Tableau: scatter with a reference diagonal, colour = dimension.*

In [ ]:
S18_DIMS = {"Sep status": ("STATUS_GRP_SEP", STATUS_GROUP_ORDER), "Limit band": ("LIMIT_BAND", LIMIT_LABELS),
            "Avg util band": ("UTIL_BAND", list(df["UTIL_BAND"].cat.categories)), "Age band": ("AGE_BAND", AGE_LABELS),
            "Education": ("EDU_LABEL", EDU_ORDER), "Sex": ("SEX_LABEL", ["Male", "Female"]), "Customer type": ("EX_CUST_TYPE", S18_TYPE_ORDER)}
def s18_seg(col, order=None, dim=None, data=None):
    d = df if data is None else data
    g = d.groupby(col, observed=True).agg(n=(TARGET, "size"), defaulters=(TARGET, "sum"), limit=("LIMIT_BAL", "sum"),
                                          exposure=("EX_EXPOSURE", "sum"), ead=("EX_EAD", "sum"), headroom=("EX_HEADROOM", "sum"),
                                          interest=("EX_INT_REV", "sum"), interchange=("EX_IX_REV", "sum"),
                                          funding=("EX_FUND_COST", "sum"), loss=("EX_LOSS", "sum"), profit=("EX_PROFIT", "sum"))
    if order is not None:
        g = g.reindex([o for o in order if o in set(g.index)])
    g.index = g.index.astype(str); g.index.name = "segment"
    g = g.reset_index(); g.insert(0, "dimension", dim or col)
    g["default_rate"] = g["defaulters"] / g["n"]
    g["share_clients"] = g["n"] / S18_N
    g["share_defaulters"] = g["defaulters"] / S18_Y.sum()
    g["share_exposure"] = g["exposure"] / S18_EXP.sum()
    g["share_ead"] = g["ead"] / df["EX_EAD"].sum()
    g["ead_tilt"] = g["share_ead"] / g["share_defaulters"].replace(0, np.nan)
    g["loss_rate_on_exposure"] = g["ead"] / g["exposure"].replace(0, np.nan)
    g["ead_per_defaulter"] = g["ead"] / g["defaulters"].replace(0, np.nan)
    g["revenue"] = g["interest"] + g["interchange"]
    g["revenue_per_client"] = g["revenue"] / g["n"]
    g["cost_per_client"] = (g["loss"] + g["funding"]) / g["n"]
    g["loss_per_client"] = g["loss"] / g["n"]
    g["profit_per_client"] = g["profit"] / g["n"]
    return g
S18_SEG = pd.concat([s18_seg(c, o, k) for k, (c, o) in S18_DIMS.items()], ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5), gridspec_kw={"width_ratios": [1.1, 1]})
s18_pal = dict(zip(S18_DIMS, sns.color_palette("tab10", len(S18_DIMS))))
for dname, sub in S18_SEG[S18_SEG.dimension != "Customer type"].groupby("dimension"):
    axes[0].scatter(sub["share_defaulters"], sub["share_ead"], s=30 + 400 * sub["share_clients"], color=s18_pal[dname], alpha=.75, label=dname, edgecolor="k", lw=.4)
    for _, r in sub.iterrows():
        if r["share_defaulters"] > 0.03 and abs(np.log(r["ead_tilt"])) > 0.25:
            axes[0].annotate(f"{r['segment']}", (r["share_defaulters"], r["share_ead"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
s18_mx = float(S18_SEG[["share_defaulters", "share_ead"]].max().max()) * 1.05
axes[0].plot([0, s18_mx], [0, s18_mx], ls="--", color="k", lw=1)
axes[0].set_xlabel("share of all defaulters"); axes[0].set_ylabel("share of all EAD (NT$)")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].set_title("Headcount at risk vs money at risk, by segment (size = share of clients)"); axes[0].legend(fontsize=8)
s18_t = S18_SEG[(S18_SEG.dimension != "Customer type") & (S18_SEG.share_defaulters >= 0.01)].copy()
s18_t["label"] = s18_t["dimension"] + ": " + s18_t["segment"]
s18_t = s18_t.sort_values("ead_tilt")
axes[1].barh(s18_t["label"], s18_t["ead_tilt"], color=np.where(s18_t["ead_tilt"] > 1, C_DEF, C_OK))
axes[1].axvline(1, color="k", lw=1); axes[1].tick_params(axis="y", labelsize=7)
axes[1].set_xlabel("EAD tilt = share of NT$ at risk / share of defaulters"); axes[1].set_title("NT$ tilt by segment (segments with >= 1% of defaulters)")
plt.tight_layout(); savefig("s18_money_map"); plt.show()
display(S18_SEG[["dimension", "segment", "n", "default_rate", "share_clients", "share_defaulters", "share_exposure", "share_ead", "ead_tilt", "ead_per_defaulter"]].round(3))
s18_tt = s18_t[s18_t["segment"] != "<=0"]
s18_hi, s18_lo = s18_tt.iloc[-1], s18_tt.iloc[0]
add_fact("18.1", "exposure", f"Money at risk and headcount at risk diverge: '{s18_hi['label']}' holds {s18_hi['share_ead']:.1%} of NT$ EAD with {s18_hi['share_defaulters']:.1%} of defaulters (tilt {s18_hi['ead_tilt']:.2f}); '{s18_lo['label']}' holds {s18_lo['share_defaulters']:.1%} of defaulters but only {s18_lo['share_ead']:.1%} of EAD (tilt {s18_lo['ead_tilt']:.2f})", float(s18_hi["ead_tilt"]), "s18_money_map", 3)

**Observations:**
- Utilisation drives the NT$ tilt. Clients above 100% average utilisation hold 7.2% of EAD with 3.3% of defaulters (tilt 2.19). The 0-10% band holds 25.8% of defaulters but only 2.4% of EAD (tilt 0.09).
- Limit band tilts strongly too. The tilt rises from 0.44 (<=50k) to 2.01 (300-500k) and 6.3 (>500k, only 23 defaulters).
- By September status, "Revolving" defaulters (tilt 1.49) and "Late 2m" defaulters (1.34) carry the money. Inactive and paid-in-full defaulters (tilt 0.15) are many but nearly empty.
- Story use: "who defaults" and "where the money goes" are different maps. Tableau should show both side by side.

#### 18.1.3 Inside each segment, do defaulters carry bigger balances than their non-defaulting peers?
NT$ at risk = frequency x severity. The scatter above mixes both. Here we hold the segment fixed and compare the **mean September balance of defaulters (red) vs non-defaulters (blue)**. If defaulters are "maxed out", the red dot sits right of the blue one. If they are small-balance clients, it sits left.

In [ ]:
s18_rows = []
for dname, (c, o) in S18_DIMS.items():
    g = df.groupby([c, TARGET], observed=True)["EX_EXPOSURE"].agg(["mean", "median", "size"]).unstack(TARGET)
    g = g.reindex([x for x in o if x in set(g.index)])
    for seg_, r in g.iterrows():
        s18_rows.append({"dimension": dname, "segment": str(seg_), "mean_bal_payers": r[("mean", 0)], "mean_bal_defaulters": r[("mean", 1)],
                         "median_bal_payers": r[("median", 0)], "median_bal_defaulters": r[("median", 1)], "n_defaulters": r[("size", 1)]})
S18_SEV = pd.DataFrame(s18_rows)
S18_SEV["severity_ratio"] = S18_SEV["mean_bal_defaulters"] / S18_SEV["mean_bal_payers"]
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, dname in zip(axes.ravel(), S18_DIMS):
    s = S18_SEV[S18_SEV.dimension == dname].reset_index(drop=True)
    y = np.arange(len(s))
    ax.hlines(y, s["mean_bal_payers"] / 1e3, s["mean_bal_defaulters"] / 1e3, color=C_NEU, lw=2)
    ax.scatter(s["mean_bal_payers"] / 1e3, y, color=C_OK, s=40, zorder=3, label="non-defaulters")
    ax.scatter(s["mean_bal_defaulters"] / 1e3, y, color=C_DEF, s=40, zorder=3, label="defaulters")
    ax.set_yticks(y); ax.set_yticklabels(s["segment"], fontsize=8); ax.invert_yaxis()
    ax.set_title(dname); ax.set_xlabel("mean Sep balance, NT$ thousand")
axes[0, 0].legend(fontsize=8)
s18_allm = df.groupby(TARGET)["EX_EXPOSURE"].mean()
axes[1, 3].bar(["non-defaulters", "defaulters"], s18_allm.values / 1e3, color=[C_OK, C_DEF]); axes[1, 3].set_title("All clients"); axes[1, 3].set_ylabel("NT$ thousand")
plt.suptitle("Severity: mean September balance of defaulters vs non-defaulters within each segment", fontweight="bold")
plt.tight_layout(); savefig("s18_severity_dumbbell"); plt.show()
display(S18_SEV.round(2))
s18_sv = S18_SEV[S18_SEV.n_defaulters >= 50].sort_values("severity_ratio")
print(f"All clients: defaulters' mean balance NT${s18_allm[1]:,.0f} vs NT${s18_allm[0]:,.0f} (ratio {s18_allm[1]/s18_allm[0]:.2f}); "
      f"segments with defaulters richer than peers: {(s18_sv.severity_ratio > 1).sum()} of {len(s18_sv)}")
add_fact("18.1", "severity", f"Across the portfolio a defaulter's mean Sep balance is NT${s18_allm[1]:,.0f} vs NT${s18_allm[0]:,.0f} for payers (x{s18_allm[1]/s18_allm[0]:.2f}); within segments the ratio ranges from {s18_sv.severity_ratio.min():.2f} ({s18_sv.iloc[0]['dimension']}: {s18_sv.iloc[0]['segment']}) to {s18_sv.severity_ratio.max():.2f} ({s18_sv.iloc[-1]['dimension']}: {s18_sv.iloc[-1]['segment']})", float(s18_allm[1] / s18_allm[0]), "s18_severity_dumbbell", 2)

**Observations:**
- On average defaulters do **not** owe more. Their mean September balance is NT$48,516 vs NT$52,021 for payers (ratio 0.93).
- Within utilisation bands, defaulters owe less than their peers (ratio 0.56 in 0-10%, 0.77-0.94 at 50-100%+). Within limit bands they owe more, and the gap grows with the limit (0.92 at <=50k, 1.25 at 300-500k).
- Late-1m defaulters owe 1.20x their late-1m peers. Revolving defaulters owe 0.91x.
- Only 11 of 37 segments (with 50+ defaulters) show defaulters richer than their peers.
- Story use: default is not a "maxed-out big balance" event. Severity comes from the limit, not from the act of defaulting.

#### 18.1.4 How concentrated is the defaulters' money?
Left: Lorenz-style curve **within defaulters** (largest balance first). What share of all EAD do the top x% of defaulters hold? The same curve for payers is shown for reference. Right: rank **all** clients by balance and track how much of the defaulters' money sits in the top x% of clients. The table answers the brief directly.

In [ ]:
def s18_gini_desc(v):
    v = np.sort(np.asarray(v, float))
    cum = np.concatenate([[0], np.cumsum(v)]) / max(v.sum(), 1e-9)
    x = np.linspace(0, 1, len(cum))
    return 1 - 2 * float(((cum[1:] + cum[:-1]) / 2 * np.diff(x)).sum())
s18_d = np.sort(S18_EXP[S18_Y == 1])[::-1]; s18_nd = np.sort(S18_EXP[S18_Y == 0])[::-1]
s18_cd = np.cumsum(s18_d) / s18_d.sum(); s18_cnd = np.cumsum(s18_nd) / s18_nd.sum()
s18_o = np.argsort(-S18_EXP, kind="stable")
s18_cead = np.cumsum(df["EX_EAD"].to_numpy()[s18_o]) / df["EX_EAD"].sum()
s18_cdef = np.cumsum(S18_Y[s18_o]) / S18_Y.sum()
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2))
axes[0].plot(np.arange(1, len(s18_d) + 1) / len(s18_d), s18_cd, color=C_DEF, lw=2, label=f"defaulters (Gini {s18_gini_desc(s18_d):.2f})")
axes[0].plot(np.arange(1, len(s18_nd) + 1) / len(s18_nd), s18_cnd, color=C_OK, lw=2, label=f"non-defaulters (Gini {s18_gini_desc(s18_nd):.2f})")
axes[0].plot([0, 1], [0, 1], ls="--", color="k", lw=1); axes[0].legend()
axes[0].set_xlabel("top x% of the group by Sep balance"); axes[0].set_ylabel("cumulative share of the group's balance")
axes[0].set_title("Concentration within each group (largest balance first)")
s18_xr = np.arange(1, S18_N + 1) / S18_N
axes[1].plot(s18_xr, s18_cead, color=C_DEF, lw=2, label="share of defaulters' NT$ (EAD)")
axes[1].plot(s18_xr, s18_cdef, color="#fdae6b", lw=2, label="share of defaulters (count)")
axes[1].plot([0, 1], [0, 1], ls="--", color="k", lw=1); axes[1].legend()
axes[1].set_xlabel("top x% of ALL clients by Sep balance"); axes[1].set_title("Where the defaulters' money sits in the whole book")
for ax in axes:
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
plt.tight_layout(); savefig("s18_ead_concentration"); plt.show()
s18_rows = []
for q in [0.01, 0.05, 0.10, 0.20, 0.50]:
    kd, ka = int(round(q * len(s18_d))), int(round(q * S18_N))
    s18_rows.append({"top_x": f"{q:.0%}", "share_EAD_top_x_of_defaulters": s18_cd[kd - 1], "share_balance_top_x_of_payers": s18_cnd[int(round(q * len(s18_nd))) - 1],
                     "share_EAD_top_x_of_all_clients": s18_cead[ka - 1], "share_defaulters_top_x_of_all_clients": s18_cdef[ka - 1],
                     "min_balance_in_top_x_defaulters_NT$": s18_d[kd - 1]})
S18_CONC = pd.DataFrame(s18_rows)
display(S18_CONC.round(3))
print(f"Defaulters with zero or credit Sep balance: {(s18_d == 0).sum():,} ({(s18_d == 0).mean():.1%}); median defaulter balance NT${np.median(s18_d):,.0f}")
add_fact("18.1", "concentration", f"The top 10% of defaulters by balance hold {S18_CONC.loc[2, 'share_EAD_top_x_of_defaulters']:.0%} of all NT$ at default and the top 20% hold {S18_CONC.loc[3, 'share_EAD_top_x_of_defaulters']:.0%}; the top 10% of all clients by balance hold {S18_CONC.loc[2, 'share_EAD_top_x_of_all_clients']:.0%} of EAD but only {S18_CONC.loc[2, 'share_defaulters_top_x_of_all_clients']:.0%} of defaulters", float(S18_CONC.loc[3, "share_EAD_top_x_of_defaulters"]), "s18_ead_concentration", 3)

**Observations:**
- Defaulters' money is concentrated. The top 10% of defaulters by balance hold 47% of EAD, and the top 20% hold 68% (Gini similar to payers').
- 9.7% of defaulters (643) have a zero or credit balance. The median defaulter owes NT$20,185. Entering the top 10% takes at least NT$132k.
- The top 10% of *all* clients by balance hold 44% of EAD but only 9% of defaulters.
- Story use: a few hundred large-balance defaulters carry half of the money at risk.

#### 18.1.5 Risk-first or money-first: if collections works down a list, what does it cover?
Two simple orderings, no model. **Risk-first**: sort by September status, then number of late months, then September utilisation (all descending). **Money-first**: sort by September balance. For the top x% of the list we track the share of defaulters, of NT$ EAD and of revenue touched. A good list covers defaulters and EAD fast and revenue slowly.

In [ ]:
s18_risk_order = np.lexsort((-df["UTIL_1"].to_numpy(), -df["N_LATE"].to_numpy(), -S18_S[:, 0]))
s18_money_order = np.argsort(-S18_EXP, kind="stable")
s18_rows = []
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2), sharey=True)
for ax, (oname, order) in zip(axes, [("Risk-first (status, late months, util)", s18_risk_order), ("Money-first (Sep balance)", s18_money_order)]):
    cdef = np.cumsum(S18_Y[order]) / S18_Y.sum()
    cead = np.cumsum(df["EX_EAD"].to_numpy()[order]) / df["EX_EAD"].sum()
    crev = np.cumsum(S18_REV[order]) / S18_REV.sum()
    for v, lab, col in [(cdef, "defaulters", "#fdae6b"), (cead, "NT$ EAD", C_DEF), (crev, "revenue", C_OK)]:
        ax.plot(s18_xr, v, lw=2, color=col, label=lab)
    ax.plot([0, 1], [0, 1], ls="--", color="k", lw=1); ax.set_title(oname); ax.legend()
    ax.set_xlabel("top x% of clients worked"); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    for q in [0.05, 0.10, 0.20, 0.30]:
        k = int(round(q * S18_N)) - 1
        s18_rows.append({"ordering": oname, "top_x": q, "share_defaulters": cdef[k], "share_EAD": cead[k], "share_revenue": crev[k]})
axes[0].set_ylabel("cumulative share covered")
plt.tight_layout(); savefig("s18_capture_risk_vs_money"); plt.show()
S18_CAPT = pd.DataFrame(s18_rows)
display(S18_CAPT.round(3))
s18_r10 = S18_CAPT[(S18_CAPT.top_x == 0.10)].set_index("ordering")
add_fact("18.1", "targeting", f"Working the riskiest 10% of clients (by Sep status) reaches {s18_r10.iloc[0]['share_defaulters']:.0%} of defaulters and {s18_r10.iloc[0]['share_EAD']:.0%} of NT$ EAD; working the largest 10% of balances reaches {s18_r10.iloc[1]['share_defaulters']:.0%} of defaulters and {s18_r10.iloc[1]['share_EAD']:.0%} of EAD", float(s18_r10.iloc[0]["share_EAD"]), "s18_capture_risk_vs_money", 2)

**Observations:**
- Risk-first ordering (Sep status, then late months, then utilisation): the top 10% reach 32% of defaulters and 43% of EAD but touch only 14% of revenue.
- Money-first ordering (by balance): the top 10% reach 44% of EAD but only 9% of defaulters, and touch 43% of revenue.
- At 30% coverage, risk-first reaches 60% of defaulters and 63% of EAD for 29% of revenue. Money-first reaches 77% of EAD but 77% of revenue too.
- Story use: a status-ordered list reaches the money almost as fast as a balance list and spares the revenue.

### 18.2 Expected loss: do the segments with the most defaulters also lose the most money?
Loss = `DEFAULT x LGD x Sep exposure` (LGD 80%). This is the *realised* loss proxy. Its expectation inside a segment equals the default rate x mean exposure x LGD, so it is also the expected-loss view. LGD only scales the totals and never changes rankings.

#### 18.2.1 Rank by defaulters vs rank by NT$ loss (Sep status x limit band cells, n >= 50)
Slopegraph: left = rank by number of defaulters, right = rank by NT$ loss. Red lines climb: the cell loses more money than its headcount suggests. Blue lines fall.

In [ ]:
s18_sl = df.groupby(["STATUS_GRP_SEP", "LIMIT_BAND"], observed=True).agg(n=(TARGET, "size"), defaulters=(TARGET, "sum"),
                                                                          exposure=("EX_EXPOSURE", "sum"), loss=("EX_LOSS", "sum")).reset_index()
s18_sl = s18_sl[s18_sl.n >= 50].copy()
s18_sl["cell"] = s18_sl["STATUS_GRP_SEP"].astype(str) + " | " + s18_sl["LIMIT_BAND"].astype(str)
s18_sl["rank_defaulters"] = s18_sl["defaulters"].rank(ascending=False, method="first").astype(int)
s18_sl["rank_loss"] = s18_sl["loss"].rank(ascending=False, method="first").astype(int)
s18_sl["default_rate"] = s18_sl["defaulters"] / s18_sl["n"]
s18_sl["loss_per_defaulter"] = s18_sl["loss"] / s18_sl["defaulters"].replace(0, np.nan)
s18_sl["share_defaulters"] = s18_sl["defaulters"] / S18_Y.sum(); s18_sl["share_loss"] = s18_sl["loss"] / df["EX_LOSS"].sum()
s18_top = s18_sl[(s18_sl.rank_defaulters <= 12) | (s18_sl.rank_loss <= 12)]
fig, ax = plt.subplots(figsize=(11, 8))
for _, r in s18_top.iterrows():
    col = C_DEF if r["rank_loss"] < r["rank_defaulters"] else (C_OK if r["rank_loss"] > r["rank_defaulters"] else C_NEU)
    ax.plot([0, 1], [r["rank_defaulters"], r["rank_loss"]], color=col, lw=1.8, marker="o")
    ax.text(-0.03, r["rank_defaulters"], f"{r['cell']} ({int(r['defaulters'])})", ha="right", va="center", fontsize=8)
    ax.text(1.03, r["rank_loss"], f"{r['cell']} (NT${r['loss']/1e6:.0f}m)", ha="left", va="center", fontsize=8)
ax.invert_yaxis(); ax.set_xlim(-0.6, 1.6); ax.set_xticks([0, 1]); ax.set_xticklabels(["rank by # defaulters", "rank by NT$ loss"])
ax.set_ylabel("rank (1 = largest)"); ax.set_title("Most defaulters vs most money lost: Sep status | limit band cells"); ax.grid(False)
plt.tight_layout(); savefig("s18_rank_slopegraph"); plt.show()
s18_cols = ["cell", "n", "default_rate", "defaulters", "share_defaulters", "loss", "share_loss", "loss_per_defaulter", "rank_defaulters", "rank_loss"]
print("Top 10 by defaulters:"); display(s18_sl.sort_values("rank_defaulters").head(10)[s18_cols].round(3))
print("Top 10 by NT$ loss:"); display(s18_sl.sort_values("rank_loss").head(10)[s18_cols].round(3))
s18_ov = len(set(s18_sl.nsmallest(10, "rank_defaulters")["cell"]) & set(s18_sl.nsmallest(10, "rank_loss")["cell"]))
s18_rho = stats.spearmanr(s18_sl["rank_defaulters"], s18_sl["rank_loss"])[0]
print(f"Overlap of the two top-10 lists: {s18_ov}/10 | Spearman rho of ranks: {s18_rho:.2f} | cells: {len(s18_sl)}")
s18_l1, s18_d1 = s18_sl.sort_values("rank_loss").iloc[0], s18_sl.sort_values("rank_defaulters").iloc[0]
add_fact("18.2", "expected loss", f"The cell with most defaulters is '{s18_d1['cell']}' ({int(s18_d1['defaulters']):,}, {s18_d1['share_defaulters']:.0%} of all), but the biggest NT$ loss is '{s18_l1['cell']}' (NT${s18_l1['loss']/1e6:.0f}m, {s18_l1['share_loss']:.0%} of loss); only {s18_ov} of the top-10 cells by headcount are also top-10 by money", float(s18_l1["share_loss"]), "s18_rank_slopegraph", 3)

**Observations:**
- The cell with most defaulters is Revolving | <=50k (807, 12% of all). It ranks only 6th on NT$ loss.
- The biggest NT$ losses are Revolving | 300-500k (NT$22m, 9%) and Revolving | 200-300k (NT$21m). These are low-rate cells (7-8% default) with large balances (NT$176k and NT$124k lost per defaulter).
- Only 5 of the top-10 cells by headcount are also top-10 by money (Spearman rho of ranks 0.61).
- Late 2m | 50-100k is in both lists: 455 defaulters at 72% and NT$20m.
- Story use: the low-risk, high-limit revolvers are the largest single source of NT$ loss.

#### 18.2.2 Frequency vs intensity: average utilisation band x limit band
Three views of the same grid (cells n >= 30): count default rate, **NT$ loss per client**, and **loss rate on drawn NT$** (EAD / exposure). The frequency map and the money maps peak in different places. *Tableau: three highlight tables with a shared row and column order.*

In [ ]:
s18_ul = df.groupby(["UTIL_BAND", "LIMIT_BAND"], observed=True).agg(n=(TARGET, "size"), rate=(TARGET, "mean"), loss=("EX_LOSS", "sum"),
                                                                     ead=("EX_EAD", "sum"), exposure=("EX_EXPOSURE", "sum")).reset_index()
s18_ul["loss_per_client"] = s18_ul["loss"] / s18_ul["n"]
s18_ul["loss_rate_on_exposure"] = s18_ul["ead"] / s18_ul["exposure"].replace(0, np.nan)
s18_uo = list(df["UTIL_BAND"].cat.categories)
def s18_piv(frame, v, idx, col, io, co):
    return frame.pivot(index=idx, columns=col, values=v).reindex(index=io, columns=co)
s18_n = s18_piv(s18_ul, "n", "UTIL_BAND", "LIMIT_BAND", s18_uo, LIMIT_LABELS)
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, (v, ttl, fmt, cm) in zip(axes, [("rate", "Default rate", lambda x: f"{x:.0%}", CMAP_RATE),
                                       ("loss_per_client", "NT$ loss per client (thousand)", lambda x: f"{x/1e3:.1f}", "Reds"),
                                       ("loss_rate_on_exposure", "Loss rate on drawn NT$ (EAD / exposure)", lambda x: f"{x:.0%}", "Purples")]):
    m = s18_piv(s18_ul, v, "UTIL_BAND", "LIMIT_BAND", s18_uo, LIMIT_LABELS).where(s18_n >= 30)
    ann = m.apply(lambda col: col.map(lambda x: "" if pd.isna(x) else fmt(x)))
    sns.heatmap(m.astype(float), annot=ann, fmt="", cmap=cm, ax=ax, annot_kws={"fontsize": 8}, cbar=True)
    ax.set_title(ttl); ax.set_xlabel("limit band"); ax.set_ylabel("average utilisation band" if v == "rate" else "")
plt.tight_layout(); savefig("s18_freq_vs_intensity"); plt.show()
s18_ulv = s18_ul[(s18_ul.n >= 30) & (s18_ul["UTIL_BAND"].astype(str) != "<=0")]   # <=0 = credit balances, no money at risk
s18_a, s18_b = s18_ulv.sort_values("rate").iloc[-1], s18_ulv.sort_values("loss_per_client").iloc[-1]
display(s18_ulv.sort_values("loss_per_client", ascending=False).head(10).round(3))
print(f"Highest default rate: util {s18_a['UTIL_BAND']} | limit {s18_a['LIMIT_BAND']} ({s18_a['rate']:.0%}, NT${s18_a['loss_per_client']:,.0f}/client); "
      f"highest loss per client: util {s18_b['UTIL_BAND']} | limit {s18_b['LIMIT_BAND']} ({s18_b['rate']:.0%}, NT${s18_b['loss_per_client']:,.0f}/client)")
add_fact("18.2", "expected loss", f"Frequency and money peak in different cells: the highest default rate is util {s18_a['UTIL_BAND']} x limit {s18_a['LIMIT_BAND']} ({s18_a['rate']:.0%}, NT${s18_a['loss_per_client']:,.0f} lost per client), the highest loss per client is util {s18_b['UTIL_BAND']} x limit {s18_b['LIMIT_BAND']} (NT${s18_b['loss_per_client']:,.0f} per client at {s18_b['rate']:.0%} default)", float(s18_b["loss_per_client"]), "s18_freq_vs_intensity", 2)

**Observations:**
- Default frequency peaks at low limits and very high utilisation: util >100% x limit 50-100k defaults at 39%.
- Loss per client peaks elsewhere: util 90-100% x limit 300-500k loses NT$110k per client at 36% default (92 clients).
- Loss per client rises with both utilisation and limit. The loss rate on drawn NT$ follows default frequency closely (36% in the top cell).
- Story use: a two-axis highlight table (utilisation x limit) shows how frequency and money separate.

### 18.3 The revenue side: who pays for the losses?
A card book earns **interest** from revolvers and **interchange** from spenders. Under the 18.0 assumptions: revolvers (status 0) and late clients accrue interest on the unpaid balance, transactors (status -1) earn interchange only, inactive months (-2) earn nothing. **All NT$ revenue below is an assumption-driven illustration**, useful for direction and ranking, not for absolute profit.

#### 18.3.1 What do the four customer types look like in NT$?
September type vs the dominant type over six months, with balances, interest-bearing balance, purchase volume, revenue and default.

In [ ]:
s18_ct = df.groupby("EX_CUST_TYPE", observed=True).agg(n=(TARGET, "size"), default_rate=(TARGET, "mean"), avg_limit=("LIMIT_BAL", "mean"),
                                                       avg_sep_balance=("EX_EXPOSURE", "mean"), avg_headroom=("EX_HEADROOM", "mean"),
                                                       revenue=("EX_REVENUE", "sum"), loss=("EX_LOSS", "sum"), profit=("EX_PROFIT", "sum"))
s18_ct["share_clients"] = s18_ct["n"] / S18_N
s18_ct["avg_interest_bearing_bal"] = pd.Series(S18_INT1).groupby(df["EX_CUST_TYPE"].astype(str).values).mean().reindex(s18_ct.index.astype(str)).values
s18_ct["avg_annual_purchases"] = pd.Series(S18_IX1).groupby(df["EX_CUST_TYPE"].astype(str).values).mean().reindex(s18_ct.index.astype(str)).values
s18_ct["share_revenue"] = s18_ct["revenue"] / s18_ct["revenue"].sum()
s18_ct["share_loss"] = s18_ct["loss"] / s18_ct["loss"].sum()
display(s18_ct.round(3))
print("Sep type (rows) vs dominant six-month type (columns):")
display(pd.crosstab(df["EX_CUST_TYPE"], df["EX_CUST_TYPE_6M"], margins=True))
print("Default rate by six-month type:"); display(rate_table(df, "EX_CUST_TYPE_6M")[["EX_CUST_TYPE_6M", "n", "rate", "share_clients"]].round(3))
add_fact("18.3", "revenue", f"Revolvers are {s18_ct.loc['Revolver', 'share_clients']:.0%} of clients and generate {s18_ct.loc['Revolver', 'share_revenue']:.0%} of assumed revenue; transactors are {s18_ct.loc['Transactor', 'share_clients']:.0%} of clients but earn only {s18_ct.loc['Transactor', 'share_revenue']:.0%}; delinquent clients carry {s18_ct.loc['Delinquent', 'share_loss']:.0%} of NT$ credit loss", float(s18_ct.loc["Revolver", "share_revenue"]), None, 2)

**Observations:**
- Revolvers are 49% of clients and earn 74% of assumed revenue. Transactors are 19% of clients and earn 4%. Inactive clients earn 0.5%.
- Delinquent clients (23% of clients, 50% default) carry 55% of credit loss but still generate 21% of revenue.
- Types are sticky. 94% of September revolvers and 77% of transactors are the same type over six months. But 1,021 September-delinquent clients were mostly inactive before.
- By six-month type, the dominant delinquents default at 59% and every other type at 16-19%.
- Story use: the revolver is the economic engine, and the delinquent consumes it.

#### 18.3.2 Per-client P&L by customer type: does anyone besides the revolver pay their way?
Bars per client per year: interest and interchange (up), cost of funds and credit loss (down), net (black dot). Left: September type. Right: dominant six-month type.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2), sharey=True)
s18_pl = {}
for ax, col in zip(axes, ["EX_CUST_TYPE", "EX_CUST_TYPE_6M"]):
    p = df.groupby(col, observed=True)[["EX_INT_REV", "EX_IX_REV", "EX_FUND_COST", "EX_LOSS", "EX_PROFIT"]].mean().reindex(S18_TYPE_ORDER)
    s18_pl[col] = p
    x = np.arange(len(p))
    ax.bar(x, p["EX_INT_REV"], color=C_OK, label="interest")
    ax.bar(x, p["EX_IX_REV"], bottom=p["EX_INT_REV"], color="#9ecae1", label="interchange")
    ax.bar(x, -p["EX_FUND_COST"], color=C_NEU, label="cost of funds")
    ax.bar(x, -p["EX_LOSS"], bottom=-p["EX_FUND_COST"], color=C_DEF, label="credit loss")
    ax.scatter(x, p["EX_PROFIT"], color="k", zorder=4, s=60, label="net")
    for xi, v in zip(x, p["EX_PROFIT"]):
        ax.text(xi + 0.12, v, f"NT${v:,.0f}", fontsize=8, va="center")
    ax.axhline(0, color="k", lw=.8); ax.set_xticks(x); ax.set_xticklabels(S18_TYPE_ORDER)
    ax.set_title(f"Annual P&L per client by {'Sep' if col == 'EX_CUST_TYPE' else 'dominant 6-month'} type")
axes[0].set_ylabel("NT$ per client per year"); axes[0].legend(fontsize=8)
plt.tight_layout(); savefig("s18_pnl_per_client_type"); plt.show()
for k, v in s18_pl.items():
    print(k); display(v.round(0))
s18_pp = s18_pl["EX_CUST_TYPE"]["EX_PROFIT"]
add_fact("18.3", "profitability", f"Under the base assumptions a September revolver nets NT${s18_pp['Revolver']:,.0f} a year, a transactor NT${s18_pp['Transactor']:,.0f}, an inactive client NT${s18_pp['Inactive']:,.0f}, and a delinquent client NT${s18_pp['Delinquent']:,.0f}", float(s18_pp["Delinquent"]), "s18_pnl_per_client_type", 3)

**Observations:**
- Per client per year: a revolver nets NT$3,555, a transactor NT$633, an inactive client -NT$518, and a delinquent client -NT$14,017.
- A delinquent client earns NT$6,633 interest, close to a revolver's NT$10,450, but loses NT$20,623 to default.
- By dominant six-month type, a revolver nets only NT$1,424, because loss per revolver rises to NT$9,422.
- Even inactive clients lose money: no revenue, plus a 13% default rate on small balances.

#### 18.3.3 Portfolio P&L waterfall: does the book make money at all?
Interest + interchange - cost of funds - credit loss = profit proxy, in NT$ million per year, with the delinquent clients' contribution split out.

In [ ]:
s18_tot = {k: float(v.sum()) / 1e6 for k, v in S18_E.items()}
s18_steps = [("interest", s18_tot["interest"]), ("interchange", s18_tot["interchange"]), ("cost of funds", -s18_tot["funding"]), ("credit loss", -s18_tot["loss"])]
fig, axes = plt.subplots(1, 2, figsize=(16, 5), gridspec_kw={"width_ratios": [1.2, 1]})
s18_c = 0
for i, (lab, v) in enumerate(s18_steps):
    axes[0].bar(i, v, bottom=s18_c if v >= 0 else s18_c + v, color=C_OK if v >= 0 else C_DEF)
    axes[0].text(i, s18_c + v / 2 if v >= 0 else s18_c + v / 2, f"{v:+,.0f}", ha="center", va="center", color="white", fontweight="bold")
    s18_c += v
axes[0].bar(len(s18_steps), s18_c, color="k" if s18_c >= 0 else "#67000d")
axes[0].text(len(s18_steps), s18_c, f"{s18_c:,.0f}", ha="center", va="bottom" if s18_c >= 0 else "top")
axes[0].set_xticks(range(len(s18_steps) + 1)); axes[0].set_xticklabels([s for s, _ in s18_steps] + ["profit proxy"])
axes[0].axhline(0, color="k", lw=.8); axes[0].set_ylabel("NT$ million per year"); axes[0].set_title("Portfolio P&L waterfall (base assumptions)")
s18_bt = df.groupby("EX_CUST_TYPE", observed=True)[["EX_INT_REV", "EX_IX_REV", "EX_FUND_COST", "EX_LOSS", "EX_PROFIT"]].sum().reindex(S18_TYPE_ORDER) / 1e6
axes[1].bar(S18_TYPE_ORDER, s18_bt["EX_PROFIT"], color=np.where(s18_bt["EX_PROFIT"] >= 0, C_OK, C_DEF))
for i, v in enumerate(s18_bt["EX_PROFIT"]):
    axes[1].text(i, v, f"{v:,.0f}", ha="center", va="bottom" if v >= 0 else "top")
axes[1].axhline(0, color="k", lw=.8); axes[1].set_title("Profit proxy by Sep customer type, NT$ million"); axes[1].set_ylabel("NT$ million")
plt.tight_layout(); savefig("s18_pnl_waterfall"); plt.show()
display(s18_bt.round(1))
print(f"Profit margin on revenue: {s18_c / (s18_tot['interest'] + s18_tot['interchange']):.1%} | credit loss as % of revenue: {s18_tot['loss'] / (s18_tot['interest'] + s18_tot['interchange']):.0%} | loss as % of drawn: {s18_tot['loss'] * 1e6 / S18_EXP.sum():.1%}")
add_fact("18.3", "profitability", f"Base case: NT${s18_tot['interest'] + s18_tot['interchange']:,.0f}m annual revenue vs NT${s18_tot['loss']:,.0f}m credit loss; the book nets NT${s18_c:,.0f}m, and delinquent clients alone net NT${s18_bt.loc['Delinquent', 'EX_PROFIT']:,.0f}m", float(s18_c), "s18_pnl_waterfall", 3)

**Observations:**
- Base waterfall: interest NT$204m + interchange NT$40m - funding NT$27m - credit loss NT$258m = **-NT$41m** (margin -17% on revenue).
- Credit loss is 106% of revenue and 16.8% of the drawn balance.
- Revolvers net +NT$52m and transactors +NT$4m. Delinquent clients net -NT$96m and inactive clients -NT$1m.
- Story use: under any realistic card assumptions, the September-delinquent group alone wipes out the book's earnings.

#### 18.3.4 Risk vs revenue: which segments sit above the break-even line?
Each bubble is a segment from 18.1.2. x = revenue per client, y = cost per client (credit loss + funding). Below the dashed diagonal = profitable. Bubble size = clients. *Tableau: scatter with a 45-degree reference line, filter on dimension.*

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7.5))
s18_sc = S18_SEG[S18_SEG.dimension != "Customer type"]
for dname, sub in s18_sc.groupby("dimension"):
    ax.scatter(sub["revenue_per_client"] / 1e3, sub["cost_per_client"] / 1e3, s=20 + 1500 * sub["share_clients"], color=s18_pal[dname], alpha=.7, edgecolor="k", lw=.4, label=dname)
    for _, r in sub.iterrows():
        if r["cost_per_client"] > 1.3 * r["revenue_per_client"] or r["revenue_per_client"] > 1.3 * r["cost_per_client"] or r["share_clients"] > .25:
            ax.annotate(r["segment"], (r["revenue_per_client"] / 1e3, r["cost_per_client"] / 1e3), fontsize=7, xytext=(3, 3), textcoords="offset points")
s18_m = float(max(s18_sc["revenue_per_client"].max(), s18_sc["cost_per_client"].max())) / 1e3 * 1.05
ax.plot([0, s18_m], [0, s18_m], ls="--", color="k", lw=1); ax.text(s18_m * .7, s18_m * .62, "break-even", rotation=30, fontsize=9)
ax.set_xlabel("revenue per client, NT$ thousand / year"); ax.set_ylabel("credit loss + funding per client, NT$ thousand / year")
ax.set_title("Risk vs revenue by segment (base assumptions)"); ax.legend(fontsize=8, loc="upper left")
plt.tight_layout(); savefig("s18_risk_vs_revenue"); plt.show()
s18_scd = s18_sc[["dimension", "segment", "n", "default_rate", "revenue_per_client", "cost_per_client", "profit_per_client"]].sort_values("profit_per_client").copy()
s18_scd[["revenue_per_client", "cost_per_client", "profit_per_client"]] = s18_scd[["revenue_per_client", "cost_per_client", "profit_per_client"]].round(0)
display(s18_scd.round(3))

**Observations:**
- Late 2m (-NT$26k per client) and Late 3m+ (-NT$23k) sit furthest above break-even, followed by utilisation >100% (-NT$12k) and 90-100% (-NT$8k).
- Revolving (+NT$3.6k), the 10-30% utilisation band (+NT$1.6k) and Education Other/Unknown (+NT$4.1k) are the clearest profit makers.
- Most demographic segments (age, sex, education) cluster just above the diagonal, from -NT$2.2k to -NT$0.3k. Demographics barely separate economics.

#### 18.3.5 Which segments make or lose money, and where in the status x limit grid?
Left: profit per client for every single-dimension segment (green earns, red loses). Right: profit per client for Sep status x limit band (cells n >= 30, NT$ thousand).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 9), gridspec_kw={"width_ratios": [1, 1.1]})
s18_pf = S18_SEG[S18_SEG.dimension != "Customer type"].copy()
s18_pf["label"] = s18_pf["dimension"] + ": " + s18_pf["segment"]
s18_pf = s18_pf.sort_values("profit_per_client")
axes[0].barh(s18_pf["label"], s18_pf["profit_per_client"], color=np.where(s18_pf["profit_per_client"] >= 0, C_ACC, C_DEF))
axes[0].axvline(0, color="k", lw=.8); axes[0].tick_params(axis="y", labelsize=7); axes[0].set_xlabel("profit proxy per client, NT$ / year")
axes[0].set_title("Profit per client by segment")
s18_sg = df.groupby(["STATUS_GRP_SEP", "LIMIT_BAND"], observed=True).agg(n=(TARGET, "size"), ppc=("EX_PROFIT", "mean"), tot=("EX_PROFIT", "sum")).reset_index()
s18_mp = s18_sg.pivot(index="STATUS_GRP_SEP", columns="LIMIT_BAND", values="ppc").reindex(index=STATUS_GROUP_ORDER, columns=LIMIT_LABELS)
s18_mn = s18_sg.pivot(index="STATUS_GRP_SEP", columns="LIMIT_BAND", values="n").reindex(index=STATUS_GROUP_ORDER, columns=LIMIT_LABELS)
s18_mp = s18_mp.where(s18_mn >= 30)
s18_lim = float(np.nanmax(np.abs(s18_mp.values))) / 1e3
sns.heatmap(s18_mp / 1e3, annot=True, fmt=".1f", cmap="RdYlGn", center=0, vmin=-s18_lim, vmax=s18_lim, ax=axes[1], annot_kws={"fontsize": 8},
            cbar_kws={"label": "NT$ thousand per client per year"})
axes[1].set_title("Profit per client: Sep status x limit band (n >= 30)"); axes[1].set_xlabel("limit band"); axes[1].set_ylabel("")
plt.tight_layout(); savefig("s18_segment_profit"); plt.show()
s18_neg = s18_pf[s18_pf.profit_per_client < 0]
print(f"Loss-making single-dimension segments: {len(s18_neg)} of {len(s18_pf)} -> {', '.join(s18_neg['label'])}")
display(s18_sg.sort_values("tot").head(6).round(0)); display(s18_sg.sort_values("tot").tail(6).round(0))
s18_best = s18_sg[s18_sg.n >= 30].sort_values("tot").iloc[-1]; s18_worst = s18_sg[s18_sg.n >= 30].sort_values("tot").iloc[0]
add_fact("18.3", "profitability", f"{len(s18_neg)} of {len(s18_pf)} single-dimension segments lose money under the base assumptions; the biggest NT$ earner is '{s18_best['STATUS_GRP_SEP']} | {s18_best['LIMIT_BAND']}' (NT${s18_best['tot']/1e6:,.0f}m) and the biggest destroyer is '{s18_worst['STATUS_GRP_SEP']} | {s18_worst['LIMIT_BAND']}' (NT${s18_worst['tot']/1e6:,.0f}m)", float(s18_worst["tot"]), "s18_segment_profit", 2)

**Observations:**
- 26 of 35 single-dimension segments lose money under the base case, mostly because the whole book is slightly negative.
- In the status x limit grid, every Late-2m cell loses money. The loss per client grows with the limit, from -NT$9k (<=50k) to -NT$78k (300-500k).
- The largest NT$ earners are Revolving | 300-500k (+NT$16.6m) and Revolving | 200-300k (+NT$15.8m). The same cells are the biggest NT$ loss sources in 18.2.1: high revenue offsets them.
- Story use: limit size multiplies both profit and loss. The status decides the sign.

#### 18.3.6 How sensitive is the verdict to APR, interchange, LGD and funding cost?
Left: portfolio profit proxy (NT$ million) over an APR x LGD grid at base interchange and funding cost. Right: tornado chart. Each assumption moves alone across its range while the others stay at base.

In [ ]:
S18_APRS = [0.10, 0.12, 0.15, 0.18, 0.20]
S18_LGDS = [0.45, 0.60, 0.75, 0.80, 0.90, 1.00]
s18_grid = pd.DataFrame([[s18_econ(apr=a, lgd=l)["profit"].sum() / 1e6 for a in S18_APRS] for l in S18_LGDS],
                        index=[f"LGD {l:.0%}" for l in S18_LGDS], columns=[f"APR {a:.0%}" for a in S18_APRS])
s18_base = s18_econ()["profit"].sum() / 1e6
s18_torn = []
for nm, key, lo, hi in [("APR", "apr", 0.10, 0.20), ("Interchange", "ix", 0.01, 0.02), ("LGD", "lgd", 1.00, 0.45), ("Cost of funds", "cof", 0.03, 0.01), ("CCF on headroom", "ccf", 0.5, 0.0)]:
    vlo = s18_econ(**{key: lo})["profit"].sum() / 1e6; vhi = s18_econ(**{key: hi})["profit"].sum() / 1e6
    s18_torn.append({"assumption": nm, "worse_value": lo, "better_value": hi, "profit_worse": vlo, "profit_better": vhi, "swing": abs(vhi - vlo)})
s18_torn = pd.DataFrame(s18_torn).sort_values("swing")
fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))
s18_v = float(np.abs(s18_grid.values).max())
sns.heatmap(s18_grid, annot=True, fmt=",.0f", cmap="RdYlGn", center=0, vmin=-s18_v, vmax=s18_v, ax=axes[0], cbar_kws={"label": "NT$ million / year"})
axes[0].set_title("Portfolio profit proxy: APR x LGD (interchange 1.5%, funding 2%)")
y = np.arange(len(s18_torn))
axes[1].barh(y, s18_torn["profit_worse"] - s18_base, left=s18_base, color=C_DEF, label="worse end")
axes[1].barh(y, s18_torn["profit_better"] - s18_base, left=s18_base, color=C_ACC, label="better end")
axes[1].set_yticks(y); axes[1].set_yticklabels([f"{r.assumption} ({r.worse_value:g} -> {r.better_value:g})" for r in s18_torn.itertuples()])
axes[1].axvline(s18_base, color="k", lw=1); axes[1].axvline(0, color="k", lw=.8, ls=":")
axes[1].set_xlabel("portfolio profit proxy, NT$ million"); axes[1].set_title(f"Tornado: one assumption at a time (base NT${s18_base:,.0f}m)"); axes[1].legend()
plt.tight_layout(); savefig("s18_sensitivity"); plt.show()
display(s18_grid.round(0)); display(s18_torn.round(1))
S18_SENS = s18_grid.reset_index().melt(id_vars="index", var_name="APR", value_name="profit_NT$m").rename(columns={"index": "LGD"})
s18_pos = (s18_grid > 0).values.sum()
add_fact("18.3", "sensitivity", f"The book's profit sign depends on the assumptions: {s18_pos} of {s18_grid.size} APR x LGD combinations are profitable (from NT${s18_grid.values.min():,.0f}m to NT${s18_grid.values.max():,.0f}m); the largest swing comes from {s18_torn.iloc[-1]['assumption']} (NT${s18_torn.iloc[-1]['swing']:,.0f}m)", int(s18_pos), "s18_sensitivity", 2)

**Observations:**
- Only 6 of 30 APR x LGD combinations are profitable. The book needs LGD <= 60% with APR >= 18%, or LGD 45% with APR >= 12%.
- The range runs from -NT$196m (APR 10%, LGD 100%) to +NT$94m (APR 20%, LGD 45%).
- One at a time, LGD has the largest swing (NT$177m), then APR (NT$113m). Interchange and funding cost move the result by about NT$27m each.
- If defaulters drew half their headroom before default (CCF 50%), the profit proxy falls to -NT$260m.
- Story use: the verdict "the book loses money" depends on assumptions. The *ranking* of segments does not.

#### 18.3.7 Break-even LGD and break-even APR by segment: how much room does each segment have?
Break-even LGD = (revenue - funding) / EAD. A segment profits if real LGD is below it. Break-even APR = (loss + funding - interchange) / interest-bearing balance. A segment profits if the real APR is above it. The shaded band is the plausible range.

In [ ]:
s18_be = S18_SEG.copy()
s18_ibs = []
for dname, (c, o) in S18_DIMS.items():
    t = pd.Series(S18_INT1, index=df.index).groupby(df[c].astype(str)).sum()
    s18_ibs.append(pd.DataFrame({"dimension": dname, "segment": t.index, "int_bal": t.values}))
s18_be = s18_be.merge(pd.concat(s18_ibs), on=["dimension", "segment"], how="left")
s18_be["breakeven_LGD"] = (s18_be["revenue"] - s18_be["funding"]) / s18_be["ead"].replace(0, np.nan)
s18_be["breakeven_APR"] = (s18_be["loss"] + s18_be["funding"] - s18_be["interchange"]) / s18_be["int_bal"].replace(0, np.nan)
s18_be = s18_be[s18_be.dimension != "Customer type"].copy()
s18_be["label"] = s18_be["dimension"] + ": " + s18_be["segment"]
fig, axes = plt.subplots(1, 2, figsize=(18, 9), sharey=True)
s18_be = s18_be.sort_values("breakeven_LGD")
y = np.arange(len(s18_be))
axes[0].axvspan(0.45, 1.0, color="#fee0d2", alpha=.6, label="plausible LGD 45-100%")
axes[0].scatter(s18_be["breakeven_LGD"].clip(upper=3), y, color=np.where(s18_be["breakeven_LGD"] > S18_A["LGD"], C_ACC, C_DEF), zorder=3)
axes[0].axvline(S18_A["LGD"], color="k", ls="--", lw=1)
axes[0].set_yticks(y); axes[0].set_yticklabels(s18_be["label"], fontsize=7); axes[0].set_xlabel("break-even LGD (clipped at 300%)")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].legend(fontsize=8); axes[0].set_title("Break-even LGD (green = profitable at LGD 80%)")
axes[1].axvspan(0.10, 0.20, color="#e5f5e0", alpha=.6, label="plausible APR 10-20%")
axes[1].scatter(s18_be["breakeven_APR"].clip(-0.1, 1.0), y, color=np.where(s18_be["breakeven_APR"] < S18_A["APR"], C_ACC, C_DEF), zorder=3)
axes[1].axvline(S18_A["APR"], color="k", ls="--", lw=1)
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_xlabel("break-even APR (clipped to -10%..100%)"); axes[1].legend(fontsize=8)
axes[1].set_title("Break-even APR (green = profitable at APR 18%)")
plt.tight_layout(); savefig("s18_breakeven"); plt.show()
display(s18_be[["label", "default_rate", "breakeven_LGD", "breakeven_APR"]].round(3))
s18_never = s18_be[s18_be["breakeven_LGD"] < 0.45]["label"].tolist(); s18_always = s18_be[s18_be["breakeven_LGD"] > 1.0]["label"].tolist()
print(f"Lose money even at LGD 45%: {s18_never}")
print(f"Profitable even at LGD 100%: {s18_always}")
add_fact("18.3", "sensitivity", f"{len(s18_always)} of {len(s18_be)} segments stay profitable even at LGD 100% and {len(s18_never)} lose money even at LGD 45% ({', '.join(s18_never[:4])})", len(s18_never), "s18_breakeven", 2)

**Observations:**
- All four non-current September groups (Late 1m, 2m, 3m+, and Inactive) lose money even at LGD 45%. Their break-even LGDs are 20-40%.
- Five segments stay profitable even at LGD 100%: Revolving, Paid in full, utilisation 0-10% and 10-30%, and Education Other/Unknown.
- Break-even APR is 65-70% for Late 2m/3m+ clients and below 12% for revolvers. No legal APR prices the late groups.
- Story use: no realistic price makes September-delinquent clients profitable. This is a limit-management problem, not a pricing one.

#### 18.3.8 Client-level profit concentration: how few clients carry the book?
Stobachoff-style curve: clients sorted from most to least profitable, cumulative profit as a share of the **positive profit pool** (sum of all client profits above zero). The curve peaks at 100% where profitable clients end; the drop after the peak is the profit destroyed by loss-making clients, and the end point is the net result. New column `EX_PROFIT_BAND`.

In [ ]:
df["EX_PROFIT_BAND"] = pd.cut(df["EX_PROFIT"], [-np.inf, -10_000, -1_000, 1_000, 5_000, np.inf],
                               labels=["Heavy loss (< -10k)", "Loss (-10k to -1k)", "Break-even (+/-1k)", "Profit (1k-5k)", "High profit (> 5k)"])
s18_pr = np.sort(df["EX_PROFIT"].to_numpy())[::-1]
s18_pool = s18_pr[s18_pr > 0].sum()
s18_cp = np.cumsum(s18_pr) / s18_pool
fig, axes = plt.subplots(1, 2, figsize=(16, 5.2))
axes[0].plot(s18_xr, s18_cp, color="k", lw=2)
axes[0].axhline(1, color=C_NEU, ls="--"); axes[0].axvline((s18_pr > 0).mean(), color=C_DEF, ls=":", lw=1)
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].axhline(0, color="k", lw=.8)
axes[0].set_xlabel("clients, most profitable first"); axes[0].set_ylabel("cumulative profit / positive profit pool")
axes[0].set_title(f"Profit concentration: pool NT${s18_pool/1e6:,.0f}m, net NT${s18_pr.sum()/1e6:,.0f}m ({s18_cp[-1]:.0%} of pool)")
s18_pbt = pd.crosstab(df["EX_PROFIT_BAND"], df["EX_CUST_TYPE"], normalize="columns").reindex(columns=S18_TYPE_ORDER)
s18_pbt.T.plot(kind="bar", stacked=True, ax=axes[1], color=[C_DEF, "#fc9272", C_NEU, "#a1d99b", C_ACC], width=.7)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("Profit band mix by Sep customer type"); axes[1].legend(fontsize=7, loc="upper left", bbox_to_anchor=(1, 1))
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); savefig("s18_profit_concentration"); plt.show()
s18_pb = df.groupby("EX_PROFIT_BAND", observed=True).agg(n=(TARGET, "size"), default_rate=(TARGET, "mean"), profit_NTm=("EX_PROFIT", lambda v: v.sum() / 1e6))
s18_pb["share_clients"] = s18_pb["n"] / S18_N
display(s18_pb.round(3)); display(s18_pbt.round(3))
s18_k20 = s18_cp[int(0.2 * S18_N) - 1]
s18_negpool = -s18_pr[s18_pr < 0].sum()
print(f"Positive pool NT${s18_pool/1e6:,.1f}m | top 20% of clients earn {s18_k20:.0%} of it | loss-making clients {(s18_pr < 0).mean():.1%} destroy NT${s18_negpool/1e6:,.1f}m ({s18_negpool/s18_pool:.0%} of the pool) | "
      f"heavy-loss clients that defaulted: {df.loc[df['EX_PROFIT'] < -10_000, TARGET].mean():.1%}")
add_fact("18.3", "profitability", f"The most profitable 20% of clients earn {s18_k20:.0%} of the positive profit pool (NT${s18_pool/1e6:,.0f}m); the {(s18_pr < 0).mean():.0%} of clients who lose money destroy NT${s18_negpool/1e6:,.0f}m, {s18_negpool/s18_pool:.0%} of that pool", float(s18_negpool / s18_pool), "s18_profit_concentration", 2)

**Observations:**
- The most profitable 20% of clients earn 74% of the positive profit pool (NT$170m).
- The 26% of clients who lose money destroy NT$211m, 124% of that pool. All heavy-loss clients (< -NT$10k) are defaulters.
- 54% of revolvers are high-profit (> NT$5k). 89% of inactive clients are break-even.
- Story use: a Stobachoff curve tells the "few earn, few destroy" story in one chart.

### 18.4 What-if credit-line policies: what would a simple rule have done?
**Design (descriptive simulation, not a model).** Each rule flags clients from information available at the September statement. A **limit cut** of x% removes `min(headroom, x x limit)` of undrawn line. A **freeze** removes all headroom. A **close** removes all headroom and loses the client's revenue. None of these reduces the balance already drawn.

For each rule:
* clients affected, defaulters caught (capture), good clients hit
* **NT$ undrawn exposure removed**, total and from defaulters
* **loss avoided** = LGD x draw-before-default (50%) x headroom removed from defaulters
* **revenue lost** from good clients. Cut or freeze: the share of own headroom removed x (interchange + 25% of interest). Close: all revenue.
* **net** = loss avoided - revenue lost

All three money columns rest on the 18.0 assumptions.

#### 18.4.1 Five early-warning signs and eight candidate rules
The signs are recomputed here from September-time data (Section 17 has the full catalogue). `EX_N_SIGNS` counts how many fire for each client.

In [ ]:
s18_u1 = S18_B[:, 0] / S18_L
S18_SIGNS = {
    "late_in_sep": S18_S[:, 0] >= 1,
    "zero_pay_sep_on_pos_aug_bill": (S18_P[:, 0] == 0) & (S18_B[:, 1] > 0),
    "late_2plus_months": (S18_S >= 1).sum(1) >= 2,
    "maxed_and_min_pay": (s18_u1 > 0.9) & (S18_B[:, 1] > 0) & (S18_P[:, 0] < 0.1 * S18_B[:, 1]),
    "util_up_20pp_apr_sep": (S18_B[:, 0] - S18_B[:, 5]) / S18_L > 0.2,
}
df["EX_N_SIGNS"] = np.sum([v.astype(int) for v in S18_SIGNS.values()], axis=0)
s18_rows = []
for k, v in S18_SIGNS.items():
    s18_rows.append({"sign": k, "prevalence": v.mean(), "hit_rate": S18_Y[v].mean(), "capture": S18_Y[v].sum() / S18_Y.sum(),
                     "share_EAD_flagged": df["EX_EAD"].to_numpy()[v].sum() / df["EX_EAD"].sum()})
display(pd.DataFrame(s18_rows).round(3))
display(rate_table(df, "EX_N_SIGNS")[["EX_N_SIGNS", "n", "rate", "share_clients", "share_defaults"]].round(3))

def s18_policy_eval(mask, kind, x=None, lgd=None, ccf=None, ih=None):
    lgd = S18_A["LGD"] if lgd is None else lgd; ccf = S18_A["CCF"] if ccf is None else ccf; ih = S18_A["INT_HAIRCUT"] if ih is None else ih
    mask = np.asarray(mask, bool)
    rem = np.minimum(S18_HR, x * S18_L) if kind == "cut" else S18_HR.copy()
    rem = rem * mask
    frac = np.divide(rem, S18_HR, out=np.zeros_like(rem), where=S18_HR > 0)
    rl = S18_REV * mask if kind == "close" else frac * (S18_IXREV + ih * S18_INTREV)
    bad, good = mask & (S18_Y == 1), mask & (S18_Y == 0)
    la = lgd * ccf * rem[bad].sum(); rlost = rl[good].sum()
    return {"clients_affected": int(mask.sum()), "share_clients": mask.mean(), "defaulters_caught": int(bad.sum()),
            "capture": bad.sum() / S18_Y.sum(), "good_hit": int(good.sum()), "good_hit_share": good.sum() / (S18_Y == 0).sum(),
            "precision": bad.sum() / max(mask.sum(), 1), "headroom_removed_NTm": rem.sum() / 1e6, "headroom_removed_defaulters_NTm": rem[bad].sum() / 1e6,
            "EAD_flagged_NTm": S18_EXP[bad].sum() / 1e6, "loss_avoided_NTm": la / 1e6, "revenue_lost_NTm": rlost / 1e6,
            "net_NTm": (la - rlost) / 1e6, "profit_of_goods_hit_NTm": df["EX_PROFIT"].to_numpy()[good].sum() / 1e6}

s18_nsig = df["EX_N_SIGNS"].to_numpy()
S18_POLICIES = [
    ("A. Cut limit 30% if Sep status >= 2", "cut", 0.30, S18_S[:, 0] >= 2),
    ("B. Cut limit 50% if Sep status >= 1", "cut", 0.50, S18_S[:, 0] >= 1),
    ("C. Freeze line if Sep util > 90%", "freeze", None, s18_u1 > 0.9),
    ("D. Freeze if util > 70% and ever late", "freeze", None, (s18_u1 > 0.7) & ((S18_S >= 1).sum(1) >= 1)),
    ("E. Cut 20% if util > 70% and paid < 10% of Aug bill", "cut", 0.20, (s18_u1 > 0.7) & (S18_B[:, 1] > 0) & (S18_P[:, 0] < 0.1 * S18_B[:, 1])),
    ("F. Freeze if zero Sep payment on a positive Aug bill", "freeze", None, S18_SIGNS["zero_pay_sep_on_pos_aug_bill"]),
    ("G. Close line if >= 2 warning signs", "close", None, s18_nsig >= 2),
    ("H. Close line if >= 3 warning signs", "close", None, s18_nsig >= 3),
]
S18_POL = pd.DataFrame([{"policy": nm, "action": kd, **s18_policy_eval(mk, kd, x)} for nm, kd, x, mk in S18_POLICIES])
display(S18_POL.round(3))
for nm, kd, x, mk in S18_POLICIES:
    df["EX_POL_" + nm[0]] = mk.astype(np.int8)
s18_bp = S18_POL.sort_values("net_NTm").iloc[-1]
add_fact("18.4", "policy", f"Best rule under base assumptions: '{s18_bp['policy']}' affects {s18_bp['share_clients']:.0%} of clients, catches {s18_bp['capture']:.0%} of defaulters, hits {s18_bp['good_hit']:,} good clients, removes NT${s18_bp['headroom_removed_defaulters_NTm']:,.0f}m of defaulters' undrawn line and nets NT${s18_bp['net_NTm']:,.1f}m", float(s18_bp["net_NTm"]), "s18_policy_frontier", 3)
s18_fz = S18_POL.set_index("policy").loc["C. Freeze line if Sep util > 90%"]
add_fact("18.4", "policy", f"Freezing every line above 90% utilisation affects {s18_fz['share_clients']:.0%} of clients but catches only {s18_fz['capture']:.0%} of defaulters (precision {s18_fz['precision']:.0%}); maxed-out lines have little headroom left to remove (NT${s18_fz['headroom_removed_NTm']:,.0f}m)", float(s18_fz["capture"]), "s18_policy_frontier", 2)

**Observations:**
- Signs: late in Sep (hit rate 50%, capture 52%) and late 2+ months (53%, 45%) are the strongest. Maxed + min-pay and a 20-point utilisation rise flag 49% and 43% of EAD, but at only 27% and 21% hit rates.
- Default rises from 12% (0 signs) to 58% (3 signs), then dips to 48-51% for 4-5 signs.
- Rule B (cut 50% if Sep status >= 1) nets the most: +NT$50m. It catches 52% of defaulters, hits 3,389 good clients (15%) and removes NT$137m of defaulters' headroom.
- Rule F (freeze on zero September payment) nets +NT$23m with only 1,054 good clients hit. Rule A (cut 30% at status >= 2) nets +NT$17m at 70% precision.
- Utilisation rules lose money. C (freeze > 90%) nets -NT$15m and E nets -NT$20m. Maxed lines have almost no headroom left to remove, and the rules hit profitable revolvers.
- Story use: act on delinquency and missed payments, not on high utilisation.

#### 18.4.2 The policy frontier: how many good clients must be hit to catch a defaulter, and is it worth it in NT$?
Three rule families are swept across their thresholds: freeze if Sep util > t, cut 50% if Sep status >= k, close if >= k warning signs. The eight named rules are labelled points. Left: % of defaulters caught vs % of good clients hit (the diagonal = random targeting). Right: NT$ revenue lost vs NT$ loss avoided. Above the diagonal the rule pays for itself.

In [ ]:
s18_fr = []
for t in np.round(np.arange(0.5, 1.051, 0.05), 2):
    s18_fr.append({"family": "Freeze if Sep util > t", "threshold": t, **s18_policy_eval(s18_u1 > t, "freeze")})
for k in [0, 1, 2, 3, 4]:
    s18_fr.append({"family": "Cut 50% if Sep status >= k", "threshold": k, **s18_policy_eval(S18_S[:, 0] >= k, "cut", 0.5)})
for k in [1, 2, 3, 4, 5]:
    s18_fr.append({"family": "Close if >= k signs", "threshold": k, **s18_policy_eval(s18_nsig >= k, "close")})
S18_FRONT = pd.DataFrame(s18_fr)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
s18_fc = dict(zip(S18_FRONT["family"].unique(), [C_OK, C_DEF, "#756bb1"]))
for fam, sub in S18_FRONT.groupby("family"):
    axes[0].plot(sub["good_hit_share"], sub["capture"], marker="o", ms=4, color=s18_fc[fam], label=fam, lw=1.5)
    axes[1].plot(sub["revenue_lost_NTm"], sub["loss_avoided_NTm"], marker="o", ms=4, color=s18_fc[fam], label=fam, lw=1.5)
    for _, r in sub.iterrows():
        axes[0].annotate(f"{r['threshold']:g}", (r["good_hit_share"], r["capture"]), fontsize=6, xytext=(2, -8), textcoords="offset points", color=s18_fc[fam])
axes[0].scatter(S18_POL["good_hit_share"], S18_POL["capture"], s=90, color="k", marker="*", zorder=5, label="named rules")
axes[1].scatter(S18_POL["revenue_lost_NTm"], S18_POL["loss_avoided_NTm"], s=90, color="k", marker="*", zorder=5, label="named rules")
for _, r in S18_POL.iterrows():
    axes[0].annotate(r["policy"][0], (r["good_hit_share"], r["capture"]), fontsize=10, fontweight="bold", xytext=(4, 4), textcoords="offset points")
    axes[1].annotate(r["policy"][0], (r["revenue_lost_NTm"], r["loss_avoided_NTm"]), fontsize=10, fontweight="bold", xytext=(4, 4), textcoords="offset points")
axes[0].plot([0, 1], [0, 1], ls="--", color="k", lw=1)
axes[0].set_xlabel("good clients hit (% of non-defaulters)"); axes[0].set_ylabel("defaulters caught (% of defaulters)")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].set_title("Policy frontier: catch vs collateral damage"); axes[0].legend(fontsize=8, loc="lower right")
s18_m2 = float(max(S18_FRONT["revenue_lost_NTm"].max(), S18_FRONT["loss_avoided_NTm"].max(), S18_POL["revenue_lost_NTm"].max()))
axes[1].plot([0, s18_m2], [0, s18_m2], ls="--", color="k", lw=1); axes[1].set_xscale("symlog", linthresh=1); axes[1].set_yscale("symlog", linthresh=1)
axes[1].set_xlabel("revenue lost from good clients, NT$ million / year (symlog)"); axes[1].set_ylabel("credit loss avoided, NT$ million (symlog)")
axes[1].set_title("Policy economics: above the diagonal pays for itself"); axes[1].legend(fontsize=8)
plt.tight_layout(); savefig("s18_policy_frontier"); plt.show()
display(S18_FRONT[["family", "threshold", "share_clients", "capture", "good_hit_share", "precision", "headroom_removed_defaulters_NTm", "loss_avoided_NTm", "revenue_lost_NTm", "net_NTm"]].round(3))
s18_bf = S18_FRONT.sort_values("net_NTm").iloc[-1]
add_fact("18.4", "policy", f"Across {len(S18_FRONT)} swept thresholds the best net result is '{s18_bf['family']}' at {s18_bf['threshold']:g} (net NT${s18_bf['net_NTm']:,.1f}m, {s18_bf['capture']:.0%} of defaulters caught, {s18_bf['good_hit_share']:.0%} of good clients hit); {int((S18_FRONT['net_NTm'] > 0).sum())} of {len(S18_FRONT)} swept rules have a positive net", float(s18_bf["net_NTm"]), "s18_policy_frontier", 2)

**Observations:**
- In the catch-vs-damage view, status-based cuts sit far above the diagonal (status >= 1: 52% caught for 15% of good clients). Utilisation freezes lie close to it (util > 90%: 24% vs 19%).
- Only 6 of 22 swept rules have a positive net. Cut 50% at status >= 1 is best (+NT$50m). At status >= 0 the net falls to +NT$27m because revenue loss rises to NT$51m.
- Every utilisation-freeze threshold from 50% to 95% is net-negative (-NT$10m to -NT$22m).
- Closing lines at >= 1 or >= 2 signs is slightly negative (-NT$6m): it catches many defaulters but gives up NT$57-114m of revenue.
- Story use: the policy frontier chart is a hero chart for the "what should the bank do" part of the story.

#### 18.4.3 Does the policy ranking survive other assumptions?
Net NT$ (loss avoided - revenue lost) for each named rule under nine draw-before-default x LGD scenarios, plus two interest-haircut variants. Green = the rule pays for itself.

In [ ]:
s18_sc_rows = []
for nm, kd, x, mk in S18_POLICIES:
    row = {"policy": nm}
    for ccf in [0.25, 0.5, 1.0]:
        for lgd in [0.6, 0.8, 1.0]:
            row[f"CCF {ccf:.0%} / LGD {lgd:.0%}"] = s18_policy_eval(mk, kd, x, lgd=lgd, ccf=ccf)["net_NTm"]
    row["base, int haircut 0%"] = s18_policy_eval(mk, kd, x, ih=0.0)["net_NTm"]
    row["base, int haircut 100%"] = s18_policy_eval(mk, kd, x, ih=1.0)["net_NTm"]
    s18_sc_rows.append(row)
S18_POLSENS = pd.DataFrame(s18_sc_rows).set_index("policy")
fig, ax = plt.subplots(figsize=(17, 5.5))
s18_v = float(np.nanmax(np.abs(S18_POLSENS.values)))
sns.heatmap(S18_POLSENS, annot=True, fmt=".1f", cmap="RdYlGn", center=0, vmin=-s18_v, vmax=s18_v, ax=ax, annot_kws={"fontsize": 8}, cbar_kws={"label": "net NT$ million"})
ax.set_title("Net benefit of each rule under alternative assumptions (NT$ million)"); ax.set_ylabel(""); ax.tick_params(axis="x", rotation=35)
plt.tight_layout(); savefig("s18_policy_sensitivity"); plt.show()
s18_rob = (S18_POLSENS > 0).sum(1)
display(pd.DataFrame({"scenarios_positive": s18_rob, "of": S18_POLSENS.shape[1], "min_net": S18_POLSENS.min(1), "max_net": S18_POLSENS.max(1)}).round(1))
s18_all = s18_rob[s18_rob == S18_POLSENS.shape[1]].index.tolist(); s18_none = s18_rob[s18_rob == 0].index.tolist()
add_fact("18.4", "policy", f"Net-positive in all {S18_POLSENS.shape[1]} assumption scenarios: {', '.join(p[0] for p in s18_all)} (status- and payment-based limit actions); never positive: {', '.join(p[0] for p in s18_none) or 'none'} (utilisation-based freezes and cuts)", len(s18_all), "s18_policy_sensitivity", 2)

**Observations:**
- Rules A, B and F stay net-positive in all 11 assumption scenarios. B ranges from +NT$16m to +NT$132m.
- Rules C and E are never positive (as low as -NT$46m and -NT$70m).
- Rule H (close at >= 3 signs) is positive in 8 of 11 scenarios, rule G in 4 and rule D in 6.
- Story use: the recommendation (act on status and missed payments) does not depend on the APR, LGD or draw assumptions.

#### 18.4.4 Collateral damage: which good clients would each rule hit?
For each rule: the good (non-defaulting) clients flagged, split by September customer type, and the share of the book's profit they generate. A rule that mostly hits profitable revolvers is costly even when its headcount looks small.

In [ ]:
s18_cd_rows = []
s18_prof_tot = df["EX_PROFIT"].clip(lower=0).sum()
for nm, kd, x, mk in S18_POLICIES:
    good = mk & (S18_Y == 0)
    comp = df.loc[good, "EX_CUST_TYPE"].value_counts(normalize=True).reindex(S18_TYPE_ORDER).fillna(0)
    s18_cd_rows.append({"policy": nm, "good_hit": int(good.sum()), **{f"share_{t}": comp[t] for t in S18_TYPE_ORDER},
                        "mean_profit_of_goods_hit": df.loc[good, "EX_PROFIT"].mean(), "share_of_positive_profit_pool": df.loc[good, "EX_PROFIT"].clip(lower=0).sum() / s18_prof_tot})
S18_COLL = pd.DataFrame(s18_cd_rows).set_index("policy")
fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={"width_ratios": [1.3, 1]})
S18_COLL[[f"share_{t}" for t in S18_TYPE_ORDER]].plot(kind="barh", stacked=True, ax=axes[0], color=[C_NEU, C_OK, "#fdae6b", C_DEF], width=.75)
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].legend(S18_TYPE_ORDER, fontsize=8, loc="lower right"); axes[0].invert_yaxis()
axes[0].set_title("Good clients hit, by Sep customer type"); axes[0].set_ylabel("")
axes[1].barh(S18_COLL.index, S18_COLL["share_of_positive_profit_pool"], color=C_ACC); axes[1].invert_yaxis(); axes[1].set_yticklabels([])
axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("Share of the positive profit pool sitting with good clients hit")
plt.tight_layout(); savefig("s18_policy_collateral"); plt.show()
display(S18_COLL.round(3))

**Observations:**
- Rules A and B hit only September-delinquent good clients, by construction. Their collateral is 5% and 11% of the positive profit pool.
- Rules C and E hit mostly revolvers (82% of good clients hit) who hold 38% and 55% of the profit pool. This is why they fail.
- Rule F hits delinquent (79%) and transactor (21%) good clients, only 4% of the pool. It is the cheapest rule.
- Story use: collateral damage measured in profit, not headcount, explains why utilisation rules are a trap.

### 18.5 Limit headroom: how much more could defaulters still draw?
The September balance is what is owed today. The **undrawn line** is what a client could still add before the account stops. For a defaulter this is extra draw risk that no balance-based view shows.

#### 18.5.1 Do defaulters have less room left, and how much NT$ is it?
ECDFs of headroom in NT$ and as a share of the client's own limit, defaulters vs non-defaulters, plus a summary table.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
for lab, y_, col in [("non-defaulters", 0, C_OK), ("defaulters", 1, C_DEF)]:
    for ax, v in zip(axes, [df.loc[df[TARGET] == y_, "EX_HEADROOM"] / 1e3, df.loc[df[TARGET] == y_, "EX_HEADROOM_PCT"]]):
        s = np.sort(v.to_numpy()); ax.plot(s, np.arange(1, len(s) + 1) / len(s), color=col, lw=2, label=lab)
axes[0].set_xscale("symlog", linthresh=1); axes[0].set_xlabel("undrawn headroom, NT$ thousand (symlog)"); axes[0].set_title("ECDF of headroom (NT$)")
axes[1].set_xlabel("headroom as share of own limit"); axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[1].set_title("ECDF of headroom (% of limit)")
for ax in axes:
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_ylabel("cumulative share of clients"); ax.legend()
plt.tight_layout(); savefig("s18_headroom_ecdf"); plt.show()
S18_HRT = df.groupby("DEFAULT_LABEL").agg(n=(TARGET, "size"), total_headroom_NTm=("EX_HEADROOM", lambda v: v.sum() / 1e6), mean_headroom=("EX_HEADROOM", "mean"),
                                          median_headroom=("EX_HEADROOM", "median"), median_headroom_pct=("EX_HEADROOM_PCT", "median"),
                                          share_lt10pct=("EX_HEADROOM_PCT", lambda v: (v < 0.1).mean()), share_zero=("EX_HEADROOM", lambda v: (v == 0).mean()),
                                          total_limit_NTm=("LIMIT_BAL", lambda v: v.sum() / 1e6))
S18_HRT["unused_share_of_limit"] = S18_HRT["total_headroom_NTm"] / S18_HRT["total_limit_NTm"]
display(S18_HRT.round(3))
s18_hd = S18_HRT.loc["Default"]
add_fact("18.5", "headroom", f"Defaulters still had NT${s18_hd['total_headroom_NTm']:,.0f}m of undrawn line in September ({s18_hd['unused_share_of_limit']:.0%} of their limits, median NT${s18_hd['median_headroom']:,.0f} per defaulter); {s18_hd['share_lt10pct']:.0%} of defaulters had less than 10% of their line left vs {S18_HRT.loc['No default', 'share_lt10pct']:.0%} of payers", float(s18_hd["total_headroom_NTm"]), "s18_headroom_ecdf", 3)

**Observations:**
- Defaulters still had NT$548m of undrawn line in September: 64% of their limits, median NT$29,394 per defaulter vs NT$80,642 for payers.
- 24% of defaulters had less than 10% of the line left (19% of payers). 9.6% had zero headroom (6.4% of payers).
- Defaulters have less room in NT$ (mean NT$83k vs NT$127k), mainly because their limits are smaller. In share-of-limit terms they still have half the line open.
- Story use: most defaulters could still draw substantially more. The open line is unmanaged draw risk.

#### 18.5.2 How much would EAD grow if defaulters drew part of their remaining line?
By September status group: the defaulters' current EAD (Sep balance) plus the extra EAD if they drew 25%, 50% or 100% of their headroom before default (credit conversion factor, CCF). The status groups that look small on drawn balance may hold large undrawn lines.

In [ ]:
s18_hs = df[df[TARGET] == 1].groupby("STATUS_GRP_SEP").agg(defaulters=(TARGET, "size"), EAD=("EX_EXPOSURE", "sum"), headroom=("EX_HEADROOM", "sum")).reindex(STATUS_GROUP_ORDER) / [1, 1e6, 1e6]
for c in [0.25, 0.5, 1.0]:
    s18_hs[f"EAD_CCF{int(c*100)}"] = s18_hs["EAD"] + c * s18_hs["headroom"]
s18_hs["uplift_CCF50"] = s18_hs["EAD_CCF50"] / s18_hs["EAD"] - 1
s18_hs["headroom_per_defaulter_NT$"] = s18_hs["headroom"] * 1e6 / s18_hs["defaulters"]
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(s18_hs))
ax.bar(x, s18_hs["EAD"], color=C_DEF, label="Sep balance (EAD)")
ax.bar(x, 0.5 * s18_hs["headroom"], bottom=s18_hs["EAD"], color="#fc9272", label="+ 50% of headroom")
ax.bar(x, 0.5 * s18_hs["headroom"], bottom=s18_hs["EAD"] + 0.5 * s18_hs["headroom"], color="#fee0d2", label="+ remaining headroom (CCF 100%)")
for xi, (e, u) in enumerate(zip(s18_hs["EAD"], s18_hs["uplift_CCF50"])):
    ax.text(xi, e, f"+{u:.0%}\n@CCF50", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(s18_hs.index); ax.set_ylabel("NT$ million"); ax.legend()
ax.set_title("Defaulters' NT$ at risk: drawn balance plus undrawn line, by Sep status")
plt.tight_layout(); savefig("s18_headroom_ead_uplift"); plt.show()
display(s18_hs.round(2))
s18_tot_up = s18_hs["EAD_CCF50"].sum() / s18_hs["EAD"].sum() - 1
add_fact("18.5", "headroom", f"If defaulters had drawn half of their remaining line before default, EAD would rise by {s18_tot_up:.0%} overall; the uplift is largest for '{s18_hs['uplift_CCF50'].idxmax()}' defaulters (+{s18_hs['uplift_CCF50'].max():.0%}) and smallest for '{s18_hs['uplift_CCF50'].idxmin()}' (+{s18_hs['uplift_CCF50'].min():.0%})", float(s18_tot_up), "s18_headroom_ead_uplift", 2)

**Observations:**
- If defaulters drew 50% of their remaining line before charge-off, EAD would rise by 85% overall, from NT$322m to about NT$596m.
- The uplift is extreme for Inactive (+1,752%) and Paid-in-full (+1,098%) defaulters. They owe NT$10m together but hold NT$246m of open line.
- Revolving (+30%) and Late 2m/3m+ (+36-37%) defaulters have already used most of their lines.
- Story use: the "silent" defaulters (inactive or paying in full) are a hidden draw risk, invisible on balance-based views.

#### 18.5.3 Did defaulters use up their line over the six months?
Aggregate headroom by month (sum of undrawn line, NT$) for defaulters and non-defaulters, indexed to April = 100, next to aggregate utilisation (sum of positive balances / sum of limits). A falling headroom index means the group is drawing down its line.

In [ ]:
s18_hm = []
for k in range(6)[::-1]:
    hr_k = np.clip(S18_L - np.clip(S18_B[:, k], 0, None), 0, None)
    for y_, lab in [(0, "No default"), (1, "Default")]:
        m = S18_Y == y_
        s18_hm.append({"month": MONTH[k + 1], "group": lab, "headroom_NTm": hr_k[m].sum() / 1e6,
                       "agg_util": np.clip(S18_B[m, k], 0, None).sum() / S18_L[m].sum(), "share_zero_headroom": (hr_k[m] == 0).mean()})
S18_HM = pd.DataFrame(s18_hm)
S18_HM["headroom_index"] = S18_HM["headroom_NTm"] / S18_HM.groupby("group")["headroom_NTm"].transform("first") * 100
fig, axes = plt.subplots(1, 3, figsize=(19, 4.6))
for lab, col in [("No default", C_OK), ("Default", C_DEF)]:
    s = S18_HM[S18_HM.group == lab]
    axes[0].plot(s["month"], s["headroom_index"], marker="o", color=col, lw=2, label=lab)
    axes[1].plot(s["month"], s["agg_util"], marker="o", color=col, lw=2, label=lab)
    axes[2].plot(s["month"], s["share_zero_headroom"], marker="o", color=col, lw=2, label=lab)
axes[0].set_title("Undrawn headroom index (Apr = 100)"); axes[1].set_title("Aggregate utilisation (drawn / limit)"); axes[2].set_title("Share of clients with no headroom left")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[2].yaxis.set_major_formatter(mtick.PercentFormatter(1))
for ax in axes:
    ax.legend()
plt.tight_layout(); savefig("s18_headroom_path"); plt.show()
display(S18_HM.round(3))
s18_hdd = S18_HM[(S18_HM.group == "Default")].set_index("month"); s18_hnd = S18_HM[(S18_HM.group == "No default")].set_index("month")
add_fact("18.5", "headroom", f"Defaulters' aggregate utilisation went from {s18_hdd.loc['Apr', 'agg_util']:.0%} in April to {s18_hdd.loc['Sep', 'agg_util']:.0%} in September (payers {s18_hnd.loc['Apr', 'agg_util']:.0%} -> {s18_hnd.loc['Sep', 'agg_util']:.0%}); their headroom index ends at {s18_hdd.loc['Sep', 'headroom_index']:.0f} vs {s18_hnd.loc['Sep', 'headroom_index']:.0f}", float(s18_hdd.loc["Sep", "agg_util"]), "s18_headroom_path", 2)

**Observations:**
- Both groups draw down their lines at a similar pace. Headroom falls to 90 (defaulters) vs 91 (payers) from April = 100.
- Defaulters' aggregate utilisation rises from 29.5% to 37.3%, payers' from 22% to 29%. The gap stays at about 7-8 points throughout.
- The share of defaulters with no headroom left doubles from 4.4% to 9.6%, vs 2.3% to 6.4% for payers.
- Story use: no last-minute draw-down "run" before default. Defaulters sit higher all along.

#### 18.5.4 Where is the dangerous headroom: large unused lines held by risky clients?
Headroom band x September status. Left: default rate (cells n >= 30). Right: NT$ million of undrawn line held by **defaulters** in each cell. New column `EX_HEADROOM_BAND`. *Tableau: two highlight tables, or one with colour = rate and size = NT$.*

In [ ]:
S18_HR_LABELS = ["0 (at/over limit)", "<10k", "10-50k", "50-100k", "100-200k", ">200k"]
df["EX_HEADROOM_BAND"] = pd.cut(df["EX_HEADROOM"], [-1, 0, 10_000, 50_000, 100_000, 200_000, np.inf], labels=S18_HR_LABELS)
S18_HG = df.groupby(["EX_HEADROOM_BAND", "STATUS_GRP_SEP"], observed=True).agg(n=(TARGET, "size"), rate=(TARGET, "mean"),
          headroom_def_NTm=("EX_HEADROOM", lambda v: v[df.loc[v.index, TARGET] == 1].sum() / 1e6)).reset_index()
s18_r = S18_HG.pivot(index="EX_HEADROOM_BAND", columns="STATUS_GRP_SEP", values="rate").reindex(index=S18_HR_LABELS, columns=STATUS_GROUP_ORDER)
s18_nn = S18_HG.pivot(index="EX_HEADROOM_BAND", columns="STATUS_GRP_SEP", values="n").reindex(index=S18_HR_LABELS, columns=STATUS_GROUP_ORDER)
s18_h = S18_HG.pivot(index="EX_HEADROOM_BAND", columns="STATUS_GRP_SEP", values="headroom_def_NTm").reindex(index=S18_HR_LABELS, columns=STATUS_GROUP_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))
sns.heatmap(s18_r.where(s18_nn >= 30).astype(float), annot=True, fmt=".0%", cmap=CMAP_RATE, ax=axes[0], cbar_kws={"format": mtick.PercentFormatter(1)})
axes[0].set_title("Default rate: headroom band x Sep status (n >= 30)"); axes[0].set_xlabel(""); axes[0].set_ylabel("undrawn headroom (NT$)")
sns.heatmap(s18_h.astype(float), annot=True, fmt=".0f", cmap="Reds", ax=axes[1], cbar_kws={"label": "NT$ million"})
axes[1].set_title("Undrawn line held by defaulters, NT$ million"); axes[1].set_xlabel(""); axes[1].set_ylabel("")
plt.tight_layout(); savefig("s18_headroom_grid"); plt.show()
display(rate_table(df, "EX_HEADROOM_BAND")[["EX_HEADROOM_BAND", "n", "rate", "share_clients", "share_defaults", "lift"]].round(3))
s18_late_big = df[(S18_S[:, 0] >= 2) & (df["EX_HEADROOM"] >= 50_000)]
print(f"Clients 2+ months late in Sep with >= NT$50k headroom: {len(s18_late_big):,}; default rate {s18_late_big[TARGET].mean():.0%}; "
      f"their defaulters hold NT${s18_late_big.loc[s18_late_big[TARGET] == 1, 'EX_HEADROOM'].sum()/1e6:,.0f}m of undrawn line")
add_fact("18.5", "headroom", f"{len(s18_late_big):,} clients were 2+ months late in September yet still had at least NT$50k of open line; {s18_late_big[TARGET].mean():.0%} of them defaulted, and those defaulters held NT${s18_late_big.loc[s18_late_big[TARGET] == 1, 'EX_HEADROOM'].sum()/1e6:,.0f}m of undrawn credit", float(s18_late_big[TARGET].mean()), "s18_headroom_grid", 2)

**Observations:**
- Default rate falls with headroom: 30% at zero headroom, 29% under NT$10k, 14% above NT$200k. The steepest drop is above NT$50k.
- Within a September status, headroom still matters for current clients: Revolving falls from 17-18% (under NT$10k) to 5% (above NT$200k), Late 1m from 44% to 26%. For Late 2m it hardly moves (62-74% in every band), so once a client is 2+ months late the open line is no comfort.
- 804 clients were 2+ months late in September yet still had >= NT$50k of open line. 69% defaulted, and those defaulters held NT$78m of undrawn credit.
- Story use: "late and still open" (804 clients, NT$78m of open line) is the most direct target for a limit action.

### 18.6 Section 18 wrap-up: exports, story facts and new columns
Tidy Tableau tables: segment economics (shares, tilt, P&L, break-evens), concentration, capture curves, sensitivity grid, policies and frontier, policy sensitivity, headroom paths and grid, and a client-level economics file.

In [ ]:
export(S18_SEG, "s18_segment_economics")
export(s18_be[["dimension", "segment", "breakeven_LGD", "breakeven_APR"]], "s18_breakeven")
export(S18_SEV, "s18_severity")
export(S18_CONC, "s18_ead_concentration")
export(S18_CAPT, "s18_capture_risk_vs_money")
export(s18_sl, "s18_status_limit_loss_ranks")
export(s18_ul, "s18_util_limit_loss_grid")
export(S18_SENS, "s18_sensitivity_apr_lgd")
export(S18_POL, "s18_policies")
export(S18_FRONT, "s18_policy_frontier")
export(S18_POLSENS.reset_index(), "s18_policy_sensitivity")
export(S18_COLL.reset_index(), "s18_policy_collateral")
export(S18_HM, "s18_headroom_by_month")
export(S18_HG, "s18_headroom_grid")
s18_cli = ["ID", TARGET, "DEFAULT_LABEL", "STATUS_GRP_SEP", "LIMIT_BAND", "UTIL_BAND", "AGE_BAND", "EDU_LABEL", "SEX_LABEL", "LIMIT_BAL"] + \
          [c for c in df.columns if c.startswith("EX_")]
export(df[s18_cli], "s18_client_economics")
print(f"Section 18 runtime: {time.time() - S18_T0:.1f}s")
S18_FACTS = pd.DataFrame([f for f in STORY_FACTS if f["section"].startswith("18")])
with pd.option_context("display.max_colwidth", 400):
    display(S18_FACTS)

**New `df` columns (prefix `EX_`)**

| Column | Definition |
|---|---|
| `EX_EXPOSURE` | September drawn balance, `max(BILL_AMT1, 0)` (NT$) |
| `EX_HEADROOM`, `EX_HEADROOM_PCT` | undrawn line `max(LIMIT_BAL - EX_EXPOSURE, 0)`, and as a share of the limit |
| `EX_HEADROOM_BAND` | headroom band: 0 (at/over limit), <10k, 10-50k, 50-100k, 100-200k, >200k |
| `EX_EAD` | exposure at default = `EX_EXPOSURE x DEFAULT` |
| `EX_INT_REV` | annual interest proxy: APR 18% x average unpaid previous balance in months with status >= 0 |
| `EX_IX_REV` | annual interchange proxy: 1.5% x annualised positive implied spend (status != -2) |
| `EX_REVENUE` | `EX_INT_REV + EX_IX_REV` |
| `EX_FUND_COST` | 2% x average positive balance |
| `EX_LOSS` | `DEFAULT x LGD 80% x EX_EXPOSURE` |
| `EX_PROFIT`, `EX_PROFIT_BAND` | revenue - funding - loss (assumption-driven), and its band |
| `EX_CUST_TYPE` | Sep type: Inactive (-2), Transactor (-1), Revolver (0), Delinquent (>= 1) |
| `EX_CUST_TYPE_6M` | dominant type over Apr-Sep (ties go to the riskier type) |
| `EX_N_SIGNS` | number of five September-time warning signs that fire (18.4.1) |
| `EX_POL_A` .. `EX_POL_H` | 1 if the client is flagged by named policy rule A..H (18.4.1) |

All NT$ revenue, profit and policy-benefit numbers depend on the assumptions in 18.0. Use them for direction and ranking, not as accounting figures.

## 19. Case studies: twelve groups that break the rules

**Why this section matters for the story.** Aggregates say "late payers default, clean payers do not". A data story needs
*people*, and the most memorable people are the ones who break that rule: the client who paid every bill in full and
still defaulted, the client two months late who quietly recovered. This section defines **twelve groups from the data**
(each a rule a banker would believe), measures how often the rule holds, and then opens the files of **real clients**
inside each group, month by month, Apr → Sep.

How to read each case study:
* **Rule and expectation.** Each group is a population defined only by inputs (never by the outcome). Each carries an
  expectation (*safe* or *risky*). The **rule-breakers** are the defaulters in a safe group or the survivors in a risky group.
* **Profile strip.** Standardised mean difference (SMD) of the group vs everyone else on 14 key variables (|SMD| > 0.2 is noticeable).
* **Case cards.** Three real clients: the *typical* rule-breaker (closest to the breakers' median profile), the *extreme*
  rule-breaker (most extreme on the group's defining dimension) and the *twin* (the group-mate who most resembles the typical
  breaker month by month but had the opposite outcome).
* **Story role.** *Myth* (the rule does not hold), *Warning* (a risky signal that really bites) or *Surprise* (the rule holds
  on average, but a sizeable minority breaks it).

Conventions: month arrays are chronological (**Apr → Sep**). The payment shown under a month is the payment made in that
month, which settles the **previous** month's bill, so "paid % of prev bill" is payment(t) / bill(t−1).
New `df` columns carry the prefix `CS_`; section globals the prefix `S19_` / `s19_`.

**Section guide**

- **What it does:** Defines twelve client groups, each based on a rule a banker would believe (e.g. "never late = safe"). It measures how often each rule holds and opens the monthly files of real clients who break it.
- **Question it answers:** Which common beliefs about default are true, which are myths, and who are the people who break the rules?
- **Inputs / new columns:** monthly status, bill and payment columns, `LIMIT_BAL`, `AGE`, demographics, utilisation. Adds 17 columns with prefix `CS_` (one flag per group `CS_G01`-`CS_G12`, plus counts, `CS_RULE_BREAKER` and `CS_PRIMARY_GROUP`; full list in 19.20).
- **How to read the charts:** profile strips: bars = how far the group differs from everyone else (SMD, standardised difference; beyond ±0.2 is noticeable). Case cards: one client per panel, Apr → Sep, with bill, payment and status per month. Gallery: x = group size (log), y = default rate with 95% CI, colour = story role.
- **Terms:** rule-breaker = a defaulter in a "safe" group or a survivor in a "risky" group. Myth = the rule does not hold; Warning = the risk signal really bites; Surprise = the rule holds on average but a sizeable minority breaks it.
- **Key takeaways:**
  - Myths: "maxed out but never late" defaults at only 13.1%; "big limit + young" at 11.6%; "cured" clients at 22.5% (same as base).
  - Warnings: 2+ months late in Sep 69.6%; Sep payment stop 46.0%; sudden late after five clean months 36.0%.
  - Clean payers default at 11.7% but still supply 35% of all defaulters; 725 full payers still default.
  - Rule-breakers are mostly invisible in the data (within-group AUC 0.49-0.61) (number from the pre-fix run; refresh after the next Kaggle run).
- **Where it feeds:** the case-study gallery and persona cards in Tableau (`s19_group_definitions.csv`, `s19_case_cards_long.csv`); its "meaning" column supplies captions.

### 19.0 Building blocks
Chronological matrices, a 14-variable profile used for every SMD strip, and helpers for SMD, case-card tables and
plain-language readings of a single client.

In [ ]:
S19_T0 = time.time()
S19_Y = df[TARGET].to_numpy()
S19_N = len(df)
S19_M = list(MONTH_ORDER)                                     # Apr..Sep
S19_ST = df[PAY_COLS].to_numpy(int)[:, ::-1].copy()           # status, Apr..Sep
S19_B = df[BILL_COLS].to_numpy(float)[:, ::-1].copy()         # bills, Apr..Sep
S19_P = df[PAMT_COLS].to_numpy(float)[:, ::-1].copy()         # payments, Apr..Sep (payment in month t settles bill t-1)
S19_LIM = df["LIMIT_BAL"].to_numpy(float)
S19_U = S19_B / S19_LIM[:, None]
S19_PR = np.full_like(S19_B, np.nan)                          # paid share of previous bill (May..Sep)
_s19_prev = S19_B[:, :-1]
S19_PR[:, 1:] = np.where(_s19_prev > 0, S19_P[:, 1:] / np.where(_s19_prev > 0, _s19_prev, 1), np.nan)

S19_PROF = pd.DataFrame({
    "log10 limit": np.log10(df["LIMIT_BAL"]), "Age": df["AGE"], "Female": (df["SEX"] == 2).astype(float),
    "Grad school": (df["EDUCATION"] == 1).astype(float), "Married": (df["MARRIAGE"] == 1).astype(float),
    "Avg utilisation": df["AVG_UTIL"].clip(-0.5, 2), "Share of bills repaid (5 statements, aligned)": df["PAY_TO_BILL_5M"].clip(0, 2),
    "Months late": df["N_LATE"], "Worst delay": df["MAX_DELAY"], "Months paid in full": df["N_PAIDFULL"],
    "Months inactive": df["N_INACTIVE"], "Months zero pay": df["N_ZERO_PAY"],
    "log10 avg bill": np.log10(1 + df["AVG_BILL"].clip(lower=0)), "log10 avg pay": np.log10(1 + df["AVG_PAY"])})
S19_PROF_Z = ((S19_PROF - S19_PROF.mean()) / S19_PROF.std()).fillna(0).to_numpy()
# month-by-month "shape" space for twin matching: utilisation path + status path + limit + age
_s19_shape = np.column_stack([np.clip(S19_U, -0.5, 2), S19_ST, np.log10(S19_LIM), df["AGE"].to_numpy(float)])
S19_SHAPE_Z = (_s19_shape - _s19_shape.mean(0)) / _s19_shape.std(0)

def s19_smd(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    sd = np.sqrt((a.var(ddof=1) + b.var(ddof=1)) / 2)
    return float((a.mean() - b.mean()) / sd) if sd > 0 else 0.0

def s19_smd_strip(mask_a, mask_b):
    return pd.Series({c: s19_smd(S19_PROF[c].to_numpy()[mask_a], S19_PROF[c].to_numpy()[mask_b]) for c in S19_PROF.columns})

S19_STATUS_SHORT = {-2: "-2 none", -1: "-1 full", 0: "0 min"}
def s19_card(i):
    # one client, month by month (Apr..Sep), as a small printable table
    st = [S19_STATUS_SHORT.get(int(v), f"{int(v)} late") for v in S19_ST[i]]
    return pd.DataFrame({"status": st,
                         "bill NT$": [f"{v:,.0f}" for v in S19_B[i]],
                         "payment NT$": [f"{v:,.0f}" for v in S19_P[i]],
                         "util": [f"{v:.0%}" for v in S19_U[i]],
                         "paid % prev bill": ["-" if np.isnan(v) else f"{v:.0%}" for v in S19_PR[i]]}, index=S19_M).T

def s19_reading(i):
    r = df.iloc[i]
    st = S19_ST[i]
    late_m = [S19_M[k] for k in range(6) if st[k] >= 1]
    s_txt = "never late" if not late_m else f"late in {', '.join(late_m)} (worst {int(st.max())}m)"
    pos = np.clip(S19_B[i, :-1], 0, None).sum()              # Apr..Aug bills, settled by the May..Sep payments
    p_txt = f"repaid {S19_P[i, 1:].sum() / pos:.0%} of the Apr-Aug bills (5 statements, aligned)" if pos > 0 else "had no positive bill Apr-Aug"
    z = int((S19_P[i, 1:] == 0).sum())
    out = "DEFAULTED in October" if S19_Y[i] == 1 else "did NOT default"
    return (f"{r['SEX_LABEL']}, {int(r['AGE'])}, {r['EDU_LABEL']}, {r['MAR_LABEL']}, limit NT${r['LIMIT_BAL']:,.0f}: {s_txt}; "
            f"utilisation {S19_U[i, 0]:.0%} in Apr -> {S19_U[i, -1]:.0%} in Sep; {p_txt}; {z} zero-payment month(s) May-Sep; {out}.")
print("building blocks ready:", S19_PROF.shape[1], "profile variables")

### 19.1 The twelve groups: what rule does each one test, and how often does the rule hold?
Every group is defined from inputs only. Thresholds that are not natural codes come from the data (quantiles), not from
taste. Group 12 uses a demographic-only risk score (logistic fit on sex, education, marriage, age band and limit band):
its bottom 10% is the profile a demographics-led lender would call "safest". Education "Other/Unknown" is excluded
from group 12 because Section 1 showed it is label noise with an artificially low rate.

In [ ]:
s19_never_late = S19_ST.max(1) <= 0
s19_prev_pos = S19_B[:, :-1] > 0
s19_paid_amount = (S19_P[:, 1:] >= np.clip(S19_B[:, :-1], 0, None)).all(1) & s19_prev_pos.any(1)
S19_LIM_HI = float(np.quantile(S19_LIM, 0.8)); S19_AGE_LO = float(np.quantile(df["AGE"], 0.2))
s19_pay_collapse = (S19_P[:, 1:5] > 0).all(1) & (S19_P[:, 5] == 0) & (S19_B[:, 4] > 0)

# demographic-only score for group 12
s19_demo_X = pd.get_dummies(df[["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND"]].astype(str)).astype(float)
try:
    from sklearn.linear_model import LogisticRegression
    S19_DEMO_SCORE = LogisticRegression(max_iter=3000).fit(s19_demo_X.to_numpy(), S19_Y).predict_proba(s19_demo_X.to_numpy())[:, 1]
    S19_SCORE_SRC = "logistic regression"
except Exception as s19_e:
    _te = np.zeros(S19_N)
    for _c in ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND"]:
        _te += df.groupby(_c, observed=True)[TARGET].transform("mean").to_numpy() - BASE_RATE
    S19_DEMO_SCORE = BASE_RATE + _te
    S19_SCORE_SRC = f"additive target encoding (sklearn unavailable: {s19_e})"
s19_elig = (df["EDU_LABEL"] != "Other/Unknown").to_numpy()
S19_SAFE_THR = float(np.quantile(S19_DEMO_SCORE[s19_elig], 0.10))
s19_safest = s19_elig & (S19_DEMO_SCORE <= S19_SAFE_THR)

S19_G = {
 "G01": dict(col="CS_G01_CLEAN_PAYER", name="Clean payers", expect="safe",
             rule="status <= 0 in all six months (never late)", mask=s19_never_late),
 "G02": dict(col="CS_G02_LATE_SEP_2P", name="2m+ late in Sep", expect="risky",
             rule="Sep status >= 2 (two or more months behind at the last statement)", mask=S19_ST[:, 5] >= 2),
 "G03": dict(col="CS_G03_PAID_FULL_AMT", name="Paid in full every month", expect="safe",
             rule="every May-Sep payment >= the previous bill (by amount), at least one positive bill", mask=s19_paid_amount),
 "G04": dict(col="CS_G04_NEVER_USED", name="Never-used card", expect="safe",
             rule="status = -2 (no consumption) in all six months", mask=(S19_ST == -2).all(1)),
 "G05": dict(col="CS_G05_MAXED_CLEAN", name="Maxed out, never late", expect="risky",
             rule="utilisation > 95% in all six months AND never late", mask=(S19_U > 0.95).all(1) & s19_never_late),
 "G06": dict(col="CS_G06_BIG_YOUNG", name="Big limit + young", expect="risky",
             rule=f"limit >= NT${S19_LIM_HI:,.0f} (top 20%) AND age <= {S19_AGE_LO:.0f} (youngest 20%)",
             mask=(S19_LIM >= S19_LIM_HI) & (df["AGE"].to_numpy() <= S19_AGE_LO)),
 "G07": dict(col="CS_G07_DEEP_OVERLIMIT", name="Deep over-limit", expect="risky",
             rule="bill > 110% of limit in at least one month", mask=S19_U.max(1) > 1.10),
 "G08": dict(col="CS_G08_SUDDEN_COLLAPSE", name="Sudden collapse", expect="risky",
             rule="never late Apr-Aug, then late (status >= 1) in Sep", mask=(S19_ST[:, :5].max(1) <= 0) & (S19_ST[:, 5] >= 1)),
 "G09": dict(col="CS_G09_CURED", name="Cured", expect="safe",
             rule="late at least once in Apr-Jun, never late in Jul-Sep", mask=(S19_ST[:, :3].max(1) >= 1) & (S19_ST[:, 3:].max(1) <= 0)),
 "G10": dict(col="CS_G10_PAY_COLLAPSE", name="Sep payment collapse", expect="risky",
             rule="paid something every month May-Aug, paid nothing in Sep although the Aug bill was positive", mask=s19_pay_collapse),
 "G11": dict(col="CS_G11_CREDIT_BAL", name="Credit balance", expect="safe",
             rule="bill < 0 (bank owes the client) in at least one month", mask=(S19_B < 0).any(1)),
 "G12": dict(col="CS_G12_SAFEST_DEMO", name="Safest demographic profile", expect="safe",
             rule=f"bottom 10% of a demographic-only risk score (score <= {S19_SAFE_THR:.3f}; education known)", mask=s19_safest),
}
# extreme-case key (largest value wins) and within-group split, per group
_s19_negdepth = -S19_B.min(1)
S19_EXT = {"G01": (S19_LIM, "largest credit line"), "G02": (S19_ST[:, 5] * 1e7 + S19_B[:, 5], "worst Sep delay, then biggest Sep bill"),
           "G03": (S19_LIM, "largest credit line"), "G04": (S19_LIM, "largest credit line"), "G05": (S19_U.mean(1), "highest average utilisation"),
           "G06": (S19_LIM, "largest credit line"), "G07": (S19_U.max(1), "deepest over-limit"), "G08": (S19_ST[:, 5] * 1e7 + S19_B[:, 5], "worst Sep delay, then biggest Sep bill"),
           "G09": (S19_ST[:, :3].max(1) * 1e7 + S19_LIM, "worst early delay, then largest limit"), "G10": (S19_P[:, 1:5].mean(1), "largest regular payment that stopped"),
           "G11": (_s19_negdepth, "deepest credit balance"), "G12": (-S19_DEMO_SCORE, "lowest demographic risk score")}
_s19_nneg = (S19_B < 0).sum(1)
S19_SPLIT = {"G01": (df["STATUS_GRP_SEP"].to_numpy(), STATUS_GROUP_ORDER, "Sep status group"),
             "G02": (df["N_LATE_BAND"].astype(str).to_numpy(), ["0", "1", "2", "3", "4-6"], "months late (of 6)"),
             "G03": (df["LIMIT_BAND"].astype(str).to_numpy(), LIMIT_LABELS, "limit band"),
             "G04": (df["AGE_BAND"].astype(str).to_numpy(), AGE_LABELS, "age band"),
             "G05": (df["LIMIT_BAND"].astype(str).to_numpy(), LIMIT_LABELS, "limit band"),
             "G06": (df["EDU_LABEL"].to_numpy(), EDU_ORDER, "education"),
             "G07": (df["STATUS_GRP_SEP"].to_numpy(), STATUS_GROUP_ORDER, "Sep status group"),
             "G08": (df["STATUS_GRP_SEP"].to_numpy(), STATUS_GROUP_ORDER, "Sep status group"),
             "G09": (df["STATUS_GRP_SEP"].to_numpy(), STATUS_GROUP_ORDER, "Sep status group"),
             "G10": (df["STATUS_GRP_SEP"].to_numpy(), STATUS_GROUP_ORDER, "Sep status group"),
             "G11": (np.where(_s19_nneg >= 3, "3+", _s19_nneg.astype(str)), ["1", "2", "3+"], "months in credit"),
             "G12": (df["STATUS_GRP_SEP"].to_numpy(), STATUS_GROUP_ORDER, "Sep status group")}

def s19_role(expect, lift):
    if expect == "safe":
        return "Myth" if lift >= 0.9 else "Surprise"
    if lift <= 1.1:
        return "Myth"
    return "Warning" if lift >= 1.5 else "Surprise"

s19_rows = []
for s19_gid, s19_g in S19_G.items():
    _m = s19_g["mask"]; _n = int(_m.sum()); _k = int(S19_Y[_m].sum()); _r = _k / _n; _lo, _hi = wilson_ci(_k, _n)
    _brk = _m & ((S19_Y == 1) if s19_g["expect"] == "safe" else (S19_Y == 0))
    s19_g["breakers"] = _brk
    df[s19_g["col"]] = _m.astype(int)
    s19_rows.append({"group_id": s19_gid, "group": s19_g["name"], "rule": s19_g["rule"], "expectation": s19_g["expect"],
                     "n": _n, "share_clients": _n / S19_N, "defaults": _k, "default_rate": _r, "ci_lo": float(_lo), "ci_hi": float(_hi),
                     "lift": _r / BASE_RATE, "share_of_all_defaulters": _k / S19_Y.sum(),
                     "rule_breaker": "defaulted" if s19_g["expect"] == "safe" else "survived", "n_breakers": int(_brk.sum()),
                     "breaker_share_of_group": _brk.sum() / _n, "role": s19_role(s19_g["expect"], _r / BASE_RATE)})
S19_DEF = pd.DataFrame(s19_rows)
with pd.option_context("display.max_colwidth", 90):
    display(S19_DEF.drop(columns=["ci_lo", "ci_hi"]).round(3))
print(f"Base rate {BASE_RATE:.2%} | demographic score source: {S19_SCORE_SRC} | G06 thresholds: limit >= {S19_LIM_HI:,.0f}, age <= {S19_AGE_LO:.0f}")
export(S19_DEF, "s19_group_definitions")
_myth = S19_DEF[S19_DEF["role"] == "Myth"]
add_fact("19.1", "case studies", f"Of 12 banker rules tested as data-defined groups, {len(_myth)} turn out to be myths ({', '.join(_myth['group'])}); "
         f"the riskiest group is '{S19_DEF.loc[S19_DEF['default_rate'].idxmax(), 'group']}' at {S19_DEF['default_rate'].max():.1%} and the safest "
         f"'{S19_DEF.loc[S19_DEF['default_rate'].idxmin(), 'group']}' at {S19_DEF['default_rate'].min():.1%}", float(len(_myth)), "s19_gallery", 3)

**Observations:**
* Three of the twelve rules are myths: "maxed out, never late" (13.1%), "big limit + young" (11.6%) and "cured" (22.5%, the same as the 22.1% base).
* Three rules are real warnings: 2m+ late in Sep (69.6%, lift 3.14), Sep payment collapse (46.0%, lift 2.08) and sudden collapse (36.0%, lift 1.63).
* The safe rules hold on average but leak. Clean payers default at 11.7% and still supply 2,334 defaulters, 35% of all defaulters.
* Paying in full by amount (every bill cleared) gives 5,339 clients and exactly 725 defaulters (13.6%). This matches the round-1 count.
* Story use: the safest group is "big limit + young" at 11.6%. The bank's own credit-line decision beats the age stereotype.

### 19.2 Do the groups describe different people, or the same people twice?
Two 12 × 12 matrices: **Jaccard overlap** (shared members / union) and **conditional membership** P(column group | row group).
Then the default rate by the **number of groups** a client belongs to: are clients who fit several rules special?

In [ ]:
S19_IDS = list(S19_G)
S19_MEM = np.column_stack([S19_G[g]["mask"] for g in S19_IDS]).astype(int)
_inter = S19_MEM.T @ S19_MEM
_sz = S19_MEM.sum(0)
S19_JAC = pd.DataFrame(_inter / (_sz[:, None] + _sz[None, :] - _inter), index=S19_IDS, columns=S19_IDS)
S19_COND = pd.DataFrame(_inter / _sz[:, None], index=S19_IDS, columns=S19_IDS)
_lab = [f"{g} {S19_G[g]['name']}" for g in S19_IDS]
fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for ax, mat, ttl in [(axes[0], S19_JAC, "Jaccard overlap"), (axes[1], S19_COND, "P(column group | row group)")]:
    sns.heatmap(mat, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1, ax=ax, cbar=False, annot_kws={"size": 7},
                xticklabels=S19_IDS, yticklabels=_lab)
    ax.set_title(ttl)
fig.suptitle("19.2 How much do the twelve case-study groups overlap?", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s19_overlap"); plt.show()
df["CS_N_GROUPS"] = S19_MEM.sum(1)
S19_NG = rate_table(df.assign(CS_N_GROUPS_BAND=np.where(df["CS_N_GROUPS"] >= 4, "4+", df["CS_N_GROUPS"].astype(str))), "CS_N_GROUPS_BAND")
display(S19_NG.round(3))
_off = S19_JAC.where(~np.eye(12, dtype=bool)).stack().sort_values(ascending=False)
print("Largest overlaps (Jaccard):"); print(_off.iloc[::2].head(6).round(3).to_string())
print(f"Clients in no group: {(df['CS_N_GROUPS'] == 0).sum():,} ({(df['CS_N_GROUPS'] == 0).mean():.1%})")
export(S19_JAC.reset_index().rename(columns={"index": "group_id"}), "s19_overlap_jaccard")
_a, _b = _off.index[0]
add_fact("19.2", "case studies", f"The twelve groups are mostly distinct people: the largest overlap is {_a} with {_b} (Jaccard {_off.iloc[0]:.2f}); "
         f"{(df['CS_N_GROUPS'] == 0).mean():.0%} of clients fit none of the twelve rules and default at {S19_Y[df['CS_N_GROUPS'] == 0].mean():.1%}", float(_off.iloc[0]), "s19_overlap", 1)

**Observations:**
* Most groups describe different people. The largest overlap is paid-in-full with never-used (Jaccard 0.26), then clean payers with paid-in-full (0.22).
* Default falls steadily with the number of groups a client fits: 21.8% for one group, 15.4% for three and 11.2% for four or more.
* The 2,721 clients (9.1%) who fit no rule default at 37.7% (lift 1.71). The twelve stories leave a risky remainder: chronic revolvers who are late but not in the late-Sep groups.
* Story use: the case studies cover 91% of clients, so the gallery is close to a full map of the portfolio.

#### The case-study engine
One function tells each group's story: headline numbers, the profile strip, the within-group split, the mean utilisation
path of breakers vs group-mates, and three case cards with a one-line plain reading each. The case-card rows are kept in
long format for Tableau.

In [ ]:
S19_CARDS = {}
S19_RES = {}
S19_ROLE_COL = {"typical breaker": C_DEF, "extreme breaker": "#7a0177", "twin (opposite outcome)": C_OK}

def s19_pick_cases(gid):
    g = S19_G[gid]; brk = np.where(g["breakers"])[0]; mates = np.where(g["mask"] & ~g["breakers"])[0]
    zb = S19_PROF_Z[brk]
    typ = brk[np.argmin(np.abs(zb - np.median(zb, 0)).sum(1))]
    key = S19_EXT[gid][0][brk]
    ext = next(i for i in brk[np.argsort(-key, kind="stable")] if i != typ)
    out = [(typ, "typical breaker"), (ext, "extreme breaker")]
    if len(mates):
        twin = mates[np.argmin(((S19_SHAPE_Z[mates] - S19_SHAPE_Z[typ]) ** 2).sum(1))]
        out.append((twin, "twin (opposite outcome)"))
    return out

def s19_story(gid, sec):
    g = S19_G[gid]; m = g["mask"]; brk = g["breakers"]; mates = m & ~brk
    n = int(m.sum()); k = int(S19_Y[m].sum()); rate = k / n; lo, hi = wilson_ci(k, n); lift = rate / BASE_RATE
    role = S19_DEF.set_index("group_id").loc[gid, "role"]
    print(f"{gid} {g['name']} | rule: {g['rule']} | expectation: {g['expect']} | role: {role}")
    print(f"n = {n:,} ({n / S19_N:.1%} of clients) | default rate {rate:.1%} [{float(lo):.1%}, {float(hi):.1%}] | lift {lift:.2f} | "
          f"rule-breakers ({'defaulted' if g['expect'] == 'safe' else 'survived'}): {int(brk.sum()):,} = {brk.sum() / n:.0%} of the group")
    smd_g = s19_smd_strip(m, ~m).sort_values()
    smd_b = s19_smd_strip(brk, mates)
    tb = smd_b.abs().sort_values(ascending=False)
    print("Group vs rest, largest |SMD|: " + ", ".join(f"{c} {smd_g[c]:+.2f}" for c in smd_g.abs().sort_values(ascending=False).index[:4]))
    print("Breakers vs group-mates, largest |SMD|: " + ", ".join(f"{c} {smd_b[c]:+.2f}" for c in tb.index[:4]))
    lab, order, sname = S19_SPLIT[gid]
    sp = pd.DataFrame({"lev": lab[m], "y": S19_Y[m]}).groupby("lev").agg(n=("y", "size"), rate=("y", "mean"))
    sp = sp.reindex([o for o in order if o in sp.index]); sp = sp[sp["n"] >= 20]
    print(f"Within-group default rate by {sname} (n >= 20):"); print(sp.T.round(3).to_string())
    cases = s19_pick_cases(gid)

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    ax = axes[0, 0]
    ax.hlines(smd_g.index, 0, smd_g.values, color=C_NEU)
    ax.scatter(smd_g.values, smd_g.index, c=np.where(smd_g.abs() >= .2, C_DEF, C_NEU), zorder=3)
    for _v in (-.2, .2): ax.axvline(_v, ls=":", color=C_NEU)
    ax.axvline(0, color="k", lw=.8); ax.set_xlabel("standardised mean difference (group minus rest)"); ax.set_title("Profile: how the group differs from everyone else")
    ax = axes[0, 1]
    if len(sp):
        lo2, hi2 = wilson_ci(sp["rate"] * sp["n"], sp["n"]); x = np.arange(len(sp))
        ax.bar(x, sp["rate"], color=[C_DEF if r > BASE_RATE else C_OK for r in sp["rate"]], alpha=.8)
        ax.errorbar(x, sp["rate"], yerr=[sp["rate"] - lo2, hi2 - sp["rate"]], fmt="none", color="k", capsize=3)
        ax.set_xticks(x); ax.set_xticklabels([f"{i}\nn={c:,}" for i, c in zip(sp.index, sp["n"])], fontsize=8)
    ax.axhline(BASE_RATE, ls="--", color="k", lw=1, label=f"base {BASE_RATE:.1%}"); ax.axhline(rate, ls="-.", color=C_ACC, lw=1.2, label=f"group {rate:.1%}")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.legend(fontsize=8); ax.set_title(f"Inside the group: default rate by {sname}")
    ax = axes[1, 0]
    for msk, lb, cl in [(brk, f"rule-breakers (n={int(brk.sum()):,})", C_DEF), (mates, f"group-mates (n={int(mates.sum()):,})", C_OK)]:
        if msk.sum():
            ax.plot(S19_M, np.median(S19_U[msk], 0), marker="o", color=cl, lw=2, label=lb + " median")
            ax.fill_between(S19_M, np.quantile(S19_U[msk], .25, 0), np.quantile(S19_U[msk], .75, 0), color=cl, alpha=.12)
    ax.plot(S19_M, np.median(S19_U, 0), ls="--", color="k", lw=1, label="all clients median")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.legend(fontsize=8); ax.set_title("Utilisation path, Apr -> Sep (median, IQR band)")
    ax = axes[1, 1]
    for i, rl in cases:
        ax.plot(S19_M, S19_U[i], color=S19_ROLE_COL[rl], lw=2, label=f"{rl}: ID {int(df['ID'].iat[i])} ({'default' if S19_Y[i] else 'no default'})")
        late = S19_ST[i] >= 1
        ax.scatter(np.array(S19_M)[late], S19_U[i][late], marker="X", s=110, color="k", zorder=4)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.legend(fontsize=8); ax.set_title("Case cards: utilisation path (X = month late)")
    fig.suptitle(f"19.{sec} {gid} {g['name']}: {rate:.1%} default (lift {lift:.2f}), role: {role}", fontsize=14, fontweight="bold")
    plt.tight_layout(); savefig(f"s19_{gid.lower()}_case"); plt.show()

    rows = []
    for i, rl in cases:
        r = df.iloc[i]
        print(f"\n--- {rl.upper()} | client ID {int(r['ID'])} | outcome: {'DEFAULT' if S19_Y[i] else 'no default'} ---")
        print(s19_card(i).to_string())
        print("Reading: " + s19_reading(i))
        for kk, mo in enumerate(S19_M):
            for met, val in [("status", S19_ST[i, kk]), ("bill", S19_B[i, kk]), ("payment", S19_P[i, kk]), ("util", S19_U[i, kk]), ("paid_share_prev_bill", S19_PR[i, kk])]:
                rows.append({"group_id": gid, "group": g["name"], "role": role, "card_role": rl, "client_id": int(r["ID"]), "month": mo, "month_num": kk + 1,
                             "metric": met, "value": float(val), "default": int(S19_Y[i]), "limit": float(r["LIMIT_BAL"]), "age": int(r["AGE"]),
                             "sex": r["SEX_LABEL"], "education": r["EDU_LABEL"], "marriage": r["MAR_LABEL"], "reading": s19_reading(i)})
    S19_CARDS[gid] = rows
    S19_RES[gid] = dict(n=n, rate=rate, lift=lift, role=role, n_brk=int(brk.sum()), smd_b=smd_b, smd_g=smd_g, split=sp)
    add_fact(f"19.{sec}", "case studies", f"{g['name']} ({g['rule']}): {n:,} clients default at {rate:.1%} (lift {lift:.2f}, {role.lower()}); "
             f"{int(brk.sum()):,} of them {'defaulted anyway' if g['expect'] == 'safe' else 'survived anyway'}", float(rate), f"s19_{gid.lower()}_case", 2)
    add_fact(f"19.{sec}", "case studies", f"Inside '{g['name']}', what separates the rule-breakers from their group-mates most is {tb.index[0]} "
             f"(SMD {smd_b[tb.index[0]]:+.2f}), then {tb.index[1]} ({smd_b[tb.index[1]]:+.2f})", float(smd_b[tb.index[0]]), f"s19_{gid.lower()}_case", 1)
    return S19_RES[gid]
print("engine ready")

### 19.3 G01 Clean payers: if a client is never late in six months, why would they default?
**Rule:** status ≤ 0 in all six months. **Expectation:** safe. **Breakers:** defaulters. Section 17.9 counted these
"silent defaulters"; here we ask *which* clean payers default (split by September status: inactive, paid in full or
revolving) and read their files. If the breakers are revolvers carrying high balances, the warning is in the money, not
in the status codes.

In [ ]:
s19_r = s19_story("G01", 3)
_sp = s19_r["split"]
if "Revolving" in _sp.index and "Paid in full" in _sp.index:
    print(f"\nRevolving-in-Sep clean payers default at {_sp.loc['Revolving', 'rate']:.1%} vs {_sp.loc['Paid in full', 'rate']:.1%} for paid-in-full-in-Sep clean payers")

**Observations:**
* Clean payers are two-thirds of the book (19,931 clients) and default at 11.7%. They still produce 2,334 defaulters, 35% of the total.
* Inside the group, revolvers default at 10.7%, while clients who paid in full in Sep or were inactive default more (13.8% and 12.9%). Inside the clean group, "paid in full" is not the safest status.
* Breakers differ only mildly from their group-mates: they pay less (log avg pay SMD -0.34), carry smaller bills (-0.32) and have smaller limits (-0.26).
* The typical breaker (ID 240) revolves a NT$16-23k bill at 12-16% utilisation and repays 6-18% of it each month. Her twin (ID 8127) looks almost the same but repays 14-25%. The gap between them is only a few percentage points of repayment share (number from the pre-fix run; refresh after the next Kaggle run).
* The extreme breaker (ID 26922, limit NT$720k) lets the bill climb from 85k to 308k while paying a flat 4% minimum. The status code never flags it.
* Story use: Surprise. "Never late" is not "safe". The warning sits in the payment amounts, not in the status codes.

### 19.4 G02 Two or more months late in September: who survives being that far behind?
**Rule:** September status ≥ 2. **Expectation:** risky. **Breakers:** survivors. This is the single strongest risk
signal in the data (PAY_1). The survivors are the interesting people: is survival tied to a short delinquency history
(first-time late) or to continued payments while behind?

In [ ]:
s19_r = s19_story("G02", 4)
_m = S19_G["G02"]["mask"]
_pay_sep = S19_P[:, 5] > 0
print(f"\nInside G02: paid something in Sep -> default {S19_Y[_m & _pay_sep].mean():.1%} (n={int((_m & _pay_sep).sum()):,}); paid nothing -> {S19_Y[_m & ~_pay_sep].mean():.1%} (n={int((_m & ~_pay_sep).sum()):,})")
add_fact("19.4", "case studies", f"Among clients 2m+ late in Sep, those who still paid something in Sep default at {S19_Y[_m & _pay_sep].mean():.1%} vs {S19_Y[_m & ~_pay_sep].mean():.1%} for those who paid nothing", float(S19_Y[_m & _pay_sep].mean()), "s19_g02_case", 2)

**Observations:**
* Being 2+ months late in Sep is the strongest warning: 69.6% default (lift 3.14). Even so, 953 clients (30%) survive.
* Survivors repay a slightly larger share (SMD +0.28) and have more paid-in-full months (+0.28). Their default rate rises with history, from 62.0% at 1 late month to 74.2% at 4-6 (number from the pre-fix run; refresh after the next Kaggle run).
* Paying something in Sep barely helps: 69.0% if they paid vs 70.9% if they paid nothing.
* Card 6233 (survived) and her twin 4065 (defaulted) have the same pattern: status 2 for Jul-Sep while paying about 4% of the bill. The status code cannot separate them (number from the pre-fix run; refresh after the next Kaggle run).
* The extreme survivor (ID 8998) is 8 months late, at 158% utilisation, with zero payments for six months, and did not default. This points to label noise or a write-off that was coded differently.
* Story use: Warning. Act on a September delay. The 30% who recover are not visible in this data (within-group AUC 0.61, see 19.16) (number from the pre-fix run; refresh after the next Kaggle run).

### 19.5 G03 Paid in full every month (by amount): how can someone who clears every bill default?
**Rule:** every May–Sep payment ≥ the previous month's bill. **Expectation:** safe. This is the amount-based version of
"paid in full"; it recovers the round-1 fact that hundreds of full payers still default. Is it bigger lines, older
clients, or clients whose bills were tiny (so "paying in full" cost almost nothing)?

In [ ]:
s19_r = s19_story("G03", 5)
_m = S19_G["G03"]["mask"]; _bb = np.clip(S19_B[:, :5], 0, None).mean(1)
_q = np.quantile(_bb[_m], [0.25, 0.5, 0.75])
_band = np.digitize(_bb, _q)
_t = pd.DataFrame({"bill_quartile": _band[_m], "y": S19_Y[_m]}).groupby("bill_quartile")["y"].agg(["size", "mean"])
_t.index = ["Q1 smallest", "Q2", "Q3", "Q4 largest"][:len(_t)]
print("\nDefault rate of full payers by quartile of their average Apr-Aug bill:"); print(_t.round(3).to_string())

**Observations:**
* 5,339 clients cleared every bill by amount, and 725 (13.6%) still defaulted.
* Breakers are the small spenders: they have lower payments (SMD -0.39) and lower bills (-0.37). Default falls from 20.3% in the smallest-bill quartile to 7.6% in the largest.
* The rate also falls with limit, from 18.2% at <=50k to 4.4% above 500k.
* The typical breaker (ID 11545) has a NT$1,000 bill paid in full every month on a NT$300k line, and still defaulted. Extreme case 13884 paid 100% every month, yet her Sep status reads "2 late". Status and amount disagree (number from the pre-fix run; refresh after the next Kaggle run).
* Story use: Surprise. For a full payer, a tiny, dormant-looking balance is the risk, not a large one. This is likely tied to the label semantics in Section 12.

### 19.6 G04 Never-used card: what does a default mean for a client who never spent?
**Rule:** status −2 (no consumption) in all six months. **Expectation:** safe (no balance, little to lose). A default here
cannot be a classic "could not pay the bill" story. The case cards show whether these clients had bills at all.

In [ ]:
s19_r = s19_story("G04", 6)
_m = S19_G["G04"]["mask"]
_zero = (S19_B == 0).all(1)
print(f"\nNever-used clients with all six bills exactly zero: {int((_m & _zero).sum()):,}, default {S19_Y[_m & _zero].mean():.1%}; "
      f"with some non-zero bill: {int((_m & ~_zero).sum()):,}, default {S19_Y[_m & ~_zero].mean():.1%}")
add_fact("19.6", "case studies", f"Never-used cards with six zero bills still default at {S19_Y[_m & _zero].mean():.1%}; with some non-zero bill the rate is {S19_Y[_m & ~_zero].mean():.1%}", float(S19_Y[_m & _zero].mean()), "s19_g04_case", 2)

**Observations:**
* Never-used cards default at 13.5% (lift 0.61) overall.
* The headline is the split. The 325 clients with six bills of exactly zero default at 33.2%, against 9.9% for the 1,784 with some small bill activity.
* Breakers pay and are billed much less (SMD -0.66 each) and have more zero-pay months (+0.58). Default rises with age, to 19.9% at 51-60 and 27.3% at 61+.
* The typical breaker (ID 13316) had one NT$1,306 bill in April, paid it in May, and then nothing. She defaulted with no balance to default on (number from the pre-fix run; refresh after the next Kaggle run).
* Story use: Myth inside a surprise. A completely dormant card is 1.5x base risk. A "default" with no balance points to account closure or label semantics, not to inability to pay.

### 19.7 G05 Maxed out all half-year but never late: is a full card a danger sign on its own?
**Rule:** utilisation > 95% in every month and never late. **Expectation:** risky (no headroom). **Breakers:** survivors.
If this group defaults *below* base, high utilisation without delinquency is a myth of risk: the client is a heavy but
disciplined revolver.

In [ ]:
s19_r = s19_story("G05", 7)
_mx = (S19_U > 0.95).all(1)
print(f"\nMaxed all six months, any status: n={int(_mx.sum()):,}, default {S19_Y[_mx].mean():.1%}; of which ever late: n={int((_mx & ~s19_never_late).sum()):,}, default {S19_Y[_mx & ~s19_never_late].mean():.1%}")
add_fact("19.7", "case studies", f"Maxed-out clients (util > 95% all six months) default at {S19_Y[_mx].mean():.1%} overall, but only {S19_Y[_mx & s19_never_late].mean():.1%} when never late vs {S19_Y[_mx & ~s19_never_late].mean():.1%} when ever late", float(S19_Y[_mx & s19_never_late].mean()), "s19_g05_case", 3)

**Observations:**
* Maxed out all half-year but never late: 350 clients default at only 13.1% (lift 0.59). Utilisation alone is a myth of risk.
* The same maxed-out clients default at 51.8% once they have been late at least once. Delinquency drives the risk; utilisation does not.
* Breakers cannot be told apart from group-mates (within-group AUC 0.49; the largest SMD is sex at +0.26) (number from the pre-fix run; refresh after the next Kaggle run).
* Typical survivor 14694 and defaulting twin 28686 are near copies: both sit at about 100% of a NT$100k limit and pay about 4% a month without fail (number from the pre-fix run; refresh after the next Kaggle run).
* Extreme survivor 3797 sits at 389-646% of a NT$20k limit and never misses a minimum.
* Story use: Myth. A full card with disciplined minimum payments is a profitable revolver, not a defaulter in waiting.

### 19.8 G06 Big limit and young: does youth make a large credit line risky?
**Rule:** limit in the top 20% and age in the youngest 20%. **Expectation:** risky (a large line in inexperienced hands).
The bank chose to give these clients big lines; the case asks whether that screening worked.

In [ ]:
s19_r = s19_story("G06", 8)
_m = S19_G["G06"]["mask"]; _young = df["AGE"].to_numpy() <= S19_AGE_LO; _big = S19_LIM >= S19_LIM_HI
print(f"\nYoung (any limit): {S19_Y[_young].mean():.1%} | big limit (any age): {S19_Y[_big].mean():.1%} | young AND big: {S19_Y[_m].mean():.1%} | young, small limit (< NT${S19_LIM_HI:,.0f}): {S19_Y[_young & ~_big].mean():.1%}")

**Observations:**
* Young clients with big lines (limit >= NT$270k, age <= 27) default at 11.6%, the lowest of all twelve groups.
* Young clients overall default at 24.2%, and young clients with smaller lines at 25.1%. The limit, not the age, carries the signal.
* The 52 defaulters stand out on behaviour: they have a worse delay (SMD -0.78 for survivors) and more late months (-0.75). Education makes no difference (11.6-12.0%).
* Twin 22894 defaulted after paying down from 7% to 2% utilisation with a large Sep payment. Nothing in her file looks risky.
* Story use: Myth. "Young = risky" disappears once the bank has screened a client into a large line. This supports the Section 15 finding that the limit absorbs most of the demographic signal.

### 19.9 G07 Deep over-limit: who goes more than 10% past the limit and still pays?
**Rule:** bill > 110% of limit in at least one month. **Expectation:** risky. **Breakers:** survivors. Over-limit
balances should be impossible without the bank's tolerance; survivors may be clients the bank trusted to overspend.

In [ ]:
s19_r = s19_story("G07", 9)
_m = S19_G["G07"]["mask"]; _nol = (S19_U > 1.0).sum(1)
_t = pd.DataFrame({"months_over_limit": np.where(_nol[_m] >= 4, "4-6", _nol[_m].astype(str)), "y": S19_Y[_m]}).groupby("months_over_limit")["y"].agg(["size", "mean"])
print("\nDeep over-limit clients: default rate by number of months over 100% of limit"); print(_t.round(3).to_string())

**Observations:**
* Deep over-limit clients (bill > 110% of limit at some point) default at 29.8% (lift 1.35). Even so, 70% survive.
* Risk builds with persistence: 20.7% after one over-limit month, rising to 36.0% after 4-6 months.
* Status in Sep separates them sharply: revolvers 18.2% vs 2m-late 66.1%. Survivors have fewer late months (SMD -0.71).
* Typical survivor 24762 and defaulting twin 26377 both spike above 115% of the limit in July while paying 3-9%. The twin ends Sep at 102%, the survivor at 92% (number from the pre-fix run; refresh after the next Kaggle run).
* Extreme case 25732 shows a 1069% July bill that she cleared with a NT$1.02m payment. This is a one-off transaction, not distress.
* Story use: Surprise. Going over the limit is a mild signal. It matters only when it repeats or comes with lateness.

### 19.10 G08 Sudden collapse: clean for five months, then late in September. A blip or a cliff?
**Rule:** never late Apr–Aug, status ≥ 1 in Sep. **Expectation:** risky. Status 1 exists almost only in September, so
most of these are "1 month late"; the case cards show whether a single late month after a clean record is a warning or noise.

*Footnote:* In September, code 1 mixes clients who were paid-up or inactive in August with clients who were 2+ months late in August; it is not a uniform first delay. (Here August is clean by construction, so the group's code-1 clients are the paid-up/inactive kind.)

In [ ]:
s19_r = s19_story("G08", 10)
_m = S19_G["G08"]["mask"]
for _v, _lb in [(1, "1 month late"), (2, "2 months late")]:
    _mm = _m & (S19_ST[:, 5] == _v)
    print(f"Sudden collapse with Sep status {_lb}: n={int(_mm.sum()):,}, default {S19_Y[_mm].mean():.1%}")
_cl = s19_never_late
print(f"Compare: clean all six months default {S19_Y[_cl].mean():.1%}")
add_fact("19.10", "case studies", f"A single late September after five clean months lifts default from {S19_Y[_cl].mean():.1%} (always clean) to {S19_Y[_m].mean():.1%}", float(S19_Y[_m].mean() / S19_Y[_cl].mean()), "s19_g08_case", 2)

**Observations:**
* Five clean months followed by a late Sep: 2,369 clients default at 36.0%, against 11.7% for the always-clean.
* The size of the delay matters: 1 month late in Sep gives 25.1%, 2 months late gives 62.0%.
* Survivors have a smaller worst delay (SMD -0.75), repay a larger share (+0.74) and have lower utilisation (-0.58) (number from the pre-fix run; refresh after the next Kaggle run).
* Typical survivor 11755 and defaulting twin 3206 both have a near-zero or negative balance and get a "1 late" flag on an account with nothing owed. The flag looks like noise there (number from the pre-fix run; refresh after the next Kaggle run).
* Extreme survivor 11856 slides to 124% of a NT$420k line while paying 3-4%, is 2 months late in Sep, and still did not default.
* Story use: Warning. One recent miss after a clean record more than triples risk. The "1 month late" flag on a zero balance is mostly noise, though.

### 19.11 G09 Cured: late in spring, clean all summer. Is the past forgiven?
**Rule:** late at least once in Apr–Jun, never late in Jul–Sep. **Expectation:** safe (three clean months). If cured
clients still default near or above base, delinquency history leaves a scar the status codes no longer show.

In [ ]:
s19_r = s19_story("G09", 11)
_m = S19_G["G09"]["mask"]
_never = s19_never_late; _recent_clean_all = S19_ST[:, 3:].max(1) <= 0
print(f"\nClean in Jul-Sep, never late before: {S19_Y[_recent_clean_all & _never].mean():.1%} | clean in Jul-Sep, cured: {S19_Y[_m].mean():.1%} (n={int(_m.sum()):,})")
_w = S19_ST[:, :3].max(1)
_t = pd.DataFrame({"worst_early_delay": np.where(_w[_m] >= 3, "3+", _w[_m].astype(str)), "y": S19_Y[_m]}).groupby("worst_early_delay")["y"].agg(["size", "mean"])
print("Cured clients by worst Apr-Jun delay:"); print(_t.round(3).to_string())
add_fact("19.11", "case studies", f"Three clean months do not erase the past: cured clients default at {S19_Y[_m].mean():.1%} vs {S19_Y[_recent_clean_all & _never].mean():.1%} for clients clean in Jul-Sep who were never late", float(S19_Y[_m].mean()), "s19_g09_case", 3)

**Observations:**
* Cured clients (late in Apr-Jun, clean Jul-Sep) default at 22.5%, about the same as base. Clients clean in Jul-Sep who were never late default at 11.7%.
* A deeper early delay leaves a larger scar: 21.8% if the worst delay was 2 months, 38.1% if it was 3+.
* Breakers are hard to separate from group-mates (AUC 0.57; the largest SMD is pay at -0.25) (number from the pre-fix run; refresh after the next Kaggle run).
* Typical breaker 6404 and survivor twin 11257 are near identical: late 2m in April, then about 40% utilisation, paying 4-9% each month (number from the pre-fix run; refresh after the next Kaggle run).
* Extreme breaker 20674 was 7 months late in spring, cut utilisation from 103% to 34%, and still defaulted.
* Story use: Myth. Three clean months do not erase a delinquency. History doubles the risk of an otherwise clean recent record.

### 19.12 G10 September payment collapse: a regular payer stops paying. Who recovers?
**Rule:** a payment in every month May–Aug, nothing in Sep although the August bill was positive. **Expectation:** risky.
This is a money-side signal that the status code may not show yet.

In [ ]:
s19_r = s19_story("G10", 12)
_m = S19_G["G10"]["mask"]
_t = pd.DataFrame({"sep_status": df["STATUS_GRP_SEP"].to_numpy()[_m], "y": S19_Y[_m]}).groupby("sep_status")["y"].agg(["size", "mean"])
print("\nPayment collapse: share of the group by Sep status (does the status code see it?)"); print((_t["size"] / _t["size"].sum()).round(3).to_string())
_vis = _m & (S19_ST[:, 5] >= 1)
print(f"Collapse WITHOUT a late Sep status: n={int((_m & ~_vis).sum()):,}, default {S19_Y[_m & ~_vis].mean():.1%}; with a late status: n={int(_vis.sum()):,}, default {S19_Y[_vis].mean():.1%}")

**Observations:**
* A regular payer who stops paying in Sep defaults at 46.0% (lift 2.08). This group has 781 clients.
* The status code mostly sees it already: 89% of the group are late in Sep. The 89 clients without a late status still default at 34.8%.
* Survivors have fewer late months (SMD -0.33) and slightly higher repayment shares (+0.25) (number from the pre-fix run; refresh after the next Kaggle run).
* Typical survivor 11397 and defaulting twin 29248 both run utilisation up to 99% by Sep and then pay nothing (number from the pre-fix run; refresh after the next Kaggle run).
* Extreme survivor 29383 paid NT$202k and 180k in May-Jun, then dropped to small payments and zero.
* Story use: Warning. A stopped payment is a money-side alarm that adds risk even when the status code looks fine. This rule depends on its threshold, though (19.17).

### 19.13 G11 Credit balance: the bank owes the client money. How can such a client default?
**Rule:** bill < 0 in at least one month. **Expectation:** safe. Negative bills arise from overpayment or refunds;
a default afterwards means the balance swung back. The case cards trace that swing.

In [ ]:
s19_r = s19_story("G11", 13)
_m = S19_G["G11"]["mask"]; _negsep = S19_B[:, 5] < 0
print(f"\nCredit balance still in Sep: n={int((_m & _negsep).sum()):,}, default {S19_Y[_m & _negsep].mean():.1%}; credit balance earlier, positive by Sep: n={int((_m & ~_negsep).sum()):,}, default {S19_Y[_m & ~_negsep].mean():.1%}")

**Observations:**
* Clients with a credit balance default at 16.5% (lift 0.74). Even here, 318 default.
* More months in credit means more risk: 14.2% for 1 month, 21.0% for 3+. A credit balance still present in Sep gives 18.5%, against 15.6% when it was gone by Sep.
* Breakers show more lateness (months late SMD +0.46) and more zero-pay months (+0.38).
* Extreme breaker 291 overpaid by NT$345k in April (a -121% bill), then kept tiny bills paid in full, and defaulted.
* Story use: Surprise. "The bank owes the client" is not protective. Long or recurring credit balances behave like dormant accounts.

### 19.14 G12 The safest demographic profile: does a 'good' profile protect against default?
**Rule:** bottom 10% of a demographic-only risk score. **Expectation:** safe. A lender who screened on who the client *is*
would love this group. The case cards ask whether behaviour overrides the profile.

In [ ]:
s19_r = s19_story("G12", 14)
_m = S19_G["G12"]["mask"]
print("\nModal profile of the safest group:")
for _c in ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND"]:
    _vc = df.loc[_m, _c].astype(str).value_counts(normalize=True)
    print(f"  {_c}: {_vc.index[0]} ({_vc.iloc[0]:.0%})")
_late = S19_ST[:, 5] >= 1
print(f"Safest profile but late in Sep: n={int((_m & _late).sum()):,}, default {S19_Y[_m & _late].mean():.1%} | everyone late in Sep: {S19_Y[_late].mean():.1%}")
add_fact("19.14", "case studies", f"The safest demographic profile defaults at {S19_Y[_m].mean():.1%}, but once late in Sep it defaults at {S19_Y[_m & _late].mean():.1%}, close to the {S19_Y[_late].mean():.1%} of all late clients: behaviour overrides profile", float(S19_Y[_m & _late].mean()), "s19_g12_case", 3)

**Observations:**
* The safest demographic profile defaults at 12.4%. Its typical member is a young, single, female graduate-school client with a 300-500k limit.
* Behaviour overrides the profile. Inside the group, revolvers default at 7.0%, but 1-month-late clients default at 24.7% and 2-months-late at 53.3%. If late in Sep, the group defaults at 33.9%.
* Breakers here are the easiest to spot (AUC 0.75). They have more zero-pay months (SMD +0.64), more late months (+0.58) and lower payments (-0.56) (number from the pre-fix run; refresh after the next Kaggle run).
* Extreme breaker 5287 was never late, yet her bill rose from 191k to 485k (93% utilisation) while her repayment share fell from 18% to 4%.
* Story use: Surprise. A good profile lowers risk, but once a client is late it no longer protects them.

### 19.15 The gallery: all twelve groups on one page
Left: group size (log scale) vs default rate with 95% Wilson intervals (confidence range for each rate), coloured by story role. Right: lift vs base, with
an arrow showing which way the rule *expected* the group to go. This is the natural Tableau "case-study gallery" sheet.

In [ ]:
S19_ROLE_COLORS = {"Myth": "#984ea3", "Warning": C_DEF, "Surprise": C_OK}
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
ax = axes[0]
for _, s19_rw in S19_DEF.iterrows():
    ax.errorbar(s19_rw["n"], s19_rw["default_rate"], yerr=[[s19_rw["default_rate"] - s19_rw["ci_lo"]], [s19_rw["ci_hi"] - s19_rw["default_rate"]]], fmt="o",
                ms=6 + 10 * np.sqrt(s19_rw["share_clients"]), color=S19_ROLE_COLORS[s19_rw["role"]], capsize=3)
    ax.annotate(f"{s19_rw['group_id']} {s19_rw['group']}", (s19_rw["n"], s19_rw["default_rate"]), xytext=(6, 4), textcoords="offset points", fontsize=8)
ax.axhline(BASE_RATE, ls="--", color="k", lw=1); ax.set_xscale("log")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("clients in group (log scale)"); ax.set_ylabel("default rate")
for s19_rl, s19_cl in S19_ROLE_COLORS.items(): ax.scatter([], [], color=s19_cl, label=s19_rl)
ax.legend(title="story role"); ax.set_title("Size vs default rate")
ax = axes[1]
_t = S19_DEF.sort_values("lift")
ax.barh(_t["group_id"] + " " + _t["group"], _t["lift"], color=[S19_ROLE_COLORS[_r] for _r in _t["role"]], alpha=.85)
for s19_yy, (s19_lf, s19_ex) in enumerate(zip(_t["lift"], _t["expectation"])):
    ax.annotate("expected low" if s19_ex == "safe" else "expected high", (s19_lf, s19_yy), xytext=(5, -3), textcoords="offset points", fontsize=7, color="k")
ax.axvline(1, ls="--", color="k"); ax.set_xlabel("lift (group default rate / base rate)"); ax.set_title("Lift vs base, with the expectation each rule makes")
fig.suptitle("19.15 Case-study gallery: twelve rules, how they hold up", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s19_gallery"); plt.show()
print(S19_DEF[["group_id", "group", "expectation", "n", "default_rate", "lift", "n_breakers", "role"]].round(3).to_string(index=False))
add_fact("19.15", "case studies", f"The gallery spans default rates from {S19_DEF['default_rate'].min():.1%} to {S19_DEF['default_rate'].max():.1%}; "
         f"{(S19_DEF['role'] == 'Warning').sum()} rules are real warnings, {(S19_DEF['role'] == 'Surprise').sum()} hold with a large minority of exceptions and "
         f"{(S19_DEF['role'] == 'Myth').sum()} are myths", float(S19_DEF["default_rate"].max() - S19_DEF["default_rate"].min()), "s19_gallery", 3)

**Observations:**
* Default rates across the twelve groups range from 11.6% (big limit + young) to 69.6% (2m+ late in Sep).
* Every "risky" rule built on money alone (maxed out, over-limit) is weaker than the rules built on delinquency.
* The three warnings (G02, G08, G10) all centre on September. The myths (G05, G06, G09) are rules about utilisation, age and recovery.
* Story use: hero chart for the case-study panel (size vs default rate, coloured by role).

### 19.16 Can the rule-breakers be told apart from their group-mates?
For each group: the SMD of breakers vs group-mates on the 14 profile variables (heatmap), and a cross-validated
within-group AUC of a simple logistic lens on profile + monthly status and utilisation. An AUC near 0.5 means the breakers
are invisible in this data: they are the irreducible part of the story, not a missed signal.

In [ ]:
S19_BRK_SMD = pd.DataFrame({_g: S19_RES[_g]["smd_b"] for _g in S19_IDS}).T
fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(S19_BRK_SMD.astype(float), annot=True, fmt=".2f", cmap=CMAP_DIV, center=0, vmin=-1.5, vmax=1.5, ax=ax, annot_kws={"size": 7},
            yticklabels=[f"{_g} {S19_G[_g]['name']}" for _g in S19_IDS])
ax.set_title("19.16 Rule-breakers minus group-mates (SMD); red = breakers higher"); plt.tight_layout(); savefig("s19_breaker_smd"); plt.show()
s19_auc_rows = []
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score, StratifiedKFold
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    _X_all = np.column_stack([S19_PROF.fillna(S19_PROF.median()).to_numpy(), S19_ST, np.clip(S19_U, -0.5, 2)])
    for s19_gg in S19_IDS:
        _m = S19_G[s19_gg]["mask"]; _yy = S19_G[s19_gg]["breakers"][_m].astype(int)
        if min(_yy.sum(), len(_yy) - _yy.sum()) >= 15:
            _cv = StratifiedKFold(5, shuffle=True, random_state=42)
            _auc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)), _X_all[_m], _yy, cv=_cv, scoring="roc_auc").mean()
        else:
            _auc = np.nan
        s19_auc_rows.append({"group_id": s19_gg, "group": S19_G[s19_gg]["name"], "n": int(_m.sum()), "n_breakers": int(_yy.sum()), "within_group_auc": _auc,
                             "top_separator": S19_BRK_SMD.loc[s19_gg].abs().idxmax(), "top_smd": S19_BRK_SMD.loc[s19_gg, S19_BRK_SMD.loc[s19_gg].abs().idxmax()]})
except Exception as s19_e:
    print("sklearn lens skipped:", s19_e)
S19_AUC = pd.DataFrame(s19_auc_rows)
if len(S19_AUC):
    display(S19_AUC.round(3)); export(S19_AUC, "s19_breaker_separability")
    _lo_auc = S19_AUC.sort_values("within_group_auc").iloc[0]; _hi_auc = S19_AUC.sort_values("within_group_auc").iloc[-1]
    add_fact("19.16", "case studies", f"Rule-breakers are hardest to spot in '{_lo_auc['group']}' (within-group AUC {_lo_auc['within_group_auc']:.2f}) and easiest in "
             f"'{_hi_auc['group']}' (AUC {_hi_auc['within_group_auc']:.2f}, top separator {_hi_auc['top_separator']})", float(_lo_auc["within_group_auc"]), "s19_breaker_smd", 2)
export(S19_BRK_SMD.reset_index().rename(columns={"index": "group_id"}), "s19_breaker_smd")

**Observations:**
* Rule-breakers are mostly hard to tell apart. Within-group AUC is 0.49 for maxed-out, 0.57 for cured and 0.61 for 2m+ late, all near chance (number from the pre-fix run; refresh after the next Kaggle run).
* They are easiest to spot in the safest demographic profile (0.75) and sudden collapse (0.73). In both, delay and payment history separate them (number from the pre-fix run; refresh after the next Kaggle run).
* Lower payment amounts are the most common separator. Pay is the top separator for clean payers, full payers, never-used cards and cured clients.
* Story use: much of each surprise is irreducible in six months of data. Present the breakers as real uncertainty, not as a missed signal.

### 19.17 Are the findings an artefact of the thresholds?
Five groups use a cut-off that could have been chosen differently. Each is re-run across a range of cut-offs; if the
default rate barely moves, the case study is robust.

In [ ]:
s19_sens = []
for s19_t in [0.8, 0.85, 0.9, 0.95, 1.0]:
    _mm = (S19_U > s19_t).all(1) & s19_never_late; s19_sens.append({"group_id": "G05", "parameter": "util threshold (all 6 months, never late)", "value": s19_t, "n": int(_mm.sum()), "rate": S19_Y[_mm].mean()})
for s19_t in [1.0, 1.05, 1.1, 1.2, 1.3, 1.5]:
    _mm = S19_U.max(1) > s19_t; s19_sens.append({"group_id": "G07", "parameter": "max utilisation above", "value": s19_t, "n": int(_mm.sum()), "rate": S19_Y[_mm].mean()})
for s19_q in [0.7, 0.8, 0.9]:
    for s19_qa in [0.1, 0.2, 0.3]:
        _mm = (S19_LIM >= np.quantile(S19_LIM, s19_q)) & (df["AGE"].to_numpy() <= np.quantile(df["AGE"], s19_qa))
        s19_sens.append({"group_id": "G06", "parameter": f"limit quantile >= {s19_q:.1f}, age quantile <=", "value": s19_qa, "n": int(_mm.sum()), "rate": S19_Y[_mm].mean() if _mm.sum() else np.nan})
for s19_qs in [0.05, 0.1, 0.2, 0.3, 0.5]:
    _mm = s19_elig & (S19_DEMO_SCORE <= np.quantile(S19_DEMO_SCORE[s19_elig], s19_qs))
    s19_sens.append({"group_id": "G12", "parameter": "bottom share of demographic score", "value": s19_qs, "n": int(_mm.sum()), "rate": S19_Y[_mm].mean()})
_prior = S19_P[:, 1:5].mean(1)
for s19_f in [0.0, 0.1, 0.25, 0.5]:
    _mm = (S19_P[:, 1:5] > 0).all(1) & (S19_P[:, 5] <= s19_f * _prior) & (S19_B[:, 4] > 0)
    s19_sens.append({"group_id": "G10", "parameter": "Sep payment <= share of May-Aug mean", "value": s19_f, "n": int(_mm.sum()), "rate": S19_Y[_mm].mean()})
S19_SENS = pd.DataFrame(s19_sens)
fig, axes = plt.subplots(1, 5, figsize=(24, 4.8))
for ax, s19_gg in zip(axes, ["G05", "G07", "G06", "G12", "G10"]):
    _t = S19_SENS[S19_SENS["group_id"] == s19_gg]
    for s19_pp, s19_tt in _t.groupby("parameter", sort=False):
        ax.plot(s19_tt["value"], s19_tt["rate"], marker="o", label=s19_pp if s19_gg == "G06" else None)
        for s19_xv, s19_yv, s19_nv in zip(s19_tt["value"], s19_tt["rate"], s19_tt["n"]):
            ax.annotate(f"{s19_nv:,}", (s19_xv, s19_yv), xytext=(0, 5), textcoords="offset points", fontsize=6, ha="center")
    ax.axhline(BASE_RATE, ls="--", color="k", lw=1); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(f"{s19_gg} {S19_G[s19_gg]['name']}", fontsize=10); ax.set_xlabel(_t["parameter"].iloc[-1], fontsize=8)
    if s19_gg == "G06": ax.legend(fontsize=6)
fig.suptitle("19.17 Threshold sensitivity (labels = group size)", fontsize=14, fontweight="bold")
plt.tight_layout(); savefig("s19_threshold_sensitivity"); plt.show()
S19_SENS_SUM = S19_SENS.groupby("group_id").agg(min_rate=("rate", "min"), max_rate=("rate", "max"), min_n=("n", "min"), max_n=("n", "max"))
S19_SENS_SUM["always_same_side_of_base"] = S19_SENS.groupby("group_id")["rate"].apply(lambda s: bool((s.dropna() > BASE_RATE).all() or (s.dropna() < BASE_RATE).all()))
display(S19_SENS_SUM.round(3)); export(S19_SENS, "s19_threshold_sensitivity")
add_fact("19.17", "case studies", f"{int(S19_SENS_SUM['always_same_side_of_base'].sum())} of {len(S19_SENS_SUM)} threshold-based groups stay on the same side of the base rate across every cut-off tried", float(S19_SENS_SUM["always_same_side_of_base"].mean()), "s19_threshold_sensitivity", 1)

**Observations:**
* Four of the five threshold-based groups stay on the same side of base at every cut-off: G05 13.1-18.2%, G06 11.4-17.7%, G07 25.3-31.5% and G12 12.4-15.9%.
* G10 is the exception. With a strict "paid nothing" rule it reaches 46.0%, but at "paid at most half the usual" it falls to 19.6%, below base. Only a true stop is a warning.
* Story use: the myths and surprises do not depend on the thresholds. The payment-collapse warning applies only to clients who stop paying completely.

### 19.18 What does each case mean for the story?
One line per group, built from the numbers: is the rule a **myth**, a **warning**, or a **surprise**? This table is the
script for the case-study panel of the dashboard.

In [ ]:
def s19_meaning(r):
    if r["role"] == "Myth":
        return (f"Myth: expected {'low' if r['expectation'] == 'safe' else 'high'} risk, but the group defaults at {r['default_rate']:.1%} "
                f"(lift {r['lift']:.2f}); the rule does not hold.")
    if r["role"] == "Warning":
        return f"Warning: {r['default_rate']:.1%} default ({r['lift']:.1f}x base); act on it, even though {r['n_breakers']:,} clients ({r['breaker_share_of_group']:.0%}) survive."
    return (f"Surprise: the rule holds on average ({r['default_rate']:.1%}, lift {r['lift']:.2f}), yet {r['n_breakers']:,} clients "
            f"({r['breaker_share_of_group']:.0%} of the group) {r['rule_breaker']} against it.")
S19_DEF["meaning"] = S19_DEF.apply(s19_meaning, axis=1)
if len(S19_AUC):
    S19_DEF = S19_DEF.drop(columns=[c for c in ["within_group_auc", "top_separator"] if c in S19_DEF.columns]).merge(S19_AUC[["group_id", "within_group_auc", "top_separator"]], on="group_id", how="left")
with pd.option_context("display.max_colwidth", 200):
    display(S19_DEF[["group_id", "group", "role", "meaning"]])
export(S19_DEF, "s19_group_definitions")

**Observations:**
* The 12 groups split into 3 warnings, 3 myths and 6 surprises.
* Every surprise is a "safe" rule that leaks 12-16% defaulters, except deep over-limit, where 70% survive a "risky" rule.
* Story use: use the meaning column as captions in the Tableau case-study panel.

### 19.19 Client-level flags and Tableau exports
`CS_RULE_BREAKER` marks a client who breaks at least one rule; `CS_PRIMARY_GROUP` assigns each client to the *smallest*
group it belongs to (the most specific story). Exports: `s19_case_cards_long` (one row per client × month × metric, for a
Tableau small-multiple case card), `s19_client_groups` (client-level membership) and the group-definition table.

In [ ]:
S19_BRK = np.column_stack([S19_G[g]["breakers"] for g in S19_IDS]).astype(int)
df["CS_N_BREAKS"] = S19_BRK.sum(1)
df["CS_RULE_BREAKER"] = (df["CS_N_BREAKS"] > 0).astype(int)
df["CS_DEMO_SCORE"] = S19_DEMO_SCORE
_order = np.argsort(_sz)                                    # smallest group first
_prim = np.full(S19_N, "None", dtype=object)
for s19_j in _order[::-1]:
    _prim[S19_MEM[:, s19_j] == 1] = f"{S19_IDS[s19_j]} {S19_G[S19_IDS[s19_j]]['name']}"
df["CS_PRIMARY_GROUP"] = _prim
S19_PRIM = rate_table(df, "CS_PRIMARY_GROUP")
display(S19_PRIM.round(3))
print(f"Rule-breakers in at least one group: {df['CS_RULE_BREAKER'].sum():,} ({df['CS_RULE_BREAKER'].mean():.1%} of clients)")
S19_CARDS_LONG = pd.DataFrame([_r for _g in S19_IDS for _r in S19_CARDS.get(_g, [])])
print(S19_CARDS_LONG.groupby(["group_id", "card_role"])["client_id"].first().unstack().to_string())
export(S19_CARDS_LONG, "s19_case_cards_long")
S19_CS_COLS = [S19_G[g]["col"] for g in S19_IDS] + ["CS_N_GROUPS", "CS_N_BREAKS", "CS_RULE_BREAKER", "CS_PRIMARY_GROUP", "CS_DEMO_SCORE"]
export(df[["ID", TARGET] + S19_CS_COLS], "s19_client_groups")
add_fact("19.19", "case studies", f"{df['CS_RULE_BREAKER'].mean():.0%} of all clients break at least one of the twelve rules", float(df["CS_RULE_BREAKER"].mean()), "s19_gallery", 1)

**Observations:**
* 6,794 clients (22.6%) break at least one rule.
* Assigning each client to their most specific group gives a clean ladder: safest demographic profile 8.4%, clean payers 11.3%, then 2m+ late in Sep 73.0%. The "None" group sits at 37.7%.
* The case-card table holds 36 clients (12 groups x 3 roles) in 1,080 long-format rows, ready for a Tableau small-multiple card.

### 19.20 Section 19 facts and new columns

| Column | Meaning |
|---|---|
| `CS_G01_CLEAN_PAYER` | 1 = status ≤ 0 in all six months (never late) |
| `CS_G02_LATE_SEP_2P` | 1 = September status ≥ 2 |
| `CS_G03_PAID_FULL_AMT` | 1 = every May–Sep payment ≥ the previous bill (by amount), at least one positive bill |
| `CS_G04_NEVER_USED` | 1 = status −2 (no consumption) in all six months |
| `CS_G05_MAXED_CLEAN` | 1 = utilisation > 95% in all six months and never late |
| `CS_G06_BIG_YOUNG` | 1 = limit in the top 20% and age in the youngest 20% |
| `CS_G07_DEEP_OVERLIMIT` | 1 = bill > 110% of limit in at least one month |
| `CS_G08_SUDDEN_COLLAPSE` | 1 = never late Apr–Aug, late (status ≥ 1) in Sep |
| `CS_G09_CURED` | 1 = late at least once in Apr–Jun, never late Jul–Sep |
| `CS_G10_PAY_COLLAPSE` | 1 = paid every month May–Aug, paid nothing in Sep with a positive Aug bill |
| `CS_G11_CREDIT_BAL` | 1 = negative bill (credit balance) in at least one month |
| `CS_G12_SAFEST_DEMO` | 1 = bottom 10% of the demographic-only risk score (education known) |
| `CS_N_GROUPS` | number of the twelve groups the client belongs to |
| `CS_N_BREAKS` | number of groups in which the client is a rule-breaker (defaulted in a safe group / survived in a risky one) |
| `CS_RULE_BREAKER` | 1 = rule-breaker in at least one group |
| `CS_PRIMARY_GROUP` | smallest (most specific) group the client belongs to, or "None" |
| `CS_DEMO_SCORE` | demographic-only default score (logistic on sex, education, marriage, age band, limit band; in-sample, descriptive) |

**Key Tableau exports:** `s19_group_definitions` (rules, n, rate, CI, lift, role, meaning), `s19_case_cards_long`
(case-card rows, long format), `s19_client_groups`, `s19_overlap_jaccard`, `s19_breaker_smd`, `s19_breaker_separability`,
`s19_threshold_sensitivity`.

In [ ]:
print(f"{len(S19_CS_COLS)} CS_ columns added:", S19_CS_COLS)
S19_FACTS = pd.DataFrame([f for f in STORY_FACTS if f["section"].startswith("19")])
with pd.option_context("display.max_colwidth", None):
    display(S19_FACTS)
print(f"Section 19 runtime: {time.time() - S19_T0:.0f}s")

**Observations:**
* The section added 17 CS_ columns and 36 story facts, and ran in about 9 s on CPU.
* Story use: the headline facts are the 725 full payers who default, maxed-out-but-never-late at 13.1%, cured at 22.5%, and the safest demographic profile (12.4% overall) jumping to 33.9% for its members who were late in September.

## 19B. Ideas borrowed from AmEx, Home Credit and other competitions

**Why this section matters for the story.** Thousands of analysts have studied credit default on Kaggle. The best of them
(American Express Default Prediction 2022, Home Credit Default Risk 2018, Give Me Some Credit 2011, and the top notebooks on
this very dataset) found a handful of *ways of looking* that keep working. This section borrows only the ideas that the
earlier sections of this notebook have **not** already used, and asks whether each one tells us something new about these
30,000 Taiwanese card holders. Nothing here is a prediction deliverable: small models appear only as a lens.

| Idea | Borrowed from | Subsection |
|---|---|---|
| Summarise every monthly series with `last, first, mean, min, max, std, last-mean, last-first` and ask which summary carries the signal | AmEx 2022 (Chris Deotte's XGBoost starter, 2nd place JuneHomes, 21st place write-up) | 19B.1 |
| Does an extra summary still help once the last statement is known? | AmEx "last vs aggregate" debate | 19B.2 |
| Score every driver with AmEx's own metric: top-4% capture and Gini | AmEx competition metric | 19B.3 |
| "After-pay" balance: what is left of a bill once the payment is made | AmEx public notebooks (`B_x - P_2` features) | 19B.4 |
| Statement-level view versus client-level view, and honest error bars | AmEx data layout (13 statements per customer) | 19B.5 |
| Missing statements as signal: the shape of empty months | AmEx (raddar, "Understanding NA values") | 19B.6 |
| Is there injected noise or heaping in the amounts? | AmEx (raddar's de-noised data, ambrosm's EDA) | 19B.7 |
| Drift between periods with adversarial validation and PSI | AmEx and Home Credit train/test drift checks | 19B.8 |
| Does the row order (ID) hide a leak? | Standard Kaggle leak hunt | 19B.9 |
| Sentinel codes and late counts split by severity | Give Me Some Credit (96/98 past-due codes, 30-59/60-89/90+ counts) | 19B.10 |
| Domain ratios: repayment burden and pay-off horizon, read with KDEs by target | Home Credit (credit/annuity, annuity/income) | 19B.11 |

Column prefix for new `df` columns: `CX_`. The full research note is `build/RESEARCH_competitions.md`.

### 19B.0 Building blocks
Chronological matrices (column 0 = Apr ... column 5 = Sep), the AmEx metric, a ranked AUC helper and an out-of-fold
binned-rate scorer. The scorer turns one or two variables into deciles, learns the default rate of each cell on four
fifths of the clients and scores the remaining fifth. It needs no model and cannot overfit much.

In [ ]:
from sklearn.metrics import roc_auc_score
S19B_T0 = time.time()
s19b_y = df[TARGET].to_numpy()
s19b_L = df["LIMIT_BAL"].to_numpy(float)[:, None]
S19B_S = df[[f"PAY_{i}" for i in IDX[::-1]]].to_numpy(float)          # status Apr..Sep
S19B_B = df[[f"BILL_AMT{i}" for i in IDX[::-1]]].to_numpy(float)     # bill Apr..Sep
S19B_P = df[[f"PAY_AMT{i}" for i in IDX[::-1]]].to_numpy(float)      # payment made in Apr..Sep
S19B_U = S19B_B / s19b_L
S19B_AP = S19B_B[:, :5] - S19B_P[:, 1:]                              # after-pay: bill of month m minus payment made in m+1 (May..Sep)
S19B_APU = S19B_AP / s19b_L
S19B_PL = S19B_P / s19b_L                                            # payment as share of the line
S19B_SP = df[[f"SPEND_{i}" for i in IDX[:-1][::-1]]].to_numpy(float) # implied spend May..Sep
S19B_MON5 = MONTH_ORDER[1:]                                          # payment months for after-pay / spend

def s19b_auc(x, y=None):
    # univariate AUC oriented so that 'higher = riskier'; NaN replaced by the median
    y = s19b_y if y is None else y
    x = np.asarray(x, float)
    x = np.where(np.isnan(x), np.nanmedian(x), x)
    a = roc_auc_score(y, x)
    return max(a, 1 - a), (1 if a >= 0.5 else -1)

def s19b_amex(y, score, seed=0):
    # AmEx 2022 metric: M = 0.5 * (normalised weighted Gini + share of defaulters captured in the top 4% of weight)
    # non-defaulters weigh 20 (AmEx had down-sampled them 5%); ties are broken at random so discrete scores are fair
    y = np.asarray(y, int); score = np.asarray(score, float)
    rnd = np.random.default_rng(seed).random(len(y))
    def _parts(s):
        o = np.lexsort((rnd, -s)); yy = y[o]; w = np.where(yy == 0, 20.0, 1.0)
        top = np.cumsum(w) <= 0.04 * w.sum()
        d = yy[top].sum() / yy.sum()
        rand = np.cumsum(w / w.sum()); lor = np.cumsum(yy * w) / (yy * w).sum()
        return d, ((lor - rand) * w).sum()
    d, g = _parts(score); _, gmax = _parts(y.astype(float))
    return 0.5 * (g / gmax + d), g / gmax, d

def s19b_top_capture(y, score, q=0.04, seed=0):
    # plain version: share of all defaulters found in the riskiest q of clients, and the default rate there
    rnd = np.random.default_rng(seed).random(len(y))
    o = np.lexsort((rnd, -np.asarray(score, float))); k = int(round(q * len(y)))
    return y[o[:k]].sum() / y.sum(), y[o[:k]].mean()

def s19b_codes(x, q=10):
    x = pd.Series(np.asarray(x, float))
    c = pd.qcut(x, q, labels=False, duplicates="drop") if x.nunique() > q else pd.Series(pd.factorize(x)[0], dtype=float)
    return c.fillna(-1).astype(int).to_numpy()

S19B_FOLD = RNG.permutation(len(df)) % 5
def s19b_oof_auc(*xs, q=10, a=20.0):
    codes = np.zeros(len(df), np.int64)
    for x in xs:
        c = s19b_codes(x, q); codes = codes * 1000 + (c + 1)
    codes = pd.factorize(codes)[0]; K = codes.max() + 1
    pred = np.zeros(len(df))
    for f in range(5):
        tr = S19B_FOLD != f
        n = np.bincount(codes[tr], minlength=K); k = np.bincount(codes[tr], weights=s19b_y[tr], minlength=K)
        p0 = s19b_y[tr].mean()
        pred[~tr] = ((k + a * p0) / (n + a))[codes[~tr]]
    return roc_auc_score(s19b_y, pred)

_m, _g, _d = s19b_amex(s19b_y, df["PAY_1"].to_numpy(float))
print(f"sanity: AmEx metric of Sep status alone = {_m:.3f} (Gini {_g:.3f}, top-4% capture {_d:.3f}); OOF binned AUC of Sep status = {s19b_oof_auc(df['PAY_1']):.3f}")

### 19B.1 Which summary of six statements carries the signal: the last one, the average, or the extremes?
**The idea.** In the AmEx competition every customer had up to 13 monthly statements. Almost every top solution reduced each
monthly column to the same family of summaries (`last, first, mean, min, max, std`, plus `last - mean` and `last - first`) and
let the model choose. Chris Deotte's starter notebook and the 2nd-place team both reported that `last` and `mean` carry most of
the weight. ([competition](https://www.kaggle.com/c/amex-default-prediction), [21st place write-up](https://www.kaggle.com/competitions/amex-default-prediction/writeups/angus-ryota-joe-21st-solution-and-code-sharing))

**Why it might matter here.** Section 16.11 built a shape-oriented bank (slope, r², autocorrelation) and Section 11.11 compared
September with the six-month mean. Neither looked at **min and max**, at the AmEx `last - mean` delta, or at the two series
this section adds: the after-pay balance and the payment as a share of the line.

**How to read it.** Each cell is the univariate AUC (0.5 = no signal, oriented so that higher always means riskier) of one
summary of one series. Read across a row to see which summary wins for that series; read down a column to see which series
benefits most from that summary.

In [ ]:
S19B_SERIES = {"status": S19B_S, "bill": S19B_B, "payment": S19B_P, "utilisation": S19B_U, "after-pay bal": S19B_AP,
               "after-pay util": S19B_APU, "pay / limit": S19B_PL, "spend": S19B_SP}
S19B_AGGS = ["last", "first", "mean", "min", "max", "std", "last-mean", "last-first"]
def s19b_aggregate(M):
    mean = M.mean(1)
    return {"last": M[:, -1], "first": M[:, 0], "mean": mean, "min": M.min(1), "max": M.max(1), "std": M.std(1),
            "last-mean": M[:, -1] - mean, "last-first": M[:, -1] - M[:, 0]}
_cols, _rows = {}, []
for s_name, M in S19B_SERIES.items():
    for a_name, v in s19b_aggregate(M).items():
        key = f"{s_name} | {a_name}"; _cols[key] = v
        auc, sign = s19b_auc(v)
        m, g, d = s19b_amex(s19b_y, sign * v)
        _rows.append({"series": s_name, "aggregate": a_name, "feature": key, "AUC": auc, "direction": "+" if sign > 0 else "-",
                      "amex_M": m, "gini_w": g, "top4_capture_w": d})
S19B_AGG = pd.DataFrame(_cols)
S19B_AGG_RANK = pd.DataFrame(_rows).sort_values("AUC", ascending=False).reset_index(drop=True)
S19B_AGG_GRID = S19B_AGG_RANK.pivot(index="series", columns="aggregate", values="AUC").reindex(index=list(S19B_SERIES), columns=S19B_AGGS)
print(f"{S19B_AGG.shape[1]} AmEx-style aggregates computed")
display(S19B_AGG_GRID.round(3))

The heatmap is the series x summary grid. The bar chart on the right asks the AmEx question directly: for each series,
how much does the best non-`last` summary beat `last`? A positive bar means history adds something the latest statement
does not carry on its own; a negative bar means the latest statement is already the best summary.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 5.8), gridspec_kw={"width_ratios": [1.5, 1]})
sns.heatmap(S19B_AGG_GRID, annot=True, fmt=".3f", cmap="YlOrRd", vmin=0.5, ax=axes[0], linewidths=.5, cbar_kws={"label": "AUC"})
axes[0].set_title("Univariate AUC of every AmEx-style summary (rows = monthly series)"); axes[0].set_xlabel(""); axes[0].set_ylabel("")
_best = S19B_AGG_RANK[S19B_AGG_RANK["aggregate"] != "last"].groupby("series").first()
_vs = pd.DataFrame({"last": S19B_AGG_GRID["last"], "best_other": _best["AUC"], "best_other_agg": _best["aggregate"]}).reindex(list(S19B_SERIES))
_vs["gain"] = _vs["best_other"] - _vs["last"]
axes[1].barh(_vs.index[::-1], _vs["gain"][::-1], color=[C_DEF if g > 0 else C_OK for g in _vs["gain"][::-1]])
for i, (g, a) in enumerate(zip(_vs["gain"][::-1], _vs["best_other_agg"][::-1])):
    axes[1].text(g, i, f"  {a}" if g >= 0 else f"{a}  ", va="center", ha="left" if g >= 0 else "right", fontsize=9)
axes[1].axvline(0, color="k", lw=.8); axes[1].set_xlabel("AUC of best other summary minus AUC of 'last'")
axes[1].set_title("Does any summary beat the last statement?")
plt.tight_layout(); savefig("s19b_amex_aggregates"); plt.show()
S19B_FAM = S19B_AGG_RANK.groupby("aggregate").agg(AUC_mean=("AUC", "mean"), AUC_max=("AUC", "max"), best_feature=("feature", "first")).reindex(S19B_AGGS)
S19B_FAM["rank_by_mean"] = S19B_FAM["AUC_mean"].rank(ascending=False).astype(int)
display(S19B_FAM.round(3)); display(_vs.round(3))
S19B_VS_LAST = _vs
export(S19B_AGG_RANK, "s19b_amex_aggregate_rank")
_fb = S19B_FAM["AUC_mean"].idxmax(); _top = S19B_AGG_RANK.iloc[0]
add_fact("19B.1", "AmEx aggregates", f"Across 8 monthly series x 8 AmEx summaries, the '{_fb}' summary has the highest mean AUC ({S19B_FAM.loc[_fb, 'AUC_mean']:.3f}); "
         f"the single best is {_top['feature']} (AUC {_top['AUC']:.3f})", float(S19B_FAM.loc[_fb, "AUC_mean"]), "s19b_amex_aggregates", 2)
_win = _vs[_vs["gain"] > 0.005]
add_fact("19B.1", "AmEx aggregates", f"A non-last summary beats the last statement by more than 0.005 AUC for {len(_win)} of 8 series "
         f"({', '.join(f'{s}: {r.best_other_agg} +{r.gain:.3f}' for s, r in _win.iterrows()) or 'none'})", len(_win), "s19b_amex_aggregates", 2)

**Observations:**
- Status summaries lead every column: `max` status has AUC 0.708, `last` (September) 0.690, `mean` 0.675.
- Money series are weak one by one (AUC 0.50-0.64). The best money summaries are `payment | min` (0.636), `spend | mean/max` (0.631) and `pay / limit | min` (0.619).
- `min` has the highest average AUC across all eight series (0.581). The smallest payment in six months says more than the latest one.
- A non-`last` summary beats `last` for all 8 series. The gains are large for the bill (`std`, +0.085) and pay/limit (`min`, +0.081), and small for status (+0.018).
- `last - mean` and `last - first`, the AmEx favourites, are the weakest families here (mean AUC 0.53-0.54).
- Story use: "the worst month and the smallest payment matter more than the latest bill"; for money, show history, not only September.

### 19B.2 Once the last statement is known, which extra summary still helps?
**The idea.** AmEx teams found that adding `mean`, `max` or `last - mean` on top of `last` kept lifting the score even though
each was correlated with `last`. Here we test that claim without a model: both variables are cut into deciles, the default
rate of every decile x decile cell is learned on four fifths of the clients, and the AUC is measured on the remaining fifth
(5-fold, out-of-fold). The lift is the AUC of `last + summary` minus the AUC of `last` alone, scored the same way.

**Why it might matter here.** If the lift is near zero the story can stay simple: "the latest month says it all". If it is
large for some series, Tableau should show that series as history, not as a snapshot.

**How to read it.** Each cell is the out-of-fold AUC lift in points of AUC (x 100). Above about 0.5 points is worth a chart.

In [ ]:
_rows = []
for s_name, M in S19B_SERIES.items():
    ag = s19b_aggregate(M); base = s19b_oof_auc(ag["last"])
    for a_name in S19B_AGGS[1:]:
        _rows.append({"series": s_name, "aggregate": a_name, "AUC_last": base, "AUC_last_plus": s19b_oof_auc(ag["last"], ag[a_name])})
S19B_INC = pd.DataFrame(_rows); S19B_INC["lift_pts"] = 100 * (S19B_INC["AUC_last_plus"] - S19B_INC["AUC_last"])
_grid = S19B_INC.pivot(index="series", columns="aggregate", values="lift_pts").reindex(index=list(S19B_SERIES), columns=S19B_AGGS[1:])
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(_grid, annot=True, fmt=".1f", cmap=CMAP_DIV, center=0, ax=ax, linewidths=.5, cbar_kws={"label": "AUC lift (points)"})
ax.set_title("Out-of-fold AUC lift from adding one summary to the last statement (x100)"); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); savefig("s19b_lift_over_last"); plt.show()
display(S19B_INC.sort_values("lift_pts", ascending=False).head(12).round(4))
export(S19B_INC, "s19b_lift_over_last")
_b = S19B_INC.sort_values("lift_pts", ascending=False).iloc[0]
_st = S19B_INC[S19B_INC["series"] == "status"]["lift_pts"].max()
add_fact("19B.2", "AmEx aggregates", f"The largest gain from adding a summary to the last statement is {_b['series']} + {_b['aggregate']} "
         f"(+{_b['lift_pts']:.1f} AUC points, {_b['AUC_last']:.3f} -> {_b['AUC_last_plus']:.3f}); for status the best addition gives only +{_st:.1f} points",
         float(_b["lift_pts"]), "s19b_lift_over_last", 2)

**Observations:**
- Adding a second summary to the last bill lifts the out-of-fold AUC the most: `bill + std` goes from 0.525 to 0.627 (+10.2 points).
- Utilisation and after-pay utilisation gain about 6 points from `last - mean`. A balance that has moved away from its own average is a signal.
- Status gains at most +3.4 points, so the September code already holds most of what its history knows.
- `payment + min` adds 5.0 points: one skipped or tiny payment in the window carries risk.
- Story use: in Tableau, show money columns as a 6-month band (volatility, deviation from own mean) and keep status as a snapshot.

### 19B.3 How do our usual drivers look through AmEx's own scoring rule?
**The idea.** AmEx did not score the competition on AUC. Its metric is the average of two parts: the normalised Gini, and
the **share of all defaulters caught in the riskiest 4% of the book** (non-defaulters weighted 20x, because AmEx had
down-sampled them). The 4% part rewards a variable that finds the worst clients at the very top, which is what a collections
team cares about. ([metric notebook](https://www.kaggle.com/code/inversion/amex-competition-metric-python))

**Why it might matter here.** A variable can have a fine AUC yet be useless at the very top (it ranks the middle well), and
the reverse. Here we also give an unweighted, plain-English version: of the 4% riskiest clients by this variable, what
share default, and what share of all defaulters are they?

**How to read it.** In the scatter, the x-axis is Gini (whole ranking), the y-axis the weighted top-4% capture.
Points high and to the left are "sharp at the top"; points low and to the right are "good on average, blunt at the top".

In [ ]:
S19B_DRIVERS = {"Sep status (PAY_1)": df["PAY_1"], "Max delay 6m": df["MAX_DELAY"], "Months late": df["N_LATE"],
                 "Late streak (recent)": df["LATE_STREAK_RECENT"], "Avg utilisation": df["AVG_UTIL"], "Credit limit": df["LIMIT_BAL"],
                 "Bills repaid (5 statements, aligned)": df["PAY_TO_BILL_5M"], "Zero-pay months": df["N_ZERO_PAY"], "Age": df["AGE"],
                 "status | mean": S19B_AGG["status | mean"], "status | max": S19B_AGG["status | max"],
                 "after-pay util | last": S19B_AGG["after-pay util | last"], "pay / limit | mean": S19B_AGG["pay / limit | mean"],
                 "payment | max": S19B_AGG["payment | max"], "utilisation | min": S19B_AGG["utilisation | min"]}
_rows = []
for nm, v in S19B_DRIVERS.items():
    v = np.asarray(v, float); auc, sign = s19b_auc(v); sc = sign * np.where(np.isnan(v), np.nanmedian(v), v)
    m, g, d = s19b_amex(s19b_y, sc); cap, prec = s19b_top_capture(s19b_y, sc)
    _rows.append({"driver": nm, "AUC": auc, "amex_M": m, "gini_w": g, "top4_capture_w": d, "top4pct_share_of_defaulters": cap, "top4pct_default_rate": prec})
S19B_AMEX = pd.DataFrame(_rows).sort_values("amex_M", ascending=False).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 6.5))
ax.scatter(S19B_AMEX["gini_w"], S19B_AMEX["top4_capture_w"], s=60, color=C_DEF)
for _, r in S19B_AMEX.iterrows():
    ax.annotate(r["driver"], (r["gini_w"], r["top4_capture_w"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
ax.set_xlabel("normalised weighted Gini"); ax.set_ylabel("weighted top-4% capture (share of defaulters)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_title("AmEx metric view of the main drivers: whole-book ranking vs sharpness at the top")
plt.tight_layout(); savefig("s19b_amex_metric"); plt.show()
display(S19B_AMEX.round(3))
export(S19B_AMEX, "s19b_amex_metric_scoreboard")
_r0 = S19B_AMEX.iloc[0]
add_fact("19B.3", "AmEx metric", f"Scored with AmEx's metric, {_r0['driver']} leads (M = {_r0['amex_M']:.3f}); its riskiest 4% of clients default at "
         f"{_r0['top4pct_default_rate']:.0%} vs a {BASE_RATE:.0%} base rate", float(_r0["amex_M"]), "s19b_amex_metric", 2)
_r1 = S19B_AMEX.sort_values("top4pct_default_rate", ascending=False).iloc[0]
add_fact("19B.3", "AmEx metric", f"The sharpest variable at the very top is {_r1['driver']}: {_r1['top4pct_default_rate']:.0%} of its top-4% clients default, "
         f"catching {_r1['top4pct_share_of_defaulters']:.1%} of all defaulters", float(_r1["top4pct_default_rate"]), "s19b_amex_metric", 1)

**Observations:**
- On AmEx's metric, months late leads (M = 0.343), ahead of Sep status (0.334) and the recent late streak (0.313).
- Sep status catches the most defaulters in the weighted top 4% (29.2%). Months late has the best Gini (0.450).
- Every status-based driver puts about 70% defaulters in its riskiest 4% (status mean: 72.9%). No money driver gets above 39% (number from the pre-fix run; refresh after the next Kaggle run).
- Credit limit, zero-pay months and max payment reach only 37-39% at the top. They rank the middle of the book, not the extremes.
- Age is close to random on this metric (M = 0.028).
- Story use: for "who to call first", use delinquency counts; money variables help rank the middle, not the top.

### 19B.4 Is it the bill, or what is left after paying it, that predicts default?
**The idea.** AmEx public notebooks popularised "after-pay" features: a balance column minus a payment column
(`B_x - P_2`). The thinking is that a large bill that is paid off is harmless, while the same bill with little paid is debt.

**Why it might matter here.** Section 12.1 proved that `PAY_AMT(i)` settles `BILL_AMT(i+1)`, so the right after-pay balance
here is `bill of month m - payment made in month m+1` (`CX_AP_*`). Naive copies of the AmEx feature subtract the payment of the
*same* row. We compute both and let the data say which timing carries signal. Section 7's `FE_REVOLVE_BAL` is the mean of the
positive part of this balance; here we look at it month by month, as utilisation, and against the bill itself.

**How to read it.** Left: AUC per month for the bill, the correctly-timed after-pay balance and the naive same-row version,
all as a share of the limit. Right: default rate by Aug bill utilisation band (columns) and by how much of that bill was
left after the Sep payment (rows). Reading down a column holds the bill fixed and varies only what was paid.

In [ ]:
for k, m in enumerate(S19B_MON5):
    df[f"CX_AP_{m.upper()}"] = S19B_AP[:, k]
df["CX_AP_UTIL_LAST"] = S19B_APU[:, -1]
df["CX_AP_UTIL_MEAN"] = S19B_APU.mean(1)
_naive = (S19B_B - S19B_P) / s19b_L
_rows = []
for k, m in enumerate(MONTH_ORDER):
    r = {"month": m, "bill util": s19b_auc(S19B_U[:, k])[0], "after-pay util, naive same-row": s19b_auc(_naive[:, k])[0]}
    if k < 5:
        r["after-pay util, correct timing (bill m - pay m+1)"] = s19b_auc(S19B_APU[:, k])[0]
    _rows.append(r)
S19B_AP_AUC = pd.DataFrame(_rows).set_index("month")

_aug_u = S19B_U[:, 4]; _left = np.where(S19B_B[:, 4] > 0, S19B_AP[:, 4] / np.where(S19B_B[:, 4] > 0, S19B_B[:, 4], 1), np.nan)
_t = pd.DataFrame({"bill_band": pd.cut(_aug_u, [-np.inf, 0, .3, .6, .9, np.inf], labels=["<=0", "0-30%", "30-60%", "60-90%", ">90%"]),
                   "left_band": pd.cut(_left, [-np.inf, 0, .5, .9, .99, np.inf], labels=["overpaid (<0)", "0-50% left", "50-90% left", "90-99% left", "~all left (>=99%)"]),
                   TARGET: s19b_y})
S19B_AP_GRID = _t.groupby(["left_band", "bill_band"], observed=True)[TARGET].agg(["mean", "size"]).reset_index()
_pv = S19B_AP_GRID.pivot(index="left_band", columns="bill_band", values="mean")
_nv = S19B_AP_GRID.pivot(index="left_band", columns="bill_band", values="size")

fig, axes = plt.subplots(1, 2, figsize=(17, 5.5), gridspec_kw={"width_ratios": [1, 1.2]})
for c, col in zip(S19B_AP_AUC.columns, [C_NEU, C_DEF, C_ACC]):
    axes[0].plot(S19B_AP_AUC.index, S19B_AP_AUC[c], marker="o", lw=2, label=c, color=col)
axes[0].set_ylabel("univariate AUC"); axes[0].set_title("Bill vs after-pay balance (both / limit), by month"); axes[0].legend(fontsize=8)
sns.heatmap(_pv, annot=True, fmt=".0%", cmap=CMAP_RATE, ax=axes[1], cbar_kws={"label": "default rate"})
axes[1].set_title("Default rate: Aug bill utilisation (cols) x share of it left after the Sep payment (rows)")
axes[1].set_xlabel("Aug bill / limit"); axes[1].set_ylabel("")
plt.tight_layout(); savefig("s19b_after_pay"); plt.show()
display(S19B_AP_AUC.round(3)); print("cell sizes:"); display(_nv)
export(S19B_AP_GRID, "s19b_after_pay_grid")
_hi = S19B_AP_GRID[(S19B_AP_GRID["bill_band"] == "60-90%")].set_index("left_band")["mean"]
add_fact("19B.4", "after-pay", f"Among clients with an Aug bill at 60-90% of the limit, default runs from {_hi.min():.0%} to {_hi.max():.0%} depending only on how much "
         f"of that bill was left after the Sep payment", float(_hi.max() - _hi.min()), "s19b_after_pay", 3)
add_fact("19B.4", "after-pay", f"The correctly timed after-pay utilisation (Aug bill - Sep payment) has AUC {S19B_AP_AUC.loc['Aug'].iloc[2]:.3f} vs "
         f"{S19B_AP_AUC.loc['Aug', 'bill util']:.3f} for the Aug bill alone and {S19B_AP_AUC.loc['Aug', 'after-pay util, naive same-row']:.3f} for the naive same-row version",
         float(S19B_AP_AUC.loc["Aug"].iloc[2]), "s19b_after_pay", 2)

**Observations:**
- With correct timing (Aug bill minus Sep payment), after-pay utilisation beats the bill every month: AUC 0.582-0.587 vs 0.551-0.566.
- The naive same-row version sits between them (0.564-0.578). Matching the payment to the bill it settles adds signal.
- For an Aug bill at 60-90% of the limit, default runs from 10.7% (overpaid) to 58.4% (about all of the bill left unpaid).
- Clients who left about all of the bill unpaid (>= 99%) default at 47-58% in every bill band. The bill size barely matters.
- Story use: "it is not how much you owe, it is how much you leave": a heatmap with the bill held fixed and only the repayment changing.

#### 19B.4.1 Over-limit after paying: the Give-Me-Some-Credit "utilisation above 1" view
Give Me Some Credit's `RevolvingUtilizationOfUnsecuredLines` had values far above 1 that turned out to be both real and very
risky. The analogue here is an after-pay balance above the limit: even after paying, the client still owes more than the line.
Bars: share of clients in each Sep after-pay band; line: default rate.

In [ ]:
df["CX_AP_UTIL_BAND"] = pd.cut(df["CX_AP_UTIL_LAST"], [-np.inf, -0.0001, 0.0001, .3, .6, .9, 1.0, 1.1, np.inf],
                                labels=["credit (<0)", "zero", "0-30%", "30-60%", "60-90%", "90-100%", "100-110%", ">110%"])
fig, ax = plt.subplots(figsize=(11, 4.5))
S19B_AP_BAND = plot_rate(df, "CX_AP_UTIL_BAND", ax=ax, title="Default rate by after-pay utilisation (Aug bill - Sep payment) / limit")
plt.tight_layout(); savefig("s19b_after_pay_band"); plt.show()
display(S19B_AP_BAND[["CX_AP_UTIL_BAND", "n", "rate", "ci_lo", "ci_hi", "lift"]].round(3))
export(S19B_AP_BAND, "s19b_after_pay_band")
_ov = S19B_AP_BAND[S19B_AP_BAND["CX_AP_UTIL_BAND"].astype(str).isin(["100-110%", ">110%"])]
add_fact("19B.4", "after-pay", f"{int(_ov['n'].sum()):,} clients still owe more than their limit after the Sep payment; they default at "
         f"{_ov['defaults'].sum() / _ov['n'].sum():.1%}", float(_ov["defaults"].sum() / _ov["n"].sum()), "s19b_after_pay_band", 2)

**Observations:**
- Default rises steadily with after-pay utilisation: 12.6% when the client is in credit, 18.1% at zero, 27.8% at 60-90%.
- 652 clients still owe more than the limit after paying. Those at 100-110% default at 40.5%.
- Above 110% the rate falls back to 29.2% (n = 319). Like GMSC's extreme utilisation, the far tail mixes real risk with odd accounts.
- Story use: "over the limit even after paying" is a clear alarm at 100-110%; do not extrapolate it to the extreme tail.

### 19B.5 Do we learn the same thing from 180,000 statements as from 30,000 clients?
**The idea.** AmEx shipped the data at statement level (one row per customer per month) and the label at customer level.
Many early notebooks plotted statement rows directly, which repeats each customer up to 13 times. Top notebooks always
aggregated to the customer first, or at least used customer-level error bars.

**Why it might matter here.** Section 5.1 and Section 21 export a long client x month table for Tableau. A Tableau user who
drags `utilisation` onto a chart with the long table is counting statements, not people. This subsection shows how much the
picture and its confidence intervals change.

**How to read it.** Left: default rate by utilisation band, counted three ways: every statement (180k rows), September only
(one row per client) and the client's six-month mean. Right: the width of the 95% CI per band, naive (statements treated as
independent) versus a client-cluster bootstrap. A ratio above 1 means the naive interval is too narrow.

In [ ]:
_bands = [-np.inf, 0, .1, .3, .5, .7, .9, 1.0, np.inf]; _lab = ["<=0", "0-10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%", ">100%"]
_stmt = pd.DataFrame({"band": pd.cut(S19B_U.ravel(), _bands, labels=_lab), "y": np.repeat(s19b_y, 6), "client": np.repeat(np.arange(len(df)), 6)})
_v_stmt = _stmt.groupby("band", observed=True)["y"].agg(["mean", "size"])
_v_sep = pd.DataFrame({"band": pd.cut(S19B_U[:, -1], _bands, labels=_lab), "y": s19b_y}).groupby("band", observed=True)["y"].agg(["mean", "size"])
_v_cli = pd.DataFrame({"band": pd.cut(S19B_U.mean(1), _bands, labels=_lab), "y": s19b_y}).groupby("band", observed=True)["y"].agg(["mean", "size"])
# naive Wilson CI vs client-cluster bootstrap for the statement-level rates
_codes = _stmt["band"].cat.codes.to_numpy(); _nb = len(_lab)
_cnt = np.zeros((len(df), _nb)); np.add.at(_cnt, (_stmt["client"].to_numpy(), _codes), 1)
_dft = _cnt * s19b_y[:, None]
_boot = np.empty((200, _nb))
for b in range(200):
    w = np.bincount(RNG.integers(0, len(df), len(df)), minlength=len(df)).astype(float)
    _boot[b] = (w @ _dft) / np.maximum(w @ _cnt, 1)
_lo, _hi = wilson_ci(_v_stmt["mean"].to_numpy() * _v_stmt["size"].to_numpy(), _v_stmt["size"].to_numpy())
S19B_STMT = pd.DataFrame({"band": _lab, "rate_statement": _v_stmt["mean"].reindex(_lab).to_numpy(), "n_statements": _v_stmt["size"].reindex(_lab).to_numpy(),
                          "rate_sep_only": _v_sep["mean"].reindex(_lab).to_numpy(), "n_sep": _v_sep["size"].reindex(_lab).to_numpy(),
                          "rate_client_mean": _v_cli["mean"].reindex(_lab).to_numpy(), "n_client_mean": _v_cli["size"].reindex(_lab).to_numpy(),
                          "ci_width_naive": (_hi - _lo), "ci_width_cluster": np.percentile(_boot, 97.5, 0) - np.percentile(_boot, 2.5, 0)})
S19B_STMT["ci_inflation"] = S19B_STMT["ci_width_cluster"] / S19B_STMT["ci_width_naive"]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
x = np.arange(_nb)
for c, lab, col in [("rate_statement", "every statement (180k rows)", C_NEU), ("rate_sep_only", "September only", C_DEF), ("rate_client_mean", "client 6-month mean", C_OK)]:
    axes[0].plot(x, S19B_STMT[c], marker="o", lw=2, label=lab, color=col)
axes[0].set_xticks(x); axes[0].set_xticklabels(_lab); axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].axhline(BASE_RATE, ls="--", color="k", lw=1); axes[0].legend(); axes[0].set_title("Default rate by utilisation band: three ways to count")
axes[1].bar(x, S19B_STMT["ci_inflation"], color=C_ACC); axes[1].axhline(1, color="k", lw=1)
axes[1].set_xticks(x); axes[1].set_xticklabels(_lab); axes[1].set_ylabel("cluster CI width / naive CI width")
axes[1].set_title("How much too narrow are statement-level error bars?")
plt.tight_layout(); savefig("s19b_statement_vs_client"); plt.show()
_auc_stmt = roc_auc_score(np.repeat(s19b_y, 6), S19B_U.ravel())
print(f"AUC of utilisation: statement level {_auc_stmt:.3f} | Sep only {s19b_auc(S19B_U[:, -1])[0]:.3f} | client mean {s19b_auc(S19B_U.mean(1))[0]:.3f}")
display(S19B_STMT.round(4))
export(S19B_STMT, "s19b_statement_vs_client")
add_fact("19B.5", "statement vs client", f"Treating the 180,000 client-months as independent makes the error bars too narrow by a factor of "
         f"{S19B_STMT['ci_inflation'].median():.1f} (median over utilisation bands): each client is counted six times", float(S19B_STMT["ci_inflation"].median()), "s19b_statement_vs_client", 2)

**Observations:**
- The three counting methods agree on the shape: default rises from about 16-17% at 0-30% utilisation to 30-34% above 100%.
- They disagree at the edges. For "<= 0" the client-mean view gives 34.4% vs 23.5% per statement. For 90-100%, it gives 32.2% vs 24.3% for September only.
- Statement-level error bars are too narrow by a median factor of 1.9 (range 1.7-2.2), because each client is counted six times.
- Pooled statement AUC (0.561) is close to the client-mean AUC (0.564). Pooling does not add information, it only fakes precision.
- Story use: build Tableau rate charts from the client table; if the long table is used, count distinct clients and show cluster-honest bands.

### 19B.6 Does an empty statement carry signal, the way missing values did at AmEx?
**The idea.** In AmEx, missing values were not random: raddar's notebook showed that *which* columns were empty, and how many
statements a customer had, were strong signals in their own right (new customers with few statements behaved differently).
([Understanding NA values in AMEX](https://www.kaggle.com/code/raddar/understanding-na-values-in-amex-competition))

**Why it might matter here.** This file has no NaNs, but it has their twin: an **empty statement**, where the bill is 0 and
nothing is paid. Section 1 counted zeros; here we look at the **pattern** of empty months over time. Empty months at the start
look like a new or re-opened account; at the end, like a card that went quiet; in the middle, like intermittent use.

**How to read it.** Each pattern reads Apr -> Sep; `E` = empty month, `.` = active month. The chart shows default rate by
pattern type and by the number of active statements (the AmEx "statement count").

In [ ]:
_E = (S19B_B == 0) & (S19B_P == 0)
df["CX_N_ACTIVE_STMT"] = 6 - _E.sum(1)
df["CX_EMPTY_PATTERN"] = ["".join("E" if e else "." for e in r) for r in _E]
def s19b_ptype(p):
    if "E" not in p: return "All active"
    if "." not in p: return "Never active"
    s = p.strip("E")
    if "E" in s: return "Gaps (intermittent)"
    if p.startswith("E") and p.endswith("E"): return "Active only mid-period"
    if p.startswith("E"): return "Empty early, active late (new/re-opened)"
    return "Active early, empty late (went quiet)"
df["CX_EMPTY_TYPE"] = df["CX_EMPTY_PATTERN"].map(s19b_ptype)
S19B_ETYPE_ORDER = ["All active", "Empty early, active late (new/re-opened)", "Gaps (intermittent)", "Active only mid-period", "Active early, empty late (went quiet)", "Never active"]
fig, axes = plt.subplots(1, 2, figsize=(17, 5), gridspec_kw={"width_ratios": [1.4, 1]})
S19B_ETYPE = plot_rate(df, "CX_EMPTY_TYPE", ax=axes[0], order=S19B_ETYPE_ORDER, rotate=20, title="Default rate by shape of empty months")
S19B_NACT = plot_rate(df, "CX_N_ACTIVE_STMT", ax=axes[1], title="Default rate by number of active statements (of 6)")
plt.tight_layout(); savefig("s19b_empty_statements"); plt.show()
display(S19B_ETYPE[["CX_EMPTY_TYPE", "n", "rate", "ci_lo", "ci_hi", "lift"]].round(3))
S19B_EPAT = rate_table(df, "CX_EMPTY_PATTERN", min_n=80).sort_values("n", ascending=False)
display(S19B_EPAT[["CX_EMPTY_PATTERN", "n", "rate", "ci_lo", "ci_hi", "lift"]].head(16).round(3))
export(S19B_ETYPE, "s19b_empty_type"); export(S19B_EPAT, "s19b_empty_pattern")
_et = S19B_ETYPE.set_index("CX_EMPTY_TYPE")
add_fact("19B.6", "empty statements", f"Clients with no activity in any of the six months default at {_et.loc['Never active', 'rate']:.1%} "
         f"({int(_et.loc['Never active', 'n']):,} clients), above the {_et.loc['All active', 'rate']:.1%} of always-active clients: an empty card is not a safe card",
         float(_et.loc["Never active", "rate"]), "s19b_empty_statements", 3)
add_fact("19B.6", "empty statements", f"Accounts that start empty and become active later (new or re-opened) default at {_et.loc['Empty early, active late (new/re-opened)', 'rate']:.1%}; "
         f"those that go quiet late default at {_et.loc['Active early, empty late (went quiet)', 'rate']:.1%}",
         float(_et.loc["Empty early, active late (new/re-opened)", "rate"]), "s19b_empty_statements", 2)

**Observations:**
- 84.7% of clients (25,415) are active in all six months and default at 21.9%, the base rate.
- The 795 never-active clients default at 37.6%, 1.7x the base. An empty card is not a safe card.
- New or re-opened accounts (empty early, active late, n = 2,286) default at 21.8%, no different from always-active.
- Intermittent users (gaps, n = 863) default least, at 15.6%. Accounts that went quiet late default at 20.4%.
- Story use: "never used" is its own risk segment; it is invisible to every money-based chart because all its amounts are zero.

#### 19B.6.1 What do never-active defaulters look like?
If a client never used the card, what does their status code say? This table shows the status codes and limits of the
never-active group by outcome. A default with zero balance is a label we cannot explain from the money columns.

In [ ]:
_na = df[df["CX_EMPTY_TYPE"] == "Never active"]
_t = pd.DataFrame({"share Sep status -2": _na.groupby(TARGET)["PAY_1"].apply(lambda s: (s == -2).mean()),
                   "share all six status -2": _na.groupby(TARGET)["N_INACTIVE"].apply(lambda s: (s == 6).mean()),
                   "median limit": _na.groupby(TARGET)["LIMIT_BAL"].median(), "median age": _na.groupby(TARGET)["AGE"].median(),
                   "n": _na.groupby(TARGET).size()})
display(_t.round(3))
print("Sep status of never-active clients:"); display(pd.crosstab(_na["PAY_1"], _na[TARGET], margins=True))
display(rate_table(_na, "LIMIT_BAND")[["LIMIT_BAND", "n", "rate", "ci_lo", "ci_hi"]].round(3))

**Observations:**
- Only 34-40% of never-active clients show status -2 in every month. In September, 495 of the 795 (62%) are coded '1 Late 1m' with a zero bill and no payment.
- Never-active defaulters have higher limits (median 230k vs 200k) and are older (37 vs 33.5).
- The default rate climbs with the limit, up to 52.8% for 300-500k lines (n = 178).
- These defaults cannot come from card spending. They point to other debt at the same bank or to label noise (see 12.5).
- Story use: a caveat slide; high-limit dormant cards defaulting at over 50% are a puzzle the data cannot solve.

### 19B.7 Was noise injected into the amounts, and which exact amounts repeat too often?
**The idea.** AmEx added small random noise to every float to anonymise the data; raddar and ambrosm detected it by looking at
the fine structure of the values (gaps, terminal digits) and released a de-noised version that helped every team.
([AMEX EDA which makes sense](https://www.kaggle.com/code/ambrosm/amex-eda-which-makes-sense))

**Why it might matter here.** Section 12 showed that payments heap on round thousands. Bills should **not** heap: a statement
balance is the sum of many purchases. Any bill value that repeats far more often than its neighbours is a fixed charge (a fee,
an instalment or a standing order), and tells us about the product, not about spending.

**How to read it.** Left: share of non-zero amounts by last digit (uniform = 10%). Right: the most repeated exact bill values,
with the default rate of clients who had that bill in any month.

In [ ]:
_bn = S19B_B[S19B_B != 0]; _pn = S19B_P[S19B_P > 0]
_dig = pd.DataFrame({"bill": pd.Series(np.abs(_bn).astype(np.int64) % 10).value_counts(normalize=True).sort_index(),
                     "payment": pd.Series(_pn.astype(np.int64) % 10).value_counts(normalize=True).sort_index()}).fillna(0)
_chi_b = stats.chisquare(np.bincount(np.abs(_bn).astype(np.int64) % 10, minlength=10)).pvalue
print(f"Non-integer bills: {int((S19B_B % 1 != 0).sum())} | non-integer payments: {int((S19B_P % 1 != 0).sum())}  -> no AmEx-style float noise")
print(f"bill last digit: share ending in 0 = {_dig.loc[0, 'bill']:.1%} (chi-square p = {_chi_b:.2g}); payment ending in 0 = {_dig.loc[0, 'payment']:.1%}")
_vc = pd.Series(_bn).value_counts()
_heap = []
for v, n in _vc.head(12).items():
    m = (S19B_B == v).any(1)
    near = pd.Series(_bn)[(np.abs(_bn - v) <= max(20, abs(v) * 0.05)) & (_bn != v)].value_counts()
    _heap.append({"bill_value": v, "statements": int(n), "clients": int(m.sum()), "neighbour_typical_count": float(near.median()) if len(near) else 0.0,
                  "heap_ratio": n / max(1.0, float(near.median()) if len(near) else 1.0), "default_rate": s19b_y[m].mean(),
                  "share_paid_exactly_this": float((S19B_P[:, 1:][S19B_B[:, :5] == v] == v).mean()) if (S19B_B[:, :5] == v).any() else np.nan})
S19B_HEAP = pd.DataFrame(_heap)
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
_dig.plot.bar(ax=axes[0], color=[C_OK, C_DEF]); axes[0].axhline(.1, ls="--", color="k", lw=1)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1)); axes[0].set_title("Last digit of non-zero amounts"); axes[0].set_xlabel("last digit")
axes[1].bar(S19B_HEAP["bill_value"].astype(int).astype(str), S19B_HEAP["statements"], color=C_NEU)
ax2 = axes[1].twinx(); ax2.plot(S19B_HEAP["bill_value"].astype(int).astype(str), S19B_HEAP["default_rate"], "o-", color=C_DEF)
ax2.axhline(BASE_RATE, ls="--", color="k", lw=1); ax2.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax2.grid(False)
axes[1].set_title("Most repeated exact bill values (bars) and default rate of their clients (line)"); axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); savefig("s19b_heaping"); plt.show()
display(S19B_HEAP.round(3))
export(S19B_HEAP, "s19b_bill_heaps")
df["CX_FIXED_390"] = ((S19B_B == 390) | (S19B_B == 780)).any(1).astype(int)
_h0 = S19B_HEAP.iloc[0]
add_fact("19B.7", "heaping", f"The bill value NT$ {_h0['bill_value']:,.0f} appears on {int(_h0['statements']):,} statements ({_h0['heap_ratio']:.0f}x its neighbours) and "
         f"{_h0['share_paid_exactly_this']:.0%} of the following payments are exactly that amount: a fixed recurring charge, not spending", float(_h0["heap_ratio"]), "s19b_heaping", 2)

**Observations:**
- No float noise: every bill and payment is an integer, so AmEx-style de-noising does not apply.
- Bill last digits are not uniform (15.1% end in 0, chi-square p ~ 0). Payments heap much harder (51.2% end in 0).
- NT$ 390 appears on 1,438 statements, 180x its neighbours, and 80% of the following payments are exactly 390. So do 780 (= 2 x 390), 316, 326 and 396.
- These fixed charges look like instalments or fees, not spending. Clients with 2,400 and 1,050 heaps default at 74% and 68%, but few clients hold them.
- Story use: a small "data detective" note: some bills are fixed charges, so utilisation for low-balance clients partly measures a fee, not use.

### 19B.8 Is Jul-Sep a different world from Apr-Jun? Adversarial validation and PSI
**The idea.** In AmEx and Home Credit, teams checked whether the test period looked like the training period by training a
classifier to tell the two apart ("adversarial validation"). An AUC near 0.5 means the periods are interchangeable; near 1
means they differ. The Population Stability Index (PSI) is the credit-risk industry's version of the same check
(PSI < 0.1 stable, 0.1-0.25 some shift, > 0.25 major shift).

**Why it might matter here.** Section 11.5 showed portfolio averages month by month. Averages can stay flat while the shape of
a distribution changes. If the early and late months are distinguishable, a Tableau story that pools all six months mixes two
regimes. The classifier is only a lens (gradient boosting, client-grouped 3-fold CV so the same client never sits in both
training and scoring folds).

**How to read it.** Bars: adversarial AUC with all statement features, without the status code, and status alone. Heatmap:
PSI of each feature in each month against April.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
_feat = {"status": S19B_S, "utilisation": S19B_U, "log bill": np.sign(S19B_B) * np.log1p(np.abs(S19B_B)), "log payment": np.log1p(S19B_P),
         "pay / limit": S19B_PL, "empty": _E.astype(float)}
_long = pd.DataFrame({k: v.ravel() for k, v in _feat.items()})
_long["month_k"] = np.tile(np.arange(6), len(df)); _long["client"] = np.repeat(np.arange(len(df)), 6)
_samp = RNG.choice(len(df), 5000, replace=False)
_sub = _long[_long["client"].isin(_samp)].reset_index(drop=True)
_sub["late_period"] = (_sub["month_k"] >= 3).astype(int)
try:
    from threadpoolctl import threadpool_limits      # one thread per fit: far faster on a busy or small machine
except Exception:
    threadpool_limits = None
def s19b_adv(cols, data=_sub, label="late_period"):
    clf = HistGradientBoostingClassifier(max_iter=60, learning_rate=0.15, max_leaf_nodes=15, random_state=0)
    def _run():
        return cross_val_predict(clf, data[cols], data[label], groups=data["client"], cv=GroupKFold(3), method="predict_proba")[:, 1]
    if threadpool_limits is not None:
        with threadpool_limits(limits=1):
            p = _run()
    else:
        p = _run()
    return roc_auc_score(data[label], p)
_fc = list(_feat)
S19B_ADV = pd.DataFrame([{"feature set": "all six statement features", "AUC": s19b_adv(_fc)},
                         {"feature set": "without status", "AUC": s19b_adv([c for c in _fc if c != "status"])},
                         {"feature set": "status only", "AUC": s19b_adv(["status"])}]
                        + [{"feature set": f"{c} only (rank AUC)", "AUC": s19b_auc(_sub[c].to_numpy(), _sub["late_period"].to_numpy())[0]} for c in _fc if c != "status"])
_rows = []
for k in range(1, 6):
    d2 = _sub[_sub["month_k"].isin([0, k])].copy(); d2["is_k"] = (d2["month_k"] == k).astype(int)
    _rows.append({"month": MONTH_ORDER[k], "AUC_vs_Apr_all": s19b_adv(_fc, d2, "is_k"), "AUC_vs_Apr_no_status": s19b_adv([c for c in _fc if c != "status"], d2, "is_k")})
S19B_ADV_M = pd.DataFrame(_rows)

def s19b_psi(ref, cur, bins=10):
    ref = np.asarray(ref, float); cur = np.asarray(cur, float)
    if len(np.unique(ref)) <= 15:
        cats = np.union1d(np.unique(ref), np.unique(cur)); e = np.array([(ref == c).mean() for c in cats]); a = np.array([(cur == c).mean() for c in cats])
    else:
        edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1))); edges[0], edges[-1] = -np.inf, np.inf
        e = np.histogram(ref, edges)[0] / len(ref); a = np.histogram(cur, edges)[0] / len(cur)
    e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    return float(((a - e) * np.log(a / e)).sum())
S19B_PSI = pd.DataFrame({MONTH_ORDER[k]: {f: s19b_psi(M[:, 0], M[:, k]) for f, M in _feat.items()} for k in range(1, 6)})

fig, axes = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw={"width_ratios": [1.2, 1, 1]})
axes[0].barh(S19B_ADV["feature set"][::-1], S19B_ADV["AUC"][::-1], color=[C_DEF if a > .6 else C_NEU for a in S19B_ADV["AUC"][::-1]])
axes[0].axvline(.5, color="k", lw=1); axes[0].set_xlim(.45, max(.75, S19B_ADV["AUC"].max() + .03))
axes[0].set_title("Adversarial AUC: Apr-Jun vs Jul-Sep statements"); axes[0].set_xlabel("AUC (0.5 = indistinguishable)")
axes[1].plot(S19B_ADV_M["month"], S19B_ADV_M["AUC_vs_Apr_all"], "o-", color=C_DEF, label="all features")
axes[1].plot(S19B_ADV_M["month"], S19B_ADV_M["AUC_vs_Apr_no_status"], "o-", color=C_OK, label="without status")
axes[1].axhline(.5, color="k", lw=1); axes[1].legend(); axes[1].set_title("Each month vs April: how different?")
sns.heatmap(S19B_PSI, annot=True, fmt=".3f", cmap="YlOrRd", vmin=0, vmax=.25, ax=axes[2], cbar_kws={"label": "PSI vs April"})
axes[2].set_title("Population Stability Index vs April")
plt.tight_layout(); savefig("s19b_drift"); plt.show()
display(S19B_ADV.round(3)); display(S19B_ADV_M.round(3)); display(S19B_PSI.round(3))
export(S19B_ADV, "s19b_adversarial_auc"); export(S19B_PSI.reset_index().rename(columns={"index": "feature"}), "s19b_psi_vs_april")
_adv = S19B_ADV.set_index("feature set")["AUC"]
add_fact("19B.8", "drift", f"A classifier tells Jul-Sep statements from Apr-Jun ones with AUC {_adv['all six statement features']:.2f} "
         f"({_adv['without status']:.2f} without status, {_adv['status only']:.2f} with status alone); Sep vs Apr alone reaches "
         f"{S19B_ADV_M.set_index('month').loc['Sep', 'AUC_vs_Apr_all']:.2f}: the money columns drift steadily towards September",
         float(_adv["all six statement features"]), "s19b_drift", 3)
_mx = S19B_PSI.stack().idxmax()
add_fact("19B.8", "drift", f"The largest PSI against April is {_mx[0]} in {_mx[1]} ({S19B_PSI.loc[_mx]:.3f}); "
         f"{int((S19B_PSI > 0.1).to_numpy().sum())} of {S19B_PSI.size} feature-months exceed 0.1", float(S19B_PSI.loc[_mx]), "s19b_drift", 2)

**Observations:**
- A classifier tells Jul-Sep statements from Apr-Jun ones with AUC about 0.69; without status it still reaches about 0.65 (exact values move by 0.002 between library versions).
- Drift grows with distance from April: May 0.51, Jun 0.54, Jul 0.61, Aug 0.69, Sep 0.77 (all features).
- PSI shows September status as the one major shift (0.926), driven by code 1, which exists only in September.
- Utilisation, payment and pay/limit drift moderately by Aug-Sep (PSI 0.10-0.14). Bills drift little (0.07).
- Story use: the six months are not interchangeable; label pooled charts "Apr-Sep" and treat September status as a different coding.

#### 19B.8.1 Concept drift: does the same utilisation mean the same risk in April and in September?
Data drift asks whether the inputs move; concept drift asks whether their **meaning** moves. For each month: default rate by
utilisation band of that month's statement. If the curves lie on top of each other, the relationship is stable.

In [ ]:
_rows = []
for k, m in enumerate(MONTH_ORDER):
    t = pd.DataFrame({"band": pd.cut(S19B_U[:, k], _bands, labels=_lab), "y": s19b_y}).groupby("band", observed=True)["y"].agg(["mean", "size"])
    for b, r in t.iterrows():
        _rows.append({"month": m, "util_band": b, "rate": r["mean"], "n": int(r["size"])})
S19B_CONCEPT = pd.DataFrame(_rows)
fig, ax = plt.subplots(figsize=(11, 5))
_pal = sns.color_palette("viridis", 6)
for c, m in zip(_pal, MONTH_ORDER):
    t = S19B_CONCEPT[S19B_CONCEPT["month"] == m]; ax.plot(t["util_band"].astype(str), t["rate"], "o-", color=c, label=m)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.axhline(BASE_RATE, ls="--", color="k", lw=1); ax.legend(title="statement month")
ax.set_title("Default rate by utilisation band of each month's statement"); plt.tight_layout(); savefig("s19b_concept_drift"); plt.show()
_cp = S19B_CONCEPT.pivot(index="util_band", columns="month", values="rate")[MONTH_ORDER]
_cp["Sep - Apr"] = _cp["Sep"] - _cp["Apr"]; display(_cp.round(3))
export(S19B_CONCEPT, "s19b_concept_drift")

**Observations:**
- The utilisation-to-risk link holds below 70%: the monthly curves overlap within about 2 points.
- At high utilisation the link weakens over time: 90-100% utilisation means 33.5% default in April but 24.3% in September (-9.2 points).
- Above 100%, April gives 35.6% vs 30.1% for September.
- A full card in April is a stronger warning than a full card in September. Older maxed-out balances persist, while recent ones include clients who pay them down.
- Story use: a full card is riskier the longer it has been full; pair it with the persistence features in Section 16.

### 19B.9 Does the row order (ID) hide a leak?
**The idea.** A standard Kaggle check: if rows were sorted by date, branch or outcome before the IDs were assigned, the ID
itself predicts the target (famous leaks include Santander and Instacart). Home Credit and AmEx kernels routinely checked
`SK_ID_CURR` / `customer_ID` order before modelling.

**Why it might matter here.** If default clusters by ID, part of the file was assembled differently (another branch, another
period), and pooled rates could hide it.

**How to read it.** Default rate by ID decile with 95% CI. A flat line inside the band means the order is clean.

In [ ]:
df["CX_ID_DECILE"] = pd.qcut(df["ID"], 10, labels=[f"D{i}" for i in range(1, 11)])
fig, ax = plt.subplots(figsize=(11, 4.2))
S19B_ID = plot_rate(df, "CX_ID_DECILE", ax=ax, title="Default rate by ID decile (row order)")
plt.tight_layout(); savefig("s19b_id_order"); plt.show()
_chi = stats.chi2_contingency(pd.crosstab(df["CX_ID_DECILE"], df[TARGET]))
_sp = pd.Series({c: stats.spearmanr(df["ID"], df[c])[0] for c in ["LIMIT_BAL", "AGE", "PAY_1", "BILL_AMT1", "PAY_AMT1", "N_LATE", TARGET]})
print(f"chi-square ID decile x default: p = {_chi[1]:.3f} | AUC of ID alone: {s19b_auc(df['ID'])[0]:.3f} | out-of-fold binned AUC: {s19b_oof_auc(df['ID']):.3f}")
print("Spearman correlation of ID with key columns:"); print(_sp.round(4).to_string())
export(S19B_ID, "s19b_id_order")
add_fact("19B.9", "leak check", f"Default by ID decile ranges {S19B_ID['rate'].min():.1%}-{S19B_ID['rate'].max():.1%} (chi-square p = {_chi[1]:.3f}), "
         f"but ID alone ranks clients with AUC {s19b_auc(df['ID'])[0]:.3f} and its largest |Spearman| with any key column is {_sp.abs().max():.3f}: "
         f"a mild ordering effect, not a usable leak", float(_chi[1]), "s19b_id_order", 1)

**Observations:**
- Default by ID decile ranges from 19.1% to 23.6%, and the chi-square test rejects equality (p < 0.001).
- ID alone ranks clients at AUC 0.510 (out-of-fold 0.521). Its largest Spearman correlation with any key column is 0.031.
- The file shows a mild ordering effect, probably from how rows were batched, but nothing a chart or model could exploit.
- Story use: none for the dashboard; one line in the data-quality appendix.

### 19B.10 Does this data have its own "96/98": codes that look like placeholders?
**The idea.** In Give Me Some Credit, the past-due counts contained 96 and 98, impossible values that were really placeholder
codes. Every top notebook found them, flagged them, and treated them apart. The same competition split lateness into three
severity counts (30-59, 60-89, 90+ days), which became the strongest features.

**Why it might matter here.** A status code counts months of delay, so it should climb by one each month a client stays
unpaid. A client who sits at the **same** high code (3+) for several months in a row is behaving like a placeholder: the
counter has stopped. Section 1.6 flagged impossible jumps; this subsection looks at the opposite, the frozen counter.

**How to read it.** Left: default rate by the number of month-pairs where a code of 3 or more stayed frozen. Right: the GMSC-style
severity grid, months at code 1-2 by months at code 3+. We do not translate codes into days: code 1 exists almost only in September, and in Apr-Aug a
first delay is coded 2, so code 3 is roughly 60 days late there, not 90+. (The column names `CX_N_30_60` / `CX_N_90P` are kept for continuity; read them as "code 1-2" / "code 3+".)

In [ ]:
_frozen = ((S19B_S[:, 1:] == S19B_S[:, :-1]) & (S19B_S[:, 1:] >= 3)).sum(1)
df["CX_STUCK_HIGH_CODE"] = _frozen
df["CX_N_30_60"] = ((S19B_S >= 1) & (S19B_S <= 2)).sum(1)
df["CX_N_90P"] = (S19B_S >= 3).sum(1)
df["CX_FROZEN_BAND"] = pd.cut(df["CX_STUCK_HIGH_CODE"], [-1, 0, 1, 2, 6], labels=["0", "1", "2", "3+"])
_g = df.assign(_a=pd.cut(df["CX_N_30_60"], [-1, 0, 1, 2, 3, 6], labels=["0", "1", "2", "3", "4-6"]),
               _b=pd.cut(df["CX_N_90P"], [-1, 0, 1, 2, 6], labels=["0", "1", "2", "3-6"]))
S19B_SEV = rate_table(_g, ["_a", "_b"]).rename(columns={"_a": "months_code_1_2", "_b": "months_code_3plus"})
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={"width_ratios": [1, 1.2]})
S19B_FROZ = plot_rate(df, "CX_FROZEN_BAND", ax=axes[0], title="Default rate by stuck high-code month-pairs (CX_STUCK_HIGH_CODE)")
_pv = S19B_SEV.pivot(index="months_code_3plus", columns="months_code_1_2", values="rate"); _nv = S19B_SEV.pivot(index="months_code_3plus", columns="months_code_1_2", values="n")
sns.heatmap(_pv, annot=True, fmt=".0%", cmap=CMAP_RATE, ax=axes[1], cbar_kws={"label": "default rate"})
axes[1].set_title("GMSC-style severity grid: months at code 1-2 (cols) x months at code 3+ (rows)")
plt.tight_layout(); savefig("s19b_sentinel_severity"); plt.show()
display(S19B_FROZ[["CX_FROZEN_BAND", "n", "rate", "ci_lo", "ci_hi"]].round(3)); print("cell sizes:"); display(_nv)
_fz = df[df["CX_STUCK_HIGH_CODE"] >= 2]
print(f"Frozen >= 2: {len(_fz)} clients | median Sep bill {_fz['BILL_AMT1'].median():,.0f} | share with zero payments all 6 months {(_fz['N_ZERO_PAY'] == 6).mean():.0%} "
      f"| share whose bill changed by < 2% Apr->Sep {(np.abs(_fz['BILL_AMT1'] - _fz['BILL_AMT6']) <= 0.02 * _fz['BILL_AMT6'].abs().clip(lower=1)).mean():.0%}")
print("most common frozen codes:", pd.Series(S19B_S[:, 1:][(S19B_S[:, 1:] == S19B_S[:, :-1]) & (S19B_S[:, 1:] >= 3)]).value_counts().head(5).to_dict())
export(S19B_SEV, "s19b_severity_grid"); export(S19B_FROZ, "s19b_frozen_codes")
_f = S19B_FROZ.set_index("CX_FROZEN_BAND")
add_fact("19B.10", "sentinel codes", f"{int(_f.loc[['1', '2', '3+'], 'n'].sum()):,} clients have a status code of 3+ that stays frozen for at least one month; "
         f"with two or more frozen pairs they default at {(_fz[TARGET].mean()):.0%}", float(_fz[TARGET].mean()), "s19b_sentinel_severity", 2)

**Observations:**
- 249 clients have a status of 3+ that stays frozen for at least one month. With one frozen pair they default at 57.4%; with two or more, at 79-82%.
- The most common frozen codes are 3 (176 pairs), 7 (126) and 4 (106).
- 72% of the 120 heavily frozen clients made no payment in six months, and 72% have a bill that barely moved (median Sep bill NT$ 2,400). These look like written-off accounts, like GMSC's 96/98 placeholders.
- In the severity grid, one month at 3+ with no 1-2 month lates already means 29% default. Four or more months at code 1-2 with no month at code 3+ means 63%.
- Story use: "frozen" accounts are a tiny, near-certain-default group; flag them apart so they do not inflate the 'Late 3m+' story.

### 19B.11 At today's pace, how many months to clear the balance? Home-Credit-style ratios
**The idea.** The single most useful engineered feature in Home Credit was `AMT_CREDIT / AMT_ANNUITY`, the loan term in
months; close behind were `AMT_ANNUITY / AMT_INCOME_TOTAL` (repayment burden). Will Koehrsen's "Start Here" notebook also made
the **KDE by target** the standard way to read such ratios. ([Start Here: A Gentle Introduction](https://www.kaggle.com/code/willkoehrsen/start-here-a-gentle-introduction))

**Why it might matter here.** A card has no fixed term, but we can compute the same idea:
* `CX_PAYOFF_MONTHS` = mean positive after-pay balance / mean monthly payment (May-Sep), capped at 120; "never" when nothing is paid.
* `CX_PAY_BURDEN` = mean monthly payment / credit limit (the annuity-to-income analogue, the line standing in for income).

**How to read it.** Top: KDEs of log ratios for defaulters and non-defaulters (separated curves = signal). Bottom: default rate
by pay-off band.

In [ ]:
_apm = np.clip(S19B_AP, 0, None).mean(1); _pm = S19B_P[:, 1:].mean(1)
df["CX_PAYOFF_MONTHS"] = np.where(_pm > 0, np.minimum(_apm / np.where(_pm > 0, _pm, 1), 120), np.nan)
df.loc[(_pm == 0) & (_apm == 0), "CX_PAYOFF_MONTHS"] = 0.0
df["CX_PAY_BURDEN"] = S19B_P.mean(1) / df["LIMIT_BAL"]
df["CX_PAYOFF_BAND"] = pd.cut(df["CX_PAYOFF_MONTHS"], [-0.001, 0, 1, 3, 6, 12, 24, 60, 121],
                              labels=["0 (no balance)", "<=1", "1-3", "3-6", "6-12", "12-24", "24-60", "60+"]).astype(object)
df.loc[df["CX_PAYOFF_MONTHS"].isna(), "CX_PAYOFF_BAND"] = "never (no payment)"
S19B_PB_ORDER = ["0 (no balance)", "<=1", "1-3", "3-6", "6-12", "12-24", "24-60", "60+", "never (no payment)"]
fig = plt.figure(figsize=(17, 9)); gs = fig.add_gridspec(2, 2)
for j, (c, lab) in enumerate([("CX_PAYOFF_MONTHS", "log10(1 + pay-off months)"), ("CX_PAY_BURDEN", "log10(payment / limit), payers only")]):
    ax = fig.add_subplot(gs[0, j])
    v = df[c]; v = np.log10(1 + v) if c == "CX_PAYOFF_MONTHS" else np.log10(v.where(v > 0))
    for yv, col, nm in [(0, C_OK, "No default"), (1, C_DEF, "Default")]:
        sns.kdeplot(v[df[TARGET] == yv].dropna(), ax=ax, color=col, fill=True, alpha=.3, label=nm, common_norm=False)
    ax.set_xlabel(lab); ax.legend(); ax.set_title(f"KDE by target: {c}")
ax = fig.add_subplot(gs[1, :])
S19B_PAYOFF = plot_rate(df, "CX_PAYOFF_BAND", ax=ax, order=S19B_PB_ORDER, title="Default rate by pay-off horizon at the current pace (months)")
plt.tight_layout(); savefig("s19b_home_credit_ratios"); plt.show()
display(S19B_PAYOFF[["CX_PAYOFF_BAND", "n", "rate", "ci_lo", "ci_hi", "lift"]].round(3))
for c in ["CX_PAYOFF_MONTHS", "CX_PAY_BURDEN"]:
    print(f"{c}: AUC {s19b_auc(df[c].fillna(df[c].max() if c == 'CX_PAYOFF_MONTHS' else 0))[0]:.3f}, IV {woe_iv(df.assign(_x=df[c].fillna(999)), '_x')[1]:.3f}")
export(S19B_PAYOFF, "s19b_payoff_horizon")
_po = S19B_PAYOFF.set_index("CX_PAYOFF_BAND")
add_fact("19B.11", "pay-off horizon", f"At their current pace, clients who would need 24-60 months to clear the balance default at {_po.loc['24-60', 'rate']:.1%}, "
         f"vs {_po.loc['<=1', 'rate']:.1%} for those who clear it within a month; clients who pay nothing default at {_po.loc['never (no payment)', 'rate']:.1%}",
         float(_po.loc["24-60", "rate"]), "s19b_home_credit_ratios", 3)

**Observations:**
- The pay-off horizon is monotone from 3-6 months onward: 16.8% (3-6), 23.9% (6-12), 25.2% (12-24), 30.2% (24-60), 55.6% (60+).
- The 230 clients who paid nothing but carry a balance default at 73.0%.
- Clients with no balance (17.2%) or a balance cleared within 1-3 months (14.5-17.5%) are the safest.
- AUC 0.588 and IV 0.119 for the horizon; the payment burden (pay/limit) is weak (AUC 0.521, IV 0.049).
- Story use: "months to clear the balance at today's pace" is an intuitive, Home-Credit-style KPI for a Tableau tooltip or band filter.

#### 19B.11.1 Does the pay-off horizon add anything once we know the September status?
Home Credit's ratios survived because they carried information beyond `EXT_SOURCE`. The analogue here: default rate by
pay-off band **within** each September status group. If the rows still rise from left to right, the ratio is a new lens.

In [ ]:
_g = rate_table(df, ["STATUS_GRP_SEP", "CX_PAYOFF_BAND"], min_n=60)
_pv = _g.pivot(index="STATUS_GRP_SEP", columns="CX_PAYOFF_BAND", values="rate").reindex(index=STATUS_GROUP_ORDER, columns=S19B_PB_ORDER)
fig, ax = plt.subplots(figsize=(14, 4.6))
sns.heatmap(_pv, annot=True, fmt=".0%", cmap=CMAP_RATE, ax=ax, cbar_kws={"label": "default rate"})
ax.set_title("Default rate by Sep status group (rows) x pay-off horizon (cols); cells with n >= 60"); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); savefig("s19b_payoff_within_status"); plt.show()
export(_g, "s19b_payoff_within_status")
_rv = _pv.loc["Revolving"].dropna()
print(f"Within 'Revolving' (status 0): default runs from {_rv.min():.1%} to {_rv.max():.1%} across pay-off bands")
add_fact("19B.11", "pay-off horizon", f"Inside the September 'Revolving' group alone, default ranges from {_rv.min():.1%} to {_rv.max():.1%} across pay-off bands, "
         f"so the horizon adds information that the status code does not carry", float(_rv.max() - _rv.min()), "s19b_payoff_within_status", 2)

**Observations:**
- Within 'Revolving' (status 0) the horizon still moves default from 9.4% to 14.8%.
- Within 'Late 2m', paying nothing pushes default to 85%.
- Within 'Late 1m' it spans 22.7% to 48.8%, and within 'Paid in full' 14.8% to 29.8%. Within 'Inactive' it spans only 10.7% to 13.9%.
- The horizon adds most where the client carries a balance, which is where it should.
- Story use: a secondary split under the status chart for revolvers: "same status, different pace".

### 19B.12 Section 19B wrap-up: which borrowed ideas earned a place in the story?
One row per borrowed idea, its headline number, and whether it should appear in Tableau. Then the facts table, the new
columns and the client-level export.

In [ ]:
_top_agg = S19B_AGG_RANK.head(5)["feature"].tolist()
for f in _top_agg:
    df["CX_AGG_" + f.upper().replace(" | ", "_").replace(" ", "_").replace("/", "PER").replace("-", "_")] = S19B_AGG[f].to_numpy()
S19B_CX = [c for c in df.columns if c.startswith("CX_")]
export(df[["ID", TARGET, "DEFAULT_LABEL", "STATUS_GRP_SEP", "LIMIT_BAND", "AGE_BAND"] + S19B_CX], "s19b_client_competition_features")
print(f"Section 19B runtime: {time.time() - S19B_T0:.1f}s | new columns: {len(S19B_CX)}")
S19B_FACTS = pd.DataFrame([f for f in STORY_FACTS if f["section"].startswith("19B")])
with pd.option_context("display.max_colwidth", 400):
    display(S19B_FACTS)
export(S19B_FACTS, "s19b_facts")

**Observations:**
- 18 facts registered, 25 new `CX_` columns, 19 Tableau tables exported.
- The ideas that change the story are after-pay (19B.4), never-active cards (19B.6), the pay-off horizon (19B.11), frozen codes (19B.10) and cluster-honest error bars (19B.5).
- The AmEx aggregates confirm the notebook's spine (status first) and add one twist: `min` payment and bill volatility beat the latest money values.
- Drift (19B.8) is a caveat for every pooled chart. The ID check (19B.9) is a clean bill of health.
- Story use: these five ideas can each fill one Tableau story point; the rest belong in the appendix.

**New `df` columns (prefix `CX_`)**

| Column | Definition |
|---|---|
| `CX_AP_MAY` .. `CX_AP_SEP` | after-pay balance: bill of the previous month minus the payment made in this month (NT$) |
| `CX_AP_UTIL_LAST`, `CX_AP_UTIL_MEAN` | after-pay balance / limit, Sep value and May-Sep mean |
| `CX_AP_UTIL_BAND` | band of `CX_AP_UTIL_LAST` (credit, zero, 0-30% ... >110%) |
| `CX_N_ACTIVE_STMT` | months (of 6) that are not empty (empty = bill 0 and payment 0) |
| `CX_EMPTY_PATTERN` | 6-character pattern Apr -> Sep, `E` = empty month, `.` = active |
| `CX_EMPTY_TYPE` | All active / Empty early, active late / Gaps / Active only mid-period / Active early, empty late / Never active |
| `CX_FIXED_390` | 1 if any bill equals NT$ 390 or 780 (the heaped fixed charge) |
| `CX_ID_DECILE` | decile of `ID` (row-order leak check) |
| `CX_STUCK_HIGH_CODE`, `CX_FROZEN_BAND` | month-pairs where a status of 3+ stayed the same (stuck counter; renamed from `CX_FROZEN_HIGH`, not the s16 "frozen" balance), and its band |
| `CX_N_30_60`, `CX_N_90P` | months at status code 1-2 and at code 3+ (Give-Me-Some-Credit-style severity counts; labels are codes, not day counts) |
| `CX_PAYOFF_MONTHS`, `CX_PAYOFF_BAND` | mean positive after-pay balance / mean payment (May-Sep), capped at 120; NaN = never pays |
| `CX_PAY_BURDEN` | mean monthly payment / limit |
| `CX_AGG_*` | the five strongest AmEx-style aggregates from 19B.1 (series_summary) |

The full 64-column aggregate bank stays in `S19B_AGG` (not in `df`).

## 20. Story synthesis: narrative arc, hero charts and storyboard

**Why this section matters for the story.** Sections 1-19 produce hundreds of charts; a dashboard that shows all of them tells
nothing. This section turns the analysis into a **narrative**: one spine (3 acts, 7 chapters), one *hero chart* per chapter,
a ranked ledger of every headline fact registered by the other sections (`STORY_FACTS`), and a storyboard that maps directly
onto **Tableau Story Points**. It runs after every other section, but it also computes its own core facts from `df`, so it is
never empty when run alone.

**Design references used here (briefly).**
* Segel & Heer, *Narrative Visualization: Telling Stories with Data* (IEEE TVCG 2010): seven genres (annotated chart, slide show,
  partitioned poster ...) and the **author-driven vs reader-driven** spectrum. Their **martini-glass** pattern (a tight
  author-driven stem that opens into free exploration) and the **interactive slideshow** are exactly what Tableau Story Points
  plus a final explorer dashboard give us.
* Cole Nussbaumer Knaflic, *Storytelling with Data*: understand the context, choose the right display, remove clutter, focus
  attention (pre-attentive colour: grey everything, one accent), think like a designer, tell a story (action titles).
* Tableau help, *Best Practices for Telling Great Stories*: decide between "build up to the conclusion" and "conclusion first"
  (better for a busy audience), keep each story point to one idea, do not cram views, delete every legend / grid line that
  does not serve the point.
* Backdrop: the data (Apr-Sep 2005, default flag for Oct 2005) sits at the **peak of Taiwan's credit-card and cash-card debt
  crisis**. Card debt peaked in the second half of 2005 (about NT$ 800 bn of card + cash-card loans; ~133 cards per 100 adults);
  by early 2006 regulators counted roughly half a million delinquent "card slaves" and launched a debt-negotiation platform
  (sources: BIS Quarterly Review June 2007, Kang & Ma, *Recent episodes of credit card distress in Asia*; National Policy
  Foundation (Taiwan) 2006 review of the 2005-06 card-debt storm; Electronic Payments International, Feb 2007).
  A ~22% default rate is therefore a crisis-period number for one bank, not a normal-times benchmark.

Globals in this section are prefixed `S20_` / `s20_`. **No new `df` columns are created.**

**Section guide**

* **What this section does.** It turns the whole notebook into one story for the presentation and the Tableau dashboard. It does not find new facts. It picks, checks and orders the facts from Sections 1-19.
* **The question it answers.** "If we have five minutes, what do we say, in what order, with which chart?"
* **Inputs.** The raw columns and Section 0 features (so it also runs alone), plus `STORY_FACTS`, the list of facts every section saves with `add_fact`. New columns: none.
* **How to read it.**
  * 20.0 is the story in one page. Read it first.
  * 20.1 is the plan: 3 acts, 7 chapters.
  * 20.4 has the nine main ("hero") charts. Each title states the finding. Grey = context, red = the point, dashed line = the 22.1% average.
  * 20.7 tests 23 common beliefs: Confirmed, Reversed or Weak.
  * 20.8 is the Tableau storyboard.
  * 20.9 lists what the story cannot claim.
* **Key takeaways.**
  * 22 in 100 clients defaulted. Demographics move risk by 7 points at most. Behaviour moves it by 59.
  * Two months late in September gives 69% default. One month late gives 34%.
  * 26% of defaulters gave no warning sign.
  * 17% of clients hold 36% of defaulters and 43% of the money at risk.
* **Where it goes.** Every Tableau dashboard and story point comes from here. The CSV files are listed in 20.11.

### 20.0 The story in one page (written from the run)

> **Headline: watch how they pay, not who they are. But one in three defaulters gives no warning.**

**Act I. The set-up**

1. **The scale.** In one Taiwanese bank in 2005, 6,636 of 30,000 cardholders (22%) defaulted the next month. They owed NT$ 322 M,
   21% of all balances. This was the peak of Taiwan's card-debt bubble, so the rate is a crisis rate.
2. **The usual suspects fail.** Sex, age, education and marriage move the default rate by 7 points at most.
   Men default at 24% and women at 21%. Risk is U-shaped in age, lowest at about 31 (20%) and highest at 22 (29%).
   Most of these gaps are behaviour in disguise: once behaviour is held equal, the age spread falls from 9 to 3 points
   and the education gap is fully explained. Only about half of the sex gap is explained.

**Act II. The confrontation**

3. **Behaviour decides.** September status alone moves the rate by 59 points. Revolvers default at 13%.
   One month late gives 34%; two months late gives 69%. The step from one to two months late is the biggest cliff in the data (+35 points).
   (Footnote: September's code 1 mixes two kinds of client: half were paid-up or inactive in August, half were already 2+ months late in August.)
   The 10% of clients who are 2+ months late in September produce 33% of all defaulters.
   The deepest trouble is a code that does not move: 120 clients stuck at 3+ months late for two months or more default at about 80% (Section 19B).
4. **The clock.** The warning was there early. In April, 24% of future defaulters were already 2+ months late, against 6% of the others.
   Careful with the month-to-month comparison: code 1 ("one month late") exists only in September, so "2+ months late" is a
   stricter test in September (69%) than in April (52%). Compared fairly (any delay), late in September gives 50% and late in April 52%.
   What really shows recency is the **scar**: last late in September 50%, one clean month since 38%, five clean months 19%,
   never late 12%. Old trouble fades, but never fully. September alone carries 94% of the signal of all six months.
5. **The wallet.** Paying nothing against a bill is the loudest money signal (53% default). Usage matters much less:
   heavy users who repay are safe. Quarterly paydown (three months of payments over the three statements they settle)
   tells it simply: paying back 0-5% of the debt gives 26% default, paying it all off gives 14%. For a client who is already
   two months late, paying the quarter off cuts the risk from 69% to 41%. The scary "rising balance, falling repayment" pattern does not predict default: 21% with falling payment amounts,
   17% with a falling repayment share (Section 6), against the 22% average.
   Big spenders are the safest quarter (14%).
   Size of wallet (Section 6.10b) can only be estimated here (no income or bureau data). Wallet size barely moves risk (21-27%),
   but clients whose estimated wallet is at least twice their line ("under-served", mostly lines of NT$ 50k or less) default at 31%:
   the bank's own small line is the warning, not the wallet. The danger is a **quiet card**: a balance with no new spending in all five months gives 37%.
   And the "debt spiral" is not the danger: balances that grew three or more months in a row default at 16%,
   while balances that stay flat ("treading water") default at 28% (Section 16).

**Act III. The resolution**

6. **The twists.**
   - "Paid in full" is not the safest status: clients **coded** paid in full in September default at 17%, revolvers at 13%.
     Even clients who **cleared every bill by amount** for five months still produce 725 defaulters (13.6%).
   - 48% of defaulters were not late in September and 35% were never late in six months. 550 defaulters fired none of 22 warning signs,
     and they look almost the same as good clients.
   - The status labels only partly mean what they say: only 76% of "paid in full" months really pay the full bill,
     and only 61% of "no consumption" months have no bill (section 12).
   - Case studies (section 19): a client clean for five months and late only in September defaults at 36%.
     A "cured" client (late in Apr-Jun, clean Jul-Sep) still defaults at the base rate (23%), not below it.
     A card with zero bills in all six months defaults at 37% (866 clients): silence is not safety.
   - The tiny "Other/Unknown" education group defaults at only 7%. It is a product or channel artefact, not a demographic finding.
   - 5 of 23 common beliefs are reversed by the data, and 5 more are too weak to use.
7. **So what.** Defaulter counts and money tell different stories. The largest pool of money at risk (20%) sits with clean revolvers
   on NT$ 100-300k lines, not with the late payers. Four big, late-in-September segments (the "act" quadrant of Hero 9) are 17% of clients
   and hold 36% of defaulters and 43% of the money at risk. A transparent rule (September status + August status + months late)
   finds 97% of what a machine-learning model finds.
   **Policy what-ifs (section 18, illustrative assumptions):** cutting the credit line by half for clients late in September
   catches 52% of defaulters, touches 15% of good clients, and saves about NT$ 50 M. Freezing every line above 90% utilisation
   loses money in all 11 scenarios tested, because it mostly hits profitable revolvers. Defaulters still had NT$ 548 M of unused
   line (64% of their limits): the bank's open lines, not only its balances, are the risk. Even perfect use of this data puts only 32% of defaulters in the riskiest 10%:
   the rest of default is not visible in six months of card data.

**The one sentence for the last slide:** *Default here is announced by how people pay, mostly in the last month, and mostly loudly. The bank
should act on the loud 17%, and accept that the quiet third of defaults needs data this file does not have (income, other debts).*

_Numbers come from the local run of this notebook on the same 30,000-row file. Sections 18 and 19 add the policy what-ifs and case studies._

### 20.1 What is the story? A three-act spine with seven chapters
Each chapter is **one question → one hero chart → one takeaway**. Takeaways are placeholders here; the numbers are filled by
code in 20.2 and 20.8 so the text never drifts from the data.

| Act | Chapter | Question | Hero chart | Takeaway (to be confirmed after the run) |
|---|---|---|---|---|
| **I. Set-up** | 1. The setting | How big is the problem? | Unit / waffle chart "x in 100" | _Taiwan 2005, 30,000 clients, about 1 in 5 defaults._ |
| | 2. The usual suspects | Do age, sex, education, marriage explain it? | Dumbbell: demographic gaps vs behaviour gaps on one scale | _Demographics barely move the needle._ |
| **II. Confrontation** | 3. Behaviour is destiny | What does the latest repayment status say? | Status-cliff bars with the 50% line | _Two months late in September flips the odds._ |
| | 4. The clock | When does the risk become visible? | Small-multiple timeline, defaulters vs others | _The gap is visible in April and widens every month._ |
| | 5. The wallet | Does how much of the line is used / repaid matter? | Utilisation × repayment-share heatmap | _Usage hurts only when repayment is thin._ |
| **III. Resolution** | 6. The twist(s) | Where does intuition fail? | Slope chart crude vs adjusted + warning funnel | _Clean payers who default; "stress" that is not; safe unknowns._ |
| | 7. So what | Where is the money, and what can be done? | Pareto of NT$ at risk + size-vs-rate quadrant | _A few segments hold most of the exposure._ |

The same arc as a DataFrame (`s20_storyboard`) so it can be exported and extended with computed numbers later.

In [ ]:
S20_CHAPTERS = ["1 Setting", "2 Usual suspects", "3 Behaviour is destiny", "4 The clock", "5 The wallet", "6 The twist", "7 So what"]
s20_storyboard = pd.DataFrame([
    dict(act="I Set-up", chapter=S20_CHAPTERS[0], question="How big is the problem?",
         hero_chart="s20_hero_1_waffle", supporting_charts="KPI tiles; s02 distributions; s01 data quality",
         key_number_field="base_rate", tableau_dashboard="D1 Overview", interaction="none (title slide); hover = counts"),
    dict(act="I Set-up", chapter=S20_CHAPTERS[1], question="Do demographics explain default?",
         hero_chart="s20_hero_2_gap_dumbbell", supporting_charts="s03_rate_by_level; s03_demographic_gaps; s20_hero_6_slope",
         key_number_field="demo_max_gap_pp", tableau_dashboard="D2 Who (demographics)", interaction="parameter: pick demographic; highlight level"),
    dict(act="II Confrontation", chapter=S20_CHAPTERS[2], question="What does the September status say?",
         hero_chart="s20_hero_3_status_cliff", supporting_charts="s03_woe_bins; s03_iv_ranking; s04 status x limit heatmap",
         key_number_field="rate_sep_late2", tableau_dashboard="D3 Drivers (behaviour)", interaction="click a status bar -> filters the page"),
    dict(act="II Confrontation", chapter=S20_CHAPTERS[3], question="When does risk become visible?",
         hero_chart="s20_hero_4_timeline", supporting_charts="s05_roll_rates; s05_sankey_flows; s05_onset_recency_rates",
         key_number_field="late2_apr_defaulters", tableau_dashboard="D4 Time (six months)", interaction="pages / month slider; highlight defaulters"),
    dict(act="II Confrontation", chapter=S20_CHAPTERS[4], question="Does the wallet matter?",
         hero_chart="s20_hero_5_wallet_heatmap", supporting_charts="s06_sow_grid; s06_sow_segments; utilisation by status",
         key_number_field="wallet_hi_rate", tableau_dashboard="D5 Wallet", interaction="filter by Sep status to show the effect inside a status"),
    dict(act="III Resolution", chapter=S20_CHAPTERS[5], question="Where does intuition fail?",
         hero_chart="s20_hero_6_slope + s20_hero_7_warning_funnel", supporting_charts="s20_myths; s04_paradox_segments; s20_explanation_budget",
         key_number_field="share_def_clean_sep", tableau_dashboard="D6 Twists", interaction="myth selector (parameter) swaps the comparison shown"),
    dict(act="III Resolution", chapter=S20_CHAPTERS[6], question="Where is the money and what can be done?",
         hero_chart="s20_hero_8_pareto + s20_hero_9_quadrant", supporting_charts="s08_risk_tiers; s08_client_segments; s20_risk_ladders",
         key_number_field="ead_def_total", tableau_dashboard="D7 Action + free explorer", interaction="all filters unlocked (martini-glass bowl)"),
])
with pd.option_context("display.max_colwidth", 70):
    display(s20_storyboard)

**Observations:**
- The spine has 3 acts and 7 chapters. Each chapter owns one question, one hero chart and one number.
- Act I sets the scale and kills the "who you are" idea. Act II shows that "how you paid" decides. Act III gives the twists and the money.
- Story use: this table is the Tableau Story Points plan. Keep it as the contents slide.

### 20.2 Which numbers carry the story? Core facts computed here
Everything the hero charts and captions need is computed once into the dictionary `S20_K` and registered with `add_fact`,
so this section is self-sufficient. Definitions used throughout Section 20:
* **late 2+** = status code ≥ 2 (comparable across months; code 1 is essentially a September-only code).
* **clean September** = `PAY_1 ≤ 0`. **NT$ at risk** = positive September statement balance (`BILL_AMT1`) of clients who default
  (an exposure proxy: no recovery, no interest).

In [ ]:
def s20_rate(mask):
    # default rate and size of a boolean client mask
    mask = np.asarray(mask, bool)
    n = int(mask.sum())
    return (float(df.loc[mask, TARGET].mean()) if n else np.nan), n

def s20_two_prop(mask_a, mask_b):
    # difference in default rate (a - b) with a two-proportion z test
    ra, na = s20_rate(mask_a); rb, nb = s20_rate(mask_b)
    if min(na, nb) == 0:
        return ra, na, rb, nb, np.nan, np.nan
    pp = (ra * na + rb * nb) / (na + nb)
    se = math.sqrt(max(pp * (1 - pp) * (1 / na + 1 / nb), 1e-18))
    z = (ra - rb) / se
    return ra, na, rb, nb, ra - rb, float(2 * stats.norm.sf(abs(z)))

S20_y = df[TARGET].to_numpy()
S20_isdef = S20_y == 1
S20_EAD = df["BILL_AMT1"].clip(lower=0).astype(float)          # exposure proxy (NT$)
S20_K = {}
K = S20_K                                                       # short alias (not a global named C)

# --- setting
K["n"] = len(df); K["n_def"] = int(S20_isdef.sum()); K["base_rate"] = float(BASE_RATE)
K["ead_total"] = float(S20_EAD.sum()); K["ead_def_total"] = float(S20_EAD[S20_isdef].sum())
K["ead_def_share"] = K["ead_def_total"] / K["ead_total"]
K["limit_def_total"] = float(df.loc[S20_isdef, "LIMIT_BAL"].sum())

# --- behaviour: the September status cliff
K["rate_sep_clean"], K["n_sep_clean"] = s20_rate(df["PAY_1"] <= 0)
K["rate_sep_late1"], K["n_sep_late1"] = s20_rate(df["PAY_1"] == 1)
K["rate_sep_late2"], K["n_sep_late2"] = s20_rate(df["PAY_1"] == 2)
K["rate_sep_late2p"], K["n_sep_late2p"] = s20_rate(df["PAY_1"] >= 2)
K["rate_sep_late3p"], K["n_sep_late3p"] = s20_rate(df["PAY_1"] >= 3)
K["share_clients_late2p"] = K["n_sep_late2p"] / K["n"]
K["share_def_late2p"] = float((df.loc[S20_isdef, "PAY_1"] >= 2).mean())
K["share_def_clean_sep"] = float((df.loc[S20_isdef, "PAY_1"] <= 0).mean())
K["n_def_clean_sep"] = int((S20_isdef & (df["PAY_1"] <= 0)).sum())
K["share_def_never_late"] = float((df.loc[S20_isdef, "N_LATE"] == 0).mean())
K["n_def_never_late"] = int((S20_isdef & (df["N_LATE"] == 0)).sum())
K["rate_never_late"], K["n_never_late"] = s20_rate(df["N_LATE"] == 0)
K["rate_late4p"], K["n_late4p"] = s20_rate(df["N_LATE"] >= 4)
K["rate_allfull"], K["n_allfull"] = s20_rate(df["N_PAIDFULL"] == 6)
K["n_def_allfull"] = int((S20_isdef & (df["N_PAIDFULL"] == 6)).sum())
S20_status_rates = rate_table(df, "STATUS_GRP_SEP").set_index("STATUS_GRP_SEP").reindex(STATUS_GROUP_ORDER)
K["rate_sep_inactive"] = float(S20_status_rates.loc["Inactive", "rate"])
K["rate_sep_paidfull"] = float(S20_status_rates.loc["Paid in full", "rate"])
K["rate_sep_revolving"] = float(S20_status_rates.loc["Revolving", "rate"])

# --- demographics and credit line: largest gap between two levels (levels with n >= 200)
def s20_gap(key, min_n=200, exclude=()):
    t = rate_table(df.assign(_k=key), "_k", min_n=min_n)
    t = t[~t["_k"].astype(str).isin(exclude)]
    lo, hi = t.loc[t["rate"].idxmin()], t.loc[t["rate"].idxmax()]
    return dict(lo_level=str(lo["_k"]), lo_rate=float(lo["rate"]), lo_n=int(lo["n"]),
                hi_level=str(hi["_k"]), hi_rate=float(hi["rate"]), hi_n=int(hi["n"]), gap=float(hi["rate"] - lo["rate"]))
S20_GAP_SPECS = [  # label, key series, kind, excluded levels
    ("Sex", df["SEX_LABEL"], "demographic", ()),
    ("Marital status", df["MAR_LABEL"], "demographic", ()),
    ("Education (documented levels)", df["EDU_LABEL"], "demographic", ("Other/Unknown",)),
    ("Age band", df["AGE_BAND"], "demographic", ()),
    ("Credit limit band", df["LIMIT_BAND"], "account", ()),
    ("Average utilisation band", df["UTIL_BAND"], "behaviour", ()),
    ("Share of bills repaid (5 statements)", df["PAYRATIO_BAND"], "behaviour", ()),
    ("Months late in six", df["N_LATE_BAND"], "behaviour", ()),
    ("September status", df["STATUS_GRP_SEP"], "behaviour", ()),
]
s20_gaps = pd.DataFrame([dict(variable=lab, kind=kind, **s20_gap(key, exclude=exc)) for lab, key, kind, exc in S20_GAP_SPECS])
K["demo_max_gap_pp"] = float(s20_gaps.loc[s20_gaps["kind"] == "demographic", "gap"].max() * 100)
K["beh_max_gap_pp"] = float(s20_gaps.loc[s20_gaps["kind"] == "behaviour", "gap"].max() * 100)
K["rate_female"], _ = s20_rate(df["SEX_LABEL"] == "Female"); K["rate_male"], _ = s20_rate(df["SEX_LABEL"] == "Male")
K["rate_edu_unknown"], K["n_edu_unknown"] = s20_rate(df["EDU_LABEL"] == "Other/Unknown")
K["rate_limit_low"], K["n_limit_low"] = s20_rate(df["LIMIT_BAL"] <= 50_000)
K["rate_limit_high"], K["n_limit_high"] = s20_rate(df["LIMIT_BAL"] > 300_000)

# --- the clock: how informative is a 2+ month delay, month by month, and how early is the gap visible
s20_recency = pd.DataFrame([dict(month=MONTH[i], idx=i,
                                 rate_if_late2=s20_rate(df[f"PAY_{i}"] >= 2)[0], n_late2=s20_rate(df[f"PAY_{i}"] >= 2)[1],
                                 late2_share_defaulters=float((df.loc[S20_isdef, f"PAY_{i}"] >= 2).mean()),
                                 late2_share_others=float((df.loc[~S20_isdef, f"PAY_{i}"] >= 2).mean())) for i in IDX])
s20_recency["month"] = pd.Categorical(s20_recency["month"], MONTH_ORDER, ordered=True)
s20_recency = s20_recency.sort_values("month").reset_index(drop=True)
K["rate_late2_apr"] = float(s20_recency.loc[s20_recency["idx"] == 6, "rate_if_late2"].iloc[0])
K["rate_late2_sep"] = float(s20_recency.loc[s20_recency["idx"] == 1, "rate_if_late2"].iloc[0])
K["late2_apr_defaulters"] = float(s20_recency.loc[s20_recency["idx"] == 6, "late2_share_defaulters"].iloc[0])
K["late2_apr_others"] = float(s20_recency.loc[s20_recency["idx"] == 6, "late2_share_others"].iloc[0])
K["late2_sep_defaulters"] = float(s20_recency.loc[s20_recency["idx"] == 1, "late2_share_defaulters"].iloc[0])
K["late2_sep_others"] = float(s20_recency.loc[s20_recency["idx"] == 1, "late2_share_others"].iloc[0])
# Recency without the code-1 trap: "late" = status >= 1 in every month (code 1 exists only in Sep, so ">= 2" is a
# stricter test in Sep than in Apr-Aug and must not be compared across months).
K["rate_anylate_sep"], K["n_anylate_sep"] = s20_rate(df["PAY_1"] >= 1)
K["rate_anylate_apr"], K["n_anylate_apr"] = s20_rate(df["PAY_6"] >= 1)
# months since the last late month: 0 = late in Sep, 5 = last late in Apr and clean May-Sep, NaN = never late
S20_late_mat = np.column_stack([(df[f"PAY_{i}"] >= 1).to_numpy() for i in IDX])   # col 0 = Sep
S20_msl = pd.Series(np.where(S20_late_mat.any(1), S20_late_mat.argmax(1), np.nan), index=df.index)
for _k in range(6):
    K[f"rate_msl_{_k}"], K[f"n_msl_{_k}"] = s20_rate(S20_msl == _k)
K["rate_never_late"], K["n_never_late"] = s20_rate(S20_msl.isna())
S20_late_only_apr = S20_msl == 5                      # late in Apr, clean May-Sep (same rule as Section 5)
K["rate_late_apr_then_clean"], K["n_late_apr_then_clean"] = s20_rate(S20_late_only_apr)

# --- the wallet
K["rate_util_hi"], K["n_util_hi"] = s20_rate(df["AVG_UTIL"] > 0.9)
K["rate_util_lo"], K["n_util_lo"] = s20_rate((df["AVG_UTIL"] > 0) & (df["AVG_UTIL"] <= 0.1))
S20_rev = df["PAY_1"] == 0
K["rate_rev_util_hi"], K["n_rev_util_hi"] = s20_rate(S20_rev & (df["UTIL_1"] > 0.9))
K["rate_rev_util_lo"], K["n_rev_util_lo"] = s20_rate(S20_rev & (df["UTIL_1"] <= 0.3))
S20_PTB = "PAY_TO_BILL_5M" if "PAY_TO_BILL_5M" in df.columns else "PAY_TO_BILL_6M"   # aligned version: payments settle the previous statement
K["rate_payratio_lo"], K["n_payratio_lo"] = s20_rate(df[S20_PTB] < 0.05)
K["rate_payratio_hi"], K["n_payratio_hi"] = s20_rate(df[S20_PTB] >= 0.5)
K["rate_zero_pay_sep"], K["n_zero_pay_sep"] = s20_rate((df["PAY_AMT1"] == 0) & (df["BILL_AMT2"] > 0))
S20_stress = (df["UTIL_TREND"] > 0) & (df["PAY_TREND"] < 0)   # payment AMOUNT trend, not repayment share (Section 6 SOW_STRESS_FLAG is the share-based version)
K["rate_stress"], K["n_stress"] = s20_rate(S20_stress)

# --- exposure concentration among defaulters
S20_ead_def_sorted = np.sort(S20_EAD[S20_isdef].to_numpy())[::-1]
K["ead_top10_share"] = float(S20_ead_def_sorted[: max(1, int(0.10 * len(S20_ead_def_sorted)))].sum() / max(S20_ead_def_sorted.sum(), 1))
K["ead_def_median"] = float(np.median(S20_ead_def_sorted))
K["ead_def_clean_sep_share"] = float(S20_EAD[S20_isdef & (df["PAY_1"] <= 0).to_numpy()].sum() / max(K["ead_def_total"], 1))

# --- data quirks worth a caveat
K["n_code1_sep"] = int((df["PAY_1"] == 1).sum())
K["n_code1_other"] = int(sum((df[f"PAY_{i}"] == 1).sum() for i in IDX[1:]))
K["n_dup_rows"] = int(df.duplicated(subset=["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"] + PAY_COLS + BILL_COLS + PAMT_COLS, keep=False).sum())

s20_key_numbers = pd.DataFrame({"key": list(K.keys()), "value": list(K.values())})
print(f"{len(K)} key numbers computed")
display(s20_key_numbers)
display(s20_gaps.round(4))
display(s20_recency.round(4))

Register the core facts (all f-strings from the dictionary above). Themes are chosen so that the chapter mapper in 20.3
places them correctly.

In [ ]:
S20_n_before = len(STORY_FACTS)
add_fact("20.2", "setting", f"{K['n_def']:,} of {K['n']:,} clients ({K['base_rate']:.1%}) defaulted in October 2005: about {K['base_rate']*100:.0f} in every 100", K["base_rate"], "s20_hero_1_waffle", 3)
add_fact("20.2", "setting", f"Defaulters owed NT$ {K['ead_def_total']/1e6:,.0f} M on their September statements, {K['ead_def_share']:.1%} of all outstanding balances (NT$ {K['ead_total']/1e6:,.0f} M)", K["ead_def_total"], "s20_hero_8_pareto", 3)
add_fact("20.2", "demographics", f"The widest demographic gap is {K['demo_max_gap_pp']:.1f} pp; the widest behavioural gap is {K['beh_max_gap_pp']:.1f} pp ({K['beh_max_gap_pp']/max(K['demo_max_gap_pp'],1e-9):.1f}x larger)", K["demo_max_gap_pp"], "s20_hero_2_gap_dumbbell", 3)
add_fact("20.2", "demographics", f"Men default at {K['rate_male']:.1%}, women at {K['rate_female']:.1%}: a {abs(K['rate_male']-K['rate_female'])*100:.1f} pp gap", K["rate_male"] - K["rate_female"], "s20_hero_2_gap_dumbbell", 2)
add_fact("20.2", "credit limit", f"Credit line gradient: {K['rate_limit_low']:.1%} default at limits up to NT$ 50k ({K['n_limit_low']:,} clients) vs {K['rate_limit_high']:.1%} above NT$ 300k ({K['n_limit_high']:,})", K["rate_limit_low"] - K["rate_limit_high"], "s20_hero_2_gap_dumbbell", 2)
add_fact("20.2", "status cliff", f"Two months late in September: {K['rate_sep_late2']:.1%} default ({K['n_sep_late2']:,} clients); not late: {K['rate_sep_clean']:.1%} ({K['n_sep_clean']:,})", K["rate_sep_late2"], "s20_hero_3_status_cliff", 3)
add_fact("20.2", "status cliff", f"The {K['share_clients_late2p']:.1%} of clients who were 2+ months late in September produce {K['share_def_late2p']:.1%} of all defaulters (rate {K['rate_sep_late2p']:.1%})", K["share_def_late2p"], "s20_hero_3_status_cliff", 3)
add_fact("20.2", "status cliff", f"One month late in September (code 1, {K['n_sep_late1']:,} clients) defaults at {K['rate_sep_late1']:.1%}: above base but far below the {K['rate_sep_late2']:.1%} of two months late", K["rate_sep_late1"], "s20_hero_3_status_cliff", 2)
add_fact("20.2", "status cliff", f"September status groups: inactive {K['rate_sep_inactive']:.1%}, paid in full {K['rate_sep_paidfull']:.1%}, revolving {K['rate_sep_revolving']:.1%}", K["rate_sep_revolving"], "s20_hero_3_status_cliff", 2)
add_fact("20.2", "delinquency history", f"Never late in six months: {K['rate_never_late']:.1%} default ({K['n_never_late']:,} clients); late in 4+ of 6 months: {K['rate_late4p']:.1%} ({K['n_late4p']:,})", K["rate_late4p"], "s20_hero_2_gap_dumbbell", 2)
add_fact("20.2", "recency", f"Compared like with like (any delay, status >= 1), being late in September gives {K['rate_anylate_sep']:.1%} default and being late in April {K['rate_anylate_apr']:.1%}; the often-quoted {K['rate_late2_sep']:.1%} vs {K['rate_late2_apr']:.1%} for '2+' is a coding artefact (code 1 exists only in September)", K["rate_anylate_sep"], "s20_hero_4_timeline", 3)
add_fact("20.2", "recency", f"The scar fades with clean months: last late in September {K['rate_msl_0']:.1%}, one clean month since {K['rate_msl_1']:.1%}, five clean months {K['rate_msl_5']:.1%}, never late {K['rate_never_late']:.1%}", K["rate_msl_5"], "s20_hero_4_timeline", 3)
add_fact("20.2", "recency", f"Already in April, {K['late2_apr_defaulters']:.1%} of future defaulters were 2+ months late vs {K['late2_apr_others']:.1%} of the others; by September {K['late2_sep_defaulters']:.1%} vs {K['late2_sep_others']:.1%}", K["late2_apr_defaulters"], "s20_hero_4_timeline", 3)
add_fact("20.2", "recency", f"Clients late in April but clean from May to September default at {K['rate_late_apr_then_clean']:.1%} ({K['n_late_apr_then_clean']:,} clients): old trouble fades, but not to the never-late {K['rate_never_late']:.1%}", K["rate_late_apr_then_clean"], "s20_hero_4_timeline", 2)
add_fact("20.2", "wallet utilisation", f"Average utilisation above 90%: {K['rate_util_hi']:.1%} default ({K['n_util_hi']:,}); between 0 and 10%: {K['rate_util_lo']:.1%} ({K['n_util_lo']:,})", K["rate_util_hi"], "s20_hero_5_wallet_heatmap", 2)
add_fact("20.2", "wallet utilisation", f"Inside September revolvers, utilisation above 90% means {K['rate_rev_util_hi']:.1%} default vs {K['rate_rev_util_lo']:.1%} at 30% or less: usage still matters once status is fixed", K["rate_rev_util_hi"] - K["rate_rev_util_lo"], "s20_hero_5_wallet_heatmap", 2)
add_fact("20.2", "wallet repayment share", f"Repaying under 5% of the last five statements: {K['rate_payratio_lo']:.1%} default ({K['n_payratio_lo']:,}); repaying half or more: {K['rate_payratio_hi']:.1%} ({K['n_payratio_hi']:,})", K["rate_payratio_lo"], "s20_hero_5_wallet_heatmap", 3)
add_fact("20.2", "wallet repayment share", f"Paying nothing in September against a positive August bill: {K['rate_zero_pay_sep']:.1%} default ({K['n_zero_pay_sep']:,} clients)", K["rate_zero_pay_sep"], "s20_hero_5_wallet_heatmap", 2)
add_fact("20.2", "paradox", f"{K['share_def_clean_sep']:.1%} of defaulters ({K['n_def_clean_sep']:,}) were NOT late in September; {K['share_def_never_late']:.1%} ({K['n_def_never_late']:,}) were never late in six months", K["share_def_clean_sep"], "s20_hero_7_warning_funnel", 3)
add_fact("20.2", "paradox", f"{K['n_def_allfull']:,} clients coded 'paid in full' in all six months still defaulted (rate {K['rate_allfull']:.1%} among {K['n_allfull']:,})", K["n_def_allfull"], "s20_hero_7_warning_funnel", 2)
add_fact("20.2", "paradox", f"Rising utilisation with falling payment amounts ({K['n_stress']:,} clients) defaults at {K['rate_stress']:.1%} vs base {K['base_rate']:.1%}: the textbook stress signature is {'not ' if K['rate_stress'] <= K['base_rate'] else ''}riskier than average", K["rate_stress"], "s20_myths", 3)
add_fact("20.2", "paradox", f"'Other/Unknown' education ({K['n_edu_unknown']:,} clients) defaults at only {K['rate_edu_unknown']:.1%}, far {'below' if K['rate_edu_unknown'] < K['base_rate'] else 'above'} the base rate", K["rate_edu_unknown"], "s20_myths", 2)
add_fact("20.2", "exposure", f"The top 10% of defaulters by balance hold {K['ead_top10_share']:.1%} of the NT$ at risk; the median defaulter owes NT$ {K['ead_def_median']:,.0f}", K["ead_top10_share"], "s20_hero_8_pareto", 3)
add_fact("20.2", "exposure", f"{K['ead_def_clean_sep_share']:.1%} of the NT$ at risk sits with defaulters who were not late in September", K["ead_def_clean_sep_share"], "s20_hero_8_pareto", 2)
add_fact("20.2", "exposure", f"Credit lines granted to eventual defaulters total NT$ {K['limit_def_total']/1e6:,.0f} M; their September balances use {K['ead_def_total']/max(K['limit_def_total'],1):.1%} of it", K["limit_def_total"], "s20_hero_8_pareto", 1)
add_fact("20.2", "data caveat", f"Status code 1 appears {K['n_code1_sep']:,} times in September but only {K['n_code1_other']:,} times in April-August combined: 'one month late' is a September-only code", K["n_code1_sep"], None, 2)
print(f"registered {len(STORY_FACTS) - S20_n_before} core facts (STORY_FACTS now has {len(STORY_FACTS)})")

**Observations:**
- 6,636 of 30,000 clients defaulted (22.1%). They owed NT$ 322 M on their September statements, 20.9% of all balances.
- The widest demographic gap is 7.4 pp (age band). The widest behaviour gap is 59.1 pp (September status). Behaviour is 8x stronger.
- The same 2+ month delay means 69.6% default in September but 52.3% in April. Recent trouble counts more.
- 48.3% of defaulters were NOT late in September, and 35.2% were never late in six months.
- Story use: these 73 numbers feed every caption. Quote them from here, never retype them.

### 20.3 What did every section find? The facts ledger
`STORY_FACTS` (filled by all sections through `add_fact`) becomes one table: de-duplicated, ranked by strength
(3 = headline), and assigned to a chapter with a keyword dictionary applied first to the theme tag and then to the fact
text (fallback: `unassigned`). The ranking interleaves chapters so the top of the list is not monopolised by one topic.

In [ ]:
S20_CHAPTER_KEYWORDS = [   # order matters: first match wins (specific before generic)
    ("6 The twist",            ["paradox", "twist", "myth", "simpson", "revers", "conflict", "noise", "duplicate", "unknown", "surpris",
                                "counter", "clean payer", "label", "semantic", "forensic", "quirk", "anomal", "caveat", "identity"]),
    ("7 So what",              ["exposure", "nt$", "econom", "policy", "scenario", "what-if", "revenue", "segment", "persona", "tier",
                                "case stud", "sub-pop", "subpop", "lever", "loss", "action", "cluster"]),
    ("4 The clock",            ["recency", "clock", "time", "temporal", "panel", "snapshot", "trajector", "transition", "roll", "early warning",
                                "warning", "threshold", "tipping", "streak", "trend", "dynamics", "shape", "month", "lag", "onset", "persisten"]),
    ("5 The wallet",           ["wallet", "utili", "repay", "payment", "pay ratio", "bill", "spend", "money", "balance", "flow", "amount"]),
    ("3 Behaviour is destiny", ["status", "cliff", "pay_1", "delinquen", "late", "behaviour", "behavior", "driver", "iv", "association",
                                "interaction", "conditional", "atlas", "network"]),
    ("2 Usual suspects",       ["demograph", "sex", "gender", "age", "education", "marriage", "marital", "lifecycle", "credit limit",
                                "credit line", "limit", "gap decomposition"]),
    ("1 Setting",              ["setting", "base", "overview", "quality", "sample", "dataset", "context", "structure", "column"]),
]
def s20_chapter(theme, fact):
    for text in (str(theme).lower(), str(fact).lower()):
        for chap, kws in S20_CHAPTER_KEYWORDS:
            if any(k in text for k in kws):
                return chap
    return "unassigned"

S20_FACT_COLS = ["section", "theme", "fact", "value", "chart", "strength"]
s20_facts_raw = pd.DataFrame(STORY_FACTS, columns=S20_FACT_COLS)
s20_story_facts = s20_facts_raw.copy()
s20_story_facts["fact"] = s20_story_facts["fact"].astype(str).str.strip()
s20_story_facts["_key"] = s20_story_facts["fact"].str.lower().str.replace(r"\s+", " ", regex=True)
s20_story_facts["strength"] = pd.to_numeric(s20_story_facts["strength"], errors="coerce").fillna(1).clip(1, 3).astype(int)
s20_story_facts = (s20_story_facts.sort_values("strength", ascending=False, kind="mergesort")
                   .drop_duplicates("_key").drop(columns="_key").reset_index(drop=True))
s20_story_facts["value"] = pd.to_numeric(s20_story_facts["value"], errors="coerce")
s20_story_facts["section_major"] = s20_story_facts["section"].astype(str).str.extract(r"^(\d+)", expand=False).fillna("?")
s20_story_facts["chapter"] = [s20_chapter(t, f) for t, f in zip(s20_story_facts["theme"], s20_story_facts["fact"])]
s20_story_facts["has_chart"] = s20_story_facts["chart"].notna() & (s20_story_facts["chart"].astype(str) != "None")
# rank: strength first, then round-robin across chapters, then facts that point to a chart
s20_story_facts = s20_story_facts.sort_values(["strength", "has_chart"], ascending=[False, False], kind="mergesort").reset_index(drop=True)
s20_story_facts["rank_in_chapter"] = s20_story_facts.groupby("chapter").cumcount() + 1
s20_story_facts = s20_story_facts.sort_values(["strength", "rank_in_chapter", "chapter"], ascending=[False, True, True], kind="mergesort").reset_index(drop=True)
s20_story_facts.insert(0, "rank", np.arange(1, len(s20_story_facts) + 1))
print(f"facts registered: {len(s20_facts_raw)} | after de-duplication: {len(s20_story_facts)} | "
      f"sections contributing: {s20_story_facts['section_major'].nunique()} | unassigned: {(s20_story_facts['chapter'] == 'unassigned').sum()}")
export(s20_story_facts[["rank", "chapter", "section", "theme", "fact", "value", "chart", "strength", "rank_in_chapter"]], "s20_story_facts")

Who contributes what: facts per section and per chapter (rows = notebook section, columns = story chapter). A section with
zero facts is a section the story currently ignores; a chapter with few facts is thinly supported.

In [ ]:
s20_facts_by_section = (pd.crosstab(s20_story_facts["section_major"], s20_story_facts["chapter"])
                        .reindex(columns=[c for c in S20_CHAPTERS + ["unassigned"] if c in set(s20_story_facts["chapter"])]))
s20_facts_by_section = s20_facts_by_section.loc[sorted(s20_facts_by_section.index, key=lambda v: int(v) if str(v).isdigit() else 999)]
s20_facts_by_section["total"] = s20_facts_by_section.sum(axis=1)
display(s20_facts_by_section)
s20_facts_by_chapter = (s20_story_facts.groupby("chapter").agg(facts=("fact", "size"), headline=("strength", lambda s: int((s == 3).sum())),
                                                               sections=("section_major", "nunique"), themes=("theme", "nunique"))
                        .reindex([c for c in S20_CHAPTERS + ["unassigned"] if c in set(s20_story_facts["chapter"])]))
display(s20_facts_by_chapter)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
ax.bar(["S" + str(s) for s in s20_facts_by_section.index], s20_facts_by_section["total"], color=C_NEU)
for x, v in enumerate(s20_facts_by_section["total"]):
    ax.text(x, v, str(int(v)), ha="center", va="bottom", fontsize=9)
ax.set_title("Facts registered per notebook section"); ax.set_ylabel("facts"); ax.grid(False)
ax = axes[1]
ax.barh(s20_facts_by_chapter.index[::-1], s20_facts_by_chapter["facts"][::-1], color=C_NEU, label="all facts")
ax.barh(s20_facts_by_chapter.index[::-1], s20_facts_by_chapter["headline"][::-1], color=C_DEF, label="headline (strength 3)")
ax.set_title("Facts per story chapter (red = headline)"); ax.grid(False)
plt.tight_layout(); savefig("s20_facts_coverage"); plt.show()

Top facts per chapter (up to six each): this is the raw material for captions, annotations and tooltips.

In [ ]:
s20_top_by_chapter = (s20_story_facts[s20_story_facts["rank_in_chapter"] <= 6]
                      .sort_values(["chapter", "rank_in_chapter"])[["chapter", "rank_in_chapter", "strength", "section", "theme", "fact", "chart"]])
with pd.option_context("display.max_colwidth", None, "display.max_rows", 200):
    for S20_ch in [c for c in S20_CHAPTERS + ["unassigned"] if c in set(s20_top_by_chapter["chapter"])]:
        print(f"\n=== {S20_ch} ===")
        for _, r in s20_top_by_chapter[s20_top_by_chapter["chapter"] == S20_ch].iterrows():
            print(f"  [{r['strength']}] (S{r['section']}, {r['theme']}) {r['fact']}")

**Observations:**
- Every chapter has at least 2 headline facts, so no chapter is thin.
- Chapter 6 (twists) has the most surprising material: clean payers who default, "paid in full" riskier than revolvers, and safe "unknown" education.
- When the full notebook runs, sections 11-19 add many more facts here. Pick the top 2 per chapter for the slides.
- Story use: this is the caption bank for tooltips and annotations.

### 20.4 Hero charts: one presentation-quality chart per chapter
House rules applied to all nine (Knaflic): the **title is the takeaway and is computed from the data**; greys for context and
**one accent colour** (`C_DEF`) for the point; direct labels instead of legends; an annotation on the key mark; a source
footnote. Each chart is also the specification for the corresponding Tableau sheet. Matplotlib only.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

S20_GREY, S20_MID, S20_DARK = "#d0d0d0", "#8c8c8c", "#3d3d3d"
S20_SOURCE = "Source: UCI / Kaggle 'Default of Credit Card Clients' (one Taiwanese bank, Apr-Sep 2005; default flag = Oct 2005). n = 30,000."
S20_CMAP = LinearSegmentedColormap.from_list("s20_rate", ["#f4f4f4", "#fcbba1", C_DEF, "#7f0000"])

def s20_hero(figsize=(11, 6), nrows=1, ncols=1, **kw):
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, **kw)
    for a in np.atleast_1d(axes).ravel():
        a.grid(False)
        for sp in ("top", "right"):
            a.spines[sp].set_visible(False)
        a.tick_params(colors=S20_DARK, labelsize=10)
    return fig, axes

def s20_finish(fig, title, subtitle=None, name=None, top=0.86):
    # action title (left aligned), grey subtitle, source footnote, save
    fig.subplots_adjust(top=top)
    fig.text(0.01, 0.985, title, fontsize=15.5, fontweight="bold", color="#111111", ha="left", va="top")
    if subtitle:
        fig.text(0.01, 0.925, subtitle, fontsize=10.5, color=S20_MID, ha="left", va="top")
    fig.text(0.01, -0.01, S20_SOURCE, fontsize=8, color=S20_MID, ha="left", va="top")
    if name:
        savefig(name, fig)
    plt.show()

S20_HERO_TITLES = {}
print("hero-chart helpers ready")

**Hero 1 (chapter 1, the setting): how big is the problem?** A unit chart makes the base rate concrete: 100 squares, one per
"client in a hundred". Tableau: 10×10 grid from a scaffold table, colour by `is_default`.

In [ ]:
S20_nred = int(round(K["base_rate"] * 100))
fig, ax = s20_hero(figsize=(10.5, 6.2))
S20_xy = [(c, r) for r in range(10) for c in range(10)]
for j, (cx, cy) in enumerate(S20_xy):
    ax.add_patch(Rectangle((cx + 0.08, cy + 0.08), 0.84, 0.84, color=C_DEF if j < S20_nred else S20_GREY, lw=0))
ax.set_xlim(0, 17); ax.set_ylim(-0.2, 10.2); ax.set_aspect("equal"); ax.axis("off")
ax.text(10.7, 8.6, f"{K['base_rate']:.1%}", fontsize=44, fontweight="bold", color=C_DEF, va="center")
ax.text(10.7, 7.0, f"{K['n_def']:,} of {K['n']:,} clients\ndefaulted in October 2005", fontsize=12.5, color=S20_DARK, va="center")
ax.text(10.7, 4.6, f"NT$ {K['ead_def_total']/1e6:,.0f} M", fontsize=26, fontweight="bold", color=S20_DARK, va="center")
ax.text(10.7, 3.4, f"owed by them in September\n({K['ead_def_share']:.0%} of all card balances)", fontsize=12.5, color=S20_DARK, va="center")
ax.annotate("each square = 1 client in 100", xy=(S20_nred % 10 - 0.5 if S20_nred % 10 else 9.5, S20_nred // 10 + (0.5 if S20_nred % 10 else -0.5)),
            xytext=(10.7, 1.2), fontsize=10, color=S20_MID, arrowprops=dict(arrowstyle="->", color=S20_MID))
S20_HERO_TITLES[1] = f"{S20_nred} in every 100 cardholders defaulted the following month"
s20_finish(fig, S20_HERO_TITLES[1], "Taiwan, one bank, at the peak of the 2005 card-debt boom: a crisis-level base rate", "s20_hero_1_waffle", top=0.88)

**Observations:**
- 22 of 100 squares are red. One in five cardholders defaulted the next month.
- This is a crisis-level rate. Taiwan was at the peak of its 2005 card-debt bubble.
- Story use: opening slide. Say the number, then ask "who are these 22?".

**Hero 2 (chapter 2, the usual suspects): do demographics explain default?** One dumbbell per variable, lowest-rate level to
highest-rate level (levels with n ≥ 200), **all on the same axis**. If demographics mattered, their dumbbells would be as
long as the behavioural ones. Tableau: dual-axis Gantt/circle dumbbell from `s20_gap_dumbbell.csv`.

In [ ]:
S20_g = s20_gaps.sort_values("gap").reset_index(drop=True)
S20_kind_col = {"demographic": S20_MID, "account": S20_DARK, "behaviour": C_DEF}
fig, ax = s20_hero(figsize=(11.5, 6.4))
for yy, r in S20_g.iterrows():
    col = S20_kind_col[r["kind"]]
    ax.plot([r["lo_rate"], r["hi_rate"]], [yy, yy], color=col, lw=3.2, solid_capstyle="round", zorder=2)
    ax.scatter([r["lo_rate"], r["hi_rate"]], [yy, yy], s=95, color=col, zorder=3, edgecolor="white", lw=1)
    ax.text(r["lo_rate"] - 0.012, yy, f"{r['lo_level']}  {r['lo_rate']:.0%}", ha="right", va="center", fontsize=9, color=S20_DARK)
    ax.text(r["hi_rate"] + 0.012, yy, f"{r['hi_rate']:.0%}  {r['hi_level']}", ha="left", va="center", fontsize=9, color=S20_DARK)
    ax.text(1.0, yy, f"{r['gap']*100:.0f} pp", transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=11, fontweight="bold", color=col)
ax.axvline(K["base_rate"], color=S20_MID, ls=":", lw=1)
ax.text(K["base_rate"], len(S20_g) - 0.35, f"base {K['base_rate']:.0%}", ha="center", fontsize=9, color=S20_MID)
ax.set_yticks(range(len(S20_g))); ax.set_yticklabels(S20_g["variable"], fontsize=10.5)
for lab, kind in zip(ax.get_yticklabels(), S20_g["kind"]):
    lab.set_color(S20_kind_col[kind]); lab.set_fontweight("bold" if kind == "behaviour" else "normal")
ax.set_xlim(-0.16, max(0.9, S20_g["hi_rate"].max() + 0.22)); ax.set_ylim(-0.6, len(S20_g) - 0.1)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_xlabel("default rate: lowest-rate level to highest-rate level", color=S20_DARK)
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
S20_HERO_TITLES[2] = (f"Who you are moves default by at most {K['demo_max_gap_pp']:.0f} pp; "
                      f"how you pay moves it by {K['beh_max_gap_pp']:.0f} pp")
s20_finish(fig, S20_HERO_TITLES[2], "Gap between the safest and the riskiest level of each variable (grey = demographics, dark = credit line, red = behaviour)",
           "s20_hero_2_gap_dumbbell")
export(s20_gaps.assign(gap_pp=s20_gaps["gap"] * 100), "s20_gap_dumbbell")

**Observations:**
- Demographic dumbbells are short: sex 3.4 pp, marital status 2.7 pp, education 5.9 pp, age 7.4 pp.
- Behaviour dumbbells are long: share of bills repaid 25.2 pp, months late 52.3 pp, September status 59.1 pp.
- Credit line sits in between (20.6 pp). It is the bank's own judgement of the client, so it carries hidden information.
- Story use: the "who vs how" reveal. Show the grey rows first, then animate in the red rows.

**Hero 3 (chapter 3, behaviour is destiny): what does the September status say?** Default rate by the latest repayment status
code with the coin-flip (50%) line. Bars above the line are accented. Tableau: bar chart on `PAY_1` (codes ≥ 4 grouped),
reference lines at 50% and at the base rate.

In [ ]:
S20_code = df["PAY_1"].clip(upper=4)
S20_code_lab = {-2: "No\nconsumption", -1: "Paid\nin full", 0: "Revolving\n(min paid)", 1: "1 month\nlate", 2: "2 months\nlate", 3: "3 months\nlate", 4: "4+ months\nlate"}
s20_cliff = rate_table(df.assign(_k=S20_code), "_k").rename(columns={"_k": "PAY_1_code"})
s20_cliff["label"] = s20_cliff["PAY_1_code"].map(S20_code_lab)
fig, ax = s20_hero(figsize=(11.5, 6.2))
S20_x = np.arange(len(s20_cliff))
S20_cols = [C_DEF if r >= 0.5 else S20_GREY for r in s20_cliff["rate"]]
ax.bar(S20_x, s20_cliff["rate"], color=S20_cols, width=0.72)
ax.errorbar(S20_x, s20_cliff["rate"], yerr=[s20_cliff["rate"] - s20_cliff["ci_lo"], s20_cliff["ci_hi"] - s20_cliff["rate"]], fmt="none", ecolor=S20_DARK, lw=1, capsize=3)
for xx, r in zip(S20_x, s20_cliff.itertuples()):
    ax.text(xx, r.ci_hi + 0.015, f"{r.rate:.0%}", ha="center", fontsize=13, fontweight="bold", color=C_DEF if r.rate >= 0.5 else S20_DARK)
    ax.text(xx, -0.045, f"n = {r.n:,}\n{r.share_clients:.0%} of clients", ha="center", va="top", fontsize=8.5, color=S20_MID, transform=ax.get_xaxis_transform())
ax.axhline(0.5, color=S20_DARK, lw=1.2, ls="--"); ax.text(-0.45, 0.515, "coin flip (50%)", fontsize=9.5, color=S20_DARK)
ax.axhline(K["base_rate"], color=S20_MID, lw=1, ls=":"); ax.text(len(S20_x) - 0.55, K["base_rate"] + 0.012, f"base {K['base_rate']:.0%}", fontsize=9, color=S20_MID, ha="right")
S20_i1 = int(np.where(s20_cliff["PAY_1_code"] == 1)[0][0]); S20_i2 = int(np.where(s20_cliff["PAY_1_code"] == 2)[0][0])
ax.annotate(f"The cliff: +{(K['rate_sep_late2'] - K['rate_sep_late1'])*100:.0f} pp\nfrom one to two months late",
            xy=(S20_i2 - 0.36, K["rate_sep_late2"] * 0.97), xytext=(S20_i1 - 1.9, 0.72), fontsize=11, color=C_DEF, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color=C_DEF, lw=1.5))
ax.set_xticks(S20_x); ax.set_xticklabels(s20_cliff["label"], fontsize=10); ax.tick_params(axis="x", pad=2, length=0)
ax.set_ylim(0, 1.0); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_ylabel("default rate", color=S20_DARK)
fig.subplots_adjust(bottom=0.2)
S20_HERO_TITLES[3] = f"Two months late in September → {K['rate_sep_late2']:.0%} default; not late → {K['rate_sep_clean']:.0%}"
s20_finish(fig, S20_HERO_TITLES[3], f"Default rate by September repayment status (95% Wilson CI). {K['share_clients_late2p']:.0%} of clients sit right of the cliff "
           f"and supply {K['share_def_late2p']:.0%} of defaulters", "s20_hero_3_status_cliff")
export(s20_cliff.assign(label=s20_cliff["label"].str.replace("\n", " ")), "s20_status_cliff")

**Observations:**
- Revolving (minimum paid) clients default at 12.8%. Paid-in-full clients default at 16.8%. Inactive clients at 13.2%.
- One month late (code 1) gives 33.9%. Two months late gives 69.1%. The step 1 → 2 is the largest cliff in the data (+35 pp).
- The 10.4% of clients who are 2+ months late in September produce 32.8% of all defaulters.
- Story use: the cliff. Annotate the 1 → 2 step with "+35 pp".

**Hero 4 (chapter 4, the clock): when does the risk become visible?** Three small multiples over the six months, future
defaulters (accent) vs everyone else (grey): share 2+ months late, median utilisation, share paying nothing. Direct labels at the
line ends. Tableau: three line sheets on the client-month panel, colour = `DEFAULT_LABEL`.

In [ ]:
S20_tl_rows = []
for i in IDX:
    for g, glab in ((1, "Defaulters"), (0, "Others")):
        m = S20_y == g
        S20_tl_rows.append(dict(month=MONTH[i], group=glab,
                                late2_share=float((df.loc[m, f"PAY_{i}"] >= 2).mean()),
                                median_util=float(df.loc[m, f"UTIL_{i}"].median()),
                                zero_pay_share=float((df.loc[m, f"PAY_AMT{i}"] == 0).mean())))
s20_timeline = pd.DataFrame(S20_tl_rows)
s20_timeline["month"] = pd.Categorical(s20_timeline["month"], MONTH_ORDER, ordered=True)
s20_timeline = s20_timeline.sort_values(["group", "month"]).reset_index(drop=True)
S20_panels = [("late2_share", "Share 2+ months late"), ("median_util", "Median utilisation of the credit line"), ("zero_pay_share", "Share paying nothing that month")]
fig, axes = s20_hero(figsize=(13.5, 5.6), ncols=3)
for ax, (colname, ttl) in zip(axes, S20_panels):
    for glab, col, lw in (("Others", S20_MID, 2.2), ("Defaulters", C_DEF, 3.2)):
        t = s20_timeline[s20_timeline["group"] == glab]
        ax.plot(range(6), t[colname], color=col, lw=lw, marker="o", ms=5)
        ax.text(5.12, t[colname].iloc[-1], f"{glab}\n{t[colname].iloc[-1]:.0%}", color=col, fontsize=10, fontweight="bold", va="center")
        ax.text(-0.12, t[colname].iloc[0], f"{t[colname].iloc[0]:.0%}", color=col, fontsize=9.5, va="center", ha="right")
    ax.set_xticks(range(6)); ax.set_xticklabels(MONTH_ORDER); ax.set_xlim(-0.7, 6.6)
    ax.set_ylim(0, max(0.05, s20_timeline[colname].max() * 1.25)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(ttl, loc="left", fontsize=11.5, color=S20_DARK)
S20_gap_apr = K["late2_apr_defaulters"] - K["late2_apr_others"]; S20_gap_sep = K["late2_sep_defaulters"] - K["late2_sep_others"]
axes[0].annotate(f"gap {S20_gap_apr*100:.0f} pp in April\n→ {S20_gap_sep*100:.0f} pp in September", xy=(4.6, K["late2_sep_defaulters"] * 0.93),
                 xytext=(0.2, s20_timeline["late2_share"].max() * 1.08), fontsize=10, color=C_DEF, arrowprops=dict(arrowstyle="->", color=C_DEF))
S20_HERO_TITLES[4] = (f"The warning was there in April: {K['late2_apr_defaulters']:.0%} of future defaulters already 2+ months late "
                      f"(others {K['late2_apr_others']:.0%}); {K['late2_sep_defaulters']:.0%} by September")
s20_finish(fig, S20_HERO_TITLES[4], "Six monthly snapshots, future defaulters vs everyone else", "s20_hero_4_timeline", top=0.80)
export(s20_timeline, "s20_timeline")

**Observations:**
- In April, 24.3% of future defaulters were already 2+ months late, against 6.3% of others. The gap was visible six months early.
- The defaulter line rises to 37.2% in August. It dips to 32.8% in September only because the new code 1 catches first delays in September (a coding quirk, not an improvement).
- Others fall from 6.3% to 4.1%. The two groups separate a little more each month.
- Story use: "the warning was there in April". Put a note on the September dip.

**Hero 5 (chapter 5, the wallet): does usage matter, or only repayment?** Default rate on the grid *average utilisation* ×
*share of six months of bills repaid*. Cells with fewer than 100 clients are blanked. Tableau: highlight table from
`s20_wallet_heatmap.csv`, colour = rate, label = rate and n.

In [ ]:
S20_PR_ORDER = ["0", "0-5%", "5-10%", "10-25%", "25-50%", "50-100%", ">100%"]
S20_UT_ORDER = ["0-10%", "10-30%", "30-50%", "50-70%", "70-90%", "90-100%", ">100%"]
s20_wallet = rate_table(df[df["UTIL_BAND"].isin(S20_UT_ORDER) & df["PAYRATIO_BAND"].isin(S20_PR_ORDER)], ["UTIL_BAND", "PAYRATIO_BAND"])
s20_wallet["UTIL_BAND"] = s20_wallet["UTIL_BAND"].astype(str); s20_wallet["PAYRATIO_BAND"] = s20_wallet["PAYRATIO_BAND"].astype(str)
S20_wr = s20_wallet.pivot(index="UTIL_BAND", columns="PAYRATIO_BAND", values="rate").reindex(index=S20_UT_ORDER[::-1], columns=S20_PR_ORDER)
S20_wn = s20_wallet.pivot(index="UTIL_BAND", columns="PAYRATIO_BAND", values="n").reindex(index=S20_UT_ORDER[::-1], columns=S20_PR_ORDER).fillna(0)
S20_wm = S20_wr.where(S20_wn >= 100)
fig, ax = s20_hero(figsize=(11.5, 6.6))
im = ax.imshow(S20_wm.to_numpy(float), cmap=S20_CMAP, vmin=0, vmax=max(0.5, float(np.nanmax(S20_wm.to_numpy(float)))), aspect="auto")
for a in range(S20_wm.shape[0]):
    for b in range(S20_wm.shape[1]):
        v = S20_wm.iat[a, b]
        if pd.notna(v):
            ax.text(b, a - 0.08, f"{v:.0%}", ha="center", va="center", fontsize=11.5, fontweight="bold", color="white" if v > 0.33 else S20_DARK)
            ax.text(b, a + 0.28, f"n={int(S20_wn.iat[a, b]):,}", ha="center", va="center", fontsize=7.5, color="white" if v > 0.33 else S20_MID)
        else:
            ax.text(b, a, "n<100", ha="center", va="center", fontsize=7.5, color=S20_GREY)
S20_wflat = S20_wm.stack()
(S20_hi_u, S20_hi_p), (S20_lo_u, S20_lo_p) = S20_wflat.idxmax(), S20_wflat.idxmin()
K["wallet_hi_rate"], K["wallet_lo_rate"] = float(S20_wflat.max()), float(S20_wflat.min())
for (uu, pp_), colr in (((S20_hi_u, S20_hi_p), "#111111"), ((S20_lo_u, S20_lo_p), C_OK)):
    ax.add_patch(Rectangle((S20_PR_ORDER.index(pp_) - 0.5, S20_UT_ORDER[::-1].index(uu) - 0.5), 1, 1, fill=False, ec=colr, lw=2.5))
ax.set_xticks(range(len(S20_PR_ORDER))); ax.set_xticklabels(S20_PR_ORDER); ax.set_yticks(range(len(S20_UT_ORDER))); ax.set_yticklabels(S20_UT_ORDER[::-1])
ax.set_xlabel("share of six months of bills repaid  →  pays more", color=S20_DARK); ax.set_ylabel("average utilisation of the credit line  →  uses more", color=S20_DARK)
for sp in ax.spines.values():
    sp.set_visible(False)
ax.tick_params(length=0)
# how much does each axis move the rate on its own (range of marginal rates)?
S20_marg_u = rate_table(df[df["UTIL_BAND"].isin(S20_UT_ORDER)], "UTIL_BAND", min_n=200)["rate"]
S20_marg_p = rate_table(df[df["PAYRATIO_BAND"].isin(S20_PR_ORDER)], "PAYRATIO_BAND", min_n=200)["rate"]
K["wallet_util_range_pp"] = float((S20_marg_u.max() - S20_marg_u.min()) * 100); K["wallet_pay_range_pp"] = float((S20_marg_p.max() - S20_marg_p.min()) * 100)
S20_HERO_TITLES[5] = (f"The wallet in one grid: {K['wallet_hi_rate']:.0%} default at {S20_hi_u} utilisation / {S20_hi_p} repaid "
                      f"vs {K['wallet_lo_rate']:.0%} at {S20_lo_u} / {S20_lo_p}")
s20_finish(fig, S20_HERO_TITLES[5], f"Default rate by utilisation and repayment share (black box = riskiest cell, blue box = safest). Marginal range: "
           f"utilisation {K['wallet_util_range_pp']:.0f} pp, repayment share {K['wallet_pay_range_pp']:.0f} pp", "s20_hero_5_wallet_heatmap")
export(s20_wallet, "s20_wallet_heatmap")
add_fact("20.4", "wallet", f"On the utilisation x repayment grid the default rate runs from {K['wallet_lo_rate']:.1%} ({S20_lo_u} used, {S20_lo_p} repaid) to {K['wallet_hi_rate']:.1%} ({S20_hi_u} used, {S20_hi_p} repaid)", K["wallet_hi_rate"], "s20_hero_5_wallet_heatmap", 3)

**Observations:**
- The riskiest cell is low utilisation (0-10%) with zero repayment: 38% default. These are small bills that the client simply stops paying.
- The safest cell is 10-30% utilisation with 50-100% of bills repaid: 9% default.
- Repayment share separates risk more than usage. Heavy use with good repayment is not dangerous.
- Story use: "it is not how much you use, it is whether you pay". Box the two extreme cells.

**Hero 6 (chapter 6, twist one): do demographic gaps survive once behaviour is held equal?** Slope chart: crude default rate →
rate **standardised to the overall September-status mix** (direct standardisation: each level is given the population's
distribution of `STATUS_GRP_SEP`). Lines that converge mean the demographic gap was mostly a behaviour-mix effect.
Tableau: slope chart from `s20_demo_adjusted.csv`.

In [ ]:
def s20_standardised(key, strat="STATUS_GRP_SEP", min_n=200):
    w = df[strat].value_counts(normalize=True)
    rows = []
    tmp = df.assign(_k=key)
    for lvl, g in tmp.groupby("_k", observed=True):
        if len(g) < min_n:
            continue
        r = g.groupby(strat, observed=True)[TARGET].mean()
        ww = w.reindex(r.index)
        rows.append(dict(level=str(lvl), n=len(g), crude=float(g[TARGET].mean()), adjusted=float((r * ww).sum() / ww.sum())))
    return pd.DataFrame(rows)

S20_age3 = pd.cut(df["AGE"], [20, 30, 45, 80], labels=["Age 21-30", "Age 31-45", "Age 46+"])
S20_lim3 = pd.cut(df["LIMIT_BAL"], [0, 100_000, 300_000, 1_000_001], labels=["Limit <=100k", "Limit 100-300k", "Limit >300k"])
S20_SLOPE_SPECS = [("Sex", df["SEX_LABEL"]), ("Education", df["EDU_LABEL"]), ("Marital status", df["MAR_LABEL"]), ("Age", S20_age3), ("Credit limit", S20_lim3)]
s20_demo_adjusted = pd.concat([s20_standardised(key).assign(variable=lab) for lab, key in S20_SLOPE_SPECS], ignore_index=True)
s20_demo_gapshrink = (s20_demo_adjusted.groupby("variable", sort=False)
                      .agg(crude_gap_pp=("crude", lambda s: (s.max() - s.min()) * 100), adjusted_gap_pp=("adjusted", lambda s: (s.max() - s.min()) * 100)))
s20_demo_gapshrink["gap_retained"] = s20_demo_gapshrink["adjusted_gap_pp"] / s20_demo_gapshrink["crude_gap_pp"]
display(s20_demo_adjusted.round(4)); display(s20_demo_gapshrink.round(3))

fig, axes = s20_hero(figsize=(14, 5.8), ncols=len(S20_SLOPE_SPECS), sharey=True)
for ax, (lab, _) in zip(axes, S20_SLOPE_SPECS):
    t = s20_demo_adjusted[s20_demo_adjusted["variable"] == lab]
    hi_lvl = t.loc[t["crude"].idxmax(), "level"]
    for r in t.itertuples():
        col = C_DEF if r.level == hi_lvl else S20_MID
        ax.plot([0, 1], [r.crude, r.adjusted], color=col, lw=2.6 if r.level == hi_lvl else 1.8, marker="o", ms=6)
        ax.text(-0.07, r.crude, f"{r.level.replace('Age ', '').replace('Limit ', '')} {r.crude:.0%}", ha="right", va="center", fontsize=8.5, color=col)
        ax.text(1.07, r.adjusted, f"{r.adjusted:.0%}", ha="left", va="center", fontsize=8.5, color=col)
    g = s20_demo_gapshrink.loc[lab]
    ax.set_title(f"{lab}\ngap {g['crude_gap_pp']:.1f} → {g['adjusted_gap_pp']:.1f} pp", fontsize=10.5, color=S20_DARK)
    ax.axhline(K["base_rate"], color=S20_GREY, lw=1, ls=":")
    ax.set_xlim(-1.25, 1.45); ax.set_xticks([0, 1]); ax.set_xticklabels(["crude", "same\nSep status mix"], fontsize=9)
    ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0, labelleft=False)
S20_big = s20_demo_gapshrink["crude_gap_pp"].idxmax()
S20_gb = s20_demo_gapshrink.loc[S20_big]
K["demo_gap_retained_median"] = float(s20_demo_gapshrink["gap_retained"].median())
S20_gsex = s20_demo_gapshrink.loc["Sex"]
S20_HERO_TITLES[6] = (f"Same September behaviour: the sex gap goes {S20_gsex['crude_gap_pp']:.1f} → {S20_gsex['adjusted_gap_pp']:.1f} pp; "
                      f"a median {K['demo_gap_retained_median']:.0%} of each gap survives, but every gap stays small")
s20_finish(fig, S20_HERO_TITLES[6], f"Crude default rate vs rate standardised to the overall September-status mix (red = riskiest level). "
           f"Median share of a gap that survives: {K['demo_gap_retained_median']:.0%}", "s20_hero_6_slope", top=0.76)
export(s20_demo_adjusted, "s20_demo_adjusted")
add_fact("20.4", "demographics adjusted", f"Standardising to the same September-status mix moves the {S20_big.lower()} gap from {S20_gb['crude_gap_pp']:.1f} pp to {S20_gb['adjusted_gap_pp']:.1f} pp; across five static variables a median {K['demo_gap_retained_median']:.0%} of the crude gap survives", K["demo_gap_retained_median"], "s20_hero_6_slope", 3)

**Observations:**
- With the same September status mix, the sex gap falls from 3.4 to 2.4 pp. Age falls from 4.4 to 3.1 pp. Marital status falls from 2.7 to 1.9 pp.
- About 70% of each demographic gap survives this simple control. But every surviving gap is under 3 pp, so it stays small.
- The education gap stays large (18 → 17 pp) only because of the tiny, very safe "Other/Unknown" group (468 clients, 7%).
- The credit-line gap halves (16.2 → 8.7 pp). Part of the line effect is behaviour, part is the bank's private knowledge.
- Story use: honest twist. Demographics are weak, and part of their weak effect is behaviour in disguise (section 15 shows more).

**Hero 7 (chapter 6, twist two): who are the defaulters, and what warning did each one give?** Every defaulter is placed in
the **first** bucket that applies (ordered, mutually exclusive): 2+ months late in Sep → 1 month late in Sep → 2+ months late
earlier (Apr-Aug) → September utilisation ≥ 90% → paid nothing in Sep against a balance → no visible warning.
The same buckets drive the "explanation budget" in 20.6. Tableau: funnel / ordered bar from `s20_explanation_budget.csv`.

In [ ]:
S20_RULES = [
    ("2+ months late in September",                 df["PAY_1"] >= 2),
    ("1 month late in September (code 1)",          df["PAY_1"] == 1),
    ("2+ months late earlier (Apr-Aug), Sep clean", df[[f"PAY_{i}" for i in IDX[1:]]].max(axis=1) >= 2),
    ("September utilisation 90% or more",           df["UTIL_1"] >= 0.9),
    ("Paid nothing in Sep against a balance",       (df["PAY_AMT1"] == 0) & (df["BILL_AMT2"] > 0)),
]
S20_NONE = "No visible warning"
S20_RULE_ORDER = [r[0] for r in S20_RULES] + [S20_NONE]
S20_bucket = pd.Series(S20_NONE, index=df.index, dtype=object)
S20_taken = np.zeros(len(df), bool)
for lab, m in S20_RULES:
    mm = np.asarray(m, bool) & ~S20_taken
    S20_bucket[mm] = lab
    S20_taken |= mm
s20_budget = rate_table(df.assign(_b=S20_bucket), "_b").rename(columns={"_b": "rule"}).set_index("rule").reindex(S20_RULE_ORDER).reset_index()
s20_budget["step"] = np.arange(1, len(s20_budget) + 1)
s20_budget["non_defaults"] = s20_budget["n"] - s20_budget["defaults"]
s20_budget["share_non_defaults"] = s20_budget["non_defaults"] / s20_budget["non_defaults"].sum()
s20_budget["ead_at_risk"] = [float(S20_EAD[(S20_bucket == r).to_numpy() & S20_isdef].sum()) for r in s20_budget["rule"]]
s20_budget["share_ead_at_risk"] = s20_budget["ead_at_risk"] / s20_budget["ead_at_risk"].sum()
s20_budget["cum_share_defaults"] = s20_budget["share_defaults"].cumsum()          # recall if we flag rules 1..k
s20_budget["cum_share_non_defaults"] = s20_budget["share_non_defaults"].cumsum()   # false-alarm rate
s20_budget["cum_share_clients"] = s20_budget["share_clients"].cumsum()
s20_budget["cum_precision"] = s20_budget["defaults"].cumsum() / s20_budget["n"].cumsum()
display(s20_budget[["step", "rule", "n", "defaults", "rate", "share_clients", "share_defaults", "share_non_defaults", "share_ead_at_risk",
                    "cum_share_defaults", "cum_share_non_defaults", "cum_precision"]].round(4))
K["share_def_no_warning"] = float(s20_budget.loc[s20_budget["rule"] == S20_NONE, "share_defaults"].iloc[0])
K["rate_no_warning"] = float(s20_budget.loc[s20_budget["rule"] == S20_NONE, "rate"].iloc[0])
K["n_def_no_warning"] = int(s20_budget.loc[s20_budget["rule"] == S20_NONE, "defaults"].iloc[0])

fig, ax = s20_hero(figsize=(12, 6))
S20_yp = np.arange(len(s20_budget))[::-1]
S20_bcols = [S20_DARK if r == S20_NONE else (C_DEF if k == 0 else S20_MID) for k, r in enumerate(s20_budget["rule"])]
ax.barh(S20_yp, s20_budget["defaults"], color=S20_bcols, height=0.66)
for yy, r in zip(S20_yp, s20_budget.itertuples()):
    ax.text(r.defaults + K["n_def"] * 0.008, yy, f"{r.defaults:,}  ({r.share_defaults:.0%} of defaulters)", va="center", fontsize=11, fontweight="bold", color=S20_DARK)
    ax.text(r.defaults + K["n_def"] * 0.008, yy - 0.3, f"{r.rate:.0%} of the {r.n:,} clients in this bucket default", va="center", fontsize=8.5, color=S20_MID)
ax.set_yticks(S20_yp); ax.set_yticklabels([f"{s}. {r}" for s, r in zip(s20_budget["step"], s20_budget["rule"])], fontsize=10.5)
ax.set_xlim(0, s20_budget["defaults"].max() * 1.55); ax.set_xlabel("defaulters (each counted once, in the first bucket that applies)", color=S20_DARK)
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
ax.annotate(f"Invisible to every simple rule,\nyet still {K['rate_no_warning']:.0%} of this bucket default", xy=(K["n_def_no_warning"] * 0.5, S20_yp[-1] + 0.36),
            xytext=(s20_budget["defaults"].max() * 0.95, S20_yp[-1] + 1.55), fontsize=10.5, color=S20_DARK, arrowprops=dict(arrowstyle="->", color=S20_DARK))
S20_HERO_TITLES[7] = (f"Of {K['n_def']:,} defaulters, {1 - K['share_def_no_warning']:.0%} showed a warning sign; "
                      f"{K['share_def_no_warning']:.0%} gave none at all")
s20_finish(fig, S20_HERO_TITLES[7], "Defaulters by the first warning they tripped (ordered, mutually exclusive rules)", "s20_hero_7_warning_funnel")
add_fact("20.4", "paradox", f"{K['share_def_no_warning']:.1%} of defaulters ({K['n_def_no_warning']:,}) tripped none of five simple warning rules; that 'quiet' bucket still defaults at {K['rate_no_warning']:.1%}", K["share_def_no_warning"], "s20_hero_7_warning_funnel", 3)

**Observations:**
- Rule 1 (2+ late in September) catches 32.8% of defaulters at 69.6% precision.
- Adding code 1, earlier delays and high utilisation brings the catch to 74%. But by then 39% of good clients are also flagged.
- 1,720 defaulters (25.9%) tripped none of these rules. They hold 14.4% of the NT$ at risk.
- Story use: "one in four defaulters gives no warning". This is the humility slide before the money slide.

**Hero 8 (chapter 7, so what): where is the money?** Pareto of NT$ at risk (September balance of defaulters) across
18 segments = September status × credit-line tier. Accent = the segments needed to reach 80% of the exposure.
Tableau: sorted bar + running-total table calc from `s20_exposure_pareto.csv`.

In [ ]:
S20_limt = pd.cut(df["LIMIT_BAL"], [0, 100_000, 300_000, 1_000_001], labels=["limit <=100k", "limit 100-300k", "limit >300k"]).astype(str)
S20_seg = df["STATUS_GRP_SEP"].astype(str) + " | " + S20_limt
s20_pareto = rate_table(df.assign(_s=S20_seg), "_s").rename(columns={"_s": "segment"})
S20_tmp = pd.DataFrame({"segment": S20_seg, "ead": S20_EAD, "ead_def": S20_EAD.where(S20_isdef, 0.0)}).groupby("segment").sum()
s20_pareto = s20_pareto.merge(S20_tmp, left_on="segment", right_index=True)
s20_pareto[["status_sep", "limit_tier"]] = s20_pareto["segment"].str.split(" | ", regex=False, expand=True)
s20_pareto = s20_pareto.sort_values("ead_def", ascending=False).reset_index(drop=True)
s20_pareto["share_ead_def"] = s20_pareto["ead_def"] / s20_pareto["ead_def"].sum()
s20_pareto["cum_share_ead_def"] = s20_pareto["share_ead_def"].cumsum()
s20_pareto["balance_weighted_rate"] = s20_pareto["ead_def"] / s20_pareto["ead"].where(s20_pareto["ead"] > 0)
s20_pareto["in_top80"] = s20_pareto["cum_share_ead_def"].shift(fill_value=0) < 0.8
K["pareto_k80"] = int(s20_pareto["in_top80"].sum()); K["pareto_n_seg"] = len(s20_pareto)
K["pareto_cum_k80"] = float(s20_pareto.loc[s20_pareto["in_top80"], "share_ead_def"].sum())
K["pareto_top_seg"] = str(s20_pareto.loc[0, "segment"]); K["pareto_top_share"] = float(s20_pareto.loc[0, "share_ead_def"])
display(s20_pareto[["segment", "n", "rate", "share_clients", "share_defaults", "ead_def", "share_ead_def", "cum_share_ead_def", "balance_weighted_rate"]].round(4))

fig, ax = s20_hero(figsize=(12, 7.2))
S20_yp = np.arange(len(s20_pareto))[::-1]
ax.barh(S20_yp, s20_pareto["ead_def"] / 1e6, color=[C_DEF if t else S20_GREY for t in s20_pareto["in_top80"]], height=0.7)
for yy, r in zip(S20_yp, s20_pareto.itertuples()):
    ax.text(r.ead_def / 1e6 + s20_pareto["ead_def"].max() / 1e6 * 0.01, yy,
            f"NT$ {r.ead_def/1e6:,.0f} M  ·  cumulative {r.cum_share_ead_def:.0%}  ·  default rate {r.rate:.0%}", va="center", fontsize=8.8,
            color=S20_DARK if r.in_top80 else S20_MID)
ax.set_yticks(S20_yp); ax.set_yticklabels(s20_pareto["segment"], fontsize=9.5)
ax.set_xlim(0, s20_pareto["ead_def"].max() / 1e6 * 1.75); ax.set_xlabel("NT$ at risk (September balance of clients who default), millions", color=S20_DARK)
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
ax.annotate(f"largest pool: {K['pareto_top_share']:.0%} of all NT$ at risk", xy=(s20_pareto['ead_def'].iloc[0] / 1e6 * 0.6, S20_yp[0] - 0.4),
            xytext=(s20_pareto["ead_def"].max() / 1e6 * 0.9, S20_yp[0] - 3.2), fontsize=10.5, color=C_DEF, fontweight="bold", arrowprops=dict(arrowstyle="->", color=C_DEF))
S20_HERO_TITLES[8] = (f"{K['pareto_k80']} of {K['pareto_n_seg']} segments hold {K['pareto_cum_k80']:.0%} of the NT$ {K['ead_def_total']/1e6:,.0f} M at risk; "
                      f"the largest pool is '{K['pareto_top_seg']}'")
s20_finish(fig, S20_HERO_TITLES[8], "NT$ at risk by September status and credit-line tier (red = segments needed to reach 80%)", "s20_hero_8_pareto", top=0.88)
export(s20_pareto.drop(columns=["ci_lo", "ci_hi"]), "s20_exposure_pareto")
add_fact("20.4", "exposure", f"{K['pareto_k80']} of {K['pareto_n_seg']} status x limit segments hold {K['pareto_cum_k80']:.0%} of NT$ at risk; the largest single pool is '{K['pareto_top_seg']}' with {K['pareto_top_share']:.1%}", K["pareto_cum_k80"], "s20_hero_8_pareto", 3)

**Observations:**
- 7 of 18 segments hold 85% of the NT$ 322 M at risk.
- The largest single pool is clean revolvers with a NT$ 100-300k line: 9.6% default, but 20% of the money at risk. Low rate times big balances equals big money.
- Late-2m clients with NT$ 100-300k lines are second (18% of the money at only 2.8% of clients).
- Story use: "count risk and money risk are different maps". Defaulter counts point to small lines; money points to mid lines.

**Hero 9 (chapter 7, so what): which segments deserve action?** Segment size (share of clients, log scale) vs default rate for
September status × September utilisation tier; bubble area = NT$ at risk. The top-right quadrant (big **and** risky) is where a
policy change moves the portfolio. Tableau: scatter with reference lines; size = exposure; label = segment.

In [ ]:
S20_ut3 = pd.cut(df["UTIL_1"], [-np.inf, 0.3, 0.8, np.inf], labels=["util <30%", "util 30-80%", "util >80%"]).astype(str)
S20_seg2 = df["STATUS_GRP_SEP"].astype(str) + " | " + S20_ut3
s20_quadrant = rate_table(df.assign(_s=S20_seg2), "_s").rename(columns={"_s": "segment"})
s20_quadrant = s20_quadrant.merge(pd.DataFrame({"segment": S20_seg2, "ead_def": S20_EAD.where(S20_isdef, 0.0)}).groupby("segment").sum(), left_on="segment", right_index=True)
s20_quadrant[["status_sep", "util_tier"]] = s20_quadrant["segment"].str.split(" | ", regex=False, expand=True)
S20_xcut = float(s20_quadrant["share_clients"].median())
s20_quadrant["quadrant"] = np.select(
    [(s20_quadrant["rate"] >= BASE_RATE) & (s20_quadrant["share_clients"] >= S20_xcut), (s20_quadrant["rate"] >= BASE_RATE),
     (s20_quadrant["share_clients"] >= S20_xcut)], ["Big and risky: act", "Small and risky: watch", "Big and safe: grow"], default="Small and safe: leave")
s20_quadrant = s20_quadrant.sort_values("ead_def", ascending=False).reset_index(drop=True)
display(s20_quadrant[["segment", "n", "share_clients", "rate", "lift", "share_defaults", "ead_def", "quadrant"]].round(4))
s20_quadrant_summary = s20_quadrant.groupby("quadrant").agg(segments=("segment", "size"), clients=("n", "sum"), defaults=("defaults", "sum"), ead_def=("ead_def", "sum"))
s20_quadrant_summary["share_clients"] = s20_quadrant_summary["clients"] / len(df)
s20_quadrant_summary["share_defaults"] = s20_quadrant_summary["defaults"] / K["n_def"]
s20_quadrant_summary["share_ead_def"] = s20_quadrant_summary["ead_def"] / K["ead_def_total"]
display(s20_quadrant_summary.round(4))

fig, ax = s20_hero(figsize=(12, 7))
S20_act = s20_quadrant["quadrant"] == "Big and risky: act"
ax.scatter(s20_quadrant["share_clients"], s20_quadrant["rate"], s=60 + 2600 * s20_quadrant["ead_def"] / s20_quadrant["ead_def"].max(),
           c=[C_DEF if a else S20_GREY for a in S20_act], alpha=0.75, edgecolor="white", lw=1.2, zorder=3)
for r in s20_quadrant.itertuples():
    if r.n >= 100:
        ax.annotate(r.segment.replace(" | ", "\n"), (r.share_clients, r.rate), fontsize=7.8, ha="center", va="center",
                    color="#111111" if r.quadrant == "Big and risky: act" else S20_DARK, zorder=4)
ax.axhline(BASE_RATE, color=S20_MID, ls=":", lw=1); ax.axvline(S20_xcut, color=S20_MID, ls=":", lw=1)
ax.set_xscale("log"); ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v, p: f"{v:.1%}" if v < 0.01 else f"{v:.0%}"))
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_ylim(0, min(1.0, s20_quadrant["rate"].max() + 0.12))
ax.set_xlabel("segment size: share of all clients (log scale)", color=S20_DARK); ax.set_ylabel("default rate", color=S20_DARK)
for txt, xx, yy, ha, va in (("BIG AND RISKY: ACT", 0.99, 0.98, "right", "top"), ("SMALL AND RISKY: WATCH", 0.01, 0.98, "left", "top"),
                            ("BIG AND SAFE: GROW", 0.99, 0.02, "right", "bottom"), ("SMALL AND SAFE", 0.01, 0.02, "left", "bottom")):
    ax.text(xx, yy, txt, transform=ax.transAxes, ha=ha, va=va, fontsize=10, fontweight="bold", color=C_DEF if "ACT" in txt else S20_MID, alpha=0.9)
S20_qa = s20_quadrant_summary.reindex(["Big and risky: act"]).fillna(0).iloc[0]
K["act_share_clients"], K["act_share_defaults"], K["act_share_ead"] = float(S20_qa["share_clients"]), float(S20_qa["share_defaults"]), float(S20_qa["share_ead_def"])
S20_HERO_TITLES[9] = (f"The action quadrant: {K['act_share_clients']:.0%} of clients, {K['act_share_defaults']:.0%} of defaulters, "
                      f"{K['act_share_ead']:.0%} of the NT$ at risk")
s20_finish(fig, S20_HERO_TITLES[9], "September status x utilisation tier. Bubble area = NT$ at risk; dotted lines = base rate and median segment size",
           "s20_hero_9_quadrant", top=0.88)
export(s20_quadrant.drop(columns=["ci_lo", "ci_hi"]), "s20_segment_quadrant")
add_fact("20.4", "segment action", f"Big-and-risky segments (status x utilisation) are {K['act_share_clients']:.1%} of clients but {K['act_share_defaults']:.1%} of defaulters and {K['act_share_ead']:.1%} of NT$ at risk", K["act_share_defaults"], "s20_hero_9_quadrant", 3)

**Observations:**
- The "act" quadrant (big and risky) has 17% of clients, 36% of defaulters and 43% of the NT$ at risk.
- The "grow" quadrant (big and safe) has 76% of clients and still 44% of the NT$ at risk, because it is so large.
- Late 2m with utilisation 30-80% is the single riskiest big segment (74.8% default).
- Story use: final action slide. The bank should act on 17% of clients, not on everyone.

All nine computed action titles in one place (these become the Tableau sheet titles / story captions).

In [ ]:
s20_hero_titles = pd.DataFrame({"hero": list(S20_HERO_TITLES.keys()), "computed_title": list(S20_HERO_TITLES.values())})
with pd.option_context("display.max_colwidth", None):
    display(s20_hero_titles)
export(s20_hero_titles, "s20_hero_titles")

### 20.5 How does risk build, one condition at a time? Risk ladders
A waterfall-style decomposition: start from all clients and add **one condition per step**; each step shows the default
rate of the clients that remain and the change in percentage points. Four illustrative paths: two climbing (the revolver
squeeze, the delinquency spiral), one descending (the safe core), and one where a "risky" demographic profile is defused by
clean behaviour. Tableau: step/waterfall chart with path as a parameter, from `s20_risk_ladders.csv`.

In [ ]:
S20_prev_late2 = df[[f"PAY_{i}" for i in IDX[1:]]].max(axis=1) >= 2
S20_LADDERS = {
    "A. Revolver squeeze (up)": [("Sep status = revolving", df["PAY_1"] == 0), ("+ Sep utilisation > 90%", df["UTIL_1"] > 0.9),
                                 ("+ 2+ months late at least once Apr-Aug", S20_prev_late2), ("+ repaid < 5% of the Aug bill", df["PAY_RATIO_1"] < 0.05)],
    "B. Delinquency spiral (up)": [("2+ months late in Sep", df["PAY_1"] >= 2), ("+ also 2+ late in Aug", df["PAY_2"] >= 2),
                                   ("+ also 2+ late in Jul", df["PAY_3"] >= 2), ("+ paid nothing in Sep", df["PAY_AMT1"] == 0)],
    "C. Safe core (down)": [("Not late in Sep", df["PAY_1"] <= 0), ("+ never late in six months", df["N_LATE"] == 0),
                            ("+ credit limit >= NT$ 200k", df["LIMIT_BAL"] >= 200_000), ("+ repaid >= 50% of the last five statements", df[S20_PTB] >= 0.5)],
    "D. Risky profile, clean behaviour": [("Age 25 or younger", df["AGE"] <= 25), ("+ credit limit <= NT$ 50k", df["LIMIT_BAL"] <= 50_000),
                                          ("+ not late in Sep", df["PAY_1"] <= 0), ("+ never late in six months", df["N_LATE"] == 0)],
}
S20_lad_rows = []
for path, steps in S20_LADDERS.items():
    m = np.ones(len(df), bool); prev = BASE_RATE
    S20_lad_rows.append(dict(path=path, step=0, condition="All clients", n=len(df), defaults=int(S20_isdef.sum()), rate=float(BASE_RATE), delta_pp=0.0))
    for s, (lab, cond) in enumerate(steps, 1):
        m = m & np.asarray(cond.fillna(False) if hasattr(cond, "fillna") else cond, bool)
        r, n = s20_rate(m)
        S20_lad_rows.append(dict(path=path, step=s, condition=lab, n=n, defaults=int(S20_isdef[m].sum()), rate=r, delta_pp=(r - prev) * 100 if n else np.nan))
        prev = r if n else prev
s20_risk_ladders = pd.DataFrame(S20_lad_rows)
s20_risk_ladders["ci_lo"], s20_risk_ladders["ci_hi"] = wilson_ci(s20_risk_ladders["defaults"], s20_risk_ladders["n"])
s20_risk_ladders["share_clients"] = s20_risk_ladders["n"] / len(df)
s20_risk_ladders["lift"] = s20_risk_ladders["rate"] / BASE_RATE
display(s20_risk_ladders.round(4))
export(s20_risk_ladders, "s20_risk_ladders")

fig, axes = s20_hero(figsize=(14, 9), nrows=2, ncols=2)
for ax, (path, t) in zip(axes.ravel(), s20_risk_ladders.groupby("path", sort=False)):
    t = t.reset_index(drop=True)
    cols = [S20_GREY] * (len(t) - 1) + [C_DEF if t["rate"].iloc[-1] >= BASE_RATE else C_OK]
    ax.bar(range(len(t)), t["rate"], color=cols, width=0.62)
    for k, r in t.iterrows():
        ax.text(k, r["rate"] + 0.02, f"{r['rate']:.0%}", ha="center", fontsize=11.5, fontweight="bold", color=S20_DARK)
        ax.text(k, r["rate"] + 0.085, f"n={r['n']:,}" + (f"  ({r['delta_pp']:+.0f} pp)" if k else ""), ha="center", fontsize=8, color=S20_MID)
    ax.axhline(BASE_RATE, color=S20_MID, ls=":", lw=1)
    ax.set_xticks(range(len(t))); ax.set_xticklabels([c.replace("+ ", "+\n").replace(" at least", "\nat least").replace(" of six", "\nof six") for c in t["condition"]], fontsize=8)
    ax.set_ylim(0, 1.08); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.set_title(path, loc="left", fontsize=11.5, color=S20_DARK)
S20_la = s20_risk_ladders[s20_risk_ladders["path"].str.startswith("B")].iloc[-1]; S20_lc = s20_risk_ladders[s20_risk_ladders["path"].str.startswith("C")].iloc[-1]
plt.tight_layout(rect=(0, 0, 1, 0.9))
s20_finish(fig, f"Four conditions take a client from {BASE_RATE:.0%} to {S20_la['rate']:.0%}, or down to {S20_lc['rate']:.0%}",
           "Default rate after adding one condition at a time (last bar red = above base, blue = below base)", "s20_risk_ladders", top=0.87)
S20_ld = s20_risk_ladders[s20_risk_ladders["path"].str.startswith("D")]
add_fact("20.5", "risk ladder segments", f"Delinquency spiral: 2+ months late in Sep, Aug and Jul with no September payment leaves {int(S20_la['n']):,} clients at {S20_la['rate']:.1%} default; the safe core (never late, limit >= 200k, repays >= 50%) is {int(S20_lc['n']):,} clients at {S20_lc['rate']:.1%}", S20_la["rate"], "s20_risk_ladders", 3)
add_fact("20.5", "demographics defused", f"Young (<= 25) low-limit (<= 50k) clients default at {S20_ld['rate'].iloc[2]:.1%}; the never-late ones among them only {S20_ld['rate'].iloc[4]:.1%} (base {BASE_RATE:.1%})", S20_ld["rate"].iloc[4], "s20_risk_ladders", 2)

**Observations:**
- Delinquency spiral: 2+ late in September gives 69.6%. Adding August and July delays adds only +3 pp more (72.9%). The first condition does almost all the work.
- Revolver squeeze: revolving + utilisation >90% + an earlier delay + tiny repayment climbs from 12.8% only to 28.4%. Money stress alone stays far below the delinquency path.
- Safe core: not late, never late, line ≥ NT$ 200k gives 9.6%. Adding high repayment does not lower it further.
- Risky profile defused: young (≤25) with a small line starts at 31.3%. Add "not late in September" and it falls to 19.8%; "never late" gives 16.4%. Behaviour beats profile.
- Story use: ladder D is the best single slide for "how you pay beats who you are".

### 20.6 How much of default can simple rules explain? The explanation budget
Reading the 20.4 buckets as a budget: each successive rule "accounts for" a share of the defaulters; what remains after the
last rule is the **unexplained residual**. The mirror bar shows the price: the share of **non-defaulters** caught by the same
rules (false alarms). A third bar shows where the NT$ at risk sits. Tableau: 100% stacked bars, colour = rule.

In [ ]:
S20_bud_cols = [C_DEF, "#f08a6c", "#f6b8a2", "#9ecae1", "#c6dbef", S20_DARK]
S20_bars = [("Defaulters", "share_defaults"), ("NT$ at risk", "share_ead_at_risk"), ("Non-defaulters\n(false alarms)", "share_non_defaults"), ("All clients", "share_clients")]
fig, ax = s20_hero(figsize=(13, 5.4))
for yy, (lab, colname) in enumerate(S20_bars[::-1]):
    left = 0.0
    for k, r in s20_budget.iterrows():
        v = float(r[colname])
        ax.barh(yy, v, left=left, color=S20_bud_cols[k], edgecolor="white", height=0.62)
        if v >= 0.035:
            ax.text(left + v / 2, yy, f"{v:.0%}", ha="center", va="center", fontsize=10.5, fontweight="bold", color="white" if k in (0, 5) else S20_DARK)
        left += v
ax.set_yticks(range(len(S20_bars))); ax.set_yticklabels([b[0] for b in S20_bars[::-1]], fontsize=11)
left = 0.0
for k, r in s20_budget.iterrows():                      # direct labels above the top bar instead of a legend
    v = float(r["share_defaults"])
    ax.text(left + v / 2, len(S20_bars) - 0.6 + (0.22 if k % 2 else 0), f"{k+1}. {r['rule'].split(' (')[0].replace(', Sep clean', '')}", ha="center", va="bottom", fontsize=7.8, color=S20_DARK)
    left += v
ax.set_xlim(0, 1); ax.set_ylim(-0.5, len(S20_bars) + 0.15); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
K["budget_recall_all_rules"] = float(1 - K["share_def_no_warning"])
K["budget_false_alarm_all_rules"] = float(s20_budget["share_non_defaults"].iloc[:-1].sum())
K["budget_recall_rule1"] = float(s20_budget["share_defaults"].iloc[0]); K["budget_false_alarm_rule1"] = float(s20_budget["share_non_defaults"].iloc[0])
s20_finish(fig, f"Five simple rules account for {K['budget_recall_all_rules']:.0%} of defaulters, at the price of flagging {K['budget_false_alarm_all_rules']:.0%} of good clients",
           f"Rule 1 alone: {K['budget_recall_rule1']:.0%} of defaulters for {K['budget_false_alarm_rule1']:.0%} false alarms. Dark = no rule fires (the unexplained residual)",
           "s20_explanation_budget", top=0.80)
export(s20_budget.drop(columns=["ci_lo", "ci_hi"]), "s20_explanation_budget")
add_fact("20.6", "early warning budget", f"Rule 1 (2+ months late in Sep) alone accounts for {K['budget_recall_rule1']:.1%} of defaulters while flagging {K['budget_false_alarm_rule1']:.1%} of non-defaulters; all five rules reach {K['budget_recall_all_rules']:.1%} at {K['budget_false_alarm_all_rules']:.1%} false alarms", K["budget_recall_rule1"], "s20_explanation_budget", 3)

Cumulative view of the same budget: if we flag rules 1..k, what share of defaulters is caught (recall), what share of
non-defaulters is bothered (false-alarm rate), and how precise is the flag?

In [ ]:
s20_budget_cum = s20_budget.iloc[:-1][["step", "rule", "cum_share_clients", "cum_share_defaults", "cum_share_non_defaults", "cum_precision"]].copy()
s20_budget_cum["defaulters_per_false_alarm"] = (s20_budget["defaults"].cumsum() / s20_budget["non_defaults"].cumsum()).iloc[:-1]
s20_budget_cum["marginal_precision"] = s20_budget["rate"].iloc[:-1]
display(s20_budget_cum.round(4))
fig, ax = s20_hero(figsize=(9.5, 5.2))
ax.plot([0] + list(s20_budget_cum["cum_share_non_defaults"]) + [1], [0] + list(s20_budget_cum["cum_share_defaults"]) + [1], color=C_DEF, lw=2.6, marker="o")
ax.plot([0, 1], [0, 1], color=S20_GREY, ls="--", lw=1)
for r in s20_budget_cum.itertuples():
    ax.annotate(f"rules 1-{r.step}: precision {r.cum_precision:.0%}", (r.cum_share_non_defaults, r.cum_share_defaults), xytext=(8, -12), textcoords="offset points", fontsize=8.5, color=S20_DARK)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1)); ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("non-defaulters flagged (false-alarm rate)"); ax.set_ylabel("defaulters caught (recall)")
ax.set_title("Each extra rule buys less recall for more false alarms", loc="left")
plt.tight_layout(); savefig("s20_budget_curve"); plt.show()

**Observations:**
- Two September rules (2+ late, then code 1) catch 51.7% of defaulters while flagging 14.5% of good clients.
- Each next rule is less precise: 69.6% → 34.0% → 26.9% → 15.2%. The fourth rule (high utilisation) catches defaulters at below the base rate.
- After all rules, 26% of defaulters remain unexplained.
- Story use: "simple rules get half the defaulters cheaply; the second half is expensive".

### 20.7 Where does intuition fail? Expectation-versus-reality pairs
Story twists need a set-up: state what a reasonable person **expects**, then show the data. Each row is a claim of the form
"group A defaults more than group B". Verdict by rule (two-proportion z test):
**Confirmed** if A − B ≥ +3 pp and p < 0.01; **Reversed** if A − B ≤ −3 pp and p < 0.01; otherwise **Weak** (the claim is not
wrong in sign, perhaps, but too small to build a story on). Exported as `s20_myths` for a Tableau "myth selector".

In [ ]:
S20_q_bill = df["BILL_AMT1"].quantile([0.25, 0.75]); S20_q_spend = df["AVG_SPEND"].quantile([0.25, 0.75])
S20_late2_sep_only = (df["PAY_1"] >= 2)
S20_late2_apr_only = S20_msl == 5          # late in Apr, clean May-Sep
S20_anylate_sep = df["PAY_1"] >= 1
S20_MYTHS = [  # (chapter, expectation, label A, mask A, label B, mask B)
    ("2 Usual suspects", "Young clients are riskier", "Age <= 25", df["AGE"] <= 25, "Age 36-45", df["AGE"].between(36, 45)),
    ("2 Usual suspects", "Older clients are riskier", "Age 56+", df["AGE"] >= 56, "Age 36-45", df["AGE"].between(36, 45)),
    ("2 Usual suspects", "Women are riskier than men", "Female", df["SEX_LABEL"] == "Female", "Male", df["SEX_LABEL"] == "Male"),
    ("2 Usual suspects", "Less education means more default", "High school", df["EDU_LABEL"] == "High school", "Graduate school", df["EDU_LABEL"] == "Graduate school"),
    ("6 The twist", "Unknown education is a red flag", "Education Other/Unknown", df["EDU_LABEL"] == "Other/Unknown", "Documented education", df["EDU_LABEL"] != "Other/Unknown"),
    ("2 Usual suspects", "Singles are riskier than married clients", "Single", df["MAR_LABEL"] == "Single", "Married", df["MAR_LABEL"] == "Married"),
    ("2 Usual suspects", "A bigger credit line means more risk", "Limit > NT$ 300k", df["LIMIT_BAL"] > 300_000, "Limit <= NT$ 50k", df["LIMIT_BAL"] <= 50_000),
    ("5 The wallet", "A bigger balance means more risk", "Sep bill in top quartile", df["BILL_AMT1"] >= S20_q_bill.loc[0.75], "Sep bill in bottom quartile", df["BILL_AMT1"] <= S20_q_bill.loc[0.25]),
    ("5 The wallet", "Big spenders are riskier", "Avg spend in top quartile", df["AVG_SPEND"] >= S20_q_spend.loc[0.75], "Avg spend in bottom quartile", df["AVG_SPEND"] <= S20_q_spend.loc[0.25]),
    ("5 The wallet", "High utilisation is risky", "Sep utilisation > 90%", df["UTIL_1"] > 0.9, "Sep utilisation 0-30%", df["UTIL_1"].between(0, 0.3)),
    ("5 The wallet", "High utilisation is risky even among revolvers", "Revolving, util > 90%", S20_rev & (df["UTIL_1"] > 0.9), "Revolving, util 0-30%", S20_rev & df["UTIL_1"].between(0, 0.3)),
    ("5 The wallet", "High utilisation is risky even among the already late", "Late 2+, util > 90%", (df["PAY_1"] >= 2) & (df["UTIL_1"] > 0.9), "Late 2+, util 0-30%", (df["PAY_1"] >= 2) & df["UTIL_1"].between(0, 0.3)),
    ("5 The wallet", "Going over the limit is a red flag", "Over limit at least once", df["OVERLIMIT_ANY"] == 1, "Never over limit", df["OVERLIMIT_ANY"] == 0),
    ("5 The wallet", "Paying nothing this month is a red flag", "Paid 0 in Sep (bill due)", (df["PAY_AMT1"] == 0) & (df["BILL_AMT2"] > 0), "Paid something in Sep (bill due)", (df["PAY_AMT1"] > 0) & (df["BILL_AMT2"] > 0)),
    ("6 The twist", "Rising utilisation with falling payment amounts signals stress", "Util rising, payment amounts falling", S20_stress, "Everyone else", ~S20_stress),
    ("6 The twist", "Revolving a balance is riskier than not using the card", "Sep revolving (min paid)", df["PAY_1"] == 0, "Sep no consumption", df["PAY_1"] == -2),
    ("6 The twist", "Revolvers are riskier than full payers", "Sep revolving (min paid)", df["PAY_1"] == 0, "Sep paid in full", df["PAY_1"] == -1),
    ("6 The twist", "Always paying in full makes a client safer than average", "Everyone else", df["N_PAIDFULL"] < 6, "Paid in full all six months", df["N_PAIDFULL"] == 6),
    ("3 Behaviour is destiny", "One month late is already alarming", "Sep 1 month late", df["PAY_1"] == 1, "Sep not late", df["PAY_1"] <= 0),
    ("3 Behaviour is destiny", "Two months late is much worse than one", "Sep 2 months late", df["PAY_1"] == 2, "Sep 1 month late", df["PAY_1"] == 1),
    ("4 The clock", "A recent delay is worse than an old one", "Late in Sep (any code)", S20_anylate_sep, "Last late in Apr, clean since", S20_late2_apr_only),
    ("4 The clock", "Once late, always risky (old delay vs never late)", "Last late in Apr, clean since", S20_late2_apr_only, "Never late in six months", df["N_LATE"] == 0),
    ("4 The clock", "Turning late is worse than recovering", "Not late in Apr, late in Sep", (df["PAY_6"] <= 0) & (df["PAY_1"] >= 1), "Late in Apr, not late in Sep", (df["PAY_6"] >= 1) & (df["PAY_1"] <= 0)),
]
S20_myth_rows = []
for chap, exp_, la, ma, lb, mb in S20_MYTHS:
    ra, na, rb, nb, d, p = s20_two_prop(ma, mb)
    if np.isnan(d) or min(na, nb) < 30:
        verdict = "Too few clients"
    elif d >= 0.03 and p < 0.01:
        verdict = "Confirmed"
    elif d <= -0.03 and p < 0.01:
        verdict = "Reversed"
    else:
        verdict = "Weak"
    S20_myth_rows.append(dict(chapter=chap, expectation=exp_, group_a=la, n_a=na, rate_a=ra, group_b=lb, n_b=nb, rate_b=rb,
                              diff_pp=d * 100 if not np.isnan(d) else np.nan, ratio=ra / rb if rb else np.nan, p_value=p, verdict=verdict))
s20_myths = pd.DataFrame(S20_myth_rows)
s20_myths["reveal"] = [f"{r.group_a}: {r.rate_a:.1%} vs {r.group_b}: {r.rate_b:.1%} ({r.diff_pp:+.1f} pp) -> {r.verdict}" for r in s20_myths.itertuples()]
with pd.option_context("display.max_colwidth", None, "display.max_rows", 100):
    display(s20_myths[["chapter", "expectation", "group_a", "n_a", "rate_a", "group_b", "n_b", "rate_b", "diff_pp", "p_value", "verdict"]].round(4))
print(s20_myths["verdict"].value_counts().to_string())
export(s20_myths, "s20_myths")

The same table as a chart: one dumbbell per expectation (A = filled dot, B = hollow dot), coloured by verdict. Reversed and
Weak rows are the candidates for "reveal" slides.

In [ ]:
S20_vcol = {"Confirmed": S20_MID, "Weak": "#f0a35e", "Reversed": C_DEF, "Too few clients": S20_GREY}
S20_m = s20_myths.sort_values("diff_pp").reset_index(drop=True)
fig, ax = s20_hero(figsize=(12.5, 0.36 * len(S20_m) + 2.6))
for yy, r in S20_m.iterrows():
    col = S20_vcol[r["verdict"]]
    ax.plot([r["rate_b"], r["rate_a"]], [yy, yy], color=col, lw=2.4, zorder=2)
    ax.scatter([r["rate_a"]], [yy], s=70, color=col, zorder=3)
    ax.scatter([r["rate_b"]], [yy], s=70, facecolor="white", edgecolor=col, lw=1.8, zorder=3)
    ax.text(1.0, yy, f"{r['diff_pp']:+.0f} pp  {r['verdict']}", transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=9, fontweight="bold", color=col)
ax.set_yticks(range(len(S20_m))); ax.set_yticklabels(S20_m["expectation"], fontsize=9)
ax.axvline(BASE_RATE, color=S20_GREY, ls=":", lw=1)
ax.set_xlim(0, max(0.95, float(S20_m[["rate_a", "rate_b"]].max().max()) + 0.25)); ax.xaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel("default rate (filled = group expected to be riskier, hollow = comparison group)", color=S20_DARK)
ax.spines["left"].set_visible(False); ax.tick_params(axis="y", length=0)
S20_vc = s20_myths["verdict"].value_counts()
s20_finish(fig, f"Of {len(s20_myths)} common-sense expectations, {int(S20_vc.get('Confirmed', 0))} hold, {int(S20_vc.get('Weak', 0))} are weak and {int(S20_vc.get('Reversed', 0))} are reversed",
           "Expectation: the filled group defaults more than the hollow group", "s20_myths", top=0.88)
for r in s20_myths[s20_myths["verdict"] == "Reversed"].itertuples():
    add_fact("20.7", "myth reversed", f"Expectation '{r.expectation}' is reversed: {r.group_a} {r.rate_a:.1%} vs {r.group_b} {r.rate_b:.1%} ({r.diff_pp:+.1f} pp)", r.diff_pp, "s20_myths", 2)
add_fact("20.7", "myth scorecard", f"Of {len(s20_myths)} common-sense expectations tested, {int(S20_vc.get('Confirmed', 0))} are confirmed, {int(S20_vc.get('Weak', 0))} are weak (< 3 pp or not significant) and {int(S20_vc.get('Reversed', 0))} are reversed", int(S20_vc.get("Reversed", 0)), "s20_myths", 3)

**Observations:**
- 13 of 23 common beliefs hold, 5 are reversed, and 5 are too weak to use.
- Reversed: women are not riskier (men 24.2% vs women 20.8%); big lines are safer (13.3% vs 31.8%); big spenders are safer (14.2% vs 35.2%); revolvers are safer than full payers (12.8% vs 16.8%); unknown education is safe (7.1%).
- Weak: "rising utilisation with falling payment amounts" (20.9% vs 22.3%) and "a bigger balance is riskier" (20.0% vs 23.0%).
- Confirmed and strong: paying nothing this month (+33 pp), two months late vs one (+35 pp), recent vs old delay (late in September 50.3% vs last late in April and clean since 18.7%, +32 pp; both use "any delay", so the September-only code 1 does not bias it).
- Story use: a "myth or fact?" quiz slide. It is the most engaging part of the deck.

### 20.8 How do we tell it in Tableau? Story Points storyboard
Ten story points, each a single idea. Captions are generated from the numbers above and limited to **12 words** (Tableau's
navigator truncates long captions). The table states the sheet type, the source CSV (all exported by this notebook), the
highlight / annotation, and the transition to the next point.

**Recommended navigation flow** (martini glass: author-driven stem, reader-driven bowl):

`Overview` → `Drivers (who vs how)` → `Time` → `Wallet` → `Twists` → `Action` → **free explorer**

**Design notes for the dashboard**
* **Colour semantics never change:** blue (`C_OK` #2b8cbe) = did not default / safe, red (`C_DEF` #e34a33) = default / risk,
  grey = context. One accent per view; sequential red ramp for rates; no rainbow palettes.
* **Annotation over legend:** label lines and bars directly; put the number in the title; remove gridlines, borders, axis titles that repeat the caption.
* **Progressive disclosure:** each story point shows one view; details (CI, n, definitions) live in tooltips; filters appear only from Act II onwards.
* **Guided filters:** at most one filter or parameter per story point, pre-set to the state that makes the point; a reset button returns to the author's view.
* **Tooltips carry the evidence:** `n`, defaults, rate, 95% Wilson CI, lift vs base, and a one-line definition of the field.
* **Consistency:** months always chronological Apr → Sep; status always in `STATUS_GROUP_ORDER`; the base rate appears as the same dotted reference line everywhere.
* **Honesty:** blank out cells with n < 100; say "association", never "cause"; the caveats of 20.9 get their own closing story point.

In [ ]:
def s20_caption(text, max_words=12):
    words = str(text).split()
    return " ".join(words[:max_words])

S20_SP = [
    dict(caption=f"{K['base_rate']*100:.0f} in 100 cardholders defaulted: Taiwan, October 2005",
         chapter=S20_CHAPTERS[0], sheet_type="Unit (waffle) chart + 3 KPI tiles", source_csv="tableau_kpis.csv; s20_hero_titles.csv",
         highlight="Red squares; NT$ at risk tile", transition="Scale set. Next question: who are they?"),
    dict(caption=f"Demographics move default by {K['demo_max_gap_pp']:.0f} points at most",
         chapter=S20_CHAPTERS[1], sheet_type="Dumbbell (range) chart, demographics only", source_csv="s20_gap_dumbbell.csv; s03_rate_by_level.csv",
         highlight="Short grey dumbbells; base-rate line", transition="Same axis, now add behaviour (animate in the red rows)"),
    dict(caption=f"Behaviour moves it by {K['beh_max_gap_pp']:.0f} points",
         chapter=S20_CHAPTERS[1], sheet_type="Same dumbbell with behaviour rows revealed", source_csv="s20_gap_dumbbell.csv",
         highlight="Red rows: September status, months late", transition="Zoom into the longest dumbbell: September status"),
    dict(caption=f"Two months late in September: {K['rate_sep_late2']:.0%} default",
         chapter=S20_CHAPTERS[2], sheet_type="Bar chart by status code + 50% reference line", source_csv="s20_status_cliff.csv; s03_woe_bins.csv",
         highlight="Annotation on the 1 -> 2 months step", transition="Was this visible earlier? Rewind six months"),
    dict(caption=f"In April {K['late2_apr_defaulters']:.0%} of future defaulters were already late",
         chapter=S20_CHAPTERS[3], sheet_type="Small-multiple lines (3 metrics x 6 months)", source_csv="s20_timeline.csv; s05_client_month_panel.csv",
         highlight="Widening gap label Apr vs Sep", transition="Status is one lens; what about money?"),
    dict(caption=f"Riskiest wallet cell ({S20_hi_u} used, {S20_hi_p} repaid): {K['wallet_hi_rate']:.0%}",
         chapter=S20_CHAPTERS[4], sheet_type="Highlight table (utilisation x repayment share)", source_csv="s20_wallet_heatmap.csv; s06_sow_grid.csv",
         highlight="Boxed riskiest and safest cells", transition="So far intuition holds. Now the surprises"),
    dict(caption=f"Twist: equal behaviour leaves {K['demo_gap_retained_median']:.0%} of demographic gaps",
         chapter=S20_CHAPTERS[5], sheet_type="Slope chart crude vs standardised", source_csv="s20_demo_adjusted.csv; s20_myths.csv",
         highlight="Converging lines; myth selector parameter", transition="If not demographics, can rules catch everyone?"),
    dict(caption=f"Twist: {K['share_def_no_warning']:.0%} of defaulters gave no warning at all",
         chapter=S20_CHAPTERS[5], sheet_type="Ordered funnel bar + 100% stacked budget bars", source_csv="s20_explanation_budget.csv",
         highlight="Dark 'no visible warning' bar", transition="Limits known. Where should the bank act?"),
    dict(caption=f"{K['pareto_k80']} of {K['pareto_n_seg']} segments hold {K['pareto_cum_k80']:.0%} of money at risk",
         chapter=S20_CHAPTERS[6], sheet_type="Pareto bar + size-vs-rate quadrant scatter", source_csv="s20_exposure_pareto.csv; s20_segment_quadrant.csv",
         highlight="Red bars to 80%; 'act' quadrant", transition="Hand over control: free exploration"),
    dict(caption="Explore: your segment, your threshold, and the caveats",
         chapter=S20_CHAPTERS[6], sheet_type="Explorer dashboard (all filters) + caveat panel", source_csv="tableau client-level extract; s20_risk_ladders.csv",
         highlight="Reset button; caveat text box", transition="End (martini-glass bowl)"),
]
s20_story_points = pd.DataFrame(S20_SP)
s20_story_points["caption"] = s20_story_points["caption"].map(s20_caption)
s20_story_points.insert(0, "story_point", np.arange(1, len(s20_story_points) + 1))
s20_story_points["caption_words"] = s20_story_points["caption"].str.split().str.len()
assert s20_story_points["caption_words"].max() <= 12
with pd.option_context("display.max_colwidth", None):
    display(s20_story_points)
export(s20_story_points, "s20_story_points")

The chapter-level storyboard of 20.1, now completed with the computed key number and the computed takeaway of each chapter.
Exported as `s20_storyboard.csv` (one row per chapter) next to `s20_story_points.csv` (one row per Tableau story point).

In [ ]:
S20_KEY_FMT = {"base_rate": f"{K['base_rate']:.1%}", "demo_max_gap_pp": f"{K['demo_max_gap_pp']:.1f} pp", "rate_sep_late2": f"{K['rate_sep_late2']:.1%}",
               "late2_apr_defaulters": f"{K['late2_apr_defaulters']:.1%}", "wallet_hi_rate": f"{K['wallet_hi_rate']:.1%}",
               "share_def_clean_sep": f"{K['share_def_clean_sep']:.1%}", "ead_def_total": f"NT$ {K['ead_def_total']/1e6:,.0f} M"}
s20_storyboard["key_number"] = s20_storyboard["key_number_field"].map(S20_KEY_FMT)
s20_storyboard["computed_takeaway"] = [
    S20_HERO_TITLES[1], S20_HERO_TITLES[2], S20_HERO_TITLES[3], S20_HERO_TITLES[4], S20_HERO_TITLES[5],
    S20_HERO_TITLES[6] + " | " + S20_HERO_TITLES[7], S20_HERO_TITLES[8] + " | " + S20_HERO_TITLES[9]]
s20_storyboard["n_facts_in_ledger"] = s20_storyboard["chapter"].map(s20_story_facts["chapter"].value_counts()).fillna(0).astype(int)
with pd.option_context("display.max_colwidth", None):
    display(s20_storyboard[["act", "chapter", "question", "key_number", "computed_takeaway", "tableau_dashboard", "n_facts_in_ledger"]])
export(s20_storyboard, "s20_storyboard")
s20_nav_flow = pd.DataFrame({
    "order": range(1, 8),
    "dashboard": ["D1 Overview", "D2 Who (demographics)", "D3 Drivers (behaviour)", "D4 Time (six months)", "D5 Wallet", "D6 Twists", "D7 Action + free explorer"],
    "mode": ["author-driven"] * 6 + ["reader-driven"],
    "filters_unlocked": ["none", "demographic selector", "status click-filter", "month pages", "status filter", "myth selector", "all"],
    "story_points": ["1", "2-3", "4", "5", "6", "7-8", "9-10"]})
display(s20_nav_flow)
export(s20_nav_flow, "s20_navigation_flow")

**Observations:**
- Ten story points, each with a caption of 12 words or fewer, all computed from the data.
- The flow is author-driven for points 1-9 and reader-driven at point 10 (the "martini glass").
- Story use: build the Tableau story in this exact order. Use the source CSV listed for each sheet.

### 20.9 What can this story NOT claim? Open questions and caveats (the honesty slide)
* **One bank, one moment.** A single Taiwanese issuer, April-September 2005, at the peak of a national card-debt bubble.
  Rates and thresholds describe that portfolio; they are not universal constants.
* **No income, no bureau data, no other debts.** Utilisation is measured against *this* card's limit only. "Share of wallet"
  is a proxy built from limit, bills and payments; true wallet share is unobserved.
* **The label is thin.** `DEFAULT` = payment default in October 2005 as recorded by the bank. No severity, no recovery, no
  cure; a client one day past due and a client who never pays again look identical.
* **Status codes are not fully documented.** −2 and 0 are undocumented in the original paper; code 1 is practically a
  September-only code (in April-August a first delay appears as 2), so "one month late" cannot be compared across months.
* **Six snapshots, not a history.** We see month-end states, not transactions; account age and origination date are unknown,
  so new and seasoned accounts are mixed (left-censoring). `PAY_AMT`/`BILL_AMT` identities hold only approximately.
* **Education / marriage "other" codes** (0, 4, 5, 6 / 0, 3) are small, undocumented and unusually safe: likely a product or
  channel artefact, not a demographic finding.
* **Label noise exists.** Some clients with identical feature rows carry different labels; some "paid in full every month"
  clients default. A residual of defaulters with no visible warning is expected with these fields.
* **Association, not causation.** Raising a credit line does not make a client safer; the bank gave larger lines to safer clients.
  Policy what-ifs (Section 18) are accounting exercises under stated assumptions, not forecasts.
* **NT$ at risk is an exposure proxy** (September balance of defaulters), not expected loss: no LGD, no interest margin.
* **Fairness.** Sex, age, marital status and education are shown to *demonstrate how little they add*; using them for
  decisions would be discriminatory in most jurisdictions and is not recommended.

The cell below quantifies the caveats that can be quantified, so the honesty slide carries numbers too.

In [ ]:
S20_feat_cols = [c for c in ["LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE"] + PAY_COLS + BILL_COLS + PAMT_COLS]
S20_dup = df.duplicated(subset=S20_feat_cols, keep=False)
S20_dup_conf = int((df[S20_dup].groupby(S20_feat_cols)[TARGET].nunique() > 1).sum())
s20_caveats = pd.DataFrame([
    dict(caveat="Status code 1 is a September-only code", metric="clients with code 1: Sep vs Apr-Aug total", value=f"{K['n_code1_sep']:,} vs {K['n_code1_other']:,}"),
    dict(caveat="Undocumented status codes dominate", metric="share of all 180,000 monthly statuses coded -2 or 0", value=f"{float(np.isin(df[PAY_COLS].to_numpy(), [-2, 0]).mean()):.1%}"),
    dict(caveat="Undocumented education codes", metric="clients with EDUCATION in {0,4,5,6} and their default rate", value=f"{K['n_edu_unknown']:,} at {K['rate_edu_unknown']:.1%}"),
    dict(caveat="Feature-duplicate rows", metric="rows sharing all 23 features with another row / groups with conflicting labels", value=f"{int(S20_dup.sum()):,} / {S20_dup_conf}"),
    dict(caveat="Clean payers who default", metric="defaulters never late in six months", value=f"{K['n_def_never_late']:,} ({K['share_def_never_late']:.1%} of defaulters)"),
    dict(caveat="Defaulters invisible to simple rules", metric="defaulters with no visible warning (20.4 buckets)", value=f"{K['n_def_no_warning']:,} ({K['share_def_no_warning']:.1%})"),
    dict(caveat="Negative balances (credits) exist", metric="clients with at least one negative bill", value=f"{int(df['NEG_BILL_ANY'].sum()):,}"),
    dict(caveat="Bills above the credit line exist", metric="clients over limit at least once", value=f"{int(df['OVERLIMIT_ANY'].sum()):,}"),
    dict(caveat="No activity at all", metric="clients with zero bills and zero payments in all six months", value=f"{int(((df[BILL_COLS] == 0).all(axis=1) & (df[PAMT_COLS] == 0).all(axis=1)).sum()):,}"),
    dict(caveat="Crisis-period base rate", metric="default rate in this sample", value=f"{K['base_rate']:.1%}"),
])
with pd.option_context("display.max_colwidth", None):
    display(s20_caveats)
export(s20_caveats, "s20_caveats")
add_fact("20.9", "data caveat", f"{int(S20_dup.sum()):,} rows share all 23 features with another row and {S20_dup_conf} such groups carry conflicting default labels: an irreducible noise floor", int(S20_dup.sum()), None, 1)

**Observations:**
- Code 1 appears 3,688 times in September but only 34 times in April-August. "One month late" is a September-only code.
- 66.9% of all monthly statuses are -2 or 0, which the original paper does not document.
- 108 rows are feature duplicates and 21 groups carry conflicting labels. 795 clients had no bills and no payments at all.
- Story use: one honest closing slide. It protects the story from the obvious questions.

### 20.10 One-page executive summary (skeleton; the filled version is 20.0)

> **Headline.** _In a 30,000-client Taiwanese card portfolio (2005), ____ % defaulted the next month; **how** clients paid in the
> last months, not **who** they are, separates the ____ % segment from the ____ % segment._

**1. The setting.** ____ of 30,000 clients defaulted (____ %), owing NT$ ____ M (____ % of balances). _[Hero 1]_

**2. The usual suspects.** Sex / age / education / marriage move the rate by at most ____ pp; behaviour by ____ pp. _[Hero 2]_

**3. Behaviour is destiny.** Two months late in September → ____ % default; not late → ____ %. ____ % of clients produce ____ % of defaulters. _[Hero 3]_

**4. The clock.** In April ____ % of future defaulters were already 2+ months late (others ____ %); a delay is worth ____ % when recent and ____ % when six months old. _[Hero 4]_

**5. The wallet.** Riskiest cell ____ % (utilisation ____, repaid ____) vs safest ____ %. _[Hero 5]_

**6. The twists.** (a) With equal September behaviour only ____ % of a demographic gap survives. (b) ____ % of defaulters gave no warning. (c) ____ of ____ common-sense expectations are weak or reversed: ____ . _[Hero 6, 7, myths]_

**7. So what.** ____ of ____ segments hold ____ % of the NT$ at risk; the action quadrant is ____ % of clients and ____ % of defaulters. Levers: ____ . _[Hero 8, 9]_

**Caveats.** One bank, 2005 crisis, no income, thin label, September code-1 quirk, six snapshots. Association, not causation.

The cell below prints the 15 highest-ranked facts of the ledger (all sections), ready to paste into the blanks.

In [ ]:
print(f"TOP 15 STORY FACTS (of {len(s20_story_facts)} in the ledger; rank = strength, then round-robin across chapters)\n")
for r in s20_story_facts.head(15).itertuples():
    print(f"{r.rank:>2}. [{r.chapter} | S{r.section} | strength {r.strength}] {r.fact}")
print("\nComputed hero-chart titles:")
for k, v in S20_HERO_TITLES.items():
    print(f"  Hero {k}: {v}")

**Observations:**
- The top 15 facts cover all seven chapters, so the summary is balanced.
- The filled one-page summary is in 20.0 at the top of this section.
- Story use: read 20.0 aloud as the 2-minute version of the presentation.

### 20.11 Facts registered by this section and exports

| Export (CSV in `tableau/`) | Content | Story chapter |
|---|---|---|
| `s20_story_facts` | ranked, de-duplicated ledger of every `add_fact` in the notebook | all |
| `s20_storyboard`, `s20_story_points`, `s20_navigation_flow`, `s20_hero_titles` | chapter storyboard, 10 Tableau story points, dashboard flow, computed titles | all |
| `s20_gap_dumbbell` | lowest vs highest default-rate level per variable | 2 |
| `s20_status_cliff` | default rate by September status code | 3 |
| `s20_timeline` | defaulters vs others, three metrics × six months | 4 |
| `s20_wallet_heatmap` | utilisation × repayment-share grid | 5 |
| `s20_demo_adjusted`, `s20_myths` | crude vs status-standardised rates; expectation-vs-reality table | 6 |
| `s20_explanation_budget` | ordered, mutually exclusive warning rules: recall, false alarms, NT$ | 6 |
| `s20_exposure_pareto`, `s20_segment_quadrant`, `s20_risk_ladders` | NT$ at risk by segment; size-vs-rate quadrant; conditional risk ladders | 7 |
| `s20_caveats` | quantified caveats | closing |

New `df` columns created by this section: **none**.

In [ ]:
s20_own_facts = pd.DataFrame([f for f in STORY_FACTS if str(f["section"]).startswith("20")])
with pd.option_context("display.max_colwidth", None, "display.max_rows", 100):
    display(s20_own_facts)

## 22. Column glossary: every column created by sections 1-20

**Why this section exists.** Sections 1-20 add many columns to `df`: Section 5 trajectory and trend columns, Section 6
share-of-wallet columns (`SOW_`, including the quarterly paydown `SOW_QPD_*`), Section 7 engineered features (`FE_`), Section 8
segments (`RISK_TIER`, `CLUSTER`, `PERSONA`, `RFM_*`), and the prefixed deep-dive columns (`PN_` panel, `CF_` column forensics,
`CI_` conditional interactions, `DM_` demographics, `MF_` money flow, `EW_` early warning, `EX_` exposure, `CS_` case
studies). This glossary is written for a reader who has never seen this data. For each column: what it is in plain
words with its unit (NT$, share of the credit line, months, count), what a high versus a low value means, one worked
example, and a one-line "watch out". It is written from the code that creates the column, not from intentions.

Quick unit guide: **NT$** = New Taiwan dollars (2005). **Utilisation / share of line** = bill / credit line, so 0.45 = 45% of the line.
**pp / points** = percentage points. **Months** are calendar months Apr-Sep 2005.

Then a profiler runs over **every** column now in `df` (raw, Section 0, Sections 1-20): type, % missing,
min / median / max (or top categories), and its value for three fixed example clients. Any column without a
definition is flagged **"definition missing"**, so new columns are caught automatically.

Raw and Section 0 columns are explained in depth in **Section 0B**; their short definitions are reused here.

**Prefix map:** `PN_` = 11 panel structure | `CF_` = 12 column forensics | `CI_` = 14 conditional interactions | `DM_` = 15 demographics |
`MF_` = 16 money flow | `EW_` = 17 early warning | `EX_` = 18 exposure economics | `CS_` = 19 case studies | `CX_` = 19B competition ideas. Sections 13 and 20 add no columns.

**Two warnings that apply to many columns below.**
1. **Columns built from the outcome** (`DM_BEH_EXPECTED*`, `DM_CELL_RATE_EB`, `CF_IMPLAUSIBLE_DEFAULT`, `EW_GROUP`, `EW_SILENT_DEFAULTER`,
   `EW_NEVER_LATE_DEFAULTER`, `CS_N_BREAKS`, `CS_RULE_BREAKER`, `SOW_EAD`, cluster ids ordered by default rate) summarise what happened.
   Never use them to "explain" default; they contain it.
2. **Month order.** Inside sections 11-17 arrays usually run Apr -> Sep (chronological), while raw indices run Sep (1) -> Apr (6). Month names in the definitions are always calendar months.

### 22.1 The glossary (written from the code), grouped by the section that creates each column
Each entry: `column: (section, plain definition, how to read it with an example, caveat)`. Families of similar
columns (`EW_SIGN_*`, `MF_TS_*`) are defined by a rule so every member gets a definition.

In [ ]:
S97_T0 = time.time()
S97_GLOSSARY = {
    # ======================= Section 5: temporal behaviour =======================
    "TRAJ_SEQ": ("5", "The client's six monthly status groups written as a 6-letter word, April first: N = inactive, P = paid in full, R = revolving (paid part), 1/2/3 = 1, 2, 3+ months late.",
                 "Read left to right like a calendar. Example: 'PPPP22' = paid in full Apr-Jul, then 2 months late in Aug and Sep. 'RRRRRR' = carried a balance all six months.", "There are hundreds of distinct words; most are rare, so group them before charting."),
    "TRAJ_N_CHANGES": ("5", "How many times the status group changed from one month to the next (0-5 changes).", "0 = same behaviour all window (stable); 5 = changed every month. Example: 'PPRRR2' -> 2 changes.", "Counts any change, good or bad, the same."),
    "TRAJ_ENTROPY": ("5", "How mixed the six status groups are, in bits (0 to about 2.6). Shannon entropy of the share of months in each group.",
                     "0 = one group all six months (very regular). Higher = more mixed behaviour. Example: 3 months P + 3 months R -> 1.0 bit.", "Ignores the order of months."),
    "_EVER_LATE": ("5", "Helper label: 'Ever late' if the client was late (status >= 1) in any month, else 'Never late'.", "Example: N_LATE = 2 -> 'Ever late'.", "Helper column for one table; underscore = technical."),
    "MONTHS_SINCE_LATE": ("5", "Months between the most recent late month and September (0 = late in Sep, 5 = last late in Apr, 7 = never late).",
                          "Low = recent trouble (riskier); 7 = clean record. Example: last late in Jul -> 2.", "7 is a code for 'never', not a distance; do not average it."),
    "FIRST_LATE_MONTH": ("5", "The first month (Apr..Sep) in which the client was late, or 'Never'.", "Example: late in Jun and Sep -> 'Jun'. 'Apr' may mean the trouble started before the window.", "April is the first month we see, so 'Apr' is left-censored."),
    "DELAY_TREND_BAND": ("5", "DELAY_TREND (Sep status code minus Apr code) clipped to -3..+3 and shown as labels.", "Positive = worse in Sep than in Apr. Example: Apr -1, Sep 2 -> '>=+3'.", "Inherits DELAY_TREND's flaw: -2 -> 0 counts as +2 without any delay."),
    "UTIL_TREND_BAND": ("5", "Direction of the utilisation trend: Flat (within +/-0.5 point of the line per month), Down / Strong down, Up / Strong up (split at the median of each side).",
                        "Example: utilisation rising 3 points a month -> 'Up' or 'Strong up'.", "'Strong' is relative to other clients, not an absolute size."),
    "BILL_TREND_BAND": ("5", "Same banding for the bill trend; Flat = within +/-NT$500 per month.", "Example: bill growing NT$5,000 a month -> 'Up' or 'Strong up'.", "In NT$, so large credit lines drift into the strong bands."),
    "PAY_TREND_BAND": ("5", "Same banding for the payment trend; Flat = within +/-NT$100 per month.", "Example: payments shrinking NT$1,000 a month -> 'Down' or 'Strong down'.", "In NT$, not scaled by the bill."),
    "BILL_CV": ("5", "How much the bill swings month to month: standard deviation / mean of the six bills (no unit). NaN if the mean bill <= 0.",
                "0 = same bill every month; 1 = swings as large as the bill itself. Example: bills 10k, 30k, 10k, 30k, 10k, 30k -> 0.5.", "Explodes when the mean bill is tiny."),
    "PAY_CV": ("5", "Same swing measure for the six payments. NaN if the client never paid.", "0 = pays the same amount every month (e.g. a fixed 3,000). High = irregular lump payments.", "Explodes for clients who paid once."),
    "BILL_CV_BAND": ("5", "BILL_CV in five equal-size groups (Q1 = steadiest ... Q5 = most volatile) plus 'Undefined (mean bill<=0)'.", "Example: BILL_CV 0.02 -> 'Q1 ...'.", "Band edges come from this data."),
    "PAY_CV_BAND": ("5", "PAY_CV in five equal-size groups plus 'No payments'.", "Example: fixed monthly payment -> Q1.", "Band edges come from this data."),
    "N_MINPAY": ("5", "Months (0-5) in which the client paid something but less than 10% of the previous bill (a 'minimum-payment' month).",
                 "0 = never a minimum payer; 5 = minimum payer every month (debt shrinks very slowly). Example: 3,000 paid on a 67,060 bill -> counts.", "10% is our threshold, not the bank's real minimum due."),
    # ======================= Section 6: share of wallet =======================
    "SOW_UTIL_SEP": ("6", "September utilisation (Sep bill / credit line) with credit balances set to 0. A share: 0.45 = 45% of the line.", "High = most of the line is used. Example: bill 45,000 on 100,000 -> 0.45.", "A credit balance shows as 0, same as an unused card."),
    "SOW_UTIL_AVG": ("6", "Average of the six monthly utilisations (credit balances set to 0).", "Example: 0.40, 0.42, 0.45, 0.47, 0.50, 0.52 -> 0.46 (46% of the line on average).", "Same as AVG_UTIL except negatives are floored at 0."),
    "SOW_UTIL_PEAK": ("6", "Highest monthly utilisation in the window (credit balances set to 0).", "Example: one month at 1.10 -> 1.10 = 10% over the line at the worst point.", "One month decides it."),
    "SOW_HEADROOM_SEP": ("6", "Room left on the card in September, in NT$: credit line - Sep bill.", "High = lots of unused credit. Negative = over the limit. Example: line 100,000, bill 45,000 -> 55,000.", "In NT$, so big lines always look roomy."),
    "SOW_HEADROOM_PCT": ("6", "Room left as a share of the line: 1 - Sep utilisation.", "0.55 = 55% of the line still free; negative = over limit.", "Uses the raw (not floored) utilisation."),
    "SOW_CAPTURE_BAND": ("6", "SOW_UTIL_AVG in bands: 0 (none/credit), 0-10%, 10-30%, 30-50%, 50-70%, 70-90%, 90-100%, >100%.", "Example: 0.46 -> '30-50%'.", "-"),
    "SOW_CAPTURE_SEP_BAND": ("6", "Same bands applied to SOW_UTIL_SEP (September only).", "Example: Sep utilisation 0.95 -> '90-100%'.", "-"),
    "SOW_HEADROOM_BAND": ("6", "SOW_HEADROOM_SEP in NT$ bands: Over limit (<0), 0-10k, 10-50k, 50-100k, 100-200k, >200k.", "Example: 55,000 -> '50-100k'.", "Absolute NT$, not a share."),
    "SOW_SPEND_SHARE_AVG": ("6", "Average monthly new charges as a share of the line, May-Sep (negative spend set to 0).", "0.05 = new charges worth 5% of the line each month. Example: 5,000 a month on a 100,000 line -> 0.05.", "Spend includes interest and fees (see 0B.6)."),
    "SOW_SPEND_HHI": ("6", "How concentrated spending is across the 5 months (Herfindahl index, 0.2 to 1). NaN if no spend.",
                      "0.2 = spread evenly over 5 months; 1 = all spending in a single month. Example: 10k in one month only -> 1.0.", "Says nothing about the amount."),
    "SOW_SPEND_ACTIVE_M": ("6", "Number of months (0-5) with positive implied spend.", "5 = card used (or charged interest) every month. Example: spends in Jul and Sep only -> 2.", "Interest alone makes a month 'active'."),
    "SOW_SPEND_TREND": ("6", "Slope of the monthly spend share May->Sep (share of line per month).", "Positive = spending more each month. Example: +0.01 = 1 extra point of the line per month.", "Five points; noisy."),
    "SOW_SPEND_BAND": ("6", "SOW_SPEND_SHARE_AVG in bands: 0 (no spend), 0-2%, 2-5%, 5-10%, 10-20%, >20%.", "Example: 0.05 -> '2-5%' (right edge included).", "-"),
    "SOW_SPEND_CONC_BAND": ("6", "SOW_SPEND_HHI in bands (spread ... concentrated) plus 'No spend'.", "Example: HHI 0.9 -> the most concentrated band.", "-"),
    "SOW_PAYSHARE_MED": ("6", "Typical share of the previous bill repaid: median over May-Sep of PAY_RATIO clipped to 0-1.", "1 = pays everything (transactor); 0.04 = ~minimum payer; 0 = pays nothing. Example: 5%, 5%, 4%, 5%, 6% -> 0.05.", "NaN if no bill was ever due; over-payment is cut to 1."),
    "SOW_PAYSHARE_MEAN": ("6", "Mean instead of median of the same monthly shares.", "Example: 1, 1, 0, 1, 1 -> 0.8 (one missed month pulls it down).", "More sensitive to one odd month than the median."),
    "SOW_PAY_LIMIT_AVG": ("6", "Average monthly payment as a share of the credit line.", "0.02 = pays about 2% of the line each month. Example: 2,000 a month on 100,000 -> 0.02.", "Mixes 'large line' with 'small payment'."),
    "SOW_N_TRANSACTOR_M": ("6", "Months (0-5) in which the client repaid at least 95% of the previous bill.", "5 = full payer every month; 0 = always carries debt.", "Months with no bill due do not count either way."),
    "SOW_PAYER_TYPE": ("6", "Payer type (by money) from SOW_PAYSHARE_MED: Transactor (>= 95%), Partial payer (6-95%; earlier runs called it 'Revolver'), Minimum payer (above 0 but under 6%), Non-payer (0), No balance due.",
                       "Example: median share 0.05 -> 'Minimum payer'.", "The 6% and 95% cut-offs are analyst assumptions. Not the same as the status-code word 'revolver' (status 0)."),
    "SOW_PAYSHARE_BAND": ("6", "SOW_PAYSHARE_MED in bands: 0%, 0-6% (~min due), 6-10%, 10-25%, 25-50%, 50-95%, >=95%, No balance due.", "Example: 0.05 -> '0-6% (~min due)'.", "-"),
    "SOW_EXPOSURE_SEP": ("6", "What the bank stands to lose in September, NT$: the Sep bill, with credit balances set to 0.", "Example: Sep bill 64,400 -> 64,400; Sep bill -2,000 -> 0.", "Ignores any unused line the client could still draw."),
    "SOW_EAD": ("6", "Exposure that actually defaulted, NT$: SOW_EXPOSURE_SEP x DEFAULT.", "Example: Sep bill 41,087 and DEFAULT = 1 -> 41,087; non-defaulters -> 0.", "Built from the outcome: a result, not a predictor."),
    "SOW_UTIL_CHG": ("6", "Change in utilisation from April to September: UTIL_1 - UTIL_6 (in share of line).", "+0.20 = uses 20 more points of the line than in April. Example: 0.40 -> 1.29 gives +0.89.", "Only the two end months count."),
    "SOW_STRETCH_M": ("6", "Months (0-6) with utilisation above 90% ('stretched').", "0 = never near the limit; 6 = maxed out all window.", "-"),
    "SOW_D_UTIL": ("6", "Utilisation in Aug-Sep minus utilisation in Apr-May (average of each pair).", "Positive = card filling up. Example: 0.30 -> 0.50 gives +0.20.", "-"),
    "SOW_D_PAYSHARE": ("6", "Repayment share in Aug-Sep minus May-Jun (average of each pair).", "Negative = paying back a smaller share than before. Example: 0.50 -> 0.10 gives -0.40.", "NaN when no bills were due in either window."),
    "SOW_DYN_QUADRANT": ("6", "Combines the two changes: 'Stress: capture up + pay down' (utilisation up > 5 pts and pay share down > 5 pts), 'Capture up, paying', 'Paying less, capture flat', 'Stable / improving', 'Insufficient history'.",
                         "Example: +0.20 utilisation and -0.40 pay share -> 'Stress'.", "Round 1 found the stress quadrant defaults LESS than the base rate; do not assume it is bad."),
    "SOW_STRESS_FLAG": ("6", "1 if the client is in the 'Stress' quadrant above, else 0.", "Example: see SOW_DYN_QUADRANT.", "Same caveat: not a reliable risk sign here."),
    "SOW_UTIL_CHG_BAND": ("6", "SOW_UTIL_CHG in bands: Falling >20pp, Falling 5-20pp, Stable (+/-5pp), Rising 5-20pp, Rising >20pp (pp = percentage points of the line).", "Example: +0.89 -> 'Rising >20pp'.", "-"),
    "SOW_STRETCH_BAND": ("6", "SOW_STRETCH_M as 0 months, 1-2 months, 3-5 months, All 6 months.", "Example: 3 -> '3-5 months'.", "-"),
    "SOW_AVG_POS_BILL": ("6", "Average monthly bill in NT$, credit balances set to 0.", "Example: bills 20k, 20k, 20k, 58k, 57k, 64k -> about 40k.", "A stock (debt), not spending."),
    "SOW_PEER_CELL_N": ("6", "Number of clients in the same peer cell (age band x education x sex x marital status).", "Example: 'Female, University, Single, 26-30' may hold ~1,500 clients.", "Cells under 30 clients fall back to a coarser age x education cell."),
    "SOW_PEER_LIMIT_MED": ("6", "Median credit line (NT$) of the client's peer cell.", "Example: 100,000 = similar people typically get 100k.", "-"),
    "SOW_PEER_LIMIT_IDX": ("6", "Own credit line / peer median line (a multiple).", "1 = typical for peers; 2 = twice the peer line; 0.5 = half. Example: 250k vs peer 125k -> 2.0.", "Peers are defined by demographics only."),
    "SOW_PEER_BILL_IDX": ("6", "Own average positive bill / peer median bill (peer median floored at NT$1,000).", ">1 = carries more debt than similar people. Example: 40k vs peer 20k -> 2.0.", "Near 0 for dormant cards."),
    "SOW_PEER_LIMIT_BAND": ("6", "SOW_PEER_LIMIT_IDX in bands: <0.5x, 0.5-0.8x, 0.8-1.25x, 1.25-2x, >2x.", "Example: 2.0 -> '1.25-2x' (right edge included).", "-"),
    "SOW_PEER_BILL_BAND": ("6", "SOW_PEER_BILL_IDX in bands: <0.1x (dormant), 0.1-0.5x, 0.5-0.8x, 0.8-1.25x, 1.25-2x, >2x.", "Example: 0.05 -> '<0.1x (dormant)'.", "-"),
    "SOW_LIMIT_RESID": ("6", "How generous the line is versus the peer prediction, in log points: log(own line) - log(predicted peer line).", "+0.69 = about double the expected line; -0.69 = about half. 0 = as expected.", "Similar idea to DM_LIMIT_RESID (Section 15) but a different model."),
    "SOW_WALLET_EST": ("6", "Estimated total card 'wallet' in NT$: the 90th percentile of average balances among the client's peers.", "Example: 120,000 = people like this client can carry about 120k of card debt.", "An assumption; we never observe income or other cards."),
    "SOW_WALLET_SHARE": ("6", "Own average positive bill / SOW_WALLET_EST: the share of that estimated wallet this card holds.", "1 = this card holds the whole estimated wallet; 0.1 = a tenth. Example: 40k / 120k -> 0.33.", "Inherits the wallet assumption."),
    "SOW_LIMIT_RESID_BAND": ("6", "SOW_LIMIT_RESID in bands: Under-limited (<-0.5), Mildly under, In line (+/-0.2), Mildly over, Over-limited (>0.5).", "Example: +0.69 -> 'Over-limited (>0.5)'.", "-"),
    "SOW_WALLET_SHARE_BAND": ("6", "SOW_WALLET_SHARE in bands: ~0 (<1%), 1-10%, 10-25%, 25-50%, 50-100%, >100%.", "Example: 0.33 -> '25-50%'.", "-"),
    "SOW_WALLET_BANK": ("6", "The bank's own view of the client's wallet: the credit line, NT$ (a copy of LIMIT_BAL).", "Example: 250,000.", "Only what the bank granted, not what the client could borrow elsewhere."),
    "SOW_WALLET_PAY_MAX": ("6", "Largest single monthly payment in the window, NT$: what the client has shown they can pay at once.", "High = can pay large sums. Example: payments 1k-37k -> 37,292.", "One bonus month can dominate."),
    "SOW_WALLET_PAY_P90": ("6", "90th percentile of the six monthly payments, NT$ (a high but not extreme month).", "Example: 3,000 every month -> 3,000.", "With six values this sits between the top two."),
    "SOW_WALLET_PAYCAP": ("6", "Yearly payment capacity, NT$: 12 x SOW_WALLET_PAY_P90.", "Example: 3,000 a month -> 36,000 a year.", "Assumes the high month could repeat all year."),
    "SOW_WALLET_BORROW": ("6", "Highest statement balance in the window, NT$ (credit balances as 0): the debt the client has shown they can carry.", "Example: bills 55k-71k -> 70,887.", "A peak, not an average."),
    "SOW_WALLET_PEER_CELL": ("6", "90th percentile of peak balance among similar clients (age band x education x sex x marital status; cells under 50 fall back to age band), NT$.", "Example: 167,000 = the top 10% of people like this client carry 167k.", "Demographics only; income is unknown."),
    "SOW_WALLET_PEER": ("6", "Same 'realistic peer wallet', estimated with a quantile regression (90th percentile) on the demographics, NT$.", "Example: 85,402 for a young woman with a 20k line.", "Model estimate; falls back to SOW_WALLET_PEER_CELL if the regression fails."),
    "SOW_WALLET_PEER_P80": ("6", "The 80th-percentile version of SOW_WALLET_PEER (a more conservative wallet), NT$.", "Lower than SOW_WALLET_PEER by construction.", "Sensitivity check only."),
    "SOW_WALLET_COMBINED": ("6", "Final wallet estimate, NT$: max(own peak balance, peer wallet).", "Example: max(542,653, 167,388) = 542,653.", "If the client's own peak is the larger one, their capture is high by construction."),
    "SOW_WALLET_SRC": ("6", "Which part won in SOW_WALLET_COMBINED: 'Own peak balance' or 'Peer realistic wallet'.", "Example: own peak 542k > peer 167k -> 'Own peak balance'.", "-"),
    "SOW_WALLET_CAPTURE": ("6", "Share of wallet this card holds: average balance (credits as 0) / SOW_WALLET_COMBINED (floor NT$1,000).", "1 = the card holds the whole estimated wallet; 0.1 = a tenth. Example: 454,099 / 542,653 = 0.84.", "Cannot exceed 1 much, because the own peak is part of the wallet."),
    "SOW_WALLET_TO_LINE": ("6", "Estimated wallet / credit line (a multiple).", ">= 2 = the client could borrow twice the line (under-served); ~1 = matched. Example: 85,402 / 20,000 = 4.3.", "-"),
    "SOW_WALLET_SIZE_BAND": ("6", "SOW_WALLET_COMBINED in NT$ bands: Small (<100k), Medium (100-175k), Large (175-300k), Very large (>=300k).", "Example: 542,653 -> 'Very large'.", "Left edge included."),
    "SOW_WALLET_SOW_BAND": ("6", "SOW_WALLET_CAPTURE in bands: <1% (dormant), 1-10%, 10-30%, 30-60%, >=60%.", "Example: 0.84 -> '>=60%'.", "Left edge included."),
    "SOW_WALLET_SEGMENT": ("6", "Over-extended (own peak above the peer wallet and capture >= 60%), Under-served (wallet >= 2x the line), else Matched.", "Example: ID 1 (wallet 4.3x her line) -> 'Under-served'.", "Built from estimates; a label for targeting ideas, not a risk score."),
    "SOW_QPD_1": ("6", "Quarterly paydown, the headline 'how much of the debt is paid down' measure: payments made Jul+Aug+Sep / statements of Jun+Jul+Aug, i.e. (PAY_AMT1+PAY_AMT2+PAY_AMT3) / (BILL_AMT2+BILL_AMT3+BILL_AMT4). Negative bills count as 0; missing if the bills sum to 0 or less; capped at 5.",
                  "1.0 = paid off everything that was billed (full payer); 0.05 = paid 5% (minimum-payer, debt barely moves); 0 = paid nothing. Example: payments 3,000 x 3 = 9,000 on bills 59,696 + 63,561 + 67,060 = 190,317 -> 0.047.",
                  "Above 1 is possible (paying older debt or prepaying); the cap at 5 stops tiny balances from giving huge ratios."),
    "SOW_QPD_2": ("6", "Same paydown one month earlier: payments Jun+Jul+Aug / statements May+Jun+Jul.", "Read like SOW_QPD_1. Example: a steady revolver has about the same value in QPD_1, _2 and _3.", "Overlaps two months with SOW_QPD_1 (rolling window)."),
    "SOW_QPD_3": ("6", "Same paydown two months earlier: payments May+Jun+Jul / statements Apr+May+Jun.", "Read like SOW_QPD_1.", "Overlaps with SOW_QPD_1 and _2."),
    "SOW_QPD_PRIOR": ("6", "Earlier, non-overlapping paydown over two months: payments May+Jun / statements Apr+May.", "Example: payments 1,000 + 1,000 on bills 20,024 + 19,619 -> 0.050.", "Only two months are possible because the March bill is not in the data."),
    "SOW_QPD_CHG": ("6", "Change in paydown: SOW_QPD_1 - SOW_QPD_PRIOR (in share of the bill).", "Positive = paying down faster lately; negative = slowing down. Example: 0.047 - 0.050 = -0.003 (stable).", "Missing if either part is missing."),
    "SOW_QPD_BAND": ("6", "SOW_QPD_1 in bands: 0 paid, 0-5%, 5-10%, 10-25%, 25-50%, 50-90%, 90-110% (paid off), >110%, No balance.", "Example: 0.047 -> '0-5%'.", "'No balance' = nothing was owed in Jun-Aug."),
    "SOW_QPD_CHG_BAND": ("6", "SOW_QPD_CHG in bands: Fell > 25 pp, Fell 5-25 pp, Stable (+/-5 pp), Rose 5-25 pp, Rose > 25 pp, n/a.", "Example: -0.003 -> 'Stable'.", "-"),
    # ======================= Section 7: engineered features =======================
    "FE_LIMIT_PER_AGE": ("7", "Credit line divided by age, NT$ per year of age.", "High = large line for a young person. Example: 200,000 / 40 -> 5,000.", "A ratio of two unrelated scales; read with care."),
    "FE_UTIL_ACCEL": ("7", "Is card use speeding up? Recent speed (Sep - Jul utilisation)/2 minus earlier speed (Jun - Apr utilisation)/2, in points of the line per month.",
                      "Positive = filling up faster lately. Example: Apr 0.40, Jun 0.39, Jul 1.15, Sep 1.29 -> (1.29-1.15)/2 - (0.39-0.40)/2 = 0.075.", "Uses only four of the six months."),
    "FE_PAY_CV": ("7", "Swing of payments: std / mean of the six payments (NaN if never paid).", "0 = same amount every month. Example: 3,000 every month -> 0.", "Same as PAY_CV (Section 5)."),
    "FE_MAX_PAY_SHARE": ("7", "Largest single payment as a share of all payments (1/6 to 1).", "1 = everything paid in one lump; ~0.17 = evenly spread. Example: 3,000 x 6 -> 0.17.", "NaN if never paid."),
    "FE_MONTHS_SINCE_FULLPAY": ("7", "Months since the last payment that covered the previous bill in full (0 = Sep payment covered the Aug bill ... 4 = May); 6 = never in the window.",
                                "Low = recently paid in full. Example: last full payment in Jul -> 2.", "Values run 0-4; 6 is a code for 'never'."),
    "FE_REVOLVE_BAL": ("7", "Average debt carried over each month, NT$: mean of max(previous bill - payment, 0) over May-Sep.", "Example: bill 67,060, payment 3,000 -> 64,060 carried that month.", "Ignores new purchases."),
    "FE_REVOLVE_UTIL": ("7", "FE_REVOLVE_BAL as a share of the credit line.", "0.25 = a quarter of the line is rolled-over debt.", "-"),
    "FE_MIN_PAYER_MONTHS": ("7", "Months (0-5) paying more than 0 but under 10% of the previous bill.", "5 = minimum payer every month. Example: 3,000 on 67,060 -> counts.", "Duplicate of N_MINPAY."),
    "FE_OVERPAY_MONTHS": ("7", "Months (0-5) paying more than the previous positive bill.", "Example: 37,292 paid on a 36,923 bill -> counts.", "Small round-ups count like big prepayments."),
    "FE_HEADROOM": ("7", "Room left on the line in Sep, NT$: LIMIT_BAL - BILL_AMT1.", "Negative = over limit. Example: 50,000 - 64,400 = -14,400.", "Same as SOW_HEADROOM_SEP."),
    "FE_HEADROOM_MIN": ("7", "Smallest room left in any month, NT$.", "Negative = over the limit at least once.", "-"),
    "FE_BILL_CV": ("7", "Swing of bills: std / mean(|bill|).", "0 = stable bill. Example: bills 20k, 20k, 20k, 58k, 57k, 64k -> about 0.5.", "Close to BILL_CV."),
    "FE_PAY_MOMENTUM": ("7", "Are payments shrinking? log(1 + mean payment Jul-Sep) - log(1 + mean payment Apr-Jun).", "Negative = paying less lately; +0.69 = recent payments about double. Example: 1,000 -> 2,000 gives about +0.69.", "Log scale: small amounts swing a lot."),
    "FE_DELAY_SEVERITY": ("7", "Sum of the delay codes over six months (negative codes count as 0): months-of-delay burden.", "0 = never late; 12 = e.g. 2 months late in six months. Example: 2, 2, 2, 0, 0, 2 -> 8.", "Treats codes as month counts; code 1 only exists in Sep."),
    "FE_LATE_RECENT3": ("7", "Late months (0-3) in Jul, Aug, Sep.", "Example: late in Sep only -> 1.", "-"),
    "FE_UTIL_VOL": ("7", "Standard deviation of the six monthly utilisations.", "0 = stable use; 0.4 = big jumps. Example: 0.40 to 1.29 jump -> about 0.4.", "-"),
    # ======================= Section 8: segmentation =======================
    "RISK_TIER": ("8", "Rule-based risk tier, first rule wins: T5 Severe = 2+ months late in Sep; T4 High = 1 month late in Sep or late in 3+ months; T3 Elevated = late 1-2 months earlier or average use >= 90%; T2 Watch = never late, use 50-90%; T1 Low = everything else.",
                  "T1 safest ... T5 riskiest. Example: never late, average use 25% -> 'T1 Low'.", "Hand-made rules."),
    "CLUSTER": ("8", "Behaviour cluster id from k-means on standardised behaviour features (outcome not used to form clusters).", "C0 = cluster with the lowest default rate ... highest number = riskiest.", "The numbering uses the default rate after clustering."),
    "PERSONA": ("8", "Readable name of CLUSTER, built from the cluster's distinctive features.", "Example: 'C2 Revolvers (high utilisation)'.", "Names are automatic summaries; check the profile table in Section 8."),
    "GMM_COMP": ("8", "Component id from a Gaussian mixture model on the same features, ordered by default rate.", "0 = safest component.", "Ordering uses the outcome; compare with CLUSTER only."),
    "RFM_R": ("8", "Recency of lateness (0-3): 0 never late, 1 last late in Apr-Jun, 2 in Jul-Aug, 3 late in Sep.", "Higher = more recent trouble. Example: late only in May -> 1.", "R = 0 exactly when F = 0."),
    "RFM_F": ("8", "Frequency of late months (0-3): 0 none, 1 one month, 2 two-three months, 3 four-six months.", "Example: N_LATE = 4 -> 3.", "-"),
    "RFM_M": ("8", "Money: tercile of the average bill (1 low, 2 mid, 3 high balance).", "Example: average bill in the top third -> 3.", "Ties broken by row order."),
    "RFM_SCORE": ("8", "The three digits R, F, M written together.", "Example: '332' = late in Sep, late often, high balance.", "A label, not a number."),
    "RFM_TOTAL": ("8", "R + F + M (1-9).", "Higher = riskier-looking. Example: '332' -> 8.", "Adds unlike scales."),
    "_DEMO_CELL": ("8", "Helper: age band | education | sex | marital status as one label.", "Example: '26-30|University|Female|Single'.", "Helper column."),
    # ---------------- 11 panel structure (PN_) ----------------
    "PN_N_PAY_MONTHS": ("11", "Number of months (0-6, Apr-Sep) in which the client paid anything at all.", "6 = paid something every month; 0 = never paid. Example: paid in Apr, Jun, Aug -> 3.",
                        "Paying NT$1 counts like paying in full."),
    "PN_PAY_RUNS": ("11", "How many unbroken stretches the paid / not-paid sequence has (1-6).", "1 = always paid or never paid; 6 = strict on/off. Example: paid, paid, no, no, paid, paid -> 3.",
                    "Says nothing about amounts."),
    "PN_RHYTHM": ("11", "Payment rhythm type, rules in order: Every month (6 paying months), Misses one month (5), Never pays (0), One lump (1), Alternating (on/off every month), One block (paying months together), else Sporadic.",
                  "Example: paid Apr-Jun only -> 'One block'; paid Apr, Jun, Aug -> 'Alternating'.", "A never-used card is also 'Never pays'."),
    "PN_BILL_AC1": ("11", "How smoothly the bill moves from month to month: lag-1 autocorrelation of the six bills (-1 to +1). NaN if the bill never changes.",
                    "Near +1 = smooth drift; negative = see-saw. Example: bills 10k, 20k, 30k, 40k, 50k, 60k -> about +0.5.", "Only six points: noisy and biased negative (about -0.2 even for random bills)."),
    "PN_PAY_AC1": ("11", "Same smoothness measure for the six payments.", "Example: 1k, 5k, 1k, 5k, 1k, 5k -> strongly negative (alternating).", "Same small-sample bias; NaN if payments never change."),
    "PN_DOM_PERIOD": ("11", "The strongest repeating cycle in the payment series (6-point Fourier transform): 6 months (slow swing), 3 months, 2 months (alternating), or Flat.",
                      "Example: 1k, 5k, 1k, 5k, 1k, 5k -> '2 months (alternating)'.", "Only three cycles exist with six points; random payments give 40/40/20% shares, so treat it as weak evidence."),
    "PN_ACTIVITY_PATTERN": ("11", "Pattern of months with a positive bill: Always active (6), Never active (0), Activated during window (switches on once), Went dormant (switches off once), else Intermittent.",
                            "Example: bills 0, 0, 0, 5k, 8k, 9k -> 'Activated during window'.", "'Active' means 'owes a balance', not 'used the card'."),
    "PN_FIRST_ACTIVE_MONTH": ("11", "First month (Apr..Sep) with a positive bill, or 'Never active'.", "Example: first bill in Jul -> 'Jul'. 'Apr' = had a balance from the start.", "April is the first month we see; the account may be much older."),
    # ---------------- 12 column forensics (CF_) ----------------
    "CF_SEP1_ORIGIN": ("12", "For clients with Sep status 1 (1 month late): their Aug status - 'from Aug -2/-1' (was clean), 'from Aug 0' (was revolving), 'from Aug late' (partial cure, e.g. 2 -> 1). Everyone else: 'not code 1 in Sep'.",
                       "Example: Aug 2 then Sep 1 -> 'from Aug late (cure 2->1 etc.)'.", "Only meaningful for the ~3,700 Sep code-1 clients."),
    "CF_N_EXACT_FULL": ("12", "Months (May-Sep, 0-5) in which the payment equals the previous bill to the dollar.",
                        "High = disciplined full payer. Example: May bill 14,307, Jun payment 14,307 (ID 31) -> counts.", "Paying 17,270 on 17,265 is 'over-payment', not exact."),
    "CF_N_MINLIKE": ("12", "Months (May-Sep) paying more than 0 but at most 10% of the previous bill.", "High = minimum payer. Example: 3,000 on 67,060 (4.5%) -> counts.", "10% is our threshold, not the bank's real minimum due."),
    "CF_N_ZERO_PAY_DUE": ("12", "Months (May-Sep) with no payment although the previous bill was positive (money was owed).", "0 = never skipped a due bill. Example: 20,000 owed, 0 paid -> counts; nothing owed -> does not count.",
                          "The clean version of N_ZERO_PAY, which also counts months with nothing owed."),
    "CF_N_OVERPAY": ("12", "Months (May-Sep) paying more than the previous bill.", "Example: 37,292 paid on 36,923 -> counts.", "Tiny round-ups count like large prepayments."),
    "CF_N_ROUND_PAY": ("12", "Months (May-Sep) with a positive payment that is a round multiple of NT$1,000 and not an exact full payment (a chosen amount).",
                       "Example: 3,000 on a 67,060 bill -> counts; 2,154 does not.", "A behaviour marker, not a risk measure by itself."),
    "CF_N_CREDIT_BAL": ("12", "Months (Apr-Sep, 0-6) with a negative bill (the bank owes the client).", "Example: one month at -2,000 -> 1.", "Same as MF_N_NEG_BILL."),
    "CF_PAY_HABIT": ("12", "The client's most common monthly habit May-Sep: Zero payment, Minimum-like (<= 10% of bill), Partial (10-90%), Near full (90-100%), Exact full, Over-payment. Ties go to the riskier habit. 'No bills due' if nothing was ever owed.",
                     "Example: pays ~5% every month -> 'Minimum-like'.", "Only months with something owed count; one such month decides the label for nearly unused cards."),
    "CF_NOISE_FLAGS": ("12", "How many of five 'hard to explain if they default' flags the client trips (0-5): N1 duplicate row with a conflicting label, N2 no activity at all, N3 owes nothing in Sep, N4 paid every due bill in full, N5 status <= -1 in all months.",
                       "0 = normal client. Example: never-used card trips N2, N3 and N5 -> 3.", "Flags mark clients whose default is surprising; they do not prove the label is wrong."),
    "CF_IMPLAUSIBLE_DEFAULT": ("12", "1 if the client defaulted AND trips at least one noise flag.", "Example: defaulter who owed nothing in Sep -> 1.",
                               "Built from the outcome. 'Implausible' is strong wording: such clients may default on charges we do not see."),
    # ---------------- 14 conditional interactions (CI_) ----------------
    "CI_SEP_CLEAN": ("14", "'Clean in Sep' (Sep status <= 0) or 'Late in Sep' (Sep status >= 1).", "Example: PAY_1 = 1 -> 'Late in Sep'.", "Puts the Sep-only code 1 together with deep delays."),
    "CI_STATUS4": ("14", "Sep status in 4 groups: Inactive (-2), Paid in full (-1), Revolving (0), Late (>= 1).", "Example: PAY_1 = 3 -> 'Late'.", "Coarser than STATUS_GRP_SEP."),
    "CI_STATUS_AUG4": ("14", "Same 4 groups for August (PAY_2).", "Example: PAY_2 = 2 -> 'Late'.", "In Aug a first delay is coded 2, not 1."),
    "CI_HIST_NLATE": ("14", "Late months in Apr-Aug only (0-5): the history before September.", "Example: late in May and Jun -> 2.", "Excludes Sep on purpose."),
    "CI_HIST_LATE3": ("14", "CI_HIST_NLATE as 0 / 1-2 / 3-5.", "Example: 2 -> '1-2'.", "-"),
    "CI_LIMIT3": ("14", "Credit line in three bands: <=100k, 100-300k, >300k NT$.", "Example: 150,000 -> '100-300k'.", "-"),
    "CI_LIMIT_HALF": ("14", "Credit line at or below vs above the median line (~140k NT$).", "Example: 140,000 -> 'Low limit (<=140k)'.", "The label text carries the median of this data."),
    "CI_AGE3": ("14", "Age 21-30 / 31-45 / 46+.", "Example: 45 -> '31-45'.", "-"),
    "CI_UTIL4": ("14", "Average utilisation in four bands: <=10%, 10-50%, 50-90%, >90% of the line.", "Example: 0.46 -> '10-50%'.", "Credit balances fall into '<=10%'."),
    "CI_PAYR5": ("14", "PAY_TO_BILL_6M in five groups: <=5%, 5-25%, 25-100%, >100%, No bill.", "Example: 0.05 -> '<=5%'.", "Inherits PAY_TO_BILL_6M's one-month misalignment (see 0B.11); SOW_QPD_BAND is the cleaner alternative."),
    "CI_ZEROPAY3": ("14", "N_ZERO_PAY as 0 / 1-2 / 3-6 months with no payment.", "Example: inactive card -> '3-6'.", "Mixes 'did not pay' with 'nothing owed'."),
    "CI_UTILTREND3": ("14", "Utilisation trend: Falling (< -2 points/month), Flat, Rising (> +2 points/month).", "Example: +0.03 a month -> 'Rising'.", "Thresholds set by hand."),
    "CI_SPEND3": ("14", "Thirds of average implied spend: Low / Mid / High spend.", "Example: top third of AVG_SPEND -> 'High spend'.", "Thousands of clients have exactly 0 spend; their third is decided by row order, not behaviour."),
    "CI_OVERLIMIT": ("14", "'Never over limit' vs 'Over limit >=1m' (bill above the line in at least one month).", "Example: one month at 129% of the line -> 'Over limit >=1m'.", "-"),
    "CI_LOG_PAY_AMT1": ("14", "Sep payment on a log scale: log(1 + PAY_AMT1).", "Each +0.69 = payment doubles. Example: 2,999 -> 8.0; 0 -> 0.", "Natural log."),
    # ---------------- 15 demographics (DM_) ----------------
    "DM_LOG_LIMIT": ("15", "Natural log of the credit line.", "+0.69 = double the line. Example: 200,000 -> 12.2.", "-"),
    "DM_LOG10_LIMIT": ("15", "Base-10 log of the credit line (for log-scale charts).", "Each +1 = ten times the line. Example: 100,000 -> 5.0.", "-"),
    "DM_LIFE_STAGE": ("15", "Life stage from age and marital status, first rule wins: Senior (60+); Other marital status (<60); Young single/married (<30); Mid-life single/married (30-44); Mature single/married (45-59).",
                      "Example: age 33, married -> 'Mid-life married (30-44)'.", "Age cut-offs set by hand."),
    "DM_LIMIT_PEER": ("15", "Credit line (NT$) the bank typically gives to people of the same sex, age band, education and marital status (regression on log line).", "Example: 180,000 = what similar people typically get.", "Fitted on this data."),
    "DM_LIMIT_RESID": ("15", "Own line versus that peer line, in log points: log(own) - log(peer).", "+0.69 = double the peer line; -0.69 = half; 0 = typical. Example: 250k vs 125k -> +0.69.",
                       "Mixes the bank's private information (income, tenure) with noise."),
    "DM_LIMIT_RESID_BAND": ("15", "DM_LIMIT_RESID in five equal-size groups: Q1 far below peers ... Q5 far above peers.", "Example: top 20% -> 'Q5 far above peers'.", "-"),
    "DM_RESID_VENTILE": ("15", "DM_LIMIT_RESID in 20 equal groups, numbered 0-19.", "19 = top 5% most generous lines relative to peers.", "A group number only."),
    "DM_BEH_CELL": ("15", "Behaviour cell: Sep status group x credit-line band (about 42 cells).", "Example: 'Revolving | 100-150k'.", "-"),
    "DM_BEH_EXPECTED": ("15", "Default rate of all clients in the same behaviour cell (what behaviour alone predicts).", "0.13 = clients behaving like this default 13% of the time.",
                        "Built from the outcome on the same data; use only to compare groups fairly, never as a predictor."),
    "DM_BEH_CELL_RICH": ("15", "DM_BEH_CELL further split by number of late months.", "Example: 'Late 2m | 50-100k | late 4-6'.", "Some cells are small."),
    "DM_BEH_EXPECTED_RICH": ("15", "Default rate of the client's richer behaviour cell.", "Read like DM_BEH_EXPECTED.", "Built from the outcome; small cells give noisy rates."),
    "DM_EDU_RAW_LABEL": ("15", "Readable label for the raw EDUCATION code, keeping 0, 4, 5, 6 separate.", "Example: 5 -> '5 (undocumented)'.", "-"),
    "DM_AGE6": ("15", "Six age bands: 21-25, 26-30, 31-35, 36-40, 41-50, 51+.", "Example: 45 -> '41-50'.", "Different from AGE_BAND (wider top bands)."),
    "DM_HH_GROUP": ("15", "Household group: Married men, Married women, Single men, Single women; everyone else 'Other marital status'.", "Example: single female -> 'Single women'.", "-"),
    "DM_FLAG_SEP2": ("15", "1 if 2 or more months late in September, else 0 (a simple behaviour-only alarm).", "Example: PAY_1 = 2 -> 1; PAY_1 = 1 -> 0.", "Misses the Sep code-1 group, which defaults at about 34%."),
    "DM_BEH_TIER": ("15", "Five behaviour tiers, first rule wins: T5 2+ months late in Sep; T4 1 month late in Sep; T3 late earlier but current in Sep; T2 revolving in Sep, never late; T1 paid in full or inactive, never late.",
                    "T1 safest ... T5 riskiest. Example: never late, paid in full in Sep -> 'T1'.", "-"),
    "DM_CELL4": ("15", "Four-way demographic cell: sex | education | marital status | age band.", "Example: 'Female | University | Single | 26-30'.", "Many cells are small."),
    "DM_CELL_RATE_EB": ("15", "Default rate of the client's demographic cell, shrunk towards the overall rate when the cell is small (empirical Bayes).",
                        "Example: a cell of 12 with 6 defaults (50%) is pulled down towards ~22%.", "Built from the outcome."),
    # ---------------- 16 money flow (MF_) ----------------
    "MF_UTIL_CURV": ("16", "Is the utilisation path bending up or down? Curvature c of a quadratic fit (utilisation clipped to -0.5..2), in points of the line per month squared.",
                     "Positive = convex (accelerating up, or bottoming out); negative = concave (levelling off or peaking); ~0 = straight line.", "Six points; one odd month dominates."),
    "MF_UTIL_MAXDD": ("16", "Largest fall in utilisation from an earlier peak (max drawdown), share of line.", "Example: 0.60 then later 0.20 -> 0.40.", "-"),
    "MF_UTIL_RUNUP": ("16", "Largest rise in utilisation from an earlier low (max run-up).", "Example: 0.10 then later 0.70 -> 0.60.", "-"),
    "MF_UTIL_RANGE": ("16", "Highest minus lowest utilisation over six months.", "0 = flat; 1 = swung across the whole line. Example: 0.20..0.90 -> 0.70.", "-"),
    "MF_UTIL_TURNS": ("16", "Number of times utilisation changed direction, counting only moves above 2 points of the line.", "0 = one-way path; 4 = zig-zag. Example: up, down, up -> 2.", "Small wiggles ignored on purpose."),
    "MF_UTIL_PEAK_MONTH": ("16", "Month with the highest utilisation, or 'Flat' if the range is under 3 points.", "'Sep' = card fullest in the latest month.", "Ties go to the earliest month."),
    "MF_UTIL_PATH_CLASS": ("16", "Direction / bend of the utilisation path: Falling, Flat or Rising (+/-1 point per month) combined with Concave, Linear or Convex.", "Example: 'Rising / Convex' = accelerating climb.", "Thresholds set by hand."),
    "MF_UTIL_SHAPE": ("16", "Named group of clients whose utilisation paths have a similar SHAPE (k-means on paths with the level removed); flat paths labelled Flat high / Flat low / Flat zero (dormant).",
                      "Example: 'Ramp up' = steady rise whatever the level.", "Group names come from the group's average shape; may differ slightly between library versions."),
    "MF_UTIL_LEVEL_CLUSTER": ("16", "Group of clients with similar raw utilisation paths (level and shape), named 'U<rank>: avg X%, rising/stable/falling'.", "Example: 'U4: avg 85%, stable'.", "Same caveat as above."),
    "MF_PAYR_SHAPE": ("16", "Shape group of the monthly repayment-share path May-Sep; flat paths labelled full / partial / minimum payer; 'No balance due in >=1 month' if any month had nothing owed.",
                      "Example: 'Falling repayment' = paying back a shrinking share.", "Clients with any month without a bill are not clustered."),
    "MF_PAYR_LEVEL_CLUSTER": ("16", "Level + shape group of the repayment-share path, named 'P<rank>: avg X%, trend'.", "Example: 'P1: avg 4%, stable' = steady minimum payer.", "-"),
    "MF_SPEND_SHAPE": ("16", "Shape group of monthly spend as a share of the line, May-Sep; flat paths 'Flat: no spend' / 'Flat: steady spend'.", "Example: 'Fading spend' = card used less and less.", "Spend includes interest (see 0B.6)."),
    "MF_SPEND_LEVEL_CLUSTER": ("16", "Level + shape group of the spend path, named 'S<rank>: ...'.", "Example: 'S1: avg 0%, stable' = no spending.", "-"),
    "MF_REGIME_SEQ": ("16", "Five letters, one per month May-Sep, for how the balance moved: B build (up > 5%), D pay-down (down > 5%), F flat, Z no balance. Change measured against max(previous bill, NT$1,000).",
                      "Example: 'BBBBB' = five straight builds; 'DDDDD' = steady deleveraging.", "-"),
    "MF_N_BUILD": ("16", "Number of build months (0-5).", "5 = balance grew every month.", "-"),
    "MF_N_PAYDOWN": ("16", "Number of pay-down months (0-5).", "5 = balance shrank every month.", "-"),
    "MF_N_FLAT": ("16", "Number of flat months (0-5).", "Example: 'FFFBD' -> 3.", "'No balance' months are not counted anywhere."),
    "MF_REGIME_LAST": ("16", "Direction of the balance from Aug to Sep: Build / Pay-down / Flat / No balance.", "Example: Sep bill 64,400 vs Aug 57,069 -> 'Build'.", "-"),
    "MF_BUILD_RUN_MAX": ("16", "Longest streak of consecutive build months.", "Example: 'BBDBB' -> 2.", "-"),
    "MF_PAYDOWN_RUN_MAX": ("16", "Longest streak of consecutive pay-down months.", "Example: 'DDDFB' -> 3.", "-"),
    "MF_FLOW_ARCHETYPE": ("16", "Money-flow type, first rule wins: No balance; Debt spiral, token payments (3+ build months in a row, paying < 10%); Debt spiral, pays but charges more; Binge then repay (one jump > 20% of line, half gone by Sep); Deleverager (3+ pay-downs in a row, Sep bill < 70% of Apr); Treading water (4+ flat months, bill > 5k, paying <= 10%); Full payer; Other / mixed.",
                          "Example: balance rising five months on ~3% payments -> 'Debt spiral, token payments'.", "Rule order decides overlaps; thresholds set by hand."),
    "MF_PAY_LAST_MOVE": ("16", "How the payment changed from Aug to Sep: Zero->zero, Stop, Drop > 50%, Drop 10-50%, Stable +/-10%, Rise 10-100%, Rise > 100%, Restart.", "Example: 3,000 -> 1,000 = 'Drop > 50%'.", "-"),
    "MF_N_PAY_SHOCK": ("16", "Months (May-Sep) in which the payment fell to under half of last month's payment (stopping included).", "0 = steady payer. Example: 3,000 then 1,000 -> 1 shock.", "A normal payment after one big lump also counts."),
    "MF_PAY_SHOCK_SEP": ("16", "1 if the Aug->Sep payment was a shock (fell by more than half).", "Example: 2,000 in Aug, 0 in Sep -> 1.", "-"),
    "MF_BIG_PAY_ANY": ("16", "1 if any payment was more than 5x the client's usual (median) payment and at least NT$5,000: a one-off lump.", "Example: usual 2,000, one payment of 40,000 -> 1.", "Clients whose usual payment is 0 can never qualify."),
    "MF_BIG_PAY_MONTH": ("16", "Month of the most recent lump payment, 'None', or 'n/a (median payment 0)'.", "Example: 'Jun'.", "-"),
    "MF_PAY_SEP_VS_PRIOR": ("16", "September payment / average Apr-Aug payment (capped at 20). NaN if nothing paid before.", "1 = usual; 0.2 = paid a fifth of the usual amount; 0 = stopped. Example: 500 vs usual 2,500 -> 0.2.", "-"),
    "MF_PAY_COLLAPSE_BAND": ("16", "MF_PAY_SEP_VS_PRIOR in bands: No prior payments, Sep = 0, <25%, 25-75%, 75-125%, 125-300%, >300%.", "Example: 0.2 -> '<25%'.", "-"),
    "MF_N_FROZEN": ("16", "Months (May-Sep) where the client owed money but added almost nothing new (new charges <= 2% of the previous bill): balance sitting, card not used.",
                    "0 = card in use; 5 = frozen all window. Example: bill stuck at 40k plus interest -> frozen.", "2% is about one month's interest; a high-interest account could look 'active'."),
    "MF_FROZEN_RECENT": ("16", "Length (0-5) of the frozen streak ending in September.", "Example: frozen Jul, Aug, Sep -> 3.", "-"),
    "MF_SPEND_LATE_RATIO": ("16", "Recent spend vs earlier spend: average spend Aug-Sep / average spend May-Jul (negative spend set to 0; capped at 20). NaN if no earlier spend.",
                            "1 = same; 0.3 = spending fell to 30%; 3 = tripled. Example: 3,000 vs 10,000 -> 0.3.", "-"),
    "MF_SPEND_SURGE_SEP": ("16", "1 if Sep spend is more than 3x the May-Aug average, at least NT$5,000 and at least 5% of the line ('last hurrah').", "Example: usual 2,000, Sep 20,000 on a 100k line -> 1.", "-"),
    "MF_SPEND_PATTERN": ("16", "Spend pattern, first rule wins: No spend; Stopped using (frozen 2+ months to Sep); Sep surge; Fading (recent < half of earlier); else Regular.", "Example: spend falling from 10k to 3k -> 'Fading'.", "-"),
    "MF_COVERAGE": ("16", "Do payments cover new charges? Total payments May-Sep / total new charges May-Sep (capped at 10). NaN if no net new charges.",
                    "1 = balance unchanged; < 1 = debt grew; > 1 = debt shrank. Example: paid 10,000 against 20,000 of charges -> 0.5.", "By accounting, coverage < 1 is the same thing as 'balance grew'."),
    "MF_COVERAGE_BAND": ("16", "MF_COVERAGE in bands: No net charges, <50%, 50-80%, 80-95%, 95-105% (balanced), 105-150%, >150%.", "Example: 0.5 -> '50-80%'.", "-"),
    "MF_OL_MAX_DEPTH": ("16", "How far over the limit at worst: max(bill / line - 1); 0 if never over.", "0.29 = at worst 29% over the line. Example: 64,400 on 50,000 -> 0.29.", "-"),
    "MF_OL_RUN_MAX": ("16", "Longest streak of consecutive over-limit months.", "Example: over in Jul, Aug, Sep -> 3.", "-"),
    "MF_OL_DEPTH_BAND": ("16", "MF_OL_MAX_DEPTH in bands: Never over, 0-2%, 2-5%, 5-10%, 10-20%, >20%.", "Example: 0.29 -> '>20%'.", "-"),
    "MF_OL_PATTERN": ("16", "Never over / Past only (back under the line by Sep) / Over in Sep (went over during the window) / Over limit all 6 months.", "Example: over Jul-Sep -> 'Over in Sep (entered in window)'.", "-"),
    "MF_N_NEG_BILL": ("16", "Months (0-6) with a negative bill (in credit).", "Example: one month at -2,000 -> 1.", "Same as CF_N_CREDIT_BAL."),
    "MF_NEG_MAX_DEPTH": ("16", "Largest credit balance as a share of the line.", "0.05 = the bank owed the client 5% of the line at most. Example: -5,000 on 100k -> 0.05.", "-"),
    "MF_NEG_PATTERN": ("16", "Never in credit / Past only / In credit in Sep / In credit all 6 months.", "Example: credit in May only -> 'Past only'.", "-"),
    # ---------------- 17 early warning (EW_) ----------------
    "EW_LENS_OOF": ("17", "Predicted default probability (0-1) from a gradient-boosting model using all six months, each client scored by a model that did not see them (5-fold).",
                    "0.6 = high-risk profile; 0.05 = low. Example: a client 2 months late in Sep typically scores above 0.5.", "A yardstick for how much signal exists, not a deliverable model."),
    "EW_UTIL_APR_BAND": ("17", "April utilisation in bands: <=0, 0-30%, 30-70%, 70-100%, >100%.", "Example: 0.40 -> '30-70%'.", "-"),
    "EW_ENTRY_SEVERITY_APR": ("17", "How late the client was in April: not late / 1 month / 2 months / 3+ months.", "Example: PAY_6 = 2 -> '2 months'.", "'1 month' is always empty in April (code 1 not used)."),
    "EW_ENTRY_SEVERITY_MAY": ("17", "Same for May.", "Example: PAY_5 = 3 -> '3+ months'.", "'1 month' is empty in May too."),
    "EW_N_SIGNS": ("17", "Number of the 26 warning signs (see EW_SIGN_*) that fired at least once Apr-Sep.", "0 = no alarm; 10+ = many alarms. Example: late twice and maxed out -> several signs.", "Signs overlap heavily, so the count double-counts."),
    "EW_N_SIGNS_CORE": ("17", "Number of the 22 core signs (delinquency, payment, utilisation, compound families) that fired.", "0 = no core alarm.", "-"),
    "EW_N_SIGNS_RECENT": ("17", "Core signs that fired in Aug or Sep.", "Example: late in Sep with zero payment -> several recent signs.", "-"),
    "EW_N_SIGNS_CAPPED": ("17", "EW_N_SIGNS_CORE capped at 12 (for charts).", "12 = 12 or more.", "-"),
    "EW_N_FAMILIES": ("17", "How many of three alarm families fired: Delinquency, Payment, Utilisation (0-3).", "Example: late and maxed out -> 2.", "-"),
    "EW_FAMILY_COMBO": ("17", "Which of those families fired, e.g. 'Delinq+Payment', or 'None'.", "Example: only utilisation alarms -> 'Util'.", "-"),
    "EW_FAMILY_PATH": ("17", "Order in which alarm families first appeared ('>' = earlier, '=' = same month): Delinq, Pay, Util, Shift.", "Example: 'Util > Delinq' = the card filled up before the first late payment.", "April is the first month we see, so 'first' means first observed."),
    "EW_FIRST_SIGN_MONTH": ("17", "Month the first warning sign fired, or 'None'.", "Example: 'Jun'.", "Many signs need the previous month and cannot fire in April."),
    "EW_LEAD_TIME": ("17", "Months between the first alarm and September (0 = first alarm in Sep, 5 = April); NaN if none.", "High = early warning. Example: first alarm in Jun -> 3.", "Capped by the start of the window."),
    "EW_GROUP": ("17", "Outcome x alarm group: Silent defaulter (defaulted, no core sign), Clean non-defaulter, Loud defaulter / False alarm (many signs, with / without default), Mid-signal defaulter / non-defaulter.",
                 "Example: no signs and defaulted -> 'Silent defaulter'.", "Built from the outcome; the 'many signs' cut-off is computed in the run."),
    "EW_SILENT_DEFAULTER": ("17", "1 if the client defaulted with zero core warning signs.", "Example: ID 46 (inactive card that defaulted) -> 1.", "Built from the outcome."),
    "EW_NEVER_LATE_DEFAULTER": ("17", "1 if the client defaulted and no delinquency sign ever fired.", "Example: a full payer who defaulted -> 1.", "Built from the outcome."),
    # ---------------- 18 exposure economics (EX_) ----------------
    "EX_EXPOSURE": ("18", "What the bank has lent out in September, NT$: max(Sep bill, 0).", "Example: Sep bill 64,400 -> 64,400; a credit balance -> 0.", "Same as SOW_EXPOSURE_SEP; ignores the unused line."),
    "EX_HEADROOM": ("18", "Unused credit line in September, NT$: max(line - exposure, 0).", "High = lots of room to borrow more. Example: line 100,000, bill 45,000 -> 55,000; over-limit -> 0.", "Floored at 0, unlike SOW_HEADROOM_SEP and FE_HEADROOM."),
    "EX_HEADROOM_PCT": ("18", "EX_HEADROOM as a share of the line.", "0.55 = 55% of the line unused; 0 = at or over the limit.", "-"),
    "EX_HEADROOM_BAND": ("18", "EX_HEADROOM in NT$ bands: 0 (at/over limit), <10k, 10-50k, 50-100k, 100-200k, >200k.", "Example: 55,000 -> '50-100k'.", "-"),
    "EX_EAD": ("18", "Exposure at default, NT$: EX_EXPOSURE x DEFAULT.", "Example: Sep bill 41,087 and defaulted -> 41,087; non-defaulters -> 0.", "Built from the outcome."),
    "EX_INT_REV": ("18", "Rough yearly interest income, NT$: 18% x average unpaid previous balance in months with status >= 0.", "Example: 60,000 carried on average -> about 10,800 a year.", "The 18% rate is an assumption; real card rates are not in the data."),
    "EX_IX_REV": ("18", "Rough yearly interchange (merchant fee) income, NT$: 1.5% x yearly positive implied spend (months not coded -2).", "Example: 6,000 spend a month -> 72,000 a year -> about 1,080.", "Assumed 1.5%; implied spend includes interest."),
    "EX_REVENUE": ("18", "EX_INT_REV + EX_IX_REV (NT$ a year).", "High = valuable client. Example: 10,800 + 1,080 = 11,880.", "Assumption-driven proxy."),
    "EX_FUND_COST": ("18", "Rough yearly funding cost, NT$: 2% x average positive balance.", "Example: 60,000 average balance -> 1,200.", "Assumed 2%."),
    "EX_LOSS": ("18", "Credit loss, NT$: DEFAULT x 80% x EX_EXPOSURE (80% = assumed loss given default).", "Example: defaulter with 41,087 exposure -> about 32,900; non-defaulters -> 0.", "Built from the outcome; the 80% is assumed."),
    "EX_PROFIT": ("18", "Revenue - funding cost - loss, NT$ (a rough yearly profit proxy).", "Positive = the client earns the bank money; negative = loses money. Example: 11,880 - 1,200 - 0 = 10,680.", "Mixes yearly revenue with a one-off loss; built from the outcome."),
    "EX_PROFIT_BAND": ("18", "EX_PROFIT in bands: < -10k, -10k to -1k, -1k to 1k, 1k to 5k, > 5k NT$.", "Example: 10,680 -> '>5k'.", "Inherits all assumptions above."),
    "EX_CUST_TYPE": ("18", "September customer type from the status code: Inactive (-2), Transactor (-1), Revolver (0), Delinquent (>= 1).", "Example: PAY_1 = 0 -> 'Revolver'.", "Status-code view; the money view is SOW_PAYER_TYPE."),
    "EX_CUST_TYPE_6M": ("18", "Most common customer type over Apr-Sep (ties go to the riskier type).", "Example: 4 revolving months and 2 late months -> 'Revolver'.", "-"),
    "EX_N_SIGNS": ("18", "Number of five September warning signs (0-5): late in Sep; zero Sep payment on a positive Aug bill; late in 2+ months; Sep use > 90% and paid < 10%; utilisation up > 20 points Apr->Sep.", "0 = no sign; 3+ = several alarms.", "Different from the 26-sign EW_N_SIGNS of Section 17."),
    # ---------------- 19B competition ideas (CX_) ----------------
    "CX_AP_UTIL_LAST": ("19B", "Debt left after the September payment as a share of the line: (Aug bill - Sep payment) / line.", "0.6 = 60% of the line still owed after paying. Example: (67,060 - 3,000) / 250,000 = 0.26.", "Negative = paid more than the bill."),
    "CX_AP_UTIL_MEAN": ("19B", "Average of that after-payment debt share over May-Sep.", "Read like CX_AP_UTIL_LAST.", "-"),
    "CX_AP_UTIL_BAND": ("19B", "CX_AP_UTIL_LAST in bands: credit, zero, 0-30%, 30-60%, 60-90%, 90-100%, 100-110%, >110%.", "Example: 0.26 -> '0-30%'.", "-"),
    "CX_N_ACTIVE_STMT": ("19B", "Months (0-6) that are not empty (empty = bill 0 and payment 0).", "6 = activity every month; 0 = completely dormant.", "-"),
    "CX_EMPTY_PATTERN": ("19B", "Six characters Apr -> Sep: 'E' = empty month, '.' = active month.", "Example: 'EEE...' = started being used in Jul.", "-"),
    "CX_EMPTY_TYPE": ("19B", "Type of that pattern: All active, Never active, Gaps (intermittent), Active only mid-period, Empty early / active late (new or re-opened), Active early / empty late (went quiet).", "Example: '...EEE' -> 'Active early, empty late (went quiet)'.", "-"),
    "CX_FIXED_390": ("19B", "1 if any bill equals exactly NT$390 or 780 (a fixed charge that appears unusually often).", "Example: a card with only a yearly fee on it -> 1.", "Interpretation (fee) is a guess."),
    "CX_ID_DECILE": ("19B", "Which tenth of the ID order the row is in (D1 ... D10): a check that row order carries no signal.", "Example: ID 15 -> 'D1'.", "Not a client trait; never use as a feature."),
    "CX_FROZEN_HIGH": ("19B", "Month pairs (0-5) where a status of 3 or more stayed exactly the same (a delay counter that stopped counting). Renamed CX_STUCK_HIGH_CODE in later runs.", "0 = normal; 2+ = code looks stuck. Example: 4, 4, 4 -> 2.", "Could be a placeholder code rather than real behaviour."),
    "CX_STUCK_HIGH_CODE": ("19B", "Month pairs (0-5) where a status of 3 or more stayed exactly the same (a delay counter that stopped counting).", "0 = normal; 2+ = code looks stuck. Example: 4, 4, 4 -> 2.", "Could be a placeholder code rather than real behaviour."),
    "CX_FROZEN_BAND": ("19B", "That count banded 0, 1, 2, 3+.", "Example: 2 -> '2'.", "-"),
    "CX_STUCK_BAND": ("19B", "That count banded 0, 1, 2, 3+.", "Example: 2 -> '2'.", "-"),
    "CX_N_30_60": ("19B", "Months (0-6) at status 1-2 (roughly 30-60 days late).", "Example: 2, 2, 0, 0, 0, 1 -> 3.", "Code 1 is Sep-only."),
    "CX_N_90P": ("19B", "Months (0-6) at status 3+ (90+ days late).", "Example: 3, 4, 5 in three months -> 3.", "-"),
    "CX_PAYOFF_MONTHS": ("19B", "Months it would take to clear the debt at the current pace: mean debt left after paying / mean monthly payment (May-Sep), capped at 120; NaN = never pays.", "Low = fast payer; 120 = would take 10+ years. Example: 64,000 left, paying 3,000 a month -> about 21 months.", "Ignores interest, so the real time is longer."),
    "CX_PAYOFF_BAND": ("19B", "CX_PAYOFF_MONTHS in bands: 0, <=1, 1-3, 3-6, 6-12, 12-24, 24-60, 60-120 months.", "Example: 21 -> '12-24'.", "-"),
    "CX_PAY_BURDEN": ("19B", "Average monthly payment as a share of the credit line.", "0.02 = pays about 2% of the line a month. Example: 3,000 on 250,000 -> 0.012.", "The line stands in for income, which we do not have."),
    # ---------------- 19 case studies (CS_) ----------------
    "CS_G01_CLEAN_PAYER": ("19", "1 = never late in any month (status <= 0 all six months).", "Expected 'safe' group. Example: ID 15 -> 1.", "-"),
    "CS_G02_LATE_SEP_2P": ("19", "1 = 2 or more months late in September.", "Expected 'risky' group. Example: ID 23 -> 1.", "-"),
    "CS_G03_PAID_FULL_AMT": ("19", "1 = every May-Sep payment covered the previous bill (by amount), with at least one bill due.", "Expected 'safe'. Example: ID 31 -> 1.", "-"),
    "CS_G04_NEVER_USED": ("19", "1 = status -2 (no consumption) in all six months.", "Expected 'safe'. Example: ID 46 -> 1 (yet defaulted).", "-2 does not always mean a zero bill (39% of -2 months have a balance)."),
    "CS_G05_MAXED_CLEAN": ("19", "1 = utilisation above 95% in all six months and never late.", "Expected 'risky'.", "-"),
    "CS_G06_BIG_YOUNG": ("19", "1 = credit line in the top 20% and age in the youngest 20%.", "Expected 'risky'.", "Cut-offs computed in the run."),
    "CS_G07_DEEP_OVERLIMIT": ("19", "1 = bill above 110% of the line in at least one month.", "Expected 'risky'. Example: ID 6 (129% of line) -> 1.", "-"),
    "CS_G08_SUDDEN_COLLAPSE": ("19", "1 = never late Apr-Aug, then late in Sep.", "Expected 'risky'.", "Mostly Sep code-1 clients."),
    "CS_G09_CURED": ("19", "1 = late at least once in Apr-Jun, never late in Jul-Sep.", "Expected 'safe'.", "-"),
    "CS_G10_PAY_COLLAPSE": ("19", "1 = paid every month May-Aug, then paid nothing in Sep although the Aug bill was positive.", "Expected 'risky'.", "-"),
    "CS_G11_CREDIT_BAL": ("19", "1 = negative bill (in credit) in at least one month.", "Expected 'safe'.", "Same as NEG_BILL_ANY."),
    "CS_G12_SAFEST_DEMO": ("19", "1 = among the 10% with the lowest demographics-only risk score (education known).", "Expected 'safe'.", "-"),
    "CS_N_GROUPS": ("19", "Number of the twelve case-study groups the client belongs to (0-12).", "Example: clean payer who paid in full -> 2.", "-"),
    "CS_N_BREAKS": ("19", "Number of groups in which the client 'breaks the rule' (defaulted in a safe group or did not default in a risky one).", "Example: ID 46 (never used, defaulted) -> at least 1.", "Built from the outcome."),
    "CS_RULE_BREAKER": ("19", "1 if the client breaks the rule in at least one group.", "Example: ID 46 -> 1.", "Built from the outcome."),
    "CS_PRIMARY_GROUP": ("19", "The smallest (most specific) group the client belongs to, or 'None'.", "Example: a client in G01 and G09 gets the smaller of the two.", "-"),
    "CS_DEMO_SCORE": ("19", "Default probability (0-1) predicted from demographics and credit-line band only (logistic regression, fitted on this data).",
                      "0.25 = demographics alone suggest a 25% rate. Example: most clients fall between 0.15 and 0.30.", "Fitted on the outcome of the same data; descriptive only."),
}
for i in IDX[:-1]:
    S97_GLOSSARY[f"SOW_SPEND_SHARE_{i}"] = ("6", f"New charges in {MONTH[i]} as a share of the credit line: max(SPEND_{i}, 0) / LIMIT_BAL.",
                                            "0.05 = charges worth 5% of the line that month. Example: 5,000 on a 100k line -> 0.05.", "Negative spend shown as 0; includes interest.")
    S97_GLOSSARY[f"SOW_PAYSHARE_{i}"] = ("6", f"Share of the {MONTH[i + 1]} bill repaid in {MONTH[i]}: PAY_RATIO_{i} clipped to 0-1.",
                                         "1 = paid in full; 0.05 = about the minimum; 0 = nothing. Example: 3,000 on 67,060 -> 0.045.", "NaN if nothing was owed; over-payment shown as 1.")


# ---------- families defined by rule ----------
S97_TS_STAT = {"mean": "mean over the months", "std": "standard deviation over the months", "cv": "std / |mean| (NaN if mean ~ 0)",
               "slope": "OLS slope per month (Apr->Sep)", "last_minus_first": "Sep value minus first value", "last_over_mean": "Sep value / own mean",
               "ac1": "lag-1 autocorrelation", "max_jump": "largest single month-on-month move (signed)", "argmax": "index of the peak month (0 = first month)",
               "n_above_mean": "number of months above the client's own mean", "r2": "share of variance explained by a straight line"}
S97_TS_SERIES = {"bill": "bill (NT$)", "pay": "payment (NT$)", "util": "utilisation (bill / limit)", "spend": "implied spend May-Sep (NT$)", "status": "status code"}
def s97_rule_definition(col):
    if col.startswith("MF_TS_"):
        rest = col[6:].lower()
        for ser in S97_TS_SERIES:
            if rest.startswith(ser + "_") and rest[len(ser) + 1:] in S97_TS_STAT:
                st = rest[len(ser) + 1:]
                return ("16", f"Time-series feature: {S97_TS_STAT[st]} of the client's {S97_TS_SERIES[ser]} series (one of the top-8 by IV in 16.11).",
                        "Compare across clients; higher/lower meaning depends on the statistic.",
                        "Which 8 features exist depends on the IV ranking of the run." + (" Status-code statistics treat codes as numbers." if ser == "status" else ""))
    if col.startswith("EX_POL_"):
        pol = {k[0]: k for k, *_ in globals().get("S18_POLICIES", [])}
        nm = pol.get(col[7:], f"policy rule {col[7:]} (see 18.4.1)")
        return ("18", f"1 if the client is flagged by the what-if credit policy '{nm}'.", "1 = this rule would act on the client (cut, freeze or close the line).", "A simulation flag, not something the bank did.")
    if col.startswith("CX_AP_") and col[6:] in ("MAY", "JUN", "JUL", "AUG", "SEP"):
        m = col[6:].title(); prev = MONTH_ORDER[MONTH_ORDER.index(m) - 1]
        return ("19B", f"Debt left after paying, NT$: the {prev} bill minus the payment made in {m}.", f"High = much of the {prev} bill still unpaid; negative = paid more than the bill. Example: 67,060 - 3,000 = 64,060.", "Ignores new purchases made in the same month.")
    if col.startswith("CX_AGG_"):
        return ("19B", f"Summary statistic of one monthly series (AmEx-style aggregate, name = series_statistic): {col[7:].lower()}.", "Compare across clients; see 19B.1 for the exact statistic.", "Which five aggregates exist depends on the ranking of the run.")
    if col.startswith("EW_SIGN_"):
        nm = col[8:]
        sg = globals().get("S17_SIGNS", {}).get(nm)
        fam = {"D": "Delinquency", "P": "Payment", "U": "Utilisation / balance", "X": "Compound", "S": "Usage shift"}
        if sg:
            return ("17", f"1 if warning sign {nm} ({fam.get(sg['family'], sg['family'])} family) fired in any month Apr-Sep: {sg['desc']}.",
                    "1 = the alarm rang at least once.", "Signs needing lagged data cannot fire in April.")
        return ("17", f"1 if warning sign {nm} fired in any month Apr-Sep (definition in 17.6).", "-", "-")
    return None

# generic one-line caveat where a column has no special one
def s97_fill(col, entry):
    sec, defin, how, cav = entry
    if cav in ("-", ""):
        cav = ("Bands hide the variation inside each band; the edges are analyst choices." if ("BAND" in col or col.endswith(("3", "4", "5", "6")) and col.startswith(("CI_", "DM_AGE")))
               else "No special trap; read it together with the general warnings at the top of this section.")
    if how in ("-", ""):
        how = "See the definition; compare the three example clients in 22.2."
    return (sec, defin, how, cav)
S97_GLOSSARY = {k: s97_fill(k, v) for k, v in S97_GLOSSARY.items()}

S97_BASE = dict(globals().get("S00B_DICT", {}))
def s97_lookup(col):
    for src, d in (("glossary (1-20)", S97_GLOSSARY), ("Section 0B", S97_BASE)):
        if col in d:
            return src, d[col]
    r = s97_rule_definition(col)
    if r:
        return "glossary rule (1-20)", s97_fill(col, r)
    return "definition missing", ("?", "definition missing", "", "")

# readable rendering of the 11-20 glossary (plus rule-defined family members present in df)
s97_rule_cols = [c for c in df.columns if c.startswith(("MF_TS_", "EW_SIGN_", "EX_POL_", "CX_AP_", "CX_AGG_")) and c not in S97_GLOSSARY]
s97_rows = [(c,) + S97_GLOSSARY[c] for c in S97_GLOSSARY] + [(c,) + s97_fill(c, s97_rule_definition(c)) for c in s97_rule_cols if s97_rule_definition(c)]
S97_GLOSS_FRAME = pd.DataFrame(s97_rows, columns=["column", "section", "definition", "how_to_read", "caveat"])
S97_GLOSS_FRAME["in_df_this_run"] = S97_GLOSS_FRAME["column"].isin(df.columns)
S97_GLOSS_FRAME["_k"] = S97_GLOSS_FRAME["section"].map(lambda v: float(str(v).replace("B", ".5")))
for _k, _g in S97_GLOSS_FRAME.sort_values("_k", kind="stable").groupby("_k", sort=True):
    _sec = str(_k).replace(".5", "B").replace(".0", "")
    _md = [f"#### Section {_sec} columns", "", "| Column | What it is (unit) | High vs low, worked example | Watch out |", "|---|---|---|---|"]
    for _, _r in _g.iterrows():
        _nm = f"`{_r['column']}`" + ("" if _r["in_df_this_run"] else " *(not in df in this run)*")
        _e = [str(_r[k]).replace("|", "\\|").replace("\n", " ") for k in ("definition", "how_to_read", "caveat")]
        _md.append(f"| {_nm} | {_e[0]} | {_e[1]} | {_e[2]} |")
    display(Markdown("\n".join(_md)))
print(f"glossary entries: {len(S97_GLOSSARY)} hand-written + {len(s97_rule_cols)} rule-defined family members in df")

### 22.2 Profile of every column now in `df`, with three fixed example clients
For each column: source of the definition, type, % missing, min / median / max (numbers) or the top three
categories (labels), and the value for three fixed clients from Section 0B: the clean revolver (ID 15), the late payer
who defaults (ID 23) and the inactive card that still defaults (ID 46). Read a row left to right to check that the
definition and the example values agree. Rows flagged **definition missing** need a glossary entry.

In [ ]:
S97_IDS = [15, 23, 46]
S97_IDS = [i for i in S97_IDS if (df["ID"] == i).any()]
s97_ex = df.set_index("ID").loc[S97_IDS]
def s97_short(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "NaN"
    if isinstance(v, (float, np.floating)):
        return f"{v:,.4g}" if abs(v) < 1e4 else f"{v:,.0f}"
    s = str(v)
    return s if len(s) <= 28 else s[:26] + ".."
s97_prof = []
for c in df.columns:
    src, (sec, defin, how, cav) = s97_lookup(c)
    s = df[c]
    row = {"column": c, "section": sec, "definition_source": src, "dtype": str(s.dtype), "pct_missing": float(s.isna().mean())}
    if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
        x = pd.to_numeric(s, errors="coerce")
        row["summary"] = f"min {s97_short(x.min())} | median {s97_short(x.median())} | max {s97_short(x.max())}"
    else:
        vc = s.astype(str).value_counts(normalize=True).head(3)
        row["summary"] = f"{s.nunique()} levels; top: " + "; ".join(f"{s97_short(k)} {v:.0%}" for k, v in vc.items())
    for i in S97_IDS:
        row[f"ID {i}"] = s97_short(s97_ex.loc[i, c]) if c != "ID" else str(i)
    row["definition"] = defin
    row["how_to_read"] = how
    row["caveat"] = cav
    s97_prof.append(row)
S97_PROFILE = pd.DataFrame(s97_prof)
S97_PROFILE["status"] = np.where(S97_PROFILE["definition_source"] == "definition missing", "definition missing", "defined")
with pd.option_context("display.max_rows", 1000, "display.max_colwidth", 60):
    display(S97_PROFILE[["column", "section", "dtype", "pct_missing", "summary"] + [f"ID {i}" for i in S97_IDS] + ["status"]].round(3))
s97_miss = S97_PROFILE.loc[S97_PROFILE["status"] == "definition missing", "column"].tolist()
print(f"columns in df: {df.shape[1]} | defined: {len(S97_PROFILE) - len(s97_miss)} | definition missing: {len(s97_miss)}")
if s97_miss:
    print("definition missing for:", ", ".join(s97_miss))
print(S97_PROFILE.groupby("definition_source").size().to_string())

### 22.3 Export
`s97_column_glossary.csv` holds one row per column in `df`: section, definition, how to read it, caveat, type, % missing,
summary and the three example values. Use it as the data dictionary sheet of the Tableau workbook.

In [ ]:
export(S97_PROFILE[["column", "section", "definition_source", "status", "definition", "how_to_read", "caveat", "dtype", "pct_missing", "summary"]
                   + [f"ID {i}" for i in S97_IDS]], "s97_column_glossary")
print(f"Section 22 runtime {time.time() - S97_T0:.1f}s")

## Appendix A. Optional automated EDA reports

**Why this appendix exists.** One-line profilers (`ydata-profiling`, `sweetviz`, `dtale`, `autoviz`) are a fair question from
any reviewer: "why not just run a profiler?". This appendix makes them available behind a switch and states honestly what
they add and what they cannot do. It is **off by default** (`RUN_AUTO_EDA = False`) so the main notebook stays fast, offline-safe
and free of extra dependencies. Globals are prefixed `S98_`.

### A.1 What automated EDA is good for, and what it cannot do

| Good for | Cannot do (why this notebook is hand-built) |
|---|---|
| A **fast sanity check** on a new file: types, missing values, constants, duplicates, cardinality | **Domain features**: utilisation, repayment share, implied spend, streaks only exist because we know what a bill, a payment and a limit are |
| Catching a **distribution you forgot to look at** (a spike at zero, a heavy tail, an undocumented code) | **Temporal structure**: a profiler sees `PAY_1..PAY_6` as six unrelated columns, not a reversed six-month sequence with transitions and roll rates |
| A generic correlation matrix and target-vs-feature comparison in minutes | **Interactions and conditioning**: "utilisation matters only inside a status" or "the gap vanishes once behaviour is equal" needs a designed comparison |
| A neutral second opinion that is not biased by the analyst's hypotheses | **Semantics of codes**: it will report −2, 0, and September's code 1 without knowing they are undocumented or month-specific |
| An HTML hand-over for people who want to browse columns | **A story**: no ordering, no annotation, no "so what"; and nothing that maps onto a Tableau sheet |

Use them as a checklist at the start (did we miss a column?), never as the analysis.

### A.2 Switch and helpers
Set `RUN_AUTO_EDA = True` to generate the reports. Libraries are imported inside `try/except`; a `pip install` is attempted
only on Kaggle and only if the internet is reachable. Every step records its status in `S98_status`.

In [ ]:
RUN_AUTO_EDA = False          # <- set True to build the optional HTML reports (adds a few minutes)

S98_status = []
def s98_internet(timeout=3):
    import socket
    try:
        socket.create_connection(("pypi.org", 443), timeout=timeout).close()
        return True
    except Exception:
        return False

def s98_import(module, pip_name=None):
    # import a module; on Kaggle with internet, try one quiet pip install if it is missing
    import importlib, subprocess
    try:
        return importlib.import_module(module)
    except Exception:
        if not (IS_KAGGLE and s98_internet()):
            return None
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name or module], check=True, timeout=600)
            return importlib.import_module(module)
        except Exception:
            return None

# a compact, labelled frame for the profilers: raw columns plus readable labels (no engineered features)
S98_cols = ["LIMIT_BAL", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE"] + PAY_COLS + BILL_COLS + PAMT_COLS + [TARGET]
S98_frame = df[S98_cols].copy()
print(f"RUN_AUTO_EDA = {RUN_AUTO_EDA} | profiling frame {S98_frame.shape} | reports would be written to {OUT}")

### A.3 `ydata-profiling` (minimal mode)
Minimal mode skips the expensive correlation / interaction blocks: types, missing values, distributions and warnings only.
Output: `autoeda_ydata_profile.html` in the output folder.

In [ ]:
if RUN_AUTO_EDA:
    try:
        S98_yp = s98_import("ydata_profiling", "ydata-profiling")
        if S98_yp is None:
            S98_status.append(("ydata-profiling", "skipped: library not available"))
        else:
            S98_t0 = time.time()
            S98_report = S98_yp.ProfileReport(S98_frame, title="Credit default - automated profile (minimal)", minimal=True, progress_bar=False)
            S98_path = OUT / "autoeda_ydata_profile.html"
            S98_report.to_file(str(S98_path), silent=True)
            S98_status.append(("ydata-profiling", f"written {S98_path} in {time.time() - S98_t0:.0f}s"))
    except Exception as e:
        S98_status.append(("ydata-profiling", f"failed: {type(e).__name__}: {str(e)[:120]}"))
else:
    S98_status.append(("ydata-profiling", "not run (RUN_AUTO_EDA = False)"))
print(S98_status[-1])

### A.4 `sweetviz` target comparison
Sweetviz is the one profiler with a useful idea for this problem: every feature shown **against the target**. Output:
`autoeda_sweetviz_target.html`. (Sweetviz is not maintained for NumPy 2; a failure here is caught and reported.)

In [ ]:
if RUN_AUTO_EDA:
    try:
        S98_sv = s98_import("sweetviz")
        if S98_sv is None:
            S98_status.append(("sweetviz", "skipped: library not available"))
        else:
            S98_t0 = time.time()
            S98_svr = S98_sv.analyze(S98_frame, target_feat=TARGET, pairwise_analysis="off")
            S98_path = OUT / "autoeda_sweetviz_target.html"
            S98_svr.show_html(str(S98_path), open_browser=False)
            S98_status.append(("sweetviz", f"written {S98_path} in {time.time() - S98_t0:.0f}s"))
    except Exception as e:
        S98_status.append(("sweetviz", f"failed: {type(e).__name__}: {str(e)[:120]}"))
else:
    S98_status.append(("sweetviz", "not run (RUN_AUTO_EDA = False)"))
print(S98_status[-1])

### A.5 `dtale` and `autoviz` (mention only)
* **D-Tale** (`dtale.show(df)`) starts a local Flask server with a spreadsheet-like explorer. It is interactive, not a report,
  and does not work inside a non-interactive Kaggle run, so it is not executed here. Useful on a laptop for ad-hoc filtering.
* **AutoViz** (`AutoViz_Class().AutoViz(...)`) emits dozens of unlabelled charts chosen by heuristics. It duplicates
  Sections 2-3 with less control over binning and labels, so it is not executed either.

Status of this appendix run:

In [ ]:
S98_status.append(("dtale", "mention only (interactive server, not suitable for a batch notebook)"))
S98_status.append(("autoviz", "mention only (superseded by Sections 2-3)"))
s98_status = pd.DataFrame(S98_status, columns=["tool", "status"])
with pd.option_context("display.max_colwidth", None):
    display(s98_status)

**Observations:** _to be written after the run (only if `RUN_AUTO_EDA` was switched on: note anything the profilers flagged
that Sections 1-19 did not already cover)._

## 21. Tableau exports, data dictionary and dashboard blueprint
This last section turns everything above into a **Tableau-ready data layer**:

| File | Grain | Use |
|---|---|---|
| `tableau_clients.csv` | 1 row per client (30,000) | main data source: raw + labels + bands + every engineered column built in Sections 0–10 |
| `tableau_client_month.csv` | client × month (180,000) | behaviour-over-time sheets (real dates Apr–Sep 2005) |
| `tableau_kpis.csv` | 1 row | KPI tiles (fast, no aggregation needed) |
| `tableau_segment_cube.csv` | 1 row per observed segment | instant interaction explorer; stores **counts**, not only rates |
| `tableau_data_dictionary.csv` | 1 row per exported column | field definitions, Tableau role, aggregation, format |

The section adapts to what ran before it: columns added by earlier sections (`SOW_*`, `FE_*`, `RISK_TIER`, `CLUSTER`, `PERSONA`, anomaly scores, 2-D embeddings, ...) are picked up automatically if present, and it still runs when only Section 0 has run.
Naming convention: **UPPER_SNAKE_CASE** everywhere, original column names kept; ordered categories get a companion `<COL>_SORT` integer so Tableau sorts them correctly (*Sort → Field → `<COL>_SORT` → Minimum*).

**Section guide**

- **What it does:** Packs everything above into clean files for Tableau: a one-row-per-client table, a client-by-month table with real dates, KPI and segment-count tables, a data dictionary, and a dashboard blueprint with ready-to-paste calculated fields.
- **Question it answers:** What exactly should Tableau connect to, and how should each sheet be built so the numbers stay correct?
- **Inputs / new columns:** every scalar column of `df`, including columns added by earlier sections (`FE_*`, `RISK_TIER`, `PERSONA`, ...). No new analysis columns; it adds `RECORD_COUNT`, readable status labels and `<COL>_SORT` keys to the exported files only.
- **How to read the outputs:** mostly tables. The "Observations" check sizes, blanks and that totals match earlier sections. In the segment cube, always compute rate as SUM(defaults) ÷ SUM(clients), never as an average of cell rates.
- **Terms:** grain = what one row stands for (a client, or a client-month). LOD expression = a Tableau calculation fixed at a chosen level of detail. Wilson CI = a 95% confidence range for a rate that behaves well for small groups.
- **Key takeaways:**
  - `tableau_clients.csv`: 30,000 rows x 224 columns; blanks are intentional (ratios are NULL when the bill is zero).
  - KPIs: 6,636 defaults (22.1%); 20.9% of the NT$1.54 bn September exposure sits with defaulters.
  - 2,810 of 3,043 cube cells have under 30 clients, so use a minimum-n filter.
  - Averaging cell rates is wrong: ages 21-25 would show 30.7% instead of the true 26.7%.
- **Where it feeds:** this is the hand-off to the Tableau dashboard; `eda_outputs.zip` holds every figure and table.

### 21.1 Client-level table `tableau_clients.csv`
Collects every scalar column of `df` (helper arrays / long embedding vectors are excluded), adds readable labels for the six monthly status codes, a `RECORD_COUNT` = 1 field, and sort keys for all ordered categoricals. Infinities are replaced by blanks; NaN is written as an empty cell (Tableau reads it as NULL).

In [ ]:
import re, csv, zipfile
from collections import OrderedDict

def _is_scalar_col(s):
    # object columns that hold lists / arrays / dicts are helper arrays, not Tableau fields
    if s.dtype != object:
        return True
    v = s.dropna()
    if v.empty:
        return True
    sample = v.iloc[:200]
    return not sample.map(lambda x: isinstance(x, (list, tuple, dict, set, np.ndarray))).any()

def _tab_name(c):
    n = re.sub(r"[^0-9A-Za-z_]+", "_", str(c)).strip("_").upper()
    return n or "COL"

# ---------- pick columns ----------
excluded = [c for c in df.columns if not _is_scalar_col(df[c])]
# long numbered vectors (e.g. EMB_0..EMB_63) bloat the extract: keep groups of <= 16 members
groups = {}
for c in df.columns:
    m = re.match(r"^(.*?)[_]?(\d+)$", str(c))
    if m and c not in excluded:
        groups.setdefault(m.group(1), []).append(c)
KNOWN_SERIES = {"PAY", "BILL_AMT", "PAY_AMT", "UTIL", "PAY_RATIO", "SPEND", "STATUS_GRP"}
for pre, cols in groups.items():
    if len(cols) > 16 and pre.upper() not in KNOWN_SERIES:
        excluded += cols
keep = [c for c in df.columns if c not in set(excluded)]
tc = df[keep].copy()

# ---------- readable status labels per month ----------
for i in IDX:
    tc[f"STATUS_LABEL_{i}"] = df[f"PAY_{i}"].map(STATUS_LABEL)
tc["RECORD_COUNT"] = 1

# ---------- Tableau-friendly names (originals kept where already clean) ----------
rename, seen = {}, set()
for c in tc.columns:
    n = _tab_name(c)
    base, k = n, 2
    while n in seen:
        n = f"{base}_{k}"; k += 1
    seen.add(n); rename[c] = n
renamed = {a: b for a, b in rename.items() if a != b}
tc = tc.rename(columns=rename)

# ---------- sort keys for ordered categoricals ----------
ORDER_LISTS = {"EDU_LABEL": EDU_ORDER, "MAR_LABEL": MAR_ORDER, "DEFAULT_LABEL": ["No default", "Default"],
               "SEX_LABEL": ["Male", "Female"]}
for i in IDX:
    ORDER_LISTS[f"STATUS_GRP_{i}"] = STATUS_GROUP_ORDER
ORDER_LISTS["STATUS_GRP_SEP"] = STATUS_GROUP_ORDER
for i in IDX:
    ORDER_LISTS[f"STATUS_LABEL_{i}"] = [STATUS_LABEL[k] for k in sorted(STATUS_LABEL)]
sort_cols = []
for c in list(tc.columns):
    s = tc[c]
    if isinstance(s.dtype, pd.CategoricalDtype):
        cats = list(s.cat.categories)
    elif c in ORDER_LISTS:
        cats = ORDER_LISTS[c]
    elif s.dtype == object and c.endswith(("_BAND", "_TIER")) and s.nunique() <= 30:
        cats = sorted(s.dropna().unique(), key=str)       # unknown band from another section: alphabetical fallback
    else:
        continue
    pos = {v: k + 1 for k, v in enumerate(cats)}
    tc[f"{c}_SORT"] = s.astype(object).map(pos).astype("Int64")
    sort_cols.append(f"{c}_SORT")

# ---------- types: categoricals -> text, bool -> 0/1, inf -> NaN ----------
for c in tc.columns:
    if isinstance(tc[c].dtype, pd.CategoricalDtype):
        tc[c] = tc[c].astype(object).where(tc[c].notna(), None)
    elif tc[c].dtype == bool:
        tc[c] = tc[c].astype(int)
num = tc.select_dtypes(include=[np.number]).columns
n_inf = int(np.isinf(tc[num].to_numpy(float, na_value=np.nan)).sum())
tc[num] = tc[num].replace([np.inf, -np.inf], np.nan)
assert not np.isinf(tc[num].to_numpy(float, na_value=np.nan)).any()
assert tc["ID"].is_unique and len(tc) == len(df)

# lead with identifiers / target / labels for a friendlier field list
lead = [c for c in ["ID", "DEFAULT", "DEFAULT_LABEL", "RECORD_COUNT", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL",
                    "AGE", "AGE_BAND", "LIMIT_BAL", "LIMIT_BAND", "STATUS_GRP_SEP"] if c in tc.columns]
tc = tc[lead + [c for c in tc.columns if c not in lead]]

export(tc.round(6), "tableau_clients")
engineered_prefix = ["SOW_", "FE_", "RISK", "CLUSTER", "PERSONA", "ANOM", "UMAP", "TSNE", "PCA", "EMB", "SHAP", "IFOREST", "LOF"]
extra = [c for c in tc.columns if c.startswith(tuple(engineered_prefix))]
print(f"columns: {tc.shape[1]} | sort keys: {len(sort_cols)} | infinities replaced: {n_inf} | "
      f"excluded helper/vector columns: {len(excluded)} | renamed: {len(renamed)}")
print("columns from later sections picked up:", extra if extra else "none (only Section 0 ran)")
na = tc.isna().sum()
print("columns with blanks (NULL in Tableau):")
display(na[na > 0].rename("n_blank").to_frame().T)
tc.head(3)

**Observations:**
- `tableau_clients.csv` has 30,000 rows and 224 columns, with 45 sort keys, no infinities and no excluded columns.
- Blanks are intentional: ratio fields are NULL when the bill is zero, for example `PAY_RATIO_5` (4,708 blanks) and `AVG_PAY_RATIO` (1,406).
- Story use: background only; the main Tableau data source.

### 21.2 Client × month long table `tableau_client_month.csv`
One row per client per statement month with a **real date** (`MONTH_DATE` = 2005-04-01 … 2005-09-01) so Tableau treats it as a continuous time axis. `PAY_AMT` in month *i* settles the bill of month *i+1*; `SPEND` and `PAY_RATIO` are NaN for April (no previous statement in the window).

In [ ]:
MONTH_DATE = {1: "2005-09-01", 2: "2005-08-01", 3: "2005-07-01", 4: "2005-06-01", 5: "2005-05-01", 6: "2005-04-01"}
static = [c for c in ["ID", "DEFAULT", "DEFAULT_LABEL", "SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE", "AGE_BAND",
                      "LIMIT_BAL", "LIMIT_BAND", "STATUS_GRP_SEP", "RISK_TIER", "CLUSTER", "PERSONA"] if c in df.columns]
parts = []
for i in IDX:
    p = df[static].copy()
    p["MONTH_IDX"] = i
    p["MONTH_NUM"] = 7 - i                                   # chronological 1 = Apr ... 6 = Sep
    p["MONTH"] = MONTH[i]
    p["MONTH_DATE"] = MONTH_DATE[i]
    p["STATUS_CODE"] = df[f"PAY_{i}"].values
    p["STATUS_LABEL"] = df[f"PAY_{i}"].map(STATUS_LABEL).values
    p["STATUS_GROUP"] = df[f"STATUS_GRP_{i}"].values
    p["IS_LATE"] = (df[f"PAY_{i}"] >= 1).astype(int).values
    p["BILL_AMT"] = df[f"BILL_AMT{i}"].values
    p["PAY_AMT"] = df[f"PAY_AMT{i}"].values
    p["UTILISATION"] = df[f"UTIL_{i}"].values
    p["SPEND"] = df[f"SPEND_{i}"].values if i < 6 else np.nan
    p["PAY_RATIO"] = df[f"PAY_RATIO_{i}"].values if i < 6 else np.nan
    p["OVER_LIMIT"] = (df[f"BILL_AMT{i}"] > df["LIMIT_BAL"]).astype(int).values
    parts.append(p)
cm = pd.concat(parts, ignore_index=True).sort_values(["ID", "MONTH_NUM"]).reset_index(drop=True)
for c in cm.columns:
    if isinstance(cm[c].dtype, pd.CategoricalDtype):
        sk = cm[c].cat.codes.add(1).astype("Int64").where(cm[c].notna())
        cm[c] = cm[c].astype(object)
        cm[f"{c}_SORT"] = sk
cm["STATUS_GROUP_SORT"] = cm["STATUS_GROUP"].map({g: k + 1 for k, g in enumerate(STATUS_GROUP_ORDER)})
cm["STATUS_CODE_SORT"] = cm["STATUS_CODE"] + 3
nc = cm.select_dtypes(include=[np.number]).columns
cm[nc] = cm[nc].replace([np.inf, -np.inf], np.nan)
assert len(cm) == 6 * len(df)
export(cm.round(6), "tableau_client_month")

# compact check: monthly portfolio trend split by outcome (what the time sheet will show)
chk = cm.groupby(["MONTH_NUM", "MONTH", "DEFAULT_LABEL"]).agg(
    late_share=("IS_LATE", "mean"), avg_util=("UTILISATION", "mean"),
    avg_bill=("BILL_AMT", "mean"), avg_pay=("PAY_AMT", "mean")).unstack("DEFAULT_LABEL")
chk

**Observations:**
- The long table has 180,000 rows (30,000 clients x 6 months) with a real date axis.
- The late share among eventual defaulters climbs from 24.3% in April to 51.7% in September; among non-defaulters it moves from 6.3% to 14.5%.
- Bills are similar (September average NT$48.5k for defaulters vs NT$52.0k), but defaulters pay much less (NT$3,397 vs NT$6,307).
- Story use: the time-series dashboard; "the gap opens in the final months, and it is about payments, not bills".

### 21.3 Pre-aggregated tables: KPIs and the segment cube
* **KPI table** — one row of headline numbers for KPI tiles (no runtime aggregation).
* **Segment cube** — counts over `SEX_LABEL × EDU_LABEL × MAR_LABEL × AGE_BAND × LIMIT_BAND × STATUS_GRP_SEP` (observed combinations only).

> **Why counts and not just rates:** a rate cannot be averaged across cells (AVG of rates weights a 3-client cell like a 3,000-client cell). The cube stores `N_CLIENTS` and `N_DEFAULTS`, so in Tableau the rate is always `SUM([N_DEFAULTS]) / SUM([N_CLIENTS])` and re-aggregates correctly at **any** level of detail (drop any dimension from the view and the rate stays exact). The `RATE` column in the file is for the full-grain cell only — never AVG it.

In [ ]:
exp_sep = df["BILL_AMT1"].clip(lower=0)
kpi = pd.DataFrame([{
    "N_CLIENTS": len(df),
    "N_DEFAULTS": int(df[TARGET].sum()),
    "DEFAULT_RATE": df[TARGET].mean(),
    "DEFAULT_RATE_CI_LO": float(wilson_ci(df[TARGET].sum(), len(df))[0]),
    "DEFAULT_RATE_CI_HI": float(wilson_ci(df[TARGET].sum(), len(df))[1]),
    "TOTAL_LIMIT": df["LIMIT_BAL"].sum(),
    "TOTAL_EXPOSURE_SEP": exp_sep.sum(),
    "DEFAULTED_EXPOSURE_SEP": exp_sep[df[TARGET] == 1].sum(),
    "EXPOSURE_DEFAULT_SHARE": exp_sep[df[TARGET] == 1].sum() / exp_sep.sum(),
    "AVG_LIMIT": df["LIMIT_BAL"].mean(),
    "MEDIAN_LIMIT": df["LIMIT_BAL"].median(),
    "AVG_UTIL": df["AVG_UTIL"].mean(),
    "AVG_UTIL_SEP": df["UTIL_1"].mean(),
    "AVG_AGE": df["AGE"].mean(),
    "SHARE_LATE_SEP": (df["PAY_1"] >= 1).mean(),
    "SHARE_EVER_LATE": (df["N_LATE"] > 0).mean(),
    "TOTAL_PAY_6M": df["TOTAL_PAY"].sum(),
    "SNAPSHOT_DATE": "2005-09-01",
}])
export(kpi, "tableau_kpis")
display(kpi.T.rename(columns={0: "value"}))

CUBE_DIMS = ["SEX_LABEL", "EDU_LABEL", "MAR_LABEL", "AGE_BAND", "LIMIT_BAND", "STATUS_GRP_SEP"]
tmp = df[CUBE_DIMS + [TARGET, "LIMIT_BAL"]].copy()
tmp["EXPOSURE_SEP"] = exp_sep
tmp["UTIL_1"] = df["UTIL_1"]
cube = tmp.groupby(CUBE_DIMS, observed=True).agg(
    N_CLIENTS=(TARGET, "size"), N_DEFAULTS=(TARGET, "sum"), SUM_LIMIT=("LIMIT_BAL", "sum"),
    SUM_EXPOSURE_SEP=("EXPOSURE_SEP", "sum"), SUM_UTIL_SEP=("UTIL_1", "sum")).reset_index()
cube["RATE"] = cube["N_DEFAULTS"] / cube["N_CLIENTS"]
cube["CI_LO"], cube["CI_HI"] = wilson_ci(cube["N_DEFAULTS"], cube["N_CLIENTS"])
cube["LIFT"] = cube["RATE"] / BASE_RATE
for c, order in [("SEX_LABEL", ["Male", "Female"]), ("EDU_LABEL", EDU_ORDER), ("MAR_LABEL", MAR_ORDER),
                 ("AGE_BAND", AGE_LABELS), ("LIMIT_BAND", LIMIT_LABELS), ("STATUS_GRP_SEP", STATUS_GROUP_ORDER)]:
    cube[c] = cube[c].astype(object)
    cube[f"{c}_SORT"] = cube[c].map({v: k + 1 for k, v in enumerate(order)})
assert cube["N_CLIENTS"].sum() == len(df) and cube["N_DEFAULTS"].sum() == df[TARGET].sum()
export(cube.round(6), "tableau_segment_cube")

# proof that re-aggregation is exact: rate by AGE_BAND from the cube == rate from raw data
re_agg = cube.groupby("AGE_BAND").agg(n=("N_CLIENTS", "sum"), d=("N_DEFAULTS", "sum"))
re_agg["rate_from_cube"] = re_agg["d"] / re_agg["n"]
re_agg["rate_from_raw"] = df.assign(AGE_BAND=df["AGE_BAND"].astype(str)).groupby("AGE_BAND")[TARGET].mean().reindex(re_agg.index).values
re_agg["naive_avg_of_cell_rates"] = cube.groupby("AGE_BAND")["RATE"].mean()
re_agg = re_agg.reindex([a for a in AGE_LABELS if a in re_agg.index])
print(f"cube cells: {len(cube):,} | cells with n < 30: {(cube['N_CLIENTS'] < 30).sum():,} "
      f"({cube.loc[cube['N_CLIENTS'] < 30, 'N_CLIENTS'].sum() / len(df):.1%} of clients)")
display(re_agg)
print("Top 10 cube cells by lift (n >= 50):")
cube[cube["N_CLIENTS"] >= 50].sort_values("LIFT", ascending=False).head(10)[CUBE_DIMS + ["N_CLIENTS", "RATE", "CI_LO", "CI_HI", "LIFT"]]

**Observations:**
- KPIs: 6,636 defaults out of 30,000 (22.1%, CI 21.7-22.6%). September exposure is NT$1.54 bn, and 20.9% of it is held by clients who defaulted.
- 33.6% of clients were late at least once and 22.7% were late in September.
- The cube has 3,043 cells, but 2,810 of them have fewer than 30 clients (52.5% of clients), so cube rates need a minimum-n filter.
- Averaging cell rates is wrong: for ages 21-25 it gives 30.7% vs the true 26.7%. Always use SUM(defaults)/SUM(clients).
- Story use: the KPI tiles and the six-way cube explorer.

### 21.3b Optional extra cubes (only if the inputs exist)
A status × month cube (always) and a persona / risk-tier cube when Section 8 produced those columns.

In [ ]:
smc = cm.groupby(["MONTH_NUM", "MONTH", "MONTH_DATE", "STATUS_GROUP", "STATUS_GROUP_SORT"]).agg(
    N_CLIENTS=("DEFAULT", "size"), N_DEFAULTS=("DEFAULT", "sum"), SUM_BILL=("BILL_AMT", "sum"),
    SUM_PAY=("PAY_AMT", "sum")).reset_index()
smc["RATE"] = smc["N_DEFAULTS"] / smc["N_CLIENTS"]
smc["SHARE_OF_MONTH"] = smc["N_CLIENTS"] / smc.groupby("MONTH_NUM")["N_CLIENTS"].transform("sum")
export(smc.round(6), "tableau_status_month_cube")
display(smc.pivot(index="STATUS_GROUP", columns="MONTH_NUM", values="RATE").reindex(STATUS_GROUP_ORDER)
        .rename(columns={7 - i: MONTH[i] for i in IDX}))

seg_dims = [c for c in ["PERSONA", "CLUSTER", "RISK_TIER"] if c in df.columns]
if seg_dims:
    for c in seg_dims:
        t = rate_table(df, c)
        t[c] = t[c].astype(str)
        export(t.round(6), f"tableau_{c.lower()}_rates")
        display(t)
else:
    print("No PERSONA / CLUSTER / RISK_TIER columns in df (Section 8 not run) - persona cube skipped.")

**Observations:**
- The status x month cube shows that late statuses get riskier closer to the outcome: Late 2m defaults at 50.7% in April but 69.1% in September.
- Clean statuses get safer: Revolving falls from 18.8% to 12.8%, and Inactive from 20.0% to 13.2%.
- The persona and risk-tier rate tables match Section 8 exactly (for example C4 53.7%, T5 69.6%).
- Story use: a status-by-month heat table; the persona and tier bars.

### 21.4 Data dictionary `tableau_data_dictionary.csv`
Every exported column of the four main files, with its source (raw / label / band / engineered / section *N*), dtype, definition, suggested **Tableau role** (dimension vs measure), default aggregation and number format. Columns created by Sections 1–10 are recognised by prefix and pointed back to the section that built them.

In [ ]:
MN = {i: MONTH[i] for i in IDX}
DEFS = {
    "ID": ("raw", "Client identifier"),
    "LIMIT_BAL": ("raw", "Credit line in NT$ (individual + family/supplementary)"),
    "SEX": ("raw", "Sex code: 1 male, 2 female"), "EDUCATION": ("raw", "Education code: 1 grad school, 2 university, 3 high school, 4 others, 0/5/6 undocumented"),
    "MARRIAGE": ("raw", "Marital code: 1 married, 2 single, 3 others, 0 undocumented"), "AGE": ("raw", "Age in years"),
    "DEFAULT": ("raw", "Target: 1 = default payment in Oct 2005. AVG = default rate; SUM = number of defaulters"),
    "DEFAULT_LABEL": ("label", "Default / No default"), "SEX_LABEL": ("label", "Male / Female"),
    "EDU_LABEL": ("label", "Education with 0/4/5/6 folded into Other/Unknown"), "MAR_LABEL": ("label", "Married / Single / Other (0 folded into Other)"),
    "AGE_BAND": ("band", "Age band 21-25 ... 61+"), "LIMIT_BAND": ("band", "Credit-limit band <=50k ... >500k"),
    "STATUS_GRP_SEP": ("band", "Coarse repayment status in Sep 2005 (latest month)"),
    "UTIL_BAND": ("band", "Band of AVG_UTIL"), "N_LATE_BAND": ("band", "Band of N_LATE"), "PAYRATIO_BAND": ("band", "Band of PAY_TO_BILL_6M; 'No bill' when nothing billed"),
    "RECORD_COUNT": ("engineered", "Constant 1; SUM = number of clients"),
    "AVG_UTIL": ("engineered", "Mean of UTIL_1..6"), "MAX_UTIL": ("engineered", "Max of UTIL_1..6"),
    "UTIL_TREND": ("engineered", "OLS slope of utilisation Apr->Sep (per month)"),
    "AVG_PAY_RATIO": ("engineered", "Mean of PAY_RATIO_1..5"), "PAY_TO_BILL_6M": ("engineered", "Total paid / total positive billed over 6 months"),
    "AVG_SPEND": ("engineered", "Mean implied spend SPEND_1..5"), "N_LATE": ("engineered", "Months with status >= 1"),
    "MAX_DELAY": ("engineered", "Worst repayment status in 6 months"), "LATE_STREAK_MAX": ("engineered", "Longest run of consecutive late months"),
    "LATE_STREAK_RECENT": ("engineered", "Run of late months ending in Sep"), "N_INACTIVE": ("engineered", "Months with status -2"),
    "N_PAIDFULL": ("engineered", "Months with status -1"), "N_REVOLVE": ("engineered", "Months with status 0"),
    "DELAY_TREND": ("engineered", "Status Sep minus status Apr (positive = deteriorating)"),
    "TOTAL_BILL": ("engineered", "Sum of BILL_AMT1..6"), "TOTAL_PAY": ("engineered", "Sum of PAY_AMT1..6"),
    "AVG_BILL": ("engineered", "Mean bill"), "AVG_PAY": ("engineered", "Mean payment"),
    "BILL_TREND": ("engineered", "OLS slope of bill Apr->Sep (NT$/month)"), "PAY_TREND": ("engineered", "OLS slope of payment Apr->Sep (NT$/month)"),
    "N_ZERO_PAY": ("engineered", "Months with zero payment"), "OVERLIMIT_ANY": ("engineered", "1 if any bill > limit"),
    "N_OVERLIMIT": ("engineered", "Months with bill > limit"), "NEG_BILL_ANY": ("engineered", "1 if any negative bill (credit balance)"),
    # long table
    "MONTH_IDX": ("raw", "Dataset month index (1 = Sep ... 6 = Apr)"), "MONTH_NUM": ("engineered", "Chronological month number (1 = Apr ... 6 = Sep)"),
    "MONTH": ("label", "Month name"), "MONTH_DATE": ("engineered", "First day of statement month (date)"),
    "STATUS_CODE": ("raw", "Repayment status code that month (PAY_i)"), "STATUS_LABEL": ("label", "Readable repayment status"),
    "STATUS_GROUP": ("band", "Coarse status group that month"), "IS_LATE": ("engineered", "1 if status >= 1 that month"),
    "BILL_AMT": ("raw", "Statement balance that month (NT$)"), "PAY_AMT": ("raw", "Amount paid that month (settles previous month's bill)"),
    "UTILISATION": ("engineered", "BILL_AMT / LIMIT_BAL that month"), "SPEND": ("engineered", "Implied new spend that month (NaN in Apr)"),
    "PAY_RATIO": ("engineered", "PAY_AMT / previous BILL_AMT, capped at 5 (NaN in Apr or prev bill <= 0)"),
    "OVER_LIMIT": ("engineered", "1 if bill > limit that month"),
    # kpi / cube
    "N_CLIENTS": ("aggregate", "Number of clients in the row"), "N_DEFAULTS": ("aggregate", "Number of defaulters in the row"),
    "RATE": ("aggregate", "N_DEFAULTS / N_CLIENTS at row grain (do not AVG; recompute)"),
    "CI_LO": ("aggregate", "Wilson 95% lower bound of RATE"), "CI_HI": ("aggregate", "Wilson 95% upper bound of RATE"),
    "LIFT": ("aggregate", "RATE / portfolio default rate"), "SUM_LIMIT": ("aggregate", "Sum of LIMIT_BAL"),
    "SUM_EXPOSURE_SEP": ("aggregate", "Sum of positive Sep balance (BILL_AMT1 clipped at 0)"), "SUM_UTIL_SEP": ("aggregate", "Sum of UTIL_1 (divide by N_CLIENTS for mean)"),
    "DEFAULT_RATE": ("aggregate", "Portfolio default rate"), "DEFAULT_RATE_CI_LO": ("aggregate", "Wilson 95% lower bound"),
    "DEFAULT_RATE_CI_HI": ("aggregate", "Wilson 95% upper bound"), "TOTAL_LIMIT": ("aggregate", "Sum of credit limits"),
    "TOTAL_EXPOSURE_SEP": ("aggregate", "Sum of positive Sep balances"), "DEFAULTED_EXPOSURE_SEP": ("aggregate", "Sep balance held by defaulters"),
    "EXPOSURE_DEFAULT_SHARE": ("aggregate", "Share of Sep balance held by defaulters"), "AVG_LIMIT": ("aggregate", "Mean credit limit"),
    "MEDIAN_LIMIT": ("aggregate", "Median credit limit"), "AVG_UTIL_SEP": ("aggregate", "Mean Sep utilisation"),
    "AVG_AGE": ("aggregate", "Mean age"), "SHARE_LATE_SEP": ("aggregate", "Share of clients late in Sep"),
    "SHARE_EVER_LATE": ("aggregate", "Share of clients late at least once"), "TOTAL_PAY_6M": ("aggregate", "Total paid over 6 months"),
    "SNAPSHOT_DATE": ("aggregate", "Latest statement month"),
}
PATTERNS = [
    (r"^PAY_(\d)$", "raw", lambda i: f"Repayment status code in {MN[int(i)]} 2005 (-2..8)"),
    (r"^BILL_AMT(\d)$", "raw", lambda i: f"Bill statement in {MN[int(i)]} 2005 (NT$)"),
    (r"^PAY_AMT(\d)$", "raw", lambda i: f"Amount paid in {MN[int(i)]} 2005 (settles the previous month's bill)"),
    (r"^UTIL_(\d)$", "engineered", lambda i: f"Utilisation in {MN[int(i)]}: BILL_AMT{i} / LIMIT_BAL"),
    (r"^PAY_RATIO_(\d)$", "engineered", lambda i: f"PAY_AMT{i} / BILL_AMT{int(i)+1} ({MN[int(i)]}), capped at 5"),
    (r"^SPEND_(\d)$", "engineered", lambda i: f"Implied spend in {MN[int(i)]}: BILL_AMT{i} - BILL_AMT{int(i)+1} + PAY_AMT{i}"),
    (r"^STATUS_GRP_(\d)$", "band", lambda i: f"Coarse repayment status group in {MN[int(i)]}"),
    (r"^STATUS_LABEL_(\d)$", "label", lambda i: f"Readable repayment status in {MN[int(i)]}"),
]
SECTION_PREFIX = [("SOW_", "section 6 (Share of Wallet)"), ("FE_", "section 7 (engineered features)"),
                  ("RISK_TIER", "section 8 (segmentation)"), ("CLUSTER", "section 8 (segmentation)"), ("PERSONA", "section 8 (segmentation)"),
                  ("SEG_", "section 8 (segmentation)"), ("ANOM", "section 9 (pattern discovery)"), ("IFOREST", "section 9 (pattern discovery)"),
                  ("LOF", "section 9 (pattern discovery)"), ("UMAP", "section 9 (pattern discovery)"), ("TSNE", "section 9 (pattern discovery)"),
                  ("PCA", "section 9 (pattern discovery)"), ("EMB", "section 9 (pattern discovery)"), ("SHAP", "section 9 (pattern discovery)"),
                  ("Q_", "section 1 (data quality)"), ("DQ_", "section 1 (data quality)"), ("TRAJ", "section 5 (temporal)"), ("T5_", "section 5 (temporal)")]
MONEY = re.compile(r"(LIMIT|BILL|PAY_AMT|SPEND|EXPOSURE|TOTAL_PAY|AVG_PAY$|^AVG_BILL$|SUM_LIMIT|BILL_TREND|PAY_TREND)")
PCT = re.compile(r"(UTIL|RATIO|TO_BILL|RATE|SHARE|CI_LO|CI_HI|_PCT|PCT_)")

def describe_col(col, s, file):
    base = col[:-5] if col.endswith("_SORT") else None
    if base is not None:
        return "sort key", f"Integer sort order for {base} (use as Tableau sort field)", "dimension", "MIN", "0"
    src, dfn = DEFS.get(col, (None, None))
    if src is None:
        for pat, srcp, f in PATTERNS:
            m = re.match(pat, col)
            if m:
                src, dfn = srcp, f(m.group(1)); break
    if src is None:
        for pre, sec in SECTION_PREFIX:
            if col.startswith(pre):
                src, dfn = sec, f"Built in {sec}; see that section's notes"; break
    if src is None:
        orig = {v: k for k, v in renamed.items()}.get(col, col)
        src, dfn = "engineered (earlier section)", f"Added to df by an earlier section (original name: {orig})"
    is_num = pd.api.types.is_numeric_dtype(s)
    is_date = col.endswith("_DATE")
    flag = is_num and s.dropna().isin([0, 1]).all() and col not in ("RECORD_COUNT",)
    code_col = col in ("ID", "SEX", "EDUCATION", "MARRIAGE", "MONTH_IDX", "MONTH_NUM", "STATUS_CODE", "CLUSTER") or re.match(r"^PAY_\d$", col)
    if code_col or is_date or not is_num:
        role = "dimension"
    else:
        role = "measure"
    if is_date:
        agg, fmt = "-", "date (month)"
    elif role == "dimension":
        agg, fmt = "-", "text" if not is_num else "0"
    elif col in ("DEFAULT",):
        agg, fmt = "AVG (= rate) / SUM (= count)", "0.0%"
    elif flag:
        agg, fmt = "AVG (= share) / SUM (= count)", "0.0%"
    elif file == "tableau_kpis":
        agg = "none (single pre-aggregated row; use MIN/ATTR)"
        fmt = "0.0%" if PCT.search(col) else ("NT$ #,##0" if MONEY.search(col) else "#,##0.00")
    elif col.startswith("N_") or col == "RECORD_COUNT":
        agg, fmt = ("SUM" if file != "tableau_clients" or col == "RECORD_COUNT" else "AVG / SUM"), "#,##0"
    elif col.startswith(("SUM_", "TOTAL_")):
        agg, fmt = "SUM", "NT$ #,##0" if MONEY.search(col) else "#,##0.00"
    elif col == "LIFT":
        agg, fmt = "recompute: [Cube Rate] / portfolio rate", "0.00x"
    elif PCT.search(col):
        agg, fmt = ("recompute from N_DEFAULTS / N_CLIENTS" if file == "tableau_segment_cube" else "AVG / MEDIAN"), "0.0%"
    elif MONEY.search(col):
        agg, fmt = "SUM / AVG", "NT$ #,##0"
    else:
        agg, fmt = "AVG / MEDIAN", "#,##0.00"
    return src, dfn, role, agg, fmt

rows = []
for file, frame in [("tableau_clients", tc), ("tableau_client_month", cm), ("tableau_kpis", kpi), ("tableau_segment_cube", cube)]:
    for col in frame.columns:
        src, dfn, role, agg, fmt = describe_col(col, frame[col], file)
        rows.append({"file": file, "column": col, "source": src, "dtype": str(frame[col].dtype),
                     "definition": dfn, "tableau_role": role, "suggested_aggregation": agg, "default_format": fmt,
                     "n_blank": int(frame[col].isna().sum()), "n_distinct": int(frame[col].nunique())})
dd = pd.DataFrame(rows)
export(dd, "tableau_data_dictionary")
print(dd.groupby(["file", "source"]).size().unstack(fill_value=0))
with pd.option_context("display.max_rows", 500, "display.max_colwidth", 90):
    display(dd)

**Observations:**
- The data dictionary has 297 rows covering the four main files, each with its source, Tableau role, aggregation and format.
- `tableau_clients` alone holds 52 Share-of-Wallet fields, 41 engineered fields, 45 sort keys and 25 raw columns.
- Story use: background only.

### 21.5 Dashboard blueprint
Five core dashboards plus an optional sixth (personas). The table generated below lists each sheet with chart type, shelves, filters, actions and the CSV that feeds it, and flags whether the required input was produced in this run.

**Data model.** Main source `tableau_clients` (1 row/client). Relate `tableau_client_month` on `ID` (Tableau relationship, many-to-one) so client filters flow into time sheets. The cube, KPI and status-month tables are standalone sources for speed. `s04_pairwise_rates_long.csv` (Section 4, if present) powers the any-pair explorer; `s06_*` files (Section 6) feed Share of Wallet.

**Calculated fields (copy-paste into Tableau).**
```
// --- core rates (client-level source) ---
[Clients]            = SUM([RECORD_COUNT])                      // or COUNTD([ID])
[Defaulters]         = SUM([DEFAULT])
[Default Rate]       = SUM([DEFAULT]) / SUM([RECORD_COUNT])
[Portfolio Rate]     = {FIXED : SUM([DEFAULT]) / SUM([RECORD_COUNT])}
[Lift vs Portfolio]  = [Default Rate] / [Portfolio Rate]
[Rate Gap (pp)]      = ([Default Rate] - [Portfolio Rate]) * 100

// --- cube source (always re-aggregate counts) ---
[Cube Rate]          = SUM([N_DEFAULTS]) / SUM([N_CLIENTS])
[Cube Lift]          = [Cube Rate] / {FIXED : SUM([N_DEFAULTS]) / SUM([N_CLIENTS])}

// --- Wilson 95% CI (works on either source; swap n/k for cube fields) ---
[n]                  = SUM([RECORD_COUNT])
[k]                  = SUM([DEFAULT])
[Wilson Centre]      = ([k]/[n] + 3.8416/(2*[n])) / (1 + 3.8416/[n])          // 3.8416 = 1.96^2
[Wilson Half]        = 1.96 * SQRT([k]/[n]*(1-[k]/[n])/[n] + 3.8416/(4*SQUARE([n]))) / (1 + 3.8416/[n])
[CI Low]             = [Wilson Centre] - [Wilson Half]
[CI High]            = [Wilson Centre] + [Wilson Half]
[Significant vs Portfolio] = IF [CI Low] > [Portfolio Rate] THEN "Above" ELSEIF [CI High] < [Portfolio Rate] THEN "Below" ELSE "n.s." END
[Small Cell]         = [n] < 30                                 // grey-out / hide unstable cells

// --- LOD (level-of-detail) expressions: computed at a fixed grain regardless of the view ---
[Age Band Rate]      = {FIXED [AGE_BAND] : AVG([DEFAULT])}
[Segment Rate]       = {FIXED [SEX_LABEL], [EDU_LABEL], [MAR_LABEL] : AVG([DEFAULT])}
[Client vs Band]     = [DEFAULT] - {FIXED [LIMIT_BAND] : AVG([DEFAULT])}
[Share of Defaults]  = SUM([DEFAULT]) / {FIXED : SUM([DEFAULT])}
[Exposure Sep]       = IIF([BILL_AMT1] > 0, [BILL_AMT1], 0)
[Exposure at Default]= SUM(IIF([DEFAULT]=1, [Exposure Sep], 0))

// --- parameters for the interaction explorer (client source) ---
// Parameter [Row Dim] / [Col Dim]: string list = Sex, Education, Marriage, Age band, Limit band, Status Sep, Util band, Late band
[Row Field] = CASE [Row Dim] WHEN "Sex" THEN [SEX_LABEL] WHEN "Education" THEN [EDU_LABEL] WHEN "Marriage" THEN [MAR_LABEL]
              WHEN "Age band" THEN [AGE_BAND] WHEN "Limit band" THEN [LIMIT_BAND] WHEN "Status Sep" THEN [STATUS_GRP_SEP]
              WHEN "Util band" THEN [UTIL_BAND] WHEN "Late band" THEN [N_LATE_BAND] END
[Col Field] = same CASE on [Col Dim]
// Pairwise long file (s04_pairwise_rates_long): filter [var_a] = [Row Dim] AND [var_b] = [Col Dim]

// --- time (client-month source) ---
[Late Share]         = AVG([IS_LATE])
[Avg Utilisation]    = AVG([UTILISATION])
[Repayment Rate]     = SUM([PAY_AMT]) / SUM(IIF([BILL_AMT] > 0, [BILL_AMT], 0))
[Status Change]      = AVG([STATUS_CODE]) - LOOKUP(AVG([STATUS_CODE]), -1)     // table calc along MONTH_DATE
```
**Design rules:** default = red `#e34a33`, no default = blue `#2b8cbe`, rate heatmaps sequential, lift diverging centred at 1.0; always show `n` in tooltips; hide or hatch cells with n < 30; sort ordered bands with their `_SORT` field.

In [ ]:
have_pairs = (TAB_DIR / "s04_pairwise_rates_long.csv").exists()
have_sow = any(c.startswith("SOW_") for c in df.columns) or bool(list(TAB_DIR.glob("s06_*.csv")))
have_persona = any(c in df.columns for c in ("PERSONA", "CLUSTER", "RISK_TIER"))
BP = [
 # dashboard, sheet, chart, columns/rows/marks, filters, actions, source
 ("1 Portfolio overview", "KPI tiles", "BANs (big numbers)", "N_CLIENTS, N_DEFAULTS, DEFAULT_RATE (+CI), TOTAL_EXPOSURE_SEP, AVG_LIMIT, AVG_UTIL", "-", "-", "tableau_kpis"),
 ("1 Portfolio overview", "Default split", "Donut / bar", "DEFAULT_LABEL x [Clients]; colour DEFAULT_LABEL", "-", "-", "tableau_clients"),
 ("1 Portfolio overview", "Limit distribution", "Histogram by outcome", "LIMIT_BAL bins x [Clients], colour DEFAULT_LABEL", "SEX/EDU/MAR/AGE_BAND", "filter all sheets", "tableau_clients"),
 ("1 Portfolio overview", "Exposure by limit band", "Bar + rate line (dual axis)", "LIMIT_BAND x SUM exposure; [Default Rate] line", "same", "highlight", "tableau_clients"),
 ("2 Drivers of default", "Rate by driver", "Bar with CI error bars + base-rate ref line", "[Row Field] x [Default Rate], [CI Low]/[CI High]", "parameter Row Dim; demographics", "click bar -> filter detail", "tableau_clients"),
 ("2 Drivers of default", "Status in Sep", "Bar (rate) + bar (n)", "STATUS_GRP_SEP x [Default Rate]; sort STATUS_GRP_SEP_SORT", "demographics", "filter", "tableau_clients"),
 ("2 Drivers of default", "Behavioural bands", "Small multiples", "UTIL_BAND / N_LATE_BAND / PAYRATIO_BAND x [Default Rate]", "demographics", "highlight", "tableau_clients"),
 ("2 Drivers of default", "Information value ranking", "Horizontal bar", "feature x IV (Section 3 export if present)", "-", "-", "s03_* (if exported)"),
 ("3 Interaction explorer", "Any-pair heatmap", "Highlight table", "[Row Field] x [Col Field], colour [Default Rate], label n", "parameters Row Dim / Col Dim; min n", "select cell -> filter client list", "tableau_clients"),
 ("3 Interaction explorer", "Pre-computed pairs", "Heatmap", "level_a x level_b, colour rate/lift", "var_a, var_b quick filters", "-", "s04_pairwise_rates_long" + ("" if have_pairs else " (NOT FOUND in this run)")),
 ("3 Interaction explorer", "Six-way cube", "Treemap / heatmap", "any cube dims; size SUM(N_CLIENTS), colour [Cube Rate]", "all 6 dims", "drill / filter", "tableau_segment_cube"),
 ("4 Behaviour over time", "Status mix by month", "100% stacked area", "MONTH_DATE x SHARE_OF_MONTH, colour STATUS_GROUP", "DEFAULT_LABEL", "-", "tableau_status_month_cube"),
 ("4 Behaviour over time", "Utilisation / late share trend", "Line by outcome", "MONTH_DATE x [Avg Utilisation], [Late Share]; colour DEFAULT_LABEL", "segments via relationship", "-", "tableau_client_month"),
 ("4 Behaviour over time", "Bill vs payment", "Dual-axis line", "MONTH_DATE x SUM(BILL_AMT), SUM(PAY_AMT)", "DEFAULT_LABEL, LIMIT_BAND", "-", "tableau_client_month"),
 ("4 Behaviour over time", "Client trajectory", "Line (spaghetti) + status heat strip", "MONTH_DATE x STATUS_CODE per ID", "ID (from action)", "target of dashboard actions", "tableau_client_month"),
 ("5 Share of Wallet", "SOW distribution by outcome", "Box / histogram", "SOW_* measures x DEFAULT_LABEL", "demographics", "highlight", "tableau_clients / s06_*" + ("" if have_sow else " (SOW not built in this run)")),
 ("5 Share of Wallet", "SOW band default rate", "Bar + CI", "SOW band x [Default Rate]", "LIMIT_BAND", "filter", "tableau_clients"),
 ("5 Share of Wallet", "Spend vs repayment", "Scatter", "AVG_SPEND vs PAY_TO_BILL_6M, colour DEFAULT_LABEL", "LIMIT_BAND", "select -> trajectory", "tableau_clients"),
 ("6 Segments / personas", "Persona profile", "Heatmap of standardised means", "PERSONA x features", "-", "select persona -> filter", "tableau_clients" + ("" if have_persona else " (PERSONA not built in this run)")),
 ("6 Segments / personas", "Persona size vs risk", "Bubble chart", "[Clients] vs [Default Rate], size exposure", "-", "filter", "tableau_persona_rates / tableau_clients"),
]
bp = pd.DataFrame(BP, columns=["dashboard", "sheet", "chart_type", "fields (shelves / marks)", "filters", "actions", "source_csv"])
export(bp, "tableau_dashboard_blueprint")
print(f"pairwise file present: {have_pairs} | SOW present: {have_sow} | personas present: {have_persona}")
with pd.option_context("display.max_colwidth", 80, "display.max_rows", 100):
    display(bp)

### 21.6 Output manifest and one-click download
Every CSV in the Tableau folder (including files exported by Sections 1–10) with rows, columns and size, then a single `eda_outputs.zip` of all figures and tables in the output root (on Kaggle: *Output* panel → `eda_outputs.zip`).

In [ ]:
def _count_rows(path):
    with open(path, "rb") as f:
        return max(0, sum(1 for _ in f) - 1)

man = []
for p in sorted(TAB_DIR.glob("*.csv")):
    with open(p, "r", encoding="utf-8", newline="") as f:
        header = next(csv.reader(f), [])
    man.append({"file": p.name, "section": p.name.split("_")[0] if p.name.startswith("s") and p.name[1:3].isdigit() else "s11",
                "rows": _count_rows(p), "columns": len(header), "size_kb": round(p.stat().st_size / 1024, 1)})
man = pd.DataFrame(man)
n_figs = len(list(FIG_DIR.glob("*.png")))
print(f"{len(man)} CSVs in {TAB_DIR} ({man['size_kb'].sum() / 1024:.1f} MB) | {n_figs} figures in {FIG_DIR}")
display(man.groupby("section").agg(files=("file", "size"), rows=("rows", "sum"), mb=("size_kb", lambda s: round(s.sum() / 1024, 2))))
with pd.option_context("display.max_rows", 300):
    display(man)

zip_path = OUT / "eda_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for d in (FIG_DIR, TAB_DIR):
        for p in sorted(d.rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(OUT))
with zipfile.ZipFile(zip_path) as z:
    n_in = len(z.namelist())
print(f"zip: {zip_path}  {zip_path.stat().st_size / 1024 / 1024:.1f} MB, {n_in} files")

**Observations:**
- 69 CSVs (120.7 MB) and 164 figures were exported; the zip holds 234 files (49.3 MB).
- The final-section `tableau_*` files are the largest group (74.3 MB), followed by Share of Wallet (20.7 MB) and the temporal panel (20.0 MB).
- Story use: background only.

**Next step in Tableau:** connect to `tableau_clients.csv`, relate `tableau_client_month.csv` on `ID`, add the cube / KPI / status-month CSVs as separate sources, paste the calculated fields from 21.5, and set every `_SORT` field as the sort for its band.